# Continuous Spatial Localization Analysis — Version 3

Rebuilt clean copy of the original Continuous_SLv3 notebook. The analysis code is preserved in its original order; cached outputs were removed so the notebook can open reliably in Jupyter.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Enable interactive plotting (optional)
%matplotlib qt

## 1. Trial-Level QC and Descriptive Checks

Subject-average trial-type quality-control pipeline, visual capture, RMSE, confidence-question summaries, and absolute-deviation diagnostics.

In [ ]:
# ============================================================
# SUBJECT-AVERAGE TRIAL-TYPE QC PIPELINE FOR CONTINUOUS SPATIAL TASK
# + VISUAL CAPTURE + RMSE + CONFIDENCE QUESTION ANALYSIS
# ============================================================
#
# Added confidence analysis:
#   1) average yes/no same-source response (post_question) by trialType/condition
#   2) average confidence rating (post_question2) by trialType/condition
#   3) same/different judgment accuracy for audiovisual trials
#   4) confidence for correct vs incorrect judgments
#   5) comparison of self-reported confidence vs response variance
#      as predictors of judgment correctness/accuracy
#
# Expected columns based on your sample file:
#   trialIndex, trialType, condLabel, isPractice, offsetDeg, sideSign, rampType,
#   visualOn, audioOn, t, mouse_x, mouse_y, blob_x, audio_x, audio_pan,
#   post_question, post_question2
#
# ------------------------------------------------------------
# FILL IN:
#   DATA_DIR = r""
# ------------------------------------------------------------

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"   
FILE_PATTERN = "*master_trials*.csv"

OUT_DIR_NAME = "spatial_behavior_qc_outputs_subject_avg_trialType"

# Resampling / plotting
RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0
TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1.0 / RESAMPLE_HZ, 1.0 / RESAMPLE_HZ)

# Trial inclusion
EXCLUDE_PRACTICE = True

# Trial types to keep
SPATIAL_TASK_TYPES = ["congruent", "incongruent_spatial", "incongruent_sweep"]
EXTRA_TRIAL_TYPES = ["audio_only", "visual_only"]
TRIAL_TYPES_TO_KEEP = SPATIAL_TASK_TYPES + EXTRA_TRIAL_TYPES

# Flagging thresholds on SUBJECT-AVERAGED TRIALTYPE traces
LOW_CORR_THRESHOLD = 0.25
LOW_VAR_THRESHOLD = 1e-4
MIN_VALID_TRIALS_PER_TRIALTYPE = 2

# Visual capture settings
MIN_ABS_DISPARITY_FOR_CAPTURE = 0.01

# Aesthetics
FIG_DPI = 140
LW_MEAN = 2.4

# ============================================================
# HELPERS
# ============================================================

def mean_abs_deviation_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return float(np.mean(np.abs(x[mask] - y[mask])))


def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()
    return Path(filepath).stem

def safe_makedirs(path):
    os.makedirs(path, exist_ok=True)

def corr_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def rmse_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return np.sqrt(np.mean((x[mask] - y[mask]) ** 2))

def sem_1d(arr, axis=0):
    arr = np.asarray(arr, dtype=float)
    n = np.sum(np.isfinite(arr), axis=axis)
    sd = np.nanstd(arr, axis=axis, ddof=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = sd / np.sqrt(n)
    return out

def clean_sorted_unique(values):
    values = pd.Series(values).dropna().astype(str).unique().tolist()
    return sorted(values)

def save_figure(fig, outpath):
    fig.tight_layout()
    fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

def plot_mean_with_sem(ax, t, mean, sem, label):
    ax.plot(t, mean, linewidth=LW_MEAN, label=label)
    ax.fill_between(t, mean - sem, mean + sem, alpha=0.22)

def resample_trial(df_trial, time_grid):
    """
    Interpolate one trial to a common time grid.
    Requires columns: t, mouse_x, blob_x, audio_x
    """
    needed = ["t", "mouse_x", "blob_x", "audio_x"]
    for c in needed:
        if c not in df_trial.columns:
            raise ValueError(f"Missing required column: {c}")

    d = df_trial.copy()
    d = d.sort_values("t")
    d = d[np.isfinite(d["t"])]
    d = d.drop_duplicates(subset="t", keep="first")

    if len(d) < 5:
        return None

    t = d["t"].values.astype(float)
    t = t - np.nanmin(t)

    if np.nanmax(t) < 1.0:
        return None

    def _interp(series_name):
        y = d[series_name].values.astype(float)
        ok = np.isfinite(t) & np.isfinite(y)
        if ok.sum() < 3:
            return np.full_like(time_grid, np.nan, dtype=float)
        return np.interp(time_grid, t[ok], y[ok], left=np.nan, right=np.nan)

    out = {
        "t_grid": time_grid.copy(),
        "mouse_x": _interp("mouse_x"),
        "blob_x": _interp("blob_x"),
        "audio_x": _interp("audio_x"),
    }
    return out

def stack_mean_sem(resampled_list, key):
    stack = np.array([r[key] for r in resampled_list], dtype=float)
    return np.nanmean(stack, axis=0), sem_1d(stack, axis=0), stack

def visual_capture_percent(mouse_x, blob_x, audio_x, min_abs_disparity=0.01):
    """
    Data-only visual capture metric:
        mean( (response - audio) / (visual - audio) )
    Valid mainly for fixed-offset spatial trials.
    """
    mouse_x = np.asarray(mouse_x, dtype=float)
    blob_x = np.asarray(blob_x, dtype=float)
    audio_x = np.asarray(audio_x, dtype=float)

    d = blob_x - audio_x
    ok = (
        np.isfinite(mouse_x) &
        np.isfinite(blob_x) &
        np.isfinite(audio_x) &
        (np.abs(d) > min_abs_disparity)
    )
    if ok.sum() == 0:
        return np.nan
    return float(np.nanmean((mouse_x[ok] - audio_x[ok]) / d[ok]))

def parse_yes_no_same_response(x):
    """
    Interprets post_question as the participant's answer to:
    'same source?' => yes=1, no=0
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["y", "yes", "1", "true", "same"]:
        return 1.0
    if s in ["n", "no", "0", "false", "different", "diff"]:
        return 0.0
    return np.nan

def safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return np.nan
    if np.nanstd(x[ok]) < 1e-12 or np.nanstd(y[ok]) < 1e-12:
        return np.nan
    return pearsonr(x[ok], y[ok])[0]

def safe_spearman(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return np.nan
    if np.nanstd(x[ok]) < 1e-12 or np.nanstd(y[ok]) < 1e-12:
        return np.nan
    return spearmanr(x[ok], y[ok]).correlation

# ============================================================
# LOAD FILES
# ============================================================

if not DATA_DIR:
    raise ValueError("Please fill in DATA_DIR before running.")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in: {DATA_DIR}")

out_root = os.path.join(DATA_DIR, OUT_DIR_NAME)
safe_makedirs(out_root)

subject_plot_dir = os.path.join(out_root, "subject_avg_trial_type_plots")
safe_makedirs(subject_plot_dir)

subject_condition_plot_dir = os.path.join(out_root, "subject_avg_condition_plots")
safe_makedirs(subject_condition_plot_dir)

group_plot_dir = os.path.join(out_root, "group_avg_trial_type_plots")
safe_makedirs(group_plot_dir)

flagged_plot_dir = os.path.join(out_root, "flagged_subject_trial_plots")
safe_makedirs(flagged_plot_dir)

metric_plot_dir = os.path.join(out_root, "metric_summary_plots")
safe_makedirs(metric_plot_dir)

capture_dir = os.path.join(out_root, "visual_capture_outputs")
safe_makedirs(capture_dir)

rmse_dir = os.path.join(out_root, "rmse_outputs")
safe_makedirs(rmse_dir)

# NEW
confidence_dir = os.path.join(out_root, "confidence_outputs")
safe_makedirs(confidence_dir)

print(f"Found {len(csv_files)} files")
print(f"Output directory: {out_root}")

# ============================================================
# READ + RESAMPLE ALL TRIALS
# ============================================================

all_trial_rows = []

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    print(f"Loading {subj}: {Path(fpath).name}")

    df = pd.read_csv(fpath)

    required_cols = [
        "trialIndex", "trialType", "condLabel", "isPractice",
        "t", "mouse_x", "blob_x", "audio_x"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"  Skipping {subj}: missing columns {missing}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    df = df[df["trialType"].isin(TRIAL_TYPES_TO_KEEP)].copy()

    if len(df) == 0:
        print(f"  Skipping {subj}: no usable rows after filtering")
        continue

    extra_cols = [c for c in [
        "offsetDeg", "sideSign", "rampType", "visualOn", "audioOn",
        "post_question", "post_question2"
    ] if c in df.columns]

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"], sort=False
    ):
        resampled = resample_trial(df_trial, TIME_GRID)
        if resampled is None:
            continue

        row = {
            "subject": subj,
            "source_file": Path(fpath).name,
            "trialIndex": trial_index,
            "trialType": trial_type,
            "condLabel": cond_label,
            "t_grid": resampled["t_grid"],
            "mouse_x": resampled["mouse_x"],
            "blob_x": resampled["blob_x"],
            "audio_x": resampled["audio_x"],
        }

        if len(df_trial) > 0:
            first_row = df_trial.iloc[0]
            for c in extra_cols:
                row[c] = first_row[c]

        all_trial_rows.append(row)

if len(all_trial_rows) == 0:
    raise RuntimeError("No valid trials were loaded after filtering/resampling.")

trial_df = pd.DataFrame(all_trial_rows)

# Ensure optional columns exist
for c in ["offsetDeg", "sideSign", "rampType", "visualOn", "audioOn", "post_question", "post_question2"]:
    if c not in trial_df.columns:
        trial_df[c] = np.nan

# ============================================================
# TRIAL-LEVEL VISUAL CAPTURE + RMSE + CONFIDENCE METRICS
# ============================================================

trial_df["midpoint_x"] = trial_df.apply(
    lambda r: (np.asarray(r["blob_x"], dtype=float) + np.asarray(r["audio_x"], dtype=float)) / 2.0,
    axis=1
)

trial_df["RMSE_Audio"] = trial_df.apply(
    lambda r: rmse_1d(r["mouse_x"], r["audio_x"]),
    axis=1
)
trial_df["RMSE_Visual"] = trial_df.apply(
    lambda r: rmse_1d(r["mouse_x"], r["blob_x"]),
    axis=1
)
trial_df["RMSE_Midpoint"] = trial_df.apply(
    lambda r: rmse_1d(r["mouse_x"], r["midpoint_x"]) if str(r["trialType"]) in ["congruent", "incongruent_spatial", "incongruent_sweep"] else np.nan,
    axis=1
)

trial_df["visual_capture_percent"] = trial_df.apply(
    lambda r: visual_capture_percent(r["mouse_x"], r["blob_x"], r["audio_x"], MIN_ABS_DISPARITY_FOR_CAPTURE)
    if str(r["trialType"]) == "incongruent_spatial" else np.nan,
    axis=1
)

# NEW: confidence question parsing
trial_df["same_response_yes"] = trial_df["post_question"].apply(parse_yes_no_same_response)
trial_df["different_response_no"] = trial_df["same_response_yes"].apply(lambda x: 1.0 - x if np.isfinite(x) else np.nan)
trial_df["confidence_rating"] = pd.to_numeric(trial_df["post_question2"], errors="coerce")

# ground truth for AV trials only
trial_df["is_av_trial"] = (
    pd.to_numeric(trial_df["audioOn"], errors="coerce").fillna(0).astype(float) == 1
) & (
    pd.to_numeric(trial_df["visualOn"], errors="coerce").fillna(0).astype(float) == 1
)

trial_df["ground_truth_same"] = np.nan
trial_df.loc[(trial_df["is_av_trial"]) & (trial_df["trialType"] == "congruent"), "ground_truth_same"] = 1.0
trial_df.loc[(trial_df["is_av_trial"]) & (trial_df["trialType"].isin(["incongruent_spatial", "incongruent_sweep"])), "ground_truth_same"] = 0.0

trial_df["same_diff_correct"] = np.where(
    np.isfinite(trial_df["same_response_yes"]) & np.isfinite(trial_df["ground_truth_same"]),
    (trial_df["same_response_yes"] == trial_df["ground_truth_same"]).astype(float),
    np.nan
)

# trial-level response variance
trial_df["response_variance_trial"] = trial_df["mouse_x"].apply(lambda x: np.nanvar(np.asarray(x, dtype=float)))

trial_metrics_extra_csv = os.path.join(out_root, "trial_level_qc_plus_capture_rmse_confidence_metrics.csv")
trial_df.drop(columns=["t_grid", "mouse_x", "blob_x", "audio_x", "midpoint_x"]).to_csv(trial_metrics_extra_csv, index=False)

# ============================================================
# BUILD SUBJECT-AVERAGED trialType TIMESERIES
# ============================================================

subject_trialtype_rows = []

for (subj, trial_type), sub in trial_df.groupby(["subject", "trialType"], sort=False):
    if len(sub) < MIN_VALID_TRIALS_PER_TRIALTYPE:
        continue

    resampled_list = []
    for _, row in sub.iterrows():
        resampled_list.append({
            "mouse_x": row["mouse_x"],
            "blob_x": row["blob_x"],
            "audio_x": row["audio_x"],
        })

    mouse_mean, mouse_sem, mouse_stack = stack_mean_sem(resampled_list, "mouse_x")
    blob_mean, blob_sem, blob_stack = stack_mean_sem(resampled_list, "blob_x")
    audio_mean, audio_sem, audio_stack = stack_mean_sem(resampled_list, "audio_x")

    subject_trialtype_rows.append({
        "subject": subj,
        "trialType": trial_type,
        "n_trials": len(resampled_list),
        "t_grid": TIME_GRID.copy(),

        "mouse_mean": mouse_mean,
        "mouse_sem": mouse_sem,
        "blob_mean": blob_mean,
        "blob_sem": blob_sem,
        "audio_mean": audio_mean,
        "audio_sem": audio_sem,

        # metrics computed ON THE SUBJECT-AVERAGED trialType traces
        "corr_mouse_visual": corr_1d(mouse_mean, blob_mean),
        "corr_mouse_audio": corr_1d(mouse_mean, audio_mean),
        "rmse_mouse_visual": rmse_1d(mouse_mean, blob_mean),
        "rmse_mouse_audio": rmse_1d(mouse_mean, audio_mean),
        "var_mouse": np.nanvar(mouse_mean),
    })

subject_trialtype_df = pd.DataFrame(subject_trialtype_rows)

if len(subject_trialtype_df) == 0:
    raise RuntimeError("No subject-level trialType averages were available. Check trialType labels and filters.")

subject_trialtype_metrics_csv = os.path.join(out_root, "subject_avg_trialType_metrics.csv")
subject_trialtype_df.drop(
    columns=["t_grid", "mouse_mean", "mouse_sem", "blob_mean", "blob_sem", "audio_mean", "audio_sem"]
).to_csv(subject_trialtype_metrics_csv, index=False)

# ============================================================
# SUBJECT-LEVEL PLOTS (MAIN FOCUS)
# ============================================================

subjects = clean_sorted_unique(subject_trialtype_df["subject"].values)

for subj in subjects:
    sub = subject_trialtype_df[subject_trialtype_df["subject"] == subj].copy()
    trial_types = clean_sorted_unique(sub["trialType"].values)

    n_panels = len(trial_types)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.7 * ncols, 4.0 * nrows), squeeze=False)
    axes = axes.flatten()

    for ax, trial_type in zip(axes, trial_types):
        row = sub[sub["trialType"] == trial_type].iloc[0]

        plot_mean_with_sem(ax, row["t_grid"], row["blob_mean"], row["blob_sem"], "Visual xV(t)")
        plot_mean_with_sem(ax, row["t_grid"], row["audio_mean"], row["audio_sem"], "Audio xA(t)")
        plot_mean_with_sem(ax, row["t_grid"], row["mouse_mean"], row["mouse_sem"], "Response mouse_x(t)")

        title = (
            f"{subj} | {trial_type}\n"
            f"n={int(row['n_trials'])} | "
            f"r(resp,V)={row['corr_mouse_visual']:.2f}, "
            f"r(resp,A)={row['corr_mouse_audio']:.2f}"
        )
        ax.set_title(title)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Position")
        ax.axhline(0, linestyle="--", linewidth=0.8, alpha=0.5)
        ax.legend(loc="best", fontsize=8)

    for ax in axes[n_panels:]:
        ax.axis("off")

    outpath = os.path.join(subject_plot_dir, f"{subj}_subject_avg_trialType.png")
    save_figure(fig, outpath)

print(f"Saved subject_avg_trialType plots -> {subject_plot_dir}")

# ============================================================
# SUBJECT-LEVEL CONDITION AVERAGE PLOTS
# ============================================================

for subj in subjects:
    sub_df = trial_df[trial_df["subject"] == subj].copy()
    cond_values = clean_sorted_unique(sub_df["condLabel"].values)

    if len(cond_values) == 0:
        continue

    n_panels = len(cond_values)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.7 * ncols, 4.0 * nrows), squeeze=False)
    axes = axes.flatten()

    for ax, cond_value in zip(axes, cond_values):
        cond_sub = sub_df[sub_df["condLabel"] == cond_value].copy()
        if len(cond_sub) == 0:
            ax.axis("off")
            continue

        resampled_list = []
        for _, row in cond_sub.iterrows():
            resampled_list.append({
                "mouse_x": row["mouse_x"],
                "blob_x": row["blob_x"],
                "audio_x": row["audio_x"],
            })

        mouse_mean, mouse_sem, _ = stack_mean_sem(resampled_list, "mouse_x")
        blob_mean, blob_sem, _ = stack_mean_sem(resampled_list, "blob_x")
        audio_mean, audio_sem, _ = stack_mean_sem(resampled_list, "audio_x")

        corr_v = corr_1d(mouse_mean, blob_mean)
        corr_a = corr_1d(mouse_mean, audio_mean)

        plot_mean_with_sem(ax, TIME_GRID, blob_mean, blob_sem, "Visual xV(t)")
        plot_mean_with_sem(ax, TIME_GRID, audio_mean, audio_sem, "Audio xA(t)")
        plot_mean_with_sem(ax, TIME_GRID, mouse_mean, mouse_sem, "Response mouse_x(t)")

        title = (
            f"{subj} | {cond_value}\n"
            f"n={len(resampled_list)} | "
            f"r(resp,V)={corr_v:.2f}, "
            f"r(resp,A)={corr_a:.2f}"
        )
        ax.set_title(title)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Position")
        ax.axhline(0, linestyle="--", linewidth=0.8, alpha=0.5)
        ax.legend(loc="best", fontsize=8)

    for ax in axes[n_panels:]:
        ax.axis("off")

    outpath = os.path.join(subject_condition_plot_dir, f"{subj}_subject_avg_condition.png")
    save_figure(fig, outpath)

print(f"Saved subject_avg_condition plots -> {subject_condition_plot_dir}")

# ============================================================
# GROUP-LEVEL trialType AVERAGES
# ============================================================

trial_types_all = clean_sorted_unique(subject_trialtype_df["trialType"].values)

n_panels = len(trial_types_all)
ncols = 3
nrows = int(np.ceil(n_panels / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.7 * ncols, 4.0 * nrows), squeeze=False)
axes = axes.flatten()

for ax, trial_type in zip(axes, trial_types_all):
    sub = subject_trialtype_df[subject_trialtype_df["trialType"] == trial_type].copy()
    if len(sub) == 0:
        ax.axis("off")
        continue

    mouse_stack = np.stack(sub["mouse_mean"].values)
    blob_stack = np.stack(sub["blob_mean"].values)
    audio_stack = np.stack(sub["audio_mean"].values)

    mouse_mean = np.nanmean(mouse_stack, axis=0)
    mouse_sem = sem_1d(mouse_stack, axis=0)

    blob_mean = np.nanmean(blob_stack, axis=0)
    blob_sem = sem_1d(blob_stack, axis=0)

    audio_mean = np.nanmean(audio_stack, axis=0)
    audio_sem = sem_1d(audio_stack, axis=0)

    plot_mean_with_sem(ax, TIME_GRID, blob_mean, blob_sem, "Visual xV(t)")
    plot_mean_with_sem(ax, TIME_GRID, audio_mean, audio_sem, "Audio xA(t)")
    plot_mean_with_sem(ax, TIME_GRID, mouse_mean, mouse_sem, "Response mouse_x(t)")

    ax.set_title(f"{trial_type}\nAverage within subject, then across subjects (n={len(sub)})")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Position")
    ax.axhline(0, linestyle="--", linewidth=0.8, alpha=0.5)
    ax.legend(loc="best", fontsize=8)

for ax in axes[n_panels:]:
    ax.axis("off")

save_figure(fig, os.path.join(group_plot_dir, "group_avg_trialType_subject_average_then_group.png"))
print(f"Saved group trialType plots -> {group_plot_dir}")

# ============================================================
# FLAGGING BASED ON SUBJECT-AVERAGED trialType TIMESERIES
# ============================================================

flag_rows = []

for subj in subjects:
    sub = subject_trialtype_df[subject_trialtype_df["subject"] == subj].copy()
    reason_list = []
    bad_trialtypes = []

    overall_corr_v = np.nanmean(sub["corr_mouse_visual"])
    overall_corr_a = np.nanmean(sub["corr_mouse_audio"])
    overall_var = np.nanmean(sub["var_mouse"])

    if np.isfinite(overall_corr_v) and np.isfinite(overall_corr_a):
        if overall_corr_v < LOW_CORR_THRESHOLD and overall_corr_a < LOW_CORR_THRESHOLD:
            reason_list.append("overall low corr to both visual and audio across subject-averaged trialTypes")

    if np.isfinite(overall_var) and overall_var < LOW_VAR_THRESHOLD:
        reason_list.append("overall near-zero variance across subject-averaged trialTypes")

    for _, row in sub.iterrows():
        low_both = (
            np.isfinite(row["corr_mouse_visual"]) and
            np.isfinite(row["corr_mouse_audio"]) and
            row["corr_mouse_visual"] < LOW_CORR_THRESHOLD and
            row["corr_mouse_audio"] < LOW_CORR_THRESHOLD
        )
        low_var = np.isfinite(row["var_mouse"]) and row["var_mouse"] < LOW_VAR_THRESHOLD

        if low_both or low_var:
            bad_trialtypes.append(str(row["trialType"]))

    if len(bad_trialtypes) > 0:
        reason_list.append("suspicious trialType(s): " + ", ".join(sorted(set(bad_trialtypes))))

    flag_rows.append({
        "subject": subj,
        "flagged": int(len(reason_list) > 0),
        "flag_reason": " | ".join(reason_list) if reason_list else "",
        "mean_corr_mouse_visual_across_trialTypes": overall_corr_v,
        "mean_corr_mouse_audio_across_trialTypes": overall_corr_a,
        "mean_var_mouse_across_trialTypes": overall_var,
        "n_trialTypes_contributing": len(sub),
    })

flag_df = pd.DataFrame(flag_rows).sort_values(["flagged", "subject"], ascending=[False, True])
flag_csv = os.path.join(out_root, "flagged_subjects_from_subject_avg_trialType.csv")
flag_df.to_csv(flag_csv, index=False)

subject_summary_df = flag_df.copy()
subject_summary_csv = os.path.join(out_root, "subject_summary_from_subject_avg_trialType.csv")
subject_summary_df.to_csv(subject_summary_csv, index=False)

print(f"Saved subject summary -> {subject_summary_csv}")
print(f"Saved flagged-subject table -> {flag_csv}")

# ============================================================
# METRIC SUMMARY PLOTS
# ============================================================

# deviation plot
deviation_plot_rows = []

for subj in subjects:
    subj_trials = trial_df[trial_df["subject"] == subj].copy()

    # Congruent: deviation between response and visual
    cong_trials = subj_trials[subj_trials["trialType"] == "congruent"].copy()
    cong_trial_devs = [
        mean_abs_deviation_1d(row["mouse_x"], row["blob_x"])
        for _, row in cong_trials.iterrows()
    ]
    congruent_deviation = np.nanmean(cong_trial_devs) if len(cong_trial_devs) > 0 else np.nan

    # Audio only: deviation between response and audio
    aud_trials = subj_trials[subj_trials["trialType"] == "audio_only"].copy()
    aud_trial_devs = [
        mean_abs_deviation_1d(row["mouse_x"], row["audio_x"])
        for _, row in aud_trials.iterrows()
    ]
    audio_only_deviation = np.nanmean(aud_trial_devs) if len(aud_trial_devs) > 0 else np.nan

    deviation_plot_rows.append({
        "subject": subj,
        "congruent_deviation": congruent_deviation,
        "audio_only_deviation": audio_only_deviation,
    })

deviation_plot_df = pd.DataFrame(deviation_plot_rows).sort_values("subject").reset_index(drop=True)

x = np.arange(len(deviation_plot_df))
w = 0.38

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, deviation_plot_df["congruent_deviation"], width=w, label="Congruent: mean |response - visual|")
ax.bar(x + w/2, deviation_plot_df["audio_only_deviation"], width=w, label="Audio only: mean |response - audio|")
ax.set_xticks(x)
ax.set_xticklabels(deviation_plot_df["subject"], rotation=90)
ax.set_ylabel("Mean absolute deviation")
ax.set_title("Average congruent and audio_only deviation by subject")
ax.legend()
save_figure(fig, os.path.join(metric_plot_dir, "subject_congruent_audio_only_deviation.png"))

# variance plot
plot_df = flag_df.sort_values("subject").copy()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(np.arange(len(plot_df)), plot_df["mean_var_mouse_across_trialTypes"])
ax.axhline(
    LOW_VAR_THRESHOLD,
    linestyle="--",
    linewidth=1.2,
    color="k",
    alpha=0.7,
    label=f"Low variance threshold = {LOW_VAR_THRESHOLD}"
)
ax.set_xticks(np.arange(len(plot_df)))
ax.set_xticklabels(plot_df["subject"], rotation=90)
ax.set_ylabel("Mean response variance")
ax.set_title("Subject response variance from subject-averaged trialType traces")
ax.legend()
save_figure(fig, os.path.join(metric_plot_dir, "subject_avg_trialType_variance.png"))

print(f"Saved metric summary plots -> {metric_plot_dir}")

# ============================================================
# FLAGGED SUBJECTS: PLOT ALL INDIVIDUAL TRIALS FOR CONFIRMATION
# ============================================================

flagged_subjects = flag_df.loc[flag_df["flagged"] == 1, "subject"].tolist()

for subj in flagged_subjects:
    sub_df = trial_df[trial_df["subject"] == subj].copy()
    if len(sub_df) == 0:
        continue

    trial_types = clean_sorted_unique(sub_df["trialType"].values)
    n_panels = len(trial_types)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.8 * ncols, 4.0 * nrows), squeeze=False)
    axes = axes.flatten()

    for ax, trial_type in zip(axes, trial_types):
        tt_sub = sub_df[sub_df["trialType"] == trial_type].copy()
        if len(tt_sub) == 0:
            ax.axis("off")
            continue

        for _, row in tt_sub.iterrows():
            ax.plot(TIME_GRID, row["blob_x"], alpha=0.18, linewidth=1.0)
            ax.plot(TIME_GRID, row["audio_x"], alpha=0.18, linewidth=1.0)
            ax.plot(TIME_GRID, row["mouse_x"], alpha=0.18, linewidth=1.0)

        avg_row = subject_trialtype_df[
            (subject_trialtype_df["subject"] == subj) &
            (subject_trialtype_df["trialType"] == trial_type)
        ]
        if len(avg_row) > 0:
            avg_row = avg_row.iloc[0]
            ax.plot(TIME_GRID, avg_row["blob_mean"], linewidth=2.4, label="Visual mean")
            ax.plot(TIME_GRID, avg_row["audio_mean"], linewidth=2.4, label="Audio mean")
            ax.plot(TIME_GRID, avg_row["mouse_mean"], linewidth=2.4, label="Response mean")

        ax.set_title(f"{subj} | {trial_type}\nAll individual trials + subject-average")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Position")
        ax.axhline(0, linestyle="--", linewidth=0.8, alpha=0.5)
        ax.legend(loc="best", fontsize=8)

    for ax in axes[n_panels:]:
        ax.axis("off")

    reason = flag_df.loc[flag_df["subject"] == subj, "flag_reason"].values[0]
    fig.suptitle(f"Flagged subject: {subj}\nReason: {reason}", y=1.02, fontsize=12)

    outpath = os.path.join(flagged_plot_dir, f"{subj}_flagged_all_trials_by_trialType.png")
    save_figure(fig, outpath)

print(f"Saved flagged-subject confirmation plots -> {flagged_plot_dir}")

# ============================================================
# VISUAL CAPTURE ANALYSIS
# ============================================================

capture_trial_df = trial_df[trial_df["trialType"] == "incongruent_spatial"].copy()

capture_trial_csv = os.path.join(capture_dir, "visual_capture_by_trial.csv")
capture_trial_df[
    ["subject", "source_file", "trialIndex", "trialType", "condLabel", "offsetDeg", "sideSign", "visual_capture_percent"]
].to_csv(capture_trial_csv, index=False)

capture_cond_df = (
    capture_trial_df
    .groupby("condLabel", as_index=False)
    .agg(
        n_trials=("visual_capture_percent", lambda x: np.sum(np.isfinite(x))),
        n_subjects=("subject", "nunique"),
        visual_capture_mean=("visual_capture_percent", "mean"),
        visual_capture_sem=("visual_capture_percent", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
    )
)

capture_cond_csv = os.path.join(capture_dir, "visual_capture_by_condition.csv")
capture_cond_df.to_csv(capture_cond_csv, index=False)

capture_offset_df = (
    capture_trial_df
    .groupby("offsetDeg", as_index=False)
    .agg(
        n_trials=("visual_capture_percent", lambda x: np.sum(np.isfinite(x))),
        n_subjects=("subject", "nunique"),
        visual_capture_mean=("visual_capture_percent", "mean"),
        visual_capture_sem=("visual_capture_percent", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
    )
    .sort_values("offsetDeg")
)

capture_offset_csv = os.path.join(capture_dir, "visual_capture_by_offset.csv")
capture_offset_df.to_csv(capture_offset_csv, index=False)

if len(capture_cond_df) > 0:
    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(capture_cond_df))
    ax.bar(x, capture_cond_df["visual_capture_mean"], yerr=capture_cond_df["visual_capture_sem"], capsize=4)
    ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.axhline(0.5, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.axhline(1.0, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(capture_cond_df["condLabel"], rotation=25)
    ax.set_ylabel("Visual capture = mean((resp - audio)/(visual - audio))")
    ax.set_title("Visual capture by fixed-offset condition")
    save_figure(fig, os.path.join(capture_dir, "visual_capture_by_condition.png"))

if len(capture_offset_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(
        capture_offset_df["offsetDeg"],
        capture_offset_df["visual_capture_mean"],
        yerr=capture_offset_df["visual_capture_sem"],
        marker="o",
        linewidth=2
    )
    ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.axhline(0.5, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.axhline(1.0, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.set_xlabel("Offset magnitude (deg)")
    ax.set_ylabel("Visual capture")
    ax.set_title("Visual capture by offset magnitude")
    save_figure(fig, os.path.join(capture_dir, "visual_capture_by_offset.png"))

subject_capture_offset_df = (
    capture_trial_df
    .groupby(["subject", "offsetDeg"], as_index=False)
    .agg(
        visual_capture_mean=("visual_capture_percent", "mean")
    )
    .sort_values(["subject", "offsetDeg"])
)

subject_capture_offset_csv = os.path.join(capture_dir, "subject_visual_capture_by_offset.csv")
subject_capture_offset_df.to_csv(subject_capture_offset_csv, index=False)

for subj in clean_sorted_unique(subject_capture_offset_df["subject"].values):
    sub = subject_capture_offset_df[subject_capture_offset_df["subject"] == subj].copy()
    if len(sub) == 0:
        continue
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(sub["offsetDeg"], sub["visual_capture_mean"], marker="o", linewidth=2)
    ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.axhline(0.5, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.axhline(1.0, linestyle="--", linewidth=1.0, alpha=0.6)
    ax.set_xlabel("Offset magnitude (deg)")
    ax.set_ylabel("Visual capture")
    ax.set_title(f"{subj}: visual capture by offset")
    save_figure(fig, os.path.join(capture_dir, f"{subj}_visual_capture_by_offset.png"))

# ============================================================
# RMSE ANALYSIS
# ============================================================

rmse_trial_csv = os.path.join(rmse_dir, "rmse_by_trial.csv")
trial_df[
    ["subject", "source_file", "trialIndex", "trialType", "condLabel", "offsetDeg", "sideSign",
     "RMSE_Audio", "RMSE_Visual", "RMSE_Midpoint"]
].to_csv(rmse_trial_csv, index=False)

rmse_cond_df = (
    trial_df
    .groupby(["trialType", "condLabel"], as_index=False)
    .agg(
        n_trials=("trialIndex", "count"),
        n_subjects=("subject", "nunique"),
        RMSE_Audio_mean=("RMSE_Audio", "mean"),
        RMSE_Audio_sem=("RMSE_Audio", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
        RMSE_Visual_mean=("RMSE_Visual", "mean"),
        RMSE_Visual_sem=("RMSE_Visual", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
        RMSE_Midpoint_mean=("RMSE_Midpoint", "mean"),
        RMSE_Midpoint_sem=("RMSE_Midpoint", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
    )
)

rmse_cond_csv = os.path.join(rmse_dir, "rmse_by_condition.csv")
rmse_cond_df.to_csv(rmse_cond_csv, index=False)

if len(rmse_cond_df) > 0:
    plot_df = rmse_cond_df.copy()
    x = np.arange(len(plot_df))
    w = 0.38

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - w/2, plot_df["RMSE_Audio_mean"], yerr=plot_df["RMSE_Audio_sem"], width=w, capsize=4, label="RMSE to Audio")
    ax.bar(x + w/2, plot_df["RMSE_Visual_mean"], yerr=plot_df["RMSE_Visual_sem"], width=w, capsize=4, label="RMSE to Visual")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["condLabel"], rotation=25)
    ax.set_ylabel("RMSE")
    ax.set_title("RMSE to audio and visual by condition")
    ax.legend()
    save_figure(fig, os.path.join(rmse_dir, "rmse_audio_visual_by_condition.png"))

rmse_multi_df = rmse_cond_df[rmse_cond_df["trialType"].isin(["congruent", "incongruent_spatial", "incongruent_sweep"])].copy()
if len(rmse_multi_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(rmse_multi_df))
    ax.bar(x, rmse_multi_df["RMSE_Midpoint_mean"], yerr=rmse_multi_df["RMSE_Midpoint_sem"], capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(rmse_multi_df["condLabel"], rotation=25)
    ax.set_ylabel("RMSE to Midpoint")
    ax.set_title("RMSE to midpoint by multisensory condition")
    save_figure(fig, os.path.join(rmse_dir, "rmse_midpoint_by_condition.png"))

subject_rmse_df = (
    trial_df
    .groupby("subject", as_index=False)
    .agg(
        RMSE_Audio_mean=("RMSE_Audio", "mean"),
        RMSE_Visual_mean=("RMSE_Visual", "mean"),
        RMSE_Midpoint_mean=("RMSE_Midpoint", "mean"),
    )
)

subject_rmse_csv = os.path.join(rmse_dir, "subject_rmse_summary.csv")
subject_rmse_df.to_csv(subject_rmse_csv, index=False)

subject_offset_rmse_df = (
    trial_df[trial_df["trialType"] == "incongruent_spatial"]
    .groupby(["subject", "offsetDeg"], as_index=False)
    .agg(
        RMSE_Audio_mean=("RMSE_Audio", "mean"),
        RMSE_Visual_mean=("RMSE_Visual", "mean"),
        RMSE_Midpoint_mean=("RMSE_Midpoint", "mean"),
    )
    .sort_values(["subject", "offsetDeg"])
)

subject_offset_rmse_csv = os.path.join(rmse_dir, "subject_fixed_offset_rmse_by_offset.csv")
subject_offset_rmse_df.to_csv(subject_offset_rmse_csv, index=False)

for subj in clean_sorted_unique(subject_offset_rmse_df["subject"].values):
    sub = subject_offset_rmse_df[subject_offset_rmse_df["subject"] == subj].copy()
    if len(sub) == 0:
        continue

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(sub["offsetDeg"], sub["RMSE_Audio_mean"], marker="o", linewidth=2, label="RMSE to Audio")
    ax.plot(sub["offsetDeg"], sub["RMSE_Visual_mean"], marker="o", linewidth=2, label="RMSE to Visual")
    ax.plot(sub["offsetDeg"], sub["RMSE_Midpoint_mean"], marker="o", linewidth=2, label="RMSE to Midpoint")
    ax.set_xlabel("Offset magnitude (deg)")
    ax.set_ylabel("RMSE")
    ax.set_title(f"{subj}: fixed-offset RMSE by offset")
    ax.legend()
    save_figure(fig, os.path.join(rmse_dir, f"{subj}_fixed_offset_rmse_by_offset.png"))

# ============================================================
# NEW: CONFIDENCE QUESTION ANALYSIS
# ============================================================

# Save trial-level confidence metrics
confidence_trial_csv = os.path.join(confidence_dir, "confidence_by_trial.csv")
trial_df[
    [
        "subject", "source_file", "trialIndex", "trialType", "condLabel",
        "visualOn", "audioOn", "offsetDeg", "sideSign",
        "post_question", "post_question2",
        "same_response_yes", "different_response_no", "confidence_rating",
        "is_av_trial", "ground_truth_same", "same_diff_correct",
        "response_variance_trial"
    ]
].to_csv(confidence_trial_csv, index=False)

# Condition-level summary of same/different and confidence
confidence_cond_df = (
    trial_df
    .groupby(["trialType", "condLabel"], as_index=False)
    .agg(
        n_trials=("trialIndex", "count"),
        n_subjects=("subject", "nunique"),
        n_valid_same_response=("same_response_yes", lambda x: np.sum(np.isfinite(x))),
        yes_same_rate=("same_response_yes", "mean"),
        no_different_rate=("different_response_no", "mean"),
        n_valid_confidence=("confidence_rating", lambda x: np.sum(np.isfinite(x))),
        confidence_mean=("confidence_rating", "mean"),
        confidence_sem=("confidence_rating", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
        n_valid_accuracy=("same_diff_correct", lambda x: np.sum(np.isfinite(x))),
        same_diff_accuracy=("same_diff_correct", "mean"),
        accuracy_sem=("same_diff_correct", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
        response_variance_mean=("response_variance_trial", "mean"),
        response_variance_sem=("response_variance_trial", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
    )
)

confidence_cond_csv = os.path.join(confidence_dir, "confidence_by_condition.csv")
confidence_cond_df.to_csv(confidence_cond_csv, index=False)

# Confidence by correctness (AV trials only)
confidence_correctness_df = (
    trial_df[np.isfinite(trial_df["same_diff_correct"])]
    .groupby(["trialType", "condLabel"], as_index=False)
    .agg(
        confidence_mean_correct=("confidence_rating", lambda x: np.nanmean(x[trial_df.loc[x.index, "same_diff_correct"] == 1]) if np.sum(trial_df.loc[x.index, "same_diff_correct"] == 1) > 0 else np.nan),
        confidence_mean_incorrect=("confidence_rating", lambda x: np.nanmean(x[trial_df.loc[x.index, "same_diff_correct"] == 0]) if np.sum(trial_df.loc[x.index, "same_diff_correct"] == 0) > 0 else np.nan),
    )
)

confidence_correctness_csv = os.path.join(confidence_dir, "confidence_correct_vs_incorrect_by_condition.csv")
confidence_correctness_df.to_csv(confidence_correctness_csv, index=False)

# Subject-condition summary for comparing confidence vs variance as confidence metrics
subject_cond_conf_df = (
    trial_df
    .groupby(["subject", "trialType", "condLabel"], as_index=False)
    .agg(
        n_trials=("trialIndex", "count"),
        mean_confidence=("confidence_rating", "mean"),
        mean_yes_same=("same_response_yes", "mean"),
        mean_accuracy=("same_diff_correct", "mean"),
        mean_response_variance=("response_variance_trial", "mean"),
    )
)

subject_cond_conf_csv = os.path.join(confidence_dir, "subject_condition_confidence_summary.csv")
subject_cond_conf_df.to_csv(subject_cond_conf_csv, index=False)

# Compare self-report confidence vs inverse variance in predicting accuracy
# Restrict to AV rows where accuracy is defined
compare_df = subject_cond_conf_df[np.isfinite(subject_cond_conf_df["mean_accuracy"])].copy()
compare_df["neg_mean_response_variance"] = -compare_df["mean_response_variance"]

comparison_rows = []

if len(compare_df) >= 3:
    comparison_rows.append({
        "level": "subject_condition",
        "metric_x": "mean_confidence",
        "metric_y": "mean_accuracy",
        "pearson_r": safe_pearson(compare_df["mean_confidence"], compare_df["mean_accuracy"]),
        "spearman_r": safe_spearman(compare_df["mean_confidence"], compare_df["mean_accuracy"]),
        "n": len(compare_df),
    })
    comparison_rows.append({
        "level": "subject_condition",
        "metric_x": "-mean_response_variance",
        "metric_y": "mean_accuracy",
        "pearson_r": safe_pearson(compare_df["neg_mean_response_variance"], compare_df["mean_accuracy"]),
        "spearman_r": safe_spearman(compare_df["neg_mean_response_variance"], compare_df["mean_accuracy"]),
        "n": len(compare_df),
    })

# Trial-level comparison as well
trial_compare_df = trial_df[np.isfinite(trial_df["same_diff_correct"])].copy()
trial_compare_df["neg_response_variance_trial"] = -trial_compare_df["response_variance_trial"]

if len(trial_compare_df) >= 3:
    comparison_rows.append({
        "level": "trial",
        "metric_x": "confidence_rating",
        "metric_y": "same_diff_correct",
        "pearson_r": safe_pearson(trial_compare_df["confidence_rating"], trial_compare_df["same_diff_correct"]),
        "spearman_r": safe_spearman(trial_compare_df["confidence_rating"], trial_compare_df["same_diff_correct"]),
        "n": len(trial_compare_df),
    })
    comparison_rows.append({
        "level": "trial",
        "metric_x": "-response_variance_trial",
        "metric_y": "same_diff_correct",
        "pearson_r": safe_pearson(trial_compare_df["neg_response_variance_trial"], trial_compare_df["same_diff_correct"]),
        "spearman_r": safe_spearman(trial_compare_df["neg_response_variance_trial"], trial_compare_df["same_diff_correct"]),
        "n": len(trial_compare_df),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_csv = os.path.join(confidence_dir, "confidence_vs_variance_comparison.csv")
comparison_df.to_csv(comparison_csv, index=False)

# -------------------------
# CONFIDENCE PLOTS
# -------------------------

# yes/no by condition
if len(confidence_cond_df) > 0:
    plot_df = confidence_cond_df.copy()
    x = np.arange(len(plot_df))
    w = 0.38

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - w/2, plot_df["yes_same_rate"], width=w, label="Mean Yes (same source)")
    ax.bar(x + w/2, plot_df["no_different_rate"], width=w, label="Mean No (different source)")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["condLabel"], rotation=25)
    ax.set_ylabel("Proportion")
    ax.set_title("Average same/different-source responses by condition")
    ax.legend()
    save_figure(fig, os.path.join(confidence_dir, "same_different_response_by_condition.png"))

# confidence rating by condition
if len(confidence_cond_df) > 0:
    plot_df = confidence_cond_df.copy()
    fig, ax = plt.subplots(figsize=(13, 5))
    x = np.arange(len(plot_df))
    ax.bar(x, plot_df["confidence_mean"], yerr=plot_df["confidence_sem"], capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["condLabel"], rotation=25)
    ax.set_ylabel("Confidence rating (1-5)")
    ax.set_title("Average self-reported confidence by condition")
    save_figure(fig, os.path.join(confidence_dir, "confidence_rating_by_condition.png"))

# same/different accuracy by condition (AV only)
acc_plot_df = confidence_cond_df[np.isfinite(confidence_cond_df["same_diff_accuracy"])].copy()
if len(acc_plot_df) > 0:
    fig, ax = plt.subplots(figsize=(13, 5))
    x = np.arange(len(acc_plot_df))
    ax.bar(x, acc_plot_df["same_diff_accuracy"], yerr=acc_plot_df["accuracy_sem"], capsize=4)
    ax.axhline(0.5, linestyle="--", linewidth=1.0, alpha=0.7, label="Chance = 0.5")
    ax.set_xticks(x)
    ax.set_xticklabels(acc_plot_df["condLabel"], rotation=25)
    ax.set_ylabel("Accuracy")
    ax.set_title("Same/different judgment accuracy by audiovisual condition")
    ax.legend()
    save_figure(fig, os.path.join(confidence_dir, "same_different_accuracy_by_condition.png"))

# confidence correct vs incorrect
if len(confidence_correctness_df) > 0:
    plot_df = confidence_correctness_df.copy()
    x = np.arange(len(plot_df))
    w = 0.38

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - w/2, plot_df["confidence_mean_correct"], width=w, label="Confidence when correct")
    ax.bar(x + w/2, plot_df["confidence_mean_incorrect"], width=w, label="Confidence when incorrect")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["condLabel"], rotation=25)
    ax.set_ylabel("Confidence rating (1-5)")
    ax.set_title("Confidence for correct vs incorrect same/different judgments")
    ax.legend()
    save_figure(fig, os.path.join(confidence_dir, "confidence_correct_vs_incorrect_by_condition.png"))

# scatter: subject-condition confidence vs accuracy
if len(compare_df) >= 3:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(compare_df["mean_confidence"], compare_df["mean_accuracy"], alpha=0.8)
    ax.set_xlabel("Mean self-reported confidence")
    ax.set_ylabel("Mean same/different accuracy")
    ax.set_title("Confidence vs accuracy (subject-condition level)")
    save_figure(fig, os.path.join(confidence_dir, "confidence_vs_accuracy_subject_condition.png"))

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(compare_df["neg_mean_response_variance"], compare_df["mean_accuracy"], alpha=0.8)
    ax.set_xlabel("- Mean response variance")
    ax.set_ylabel("Mean same/different accuracy")
    ax.set_title("Inverse response variance vs accuracy (subject-condition level)")
    save_figure(fig, os.path.join(confidence_dir, "inv_variance_vs_accuracy_subject_condition.png"))

# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n================ QC SUMMARY ================")
print(f"Subjects processed: {trial_df['subject'].nunique()}")
print(f"Subject-averaged trialType rows: {len(subject_trialtype_df)}")
print(f"Flagged subjects: {len(flagged_subjects)}")

if len(flagged_subjects) > 0:
    print("\nFlagged participants:")
    print(flag_df.loc[flag_df["flagged"] == 1, ["subject", "flag_reason"]].to_string(index=False))
else:
    print("\nNo subjects were flagged by the current thresholds.")

print("\nMain QC outputs:")
print(f"  Subject avg trialType plots:   {subject_plot_dir}")
print(f"  Subject avg condition plots:   {subject_condition_plot_dir}")
print(f"  Group avg trialType plot:      {group_plot_dir}")
print(f"  trialType metrics CSV:         {subject_trialtype_metrics_csv}")
print(f"  Subject summary CSV:           {subject_summary_csv}")
print(f"  Flagged subjects CSV:          {flag_csv}")
print(f"  Flagged confirmation plots:    {flagged_plot_dir}")

print("\nAdded visual capture outputs:")
print(f"  Capture dir:                   {capture_dir}")
print(f"  Capture by trial CSV:          {capture_trial_csv}")
print(f"  Capture by condition CSV:      {capture_cond_csv}")
print(f"  Capture by offset CSV:         {capture_offset_csv}")

print("\nAdded RMSE outputs:")
print(f"  RMSE dir:                      {rmse_dir}")
print(f"  RMSE by trial CSV:             {rmse_trial_csv}")
print(f"  RMSE by condition CSV:         {rmse_cond_csv}")
print(f"  Subject RMSE summary CSV:      {subject_rmse_csv}")
print(f"  Subject fixed-offset RMSE CSV: {subject_offset_rmse_csv}")

print("\nNEW confidence outputs:")
print(f"  Confidence dir:                        {confidence_dir}")
print(f"  Confidence by trial CSV:              {confidence_trial_csv}")
print(f"  Confidence by condition CSV:          {confidence_cond_csv}")
print(f"  Confidence correct/incorrect CSV:     {confidence_correctness_csv}")
print(f"  Subject-condition summary CSV:        {subject_cond_conf_csv}")
print(f"  Confidence vs variance comparison CSV:{comparison_csv}")

print("\nAll-trial extra metrics CSV:")
print(f"  {trial_metrics_extra_csv}")
print("============================================")

In [ ]:
# ============================================================
# STAND-ALONE ABSOLUTE DEVIATION HISTOGRAM ANALYSIS
# Runs AFTER your QC pipeline code
#
# What it does:
#   - Uses the existing trial_df already created by your QC pipeline
#   - Computes trial-level mean absolute deviation between:
#         response (mouse_x) and audio_x
#     for all multisensory trials
#   - For each participant, plots histograms of the trial-level
#     deviation distributions for:
#         10° offset   (all spatial_10_left + spatial_10_right trials)
#         20° offset   (all spatial_20_left + spatial_20_right trials)
#         30° offset   (all spatial_30_left + spatial_30_right trials)
#         40° offset   (all spatial_40_left + spatial_40_right trials)
#         sweep diverge (all left + right)
#         sweep converge (all left + right)
#
# Output:
#   - one histogram figure per participant
#   - trial-level deviation CSV
#   - subject-condition summary CSV
#
# IMPORTANT:
#   This code assumes `trial_df` already exists in memory from the
#   QC pipeline you just ran.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Check required variables from the prior pipeline
# ------------------------------------------------------------
if "trial_df" not in globals():
    raise RuntimeError("trial_df was not found in memory. Run the QC pipeline first, then run this block.")

if "out_root" not in globals():
    raise RuntimeError("out_root was not found in memory. Run the QC pipeline first, then run this block.")

# ------------------------------------------------------------
# Helper: trial-level mean absolute deviation
# ------------------------------------------------------------
def mean_abs_deviation_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return float(np.mean(np.abs(x[mask] - y[mask])))

# ------------------------------------------------------------
# Output dirs
# ------------------------------------------------------------
deviation_hist_dir = os.path.join(out_root, "absolute_deviation_histograms")
os.makedirs(deviation_hist_dir, exist_ok=True)

# ------------------------------------------------------------
# Restrict to multisensory trials and compute trial-level
# absolute deviation relative to audio_x
# ------------------------------------------------------------
multi_df = trial_df[trial_df["trialType"].isin(["congruent", "incongruent_spatial", "incongruent_sweep"])].copy()

multi_df["abs_dev_to_audio"] = multi_df.apply(
    lambda r: mean_abs_deviation_1d(r["mouse_x"], r["audio_x"]),
    axis=1
)

# Build a plotting group variable exactly as requested
def make_deviation_group(row):
    tt = str(row["trialType"])
    if tt == "incongruent_spatial":
        off = pd.to_numeric(row["offsetDeg"], errors="coerce")
        if np.isfinite(off):
            return f"offset_{int(off)}"
    elif tt == "incongruent_sweep":
        rt = str(row["rampType"]).strip().lower()
        if "diverge" in rt:
            return "sweep_diverge"
        elif "converge" in rt:
            return "sweep_converge"
    return np.nan

multi_df["deviation_group"] = multi_df.apply(make_deviation_group, axis=1)

# Keep only the requested groups
group_order = [
    "offset_10",
    "offset_20",
    "offset_30",
    "offset_40",
    "sweep_diverge",
    "sweep_converge",
]

multi_df = multi_df[multi_df["deviation_group"].isin(group_order)].copy()

# Save trial-level CSV
trial_dev_csv = os.path.join(deviation_hist_dir, "trial_level_absolute_deviation_to_audio.csv")
multi_df[[
    "subject", "source_file", "trialIndex", "trialType", "condLabel",
    "offsetDeg", "sideSign", "rampType", "deviation_group", "abs_dev_to_audio"
]].to_csv(trial_dev_csv, index=False)

# ------------------------------------------------------------
# Summary CSV
# ------------------------------------------------------------
summary_df = (
    multi_df
    .groupby(["subject", "deviation_group"], as_index=False)
    .agg(
        n_trials=("abs_dev_to_audio", lambda x: np.sum(np.isfinite(x))),
        mean_abs_dev=("abs_dev_to_audio", "mean"),
        median_abs_dev=("abs_dev_to_audio", "median"),
        sd_abs_dev=("abs_dev_to_audio", "std"),
        min_abs_dev=("abs_dev_to_audio", "min"),
        max_abs_dev=("abs_dev_to_audio", "max"),
    )
)

summary_csv = os.path.join(deviation_hist_dir, "subject_condition_absolute_deviation_summary.csv")
summary_df.to_csv(summary_csv, index=False)

# ------------------------------------------------------------
# Plot one histogram figure per participant
# ------------------------------------------------------------
subjects_for_dev = sorted(multi_df["subject"].dropna().astype(str).unique().tolist())

nice_titles = {
    "offset_10": "10° offset",
    "offset_20": "20° offset",
    "offset_30": "30° offset",
    "offset_40": "40° offset",
    "sweep_diverge": "Sweep diverge",
    "sweep_converge": "Sweep converge",
}

for subj in subjects_for_dev:
    sub = multi_df[multi_df["subject"] == subj].copy()

    # Determine common x-axis range for that subject
    subj_vals = sub["abs_dev_to_audio"].to_numpy(dtype=float)
    subj_vals = subj_vals[np.isfinite(subj_vals)]

    if len(subj_vals) == 0:
        continue

    x_min = 0
    x_max = np.nanmax(subj_vals)
    if not np.isfinite(x_max) or x_max <= 0:
        x_max = 1.0

    ncols = 3
    nrows = 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 8), squeeze=False)
    axes = axes.flatten()

    for ax, grp in zip(axes, group_order):
        grp_sub = sub[sub["deviation_group"] == grp].copy()
        vals = grp_sub["abs_dev_to_audio"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]

        if len(vals) == 0:
            ax.axis("off")
            continue

        # Histogram
        ax.hist(vals, bins="auto", alpha=0.8, edgecolor="black")

        # Reference lines
        mean_val = np.nanmean(vals)
        med_val = np.nanmedian(vals)
        ax.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {mean_val:.2f}")
        ax.axvline(med_val, linestyle=":", linewidth=2, label=f"Median = {med_val:.2f}")

        ax.set_title(f"{nice_titles[grp]}\n n = {len(vals)}")
        ax.set_xlabel("Mean absolute deviation |response - audio|")
        ax.set_ylabel("Count")
        ax.set_xlim(x_min, x_max * 1.05)
        ax.legend(fontsize=8)

    # turn off unused axes if any
    for ax in axes[len(group_order):]:
        ax.axis("off")

    fig.suptitle(
        f"{subj}: trial-level absolute deviation distributions\n"
        f"(multisensory trials, response vs audio_x)",
        fontsize=13,
        y=1.02
    )
    fig.tight_layout()

    outpath = os.path.join(deviation_hist_dir, f"{subj}_absolute_deviation_histograms.png")
    fig.savefig(outpath, dpi=140, bbox_inches="tight")
    plt.close(fig)

# ------------------------------------------------------------
# Optional pooled histogram figure across all participants
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8), squeeze=False)
axes = axes.flatten()

all_vals = multi_df["abs_dev_to_audio"].to_numpy(dtype=float)
all_vals = all_vals[np.isfinite(all_vals)]
x_min = 0
x_max = np.nanmax(all_vals) if len(all_vals) > 0 else 1.0
if not np.isfinite(x_max) or x_max <= 0:
    x_max = 1.0

for ax, grp in zip(axes, group_order):
    grp_sub = multi_df[multi_df["deviation_group"] == grp].copy()
    vals = grp_sub["abs_dev_to_audio"].to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        ax.axis("off")
        continue

    ax.hist(vals, bins="auto", alpha=0.8, edgecolor="black")
    mean_val = np.nanmean(vals)
    med_val = np.nanmedian(vals)
    ax.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {mean_val:.2f}")
    ax.axvline(med_val, linestyle=":", linewidth=2, label=f"Median = {med_val:.2f}")

    ax.set_title(f"{nice_titles[grp]}\n n = {len(vals)}")
    ax.set_xlabel("Mean absolute deviation |response - audio|")
    ax.set_ylabel("Count")
    ax.set_xlim(x_min, x_max * 1.05)
    ax.legend(fontsize=8)

for ax in axes[len(group_order):]:
    ax.axis("off")

fig.suptitle(
    "Pooled trial-level absolute deviation distributions\n"
    "(multisensory trials, response vs audio_x)",
    fontsize=13,
    y=1.02
)
fig.tight_layout()
fig.savefig(os.path.join(deviation_hist_dir, "pooled_absolute_deviation_histograms.png"), dpi=140, bbox_inches="tight")
plt.close(fig)

print("\nSaved absolute deviation histogram analysis:")
print(f"  Output directory: {deviation_hist_dir}")
print(f"  Trial-level CSV:  {trial_dev_csv}")
print(f"  Summary CSV:      {summary_csv}")
print("  Per-subject histogram PNGs")
print("  Pooled histogram PNG")

## 2. Subject-First Data Preparation and Resampling

Core subject-first preprocessing, trial loading/resampling, helper functions, and standardized data structures used by later analyses.

In [4]:
# %% CELL 1 — SETUP + HELPERS + LOAD/RESAMPLE ALL TRIALS
# ============================================================
# SUBJECT-FIRST SPATIAL LOCALIZATION ANALYSIS PIPELINE
# Tonelli-style cross-correlation + lag-corrected spatial metrics
# ============================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, linregress

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

SPATIAL_ANALYSES_ROOT = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses"
OUT_DIR_NAME = "00_core_spatial_metrics"

RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0
TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1.0 / RESAMPLE_HZ, 1.0 / RESAMPLE_HZ)

EXCLUDE_PRACTICE = True

SPATIAL_TASK_TYPES = ["congruent", "incongruent_spatial", "incongruent_sweep"]
EXTRA_TRIAL_TYPES = ["audio_only", "visual_only"]
TRIAL_TYPES_TO_KEEP = SPATIAL_TASK_TYPES + EXTRA_TRIAL_TYPES

# Cross-correlation lag search
MAX_LAG_SEC = 4.0
MIN_VALID_SAMPLES = 10

# Sliding-window ramp analysis
WINDOW_SEC = 2.0
STEP_SEC = 0.25
CAPTURE_SWITCH_THRESHOLD = 0.0
MIN_CONSECUTIVE_SWITCH_WINDOWS = 2

# Visual capture settings
MIN_ABS_DISPARITY_FOR_CAPTURE = 0.01

# Plot settings
FIG_DPI = 160
LW_MEAN = 2.4

# =========================
# OUTPUT FOLDERS
# =========================
out_root = os.path.join(SPATIAL_ANALYSES_ROOT, OUT_DIR_NAME)
trial_metric_dir = os.path.join(out_root, "trial_level_metrics")
subject_metric_dir = os.path.join(out_root, "subject_level_metrics")
ramp_dir = os.path.join(out_root, "ramp_sliding_window_outputs")
recalibration_dir = os.path.join(out_root, "recalibration_outputs")
metric_plot_dir = os.path.join(out_root, "cell7_subject_metric_condition_plots")

for d in [
    out_root,
    trial_metric_dir,
    subject_metric_dir,
    ramp_dir,
    recalibration_dir,
    metric_plot_dir,
]:
    os.makedirs(d, exist_ok=True)

# =========================
# HELPERS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()
    return Path(filepath).stem

def clean_sorted_unique(values):
    values = pd.Series(values).dropna().astype(str).unique().tolist()
    return sorted(values)

def safe_numeric(x):
    return pd.to_numeric(x, errors="coerce")

def corr_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan
    if np.nanstd(x[ok]) < 1e-12 or np.nanstd(y[ok]) < 1e-12:
        return np.nan
    return float(np.corrcoef(x[ok], y[ok])[0, 1])

def rmse_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan
    return float(np.sqrt(np.nanmean((x[ok] - y[ok]) ** 2)))

def mean_abs_deviation_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan
    return float(np.nanmean(np.abs(x[ok] - y[ok])))

def sem_1d(arr, axis=0):
    arr = np.asarray(arr, dtype=float)
    n = np.sum(np.isfinite(arr), axis=axis)
    sd = np.nanstd(arr, axis=axis, ddof=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        return sd / np.sqrt(n)

def save_figure(fig, outpath):
    fig.tight_layout()
    fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

def resample_trial(df_trial, time_grid):
    needed = ["t", "mouse_x", "blob_x", "audio_x"]
    for c in needed:
        if c not in df_trial.columns:
            raise ValueError(f"Missing required column: {c}")

    d = df_trial.copy()
    d = d.sort_values("t")
    d = d[np.isfinite(d["t"])]
    d = d.drop_duplicates(subset="t", keep="first")

    if len(d) < 5:
        return None

    t = d["t"].values.astype(float)
    t = t - np.nanmin(t)

    if np.nanmax(t) < 1.0:
        return None

    def _interp(col):
        y = d[col].values.astype(float)
        ok = np.isfinite(t) & np.isfinite(y)
        if ok.sum() < 3:
            return np.full_like(time_grid, np.nan, dtype=float)
        return np.interp(time_grid, t[ok], y[ok], left=np.nan, right=np.nan)

    return {
        "t_grid": time_grid.copy(),
        "mouse_x": _interp("mouse_x"),
        "blob_x": _interp("blob_x"),
        "audio_x": _interp("audio_x"),
    }

def lag_scanned_corr(response, stimulus, fs, max_lag_sec=4.0):
    """
    Lag-scanned correlation constrained to non-negative response delays.

    Positive lag_sec means the response lags behind the stimulus.
    Because the analysis quantifies response latency to an externally
    generated stimulus trajectory, negative response-stimulus shifts are
    not interpreted as meaningful sensory-response delays.

    No peak-correlation threshold is applied here because these lags are
    used as alignment parameters for spatial RMSE/bias analyses rather
    than as dependent variables themselves.
    """
    response = np.asarray(response, dtype=float)
    stimulus = np.asarray(stimulus, dtype=float)

    max_lag_samples = int(round(max_lag_sec * fs))
    lag_samples = np.arange(0, max_lag_samples + 1)

    corrs = []
    for lag in lag_samples:
        if lag == 0:
            r_seg = response
            s_seg = stimulus
        else:
            # Positive lag: response is delayed relative to stimulus.
            r_seg = response[lag:]
            s_seg = stimulus[:-lag]

        corrs.append(corr_1d(r_seg, s_seg))

    corrs = np.asarray(corrs, dtype=float)

    if np.all(~np.isfinite(corrs)):
        return {
            "peak_corr": np.nan,
            "lag_sec": np.nan,
            "lag_samples": np.nan,
            "all_lags_sec": lag_samples / fs,
            "all_corrs": corrs,
        }

    best_idx = np.nanargmax(corrs)
    best_lag_samples = int(lag_samples[best_idx])

    return {
        "peak_corr": float(corrs[best_idx]),
        "lag_sec": float(best_lag_samples / fs),
        "lag_samples": best_lag_samples,
        "all_lags_sec": lag_samples / fs,
        "all_corrs": corrs,
    }

def lag_correct_response(response, t_grid, lag_sec):
    """
    Shift response backward by estimated lag.
    If lag_sec > 0, response is delayed, so use response(t + lag_sec).
    """
    response = np.asarray(response, dtype=float)
    t_grid = np.asarray(t_grid, dtype=float)

    if not np.isfinite(lag_sec):
        return np.full_like(response, np.nan, dtype=float)

    shifted = np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan
    )
    return shifted

def visual_capture_percent(response, visual, audio, min_abs_disparity=0.01):
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    d = visual - audio
    ok = (
        np.isfinite(response) &
        np.isfinite(visual) &
        np.isfinite(audio) &
        (np.abs(d) > min_abs_disparity)
    )
    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan
    return float(np.nanmean((response[ok] - audio[ok]) / d[ok]))

def condition_sort_key(label):
    s = str(label).lower()

    # Put congruent first
    if "congruent" in s and "incongruent" not in s:
        return (0, 0, s)

    # Fixed spatial offsets
    m = re.search(r"(\d+)", s)
    offset = int(m.group(1)) if m else 999

    if "left" in s:
        side = 0
    elif "right" in s:
        side = 1
    else:
        side = 2

    if "sweep" in s and "diverge" in s:
        return (3, side, s)
    if "sweep" in s and "converge" in s:
        return (4, side, s)
    if "incongruent" in s:
        return (1, offset, side)

    if "audio" in s:
        return (5, 0, s)
    if "visual" in s:
        return (6, 0, s)

    return (9, 999, s)

# =========================
# LOAD FILES
# =========================
if not DATA_DIR:
    raise ValueError("Please fill in DATA_DIR before running.")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in: {DATA_DIR}")

print(f"Found {len(csv_files)} files")
print(f"Output directory: {out_root}")

all_trial_rows = []

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    print(f"Loading {subj}: {Path(fpath).name}")

    df = pd.read_csv(fpath)

    required_cols = [
        "trialIndex", "trialType", "condLabel", "isPractice",
        "t", "mouse_x", "blob_x", "audio_x"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"  Skipping {subj}: missing columns {missing}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    df = df[df["trialType"].isin(TRIAL_TYPES_TO_KEEP)].copy()

    if len(df) == 0:
        print(f"  Skipping {subj}: no usable rows after filtering")
        continue

    extra_cols = [
        c for c in [
            "offsetDeg", "sideSign", "rampType", "visualOn", "audioOn",
            "post_question", "post_question2"
        ]
        if c in df.columns
    ]

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"], sort=False
    ):
        resampled = resample_trial(df_trial, TIME_GRID)
        if resampled is None:
            continue

        row = {
            "subject": subj,
            "source_file": Path(fpath).name,
            "trialIndex": trial_index,
            "trialType": trial_type,
            "condLabel": cond_label,
            "t_grid": resampled["t_grid"],
            "mouse_x": resampled["mouse_x"],
            "blob_x": resampled["blob_x"],
            "audio_x": resampled["audio_x"],
        }

        if len(df_trial) > 0:
            first_row = df_trial.iloc[0]
            for c in extra_cols:
                row[c] = first_row[c]

        all_trial_rows.append(row)

if len(all_trial_rows) == 0:
    raise RuntimeError("No valid trials were loaded after filtering/resampling.")

trial_df = pd.DataFrame(all_trial_rows)

for c in ["offsetDeg", "sideSign", "rampType", "visualOn", "audioOn", "post_question", "post_question2"]:
    if c not in trial_df.columns:
        trial_df[c] = np.nan

trial_df["offsetDeg"] = safe_numeric(trial_df["offsetDeg"])
trial_df["sideSign"] = safe_numeric(trial_df["sideSign"])
trial_df["trial_order_within_subject"] = trial_df.groupby("subject").cumcount() + 1

trial_df["midpoint_x"] = trial_df.apply(
    lambda r: (np.asarray(r["blob_x"], dtype=float) + np.asarray(r["audio_x"], dtype=float)) / 2.0,
    axis=1
)

print(f"\nLoaded {len(trial_df)} valid trials from {trial_df['subject'].nunique()} subjects.")
print(trial_df[["subject", "trialType", "condLabel"]].head())

Found 50 files
Output directory: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics
Loading pid001: pid001_001_master_trials_LONG.csv
Loading pid003: pid003_001_master_trials_LONG.csv
Loading pid004: pid004_001_master_trials_LONG.csv
Loading pid005: pid005_001_master_trials_LONG.csv
Loading pid006: pid006_001_master_trials_LONG.csv
Loading pid007: pid007_001_master_trials_LONG.csv
Loading pid008: pid008_001_master_trials_LONG.csv
Loading pid009: pid009_001_master_trials_LONG.csv
Loading pid010: pid010_001_master_trials_LONG.csv
Loading pid011: pid011_001_master_trials_LONG.csv
Loading pid013: pid013_001_master_trials_LONG.csv
Loading pid014: pid014_001_master_trials_LONG.csv
Loading pid015: pid015_001_master_trials_LONG.csv
Loading pid016: pid016_001_master_trials_LONG.csv
Loading pid017: pid017_001_master_trials_LONG.csv
Loading pid019: pid019_001_master_trials_LONG.csv
Loading pid021: pid021_001_master_trials_LONG.csv
Lo

## 3. Same- vs Different-Source Tracking Lag

Tracking-lag analyses comparing trials explicitly judged as arising from the same versus different sources, including the collaborator-matched implementation.

In [6]:
# %% SAME- VS DIFFERENT-SOURCE TRACKING LAG ANALYSIS
# =============================================================================
# PURPOSE
# =============================================================================
# Test whether continuous spatial-localization tracking is temporally faster
# on audiovisual trials that participants ultimately judge as SAME source
# compared with physically matched trials judged as DIFFERENT sources.
#
# CENTRAL HYPOTHESIS
# ------------------
# Dynamic-BCI account:
#
#     SAME / stronger binding
#         -> greater contribution of precise visual information
#         -> shorter effective tracking lag
#
#     DIFFERENT / stronger segregation
#         -> greater reliance on temporally accumulated auditory information
#         -> longer effective tracking lag
#
# Therefore the primary directional prediction is:
#
#     Lag_Different - Lag_Same > 0
#
#
# IMPORTANT DESIGN PRINCIPLE
# --------------------------
# The critical comparison preserves physical stimulation.
#
# Same/different labels are compared:
#
#     within participant
#     within fixed audiovisual offset
#     and, for permutation inference, within offset × side
#
# Thus the analysis does NOT simply compare all SAME trials with all
# DIFFERENT trials, which would confound causal report with disparity.
#
#
# ANALYSIS SERIES
# ---------------
# A. Trial-level lag extraction from UNCORRECTED trajectories
# B. Overall descriptive SAME vs DIFFERENT lag
# C. Participant × offset matched SAME-DIFFERENT contrasts
# D. Participant-level matched effect:
#        Different - Same
# E. Whole-trial judgment-label permutation test preserving physical condition
# F. Mixed-effects model controlling offset and side
# G. Cell-averaged trajectory lag analysis
#        participant × offset × judgment trajectories are averaged FIRST,
#        then lag is estimated (higher-SNR confirmatory analysis)
# H. Tracking-quality sensitivity analysis
# I. Original ±2-s lag-window robustness analysis
# J. Secondary GEE:
#        Does trial lag predict SAME/DIFFERENT judgment above physical disparity?
#
#
# OUTPUT INTERPRETATION
# ---------------------
# Positive:
#     lag_difference_different_minus_same
#
# means tracking is SLOWER on DIFFERENT-source trials.
#
# This is the predicted dynamic-BCI result.
#
#
# INPUT
# -----
# Preferred:
#     existing trial_df from the spatial-localization notebook containing:
#         subject
#         trialIndex
#         trialType
#         condLabel
#         t_grid
#         mouse_x
#         blob_x
#         audio_x
#         post_question
#
# If the current trial_df contains only unisensory trials, this cell
# automatically reloads the master CSVs from DATA_DIR.
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import (
    ttest_1samp,
    ttest_rel,
    wilcoxon,
    t as t_dist,
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =============================================================================
# SETTINGS
# =============================================================================

# -------------------------------------------------------------------------
# These match the main spatial-localization notebook.
# -------------------------------------------------------------------------

try:
    DATA_DIR
except NameError:
    DATA_DIR = (
        r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
        r"\Continuous AVT v1\individual_data_SL"
    )

try:
    FILE_PATTERN
except NameError:
    FILE_PATTERN = "*master_trials*.csv"


RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0

TIME_GRID = np.arange(
    0.0,
    TRIAL_DURATION_SEC + (1.0 / RESAMPLE_HZ),
    1.0 / RESAMPLE_HZ,
)


# -------------------------------------------------------------------------
# Fixed-offset trials are the PRIMARY analysis.
#
# They permit the cleanest test because physical A-V disparity is constant.
# -------------------------------------------------------------------------

FIXED_OFFSETS = [10, 20, 30, 40]

EXCLUDE_PRACTICE = True


# -------------------------------------------------------------------------
# Lag-search settings
#
# PRIMARY:
# ±4 s prevents mechanically truncating auditory-like delays > 2 s.
#
# ROBUSTNESS:
# ±2 s exactly reproduces the search range used in the notebook's
# Tonelli-style lag analysis.
# -------------------------------------------------------------------------

PRIMARY_MAX_LAG_SEC = 4.0
ORIGINAL_MAX_LAG_SEC = 2.0


# -------------------------------------------------------------------------
# Full uncorrected trajectory is used because LAG is the dependent variable.
#
# Do NOT use lag-corrected trajectories here.
# -------------------------------------------------------------------------

ANALYSIS_START_SEC = 0.0
ANALYSIS_END_SEC = 18.0

MIN_VALID_SAMPLES = 20


# -------------------------------------------------------------------------
# Tracking-quality sensitivity threshold.
#
# Primary inference uses all valid trials.
# This threshold is ONLY a robustness analysis.
# -------------------------------------------------------------------------

MIN_PEAK_CORR_SENSITIVITY = 0.20


# -------------------------------------------------------------------------
# Minimum observations for various analyses
# -------------------------------------------------------------------------

MIN_TRIALS_PER_JUDGMENT_CELL = 1

# For cell-averaged trajectory analysis, 2 gives a more defensible
# judgment-specific average trajectory.
MIN_TRIALS_FOR_CELL_AVERAGE = 2

MIN_MATCHED_OFFSETS_PER_SUBJECT = 1


# -------------------------------------------------------------------------
# Permutation settings
#
# Judgment labels are shuffled across COMPLETE TRIALS within:
#
#     subject × offset × side
#
# which holds physical stimulation fixed.
# -------------------------------------------------------------------------

N_PERM = 10000
RANDOM_SEED = 12345


# -------------------------------------------------------------------------
# Plot settings
# -------------------------------------------------------------------------

FIG_DPI = 180


# =============================================================================
# OUTPUT DIRECTORIES
# =============================================================================

OUT_DIR = os.path.join(
    DATA_DIR,
    "spatial_same_different_tracking_lag_analysis",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures",
)

for _d in [
    OUT_DIR,
    CSV_DIR,
    FIG_DIR,
]:
    os.makedirs(
        _d,
        exist_ok=True,
    )


print("\n" + "=" * 80)
print("SAME VS DIFFERENT SOURCE — TRACKING LAG ANALYSIS")
print("=" * 80)

print(f"\nData directory:\n{DATA_DIR}")
print(f"\nOutput directory:\n{OUT_DIR}")

print(
    f"\nPrimary lag search: ±{PRIMARY_MAX_LAG_SEC:.1f} s"
)

print(
    f"Original-notebook robustness lag search: "
    f"±{ORIGINAL_MAX_LAG_SEC:.1f} s"
)

print(
    "\nPrimary directional prediction:"
    "\n    Lag(Different) - Lag(Same) > 0"
)


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def normalize_subject(value):
    """
    Standardize subject labels to pid### where possible.
    """

    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    m = re.search(
        r"pid\s*0*(\d+)",
        text,
    )

    if m:
        return f"pid{int(m.group(1)):03d}"

    m = re.search(
        r"(\d+)",
        text,
    )

    if m:
        return f"pid{int(m.group(1)):03d}"

    return text


def normalize_trial_id(value):
    """
    Standardize trial ID without changing its substantive value.
    """

    if pd.isna(value):
        return np.nan

    return re.sub(
        r"\.0$",
        "",
        str(value).strip(),
    )


def extract_subject_id(filepath):
    """
    Extract pid### from filename.
    """

    name = Path(filepath).name

    m = re.search(
        r"(pid\d+)",
        name,
        flags=re.IGNORECASE,
    )

    if m:
        return normalize_subject(
            m.group(1)
        )

    return Path(filepath).stem


def safe_array(value):
    """
    Convert stored trajectory to flat float array.
    """

    try:
        return np.asarray(
            value,
            dtype=float,
        ).reshape(-1)

    except Exception:
        return np.asarray(
            [],
            dtype=float,
        )


def parse_same_response(x):
    """
    Convert post-trial same/different response to:
        1 = SAME source
        0 = DIFFERENT source
    """

    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    same_tokens = [
        "same",
        "same source",
        "common",
        "common source",
        "together",
        "one",
        "1",
        "yes",
        "y",
        "true",
    ]

    different_tokens = [
        "different",
        "diff",
        "different source",
        "separate",
        "separate source",
        "apart",
        "two",
        "2",
        "0",
        "no",
        "n",
        "false",
    ]

    if s in same_tokens:
        return 1.0

    if s in different_tokens:
        return 0.0

    if (
        "same" in s
        or "common" in s
    ):
        return 1.0

    if (
        "different" in s
        or "separate" in s
        or "diff" in s
    ):
        return 0.0

    return np.nan


def judgment_label(value):
    """
    Human-readable judgment.
    """

    try:
        value = float(value)
    except Exception:
        return np.nan

    if not np.isfinite(value):
        return np.nan

    return (
        "Same"
        if int(round(value)) == 1
        else "Different"
    )


def extract_offset_abs(row):
    """
    Extract absolute fixed audiovisual offset.
    """

    if (
        "offsetDeg" in row.index
        and pd.notna(row["offsetDeg"])
    ):
        try:
            value = abs(
                float(row["offsetDeg"])
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

    cond = str(
        row.get(
            "condLabel",
            "",
        )
    ).lower()

    m = re.search(
        r"(10|20|30|40)",
        cond,
    )

    if m:
        return float(
            m.group(1)
        )

    return np.nan


def extract_side_label(row):
    """
    Preserve stimulus side for matching/permutation.
    """

    if (
        "sideSign" in row.index
        and pd.notna(row["sideSign"])
    ):
        try:
            side = float(
                row["sideSign"]
            )

            if side < 0:
                return "left"

            if side > 0:
                return "right"

            return "center"

        except Exception:
            pass

    cond = str(
        row.get(
            "condLabel",
            "",
        )
    ).lower()

    if "left" in cond:
        return "left"

    if "right" in cond:
        return "right"

    return "unknown"


def is_fixed_offset_trial(row):
    """
    Identify the fixed incongruent audiovisual trials.
    """

    trial_type = str(
        row.get(
            "trialType",
            "",
        )
    ).strip().lower()

    offset = extract_offset_abs(
        row
    )

    if not np.isfinite(offset):
        return False

    if int(round(offset)) not in FIXED_OFFSETS:
        return False

    # This is the exact trial type used throughout the notebook.
    if trial_type == "incongruent_spatial":
        return True

    # Robust fallback.
    cond = str(
        row.get(
            "condLabel",
            "",
        )
    ).lower()

    if (
        "sweep" in trial_type
        or "sweep" in cond
        or "diverge" in cond
        or "converge" in cond
    ):
        return False

    if (
        "incongruent" in trial_type
        or "fixed" in trial_type
        or "spatial" in trial_type
    ):
        return True

    return False


# =============================================================================
# RESAMPLING — MATCHES NOTEBOOK LOGIC
# =============================================================================

def resample_trial(
    df_trial,
    time_grid,
):
    """
    Resample raw trial trajectories to the same 20-Hz grid used in
    the spatial-localization notebook.
    """

    needed = [
        "t",
        "mouse_x",
        "blob_x",
        "audio_x",
    ]

    for column in needed:

        if column not in df_trial.columns:
            raise ValueError(
                f"Missing required column: {column}"
            )

    d = (
        df_trial
        .copy()
        .sort_values("t")
    )

    d = d[
        np.isfinite(
            pd.to_numeric(
                d["t"],
                errors="coerce",
            )
        )
    ]

    d = d.drop_duplicates(
        subset="t",
        keep="first",
    )

    if len(d) < 5:
        return None

    t_original = pd.to_numeric(
        d["t"],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    t_original = (
        t_original
        - np.nanmin(t_original)
    )

    if (
        len(t_original) < 3
        or np.nanmax(t_original) < 1.0
    ):
        return None

    def _interp(column):

        y = pd.to_numeric(
            d[column],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        ok = (
            np.isfinite(t_original)
            & np.isfinite(y)
        )

        if ok.sum() < 3:
            return np.full(
                len(time_grid),
                np.nan,
                dtype=float,
            )

        return np.interp(
            time_grid,
            t_original[ok],
            y[ok],
            left=np.nan,
            right=np.nan,
        )

    return {
        "t_grid": np.asarray(
            time_grid,
            dtype=float,
        ).copy(),
        "mouse_x": _interp(
            "mouse_x"
        ),
        "blob_x": _interp(
            "blob_x"
        ),
        "audio_x": _interp(
            "audio_x"
        ),
    }


# =============================================================================
# CORRELATION + LAG FUNCTIONS
# =============================================================================

def corr_1d(
    x,
    y,
    min_valid_samples=MIN_VALID_SAMPLES,
):
    """
    Pearson correlation with finite-sample checks.
    """

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    n = min(
        len(x),
        len(y),
    )

    x = x[:n]
    y = y[:n]

    ok = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    if ok.sum() < min_valid_samples:
        return np.nan

    x = x[ok]
    y = y[ok]

    if (
        np.nanstd(x) < 1e-12
        or np.nanstd(y) < 1e-12
    ):
        return np.nan

    return float(
        np.corrcoef(
            x,
            y,
        )[0, 1]
    )


def lag_scanned_corr(
    response,
    stimulus,
    fs=RESAMPLE_HZ,
    max_lag_sec=PRIMARY_MAX_LAG_SEC,
    min_valid_samples=MIN_VALID_SAMPLES,
):
    """
    Same sign convention as the notebook's Tonelli analysis.

    Positive lag_sec:
        response LAGS behind stimulus.

    Implementation:
        compares response(t + lag) against stimulus(t)
        for positive lag.
    """

    response = np.asarray(
        response,
        dtype=float,
    )

    stimulus = np.asarray(
        stimulus,
        dtype=float,
    )

    n = min(
        len(response),
        len(stimulus),
    )

    response = response[:n]
    stimulus = stimulus[:n]

    max_lag_samples = int(
        round(
            max_lag_sec * fs
        )
    )

    lag_grid = np.arange(
        -max_lag_samples,
        max_lag_samples + 1,
    )

    best_corr = np.nan
    best_lag_samples = np.nan

    corr_values = []

    for lag_samples in lag_grid:

        if lag_samples > 0:

            r = response[
                lag_samples:
            ]

            s = stimulus[
                :-lag_samples
            ]

        elif lag_samples < 0:

            r = response[
                :lag_samples
            ]

            s = stimulus[
                -lag_samples:
            ]

        else:

            r = response
            s = stimulus

        c = corr_1d(
            r,
            s,
            min_valid_samples=min_valid_samples,
        )

        corr_values.append(
            c
        )

        if np.isfinite(c):

            if (
                not np.isfinite(best_corr)
                or c > best_corr
            ):

                best_corr = c
                best_lag_samples = lag_samples

    if np.isfinite(best_lag_samples):

        lag_sec = (
            best_lag_samples
            / fs
        )

    else:

        lag_sec = np.nan

    boundary_hit = False

    if np.isfinite(best_lag_samples):

        boundary_hit = (
            abs(best_lag_samples)
            >= max_lag_samples
        )

    return {
        "peak_corr": best_corr,
        "lag_samples": best_lag_samples,
        "lag_sec": lag_sec,
        "boundary_hit": boundary_hit,
        "lag_grid_sec": (
            lag_grid / fs
        ),
        "corr_grid": np.asarray(
            corr_values,
            dtype=float,
        ),
    }


def restrict_analysis_window(
    time_grid,
    *arrays,
    start_sec=ANALYSIS_START_SEC,
    end_sec=ANALYSIS_END_SEC,
):
    """
    Restrict trajectories to analysis interval.
    """

    t = np.asarray(
        time_grid,
        dtype=float,
    )

    n = len(t)

    arrays = [
        np.asarray(
            a,
            dtype=float,
        )[:n]
        for a in arrays
    ]

    ok_time = (
        np.isfinite(t)
        & (t >= start_sec)
        & (t <= end_sec)
    )

    output = [
        t[ok_time]
    ]

    for arr in arrays:
        output.append(
            arr[ok_time]
        )

    return tuple(
        output
    )


# =============================================================================
# STATISTICAL HELPERS
# =============================================================================

def mean_ci_t(
    values,
    alpha=0.05,
):
    """
    Mean, t CI, SE, N.
    """

    vals = (
        pd.Series(values)
        .dropna()
        .astype(float)
        .to_numpy()
    )

    n = len(vals)

    if n == 0:

        return {
            "n": 0,
            "mean": np.nan,
            "sd": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    mean = float(
        np.mean(vals)
    )

    sd = (
        float(
            np.std(
                vals,
                ddof=1,
            )
        )
        if n >= 2
        else np.nan
    )

    se = (
        sd / np.sqrt(n)
        if n >= 2
        else np.nan
    )

    if n >= 2:

        crit = t_dist.ppf(
            1 - alpha / 2,
            df=n - 1,
        )

        ci_low = mean - crit * se
        ci_high = mean + crit * se

    else:

        ci_low = np.nan
        ci_high = np.nan

    return {
        "n": n,
        "mean": mean,
        "sd": sd,
        "se": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }


def cohens_dz(
    differences,
):
    """
    Paired-sample Cohen dz.
    """

    x = (
        pd.Series(differences)
        .dropna()
        .astype(float)
        .to_numpy()
    )

    if len(x) < 2:
        return np.nan

    sd = np.std(
        x,
        ddof=1,
    )

    if (
        not np.isfinite(sd)
        or sd <= 0
    ):
        return np.nan

    return float(
        np.mean(x) / sd
    )


def sign_flip_test(
    values,
    popmean=0.0,
    n_perm=N_PERM,
    seed=RANDOM_SEED,
    alternative="two-sided",
):
    """
    Participant-level sign-flip permutation test.
    """

    values = (
        pd.Series(values)
        .dropna()
        .astype(float)
        .to_numpy()
    )

    differences = (
        values
        - popmean
    )

    n = len(differences)

    if n < 2:

        return {
            "p_perm": np.nan,
            "observed_mean": np.nan,
            "n": n,
        }

    observed = float(
        np.mean(
            differences
        )
    )

    rng = np.random.default_rng(
        seed
    )

    # Avoid a huge allocation if N becomes large.
    null_values = np.empty(
        n_perm,
        dtype=float,
    )

    for p in range(n_perm):

        signs = rng.choice(
            [-1.0, 1.0],
            size=n,
        )

        null_values[p] = np.mean(
            signs * differences
        )

    if alternative == "greater":

        pvalue = (
            np.sum(
                null_values >= observed
            )
            + 1
        ) / (
            n_perm + 1
        )

    elif alternative == "less":

        pvalue = (
            np.sum(
                null_values <= observed
            )
            + 1
        ) / (
            n_perm + 1
        )

    else:

        pvalue = (
            np.sum(
                np.abs(null_values)
                >= abs(observed)
            )
            + 1
        ) / (
            n_perm + 1
        )

    return {
        "p_perm": float(pvalue),
        "observed_mean": observed,
        "n": n,
    }


def p_fmt(p):
    """
    APA-ish p formatting.
    """

    if not np.isfinite(p):
        return "NA"

    if p < .001:
        return "< .001"

    return f"= {p:.4f}"


# =============================================================================
# STEP 1 — OBTAIN COMPLETE FIXED-OFFSET TRIAL DATASET
# =============================================================================

def existing_trial_df_is_suitable(
    dataframe,
):
    """
    Check whether the currently loaded trial_df contains the relevant AV trials.
    """

    if (
        dataframe is None
        or not isinstance(
            dataframe,
            pd.DataFrame,
        )
        or dataframe.empty
    ):
        return False

    required = {
        "subject",
        "trialIndex",
        "trialType",
        "t_grid",
        "mouse_x",
        "blob_x",
        "audio_x",
    }

    if not required.issubset(
        dataframe.columns
    ):
        return False

    try:

        mask = dataframe.apply(
            is_fixed_offset_trial,
            axis=1,
        )

        return bool(
            mask.sum() > 0
        )

    except Exception:

        return False


USE_EXISTING_TRIAL_DF = (
    "trial_df" in globals()
    and existing_trial_df_is_suitable(
        trial_df
    )
)


if USE_EXISTING_TRIAL_DF:

    print(
        "\nUsing existing trial_df from notebook memory."
    )

    lag_source_df = (
        trial_df
        .copy()
    )

else:

    print(
        "\nCurrent trial_df does not contain the complete fixed-offset dataset."
    )

    print(
        "Reloading fixed-offset trials directly from master CSV files..."
    )

    csv_files = sorted(
        glob.glob(
            os.path.join(
                DATA_DIR,
                FILE_PATTERN,
            )
        )
    )

    if len(csv_files) == 0:

        raise FileNotFoundError(
            f"No files matching {FILE_PATTERN} found in:\n{DATA_DIR}"
        )

    trial_rows = []

    for filepath in csv_files:

        subject = extract_subject_id(
            filepath
        )

        raw = pd.read_csv(
            filepath
        )

        required_raw = [
            "trialIndex",
            "trialType",
            "condLabel",
            "t",
            "mouse_x",
            "blob_x",
            "audio_x",
        ]

        missing = [
            c
            for c in required_raw
            if c not in raw.columns
        ]

        if missing:

            print(
                f"Skipping {subject}: missing {missing}"
            )

            continue

        if (
            EXCLUDE_PRACTICE
            and "isPractice" in raw.columns
        ):

            practice_numeric = pd.to_numeric(
                raw["isPractice"],
                errors="coerce",
            )

            raw = raw[
                practice_numeric.fillna(0) == 0
            ].copy()

        # Keep fixed spatial AV trials only.
        raw = raw[
            raw["trialType"]
            .astype(str)
            .str.lower()
            .eq("incongruent_spatial")
        ].copy()

        if raw.empty:
            continue

        grouping_columns = [
            "trialIndex",
            "trialType",
            "condLabel",
        ]

        extra_columns = [
            c
            for c in [
                "offsetDeg",
                "sideSign",
                "rampType",
                "visualOn",
                "audioOn",
                "post_question",
                "post_question2",
            ]
            if c in raw.columns
        ]

        for group_keys, trial_raw in raw.groupby(
            grouping_columns,
            sort=False,
        ):

            trial_index, trial_type, cond_label = group_keys

            resampled = resample_trial(
                trial_raw,
                TIME_GRID,
            )

            if resampled is None:
                continue

            first = trial_raw.iloc[0]

            row = {
                "subject": subject,
                "trialIndex": trial_index,
                "trialType": trial_type,
                "condLabel": cond_label,
                "t_grid": resampled["t_grid"],
                "mouse_x": resampled["mouse_x"],
                "blob_x": resampled["blob_x"],
                "audio_x": resampled["audio_x"],
            }

            for column in extra_columns:

                row[column] = first[column]

            trial_rows.append(
                row
            )

    lag_source_df = pd.DataFrame(
        trial_rows
    )

    if lag_source_df.empty:

        raise RuntimeError(
            "No fixed incongruent spatial trials could be reconstructed."
        )


# =============================================================================
# STEP 2 — STANDARDIZE METADATA
# =============================================================================

lag_source_df = (
    lag_source_df
    .copy()
    .reset_index(drop=True)
)

lag_source_df["subject"] = (
    lag_source_df["subject"]
    .apply(normalize_subject)
)

lag_source_df["trial_id"] = (
    lag_source_df["trialIndex"]
    .apply(normalize_trial_id)
)

lag_source_df["offset_deg"] = (
    lag_source_df
    .apply(
        extract_offset_abs,
        axis=1,
    )
)

lag_source_df["side_label"] = (
    lag_source_df
    .apply(
        extract_side_label,
        axis=1,
    )
)


# Find / create same-response variable.
if "same_response_yes" in lag_source_df.columns:

    lag_source_df["same_response"] = pd.to_numeric(
        lag_source_df["same_response_yes"],
        errors="coerce",
    )

elif "post_question" in lag_source_df.columns:

    lag_source_df["same_response"] = (
        lag_source_df["post_question"]
        .apply(parse_same_response)
    )

else:

    raise RuntimeError(
        "Could not find same/different report. "
        "Need post_question or same_response_yes."
    )


lag_source_df["judgment"] = (
    lag_source_df["same_response"]
    .apply(judgment_label)
)


# Restrict to exact fixed offsets.
lag_source_df = lag_source_df[
    lag_source_df["offset_deg"]
    .isin(FIXED_OFFSETS)
].copy()


lag_source_df = lag_source_df[
    lag_source_df["same_response"]
    .isin([0.0, 1.0])
].copy()


lag_source_df = (
    lag_source_df
    .sort_values(
        [
            "subject",
            "trialIndex",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"\nFixed-offset trials with valid judgment: "
    f"{len(lag_source_df)}"
)

print(
    f"Participants: "
    f"{lag_source_df['subject'].nunique()}"
)

print("\nJudgment counts:")
print(
    lag_source_df[
        "judgment"
    ].value_counts()
)

print("\nJudgment counts by offset:")
print(
    pd.crosstab(
        lag_source_df[
            "offset_deg"
        ],
        lag_source_df[
            "judgment"
        ],
    )
)


# =============================================================================
# STEP 3 — TRIAL-LEVEL LAG EXTRACTION
# =============================================================================

trial_lag_rows = []


for _, row in lag_source_df.iterrows():

    t = safe_array(
        row["t_grid"]
    )

    response = safe_array(
        row["mouse_x"]
    )

    audio = safe_array(
        row["audio_x"]
    )

    visual = safe_array(
        row["blob_x"]
    )

    n = min(
        len(t),
        len(response),
        len(audio),
        len(visual),
    )

    if n < MIN_VALID_SAMPLES:
        continue

    t = t[:n]
    response = response[:n]
    audio = audio[:n]
    visual = visual[:n]

    (
        t_use,
        response_use,
        audio_use,
        visual_use,
    ) = restrict_analysis_window(
        t,
        response,
        audio,
        visual,
    )

    # ---------------------------------------------------------------------
    # PRIMARY lag:
    # Response relative to auditory trajectory.
    #
    # In fixed-offset trials the auditory and visual trajectories have the
    # same underlying motion structure with a spatial displacement, so their
    # temporal lag estimates should be highly similar. We still compute both
    # as a diagnostic.
    # ---------------------------------------------------------------------

    lag_audio_4s = lag_scanned_corr(
        response=response_use,
        stimulus=audio_use,
        fs=RESAMPLE_HZ,
        max_lag_sec=PRIMARY_MAX_LAG_SEC,
    )

    lag_visual_4s = lag_scanned_corr(
        response=response_use,
        stimulus=visual_use,
        fs=RESAMPLE_HZ,
        max_lag_sec=PRIMARY_MAX_LAG_SEC,
    )

    # Exact original-notebook ±2-s robustness.
    lag_audio_2s = lag_scanned_corr(
        response=response_use,
        stimulus=audio_use,
        fs=RESAMPLE_HZ,
        max_lag_sec=ORIGINAL_MAX_LAG_SEC,
    )

    lag_visual_2s = lag_scanned_corr(
        response=response_use,
        stimulus=visual_use,
        fs=RESAMPLE_HZ,
        max_lag_sec=ORIGINAL_MAX_LAG_SEC,
    )

    trial_lag_rows.append(
        {
            "subject": row["subject"],
            "trialIndex": row["trialIndex"],
            "trial_id": row["trial_id"],
            "trialType": row["trialType"],
            "condLabel": row.get(
                "condLabel",
                np.nan,
            ),
            "offset_deg": row["offset_deg"],
            "side_label": row["side_label"],
            "same_response": row["same_response"],
            "judgment": row["judgment"],

            # Primary auditory-relative lag
            "lag_audio_sec": lag_audio_4s["lag_sec"],
            "peak_audio_corr": lag_audio_4s["peak_corr"],
            "audio_boundary_hit": lag_audio_4s["boundary_hit"],

            # Visual-relative temporal diagnostic
            "lag_visual_sec": lag_visual_4s["lag_sec"],
            "peak_visual_corr": lag_visual_4s["peak_corr"],
            "visual_boundary_hit": lag_visual_4s["boundary_hit"],

            # Original ±2-s robustness
            "lag_audio_sec_max2": lag_audio_2s["lag_sec"],
            "peak_audio_corr_max2": lag_audio_2s["peak_corr"],
            "audio_boundary_hit_max2": lag_audio_2s["boundary_hit"],

            "lag_visual_sec_max2": lag_visual_2s["lag_sec"],
            "peak_visual_corr_max2": lag_visual_2s["peak_corr"],
            "visual_boundary_hit_max2": lag_visual_2s["boundary_hit"],
        }
    )


trial_lag_df = pd.DataFrame(
    trial_lag_rows
)


if trial_lag_df.empty:

    raise RuntimeError(
        "No valid trial-level lag estimates were produced."
    )


trial_lag_path = os.path.join(
    CSV_DIR,
    "trial_level_same_different_tracking_lag.csv",
)

trial_lag_df.to_csv(
    trial_lag_path,
    index=False,
)


print(
    f"\nSaved trial-level lag estimates:\n{trial_lag_path}"
)


# =============================================================================
# STEP 4 — BASIC DIAGNOSTICS
# =============================================================================

n_total = len(
    trial_lag_df
)

n_valid_lag = int(
    np.isfinite(
        trial_lag_df[
            "lag_audio_sec"
        ]
    ).sum()
)

n_boundary = int(
    trial_lag_df[
        "audio_boundary_hit"
    ]
    .fillna(False)
    .sum()
)


print("\n" + "=" * 80)
print("TRIAL-LEVEL LAG DIAGNOSTICS")
print("=" * 80)

print(
    f"\nValid auditory-relative lags: "
    f"{n_valid_lag}/{n_total}"
)

print(
    f"Primary ±{PRIMARY_MAX_LAG_SEC:g}-s "
    f"boundary hits: {n_boundary}/{n_total} "
    f"({100*n_boundary/max(n_total,1):.2f}%)"
)


# Correlation between lag computed relative to audio and vision.
lag_modality_diag = trial_lag_df[
    [
        "lag_audio_sec",
        "lag_visual_sec",
    ]
].dropna()


if len(lag_modality_diag) >= 3:

    r_av, p_av = stats.pearsonr(
        lag_modality_diag[
            "lag_audio_sec"
        ],
        lag_modality_diag[
            "lag_visual_sec"
        ],
    )

    print(
        "\nAudio-relative vs visual-relative lag diagnostic:"
    )

    print(
        f"r = {r_av:.4f}, "
        f"p {p_fmt(p_av)}, "
        f"N trials = {len(lag_modality_diag)}"
    )


# =============================================================================
# STEP 5 — DESCRIPTIVE RAW SAME VS DIFFERENT COMPARISON
# =============================================================================
#
# IMPORTANT:
# This is descriptive only because SAME and DIFFERENT frequencies vary
# strongly with physical disparity.
#
# Primary inference comes later from physically matched contrasts.
# =============================================================================

raw_descriptive_rows = []


for judgment in [
    "Same",
    "Different",
]:

    sub = trial_lag_df[
        trial_lag_df[
            "judgment"
        ] == judgment
    ]

    summary = mean_ci_t(
        sub[
            "lag_audio_sec"
        ]
    )

    raw_descriptive_rows.append(
        {
            "judgment": judgment,
            **summary,
            "mean_peak_corr": sub[
                "peak_audio_corr"
            ].mean(),
            "n_trials": len(sub),
        }
    )


raw_descriptive_df = pd.DataFrame(
    raw_descriptive_rows
)


raw_descriptive_df.to_csv(
    os.path.join(
        CSV_DIR,
        "descriptive_unmatched_same_different_lag.csv",
    ),
    index=False,
)


print("\n" + "=" * 80)
print("DESCRIPTIVE UNMATCHED SAME VS DIFFERENT")
print("DO NOT USE THIS AS THE PRIMARY CAUSAL-REPORT TEST")
print("=" * 80)

print(
    raw_descriptive_df.to_string(
        index=False
    )
)


# =============================================================================
# STEP 6 — PARTICIPANT × OFFSET × JUDGMENT SUMMARY
# =============================================================================

subject_offset_judgment_df = (
    trial_lag_df
    .groupby(
        [
            "subject",
            "offset_deg",
            "judgment",
        ],
        as_index=False,
    )
    .agg(
        n_trials=(
            "trialIndex",
            "count",
        ),
        lag_mean=(
            "lag_audio_sec",
            "mean",
        ),
        lag_median=(
            "lag_audio_sec",
            "median",
        ),
        peak_corr_mean=(
            "peak_audio_corr",
            "mean",
        ),
        lag_max2_mean=(
            "lag_audio_sec_max2",
            "mean",
        ),
    )
)


subject_offset_judgment_df.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_offset_judgment_lag_summary.csv",
    ),
    index=False,
)


# =============================================================================
# STEP 7 — EXACT MATCHED SAME-DIFFERENT CONTRAST WITHIN SUBJECT × OFFSET
# =============================================================================

matched_wide = (
    subject_offset_judgment_df
    .pivot_table(
        index=[
            "subject",
            "offset_deg",
        ],
        columns="judgment",
        values=[
            "lag_mean",
            "lag_max2_mean",
            "n_trials",
            "peak_corr_mean",
        ],
        aggfunc="first",
    )
)


# Flatten multi-index columns.
matched_wide.columns = [
    f"{metric}_{judgment}"
    for metric, judgment
    in matched_wide.columns
]

matched_wide = (
    matched_wide
    .reset_index()
)


required_matched_columns = [
    "lag_mean_Same",
    "lag_mean_Different",
]


for column in required_matched_columns:

    if column not in matched_wide.columns:

        matched_wide[column] = np.nan


matched_wide = matched_wide[
    np.isfinite(
        matched_wide[
            "lag_mean_Same"
        ]
    )
    & np.isfinite(
        matched_wide[
            "lag_mean_Different"
        ]
    )
].copy()


matched_wide[
    "lag_difference_different_minus_same"
] = (
    matched_wide[
        "lag_mean_Different"
    ]
    - matched_wide[
        "lag_mean_Same"
    ]
)


if (
    "lag_max2_mean_Different" in matched_wide.columns
    and "lag_max2_mean_Same" in matched_wide.columns
):

    matched_wide[
        "lag_difference_different_minus_same_max2"
    ] = (
        matched_wide[
            "lag_max2_mean_Different"
        ]
        - matched_wide[
            "lag_max2_mean_Same"
        ]
    )


matched_wide.to_csv(
    os.path.join(
        CSV_DIR,
        "matched_subject_offset_same_different_lag.csv",
    ),
    index=False,
)


print("\n" + "=" * 80)
print("MATCHED SUBJECT × OFFSET CELLS")
print("=" * 80)

print(
    f"\nMatched subject × offset cells: "
    f"{len(matched_wide)}"
)

print(
    f"Participants contributing ≥1 matched offset: "
    f"{matched_wide['subject'].nunique()}"
)


# =============================================================================
# STEP 8 — OFFSET-SPECIFIC MATCHED TESTS
# =============================================================================

offset_result_rows = []


for offset in FIXED_OFFSETS:

    sub = matched_wide[
        matched_wide[
            "offset_deg"
        ] == offset
    ].copy()

    diff = sub[
        "lag_difference_different_minus_same"
    ].dropna()

    summary = mean_ci_t(
        diff
    )

    if len(diff) >= 2:

        t_res = ttest_1samp(
            diff,
            popmean=0.0,
            nan_policy="omit",
        )

        perm = sign_flip_test(
            diff,
            popmean=0.0,
            alternative="greater",
            seed=RANDOM_SEED + int(offset),
        )

        dz = cohens_dz(
            diff
        )

    else:

        t_res = None

        perm = {
            "p_perm": np.nan,
        }

        dz = np.nan

    offset_result_rows.append(
        {
            "offset_deg": offset,
            "n_matched_subjects": summary["n"],
            "different_minus_same_mean_sec": summary["mean"],
            "ci_low_sec": summary["ci_low"],
            "ci_high_sec": summary["ci_high"],
            "sd_sec": summary["sd"],
            "se_sec": summary["se"],
            "t": (
                t_res.statistic
                if t_res is not None
                else np.nan
            ),
            "df": (
                len(diff) - 1
                if len(diff) >= 2
                else np.nan
            ),
            "p_two_sided_t": (
                t_res.pvalue
                if t_res is not None
                else np.nan
            ),
            "p_directional_signflip": perm["p_perm"],
            "cohens_dz": dz,
        }
    )


offset_results_df = pd.DataFrame(
    offset_result_rows
)


offset_results_df.to_csv(
    os.path.join(
        CSV_DIR,
        "offset_specific_same_different_lag_tests.csv",
    ),
    index=False,
)


print("\nOffset-specific matched effects:")
print(
    offset_results_df.to_string(
        index=False
    )
)


# =============================================================================
# STEP 9 — PRIMARY PARTICIPANT-LEVEL MATCHED EFFECT
# =============================================================================
#
# IMPORTANT:
# First average DIFFERENT-SAME contrasts over physically matched offsets
# WITHIN participant.
#
# Participants, not subject × offset cells, are the inferential unit.
# =============================================================================

participant_effect_df = (
    matched_wide
    .groupby(
        "subject",
        as_index=False,
    )
    .agg(
        n_matched_offsets=(
            "offset_deg",
            "nunique",
        ),
        lag_same_sec=(
            "lag_mean_Same",
            "mean",
        ),
        lag_different_sec=(
            "lag_mean_Different",
            "mean",
        ),
        lag_difference_different_minus_same=(
            "lag_difference_different_minus_same",
            "mean",
        ),
    )
)


participant_effect_df = participant_effect_df[
    participant_effect_df[
        "n_matched_offsets"
    ] >= MIN_MATCHED_OFFSETS_PER_SUBJECT
].copy()


participant_effect_df.to_csv(
    os.path.join(
        CSV_DIR,
        "participant_level_primary_same_different_lag_effect.csv",
    ),
    index=False,
)


primary_diff = participant_effect_df[
    "lag_difference_different_minus_same"
].dropna()


primary_summary = mean_ci_t(
    primary_diff
)


if len(primary_diff) >= 2:

    primary_t = ttest_1samp(
        primary_diff,
        popmean=0.0,
        nan_policy="omit",
    )

    primary_w = wilcoxon(
        primary_diff,
        alternative="greater",
        zero_method="wilcox",
    )

    primary_perm_two = sign_flip_test(
        primary_diff,
        alternative="two-sided",
        seed=RANDOM_SEED,
    )

    primary_perm_directional = sign_flip_test(
        primary_diff,
        alternative="greater",
        seed=RANDOM_SEED,
    )

else:

    primary_t = None
    primary_w = None

    primary_perm_two = {
        "p_perm": np.nan,
    }

    primary_perm_directional = {
        "p_perm": np.nan,
    }


primary_dz = cohens_dz(
    primary_diff
)


primary_results_df = pd.DataFrame(
    [
        {
            "analysis": (
                "participant-level matched fixed-offset effect"
            ),
            "contrast": (
                "Different minus Same tracking lag"
            ),
            "n_subjects": primary_summary["n"],
            "mean_difference_sec": primary_summary["mean"],
            "ci_low_sec": primary_summary["ci_low"],
            "ci_high_sec": primary_summary["ci_high"],
            "sd_difference_sec": primary_summary["sd"],
            "t": (
                primary_t.statistic
                if primary_t is not None
                else np.nan
            ),
            "df": (
                len(primary_diff) - 1
                if len(primary_diff) >= 2
                else np.nan
            ),
            "p_two_sided_t": (
                primary_t.pvalue
                if primary_t is not None
                else np.nan
            ),
            "p_two_sided_signflip": primary_perm_two[
                "p_perm"
            ],
            "p_directional_signflip": primary_perm_directional[
                "p_perm"
            ],
            "p_directional_wilcoxon": (
                primary_w.pvalue
                if primary_w is not None
                else np.nan
            ),
            "cohens_dz": primary_dz,
        }
    ]
)


primary_results_df.to_csv(
    os.path.join(
        CSV_DIR,
        "PRIMARY_participant_level_same_different_lag_test.csv",
    ),
    index=False,
)


print("\n" + "=" * 80)
print("PRIMARY PARTICIPANT-LEVEL MATCHED RESULT")
print("=" * 80)


same_summary = mean_ci_t(
    participant_effect_df[
        "lag_same_sec"
    ]
)

different_summary = mean_ci_t(
    participant_effect_df[
        "lag_different_sec"
    ]
)


print(
    f"\nSAME-source lag:"
    f"\n    M = {same_summary['mean']:.3f} s"
    f"\n    95% CI "
    f"[{same_summary['ci_low']:.3f}, "
    f"{same_summary['ci_high']:.3f}]"
)


print(
    f"\nDIFFERENT-source lag:"
    f"\n    M = {different_summary['mean']:.3f} s"
    f"\n    95% CI "
    f"[{different_summary['ci_low']:.3f}, "
    f"{different_summary['ci_high']:.3f}]"
)


print(
    f"\nDIFFERENT - SAME:"
    f"\n    M = {primary_summary['mean']:.3f} s"
    f"\n    95% CI "
    f"[{primary_summary['ci_low']:.3f}, "
    f"{primary_summary['ci_high']:.3f}]"
)


if primary_t is not None:

    print(
        f"\n    t({len(primary_diff)-1}) "
        f"= {primary_t.statistic:.3f}, "
        f"two-sided p {p_fmt(primary_t.pvalue)}"
    )


print(
    f"\n    directional sign-flip p "
    f"{p_fmt(primary_perm_directional['p_perm'])}"
)

print(
    f"    two-sided sign-flip p "
    f"{p_fmt(primary_perm_two['p_perm'])}"
)

print(
    f"    Cohen dz = {primary_dz:.3f}"
)


# =============================================================================
# STEP 10 — WHOLE-TRIAL JUDGMENT-LABEL PERMUTATION
# =============================================================================
#
# This is closely aligned with the judgment-label permutation logic already
# used in the manuscript.
#
# Labels are shuffled WITHIN:
#
#     participant × offset × side
#
# Complete trial-level lag estimates stay intact.
#
# This preserves:
#     - participant
#     - physical disparity
#     - stimulus side
#     - number of SAME / DIFFERENT trials in each stratum
#
# while breaking:
#     causal report <-> lag mapping.
#
# Statistic:
#     subject-first mean Different - Same lag,
#     averaged across matched offsets.
# =============================================================================

def compute_subject_first_matched_effect(
    dataframe,
    judgment_column="same_response",
):
    """
    Recreate the participant-level physically matched lag effect.
    """

    work = dataframe[
        [
            "subject",
            "offset_deg",
            "lag_audio_sec",
            judgment_column,
        ]
    ].copy()

    work = work[
        np.isfinite(
            pd.to_numeric(
                work["lag_audio_sec"],
                errors="coerce",
            )
        )
    ]

    work = work[
        work[
            judgment_column
        ].isin([0.0, 1.0])
    ]

    if work.empty:
        return pd.DataFrame()

    cell = (
        work
        .groupby(
            [
                "subject",
                "offset_deg",
                judgment_column,
            ],
            as_index=False,
        )
        .agg(
            lag_mean=(
                "lag_audio_sec",
                "mean",
            )
        )
    )

    wide = (
        cell
        .pivot_table(
            index=[
                "subject",
                "offset_deg",
            ],
            columns=judgment_column,
            values="lag_mean",
            aggfunc="first",
        )
        .reset_index()
    )

    if (
        0.0 not in wide.columns
        or 1.0 not in wide.columns
    ):
        return pd.DataFrame()

    wide = wide[
        np.isfinite(
            wide[0.0]
        )
        & np.isfinite(
            wide[1.0]
        )
    ].copy()

    if wide.empty:
        return pd.DataFrame()

    wide[
        "difference"
    ] = (
        wide[0.0]
        - wide[1.0]
    )

    subject_effect = (
        wide
        .groupby(
            "subject",
            as_index=False,
        )
        .agg(
            n_matched_offsets=(
                "offset_deg",
                "nunique",
            ),
            difference=(
                "difference",
                "mean",
            ),
        )
    )

    return subject_effect


observed_subject_effect = (
    compute_subject_first_matched_effect(
        trial_lag_df,
        judgment_column="same_response",
    )
)


observed_perm_stat = (
    observed_subject_effect[
        "difference"
    ].mean()
)


rng = np.random.default_rng(
    RANDOM_SEED
)


# Define physical matching strata.
perm_df = (
    trial_lag_df
    .copy()
)


perm_df[
    "perm_stratum"
] = (
    perm_df[
        "subject"
    ].astype(str)
    + "|"
    + perm_df[
        "offset_deg"
    ].astype(str)
    + "|"
    + perm_df[
        "side_label"
    ].astype(str)
)


group_indices = [
    np.asarray(
        indices,
        dtype=int,
    )
    for indices
    in perm_df.groupby(
        "perm_stratum"
    ).indices.values()
]


original_labels = (
    perm_df[
        "same_response"
    ]
    .to_numpy(
        dtype=float
    )
)


null_stats = np.full(
    N_PERM,
    np.nan,
    dtype=float,
)


print("\n" + "=" * 80)
print("WHOLE-TRIAL JUDGMENT-LABEL PERMUTATION")
print("=" * 80)

print(
    f"\nPermutations: {N_PERM}"
)

print(
    "Shuffling SAME/DIFFERENT labels within "
    "subject × offset × side."
)


for perm_i in range(
    N_PERM
):

    shuffled = (
        original_labels
        .copy()
    )

    for indices in group_indices:

        if len(indices) < 2:
            continue

        shuffled[
            indices
        ] = rng.permutation(
            shuffled[
                indices
            ]
        )

    perm_df[
        "_same_perm"
    ] = shuffled

    subject_effect_perm = (
        compute_subject_first_matched_effect(
            perm_df,
            judgment_column="_same_perm",
        )
    )

    if not subject_effect_perm.empty:

        null_stats[
            perm_i
        ] = subject_effect_perm[
            "difference"
        ].mean()


valid_null = null_stats[
    np.isfinite(
        null_stats
    )
]


if len(valid_null) > 0:

    perm_p_directional = (
        np.sum(
            valid_null
            >= observed_perm_stat
        )
        + 1
    ) / (
        len(valid_null) + 1
    )

    perm_p_two = (
        np.sum(
            np.abs(valid_null)
            >= abs(observed_perm_stat)
        )
        + 1
    ) / (
        len(valid_null) + 1
    )

else:

    perm_p_directional = np.nan
    perm_p_two = np.nan


whole_trial_perm_result_df = pd.DataFrame(
    [
        {
            "observed_mean_Different_minus_Same_sec": observed_perm_stat,
            "n_valid_permutations": len(valid_null),
            "p_directional_greater": perm_p_directional,
            "p_two_sided": perm_p_two,
            "shuffle_strata": (
                "subject x offset_deg x side_label"
            ),
        }
    ]
)


whole_trial_perm_result_df.to_csv(
    os.path.join(
        CSV_DIR,
        "whole_trial_judgment_label_permutation_result.csv",
    ),
    index=False,
)


pd.DataFrame(
    {
        "null_mean_Different_minus_Same_sec": valid_null
    }
).to_csv(
    os.path.join(
        CSV_DIR,
        "whole_trial_judgment_label_permutation_null.csv",
    ),
    index=False,
)


print(
    f"\nObserved participant-first effect = "
    f"{observed_perm_stat:.4f} s"
)

print(
    f"Directional whole-trial permutation p "
    f"{p_fmt(perm_p_directional)}"
)

print(
    f"Two-sided whole-trial permutation p "
    f"{p_fmt(perm_p_two)}"
)


# =============================================================================
# STEP 11 — MIXED-EFFECTS MODEL
# =============================================================================
#
# Trial-level model:
#
# lag ~ causal judgment + physical offset + side
#
# Random effects:
#     participant intercept + judgment slope
#
# If the random-slope model fails, automatically fall back to
# participant random intercept.
#
# Coding:
#
#     reported_different = 1 for DIFFERENT
#                          0 for SAME
#
# Therefore a POSITIVE coefficient means DIFFERENT trials have longer lag.
# =============================================================================

mixed_df = (
    trial_lag_df
    .copy()
)


mixed_df[
    "reported_different"
] = (
    1.0
    - mixed_df[
        "same_response"
    ]
)


mixed_df[
    "offset_factor"
] = (
    mixed_df[
        "offset_deg"
    ]
    .astype(int)
    .astype(str)
)


mixed_df = mixed_df.dropna(
    subset=[
        "lag_audio_sec",
        "reported_different",
        "subject",
        "offset_factor",
        "side_label",
    ]
).copy()


mixed_formula = (
    "lag_audio_sec ~ "
    "reported_different "
    "+ C(offset_factor) "
    "+ C(side_label)"
)


mixed_result = None
mixed_model_type = None
mixed_error = ""


# -------------------------------------------------------------------------
# Try random intercept + random judgment slope.
# -------------------------------------------------------------------------

try:

    mixed_model = smf.mixedlm(
        mixed_formula,
        data=mixed_df,
        groups=mixed_df[
            "subject"
        ],
        re_formula="~reported_different",
    )

    mixed_result = mixed_model.fit(
        reml=False,
        method="lbfgs",
        maxiter=2000,
        disp=False,
    )

    mixed_model_type = (
        "random intercept + judgment slope"
    )

except Exception as e:

    mixed_error = str(e)

    print(
        "\nRandom-slope MixedLM failed."
        "\nTrying random-intercept-only MixedLM..."
    )

    try:

        mixed_model = smf.mixedlm(
            mixed_formula,
            data=mixed_df,
            groups=mixed_df[
                "subject"
            ],
            re_formula="1",
        )

        mixed_result = mixed_model.fit(
            reml=False,
            method="lbfgs",
            maxiter=2000,
            disp=False,
        )

        mixed_model_type = (
            "random intercept only"
        )

    except Exception as e2:

        mixed_error += (
            "\nFallback error: "
            + str(e2)
        )

        mixed_result = None


if mixed_result is not None:

    print("\n" + "=" * 80)
    print("MIXED-EFFECTS MODEL")
    print("=" * 80)

    print(
        f"\nModel type: {mixed_model_type}"
    )

    print(
        f"Formula: {mixed_formula}"
    )

    print(
        mixed_result.summary()
    )

    coef = mixed_result.params.get(
        "reported_different",
        np.nan,
    )

    se = mixed_result.bse.get(
        "reported_different",
        np.nan,
    )

    z = (
        coef / se
        if np.isfinite(coef)
        and np.isfinite(se)
        and se > 0
        else np.nan
    )

    p = mixed_result.pvalues.get(
        "reported_different",
        np.nan,
    )

    ci = mixed_result.conf_int()

    if (
        "reported_different"
        in ci.index
    ):

        ci_low = ci.loc[
            "reported_different",
            0,
        ]

        ci_high = ci.loc[
            "reported_different",
            1,
        ]

    else:

        ci_low = np.nan
        ci_high = np.nan


    mixed_primary_df = pd.DataFrame(
        [
            {
                "model_type": mixed_model_type,
                "formula": mixed_formula,
                "n_trial_rows": len(mixed_df),
                "n_subjects": mixed_df[
                    "subject"
                ].nunique(),
                "coefficient_reported_different_sec": coef,
                "SE": se,
                "z": z,
                "p": p,
                "CI_low": ci_low,
                "CI_high": ci_high,
                "AIC": mixed_result.aic,
                "BIC": mixed_result.bic,
                "converged": getattr(
                    mixed_result,
                    "converged",
                    np.nan,
                ),
            }
        ]
    )

else:

    print(
        "\nWARNING: MixedLM could not be fit."
    )

    print(
        mixed_error
    )

    mixed_primary_df = pd.DataFrame(
        [
            {
                "model_type": "FAILED",
                "formula": mixed_formula,
                "error": mixed_error,
            }
        ]
    )


mixed_primary_df.to_csv(
    os.path.join(
        CSV_DIR,
        "mixed_effects_same_different_lag_model.csv",
    ),
    index=False,
)


# =============================================================================
# STEP 12 — CELL-AVERAGED TRAJECTORY ANALYSIS
# =============================================================================
#
# This is an especially important confirmatory analysis because trial-level
# cross-correlation maxima can be noisy.
#
# Procedure:
#
#   participant × offset × judgment
#       -> average raw response trajectory
#       -> average corresponding auditory trajectory
#       -> estimate ONE lag from the averaged trajectories
#
# Then:
#
#   Different lag - Same lag
#
# is calculated within participant × offset and averaged subject-first.
# =============================================================================

cell_trajectory_rows = []


for (
    subject,
    offset,
    judgment,
), group in lag_source_df.groupby(
    [
        "subject",
        "offset_deg",
        "judgment",
    ],
    sort=False,
):

    if (
        len(group)
        < MIN_TRIALS_FOR_CELL_AVERAGE
    ):
        continue

    response_list = []
    audio_list = []
    visual_list = []
    time_reference = None

    for _, row in group.iterrows():

        t = safe_array(
            row["t_grid"]
        )

        response = safe_array(
            row["mouse_x"]
        )

        audio = safe_array(
            row["audio_x"]
        )

        visual = safe_array(
            row["blob_x"]
        )

        n = min(
            len(t),
            len(response),
            len(audio),
            len(visual),
        )

        if n < MIN_VALID_SAMPLES:
            continue

        t = t[:n]
        response = response[:n]
        audio = audio[:n]
        visual = visual[:n]

        (
            t_use,
            response_use,
            audio_use,
            visual_use,
        ) = restrict_analysis_window(
            t,
            response,
            audio,
            visual,
        )

        if (
            len(response_use)
            < MIN_VALID_SAMPLES
        ):
            continue

        if time_reference is None:

            time_reference = (
                t_use.copy()
            )

        n_common = min(
            len(time_reference),
            len(response_use),
            len(audio_use),
            len(visual_use),
        )

        response_list.append(
            response_use[
                :n_common
            ]
        )

        audio_list.append(
            audio_use[
                :n_common
            ]
        )

        visual_list.append(
            visual_use[
                :n_common
            ]
        )

    if (
        len(response_list)
        < MIN_TRIALS_FOR_CELL_AVERAGE
    ):
        continue

    min_len = min(
        min(
            len(x)
            for x in response_list
        ),
        min(
            len(x)
            for x in audio_list
        ),
        min(
            len(x)
            for x in visual_list
        ),
    )

    response_matrix = np.vstack(
        [
            x[:min_len]
            for x in response_list
        ]
    )

    audio_matrix = np.vstack(
        [
            x[:min_len]
            for x in audio_list
        ]
    )

    visual_matrix = np.vstack(
        [
            x[:min_len]
            for x in visual_list
        ]
    )

    mean_response = np.nanmean(
        response_matrix,
        axis=0,
    )

    mean_audio = np.nanmean(
        audio_matrix,
        axis=0,
    )

    mean_visual = np.nanmean(
        visual_matrix,
        axis=0,
    )

    lag_audio = lag_scanned_corr(
        mean_response,
        mean_audio,
        fs=RESAMPLE_HZ,
        max_lag_sec=PRIMARY_MAX_LAG_SEC,
    )

    lag_visual = lag_scanned_corr(
        mean_response,
        mean_visual,
        fs=RESAMPLE_HZ,
        max_lag_sec=PRIMARY_MAX_LAG_SEC,
    )

    cell_trajectory_rows.append(
        {
            "subject": subject,
            "offset_deg": offset,
            "judgment": judgment,
            "n_trials": len(response_list),
            "lag_audio_sec": lag_audio[
                "lag_sec"
            ],
            "peak_audio_corr": lag_audio[
                "peak_corr"
            ],
            "audio_boundary_hit": lag_audio[
                "boundary_hit"
            ],
            "lag_visual_sec": lag_visual[
                "lag_sec"
            ],
            "peak_visual_corr": lag_visual[
                "peak_corr"
            ],
        }
    )


cell_trajectory_lag_df = pd.DataFrame(
    cell_trajectory_rows
)


cell_trajectory_lag_df.to_csv(
    os.path.join(
        CSV_DIR,
        "cell_averaged_trajectory_lag_by_subject_offset_judgment.csv",
    ),
    index=False,
)


if not cell_trajectory_lag_df.empty:

    cell_wide = (
        cell_trajectory_lag_df
        .pivot_table(
            index=[
                "subject",
                "offset_deg",
            ],
            columns="judgment",
            values="lag_audio_sec",
            aggfunc="first",
        )
        .reset_index()
    )

    if (
        "Same" in cell_wide.columns
        and "Different" in cell_wide.columns
    ):

        cell_wide = cell_wide[
            np.isfinite(
                cell_wide[
                    "Same"
                ]
            )
            & np.isfinite(
                cell_wide[
                    "Different"
                ]
            )
        ].copy()

        cell_wide[
            "lag_difference_different_minus_same"
        ] = (
            cell_wide[
                "Different"
            ]
            - cell_wide[
                "Same"
            ]
        )

        cell_subject_effect_df = (
            cell_wide
            .groupby(
                "subject",
                as_index=False,
            )
            .agg(
                n_matched_offsets=(
                    "offset_deg",
                    "nunique",
                ),
                lag_same_sec=(
                    "Same",
                    "mean",
                ),
                lag_different_sec=(
                    "Different",
                    "mean",
                ),
                lag_difference_different_minus_same=(
                    "lag_difference_different_minus_same",
                    "mean",
                ),
            )
        )

        cell_subject_effect_df.to_csv(
            os.path.join(
                CSV_DIR,
                "cell_averaged_PRIMARY_participant_lag_effect.csv",
            ),
            index=False,
        )

        cell_diff = (
            cell_subject_effect_df[
                "lag_difference_different_minus_same"
            ]
            .dropna()
        )

        cell_summary = mean_ci_t(
            cell_diff
        )

        if len(cell_diff) >= 2:

            cell_t = ttest_1samp(
                cell_diff,
                0.0,
                nan_policy="omit",
            )

            cell_perm = sign_flip_test(
                cell_diff,
                alternative="greater",
                seed=RANDOM_SEED + 999,
            )

            cell_dz = cohens_dz(
                cell_diff
            )

        else:

            cell_t = None

            cell_perm = {
                "p_perm": np.nan,
            }

            cell_dz = np.nan


        cell_results_df = pd.DataFrame(
            [
                {
                    "analysis": (
                        "cell-averaged trajectories"
                    ),
                    "n_subjects": cell_summary[
                        "n"
                    ],
                    "mean_Different_minus_Same_sec": cell_summary[
                        "mean"
                    ],
                    "CI_low": cell_summary[
                        "ci_low"
                    ],
                    "CI_high": cell_summary[
                        "ci_high"
                    ],
                    "t": (
                        cell_t.statistic
                        if cell_t is not None
                        else np.nan
                    ),
                    "df": (
                        len(cell_diff) - 1
                        if len(cell_diff) >= 2
                        else np.nan
                    ),
                    "p_two_sided_t": (
                        cell_t.pvalue
                        if cell_t is not None
                        else np.nan
                    ),
                    "p_directional_signflip": cell_perm[
                        "p_perm"
                    ],
                    "cohens_dz": cell_dz,
                }
            ]
        )

        cell_results_df.to_csv(
            os.path.join(
                CSV_DIR,
                "cell_averaged_trajectory_lag_test.csv",
            ),
            index=False,
        )


        print("\n" + "=" * 80)
        print("CELL-AVERAGED TRAJECTORY CONFIRMATORY RESULT")
        print("=" * 80)

        print(
            f"\nN participants = "
            f"{cell_summary['n']}"
        )

        print(
            f"Different - Same = "
            f"{cell_summary['mean']:.3f} s"
        )

        print(
            f"95% CI "
            f"[{cell_summary['ci_low']:.3f}, "
            f"{cell_summary['ci_high']:.3f}]"
        )

        print(
            f"Directional sign-flip p "
            f"{p_fmt(cell_perm['p_perm'])}"
        )


# =============================================================================
# STEP 13 — TRACKING-QUALITY SENSITIVITY ANALYSIS
# =============================================================================
#
# Primary analysis does NOT discard low-correlation trials.
#
# This sensitivity analysis asks whether the conclusion survives when only
# trials showing at least modest response-stimulus coupling are retained.
# =============================================================================

quality_df = trial_lag_df[
    np.isfinite(
        trial_lag_df[
            "peak_audio_corr"
        ]
    )
    & (
        trial_lag_df[
            "peak_audio_corr"
        ]
        >= MIN_PEAK_CORR_SENSITIVITY
    )
].copy()


quality_subject_effect = (
    compute_subject_first_matched_effect(
        quality_df,
        judgment_column="same_response",
    )
)


if not quality_subject_effect.empty:

    quality_diff = (
        quality_subject_effect[
            "difference"
        ]
        .dropna()
    )

    quality_summary = mean_ci_t(
        quality_diff
    )

    quality_perm = sign_flip_test(
        quality_diff,
        alternative="greater",
        seed=RANDOM_SEED + 2000,
    )

else:

    quality_diff = pd.Series(
        dtype=float
    )

    quality_summary = mean_ci_t(
        []
    )

    quality_perm = {
        "p_perm": np.nan,
    }


quality_result_df = pd.DataFrame(
    [
        {
            "peak_corr_threshold": MIN_PEAK_CORR_SENSITIVITY,
            "n_trials_retained": len(quality_df),
            "n_subjects_with_matched_effect": quality_summary[
                "n"
            ],
            "mean_Different_minus_Same_sec": quality_summary[
                "mean"
            ],
            "CI_low": quality_summary[
                "ci_low"
            ],
            "CI_high": quality_summary[
                "ci_high"
            ],
            "p_directional_signflip": quality_perm[
                "p_perm"
            ],
        }
    ]
)


quality_result_df.to_csv(
    os.path.join(
        CSV_DIR,
        "tracking_quality_sensitivity_analysis.csv",
    ),
    index=False,
)


# =============================================================================
# STEP 14 — ORIGINAL ±2-S SEARCH-WINDOW ROBUSTNESS
# =============================================================================

max2_cell = (
    trial_lag_df
    .groupby(
        [
            "subject",
            "offset_deg",
            "same_response",
        ],
        as_index=False,
    )
    .agg(
        lag_mean=(
            "lag_audio_sec_max2",
            "mean",
        )
    )
)


max2_wide = (
    max2_cell
    .pivot_table(
        index=[
            "subject",
            "offset_deg",
        ],
        columns="same_response",
        values="lag_mean",
        aggfunc="first",
    )
    .reset_index()
)


if (
    0.0 in max2_wide.columns
    and 1.0 in max2_wide.columns
):

    max2_wide = max2_wide[
        np.isfinite(
            max2_wide[0.0]
        )
        & np.isfinite(
            max2_wide[1.0]
        )
    ].copy()

    max2_wide[
        "difference"
    ] = (
        max2_wide[0.0]
        - max2_wide[1.0]
    )

    max2_subject = (
        max2_wide
        .groupby(
            "subject",
            as_index=False,
        )
        .agg(
            difference=(
                "difference",
                "mean",
            )
        )
    )

    max2_diff = (
        max2_subject[
            "difference"
        ]
        .dropna()
    )

    max2_summary = mean_ci_t(
        max2_diff
    )

    max2_perm = sign_flip_test(
        max2_diff,
        alternative="greater",
        seed=RANDOM_SEED + 3000,
    )

else:

    max2_summary = mean_ci_t(
        []
    )

    max2_perm = {
        "p_perm": np.nan,
    }


max2_result_df = pd.DataFrame(
    [
        {
            "lag_search_window_sec": ORIGINAL_MAX_LAG_SEC,
            "n_subjects": max2_summary[
                "n"
            ],
            "mean_Different_minus_Same_sec": max2_summary[
                "mean"
            ],
            "CI_low": max2_summary[
                "ci_low"
            ],
            "CI_high": max2_summary[
                "ci_high"
            ],
            "p_directional_signflip": max2_perm[
                "p_perm"
            ],
        }
    ]
)


max2_result_df.to_csv(
    os.path.join(
        CSV_DIR,
        "original_2sec_lag_window_robustness.csv",
    ),
    index=False,
)


# =============================================================================
# STEP 15 — SECONDARY ANALYSIS:
# DOES TRACKING LAG PREDICT CAUSAL REPORT ABOVE PHYSICAL DISPARITY?
# =============================================================================
#
# GEE logistic model:
#
#   P(SAME) ~ lag + offset + side
#
# clustered by participant.
#
# Prediction:
#
#   lag coefficient < 0
#
# Longer/slower tracking should predict a lower probability of reporting SAME.
#
# This is SECONDARY ASSOCIATIONAL inference:
# do not interpret it as lag causally producing the judgment.
# =============================================================================

gee_df = mixed_df[
    [
        "subject",
        "same_response",
        "lag_audio_sec",
        "offset_factor",
        "side_label",
        "peak_audio_corr",
    ]
].dropna(
    subset=[
        "subject",
        "same_response",
        "lag_audio_sec",
        "offset_factor",
        "side_label",
    ]
).copy()


gee_result = None
gee_error = ""


try:

    gee_model = smf.gee(
        formula=(
            "same_response ~ "
            "lag_audio_sec "
            "+ C(offset_factor) "
            "+ C(side_label)"
        ),
        groups="subject",
        data=gee_df,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable(),
    )

    gee_result = gee_model.fit()

except Exception as e:

    gee_error = str(e)


if gee_result is not None:

    print("\n" + "=" * 80)
    print("SECONDARY GEE — DOES LAG PREDICT CAUSAL REPORT?")
    print("=" * 80)

    print(
        gee_result.summary()
    )

    gee_coef = gee_result.params.get(
        "lag_audio_sec",
        np.nan,
    )

    gee_se = gee_result.bse.get(
        "lag_audio_sec",
        np.nan,
    )

    gee_p = gee_result.pvalues.get(
        "lag_audio_sec",
        np.nan,
    )

    gee_ci = gee_result.conf_int()

    if "lag_audio_sec" in gee_ci.index:

        gee_ci_low = gee_ci.loc[
            "lag_audio_sec",
            0,
        ]

        gee_ci_high = gee_ci.loc[
            "lag_audio_sec",
            1,
        ]

    else:

        gee_ci_low = np.nan
        gee_ci_high = np.nan


    odds_ratio = (
        np.exp(gee_coef)
        if np.isfinite(gee_coef)
        else np.nan
    )


    gee_summary_df = pd.DataFrame(
        [
            {
                "outcome": "P(Same)",
                "predictor": "tracking lag (seconds)",
                "coefficient_log_odds": gee_coef,
                "SE": gee_se,
                "p": gee_p,
                "CI_low_log_odds": gee_ci_low,
                "CI_high_log_odds": gee_ci_high,
                "odds_ratio_per_1sec_lag": odds_ratio,
                "n_trials": len(gee_df),
                "n_subjects": gee_df[
                    "subject"
                ].nunique(),
            }
        ]
    )

else:

    print(
        "\nWARNING: secondary GEE failed:"
    )

    print(
        gee_error
    )

    gee_summary_df = pd.DataFrame(
        [
            {
                "outcome": "P(Same)",
                "predictor": "tracking lag (seconds)",
                "error": gee_error,
            }
        ]
    )


gee_summary_df.to_csv(
    os.path.join(
        CSV_DIR,
        "secondary_GEE_lag_predicts_same_report.csv",
    ),
    index=False,
)


# =============================================================================
# STEP 16 — PLOTS
# =============================================================================

# -------------------------------------------------------------------------
# FIGURE 1:
# Participant-level paired SAME vs DIFFERENT lag.
# -------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(7.0, 6.0)
)


for _, row in participant_effect_df.iterrows():

    if (
        np.isfinite(
            row["lag_same_sec"]
        )
        and np.isfinite(
            row["lag_different_sec"]
        )
    ):

        ax.plot(
            [0, 1],
            [
                row["lag_same_sec"],
                row["lag_different_sec"],
            ],
            marker="o",
            alpha=0.30,
            linewidth=1.0,
        )


group_means = [
    participant_effect_df[
        "lag_same_sec"
    ].mean(),
    participant_effect_df[
        "lag_different_sec"
    ].mean(),
]


ax.plot(
    [0, 1],
    group_means,
    marker="o",
    linewidth=3.0,
    markersize=9,
    label="Group mean",
)


ax.set_xticks(
    [0, 1]
)

ax.set_xticklabels(
    [
        "Same source",
        "Different sources",
    ]
)

ax.set_ylabel(
    "Tracking lag relative to auditory trajectory (s)"
)

ax.set_title(
    "Physically matched tracking lag by causal judgment"
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

ax.legend(
    frameon=False
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "participant_paired_same_different_tracking_lag.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


# -------------------------------------------------------------------------
# FIGURE 2:
# Distribution of primary participant-level DIFFERENT - SAME effect.
# -------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(7.0, 5.5)
)


ax.hist(
    primary_diff,
    bins=min(
        15,
        max(
            5,
            len(primary_diff) // 3,
        ),
    ),
    alpha=0.75,
)


ax.axvline(
    0.0,
    linestyle="--",
    linewidth=1.5,
)


if np.isfinite(
    primary_summary[
        "mean"
    ]
):

    ax.axvline(
        primary_summary[
            "mean"
        ],
        linewidth=2.5,
        label=(
            "Mean difference = "
            f"{primary_summary['mean']:.3f} s"
        ),
    )


ax.set_xlabel(
    "Different − Same tracking lag (s)"
)

ax.set_ylabel(
    "Number of participants"
)

ax.set_title(
    "Participant-level causal-judgment lag effect"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "participant_lag_difference_histogram.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


# -------------------------------------------------------------------------
# FIGURE 3:
# Offset-resolved physically matched effect.
# -------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.0, 5.5)
)


plot_offsets = []
plot_means = []
plot_low = []
plot_high = []


for _, row in offset_results_df.iterrows():

    if not np.isfinite(
        row[
            "different_minus_same_mean_sec"
        ]
    ):
        continue

    plot_offsets.append(
        row["offset_deg"]
    )

    plot_means.append(
        row[
            "different_minus_same_mean_sec"
        ]
    )

    plot_low.append(
        row[
            "different_minus_same_mean_sec"
        ]
        - row["ci_low_sec"]
    )

    plot_high.append(
        row["ci_high_sec"]
        - row[
            "different_minus_same_mean_sec"
        ]
    )


if len(plot_offsets) > 0:

    ax.errorbar(
        plot_offsets,
        plot_means,
        yerr=[
            plot_low,
            plot_high,
        ],
        marker="o",
        capsize=4,
        linewidth=2,
    )


ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
)


ax.set_xticks(
    FIXED_OFFSETS
)

ax.set_xlabel(
    "Audiovisual disparity (deg)"
)

ax.set_ylabel(
    "Different − Same tracking lag (s)"
)

ax.set_title(
    "Causal-judgment tracking-lag effect by physical disparity"
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "offset_resolved_same_different_lag_effect.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


# -------------------------------------------------------------------------
# FIGURE 4:
# Whole-trial permutation null distribution.
# -------------------------------------------------------------------------

if len(valid_null) > 0:

    fig, ax = plt.subplots(
        figsize=(7.0, 5.5)
    )

    ax.hist(
        valid_null,
        bins=40,
        alpha=0.75,
    )

    ax.axvline(
        observed_perm_stat,
        linewidth=2.5,
        label=(
            "Observed = "
            f"{observed_perm_stat:.3f} s"
        ),
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_xlabel(
        "Permuted participant-first Different − Same lag (s)"
    )

    ax.set_ylabel(
        "Permutation count"
    )

    ax.set_title(
        "Whole-trial judgment-label permutation null"
    )

    ax.legend(
        frameon=False
    )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            FIG_DIR,
            "whole_trial_judgment_permutation_null.png",
        ),
        dpi=FIG_DPI,
        bbox_inches="tight",
    )

    plt.show()


# =============================================================================
# STEP 17 — COMPACT MASTER SUMMARY
# =============================================================================

master_summary_rows = []


master_summary_rows.append(
    {
        "analysis": (
            "PRIMARY participant matched"
        ),
        "n_subjects": primary_summary[
            "n"
        ],
        "effect_Different_minus_Same_sec": primary_summary[
            "mean"
        ],
        "CI_low": primary_summary[
            "ci_low"
        ],
        "CI_high": primary_summary[
            "ci_high"
        ],
        "p_directional": primary_perm_directional[
            "p_perm"
        ],
        "notes": (
            "Subject-first mean across physically matched offsets"
        ),
    }
)


master_summary_rows.append(
    {
        "analysis": (
            "whole-trial judgment permutation"
        ),
        "n_subjects": observed_subject_effect[
            "subject"
        ].nunique(),
        "effect_Different_minus_Same_sec": observed_perm_stat,
        "CI_low": np.nan,
        "CI_high": np.nan,
        "p_directional": perm_p_directional,
        "notes": (
            "Judgment labels shuffled within subject x offset x side"
        ),
    }
)


master_summary_rows.append(
    {
        "analysis": (
            "tracking-quality sensitivity"
        ),
        "n_subjects": quality_summary[
            "n"
        ],
        "effect_Different_minus_Same_sec": quality_summary[
            "mean"
        ],
        "CI_low": quality_summary[
            "ci_low"
        ],
        "CI_high": quality_summary[
            "ci_high"
        ],
        "p_directional": quality_perm[
            "p_perm"
        ],
        "notes": (
            f"Peak correlation >= "
            f"{MIN_PEAK_CORR_SENSITIVITY}"
        ),
    }
)


master_summary_rows.append(
    {
        "analysis": (
            "original +/-2 s lag search"
        ),
        "n_subjects": max2_summary[
            "n"
        ],
        "effect_Different_minus_Same_sec": max2_summary[
            "mean"
        ],
        "CI_low": max2_summary[
            "ci_low"
        ],
        "CI_high": max2_summary[
            "ci_high"
        ],
        "p_directional": max2_perm[
            "p_perm"
        ],
        "notes": (
            "Exact original notebook lag-search range"
        ),
    }
)


if (
    "cell_results_df" in globals()
    and not cell_results_df.empty
):

    r0 = cell_results_df.iloc[0]

    master_summary_rows.append(
        {
            "analysis": (
                "cell-averaged trajectory"
            ),
            "n_subjects": r0[
                "n_subjects"
            ],
            "effect_Different_minus_Same_sec": r0[
                "mean_Different_minus_Same_sec"
            ],
            "CI_low": r0[
                "CI_low"
            ],
            "CI_high": r0[
                "CI_high"
            ],
            "p_directional": r0[
                "p_directional_signflip"
            ],
            "notes": (
                "Lag estimated after averaging raw trajectories within "
                "subject x offset x judgment"
            ),
        }
    )


master_summary_df = pd.DataFrame(
    master_summary_rows
)


master_summary_df.to_csv(
    os.path.join(
        CSV_DIR,
        "MASTER_same_different_lag_analysis_summary.csv",
    ),
    index=False,
)


print("\n" + "=" * 80)
print("MASTER ANALYSIS SUMMARY")
print("=" * 80)

print(
    master_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# STEP 18 — AUTOMATIC INTERPRETATION CHECK
# =============================================================================

print("\n" + "=" * 80)
print("AUTOMATIC INTERPRETATION")
print("=" * 80)


if np.isfinite(
    primary_summary[
        "mean"
    ]
):

    if (
        primary_summary[
            "mean"
        ] > 0
    ):

        print(
            "\nDirection matches the dynamic-BCI prediction:"
        )

        print(
            "DIFFERENT-source trials show LONGER tracking lag than "
            "SAME-source trials after matching physical disparity."
        )

    elif (
        primary_summary[
            "mean"
        ] < 0
    ):

        print(
            "\nDirection is opposite the dynamic-BCI prediction:"
        )

        print(
            "DIFFERENT-source trials show SHORTER tracking lag than "
            "SAME-source trials."
        )

    else:

        print(
            "\nNo mean SAME/DIFFERENT lag difference was observed."
        )


print(
    "\nPrimary manuscript-level result should be based on:"
)

print(
    "  1. participant-level physically matched effect"
)

print(
    "  2. whole-trial judgment-label permutation"
)

print(
    "  3. cell-averaged trajectory confirmation"
)

print(
    "\nMixedLM, quality threshold, ±2-s search, and GEE are "
    "supporting/sensitivity analyses."
)


print("\nFiles saved to:")
print(
    OUT_DIR
)

print("\nDone.")


SAME VS DIFFERENT SOURCE — TRACKING LAG ANALYSIS

Data directory:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL

Output directory:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_same_different_tracking_lag_analysis

Primary lag search: ±4.0 s
Original-notebook robustness lag search: ±2.0 s

Primary directional prediction:
    Lag(Different) - Lag(Same) > 0

Using existing trial_df from notebook memory.

Fixed-offset trials with valid judgment: 5000
Participants: 50

Judgment counts:
judgment
Different    3016
Same         1984
Name: count, dtype: int64

Judgment counts by offset:
judgment    Different  Same
offset_deg                 
10.0              268   732
20.0              409   591
30.0              693   307
40.0             1646   354

Saved trial-level lag estimates:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_same_dif

C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 128.265913
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)



MIXED-EFFECTS MODEL

Model type: random intercept + judgment slope
Formula: lag_audio_sec ~ reported_different + C(offset_factor) + C(side_label)
                  Mixed Linear Model Regression Results
Model:                  MixedLM     Dependent Variable:     lag_audio_sec
No. Observations:       4942        Method:                 ML           
No. Groups:             50          Scale:                  5.2663       
Min. group size:        59          Log-Likelihood:         -11186.3088  
Max. group size:        100         Converged:              No           
Mean group size:        98.8                                             
-------------------------------------------------------------------------
                               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                       0.667    0.116  5.758 0.000  0.440  0.894
C(offset_factor)[T.20]         -0.073    0.104 -0.699 0.4

In [7]:
# %% EXACT COLLABORATOR-MATCHED SAME/DIFFERENT AUDITORY TRACKING LAG ANALYSIS
# =============================================================================
#
# PURPOSE
# =============================================================================
# Reproduce behavioral lag analysis as closely as possible and
# then perform a stronger physically matched Same-vs-Different comparison.
#
#
# EXACT PRIMARY LAG PROCEDURE
# =============================================================================
#
# 1. Resample every trajectory to 20 Hz.
# 2. Use RAW / un-lag-corrected mouse responses.
# 3. Cross-correlate response against the AUDITORY trajectory.
# 4. Search ONLY non-negative lags:
#
#        0.00, 0.05, 0.10, ... 4.00 s
#
# 5. At lag k:
#
#        corr(response[t+k], auditory[t])
#
# 6. Trial lag = k producing maximum Pearson r.
# 7. DISCARD trial if:
#
#        peak r <= .40
#
# 8. DISCARD trial if:
#
#        SD(auditory position) < 3 degrees
#
#
# INCLUDED AUDIOVISUAL CONDITIONS
# =============================================================================
#
#     0° congruent
#     fixed 10°
#     fixed 20°
#     fixed 30°
#     fixed 40°
#     sweep diverge
#     sweep converge
#
#
# TWO PRIMARY OUTPUTS
# =============================================================================
#
# ANALYSIS 1 — EXACT COLLABORATOR REPLICATION
#
#     Within each participant:
#         median lag across all valid SAME trials
#         median lag across all valid DIFFERENT trials
#
#     Then summarize participant medians across observers.
#
#     This is the closest direct behavioral comparison with the collaborator's
#     reported approximately:
#
#         SAME      ≈ 0.62 s
#         DIFFERENT ≈ 1.21 s
#
#
# ANALYSIS 2 — PHYSICALLY MATCHED CAUSAL-STATE TEST
#
#     Within participant × physical condition:
#         median SAME lag
#         median DIFFERENT lag
#
#     Contrast:
#
#         Different - Same
#
#     Then summarize the condition-matched contrasts within participant
#     before group-level inference.
#
#     This asks whether causal judgment predicts tracking latency ABOVE AND
#     BEYOND differences in the physical audiovisual condition.
#
#
# SIGN CONVENTION
# =============================================================================
#
#     Different - Same > 0
#
# means DIFFERENT-source trials have LONGER tracking lag.
#
# This is the dynamic-BCI prediction described by the collaborator.
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import (
    ttest_1samp,
    wilcoxon,
    t as t_dist,
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)


# =============================================================================
# SETTINGS
# =============================================================================

try:
    DATA_DIR
except NameError:
    DATA_DIR = (
        r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
        r"\Continuous AVT v1\individual_data_SL"
    )

try:
    FILE_PATTERN
except NameError:
    FILE_PATTERN = "*master_trials*.csv"


# -----------------------------------------------------------------------------
# Exact collaborator sampling rate
# -----------------------------------------------------------------------------

RESAMPLE_HZ = 20.0
DT_SEC = 1.0 / RESAMPLE_HZ

TRIAL_DURATION_SEC = 18.0

TIME_GRID = np.arange(
    0.0,
    TRIAL_DURATION_SEC + DT_SEC,
    DT_SEC,
)


# -----------------------------------------------------------------------------
# Exact collaborator lag search
# -----------------------------------------------------------------------------

MIN_LAG_SEC = 0.0
MAX_LAG_SEC = 4.0

# IMPORTANT:
# Collaborator said:
# "Trials are discarded if the peak r doesn't exceed 0.4"
#
# Therefore:
#     valid iff peak r > .40
#
MIN_PEAK_CORR = 0.40


# -----------------------------------------------------------------------------
# Minimum movement criterion
# -----------------------------------------------------------------------------

MIN_STIMULUS_SD_DEG = 3.0


# -----------------------------------------------------------------------------
# Temporal analysis interval
#
# This currently uses the entire 0–18 s behavioral trial.
# If Haocheng later confirms that he cropped to another interval,
# change these two values only.
# -----------------------------------------------------------------------------

ANALYSIS_START_SEC = 0.0
ANALYSIS_END_SEC = 18.0


# -----------------------------------------------------------------------------
# Minimum number of overlapping samples needed for a correlation
# -----------------------------------------------------------------------------

MIN_VALID_SAMPLES = 20


# -----------------------------------------------------------------------------
# Audiovisual conditions
# -----------------------------------------------------------------------------

FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]

PHYSICAL_CONDITION_ORDER = [
    "congruent",
    "fixed_10",
    "fixed_20",
    "fixed_30",
    "fixed_40",
    "sweep_diverge",
    "sweep_converge",
]

PHYSICAL_CONDITION_LABELS = {
    "congruent": "0° congruent",
    "fixed_10": "10° fixed",
    "fixed_20": "20° fixed",
    "fixed_30": "30° fixed",
    "fixed_40": "40° fixed",
    "sweep_diverge": "Sweep diverge",
    "sweep_converge": "Sweep converge",
}

AV_TRIAL_TYPES_TO_KEEP = [
    "congruent",
    "incongruent_spatial",
    "incongruent_sweep",
]


EXCLUDE_PRACTICE = True


# -----------------------------------------------------------------------------
# Physically matched analysis
#
# At least one condition containing BOTH Same and Different judgments is needed
# for a participant to contribute to the matched analysis.
# -----------------------------------------------------------------------------

MIN_MATCHED_CONDITIONS_PER_SUBJECT = 1


# -----------------------------------------------------------------------------
# Permutations / bootstrap
# -----------------------------------------------------------------------------

N_PERM = 10000
N_BOOT = 10000
RANDOM_SEED = 12345


# -----------------------------------------------------------------------------
# Figures
# -----------------------------------------------------------------------------

FIG_DPI = 180


# =============================================================================
# OUTPUT DIRECTORIES
# =============================================================================

SPATIAL_ANALYSES_ROOT = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses"
OUT_DIR = os.path.join(
    SPATIAL_ANALYSES_ROOT,
    "02_same_different_tracking_lag",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures",
)

for directory in [
    OUT_DIR,
    CSV_DIR,
    FIG_DIR,
]:
    os.makedirs(
        directory,
        exist_ok=True,
    )


print("\n" + "=" * 100)
print("EXACT COLLABORATOR-MATCHED SAME/DIFFERENT TRACKING LAG ANALYSIS")
print("=" * 100)

print(
    f"\nData directory:\n{DATA_DIR}"
)

print(
    f"\nOutput directory:\n{OUT_DIR}"
)

print(
    "\nPRIMARY LAG ESTIMATOR:"
)

print(
    f"  Sampling rate                 = {RESAMPLE_HZ:.1f} Hz"
)

print(
    f"  Lag search                    = {MIN_LAG_SEC:.2f} to "
    f"{MAX_LAG_SEC:.2f} s"
)

print(
    f"  Lag increment                 = {DT_SEC:.2f} s"
)

print(
    f"  Minimum accepted peak r       > {MIN_PEAK_CORR:.2f}"
)

print(
    f"  Minimum auditory stimulus SD >= {MIN_STIMULUS_SD_DEG:.1f}°"
)

print(
    "  Within-observer summary       = MEDIAN"
)

print(
    "\nMODEL-PREDICTED DIRECTION:"
)

print(
    "  Different - Same > 0"
)


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def normalize_subject(value):

    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    match = re.search(
        r"pid\s*0*(\d+)",
        text,
    )

    if match:

        return f"pid{int(match.group(1)):03d}"

    match = re.search(
        r"(\d+)",
        text,
    )

    if match:

        return f"pid{int(match.group(1)):03d}"

    return text


def normalize_trial_id(value):

    if pd.isna(value):
        return np.nan

    return re.sub(
        r"\.0$",
        "",
        str(value).strip(),
    )


def extract_subject_id(filepath):

    name = Path(filepath).name

    match = re.search(
        r"(pid\d+)",
        name,
        flags=re.IGNORECASE,
    )

    if match:

        return normalize_subject(
            match.group(1)
        )

    return Path(filepath).stem


def safe_array(value):

    try:

        return np.asarray(
            value,
            dtype=float,
        ).reshape(-1)

    except Exception:

        return np.asarray(
            [],
            dtype=float,
        )


def parse_same_response(value):
    """
    Output:
        1 = SAME / common source
        0 = DIFFERENT / separate sources
    """

    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    same_tokens = [
        "same",
        "same source",
        "common",
        "common source",
        "together",
        "one",
        "1",
        "yes",
        "y",
        "true",
    ]

    different_tokens = [
        "different",
        "diff",
        "different source",
        "separate",
        "separate source",
        "apart",
        "two",
        "2",
        "0",
        "no",
        "n",
        "false",
    ]

    if text in same_tokens:
        return 1.0

    if text in different_tokens:
        return 0.0

    if (
        "same" in text
        or "common" in text
    ):
        return 1.0

    if (
        "different" in text
        or "separate" in text
        or "diff" in text
    ):
        return 0.0

    return np.nan


def judgment_label(value):

    try:
        value = float(value)

    except Exception:
        return np.nan

    if not np.isfinite(value):
        return np.nan

    return (
        "Same"
        if int(round(value)) == 1
        else "Different"
    )


# =============================================================================
# PHYSICAL-CONDITION HELPERS
# =============================================================================

def extract_offset_abs(row):

    trial_type = str(
        row.get(
            "trialType",
            "",
        )
    ).strip().lower()

    if trial_type == "congruent":
        return 0.0

    if trial_type == "incongruent_sweep":
        return np.nan

    if (
        "offsetDeg" in row.index
        and pd.notna(
            row["offsetDeg"]
        )
    ):

        try:

            value = abs(
                float(
                    row["offsetDeg"]
                )
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

    condition_text = str(
        row.get(
            "condLabel",
            "",
        )
    ).lower()

    match = re.search(
        r"(10|20|30|40)",
        condition_text,
    )

    if match:

        return float(
            match.group(1)
        )

    return np.nan


def extract_side_label(row):

    trial_type = str(
        row.get(
            "trialType",
            "",
        )
    ).strip().lower()

    if trial_type == "congruent":
        return "center"

    if (
        "sideSign" in row.index
        and pd.notna(
            row["sideSign"]
        )
    ):

        try:

            side = float(
                row["sideSign"]
            )

            if side < 0:
                return "left"

            if side > 0:
                return "right"

            return "center"

        except Exception:
            pass

    text = (
        f"{row.get('trialType', '')} "
        f"{row.get('condLabel', '')} "
        f"{row.get('rampType', '')}"
    ).lower()

    if (
        "left" in text
        or "negative" in text
        or "minus" in text
    ):
        return "left"

    if (
        "right" in text
        or "positive" in text
        or "plus" in text
    ):
        return "right"

    return "unknown"


def classify_sweep_direction(row):

    text = (
        f"{row.get('trialType', '')} "
        f"{row.get('condLabel', '')} "
        f"{row.get('rampType', '')}"
    ).strip().lower()

    if "diverge" in text:
        return "sweep_diverge"

    if "converge" in text:
        return "sweep_converge"

    if (
        "0_to_40" in text
        or "0to40" in text
        or "0->40" in text
        or "0 → 40" in text
        or "outward" in text
        or "apart" in text
    ):
        return "sweep_diverge"

    if (
        "40_to_0" in text
        or "40to0" in text
        or "40->0" in text
        or "40 → 0" in text
        or "inward" in text
        or "together" in text
    ):
        return "sweep_converge"

    return np.nan


def classify_physical_condition(row):

    trial_type = str(
        row.get(
            "trialType",
            "",
        )
    ).strip().lower()

    cond = str(
        row.get(
            "condLabel",
            "",
        )
    ).strip().lower()

    # Congruent
    if trial_type == "congruent":
        return "congruent"

    if (
        cond == "congruent"
        and trial_type != "incongruent_spatial"
        and trial_type != "incongruent_sweep"
    ):
        return "congruent"

    # Sweep
    if (
        trial_type == "incongruent_sweep"
        or "sweep" in trial_type
        or "sweep" in cond
        or "diverge" in cond
        or "converge" in cond
    ):

        return classify_sweep_direction(
            row
        )

    # Fixed
    if (
        trial_type == "incongruent_spatial"
        or "fixed" in trial_type
        or "static" in trial_type
    ):

        offset = extract_offset_abs(
            row
        )

        if np.isfinite(offset):

            offset_int = int(
                round(offset)
            )

            if offset_int in FIXED_OFFSETS:

                return f"fixed_{offset_int}"

    return np.nan


def is_model_matched_av_trial(row):

    condition = classify_physical_condition(
        row
    )

    return (
        condition
        in PHYSICAL_CONDITION_ORDER
    )


# =============================================================================
# RESAMPLING
# =============================================================================

def resample_trial(
    df_trial,
    time_grid,
):

    needed_columns = [
        "t",
        "mouse_x",
        "blob_x",
        "audio_x",
    ]

    for column in needed_columns:

        if column not in df_trial.columns:

            raise ValueError(
                f"Missing required column: {column}"
            )

    d = (
        df_trial
        .copy()
        .sort_values("t")
    )

    d["t"] = pd.to_numeric(
        d["t"],
        errors="coerce",
    )

    d = d[
        np.isfinite(
            d["t"]
        )
    ]

    d = d.drop_duplicates(
        subset="t",
        keep="first",
    )

    if len(d) < 5:
        return None

    t_original = (
        d["t"]
        .to_numpy(
            dtype=float
        )
    )

    t_original = (
        t_original
        - np.nanmin(
            t_original
        )
    )

    if (
        len(t_original) < 3
        or np.nanmax(
            t_original
        ) < 1.0
    ):
        return None

    def interpolate(column):

        y = pd.to_numeric(
            d[column],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        valid = (
            np.isfinite(t_original)
            & np.isfinite(y)
        )

        if valid.sum() < 3:

            return np.full(
                len(time_grid),
                np.nan,
                dtype=float,
            )

        return np.interp(
            time_grid,
            t_original[valid],
            y[valid],
            left=np.nan,
            right=np.nan,
        )

    return {
        "t_grid": np.asarray(
            time_grid,
            dtype=float,
        ).copy(),

        "mouse_x": interpolate(
            "mouse_x"
        ),

        "blob_x": interpolate(
            "blob_x"
        ),

        "audio_x": interpolate(
            "audio_x"
        ),
    }


# =============================================================================
# ANALYSIS WINDOW
# =============================================================================

def restrict_analysis_window(
    time_grid,
    *arrays,
    start_sec=ANALYSIS_START_SEC,
    end_sec=ANALYSIS_END_SEC,
):

    t = np.asarray(
        time_grid,
        dtype=float,
    )

    n = len(t)

    arrays = [
        np.asarray(
            array,
            dtype=float,
        )[:n]
        for array in arrays
    ]

    valid_time = (
        np.isfinite(t)
        & (
            t >= start_sec
        )
        & (
            t <= end_sec
        )
    )

    output = [
        t[
            valid_time
        ]
    ]

    for array in arrays:

        output.append(
            array[
                valid_time
            ]
        )

    return tuple(
        output
    )


# =============================================================================
# EXACT COLLABORATOR POSITIVE-LAG CROSS-CORRELATION
# =============================================================================

def pearson_corr(
    x,
    y,
    min_valid_samples=MIN_VALID_SAMPLES,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    n = min(
        len(x),
        len(y),
    )

    x = x[:n]
    y = y[:n]

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    if valid.sum() < min_valid_samples:
        return np.nan

    x = x[
        valid
    ]

    y = y[
        valid
    ]

    if (
        np.std(
            x,
            ddof=0,
        ) < 1e-12
        or np.std(
            y,
            ddof=0,
        ) < 1e-12
    ):
        return np.nan

    return float(
        np.corrcoef(
            x,
            y,
        )[0, 1]
    )


def positive_lag_crosscorr(
    response,
    stimulus,
    fs=RESAMPLE_HZ,
    max_lag_sec=MAX_LAG_SEC,
    min_valid_samples=MIN_VALID_SAMPLES,
):
    """
    Exact lag convention described by collaborator:

        corr(response[t+k], stimulus[t])

    Search:

        k = 0, 1, ..., 4*fs

    Positive lag means RESPONSE LAGS STIMULUS.

    NO NEGATIVE LAGS ARE CONSIDERED.
    """

    response = np.asarray(
        response,
        dtype=float,
    )

    stimulus = np.asarray(
        stimulus,
        dtype=float,
    )

    n = min(
        len(response),
        len(stimulus),
    )

    response = response[:n]
    stimulus = stimulus[:n]

    max_lag_samples = int(
        round(
            max_lag_sec
            * fs
        )
    )

    lag_samples_grid = np.arange(
        0,
        max_lag_samples + 1,
        dtype=int,
    )

    correlation_grid = np.full(
        len(lag_samples_grid),
        np.nan,
        dtype=float,
    )

    for idx, lag_samples in enumerate(
        lag_samples_grid
    ):

        if lag_samples == 0:

            shifted_response = response
            aligned_stimulus = stimulus

        else:

            shifted_response = response[
                lag_samples:
            ]

            aligned_stimulus = stimulus[
                :-lag_samples
            ]

        correlation_grid[
            idx
        ] = pearson_corr(
            shifted_response,
            aligned_stimulus,
            min_valid_samples=min_valid_samples,
        )

    if not np.isfinite(
        correlation_grid
    ).any():

        return {
            "lag_sec": np.nan,
            "lag_samples": np.nan,
            "peak_corr": np.nan,
            "boundary_hit_zero": False,
            "boundary_hit_max": False,
            "lag_grid_sec": (
                lag_samples_grid
                / fs
            ),
            "corr_grid": correlation_grid,
        }

    best_index = int(
        np.nanargmax(
            correlation_grid
        )
    )

    best_lag_samples = int(
        lag_samples_grid[
            best_index
        ]
    )

    peak_corr = float(
        correlation_grid[
            best_index
        ]
    )

    lag_sec = (
        best_lag_samples
        / fs
    )

    return {
        "lag_sec": float(
            lag_sec
        ),

        "lag_samples": best_lag_samples,

        "peak_corr": peak_corr,

        "boundary_hit_zero": (
            best_lag_samples
            == 0
        ),

        "boundary_hit_max": (
            best_lag_samples
            == max_lag_samples
        ),

        "lag_grid_sec": (
            lag_samples_grid
            / fs
        ),

        "corr_grid": correlation_grid,
    }


# =============================================================================
# STATISTICAL HELPERS
# =============================================================================

def mean_ci_t(
    values,
    alpha=0.05,
):

    values = (
        pd.Series(
            values
        )
        .dropna()
        .astype(float)
        .to_numpy()
    )

    n = len(
        values
    )

    if n == 0:

        return {
            "n": 0,
            "mean": np.nan,
            "median": np.nan,
            "sd": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    mean_value = float(
        np.mean(
            values
        )
    )

    median_value = float(
        np.median(
            values
        )
    )

    if n >= 2:

        sd = float(
            np.std(
                values,
                ddof=1,
            )
        )

        se = (
            sd
            / np.sqrt(
                n
            )
        )

        critical = t_dist.ppf(
            1 - alpha / 2,
            df=n - 1,
        )

        ci_low = (
            mean_value
            - critical * se
        )

        ci_high = (
            mean_value
            + critical * se
        )

    else:

        sd = np.nan
        se = np.nan
        ci_low = np.nan
        ci_high = np.nan

    return {
        "n": n,
        "mean": mean_value,
        "median": median_value,
        "sd": sd,
        "se": se,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }


def bootstrap_median_ci(
    values,
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
    alpha=0.05,
):

    values = (
        pd.Series(
            values
        )
        .dropna()
        .astype(float)
        .to_numpy()
    )

    n = len(
        values
    )

    if n == 0:

        return {
            "n": 0,
            "median": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    observed = float(
        np.median(
            values
        )
    )

    if n == 1:

        return {
            "n": 1,
            "median": observed,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    rng = np.random.default_rng(
        seed
    )

    bootstrap_values = np.empty(
        n_boot,
        dtype=float,
    )

    for b in range(
        n_boot
    ):

        sample = rng.choice(
            values,
            size=n,
            replace=True,
        )

        bootstrap_values[
            b
        ] = np.median(
            sample
        )

    ci_low = float(
        np.percentile(
            bootstrap_values,
            100 * alpha / 2,
        )
    )

    ci_high = float(
        np.percentile(
            bootstrap_values,
            100 * (
                1 - alpha / 2
            ),
        )
    )

    return {
        "n": n,
        "median": observed,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }


def cohens_dz(
    differences,
):

    values = (
        pd.Series(
            differences
        )
        .dropna()
        .astype(float)
        .to_numpy()
    )

    if len(values) < 2:
        return np.nan

    sd = np.std(
        values,
        ddof=1,
    )

    if (
        not np.isfinite(
            sd
        )
        or sd <= 0
    ):
        return np.nan

    return float(
        np.mean(
            values
        )
        / sd
    )


def sign_flip_test(
    values,
    popmean=0.0,
    n_perm=N_PERM,
    seed=RANDOM_SEED,
    alternative="two-sided",
):

    values = (
        pd.Series(
            values
        )
        .dropna()
        .astype(float)
        .to_numpy()
    )

    differences = (
        values
        - popmean
    )

    n = len(
        differences
    )

    if n < 2:

        return {
            "p_perm": np.nan,
            "observed_mean": np.nan,
            "n": n,
        }

    observed = float(
        np.mean(
            differences
        )
    )

    rng = np.random.default_rng(
        seed
    )

    null_distribution = np.empty(
        n_perm,
        dtype=float,
    )

    for permutation in range(
        n_perm
    ):

        signs = rng.choice(
            [
                -1.0,
                1.0,
            ],
            size=n,
        )

        null_distribution[
            permutation
        ] = np.mean(
            signs
            * differences
        )

    if alternative == "greater":

        pvalue = (
            np.sum(
                null_distribution
                >= observed
            )
            + 1
        ) / (
            n_perm
            + 1
        )

    elif alternative == "less":

        pvalue = (
            np.sum(
                null_distribution
                <= observed
            )
            + 1
        ) / (
            n_perm
            + 1
        )

    else:

        pvalue = (
            np.sum(
                np.abs(
                    null_distribution
                )
                >= abs(
                    observed
                )
            )
            + 1
        ) / (
            n_perm
            + 1
        )

    return {
        "p_perm": float(
            pvalue
        ),
        "observed_mean": observed,
        "n": n,
    }


def p_fmt(p):

    if not np.isfinite(
        p
    ):
        return "NA"

    if p < .001:
        return "< .001"

    return f"= {p:.4f}"


# =============================================================================
# STEP 1 — OBTAIN COMPLETE AUDIOVISUAL TRIAL DATASET
# =============================================================================

def existing_trial_df_is_suitable(
    dataframe,
):

    if (
        dataframe is None
        or not isinstance(
            dataframe,
            pd.DataFrame,
        )
        or dataframe.empty
    ):
        return False

    required = {
        "subject",
        "trialIndex",
        "trialType",
        "t_grid",
        "mouse_x",
        "blob_x",
        "audio_x",
    }

    if not required.issubset(
        dataframe.columns
    ):
        return False

    try:

        mask = dataframe.apply(
            is_model_matched_av_trial,
            axis=1,
        )

        conditions_found = set(
            dataframe.loc[
                mask
            ]
            .apply(
                classify_physical_condition,
                axis=1,
            )
            .dropna()
        )

        has_fixed = any(
            str(condition).startswith(
                "fixed_"
            )
            for condition
            in conditions_found
        )

        has_congruent = (
            "congruent"
            in conditions_found
        )

        has_sweep = (
            "sweep_diverge"
            in conditions_found
            or "sweep_converge"
            in conditions_found
        )

        return bool(
            mask.sum() > 0
            and has_fixed
            and has_congruent
            and has_sweep
        )

    except Exception:

        return False


USE_EXISTING_TRIAL_DF = (
    "trial_df" in globals()
    and existing_trial_df_is_suitable(
        trial_df
    )
)


if USE_EXISTING_TRIAL_DF:

    print(
        "\nUsing complete existing trial_df from notebook memory."
    )

    lag_source_df = (
        trial_df
        .copy()
    )

else:

    print(
        "\nExisting trial_df is absent or incomplete."
    )

    print(
        "Reloading congruent + fixed + sweep AV trials "
        "directly from master CSV files..."
    )

    csv_files = sorted(
        glob.glob(
            os.path.join(
                DATA_DIR,
                FILE_PATTERN,
            )
        )
    )

    if len(
        csv_files
    ) == 0:

        raise FileNotFoundError(
            f"No files matching {FILE_PATTERN} found in:\n{DATA_DIR}"
        )

    reconstructed_trials = []

    for filepath in (
        csv_files
    ):

        subject = extract_subject_id(
            filepath
        )

        raw = pd.read_csv(
            filepath
        )

        required_raw_columns = [
            "trialIndex",
            "trialType",
            "condLabel",
            "t",
            "mouse_x",
            "blob_x",
            "audio_x",
        ]

        missing = [
            column
            for column
            in required_raw_columns
            if column
            not in raw.columns
        ]

        if missing:

            print(
                f"Skipping {subject}: missing {missing}"
            )

            continue

        if (
            EXCLUDE_PRACTICE
            and "isPractice"
            in raw.columns
        ):

            practice = pd.to_numeric(
                raw[
                    "isPractice"
                ],
                errors="coerce",
            )

            raw = raw[
                practice
                .fillna(0)
                == 0
            ].copy()

        trial_type_lower = (
            raw[
                "trialType"
            ]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        raw = raw[
            trial_type_lower.isin(
                AV_TRIAL_TYPES_TO_KEEP
            )
        ].copy()

        if raw.empty:
            continue

        grouping_columns = [
            "trialIndex",
            "trialType",
            "condLabel",
        ]

        extra_columns = [
            column
            for column
            in [
                "offsetDeg",
                "sideSign",
                "rampType",
                "visualOn",
                "audioOn",
                "post_question",
                "post_question2",
                "same_response_yes",
            ]
            if column
            in raw.columns
        ]

        for (
            group_keys,
            trial_raw,
        ) in raw.groupby(
            grouping_columns,
            sort=False,
        ):

            (
                trial_index,
                trial_type,
                condition_label,
            ) = group_keys

            resampled = resample_trial(
                trial_raw,
                TIME_GRID,
            )

            if resampled is None:
                continue

            first_row = (
                trial_raw
                .iloc[0]
            )

            trial_row = {
                "subject":
                    subject,

                "trialIndex":
                    trial_index,

                "trialType":
                    trial_type,

                "condLabel":
                    condition_label,

                "t_grid":
                    resampled[
                        "t_grid"
                    ],

                "mouse_x":
                    resampled[
                        "mouse_x"
                    ],

                "blob_x":
                    resampled[
                        "blob_x"
                    ],

                "audio_x":
                    resampled[
                        "audio_x"
                    ],
            }

            for column in (
                extra_columns
            ):

                trial_row[
                    column
                ] = first_row[
                    column
                ]

            reconstructed_trials.append(
                trial_row
            )

    lag_source_df = pd.DataFrame(
        reconstructed_trials
    )

    if lag_source_df.empty:

        raise RuntimeError(
            "No audiovisual trials could be reconstructed."
        )


# =============================================================================
# STEP 2 — STANDARDIZE TRIAL METADATA
# =============================================================================

lag_source_df = (
    lag_source_df
    .copy()
    .reset_index(
        drop=True
    )
)


lag_source_df[
    "subject"
] = (
    lag_source_df[
        "subject"
    ]
    .apply(
        normalize_subject
    )
)


lag_source_df[
    "trial_id"
] = (
    lag_source_df[
        "trialIndex"
    ]
    .apply(
        normalize_trial_id
    )
)


lag_source_df[
    "physical_condition"
] = (
    lag_source_df
    .apply(
        classify_physical_condition,
        axis=1,
    )
)


lag_source_df[
    "offset_deg"
] = (
    lag_source_df
    .apply(
        extract_offset_abs,
        axis=1,
    )
)


lag_source_df[
    "side_label"
] = (
    lag_source_df
    .apply(
        extract_side_label,
        axis=1,
    )
)


if (
    "same_response_yes"
    in lag_source_df.columns
):

    lag_source_df[
        "same_response"
    ] = pd.to_numeric(
        lag_source_df[
            "same_response_yes"
        ],
        errors="coerce",
    )

elif (
    "post_question"
    in lag_source_df.columns
):

    lag_source_df[
        "same_response"
    ] = (
        lag_source_df[
            "post_question"
        ]
        .apply(
            parse_same_response
        )
    )

else:

    raise RuntimeError(
        "Could not find same/different response variable."
    )


lag_source_df[
    "judgment"
] = (
    lag_source_df[
        "same_response"
    ]
    .apply(
        judgment_label
    )
)


lag_source_df = lag_source_df[
    lag_source_df[
        "physical_condition"
    ].isin(
        PHYSICAL_CONDITION_ORDER
    )
].copy()


lag_source_df = lag_source_df[
    lag_source_df[
        "same_response"
    ].isin(
        [
            0.0,
            1.0,
        ]
    )
].copy()


lag_source_df[
    "condition_order"
] = pd.Categorical(
    lag_source_df[
        "physical_condition"
    ],
    categories=
        PHYSICAL_CONDITION_ORDER,
    ordered=True,
)


lag_source_df = (
    lag_source_df
    .sort_values(
        [
            "subject",
            "condition_order",
            "trialIndex",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 100)
print("INPUT AV DATASET")
print("=" * 100)

print(
    f"\nTotal AV trials before lag QC: "
    f"{len(lag_source_df)}"
)

print(
    f"Participants: "
    f"{lag_source_df['subject'].nunique()}"
)


print(
    "\nTrials by physical condition:"
)

print(
    lag_source_df[
        "physical_condition"
    ]
    .value_counts()
    .reindex(
        PHYSICAL_CONDITION_ORDER
    )
    .fillna(0)
    .astype(int)
)


print(
    "\nSame/Different judgments by physical condition:"
)

judgment_count_table = pd.crosstab(
    lag_source_df[
        "physical_condition"
    ],
    lag_source_df[
        "judgment"
    ],
).reindex(
    PHYSICAL_CONDITION_ORDER
)

print(
    judgment_count_table
)


judgment_count_table.to_csv(
    os.path.join(
        CSV_DIR,
        "01_input_trial_counts_by_condition_and_judgment.csv",
    )
)


# =============================================================================
# STEP 3 — EXACT TRIAL-LEVEL LAG ESTIMATION + QC
# =============================================================================

trial_lag_rows = []


for _, row in (
    lag_source_df
    .iterrows()
):

    time = safe_array(
        row[
            "t_grid"
        ]
    )

    response = safe_array(
        row[
            "mouse_x"
        ]
    )

    audio = safe_array(
        row[
            "audio_x"
        ]
    )

    visual = safe_array(
        row[
            "blob_x"
        ]
    )

    n = min(
        len(time),
        len(response),
        len(audio),
        len(visual),
    )

    if n < MIN_VALID_SAMPLES:
        continue

    time = time[:n]
    response = response[:n]
    audio = audio[:n]
    visual = visual[:n]

    (
        time_use,
        response_use,
        audio_use,
        visual_use,
    ) = restrict_analysis_window(
        time,
        response,
        audio,
        visual,
    )


    # -------------------------------------------------------------------------
    # AUDITORY-STIMULUS MOTION SD
    # -------------------------------------------------------------------------

    finite_audio = audio_use[
        np.isfinite(
            audio_use
        )
    ]

    if len(
        finite_audio
    ) >= 2:

        audio_sd_deg = float(
            np.std(
                finite_audio,
                ddof=1,
            )
        )

    else:

        audio_sd_deg = np.nan


    # -------------------------------------------------------------------------
    # EXACT POSITIVE-LAG CROSS-CORRELATION
    # -------------------------------------------------------------------------

    lag_result = positive_lag_crosscorr(
        response=response_use,
        stimulus=audio_use,
        fs=RESAMPLE_HZ,
        max_lag_sec=MAX_LAG_SEC,
    )


    peak_corr = lag_result[
        "peak_corr"
    ]

    lag_sec = lag_result[
        "lag_sec"
    ]


    # -------------------------------------------------------------------------
    # COLLABORATOR QC
    # -------------------------------------------------------------------------

    passes_motion = (
        np.isfinite(
            audio_sd_deg
        )
        and (
            audio_sd_deg
            >= MIN_STIMULUS_SD_DEG
        )
    )


    passes_peak_corr = (
        np.isfinite(
            peak_corr
        )
        and (
            peak_corr
            > MIN_PEAK_CORR
        )
    )


    valid_lag_trial = (
        passes_motion
        and passes_peak_corr
        and np.isfinite(
            lag_sec
        )
    )


    exclusion_reason = ""

    if not passes_motion:

        exclusion_reason += (
            "stimulus_sd_below_3deg;"
        )

    if not passes_peak_corr:

        exclusion_reason += (
            "peak_r_not_above_0p40;"
        )

    if not np.isfinite(
        lag_sec
    ):

        exclusion_reason += (
            "lag_not_estimable;"
        )


    trial_lag_rows.append(
        {
            "subject":
                row[
                    "subject"
                ],

            "trialIndex":
                row[
                    "trialIndex"
                ],

            "trial_id":
                row[
                    "trial_id"
                ],

            "trialType":
                row[
                    "trialType"
                ],

            "condLabel":
                row.get(
                    "condLabel",
                    np.nan,
                ),

            "physical_condition":
                row[
                    "physical_condition"
                ],

            "offset_deg":
                row[
                    "offset_deg"
                ],

            "side_label":
                row[
                    "side_label"
                ],

            "same_response":
                row[
                    "same_response"
                ],

            "judgment":
                row[
                    "judgment"
                ],

            "audio_stimulus_sd_deg":
                audio_sd_deg,

            "lag_audio_sec":
                lag_sec,

            "lag_audio_samples":
                lag_result[
                    "lag_samples"
                ],

            "peak_audio_corr":
                peak_corr,

            "lag_at_zero_boundary":
                lag_result[
                    "boundary_hit_zero"
                ],

            "lag_at_4s_boundary":
                lag_result[
                    "boundary_hit_max"
                ],

            "passes_stimulus_sd_qc":
                passes_motion,

            "passes_peak_r_qc":
                passes_peak_corr,

            "valid_lag_trial":
                valid_lag_trial,

            "exclusion_reason":
                exclusion_reason,
        }
    )


trial_lag_all_df = pd.DataFrame(
    trial_lag_rows
)


trial_lag_all_df.to_csv(
    os.path.join(
        CSV_DIR,
        "02_all_trial_lag_estimates_before_QC.csv",
    ),
    index=False,
)


# Exact collaborator-valid trials only
trial_lag_df = trial_lag_all_df[
    trial_lag_all_df[
        "valid_lag_trial"
    ]
].copy()


trial_lag_df.to_csv(
    os.path.join(
        CSV_DIR,
        "03_VALID_trial_lag_estimates_after_exact_collaborator_QC.csv",
    ),
    index=False,
)


# =============================================================================
# STEP 4 — QC DIAGNOSTICS
# =============================================================================

n_total = len(
    trial_lag_all_df
)

n_valid = len(
    trial_lag_df
)

n_failed_motion = int(
    (
        ~trial_lag_all_df[
            "passes_stimulus_sd_qc"
        ]
    ).sum()
)

n_failed_corr = int(
    (
        ~trial_lag_all_df[
            "passes_peak_r_qc"
        ]
    ).sum()
)

n_zero_boundary = int(
    trial_lag_df[
        "lag_at_zero_boundary"
    ].sum()
)

n_max_boundary = int(
    trial_lag_df[
        "lag_at_4s_boundary"
    ].sum()
)


print("\n" + "=" * 100)
print("EXACT COLLABORATOR LAG-QC DIAGNOSTICS")
print("=" * 100)


print(
    f"\nTotal AV trials evaluated: "
    f"{n_total}"
)

print(
    f"Valid trials after exact QC: "
    f"{n_valid} "
    f"({100*n_valid/max(n_total,1):.2f}%)"
)

print(
    f"\nTrials failing auditory SD >= 3°: "
    f"{n_failed_motion}"
)

print(
    f"Trials failing peak r > .40: "
    f"{n_failed_corr}"
)

print(
    f"\nValid trials with best lag = 0 s: "
    f"{n_zero_boundary}"
)

print(
    f"Valid trials with best lag = 4 s: "
    f"{n_max_boundary}"
)


qc_condition_table = (
    trial_lag_all_df
    .groupby(
        "physical_condition"
    )
    .agg(
        total_trials=(
            "trialIndex",
            "size",
        ),

        valid_trials=(
            "valid_lag_trial",
            "sum",
        ),

        mean_stimulus_sd_deg=(
            "audio_stimulus_sd_deg",
            "mean",
        ),

        median_peak_r=(
            "peak_audio_corr",
            "median",
        ),

        median_lag_before_QC=(
            "lag_audio_sec",
            "median",
        ),
    )
    .reindex(
        PHYSICAL_CONDITION_ORDER
    )
)


qc_condition_table[
    "percent_valid"
] = (
    100
    * qc_condition_table[
        "valid_trials"
    ]
    / qc_condition_table[
        "total_trials"
    ]
)


print(
    "\nQC by physical condition:"
)

print(
    qc_condition_table
)


qc_condition_table.to_csv(
    os.path.join(
        CSV_DIR,
        "04_QC_summary_by_physical_condition.csv",
    )
)


# Same vs Different retention diagnostics
qc_judgment_table = (
    trial_lag_all_df
    .groupby(
        "judgment"
    )
    .agg(
        total_trials=(
            "trialIndex",
            "size",
        ),

        valid_trials=(
            "valid_lag_trial",
            "sum",
        ),

        median_peak_r=(
            "peak_audio_corr",
            "median",
        ),

        median_stimulus_sd=(
            "audio_stimulus_sd_deg",
            "median",
        ),
    )
)


qc_judgment_table[
    "percent_valid"
] = (
    100
    * qc_judgment_table[
        "valid_trials"
    ]
    / qc_judgment_table[
        "total_trials"
    ]
)


print(
    "\nQC retention by causal judgment:"
)

print(
    qc_judgment_table
)


qc_judgment_table.to_csv(
    os.path.join(
        CSV_DIR,
        "05_QC_summary_by_judgment.csv",
    )
)


# =============================================================================
# ANALYSIS 1
# EXACT COLLABORATOR REPLICATION
# =============================================================================
#
# Within observer:
#
#     median lag for ALL valid SAME AV trials
#     median lag for ALL valid DIFFERENT AV trials
#
# Then compare / aggregate these participant medians across observers.
# =============================================================================

subject_pooled_median_df = (
    trial_lag_df
    .groupby(
        [
            "subject",
            "judgment",
        ],
        as_index=False,
    )
    .agg(
        median_lag_sec=(
            "lag_audio_sec",
            "median",
        ),

        n_valid_trials=(
            "trialIndex",
            "count",
        ),

        median_peak_r=(
            "peak_audio_corr",
            "median",
        ),

        median_stimulus_sd_deg=(
            "audio_stimulus_sd_deg",
            "median",
        ),
    )
)


subject_pooled_wide = (
    subject_pooled_median_df
    .pivot_table(
        index="subject",
        columns="judgment",
        values="median_lag_sec",
        aggfunc="first",
    )
    .reset_index()
)


if (
    "Same"
    not in subject_pooled_wide.columns
    or "Different"
    not in subject_pooled_wide.columns
):

    raise RuntimeError(
        "Could not obtain both Same and Different participant-level medians."
    )


subject_pooled_wide = (
    subject_pooled_wide[
        np.isfinite(
            subject_pooled_wide[
                "Same"
            ]
        )
        & np.isfinite(
            subject_pooled_wide[
                "Different"
            ]
        )
    ]
    .copy()
)


subject_pooled_wide[
    "Different_minus_Same"
] = (
    subject_pooled_wide[
        "Different"
    ]
    - subject_pooled_wide[
        "Same"
    ]
)


subject_pooled_wide.to_csv(
    os.path.join(
        CSV_DIR,
        "06_EXACT_COLLABORATOR_subject_median_lags.csv",
    ),
    index=False,
)


# -----------------------------------------------------------------------------
# Group summary
#
# We report BOTH:
#
#   A. median across participant medians
#      -> closest literal implementation of "median within observer,
#         then across observers"
#
#   B. mean + 95% CI across participant medians
#      -> useful conventional group description
# -----------------------------------------------------------------------------


same_group_median = bootstrap_median_ci(
    subject_pooled_wide[
        "Same"
    ],
    seed=RANDOM_SEED + 10,
)


different_group_median = bootstrap_median_ci(
    subject_pooled_wide[
        "Different"
    ],
    seed=RANDOM_SEED + 20,
)


same_group_mean = mean_ci_t(
    subject_pooled_wide[
        "Same"
    ]
)


different_group_mean = mean_ci_t(
    subject_pooled_wide[
        "Different"
    ]
)


pooled_difference = (
    subject_pooled_wide[
        "Different_minus_Same"
    ]
    .dropna()
)


pooled_difference_summary = mean_ci_t(
    pooled_difference
)


pooled_difference_median = bootstrap_median_ci(
    pooled_difference,
    seed=RANDOM_SEED + 30,
)


# Inference
pooled_t = ttest_1samp(
    pooled_difference,
    popmean=0.0,
)


try:

    pooled_wilcoxon_greater = wilcoxon(
        pooled_difference,
        alternative="greater",
        zero_method="wilcox",
    )

except Exception:

    pooled_wilcoxon_greater = None


pooled_perm_two = sign_flip_test(
    pooled_difference,
    alternative="two-sided",
    seed=RANDOM_SEED + 40,
)


pooled_perm_greater = sign_flip_test(
    pooled_difference,
    alternative="greater",
    seed=RANDOM_SEED + 50,
)


pooled_dz = cohens_dz(
    pooled_difference
)


collaborator_summary_df = pd.DataFrame(
    [
        {
            "analysis":
                "Exact collaborator pooled AV replication",

            "n_subjects":
                len(
                    subject_pooled_wide
                ),

            "same_group_median_sec":
                same_group_median[
                    "median"
                ],

            "same_median_boot_CI_low":
                same_group_median[
                    "ci_low"
                ],

            "same_median_boot_CI_high":
                same_group_median[
                    "ci_high"
                ],

            "different_group_median_sec":
                different_group_median[
                    "median"
                ],

            "different_median_boot_CI_low":
                different_group_median[
                    "ci_low"
                ],

            "different_median_boot_CI_high":
                different_group_median[
                    "ci_high"
                ],

            "same_mean_of_subject_medians_sec":
                same_group_mean[
                    "mean"
                ],

            "different_mean_of_subject_medians_sec":
                different_group_mean[
                    "mean"
                ],

            "mean_Different_minus_Same_sec":
                pooled_difference_summary[
                    "mean"
                ],

            "mean_difference_CI_low":
                pooled_difference_summary[
                    "ci_low"
                ],

            "mean_difference_CI_high":
                pooled_difference_summary[
                    "ci_high"
                ],

            "median_Different_minus_Same_sec":
                pooled_difference_median[
                    "median"
                ],

            "median_difference_boot_CI_low":
                pooled_difference_median[
                    "ci_low"
                ],

            "median_difference_boot_CI_high":
                pooled_difference_median[
                    "ci_high"
                ],

            "t":
                pooled_t.statistic,

            "df":
                len(
                    pooled_difference
                )
                - 1,

            "p_two_sided_t":
                pooled_t.pvalue,

            "p_two_sided_signflip":
                pooled_perm_two[
                    "p_perm"
                ],

            "p_directional_signflip_Different_gt_Same":
                pooled_perm_greater[
                    "p_perm"
                ],

            "p_directional_wilcoxon":
                (
                    pooled_wilcoxon_greater.pvalue
                    if pooled_wilcoxon_greater
                    is not None
                    else np.nan
                ),

            "cohens_dz":
                pooled_dz,
        }
    ]
)


collaborator_summary_df.to_csv(
    os.path.join(
        CSV_DIR,
        "07_EXACT_COLLABORATOR_group_summary.csv",
    ),
    index=False,
)


print("\n" + "=" * 100)
print("ANALYSIS 1 — EXACT COLLABORATOR REPLICATION")
print("=" * 100)


print(
    f"\nParticipants with BOTH Same and Different valid trials: "
    f"{len(subject_pooled_wide)}"
)


print(
    "\nGROUP MEDIAN OF PARTICIPANT MEDIANS:"
)


print(
    f"\nSAME:"
    f"\n    median = "
    f"{same_group_median['median']:.3f} s"
    f"\n    bootstrap 95% CI "
    f"[{same_group_median['ci_low']:.3f}, "
    f"{same_group_median['ci_high']:.3f}]"
)


print(
    f"\nDIFFERENT:"
    f"\n    median = "
    f"{different_group_median['median']:.3f} s"
    f"\n    bootstrap 95% CI "
    f"[{different_group_median['ci_low']:.3f}, "
    f"{different_group_median['ci_high']:.3f}]"
)


print(
    "\nMEAN OF PARTICIPANT MEDIANS:"
)


print(
    f"\nSAME:"
    f"\n    M = "
    f"{same_group_mean['mean']:.3f} s"
    f"\n    95% CI "
    f"[{same_group_mean['ci_low']:.3f}, "
    f"{same_group_mean['ci_high']:.3f}]"
)


print(
    f"\nDIFFERENT:"
    f"\n    M = "
    f"{different_group_mean['mean']:.3f} s"
    f"\n    95% CI "
    f"[{different_group_mean['ci_low']:.3f}, "
    f"{different_group_mean['ci_high']:.3f}]"
)


print(
    f"\nDIFFERENT - SAME:"
    f"\n    M paired difference = "
    f"{pooled_difference_summary['mean']:.3f} s"
    f"\n    95% CI "
    f"[{pooled_difference_summary['ci_low']:.3f}, "
    f"{pooled_difference_summary['ci_high']:.3f}]"
)


print(
    f"\n    Median paired difference = "
    f"{pooled_difference_median['median']:.3f} s"
)


print(
    f"\n    t({len(pooled_difference)-1}) "
    f"= {pooled_t.statistic:.3f}, "
    f"p {p_fmt(pooled_t.pvalue)}"
)


print(
    f"    two-sided sign-flip p "
    f"{p_fmt(pooled_perm_two['p_perm'])}"
)


print(
    f"    directional Different > Same sign-flip p "
    f"{p_fmt(pooled_perm_greater['p_perm'])}"
)


print(
    f"    Cohen dz = {pooled_dz:.3f}"
)


# =============================================================================
# ANALYSIS 2
# PHYSICALLY MATCHED SAME/DIFFERENT ANALYSIS
# =============================================================================
#
# EXACT SAME lag estimator + QC as Analysis 1.
#
# Difference:
#
#     SAME and DIFFERENT are first compared WITHIN:
#
#         participant × physical condition
#
# We use MEDIAN trial lag within each cell.
# =============================================================================


subject_condition_judgment_df = (
    trial_lag_df
    .groupby(
        [
            "subject",
            "physical_condition",
            "judgment",
        ],
        as_index=False,
    )
    .agg(
        median_lag_sec=(
            "lag_audio_sec",
            "median",
        ),

        n_valid_trials=(
            "trialIndex",
            "count",
        ),

        median_peak_r=(
            "peak_audio_corr",
            "median",
        ),
    )
)


subject_condition_judgment_df.to_csv(
    os.path.join(
        CSV_DIR,
        "08_MATCHED_subject_condition_judgment_medians.csv",
    ),
    index=False,
)


matched_wide = (
    subject_condition_judgment_df
    .pivot_table(
        index=[
            "subject",
            "physical_condition",
        ],

        columns="judgment",

        values="median_lag_sec",

        aggfunc="first",
    )
    .reset_index()
)


if (
    "Same"
    not in matched_wide.columns
):

    matched_wide[
        "Same"
    ] = np.nan


if (
    "Different"
    not in matched_wide.columns
):

    matched_wide[
        "Different"
    ] = np.nan


matched_wide = matched_wide[
    np.isfinite(
        matched_wide[
            "Same"
        ]
    )
    & np.isfinite(
        matched_wide[
            "Different"
        ]
    )
].copy()


matched_wide[
    "Different_minus_Same"
] = (
    matched_wide[
        "Different"
    ]
    - matched_wide[
        "Same"
    ]
)


matched_wide.to_csv(
    os.path.join(
        CSV_DIR,
        "09_MATCHED_subject_condition_contrasts.csv",
    ),
    index=False,
)


print("\n" + "=" * 100)
print("PHYSICALLY MATCHED CELLS")
print("=" * 100)


print(
    f"\nMatched participant × condition cells: "
    f"{len(matched_wide)}"
)


matched_counts = (
    matched_wide
    .groupby(
        "physical_condition"
    )[
        "subject"
    ]
    .nunique()
    .reindex(
        PHYSICAL_CONDITION_ORDER
    )
    .fillna(0)
    .astype(int)
)


print(
    "\nParticipants with both judgments in each physical condition:"
)

print(
    matched_counts
)


matched_counts.to_csv(
    os.path.join(
        CSV_DIR,
        "10_MATCHED_participant_counts_by_condition.csv",
    )
)


# =============================================================================
# CONDITION-SPECIFIC MATCHED RESULTS
# =============================================================================

condition_result_rows = []


for condition_index, condition in enumerate(
    PHYSICAL_CONDITION_ORDER
):

    condition_data = matched_wide[
        matched_wide[
            "physical_condition"
        ]
        == condition
    ].copy()

    differences = (
        condition_data[
            "Different_minus_Same"
        ]
        .dropna()
    )

    summary = mean_ci_t(
        differences
    )

    if len(
        differences
    ) >= 2:

        test = ttest_1samp(
            differences,
            popmean=0.0,
        )

        permutation_two = sign_flip_test(
            differences,
            alternative="two-sided",
            seed=(
                RANDOM_SEED
                + 100
                + condition_index
            ),
        )

        permutation_greater = sign_flip_test(
            differences,
            alternative="greater",
            seed=(
                RANDOM_SEED
                + 200
                + condition_index
            ),
        )

        dz = cohens_dz(
            differences
        )

    else:

        test = None

        permutation_two = {
            "p_perm":
                np.nan,
        }

        permutation_greater = {
            "p_perm":
                np.nan,
        }

        dz = np.nan


    condition_result_rows.append(
        {
            "physical_condition":
                condition,

            "condition_label":
                PHYSICAL_CONDITION_LABELS[
                    condition
                ],

            "n_matched_subjects":
                summary[
                    "n"
                ],

            "mean_Different_minus_Same_sec":
                summary[
                    "mean"
                ],

            "median_Different_minus_Same_sec":
                summary[
                    "median"
                ],

            "CI_low":
                summary[
                    "ci_low"
                ],

            "CI_high":
                summary[
                    "ci_high"
                ],

            "t":
                (
                    test.statistic
                    if test is not None
                    else np.nan
                ),

            "df":
                (
                    len(
                        differences
                    )
                    - 1
                    if len(
                        differences
                    )
                    >= 2
                    else np.nan
                ),

            "p_two_sided_t":
                (
                    test.pvalue
                    if test is not None
                    else np.nan
                ),

            "p_two_sided_signflip":
                permutation_two[
                    "p_perm"
                ],

            "p_directional_Different_gt_Same":
                permutation_greater[
                    "p_perm"
                ],

            "cohens_dz":
                dz,
        }
    )


condition_results_df = pd.DataFrame(
    condition_result_rows
)


condition_results_df.to_csv(
    os.path.join(
        CSV_DIR,
        "11_MATCHED_condition_specific_results.csv",
    ),
    index=False,
)


print(
    "\nCondition-specific physically matched effects:"
)

print(
    condition_results_df.to_string(
        index=False
    )
)


# =============================================================================
# PARTICIPANT-FIRST MATCHED EFFECT
# =============================================================================
#
# Each participant may contribute multiple physical conditions.
#
# We first calculate Different-Same within each condition.
#
# Then take the MEDIAN across that participant's matched condition effects.
#
# This retains the robust median logic throughout the hierarchy.
# =============================================================================

participant_matched_df = (
    matched_wide
    .groupby(
        "subject",
        as_index=False,
    )
    .agg(
        n_matched_conditions=(
            "physical_condition",
            "nunique",
        ),

        matched_effect_Different_minus_Same=(
            "Different_minus_Same",
            "median",
        ),

        median_same_lag_across_matched_conditions=(
            "Same",
            "median",
        ),

        median_different_lag_across_matched_conditions=(
            "Different",
            "median",
        ),
    )
)


participant_matched_df = participant_matched_df[
    participant_matched_df[
        "n_matched_conditions"
    ]
    >= MIN_MATCHED_CONDITIONS_PER_SUBJECT
].copy()


participant_matched_df.to_csv(
    os.path.join(
        CSV_DIR,
        "12_MATCHED_participant_level_effects.csv",
    ),
    index=False,
)


matched_difference = (
    participant_matched_df[
        "matched_effect_Different_minus_Same"
    ]
    .dropna()
)


matched_summary = mean_ci_t(
    matched_difference
)


matched_median_summary = bootstrap_median_ci(
    matched_difference,
    seed=RANDOM_SEED + 500,
)


if len(
    matched_difference
) >= 2:

    matched_t = ttest_1samp(
        matched_difference,
        popmean=0.0,
    )

    matched_perm_two = sign_flip_test(
        matched_difference,
        alternative="two-sided",
        seed=RANDOM_SEED + 510,
    )

    matched_perm_greater = sign_flip_test(
        matched_difference,
        alternative="greater",
        seed=RANDOM_SEED + 520,
    )

    try:

        matched_wilcoxon_greater = wilcoxon(
            matched_difference,
            alternative="greater",
            zero_method="wilcox",
        )

    except Exception:

        matched_wilcoxon_greater = None

    matched_dz = cohens_dz(
        matched_difference
    )

else:

    matched_t = None

    matched_perm_two = {
        "p_perm":
            np.nan,
    }

    matched_perm_greater = {
        "p_perm":
            np.nan,
    }

    matched_wilcoxon_greater = None

    matched_dz = np.nan


matched_group_results_df = pd.DataFrame(
    [
        {
            "analysis":
                "Physically matched Same/Different lag",

            "n_subjects":
                matched_summary[
                    "n"
                ],

            "mean_Different_minus_Same_sec":
                matched_summary[
                    "mean"
                ],

            "median_Different_minus_Same_sec":
                matched_median_summary[
                    "median"
                ],

            "mean_difference_CI_low":
                matched_summary[
                    "ci_low"
                ],

            "mean_difference_CI_high":
                matched_summary[
                    "ci_high"
                ],

            "median_difference_boot_CI_low":
                matched_median_summary[
                    "ci_low"
                ],

            "median_difference_boot_CI_high":
                matched_median_summary[
                    "ci_high"
                ],

            "t":
                (
                    matched_t.statistic
                    if matched_t
                    is not None
                    else np.nan
                ),

            "df":
                (
                    len(
                        matched_difference
                    )
                    - 1
                    if len(
                        matched_difference
                    )
                    >= 2
                    else np.nan
                ),

            "p_two_sided_t":
                (
                    matched_t.pvalue
                    if matched_t
                    is not None
                    else np.nan
                ),

            "p_two_sided_signflip":
                matched_perm_two[
                    "p_perm"
                ],

            "p_directional_Different_gt_Same":
                matched_perm_greater[
                    "p_perm"
                ],

            "p_directional_wilcoxon":
                (
                    matched_wilcoxon_greater.pvalue
                    if matched_wilcoxon_greater
                    is not None
                    else np.nan
                ),

            "cohens_dz":
                matched_dz,
        }
    ]
)


matched_group_results_df.to_csv(
    os.path.join(
        CSV_DIR,
        "13_MATCHED_group_result.csv",
    ),
    index=False,
)


print("\n" + "=" * 100)
print("ANALYSIS 2 — PHYSICALLY MATCHED SAME/DIFFERENT TEST")
print("=" * 100)


print(
    f"\nParticipants contributing: "
    f"{matched_summary['n']}"
)


print(
    f"\nDIFFERENT - SAME:"
    f"\n    Mean participant effect = "
    f"{matched_summary['mean']:.3f} s"
    f"\n    95% CI "
    f"[{matched_summary['ci_low']:.3f}, "
    f"{matched_summary['ci_high']:.3f}]"
)


print(
    f"\n    Median participant effect = "
    f"{matched_median_summary['median']:.3f} s"
    f"\n    bootstrap 95% CI "
    f"[{matched_median_summary['ci_low']:.3f}, "
    f"{matched_median_summary['ci_high']:.3f}]"
)


if matched_t is not None:

    print(
        f"\n    t({len(matched_difference)-1}) "
        f"= {matched_t.statistic:.3f}, "
        f"p {p_fmt(matched_t.pvalue)}"
    )


print(
    f"    two-sided sign-flip p "
    f"{p_fmt(matched_perm_two['p_perm'])}"
)


print(
    f"    directional Different > Same sign-flip p "
    f"{p_fmt(matched_perm_greater['p_perm'])}"
)


print(
    f"    Cohen dz = {matched_dz:.3f}"
)


# =============================================================================
# WHOLE-TRIAL LABEL-PERMUTATION CONTROL FOR MATCHED ANALYSIS
# =============================================================================
#
# Labels are shuffled across complete VALID trials within:
#
#     subject × physical condition × side
#
# This keeps the exact physical stimulus structure intact.
# =============================================================================

def compute_matched_subject_effect_from_labels(
    dataframe,
    judgment_column,
):

    working = dataframe[
        [
            "subject",
            "physical_condition",
            "lag_audio_sec",
            judgment_column,
        ]
    ].copy()


    working = working[
        working[
            judgment_column
        ].isin(
            [
                0.0,
                1.0,
            ]
        )
    ]


    if working.empty:

        return pd.DataFrame()


    cell = (
        working
        .groupby(
            [
                "subject",
                "physical_condition",
                judgment_column,
            ],
            as_index=False,
        )
        .agg(
            median_lag=(
                "lag_audio_sec",
                "median",
            )
        )
    )


    wide = (
        cell
        .pivot_table(
            index=[
                "subject",
                "physical_condition",
            ],

            columns=
                judgment_column,

            values=
                "median_lag",

            aggfunc="first",
        )
        .reset_index()
    )


    if (
        0.0 not in wide.columns
        or 1.0 not in wide.columns
    ):

        return pd.DataFrame()


    wide = wide[
        np.isfinite(
            wide[
                0.0
            ]
        )
        & np.isfinite(
            wide[
                1.0
            ]
        )
    ].copy()


    if wide.empty:

        return pd.DataFrame()


    wide[
        "difference"
    ] = (
        wide[
            0.0
        ]
        - wide[
            1.0
        ]
    )


    subject_effect = (
        wide
        .groupby(
            "subject",
            as_index=False,
        )
        .agg(
            difference=(
                "difference",
                "median",
            )
        )
    )


    return subject_effect


observed_matched_subject_effect = (
    compute_matched_subject_effect_from_labels(
        trial_lag_df,
        judgment_column=
            "same_response",
    )
)


observed_matched_stat = float(
    observed_matched_subject_effect[
        "difference"
    ].mean()
)


permutation_df = (
    trial_lag_df
    .copy()
)


permutation_df[
    "permutation_stratum"
] = (
    permutation_df[
        "subject"
    ].astype(str)
    + "|"
    + permutation_df[
        "physical_condition"
    ].astype(str)
    + "|"
    + permutation_df[
        "side_label"
    ].astype(str)
)


strata_indices = [
    np.asarray(
        indices,
        dtype=int,
    )

    for indices
    in permutation_df.groupby(
        "permutation_stratum"
    ).indices.values()
]


original_labels = (
    permutation_df[
        "same_response"
    ]
    .to_numpy(
        dtype=float
    )
)


rng = np.random.default_rng(
    RANDOM_SEED + 600
)


permutation_null = np.full(
    N_PERM,
    np.nan,
    dtype=float,
)


print("\n" + "=" * 100)
print("WHOLE-TRIAL PHYSICAL-CONDITION-PRESERVING PERMUTATION")
print("=" * 100)


for permutation_index in range(
    N_PERM
):

    shuffled_labels = (
        original_labels
        .copy()
    )

    for indices in (
        strata_indices
    ):

        if len(
            indices
        ) < 2:
            continue

        shuffled_labels[
            indices
        ] = rng.permutation(
            shuffled_labels[
                indices
            ]
        )


    permutation_df[
        "_shuffled_judgment"
    ] = shuffled_labels


    permutation_subject_effect = (
        compute_matched_subject_effect_from_labels(
            permutation_df,
            judgment_column=
                "_shuffled_judgment",
        )
    )


    if not (
        permutation_subject_effect.empty
    ):

        permutation_null[
            permutation_index
        ] = (
            permutation_subject_effect[
                "difference"
            ]
            .mean()
        )


valid_permutation_null = (
    permutation_null[
        np.isfinite(
            permutation_null
        )
    ]
)


if len(
    valid_permutation_null
) > 0:

    whole_trial_p_two = (
        np.sum(
            np.abs(
                valid_permutation_null
            )
            >= abs(
                observed_matched_stat
            )
        )
        + 1
    ) / (
        len(
            valid_permutation_null
        )
        + 1
    )


    whole_trial_p_greater = (
        np.sum(
            valid_permutation_null
            >= observed_matched_stat
        )
        + 1
    ) / (
        len(
            valid_permutation_null
        )
        + 1
    )

else:

    whole_trial_p_two = np.nan
    whole_trial_p_greater = np.nan


whole_trial_result_df = pd.DataFrame(
    [
        {
            "observed_Different_minus_Same_sec":
                observed_matched_stat,

            "n_valid_permutations":
                len(
                    valid_permutation_null
                ),

            "p_two_sided":
                whole_trial_p_two,

            "p_directional_Different_gt_Same":
                whole_trial_p_greater,

            "shuffle_strata":
                (
                    "subject x physical_condition x side"
                ),
        }
    ]
)


whole_trial_result_df.to_csv(
    os.path.join(
        CSV_DIR,
        "14_MATCHED_whole_trial_label_permutation.csv",
    ),
    index=False,
)


pd.DataFrame(
    {
        "permuted_Different_minus_Same":
            valid_permutation_null
    }
).to_csv(
    os.path.join(
        CSV_DIR,
        "15_MATCHED_whole_trial_permutation_null.csv",
    ),
    index=False,
)


print(
    f"\nObserved matched effect = "
    f"{observed_matched_stat:.4f} s"
)


print(
    f"Two-sided permutation p "
    f"{p_fmt(whole_trial_p_two)}"
)


print(
    f"Directional Different > Same p "
    f"{p_fmt(whole_trial_p_greater)}"
)


# =============================================================================
# OPTIONAL SUPPORTING MIXED MODEL
# =============================================================================
#
# Uses EXACT collaborator-QC-valid trial lags.
#
# This is secondary because the primary analyses use the collaborator's
# median hierarchy.
# =============================================================================

mixed_df = (
    trial_lag_df
    .copy()
)


mixed_df[
    "reported_different"
] = (
    1.0
    - mixed_df[
        "same_response"
    ]
)


mixed_df[
    "physical_condition"
] = pd.Categorical(
    mixed_df[
        "physical_condition"
    ],
    categories=
        PHYSICAL_CONDITION_ORDER,
    ordered=False,
)


mixed_formula = (
    "lag_audio_sec ~ "
    "reported_different "
    "+ C(physical_condition) "
    "+ C(side_label)"
)


mixed_result = None
mixed_type = None


# First try random intercept + judgment slope.
try:

    model = smf.mixedlm(
        mixed_formula,
        data=mixed_df,
        groups=mixed_df[
            "subject"
        ],
        re_formula=
            "~reported_different",
    )

    result = model.fit(
        reml=False,
        method="lbfgs",
        maxiter=3000,
        disp=False,
    )

    if getattr(
        result,
        "converged",
        False,
    ):

        mixed_result = result
        mixed_type = (
            "random intercept + judgment slope"
        )

except Exception:

    pass


# If not converged, explicitly refit simpler random-intercept model.
if mixed_result is None:

    try:

        model = smf.mixedlm(
            mixed_formula,
            data=mixed_df,
            groups=mixed_df[
                "subject"
            ],
            re_formula="1",
        )

        result = model.fit(
            reml=False,
            method="lbfgs",
            maxiter=3000,
            disp=False,
        )

        mixed_result = result
        mixed_type = (
            "random intercept only"
        )

    except Exception as error:

        print(
            "\nMixed model failed:"
        )

        print(
            error
        )


if mixed_result is not None:

    print("\n" + "=" * 100)
    print("SECONDARY MIXED MODEL")
    print("=" * 100)

    print(
        f"\nModel type: "
        f"{mixed_type}"
    )

    print(
        mixed_result.summary()
    )


    coefficient = mixed_result.params.get(
        "reported_different",
        np.nan,
    )

    coefficient_se = mixed_result.bse.get(
        "reported_different",
        np.nan,
    )

    coefficient_p = mixed_result.pvalues.get(
        "reported_different",
        np.nan,
    )


    confidence_interval = (
        mixed_result
        .conf_int()
    )


    if (
        "reported_different"
        in confidence_interval.index
    ):

        coefficient_ci_low = (
            confidence_interval.loc[
                "reported_different",
                0,
            ]
        )

        coefficient_ci_high = (
            confidence_interval.loc[
                "reported_different",
                1,
            ]
        )

    else:

        coefficient_ci_low = np.nan
        coefficient_ci_high = np.nan


    mixed_summary_df = pd.DataFrame(
        [
            {
                "model_type":
                    mixed_type,

                "formula":
                    mixed_formula,

                "n_trials":
                    len(
                        mixed_df
                    ),

                "n_subjects":
                    mixed_df[
                        "subject"
                    ].nunique(),

                "reported_different_beta_sec":
                    coefficient,

                "SE":
                    coefficient_se,

                "CI_low":
                    coefficient_ci_low,

                "CI_high":
                    coefficient_ci_high,

                "p":
                    coefficient_p,

                "converged":
                    getattr(
                        mixed_result,
                        "converged",
                        np.nan,
                    ),
            }
        ]
    )

else:

    mixed_summary_df = pd.DataFrame(
        [
            {
                "model_type":
                    "FAILED"
            }
        ]
    )


mixed_summary_df.to_csv(
    os.path.join(
        CSV_DIR,
        "16_SECONDARY_mixed_model.csv",
    ),
    index=False,
)


# =============================================================================
# FIGURE 1 — EXACT COLLABORATOR POOLED COMPARISON
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        7.0,
        6.0,
    )
)


for _, row in (
    subject_pooled_wide
    .iterrows()
):

    ax.plot(
        [
            0,
            1,
        ],
        [
            row[
                "Same"
            ],
            row[
                "Different"
            ],
        ],
        marker="o",
        alpha=0.25,
        linewidth=1.0,
    )


ax.plot(
    [
        0,
        1,
    ],
    [
        same_group_median[
            "median"
        ],
        different_group_median[
            "median"
        ],
    ],
    marker="o",
    linewidth=3.0,
    markersize=9,
    label="Group median",
)


ax.set_xticks(
    [
        0,
        1,
    ]
)


ax.set_xticklabels(
    [
        "Same source",
        "Different sources",
    ]
)


ax.set_ylabel(
    "Median auditory tracking lag (s)"
)


ax.set_title(
    "Exact collaborator-style pooled AV lag analysis"
)


ax.set_ylim(
    bottom=0
)


ax.legend(
    frameon=False
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        FIG_DIR,
        "01_EXACT_COLLABORATOR_pooled_same_different_lag.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)


plt.show()


# =============================================================================
# FIGURE 2 — PARTICIPANT PAIRED DIFFERENCES
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        7.0,
        5.5,
    )
)


ax.hist(
    pooled_difference,
    bins=min(
        15,
        max(
            5,
            len(
                pooled_difference
            )
            // 3,
        ),
    ),
    alpha=0.75,
)


ax.axvline(
    0,
    linestyle="--",
    linewidth=1.5,
)


ax.axvline(
    pooled_difference_summary[
        "mean"
    ],
    linewidth=2.5,
    label=(
        "Mean paired difference = "
        f"{pooled_difference_summary['mean']:.3f} s"
    ),
)


ax.set_xlabel(
    "Different − Same median tracking lag (s)"
)


ax.set_ylabel(
    "Participants"
)


ax.set_title(
    "Exact collaborator-style participant lag differences"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        FIG_DIR,
        "02_EXACT_COLLABORATOR_difference_distribution.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)


plt.show()


# =============================================================================
# FIGURE 3 — PHYSICALLY MATCHED CONDITION EFFECTS
# =============================================================================

plot_data = (
    condition_results_df
    .copy()
)


plot_data[
    "condition_index"
] = (
    plot_data[
        "physical_condition"
    ]
    .map(
        {
            condition:
                index

            for index, condition
            in enumerate(
                PHYSICAL_CONDITION_ORDER
            )
        }
    )
)


plot_data = (
    plot_data
    .sort_values(
        "condition_index"
    )
)


fig, ax = plt.subplots(
    figsize=(
        10.0,
        5.8,
    )
)


valid_plot = plot_data[
    np.isfinite(
        plot_data[
            "mean_Different_minus_Same_sec"
        ]
    )
].copy()


x = (
    valid_plot[
        "condition_index"
    ]
    .to_numpy()
)


y = (
    valid_plot[
        "mean_Different_minus_Same_sec"
    ]
    .to_numpy()
)


lower_error = (
    y
    - valid_plot[
        "CI_low"
    ]
    .to_numpy()
)


upper_error = (
    valid_plot[
        "CI_high"
    ]
    .to_numpy()
    - y
)


ax.errorbar(
    x,
    y,
    yerr=[
        lower_error,
        upper_error,
    ],
    marker="o",
    capsize=4,
    linewidth=2,
)


ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    valid_plot[
        "condition_label"
    ],
    rotation=25,
    ha="right",
)


ax.set_ylabel(
    "Different − Same median tracking lag (s)"
)


ax.set_xlabel(
    "Physical audiovisual condition"
)


ax.set_title(
    "Physically matched causal-judgment lag effects"
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        FIG_DIR,
        "03_MATCHED_condition_specific_effects.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)


plt.show()


# =============================================================================
# FIGURE 4 — WHOLE-TRIAL PERMUTATION NULL
# =============================================================================

if len(
    valid_permutation_null
) > 0:

    fig, ax = plt.subplots(
        figsize=(
            7.0,
            5.5,
        )
    )


    ax.hist(
        valid_permutation_null,
        bins=40,
        alpha=0.75,
    )


    ax.axvline(
        observed_matched_stat,
        linewidth=2.5,
        label=(
            "Observed = "
            f"{observed_matched_stat:.3f} s"
        ),
    )


    ax.axvline(
        0,
        linestyle="--",
        linewidth=1.0,
    )


    ax.set_xlabel(
        "Permuted Different − Same matched lag (s)"
    )


    ax.set_ylabel(
        "Permutation count"
    )


    ax.set_title(
        "Physical-condition-preserving judgment-label null"
    )


    ax.legend(
        frameon=False
    )


    fig.tight_layout()


    fig.savefig(
        os.path.join(
            FIG_DIR,
            "04_MATCHED_whole_trial_permutation_null.png",
        ),
        dpi=FIG_DPI,
        bbox_inches="tight",
    )


    plt.show()


# =============================================================================
# FIGURE 5 — QC RETENTION BY JUDGMENT
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        6.5,
        5.0,
    )
)


qc_plot = (
    qc_judgment_table
    .reset_index()
)


ax.bar(
    qc_plot[
        "judgment"
    ],
    qc_plot[
        "percent_valid"
    ],
)


ax.set_ylabel(
    "Trials retained after exact QC (%)"
)


ax.set_xlabel(
    "Post-trial causal judgment"
)


ax.set_title(
    "Tracking-lag trial retention after r and motion criteria"
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        FIG_DIR,
        "05_QC_retention_by_judgment.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)


plt.show()


# =============================================================================
# MASTER SUMMARY
# =============================================================================

master_summary_df = pd.DataFrame(
    [
        {
            "analysis":
                "Exact collaborator pooled AV",

            "n_subjects":
                len(
                    subject_pooled_wide
                ),

            "same_lag_sec":
                same_group_median[
                    "median"
                ],

            "different_lag_sec":
                different_group_median[
                    "median"
                ],

            "Different_minus_Same_sec":
                pooled_difference_median[
                    "median"
                ],

            "p_two_sided":
                pooled_perm_two[
                    "p_perm"
                ],

            "p_model_direction":
                pooled_perm_greater[
                    "p_perm"
                ],

            "summary_statistic":
                (
                    "Median trials within observer; "
                    "median across observers"
                ),
        },

        {
            "analysis":
                "Physically matched Same/Different",

            "n_subjects":
                matched_summary[
                    "n"
                ],

            "same_lag_sec":
                float(
                    participant_matched_df[
                        "median_same_lag_across_matched_conditions"
                    ]
                    .median()
                ),

            "different_lag_sec":
                float(
                    participant_matched_df[
                        "median_different_lag_across_matched_conditions"
                    ]
                    .median()
                ),

            "Different_minus_Same_sec":
                matched_median_summary[
                    "median"
                ],

            "p_two_sided":
                matched_perm_two[
                    "p_perm"
                ],

            "p_model_direction":
                matched_perm_greater[
                    "p_perm"
                ],

            "summary_statistic":
                (
                    "Median within subject x physical condition x judgment; "
                    "median matched condition effect within observer"
                ),
        },

        {
            "analysis":
                "Matched whole-trial label permutation",

            "n_subjects":
                observed_matched_subject_effect[
                    "subject"
                ].nunique(),

            "same_lag_sec":
                np.nan,

            "different_lag_sec":
                np.nan,

            "Different_minus_Same_sec":
                observed_matched_stat,

            "p_two_sided":
                whole_trial_p_two,

            "p_model_direction":
                whole_trial_p_greater,

            "summary_statistic":
                (
                    "Judgment labels shuffled within "
                    "subject x physical condition x side"
                ),
        },
    ]
)


master_summary_df.to_csv(
    os.path.join(
        CSV_DIR,
        "17_MASTER_RESULTS.csv",
    ),
    index=False,
)


print("\n" + "=" * 100)
print("MASTER RESULTS")
print("=" * 100)


print(
    master_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# AUTOMATIC INTERPRETATION
# =============================================================================

print("\n" + "=" * 100)
print("AUTOMATIC INTERPRETATION")
print("=" * 100)


exact_difference = (
    pooled_difference_median[
        "median"
    ]
)


matched_difference_value = (
    matched_median_summary[
        "median"
    ]
)


print(
    "\nANALYSIS 1 — EXACT COLLABORATOR REPLICATION"
)


print(
    f"\nSame median lag      = "
    f"{same_group_median['median']:.3f} s"
)


print(
    f"Different median lag = "
    f"{different_group_median['median']:.3f} s"
)


print(
    f"Different - Same     = "
    f"{exact_difference:.3f} s"
)


if exact_difference > 0:

    print(
        "\nDirection MATCHES the model prediction:"
        "\nDifferent-source trials show longer auditory tracking lag."
    )

else:

    print(
        "\nDirection DOES NOT MATCH the model prediction:"
        "\nDifferent-source trials do not show longer auditory tracking lag."
    )


print(
    "\nANALYSIS 2 — PHYSICALLY MATCHED TEST"
)


print(
    f"\nMatched Different - Same = "
    f"{matched_difference_value:.3f} s"
)


if matched_difference_value > 0:

    print(
        "\nDirection after controlling physical condition "
        "MATCHES the model prediction."
    )

else:

    print(
        "\nDirection after controlling physical condition "
        "DOES NOT MATCH the model prediction."
    )


print(
    "\nINTERPRETATION LOGIC:"
)


print(
    "\n1. If Analysis 1 reproduces ~0.62 s Same and ~1.21 s Different:"
    "\n   -> the collaborator's behavioral observation is independently replicated."
)


print(
    "\n2. If Analysis 1 is positive but Analysis 2 becomes null:"
    "\n   -> the pooled lag difference is largely explained by differences in "
    "physical AV conditions associated with Same vs Different judgments."
)


print(
    "\n3. If BOTH Analysis 1 and Analysis 2 are positive:"
    "\n   -> strong evidence that causal interpretation itself covaries with "
    "tracking latency beyond physical stimulus condition."
)


print(
    "\n4. If neither analysis is positive:"
    "\n   -> the discrepancy cannot reasonably be attributed to lag-search "
    "direction, r threshold, stimulus-motion criterion, or mean-vs-median "
    "aggregation, because those now match the collaborator's procedure."
)


print(
    "\nIMPORTANT QC CHECK:"
)


print(
    "Compare retention percentages for Same and Different trials. "
    "If r > .40 removes substantially more trials from one judgment category, "
    "that selection difference should be reported and considered when "
    "interpreting the lag contrast."
)


print(
    "\nAll outputs saved to:"
)


print(
    OUT_DIR
)


print(
    "\nDone."
)


EXACT COLLABORATOR-MATCHED SAME/DIFFERENT TRACKING LAG ANALYSIS

Data directory:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL

Output directory:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\02_same_different_tracking_lag

PRIMARY LAG ESTIMATOR:
  Sampling rate                 = 20.0 Hz
  Lag search                    = 0.00 to 4.00 s
  Lag increment                 = 0.05 s
  Minimum accepted peak r       > 0.40
  Minimum auditory stimulus SD >= 3.0°
  Within-observer summary       = MEDIAN

MODEL-PREDICTED DIRECTION:
  Different - Same > 0

Using complete existing trial_df from notebook memory.

INPUT AV DATASET

Total AV trials before lag QC: 5500
Participants: 50

Trials by physical condition:
physical_condition
congruent          500
fixed_10          1000
fixed_20          1000
fixed_30          1000
fixed_40          1000
sweep_diverge      500
sweep_converge     500
Name: count, dtype:

## 4. Kalman-Filter Precision Analysis

Model-based tracking precision/uncertainty analysis across spatial-localization conditions.

In [ ]:
# %% KALMAN FILTER PRECISION ANALYSIS FOR CONTINUOUS SPATIAL LOCALIZATION
# ============================================================
# Goal:
# Estimate model-based spatial tracking uncertainty using a Kalman-filter
# observation-noise parameter, R, across:
#
#   1. audio-only trials
#   2. visual-only trials
#   3. audiovisual congruent trials
#   4. fixed-offset audiovisual trials
#   5. dynamic sweep trials
#
# Main outputs:
#   - trial-level Kalman R, sqrt(R), and log(R)
#   - subject-level natural-condition summaries
#   - fixed-offset target-model comparisons
#   - best-fitting target-model probabilities
#   - optional sweep sliding-window Kalman uncertainty
#
# Interpretation:
#   Lower R / sqrt(R) = lower inferred tracking uncertainty / higher precision.
#
# This analysis is intended as a supporting model-based precision analysis.
# It should complement, not replace, visual capture, hysteresis, residual bias,
# and dynamic BCI analyses.
# ============================================================

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.stats import ttest_rel, ttest_1samp, t as t_dist
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================
# REQUIRED INPUT CHECKS
# ============================================================
if "trial_df" not in globals():
    raise RuntimeError(
        "trial_df not found. Run the spatial notebook loading/resampling cell first "
        "so that trial_df contains t_grid, mouse_x, audio_x, and blob_x arrays."
    )

# Use notebook variables if they already exist.
try:
    RESAMPLE_HZ
except NameError:
    RESAMPLE_HZ = 20.0

try:
    TRIAL_DURATION_SEC
except NameError:
    TRIAL_DURATION_SEC = 18.0

try:
    TIME_GRID
except NameError:
    TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1.0 / RESAMPLE_HZ, 1.0 / RESAMPLE_HZ)

try:
    MIN_VALID_SAMPLES
except NameError:
    MIN_VALID_SAMPLES = 10

try:
    MAX_LAG_SEC
except NameError:
    MAX_LAG_SEC = 2.0

try:
    out_root
except NameError:
    try:
        DATA_DIR
    except NameError:
        DATA_DIR = os.getcwd()
    out_root = os.path.join(DATA_DIR, "spatial_behavior_tonelli_lag_corrected_subject_first")

# ============================================================
# OUTPUT FOLDERS
# ============================================================
KALMAN_OUT_DIR = os.path.join(out_root, "kalman_filter_precision_analysis")
KALMAN_CSV_DIR = os.path.join(KALMAN_OUT_DIR, "csv_outputs")
KALMAN_FIG_DIR = os.path.join(KALMAN_OUT_DIR, "figures")
KALMAN_MODEL_DIR = os.path.join(KALMAN_OUT_DIR, "model_outputs")

for d in [KALMAN_OUT_DIR, KALMAN_CSV_DIR, KALMAN_FIG_DIR, KALMAN_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# USER SETTINGS
# ============================================================
RUN_FULL_TRIAL_KALMAN = True
RUN_SWEEP_WINDOW_KALMAN = True

# Full-trial response versions.
# raw = original mouse_x
# baseline_lagcorr = response shifted using subject-level baseline motor lag
RESPONSE_VERSIONS_FULL_TRIAL = ["raw", "baseline_lagcorr"]

# Sweep windows can be computationally heavier.
# To keep runtime reasonable, sweep windows use baseline-lag-corrected response by default.
RESPONSE_VERSIONS_SWEEP = ["baseline_lagcorr"]

# Kalman filter settings.
KALMAN_PROCESS_NOISE_PER_SEC = 0.25
KALMAN_INIT_STATE_VAR = 25.0
KALMAN_R_MIN = 1e-5
KALMAN_R_MAX = 1e5
KALMAN_MIN_SAMPLES = 25

# R estimation settings.
# Full trials use maximum-likelihood optimization.
# Sweep windows can use a faster residual-initialized estimate to reduce runtime.
FAST_R_FOR_SWEEP_WINDOWS = True

# Sweep sliding-window settings.
SWEEP_WINDOW_SEC = 2.0
SWEEP_STEP_SEC = 0.50
SWEEP_MIN_VALID_SAMPLES = 20

# Empirical integration target settings.
# Empirical integrated target = wV * visual + (1 - wV) * audio
# wV is estimated from each subject's fixed-offset visual capture.
CLIP_EMPIRICAL_WV_TO_0_1 = True
DEFAULT_WV_IF_MISSING = 0.50

# Candidate target models.
FIXED_OFFSET_TARGET_MODELS = [
    "audio_target",
    "visual_target",
    "midpoint_target",
    "empirical_integrated_target",
]

SWEEP_TARGET_MODELS = [
    "audio_target",
    "visual_target",
    "midpoint_target",
    "empirical_integrated_target",
]

FIG_DPI = 180

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def sem(vals):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    if len(vals) < 2:
        return np.nan
    return float(np.std(vals, ddof=1) / np.sqrt(len(vals)))

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)
    if n < 2:
        return np.nan, np.nan, np.nan, n
    m = float(np.mean(vals))
    se = float(np.std(vals, ddof=1) / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    return m, m - crit * se, m + crit * se, n

def make_safe_filename(text, max_len=120):
    """
    Convert model labels into Windows-safe ASCII filenames.
    This prevents FileNotFoundError/encoding/path issues caused by characters
    such as em dashes, slashes, colons, or very long labels.
    """
    text = str(text).lower()

    replacements = {
        "—": "_",
        "–": "_",
        "-": "_",
        " ": "_",
        "/": "_",
        "\\": "_",
        ":": "",
        ";": "",
        "~": "",
        "*": "x",
        "+": "plus",
        "(": "",
        ")": "",
        "[": "",
        "]": "",
        "{": "",
        "}": "",
        "<": "",
        ">": "",
        "|": "",
        "?": "",
        "!": "",
        ",": "",
        ".": "",
        "'": "",
        '"': "",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"[^a-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")

    if len(text) == 0:
        text = "model"

    return text[:max_len]

def parse_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return float(abs(float(row["offsetDeg"])))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return float(m.group(1))

    return np.nan

def is_fixed_offset_trial(row):
    ttype = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()

    if ttype == "incongruent_spatial":
        return True

    if ("spatial" in cond or "fixed" in cond or "static" in cond) and not ("sweep" in cond):
        if re.search(r"(10|20|30|40)", cond):
            return True

    return False

def is_sweep_trial(row):
    ttype = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    ramp = str(row.get("rampType", "")).lower()

    return (
        ttype == "incongruent_sweep"
        or "sweep" in cond
        or "sweep" in ttype
        or "diverge" in cond
        or "converge" in cond
        or "diverge" in ramp
        or "converge" in ramp
    )

def safe_array(x):
    return np.asarray(x, dtype=float)

def common_av_target(audio_x, visual_x):
    audio_x = safe_array(audio_x)
    visual_x = safe_array(visual_x)

    stacked = np.vstack([audio_x, visual_x])
    with np.errstate(invalid="ignore"):
        return np.nanmean(stacked, axis=0)

def midpoint_target(audio_x, visual_x):
    return common_av_target(audio_x, visual_x)

def visual_capture_value(response, visual, audio, min_abs_disparity=0.01):
    response = safe_array(response)
    visual = safe_array(visual)
    audio = safe_array(audio)

    denom = visual - audio
    ok = (
        np.isfinite(response)
        & np.isfinite(visual)
        & np.isfinite(audio)
        & (np.abs(denom) > min_abs_disparity)
    )

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean((response[ok] - audio[ok]) / denom[ok]))

def corr_1d_local(x, y):
    x = safe_array(x)
    y = safe_array(y)

    ok = np.isfinite(x) & np.isfinite(y)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    if np.nanstd(x[ok]) < 1e-12 or np.nanstd(y[ok]) < 1e-12:
        return np.nan

    return float(np.corrcoef(x[ok], y[ok])[0, 1])

def lag_scanned_corr_local(response, stimulus, fs=20.0, max_lag_sec=2.0):
    response = safe_array(response)
    stimulus = safe_array(stimulus)

    max_lag_samples = int(round(max_lag_sec * fs))
    lag_samples_grid = np.arange(-max_lag_samples, max_lag_samples + 1)

    best_corr = np.nan
    best_lag_samples = np.nan

    for lag_samples in lag_samples_grid:
        if lag_samples > 0:
            r = response[lag_samples:]
            s = stimulus[:-lag_samples]
        elif lag_samples < 0:
            r = response[:lag_samples]
            s = stimulus[-lag_samples:]
        else:
            r = response
            s = stimulus

        c = corr_1d_local(r, s)

        if np.isfinite(c) and (not np.isfinite(best_corr) or c > best_corr):
            best_corr = c
            best_lag_samples = lag_samples

    return {
        "peak_corr": best_corr,
        "lag_samples": best_lag_samples,
        "lag_sec": best_lag_samples / fs if np.isfinite(best_lag_samples) else np.nan,
    }

def lag_correct_response_local(response, t_grid, lag_sec):
    response = safe_array(response)
    t_grid = safe_array(t_grid)

    if not np.isfinite(lag_sec):
        return response.copy()

    return np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan
    )

def infer_natural_target_for_baseline_lag(row):
    ttype = str(row.get("trialType", "")).lower()

    audio_x = safe_array(row["audio_x"])
    visual_x = safe_array(row["blob_x"])

    if ttype == "audio_only":
        return audio_x, "audio_target"

    if ttype == "visual_only":
        return visual_x, "visual_target"

    if ttype == "congruent":
        return common_av_target(audio_x, visual_x), "common_av_target"

    return None, None

def get_subject_baseline_lags(trial_df):
    """
    Estimate subject-level baseline motor lag from audio-only, visual-only,
    and congruent trials only. This avoids estimating lag from conflict trials.
    """
    rows = []

    for idx, row in trial_df.iterrows():
        target, target_name = infer_natural_target_for_baseline_lag(row)

        if target is None:
            continue

        response = safe_array(row["mouse_x"])

        res = lag_scanned_corr_local(
            response=response,
            stimulus=target,
            fs=RESAMPLE_HZ,
            max_lag_sec=MAX_LAG_SEC
        )

        rows.append({
            "subject": row["subject"],
            "trialIndex": row["trialIndex"],
            "trialType": row["trialType"],
            "condLabel": row.get("condLabel", np.nan),
            "baseline_target_model": target_name,
            "baseline_lag_sec": res["lag_sec"],
            "baseline_peak_corr": res["peak_corr"],
        })

    lag_df = pd.DataFrame(rows)

    if len(lag_df) == 0:
        return pd.DataFrame(columns=["subject", "subject_baseline_lag_sec", "n_baseline_lag_trials"]), lag_df

    subj_lag_df = (
        lag_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["baseline_lag_sec"])
        .groupby("subject", as_index=False)
        .agg(
            subject_baseline_lag_sec=("baseline_lag_sec", "median"),
            subject_baseline_lag_mean_sec=("baseline_lag_sec", "mean"),
            n_baseline_lag_trials=("baseline_lag_sec", "count"),
            baseline_peak_corr_mean=("baseline_peak_corr", "mean"),
        )
    )

    return subj_lag_df, lag_df

def build_empirical_capture_weights(trial_df):
    """
    Estimate empirical visual capture weights from fixed-offset trials.
    These are used to construct empirical integrated targets:
        empirical_target = wV * visual + (1 - wV) * audio
    """
    rows = []

    for idx, row in trial_df.iterrows():
        if not is_fixed_offset_trial(row):
            continue

        offset_abs = parse_offset_from_row(row)

        if not np.isfinite(offset_abs):
            continue

        response = safe_array(row["mouse_x"])
        visual = safe_array(row["blob_x"])
        audio = safe_array(row["audio_x"])

        cap = visual_capture_value(response, visual, audio)

        rows.append({
            "subject": row["subject"],
            "trialIndex": row["trialIndex"],
            "condLabel": row.get("condLabel", np.nan),
            "offset_abs": offset_abs,
            "trial_visual_capture": cap,
        })

    capture_df = pd.DataFrame(rows)

    if len(capture_df) == 0:
        empty_cols = ["subject", "offset_abs", "wv_subject_offset", "n_capture_trials"]
        return pd.DataFrame(columns=empty_cols), pd.DataFrame(), pd.DataFrame(), capture_df

    capture_df = capture_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["trial_visual_capture"])

    if CLIP_EMPIRICAL_WV_TO_0_1:
        capture_df["trial_visual_capture_for_weight"] = capture_df["trial_visual_capture"].clip(0, 1)
    else:
        capture_df["trial_visual_capture_for_weight"] = capture_df["trial_visual_capture"]

    subj_offset_w = (
        capture_df
        .groupby(["subject", "offset_abs"], as_index=False)
        .agg(
            wv_subject_offset=("trial_visual_capture_for_weight", "mean"),
            wv_subject_offset_unclipped=("trial_visual_capture", "mean"),
            n_capture_trials=("trial_visual_capture", "count"),
        )
    )

    subj_w = (
        capture_df
        .groupby("subject", as_index=False)
        .agg(
            wv_subject_mean=("trial_visual_capture_for_weight", "mean"),
            wv_subject_mean_unclipped=("trial_visual_capture", "mean"),
            n_subject_capture_trials=("trial_visual_capture", "count"),
        )
    )

    group_offset_w = (
        capture_df
        .groupby("offset_abs", as_index=False)
        .agg(
            wv_group_offset=("trial_visual_capture_for_weight", "mean"),
            wv_group_offset_unclipped=("trial_visual_capture", "mean"),
            n_group_offset_capture_trials=("trial_visual_capture", "count"),
        )
    )

    return subj_offset_w, subj_w, group_offset_w, capture_df

def lookup_empirical_wv(subject, offset_abs, subj_offset_w, subj_w, group_offset_w):
    """
    Lookup order:
      1. subject × offset
      2. subject mean across fixed offsets
      3. group offset mean
      4. default .50
    """
    if np.isfinite(offset_abs):
        m = subj_offset_w[
            (subj_offset_w["subject"] == subject)
            & (np.isclose(subj_offset_w["offset_abs"], offset_abs))
        ]

        if len(m) > 0 and np.isfinite(m["wv_subject_offset"].iloc[0]):
            return float(m["wv_subject_offset"].iloc[0]), "subject_offset"

    m = subj_w[subj_w["subject"] == subject]
    if len(m) > 0 and np.isfinite(m["wv_subject_mean"].iloc[0]):
        return float(m["wv_subject_mean"].iloc[0]), "subject_mean"

    if np.isfinite(offset_abs):
        m = group_offset_w[np.isclose(group_offset_w["offset_abs"], offset_abs)]
        if len(m) > 0 and np.isfinite(m["wv_group_offset"].iloc[0]):
            return float(m["wv_group_offset"].iloc[0]), "group_offset"

    return DEFAULT_WV_IF_MISSING, "default"

def build_target_models_for_trial(row, subj_offset_w, subj_w, group_offset_w, target_model_names=None):
    """
    Returns list of dictionaries:
      target_model, target_x, empirical_wv, empirical_wv_source
    """
    ttype = str(row.get("trialType", "")).lower()
    subject = row["subject"]
    offset_abs = parse_offset_from_row(row)

    audio_x = safe_array(row["audio_x"])
    visual_x = safe_array(row["blob_x"])

    common_target = common_av_target(audio_x, visual_x)
    mid_target = midpoint_target(audio_x, visual_x)

    wv, wv_source = lookup_empirical_wv(subject, offset_abs, subj_offset_w, subj_w, group_offset_w)
    empirical_target = wv * visual_x + (1.0 - wv) * audio_x

    if ttype == "audio_only":
        return [{
            "target_model": "audio_target",
            "target_family": "natural_unimodal_congruent",
            "target_x": audio_x,
            "empirical_wv": np.nan,
            "empirical_wv_source": "not_applicable",
        }]

    if ttype == "visual_only":
        return [{
            "target_model": "visual_target",
            "target_family": "natural_unimodal_congruent",
            "target_x": visual_x,
            "empirical_wv": np.nan,
            "empirical_wv_source": "not_applicable",
        }]

    if ttype == "congruent":
        return [{
            "target_model": "common_av_target",
            "target_family": "natural_unimodal_congruent",
            "target_x": common_target,
            "empirical_wv": np.nan,
            "empirical_wv_source": "not_applicable",
        }]

    if is_fixed_offset_trial(row):
        model_map = {
            "audio_target": audio_x,
            "visual_target": visual_x,
            "midpoint_target": mid_target,
            "empirical_integrated_target": empirical_target,
        }

        names = target_model_names if target_model_names is not None else FIXED_OFFSET_TARGET_MODELS

        return [
            {
                "target_model": name,
                "target_family": "fixed_offset_candidate",
                "target_x": model_map[name],
                "empirical_wv": wv if name == "empirical_integrated_target" else np.nan,
                "empirical_wv_source": wv_source if name == "empirical_integrated_target" else "not_applicable",
            }
            for name in names
            if name in model_map
        ]

    if is_sweep_trial(row):
        model_map = {
            "audio_target": audio_x,
            "visual_target": visual_x,
            "midpoint_target": mid_target,
            "empirical_integrated_target": empirical_target,
        }

        names = target_model_names if target_model_names is not None else SWEEP_TARGET_MODELS

        return [
            {
                "target_model": name,
                "target_family": "sweep_candidate",
                "target_x": model_map[name],
                "empirical_wv": wv if name == "empirical_integrated_target" else np.nan,
                "empirical_wv_source": wv_source if name == "empirical_integrated_target" else "not_applicable",
            }
            for name in names
            if name in model_map
        ]

    return []

# ============================================================
# KALMAN FILTER FUNCTIONS
# ============================================================
def kalman_neg_loglik_response_to_target(response, target, t_grid, R, q_per_sec=0.25, init_state_var=25.0):
    """
    1D Kalman filter with known target increments as the state-transition input.
    """
    response = safe_array(response)
    target = safe_array(target)
    t_grid = safe_array(t_grid)

    ok = np.isfinite(response) & np.isfinite(target) & np.isfinite(t_grid)
    response = response[ok]
    target = target[ok]
    t_grid = t_grid[ok]

    if len(response) < KALMAN_MIN_SAMPLES:
        return np.nan

    R = float(R)
    if not np.isfinite(R) or R <= 0:
        return np.nan

    x_est = float(target[0])
    P = float(init_state_var)

    nll = 0.0
    used = 0

    for i in range(len(response)):
        if i == 0:
            x_pred = float(target[0])
            P_pred = float(P + q_per_sec * (1.0 / RESAMPLE_HZ))
        else:
            dt = float(t_grid[i] - t_grid[i - 1])
            if not np.isfinite(dt) or dt <= 0:
                dt = 1.0 / RESAMPLE_HZ

            target_delta = float(target[i] - target[i - 1])
            x_pred = float(x_est + target_delta)
            P_pred = float(P + q_per_sec * dt)

        innov = float(response[i] - x_pred)
        S = float(P_pred + R)

        if S <= 0 or not np.isfinite(S):
            return np.nan

        nll += 0.5 * (np.log(2.0 * np.pi * S) + (innov ** 2) / S)

        K = P_pred / S
        x_est = x_pred + K * innov
        P = (1.0 - K) * P_pred

        used += 1

    if used < KALMAN_MIN_SAMPLES:
        return np.nan

    return float(nll)

def estimate_kalman_R(response, target, t_grid, fast=False):
    """
    Estimate R for one response-target pair.
    """
    response = safe_array(response)
    target = safe_array(target)
    t_grid = safe_array(t_grid)

    ok = np.isfinite(response) & np.isfinite(target) & np.isfinite(t_grid)

    if ok.sum() < KALMAN_MIN_SAMPLES:
        return {
            "kalman_R": np.nan,
            "kalman_sqrt_R": np.nan,
            "kalman_log_R": np.nan,
            "kalman_nll": np.nan,
            "kalman_aic": np.nan,
            "kalman_bic": np.nan,
            "n_kalman_samples": int(ok.sum()),
            "residual_mse": np.nan,
            "residual_rmse": np.nan,
            "residual_mean_abs": np.nan,
            "fit_status": "too_few_samples",
        }

    y = response[ok]
    x = target[ok]
    t = t_grid[ok]

    resid = y - x
    residual_mse = float(np.nanmean(resid ** 2))
    residual_rmse = float(np.sqrt(residual_mse))
    residual_mean_abs = float(np.nanmean(np.abs(resid)))

    if not np.isfinite(residual_mse) or residual_mse <= 0:
        residual_mse = 1e-4

    if fast:
        R_hat = float(np.clip(residual_mse, KALMAN_R_MIN, KALMAN_R_MAX))
        nll = kalman_neg_loglik_response_to_target(
            response=y,
            target=x,
            t_grid=t,
            R=R_hat,
            q_per_sec=KALMAN_PROCESS_NOISE_PER_SEC,
            init_state_var=KALMAN_INIT_STATE_VAR
        )

        if not np.isfinite(nll):
            return {
                "kalman_R": np.nan,
                "kalman_sqrt_R": np.nan,
                "kalman_log_R": np.nan,
                "kalman_nll": np.nan,
                "kalman_aic": np.nan,
                "kalman_bic": np.nan,
                "n_kalman_samples": int(ok.sum()),
                "residual_mse": residual_mse,
                "residual_rmse": residual_rmse,
                "residual_mean_abs": residual_mean_abs,
                "fit_status": "fast_fit_failed",
            }

        n = int(ok.sum())
        p = 1
        return {
            "kalman_R": R_hat,
            "kalman_sqrt_R": float(np.sqrt(R_hat)),
            "kalman_log_R": float(np.log(R_hat)),
            "kalman_nll": float(nll),
            "kalman_aic": float(2 * p + 2 * nll),
            "kalman_bic": float(p * np.log(n) + 2 * nll),
            "n_kalman_samples": n,
            "residual_mse": residual_mse,
            "residual_rmse": residual_rmse,
            "residual_mean_abs": residual_mean_abs,
            "fit_status": "fast_residual_initialized",
        }

    lo = max(KALMAN_R_MIN, residual_mse / 1000.0)
    hi = min(KALMAN_R_MAX, max(residual_mse * 1000.0, lo * 1000.0, 1.0))

    def obj(log_R):
        R = np.exp(log_R)
        nll = kalman_neg_loglik_response_to_target(
            response=y,
            target=x,
            t_grid=t,
            R=R,
            q_per_sec=KALMAN_PROCESS_NOISE_PER_SEC,
            init_state_var=KALMAN_INIT_STATE_VAR
        )

        if not np.isfinite(nll):
            return 1e100

        return nll

    try:
        opt = minimize_scalar(
            obj,
            bounds=(np.log(lo), np.log(hi)),
            method="bounded",
            options={"xatol": 1e-4, "maxiter": 100}
        )

        fit_status = "mle_ok" if opt.success else "optimizer_not_successful"

        R_hat = float(np.exp(opt.x))
        nll = float(opt.fun)

    except Exception as e:
        return {
            "kalman_R": np.nan,
            "kalman_sqrt_R": np.nan,
            "kalman_log_R": np.nan,
            "kalman_nll": np.nan,
            "kalman_aic": np.nan,
            "kalman_bic": np.nan,
            "n_kalman_samples": int(ok.sum()),
            "residual_mse": residual_mse,
            "residual_rmse": residual_rmse,
            "residual_mean_abs": residual_mean_abs,
            "fit_status": f"mle_failed: {str(e)}",
        }

    n = int(ok.sum())
    p = 1

    return {
        "kalman_R": R_hat,
        "kalman_sqrt_R": float(np.sqrt(R_hat)),
        "kalman_log_R": float(np.log(R_hat)),
        "kalman_nll": nll,
        "kalman_aic": float(2 * p + 2 * nll),
        "kalman_bic": float(p * np.log(n) + 2 * nll),
        "n_kalman_samples": n,
        "residual_mse": residual_mse,
        "residual_rmse": residual_rmse,
        "residual_mean_abs": residual_mean_abs,
        "fit_status": fit_status,
    }

def condition_class_from_row(row):
    ttype = str(row.get("trialType", "")).lower()

    if ttype == "audio_only":
        return "Audio-only"
    if ttype == "visual_only":
        return "Visual-only"
    if ttype == "congruent":
        return "AV congruent"
    if is_fixed_offset_trial(row):
        return "Fixed offset AV"
    if is_sweep_trial(row):
        return "Sweep AV"

    return "Other"

def get_response_for_version(row, response_version, subject_lag_lookup):
    response_raw = safe_array(row["mouse_x"])
    t_grid = safe_array(row["t_grid"])

    if response_version == "raw":
        return response_raw

    if response_version == "baseline_lagcorr":
        subj = row["subject"]
        lag = subject_lag_lookup.get(subj, 0.0)
        if not np.isfinite(lag):
            lag = 0.0
        return lag_correct_response_local(response_raw, t_grid, lag)

    raise ValueError(f"Unknown response_version: {response_version}")

def fit_mixed_or_clustered_ols(formula, data, label, out_dir, groups_col="subject"):
    """
    Try random-intercept mixed model first. If it fails, use clustered OLS.
    Saves model summary and coefficient table.

    This version uses safe ASCII filenames and re-creates the output folder
    immediately before saving. This prevents Windows/OneDrive path errors from
    labels containing characters such as em dashes.
    """
    os.makedirs(out_dir, exist_ok=True)

    d = data.copy().replace([np.inf, -np.inf], np.nan)
    needed_cols = [groups_col]

    for token in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", formula):
        if token in d.columns and token not in needed_cols:
            needed_cols.append(token)

    d = d.dropna(subset=[c for c in needed_cols if c in d.columns])

    if len(d) < 10 or d[groups_col].nunique() < 3:
        return None, pd.DataFrame()

    safe_label = make_safe_filename(label, max_len=110)

    try:
        model = smf.mixedlm(formula, data=d, groups=d[groups_col], re_formula="1")
        result = model.fit(method="powell", reml=False, maxiter=1000, disp=False)
        model_type = "mixedlm_random_intercept"

        fixed_terms = list(result.fe_params.index)
        coef_df = pd.DataFrame({
            "model_label": label,
            "model_type": model_type,
            "term": fixed_terms,
            "estimate": result.fe_params.values,
            "se": result.bse.reindex(fixed_terms).values,
            "z_or_t": result.tvalues.reindex(fixed_terms).values,
            "p": result.pvalues.reindex(fixed_terms).values,
            "n_rows": len(d),
            "n_subjects": d[groups_col].nunique(),
            "formula": formula,
        })

    except Exception as e:
        result = smf.ols(formula, data=d).fit(
            cov_type="cluster",
            cov_kwds={"groups": d[groups_col]}
        )
        model_type = f"clustered_ols_fallback_after_mixedlm_error: {str(e)}"

        coef_df = pd.DataFrame({
            "model_label": label,
            "model_type": model_type,
            "term": result.params.index,
            "estimate": result.params.values,
            "se": result.bse.reindex(result.params.index).values,
            "z_or_t": result.tvalues.reindex(result.params.index).values,
            "p": result.pvalues.reindex(result.params.index).values,
            "n_rows": len(d),
            "n_subjects": d[groups_col].nunique(),
            "formula": formula,
        })

    summary_path = os.path.join(out_dir, f"{safe_label}_summary.txt")
    coef_path = os.path.join(out_dir, f"{safe_label}_coefficients.csv")

    try:
        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(str(result.summary()))
    except Exception as e:
        print(f"WARNING: Could not save model summary to {summary_path}")
        print(f"Reason: {e}")

    try:
        coef_df.to_csv(coef_path, index=False)
    except Exception as e:
        print(f"WARNING: Could not save coefficient table to {coef_path}")
        print(f"Reason: {e}")

    print("\n" + "=" * 90)
    print(label)
    print("=" * 90)
    print(f"Formula: {formula}")
    print(f"Model type: {model_type}")
    print(f"Rows: {len(d)} | Subjects: {d[groups_col].nunique()}")
    print(f"Model summary path: {summary_path}")
    print(f"Coefficient path: {coef_path}")
    print(coef_df[["term", "estimate", "se", "z_or_t", "p"]].to_string(index=False))

    return result, coef_df

# ============================================================
# PREPARE DATA
# ============================================================
trial_df = trial_df.copy()

required_cols = ["subject", "trialIndex", "trialType", "condLabel", "t_grid", "mouse_x", "audio_x", "blob_x"]
missing = [c for c in required_cols if c not in trial_df.columns]
if missing:
    raise RuntimeError(f"trial_df is missing required columns: {missing}")

for c in ["offsetDeg", "sideSign", "rampType"]:
    if c not in trial_df.columns:
        trial_df[c] = np.nan

trial_df["condition_class"] = trial_df.apply(condition_class_from_row, axis=1)
trial_df["offset_abs"] = trial_df.apply(parse_offset_from_row, axis=1)

subject_lag_df, baseline_lag_trial_df = get_subject_baseline_lags(trial_df)

baseline_lag_csv = os.path.join(KALMAN_CSV_DIR, "kalman_baseline_lag_trial_level.csv")
subject_lag_csv = os.path.join(KALMAN_CSV_DIR, "kalman_subject_baseline_lags.csv")
baseline_lag_trial_df.to_csv(baseline_lag_csv, index=False)
subject_lag_df.to_csv(subject_lag_csv, index=False)

subject_lag_lookup = dict(zip(subject_lag_df["subject"], subject_lag_df["subject_baseline_lag_sec"]))

subj_offset_w, subj_w, group_offset_w, capture_weight_trial_df = build_empirical_capture_weights(trial_df)

subj_offset_w_csv = os.path.join(KALMAN_CSV_DIR, "kalman_empirical_capture_weights_subject_offset.csv")
subj_w_csv = os.path.join(KALMAN_CSV_DIR, "kalman_empirical_capture_weights_subject.csv")
group_offset_w_csv = os.path.join(KALMAN_CSV_DIR, "kalman_empirical_capture_weights_group_offset.csv")
capture_weight_trial_csv = os.path.join(KALMAN_CSV_DIR, "kalman_empirical_capture_weights_trial_level.csv")

subj_offset_w.to_csv(subj_offset_w_csv, index=False)
subj_w.to_csv(subj_w_csv, index=False)
group_offset_w.to_csv(group_offset_w_csv, index=False)
capture_weight_trial_df.to_csv(capture_weight_trial_csv, index=False)

print("\nBaseline lag summary:")
print(subject_lag_df.describe(include="all"))

print("\nEmpirical capture weight summary:")
if len(subj_offset_w) > 0:
    print(
        subj_offset_w
        .groupby("offset_abs", observed=True)
        .agg(
            mean_wv=("wv_subject_offset", "mean"),
            se_wv=("wv_subject_offset", sem),
            n_subjects=("subject", "nunique")
        )
        .to_string()
    )
else:
    print("No fixed-offset capture weights found. Empirical integrated targets will use DEFAULT_WV_IF_MISSING.")

# ============================================================
# STEP 1 — FULL-TRIAL KALMAN FITS
# ============================================================
kalman_trial_rows = []

if RUN_FULL_TRIAL_KALMAN:
    for idx, row in trial_df.iterrows():
        t_grid = safe_array(row["t_grid"])
        audio_x = safe_array(row["audio_x"])
        visual_x = safe_array(row["blob_x"])

        mean_signed_disparity = float(np.nanmean(visual_x - audio_x))
        mean_abs_disparity = float(np.nanmean(np.abs(visual_x - audio_x)))

        targets = build_target_models_for_trial(
            row=row,
            subj_offset_w=subj_offset_w,
            subj_w=subj_w,
            group_offset_w=group_offset_w,
            target_model_names=None
        )

        if len(targets) == 0:
            continue

        for response_version in RESPONSE_VERSIONS_FULL_TRIAL:
            response = get_response_for_version(row, response_version, subject_lag_lookup)

            for target_info in targets:
                target_x = target_info["target_x"]

                fit = estimate_kalman_R(
                    response=response,
                    target=target_x,
                    t_grid=t_grid,
                    fast=False
                )

                kalman_trial_rows.append({
                    "subject": row["subject"],
                    "source_file": row.get("source_file", np.nan),
                    "trialIndex": row["trialIndex"],
                    "trial_order_within_subject": row.get("trial_order_within_subject", np.nan),
                    "trialType": row["trialType"],
                    "condLabel": row.get("condLabel", np.nan),
                    "condition_class": row["condition_class"],
                    "offset_abs": row["offset_abs"],
                    "sideSign": row.get("sideSign", np.nan),
                    "rampType": row.get("rampType", np.nan),

                    "response_version": response_version,
                    "subject_baseline_lag_sec": subject_lag_lookup.get(row["subject"], np.nan),

                    "target_family": target_info["target_family"],
                    "target_model": target_info["target_model"],
                    "empirical_wv": target_info["empirical_wv"],
                    "empirical_wv_source": target_info["empirical_wv_source"],

                    "mean_signed_disparity": mean_signed_disparity,
                    "mean_abs_disparity": mean_abs_disparity,

                    **fit,
                })

kalman_trial_df = pd.DataFrame(kalman_trial_rows)

kalman_trial_csv = os.path.join(KALMAN_CSV_DIR, "kalman_trial_level_full_trial_R_estimates.csv")
kalman_trial_df.to_csv(kalman_trial_csv, index=False)

print(f"\nSaved full-trial Kalman estimates: {kalman_trial_csv}")
print(f"Rows: {len(kalman_trial_df)}")
if len(kalman_trial_df) > 0:
    print(
        kalman_trial_df
        .groupby(["response_version", "condition_class", "target_model"], observed=True)
        .size()
    )

# ============================================================
# STEP 2 — SWEEP SLIDING-WINDOW KALMAN FITS
# ============================================================
sweep_window_rows = []

if RUN_SWEEP_WINDOW_KALMAN:
    sweep_df = trial_df[trial_df.apply(is_sweep_trial, axis=1)].copy()

    sweep_window_starts = np.arange(
        0,
        TRIAL_DURATION_SEC - SWEEP_WINDOW_SEC + 1e-9,
        SWEEP_STEP_SEC
    )

    print(f"\nRunning sweep-window Kalman fits on {len(sweep_df)} sweep trials...")

    for idx, row in sweep_df.iterrows():
        t_grid = safe_array(row["t_grid"])
        audio_x = safe_array(row["audio_x"])
        visual_x = safe_array(row["blob_x"])

        targets = build_target_models_for_trial(
            row=row,
            subj_offset_w=subj_offset_w,
            subj_w=subj_w,
            group_offset_w=group_offset_w,
            target_model_names=SWEEP_TARGET_MODELS
        )

        if len(targets) == 0:
            continue

        for response_version in RESPONSE_VERSIONS_SWEEP:
            response_full = get_response_for_version(row, response_version, subject_lag_lookup)

            for start in sweep_window_starts:
                end = start + SWEEP_WINDOW_SEC
                center = start + SWEEP_WINDOW_SEC / 2.0

                mask = (t_grid >= start) & (t_grid < end)

                if mask.sum() < SWEEP_MIN_VALID_SAMPLES:
                    continue

                t_win = t_grid[mask]
                response_win = response_full[mask]
                audio_win = audio_x[mask]
                visual_win = visual_x[mask]

                mean_signed_disparity = float(np.nanmean(visual_win - audio_win))
                mean_abs_disparity = float(np.nanmean(np.abs(visual_win - audio_win)))

                for target_info in targets:
                    target_win = safe_array(target_info["target_x"])[mask]

                    fit = estimate_kalman_R(
                        response=response_win,
                        target=target_win,
                        t_grid=t_win,
                        fast=FAST_R_FOR_SWEEP_WINDOWS
                    )

                    sweep_window_rows.append({
                        "subject": row["subject"],
                        "source_file": row.get("source_file", np.nan),
                        "trialIndex": row["trialIndex"],
                        "trial_order_within_subject": row.get("trial_order_within_subject", np.nan),
                        "trialType": row["trialType"],
                        "condLabel": row.get("condLabel", np.nan),
                        "condition_class": row["condition_class"],
                        "offset_abs": row["offset_abs"],
                        "sideSign": row.get("sideSign", np.nan),
                        "rampType": row.get("rampType", np.nan),

                        "response_version": response_version,
                        "subject_baseline_lag_sec": subject_lag_lookup.get(row["subject"], np.nan),

                        "window_start_sec": start,
                        "window_end_sec": end,
                        "window_center_sec": center,

                        "target_family": target_info["target_family"],
                        "target_model": target_info["target_model"],
                        "empirical_wv": target_info["empirical_wv"],
                        "empirical_wv_source": target_info["empirical_wv_source"],

                        "mean_signed_disparity": mean_signed_disparity,
                        "mean_abs_disparity": mean_abs_disparity,

                        **fit,
                    })

kalman_sweep_window_df = pd.DataFrame(sweep_window_rows)

kalman_sweep_window_csv = os.path.join(KALMAN_CSV_DIR, "kalman_sweep_window_level_R_estimates.csv")
kalman_sweep_window_df.to_csv(kalman_sweep_window_csv, index=False)

print(f"\nSaved sweep-window Kalman estimates: {kalman_sweep_window_csv}")
print(f"Rows: {len(kalman_sweep_window_df)}")

# ============================================================
# STEP 3 — SUBJECT-LEVEL SUMMARIES
# ============================================================
natural_df = kalman_trial_df[
    kalman_trial_df["target_family"] == "natural_unimodal_congruent"
].copy()

natural_condition_order = ["Audio-only", "Visual-only", "AV congruent"]

if len(natural_df) > 0:
    natural_df["condition_class"] = pd.Categorical(
        natural_df["condition_class"],
        categories=natural_condition_order,
        ordered=True
    )

natural_subject_df = (
    natural_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["kalman_log_R"])
    .groupby(["subject", "response_version", "condition_class"], as_index=False, observed=True)
    .agg(
        n_trials=("trialIndex", "count"),
        mean_R=("kalman_R", "mean"),
        mean_sqrt_R=("kalman_sqrt_R", "mean"),
        mean_log_R=("kalman_log_R", "mean"),
        mean_residual_rmse=("residual_rmse", "mean"),
        mean_residual_abs=("residual_mean_abs", "mean"),
    )
)

natural_subject_csv = os.path.join(KALMAN_CSV_DIR, "kalman_subject_natural_condition_summary.csv")
natural_subject_df.to_csv(natural_subject_csv, index=False)

fixed_df = kalman_trial_df[
    kalman_trial_df["target_family"] == "fixed_offset_candidate"
].copy()

fixed_subject_model_df = (
    fixed_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["kalman_log_R", "offset_abs"])
    .groupby(["subject", "response_version", "offset_abs", "target_model"], as_index=False, observed=True)
    .agg(
        n_trials=("trialIndex", "count"),
        mean_R=("kalman_R", "mean"),
        mean_sqrt_R=("kalman_sqrt_R", "mean"),
        mean_log_R=("kalman_log_R", "mean"),
        mean_bic=("kalman_bic", "mean"),
        mean_aic=("kalman_aic", "mean"),
        mean_residual_rmse=("residual_rmse", "mean"),
        mean_residual_abs=("residual_mean_abs", "mean"),
        mean_empirical_wv=("empirical_wv", "mean"),
    )
)

fixed_subject_model_csv = os.path.join(KALMAN_CSV_DIR, "kalman_subject_fixed_offset_target_model_summary.csv")
fixed_subject_model_df.to_csv(fixed_subject_model_csv, index=False)

fixed_best_rows = []

if len(fixed_df) > 0:
    for keys, sub in fixed_df.dropna(subset=["kalman_bic"]).groupby(
        ["subject", "trialIndex", "response_version"],
        dropna=False,
        observed=True
    ):
        subject, trial_index, response_version = keys
        best = sub.sort_values("kalman_bic").iloc[0]

        fixed_best_rows.append({
            "subject": subject,
            "trialIndex": trial_index,
            "response_version": response_version,
            "trialType": best["trialType"],
            "condLabel": best["condLabel"],
            "offset_abs": best["offset_abs"],
            "sideSign": best.get("sideSign", np.nan),
            "best_target_model": best["target_model"],
            "best_kalman_bic": best["kalman_bic"],
            "best_kalman_log_R": best["kalman_log_R"],
            "best_kalman_sqrt_R": best["kalman_sqrt_R"],
        })

fixed_best_df = pd.DataFrame(fixed_best_rows)

fixed_best_csv = os.path.join(KALMAN_CSV_DIR, "kalman_fixed_offset_best_target_model_trial_level.csv")
fixed_best_df.to_csv(fixed_best_csv, index=False)

if len(fixed_best_df) > 0:
    fixed_best_subject_df = (
        fixed_best_df
        .groupby(["subject", "response_version", "offset_abs", "best_target_model"], as_index=False, observed=True)
        .agg(n_best=("trialIndex", "count"))
    )

    total_best = (
        fixed_best_df
        .groupby(["subject", "response_version", "offset_abs"], as_index=False, observed=True)
        .agg(n_total=("trialIndex", "count"))
    )

    fixed_best_subject_df = fixed_best_subject_df.merge(
        total_best,
        on=["subject", "response_version", "offset_abs"],
        how="left"
    )

    fixed_best_subject_df["best_model_probability"] = (
        fixed_best_subject_df["n_best"] / fixed_best_subject_df["n_total"]
    )
else:
    fixed_best_subject_df = pd.DataFrame()

fixed_best_subject_csv = os.path.join(KALMAN_CSV_DIR, "kalman_fixed_offset_best_target_model_subject_summary.csv")
fixed_best_subject_df.to_csv(fixed_best_subject_csv, index=False)

# Robust sweep window subject summary.
# This fixes pandas categorical groupby length-mismatch errors.
if len(kalman_sweep_window_df) > 0:
    sweep_tmp = (
        kalman_sweep_window_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["kalman_log_R", "mean_abs_disparity"])
        .copy()
    )

    if "rampType" not in sweep_tmp.columns:
        sweep_tmp["rampType"] = "unknown"

    sweep_tmp["rampType"] = (
        sweep_tmp["rampType"]
        .astype("object")
        .where(pd.notna(sweep_tmp["rampType"]), "unknown")
        .astype(str)
    )

    sweep_tmp["abs_disparity_bin"] = pd.cut(
        sweep_tmp["mean_abs_disparity"],
        bins=[-np.inf, 5, 10, 20, 30, 40, np.inf],
        labels=["0–5", "5–10", "10–20", "20–30", "30–40", "40+"]
    )

    sweep_tmp = sweep_tmp.dropna(subset=["abs_disparity_bin"]).copy()
    sweep_tmp["abs_disparity_bin"] = sweep_tmp["abs_disparity_bin"].astype(str)

    sweep_subject_window_df = (
        sweep_tmp
        .groupby(
            ["subject", "response_version", "target_model", "rampType", "abs_disparity_bin"],
            as_index=False,
            observed=True
        )
        .agg(
            n_windows=("window_center_sec", "count"),
            mean_R=("kalman_R", "mean"),
            mean_sqrt_R=("kalman_sqrt_R", "mean"),
            mean_log_R=("kalman_log_R", "mean"),
            mean_abs_disparity=("mean_abs_disparity", "mean"),
            mean_window_center=("window_center_sec", "mean"),
        )
        .reset_index(drop=True)
    )
else:
    sweep_subject_window_df = pd.DataFrame()

sweep_subject_window_csv = os.path.join(KALMAN_CSV_DIR, "kalman_sweep_subject_window_summary.csv")
sweep_subject_window_df.to_csv(sweep_subject_window_csv, index=False)

print(f"Saved sweep subject-window summary: {sweep_subject_window_csv}")
print(f"Sweep subject-window rows: {len(sweep_subject_window_df)}")

# ============================================================
# STEP 4 — GROUP ESTIMATES
# ============================================================
group_estimate_rows = []

def add_group_estimates(df, group_cols, value_cols, analysis_label):
    rows = []

    if len(df) == 0:
        return rows

    grouped = df.groupby(group_cols, dropna=False, observed=True)

    for keys, sub in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        base = {
            "analysis": analysis_label,
        }

        for col, val in zip(group_cols, keys):
            base[col] = val

        for value_col in value_cols:
            if value_col not in sub.columns:
                continue

            m, lo, hi, n = mean_ci(sub[value_col])

            rows.append({
                **base,
                "outcome": value_col,
                "n_subjects": n,
                "mean": m,
                "ci_low": lo,
                "ci_high": hi,
            })

    return rows

group_estimate_rows.extend(add_group_estimates(
    natural_subject_df,
    group_cols=["response_version", "condition_class"],
    value_cols=["mean_R", "mean_sqrt_R", "mean_log_R", "mean_residual_rmse"],
    analysis_label="natural_conditions"
))

group_estimate_rows.extend(add_group_estimates(
    fixed_subject_model_df,
    group_cols=["response_version", "offset_abs", "target_model"],
    value_cols=["mean_R", "mean_sqrt_R", "mean_log_R", "mean_bic", "mean_residual_rmse"],
    analysis_label="fixed_offset_target_models"
))

if len(fixed_best_subject_df) > 0:
    group_estimate_rows.extend(add_group_estimates(
        fixed_best_subject_df,
        group_cols=["response_version", "offset_abs", "best_target_model"],
        value_cols=["best_model_probability"],
        analysis_label="fixed_offset_best_target_model_probability"
    ))

if len(sweep_subject_window_df) > 0:
    group_estimate_rows.extend(add_group_estimates(
        sweep_subject_window_df,
        group_cols=["response_version", "target_model", "rampType", "abs_disparity_bin"],
        value_cols=["mean_R", "mean_sqrt_R", "mean_log_R", "mean_abs_disparity"],
        analysis_label="sweep_window_disparity_bins"
    ))

group_estimate_df = pd.DataFrame(group_estimate_rows)

group_estimate_csv = os.path.join(KALMAN_CSV_DIR, "kalman_group_estimates.csv")
group_estimate_df.to_csv(group_estimate_csv, index=False)

# ============================================================
# STEP 5 — STATISTICAL TESTS
# ============================================================
test_rows = []
model_coef_tables = []

for response_version, sub in natural_subject_df.groupby("response_version", observed=True):
    piv = sub.pivot_table(
        index="subject",
        columns="condition_class",
        values="mean_log_R",
        aggfunc="mean",
        observed=True
    )

    comparisons = [
        ("Audio-only", "Visual-only"),
        ("Audio-only", "AV congruent"),
        ("Visual-only", "AV congruent"),
    ]

    for a, b in comparisons:
        if a not in piv.columns or b not in piv.columns:
            continue

        d = piv[[a, b]].dropna()

        if len(d) < 2:
            continue

        test = ttest_rel(d[a], d[b])

        test_rows.append({
            "analysis": "paired_natural_condition_log_R",
            "response_version": response_version,
            "comparison": f"{a} vs {b}",
            "outcome": "mean_log_R",
            "n_subjects": len(d),
            "mean_a": float(d[a].mean()),
            "mean_b": float(d[b].mean()),
            "mean_diff_a_minus_b": float((d[a] - d[b]).mean()),
            "t": test.statistic,
            "df": len(d) - 1,
            "p": test.pvalue,
        })

for response_version, sub in natural_subject_df.groupby("response_version", observed=True):
    d = sub.dropna(subset=["mean_log_R"]).copy()

    if len(d) >= 10 and d["subject"].nunique() >= 3:
        d["condition_class"] = pd.Categorical(
            d["condition_class"],
            categories=natural_condition_order,
            ordered=True
        )

        result, coef = fit_mixed_or_clustered_ols(
            formula="mean_log_R ~ C(condition_class)",
            data=d,
            label=f"Natural conditions log_R {response_version}",
            out_dir=KALMAN_MODEL_DIR
        )

        if len(coef) > 0:
            model_coef_tables.append(coef)

for response_version, sub in fixed_subject_model_df.groupby("response_version", observed=True):
    d = sub.dropna(subset=["mean_log_R", "offset_abs", "target_model"]).copy()

    if len(d) >= 20 and d["subject"].nunique() >= 3:
        d["offset_c"] = d["offset_abs"] - d["offset_abs"].mean()

        result, coef = fit_mixed_or_clustered_ols(
            formula="mean_log_R ~ offset_c * C(target_model)",
            data=d,
            label=f"Fixed offset log_R by offset and target model {response_version}",
            out_dir=KALMAN_MODEL_DIR
        )

        if len(coef) > 0:
            model_coef_tables.append(coef)

if len(fixed_best_subject_df) > 0:
    for response_version, sub in fixed_best_subject_df.groupby("response_version", observed=True):
        d = sub.dropna(subset=["best_model_probability", "offset_abs", "best_target_model"]).copy()

        if len(d) >= 20 and d["subject"].nunique() >= 3:
            d["offset_c"] = d["offset_abs"] - d["offset_abs"].mean()

            result, coef = fit_mixed_or_clustered_ols(
                formula="best_model_probability ~ offset_c * C(best_target_model)",
                data=d,
                label=f"Fixed offset best target model probability {response_version}",
                out_dir=KALMAN_MODEL_DIR
            )

            if len(coef) > 0:
                model_coef_tables.append(coef)

if len(sweep_subject_window_df) > 0:
    for response_version, sub in sweep_subject_window_df.groupby("response_version", observed=True):
        d = sub.dropna(subset=["mean_log_R", "mean_abs_disparity", "target_model"]).copy()

        if len(d) >= 20 and d["subject"].nunique() >= 3:
            d["abs_disparity_c"] = d["mean_abs_disparity"] - d["mean_abs_disparity"].mean()

            if "rampType" in d.columns and d["rampType"].nunique(dropna=True) > 1:
                formula = "mean_log_R ~ abs_disparity_c * C(target_model) + C(rampType)"
            else:
                formula = "mean_log_R ~ abs_disparity_c * C(target_model)"

            result, coef = fit_mixed_or_clustered_ols(
                formula=formula,
                data=d,
                label=f"Sweep window log_R by disparity and target model {response_version}",
                out_dir=KALMAN_MODEL_DIR
            )

            if len(coef) > 0:
                model_coef_tables.append(coef)

kalman_test_df = pd.DataFrame(test_rows)

kalman_test_csv = os.path.join(KALMAN_CSV_DIR, "kalman_group_tests.csv")
kalman_test_df.to_csv(kalman_test_csv, index=False)

if len(model_coef_tables) > 0:
    kalman_model_coef_df = pd.concat(model_coef_tables, ignore_index=True)
else:
    kalman_model_coef_df = pd.DataFrame()

kalman_model_coef_csv = os.path.join(KALMAN_CSV_DIR, "kalman_model_coefficients_all.csv")
kalman_model_coef_df.to_csv(kalman_model_coef_csv, index=False)

# ============================================================
# STEP 6 — PLOTS
# ============================================================
if len(natural_subject_df) > 0:
    for response_version, sub in natural_subject_df.groupby("response_version", observed=True):
        summary = (
            sub
            .groupby("condition_class", observed=True, as_index=False)
            .agg(
                mean=("mean_sqrt_R", "mean"),
                se=("mean_sqrt_R", sem),
                n=("subject", "nunique")
            )
        )

        summary["condition_class"] = pd.Categorical(
            summary["condition_class"],
            categories=natural_condition_order,
            ordered=True
        )
        summary = summary.sort_values("condition_class")

        fig, ax = plt.subplots(figsize=(7.6, 5.2))

        ax.errorbar(
            np.arange(len(summary)),
            summary["mean"],
            yerr=summary["se"],
            marker="o",
            linewidth=2.2,
            capsize=4
        )

        ax.set_xticks(np.arange(len(summary)))
        ax.set_xticklabels(summary["condition_class"], rotation=20, ha="right")
        ax.set_ylabel("Kalman-derived spatial uncertainty, sqrt(R)")
        ax.set_title(f"Kalman uncertainty across natural conditions: {response_version}")

        fig.tight_layout()
        fig_path = os.path.join(KALMAN_FIG_DIR, f"natural_condition_sqrt_R_{response_version}.png")
        fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()

if len(fixed_subject_model_df) > 0:
    for response_version, sub in fixed_subject_model_df.groupby("response_version", observed=True):
        summary = (
            sub
            .groupby(["offset_abs", "target_model"], as_index=False, observed=True)
            .agg(
                mean=("mean_sqrt_R", "mean"),
                se=("mean_sqrt_R", sem),
                n=("subject", "nunique")
            )
            .sort_values(["target_model", "offset_abs"])
        )

        fig, ax = plt.subplots(figsize=(8.8, 5.6))

        for target_model in summary["target_model"].dropna().unique():
            s = summary[summary["target_model"] == target_model].copy()

            ax.errorbar(
                s["offset_abs"],
                s["mean"],
                yerr=s["se"],
                marker="o",
                linewidth=2.2,
                capsize=4,
                label=target_model
            )

        ax.set_xlabel("A-V offset magnitude (deg)")
        ax.set_ylabel("Kalman-derived spatial uncertainty, sqrt(R)")
        ax.set_title(f"Fixed-offset Kalman uncertainty by target model: {response_version}")
        ax.legend(frameon=False)

        fig.tight_layout()
        fig_path = os.path.join(KALMAN_FIG_DIR, f"fixed_offset_target_model_sqrt_R_{response_version}.png")
        fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()

if len(fixed_best_subject_df) > 0:
    for response_version, sub in fixed_best_subject_df.groupby("response_version", observed=True):
        summary = (
            sub
            .groupby(["offset_abs", "best_target_model"], as_index=False, observed=True)
            .agg(
                mean=("best_model_probability", "mean"),
                se=("best_model_probability", sem),
                n=("subject", "nunique")
            )
            .sort_values(["best_target_model", "offset_abs"])
        )

        fig, ax = plt.subplots(figsize=(8.8, 5.6))

        for target_model in summary["best_target_model"].dropna().unique():
            s = summary[summary["best_target_model"] == target_model].copy()

            ax.errorbar(
                s["offset_abs"],
                s["mean"],
                yerr=s["se"],
                marker="o",
                linewidth=2.2,
                capsize=4,
                label=target_model
            )

        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("A-V offset magnitude (deg)")
        ax.set_ylabel("Probability of best-fitting target model")
        ax.set_title(f"Best-fitting Kalman target model by offset: {response_version}")
        ax.legend(frameon=False)

        fig.tight_layout()
        fig_path = os.path.join(KALMAN_FIG_DIR, f"fixed_offset_best_target_model_probability_{response_version}.png")
        fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()

if len(sweep_subject_window_df) > 0:
    for response_version, sub in sweep_subject_window_df.groupby("response_version", observed=True):
        d = sub.dropna(subset=["mean_abs_disparity", "mean_sqrt_R"]).copy()

        if len(d) == 0:
            continue

        summary = (
            d
            .groupby(["target_model", "abs_disparity_bin"], as_index=False, observed=True)
            .agg(
                mean=("mean_sqrt_R", "mean"),
                se=("mean_sqrt_R", sem),
                n=("subject", "nunique"),
                mean_abs_disparity=("mean_abs_disparity", "mean")
            )
            .dropna(subset=["mean_abs_disparity"])
            .sort_values(["target_model", "mean_abs_disparity"])
        )

        fig, ax = plt.subplots(figsize=(9.0, 5.6))

        for target_model in summary["target_model"].dropna().unique():
            s = summary[summary["target_model"] == target_model].copy()

            ax.errorbar(
                s["mean_abs_disparity"],
                s["mean"],
                yerr=s["se"],
                marker="o",
                linewidth=2.2,
                capsize=4,
                label=target_model
            )

        ax.set_xlabel("Mean absolute A-V disparity in window (deg)")
        ax.set_ylabel("Kalman-derived spatial uncertainty, sqrt(R)")
        ax.set_title(f"Sweep-window uncertainty by disparity: {response_version}")
        ax.legend(frameon=False)

        fig.tight_layout()
        fig_path = os.path.join(KALMAN_FIG_DIR, f"sweep_window_sqrt_R_by_disparity_{response_version}.png")
        fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()

# ============================================================
# FINAL PRINT SUMMARY
# ============================================================
print("\n============================================================")
print("KALMAN FILTER PRECISION ANALYSIS COMPLETE")
print("============================================================")
print(f"Output folder:                                      {KALMAN_OUT_DIR}")
print(f"Trial-level full-trial R estimates:                 {kalman_trial_csv}")
print(f"Sweep-window R estimates:                           {kalman_sweep_window_csv}")
print(f"Subject natural-condition summary:                  {natural_subject_csv}")
print(f"Subject fixed-offset target-model summary:          {fixed_subject_model_csv}")
print(f"Fixed-offset best target model, trial level:        {fixed_best_csv}")
print(f"Fixed-offset best target model, subject summary:    {fixed_best_subject_csv}")
print(f"Sweep subject window summary:                       {sweep_subject_window_csv}")
print(f"Group estimates:                                    {group_estimate_csv}")
print(f"Paired tests:                                       {kalman_test_csv}")
print(f"Model coefficients:                                 {kalman_model_coef_csv}")
print(f"Figures folder:                                     {KALMAN_FIG_DIR}")
print(f"Model summaries folder:                             {KALMAN_MODEL_DIR}")

print("\n============================================================")
print("NATURAL CONDITION SUMMARY")
print("============================================================")
if len(natural_subject_df) > 0:
    print(
        natural_subject_df
        .groupby(["response_version", "condition_class"], observed=True)
        .agg(
            mean_sqrt_R=("mean_sqrt_R", "mean"),
            se_sqrt_R=("mean_sqrt_R", sem),
            mean_log_R=("mean_log_R", "mean"),
            n_subjects=("subject", "nunique")
        )
        .to_string()
    )
else:
    print("No natural-condition Kalman rows found.")

print("\n============================================================")
print("FIXED-OFFSET TARGET MODEL SUMMARY")
print("============================================================")
if len(fixed_subject_model_df) > 0:
    print(
        fixed_subject_model_df
        .groupby(["response_version", "offset_abs", "target_model"], observed=True)
        .agg(
            mean_sqrt_R=("mean_sqrt_R", "mean"),
            se_sqrt_R=("mean_sqrt_R", sem),
            mean_log_R=("mean_log_R", "mean"),
            n_subjects=("subject", "nunique")
        )
        .to_string()
    )
else:
    print("No fixed-offset Kalman rows found.")

print("\n============================================================")
print("BEST TARGET MODEL PROBABILITY SUMMARY")
print("============================================================")
if len(fixed_best_subject_df) > 0:
    print(
        fixed_best_subject_df
        .groupby(["response_version", "offset_abs", "best_target_model"], observed=True)
        .agg(
            mean_probability=("best_model_probability", "mean"),
            se_probability=("best_model_probability", sem),
            n_subjects=("subject", "nunique")
        )
        .to_string()
    )
else:
    print("No fixed-offset best-target-model rows found.")

print("\n============================================================")
print("PAIRED NATURAL-CONDITION TESTS")
print("============================================================")
if len(kalman_test_df) > 0:
    print(kalman_test_df.to_string(index=False))
else:
    print("No paired tests were generated.")

print("\n============================================================")
print("INTERPRETATION GUIDE")
print("============================================================")
print("- kalman_R = estimated observation/tracking noise around a candidate target trajectory.")
print("- kalman_sqrt_R = spatial uncertainty in the same units as the response trajectory.")
print("- kalman_log_R = log-transformed uncertainty used for statistical models.")
print("- Lower R / sqrt(R) means higher inferred tracking precision.")
print("- For audio-only, visual-only, and AV congruent trials, the target model is the natural stimulus trajectory.")
print("- For fixed-offset AV trials, audio, visual, midpoint, and empirical integrated targets are compared.")
print("- empirical_integrated_target = wV * visual + (1 - wV) * audio, where wV is estimated from subject-level visual capture.")
print("- fixed_offset_best_target_model_trial_level.csv tells which candidate target best explained each fixed-offset trial.")
print("- sweep-window outputs estimate how uncertainty changes as dynamic A-V disparity changes over time.")
print("- baseline_lagcorr uses subject-level motor lag estimated only from audio-only, visual-only, and congruent trials.")
print("- Treat this analysis as a supporting model-based precision analysis, not as a replacement for visual capture, hysteresis, or dynamic BCI analyses.")

## 5. Tonelli-Style Cross-Correlation and Lag Metrics

Per-trial auditory/visual peak correlation and lag estimation, followed by lag-corrected spatial accuracy and visual-capture metrics.

In [8]:
# %% CELL 2 — TONELLI-STYLE CROSS-CORRELATION METRICS PER SUBJECT/TRIAL
# ============================================================
# Computes PeakA/LagA and PeakV/LagV for every trial
# ============================================================

xcorr_rows = []

for idx, row in trial_df.iterrows():
    mouse_x = row["mouse_x"]
    audio_x = row["audio_x"]
    blob_x = row["blob_x"]

    audio_res = lag_scanned_corr(
        response=mouse_x,
        stimulus=audio_x,
        fs=RESAMPLE_HZ,
        max_lag_sec=MAX_LAG_SEC
    )

    visual_res = lag_scanned_corr(
        response=mouse_x,
        stimulus=blob_x,
        fs=RESAMPLE_HZ,
        max_lag_sec=MAX_LAG_SEC
    )

    xcorr_rows.append({
        "trial_df_index": idx,
        "subject": row["subject"],
        "source_file": row["source_file"],
        "trialIndex": row["trialIndex"],
        "trial_order_within_subject": row["trial_order_within_subject"],
        "trialType": row["trialType"],
        "condLabel": row["condLabel"],
        "offsetDeg": row["offsetDeg"],
        "sideSign": row["sideSign"],
        "rampType": row["rampType"],

        "PeakA": audio_res["peak_corr"],
        "LagA_sec": audio_res["lag_sec"],
        "LagA_samples": audio_res["lag_samples"],

        "PeakV": visual_res["peak_corr"],
        "LagV_sec": visual_res["lag_sec"],
        "LagV_samples": visual_res["lag_samples"],

        "Capture_peak": visual_res["peak_corr"] - audio_res["peak_corr"]
        if np.isfinite(visual_res["peak_corr"]) and np.isfinite(audio_res["peak_corr"])
        else np.nan,
    })

xcorr_df = pd.DataFrame(xcorr_rows)

# Merge back into trial_df
for col in [
    "PeakA", "LagA_sec", "LagA_samples",
    "PeakV", "LagV_sec", "LagV_samples",
    "Capture_peak"
]:
    trial_df[col] = xcorr_df.set_index("trial_df_index").loc[trial_df.index, col].values

xcorr_csv = os.path.join(trial_metric_dir, "cell2_trial_level_xcorr_peak_lag_metrics.csv")
xcorr_df.to_csv(xcorr_csv, index=False)

print(f"Saved cross-correlation metrics -> {xcorr_csv}")
print(xcorr_df.head())

Saved cross-correlation metrics -> C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\trial_level_metrics\cell2_trial_level_xcorr_peak_lag_metrics.csv
   trial_df_index subject                        source_file  trialIndex  \
0               0  pid001  pid001_001_master_trials_LONG.csv           2   
1               1  pid001  pid001_001_master_trials_LONG.csv           3   
2               2  pid001  pid001_001_master_trials_LONG.csv           4   
3               3  pid001  pid001_001_master_trials_LONG.csv           5   
4               4  pid001  pid001_001_master_trials_LONG.csv           6   

   trial_order_within_subject            trialType             condLabel  \
0                           1    incongruent_sweep  sweep_converge_right   
1                           2          visual_only           visual_only   
2                           3  incongruent_spatial      spatial_20_right   
3                       

In [9]:
# %% CELL 3 — LAG-CORRECTED SPATIAL ACCURACY + VISUAL CAPTURE METRICS
# ============================================================
# Computes raw and lag-corrected RMSE/deviation/capture per subject/trial
# ============================================================

metric_rows = []

for idx, row in trial_df.iterrows():
    t = row["t_grid"]
    mouse_x = np.asarray(row["mouse_x"], dtype=float)
    audio_x = np.asarray(row["audio_x"], dtype=float)
    blob_x = np.asarray(row["blob_x"], dtype=float)
    midpoint_x = np.asarray(row["midpoint_x"], dtype=float)

    # Lag-corrected responses
    mouse_lagcorr_audio = lag_correct_response(mouse_x, t, row["LagA_sec"])
    mouse_lagcorr_visual = lag_correct_response(mouse_x, t, row["LagV_sec"])

    # Raw metrics
    raw_rmse_audio = rmse_1d(mouse_x, audio_x)
    raw_rmse_visual = rmse_1d(mouse_x, blob_x)
    raw_rmse_midpoint = rmse_1d(mouse_x, midpoint_x)

    raw_dev_audio = mean_abs_deviation_1d(mouse_x, audio_x)
    raw_dev_visual = mean_abs_deviation_1d(mouse_x, blob_x)
    raw_dev_midpoint = mean_abs_deviation_1d(mouse_x, midpoint_x)

    raw_capture_percent = visual_capture_percent(
        response=mouse_x,
        visual=blob_x,
        audio=audio_x,
        min_abs_disparity=MIN_ABS_DISPARITY_FOR_CAPTURE
    )

    # Lag-corrected metrics
    lag_rmse_audio = rmse_1d(mouse_lagcorr_audio, audio_x)
    lag_rmse_visual = rmse_1d(mouse_lagcorr_visual, blob_x)

    # Midpoint has no single natural lag; use audio lag and visual lag versions
    lag_rmse_midpoint_audio_lag = rmse_1d(mouse_lagcorr_audio, midpoint_x)
    lag_rmse_midpoint_visual_lag = rmse_1d(mouse_lagcorr_visual, midpoint_x)

    lag_dev_audio = mean_abs_deviation_1d(mouse_lagcorr_audio, audio_x)
    lag_dev_visual = mean_abs_deviation_1d(mouse_lagcorr_visual, blob_x)

    lag_dev_midpoint_audio_lag = mean_abs_deviation_1d(mouse_lagcorr_audio, midpoint_x)
    lag_dev_midpoint_visual_lag = mean_abs_deviation_1d(mouse_lagcorr_visual, midpoint_x)

    lag_capture_percent_audio_lag = visual_capture_percent(
        response=mouse_lagcorr_audio,
        visual=blob_x,
        audio=audio_x,
        min_abs_disparity=MIN_ABS_DISPARITY_FOR_CAPTURE
    )

    lag_capture_percent_visual_lag = visual_capture_percent(
        response=mouse_lagcorr_visual,
        visual=blob_x,
        audio=audio_x,
        min_abs_disparity=MIN_ABS_DISPARITY_FOR_CAPTURE
    )

    metric_rows.append({
        "trial_df_index": idx,
        "subject": row["subject"],
        "source_file": row["source_file"],
        "trialIndex": row["trialIndex"],
        "trial_order_within_subject": row["trial_order_within_subject"],
        "trialType": row["trialType"],
        "condLabel": row["condLabel"],
        "offsetDeg": row["offsetDeg"],
        "sideSign": row["sideSign"],
        "rampType": row["rampType"],

        "PeakA": row["PeakA"],
        "LagA_sec": row["LagA_sec"],
        "PeakV": row["PeakV"],
        "LagV_sec": row["LagV_sec"],
        "Capture_peak": row["Capture_peak"],

        "RMSE_A_raw": raw_rmse_audio,
        "RMSE_V_raw": raw_rmse_visual,
        "RMSE_Mid_raw": raw_rmse_midpoint,

        "Dev_A_raw": raw_dev_audio,
        "Dev_V_raw": raw_dev_visual,
        "Dev_Mid_raw": raw_dev_midpoint,

        "VisualCapture_raw": raw_capture_percent,

        "RMSE_A_lagcorr": lag_rmse_audio,
        "RMSE_V_lagcorr": lag_rmse_visual,
        "RMSE_Mid_audioLag": lag_rmse_midpoint_audio_lag,
        "RMSE_Mid_visualLag": lag_rmse_midpoint_visual_lag,

        "Dev_A_lagcorr": lag_dev_audio,
        "Dev_V_lagcorr": lag_dev_visual,
        "Dev_Mid_audioLag": lag_dev_midpoint_audio_lag,
        "Dev_Mid_visualLag": lag_dev_midpoint_visual_lag,

        "VisualCapture_audioLag": lag_capture_percent_audio_lag,
        "VisualCapture_visualLag": lag_capture_percent_visual_lag,

        "Capture_rmse_lagcorr": lag_rmse_audio - lag_rmse_visual
        if np.isfinite(lag_rmse_audio) and np.isfinite(lag_rmse_visual)
        else np.nan,

        "Capture_dev_lagcorr": lag_dev_audio - lag_dev_visual
        if np.isfinite(lag_dev_audio) and np.isfinite(lag_dev_visual)
        else np.nan,

        "response_variance_trial": np.nanvar(mouse_x),
    })

trial_metrics_df = pd.DataFrame(metric_rows)

# Merge selected metrics back into trial_df
metrics_to_merge = [
    "RMSE_A_raw", "RMSE_V_raw", "RMSE_Mid_raw",
    "Dev_A_raw", "Dev_V_raw", "Dev_Mid_raw",
    "VisualCapture_raw",
    "RMSE_A_lagcorr", "RMSE_V_lagcorr",
    "RMSE_Mid_audioLag", "RMSE_Mid_visualLag",
    "Dev_A_lagcorr", "Dev_V_lagcorr",
    "Dev_Mid_audioLag", "Dev_Mid_visualLag",
    "VisualCapture_audioLag", "VisualCapture_visualLag",
    "Capture_rmse_lagcorr", "Capture_dev_lagcorr",
    "response_variance_trial",
]

for col in metrics_to_merge:
    trial_df[col] = trial_metrics_df.set_index("trial_df_index").loc[trial_df.index, col].values

trial_metrics_csv = os.path.join(trial_metric_dir, "cell3_trial_level_lag_corrected_spatial_metrics.csv")
trial_metrics_df.to_csv(trial_metrics_csv, index=False)

print(f"Saved lag-corrected spatial metrics -> {trial_metrics_csv}")
print(trial_metrics_df.head())

Saved lag-corrected spatial metrics -> C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\trial_level_metrics\cell3_trial_level_lag_corrected_spatial_metrics.csv
   trial_df_index subject                        source_file  trialIndex  \
0               0  pid001  pid001_001_master_trials_LONG.csv           2   
1               1  pid001  pid001_001_master_trials_LONG.csv           3   
2               2  pid001  pid001_001_master_trials_LONG.csv           4   
3               3  pid001  pid001_001_master_trials_LONG.csv           5   
4               4  pid001  pid001_001_master_trials_LONG.csv           6   

   trial_order_within_subject            trialType             condLabel  \
0                           1    incongruent_sweep  sweep_converge_right   
1                           2          visual_only           visual_only   
2                           3  incongruent_spatial      spatial_20_right   
3            

## 6. Whole-Trial Mixed Models and Permutation Inference

Linear mixed-effects tests and condition-label shuffle controls for whole-trial localization/capture effects.

In [10]:
# %% CELL 3 — LINEAR MIXED MODELS: SAFE VERSION FOR PARTIALLY CROSSED DESIGN
# ============================================================
# Fixes singular matrix issue by avoiding impossible interaction cells.
#
# Runs:
#   Model A: all valid cells, additive only
#       y ~ C(condition_detailed) + C(modality) + (1|subject)
#
#   Model B: AV-only fully crossed interaction
#       y ~ C(condition_detailed) * C(modality) + (1|subject)
#       excluding audio_only and visual_only
#
#   Model C: separate auditory and visual models
#       y ~ C(condition_detailed) + (1|subject)
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# -------------------------
# LOAD TRIAL METRICS IF NEEDED
# -------------------------
if "trial_metrics_df" not in globals():
    try:
        trial_metrics_csv = os.path.join(
            trial_metric_dir,
            "cell3_trial_level_lag_corrected_spatial_metrics.csv"
        )
    except NameError:
        trial_metrics_csv = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_behavior_tonelli_lag_corrected_subject_first\trial_level_metrics\cell3_trial_level_lag_corrected_spatial_metrics.csv"

    trial_metrics_df = pd.read_csv(trial_metrics_csv)
    print(f"Loaded trial_metrics_df from: {trial_metrics_csv}")
else:
    print("Using existing trial_metrics_df from memory.")

try:
    lmm_out_dir = os.path.join(out_root, "safe_valid_cell_linear_mixed_models")
except NameError:
    lmm_out_dir = os.path.join(os.path.dirname(trial_metrics_csv), "safe_valid_cell_linear_mixed_models")

os.makedirs(lmm_out_dir, exist_ok=True)

# -------------------------
# CONDITION MAPPING
# -------------------------
def extract_offset(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def map_condition_detailed(row):
    trial_type = str(row["trialType"]).lower()
    cond = str(row.get("condLabel", "")).lower()

    if trial_type == "audio_only":
        return "audio_only"
    if trial_type == "visual_only":
        return "visual_only"
    if trial_type == "congruent":
        return "congruent"

    if trial_type == "incongruent_spatial":
        off = extract_offset(row)
        if np.isfinite(off):
            return f"offset_{int(off)}"
        return "offset_unknown"

    if trial_type == "incongruent_sweep":
        if "diverge" in cond:
            return "sweep_diverge"
        if "converge" in cond:
            return "sweep_converge"
        return "sweep_unknown"

    return np.nan

df = trial_metrics_df.copy()
df["condition_detailed"] = df.apply(map_condition_detailed, axis=1)

condition_order = [
    "audio_only",
    "visual_only",
    "congruent",
    "offset_10",
    "offset_20",
    "offset_30",
    "offset_40",
    "sweep_diverge",
    "sweep_converge",
]

av_condition_order = [
    "congruent",
    "offset_10",
    "offset_20",
    "offset_30",
    "offset_40",
    "sweep_diverge",
    "sweep_converge",
]

modality_order = ["audio", "visual"]

# -------------------------
# BUILD VALID-CELL LONG DATA
# -------------------------
long_rows = []

for _, row in df.iterrows():
    condition = row["condition_detailed"]

    if pd.isna(condition):
        continue

    trial_type = str(row["trialType"]).lower()

    if trial_type == "audio_only":
        valid_modalities = ["audio"]
    elif trial_type == "visual_only":
        valid_modalities = ["visual"]
    elif trial_type in ["congruent", "incongruent_spatial", "incongruent_sweep"]:
        valid_modalities = ["audio", "visual"]
    else:
        valid_modalities = []

    for modality in valid_modalities:
        if modality == "audio":
            long_rows.append({
                "subject": row["subject"],
                "trialIndex": row["trialIndex"],
                "trialType": row["trialType"],
                "condLabel": row.get("condLabel", np.nan),
                "condition_detailed": condition,
                "modality": "audio",
                "peak": row.get("PeakA", np.nan),
                "lag": row.get("LagA_sec", np.nan),
                "rmse_raw": row.get("RMSE_A_raw", np.nan),
                "rmse_lagcorr": row.get("RMSE_A_lagcorr", np.nan),
            })

        elif modality == "visual":
            long_rows.append({
                "subject": row["subject"],
                "trialIndex": row["trialIndex"],
                "trialType": row["trialType"],
                "condLabel": row.get("condLabel", np.nan),
                "condition_detailed": condition,
                "modality": "visual",
                "peak": row.get("PeakV", np.nan),
                "lag": row.get("LagV_sec", np.nan),
                "rmse_raw": row.get("RMSE_V_raw", np.nan),
                "rmse_lagcorr": row.get("RMSE_V_lagcorr", np.nan),
            })

long_df = pd.DataFrame(long_rows)

long_df["condition_detailed"] = pd.Categorical(
    long_df["condition_detailed"],
    categories=condition_order,
    ordered=True
)

long_df["modality"] = pd.Categorical(
    long_df["modality"],
    categories=modality_order,
    ordered=True
)

long_csv = os.path.join(lmm_out_dir, "valid_cell_trial_level_long_metrics_safe.csv")
long_df.to_csv(long_csv, index=False)

print(f"Saved valid-cell long dataframe: {long_csv}")
print("\nCell counts:")
print(long_df.groupby(["condition_detailed", "modality"], observed=False).size())

# -------------------------
# MODEL FITTING HELPERS
# -------------------------
def fit_mixed_model(use, formula, metric, model_name, out_dir):
    print("\n" + "=" * 100)
    print(f"{model_name}: {metric}")
    print("=" * 100)
    print(f"Formula: {formula}")
    print(f"Rows: {len(use)}")
    print(f"Subjects: {use['subject'].nunique()}")

    if len(use) == 0 or use["subject"].nunique() < 2:
        print("Skipping: not enough data.")
        return {
            "metric": metric,
            "model_name": model_name,
            "converged": False,
            "error": "not enough data"
        }

    try:
        model = smf.mixedlm(
            formula,
            data=use,
            groups=use["subject"],
            re_formula="1"
        )

        # Powell is slower but often more stable for singular/boundary problems.
        result = model.fit(method="powell", reml=False, maxiter=1000, disp=False)

        print(result.summary())

        safe_model_name = model_name.replace(" ", "_").replace("/", "_")
        summary_text_path = os.path.join(out_dir, f"{safe_model_name}_{metric}_summary.txt")
        with open(summary_text_path, "w") as f:
            f.write(str(result.summary()))

        coef_df = pd.DataFrame({
            "term": result.params.index,
            "estimate": result.params.values,
            "se": result.bse.reindex(result.params.index).values,
            "z": result.tvalues.reindex(result.params.index).values,
            "p": result.pvalues.reindex(result.params.index).values,
        })

        coef_csv = os.path.join(out_dir, f"{safe_model_name}_{metric}_coefficients.csv")
        coef_df.to_csv(coef_csv, index=False)

        return {
            "metric": metric,
            "model_name": model_name,
            "n_rows": len(use),
            "n_subjects": use["subject"].nunique(),
            "converged": result.converged,
            "llf": result.llf,
            "aic": result.aic,
            "bic": result.bic,
            "summary_txt": summary_text_path,
            "coef_csv": coef_csv,
        }

    except Exception as e:
        print(f"Model failed: {e}")
        return {
            "metric": metric,
            "model_name": model_name,
            "n_rows": len(use),
            "n_subjects": use["subject"].nunique(),
            "converged": False,
            "error": str(e),
        }

# -------------------------
# FIT MODELS
# -------------------------
metric_list = ["peak", "lag", "rmse_raw", "rmse_lagcorr"]
model_summaries = []

for metric in metric_list:

    # ========================================================
    # MODEL A: ALL VALID CELLS, ADDITIVE ONLY
    # ========================================================
    use_all = long_df[["subject", "condition_detailed", "modality", metric]].dropna().copy()
    use_all = use_all.rename(columns={metric: "y"})

    use_all["condition_detailed"] = pd.Categorical(
        use_all["condition_detailed"],
        categories=condition_order,
        ordered=True
    ).remove_unused_categories()

    use_all["modality"] = pd.Categorical(
        use_all["modality"],
        categories=modality_order,
        ordered=True
    ).remove_unused_categories()

    formula_additive = "y ~ C(condition_detailed) + C(modality)"

    model_summaries.append(
        fit_mixed_model(
            use=use_all,
            formula=formula_additive,
            metric=metric,
            model_name="all_valid_cells_additive_condition_plus_modality",
            out_dir=lmm_out_dir
        )
    )

    # ========================================================
    # MODEL B: AV-ONLY INTERACTION MODEL
    # ========================================================
    use_av = long_df[
        long_df["condition_detailed"].isin(av_condition_order)
    ][["subject", "condition_detailed", "modality", metric]].dropna().copy()

    use_av = use_av.rename(columns={metric: "y"})

    use_av["condition_detailed"] = pd.Categorical(
        use_av["condition_detailed"],
        categories=av_condition_order,
        ordered=True
    ).remove_unused_categories()

    use_av["modality"] = pd.Categorical(
        use_av["modality"],
        categories=modality_order,
        ordered=True
    ).remove_unused_categories()

    formula_interaction = "y ~ C(condition_detailed) * C(modality)"

    model_summaries.append(
        fit_mixed_model(
            use=use_av,
            formula=formula_interaction,
            metric=metric,
            model_name="av_only_condition_by_modality_interaction",
            out_dir=lmm_out_dir
        )
    )

    # ========================================================
    # MODEL C1: AUDIO-ONLY ROWS ACROSS AUDIO CONDITIONS
    # ========================================================
    use_audio = long_df[
        long_df["modality"] == "audio"
    ][["subject", "condition_detailed", metric]].dropna().copy()

    use_audio = use_audio.rename(columns={metric: "y"})

    audio_condition_order = [
        "audio_only",
        "congruent",
        "offset_10",
        "offset_20",
        "offset_30",
        "offset_40",
        "sweep_diverge",
        "sweep_converge",
    ]

    use_audio["condition_detailed"] = pd.Categorical(
        use_audio["condition_detailed"],
        categories=audio_condition_order,
        ordered=True
    ).remove_unused_categories()

    formula_audio = "y ~ C(condition_detailed)"

    model_summaries.append(
        fit_mixed_model(
            use=use_audio,
            formula=formula_audio,
            metric=metric,
            model_name="audio_rows_condition_only",
            out_dir=lmm_out_dir
        )
    )

    # ========================================================
    # MODEL C2: VISUAL-ONLY ROWS ACROSS VISUAL CONDITIONS
    # ========================================================
    use_visual = long_df[
        long_df["modality"] == "visual"
    ][["subject", "condition_detailed", metric]].dropna().copy()

    use_visual = use_visual.rename(columns={metric: "y"})

    visual_condition_order = [
        "visual_only",
        "congruent",
        "offset_10",
        "offset_20",
        "offset_30",
        "offset_40",
        "sweep_diverge",
        "sweep_converge",
    ]

    use_visual["condition_detailed"] = pd.Categorical(
        use_visual["condition_detailed"],
        categories=visual_condition_order,
        ordered=True
    ).remove_unused_categories()

    formula_visual = "y ~ C(condition_detailed)"

    model_summaries.append(
        fit_mixed_model(
            use=use_visual,
            formula=formula_visual,
            metric=metric,
            model_name="visual_rows_condition_only",
            out_dir=lmm_out_dir
        )
    )

# -------------------------
# SAVE SUMMARY
# -------------------------
model_summary_df = pd.DataFrame(model_summaries)
model_summary_csv = os.path.join(lmm_out_dir, "safe_mixedlm_model_summary_index.csv")
model_summary_df.to_csv(model_summary_csv, index=False)

mapping_csv = os.path.join(lmm_out_dir, "condition_mapping_check.csv")
df[["subject", "trialIndex", "trialType", "condLabel", "offsetDeg", "condition_detailed"]].to_csv(mapping_csv, index=False)

print("\nSaved safe mixed-model outputs to:")
print(lmm_out_dir)
print(f"Model summary index: {model_summary_csv}")
print(f"Condition mapping check: {mapping_csv}")

Using existing trial_metrics_df from memory.
Saved valid-cell long dataframe: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\safe_valid_cell_linear_mixed_models\valid_cell_trial_level_long_metrics_safe.csv

Cell counts:
condition_detailed  modality
audio_only          audio        500
                    visual         0
visual_only         audio          0
                    visual       500
congruent           audio        500
                    visual       500
offset_10           audio       1000
                    visual      1000
offset_20           audio       1000
                    visual      1000
offset_30           audio       1000
                    visual      1000
offset_40           audio       1000
                    visual      1000
sweep_diverge       audio        500
                    visual       500
sweep_converge      audio        500
                    visual       500
dtype: int64

all

C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                       Mixed Linear Model Regression Results
Model:                      MixedLM          Dependent Variable:          y        
No. Observations:           11864            Method:                      ML       
No. Groups:                 50               Scale:                       0.0664   
Min. group size:            146              Log-Likelihood:              -819.6571
Max. group size:            240              Converged:                   Yes      
Mean group size:            237.3                                                  
-----------------------------------------------------------------------------------
                                        Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------------------
Intercept                                0.799    0.016  50.841 0.000  0.768  0.829
C(condition_detailed)[T.visual_only]     0.230    0.017  13.492 0.000  0.197  0.264
C(condition_det

C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                                  Mixed Linear Model Regression Results
Model:                              MixedLM                 Dependent Variable:                 y        
No. Observations:                   10868                   Method:                             ML       
No. Groups:                         50                      Scale:                              0.0674   
Min. group size:                    126                     Log-Likelihood:                     -839.7539
Max. group size:                    220                     Converged:                          Yes      
Mean group size:                    217.4                                                                
---------------------------------------------------------------------------------------------------------
                                                              Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------------

C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                       Mixed Linear Model Regression Results
Model:                      MixedLM           Dependent Variable:           y      
No. Observations:           5932              Method:                       ML     
No. Groups:                 50                Scale:                        0.0555 
Min. group size:            73                Log-Likelihood:               99.4556
Max. group size:            120               Converged:                    Yes    
Mean group size:            118.6                                                  
-----------------------------------------------------------------------------------
                                        Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------------------
Intercept                                0.990    0.014  68.530 0.000  0.962  1.018
C(condition_detailed)[T.congruent]      -0.104    0.015  -6.923 0.000 -0.133 -0.074
C(condition_det

In [ ]:
# %% ADD-ON — WHOLE-TRIAL CONDITION-LABEL SHUFFLE PERMUTATION TESTS
# ============================================================
# Purpose:
#   Permutation inference for the LMM validation claims that
#   auditory-relative lag-corrected RMSE increases relative to congruent AV
#   across fixed-offset and sweep audiovisual conditions.
#
# Logic:
#   Within each participant, shuffle whole-trial condition labels across AV trials
#   while preserving the number of trials per condition.
#
#   For each permutation:
#       1. Recompute subject-level condition means.
#       2. Recompute planned contrasts:
#              condition RMSE_A_lagcorr - congruent RMSE_A_lagcorr
#       3. Compute group-level t statistics.
#       4. Store the maximum t statistic across planned contrasts for max-T correction.
#
# Main contrasts:
#   offset_10       - congruent
#   offset_20       - congruent
#   offset_30       - congruent
#   offset_40       - congruent
#   sweep_diverge   - congruent
#   sweep_converge  - congruent
#
# Directional hypothesis:
#   condition RMSE_A_lagcorr > congruent RMSE_A_lagcorr
#
# Output:
#   All CSVs and figures are saved directly into lmm_out_dir.
#   No nested subfolders are created.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t as t_dist
import matplotlib.pyplot as plt

# -------------------------
# SETTINGS
# -------------------------
N_PERM_CONDITION_LABEL = 10000
RANDOM_SEED_CONDITION_LABEL = 12345

PRIMARY_METRIC_COL = "RMSE_A_lagcorr"
PRIMARY_METRIC_NAME = "lagcorr_rmse_audio_relative"

REFERENCE_CONDITION = "congruent"

PLANNED_CONDITION_CONTRASTS = [
    "offset_10",
    "offset_20",
    "offset_30",
    "offset_40",
    "sweep_diverge",
    "sweep_converge",
]

FIXED_OFFSET_CONDITION_TO_DEG = {
    "congruent": 0,
    "offset_10": 10,
    "offset_20": 20,
    "offset_30": 30,
    "offset_40": 40,
}

ALTERNATIVE = "greater"
FIG_DPI = 180
SAVE_NULL_LONG = True

# -------------------------
# OUTPUT DIRECTORY
# -------------------------
# IMPORTANT:
# This version saves directly into lmm_out_dir and does not create nested folders.
if "lmm_out_dir" in globals():
    perm_lmm_out_dir = lmm_out_dir
else:
    try:
        perm_lmm_out_dir = os.path.join(out_root, "safe_valid_cell_linear_mixed_models")
    except NameError:
        try:
            perm_lmm_out_dir = os.path.dirname(trial_metrics_csv)
        except NameError:
            perm_lmm_out_dir = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_behavior_tonelli_lag_corrected_subject_first\trial_level_metrics\safe_valid_cell_linear_mixed_models"

os.makedirs(perm_lmm_out_dir, exist_ok=True)

print("\n============================================================")
print("WHOLE-TRIAL CONDITION-LABEL SHUFFLE PERMUTATION TESTS")
print("============================================================")
print(f"Output directory:\n{perm_lmm_out_dir}")
print("All outputs will be saved directly into this folder.")
print(f"Metric: {PRIMARY_METRIC_COL}")
print(f"Permutations: {N_PERM_CONDITION_LABEL}")
print(f"Reference condition: {REFERENCE_CONDITION}")
print(f"Alternative: {ALTERNATIVE}")

# ============================================================
# SAFE SAVE HELPERS
# ============================================================

def safe_csv_path(filename):
    """
    Save directly into perm_lmm_out_dir.
    Does not create nested subfolders.
    """
    os.makedirs(perm_lmm_out_dir, exist_ok=True)
    return os.path.join(perm_lmm_out_dir, filename)


def save_df(df_to_save, filename, index=False):
    path = safe_csv_path(filename)
    df_to_save.to_csv(path, index=index)
    return path


def save_fig(fig, filename):
    path = safe_csv_path(filename)
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    return path

# ============================================================
# CONDITION MAPPING FALLBACK
# ============================================================

def extract_offset_perm(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan


def map_condition_detailed_perm(row):
    trial_type = str(row["trialType"]).lower()
    cond = str(row.get("condLabel", "")).lower()

    if trial_type == "audio_only":
        return "audio_only"

    if trial_type == "visual_only":
        return "visual_only"

    if trial_type == "congruent":
        return "congruent"

    if trial_type == "incongruent_spatial":
        off = extract_offset_perm(row)
        if np.isfinite(off):
            return f"offset_{int(off)}"
        return "offset_unknown"

    if trial_type == "incongruent_sweep":
        if "diverge" in cond:
            return "sweep_diverge"
        if "converge" in cond:
            return "sweep_converge"
        return "sweep_unknown"

    return np.nan

# ============================================================
# LOAD / PREPARE TRIAL-LEVEL DATA
# ============================================================

if "df" in globals() and isinstance(df, pd.DataFrame):
    base_df = df.copy()
    print("Using existing df from memory.")
elif "trial_metrics_df" in globals() and isinstance(trial_metrics_df, pd.DataFrame):
    base_df = trial_metrics_df.copy()
    print("Using existing trial_metrics_df from memory.")
else:
    try:
        trial_metrics_csv = os.path.join(
            trial_metric_dir,
            "cell3_trial_level_lag_corrected_spatial_metrics.csv"
        )
    except NameError:
        trial_metrics_csv = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_behavior_tonelli_lag_corrected_subject_first\trial_level_metrics\cell3_trial_level_lag_corrected_spatial_metrics.csv"

    base_df = pd.read_csv(trial_metrics_csv)
    print(f"Loaded trial-level metrics from:\n{trial_metrics_csv}")

if "condition_detailed" not in base_df.columns:
    base_df["condition_detailed"] = base_df.apply(map_condition_detailed_perm, axis=1)

base_df["condition_detailed"] = base_df["condition_detailed"].astype(str)

needed_cols = ["subject", "trialIndex", "trialType", "condition_detailed", PRIMARY_METRIC_COL]
missing_cols = [c for c in needed_cols if c not in base_df.columns]

if missing_cols:
    raise RuntimeError(
        f"Missing required columns for permutation analysis: {missing_cols}"
    )

av_condition_order_perm = [
    "congruent",
    "offset_10",
    "offset_20",
    "offset_30",
    "offset_40",
    "sweep_diverge",
    "sweep_converge",
]

av_trial_df = base_df[
    base_df["condition_detailed"].isin(av_condition_order_perm)
].copy()

av_trial_df = av_trial_df[
    np.isfinite(av_trial_df[PRIMARY_METRIC_COL])
].copy()

# Ensure one row per participant × trial.
trial_key_cols = ["subject", "trialIndex"]

if "source_file" in av_trial_df.columns:
    trial_key_cols = ["subject", "source_file", "trialIndex"]

av_trial_df = av_trial_df.drop_duplicates(subset=trial_key_cols).copy()

available_conditions = sorted(av_trial_df["condition_detailed"].dropna().unique())

planned_conditions_available = [
    c for c in PLANNED_CONDITION_CONTRASTS
    if c in available_conditions
]

if REFERENCE_CONDITION not in available_conditions:
    raise RuntimeError(
        f"Reference condition '{REFERENCE_CONDITION}' not found in AV trial data."
    )

if len(planned_conditions_available) == 0:
    raise RuntimeError("No planned contrast conditions were found in AV trial data.")

print("\nAV trial data prepared.")
print(f"Rows: {len(av_trial_df)}")
print(f"Subjects: {av_trial_df['subject'].nunique()}")
print("\nCondition counts:")
print(av_trial_df.groupby("condition_detailed").size().reindex(av_condition_order_perm))

av_trial_input_csv = save_df(
    av_trial_df,
    "perm_condition_label_shuffle_av_trial_input.csv",
    index=False
)

print(f"\nSaved AV trial input: {av_trial_input_csv}")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def mean_ci(values, alpha=0.05):
    vals = (
        pd.Series(values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
        .values
    )

    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "se": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "t": np.nan,
            "p_two_sided": np.nan,
            "p_greater": np.nan,
            "p_less": np.nan,
        }

    m = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1))
    se = float(sd / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    t_stat, p_two = stats.ttest_1samp(vals, popmean=0.0, nan_policy="omit")

    return {
        "n": int(n),
        "mean": m,
        "se": se,
        "ci95_low": float(m - crit * se),
        "ci95_high": float(m + crit * se),
        "t": float(t_stat),
        "p_two_sided": float(p_two),
        "p_greater": float(stats.t.sf(t_stat, df=n - 1)),
        "p_less": float(stats.t.cdf(t_stat, df=n - 1)),
    }


def empirical_pvalue(observed, null_values, alternative="greater"):
    null_values = (
        pd.Series(null_values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
        .values
    )

    if len(null_values) == 0 or not np.isfinite(observed):
        return np.nan

    if alternative == "greater":
        return float((np.sum(null_values >= observed) + 1) / (len(null_values) + 1))

    if alternative == "less":
        return float((np.sum(null_values <= observed) + 1) / (len(null_values) + 1))

    if alternative == "two-sided":
        null_center = np.nanmean(null_values)
        observed_distance = abs(observed - null_center)
        null_distance = np.abs(null_values - null_center)
        return float((np.sum(null_distance >= observed_distance) + 1) / (len(null_values) + 1))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")


def fdr_bh(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return qvals

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    q = np.empty_like(q_ranked)
    q[order] = q_ranked
    qvals[finite] = q

    return qvals


def holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    adj = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return adj

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    ranked_adj = np.empty(m)

    for i, pval in enumerate(ranked):
        ranked_adj[i] = (m - i) * pval

    ranked_adj = np.maximum.accumulate(ranked_adj)
    ranked_adj = np.clip(ranked_adj, 0, 1)

    out = np.empty_like(ranked_adj)
    out[order] = ranked_adj
    adj[finite] = out

    return adj


def compute_subject_condition_means(trial_df, condition_col, metric_col):
    use = trial_df.copy()
    use[condition_col] = use[condition_col].astype(str)

    subject_condition_means = (
        use
        .groupby(["subject", condition_col], as_index=False)
        .agg(
            mean_metric=(metric_col, "mean"),
            n_trials=(metric_col, "count")
        )
        .rename(columns={condition_col: "condition_for_test"})
    )

    return subject_condition_means


def compute_subject_contrasts(
    trial_df,
    condition_col,
    metric_col,
    reference_condition,
    contrast_conditions
):
    subject_condition_means = compute_subject_condition_means(
        trial_df=trial_df,
        condition_col=condition_col,
        metric_col=metric_col
    )

    wide = subject_condition_means.pivot(
        index="subject",
        columns="condition_for_test",
        values="mean_metric"
    )

    rows = []

    for cond in contrast_conditions:
        if reference_condition not in wide.columns or cond not in wide.columns:
            continue

        diff = wide[cond] - wide[reference_condition]

        for subj, val in diff.items():
            rows.append({
                "subject": subj,
                "contrast_condition": cond,
                "reference_condition": reference_condition,
                "contrast_label": f"{cond}_minus_{reference_condition}",
                "contrast_value": val,
                "metric": metric_col,
            })

    return pd.DataFrame(rows), subject_condition_means


def summarize_contrasts(contrast_df):
    rows = []

    if contrast_df.empty:
        return pd.DataFrame()

    for cond, g in contrast_df.groupby("contrast_condition"):
        vals = (
            g["contrast_value"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .astype(float)
            .values
        )

        stats_dict = mean_ci(vals)

        rows.append({
            "contrast_condition": cond,
            "contrast_label": f"{cond}_minus_{REFERENCE_CONDITION}",
            "n_subjects": stats_dict["n"],
            "mean_difference": stats_dict["mean"],
            "se": stats_dict["se"],
            "ci95_low": stats_dict["ci95_low"],
            "ci95_high": stats_dict["ci95_high"],
            "t": stats_dict["t"],
            "parametric_p_two_sided": stats_dict["p_two_sided"],
            "parametric_p_greater": stats_dict["p_greater"],
            "parametric_p_less": stats_dict["p_less"],
        })

    out = pd.DataFrame(rows)

    if not out.empty:
        out["contrast_condition"] = pd.Categorical(
            out["contrast_condition"],
            categories=planned_conditions_available,
            ordered=True
        )
        out = out.sort_values("contrast_condition").reset_index(drop=True)
        out["contrast_condition"] = out["contrast_condition"].astype(str)

    return out


def shuffle_condition_labels_within_subject(trial_df, condition_col, rng):
    """
    Whole-trial condition-label shuffle.
    Keeps each trial's observed metric value intact but randomly reassigns
    the condition label within each participant.
    """
    out = trial_df.copy()
    out["permuted_condition_detailed"] = np.nan

    for subj, idx in out.groupby("subject").groups.items():
        idx = list(idx)
        labels = out.loc[idx, condition_col].astype(str).values.copy()

        if len(labels) > 1:
            out.loc[idx, "permuted_condition_detailed"] = rng.permutation(labels)
        else:
            out.loc[idx, "permuted_condition_detailed"] = labels

    out["permuted_condition_detailed"] = out["permuted_condition_detailed"].astype(str)

    return out


def compute_t_vector_from_contrasts(contrast_df, condition_order):
    summary = summarize_contrasts(contrast_df)

    t_vec = []

    for cond in condition_order:
        row = summary[summary["contrast_condition"] == cond]

        if len(row) == 0:
            t_vec.append(np.nan)
        else:
            t_vec.append(float(row["t"].iloc[0]))

    return np.array(t_vec, dtype=float), summary


def compute_fixed_offset_slopes(
    trial_df,
    condition_col,
    metric_col,
    condition_to_deg
):
    use = trial_df[
        trial_df[condition_col].astype(str).isin(condition_to_deg.keys())
    ].copy()

    if use.empty:
        return pd.DataFrame()

    subject_condition_means = compute_subject_condition_means(
        trial_df=use,
        condition_col=condition_col,
        metric_col=metric_col
    )

    rows = []

    for subj, g in subject_condition_means.groupby("subject"):
        g = g.copy()
        g["offset_deg"] = g["condition_for_test"].map(condition_to_deg).astype(float)
        g = g.dropna(subset=["offset_deg", "mean_metric"])

        if len(g) < 3:
            continue

        x = g["offset_deg"].values.astype(float)
        y = g["mean_metric"].values.astype(float)

        slope, intercept = np.polyfit(x, y, deg=1)

        rows.append({
            "subject": subj,
            "n_conditions": len(g),
            "fixed_offset_slope_per_deg": float(slope),
            "fixed_offset_intercept": float(intercept),
        })

    return pd.DataFrame(rows)

# ============================================================
# OBSERVED CONTRASTS
# ============================================================

observed_contrasts, observed_subject_condition_means = compute_subject_contrasts(
    trial_df=av_trial_df,
    condition_col="condition_detailed",
    metric_col=PRIMARY_METRIC_COL,
    reference_condition=REFERENCE_CONDITION,
    contrast_conditions=planned_conditions_available
)

observed_summary = summarize_contrasts(observed_contrasts)

observed_contrasts_csv = save_df(
    observed_contrasts,
    "perm_condition_label_shuffle_observed_subject_level_condition_contrasts.csv",
    index=False
)

observed_summary_csv = save_df(
    observed_summary,
    "perm_condition_label_shuffle_observed_condition_contrast_summary.csv",
    index=False
)

observed_subject_condition_means_csv = save_df(
    observed_subject_condition_means,
    "perm_condition_label_shuffle_observed_subject_condition_means.csv",
    index=False
)

print("\nObserved planned condition contrasts:")
try:
    display(observed_summary)
except NameError:
    print(observed_summary.to_string(index=False))

# ============================================================
# PERMUTATION LOOP: CONDITION LABEL SHUFFLE
# ============================================================

rng = np.random.default_rng(RANDOM_SEED_CONDITION_LABEL)

null_long_rows = []
max_t_rows = []

for perm_i in range(N_PERM_CONDITION_LABEL):

    perm_df = shuffle_condition_labels_within_subject(
        trial_df=av_trial_df,
        condition_col="condition_detailed",
        rng=rng
    )

    perm_contrasts, _ = compute_subject_contrasts(
        trial_df=perm_df,
        condition_col="permuted_condition_detailed",
        metric_col=PRIMARY_METRIC_COL,
        reference_condition=REFERENCE_CONDITION,
        contrast_conditions=planned_conditions_available
    )

    perm_t_vec, perm_summary = compute_t_vector_from_contrasts(
        contrast_df=perm_contrasts,
        condition_order=planned_conditions_available
    )

    max_t = np.nanmax(perm_t_vec) if np.any(np.isfinite(perm_t_vec)) else np.nan

    max_t_rows.append({
        "perm_i": perm_i,
        "max_t": max_t,
        "metric": PRIMARY_METRIC_NAME,
        "alternative": ALTERNATIVE,
    })

    if SAVE_NULL_LONG and not perm_summary.empty:
        for _, row in perm_summary.iterrows():
            null_long_rows.append({
                "perm_i": perm_i,
                "metric": PRIMARY_METRIC_NAME,
                "contrast_condition": row["contrast_condition"],
                "contrast_label": row["contrast_label"],
                "mean_difference": row["mean_difference"],
                "t": row["t"],
                "n_subjects": row["n_subjects"],
            })

    if (perm_i + 1) % 250 == 0:
        print(f"Completed condition-label permutation {perm_i + 1}/{N_PERM_CONDITION_LABEL}")

condition_null_long = pd.DataFrame(null_long_rows)
condition_max_t_null = pd.DataFrame(max_t_rows)

if SAVE_NULL_LONG:
    condition_null_long_csv = save_df(
        condition_null_long,
        "perm_condition_label_shuffle_null_long.csv",
        index=False
    )
else:
    condition_null_long_csv = "not saved"

condition_max_t_null_csv = save_df(
    condition_max_t_null,
    "perm_condition_label_shuffle_maxT_null.csv",
    index=False
)

# ============================================================
# PERMUTATION P VALUES FOR PLANNED CONTRASTS
# ============================================================

test_rows = []

for _, obs in observed_summary.iterrows():
    cond = obs["contrast_condition"]
    obs_t = float(obs["t"])
    obs_mean = float(obs["mean_difference"])

    cond_null = condition_null_long[
        condition_null_long["contrast_condition"] == cond
    ].copy()

    null_t = cond_null["t"].dropna().astype(float).values
    null_mean = cond_null["mean_difference"].dropna().astype(float).values
    max_t_null = condition_max_t_null["max_t"].dropna().astype(float).values

    p_perm_t_uncorrected = empirical_pvalue(
        observed=obs_t,
        null_values=null_t,
        alternative="greater"
    )

    p_perm_mean_uncorrected = empirical_pvalue(
        observed=obs_mean,
        null_values=null_mean,
        alternative="greater"
    )

    p_perm_maxT = empirical_pvalue(
        observed=obs_t,
        null_values=max_t_null,
        alternative="greater"
    )

    if len(null_t) > 0:
        null_t_mean = float(np.nanmean(null_t))
        null_t_sd = float(np.nanstd(null_t, ddof=1))
    else:
        null_t_mean = np.nan
        null_t_sd = np.nan

    if len(null_mean) > 0:
        null_mean_mean = float(np.nanmean(null_mean))
        null_mean_sd = float(np.nanstd(null_mean, ddof=1))
        null_mean_ci95_low = float(np.nanpercentile(null_mean, 2.5))
        null_mean_ci95_high = float(np.nanpercentile(null_mean, 97.5))
    else:
        null_mean_mean = np.nan
        null_mean_sd = np.nan
        null_mean_ci95_low = np.nan
        null_mean_ci95_high = np.nan

    test_rows.append({
        "metric": PRIMARY_METRIC_NAME,
        "metric_column": PRIMARY_METRIC_COL,
        "null_type": "within_subject_whole_trial_condition_label_shuffle",
        "contrast_condition": cond,
        "contrast_label": obs["contrast_label"],
        "observed_mean_difference": obs_mean,
        "observed_se": obs["se"],
        "observed_ci95_low": obs["ci95_low"],
        "observed_ci95_high": obs["ci95_high"],
        "observed_t": obs_t,
        "observed_df": int(obs["n_subjects"] - 1),
        "observed_n_subjects": int(obs["n_subjects"]),
        "parametric_p_greater": obs["parametric_p_greater"],
        "parametric_p_two_sided": obs["parametric_p_two_sided"],
        "null_mean_difference_mean": null_mean_mean,
        "null_mean_difference_sd": null_mean_sd,
        "null_mean_difference_ci95_low": null_mean_ci95_low,
        "null_mean_difference_ci95_high": null_mean_ci95_high,
        "null_t_mean": null_t_mean,
        "null_t_sd": null_t_sd,
        "permutation_p_t_uncorrected": p_perm_t_uncorrected,
        "permutation_p_mean_uncorrected": p_perm_mean_uncorrected,
        "permutation_p_maxT": p_perm_maxT,
        "n_permutations": N_PERM_CONDITION_LABEL,
        "alternative": "greater",
    })

condition_label_shuffle_tests = pd.DataFrame(test_rows)

condition_label_shuffle_tests["permutation_p_t_fdr"] = fdr_bh(
    condition_label_shuffle_tests["permutation_p_t_uncorrected"].values
)

condition_label_shuffle_tests["permutation_p_t_holm"] = holm_bonferroni(
    condition_label_shuffle_tests["permutation_p_t_uncorrected"].values
)

condition_label_shuffle_tests["permutation_p_mean_fdr"] = fdr_bh(
    condition_label_shuffle_tests["permutation_p_mean_uncorrected"].values
)

condition_label_shuffle_tests["permutation_p_mean_holm"] = holm_bonferroni(
    condition_label_shuffle_tests["permutation_p_mean_uncorrected"].values
)

condition_label_shuffle_tests_csv = save_df(
    condition_label_shuffle_tests,
    "perm_condition_label_shuffle_planned_contrast_tests.csv",
    index=False
)

print("\nWhole-trial condition-label shuffle tests:")
try:
    display(condition_label_shuffle_tests)
except NameError:
    print(condition_label_shuffle_tests.to_string(index=False))

# ============================================================
# FIXED-OFFSET MONOTONIC SLOPE TEST
# ============================================================

fixed_offset_trial_df = av_trial_df[
    av_trial_df["condition_detailed"].isin(FIXED_OFFSET_CONDITION_TO_DEG.keys())
].copy()

observed_slope_df = compute_fixed_offset_slopes(
    trial_df=fixed_offset_trial_df,
    condition_col="condition_detailed",
    metric_col=PRIMARY_METRIC_COL,
    condition_to_deg=FIXED_OFFSET_CONDITION_TO_DEG
)

observed_slope_stats = mean_ci(observed_slope_df["fixed_offset_slope_per_deg"])

slope_null_rows = []

for perm_i in range(N_PERM_CONDITION_LABEL):

    perm_fixed_df = shuffle_condition_labels_within_subject(
        trial_df=fixed_offset_trial_df,
        condition_col="condition_detailed",
        rng=rng
    )

    perm_slope_df = compute_fixed_offset_slopes(
        trial_df=perm_fixed_df,
        condition_col="permuted_condition_detailed",
        metric_col=PRIMARY_METRIC_COL,
        condition_to_deg=FIXED_OFFSET_CONDITION_TO_DEG
    )

    perm_slope_stats = mean_ci(perm_slope_df["fixed_offset_slope_per_deg"])

    slope_null_rows.append({
        "perm_i": perm_i,
        "mean_slope_per_deg": perm_slope_stats["mean"],
        "t": perm_slope_stats["t"],
        "n_subjects": perm_slope_stats["n"],
    })

    if (perm_i + 1) % 250 == 0:
        print(f"Completed fixed-offset slope permutation {perm_i + 1}/{N_PERM_CONDITION_LABEL}")

slope_null_df = pd.DataFrame(slope_null_rows)

slope_p_t = empirical_pvalue(
    observed=observed_slope_stats["t"],
    null_values=slope_null_df["t"].values,
    alternative="greater"
)

slope_p_mean = empirical_pvalue(
    observed=observed_slope_stats["mean"],
    null_values=slope_null_df["mean_slope_per_deg"].values,
    alternative="greater"
)

slope_test_df = pd.DataFrame([{
    "metric": PRIMARY_METRIC_NAME,
    "metric_column": PRIMARY_METRIC_COL,
    "null_type": "within_subject_whole_trial_condition_label_shuffle_fixed_offset_conditions",
    "test": "fixed_offset_linear_slope_0_10_20_30_40_deg",
    "observed_mean_slope_per_deg": observed_slope_stats["mean"],
    "observed_se": observed_slope_stats["se"],
    "observed_ci95_low": observed_slope_stats["ci95_low"],
    "observed_ci95_high": observed_slope_stats["ci95_high"],
    "observed_t": observed_slope_stats["t"],
    "observed_df": int(observed_slope_stats["n"] - 1),
    "observed_n_subjects": int(observed_slope_stats["n"]),
    "parametric_p_greater": observed_slope_stats["p_greater"],
    "parametric_p_two_sided": observed_slope_stats["p_two_sided"],
    "permutation_p_t": slope_p_t,
    "permutation_p_mean": slope_p_mean,
    "n_permutations": N_PERM_CONDITION_LABEL,
    "alternative": "greater",
}])

observed_slope_csv = save_df(
    observed_slope_df,
    "perm_condition_label_shuffle_fixed_offset_observed_subject_slopes.csv",
    index=False
)

slope_null_csv = save_df(
    slope_null_df,
    "perm_condition_label_shuffle_fixed_offset_slope_null.csv",
    index=False
)

slope_test_csv = save_df(
    slope_test_df,
    "perm_condition_label_shuffle_fixed_offset_slope_test.csv",
    index=False
)

print("\nFixed-offset monotonic slope test:")
try:
    display(slope_test_df)
except NameError:
    print(slope_test_df.to_string(index=False))

# ============================================================
# NULL DISTRIBUTION PLOTS
# ============================================================

for _, row in condition_label_shuffle_tests.iterrows():
    cond = row["contrast_condition"]
    obs_mean = row["observed_mean_difference"]

    cond_null = condition_null_long[
        condition_null_long["contrast_condition"] == cond
    ].copy()

    if cond_null.empty:
        continue

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(cond_null["mean_difference"].dropna().astype(float).values, bins=40, alpha=0.75)
    ax.axvline(0, linewidth=1)
    ax.axvline(obs_mean, linestyle="--", linewidth=2)
    ax.set_xlabel(f"{cond} − {REFERENCE_CONDITION} mean difference")
    ax.set_ylabel("Permutation count")
    ax.set_title(f"Condition-label shuffle null: {cond} − {REFERENCE_CONDITION}")

    save_fig(
        fig,
        f"perm_condition_label_shuffle_null_{cond}_minus_{REFERENCE_CONDITION}.png"
    )
    plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(condition_max_t_null["max_t"].dropna().astype(float).values, bins=40, alpha=0.75)

for _, row in condition_label_shuffle_tests.iterrows():
    ax.axvline(row["observed_t"], linestyle="--", linewidth=1.5)

ax.set_xlabel("Maximum t across planned contrasts")
ax.set_ylabel("Permutation count")
ax.set_title("Max-T null distribution: condition-label shuffle")

save_fig(
    fig,
    "perm_condition_label_shuffle_maxT_null_distribution.png"
)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(slope_null_df["mean_slope_per_deg"].dropna().astype(float).values, bins=40, alpha=0.75)
ax.axvline(0, linewidth=1)
ax.axvline(observed_slope_stats["mean"], linestyle="--", linewidth=2)
ax.set_xlabel("Fixed-offset slope, RMSE per degree")
ax.set_ylabel("Permutation count")
ax.set_title("Condition-label shuffle null: fixed-offset RMSE slope")

save_fig(
    fig,
    "perm_condition_label_shuffle_fixed_offset_slope_null_distribution.png"
)
plt.show()

# ============================================================
# FINAL PRINT SUMMARY
# ============================================================

print("\n============================================================")
print("CONDITION-LABEL SHUFFLE PERMUTATION COMPLETE")
print("============================================================")

print("\nInterpretation:")
print("- The LMM estimates the condition-level effects.")
print("- This permutation test provides subject-first nonparametric inference for planned condition contrasts.")
print("- Whole-trial condition labels were shuffled within participant, preserving condition counts.")
print("- The max-T p value corrects across the planned condition contrasts.")

print("\nMain output files:")
print(f"Observed subject contrasts:        {observed_contrasts_csv}")
print(f"Observed contrast summary:         {observed_summary_csv}")
print(f"Condition-label null long:         {condition_null_long_csv}")
print(f"Max-T null:                        {condition_max_t_null_csv}")
print(f"Planned contrast tests:            {condition_label_shuffle_tests_csv}")
print(f"Observed fixed-offset slopes:      {observed_slope_csv}")
print(f"Fixed-offset slope null:           {slope_null_csv}")
print(f"Fixed-offset slope test:           {slope_test_csv}")

print("\nPlanned contrast permutation results:")
print(
    condition_label_shuffle_tests[
        [
            "contrast_condition",
            "observed_mean_difference",
            "observed_ci95_low",
            "observed_ci95_high",
            "observed_t",
            "permutation_p_t_uncorrected",
            "permutation_p_t_holm",
            "permutation_p_maxT",
        ]
    ].to_string(index=False)
)

print("\nFixed-offset slope permutation result:")
print(
    slope_test_df[
        [
            "observed_mean_slope_per_deg",
            "observed_ci95_low",
            "observed_ci95_high",
            "observed_t",
            "permutation_p_t",
            "permutation_p_mean",
        ]
    ].to_string(index=False)
)

## 7. Subject-Level Condition Aggregation and Unisensory Validation

Participant-by-condition aggregation and Tonelli-style audio-only versus visual-only tracking-lag validation.

In [ ]:
# %% CELL 4 — SUBJECT-LEVEL CONDITION AGGREGATION
# ============================================================
# Average trial-level metrics within each participant × condition
# This is the main dataset for group-level inference.
# ============================================================

metric_cols = [
    "PeakA", "LagA_sec",
    "PeakV", "LagV_sec",
    "Capture_peak",

    "RMSE_A_raw", "RMSE_V_raw", "RMSE_Mid_raw",
    "Dev_A_raw", "Dev_V_raw", "Dev_Mid_raw",
    "VisualCapture_raw",

    "RMSE_A_lagcorr", "RMSE_V_lagcorr",
    "RMSE_Mid_audioLag", "RMSE_Mid_visualLag",

    "Dev_A_lagcorr", "Dev_V_lagcorr",
    "Dev_Mid_audioLag", "Dev_Mid_visualLag",

    "VisualCapture_audioLag",
    "VisualCapture_visualLag",
    "Capture_rmse_lagcorr",
    "Capture_dev_lagcorr",

    "response_variance_trial",
]

subject_condition_df = (
    trial_metrics_df
    .groupby(
        ["subject", "trialType", "condLabel", "offsetDeg", "sideSign", "rampType"],
        dropna=False,
        as_index=False
    )
    .agg(
        n_trials=("trialIndex", "count"),
        **{f"{m}_mean": (m, "mean") for m in metric_cols},
        **{f"{m}_sd": (m, "std") for m in metric_cols},
    )
)

subject_condition_df["condition_sort"] = subject_condition_df["condLabel"].apply(condition_sort_key)
subject_condition_df = subject_condition_df.sort_values(["subject", "condition_sort"]).drop(columns=["condition_sort"])

subject_condition_csv = os.path.join(subject_metric_dir, "cell4_subject_condition_metrics.csv")
subject_condition_df.to_csv(subject_condition_csv, index=False)

# Also create a subject × offset summary for fixed incongruent spatial trials
subject_offset_df = (
    trial_metrics_df[trial_metrics_df["trialType"] == "incongruent_spatial"]
    .groupby(["subject", "offsetDeg", "sideSign"], dropna=False, as_index=False)
    .agg(
        n_trials=("trialIndex", "count"),
        **{f"{m}_mean": (m, "mean") for m in metric_cols},
    )
    .sort_values(["subject", "offsetDeg", "sideSign"])
)

subject_offset_csv = os.path.join(subject_metric_dir, "cell4_subject_fixed_offset_metrics.csv")
subject_offset_df.to_csv(subject_offset_csv, index=False)

print(f"Saved subject-condition metrics -> {subject_condition_csv}")
print(f"Saved subject fixed-offset metrics -> {subject_offset_csv}")
print(subject_condition_df.head())

In [ ]:
# %% TONELLI-STYLE AUDIO-ONLY VS VISUAL-ONLY LAG ANALYSIS
# ============================================================
# Goal:
# Replicate the core Tonelli-style modality lag question:
#
#   Does auditory tracking lag behind visual tracking?
#
# Main test:
#   LagDiff_AminusV = Lag_audio_only - Lag_visual_only
#
# Interpretation:
#   Positive LagDiff_AminusV:
#       audio-only tracking has a larger lag than visual-only tracking
#
#   Zero:
#       no modality-specific lag difference
#
#   Negative:
#       visual-only tracking lags more than audio-only tracking
#
# This analysis isolates modality-specific temporal tracking because
# audio-only trials are guided only by audio and visual-only trials
# are guided only by vision.
# ============================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_rel, ttest_1samp, t as t_dist, wilcoxon

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

SPATIAL_ANALYSES_ROOT = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses"
OUT_DIR_NAME = "01_unisensory_audio_visual_lag"

RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0
TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1.0 / RESAMPLE_HZ, 1.0 / RESAMPLE_HZ)

EXCLUDE_PRACTICE = True

# Tonelli-style lag scan
MAX_LAG_SEC = 4.0
MIN_VALID_SAMPLES = 8

# QC is applied ONLY because lag itself is the dependent variable here.
# A moderate threshold is used to avoid interpreting an argmax from very weak
# response-stimulus coupling as a meaningful latency estimate.
MIN_PEAK_R_FOR_LAG = 0.30
MIN_STIMULUS_SD_DEG = 3.0

# Optional: exclude first second if participants often settle into the trial
ANALYSIS_START_SEC = 0.0
ANALYSIS_END_SEC = 18.0

FIG_DPI = 180

# =========================
# OUTPUT DIRECTORIES
# =========================
out_root = os.path.join(SPATIAL_ANALYSES_ROOT, OUT_DIR_NAME)
trial_metric_dir = os.path.join(out_root, "trial_level_metrics")
subject_metric_dir = os.path.join(out_root, "subject_level_metrics")
plot_dir = os.path.join(out_root, "plots")

for d in [out_root, trial_metric_dir, subject_metric_dir, plot_dir]:
    os.makedirs(d, exist_ok=True)

# =========================
# HELPER FUNCTIONS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()
    return Path(filepath).stem

def resample_trial(df_trial, time_grid):
    needed = ["t", "mouse_x", "blob_x", "audio_x"]

    for c in needed:
        if c not in df_trial.columns:
            raise ValueError(f"Missing required column: {c}")

    d = df_trial.copy()
    d = d.sort_values("t")
    d = d[np.isfinite(d["t"])]
    d = d.drop_duplicates(subset="t", keep="first")

    if len(d) < 5:
        return None

    t_original = d["t"].values.astype(float)
    t_original = t_original - np.nanmin(t_original)

    if np.nanmax(t_original) < 1.0:
        return None

    def _interp(col):
        y = d[col].values.astype(float)
        ok = np.isfinite(t_original) & np.isfinite(y)

        if ok.sum() < 3:
            return np.full_like(time_grid, np.nan, dtype=float)

        return np.interp(
            time_grid,
            t_original[ok],
            y[ok],
            left=np.nan,
            right=np.nan
        )

    return {
        "t_grid": time_grid.copy(),
        "mouse_x": _interp("mouse_x"),
        "blob_x": _interp("blob_x"),
        "audio_x": _interp("audio_x"),
    }

def corr_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    x = x[ok]
    y = y[ok]

    if np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])

def lag_scanned_corr(response, stimulus, fs=20.0, max_lag_sec=4.0):
    """
    Lag-scanned correlation constrained to non-negative response delays.

    Positive lag_sec = response lags behind stimulus.

    Because this analysis quantifies response latency to an externally
    generated trajectory, negative response-stimulus shifts are not treated
    as meaningful sensory-response delays.
    """
    response = np.asarray(response, dtype=float)
    stimulus = np.asarray(stimulus, dtype=float)

    max_lag_samples = int(round(max_lag_sec * fs))
    lag_samples_grid = np.arange(0, max_lag_samples + 1)

    best_corr = np.nan
    best_lag_samples = np.nan

    for lag_samples in lag_samples_grid:
        if lag_samples == 0:
            r = response
            st = stimulus
        else:
            r = response[lag_samples:]
            st = stimulus[:-lag_samples]

        c = corr_1d(r, st)

        if np.isfinite(c) and (not np.isfinite(best_corr) or c > best_corr):
            best_corr = c
            best_lag_samples = lag_samples

    if np.isfinite(best_lag_samples):
        best_lag_sec = best_lag_samples / fs
    else:
        best_lag_sec = np.nan

    return {
        "peak_corr": best_corr,
        "lag_samples": best_lag_samples,
        "lag_sec": best_lag_sec,
    }

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)

    return m, m - crit * se, m + crit * se, n

def sem(vals):
    vals = pd.Series(vals).dropna().astype(float).values

    if len(vals) < 2:
        return np.nan

    return np.std(vals, ddof=1) / np.sqrt(len(vals))

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def condition_sort_key(label):
    s = str(label).lower()

    if "audio" in s and "only" in s:
        return 0
    if "visual" in s and "only" in s:
        return 1

    m = re.search(r"(10|20|30|40)", s)
    if m:
        return int(m.group(1))

    return 999

def cohens_d_paired(diff):
    diff = pd.Series(diff).dropna().astype(float).values

    if len(diff) < 2:
        return np.nan

    sd = np.std(diff, ddof=1)

    if sd == 0 or not np.isfinite(sd):
        return np.nan

    return np.mean(diff) / sd

# =========================
# STEP 1 — LOAD AUDIO-ONLY AND VISUAL-ONLY TRIALS
# =========================
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in {DATA_DIR}")

trial_rows = []

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    df = pd.read_csv(fpath)

    required_cols = [
        "trialIndex", "trialType", "condLabel", "isPractice",
        "t", "mouse_x", "blob_x", "audio_x"
    ]

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        print(f"Skipping {subj}: missing columns {missing}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    df = df[df["trialType"].astype(str).isin(["audio_only", "visual_only"])].copy()

    if len(df) == 0:
        continue

    extra_cols = [
        c for c in [
            "offsetDeg", "sideSign", "rampType", "visualOn", "audioOn",
            "post_question", "post_question2"
        ]
        if c in df.columns
    ]

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"],
        sort=False
    ):
        resampled = resample_trial(df_trial, TIME_GRID)

        if resampled is None:
            continue

        row = {
            "subject": subj,
            "source_file": Path(fpath).name,
            "trialIndex": trial_index,
            "trialType": trial_type,
            "condLabel": cond_label,
            "t_grid": resampled["t_grid"],
            "mouse_x": resampled["mouse_x"],
            "blob_x": resampled["blob_x"],
            "audio_x": resampled["audio_x"],
        }

        first_row = df_trial.iloc[0]
        for c in extra_cols:
            row[c] = first_row[c]

        trial_rows.append(row)

trial_df = pd.DataFrame(trial_rows)

if len(trial_df) == 0:
    raise RuntimeError("No audio_only or visual_only trials were loaded.")

trial_df = trial_df.sort_values(["subject", "trialIndex"]).reset_index(drop=True)

trial_df["trial_order_within_subject"] = (
    trial_df
    .groupby("subject")
    .cumcount() + 1
)

for c in ["offsetDeg", "sideSign", "rampType"]:
    if c not in trial_df.columns:
        trial_df[c] = np.nan

print(f"Loaded {len(trial_df)} audio-only / visual-only trials from {trial_df['subject'].nunique()} subjects.")

# =========================
# STEP 2 — TONELLI-STYLE CROSS-CORRELATION PER TRIAL
# =========================
xcorr_rows = []

analysis_mask = (
    (TIME_GRID >= ANALYSIS_START_SEC)
    & (TIME_GRID <= ANALYSIS_END_SEC)
)

for idx, row in trial_df.iterrows():
    mouse_x = np.asarray(row["mouse_x"], dtype=float)[analysis_mask]
    audio_x = np.asarray(row["audio_x"], dtype=float)[analysis_mask]
    blob_x = np.asarray(row["blob_x"], dtype=float)[analysis_mask]

    # Always compute both for transparency,
    # but primary modality-specific lag uses:
    #   audio_only: response vs audio
    #   visual_only: response vs visual
    audio_res = lag_scanned_corr(
        response=mouse_x,
        stimulus=audio_x,
        fs=RESAMPLE_HZ,
        max_lag_sec=MAX_LAG_SEC
    )

    visual_res = lag_scanned_corr(
        response=mouse_x,
        stimulus=blob_x,
        fs=RESAMPLE_HZ,
        max_lag_sec=MAX_LAG_SEC
    )

    if row["trialType"] == "audio_only":
        primary_peak = audio_res["peak_corr"]
        primary_lag_sec = audio_res["lag_sec"]
        primary_lag_samples = audio_res["lag_samples"]
        primary_target = "audio"
        primary_stimulus_sd = float(np.nanstd(audio_x, ddof=1))
    elif row["trialType"] == "visual_only":
        primary_peak = visual_res["peak_corr"]
        primary_lag_sec = visual_res["lag_sec"]
        primary_lag_samples = visual_res["lag_samples"]
        primary_target = "visual"
        primary_stimulus_sd = float(np.nanstd(blob_x, ddof=1))
    else:
        primary_peak = np.nan
        primary_lag_sec = np.nan
        primary_lag_samples = np.nan
        primary_target = "unknown"
        primary_stimulus_sd = np.nan

    # QC applies to the lag outcome only. Peak correlation remains available
    # unfiltered for the separate response-stimulus coupling analysis.
    lag_pass_peak_r = bool(
        np.isfinite(primary_peak) and primary_peak > MIN_PEAK_R_FOR_LAG
    )
    lag_pass_motion = bool(
        np.isfinite(primary_stimulus_sd)
        and primary_stimulus_sd >= MIN_STIMULUS_SD_DEG
    )
    lag_valid = bool(
        np.isfinite(primary_lag_sec)
        and lag_pass_peak_r
        and lag_pass_motion
    )
    primary_lag_valid_sec = primary_lag_sec if lag_valid else np.nan

    xcorr_rows.append({
        "trial_df_index": idx,
        "subject": row["subject"],
        "source_file": row["source_file"],
        "trialIndex": row["trialIndex"],
        "trial_order_within_subject": row["trial_order_within_subject"],
        "trialType": row["trialType"],
        "condLabel": row["condLabel"],
        "offsetDeg": row["offsetDeg"],
        "sideSign": row["sideSign"],
        "rampType": row["rampType"],

        "PeakA": audio_res["peak_corr"],
        "LagA_sec": audio_res["lag_sec"],
        "LagA_samples": audio_res["lag_samples"],

        "PeakV": visual_res["peak_corr"],
        "LagV_sec": visual_res["lag_sec"],
        "LagV_samples": visual_res["lag_samples"],

        "PrimaryTarget": primary_target,
        "PrimaryPeak": primary_peak,
        "PrimaryLag_sec": primary_lag_sec,
        "PrimaryLag_samples": primary_lag_samples,
        "PrimaryStimulusSD_deg": primary_stimulus_sd,
        "LagPassPeakR": lag_pass_peak_r,
        "LagPassMotion": lag_pass_motion,
        "LagValid": lag_valid,
        "PrimaryLag_valid_sec": primary_lag_valid_sec,
    })

xcorr_df = pd.DataFrame(xcorr_rows)

# Merge into trial_df
for col in [
    "PeakA", "LagA_sec", "LagA_samples",
    "PeakV", "LagV_sec", "LagV_samples",
    "PrimaryTarget", "PrimaryPeak",
    "PrimaryLag_sec", "PrimaryLag_samples",
    "PrimaryStimulusSD_deg", "LagPassPeakR", "LagPassMotion",
    "LagValid", "PrimaryLag_valid_sec"
]:
    trial_df[col] = xcorr_df.set_index("trial_df_index").loc[trial_df.index, col].values

xcorr_csv = os.path.join(trial_metric_dir, "trial_level_tonelli_audio_visual_only_xcorr_peak_lag_metrics.csv")
xcorr_df.to_csv(xcorr_csv, index=False)

trial_with_metrics_csv = os.path.join(trial_metric_dir, "trial_level_audio_visual_only_with_lag_metrics.csv")
trial_df.drop(columns=["t_grid", "mouse_x", "blob_x", "audio_x"]).to_csv(trial_with_metrics_csv, index=False)

print(f"Saved trial-level cross-correlation metrics -> {xcorr_csv}")
print(f"Saved trial-level lag metrics -> {trial_with_metrics_csv}")
print(xcorr_df.head())

# =========================
# STEP 3 — SUBJECT-LEVEL AGGREGATION
# =========================
# Trial-level lag estimates are skewed/bounded argmax quantities, so the
# participant-level lag is summarized by the MEDIAN of QC-valid trials.
# Peak correlation remains an unfiltered tracking-quality outcome and is
# summarized by the mean, matching the separate coupling analysis.

subject_condition_df = (
    trial_df
    .groupby(["subject", "trialType", "condLabel"], dropna=False, as_index=False)
    .agg(
        n_trials=("trialIndex", "count"),
        n_valid_lag=("LagValid", "sum"),

        PrimaryLag_sec_median=("PrimaryLag_valid_sec", "median"),
        PrimaryLag_sec_mean_valid=("PrimaryLag_valid_sec", "mean"),
        PrimaryPeak_mean=("PrimaryPeak", "mean"),
        PrimaryPeak_sd=("PrimaryPeak", "std"),
        PrimaryStimulusSD_deg_mean=("PrimaryStimulusSD_deg", "mean"),

        LagA_sec_median=("LagA_sec", "median"),
        LagA_sec_mean=("LagA_sec", "mean"),
        PeakA_mean=("PeakA", "mean"),

        LagV_sec_median=("LagV_sec", "median"),
        LagV_sec_mean=("LagV_sec", "mean"),
        PeakV_mean=("PeakV", "mean"),
    )
)

subject_condition_df["lag_retention_rate"] = (
    subject_condition_df["n_valid_lag"] / subject_condition_df["n_trials"]
)

subject_condition_df["condition_sort"] = subject_condition_df["condLabel"].apply(condition_sort_key)
subject_condition_df = (
    subject_condition_df
    .sort_values(["subject", "condition_sort"])
    .drop(columns=["condition_sort"])
)

subject_condition_csv = os.path.join(
    subject_metric_dir,
    "subject_condition_audio_visual_only_lag_metrics.csv"
)
subject_condition_df.to_csv(subject_condition_csv, index=False)

# Main paired subject-level table: one row per subject.
# Use a direct median across all QC-valid trials of the relevant modality.
audio_trials = trial_df[trial_df["trialType"] == "audio_only"].copy()
visual_trials = trial_df[trial_df["trialType"] == "visual_only"].copy()

audio_sub = (
    audio_trials
    .groupby("subject", as_index=False)
    .agg(
        n_audio_trials=("trialIndex", "count"),
        n_audio_valid_lag=("LagValid", "sum"),
        Lag_audio_only_sec=("PrimaryLag_valid_sec", "median"),
        Peak_audio_only=("PrimaryPeak", "mean"),
    )
)

visual_sub = (
    visual_trials
    .groupby("subject", as_index=False)
    .agg(
        n_visual_trials=("trialIndex", "count"),
        n_visual_valid_lag=("LagValid", "sum"),
        Lag_visual_only_sec=("PrimaryLag_valid_sec", "median"),
        Peak_visual_only=("PrimaryPeak", "mean"),
    )
)

audio_sub["audio_lag_retention_rate"] = (
    audio_sub["n_audio_valid_lag"] / audio_sub["n_audio_trials"]
)
visual_sub["visual_lag_retention_rate"] = (
    visual_sub["n_visual_valid_lag"] / visual_sub["n_visual_trials"]
)

subject_lag_df = pd.merge(audio_sub, visual_sub, on="subject", how="inner")

subject_lag_df["LagDiff_AminusV_sec"] = (
    subject_lag_df["Lag_audio_only_sec"] - subject_lag_df["Lag_visual_only_sec"]
)

subject_lag_df["PeakDiff_AminusV"] = (
    subject_lag_df["Peak_audio_only"] - subject_lag_df["Peak_visual_only"]
)

subject_lag_csv = os.path.join(
    subject_metric_dir,
    "subject_audio_minus_visual_lag_difference.csv"
)
subject_lag_df.to_csv(subject_lag_csv, index=False)

# QC summary by modality
qc_summary = (
    trial_df
    .groupby("trialType", as_index=False)
    .agg(
        n_trials=("trialIndex", "count"),
        n_valid_lag=("LagValid", "sum"),
        mean_peak_r=("PrimaryPeak", "mean"),
        median_peak_r=("PrimaryPeak", "median"),
        mean_stimulus_sd_deg=("PrimaryStimulusSD_deg", "mean"),
        median_stimulus_sd_deg=("PrimaryStimulusSD_deg", "median"),
    )
)
qc_summary["retention_rate"] = qc_summary["n_valid_lag"] / qc_summary["n_trials"]

qc_summary_csv = os.path.join(
    subject_metric_dir,
    "lag_qc_summary_by_modality.csv"
)
qc_summary.to_csv(qc_summary_csv, index=False)

print(f"Saved subject-condition metrics -> {subject_condition_csv}")
print(f"Saved subject audio-minus-visual lag differences -> {subject_lag_csv}")
print(f"Saved lag QC summary -> {qc_summary_csv}")
print("\nLag QC summary by modality:")
print(qc_summary.to_string(index=False))
print(subject_lag_df.head())

# =========================
# STEP 4 — GROUP-LEVEL TONELLI-STYLE TESTS
# =========================
valid = subject_lag_df.dropna(
    subset=["Lag_audio_only_sec", "Lag_visual_only_sec", "LagDiff_AminusV_sec"]
).copy()

if len(valid) < 2:
    raise RuntimeError("Not enough paired audio-only and visual-only subject data for group test.")

lag_audio = valid["Lag_audio_only_sec"].astype(float)
lag_visual = valid["Lag_visual_only_sec"].astype(float)
lag_diff = valid["LagDiff_AminusV_sec"].astype(float)

audio_mean, audio_lo, audio_hi, n_audio = mean_ci(lag_audio)
visual_mean, visual_lo, visual_hi, n_visual = mean_ci(lag_visual)
diff_mean, diff_lo, diff_hi, n_diff = mean_ci(lag_diff)

paired_t = ttest_rel(lag_audio, lag_visual)
one_sample_t = ttest_1samp(lag_diff, 0.0)

# One-sided test for audio lag > visual lag
if one_sample_t.statistic >= 0:
    one_sided_p_audio_greater = one_sample_t.pvalue / 2
else:
    one_sided_p_audio_greater = 1 - (one_sample_t.pvalue / 2)

try:
    wilcox = wilcoxon(lag_diff)
    wilcox_p = wilcox.pvalue
except Exception:
    wilcox_p = np.nan

d_paired = cohens_d_paired(lag_diff)

group_stats = pd.DataFrame([
    {
        "metric": "Lag_audio_only_sec",
        "n": n_audio,
        "mean": audio_mean,
        "ci_low": audio_lo,
        "ci_high": audio_hi,
        "test": "one_sample_vs_0",
        "t": ttest_1samp(lag_audio, 0.0).statistic,
        "df": n_audio - 1,
        "p_two_sided": ttest_1samp(lag_audio, 0.0).pvalue,
        "p_one_sided_audio_greater": np.nan,
        "cohens_d_paired": np.nan,
    },
    {
        "metric": "Lag_visual_only_sec",
        "n": n_visual,
        "mean": visual_mean,
        "ci_low": visual_lo,
        "ci_high": visual_hi,
        "test": "one_sample_vs_0",
        "t": ttest_1samp(lag_visual, 0.0).statistic,
        "df": n_visual - 1,
        "p_two_sided": ttest_1samp(lag_visual, 0.0).pvalue,
        "p_one_sided_audio_greater": np.nan,
        "cohens_d_paired": np.nan,
    },
    {
        "metric": "LagDiff_AminusV_sec",
        "n": n_diff,
        "mean": diff_mean,
        "ci_low": diff_lo,
        "ci_high": diff_hi,
        "test": "paired_audio_minus_visual",
        "t": paired_t.statistic,
        "df": n_diff - 1,
        "p_two_sided": paired_t.pvalue,
        "p_one_sided_audio_greater": one_sided_p_audio_greater,
        "wilcoxon_p_two_sided": wilcox_p,
        "cohens_d_paired": d_paired,
    }
])

group_stats_csv = os.path.join(subject_metric_dir, "group_audio_visual_lag_difference_stats.csv")
group_stats.to_csv(group_stats_csv, index=False)

print("\n============================================================")
print("TONELLI-STYLE AUDIO-ONLY VS VISUAL-ONLY LAG RESULTS")
print("============================================================")

print("\nAudio-only tracking lag")
print("-----------------------")
print(f"n = {n_audio}")
print(f"Mean = {audio_mean:.4f} s")
print(f"95% CI = [{audio_lo:.4f}, {audio_hi:.4f}] s")

print("\nVisual-only tracking lag")
print("------------------------")
print(f"n = {n_visual}")
print(f"Mean = {visual_mean:.4f} s")
print(f"95% CI = [{visual_lo:.4f}, {visual_hi:.4f}] s")

print("\nAudio-minus-visual lag difference")
print("---------------------------------")
print(f"n = {n_diff}")
print(f"Mean difference = {diff_mean:.4f} s")
print(f"95% CI = [{diff_lo:.4f}, {diff_hi:.4f}] s")
print(f"paired t({n_diff - 1}) = {paired_t.statistic:.4f}, p(two-sided) {p_fmt(paired_t.pvalue)}")
print(f"one-sided p(audio > visual) {p_fmt(one_sided_p_audio_greater)}")
print(f"paired Cohen's dz = {d_paired:.4f}")
print(f"Wilcoxon p(two-sided) {p_fmt(wilcox_p)}")

# =========================
# STEP 5 — PLOTS
# =========================

# Paired line plot
plot_df = valid.sort_values("LagDiff_AminusV_sec").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(6.8, 5.8))

x_audio = np.zeros(len(plot_df))
x_visual = np.ones(len(plot_df))

for _, r in plot_df.iterrows():
    ax.plot(
        [0, 1],
        [r["Lag_audio_only_sec"], r["Lag_visual_only_sec"]],
        alpha=0.35,
        linewidth=1.0
    )

ax.scatter(
    x_audio,
    plot_df["Lag_audio_only_sec"],
    alpha=0.75,
    label="Audio-only"
)

ax.scatter(
    x_visual,
    plot_df["Lag_visual_only_sec"],
    alpha=0.75,
    label="Visual-only"
)

ax.errorbar(
    [0],
    [audio_mean],
    yerr=[[audio_mean - audio_lo], [audio_hi - audio_mean]],
    fmt="o",
    markersize=10,
    capsize=5,
    linewidth=2.2
)

ax.errorbar(
    [1],
    [visual_mean],
    yerr=[[visual_mean - visual_lo], [visual_hi - visual_mean]],
    fmt="o",
    markersize=10,
    capsize=5,
    linewidth=2.2
)

ax.axhline(0, linestyle="--", linewidth=1.0, alpha=0.7)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Audio-only", "Visual-only"])
ax.set_ylabel("Best tracking lag (s)")
ax.set_title("Tonelli-style modality-specific tracking lag\nAudio-only vs Visual-only")

txt = (
    f"Audio − Visual = {diff_mean:.3f} s\n"
    f"95% CI [{diff_lo:.3f}, {diff_hi:.3f}]\n"
    f"t({n_diff - 1}) = {paired_t.statistic:.2f}, p {p_fmt(paired_t.pvalue)}"
)

ax.text(
    0.04,
    0.96,
    txt,
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)

fig.tight_layout()
paired_plot_path = os.path.join(plot_dir, "paired_audio_only_vs_visual_only_lag.png")
fig.savefig(paired_plot_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# Difference plot
fig, ax = plt.subplots(figsize=(7.0, 5.4))

ax.axhline(0, linestyle="--", linewidth=1.0, alpha=0.8)

x = np.arange(len(plot_df))
ax.scatter(
    x,
    plot_df["LagDiff_AminusV_sec"],
    alpha=0.80
)

ax.errorbar(
    [len(plot_df) + 1],
    [diff_mean],
    yerr=[[diff_mean - diff_lo], [diff_hi - diff_mean]],
    fmt="o",
    markersize=10,
    capsize=5,
    linewidth=2.2,
    label="Group mean"
)

ax.set_xlabel("Subject sorted by audio − visual lag")
ax.set_ylabel("Audio-only lag − Visual-only lag (s)")
ax.set_title("Subject-level modality lag difference")
ax.legend(frameon=False)

fig.tight_layout()
diff_plot_path = os.path.join(plot_dir, "subject_audio_minus_visual_lag_difference.png")
fig.savefig(diff_plot_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# Histogram
fig, ax = plt.subplots(figsize=(6.8, 5.2))

ax.hist(plot_df["LagDiff_AminusV_sec"].dropna(), bins=14, alpha=0.85)
ax.axvline(0, linestyle="--", linewidth=1.2, label="No lag difference")
ax.axvline(diff_mean, linewidth=2.0, label=f"Mean = {diff_mean:.3f}s")

ax.set_xlabel("Audio-only lag − Visual-only lag (s)")
ax.set_ylabel("Number of subjects")
ax.set_title("Distribution of modality-specific lag differences")
ax.legend(frameon=False)

fig.tight_layout()
hist_plot_path = os.path.join(plot_dir, "hist_audio_minus_visual_lag_difference.png")
fig.savefig(hist_plot_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# Peak correlation comparison plot
valid_peak = subject_lag_df.dropna(
    subset=["Peak_audio_only", "Peak_visual_only"]
).copy()

if len(valid_peak) >= 2:
    peak_audio_mean, peak_audio_lo, peak_audio_hi, _ = mean_ci(valid_peak["Peak_audio_only"])
    peak_visual_mean, peak_visual_lo, peak_visual_hi, _ = mean_ci(valid_peak["Peak_visual_only"])

    peak_t = ttest_rel(valid_peak["Peak_audio_only"], valid_peak["Peak_visual_only"])

    fig, ax = plt.subplots(figsize=(6.8, 5.6))

    for _, r in valid_peak.iterrows():
        ax.plot(
            [0, 1],
            [r["Peak_audio_only"], r["Peak_visual_only"]],
            alpha=0.35,
            linewidth=1.0
        )

    ax.scatter(
        np.zeros(len(valid_peak)),
        valid_peak["Peak_audio_only"],
        alpha=0.75,
        label="Audio-only"
    )

    ax.scatter(
        np.ones(len(valid_peak)),
        valid_peak["Peak_visual_only"],
        alpha=0.75,
        label="Visual-only"
    )

    ax.errorbar(
        [0],
        [peak_audio_mean],
        yerr=[[peak_audio_mean - peak_audio_lo], [peak_audio_hi - peak_audio_mean]],
        fmt="o",
        markersize=10,
        capsize=5,
        linewidth=2.2
    )

    ax.errorbar(
        [1],
        [peak_visual_mean],
        yerr=[[peak_visual_mean - peak_visual_lo], [peak_visual_hi - peak_visual_mean]],
        fmt="o",
        markersize=10,
        capsize=5,
        linewidth=2.2
    )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Audio-only", "Visual-only"])
    ax.set_ylabel("Peak lag-scanned correlation")
    ax.set_title("Peak tracking correlation by modality")

    txt = (
        f"Audio − Visual peak = {(valid_peak['Peak_audio_only'] - valid_peak['Peak_visual_only']).mean():.3f}\n"
        f"t({len(valid_peak)-1}) = {peak_t.statistic:.2f}, p {p_fmt(peak_t.pvalue)}"
    )

    ax.text(
        0.04,
        0.96,
        txt,
        transform=ax.transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
    )

    ax.legend(frameon=False)
    fig.tight_layout()

    peak_plot_path = os.path.join(plot_dir, "paired_audio_only_vs_visual_only_peak_corr.png")
    fig.savefig(peak_plot_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
else:
    peak_plot_path = None

# =========================
# FINAL SUMMARY
# =========================
print("\n============================================================")
print("TONELLI-STYLE LAG ANALYSIS COMPLETE")
print("============================================================")
print(f"Trial-level xcorr metrics:       {xcorr_csv}")
print(f"Trial-level lag metrics:         {trial_with_metrics_csv}")
print(f"Subject-condition lag metrics:   {subject_condition_csv}")
print(f"Subject lag-difference table:    {subject_lag_csv}")
print(f"Group stats:                     {group_stats_csv}")
print(f"Lag QC summary:                  {qc_summary_csv}")
print(f"Paired lag plot:                 {paired_plot_path}")
print(f"Lag difference plot:             {diff_plot_path}")
print(f"Lag histogram:                   {hist_plot_path}")
if peak_plot_path is not None:
    print(f"Peak correlation plot:           {peak_plot_path}")

print("\nInterpretation guide:")
print("- Positive LagDiff_AminusV_sec means auditory tracking lagged behind visual tracking.")
print("- Participant lag estimates are medians across QC-valid trials.")
print(f"- Lag QC: peak r > {MIN_PEAK_R_FOR_LAG:.2f} and stimulus SD >= {MIN_STIMULUS_SD_DEG:.1f} deg.")
print("- Peak-correlation comparisons remain unfiltered by the lag QC threshold.")
print("- The main inferential test is the paired t-test comparing audio-only lag to visual-only lag across subjects.")
print("- The one-sided p-value tests Tonelli-style directional prediction: audio lag > visual lag.")
print("- Peak correlations are included as a tracking-quality diagnostic, not the main lag result.")

## 8. Time-Resolved RMSE and Bias

Timepoint-resolved raw and lag-corrected RMSE, stable-window summaries, and permutation inference.

In [11]:
# %% TIME-RESOLVED RAW AND LAG-CORRECTED RMSE BY CONDITION
# ============================================================
# Computes timepoint-by-timepoint RMSE to audio and visual.
#
# Conditions:
#   offset_10, offset_20, offset_30, offset_40,
#   sweep_diverge, sweep_converge
#
# Subject-first pipeline:
#   trial-level squared error at each timepoint
#   -> average within subject × condition
#   -> sqrt = subject-level time-resolved RMSE
#   -> average subject curves for group-level plots
#
# Also includes:
#   Stable-window inferential summaries over 0.5–17.5 s.
#
# IMPORTANT UPDATE
# ============================================================
# This version uses ROBUST condition mapping.
#
# It recognizes condition information from:
#   trialType
#   condLabel
#   offsetDeg
#   physical_condition
#   condition
#   condition_detailed
#
# and recognizes BOTH naming conventions:
#
#   fixed_10 / fixed_20 / fixed_30 / fixed_40
#   offset_10 / offset_20 / offset_30 / offset_40
#
# plus:
#
#   sweep_diverge
#   sweep_converge
#
# It therefore does not depend on one exact condLabel format.
#
# Lag handling:
#   - uses existing LagA_sec / LagV_sec when available
#   - otherwise checks trial_metrics_df
#   - otherwise computes non-negative 0–4 s xcorr lag
#
# NOTE:
# No peak-r exclusion is applied to the spatial RMSE/bias analysis itself.
# Weak AV response-stimulus coupling can be part of the behavioral effect.
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    ttest_1samp,
    ttest_rel,
    t as t_dist,
)


# ============================================================
# SETTINGS
# ============================================================

FIG_DPI = 180

SPATIAL_ANALYSES_ROOT = (
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1\spatial_analyses"
)

OUT_DIR = os.path.join(
    SPATIAL_ANALYSES_ROOT,
    "03_time_resolved_rmse_bias",
)

SUBJECT_PLOT_DIR = os.path.join(
    OUT_DIR,
    "participant_plots",
)

GROUP_PLOT_DIR = os.path.join(
    OUT_DIR,
    "group_plots",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

for d in [
    OUT_DIR,
    SUBJECT_PLOT_DIR,
    GROUP_PLOT_DIR,
    CSV_DIR,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Sampling
# ------------------------------------------------------------

try:
    RESAMPLE_HZ
except NameError:
    RESAMPLE_HZ = 20.0


# Revised lag range
MAX_LAG_SEC = 4.0

MIN_LAG_VALID_SAMPLES = 10


# ============================================================
# LOAD DATA IF NEEDED
# ============================================================

if "trial_df" not in globals():

    raise RuntimeError(
        "trial_df not found.\n\n"
        "Run your load/resample cell first. This analysis expects "
        "trial_df to contain one row per trial with array columns:\n"
        "t_grid, mouse_x, audio_x, blob_x."
    )


# ------------------------------------------------------------
# Trial metrics
# ------------------------------------------------------------

if "trial_metrics_df" not in globals():

    possible_metric_files = [

        os.path.join(
            SPATIAL_ANALYSES_ROOT,
            "00_core_spatial_metrics",
            "trial_level_metrics",
            "cell3_trial_level_lag_corrected_spatial_metrics.csv",
        ),

        (
            r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
            r"\Continuous AVT v1\individual_data_SL"
            r"\spatial_behavior_tonelli_lag_corrected_subject_first"
            r"\trial_level_metrics"
            r"\cell3_trial_level_lag_corrected_spatial_metrics.csv"
        ),
    ]

    trial_metrics_df = pd.DataFrame()

    for candidate in possible_metric_files:

        if os.path.exists(candidate):

            trial_metrics_df = pd.read_csv(
                candidate
            )

            print(
                f"Loaded trial_metrics_df from:\n{candidate}"
            )

            break

    if len(trial_metrics_df) == 0:

        print(
            "\nWARNING: trial_metrics_df was not found."
        )

        print(
            "Missing trial-specific lags will be computed directly "
            "from trial_df."
        )

else:

    print(
        "Using existing trial_metrics_df from memory."
    )


# ============================================================
# GENERAL HELPER FUNCTIONS
# ============================================================

def normalize_text(x):
    """
    Normalize labels so condition mapping is insensitive to capitalization,
    spaces, hyphens, etc.
    """

    if pd.isna(x):
        return ""

    s = str(x).strip().lower()

    s = s.replace("°", "")
    s = s.replace("-", "_")
    s = s.replace(" ", "_")
    s = s.replace("/", "_")

    while "__" in s:
        s = s.replace("__", "_")

    return s


def safe_numeric(x):

    try:

        value = float(x)

        if np.isfinite(value):
            return value

    except Exception:
        pass

    return np.nan


def extract_numeric_offset_from_text(x):
    """
    Extract 10, 20, 30, or 40 from arbitrary condition text.
    """

    s = normalize_text(x)

    # Use boundaries that are compatible with underscores.
    m = re.search(
        r"(?:^|[^0-9])(10|20|30|40)(?:[^0-9]|$)",
        s,
    )

    if m:

        return float(
            m.group(1)
        )

    return np.nan


# ============================================================
# ROBUST OFFSET EXTRACTION
# ============================================================

def extract_offset(row):
    """
    Recover absolute fixed disparity from any available source.

    Priority:
        1. offsetDeg
        2. disparity_deg
        3. offset_deg
        4. physical_condition
        5. condition
        6. condition_detailed
        7. condLabel
        8. trialType
    """

    # --------------------------------------------------------
    # Direct numeric columns
    # --------------------------------------------------------

    numeric_candidates = [
        "offsetDeg",
        "offset_deg",
        "disparity_deg",
        "disparity",
        "offset",
    ]

    for col in numeric_candidates:

        if (
            col in row.index
            and pd.notna(
                row[col]
            )
        ):

            value = safe_numeric(
                row[col]
            )

            if np.isfinite(
                value
            ):

                value = abs(
                    value
                )

                # Round because floating-point representations can occur.
                rounded = int(
                    round(
                        value
                    )
                )

                if rounded in [
                    10,
                    20,
                    30,
                    40,
                ]:

                    return rounded


    # --------------------------------------------------------
    # Text columns
    # --------------------------------------------------------

    text_candidates = [
        "physical_condition",
        "condition",
        "condition_detailed",
        "condLabel",
        "trialType",
    ]

    for col in text_candidates:

        if col not in row.index:
            continue

        value = extract_numeric_offset_from_text(
            row[col]
        )

        if np.isfinite(
            value
        ):

            rounded = int(
                round(
                    value
                )
            )

            if rounded in [
                10,
                20,
                30,
                40,
            ]:

                return rounded


    return np.nan


# ============================================================
# ROBUST CONDITION MAPPING
# ============================================================

def map_condition_detailed(row):
    """
    Map each trial into one of:

        offset_10
        offset_20
        offset_30
        offset_40
        sweep_diverge
        sweep_converge

    This deliberately does NOT assume one exact trialType/condLabel syntax.
    """

    # --------------------------------------------------------
    # Collect every potentially informative label
    # --------------------------------------------------------

    candidate_columns = [
        "condition_detailed",
        "physical_condition",
        "condition",
        "condLabel",
        "trialType",
        "rampType",
    ]

    labels = []

    for col in candidate_columns:

        if (
            col in row.index
            and pd.notna(
                row[col]
            )
        ):

            labels.append(
                normalize_text(
                    row[col]
                )
            )


    combined = "_".join(
        labels
    )


    # ========================================================
    # SWEEPS FIRST
    # ========================================================

    # Diverging
    diverge_tokens = [
        "sweep_diverge",
        "sweep_diverging",
        "diverge",
        "diverging",
        "divergent",
        "ramp_out",
        "rampout",
        "outward",
    ]

    if any(
        token in combined
        for token in diverge_tokens
    ):

        return "sweep_diverge"


    # Converging
    converge_tokens = [
        "sweep_converge",
        "sweep_converging",
        "converge",
        "converging",
        "convergent",
        "ramp_in",
        "rampin",
        "inward",
    ]

    if any(
        token in combined
        for token in converge_tokens
    ):

        return "sweep_converge"


    # ========================================================
    # FIXED OFFSETS
    # ========================================================

    # Explicit standardized names first.
    explicit_fixed_mapping = {

        "fixed_10": "offset_10",
        "offset_10": "offset_10",

        "fixed_20": "offset_20",
        "offset_20": "offset_20",

        "fixed_30": "offset_30",
        "offset_30": "offset_30",

        "fixed_40": "offset_40",
        "offset_40": "offset_40",
    }

    for token, mapped in explicit_fixed_mapping.items():

        if token in combined:

            return mapped


    # --------------------------------------------------------
    # Otherwise infer from numeric offset.
    # --------------------------------------------------------

    off = extract_offset(
        row
    )

    if np.isfinite(
        off
    ):

        off = int(
            round(
                off
            )
        )

        if off in [
            10,
            20,
            30,
            40,
        ]:

            return f"offset_{off}"


    return np.nan


# ============================================================
# ARRAY / CORRELATION HELPERS
# ============================================================

def safe_array(x):

    return np.asarray(
        x,
        dtype=float,
    )


def corr_1d(
    x,
    y,
):

    x = safe_array(
        x
    )

    y = safe_array(
        y
    )

    ok = (
        np.isfinite(
            x
        )
        &
        np.isfinite(
            y
        )
    )

    if ok.sum() < MIN_LAG_VALID_SAMPLES:

        return np.nan

    if (
        np.nanstd(
            x[
                ok
            ]
        ) < 1e-12
        or
        np.nanstd(
            y[
                ok
            ]
        ) < 1e-12
    ):

        return np.nan

    return float(
        np.corrcoef(
            x[
                ok
            ],
            y[
                ok
            ],
        )[0, 1]
    )


# ============================================================
# NON-NEGATIVE CROSS-CORRELATION LAG
# ============================================================

def estimate_lag_by_xcorr(
    response,
    stimulus,
    t_grid=None,
    max_lag_sec=4.0,
    fs=None,
):
    """
    Estimate response delay from 0 to max_lag_sec.

    Positive lag:
        response follows stimulus.

    Negative lags are intentionally NOT searched.

    No peak-correlation exclusion is applied here because this lag is being
    used as a nuisance temporal-alignment parameter for spatial RMSE/bias.
    Low auditory coupling during AV conflict may itself be behaviorally
    meaningful and should not cause the trial to be deleted.
    """

    response = np.asarray(
        response,
        dtype=float,
    )

    stimulus = np.asarray(
        stimulus,
        dtype=float,
    )


    # --------------------------------------------------------
    # Determine sampling frequency
    # --------------------------------------------------------

    if fs is None:

        if t_grid is None:

            fs = RESAMPLE_HZ

        else:

            tt = np.asarray(
                t_grid,
                dtype=float,
            )

            finite_t = tt[
                np.isfinite(
                    tt
                )
            ]

            if len(
                finite_t
            ) >= 2:

                dt = np.nanmedian(
                    np.diff(
                        finite_t
                    )
                )

            else:

                dt = np.nan


            if (
                np.isfinite(
                    dt
                )
                and dt > 0
            ):

                fs = 1.0 / dt

            else:

                fs = RESAMPLE_HZ


    max_lag_samples = int(
        round(
            max_lag_sec
            *
            fs
        )
    )


    best_corr = np.nan
    best_lag_samples = np.nan


    for lag_samples in range(
        0,
        max_lag_samples + 1,
    ):

        if lag_samples == 0:

            r = response
            st = stimulus

        else:

            r = response[
                lag_samples:
            ]

            st = stimulus[
                :-lag_samples
            ]


        c = corr_1d(
            r,
            st,
        )


        if (
            np.isfinite(
                c
            )
            and
            (
                not np.isfinite(
                    best_corr
                )
                or c > best_corr
            )
        ):

            best_corr = c
            best_lag_samples = lag_samples


    if not np.isfinite(
        best_lag_samples
    ):

        return {
            "peak_corr": np.nan,
            "lag_sec": np.nan,
            "lag_samples": np.nan,
        }


    return {

        "peak_corr":
            float(
                best_corr
            ),

        "lag_sec":
            float(
                best_lag_samples
                /
                fs
            ),

        "lag_samples":
            int(
                best_lag_samples
            ),
    }


# ============================================================
# SHIFT RESPONSE
# ============================================================

def shift_response_by_lag(
    response,
    t_grid,
    lag_sec,
):
    """
    Align delayed response to stimulus time.

    If response lags the stimulus by +L seconds:

        corrected_response(t) = response(t + L)
    """

    response = safe_array(
        response
    )

    t_grid = safe_array(
        t_grid
    )


    if not np.isfinite(
        lag_sec
    ):

        return response.copy()


    return np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan,
    )


# ============================================================
# SUMMARY HELPERS
# ============================================================

def mean_sem_ci(
    arr,
    axis=0,
):

    arr = np.asarray(
        arr,
        dtype=float,
    )

    mean = np.nanmean(
        arr,
        axis=axis,
    )

    n = np.sum(
        np.isfinite(
            arr
        ),
        axis=axis,
    )

    sd = np.nanstd(
        arr,
        axis=axis,
        ddof=1,
    )

    with np.errstate(
        invalid="ignore",
        divide="ignore",
    ):

        sem = (
            sd
            /
            np.sqrt(
                n
            )
        )

    ci95 = (
        1.96
        *
        sem
    )

    return (
        mean,
        sem,
        ci95,
        n,
    )


def save_fig(
    fig,
    path,
):

    fig.tight_layout()

    fig.savefig(
        path,
        dpi=FIG_DPI,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )


def first_existing_column(
    df,
    candidates,
):

    for c in candidates:

        if c in df.columns:

            return c

    return None


# ============================================================
# STANDARDIZE LAG COLUMNS FROM TRIAL METRICS
# ============================================================

def standardize_lag_columns_from_metrics(
    trial_metrics_df
):
    """
    Return:

        subject
        trialIndex
        LagA_sec
        LagV_sec

    regardless of the exact lag-column names used upstream.
    """

    merge_cols = [
        "subject",
        "trialIndex",
    ]


    if (
        trial_metrics_df is None
        or len(
            trial_metrics_df
        ) == 0
    ):

        return pd.DataFrame(
            columns=
                merge_cols
                +
                [
                    "LagA_sec",
                    "LagV_sec",
                ]
        )


    if not all(
        c in trial_metrics_df.columns
        for c in merge_cols
    ):

        print(
            "\nWARNING: trial_metrics_df does not contain "
            "subject and trialIndex."
        )

        print(
            "Skipping lag merge."
        )

        return pd.DataFrame(
            columns=
                merge_cols
                +
                [
                    "LagA_sec",
                    "LagV_sec",
                ]
        )


    audio_lag_candidates = [

        "LagA_sec",
        "lagA_sec",

        "LagAudio_sec",
        "AudioLag_sec",

        "audio_lag_sec",
        "lag_audio_sec",

        "best_audio_lag_sec",
        "audio_best_lag_sec",

        "A_lag_sec",
        "lag_a_sec",
    ]


    visual_lag_candidates = [

        "LagV_sec",
        "lagV_sec",

        "LagVisual_sec",
        "VisualLag_sec",

        "visual_lag_sec",
        "lag_visual_sec",

        "best_visual_lag_sec",
        "visual_best_lag_sec",

        "V_lag_sec",
        "lag_v_sec",
    ]


    audio_col = first_existing_column(
        trial_metrics_df,
        audio_lag_candidates,
    )

    visual_col = first_existing_column(
        trial_metrics_df,
        visual_lag_candidates,
    )


    metric_lags = (
        trial_metrics_df[
            merge_cols
        ]
        .drop_duplicates(
            subset=merge_cols,
            keep="first",
        )
        .copy()
    )


    # --------------------------------------------------------
    # Audio
    # --------------------------------------------------------

    if audio_col is not None:

        tmp = (
            trial_metrics_df[
                merge_cols
                +
                [
                    audio_col
                ]
            ]
            .drop_duplicates(
                subset=merge_cols,
                keep="first",
            )
            .copy()
        )

        tmp = tmp.rename(
            columns={
                audio_col:
                    "LagA_sec"
            }
        )

        metric_lags = metric_lags.merge(
            tmp,
            on=merge_cols,
            how="left",
        )

        metric_lags[
            "LagA_sec"
        ] = pd.to_numeric(
            metric_lags[
                "LagA_sec"
            ],
            errors="coerce",
        )

    else:

        metric_lags[
            "LagA_sec"
        ] = np.nan


    # --------------------------------------------------------
    # Visual
    # --------------------------------------------------------

    if visual_col is not None:

        tmp = (
            trial_metrics_df[
                merge_cols
                +
                [
                    visual_col
                ]
            ]
            .drop_duplicates(
                subset=merge_cols,
                keep="first",
            )
            .copy()
        )

        tmp = tmp.rename(
            columns={
                visual_col:
                    "LagV_sec"
            }
        )

        metric_lags = metric_lags.merge(
            tmp,
            on=merge_cols,
            how="left",
        )

        metric_lags[
            "LagV_sec"
        ] = pd.to_numeric(
            metric_lags[
                "LagV_sec"
            ],
            errors="coerce",
        )

    else:

        metric_lags[
            "LagV_sec"
        ] = np.nan


    print(
        "\nLag-column detection:"
    )

    print(
        f"Audio lag column found:  {audio_col}"
    )

    print(
        f"Visual lag column found: {visual_col}"
    )


    return metric_lags[
        merge_cols
        +
        [
            "LagA_sec",
            "LagV_sec",
        ]
    ].copy()


# ============================================================
# PREPARE DATA
# ============================================================

df = trial_df.copy()


# ------------------------------------------------------------
# Required structural columns
# ------------------------------------------------------------

required_cols = [
    "subject",
    "trialIndex",
    "t_grid",
    "mouse_x",
    "audio_x",
    "blob_x",
]

missing = [
    c for c in required_cols
    if c not in df.columns
]

if missing:

    raise RuntimeError(
        "trial_df is missing required columns:\n"
        f"{missing}"
    )


# ============================================================
# IMPORTANT DIAGNOSTIC: SHOW ORIGINAL CONDITION STRUCTURE
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "ORIGINAL TRIAL_DF CONDITION STRUCTURE"
)

print(
    "=" * 80
)


for col in [
    "trialType",
    "condLabel",
    "physical_condition",
    "condition",
    "condition_detailed",
    "offsetDeg",
    "offset_deg",
    "rampType",
]:

    if col in df.columns:

        print(
            f"\n{col}:"
        )

        if df[
            col
        ].apply(
            lambda x:
                isinstance(
                    x,
                    (
                        list,
                        tuple,
                        np.ndarray,
                    )
                )
        ).any():

            print(
                "  [array-valued column]"
            )

        else:

            print(
                df[
                    col
                ]
                .value_counts(
                    dropna=False
                )
                .head(
                    30
                )
                .to_string()
            )


# ============================================================
# MAP CONDITIONS
# ============================================================

# IMPORTANT:
# Do NOT overwrite an existing condition_detailed column before the mapper
# has had a chance to use it.
#
# Save it temporarily if it already exists.

if "condition_detailed" in df.columns:

    df[
        "_existing_condition_detailed"
    ] = df[
        "condition_detailed"
    ]

else:

    df[
        "_existing_condition_detailed"
    ] = np.nan


# The mapper checks all available condition information.
def map_condition_final(row):

    # Temporarily make the pre-existing condition available to mapper.
    row_copy = row.copy()

    if pd.notna(
        row.get(
            "_existing_condition_detailed",
            np.nan,
        )
    ):

        row_copy[
            "condition_detailed"
        ] = row[
            "_existing_condition_detailed"
        ]

    return map_condition_detailed(
        row_copy
    )


df[
    "condition_detailed"
] = df.apply(
    map_condition_final,
    axis=1,
)


# ============================================================
# CONDITION-MAPPING DIAGNOSTICS
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "CONDITION MAPPING RESULTS"
)

print(
    "=" * 80
)


print(
    "\nMapped condition counts:"
)

print(
    df[
        "condition_detailed"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ------------------------------------------------------------
# Show unmapped examples
# ------------------------------------------------------------

unmapped = df[
    df[
        "condition_detailed"
    ].isna()
].copy()


if len(
    unmapped
) > 0:

    diagnostic_cols = [
        c for c in [
            "subject",
            "trialIndex",
            "trialType",
            "condLabel",
            "physical_condition",
            "condition",
            "_existing_condition_detailed",
            "offsetDeg",
            "offset_deg",
            "rampType",
        ]
        if c in unmapped.columns
    ]


    print(
        f"\nUnmapped trials: {len(unmapped)}"
    )

    print(
        "\nFirst 20 unmapped examples:"
    )

    print(
        unmapped[
            diagnostic_cols
        ]
        .head(
            20
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# KEEP TARGET CONDITIONS
# ============================================================

conditions = [

    "offset_10",
    "offset_20",
    "offset_30",
    "offset_40",

    "sweep_diverge",
    "sweep_converge",
]


df = df[
    df[
        "condition_detailed"
    ].isin(
        conditions
    )
].copy()


if len(
    df
) == 0:

    raise RuntimeError(
        "\nCondition mapping still returned zero target trials.\n\n"
        "The diagnostic output immediately above shows the actual "
        "trialType/condLabel/condition values in trial_df. "
        "Those values can then be mapped explicitly."
    )


print(
    "\n"
    + "=" * 80
)

print(
    "TARGET TRIALS RETAINED"
)

print(
    "=" * 80
)


print(
    f"\nTotal target trials: {len(df)}"
)

print(
    f"Subjects: {df['subject'].nunique()}"
)

print(
    "\nTrials by condition:"
)

print(
    df[
        "condition_detailed"
    ]
    .value_counts()
    .reindex(
        conditions,
        fill_value=0,
    )
    .to_string()
)


# Remove temporary column
if "_existing_condition_detailed" in df.columns:

    df = df.drop(
        columns=[
            "_existing_condition_detailed"
        ]
    )


# ============================================================
# PREPARE EXISTING LAG COLUMNS
# ============================================================

if "LagA_sec" not in df.columns:

    df[
        "LagA_sec"
    ] = np.nan

else:

    df[
        "LagA_sec"
    ] = pd.to_numeric(
        df[
            "LagA_sec"
        ],
        errors="coerce",
    )


if "LagV_sec" not in df.columns:

    df[
        "LagV_sec"
    ] = np.nan

else:

    df[
        "LagV_sec"
    ] = pd.to_numeric(
        df[
            "LagV_sec"
        ],
        errors="coerce",
    )


df[
    "LagA_source"
] = np.where(
    np.isfinite(
        df[
            "LagA_sec"
        ]
    ),
    "trial_df",
    "missing",
)


df[
    "LagV_source"
] = np.where(
    np.isfinite(
        df[
            "LagV_sec"
        ]
    ),
    "trial_df",
    "missing",
)


# ============================================================
# STANDARDIZE MERGE KEYS
# ============================================================

df[
    "subject"
] = (
    df[
        "subject"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

df[
    "trialIndex"
] = pd.to_numeric(
    df[
        "trialIndex"
    ],
    errors="coerce",
)


if (
    trial_metrics_df is not None
    and len(
        trial_metrics_df
    ) > 0
):

    trial_metrics_df = (
        trial_metrics_df.copy()
    )

    if "subject" in trial_metrics_df.columns:

        trial_metrics_df[
            "subject"
        ] = (
            trial_metrics_df[
                "subject"
            ]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    if "trialIndex" in trial_metrics_df.columns:

        trial_metrics_df[
            "trialIndex"
        ] = pd.to_numeric(
            trial_metrics_df[
                "trialIndex"
            ],
            errors="coerce",
        )


# ============================================================
# MERGE LAG VALUES FROM TRIAL_METRICS_DF
# ============================================================

metric_lags = standardize_lag_columns_from_metrics(
    trial_metrics_df
)


if len(
    metric_lags
) > 0:

    df = df.merge(
        metric_lags,
        on=[
            "subject",
            "trialIndex",
        ],
        how="left",
        suffixes=(
            "",
            "_metric",
        ),
    )


    # --------------------------------------------------------
    # Audio
    # --------------------------------------------------------

    if "LagA_sec_metric" in df.columns:

        use_metric_a = (
            ~np.isfinite(
                df[
                    "LagA_sec"
                ]
            )
            &
            np.isfinite(
                df[
                    "LagA_sec_metric"
                ]
            )
        )

        df.loc[
            use_metric_a,
            "LagA_sec",
        ] = df.loc[
            use_metric_a,
            "LagA_sec_metric",
        ]

        df.loc[
            use_metric_a,
            "LagA_source",
        ] = "trial_metrics_df"

        df = df.drop(
            columns=[
                "LagA_sec_metric"
            ]
        )


    # --------------------------------------------------------
    # Visual
    # --------------------------------------------------------

    if "LagV_sec_metric" in df.columns:

        use_metric_v = (
            ~np.isfinite(
                df[
                    "LagV_sec"
                ]
            )
            &
            np.isfinite(
                df[
                    "LagV_sec_metric"
                ]
            )
        )

        df.loc[
            use_metric_v,
            "LagV_sec",
        ] = df.loc[
            use_metric_v,
            "LagV_sec_metric",
        ]

        df.loc[
            use_metric_v,
            "LagV_source",
        ] = "trial_metrics_df"

        df = df.drop(
            columns=[
                "LagV_sec_metric"
            ]
        )


# ============================================================
# INVALIDATE OLD OUT-OF-RANGE LAGS
# ============================================================

for lag_col in [
    "LagA_sec",
    "LagV_sec",
]:

    bad = (
        np.isfinite(
            df[
                lag_col
            ]
        )
        &
        (
            (
                df[
                    lag_col
                ] < 0.0
            )
            |
            (
                df[
                    lag_col
                ] > MAX_LAG_SEC
            )
        )
    )


    if bad.any():

        print(
            f"\nRecomputing {int(bad.sum())} "
            f"out-of-range values in {lag_col}."
        )

        df.loc[
            bad,
            lag_col,
        ] = np.nan


# ============================================================
# COMPUTE MISSING LAGS
# ============================================================

need_audio_lag = (
    ~np.isfinite(
        df[
            "LagA_sec"
        ]
    )
)

need_visual_lag = (
    ~np.isfinite(
        df[
            "LagV_sec"
        ]
    )
)


computed_lag_rows = []


if (
    need_audio_lag.any()
    or need_visual_lag.any()
):

    print(
        "\nSome lag values are missing."
    )

    print(
        "Computing them with non-negative "
        "response-stimulus cross-correlation..."
    )

    print(
        f"Missing audio lags:  "
        f"{int(need_audio_lag.sum())}"
    )

    print(
        f"Missing visual lags: "
        f"{int(need_visual_lag.sum())}"
    )


for idx, row in df.iterrows():

    t = safe_array(
        row[
            "t_grid"
        ]
    )

    resp = safe_array(
        row[
            "mouse_x"
        ]
    )

    aud = safe_array(
        row[
            "audio_x"
        ]
    )

    vis = safe_array(
        row[
            "blob_x"
        ]
    )


    # --------------------------------------------------------
    # Audio lag
    # --------------------------------------------------------

    if not np.isfinite(
        row[
            "LagA_sec"
        ]
    ):

        lag_a_res = estimate_lag_by_xcorr(
            response=resp,
            stimulus=aud,
            t_grid=t,
            max_lag_sec=MAX_LAG_SEC,
        )

        df.at[
            idx,
            "LagA_sec",
        ] = lag_a_res[
            "lag_sec"
        ]

        df.at[
            idx,
            "LagA_source",
        ] = "computed_xcorr_audio"

    else:

        lag_a_res = {

            "lag_sec":
                row[
                    "LagA_sec"
                ],

            "peak_corr":
                np.nan,

            "lag_samples":
                np.nan,
        }


    # --------------------------------------------------------
    # Visual lag
    # --------------------------------------------------------

    if not np.isfinite(
        row[
            "LagV_sec"
        ]
    ):

        lag_v_res = estimate_lag_by_xcorr(
            response=resp,
            stimulus=vis,
            t_grid=t,
            max_lag_sec=MAX_LAG_SEC,
        )

        df.at[
            idx,
            "LagV_sec",
        ] = lag_v_res[
            "lag_sec"
        ]

        df.at[
            idx,
            "LagV_source",
        ] = "computed_xcorr_visual"

    else:

        lag_v_res = {

            "lag_sec":
                row[
                    "LagV_sec"
                ],

            "peak_corr":
                np.nan,

            "lag_samples":
                np.nan,
        }


    computed_lag_rows.append(
        {
            "subject":
                row[
                    "subject"
                ],

            "trialIndex":
                row[
                    "trialIndex"
                ],

            "condition":
                row[
                    "condition_detailed"
                ],

            "LagA_sec":
                df.at[
                    idx,
                    "LagA_sec",
                ],

            "LagV_sec":
                df.at[
                    idx,
                    "LagV_sec",
                ],

            "LagA_source":
                df.at[
                    idx,
                    "LagA_source",
                ],

            "LagV_source":
                df.at[
                    idx,
                    "LagV_source",
                ],

            "computed_audio_peak_corr":
                lag_a_res.get(
                    "peak_corr",
                    np.nan,
                ),

            "computed_visual_peak_corr":
                lag_v_res.get(
                    "peak_corr",
                    np.nan,
                ),
        }
    )


computed_lags_df = pd.DataFrame(
    computed_lag_rows
)


# ============================================================
# SAVE LAG DIAGNOSTICS
# ============================================================

lag_diagnostics_csv = os.path.join(
    CSV_DIR,
    "time_resolved_rmse_lag_diagnostics.csv",
)

computed_lags_df.to_csv(
    lag_diagnostics_csv,
    index=False,
)


print(
    "\nTrials included by condition:"
)

print(
    df[
        "condition_detailed"
    ]
    .value_counts()
    .reindex(
        conditions,
        fill_value=0,
    )
    .to_string()
)


print(
    "\nLag-source counts:"
)

print(
    df[
        [
            "LagA_source",
            "LagV_source",
        ]
    ]
    .value_counts(
        dropna=False
    )
)


print(
    f"\nSaved lag diagnostics:\n"
    f"{lag_diagnostics_csv}"
)


# ============================================================
# TIME GRID
# ============================================================

first_t = np.asarray(
    df.iloc[
        0
    ][
        "t_grid"
    ],
    dtype=float,
)

time_grid = (
    first_t.copy()
)


# ============================================================
# TRIAL-LEVEL TIMEPOINT ERRORS
# ============================================================

trial_error_rows = []


for _, row in df.iterrows():

    subj = row[
        "subject"
    ]

    trial_index = row[
        "trialIndex"
    ]

    cond = row[
        "condition_detailed"
    ]


    t = safe_array(
        row[
            "t_grid"
        ]
    )

    resp = safe_array(
        row[
            "mouse_x"
        ]
    )

    aud = safe_array(
        row[
            "audio_x"
        ]
    )

    vis = safe_array(
        row[
            "blob_x"
        ]
    )


    lag_a = row.get(
        "LagA_sec",
        np.nan,
    )

    lag_v = row.get(
        "LagV_sec",
        np.nan,
    )


    # --------------------------------------------------------
    # Lag-corrected response trajectories
    # --------------------------------------------------------

    resp_a_lagcorr = shift_response_by_lag(
        resp,
        t,
        lag_a,
    )

    resp_v_lagcorr = shift_response_by_lag(
        resp,
        t,
        lag_v,
    )


    # --------------------------------------------------------
    # Squared error
    # --------------------------------------------------------

    audio_sqerr_raw = (
        resp
        -
        aud
    ) ** 2

    visual_sqerr_raw = (
        resp
        -
        vis
    ) ** 2


    audio_sqerr_lagcorr = (
        resp_a_lagcorr
        -
        aud
    ) ** 2

    visual_sqerr_lagcorr = (
        resp_v_lagcorr
        -
        vis
    ) ** 2


    # --------------------------------------------------------
    # Long format
    # --------------------------------------------------------

    n_time = min(
        len(
            t
        ),
        len(
            audio_sqerr_raw
        ),
        len(
            visual_sqerr_raw
        ),
        len(
            audio_sqerr_lagcorr
        ),
        len(
            visual_sqerr_lagcorr
        ),
    )


    for i in range(
        n_time
    ):

        time_sec = t[
            i
        ]


        trial_error_rows.append(
            {
                "subject":
                    subj,

                "trialIndex":
                    trial_index,

                "condition":
                    cond,

                "time_sec":
                    time_sec,

                "LagA_sec":
                    lag_a,

                "LagV_sec":
                    lag_v,

                "LagA_source":
                    row.get(
                        "LagA_source",
                        np.nan,
                    ),

                "LagV_source":
                    row.get(
                        "LagV_source",
                        np.nan,
                    ),

                "audio_sqerr_raw":
                    audio_sqerr_raw[
                        i
                    ],

                "visual_sqerr_raw":
                    visual_sqerr_raw[
                        i
                    ],

                "audio_sqerr_lagcorr":
                    audio_sqerr_lagcorr[
                        i
                    ],

                "visual_sqerr_lagcorr":
                    visual_sqerr_lagcorr[
                        i
                    ],

                "audio_abs_raw":
                    (
                        np.sqrt(
                            audio_sqerr_raw[
                                i
                            ]
                        )
                        if np.isfinite(
                            audio_sqerr_raw[
                                i
                            ]
                        )
                        else np.nan
                    ),

                "visual_abs_raw":
                    (
                        np.sqrt(
                            visual_sqerr_raw[
                                i
                            ]
                        )
                        if np.isfinite(
                            visual_sqerr_raw[
                                i
                            ]
                        )
                        else np.nan
                    ),

                "audio_abs_lagcorr":
                    (
                        np.sqrt(
                            audio_sqerr_lagcorr[
                                i
                            ]
                        )
                        if np.isfinite(
                            audio_sqerr_lagcorr[
                                i
                            ]
                        )
                        else np.nan
                    ),

                "visual_abs_lagcorr":
                    (
                        np.sqrt(
                            visual_sqerr_lagcorr[
                                i
                            ]
                        )
                        if np.isfinite(
                            visual_sqerr_lagcorr[
                                i
                            ]
                        )
                        else np.nan
                    ),
            }
        )


trial_error_df = pd.DataFrame(
    trial_error_rows
)


trial_error_csv = os.path.join(
    CSV_DIR,
    "trial_timepoint_squared_errors_long.csv",
)

trial_error_df.to_csv(
    trial_error_csv,
    index=False,
)


print(
    f"\nSaved trial-level timepoint errors:\n"
    f"{trial_error_csv}"
)


# ============================================================
# SUBJECT × CONDITION × TIME RMSE
# ============================================================

subject_time_df = (
    trial_error_df
    .groupby(
        [
            "subject",
            "condition",
            "time_sec",
        ],
        as_index=False,
    )
    .agg(

        audio_rmse_raw=(
            "audio_sqerr_raw",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        visual_rmse_raw=(
            "visual_sqerr_raw",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        audio_rmse_lagcorr=(
            "audio_sqerr_lagcorr",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        visual_rmse_lagcorr=(
            "visual_sqerr_lagcorr",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        audio_abs_raw=(
            "audio_abs_raw",
            "mean",
        ),

        visual_abs_raw=(
            "visual_abs_raw",
            "mean",
        ),

        audio_abs_lagcorr=(
            "audio_abs_lagcorr",
            "mean",
        ),

        visual_abs_lagcorr=(
            "visual_abs_lagcorr",
            "mean",
        ),

        n_trials=(
            "trialIndex",
            "nunique",
        ),
    )
)


# ============================================================
# VISUAL BIAS
# ============================================================

subject_time_df[
    "rmse_bias_raw_AminusV"
] = (
    subject_time_df[
        "audio_rmse_raw"
    ]
    -
    subject_time_df[
        "visual_rmse_raw"
    ]
)


subject_time_df[
    "rmse_bias_lagcorr_AminusV"
] = (
    subject_time_df[
        "audio_rmse_lagcorr"
    ]
    -
    subject_time_df[
        "visual_rmse_lagcorr"
    ]
)


subject_time_csv = os.path.join(
    CSV_DIR,
    "subject_condition_time_resolved_rmse.csv",
)

subject_time_df.to_csv(
    subject_time_csv,
    index=False,
)


print(
    f"Saved subject-level time-resolved RMSE:\n"
    f"{subject_time_csv}"
)


# ============================================================
# GROUP-LEVEL TIME-RESOLVED RMSE
# ============================================================

group_rows = []


for cond in conditions:

    sub_cond = subject_time_df[
        subject_time_df[
            "condition"
        ] == cond
    ].copy()


    for time_sec, sub_time in sub_cond.groupby(
        "time_sec"
    ):

        row = {

            "condition":
                cond,

            "time_sec":
                time_sec,

            "n_subjects":
                sub_time[
                    "subject"
                ].nunique(),
        }


        for metric in [

            "audio_rmse_raw",
            "visual_rmse_raw",

            "audio_rmse_lagcorr",
            "visual_rmse_lagcorr",

            "rmse_bias_raw_AminusV",
            "rmse_bias_lagcorr_AminusV",
        ]:

            vals = (
                sub_time[
                    metric
                ]
                .astype(
                    float
                )
                .values
            )


            mean, se, ci95, n = mean_sem_ci(
                vals,
                axis=0,
            )


            row[
                f"{metric}_mean"
            ] = mean

            row[
                f"{metric}_sem"
            ] = se

            row[
                f"{metric}_ci95"
            ] = ci95

            row[
                f"{metric}_n"
            ] = n


        group_rows.append(
            row
        )


group_time_df = pd.DataFrame(
    group_rows
)


group_time_csv = os.path.join(
    CSV_DIR,
    "group_condition_time_resolved_rmse.csv",
)

group_time_df.to_csv(
    group_time_csv,
    index=False,
)


print(
    f"Saved group-level time-resolved RMSE:\n"
    f"{group_time_csv}"
)


# ============================================================
# PARTICIPANT PLOTS
# ============================================================

for subj in sorted(
    subject_time_df[
        "subject"
    ].unique()
):

    sdf = subject_time_df[
        subject_time_df[
            "subject"
        ] == subj
    ].copy()


    for metric_type in [
        "raw",
        "lagcorr",
    ]:

        fig, axes = plt.subplots(
            2,
            3,
            figsize=(
                17,
                8,
            ),
            sharex=True,
            sharey=False,
        )

        axes = axes.flatten()


        for ax, cond in zip(
            axes,
            conditions,
        ):

            cdf = (
                sdf[
                    sdf[
                        "condition"
                    ] == cond
                ]
                .sort_values(
                    "time_sec"
                )
            )


            if len(
                cdf
            ) == 0:

                ax.axis(
                    "off"
                )

                continue


            if metric_type == "raw":

                audio_col = (
                    "audio_rmse_raw"
                )

                visual_col = (
                    "visual_rmse_raw"
                )

                title_suffix = (
                    "Raw RMSE"
                )

            else:

                audio_col = (
                    "audio_rmse_lagcorr"
                )

                visual_col = (
                    "visual_rmse_lagcorr"
                )

                title_suffix = (
                    "Lag-corrected RMSE"
                )


            ax.plot(
                cdf[
                    "time_sec"
                ],
                cdf[
                    audio_col
                ],
                linewidth=2,
                label="Audio RMSE",
            )

            ax.plot(
                cdf[
                    "time_sec"
                ],
                cdf[
                    visual_col
                ],
                linewidth=2,
                label="Visual RMSE",
            )


            if "sweep" in cond:

                ax.axvline(
                    12,
                    linestyle="--",
                    linewidth=1,
                    alpha=0.7,
                )


            ax.set_title(
                f"{cond}: {title_suffix}"
            )

            ax.set_xlabel(
                "Time (s)"
            )

            ax.set_ylabel(
                "RMSE (deg)"
            )

            ax.legend(
                fontsize=8
            )


        fig.suptitle(
            (
                f"{subj}: time-resolved "
                f"{title_suffix} to audio and visual"
            ),
            fontsize=15,
            y=1.02,
        )


        outpath = os.path.join(
            SUBJECT_PLOT_DIR,
            (
                f"{subj}_time_resolved_"
                f"{metric_type}_audio_visual_rmse.png"
            ),
        )


        save_fig(
            fig,
            outpath,
        )


print(
    f"Saved participant plots to:\n"
    f"{SUBJECT_PLOT_DIR}"
)


# ============================================================
# GROUP PLOTS — AUDIO AND VISUAL RMSE
# ============================================================

for metric_type in [
    "raw",
    "lagcorr",
]:

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(
            17,
            8,
        ),
        sharex=True,
        sharey=False,
    )

    axes = axes.flatten()


    for ax, cond in zip(
        axes,
        conditions,
    ):

        cdf = (
            group_time_df[
                group_time_df[
                    "condition"
                ] == cond
            ]
            .sort_values(
                "time_sec"
            )
        )


        if len(
            cdf
        ) == 0:

            ax.axis(
                "off"
            )

            continue


        if metric_type == "raw":

            audio_mean = (
                "audio_rmse_raw_mean"
            )

            audio_ci = (
                "audio_rmse_raw_ci95"
            )

            visual_mean = (
                "visual_rmse_raw_mean"
            )

            visual_ci = (
                "visual_rmse_raw_ci95"
            )

            title_suffix = (
                "Raw RMSE"
            )

        else:

            audio_mean = (
                "audio_rmse_lagcorr_mean"
            )

            audio_ci = (
                "audio_rmse_lagcorr_ci95"
            )

            visual_mean = (
                "visual_rmse_lagcorr_mean"
            )

            visual_ci = (
                "visual_rmse_lagcorr_ci95"
            )

            title_suffix = (
                "Lag-corrected RMSE"
            )


        x = cdf[
            "time_sec"
        ].values.astype(
            float
        )

        a_mean = cdf[
            audio_mean
        ].values.astype(
            float
        )

        a_ci = cdf[
            audio_ci
        ].values.astype(
            float
        )

        v_mean = cdf[
            visual_mean
        ].values.astype(
            float
        )

        v_ci = cdf[
            visual_ci
        ].values.astype(
            float
        )


        ax.plot(
            x,
            a_mean,
            linewidth=2.3,
            label="Audio RMSE",
        )

        ax.fill_between(
            x,
            a_mean - a_ci,
            a_mean + a_ci,
            alpha=0.18,
        )


        ax.plot(
            x,
            v_mean,
            linewidth=2.3,
            label="Visual RMSE",
        )

        ax.fill_between(
            x,
            v_mean - v_ci,
            v_mean + v_ci,
            alpha=0.18,
        )


        if "sweep" in cond:

            ax.axvline(
                12,
                linestyle="--",
                linewidth=1,
                alpha=0.7,
            )


        ax.set_title(
            f"{cond}: {title_suffix}"
        )

        ax.set_xlabel(
            "Time (s)"
        )

        ax.set_ylabel(
            "RMSE (deg)"
        )

        ax.legend(
            fontsize=8
        )


    fig.suptitle(
        (
            f"Group-level time-resolved "
            f"{title_suffix}: audio vs visual"
        ),
        fontsize=15,
        y=1.02,
    )


    outpath = os.path.join(
        GROUP_PLOT_DIR,
        (
            f"group_time_resolved_{metric_type}_"
            f"audio_visual_rmse.png"
        ),
    )


    save_fig(
        fig,
        outpath,
    )


# ============================================================
# GROUP PLOTS — VISUAL BIAS
# ============================================================

for metric_type in [
    "raw",
    "lagcorr",
]:

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(
            17,
            8,
        ),
        sharex=True,
        sharey=False,
    )

    axes = axes.flatten()


    for ax, cond in zip(
        axes,
        conditions,
    ):

        cdf = (
            group_time_df[
                group_time_df[
                    "condition"
                ] == cond
            ]
            .sort_values(
                "time_sec"
            )
        )


        if len(
            cdf
        ) == 0:

            ax.axis(
                "off"
            )

            continue


        if metric_type == "raw":

            mean_col = (
                "rmse_bias_raw_AminusV_mean"
            )

            ci_col = (
                "rmse_bias_raw_AminusV_ci95"
            )

            title_suffix = (
                "Raw Bias"
            )

        else:

            mean_col = (
                "rmse_bias_lagcorr_AminusV_mean"
            )

            ci_col = (
                "rmse_bias_lagcorr_AminusV_ci95"
            )

            title_suffix = (
                "Lag-corrected Bias"
            )


        x = cdf[
            "time_sec"
        ].values.astype(
            float
        )

        y = cdf[
            mean_col
        ].values.astype(
            float
        )

        ci = cdf[
            ci_col
        ].values.astype(
            float
        )


        ax.plot(
            x,
            y,
            linewidth=2.3,
            label="Audio RMSE - Visual RMSE",
        )

        ax.fill_between(
            x,
            y - ci,
            y + ci,
            alpha=0.18,
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1,
            alpha=0.7,
        )


        if "sweep" in cond:

            ax.axvline(
                12,
                linestyle="--",
                linewidth=1,
                alpha=0.7,
            )


        ax.set_title(
            f"{cond}: {title_suffix}"
        )

        ax.set_xlabel(
            "Time (s)"
        )

        ax.set_ylabel(
            "Bias: Audio RMSE - Visual RMSE"
        )

        ax.legend(
            fontsize=8
        )


    fig.suptitle(
        (
            f"Group-level time-resolved "
            f"RMSE bias: {title_suffix}"
        ),
        fontsize=15,
        y=1.02,
    )


    outpath = os.path.join(
        GROUP_PLOT_DIR,
        (
            f"group_time_resolved_{metric_type}_"
            f"rmse_bias_AminusV.png"
        ),
    )


    save_fig(
        fig,
        outpath,
    )


print(
    f"Saved group plots to:\n"
    f"{GROUP_PLOT_DIR}"
)


# ============================================================
# STABLE-WINDOW ANALYSIS
# ============================================================

STABLE_START_SEC = 0.5
STABLE_END_SEC = 17.5


STABLE_OUT_DIR = os.path.join(
    OUT_DIR,
    "stable_0p5_17p5_outputs",
)

STABLE_CSV_DIR = os.path.join(
    STABLE_OUT_DIR,
    "csv_outputs",
)

STABLE_FIG_DIR = os.path.join(
    STABLE_OUT_DIR,
    "figures",
)


for d in [
    STABLE_OUT_DIR,
    STABLE_CSV_DIR,
    STABLE_FIG_DIR,
]:

    os.makedirs(
        d,
        exist_ok=True,
    )


# ============================================================
# STABLE-WINDOW HELPERS
# ============================================================

def _mean_ci(
    vals,
    alpha=0.05,
):

    vals = (
        pd.Series(
            vals
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .astype(
            float
        )
        .values
    )

    n = len(
        vals
    )


    if n < 2:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            n,
        )


    m = float(
        np.mean(
            vals
        )
    )

    se = float(
        np.std(
            vals,
            ddof=1,
        )
        /
        np.sqrt(
            n
        )
    )

    crit = float(
        t_dist.ppf(
            1 - alpha / 2,
            df=n - 1,
        )
    )


    return (
        m,
        m - crit * se,
        m + crit * se,
        se,
        n,
    )


def _one_sample_test(
    vals,
    popmean=0.0,
):

    vals = (
        pd.Series(
            vals
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .astype(
            float
        )
        .values
    )


    n = len(
        vals
    )


    if n < 2:

        return {
            "n_subjects":
                n,

            "mean":
                np.nan,

            "se":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "t":
                np.nan,

            "df":
                np.nan,

            "p":
                np.nan,
        }


    m, lo, hi, se, n = _mean_ci(
        vals
    )

    test = ttest_1samp(
        vals,
        popmean,
    )


    return {

        "n_subjects":
            n,

        "mean":
            m,

        "se":
            se,

        "ci_low":
            lo,

        "ci_high":
            hi,

        "t":
            float(
                test.statistic
            ),

        "df":
            n - 1,

        "p":
            float(
                test.pvalue
            ),
    }


# ============================================================
# FILTER STABLE WINDOW
# ============================================================

stable_trial_error_df = (
    trial_error_df
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .query(
        "time_sec >= @STABLE_START_SEC "
        "and time_sec <= @STABLE_END_SEC"
    )
    .copy()
)


if len(
    stable_trial_error_df
) == 0:

    raise RuntimeError(
        "No rows found in the 0.5–17.5 s stable window. "
        "Check that time_sec is expressed in seconds."
    )


stable_trial_error_csv = os.path.join(
    STABLE_CSV_DIR,
    "stable_0p5_17p5_trial_timepoint_errors.csv",
)

stable_trial_error_df.to_csv(
    stable_trial_error_csv,
    index=False,
)


print(
    "\nStable-window rows:"
)

print(
    f"Rows: {len(stable_trial_error_df)}"
)

print(
    f"Time range: "
    f"{stable_trial_error_df['time_sec'].min():.3f}–"
    f"{stable_trial_error_df['time_sec'].max():.3f} s"
)

print(
    f"Unique timepoints: "
    f"{stable_trial_error_df['time_sec'].nunique()}"
)


# ============================================================
# SUBJECT × CONDITION STABLE-WINDOW SUMMARY
# ============================================================

stable_subject_condition = (
    stable_trial_error_df
    .groupby(
        [
            "subject",
            "condition",
        ],
        as_index=False,
    )
    .agg(

        audio_rmse_raw=(
            "audio_sqerr_raw",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        visual_rmse_raw=(
            "visual_sqerr_raw",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        audio_rmse_lagcorr=(
            "audio_sqerr_lagcorr",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        visual_rmse_lagcorr=(
            "visual_sqerr_lagcorr",
            lambda x:
                np.sqrt(
                    np.nanmean(
                        x
                    )
                ),
        ),

        n_trials=(
            "trialIndex",
            "nunique",
        ),
    )
)


stable_subject_condition[
    "rmse_bias_raw_AminusV"
] = (
    stable_subject_condition[
        "audio_rmse_raw"
    ]
    -
    stable_subject_condition[
        "visual_rmse_raw"
    ]
)


stable_subject_condition[
    "rmse_bias_lagcorr_AminusV"
] = (
    stable_subject_condition[
        "audio_rmse_lagcorr"
    ]
    -
    stable_subject_condition[
        "visual_rmse_lagcorr"
    ]
)


stable_subject_csv = os.path.join(
    STABLE_CSV_DIR,
    "subject_condition_stable_window_rmse.csv",
)

stable_subject_condition.to_csv(
    stable_subject_csv,
    index=False,
)


# ============================================================
# GROUP INFERENCE
# ============================================================

stable_group_rows = []


for cond in conditions:

    cdf = stable_subject_condition[
        stable_subject_condition[
            "condition"
        ] == cond
    ]


    for metric in [
        "rmse_bias_raw_AminusV",
        "rmse_bias_lagcorr_AminusV",
    ]:

        result = _one_sample_test(
            cdf[
                metric
            ],
            popmean=0.0,
        )


        stable_group_rows.append(
            {
                "condition":
                    cond,

                "metric":
                    metric,

                **result,
            }
        )


stable_group_results = pd.DataFrame(
    stable_group_rows
)


stable_group_csv = os.path.join(
    STABLE_CSV_DIR,
    "group_stable_window_bias_tests.csv",
)

stable_group_results.to_csv(
    stable_group_csv,
    index=False,
)


print(
    "\n"
    + "=" * 80
)

print(
    "STABLE-WINDOW VISUAL-BIAS RESULTS"
)

print(
    "=" * 80
)


print(
    stable_group_results.to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "TIME-RESOLVED RMSE / VISUAL-BIAS ANALYSIS COMPLETE"
)

print(
    "=" * 80
)


print(
    "\nKey output files:"
)

print(
    f"  {lag_diagnostics_csv}"
)

print(
    f"  {trial_error_csv}"
)

print(
    f"  {subject_time_csv}"
)

print(
    f"  {group_time_csv}"
)

print(
    f"  {stable_subject_csv}"
)

print(
    f"  {stable_group_csv}"
)

Using existing trial_metrics_df from memory.

ORIGINAL TRIAL_DF CONDITION STRUCTURE

trialType:
trialType
incongruent_spatial    4000
incongruent_sweep      1000
visual_only             500
congruent               500
audio_only              500

condLabel:
condLabel
visual_only             500
spatial_20_right        500
spatial_40_right        500
spatial_40_left         500
congruent               500
spatial_30_left         500
spatial_10_right        500
spatial_20_left         500
spatial_30_right        500
audio_only              500
spatial_10_left         500
sweep_converge_right    250
sweep_diverge_left      250
sweep_converge_left     250
sweep_diverge_right     250

offsetDeg:
offsetDeg
40.0    2000
0.0     1500
20.0    1000
30.0    1000
10.0    1000

rampType:
rampType
none        5500
converge     500
diverge      500

CONDITION MAPPING RESULTS

Mapped condition counts:
condition_detailed
NaN               1500
offset_20         1000
offset_40         1000
offset_30    

In [12]:
# %% ADD-ON: STABLE WINDOW SUMMARY AND INFERENCE FOR TIME-RESOLVED RMSE BIAS
# ============================================================
# Stable-window inferential analysis.
#
# Window:
#   0.5–17.5 s
#
# Main outcome:
#   RMSE bias AminusV = Audio RMSE - Visual RMSE
#
# Interpretation:
#   Positive bias = response is closer to visual trajectory than audio trajectory
#   Negative bias = response is closer to auditory trajectory than visual trajectory
#
# This edited version fixes FileNotFoundError during CSV saving by:
#   1. Recreating output folders immediately before each save.
#   2. Using a safe_to_csv() helper.
#   3. Falling back to CSV_DIR if the nested stable-output folder fails.
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_1samp, ttest_rel, t as t_dist

# -------------------------
# REQUIRED DATA CHECKS
# -------------------------
if "trial_error_df" not in globals():
    raise RuntimeError(
        "trial_error_df not found. Run the main time-resolved RMSE/bias analysis first. "
        "This add-on expects trial_error_df to contain trial-level timepoint errors."
    )

if "CSV_DIR" not in globals():
    raise RuntimeError("CSV_DIR not found. Run the main time-resolved RMSE/bias analysis first.")

try:
    OUT_DIR
except NameError:
    OUT_DIR = os.path.abspath(os.path.join(CSV_DIR, ".."))

try:
    FIG_DPI
except NameError:
    FIG_DPI = 180

# -------------------------
# STABLE WINDOW SETTINGS
# -------------------------
STABLE_START_SEC = 0.5
STABLE_END_SEC = 17.5

# -------------------------
# SAFE STABLE-WINDOW OUTPUT PATHS
# -------------------------
# Use shorter, flatter folders inside already-working output directories.
# This avoids intermittent OneDrive failures with deeply nested paths.

STABLE_OUT_DIR = OUT_DIR
STABLE_CSV_DIR = os.path.join(CSV_DIR, "stable_0p5_17p5_csv")
STABLE_FIG_DIR = os.path.join(GROUP_PLOT_DIR, "stable_0p5_17p5_figures")

# Extra-simple fallback: save directly into CSV_DIR if the stable subfolder fails.
STABLE_FALLBACK_CSV_DIR = CSV_DIR
STABLE_FALLBACK_FIG_DIR = GROUP_PLOT_DIR

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

for d in [STABLE_CSV_DIR, STABLE_FIG_DIR, STABLE_FALLBACK_CSV_DIR, STABLE_FALLBACK_FIG_DIR]:
    ensure_dir(d)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return path

def safe_to_csv(df, filepath, index=False):
    """
    Save CSV robustly.

    First tries the intended filepath.
    If that fails for any reason, saves to a flat fallback folder inside CSV_DIR.
    """
    filepath = os.path.abspath(filepath)
    parent = os.path.dirname(filepath)

    try:
        os.makedirs(parent, exist_ok=True)
        df.to_csv(filepath, index=index)
        return filepath

    except Exception as e1:
        print("\nWARNING: Could not save to intended path:")
        print(filepath)
        print(f"Reason: {repr(e1)}")
        print("Trying flat fallback path inside CSV_DIR...")

        fallback_parent = os.path.abspath(STABLE_FALLBACK_CSV_DIR)
        os.makedirs(fallback_parent, exist_ok=True)

        fallback_path = os.path.join(fallback_parent, os.path.basename(filepath))

        try:
            df.to_csv(fallback_path, index=index)
            print(f"Saved fallback CSV: {fallback_path}")
            return fallback_path

        except Exception as e2:
            print("\nERROR: Fallback save also failed.")
            print(f"Fallback path: {fallback_path}")
            print(f"Fallback reason: {repr(e2)}")
            raise

def safe_save_fig(fig, filepath):
    """
    Save figure robustly.

    First tries the intended filepath.
    If that fails, saves to a flat fallback folder inside GROUP_PLOT_DIR.
    """
    filepath = os.path.abspath(filepath)
    parent = os.path.dirname(filepath)

    try:
        os.makedirs(parent, exist_ok=True)
        fig.tight_layout()
        fig.savefig(filepath, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        return filepath

    except Exception as e1:
        print("\nWARNING: Could not save figure to intended path:")
        print(filepath)
        print(f"Reason: {repr(e1)}")
        print("Trying flat fallback path inside GROUP_PLOT_DIR...")

        fallback_parent = os.path.abspath(STABLE_FALLBACK_FIG_DIR)
        os.makedirs(fallback_parent, exist_ok=True)

        fallback_path = os.path.join(fallback_parent, os.path.basename(filepath))

        try:
            fig.tight_layout()
            fig.savefig(fallback_path, dpi=FIG_DPI, bbox_inches="tight")
            plt.close(fig)
            print(f"Saved fallback figure: {fallback_path}")
            return fallback_path

        except Exception as e2:
            print("\nERROR: Fallback figure save also failed.")
            print(f"Fallback path: {fallback_path}")
            print(f"Fallback reason: {repr(e2)}")
            plt.close(fig)
            raise

for d in [STABLE_OUT_DIR, STABLE_CSV_DIR, STABLE_FIG_DIR, STABLE_FALLBACK_CSV_DIR, STABLE_FALLBACK_FIG_DIR]:
    ensure_dir(d)

if "conditions" not in globals():
    conditions = [
        "offset_10",
        "offset_20",
        "offset_30",
        "offset_40",
        "sweep_diverge",
        "sweep_converge",
    ]

# -------------------------
# HELPER FUNCTIONS
# -------------------------
def _sem(vals):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    if len(vals) < 2:
        return np.nan
    return float(np.std(vals, ddof=1) / np.sqrt(len(vals)))

def _mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, np.nan, n

    m = float(np.mean(vals))
    se = float(np.std(vals, ddof=1) / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    return m, m - crit * se, m + crit * se, se, n

def _one_sample_test(vals, popmean=0.0, alpha=0.05):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return {
            "n_subjects": n,
            "mean": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
        }

    m, lo, hi, se, n = _mean_ci(vals, alpha=alpha)
    test = ttest_1samp(vals, popmean)

    return {
        "n_subjects": n,
        "mean": m,
        "se": se,
        "ci_low": lo,
        "ci_high": hi,
        "t": float(test.statistic),
        "df": n - 1,
        "p": float(test.pvalue),
    }

def _paired_test(x, y, alpha=0.05):
    tmp = (
        pd.DataFrame({"x": x, "y": y})
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    n = len(tmp)

    if n < 2:
        return {
            "n_subjects": n,
            "mean_x": np.nan,
            "mean_y": np.nan,
            "mean_diff_x_minus_y": np.nan,
            "se_diff": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
        }

    diff = tmp["x"].astype(float).values - tmp["y"].astype(float).values
    m, lo, hi, se, n = _mean_ci(diff, alpha=alpha)
    test = ttest_rel(tmp["x"], tmp["y"])

    return {
        "n_subjects": n,
        "mean_x": float(tmp["x"].mean()),
        "mean_y": float(tmp["y"].mean()),
        "mean_diff_x_minus_y": m,
        "se_diff": se,
        "ci_low": lo,
        "ci_high": hi,
        "t": float(test.statistic),
        "df": n - 1,
        "p": float(test.pvalue),
    }

def _condition_label(cond):
    label_map = {
        "offset_10": "10° offset",
        "offset_20": "20° offset",
        "offset_30": "30° offset",
        "offset_40": "40° offset",
        "sweep_diverge": "Sweep diverge",
        "sweep_converge": "Sweep converge",
    }
    return label_map.get(cond, str(cond))

def _condition_sort_key(cond):
    order = {
        "offset_10": 1,
        "offset_20": 2,
        "offset_30": 3,
        "offset_40": 4,
        "sweep_diverge": 5,
        "sweep_converge": 6,
    }
    return order.get(cond, 999)

# -------------------------
# FILTER TO STABLE WINDOW
# -------------------------
stable_trial_error_df = (
    trial_error_df
    .replace([np.inf, -np.inf], np.nan)
    .query("time_sec >= @STABLE_START_SEC and time_sec <= @STABLE_END_SEC")
    .copy()
)

if len(stable_trial_error_df) == 0:
    raise RuntimeError(
        "No rows found in the 0.5–17.5 s stable window. "
        "Check that trial_error_df['time_sec'] is in seconds."
    )

stable_trial_error_csv = safe_to_csv(
    stable_trial_error_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_trial_timepoint_errors.csv"),
    index=False
)

print("\nStable-window rows:")
print(f"Rows: {len(stable_trial_error_df)}")
print(f"Time range: {stable_trial_error_df['time_sec'].min():.3f}–{stable_trial_error_df['time_sec'].max():.3f} s")
print(f"Unique timepoints: {stable_trial_error_df['time_sec'].nunique()}")
print(f"Saved stable trial-timepoint CSV: {stable_trial_error_csv}")

# -------------------------
# SUBJECT × CONDITION STABLE-WINDOW SUMMARY
# -------------------------
stable_subject_condition_df = (
    stable_trial_error_df
    .groupby(["subject", "condition"], as_index=False)
    .agg(
        n_trials=("trialIndex", "nunique"),
        n_timepoint_rows=("time_sec", "count"),
        n_timepoints=("time_sec", "nunique"),

        audio_rmse_raw_stable=("audio_sqerr_raw", lambda x: np.sqrt(np.nanmean(x))),
        visual_rmse_raw_stable=("visual_sqerr_raw", lambda x: np.sqrt(np.nanmean(x))),
        audio_rmse_lagcorr_stable=("audio_sqerr_lagcorr", lambda x: np.sqrt(np.nanmean(x))),
        visual_rmse_lagcorr_stable=("visual_sqerr_lagcorr", lambda x: np.sqrt(np.nanmean(x))),

        audio_abs_raw_stable=("audio_abs_raw", "mean"),
        visual_abs_raw_stable=("visual_abs_raw", "mean"),
        audio_abs_lagcorr_stable=("audio_abs_lagcorr", "mean"),
        visual_abs_lagcorr_stable=("visual_abs_lagcorr", "mean"),
    )
)

stable_subject_condition_df["rmse_bias_raw_AminusV_stable"] = (
    stable_subject_condition_df["audio_rmse_raw_stable"]
    - stable_subject_condition_df["visual_rmse_raw_stable"]
)

stable_subject_condition_df["rmse_bias_lagcorr_AminusV_stable"] = (
    stable_subject_condition_df["audio_rmse_lagcorr_stable"]
    - stable_subject_condition_df["visual_rmse_lagcorr_stable"]
)

stable_subject_condition_df["abs_bias_raw_AminusV_stable"] = (
    stable_subject_condition_df["audio_abs_raw_stable"]
    - stable_subject_condition_df["visual_abs_raw_stable"]
)

stable_subject_condition_df["abs_bias_lagcorr_AminusV_stable"] = (
    stable_subject_condition_df["audio_abs_lagcorr_stable"]
    - stable_subject_condition_df["visual_abs_lagcorr_stable"]
)

stable_subject_condition_df["condition_sort"] = stable_subject_condition_df["condition"].map(_condition_sort_key)

stable_subject_condition_df = (
    stable_subject_condition_df
    .sort_values(["condition_sort", "subject"])
    .drop(columns=["condition_sort"])
)

stable_subject_condition_csv = safe_to_csv(
    stable_subject_condition_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_subject_condition_rmse_bias.csv"),
    index=False
)

# -------------------------
# GROUP ESTIMATES BY CONDITION
# -------------------------
estimate_metrics = [
    "audio_rmse_raw_stable",
    "visual_rmse_raw_stable",
    "audio_rmse_lagcorr_stable",
    "visual_rmse_lagcorr_stable",
    "rmse_bias_raw_AminusV_stable",
    "rmse_bias_lagcorr_AminusV_stable",
    "audio_abs_raw_stable",
    "visual_abs_raw_stable",
    "audio_abs_lagcorr_stable",
    "visual_abs_lagcorr_stable",
    "abs_bias_raw_AminusV_stable",
    "abs_bias_lagcorr_AminusV_stable",
]

group_estimate_rows = []

for condition, sub in stable_subject_condition_df.groupby("condition"):
    for metric in estimate_metrics:
        m, lo, hi, se, n = _mean_ci(sub[metric])

        group_estimate_rows.append({
            "analysis": "stable_0p5_17p5_group_estimate",
            "window_start_sec": STABLE_START_SEC,
            "window_end_sec": STABLE_END_SEC,
            "condition": condition,
            "condition_label": _condition_label(condition),
            "outcome": metric,
            "n_subjects": n,
            "mean": m,
            "se": se,
            "ci_low": lo,
            "ci_high": hi,
        })

stable_group_estimates_df = pd.DataFrame(group_estimate_rows)
stable_group_estimates_df["condition_sort"] = stable_group_estimates_df["condition"].map(_condition_sort_key)
stable_group_estimates_df = (
    stable_group_estimates_df
    .sort_values(["condition_sort", "outcome"])
    .drop(columns=["condition_sort"])
)

stable_group_estimates_csv = safe_to_csv(
    stable_group_estimates_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_group_estimates.csv"),
    index=False
)

# -------------------------
# ONE-SAMPLE TESTS OF BIAS AGAINST ZERO
# -------------------------
bias_test_metrics = [
    "rmse_bias_raw_AminusV_stable",
    "rmse_bias_lagcorr_AminusV_stable",
    "abs_bias_raw_AminusV_stable",
    "abs_bias_lagcorr_AminusV_stable",
]

bias_test_rows = []

for condition, sub in stable_subject_condition_df.groupby("condition"):
    for metric in bias_test_metrics:
        stats = _one_sample_test(sub[metric], popmean=0.0)

        bias_test_rows.append({
            "analysis": "stable_0p5_17p5_bias_vs_zero",
            "window_start_sec": STABLE_START_SEC,
            "window_end_sec": STABLE_END_SEC,
            "condition": condition,
            "condition_label": _condition_label(condition),
            "outcome": metric,
            **stats,
        })

stable_bias_tests_df = pd.DataFrame(bias_test_rows)
stable_bias_tests_df["condition_sort"] = stable_bias_tests_df["condition"].map(_condition_sort_key)
stable_bias_tests_df = (
    stable_bias_tests_df
    .sort_values(["condition_sort", "outcome"])
    .drop(columns=["condition_sort"])
)

stable_bias_tests_csv = safe_to_csv(
    stable_bias_tests_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_bias_vs_zero_tests.csv"),
    index=False
)

# -------------------------
# PAIRED AUDIO-RMSE VS VISUAL-RMSE TESTS
# -------------------------
paired_test_specs = [
    {
        "analysis": "stable_0p5_17p5_raw_audio_vs_visual_rmse",
        "metric_type": "raw",
        "audio_col": "audio_rmse_raw_stable",
        "visual_col": "visual_rmse_raw_stable",
    },
    {
        "analysis": "stable_0p5_17p5_lagcorr_audio_vs_visual_rmse",
        "metric_type": "lagcorr",
        "audio_col": "audio_rmse_lagcorr_stable",
        "visual_col": "visual_rmse_lagcorr_stable",
    },
    {
        "analysis": "stable_0p5_17p5_raw_audio_vs_visual_abs_error",
        "metric_type": "raw_abs",
        "audio_col": "audio_abs_raw_stable",
        "visual_col": "visual_abs_raw_stable",
    },
    {
        "analysis": "stable_0p5_17p5_lagcorr_audio_vs_visual_abs_error",
        "metric_type": "lagcorr_abs",
        "audio_col": "audio_abs_lagcorr_stable",
        "visual_col": "visual_abs_lagcorr_stable",
    },
]

paired_test_rows = []

for condition, sub in stable_subject_condition_df.groupby("condition"):
    for spec in paired_test_specs:
        stats = _paired_test(sub[spec["audio_col"]], sub[spec["visual_col"]])

        paired_test_rows.append({
            "analysis": spec["analysis"],
            "window_start_sec": STABLE_START_SEC,
            "window_end_sec": STABLE_END_SEC,
            "condition": condition,
            "condition_label": _condition_label(condition),
            "metric_type": spec["metric_type"],
            "audio_col": spec["audio_col"],
            "visual_col": spec["visual_col"],
            **stats,
        })

stable_audio_visual_paired_tests_df = pd.DataFrame(paired_test_rows)
stable_audio_visual_paired_tests_df["condition_sort"] = stable_audio_visual_paired_tests_df["condition"].map(_condition_sort_key)
stable_audio_visual_paired_tests_df = (
    stable_audio_visual_paired_tests_df
    .sort_values(["condition_sort", "analysis"])
    .drop(columns=["condition_sort"])
)

stable_audio_visual_paired_tests_csv = safe_to_csv(
    stable_audio_visual_paired_tests_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_audio_vs_visual_paired_tests.csv"),
    index=False
)

# -------------------------
# FIXED-OFFSET ONLY: LINEAR OFFSET TREND TESTS
# -------------------------
offset_df = stable_subject_condition_df[
    stable_subject_condition_df["condition"].isin(["offset_10", "offset_20", "offset_30", "offset_40"])
].copy()

offset_df["offset_deg"] = (
    offset_df["condition"]
    .astype(str)
    .str.extract(r"offset_(\d+)", expand=False)
    .astype(float)
)

offset_trend_rows = []

if len(offset_df) > 0:
    try:
        import statsmodels.formula.api as smf

        for metric in [
            "rmse_bias_raw_AminusV_stable",
            "rmse_bias_lagcorr_AminusV_stable",
            "abs_bias_raw_AminusV_stable",
            "abs_bias_lagcorr_AminusV_stable",
        ]:
            d = offset_df.dropna(subset=[metric, "offset_deg", "subject"]).copy()

            if len(d) >= 20 and d["subject"].nunique() >= 3:
                d["offset_c"] = d["offset_deg"] - d["offset_deg"].mean()

                try:
                    model = smf.mixedlm(
                        f"{metric} ~ offset_c",
                        data=d,
                        groups=d["subject"],
                        re_formula="1"
                    )
                    result = model.fit(method="powell", reml=False, maxiter=1000, disp=False)
                    model_type = "mixedlm_random_intercept"

                    estimate = float(result.fe_params["offset_c"])
                    se = float(result.bse["offset_c"])
                    stat = float(result.tvalues["offset_c"])
                    p = float(result.pvalues["offset_c"])

                except Exception as e:
                    result = smf.ols(
                        f"{metric} ~ offset_c",
                        data=d
                    ).fit(
                        cov_type="cluster",
                        cov_kwds={"groups": d["subject"]}
                    )
                    model_type = f"clustered_ols_fallback_after_mixedlm_error: {str(e)}"

                    estimate = float(result.params["offset_c"])
                    se = float(result.bse["offset_c"])
                    stat = float(result.tvalues["offset_c"])
                    p = float(result.pvalues["offset_c"])

                offset_trend_rows.append({
                    "analysis": "stable_0p5_17p5_fixed_offset_linear_trend",
                    "window_start_sec": STABLE_START_SEC,
                    "window_end_sec": STABLE_END_SEC,
                    "outcome": metric,
                    "predictor": "offset_c",
                    "model_type": model_type,
                    "n_rows": len(d),
                    "n_subjects": d["subject"].nunique(),
                    "estimate_per_degree": estimate,
                    "se": se,
                    "z_or_t": stat,
                    "p": p,
                })

    except ImportError:
        print("statsmodels not available; skipping fixed-offset linear trend tests.")

stable_offset_trend_tests_df = pd.DataFrame(offset_trend_rows)

stable_offset_trend_tests_csv = safe_to_csv(
    stable_offset_trend_tests_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_fixed_offset_linear_trend_tests.csv"),
    index=False
)

# -------------------------
# SUBJECT-FIRST COLLAPSE ACROSS FIXED OFFSETS
# -------------------------
if len(offset_df) > 0:
    stable_subject_fixed_overall_df = (
        offset_df
        .groupby("subject", as_index=False)
        .agg(
            n_offset_conditions=("condition", "nunique"),
            n_trials=("n_trials", "sum"),

            audio_rmse_raw_stable=("audio_rmse_raw_stable", "mean"),
            visual_rmse_raw_stable=("visual_rmse_raw_stable", "mean"),
            audio_rmse_lagcorr_stable=("audio_rmse_lagcorr_stable", "mean"),
            visual_rmse_lagcorr_stable=("visual_rmse_lagcorr_stable", "mean"),

            rmse_bias_raw_AminusV_stable=("rmse_bias_raw_AminusV_stable", "mean"),
            rmse_bias_lagcorr_AminusV_stable=("rmse_bias_lagcorr_AminusV_stable", "mean"),

            abs_bias_raw_AminusV_stable=("abs_bias_raw_AminusV_stable", "mean"),
            abs_bias_lagcorr_AminusV_stable=("abs_bias_lagcorr_AminusV_stable", "mean"),
        )
    )

else:
    stable_subject_fixed_overall_df = pd.DataFrame()

stable_subject_fixed_overall_csv = safe_to_csv(
    stable_subject_fixed_overall_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_subject_fixed_offset_overall_rmse_bias.csv"),
    index=False
)

fixed_overall_rows = []

if len(stable_subject_fixed_overall_df) > 0:
    for metric in [
        "rmse_bias_raw_AminusV_stable",
        "rmse_bias_lagcorr_AminusV_stable",
        "abs_bias_raw_AminusV_stable",
        "abs_bias_lagcorr_AminusV_stable",
    ]:
        stats = _one_sample_test(stable_subject_fixed_overall_df[metric], popmean=0.0)

        fixed_overall_rows.append({
            "analysis": "stable_0p5_17p5_fixed_offset_overall_bias_vs_zero",
            "window_start_sec": STABLE_START_SEC,
            "window_end_sec": STABLE_END_SEC,
            "outcome": metric,
            **stats,
        })

stable_fixed_overall_tests_df = pd.DataFrame(fixed_overall_rows)

stable_fixed_overall_tests_csv = safe_to_csv(
    stable_fixed_overall_tests_df,
    os.path.join(STABLE_CSV_DIR, "stable_0p5_17p5_fixed_offset_overall_bias_tests.csv"),
    index=False
)

# -------------------------
# PLOTS: STABLE-WINDOW BIAS BY CONDITION
# -------------------------
for metric_type, bias_col, title_suffix, out_name in [
    (
        "raw",
        "rmse_bias_raw_AminusV_stable",
        "Raw RMSE bias",
        "stable_0p5_17p5_raw_rmse_bias_by_condition.png",
    ),
    (
        "lagcorr",
        "rmse_bias_lagcorr_AminusV_stable",
        "Lag-corrected RMSE bias",
        "stable_0p5_17p5_lagcorr_rmse_bias_by_condition.png",
    ),
    (
        "raw_abs",
        "abs_bias_raw_AminusV_stable",
        "Raw mean absolute-error bias",
        "stable_0p5_17p5_raw_abs_bias_by_condition.png",
    ),
    (
        "lagcorr_abs",
        "abs_bias_lagcorr_AminusV_stable",
        "Lag-corrected mean absolute-error bias",
        "stable_0p5_17p5_lagcorr_abs_bias_by_condition.png",
    ),
]:
    plot_df = stable_group_estimates_df[
        stable_group_estimates_df["outcome"] == bias_col
    ].copy()

    if len(plot_df) == 0:
        continue

    plot_df["condition_sort"] = plot_df["condition"].map(_condition_sort_key)
    plot_df = plot_df.sort_values("condition_sort")

    x = np.arange(len(plot_df))
    y = plot_df["mean"].astype(float).values
    yerr = (plot_df["ci_high"].astype(float).values - plot_df["ci_low"].astype(float).values) / 2.0
    labels = plot_df["condition_label"].tolist()

    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.errorbar(
        x,
        y,
        yerr=yerr,
        marker="o",
        linewidth=2.2,
        capsize=4
    )

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylabel("Bias A−V: Audio error − Visual error")
    ax.set_title(f"Stable-window {title_suffix} by condition\n0.5–17.5 s")

    safe_save_fig(fig, os.path.join(STABLE_FIG_DIR, out_name))

# -------------------------
# PRINT SUMMARY
# -------------------------
print("\n============================================================")
print("STABLE WINDOW TIME-RESOLVED RMSE/BIAS ANALYSIS COMPLETE")
print("============================================================")
print(f"Stable window:                         {STABLE_START_SEC}–{STABLE_END_SEC} s")
print(f"Stable output folder:                  {STABLE_OUT_DIR}")
print(f"Stable trial-timepoint CSV:            {stable_trial_error_csv}")
print(f"Subject × condition CSV:               {stable_subject_condition_csv}")
print(f"Group estimates CSV:                   {stable_group_estimates_csv}")
print(f"Bias vs zero tests CSV:                {stable_bias_tests_csv}")
print(f"Audio vs visual paired tests CSV:      {stable_audio_visual_paired_tests_csv}")
print(f"Fixed-offset trend tests CSV:          {stable_offset_trend_tests_csv}")
print(f"Fixed-offset overall subject CSV:      {stable_subject_fixed_overall_csv}")
print(f"Fixed-offset overall tests CSV:        {stable_fixed_overall_tests_csv}")
print(f"Figures saved to:                      {STABLE_FIG_DIR}")

print("\n============================================================")
print("PRIMARY LAG-CORRECTED RMSE BIAS TESTS")
print("Positive values mean responses were closer to visual than audio.")
print("============================================================")
primary_tests = stable_bias_tests_df[
    stable_bias_tests_df["outcome"] == "rmse_bias_lagcorr_AminusV_stable"
].copy()

if len(primary_tests) > 0:
    primary_tests["condition_sort"] = primary_tests["condition"].map(_condition_sort_key)
    primary_tests = primary_tests.sort_values("condition_sort").drop(columns=["condition_sort"])

    print(
        primary_tests[
            [
                "condition_label",
                "n_subjects",
                "mean",
                "ci_low",
                "ci_high",
                "t",
                "df",
                "p",
            ]
        ].to_string(index=False)
    )
else:
    print("No lag-corrected RMSE bias tests found.")

print("\n============================================================")
print("AUDIO VS VISUAL LAG-CORRECTED RMSE PAIRED TESTS")
print("Positive mean_diff_x_minus_y means audio RMSE > visual RMSE.")
print("============================================================")
paired_primary = stable_audio_visual_paired_tests_df[
    stable_audio_visual_paired_tests_df["metric_type"] == "lagcorr"
].copy()

if len(paired_primary) > 0:
    paired_primary["condition_sort"] = paired_primary["condition"].map(_condition_sort_key)
    paired_primary = paired_primary.sort_values("condition_sort").drop(columns=["condition_sort"])

    print(
        paired_primary[
            [
                "condition_label",
                "n_subjects",
                "mean_x",
                "mean_y",
                "mean_diff_x_minus_y",
                "ci_low",
                "ci_high",
                "t",
                "df",
                "p",
            ]
        ].to_string(index=False)
    )
else:
    print("No lag-corrected paired tests found.")

print("\n============================================================")
print("FIXED-OFFSET OVERALL LAG-CORRECTED BIAS")
print("Subject-first collapse across offset_10, offset_20, offset_30, offset_40.")
print("============================================================")
if len(stable_fixed_overall_tests_df) > 0:
    fixed_primary = stable_fixed_overall_tests_df[
        stable_fixed_overall_tests_df["outcome"] == "rmse_bias_lagcorr_AminusV_stable"
    ]

    if len(fixed_primary) > 0:
        print(
            fixed_primary[
                [
                    "n_subjects",
                    "mean",
                    "ci_low",
                    "ci_high",
                    "t",
                    "df",
                    "p",
                ]
            ].to_string(index=False)
        )
    else:
        print("No fixed-offset overall lag-corrected RMSE bias test found.")
else:
    print("No fixed-offset overall tests were generated.")

print("\n============================================================")
print("INTERPRETATION GUIDE")
print("============================================================")
print("- Full time-resolved plots still use the full 0–18 s trial.")
print("- Inferential summaries use the 0.5–17.5 s stable window.")
print("- RMSE bias AminusV = Audio RMSE - Visual RMSE.")
print("- Positive RMSE bias means responses were closer to the visual trajectory.")
print("- Negative RMSE bias means responses were closer to the auditory trajectory.")
print("- If saving to the intended OneDrive folder fails, files are saved to fallback folders inside CSV_DIR.")


Stable-window rows:
Rows: 1705000
Time range: 0.500–17.500 s
Unique timepoints: 341
Saved stable trial-timepoint CSV: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\03_time_resolved_rmse_bias\csv_outputs\stable_0p5_17p5_csv\stable_0p5_17p5_trial_timepoint_errors.csv

STABLE WINDOW TIME-RESOLVED RMSE/BIAS ANALYSIS COMPLETE
Stable window:                         0.5–17.5 s
Stable output folder:                  C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\03_time_resolved_rmse_bias
Stable trial-timepoint CSV:            C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\03_time_resolved_rmse_bias\csv_outputs\stable_0p5_17p5_csv\stable_0p5_17p5_trial_timepoint_errors.csv
Subject × condition CSV:               C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\03_time_resolved_rmse_bias\csv_outputs\stable_0p5_17p5_csv\sta

In [ ]:
# %% ADD-ON: PERMUTATION INFERENCE FOR STABLE-WINDOW TIME-RESOLVED RMSE BIAS
# ============================================================
# Run this AFTER the stable-window RMSE/bias analysis has run.
#
# This add-on performs two complementary permutation analyses:
#
# 1) Whole-trial condition-label shuffle permutation
#    ------------------------------------------------
#    Tests whether the empirical stable-window condition-level values are
#    larger/different than expected if condition labels were exchangeable
#    across whole trials within each participant.
#
#    This preserves:
#       - participant structure
#       - whole-trial temporal structure
#       - number of trials per condition within participant
#
#    It tests condition-specific stable-window metrics against label-shuffle
#    null distributions.
#
# 2) Cluster-corrected sign-flip permutation across time
#    ---------------------------------------------------
#    Tests whether time-resolved RMSE bias is reliably > 0 across contiguous
#    time ranges within each condition.
#
#    This preserves:
#       - participant structure
#       - the full within-subject time-series shape
#       - temporal dependence across neighboring timepoints
#
# Primary outcome:
#   rmse_bias_lagcorr_AminusV
#
# Interpretation:
#   Positive values = Audio RMSE > Visual RMSE
#                   = responses closer to the visual trajectory.
#
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import t as t_dist

# ============================================================
# SETTINGS
# ============================================================

N_PERM_LABEL_SHUFFLE = 10000
N_PERM_CLUSTER = 10000
RANDOM_SEED_STABLE_PERM = 12345

CLUSTER_FORMING_ALPHA = 0.05

# Stable window should match your stable-window analysis.
try:
    STABLE_START_SEC
except NameError:
    STABLE_START_SEC = 0.5

try:
    STABLE_END_SEC
except NameError:
    STABLE_END_SEC = 17.5

try:
    FIG_DPI
except NameError:
    FIG_DPI = 180

# Conditions should match the time-resolved RMSE analysis.
if "conditions" not in globals():
    conditions = [
        "offset_10",
        "offset_20",
        "offset_30",
        "offset_40",
        "sweep_diverge",
        "sweep_converge",
    ]

# Outcomes for stable-window condition-label shuffle.
# Primary = lag-corrected RMSE bias.
STABLE_LABEL_SHUFFLE_OUTCOMES = [
    "rmse_bias_lagcorr_AminusV_stable",
    "rmse_bias_raw_AminusV_stable",
    "abs_bias_lagcorr_AminusV_stable",
    "abs_bias_raw_AminusV_stable",
]

PRIMARY_STABLE_OUTCOME = "rmse_bias_lagcorr_AminusV_stable"

# Outcomes for cluster-corrected time-resolved sign-flip analysis.
# These are column names in subject_time_df.
TIME_CLUSTER_OUTCOMES = [
    "rmse_bias_lagcorr_AminusV",
]

# Directional hypothesis for RMSE bias:
#   Audio RMSE - Visual RMSE > 0
#   i.e., responses are closer to the visual trajectory.
LABEL_SHUFFLE_ALTERNATIVE = "greater"
CLUSTER_ALTERNATIVE = "greater"

# Save all outputs into the stable CSV/figure folders from your stable analysis.
if "STABLE_CSV_DIR" not in globals():
    if "CSV_DIR" in globals():
        STABLE_CSV_DIR = CSV_DIR
    else:
        STABLE_CSV_DIR = os.getcwd()

if "STABLE_FIG_DIR" not in globals():
    if "GROUP_PLOT_DIR" in globals():
        STABLE_FIG_DIR = GROUP_PLOT_DIR
    else:
        STABLE_FIG_DIR = os.getcwd()

os.makedirs(STABLE_CSV_DIR, exist_ok=True)
os.makedirs(STABLE_FIG_DIR, exist_ok=True)

print("\n============================================================")
print("STABLE-WINDOW RMSE BIAS PERMUTATION ADD-ON")
print("============================================================")
print(f"Stable window: {STABLE_START_SEC:.2f}–{STABLE_END_SEC:.2f} s")
print(f"Condition-label shuffle permutations: {N_PERM_LABEL_SHUFFLE}")
print(f"Cluster sign-flip permutations: {N_PERM_CLUSTER}")
print(f"CSV output folder: {STABLE_CSV_DIR}")
print(f"Figure output folder: {STABLE_FIG_DIR}")

# ============================================================
# SAFE SAVE HELPERS
# ============================================================

def _ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path

def _safe_to_csv_perm(df, filepath, index=False):
    """
    Uses your existing safe_to_csv if available; otherwise falls back to normal to_csv.
    """
    filepath = os.path.abspath(filepath)
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    if "safe_to_csv" in globals():
        return safe_to_csv(df, filepath, index=index)

    df.to_csv(filepath, index=index)
    return filepath

def _safe_save_fig_perm(fig, filepath):
    """
    Uses your existing safe_save_fig if available; otherwise falls back to normal savefig.
    """
    filepath = os.path.abspath(filepath)
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    if "safe_save_fig" in globals():
        return safe_save_fig(fig, filepath)

    fig.tight_layout()
    fig.savefig(filepath, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return filepath

# ============================================================
# REQUIRED DATA CHECKS
# ============================================================

if "stable_trial_error_df" not in globals():
    if "trial_error_df" not in globals():
        raise RuntimeError(
            "stable_trial_error_df and trial_error_df were not found. "
            "Run the main time-resolved RMSE analysis and stable-window analysis first."
        )

    stable_trial_error_df = (
        trial_error_df
        .replace([np.inf, -np.inf], np.nan)
        .query("time_sec >= @STABLE_START_SEC and time_sec <= @STABLE_END_SEC")
        .copy()
    )

if len(stable_trial_error_df) == 0:
    raise RuntimeError("stable_trial_error_df is empty.")

required_trial_error_cols = [
    "subject",
    "trialIndex",
    "condition",
    "time_sec",
    "audio_sqerr_raw",
    "visual_sqerr_raw",
    "audio_sqerr_lagcorr",
    "visual_sqerr_lagcorr",
    "audio_abs_raw",
    "visual_abs_raw",
    "audio_abs_lagcorr",
    "visual_abs_lagcorr",
]

missing_cols = [c for c in required_trial_error_cols if c not in stable_trial_error_df.columns]

if missing_cols:
    raise RuntimeError(
        f"stable_trial_error_df is missing required columns: {missing_cols}"
    )

# ============================================================
# GENERAL STATS HELPERS
# ============================================================

def _clean_numeric(values):
    return (
        pd.Series(values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
        .values
    )

def _mean_ci_perm(values, alpha=0.05):
    vals = _clean_numeric(values)
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "se": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p_two_sided": np.nan,
            "p_greater": np.nan,
            "p_less": np.nan,
        }

    m = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1))
    se = float(sd / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    t_stat, p_two = stats.ttest_1samp(vals, popmean=0.0, nan_policy="omit")

    return {
        "n": int(n),
        "mean": m,
        "se": se,
        "ci95_low": float(m - crit * se),
        "ci95_high": float(m + crit * se),
        "t": float(t_stat),
        "df": int(n - 1),
        "p_two_sided": float(p_two),
        "p_greater": float(stats.t.sf(t_stat, df=n - 1)),
        "p_less": float(stats.t.cdf(t_stat, df=n - 1)),
    }

def _empirical_pvalue(observed, null_values, alternative="two-sided"):
    null_values = _clean_numeric(null_values)

    if len(null_values) == 0 or not np.isfinite(observed):
        return np.nan

    if alternative == "greater":
        return float((np.sum(null_values >= observed) + 1) / (len(null_values) + 1))

    if alternative == "less":
        return float((np.sum(null_values <= observed) + 1) / (len(null_values) + 1))

    if alternative == "two-sided":
        null_center = float(np.nanmean(null_values))
        observed_distance = abs(observed - null_center)
        null_distance = np.abs(null_values - null_center)
        return float((np.sum(null_distance >= observed_distance) + 1) / (len(null_values) + 1))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")

def _fdr_bh(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return qvals

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    q = np.empty_like(q_ranked)
    q[order] = q_ranked
    qvals[finite] = q

    return qvals

def _holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    adj = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return adj

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    ranked_adj = np.empty(m)

    for i, pval in enumerate(ranked):
        ranked_adj[i] = (m - i) * pval

    ranked_adj = np.maximum.accumulate(ranked_adj)
    ranked_adj = np.clip(ranked_adj, 0, 1)

    out = np.empty_like(ranked_adj)
    out[order] = ranked_adj
    adj[finite] = out

    return adj

def _condition_label_perm(cond):
    label_map = {
        "offset_10": "10° offset",
        "offset_20": "20° offset",
        "offset_30": "30° offset",
        "offset_40": "40° offset",
        "sweep_diverge": "Sweep diverge",
        "sweep_converge": "Sweep converge",
    }
    return label_map.get(cond, str(cond))

def _condition_sort_key_perm(cond):
    order = {
        "offset_10": 1,
        "offset_20": 2,
        "offset_30": 3,
        "offset_40": 4,
        "sweep_diverge": 5,
        "sweep_converge": 6,
    }
    return order.get(cond, 999)

# ============================================================
# PART 1 — BUILD TRIAL-LEVEL STABLE-WINDOW INPUT
# ============================================================
# This keeps whole trials as the exchangeability unit for label shuffling.
# It aggregates timepoint errors per subject × trial × condition, retaining
# sums/counts so subject × condition RMSE can be recomputed exactly after
# label shuffling.

def _finite_sum(x):
    vals = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    return float(vals.sum())

def _finite_count(x):
    vals = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
    return int(len(vals))

stable_trial_perm_input = (
    stable_trial_error_df
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

# Keep target conditions only.
stable_trial_perm_input = stable_trial_perm_input[
    stable_trial_perm_input["condition"].isin(conditions)
].copy()

if len(stable_trial_perm_input) == 0:
    raise RuntimeError(
        "No rows from the target conditions were found in stable_trial_error_df."
    )

trial_group_cols = ["subject", "trialIndex", "condition"]

stable_trial_summary_for_perm = (
    stable_trial_perm_input
    .groupby(trial_group_cols, as_index=False)
    .agg(
        n_timepoint_rows=("time_sec", "count"),
        n_timepoints=("time_sec", "nunique"),

        audio_sqerr_raw_sum=("audio_sqerr_raw", _finite_sum),
        audio_sqerr_raw_n=("audio_sqerr_raw", _finite_count),
        visual_sqerr_raw_sum=("visual_sqerr_raw", _finite_sum),
        visual_sqerr_raw_n=("visual_sqerr_raw", _finite_count),

        audio_sqerr_lagcorr_sum=("audio_sqerr_lagcorr", _finite_sum),
        audio_sqerr_lagcorr_n=("audio_sqerr_lagcorr", _finite_count),
        visual_sqerr_lagcorr_sum=("visual_sqerr_lagcorr", _finite_sum),
        visual_sqerr_lagcorr_n=("visual_sqerr_lagcorr", _finite_count),

        audio_abs_raw_sum=("audio_abs_raw", _finite_sum),
        audio_abs_raw_n=("audio_abs_raw", _finite_count),
        visual_abs_raw_sum=("visual_abs_raw", _finite_sum),
        visual_abs_raw_n=("visual_abs_raw", _finite_count),

        audio_abs_lagcorr_sum=("audio_abs_lagcorr", _finite_sum),
        audio_abs_lagcorr_n=("audio_abs_lagcorr", _finite_count),
        visual_abs_lagcorr_sum=("visual_abs_lagcorr", _finite_sum),
        visual_abs_lagcorr_n=("visual_abs_lagcorr", _finite_count),
    )
)

stable_trial_summary_for_perm_path = _safe_to_csv_perm(
    stable_trial_summary_for_perm,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_trial_summary_for_label_shuffle.csv"),
    index=False
)

print("\nStable trial-level permutation input:")
print(f"Rows: {len(stable_trial_summary_for_perm)}")
print(f"Subjects: {stable_trial_summary_for_perm['subject'].nunique()}")
print("Condition counts:")
print(stable_trial_summary_for_perm["condition"].value_counts().sort_index())
print(f"Saved: {stable_trial_summary_for_perm_path}")

# ============================================================
# PART 2 — WHOLE-TRIAL CONDITION-LABEL SHUFFLE
# ============================================================

def _compute_subject_condition_stable_metrics_from_trial_summary(
    trial_summary_df,
    condition_col="condition"
):
    """
    Computes subject × condition stable-window metrics from trial-level
    stable-window sums/counts.

    This mirrors the stable-window analysis:
        RMSE = sqrt(sum squared error / count)
        Abs error = sum absolute error / count
        Bias = audio metric − visual metric
    """
    use = trial_summary_df.copy()
    use[condition_col] = use[condition_col].astype(str)

    grouped = (
        use
        .groupby(["subject", condition_col], as_index=False)
        .agg(
            n_trials=("trialIndex", "nunique"),
            n_timepoint_rows=("n_timepoint_rows", "sum"),

            audio_sqerr_raw_sum=("audio_sqerr_raw_sum", "sum"),
            audio_sqerr_raw_n=("audio_sqerr_raw_n", "sum"),
            visual_sqerr_raw_sum=("visual_sqerr_raw_sum", "sum"),
            visual_sqerr_raw_n=("visual_sqerr_raw_n", "sum"),

            audio_sqerr_lagcorr_sum=("audio_sqerr_lagcorr_sum", "sum"),
            audio_sqerr_lagcorr_n=("audio_sqerr_lagcorr_n", "sum"),
            visual_sqerr_lagcorr_sum=("visual_sqerr_lagcorr_sum", "sum"),
            visual_sqerr_lagcorr_n=("visual_sqerr_lagcorr_n", "sum"),

            audio_abs_raw_sum=("audio_abs_raw_sum", "sum"),
            audio_abs_raw_n=("audio_abs_raw_n", "sum"),
            visual_abs_raw_sum=("visual_abs_raw_sum", "sum"),
            visual_abs_raw_n=("visual_abs_raw_n", "sum"),

            audio_abs_lagcorr_sum=("audio_abs_lagcorr_sum", "sum"),
            audio_abs_lagcorr_n=("audio_abs_lagcorr_n", "sum"),
            visual_abs_lagcorr_sum=("visual_abs_lagcorr_sum", "sum"),
            visual_abs_lagcorr_n=("visual_abs_lagcorr_n", "sum"),
        )
        .rename(columns={condition_col: "condition"})
    )

    with np.errstate(invalid="ignore", divide="ignore"):
        grouped["audio_rmse_raw_stable"] = np.sqrt(
            grouped["audio_sqerr_raw_sum"] / grouped["audio_sqerr_raw_n"]
        )
        grouped["visual_rmse_raw_stable"] = np.sqrt(
            grouped["visual_sqerr_raw_sum"] / grouped["visual_sqerr_raw_n"]
        )

        grouped["audio_rmse_lagcorr_stable"] = np.sqrt(
            grouped["audio_sqerr_lagcorr_sum"] / grouped["audio_sqerr_lagcorr_n"]
        )
        grouped["visual_rmse_lagcorr_stable"] = np.sqrt(
            grouped["visual_sqerr_lagcorr_sum"] / grouped["visual_sqerr_lagcorr_n"]
        )

        grouped["audio_abs_raw_stable"] = (
            grouped["audio_abs_raw_sum"] / grouped["audio_abs_raw_n"]
        )
        grouped["visual_abs_raw_stable"] = (
            grouped["visual_abs_raw_sum"] / grouped["visual_abs_raw_n"]
        )

        grouped["audio_abs_lagcorr_stable"] = (
            grouped["audio_abs_lagcorr_sum"] / grouped["audio_abs_lagcorr_n"]
        )
        grouped["visual_abs_lagcorr_stable"] = (
            grouped["visual_abs_lagcorr_sum"] / grouped["visual_abs_lagcorr_n"]
        )

    grouped["rmse_bias_raw_AminusV_stable"] = (
        grouped["audio_rmse_raw_stable"] - grouped["visual_rmse_raw_stable"]
    )

    grouped["rmse_bias_lagcorr_AminusV_stable"] = (
        grouped["audio_rmse_lagcorr_stable"] - grouped["visual_rmse_lagcorr_stable"]
    )

    grouped["abs_bias_raw_AminusV_stable"] = (
        grouped["audio_abs_raw_stable"] - grouped["visual_abs_raw_stable"]
    )

    grouped["abs_bias_lagcorr_AminusV_stable"] = (
        grouped["audio_abs_lagcorr_stable"] - grouped["visual_abs_lagcorr_stable"]
    )

    return grouped


def _summarize_stable_subject_condition_metrics(subject_condition_df, outcomes):
    rows = []

    for condition, sub in subject_condition_df.groupby("condition"):
        for outcome in outcomes:
            if outcome not in sub.columns:
                continue

            stats_dict = _mean_ci_perm(sub[outcome])

            rows.append({
                "condition": condition,
                "condition_label": _condition_label_perm(condition),
                "outcome": outcome,
                "n_subjects": stats_dict["n"],
                "mean": stats_dict["mean"],
                "se": stats_dict["se"],
                "ci95_low": stats_dict["ci95_low"],
                "ci95_high": stats_dict["ci95_high"],
                "t": stats_dict["t"],
                "df": stats_dict["df"],
                "parametric_p_two_sided": stats_dict["p_two_sided"],
                "parametric_p_greater": stats_dict["p_greater"],
                "parametric_p_less": stats_dict["p_less"],
            })

    out = pd.DataFrame(rows)

    if not out.empty:
        out["condition_sort"] = out["condition"].map(_condition_sort_key_perm)
        out = out.sort_values(["condition_sort", "outcome"]).drop(columns=["condition_sort"])
        out = out.reset_index(drop=True)

    return out


def _shuffle_condition_labels_whole_trial(trial_summary_df, rng):
    """
    Whole-trial condition-label shuffle within subject.

    The stable-window data for a trial stay intact. Only the condition label is
    permuted across complete trials within the same participant.
    """
    out = trial_summary_df.copy()
    out["permuted_condition"] = np.nan

    for subj, idx in out.groupby("subject").groups.items():
        idx = list(idx)
        labels = out.loc[idx, "condition"].astype(str).values.copy()

        if len(labels) > 1:
            out.loc[idx, "permuted_condition"] = rng.permutation(labels)
        else:
            out.loc[idx, "permuted_condition"] = labels

    out["permuted_condition"] = out["permuted_condition"].astype(str)
    return out


# Observed values from the same trial-summary pipeline used for permutations.
observed_subject_condition_perm = _compute_subject_condition_stable_metrics_from_trial_summary(
    stable_trial_summary_for_perm,
    condition_col="condition"
)

observed_stable_perm_summary = _summarize_stable_subject_condition_metrics(
    observed_subject_condition_perm,
    outcomes=STABLE_LABEL_SHUFFLE_OUTCOMES
)

observed_subject_condition_perm_path = _safe_to_csv_perm(
    observed_subject_condition_perm,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_observed_subject_condition_metrics.csv"),
    index=False
)

observed_stable_perm_summary_path = _safe_to_csv_perm(
    observed_stable_perm_summary,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_observed_condition_metric_summary.csv"),
    index=False
)

print("\nObserved stable-window condition metrics:")
try:
    display(observed_stable_perm_summary)
except NameError:
    print(observed_stable_perm_summary.to_string(index=False))

# Run whole-trial label-shuffle null.
rng_label = np.random.default_rng(RANDOM_SEED_STABLE_PERM)

label_shuffle_rows = []

for perm_i in range(N_PERM_LABEL_SHUFFLE):
    perm_trial_summary = _shuffle_condition_labels_whole_trial(
        stable_trial_summary_for_perm,
        rng=rng_label
    )

    perm_subject_condition = _compute_subject_condition_stable_metrics_from_trial_summary(
        perm_trial_summary,
        condition_col="permuted_condition"
    )

    perm_summary = _summarize_stable_subject_condition_metrics(
        perm_subject_condition,
        outcomes=STABLE_LABEL_SHUFFLE_OUTCOMES
    )

    if not perm_summary.empty:
        for _, row in perm_summary.iterrows():
            label_shuffle_rows.append({
                "perm_i": perm_i,
                "condition": row["condition"],
                "condition_label": row["condition_label"],
                "outcome": row["outcome"],
                "n_subjects": row["n_subjects"],
                "mean": row["mean"],
                "t": row["t"],
            })

    if (perm_i + 1) % 250 == 0:
        print(f"Completed stable-window condition-label shuffle {perm_i + 1}/{N_PERM_LABEL_SHUFFLE}")

stable_label_shuffle_null = pd.DataFrame(label_shuffle_rows)

stable_label_shuffle_null_path = _safe_to_csv_perm(
    stable_label_shuffle_null,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_condition_label_shuffle_null_long.csv"),
    index=False
)

# Test observed empirical values against label-shuffle null distributions.
label_shuffle_test_rows = []

for _, obs in observed_stable_perm_summary.iterrows():
    condition = obs["condition"]
    outcome = obs["outcome"]

    null_sub = stable_label_shuffle_null[
        (stable_label_shuffle_null["condition"] == condition)
        & (stable_label_shuffle_null["outcome"] == outcome)
    ].copy()

    null_means = _clean_numeric(null_sub["mean"])
    null_t = _clean_numeric(null_sub["t"])

    obs_mean = float(obs["mean"])
    obs_t = float(obs["t"])

    if len(null_means) > 0:
        null_mean_mean = float(np.nanmean(null_means))
        null_mean_sd = float(np.nanstd(null_means, ddof=1))
        null_mean_ci95_low = float(np.nanpercentile(null_means, 2.5))
        null_mean_ci95_high = float(np.nanpercentile(null_means, 97.5))
    else:
        null_mean_mean = np.nan
        null_mean_sd = np.nan
        null_mean_ci95_low = np.nan
        null_mean_ci95_high = np.nan

    if len(null_t) > 0:
        null_t_mean = float(np.nanmean(null_t))
        null_t_sd = float(np.nanstd(null_t, ddof=1))
    else:
        null_t_mean = np.nan
        null_t_sd = np.nan

    p_mean_greater = _empirical_pvalue(
        obs_mean,
        null_means,
        alternative="greater"
    )

    p_mean_two_sided = _empirical_pvalue(
        obs_mean,
        null_means,
        alternative="two-sided"
    )

    p_t_greater = _empirical_pvalue(
        obs_t,
        null_t,
        alternative="greater"
    )

    p_t_two_sided = _empirical_pvalue(
        obs_t,
        null_t,
        alternative="two-sided"
    )

    label_shuffle_test_rows.append({
        "null_type": "within_subject_whole_trial_condition_label_shuffle",
        "condition": condition,
        "condition_label": obs["condition_label"],
        "outcome": outcome,
        "observed_mean": obs_mean,
        "observed_se": obs["se"],
        "observed_ci95_low": obs["ci95_low"],
        "observed_ci95_high": obs["ci95_high"],
        "observed_t": obs_t,
        "observed_df": obs["df"],
        "observed_n_subjects": obs["n_subjects"],
        "parametric_p_greater_vs_zero": obs["parametric_p_greater"],
        "parametric_p_two_sided_vs_zero": obs["parametric_p_two_sided"],
        "null_mean_mean": null_mean_mean,
        "null_mean_sd": null_mean_sd,
        "null_mean_ci95_low": null_mean_ci95_low,
        "null_mean_ci95_high": null_mean_ci95_high,
        "null_t_mean": null_t_mean,
        "null_t_sd": null_t_sd,
        "permutation_p_mean_greater_than_label_null": p_mean_greater,
        "permutation_p_mean_two_sided_vs_label_null": p_mean_two_sided,
        "permutation_p_t_greater_than_label_null": p_t_greater,
        "permutation_p_t_two_sided_vs_label_null": p_t_two_sided,
        "n_permutations": N_PERM_LABEL_SHUFFLE,
        "directional_interpretation": (
            "greater means more positive Audio-minus-Visual bias than label-shuffle null; "
            "positive bias means closer to visual"
        ),
    })

stable_label_shuffle_tests = pd.DataFrame(label_shuffle_test_rows)

# FDR/Holm corrections across each p-value family.
for p_col in [
    "permutation_p_mean_greater_than_label_null",
    "permutation_p_mean_two_sided_vs_label_null",
    "permutation_p_t_greater_than_label_null",
    "permutation_p_t_two_sided_vs_label_null",
]:
    stable_label_shuffle_tests[f"{p_col}_fdr"] = _fdr_bh(
        stable_label_shuffle_tests[p_col].values
    )
    stable_label_shuffle_tests[f"{p_col}_holm"] = _holm_bonferroni(
        stable_label_shuffle_tests[p_col].values
    )

stable_label_shuffle_tests_path = _safe_to_csv_perm(
    stable_label_shuffle_tests,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_condition_label_shuffle_tests.csv"),
    index=False
)

print("\nWhole-trial condition-label shuffle tests:")
try:
    display(stable_label_shuffle_tests)
except NameError:
    print(stable_label_shuffle_tests.to_string(index=False))

# ============================================================
# PART 3 — CLUSTER-CORRECTED SIGN-FLIP ACROSS TIME
# ============================================================

# Build subject_time_df if it is not already available.
if "subject_time_df" not in globals():
    subject_time_df = (
        stable_trial_error_df
        .groupby(["subject", "condition", "time_sec"], as_index=False)
        .agg(
            audio_rmse_raw=("audio_sqerr_raw", lambda x: np.sqrt(np.nanmean(x))),
            visual_rmse_raw=("visual_sqerr_raw", lambda x: np.sqrt(np.nanmean(x))),
            audio_rmse_lagcorr=("audio_sqerr_lagcorr", lambda x: np.sqrt(np.nanmean(x))),
            visual_rmse_lagcorr=("visual_sqerr_lagcorr", lambda x: np.sqrt(np.nanmean(x))),
            audio_abs_raw=("audio_abs_raw", "mean"),
            visual_abs_raw=("visual_abs_raw", "mean"),
            audio_abs_lagcorr=("audio_abs_lagcorr", "mean"),
            visual_abs_lagcorr=("visual_abs_lagcorr", "mean"),
            n_trials=("trialIndex", "nunique"),
        )
    )

    subject_time_df["rmse_bias_raw_AminusV"] = (
        subject_time_df["audio_rmse_raw"] - subject_time_df["visual_rmse_raw"]
    )
    subject_time_df["rmse_bias_lagcorr_AminusV"] = (
        subject_time_df["audio_rmse_lagcorr"] - subject_time_df["visual_rmse_lagcorr"]
    )

# Restrict subject_time_df to the stable window.
subject_time_stable_for_perm = (
    subject_time_df
    .replace([np.inf, -np.inf], np.nan)
    .query("time_sec >= @STABLE_START_SEC and time_sec <= @STABLE_END_SEC")
    .copy()
)

subject_time_stable_for_perm = subject_time_stable_for_perm[
    subject_time_stable_for_perm["condition"].isin(conditions)
].copy()

subject_time_stable_for_perm_path = _safe_to_csv_perm(
    subject_time_stable_for_perm,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_subject_time_input_for_cluster_tests.csv"),
    index=False
)

def _one_sample_t_by_time(matrix):
    """
    matrix: subjects × timepoints
    """
    matrix = np.asarray(matrix, dtype=float)
    n_time = matrix.shape[1]

    means = np.full(n_time, np.nan)
    t_vals = np.full(n_time, np.nan)
    p_two_vals = np.full(n_time, np.nan)
    ns = np.zeros(n_time, dtype=int)

    for j in range(n_time):
        vals = matrix[:, j]
        vals = vals[np.isfinite(vals)]
        ns[j] = len(vals)

        if len(vals) < 2:
            continue

        means[j] = float(np.nanmean(vals))

        if np.nanstd(vals, ddof=1) == 0:
            continue

        t_stat, p_two = stats.ttest_1samp(vals, popmean=0.0, nan_policy="omit")
        t_vals[j] = float(t_stat)
        p_two_vals[j] = float(p_two)

    return means, t_vals, p_two_vals, ns

def _p_from_t_for_alternative(t_vals, ns, alternative="greater"):
    t_vals = np.asarray(t_vals, dtype=float)
    ns = np.asarray(ns, dtype=int)

    p_vals = np.full_like(t_vals, np.nan, dtype=float)

    for j, t_stat in enumerate(t_vals):
        if not np.isfinite(t_stat) or ns[j] < 2:
            continue

        df = ns[j] - 1

        if alternative == "two-sided":
            p_vals[j] = 2 * stats.t.sf(abs(t_stat), df=df)
        elif alternative == "greater":
            p_vals[j] = stats.t.sf(t_stat, df=df)
        elif alternative == "less":
            p_vals[j] = stats.t.cdf(t_stat, df=df)
        else:
            raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")

    return p_vals

def _find_contiguous_clusters(mask):
    mask = np.asarray(mask, dtype=bool)
    clusters = []
    start = None

    for i, val in enumerate(mask):
        if val and start is None:
            start = i

        if (not val or i == len(mask) - 1) and start is not None:
            end = i if not val else i + 1
            clusters.append(np.arange(start, end))
            start = None

    return clusters

def _cluster_stat_from_t(t_vals, cluster_idx, alternative="greater"):
    vals = np.asarray(t_vals, dtype=float)[cluster_idx]
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan

    if alternative == "greater":
        return float(np.sum(vals))

    if alternative == "less":
        return float(np.sum(-vals))

    if alternative == "two-sided":
        return float(np.sum(np.abs(vals)))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")

def _build_subject_time_matrix(df, condition, outcome):
    d = df[
        (df["condition"] == condition)
    ].copy()

    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=["subject", "time_sec", outcome]).copy()

    if d.empty:
        return np.array([]), np.empty((0, 0)), []

    # Round time to reduce floating-point merge/pivot issues.
    d["time_sec_rounded"] = d["time_sec"].astype(float).round(6)

    all_times = np.sort(d["time_sec_rounded"].unique().astype(float))
    all_subjects = np.sort(d["subject"].unique())

    wide = d.pivot_table(
        index="subject",
        columns="time_sec_rounded",
        values=outcome,
        aggfunc="mean"
    )

    wide = wide.reindex(index=all_subjects, columns=all_times)

    matrix = wide.values.astype(float)

    return all_times, matrix, list(all_subjects)

def _cluster_corrected_signflip_time_test(
    matrix,
    timepoints,
    alternative="greater",
    cluster_alpha=0.05,
    n_perm=10000,
    seed=12345
):
    """
    Cluster-corrected sign-flip permutation test across ordered timepoints.

    The null flips each participant's entire time-resolved bias curve,
    preserving temporal dependence within participant.
    """
    rng = np.random.default_rng(seed)

    matrix = np.asarray(matrix, dtype=float)
    timepoints = np.asarray(timepoints, dtype=float)

    if matrix.size == 0 or matrix.shape[1] == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    valid_time_n = np.sum(np.isfinite(matrix), axis=0)
    valid_times = valid_time_n >= 2

    matrix_valid = matrix[:, valid_times]
    timepoints_valid = timepoints[valid_times]

    if matrix_valid.shape[1] == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    means, t_vals, p_two, ns = _one_sample_t_by_time(matrix_valid)
    p_vals = _p_from_t_for_alternative(t_vals, ns, alternative=alternative)

    cluster_mask = p_vals < cluster_alpha

    if alternative == "greater":
        cluster_mask = cluster_mask & (t_vals > 0)
    elif alternative == "less":
        cluster_mask = cluster_mask & (t_vals < 0)

    obs_clusters = _find_contiguous_clusters(cluster_mask)

    obs_cluster_rows = []

    for cluster_i, idx in enumerate(obs_clusters, start=1):
        c_stat = _cluster_stat_from_t(t_vals, idx, alternative=alternative)

        obs_cluster_rows.append({
            "cluster_id": cluster_i,
            "start_time_sec": float(timepoints_valid[idx[0]]),
            "end_time_sec": float(timepoints_valid[idx[-1]]),
            "n_timepoints": int(len(idx)),
            "cluster_stat": c_stat,
            "mean_difference_in_cluster": float(np.nanmean(means[idx])),
            "max_abs_t_in_cluster": float(np.nanmax(np.abs(t_vals[idx]))),
            "peak_t_in_cluster": float(np.nanmax(t_vals[idx])),
        })

    n_subjects = matrix_valid.shape[0]
    max_cluster_stats = np.zeros(n_perm, dtype=float)

    for perm_i in range(n_perm):
        signs = rng.choice([-1, 1], size=n_subjects)
        perm_matrix = matrix_valid * signs[:, None]

        _, perm_t_vals, _, perm_ns = _one_sample_t_by_time(perm_matrix)

        perm_p_vals = _p_from_t_for_alternative(
            perm_t_vals,
            perm_ns,
            alternative=alternative
        )

        perm_mask = perm_p_vals < cluster_alpha

        if alternative == "greater":
            perm_mask = perm_mask & (perm_t_vals > 0)
        elif alternative == "less":
            perm_mask = perm_mask & (perm_t_vals < 0)

        perm_clusters = _find_contiguous_clusters(perm_mask)

        if len(perm_clusters) == 0:
            max_cluster_stats[perm_i] = 0.0
        else:
            perm_cluster_stats = [
                _cluster_stat_from_t(perm_t_vals, idx, alternative=alternative)
                for idx in perm_clusters
            ]
            max_cluster_stats[perm_i] = float(np.nanmax(perm_cluster_stats))

    obs_cluster_df = pd.DataFrame(obs_cluster_rows)

    if not obs_cluster_df.empty:
        obs_cluster_df["cluster_corrected_p"] = [
            float((np.sum(max_cluster_stats >= stat) + 1) / (n_perm + 1))
            for stat in obs_cluster_df["cluster_stat"].values
        ]

    binwise_df = pd.DataFrame({
        "time_sec": timepoints_valid,
        "mean_difference": means,
        "t": t_vals,
        "p_uncorrected": p_vals,
        "n_subjects": ns,
        "passes_cluster_forming_threshold": cluster_mask,
    })

    null_df = pd.DataFrame({
        "perm_i": np.arange(n_perm),
        "max_cluster_stat": max_cluster_stats,
    })

    return binwise_df, obs_cluster_df, null_df

# Run cluster-corrected tests.
cluster_binwise_all = []
cluster_corrected_all = []
cluster_null_all = []

for outcome in TIME_CLUSTER_OUTCOMES:
    if outcome not in subject_time_stable_for_perm.columns:
        print(f"Skipping cluster tests for missing outcome: {outcome}")
        continue

    for condition in conditions:
        timepoints, matrix, subjects_used = _build_subject_time_matrix(
            subject_time_stable_for_perm,
            condition=condition,
            outcome=outcome
        )

        if matrix.size == 0:
            print(f"Skipping cluster test: no matrix for {condition}, {outcome}")
            continue

        binwise_df, cluster_df, null_df = _cluster_corrected_signflip_time_test(
            matrix=matrix,
            timepoints=timepoints,
            alternative=CLUSTER_ALTERNATIVE,
            cluster_alpha=CLUSTER_FORMING_ALPHA,
            n_perm=N_PERM_CLUSTER,
            seed=RANDOM_SEED_STABLE_PERM + 1000 + _condition_sort_key_perm(condition)
        )

        if not binwise_df.empty:
            binwise_df["condition"] = condition
            binwise_df["condition_label"] = _condition_label_perm(condition)
            binwise_df["outcome"] = outcome
            binwise_df["alternative"] = CLUSTER_ALTERNATIVE
            binwise_df["n_subjects_total"] = len(subjects_used)
            cluster_binwise_all.append(binwise_df)

        if not cluster_df.empty:
            cluster_df["condition"] = condition
            cluster_df["condition_label"] = _condition_label_perm(condition)
            cluster_df["outcome"] = outcome
            cluster_df["alternative"] = CLUSTER_ALTERNATIVE
            cluster_df["n_subjects_total"] = len(subjects_used)
            cluster_corrected_all.append(cluster_df)

        if not null_df.empty:
            null_df["condition"] = condition
            null_df["condition_label"] = _condition_label_perm(condition)
            null_df["outcome"] = outcome
            null_df["alternative"] = CLUSTER_ALTERNATIVE
            cluster_null_all.append(null_df)

        print(f"Completed cluster test: {condition}, {outcome}")

stable_cluster_binwise_tests = (
    pd.concat(cluster_binwise_all, ignore_index=True)
    if len(cluster_binwise_all) > 0
    else pd.DataFrame()
)

stable_cluster_corrected_tests = (
    pd.concat(cluster_corrected_all, ignore_index=True)
    if len(cluster_corrected_all) > 0
    else pd.DataFrame()
)

stable_cluster_null_distribution = (
    pd.concat(cluster_null_all, ignore_index=True)
    if len(cluster_null_all) > 0
    else pd.DataFrame()
)

stable_cluster_binwise_tests_path = _safe_to_csv_perm(
    stable_cluster_binwise_tests,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_cluster_binwise_time_tests.csv"),
    index=False
)

stable_cluster_corrected_tests_path = _safe_to_csv_perm(
    stable_cluster_corrected_tests,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_cluster_corrected_time_tests.csv"),
    index=False
)

stable_cluster_null_distribution_path = _safe_to_csv_perm(
    stable_cluster_null_distribution,
    os.path.join(STABLE_CSV_DIR, "perm_stable_0p5_17p5_cluster_null_distribution.csv"),
    index=False
)

print("\nCluster-corrected sign-flip tests:")
try:
    display(stable_cluster_corrected_tests)
except NameError:
    print(stable_cluster_corrected_tests.to_string(index=False))

# ============================================================
# PART 4 — PLOTS
# ============================================================

# Label-shuffle null plots for the primary stable outcome.
primary_label_tests = stable_label_shuffle_tests[
    stable_label_shuffle_tests["outcome"] == PRIMARY_STABLE_OUTCOME
].copy()

for _, row in primary_label_tests.iterrows():
    cond = row["condition"]
    outcome = row["outcome"]
    observed = row["observed_mean"]

    null_sub = stable_label_shuffle_null[
        (stable_label_shuffle_null["condition"] == cond)
        & (stable_label_shuffle_null["outcome"] == outcome)
    ].copy()

    if len(null_sub) == 0:
        continue

    null_vals = _clean_numeric(null_sub["mean"])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(null_vals, bins=40, alpha=0.75)
    ax.axvline(0, linewidth=1)
    ax.axvline(observed, linestyle="--", linewidth=2)
    ax.set_xlabel(f"{outcome}: {cond}")
    ax.set_ylabel("Permutation count")
    ax.set_title(f"Condition-label shuffle null\n{_condition_label_perm(cond)}")

    _safe_save_fig_perm(
        fig,
        os.path.join(
            STABLE_FIG_DIR,
            f"perm_stable_label_shuffle_null_{cond}_{outcome}.png"
        )
    )

# Cluster-corrected time-series plots for primary time outcome.
for outcome in TIME_CLUSTER_OUTCOMES:
    if outcome not in subject_time_stable_for_perm.columns:
        continue

    for condition in conditions:
        d = subject_time_stable_for_perm[
            (subject_time_stable_for_perm["condition"] == condition)
        ].copy()

        if d.empty:
            continue

        # Subject-first group estimate by timepoint.
        curve_rows = []

        for time_sec, sub in d.groupby("time_sec"):
            stats_dict = _mean_ci_perm(sub[outcome])

            curve_rows.append({
                "time_sec": float(time_sec),
                "mean": stats_dict["mean"],
                "ci95_low": stats_dict["ci95_low"],
                "ci95_high": stats_dict["ci95_high"],
                "n_subjects": stats_dict["n"],
            })

        curve_df = pd.DataFrame(curve_rows).sort_values("time_sec")

        if curve_df.empty:
            continue

        fig, ax = plt.subplots(figsize=(9, 4.8))

        x = curve_df["time_sec"].astype(float).values
        y = curve_df["mean"].astype(float).values
        lo = curve_df["ci95_low"].astype(float).values
        hi = curve_df["ci95_high"].astype(float).values

        ax.plot(x, y, linewidth=2.2)
        ax.fill_between(x, lo, hi, alpha=0.20)
        ax.axhline(0, linestyle="--", linewidth=1)

        if "sweep" in condition:
            ax.axvline(12, linestyle="--", linewidth=1, alpha=0.7)

        # Add significant cluster spans.
        sig_clusters = stable_cluster_corrected_tests[
            (stable_cluster_corrected_tests["condition"] == condition)
            & (stable_cluster_corrected_tests["outcome"] == outcome)
            & (stable_cluster_corrected_tests["cluster_corrected_p"] < 0.05)
        ].copy()

        if not sig_clusters.empty:
            y_min, y_max = ax.get_ylim()
            y_pos = y_max - 0.08 * (y_max - y_min)

            label_added = False

            for _, crow in sig_clusters.iterrows():
                ax.axvspan(
                    crow["start_time_sec"],
                    crow["end_time_sec"],
                    alpha=0.12,
                    label="cluster p < .05" if not label_added else "_nolegend_"
                )

                ax.text(
                    x=(crow["start_time_sec"] + crow["end_time_sec"]) / 2,
                    y=y_pos,
                    s=f"p={crow['cluster_corrected_p']:.4f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )

                label_added = True

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Audio RMSE − Visual RMSE")
        ax.set_title(
            f"Cluster-corrected time-resolved RMSE bias\n"
            f"{_condition_label_perm(condition)}"
        )

        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys(), frameon=False)

        _safe_save_fig_perm(
            fig,
            os.path.join(
                STABLE_FIG_DIR,
                f"perm_stable_cluster_timecourse_{condition}_{outcome}.png"
            )
        )

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n============================================================")
print("STABLE-WINDOW PERMUTATION ADD-ON COMPLETE")
print("============================================================")

print("\nWhole-trial condition-label shuffle:")
print("- Unit of permutation: complete trials within participant.")
print("- Condition labels are shuffled while trial-level error values stay intact.")
print("- This tests whether condition-specific stable-window values exceed/differ from label-shuffle nulls.")
print("- For the primary outcome, positive values mean responses were closer to visual than audio.")

print("\nCluster-corrected sign-flip:")
print("- Unit of permutation: whole subject-level time-resolved curves.")
print("- Each participant's full time series is sign-flipped, preserving temporal dependence.")
print("- This tests whether positive RMSE bias occurs across contiguous time ranges.")

print("\nKey output files:")
print(f"Trial-level permutation input:       {stable_trial_summary_for_perm_path}")
print(f"Observed subject-condition metrics:  {observed_subject_condition_perm_path}")
print(f"Observed condition summary:          {observed_stable_perm_summary_path}")
print(f"Condition-label null long:           {stable_label_shuffle_null_path}")
print(f"Condition-label tests:               {stable_label_shuffle_tests_path}")
print(f"Cluster binwise time tests:          {stable_cluster_binwise_tests_path}")
print(f"Cluster-corrected time tests:        {stable_cluster_corrected_tests_path}")
print(f"Cluster null distribution:           {stable_cluster_null_distribution_path}")

print("\nPrimary condition-label shuffle results:")
primary_print = stable_label_shuffle_tests[
    stable_label_shuffle_tests["outcome"] == PRIMARY_STABLE_OUTCOME
].copy()

if len(primary_print) > 0:
    primary_print["condition_sort"] = primary_print["condition"].map(_condition_sort_key_perm)
    primary_print = primary_print.sort_values("condition_sort").drop(columns=["condition_sort"])

    print(
        primary_print[
            [
                "condition_label",
                "observed_mean",
                "observed_ci95_low",
                "observed_ci95_high",
                "observed_t",
                "null_mean_mean",
                "null_mean_ci95_low",
                "null_mean_ci95_high",
                "permutation_p_mean_greater_than_label_null",
                "permutation_p_mean_greater_than_label_null_fdr",
                "permutation_p_mean_two_sided_vs_label_null",
                "permutation_p_mean_two_sided_vs_label_null_fdr",
            ]
        ].to_string(index=False)
    )
else:
    print("No primary condition-label shuffle results found.")

print("\nCluster-corrected time-range results:")
if len(stable_cluster_corrected_tests) > 0:
    print(
        stable_cluster_corrected_tests[
            [
                "condition_label",
                "outcome",
                "start_time_sec",
                "end_time_sec",
                "n_timepoints",
                "mean_difference_in_cluster",
                "max_abs_t_in_cluster",
                "cluster_corrected_p",
            ]
        ].to_string(index=False)
    )
else:
    print("No cluster-corrected time clusters were found.")

## 9. Time-Resolved Bias by Explicit Causal Judgment

Same- versus different-source bias trajectories, permutation inference, high-confidence different-source trials, and confidence analyses.

In [13]:
# %% ANALYSIS 4 — TIME-RESOLVED BIAS TRAJECTORIES BY SAME/DIFFERENT JUDGMENT
# ============================================================
# Goal:
# Plot Bias_AminusV(t) separately for trials judged:
#   Same Source vs Different Source.
#
# Bias_AminusV(t) = abs(response(t) - audio(t)) - abs(response(t) - visual(t))
#
# Interpretation:
#   Positive = response closer to visual trajectory
#   Negative = response closer to auditory trajectory
#
# Subject-first:
#   trial-level timepoint bias
#   -> subject × offset × judgment × time average
#   -> group-level average across subjects
#
# Outputs:
#   - trial timepoint bias CSV
#   - subject-level timepoint bias CSV
#   - group-level timepoint bias CSV
#   - group plots overall and separated by offset
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# OUTPUT DIRECTORY
# -------------------------
try:
    SPATIAL_ANALYSES_ROOT = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses"
    timebias_out_dir = os.path.join(SPATIAL_ANALYSES_ROOT, "04_causal_judgment_visual_bias")
except NameError:
    timebias_out_dir = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\04_causal_judgment_visual_bias"

csv_dir = os.path.join(timebias_out_dir, "csv_outputs")
fig_dir = os.path.join(timebias_out_dir, "figures")

for d in [timebias_out_dir, csv_dir, fig_dir]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 180

# -------------------------
# REQUIRED DATA CHECK
# -------------------------
if "trial_df" not in globals():
    raise RuntimeError("trial_df not found. Run your load/resample cell first.")

# -------------------------
# HELPERS
# -------------------------
def parse_yes_no_same_response(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["y", "yes", "1", "true", "same", "same source", "common", "common source"]:
        return 1.0
    if s in ["n", "no", "0", "false", "different", "diff", "different source", "separate", "separate source"]:
        return 0.0
    return np.nan

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def sem(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return x.std(ddof=1) / np.sqrt(len(x))

def save_fig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

# -------------------------
# PREPARE FIXED-OFFSET TRIALS
# -------------------------
df = trial_df.copy()

if "same_response_yes" not in df.columns:
    if "post_question" not in df.columns:
        raise RuntimeError("Neither same_response_yes nor post_question found in trial_df.")
    df["same_response_yes"] = df["post_question"].apply(parse_yes_no_same_response)

df["Psame"] = pd.to_numeric(df["same_response_yes"], errors="coerce")
df["judgment_label"] = np.where(df["Psame"] == 1, "Same source",
                         np.where(df["Psame"] == 0, "Different source", np.nan))

df = df[df["trialType"].astype(str).str.lower() == "incongruent_spatial"].copy()
df["offset_num"] = df.apply(extract_offset_from_row, axis=1)
df = df[df["offset_num"].isin([10, 20, 30, 40])].copy()
df = df.dropna(subset=["Psame", "judgment_label"]).copy()

if len(df) == 0:
    raise RuntimeError("No fixed-offset trials with valid same/different judgments found.")

print("Fixed-offset trials with valid judgments:")
print(df.groupby(["offset_num", "judgment_label"]).size())

# -------------------------
# TRIAL-LEVEL TIMEPOINT BIAS
# -------------------------
trial_time_rows = []

for _, row in df.iterrows():
    subj = row["subject"]
    trial_index = row["trialIndex"]
    cond = row.get("condLabel", "")
    offset = int(row["offset_num"])
    psame = float(row["Psame"])
    judgment = row["judgment_label"]

    t = np.asarray(row["t_grid"], dtype=float)
    resp = np.asarray(row["mouse_x"], dtype=float)
    aud = np.asarray(row["audio_x"], dtype=float)
    vis = np.asarray(row["blob_x"], dtype=float)

    audio_abs_error = np.abs(resp - aud)
    visual_abs_error = np.abs(resp - vis)

    bias_AminusV_t = audio_abs_error - visual_abs_error

    for i, time_sec in enumerate(t):
        trial_time_rows.append({
            "subject": subj,
            "trialIndex": trial_index,
            "condLabel": cond,
            "offset_num": offset,
            "Psame": psame,
            "judgment_label": judgment,
            "time_sec": time_sec,
            "audio_abs_error": audio_abs_error[i],
            "visual_abs_error": visual_abs_error[i],
            "Bias_AminusV_t": bias_AminusV_t[i],
        })

trial_time_df = pd.DataFrame(trial_time_rows)

trial_time_csv = os.path.join(csv_dir, "trial_timepoint_bias_by_judgment.csv")
trial_time_df.to_csv(trial_time_csv, index=False)

# -------------------------
# SUBJECT × OFFSET × JUDGMENT × TIME AVERAGE
# -------------------------
subject_time_df = (
    trial_time_df
    .groupby(["subject", "offset_num", "judgment_label", "time_sec"], as_index=False)
    .agg(
        Bias_AminusV_t=("Bias_AminusV_t", "mean"),
        audio_abs_error=("audio_abs_error", "mean"),
        visual_abs_error=("visual_abs_error", "mean"),
        Psame=("Psame", "mean"),
        n_trials=("trialIndex", "nunique")
    )
)

subject_time_csv = os.path.join(csv_dir, "subject_timepoint_bias_by_offset_judgment.csv")
subject_time_df.to_csv(subject_time_csv, index=False)

# -------------------------
# GROUP-LEVEL TIME CURVES
# -------------------------
group_time_df = (
    subject_time_df
    .groupby(["offset_num", "judgment_label", "time_sec"], as_index=False)
    .agg(
        Bias_mean=("Bias_AminusV_t", "mean"),
        Bias_sem=("Bias_AminusV_t", sem),
        n_subjects=("subject", "nunique"),
        audio_abs_error_mean=("audio_abs_error", "mean"),
        visual_abs_error_mean=("visual_abs_error", "mean")
    )
)

group_time_df["Bias_ci95"] = 1.96 * group_time_df["Bias_sem"]

group_time_csv = os.path.join(csv_dir, "group_timepoint_bias_by_offset_judgment.csv")
group_time_df.to_csv(group_time_csv, index=False)

# Also collapse across offsets subject-first.
subject_time_overall_df = (
    subject_time_df
    .groupby(["subject", "judgment_label", "time_sec"], as_index=False)
    .agg(
        Bias_AminusV_t=("Bias_AminusV_t", "mean"),
        audio_abs_error=("audio_abs_error", "mean"),
        visual_abs_error=("visual_abs_error", "mean"),
        n_trials=("n_trials", "sum")
    )
)

group_time_overall_df = (
    subject_time_overall_df
    .groupby(["judgment_label", "time_sec"], as_index=False)
    .agg(
        Bias_mean=("Bias_AminusV_t", "mean"),
        Bias_sem=("Bias_AminusV_t", sem),
        n_subjects=("subject", "nunique")
    )
)

group_time_overall_df["Bias_ci95"] = 1.96 * group_time_overall_df["Bias_sem"]

overall_csv = os.path.join(csv_dir, "group_timepoint_bias_overall_by_judgment.csv")
group_time_overall_df.to_csv(overall_csv, index=False)

# -------------------------
# PLOT 1: OVERALL SAME VS DIFFERENT
# -------------------------
fig, ax = plt.subplots(figsize=(8, 5))

for label in ["Same source", "Different source"]:
    cdf = group_time_overall_df[group_time_overall_df["judgment_label"] == label].sort_values("time_sec")
    if len(cdf) == 0:
        continue

    x = cdf["time_sec"].values.astype(float)
    y = cdf["Bias_mean"].values.astype(float)
    ci = cdf["Bias_ci95"].values.astype(float)

    ax.plot(x, y, linewidth=2.4, label=label)
    ax.fill_between(x, y - ci, y + ci, alpha=0.18)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Bias A−V(t): |response−audio| − |response−visual|")
ax.set_title("Time-resolved visual bias by explicit causal judgment")
ax.legend()
save_fig(fig, os.path.join(fig_dir, "overall_time_resolved_bias_same_vs_different.png"))

# -------------------------
# PLOT 2: OFFSET-SPECIFIC SAME VS DIFFERENT
# -------------------------
offsets = [10, 20, 30, 40]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, off in zip(axes, offsets):
    for label in ["Same source", "Different source"]:
        cdf = group_time_df[
            (group_time_df["offset_num"] == off) &
            (group_time_df["judgment_label"] == label)
        ].sort_values("time_sec")

        if len(cdf) == 0:
            continue

        x = cdf["time_sec"].values.astype(float)
        y = cdf["Bias_mean"].values.astype(float)
        ci = cdf["Bias_ci95"].values.astype(float)

        ax.plot(x, y, linewidth=2.2, label=label)
        ax.fill_between(x, y - ci, y + ci, alpha=0.18)

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(f"{off}° offset")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Bias A−V(t)")
    ax.legend(fontsize=8)

fig.suptitle(
    "Time-resolved visual bias by explicit judgment and offset",
    fontsize=15,
    y=1.02
)
save_fig(fig, os.path.join(fig_dir, "offset_specific_time_resolved_bias_same_vs_different.png"))

# -------------------------
# PLOT 3: DIFFERENT-SOURCE ONLY, OFFSET-SPECIFIC
# -------------------------
fig, ax = plt.subplots(figsize=(8, 5))

for off in offsets:
    cdf = group_time_df[
        (group_time_df["offset_num"] == off) &
        (group_time_df["judgment_label"] == "Different source")
    ].sort_values("time_sec")

    if len(cdf) == 0:
        continue

    ax.plot(
        cdf["time_sec"],
        cdf["Bias_mean"],
        linewidth=2.2,
        label=f"{off}°"
    )

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Bias A−V(t)")
ax.set_title("Residual visual bias on trials judged different source")
ax.legend(title="Offset")
save_fig(fig, os.path.join(fig_dir, "different_source_only_time_resolved_bias_by_offset.png"))

print("\nDONE.")
print(f"Trial timepoint CSV:   {trial_time_csv}")
print(f"Subject timepoint CSV: {subject_time_csv}")
print(f"Group timepoint CSV:   {group_time_csv}")
print(f"Overall group CSV:     {overall_csv}")
print(f"Figures saved to:      {fig_dir}")


# -------------------------
# INFERENTIAL SUMMARY WINDOW: 0.5–17.5 s
# -------------------------
from scipy.stats import ttest_1samp, ttest_rel, t as t_dist

STABLE_START_SEC = 0.5
STABLE_END_SEC = 17.5

def mean_ci_test(vals, popmean=0.0, alpha=0.05):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
        }

    m = float(np.mean(vals))
    se = float(np.std(vals, ddof=1) / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    test = ttest_1samp(vals, popmean)

    return {
        "n": n,
        "mean": m,
        "ci_low": m - crit * se,
        "ci_high": m + crit * se,
        "t": float(test.statistic),
        "df": n - 1,
        "p": float(test.pvalue),
    }

def paired_ci_test(x, y, alpha=0.05):
    tmp = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    diff = tmp["x"].astype(float).values - tmp["y"].astype(float).values

    n = len(diff)

    if n < 2:
        return {
            "n": n,
            "mean_x": np.nan,
            "mean_y": np.nan,
            "mean_diff": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
        }

    m = float(np.mean(diff))
    se = float(np.std(diff, ddof=1) / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    test = ttest_rel(tmp["x"], tmp["y"])

    return {
        "n": n,
        "mean_x": float(tmp["x"].mean()),
        "mean_y": float(tmp["y"].mean()),
        "mean_diff": m,
        "ci_low": m - crit * se,
        "ci_high": m + crit * se,
        "t": float(test.statistic),
        "df": n - 1,
        "p": float(test.pvalue),
    }

# Subject × offset × judgment summary over the stable inferential window.
stable_subject_offset_judgment_df = (
    subject_time_df
    .query("time_sec >= @STABLE_START_SEC and time_sec <= @STABLE_END_SEC")
    .groupby(["subject", "offset_num", "judgment_label"], as_index=False)
    .agg(
        Bias_AminusV_stable=("Bias_AminusV_t", "mean"),
        audio_abs_error_stable=("audio_abs_error", "mean"),
        visual_abs_error_stable=("visual_abs_error", "mean"),
        Psame=("Psame", "mean"),
        n_timepoints=("time_sec", "nunique"),
        n_trials=("n_trials", "sum"),
    )
)

stable_subject_offset_judgment_csv = os.path.join(
    csv_dir,
    "stable_0p5_17p5_subject_offset_judgment_bias.csv"
)
stable_subject_offset_judgment_df.to_csv(stable_subject_offset_judgment_csv, index=False)

# One-sample tests against zero, separately by offset and judgment.
stable_test_rows = []

for (offset, judgment), sub in stable_subject_offset_judgment_df.groupby(["offset_num", "judgment_label"]):
    stats = mean_ci_test(sub["Bias_AminusV_stable"], popmean=0.0)

    stable_test_rows.append({
        "analysis": "stable_0p5_17p5_offset_judgment_vs_zero",
        "window_start_sec": STABLE_START_SEC,
        "window_end_sec": STABLE_END_SEC,
        "offset_num": offset,
        "judgment_label": judgment,
        **stats,
    })

stable_offset_judgment_tests_df = pd.DataFrame(stable_test_rows)

stable_offset_judgment_tests_csv = os.path.join(
    csv_dir,
    "stable_0p5_17p5_offset_judgment_tests.csv"
)
stable_offset_judgment_tests_df.to_csv(stable_offset_judgment_tests_csv, index=False)

# Overall subject-first summary collapsed across offsets.
stable_subject_overall_judgment_df = (
    stable_subject_offset_judgment_df
    .groupby(["subject", "judgment_label"], as_index=False)
    .agg(
        Bias_AminusV_stable=("Bias_AminusV_stable", "mean"),
        n_offsets=("offset_num", "nunique"),
        n_trials=("n_trials", "sum"),
    )
)

stable_subject_overall_judgment_csv = os.path.join(
    csv_dir,
    "stable_0p5_17p5_subject_overall_judgment_bias.csv"
)
stable_subject_overall_judgment_df.to_csv(stable_subject_overall_judgment_csv, index=False)

# Overall one-sample tests against zero.
overall_rows = []

for judgment, sub in stable_subject_overall_judgment_df.groupby("judgment_label"):
    stats = mean_ci_test(sub["Bias_AminusV_stable"], popmean=0.0)

    overall_rows.append({
        "analysis": "stable_0p5_17p5_overall_judgment_vs_zero",
        "window_start_sec": STABLE_START_SEC,
        "window_end_sec": STABLE_END_SEC,
        "judgment_label": judgment,
        **stats,
    })

stable_overall_tests_df = pd.DataFrame(overall_rows)

# Paired same-minus-different test.
stable_pivot = stable_subject_overall_judgment_df.pivot(
    index="subject",
    columns="judgment_label",
    values="Bias_AminusV_stable"
)

if {"Same source", "Different source"}.issubset(stable_pivot.columns):
    paired_stats = paired_ci_test(
        stable_pivot["Same source"],
        stable_pivot["Different source"]
    )

    paired_row = {
        "analysis": "stable_0p5_17p5_same_minus_different_paired",
        "window_start_sec": STABLE_START_SEC,
        "window_end_sec": STABLE_END_SEC,
        "comparison": "Same source - Different source",
        **paired_stats,
    }

    stable_paired_tests_df = pd.DataFrame([paired_row])
else:
    stable_paired_tests_df = pd.DataFrame()

stable_overall_tests_csv = os.path.join(
    csv_dir,
    "stable_0p5_17p5_overall_judgment_tests.csv"
)
stable_overall_tests_df.to_csv(stable_overall_tests_csv, index=False)

stable_paired_tests_csv = os.path.join(
    csv_dir,
    "stable_0p5_17p5_same_minus_different_paired_test.csv"
)
stable_paired_tests_df.to_csv(stable_paired_tests_csv, index=False)

print("\n============================================================")
print("STABLE WINDOW INFERENTIAL SUMMARY: 0.5–17.5 s")
print("============================================================")
print(f"Subject × offset × judgment CSV: {stable_subject_offset_judgment_csv}")
print(f"Offset × judgment tests CSV:     {stable_offset_judgment_tests_csv}")
print(f"Subject overall judgment CSV:    {stable_subject_overall_judgment_csv}")
print(f"Overall judgment tests CSV:      {stable_overall_tests_csv}")
print(f"Paired same-different CSV:       {stable_paired_tests_csv}")

print("\nOffset × judgment tests:")
print(stable_offset_judgment_tests_df.to_string(index=False))

print("\nOverall judgment tests:")
print(stable_overall_tests_df.to_string(index=False))

print("\nPaired same-minus-different test:")
if len(stable_paired_tests_df) > 0:
    print(stable_paired_tests_df.to_string(index=False))
else:
    print("Paired test could not be computed because one or both judgment labels were missing.")

Fixed-offset trials with valid judgments:
offset_num  judgment_label  
10          Different source    268
            Same source         732
20          Different source    409
            Same source         591
30          Different source    693
            Same source         307
40          Different source    904
            Same source          96
dtype: int64

DONE.
Trial timepoint CSV:   C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\04_causal_judgment_visual_bias\csv_outputs\trial_timepoint_bias_by_judgment.csv
Subject timepoint CSV: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\04_causal_judgment_visual_bias\csv_outputs\subject_timepoint_bias_by_offset_judgment.csv
Group timepoint CSV:   C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\04_causal_judgment_visual_bias\csv_outputs\group_timepoint_bias_by_offset_judgment.csv
Overall group CSV:     

In [ ]:
# %% ADD-ON: PERMUTATION INFERENCE FOR TIME-RESOLVED BIAS BY SAME/DIFFERENT JUDGMENT
# UPDATED NULL STRUCTURE — FLAT OUTPUT-PATH VERSION
# FIXED TEST-KEY MATCHING VERSION
# ============================================================
# Run this AFTER:
#   ANALYSIS 4 — TIME-RESOLVED BIAS TRAJECTORIES BY SAME/DIFFERENT JUDGMENT
#
# This edited add-on performs:
#
# 1) Within-judgment whole-trial condition/offset shuffle
#    ----------------------------------------------------
#    Used for offset × judgment empirical values:
#       10° Same source, 20° Same source, ...
#       10° Different source, 20° Different source, ...
#
#    Offset labels are shuffled within subject × judgment_label.
#
#    Same-source trials are compared against a null created only from
#    other Same-source trials across offsets.
#
#    Different-source trials are compared against a null created only from
#    other Different-source trials across offsets.
#
# 2) Original whole-trial judgment-label shuffle
#    ----------------------------------------------------
#    Used ONLY for Same − Different contrasts.
#
#    Judgment labels are shuffled within subject × offset.
#
# 3) Cluster-corrected sign-flip permutation across time
#    ----------------------------------------------------
#    Unchanged from previous version.
#
# Output path change:
#   ALL CSV and PNG files are saved directly into the parent
#   time_resolved_bias_by_causal_judgment folder.
#   Nothing is saved into csv_outputs or figures.
#
# Key matching fix:
#   All test keys use _key_offset_value(), so 10 and 10.0 both become "10".
#   This prevents empty null_sub matches and NaN permutation p values.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import t as t_dist

# ============================================================
# SETTINGS
# ============================================================

N_PERM_CONDITION_SHUFFLE = 1000
N_PERM_LABEL_SHUFFLE = 1000
N_PERM_CLUSTER = 1000

RANDOM_SEED_JUDGMENT_PERM = 12345

STABLE_START_SEC = 0.5
STABLE_END_SEC = 17.5

CLUSTER_FORMING_ALPHA = 0.05

# Directional tests:
# Positive bias = closer to visual.
BIAS_ALTERNATIVE = "greater"

# Same − Different > 0 means stronger visual bias on Same-source trials.
SAME_MINUS_DIFFERENT_ALTERNATIVE = "greater"

JUDGMENT_LABELS = ["Same source", "Different source"]
OFFSETS = [10, 20, 30, 40]

try:
    FIG_DPI
except NameError:
    FIG_DPI = 180

# ============================================================
# FLAT OUTPUT DIRECTORY RESOLUTION
# ============================================================

TIMEBIAS_FOLDER_NAME = "04_causal_judgment_visual_bias"

def _find_ancestor_named(path, ancestor_name):
    """
    Walk upward from path until an ancestor folder named ancestor_name is found.
    """
    if path is None:
        return None

    try:
        cur = os.path.abspath(str(path))
    except Exception:
        return None

    while True:
        if os.path.basename(cur).lower() == ancestor_name.lower():
            return cur

        parent = os.path.dirname(cur)

        if parent == cur:
            return None

        cur = parent


def _resolve_flat_timebias_output_dir():
    """
    Returns the parent time_resolved_bias_by_causal_judgment folder.

    Priority:
      1. Existing timebias_out_dir, if defined.
      2. Parent of csv_dir if csv_dir ends in csv_outputs.
      3. Parent of fig_dir if fig_dir ends in figures.
      4. Ancestor named time_resolved_bias_by_causal_judgment.
      5. out_root/time_resolved_bias_by_causal_judgment.
      6. cwd/time_resolved_bias_by_causal_judgment.
    """
    if "timebias_out_dir" in globals():
        return os.path.abspath(timebias_out_dir)

    if "csv_dir" in globals():
        csv_dir_abs = os.path.abspath(csv_dir)

        if os.path.basename(csv_dir_abs).lower() == "csv_outputs":
            return os.path.dirname(csv_dir_abs)

        ancestor = _find_ancestor_named(csv_dir_abs, TIMEBIAS_FOLDER_NAME)
        if ancestor is not None:
            return ancestor

    if "fig_dir" in globals():
        fig_dir_abs = os.path.abspath(fig_dir)

        if os.path.basename(fig_dir_abs).lower() == "figures":
            return os.path.dirname(fig_dir_abs)

        ancestor = _find_ancestor_named(fig_dir_abs, TIMEBIAS_FOLDER_NAME)
        if ancestor is not None:
            return ancestor

    if "out_root" in globals():
        return os.path.join(os.path.abspath(out_root), TIMEBIAS_FOLDER_NAME)

    return os.path.join(os.getcwd(), TIMEBIAS_FOLDER_NAME)


timebias_perm_out_dir = _resolve_flat_timebias_output_dir()
os.makedirs(timebias_perm_out_dir, exist_ok=True)

# Override csv_dir and fig_dir so every save below is flat.
csv_dir = timebias_perm_out_dir
fig_dir = timebias_perm_out_dir

print("\n============================================================")
print("PERMUTATION ADD-ON: BIAS BY SAME/DIFFERENT JUDGMENT")
print("UPDATED NULL STRUCTURE — FLAT OUTPUT-PATH VERSION")
print("FIXED TEST-KEY MATCHING VERSION")
print("============================================================")
print(f"Stable window: {STABLE_START_SEC:.1f}–{STABLE_END_SEC:.1f} s")
print(f"Within-judgment condition/offset shuffles: {N_PERM_CONDITION_SHUFFLE}")
print(f"Same-minus-Different judgment-label shuffles: {N_PERM_LABEL_SHUFFLE}")
print(f"Cluster sign-flip permutations: {N_PERM_CLUSTER}")
print("\nAll CSV and PNG outputs will be saved directly into:")
print(timebias_perm_out_dir)

# ============================================================
# REQUIRED DATA CHECKS
# ============================================================

required_objects = ["trial_time_df", "subject_time_df", "subject_time_overall_df"]
missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise RuntimeError(
        "Missing required objects from the time-resolved judgment analysis: "
        f"{missing_objects}. Run the parent analysis cell first."
    )

required_trial_cols = [
    "subject",
    "trialIndex",
    "offset_num",
    "judgment_label",
    "time_sec",
    "Bias_AminusV_t",
    "audio_abs_error",
    "visual_abs_error",
]

missing_cols = [c for c in required_trial_cols if c not in trial_time_df.columns]

if missing_cols:
    raise RuntimeError(
        f"trial_time_df is missing required columns: {missing_cols}"
    )

# ============================================================
# SAFE FLAT SAVE HELPERS
# ============================================================

def _flat_output_path(filepath_or_filename):
    """
    Always save into timebias_perm_out_dir using only the basename.
    This prevents nested csv_outputs/figures folders from being used.
    """
    filename = os.path.basename(str(filepath_or_filename))
    return os.path.join(timebias_perm_out_dir, filename)


def _safe_to_csv(df, filepath_or_filename, index=False):
    outpath = _flat_output_path(filepath_or_filename)
    os.makedirs(timebias_perm_out_dir, exist_ok=True)
    df.to_csv(outpath, index=index)
    return outpath


def _safe_save_fig(fig, filepath_or_filename):
    outpath = _flat_output_path(filepath_or_filename)
    os.makedirs(timebias_perm_out_dir, exist_ok=True)
    fig.tight_layout()
    fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return outpath

# ============================================================
# GENERAL HELPERS
# ============================================================

def _clean_numeric(values):
    return (
        pd.Series(values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
        .values
    )


def _mean_ci_test(values, popmean=0.0, alpha=0.05):
    vals = _clean_numeric(values)
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "se": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p_two_sided": np.nan,
            "p_greater": np.nan,
            "p_less": np.nan,
        }

    m = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1))
    se = float(sd / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    t_stat, p_two = stats.ttest_1samp(vals, popmean=popmean, nan_policy="omit")

    return {
        "n": int(n),
        "mean": m,
        "se": se,
        "ci95_low": float(m - crit * se),
        "ci95_high": float(m + crit * se),
        "t": float(t_stat),
        "df": int(n - 1),
        "p_two_sided": float(p_two),
        "p_greater": float(stats.t.sf(t_stat, df=n - 1)),
        "p_less": float(stats.t.cdf(t_stat, df=n - 1)),
    }


def _paired_ci_test(x, y, alpha=0.05):
    tmp = (
        pd.DataFrame({"x": x, "y": y})
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    n = len(tmp)

    if n < 2:
        return {
            "n": n,
            "mean_x": np.nan,
            "mean_y": np.nan,
            "mean_diff": np.nan,
            "se_diff": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p_two_sided": np.nan,
            "p_greater": np.nan,
            "p_less": np.nan,
        }

    diff = tmp["x"].astype(float).values - tmp["y"].astype(float).values
    m = float(np.mean(diff))
    se = float(np.std(diff, ddof=1) / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    test = stats.ttest_rel(tmp["x"], tmp["y"], nan_policy="omit")

    return {
        "n": int(n),
        "mean_x": float(tmp["x"].mean()),
        "mean_y": float(tmp["y"].mean()),
        "mean_diff": m,
        "se_diff": se,
        "ci95_low": float(m - crit * se),
        "ci95_high": float(m + crit * se),
        "t": float(test.statistic),
        "df": int(n - 1),
        "p_two_sided": float(test.pvalue),
        "p_greater": float(stats.t.sf(test.statistic, df=n - 1)),
        "p_less": float(stats.t.cdf(test.statistic, df=n - 1)),
    }


def _empirical_pvalue(observed, null_values, alternative="two-sided"):
    null_values = _clean_numeric(null_values)

    if len(null_values) == 0 or not np.isfinite(observed):
        return np.nan

    if alternative == "greater":
        return float((np.sum(null_values >= observed) + 1) / (len(null_values) + 1))

    if alternative == "less":
        return float((np.sum(null_values <= observed) + 1) / (len(null_values) + 1))

    if alternative == "two-sided":
        null_center = float(np.nanmean(null_values))
        observed_distance = abs(observed - null_center)
        null_distance = np.abs(null_values - null_center)
        return float((np.sum(null_distance >= observed_distance) + 1) / (len(null_values) + 1))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")


def _fdr_bh(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return qvals

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    q = np.empty_like(q_ranked)
    q[order] = q_ranked
    qvals[finite] = q

    return qvals


def _holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    adj = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return adj

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    ranked_adj = np.empty(m)

    for i, pval in enumerate(ranked):
        ranked_adj[i] = (m - i) * pval

    ranked_adj = np.maximum.accumulate(ranked_adj)
    ranked_adj = np.clip(ranked_adj, 0, 1)

    out = np.empty_like(ranked_adj)
    out[order] = ranked_adj
    adj[finite] = out

    return adj


def _offset_label(offset):
    if pd.isna(offset):
        return "Collapsed across offsets"
    return f"{int(float(offset))}° offset"


def _key_offset_value(offset):
    """
    Consistent offset formatting for test keys.

    This fixes the common mismatch:
        observed key: 10.0
        null key:     10

    Both become:
        "10"

    NaN overall rows become:
        "NA"
    """
    if pd.isna(offset):
        return "NA"
    return str(int(float(offset)))


def _make_test_key(analysis, offset_num, judgment_label, comparison):
    """
    Build stable test keys that match observed and null rows.
    """
    return "||".join([
        str(analysis),
        _key_offset_value(offset_num),
        str(judgment_label),
        str(comparison),
    ])


def _safe_name(label):
    safe = str(label)
    safe = safe.replace(" ", "_")
    safe = safe.replace("|", "")
    safe = safe.replace(">", "gt")
    safe = safe.replace("<", "lt")
    safe = safe.replace("−", "minus")
    safe = safe.replace("-", "minus")
    safe = safe.replace("°", "deg")
    safe = safe.replace("/", "_")
    safe = safe.replace(":", "")
    safe = safe.replace("(", "")
    safe = safe.replace(")", "")
    safe = safe.replace(",", "")
    safe = safe.replace(".", "p")
    return safe


def _analysis_sort_key(row):
    analysis = row.get("analysis", "")
    offset = row.get("offset_num", np.nan)
    judgment = row.get("judgment_label", "")
    offset_val = int(float(offset)) if pd.notna(offset) else 999
    judgment_val = 0 if judgment == "Same source" else 1 if judgment == "Different source" else 9

    analysis_order = {
        "offset_judgment_vs_condition_shuffle": 1,
        "offset_same_minus_different": 2,
        "overall_same_minus_different": 3,
    }.get(analysis, 99)

    return analysis_order, offset_val, judgment_val

# ============================================================
# PART 1 — BUILD STABLE-WINDOW TRIAL-LEVEL INPUT
# ============================================================

stable_trial_time_for_perm = (
    trial_time_df
    .replace([np.inf, -np.inf], np.nan)
    .query("time_sec >= @STABLE_START_SEC and time_sec <= @STABLE_END_SEC")
    .copy()
)

stable_trial_time_for_perm = stable_trial_time_for_perm[
    stable_trial_time_for_perm["offset_num"].isin(OFFSETS)
    & stable_trial_time_for_perm["judgment_label"].isin(JUDGMENT_LABELS)
].copy()

if len(stable_trial_time_for_perm) == 0:
    raise RuntimeError(
        "No stable-window rows were found for the expected offsets and judgment labels."
    )

stable_trial_summary_for_perm = (
    stable_trial_time_for_perm
    .groupby(["subject", "trialIndex", "offset_num", "judgment_label"], as_index=False)
    .agg(
        Bias_AminusV_stable=("Bias_AminusV_t", "mean"),
        audio_abs_error_stable=("audio_abs_error", "mean"),
        visual_abs_error_stable=("visual_abs_error", "mean"),
        n_timepoints=("time_sec", "nunique"),
        n_timepoint_rows=("time_sec", "count"),
    )
)

stable_trial_summary_csv = _safe_to_csv(
    stable_trial_summary_for_perm,
    "perm_judgment_stable_trial_summary_input.csv",
    index=False
)

print("\nStable trial-level permutation input:")
print(f"Rows: {len(stable_trial_summary_for_perm)}")
print(f"Subjects: {stable_trial_summary_for_perm['subject'].nunique()}")
print("Offset × judgment counts:")
print(stable_trial_summary_for_perm.groupby(["offset_num", "judgment_label"]).size())
print(f"Saved: {stable_trial_summary_csv}")

# ============================================================
# PART 2 — SUMMARY FUNCTIONS
# ============================================================

def _compute_offset_judgment_subject_summary(
    trial_summary_df,
    offset_col="offset_num",
    judgment_col="judgment_label"
):
    use = trial_summary_df.copy()
    use[offset_col] = pd.to_numeric(use[offset_col], errors="coerce")
    use[judgment_col] = use[judgment_col].astype(str)

    out = (
        use
        .dropna(subset=[offset_col])
        .groupby(["subject", offset_col, judgment_col], as_index=False)
        .agg(
            Bias_AminusV_stable=("Bias_AminusV_stable", "mean"),
            audio_abs_error_stable=("audio_abs_error_stable", "mean"),
            visual_abs_error_stable=("visual_abs_error_stable", "mean"),
            n_trials=("trialIndex", "nunique"),
            n_timepoints=("n_timepoints", "sum"),
        )
        .rename(columns={
            offset_col: "offset_num",
            judgment_col: "judgment_label"
        })
    )

    out["offset_num"] = out["offset_num"].astype(int)

    return out


def _compute_overall_judgment_subject_summary(offset_judgment_subject_df):
    out = (
        offset_judgment_subject_df
        .groupby(["subject", "judgment_label"], as_index=False)
        .agg(
            Bias_AminusV_stable=("Bias_AminusV_stable", "mean"),
            audio_abs_error_stable=("audio_abs_error_stable", "mean"),
            visual_abs_error_stable=("visual_abs_error_stable", "mean"),
            n_offsets=("offset_num", "nunique"),
            n_trials=("n_trials", "sum"),
        )
    )

    return out


def _summarize_offset_judgment_vs_zero(
    offset_judgment_subject_df,
    analysis_name="offset_judgment_vs_zero"
):
    rows = []

    for (offset, judgment), sub in offset_judgment_subject_df.groupby(["offset_num", "judgment_label"]):
        stats_dict = _mean_ci_test(sub["Bias_AminusV_stable"], popmean=0.0)

        rows.append({
            "analysis": analysis_name,
            "offset_num": int(offset),
            "offset_label": _offset_label(offset),
            "judgment_label": judgment,
            "comparison": f"{_offset_label(offset)} | {judgment}",
            "effect_name": "mean_bias",
            **stats_dict,
        })

    return pd.DataFrame(rows)


def _summarize_offset_same_minus_different(offset_judgment_subject_df):
    rows = []

    for offset, sub in offset_judgment_subject_df.groupby("offset_num"):
        wide = sub.pivot(
            index="subject",
            columns="judgment_label",
            values="Bias_AminusV_stable"
        )

        if not {"Same source", "Different source"}.issubset(wide.columns):
            continue

        stats_dict = _paired_ci_test(
            wide["Same source"],
            wide["Different source"]
        )

        rows.append({
            "analysis": "offset_same_minus_different",
            "offset_num": int(offset),
            "offset_label": _offset_label(offset),
            "judgment_label": "Same source - Different source",
            "comparison": f"{_offset_label(offset)} | Same − Different > 0",
            "effect_name": "mean_same_minus_different",
            **stats_dict,
        })

    return pd.DataFrame(rows)


def _summarize_overall_same_minus_different(overall_judgment_subject_df):
    wide = overall_judgment_subject_df.pivot(
        index="subject",
        columns="judgment_label",
        values="Bias_AminusV_stable"
    )

    if not {"Same source", "Different source"}.issubset(wide.columns):
        return pd.DataFrame()

    stats_dict = _paired_ci_test(
        wide["Same source"],
        wide["Different source"]
    )

    return pd.DataFrame([{
        "analysis": "overall_same_minus_different",
        "offset_num": np.nan,
        "offset_label": "Collapsed across offsets",
        "judgment_label": "Same source - Different source",
        "comparison": "Overall Same − Different > 0",
        "effect_name": "mean_same_minus_different",
        **stats_dict,
    }])


def _summarize_observed_all(trial_summary_df):
    offset_judgment_subject_df = _compute_offset_judgment_subject_summary(
        trial_summary_df,
        offset_col="offset_num",
        judgment_col="judgment_label"
    )

    overall_judgment_subject_df = _compute_overall_judgment_subject_summary(
        offset_judgment_subject_df
    )

    offset_judgment_rows = _summarize_offset_judgment_vs_zero(
        offset_judgment_subject_df,
        analysis_name="offset_judgment_vs_condition_shuffle"
    )

    same_diff_rows = pd.concat(
        [
            _summarize_offset_same_minus_different(offset_judgment_subject_df),
            _summarize_overall_same_minus_different(overall_judgment_subject_df),
        ],
        ignore_index=True
    )

    out = pd.concat(
        [offset_judgment_rows, same_diff_rows],
        ignore_index=True
    )

    if not out.empty:
        sort_vals = out.apply(_analysis_sort_key, axis=1)
        out["_sort"] = list(sort_vals)
        out = out.sort_values("_sort").drop(columns=["_sort"]).reset_index(drop=True)

    return out, offset_judgment_subject_df, overall_judgment_subject_df


observed_judgment_summary_df, observed_offset_judgment_subject_df, observed_overall_judgment_subject_df = (
    _summarize_observed_all(stable_trial_summary_for_perm)
)

observed_judgment_summary_csv = _safe_to_csv(
    observed_judgment_summary_df,
    "perm_judgment_observed_stable_tests_summary_UPDATED_NULLS.csv",
    index=False
)

observed_offset_judgment_subject_csv = _safe_to_csv(
    observed_offset_judgment_subject_df,
    "perm_judgment_observed_subject_offset_judgment_bias_UPDATED_NULLS.csv",
    index=False
)

observed_overall_judgment_subject_csv = _safe_to_csv(
    observed_overall_judgment_subject_df,
    "perm_judgment_observed_subject_overall_judgment_bias_UPDATED_NULLS.csv",
    index=False
)

print("\nObserved stable-window judgment tests:")
try:
    display(observed_judgment_summary_df)
except NameError:
    print(observed_judgment_summary_df.to_string(index=False))

# ============================================================
# PART 3A — WITHIN-JUDGMENT CONDITION/OFFSET SHUFFLE
# ============================================================

def _shuffle_offset_labels_within_subject_judgment(trial_summary_df, rng):
    """
    Whole-trial condition/offset shuffle within subject × judgment.

    This preserves:
        - subject
        - judgment category
        - complete trial bias values
        - number of trials in each offset within subject × judgment

    This changes:
        - which offset condition each trial contributes to within that judgment.
    """
    out = trial_summary_df.copy()
    out["permuted_offset_num"] = np.nan

    for (subj, judgment), idx in out.groupby(["subject", "judgment_label"]).groups.items():
        idx = list(idx)
        offsets = out.loc[idx, "offset_num"].astype(int).values.copy()

        if len(offsets) > 1:
            out.loc[idx, "permuted_offset_num"] = rng.permutation(offsets)
        else:
            out.loc[idx, "permuted_offset_num"] = offsets

    out["permuted_offset_num"] = pd.to_numeric(out["permuted_offset_num"], errors="coerce")

    return out


def _summarize_condition_shuffle_offset_judgment_tests(
    trial_summary_df,
    offset_col="permuted_offset_num"
):
    offset_judgment_subject_df = _compute_offset_judgment_subject_summary(
        trial_summary_df,
        offset_col=offset_col,
        judgment_col="judgment_label"
    )

    out = _summarize_offset_judgment_vs_zero(
        offset_judgment_subject_df,
        analysis_name="offset_judgment_vs_condition_shuffle"
    )

    return out, offset_judgment_subject_df


rng_condition = np.random.default_rng(RANDOM_SEED_JUDGMENT_PERM)

condition_shuffle_null_rows = []

for perm_i in range(N_PERM_CONDITION_SHUFFLE):
    perm_trial_summary = _shuffle_offset_labels_within_subject_judgment(
        stable_trial_summary_for_perm,
        rng=rng_condition
    )

    perm_summary, _ = _summarize_condition_shuffle_offset_judgment_tests(
        perm_trial_summary,
        offset_col="permuted_offset_num"
    )

    if not perm_summary.empty:
        for _, row in perm_summary.iterrows():

            test_key = _make_test_key(
                row["analysis"],
                row["offset_num"],
                row["judgment_label"],
                row["comparison"]
            )

            condition_shuffle_null_rows.append({
                "perm_i": perm_i,
                "null_type": "within_subject_judgment_whole_trial_condition_offset_shuffle",
                "analysis": row["analysis"],
                "offset_num": row["offset_num"],
                "offset_label": row["offset_label"],
                "judgment_label": row["judgment_label"],
                "comparison": row["comparison"],
                "test_key": test_key,
                "mean": row.get("mean", np.nan),
                "t": row.get("t", np.nan),
                "n": row.get("n", np.nan),
            })

    if (perm_i + 1) % 250 == 0:
        print(
            f"Completed within-judgment condition/offset shuffle "
            f"{perm_i + 1}/{N_PERM_CONDITION_SHUFFLE}"
        )

condition_shuffle_null_df = pd.DataFrame(condition_shuffle_null_rows)

condition_shuffle_null_csv = _safe_to_csv(
    condition_shuffle_null_df,
    "perm_judgment_WITHIN_JUDGMENT_condition_offset_shuffle_null_long.csv",
    index=False
)

condition_shuffle_test_rows = []

observed_offset_judgment_rows = observed_judgment_summary_df[
    observed_judgment_summary_df["analysis"] == "offset_judgment_vs_condition_shuffle"
].copy()

for _, obs in observed_offset_judgment_rows.iterrows():

    test_key = _make_test_key(
        obs["analysis"],
        obs["offset_num"],
        obs["judgment_label"],
        obs["comparison"]
    )

    null_sub = condition_shuffle_null_df[
        condition_shuffle_null_df["test_key"] == test_key
    ].copy()

    observed_effect = obs["mean"]
    observed_t = obs["t"]

    null_effects_clean = _clean_numeric(null_sub["mean"])
    null_t_clean = _clean_numeric(null_sub["t"])

    if len(null_effects_clean) > 0:
        null_effect_mean = float(np.nanmean(null_effects_clean))
        null_effect_sd = float(np.nanstd(null_effects_clean, ddof=1))
        null_effect_ci95_low = float(np.nanpercentile(null_effects_clean, 2.5))
        null_effect_ci95_high = float(np.nanpercentile(null_effects_clean, 97.5))
    else:
        null_effect_mean = np.nan
        null_effect_sd = np.nan
        null_effect_ci95_low = np.nan
        null_effect_ci95_high = np.nan

    if len(null_t_clean) > 0:
        null_t_mean = float(np.nanmean(null_t_clean))
        null_t_sd = float(np.nanstd(null_t_clean, ddof=1))
    else:
        null_t_mean = np.nan
        null_t_sd = np.nan

    p_effect_directional = _empirical_pvalue(
        observed_effect,
        null_effects_clean,
        alternative=BIAS_ALTERNATIVE
    )

    p_effect_two_sided = _empirical_pvalue(
        observed_effect,
        null_effects_clean,
        alternative="two-sided"
    )

    p_t_directional = _empirical_pvalue(
        observed_t,
        null_t_clean,
        alternative=BIAS_ALTERNATIVE
    )

    p_t_two_sided = _empirical_pvalue(
        observed_t,
        null_t_clean,
        alternative="two-sided"
    )

    condition_shuffle_test_rows.append({
        "null_type": "within_subject_judgment_whole_trial_condition_offset_shuffle",
        "analysis": obs["analysis"],
        "offset_num": obs["offset_num"],
        "offset_label": obs["offset_label"],
        "judgment_label": obs["judgment_label"],
        "comparison": obs["comparison"],
        "test_key": test_key,
        "effect_name": "mean_bias",
        "observed_effect": observed_effect,
        "observed_ci95_low": obs.get("ci95_low", np.nan),
        "observed_ci95_high": obs.get("ci95_high", np.nan),
        "observed_t": observed_t,
        "observed_df": obs["df"],
        "observed_n": obs["n"],
        "parametric_p_two_sided": obs.get("p_two_sided", np.nan),
        "parametric_p_directional": obs.get("p_greater", np.nan),
        "null_effect_mean": null_effect_mean,
        "null_effect_sd": null_effect_sd,
        "null_effect_ci95_low": null_effect_ci95_low,
        "null_effect_ci95_high": null_effect_ci95_high,
        "null_t_mean": null_t_mean,
        "null_t_sd": null_t_sd,
        "directional_alternative": BIAS_ALTERNATIVE,
        "permutation_p_effect_directional": p_effect_directional,
        "permutation_p_effect_two_sided": p_effect_two_sided,
        "permutation_p_t_directional": p_t_directional,
        "permutation_p_t_two_sided": p_t_two_sided,
        "n_permutations": N_PERM_CONDITION_SHUFFLE,
    })

condition_shuffle_tests_df = pd.DataFrame(condition_shuffle_test_rows)

for p_col in [
    "permutation_p_effect_directional",
    "permutation_p_effect_two_sided",
    "permutation_p_t_directional",
    "permutation_p_t_two_sided",
]:
    condition_shuffle_tests_df[f"{p_col}_fdr_condition_shuffle"] = _fdr_bh(
        condition_shuffle_tests_df[p_col].values
    )
    condition_shuffle_tests_df[f"{p_col}_holm_condition_shuffle"] = _holm_bonferroni(
        condition_shuffle_tests_df[p_col].values
    )

condition_shuffle_tests_csv = _safe_to_csv(
    condition_shuffle_tests_df,
    "perm_judgment_WITHIN_JUDGMENT_condition_offset_shuffle_tests.csv",
    index=False
)

print("\nWithin-judgment condition/offset shuffle tests:")
try:
    display(condition_shuffle_tests_df)
except NameError:
    print(condition_shuffle_tests_df.to_string(index=False))

print("\nCondition-shuffle key-match diagnostic:")
print(f"Observed offset×judgment rows: {len(observed_offset_judgment_rows)}")
print(f"Null rows: {len(condition_shuffle_null_df)}")
print(f"Rows with empty null distributions: {condition_shuffle_tests_df['null_effect_mean'].isna().sum()}")

# ============================================================
# PART 3B — ORIGINAL JUDGMENT-LABEL SHUFFLE FOR SAME − DIFFERENT
# ============================================================

def _shuffle_judgment_labels_within_subject_offset(trial_summary_df, rng):
    """
    Whole-trial judgment-label shuffle within subject × offset.

    This preserves physical offset and complete-trial trajectories,
    and tests whether Same-source bias exceeds Different-source bias
    more than expected under exchangeable judgment labels.
    """
    out = trial_summary_df.copy()
    out["permuted_judgment_label"] = np.nan

    for (subj, offset), idx in out.groupby(["subject", "offset_num"]).groups.items():
        idx = list(idx)
        labels = out.loc[idx, "judgment_label"].astype(str).values.copy()

        if len(labels) > 1:
            out.loc[idx, "permuted_judgment_label"] = rng.permutation(labels)
        else:
            out.loc[idx, "permuted_judgment_label"] = labels

    out["permuted_judgment_label"] = out["permuted_judgment_label"].astype(str)

    return out


def _summarize_same_minus_different_tests(
    trial_summary_df,
    judgment_col="judgment_label"
):
    offset_judgment_subject_df = _compute_offset_judgment_subject_summary(
        trial_summary_df,
        offset_col="offset_num",
        judgment_col=judgment_col
    )

    overall_judgment_subject_df = _compute_overall_judgment_subject_summary(
        offset_judgment_subject_df
    )

    out = pd.concat(
        [
            _summarize_offset_same_minus_different(offset_judgment_subject_df),
            _summarize_overall_same_minus_different(overall_judgment_subject_df),
        ],
        ignore_index=True
    )

    return out, offset_judgment_subject_df, overall_judgment_subject_df


rng_label = np.random.default_rng(RANDOM_SEED_JUDGMENT_PERM + 50000)

label_shuffle_null_rows = []

for perm_i in range(N_PERM_LABEL_SHUFFLE):
    perm_trial_summary = _shuffle_judgment_labels_within_subject_offset(
        stable_trial_summary_for_perm,
        rng=rng_label
    )

    perm_summary, _, _ = _summarize_same_minus_different_tests(
        perm_trial_summary,
        judgment_col="permuted_judgment_label"
    )

    if not perm_summary.empty:
        for _, row in perm_summary.iterrows():

            test_key = _make_test_key(
                row["analysis"],
                row["offset_num"],
                row["judgment_label"],
                row["comparison"]
            )

            label_shuffle_null_rows.append({
                "perm_i": perm_i,
                "null_type": "within_subject_offset_whole_trial_judgment_label_shuffle",
                "analysis": row["analysis"],
                "offset_num": row["offset_num"],
                "offset_label": row["offset_label"],
                "judgment_label": row["judgment_label"],
                "comparison": row["comparison"],
                "test_key": test_key,
                "mean_diff": row.get("mean_diff", np.nan),
                "t": row.get("t", np.nan),
                "n": row.get("n", np.nan),
            })

    if (perm_i + 1) % 250 == 0:
        print(
            f"Completed Same-minus-Different judgment-label shuffle "
            f"{perm_i + 1}/{N_PERM_LABEL_SHUFFLE}"
        )

judgment_label_shuffle_null_df = pd.DataFrame(label_shuffle_null_rows)

judgment_label_shuffle_null_csv = _safe_to_csv(
    judgment_label_shuffle_null_df,
    "perm_judgment_LABEL_SHUFFLE_same_minus_different_null_long.csv",
    index=False
)

label_shuffle_test_rows = []

observed_same_diff_rows = observed_judgment_summary_df[
    observed_judgment_summary_df["analysis"].isin([
        "offset_same_minus_different",
        "overall_same_minus_different"
    ])
].copy()

for _, obs in observed_same_diff_rows.iterrows():

    test_key = _make_test_key(
        obs["analysis"],
        obs["offset_num"],
        obs["judgment_label"],
        obs["comparison"]
    )

    null_sub = judgment_label_shuffle_null_df[
        judgment_label_shuffle_null_df["test_key"] == test_key
    ].copy()

    observed_effect = obs["mean_diff"]
    observed_t = obs["t"]

    null_effects_clean = _clean_numeric(null_sub["mean_diff"])
    null_t_clean = _clean_numeric(null_sub["t"])

    if len(null_effects_clean) > 0:
        null_effect_mean = float(np.nanmean(null_effects_clean))
        null_effect_sd = float(np.nanstd(null_effects_clean, ddof=1))
        null_effect_ci95_low = float(np.nanpercentile(null_effects_clean, 2.5))
        null_effect_ci95_high = float(np.nanpercentile(null_effects_clean, 97.5))
    else:
        null_effect_mean = np.nan
        null_effect_sd = np.nan
        null_effect_ci95_low = np.nan
        null_effect_ci95_high = np.nan

    if len(null_t_clean) > 0:
        null_t_mean = float(np.nanmean(null_t_clean))
        null_t_sd = float(np.nanstd(null_t_clean, ddof=1))
    else:
        null_t_mean = np.nan
        null_t_sd = np.nan

    p_effect_directional = _empirical_pvalue(
        observed_effect,
        null_effects_clean,
        alternative=SAME_MINUS_DIFFERENT_ALTERNATIVE
    )

    p_effect_two_sided = _empirical_pvalue(
        observed_effect,
        null_effects_clean,
        alternative="two-sided"
    )

    p_t_directional = _empirical_pvalue(
        observed_t,
        null_t_clean,
        alternative=SAME_MINUS_DIFFERENT_ALTERNATIVE
    )

    p_t_two_sided = _empirical_pvalue(
        observed_t,
        null_t_clean,
        alternative="two-sided"
    )

    label_shuffle_test_rows.append({
        "null_type": "within_subject_offset_whole_trial_judgment_label_shuffle",
        "analysis": obs["analysis"],
        "offset_num": obs["offset_num"],
        "offset_label": obs["offset_label"],
        "judgment_label": obs["judgment_label"],
        "comparison": obs["comparison"],
        "test_key": test_key,
        "effect_name": "mean_same_minus_different",
        "observed_effect": observed_effect,
        "observed_ci95_low": obs.get("ci95_low", np.nan),
        "observed_ci95_high": obs.get("ci95_high", np.nan),
        "observed_t": observed_t,
        "observed_df": obs["df"],
        "observed_n": obs["n"],
        "parametric_p_two_sided": obs.get("p_two_sided", np.nan),
        "parametric_p_directional": obs.get("p_greater", np.nan),
        "null_effect_mean": null_effect_mean,
        "null_effect_sd": null_effect_sd,
        "null_effect_ci95_low": null_effect_ci95_low,
        "null_effect_ci95_high": null_effect_ci95_high,
        "null_t_mean": null_t_mean,
        "null_t_sd": null_t_sd,
        "directional_alternative": SAME_MINUS_DIFFERENT_ALTERNATIVE,
        "permutation_p_effect_directional": p_effect_directional,
        "permutation_p_effect_two_sided": p_effect_two_sided,
        "permutation_p_t_directional": p_t_directional,
        "permutation_p_t_two_sided": p_t_two_sided,
        "n_permutations": N_PERM_LABEL_SHUFFLE,
    })

judgment_label_shuffle_tests_df = pd.DataFrame(label_shuffle_test_rows)

for p_col in [
    "permutation_p_effect_directional",
    "permutation_p_effect_two_sided",
    "permutation_p_t_directional",
    "permutation_p_t_two_sided",
]:
    judgment_label_shuffle_tests_df[f"{p_col}_fdr_label_shuffle"] = _fdr_bh(
        judgment_label_shuffle_tests_df[p_col].values
    )
    judgment_label_shuffle_tests_df[f"{p_col}_holm_label_shuffle"] = _holm_bonferroni(
        judgment_label_shuffle_tests_df[p_col].values
    )

judgment_label_shuffle_tests_csv = _safe_to_csv(
    judgment_label_shuffle_tests_df,
    "perm_judgment_LABEL_SHUFFLE_same_minus_different_tests.csv",
    index=False
)

print("\nSame-minus-Different judgment-label shuffle tests:")
try:
    display(judgment_label_shuffle_tests_df)
except NameError:
    print(judgment_label_shuffle_tests_df.to_string(index=False))

print("\nLabel-shuffle key-match diagnostic:")
print(f"Observed Same-minus-Different rows: {len(observed_same_diff_rows)}")
print(f"Null rows: {len(judgment_label_shuffle_null_df)}")
print(f"Rows with empty null distributions: {judgment_label_shuffle_tests_df['null_effect_mean'].isna().sum()}")

# ============================================================
# PART 3C — COMBINED PERMUTATION TEST TABLE
# ============================================================

combined_permutation_tests_df = pd.concat(
    [
        condition_shuffle_tests_df,
        judgment_label_shuffle_tests_df,
    ],
    ignore_index=True,
    sort=False
)

for p_col in [
    "permutation_p_effect_directional",
    "permutation_p_effect_two_sided",
    "permutation_p_t_directional",
    "permutation_p_t_two_sided",
]:
    combined_permutation_tests_df[f"{p_col}_fdr_all_permutation_tests"] = _fdr_bh(
        combined_permutation_tests_df[p_col].values
    )
    combined_permutation_tests_df[f"{p_col}_holm_all_permutation_tests"] = _holm_bonferroni(
        combined_permutation_tests_df[p_col].values
    )

combined_permutation_tests_csv = _safe_to_csv(
    combined_permutation_tests_df,
    "perm_judgment_COMBINED_updated_nulls_tests.csv",
    index=False
)

print("\nCombined updated-null permutation tests:")
try:
    display(combined_permutation_tests_df)
except NameError:
    print(combined_permutation_tests_df.to_string(index=False))

# ============================================================
# PART 4 — CLUSTER-CORRECTED SIGN-FLIP PERMUTATIONS ACROSS TIME
# ============================================================

def _one_sample_t_by_time(matrix):
    matrix = np.asarray(matrix, dtype=float)
    n_time = matrix.shape[1]

    means = np.full(n_time, np.nan)
    t_vals = np.full(n_time, np.nan)
    p_two_vals = np.full(n_time, np.nan)
    ns = np.zeros(n_time, dtype=int)

    for j in range(n_time):
        vals = matrix[:, j]
        vals = vals[np.isfinite(vals)]
        ns[j] = len(vals)

        if len(vals) < 2:
            continue

        means[j] = float(np.nanmean(vals))

        if np.nanstd(vals, ddof=1) == 0:
            continue

        t_stat, p_two = stats.ttest_1samp(vals, popmean=0.0, nan_policy="omit")
        t_vals[j] = float(t_stat)
        p_two_vals[j] = float(p_two)

    return means, t_vals, p_two_vals, ns


def _p_from_t_for_alternative(t_vals, ns, alternative="greater"):
    t_vals = np.asarray(t_vals, dtype=float)
    ns = np.asarray(ns, dtype=int)

    p_vals = np.full_like(t_vals, np.nan, dtype=float)

    for j, t_stat in enumerate(t_vals):
        if not np.isfinite(t_stat) or ns[j] < 2:
            continue

        df = ns[j] - 1

        if alternative == "greater":
            p_vals[j] = stats.t.sf(t_stat, df=df)
        elif alternative == "less":
            p_vals[j] = stats.t.cdf(t_stat, df=df)
        elif alternative == "two-sided":
            p_vals[j] = 2 * stats.t.sf(abs(t_stat), df=df)
        else:
            raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")

    return p_vals


def _find_contiguous_clusters(mask):
    mask = np.asarray(mask, dtype=bool)
    clusters = []
    start = None

    for i, val in enumerate(mask):
        if val and start is None:
            start = i

        if (not val or i == len(mask) - 1) and start is not None:
            end = i if not val else i + 1
            clusters.append(np.arange(start, end))
            start = None

    return clusters


def _cluster_stat_from_t(t_vals, cluster_idx, alternative="greater"):
    vals = np.asarray(t_vals, dtype=float)[cluster_idx]
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan

    if alternative == "greater":
        return float(np.sum(vals))

    if alternative == "less":
        return float(np.sum(-vals))

    if alternative == "two-sided":
        return float(np.sum(np.abs(vals)))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")


def _cluster_corrected_signflip_time_test(
    matrix,
    timepoints,
    alternative="greater",
    cluster_alpha=0.05,
    n_perm=10000,
    seed=12345
):
    rng = np.random.default_rng(seed)

    matrix = np.asarray(matrix, dtype=float)
    timepoints = np.asarray(timepoints, dtype=float)

    if matrix.size == 0 or matrix.shape[1] == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    valid_time_n = np.sum(np.isfinite(matrix), axis=0)
    valid_times = valid_time_n >= 2

    matrix_valid = matrix[:, valid_times]
    timepoints_valid = timepoints[valid_times]

    if matrix_valid.shape[1] == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    means, t_vals, p_two, ns = _one_sample_t_by_time(matrix_valid)
    p_vals = _p_from_t_for_alternative(t_vals, ns, alternative=alternative)

    cluster_mask = p_vals < cluster_alpha

    if alternative == "greater":
        cluster_mask = cluster_mask & (t_vals > 0)
    elif alternative == "less":
        cluster_mask = cluster_mask & (t_vals < 0)

    obs_clusters = _find_contiguous_clusters(cluster_mask)

    obs_cluster_rows = []

    for cluster_i, idx in enumerate(obs_clusters, start=1):
        c_stat = _cluster_stat_from_t(t_vals, idx, alternative=alternative)

        obs_cluster_rows.append({
            "cluster_id": cluster_i,
            "start_time_sec": float(timepoints_valid[idx[0]]),
            "end_time_sec": float(timepoints_valid[idx[-1]]),
            "n_timepoints": int(len(idx)),
            "cluster_stat": c_stat,
            "mean_difference_in_cluster": float(np.nanmean(means[idx])),
            "max_abs_t_in_cluster": float(np.nanmax(np.abs(t_vals[idx]))),
            "peak_t_in_cluster": float(np.nanmax(t_vals[idx])),
            "min_t_in_cluster": float(np.nanmin(t_vals[idx])),
        })

    n_subjects = matrix_valid.shape[0]
    max_cluster_stats = np.zeros(n_perm, dtype=float)

    for perm_i in range(n_perm):
        signs = rng.choice([-1, 1], size=n_subjects)
        perm_matrix = matrix_valid * signs[:, None]

        _, perm_t_vals, _, perm_ns = _one_sample_t_by_time(perm_matrix)

        perm_p_vals = _p_from_t_for_alternative(
            perm_t_vals,
            perm_ns,
            alternative=alternative
        )

        perm_mask = perm_p_vals < cluster_alpha

        if alternative == "greater":
            perm_mask = perm_mask & (perm_t_vals > 0)
        elif alternative == "less":
            perm_mask = perm_mask & (perm_t_vals < 0)

        perm_clusters = _find_contiguous_clusters(perm_mask)

        if len(perm_clusters) == 0:
            max_cluster_stats[perm_i] = 0.0
        else:
            perm_stats = [
                _cluster_stat_from_t(perm_t_vals, idx, alternative=alternative)
                for idx in perm_clusters
            ]
            max_cluster_stats[perm_i] = float(np.nanmax(perm_stats))

    cluster_df = pd.DataFrame(obs_cluster_rows)

    if not cluster_df.empty:
        cluster_df["cluster_corrected_p"] = [
            float((np.sum(max_cluster_stats >= stat) + 1) / (n_perm + 1))
            for stat in cluster_df["cluster_stat"].values
        ]

    binwise_df = pd.DataFrame({
        "time_sec": timepoints_valid,
        "mean_difference": means,
        "t": t_vals,
        "p_uncorrected": p_vals,
        "n_subjects": ns,
        "passes_cluster_forming_threshold": cluster_mask,
    })

    null_df = pd.DataFrame({
        "perm_i": np.arange(n_perm),
        "max_cluster_stat": max_cluster_stats,
    })

    return binwise_df, cluster_df, null_df


def _build_bias_matrix_vs_zero(data_df, value_col="Bias_AminusV_t"):
    d = data_df.copy()
    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=["subject", "time_sec", value_col]).copy()

    if d.empty:
        return np.array([]), np.empty((0, 0)), []

    d["time_sec_rounded"] = d["time_sec"].astype(float).round(6)

    all_times = np.sort(d["time_sec_rounded"].unique().astype(float))
    all_subjects = np.sort(d["subject"].unique())

    wide = d.pivot_table(
        index="subject",
        columns="time_sec_rounded",
        values=value_col,
        aggfunc="mean"
    )

    wide = wide.reindex(index=all_subjects, columns=all_times)

    return all_times, wide.values.astype(float), list(all_subjects)


def _build_same_minus_different_matrix(data_df):
    d = data_df.copy()
    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=["subject", "time_sec", "judgment_label", "Bias_AminusV_t"]).copy()

    if d.empty:
        return np.array([]), np.empty((0, 0)), []

    d["time_sec_rounded"] = d["time_sec"].astype(float).round(6)

    pivot = d.pivot_table(
        index=["subject", "time_sec_rounded"],
        columns="judgment_label",
        values="Bias_AminusV_t",
        aggfunc="mean"
    ).reset_index()

    if not {"Same source", "Different source"}.issubset(pivot.columns):
        return np.array([]), np.empty((0, 0)), []

    pivot["SameMinusDifferent_Bias_AminusV_t"] = (
        pivot["Same source"] - pivot["Different source"]
    )

    all_times = np.sort(pivot["time_sec_rounded"].unique().astype(float))
    all_subjects = np.sort(pivot["subject"].unique())

    wide = pivot.pivot_table(
        index="subject",
        columns="time_sec_rounded",
        values="SameMinusDifferent_Bias_AminusV_t",
        aggfunc="mean"
    )

    wide = wide.reindex(index=all_subjects, columns=all_times)

    return all_times, wide.values.astype(float), list(all_subjects)


subject_time_cluster_df = (
    subject_time_df
    .replace([np.inf, -np.inf], np.nan)
    .query("time_sec >= @STABLE_START_SEC and time_sec <= @STABLE_END_SEC")
    .copy()
)

subject_time_overall_cluster_df = (
    subject_time_overall_df
    .replace([np.inf, -np.inf], np.nan)
    .query("time_sec >= @STABLE_START_SEC and time_sec <= @STABLE_END_SEC")
    .copy()
)

cluster_binwise_rows = []
cluster_corrected_rows = []
cluster_null_rows = []

cluster_specs = []

for offset in OFFSETS:
    for judgment in JUDGMENT_LABELS:
        cluster_specs.append({
            "analysis": "offset_judgment_vs_zero_timecourse",
            "offset_num": offset,
            "offset_label": _offset_label(offset),
            "judgment_label": judgment,
            "comparison": f"{_offset_label(offset)} | {judgment} bias > 0",
            "data_type": "bias_vs_zero",
            "data_df": subject_time_cluster_df[
                (subject_time_cluster_df["offset_num"] == offset)
                & (subject_time_cluster_df["judgment_label"] == judgment)
            ].copy(),
            "alternative": BIAS_ALTERNATIVE,
        })

for judgment in JUDGMENT_LABELS:
    cluster_specs.append({
        "analysis": "overall_judgment_vs_zero_timecourse",
        "offset_num": np.nan,
        "offset_label": "Collapsed across offsets",
        "judgment_label": judgment,
        "comparison": f"Overall {judgment} bias > 0",
        "data_type": "bias_vs_zero",
        "data_df": subject_time_overall_cluster_df[
            subject_time_overall_cluster_df["judgment_label"] == judgment
        ].copy(),
        "alternative": BIAS_ALTERNATIVE,
    })

for offset in OFFSETS:
    cluster_specs.append({
        "analysis": "offset_same_minus_different_timecourse",
        "offset_num": offset,
        "offset_label": _offset_label(offset),
        "judgment_label": "Same source - Different source",
        "comparison": f"{_offset_label(offset)} | Same − Different > 0",
        "data_type": "same_minus_different",
        "data_df": subject_time_cluster_df[
            subject_time_cluster_df["offset_num"] == offset
        ].copy(),
        "alternative": SAME_MINUS_DIFFERENT_ALTERNATIVE,
    })

cluster_specs.append({
    "analysis": "overall_same_minus_different_timecourse",
    "offset_num": np.nan,
    "offset_label": "Collapsed across offsets",
    "judgment_label": "Same source - Different source",
    "comparison": "Overall Same − Different > 0",
    "data_type": "same_minus_different",
    "data_df": subject_time_overall_cluster_df.copy(),
    "alternative": SAME_MINUS_DIFFERENT_ALTERNATIVE,
})

for spec_i, spec in enumerate(cluster_specs):
    d = spec["data_df"]

    if d.empty:
        print(f"Skipping empty cluster spec: {spec['comparison']}")
        continue

    if spec["data_type"] == "bias_vs_zero":
        timepoints, matrix, subjects_used = _build_bias_matrix_vs_zero(
            d,
            value_col="Bias_AminusV_t"
        )
    else:
        timepoints, matrix, subjects_used = _build_same_minus_different_matrix(d)

    if matrix.size == 0 or matrix.shape[1] == 0:
        print(f"Skipping cluster spec with no matrix: {spec['comparison']}")
        continue

    binwise_df, cluster_df, null_df = _cluster_corrected_signflip_time_test(
        matrix=matrix,
        timepoints=timepoints,
        alternative=spec["alternative"],
        cluster_alpha=CLUSTER_FORMING_ALPHA,
        n_perm=N_PERM_CLUSTER,
        seed=RANDOM_SEED_JUDGMENT_PERM + 1000 + spec_i
    )

    if not binwise_df.empty:
        binwise_df["analysis"] = spec["analysis"]
        binwise_df["offset_num"] = spec["offset_num"]
        binwise_df["offset_label"] = spec["offset_label"]
        binwise_df["judgment_label"] = spec["judgment_label"]
        binwise_df["comparison"] = spec["comparison"]
        binwise_df["data_type"] = spec["data_type"]
        binwise_df["alternative"] = spec["alternative"]
        binwise_df["n_subjects_total"] = len(subjects_used)
        cluster_binwise_rows.append(binwise_df)

    if not cluster_df.empty:
        cluster_df["analysis"] = spec["analysis"]
        cluster_df["offset_num"] = spec["offset_num"]
        cluster_df["offset_label"] = spec["offset_label"]
        cluster_df["judgment_label"] = spec["judgment_label"]
        cluster_df["comparison"] = spec["comparison"]
        cluster_df["data_type"] = spec["data_type"]
        cluster_df["alternative"] = spec["alternative"]
        cluster_df["n_subjects_total"] = len(subjects_used)
        cluster_corrected_rows.append(cluster_df)

    if not null_df.empty:
        null_df["analysis"] = spec["analysis"]
        null_df["offset_num"] = spec["offset_num"]
        null_df["offset_label"] = spec["offset_label"]
        null_df["judgment_label"] = spec["judgment_label"]
        null_df["comparison"] = spec["comparison"]
        null_df["data_type"] = spec["data_type"]
        null_df["alternative"] = spec["alternative"]
        cluster_null_rows.append(null_df)

    print(f"Completed cluster test: {spec['comparison']}")

judgment_cluster_binwise_df = (
    pd.concat(cluster_binwise_rows, ignore_index=True)
    if len(cluster_binwise_rows) > 0
    else pd.DataFrame()
)

judgment_cluster_corrected_df = (
    pd.concat(cluster_corrected_rows, ignore_index=True)
    if len(cluster_corrected_rows) > 0
    else pd.DataFrame()
)

judgment_cluster_null_df = (
    pd.concat(cluster_null_rows, ignore_index=True)
    if len(cluster_null_rows) > 0
    else pd.DataFrame()
)

if not judgment_cluster_corrected_df.empty:
    judgment_cluster_corrected_df["cluster_corrected_p_fdr_all_clusters"] = _fdr_bh(
        judgment_cluster_corrected_df["cluster_corrected_p"].values
    )

    judgment_cluster_corrected_df["cluster_corrected_p_holm_all_clusters"] = _holm_bonferroni(
        judgment_cluster_corrected_df["cluster_corrected_p"].values
    )

    judgment_cluster_corrected_df["cluster_corrected_p_fdr_within_analysis"] = np.nan

    for analysis_name, idx in judgment_cluster_corrected_df.groupby("analysis").groups.items():
        idx = list(idx)
        judgment_cluster_corrected_df.loc[idx, "cluster_corrected_p_fdr_within_analysis"] = _fdr_bh(
            judgment_cluster_corrected_df.loc[idx, "cluster_corrected_p"].values
        )

judgment_cluster_binwise_csv = _safe_to_csv(
    judgment_cluster_binwise_df,
    "perm_judgment_cluster_binwise_time_tests_UPDATED_NULLS.csv",
    index=False
)

judgment_cluster_corrected_csv = _safe_to_csv(
    judgment_cluster_corrected_df,
    "perm_judgment_cluster_corrected_time_tests_UPDATED_NULLS.csv",
    index=False
)

judgment_cluster_null_csv = _safe_to_csv(
    judgment_cluster_null_df,
    "perm_judgment_cluster_null_distribution_UPDATED_NULLS.csv",
    index=False
)

print("\nCluster-corrected sign-flip results:")
try:
    display(judgment_cluster_corrected_df)
except NameError:
    print(judgment_cluster_corrected_df.to_string(index=False))

# ============================================================
# PART 5 — PLOTS
# ============================================================

for _, row in condition_shuffle_tests_df.iterrows():
    comparison = str(row["comparison"])

    test_key = _make_test_key(
        row["analysis"],
        row["offset_num"],
        row["judgment_label"],
        row["comparison"]
    )

    null_sub = condition_shuffle_null_df[
        condition_shuffle_null_df["test_key"] == test_key
    ].copy()

    if null_sub.empty:
        continue

    null_vals = _clean_numeric(null_sub["mean"])
    observed = row["observed_effect"]

    if len(null_vals) == 0:
        continue

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(null_vals, bins=40, alpha=0.75)
    ax.axvline(0, linewidth=1)
    ax.axvline(observed, linestyle="--", linewidth=2)
    ax.set_xlabel("Mean Bias_AminusV_stable")
    ax.set_ylabel("Permutation count")
    ax.set_title(f"Within-judgment condition/offset null\n{comparison}")

    _safe_save_fig(
        fig,
        f"perm_judgment_WITHIN_JUDGMENT_condition_offset_null_{_safe_name(comparison)}.png"
    )

for _, row in judgment_label_shuffle_tests_df.iterrows():
    comparison = str(row["comparison"])

    test_key = _make_test_key(
        row["analysis"],
        row["offset_num"],
        row["judgment_label"],
        row["comparison"]
    )

    null_sub = judgment_label_shuffle_null_df[
        judgment_label_shuffle_null_df["test_key"] == test_key
    ].copy()

    if null_sub.empty:
        continue

    null_vals = _clean_numeric(null_sub["mean_diff"])
    observed = row["observed_effect"]

    if len(null_vals) == 0:
        continue

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(null_vals, bins=40, alpha=0.75)
    ax.axvline(0, linewidth=1)
    ax.axvline(observed, linestyle="--", linewidth=2)
    ax.set_xlabel("Mean Same − Different bias")
    ax.set_ylabel("Permutation count")
    ax.set_title(f"Judgment-label shuffle null\n{comparison}")

    _safe_save_fig(
        fig,
        f"perm_judgment_LABEL_SHUFFLE_same_minus_different_null_{_safe_name(comparison)}.png"
    )


def _plot_cluster_timecourse(spec_row):
    analysis = spec_row["analysis"]
    offset = spec_row["offset_num"]
    judgment = spec_row["judgment_label"]
    comparison = spec_row["comparison"]
    data_type = spec_row["data_type"]

    if data_type == "bias_vs_zero":
        if pd.notna(offset):
            d = subject_time_cluster_df[
                (subject_time_cluster_df["offset_num"] == int(offset))
                & (subject_time_cluster_df["judgment_label"] == judgment)
            ].copy()
        else:
            d = subject_time_overall_cluster_df[
                subject_time_overall_cluster_df["judgment_label"] == judgment
            ].copy()

        value_col = "Bias_AminusV_t"
        y_label = "Bias A−V(t)"
        curve_rows = []

        for time_sec, sub in d.groupby("time_sec"):
            stats_dict = _mean_ci_test(sub[value_col], popmean=0.0)
            curve_rows.append({
                "time_sec": float(time_sec),
                "mean": stats_dict["mean"],
                "ci95_low": stats_dict["ci95_low"],
                "ci95_high": stats_dict["ci95_high"],
                "n": stats_dict["n"],
            })

        curve_df = pd.DataFrame(curve_rows).sort_values("time_sec")

    else:
        if pd.notna(offset):
            d = subject_time_cluster_df[
                subject_time_cluster_df["offset_num"] == int(offset)
            ].copy()
        else:
            d = subject_time_overall_cluster_df.copy()

        pivot = d.pivot_table(
            index=["subject", "time_sec"],
            columns="judgment_label",
            values="Bias_AminusV_t",
            aggfunc="mean"
        ).reset_index()

        if not {"Same source", "Different source"}.issubset(pivot.columns):
            return

        pivot["SameMinusDifferent_Bias_AminusV_t"] = (
            pivot["Same source"] - pivot["Different source"]
        )

        y_label = "Same − Different Bias A−V(t)"
        curve_rows = []

        for time_sec, sub in pivot.groupby("time_sec"):
            stats_dict = _mean_ci_test(
                sub["SameMinusDifferent_Bias_AminusV_t"],
                popmean=0.0
            )

            curve_rows.append({
                "time_sec": float(time_sec),
                "mean": stats_dict["mean"],
                "ci95_low": stats_dict["ci95_low"],
                "ci95_high": stats_dict["ci95_high"],
                "n": stats_dict["n"],
            })

        curve_df = pd.DataFrame(curve_rows).sort_values("time_sec")

    if curve_df.empty:
        return

    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    x = curve_df["time_sec"].astype(float).values
    y = curve_df["mean"].astype(float).values
    lo = curve_df["ci95_low"].astype(float).values
    hi = curve_df["ci95_high"].astype(float).values

    ax.plot(x, y, linewidth=2.3)
    ax.fill_between(x, lo, hi, alpha=0.20)
    ax.axhline(0, linestyle="--", linewidth=1)

    sig_clusters = judgment_cluster_corrected_df[
        (judgment_cluster_corrected_df["analysis"] == analysis)
        & (judgment_cluster_corrected_df["comparison"] == comparison)
        & (judgment_cluster_corrected_df["cluster_corrected_p"] < 0.05)
    ].copy()

    if not sig_clusters.empty:
        y_min, y_max = ax.get_ylim()
        y_pos = y_max - 0.08 * (y_max - y_min)
        label_added = False

        for _, crow in sig_clusters.iterrows():
            ax.axvspan(
                crow["start_time_sec"],
                crow["end_time_sec"],
                alpha=0.12,
                label="cluster p < .05" if not label_added else "_nolegend_"
            )

            ax.text(
                x=(crow["start_time_sec"] + crow["end_time_sec"]) / 2,
                y=y_pos,
                s=f"p={crow['cluster_corrected_p']:.4f}",
                ha="center",
                va="center",
                fontsize=8
            )

            label_added = True

    ax.set_xlabel("Time (s)")
    ax.set_ylabel(y_label)
    ax.set_title(f"Cluster-corrected timecourse\n{comparison}")

    handles, labels = ax.get_legend_handles_labels()

    if len(handles) > 0:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), frameon=False)

    _safe_save_fig(
        fig,
        f"perm_judgment_cluster_timecourse_{_safe_name(comparison)}.png"
    )


if not judgment_cluster_binwise_df.empty:
    plot_specs = (
        judgment_cluster_binwise_df[
            ["analysis", "offset_num", "judgment_label", "comparison", "data_type"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    for _, spec_row in plot_specs.iterrows():
        _plot_cluster_timecourse(spec_row)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n============================================================")
print("JUDGMENT-BIAS PERMUTATION ADD-ON COMPLETE")
print("UPDATED NULL STRUCTURE — FLAT OUTPUT-PATH VERSION")
print("FIXED TEST-KEY MATCHING VERSION")
print("============================================================")

print("\nAll outputs were saved directly into:")
print(timebias_perm_out_dir)

print("\nWithin-judgment condition/offset shuffle:")
print("- Used for offset × judgment empirical values.")
print("- Same-source trials were permuted only across Same-source trials from other offsets/conditions.")
print("- Different-source trials were permuted only across Different-source trials from other offsets/conditions.")
print("- This avoids inflating the Different-source null with Same-source trials.")

print("\nSame-minus-Different judgment-label shuffle:")
print("- Same − Different contrasts were kept as in the original code.")
print("- Judgment labels were shuffled across complete trials within subject × offset.")

print("\nCluster-corrected sign-flip:")
print("- Subject-level time-resolved curves were sign-flipped as whole curves.")
print("- This preserves temporal dependence and tests for contiguous positive visual-bias clusters.")

print("\nKey output files:")
print(f"Stable trial input:                         {stable_trial_summary_csv}")
print(f"Observed stable summary:                    {observed_judgment_summary_csv}")
print(f"Observed subject offset summary:            {observed_offset_judgment_subject_csv}")
print(f"Observed subject overall summary:           {observed_overall_judgment_subject_csv}")
print(f"Within-judgment condition null:             {condition_shuffle_null_csv}")
print(f"Within-judgment condition tests:            {condition_shuffle_tests_csv}")
print(f"Same-minus-Different label null:            {judgment_label_shuffle_null_csv}")
print(f"Same-minus-Different label tests:           {judgment_label_shuffle_tests_csv}")
print(f"Combined updated-null tests:                {combined_permutation_tests_csv}")
print(f"Cluster binwise time tests:                 {judgment_cluster_binwise_csv}")
print(f"Cluster-corrected time tests:               {judgment_cluster_corrected_csv}")
print(f"Cluster null distribution:                  {judgment_cluster_null_csv}")

print("\nPrimary updated-null stable-window results:")

primary_cols = [
    "null_type",
    "analysis",
    "offset_label",
    "judgment_label",
    "observed_effect",
    "observed_ci95_low",
    "observed_ci95_high",
    "observed_t",
    "null_effect_mean",
    "null_effect_ci95_low",
    "null_effect_ci95_high",
    "permutation_p_effect_directional",
    "permutation_p_effect_directional_fdr_all_permutation_tests",
    "permutation_p_effect_two_sided",
    "permutation_p_effect_two_sided_fdr_all_permutation_tests",
]

available_primary_cols = [
    c for c in primary_cols
    if c in combined_permutation_tests_df.columns
]

print(
    combined_permutation_tests_df[available_primary_cols]
    .to_string(index=False)
)

print("\nCluster-corrected time-range results:")

if len(judgment_cluster_corrected_df) > 0:
    cluster_print_cols = [
        "analysis",
        "offset_label",
        "judgment_label",
        "start_time_sec",
        "end_time_sec",
        "n_timepoints",
        "mean_difference_in_cluster",
        "max_abs_t_in_cluster",
        "cluster_corrected_p",
        "cluster_corrected_p_fdr_all_clusters",
        "cluster_corrected_p_fdr_within_analysis",
    ]

    available_cluster_cols = [
        c for c in cluster_print_cols
        if c in judgment_cluster_corrected_df.columns
    ]

    print(
        judgment_cluster_corrected_df[available_cluster_cols]
        .to_string(index=False)
    )

else:
    print("No cluster-corrected time clusters were found.")

In [ ]:
# %% VISUAL BIAS ON HIGH-CONFIDENCE DIFFERENT-SOURCE TRIALS
# ============================================================
# Question:
# Does visual localization bias persist even when participants confidently
# report that the auditory and visual signals came from different sources?
#
# Core metric:
#   Bias_AminusV = RMSE_Audio - RMSE_Visual
#
# Interpretation:
#   Bias_AminusV > 0:
#       response is closer to the visual trajectory than the auditory trajectory
#
# Primary test:
#   Bias_AminusV > 0 in high-confidence different-source trials.
#
# Confidence split:
#   high confidence = post-trial confidence 4–5
#   low/medium confidence = post-trial confidence 1–3
#
# Primary stable window:
#   0.5–17.5 s
#
# Lag correction:
#   Uses LagA_sec and LagV_sec from trial_df if available.
#   Raw bias is always computed as a robustness metric.
# ============================================================

import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# -------------------------
# SETTINGS
# -------------------------
RANDOM_SEED = 12345
N_PERM = 10000
STABLE_START_SEC = 0.5
STABLE_END_SEC = 17.5
HIGH_CONF_MIN = 4
FIG_DPI = 200

MIN_SUBJECTS_FOR_TEST = 5
MIN_STABLE_SAMPLES = 5
MIN_ABS_DISPARITY_FOR_NORMALIZED_BIAS = 0.01

try:
    OUT_DIR = os.path.join(out_root, "high_confidence_different_visual_bias")
except NameError:
    OUT_DIR = os.path.join(os.getcwd(), "high_confidence_different_visual_bias")

CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

if "trial_df" not in globals():
    raise RuntimeError("trial_df not found. Run the spatial notebook load/resample cell first.")

# -------------------------
# HELPERS
# -------------------------
def parse_yes_no_same_response(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    same_tokens = [
        "y", "yes", "1", "true",
        "same", "same source", "common", "common source"
    ]

    different_tokens = [
        "n", "no", "0", "false",
        "different", "diff", "different source",
        "separate", "separate source"
    ]

    if s in same_tokens:
        return 1.0
    if s in different_tokens:
        return 0.0

    if "same" in s or "common" in s:
        return 1.0
    if "different" in s or "separate" in s or "diff" in s:
        return 0.0

    return np.nan


def parse_confidence(x):
    val = pd.to_numeric(x, errors="coerce")
    if pd.isna(val):
        return np.nan
    return float(val)


def extract_offset_abs(row):
    trial_type = str(row.get("trialType", "")).lower()

    if trial_type == "congruent":
        return 0.0

    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return abs(float(row["offsetDeg"]))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return float(m.group(1))

    return np.nan


def extract_side_label(row):
    side = row.get("sideSign", np.nan)

    try:
        side_num = float(side)
        if side_num < 0:
            return "left"
        if side_num > 0:
            return "right"
    except Exception:
        pass

    cond = str(row.get("condLabel", "")).lower()
    if "left" in cond:
        return "left"
    if "right" in cond:
        return "right"

    return "unknown"


def extract_sweep_direction(row):
    ramp = str(row.get("rampType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()

    text = f"{ramp} {cond}"

    if "converge" in text or "converging" in text:
        return "sweep_converge"
    if "diverge" in text or "diverging" in text:
        return "sweep_diverge"

    # Fallback: infer from condition text if it contains common naming.
    if "conv" in text:
        return "sweep_converge"
    if "div" in text:
        return "sweep_diverge"

    return "sweep_unknown"


def condition_group_from_row(row):
    trial_type = str(row.get("trialType", "")).lower()

    if trial_type == "incongruent_spatial":
        off = extract_offset_abs(row)
        if np.isfinite(off):
            return f"fixed_{int(off)}"
        return "fixed_unknown"

    if trial_type == "incongruent_sweep":
        return extract_sweep_direction(row)

    return np.nan


def condition_group_side_from_row(row):
    base = condition_group_from_row(row)

    if pd.isna(base):
        return np.nan

    side = extract_side_label(row)

    if str(base).startswith("fixed_"):
        return f"{base}_{side}"

    if str(base).startswith("sweep_"):
        return f"{base}_{side}"

    return base


def shift_response_by_lag(response, t_grid, lag_sec):
    """
    Lag-correct response by shifting response relative to stimulus.
    If lag_sec is positive, behavior lags stimulus, so sample response
    at t + lag_sec to align behavior to stimulus time.
    """
    response = np.asarray(response, dtype=float)
    t_grid = np.asarray(t_grid, dtype=float)

    if not np.isfinite(lag_sec):
        return np.full_like(response, np.nan, dtype=float)

    return np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan
    )


def rmse_from_arrays(x, y, mask):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    good = mask & np.isfinite(x) & np.isfinite(y)

    if good.sum() < MIN_STABLE_SAMPLES:
        return np.nan

    return float(np.sqrt(np.nanmean((x[good] - y[good]) ** 2)))


def mean_abs_from_arrays(x, y, mask):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    good = mask & np.isfinite(x) & np.isfinite(y)

    if good.sum() < MIN_STABLE_SAMPLES:
        return np.nan

    return float(np.nanmean(np.abs(x[good] - y[good])))


def mean_ci_t(values, alpha=0.05):
    vals = pd.Series(values).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)

    return m, m - crit * se, m + crit * se, se, n


def sign_flip_test(values, popmean=0.0, alternative="greater", n_perm=N_PERM, seed=RANDOM_SEED):
    vals = pd.Series(values).dropna().astype(float).values
    diffs = vals - popmean
    n = len(diffs)

    if n < 2:
        return np.nan, n

    obs = np.mean(diffs)
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1, 1], size=(n_perm, n))
    null = np.mean(signs * diffs, axis=1)

    if alternative == "greater":
        p = (np.sum(null >= obs) + 1) / (n_perm + 1)
    elif alternative == "less":
        p = (np.sum(null <= obs) + 1) / (n_perm + 1)
    else:
        p = (np.sum(np.abs(null) >= np.abs(obs)) + 1) / (n_perm + 1)

    return p, n


def fdr_bh(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)

    valid = np.isfinite(p)
    pv = p[valid]

    if len(pv) == 0:
        return out

    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)

    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)

    tmp = np.empty_like(adj)
    tmp[order] = adj
    out[valid] = tmp

    return out


# -------------------------
# BUILD TRIAL-LEVEL STABLE-WINDOW BIAS DATASET
# -------------------------
df_bias = trial_df.copy()

subject_col = "subject" if "subject" in df_bias.columns else "participant_id"
if subject_col not in df_bias.columns:
    raise RuntimeError("Could not find a subject column named 'subject' or 'participant_id'.")

if "same_response_yes" not in df_bias.columns:
    if "post_question" not in df_bias.columns:
        raise RuntimeError("Need either same_response_yes or post_question in trial_df.")
    df_bias["same_response_yes"] = df_bias["post_question"].apply(parse_yes_no_same_response)

if "confidence_rating" not in df_bias.columns:
    if "post_question2" not in df_bias.columns:
        raise RuntimeError("Need either confidence_rating or post_question2 in trial_df.")
    df_bias["confidence_rating"] = df_bias["post_question2"].apply(parse_confidence)

# Conflict AV trials only: fixed incongruent + sweeps.
conflict_mask = df_bias["trialType"].astype(str).str.lower().isin([
    "incongruent_spatial",
    "incongruent_sweep"
])

df_bias = df_bias[conflict_mask].copy()

df_bias["condition_group"] = df_bias.apply(condition_group_from_row, axis=1)
df_bias["condition_group_side"] = df_bias.apply(condition_group_side_from_row, axis=1)
df_bias["offset_abs"] = df_bias.apply(extract_offset_abs, axis=1)
df_bias["side_label"] = df_bias.apply(extract_side_label, axis=1)

df_bias = df_bias.dropna(
    subset=[subject_col, "trialIndex", "condition_group", "same_response_yes", "confidence_rating"]
).copy()

has_lag_cols = ("LagA_sec" in df_bias.columns) and ("LagV_sec" in df_bias.columns)

stable_rows = []

for _, row in df_bias.iterrows():

    subj = row[subject_col]
    trial_index = row["trialIndex"]

    t = np.asarray(row["t_grid"], dtype=float)
    resp = np.asarray(row["mouse_x"], dtype=float)
    aud = np.asarray(row["audio_x"], dtype=float)
    vis = np.asarray(row["blob_x"], dtype=float)

    if not (len(t) == len(resp) == len(aud) == len(vis)):
        continue

    stable_mask = (t >= STABLE_START_SEC) & (t <= STABLE_END_SEC)

    if stable_mask.sum() < MIN_STABLE_SAMPLES:
        continue

    rmse_a_raw = rmse_from_arrays(resp, aud, stable_mask)
    rmse_v_raw = rmse_from_arrays(resp, vis, stable_mask)

    abs_a_raw = mean_abs_from_arrays(resp, aud, stable_mask)
    abs_v_raw = mean_abs_from_arrays(resp, vis, stable_mask)

    mean_abs_disparity = mean_abs_from_arrays(aud, vis, stable_mask)

    # Lag-corrected response, if lags exist.
    rmse_a_lagcorr = np.nan
    rmse_v_lagcorr = np.nan
    abs_a_lagcorr = np.nan
    abs_v_lagcorr = np.nan

    if has_lag_cols:
        lag_a = pd.to_numeric(row.get("LagA_sec", np.nan), errors="coerce")
        lag_v = pd.to_numeric(row.get("LagV_sec", np.nan), errors="coerce")

        resp_a_lagcorr = shift_response_by_lag(resp, t, lag_a)
        resp_v_lagcorr = shift_response_by_lag(resp, t, lag_v)

        rmse_a_lagcorr = rmse_from_arrays(resp_a_lagcorr, aud, stable_mask)
        rmse_v_lagcorr = rmse_from_arrays(resp_v_lagcorr, vis, stable_mask)

        abs_a_lagcorr = mean_abs_from_arrays(resp_a_lagcorr, aud, stable_mask)
        abs_v_lagcorr = mean_abs_from_arrays(resp_v_lagcorr, vis, stable_mask)

    bias_raw = rmse_a_raw - rmse_v_raw if np.isfinite(rmse_a_raw) and np.isfinite(rmse_v_raw) else np.nan
    bias_lagcorr = (
        rmse_a_lagcorr - rmse_v_lagcorr
        if np.isfinite(rmse_a_lagcorr) and np.isfinite(rmse_v_lagcorr)
        else np.nan
    )

    absbias_raw = abs_a_raw - abs_v_raw if np.isfinite(abs_a_raw) and np.isfinite(abs_v_raw) else np.nan
    absbias_lagcorr = (
        abs_a_lagcorr - abs_v_lagcorr
        if np.isfinite(abs_a_lagcorr) and np.isfinite(abs_v_lagcorr)
        else np.nan
    )

    normalized_absbias_raw = (
        absbias_raw / mean_abs_disparity
        if np.isfinite(absbias_raw) and np.isfinite(mean_abs_disparity) and mean_abs_disparity > MIN_ABS_DISPARITY_FOR_NORMALIZED_BIAS
        else np.nan
    )

    normalized_absbias_lagcorr = (
        absbias_lagcorr / mean_abs_disparity
        if np.isfinite(absbias_lagcorr) and np.isfinite(mean_abs_disparity) and mean_abs_disparity > MIN_ABS_DISPARITY_FOR_NORMALIZED_BIAS
        else np.nan
    )

    same = float(row["same_response_yes"])
    diff = 1.0 - same
    conf = float(row["confidence_rating"])
    high_conf = conf >= HIGH_CONF_MIN

    if same == 1 and high_conf:
        report_conf_group = "high_conf_same"
    elif same == 1 and not high_conf:
        report_conf_group = "lowmed_conf_same"
    elif diff == 1 and high_conf:
        report_conf_group = "high_conf_different"
    elif diff == 1 and not high_conf:
        report_conf_group = "lowmed_conf_different"
    else:
        report_conf_group = np.nan

    stable_rows.append({
        "subject_id": subj,
        "trialIndex": trial_index,
        "trialType": row.get("trialType", np.nan),
        "condLabel": row.get("condLabel", np.nan),
        "condition_group": row["condition_group"],
        "condition_group_side": row["condition_group_side"],
        "side_label": row["side_label"],
        "offset_abs": row["offset_abs"],

        "same_response_yes": same,
        "different_report": diff,
        "confidence_rating": conf,
        "high_confidence": int(high_conf),
        "report_conf_group": report_conf_group,

        "mean_abs_disparity": mean_abs_disparity,

        "RMSE_A_raw_stable": rmse_a_raw,
        "RMSE_V_raw_stable": rmse_v_raw,
        "Bias_raw_AminusV_stable": bias_raw,

        "RMSE_A_lagcorr_stable": rmse_a_lagcorr,
        "RMSE_V_lagcorr_stable": rmse_v_lagcorr,
        "Bias_lagcorr_AminusV_stable": bias_lagcorr,

        "AbsBias_raw_AminusV_stable": absbias_raw,
        "AbsBias_lagcorr_AminusV_stable": absbias_lagcorr,

        "NormalizedAbsBias_raw_AminusV_stable": normalized_absbias_raw,
        "NormalizedAbsBias_lagcorr_AminusV_stable": normalized_absbias_lagcorr
    })

bias_trial_df = pd.DataFrame(stable_rows)

if len(bias_trial_df) == 0:
    raise RuntimeError("No stable-window bias rows were created. Check trial_df arrays and trial labels.")

bias_trial_df.to_csv(
    os.path.join(CSV_DIR, "posttrial_confidence_high_conf_diff_bias_trial_metrics.csv"),
    index=False
)

# Pick primary visual-bias column.
if bias_trial_df["Bias_lagcorr_AminusV_stable"].notna().sum() > 0:
    PRIMARY_BIAS_COL = "Bias_lagcorr_AminusV_stable"
    PRIMARY_BIAS_LABEL = "Lag-corrected RMSE bias A−V"
else:
    PRIMARY_BIAS_COL = "Bias_raw_AminusV_stable"
    PRIMARY_BIAS_LABEL = "Raw RMSE bias A−V"

print("\nBias dataset:")
print(f"Rows: {len(bias_trial_df):,}")
print(f"Subjects: {bias_trial_df['subject_id'].nunique()}")
print(f"Primary bias metric: {PRIMARY_BIAS_COL}")
print(bias_trial_df.head())


# -------------------------
# SUBJECT-FIRST GROUP MEANS
# -------------------------
subject_bias_by_group = (
    bias_trial_df
    .groupby(["subject_id", "condition_group", "report_conf_group"], as_index=False, observed=True, sort=False)
    .agg(
        n_trials=("trialIndex", "nunique"),
        confidence_mean=("confidence_rating", "mean"),
        Bias_raw_AminusV_stable=("Bias_raw_AminusV_stable", "mean"),
        Bias_lagcorr_AminusV_stable=("Bias_lagcorr_AminusV_stable", "mean"),
        AbsBias_raw_AminusV_stable=("AbsBias_raw_AminusV_stable", "mean"),
        AbsBias_lagcorr_AminusV_stable=("AbsBias_lagcorr_AminusV_stable", "mean"),
        NormalizedAbsBias_raw_AminusV_stable=("NormalizedAbsBias_raw_AminusV_stable", "mean"),
        NormalizedAbsBias_lagcorr_AminusV_stable=("NormalizedAbsBias_lagcorr_AminusV_stable", "mean")
    )
)

subject_bias_by_group.to_csv(
    os.path.join(CSV_DIR, "subject_bias_by_condition_report_conf_group.csv"),
    index=False
)

group_bias_by_group = (
    subject_bias_by_group
    .groupby(["condition_group", "report_conf_group"], as_index=False, observed=True, sort=False)
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_trials=("n_trials", "sum"),
        confidence_mean=("confidence_mean", "mean"),
        bias_mean=(PRIMARY_BIAS_COL, "mean"),
        bias_sd=(PRIMARY_BIAS_COL, "std")
    )
)

group_bias_by_group["bias_sem"] = group_bias_by_group["bias_sd"] / np.sqrt(group_bias_by_group["n_subjects"])
group_bias_by_group["ci95_low"] = group_bias_by_group.apply(
    lambda r: r["bias_mean"] - t_dist.ppf(.975, r["n_subjects"] - 1) * r["bias_sem"]
    if r["n_subjects"] > 1 else np.nan,
    axis=1
)
group_bias_by_group["ci95_high"] = group_bias_by_group.apply(
    lambda r: r["bias_mean"] + t_dist.ppf(.975, r["n_subjects"] - 1) * r["bias_sem"]
    if r["n_subjects"] > 1 else np.nan,
    axis=1
)

group_bias_by_group.to_csv(
    os.path.join(CSV_DIR, "group_bias_by_condition_report_conf_group.csv"),
    index=False
)

# -------------------------
# CRITICAL TESTS:
# Is high-confidence different-source bias > 0?
# -------------------------
high_diff_subject = subject_bias_by_group[
    subject_bias_by_group["report_conf_group"] == "high_conf_different"
].copy()

test_rows = []

bias_cols_to_test = [
    "Bias_lagcorr_AminusV_stable",
    "Bias_raw_AminusV_stable",
    "NormalizedAbsBias_lagcorr_AminusV_stable",
    "NormalizedAbsBias_raw_AminusV_stable"
]

for bias_col in bias_cols_to_test:
    if bias_col not in high_diff_subject.columns:
        continue

    for condition, g in high_diff_subject.groupby("condition_group", observed=True, sort=False):

        vals = g[bias_col].dropna().astype(float)

        if len(vals) < MIN_SUBJECTS_FOR_TEST:
            continue

        m, lo, hi, se, n = mean_ci_t(vals)
        p, _ = sign_flip_test(vals, alternative="greater", seed=RANDOM_SEED + len(test_rows) * 10)

        test_rows.append({
            "analysis": "high_confidence_different_bias_by_condition",
            "condition_group": condition,
            "bias_metric": bias_col,
            "alternative": "greater_than_zero",
            "n_subjects": n,
            "mean": m,
            "se": se,
            "ci95_low": lo,
            "ci95_high": hi,
            "p_perm": p
        })

high_conf_diff_tests = pd.DataFrame(test_rows)

if len(high_conf_diff_tests) > 0:
    high_conf_diff_tests["p_fdr"] = fdr_bh(high_conf_diff_tests["p_perm"].values)

high_conf_diff_tests.to_csv(
    os.path.join(CSV_DIR, "high_confidence_different_visual_bias_tests_by_condition.csv"),
    index=False
)

print("\nHigh-confidence different-source visual-bias tests by condition:")
try:
    display(high_conf_diff_tests)
except NameError:
    print(high_conf_diff_tests.to_string(index=False))


# -------------------------
# PLANNED AGGREGATE TESTS
# -------------------------
planned_sets = {
    "fixed_30_40": ["fixed_30", "fixed_40"],
    "fixed_40_only": ["fixed_40"],
    "sweep_all": ["sweep_converge", "sweep_diverge"],
    "all_large_conflict": ["fixed_30", "fixed_40", "sweep_converge", "sweep_diverge"],
    "all_fixed_offsets": ["fixed_10", "fixed_20", "fixed_30", "fixed_40"],
    "all_conflict_trials": ["fixed_10", "fixed_20", "fixed_30", "fixed_40", "sweep_converge", "sweep_diverge"]
}

planned_rows = []

for set_name, conds in planned_sets.items():
    set_df = bias_trial_df[
        (bias_trial_df["report_conf_group"] == "high_conf_different") &
        (bias_trial_df["condition_group"].isin(conds))
    ].copy()

    if len(set_df) == 0:
        continue

    subject_set = (
        set_df
        .groupby("subject_id", as_index=False, observed=True, sort=False)
        .agg(
            n_trials=("trialIndex", "nunique"),
            Bias_lagcorr_AminusV_stable=("Bias_lagcorr_AminusV_stable", "mean"),
            Bias_raw_AminusV_stable=("Bias_raw_AminusV_stable", "mean"),
            NormalizedAbsBias_lagcorr_AminusV_stable=("NormalizedAbsBias_lagcorr_AminusV_stable", "mean"),
            NormalizedAbsBias_raw_AminusV_stable=("NormalizedAbsBias_raw_AminusV_stable", "mean")
        )
    )

    for bias_col in bias_cols_to_test:
        vals = subject_set[bias_col].dropna().astype(float)

        if len(vals) < MIN_SUBJECTS_FOR_TEST:
            continue

        m, lo, hi, se, n = mean_ci_t(vals)
        p, _ = sign_flip_test(vals, alternative="greater", seed=RANDOM_SEED + len(planned_rows) * 10)

        planned_rows.append({
            "analysis": "planned_high_confidence_different_bias",
            "planned_set": set_name,
            "included_conditions": ",".join(conds),
            "bias_metric": bias_col,
            "alternative": "greater_than_zero",
            "n_subjects": n,
            "mean": m,
            "se": se,
            "ci95_low": lo,
            "ci95_high": hi,
            "p_perm": p,
            "total_trials": int(subject_set["n_trials"].sum())
        })

planned_tests = pd.DataFrame(planned_rows)

if len(planned_tests) > 0:
    planned_tests["p_fdr"] = fdr_bh(planned_tests["p_perm"].values)

planned_tests.to_csv(
    os.path.join(CSV_DIR, "planned_high_confidence_different_visual_bias_tests.csv"),
    index=False
)

print("\nPlanned high-confidence different-source visual-bias tests:")
try:
    display(planned_tests)
except NameError:
    print(planned_tests.to_string(index=False))


# -------------------------
# PLOTS
# -------------------------
condition_order = [
    "fixed_10",
    "fixed_20",
    "fixed_30",
    "fixed_40",
    "sweep_converge",
    "sweep_diverge"
]

group_order = [
    "lowmed_conf_same",
    "high_conf_same",
    "lowmed_conf_different",
    "high_conf_different"
]

group_labels = {
    "lowmed_conf_same": "Low/med-conf same",
    "high_conf_same": "High-conf same",
    "lowmed_conf_different": "Low/med-conf different",
    "high_conf_different": "High-conf different"
}

plot_df = group_bias_by_group[
    group_bias_by_group["condition_group"].isin(condition_order) &
    group_bias_by_group["report_conf_group"].isin(group_order)
].copy()

# Grouped bar plot for all report-confidence cells.
plt.figure(figsize=(13, 6))

x = np.arange(len(condition_order))
width = 0.18

for i, grp in enumerate(group_order):
    g = plot_df[plot_df["report_conf_group"] == grp].set_index("condition_group").reindex(condition_order)

    xpos = x + (i - 1.5) * width

    plt.bar(
        xpos,
        g["bias_mean"].values,
        width=width,
        yerr=g["bias_sem"].values,
        capsize=3,
        label=group_labels.get(grp, grp)
    )

plt.axhline(0, linestyle="--", linewidth=1)
plt.xticks(x, condition_order, rotation=35, ha="right")
plt.ylabel(f"{PRIMARY_BIAS_LABEL}\npositive = closer to visual")
plt.title("Visual bias by explicit judgment and post-trial confidence")
plt.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "visual_bias_by_report_confidence_group.png"),
    dpi=FIG_DPI,
    bbox_inches="tight"
)
plt.show()

# Critical plot: high-confidence different-source trials only.
crit_plot = plot_df[plot_df["report_conf_group"] == "high_conf_different"].copy()
crit_plot = crit_plot.set_index("condition_group").reindex(condition_order).reset_index()

plt.figure(figsize=(9, 5))

plt.bar(
    np.arange(len(condition_order)),
    crit_plot["bias_mean"].values,
    yerr=crit_plot["bias_sem"].values,
    capsize=4
)

plt.axhline(0, linestyle="--", linewidth=1)
plt.xticks(np.arange(len(condition_order)), condition_order, rotation=35, ha="right")
plt.ylabel(f"{PRIMARY_BIAS_LABEL}\npositive = closer to visual")
plt.title("Visual bias on high-confidence different-source trials")
plt.tight_layout()

plt.savefig(
    os.path.join(FIG_DIR, "high_confidence_different_visual_bias_by_condition.png"),
    dpi=FIG_DPI,
    bbox_inches="tight"
)
plt.show()


# -------------------------
# OPTIONAL CLUSTER-ROBUST MODEL
# -------------------------
# This is supplementary because the subject-first tests above are primary.
cluster_df = bias_trial_df.dropna(
    subset=[PRIMARY_BIAS_COL, "report_conf_group", "condition_group", "subject_id"]
).copy()

cluster_df["report_conf_group"] = pd.Categorical(
    cluster_df["report_conf_group"],
    categories=group_order,
    ordered=True
)

cluster_df["condition_group"] = pd.Categorical(
    cluster_df["condition_group"],
    categories=condition_order,
    ordered=True
)

try:
    m = smf.ols(
        f"{PRIMARY_BIAS_COL} ~ C(condition_group) * C(report_conf_group)",
        data=cluster_df
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": cluster_df["subject_id"]}
    )

    with open(os.path.join(CSV_DIR, "cluster_robust_bias_condition_by_report_conf_group.txt"), "w") as f:
        f.write(str(m.summary()))

    coef_rows = []
    ci = m.conf_int()

    for term in m.params.index:
        coef_rows.append({
            "term": term,
            "b": m.params[term],
            "SE_cluster": m.bse[term],
            "z": m.tvalues[term],
            "p": m.pvalues[term],
            "ci95_low": ci.loc[term, 0],
            "ci95_high": ci.loc[term, 1],
            "n_obs": int(m.nobs),
            "r2": m.rsquared,
            "adj_r2": m.rsquared_adj
        })

    coef_df = pd.DataFrame(coef_rows)
    coef_df.to_csv(
        os.path.join(CSV_DIR, "cluster_robust_bias_condition_by_report_conf_group_coefficients.csv"),
        index=False
    )

    print("\nCluster-robust robustness model:")
    print(m.summary())

except Exception as e:
    print(f"\nCluster-robust robustness model failed: {e}")


print("\nAnalysis complete.")
print(f"Outputs saved to: {OUT_DIR}")
print(f"Primary bias metric: {PRIMARY_BIAS_COL}")
print("\nKey files:")
print(f"  Trial-level bias data: {os.path.join(CSV_DIR, 'posttrial_confidence_high_conf_diff_bias_trial_metrics.csv')}")
print(f"  Subject means: {os.path.join(CSV_DIR, 'subject_bias_by_condition_report_conf_group.csv')}")
print(f"  High-confidence different tests: {os.path.join(CSV_DIR, 'high_confidence_different_visual_bias_tests_by_condition.csv')}")
print(f"  Planned aggregate tests: {os.path.join(CSV_DIR, 'planned_high_confidence_different_visual_bias_tests.csv')}")
print(f"  Critical figure: {os.path.join(FIG_DIR, 'high_confidence_different_visual_bias_by_condition.png')}")

In [ ]:
# %% POST-TRIAL CONFIDENCE AS METACOGNITIVE SUPPORT FOR EXPLICIT SAME/DIFFERENT REPORTS
# ============================================================
# Question:
# Does post-trial confidence track whether the participant's explicit
# same/different report is supported by spatial disparity?
#
# Core model, fit subject-first:
#   confidence ~ spatial_disparity * reported_different
#
# Interpretation:
#   - same-source reports should be more confident at small disparities
#   - different-source reports should be more confident at large disparities
#   - a positive interaction means confidence increasingly favors
#     "different" reports as disparity grows
#
# Complementary support-score model:
#   report_support =
#       1 - disparity/max_disparity, if reported same
#       disparity/max_disparity,     if reported different
#
#   confidence ~ report_support + reported_different
#
# Interpretation:
#   Higher confidence when the explicit report is more supported by
#   the spatial evidence.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# -------------------------
# SETTINGS
# -------------------------
RANDOM_SEED = 12345
N_PERM = 10000
MAX_FIXED_OFFSET = 40.0
MIN_OBS_PER_SUBJECT_MODEL = 12
FIG_DPI = 200

try:
    OUT_DIR = os.path.join(out_root, "posttrial_confidence_report_by_disparity")
except NameError:
    OUT_DIR = os.path.join(os.getcwd(), "posttrial_confidence_report_by_disparity")

CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

if "trial_df" not in globals():
    raise RuntimeError("trial_df not found. Run the spatial notebook load/resample cell first.")

# -------------------------
# HELPERS
# -------------------------
def parse_yes_no_same_response(x):
    """
    Interprets post_question as the participant's same-source response.
    same/common/yes = 1
    different/separate/no = 0
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    same_tokens = [
        "y", "yes", "1", "true",
        "same", "same source", "common", "common source"
    ]

    different_tokens = [
        "n", "no", "0", "false",
        "different", "diff", "different source",
        "separate", "separate source"
    ]

    if s in same_tokens:
        return 1.0
    if s in different_tokens:
        return 0.0

    if "same" in s or "common" in s:
        return 1.0
    if "different" in s or "separate" in s or "diff" in s:
        return 0.0

    return np.nan


def parse_confidence(x):
    """
    Parse post_question2 confidence rating.
    Expected scale: 1-5.
    """
    val = pd.to_numeric(x, errors="coerce")
    if pd.isna(val):
        return np.nan
    return float(val)


def extract_offset_abs(row):
    """
    Extract absolute audiovisual spatial offset.
    Congruent trials are assigned 0.
    """
    trial_type = str(row.get("trialType", "")).lower()

    if trial_type == "congruent":
        return 0.0

    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return abs(float(row["offsetDeg"]))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return float(m.group(1))

    return np.nan


def extract_side_label(row):
    """
    Extract left/right/center label for descriptive output.
    """
    trial_type = str(row.get("trialType", "")).lower()
    if trial_type == "congruent":
        return "center"

    side = row.get("sideSign", np.nan)
    try:
        side_num = float(side)
        if side_num < 0:
            return "left"
        if side_num > 0:
            return "right"
    except Exception:
        pass

    cond = str(row.get("condLabel", "")).lower()
    if "left" in cond:
        return "left"
    if "right" in cond:
        return "right"

    return "unknown"


def mean_ci_t(values, alpha=0.05):
    vals = pd.Series(values).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)

    return m, m - crit * se, m + crit * se, se, n


def sign_flip_test(values, popmean=0.0, alternative="two-sided", n_perm=N_PERM, seed=RANDOM_SEED):
    vals = pd.Series(values).dropna().astype(float).values
    diffs = vals - popmean
    n = len(diffs)

    if n < 2:
        return np.nan, n

    obs = np.mean(diffs)
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1, 1], size=(n_perm, n))
    null = np.mean(signs * diffs, axis=1)

    if alternative == "greater":
        p = (np.sum(null >= obs) + 1) / (n_perm + 1)
    elif alternative == "less":
        p = (np.sum(null <= obs) + 1) / (n_perm + 1)
    else:
        p = (np.sum(np.abs(null) >= np.abs(obs)) + 1) / (n_perm + 1)

    return p, n


def fdr_bh(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)

    valid = np.isfinite(p)
    pv = p[valid]

    if len(pv) == 0:
        return out

    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)

    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)

    tmp = np.empty_like(adj)
    tmp[order] = adj
    out[valid] = tmp

    return out


def safe_fit_subject_ols(data, formula, required_terms, min_n=MIN_OBS_PER_SUBJECT_MODEL):
    out = {
        "n_obs": len(data),
        "model_r2": np.nan,
        "model_adj_r2": np.nan,
        "model_failed": False,
        "model_error": "",
        "rank_deficient": np.nan,
        "condition_number": np.nan
    }

    for term in required_terms:
        out[term] = np.nan

    if len(data) < min_n:
        out["model_failed"] = True
        out["model_error"] = f"Too few observations: {len(data)}"
        return out

    try:
        m = smf.ols(formula, data=data).fit()

        out["model_r2"] = m.rsquared
        out["model_adj_r2"] = m.rsquared_adj

        exog = np.asarray(m.model.exog, dtype=float)
        rank = np.linalg.matrix_rank(exog)
        n_params = exog.shape[1]

        out["rank_deficient"] = bool(rank < n_params)
        out["condition_number"] = np.linalg.cond(exog)

        for term in required_terms:
            out[term] = m.params.get(term, np.nan)

    except Exception as e:
        out["model_failed"] = True
        out["model_error"] = str(e)

    return out


# -------------------------
# BUILD ANALYSIS DATASET
# -------------------------
df_conf = trial_df.copy()

subject_col = "subject" if "subject" in df_conf.columns else "participant_id"
if subject_col not in df_conf.columns:
    raise RuntimeError("Could not find a subject column named 'subject' or 'participant_id'.")

if "same_response_yes" not in df_conf.columns:
    if "post_question" not in df_conf.columns:
        raise RuntimeError("Need either same_response_yes or post_question in trial_df.")
    df_conf["same_response_yes"] = df_conf["post_question"].apply(parse_yes_no_same_response)

if "confidence_rating" not in df_conf.columns:
    if "post_question2" not in df_conf.columns:
        raise RuntimeError("Need either confidence_rating or post_question2 in trial_df.")
    df_conf["confidence_rating"] = df_conf["post_question2"].apply(parse_confidence)

df_conf["offset_abs"] = df_conf.apply(extract_offset_abs, axis=1)
df_conf["side_label"] = df_conf.apply(extract_side_label, axis=1)

# Fixed spatial evidence trials: congruent + fixed spatial offsets.
# Excludes dynamic sweeps because their disparity changes continuously.
fixed_mask = df_conf["trialType"].astype(str).str.lower().isin([
    "congruent",
    "incongruent_spatial"
])

model_df_conf = df_conf[fixed_mask].copy()

model_df_conf = model_df_conf.dropna(
    subset=[subject_col, "trialIndex", "confidence_rating", "same_response_yes", "offset_abs"]
).copy()

model_df_conf["same_report"] = model_df_conf["same_response_yes"].astype(float)
model_df_conf["different_report"] = 1.0 - model_df_conf["same_report"]

model_df_conf["report_label"] = np.where(
    model_df_conf["same_report"] == 1,
    "same",
    "different"
)

model_df_conf["offset_10"] = model_df_conf["offset_abs"] / 10.0
model_df_conf["offset_10_c"] = model_df_conf["offset_10"] - model_df_conf["offset_10"].mean()

# Evidence support for the report:
# same reports are most supported at low disparity; different reports at high disparity.
model_df_conf["offset_norm_0_1"] = model_df_conf["offset_abs"] / MAX_FIXED_OFFSET
model_df_conf["report_support"] = np.where(
    model_df_conf["different_report"] == 1,
    model_df_conf["offset_norm_0_1"],
    1.0 - model_df_conf["offset_norm_0_1"]
)

model_df_conf["condition_fixed"] = np.where(
    model_df_conf["offset_abs"] == 0,
    "congruent",
    "spatial_" + model_df_conf["offset_abs"].astype(int).astype(str)
)

model_df_conf["condition_fixed_side"] = np.where(
    model_df_conf["offset_abs"] == 0,
    "congruent",
    model_df_conf["condition_fixed"] + "_" + model_df_conf["side_label"]
)

model_df_conf = model_df_conf.rename(columns={subject_col: "subject_id"})

model_df_conf.to_csv(
    os.path.join(CSV_DIR, "posttrial_confidence_disparity_report_model_df.csv"),
    index=False
)

print("\nAnalysis dataset:")
print(f"Rows: {len(model_df_conf):,}")
print(f"Subjects: {model_df_conf['subject_id'].nunique()}")
print(model_df_conf[["subject_id", "trialIndex", "trialType", "condition_fixed_side", "offset_abs", "report_label", "confidence_rating", "report_support"]].head())


# -------------------------
# DESCRIPTIVE SUBJECT-FIRST MEANS BY OFFSET × REPORT
# -------------------------
subj_offset_report = (
    model_df_conf
    .groupby(["subject_id", "offset_abs", "report_label"], as_index=False, observed=True, sort=False)
    .agg(
        confidence_mean=("confidence_rating", "mean"),
        n_trials=("trialIndex", "nunique")
    )
)

group_offset_report = (
    subj_offset_report
    .groupby(["offset_abs", "report_label"], as_index=False, observed=True, sort=False)
    .agg(
        confidence_mean=("confidence_mean", "mean"),
        confidence_sd=("confidence_mean", "std"),
        n_subjects=("subject_id", "nunique"),
        n_trials=("n_trials", "sum")
    )
)

group_offset_report["confidence_sem"] = (
    group_offset_report["confidence_sd"] / np.sqrt(group_offset_report["n_subjects"])
)
group_offset_report["ci95_low"] = group_offset_report.apply(
    lambda r: r["confidence_mean"] - t_dist.ppf(.975, r["n_subjects"] - 1) * r["confidence_sem"]
    if r["n_subjects"] > 1 else np.nan,
    axis=1
)
group_offset_report["ci95_high"] = group_offset_report.apply(
    lambda r: r["confidence_mean"] + t_dist.ppf(.975, r["n_subjects"] - 1) * r["confidence_sem"]
    if r["n_subjects"] > 1 else np.nan,
    axis=1
)

subj_offset_report.to_csv(
    os.path.join(CSV_DIR, "subject_confidence_by_offset_and_report.csv"),
    index=False
)

group_offset_report.to_csv(
    os.path.join(CSV_DIR, "group_confidence_by_offset_and_report.csv"),
    index=False
)

# Plot confidence by offset and report.
plt.figure(figsize=(8, 5))

for label in ["same", "different"]:
    g = group_offset_report[group_offset_report["report_label"] == label].sort_values("offset_abs")
    if len(g) == 0:
        continue

    plt.plot(
        g["offset_abs"],
        g["confidence_mean"],
        marker="o",
        linewidth=2.5,
        label=f"Reported {label}"
    )

    plt.fill_between(
        g["offset_abs"].astype(float),
        g["ci95_low"].astype(float),
        g["ci95_high"].astype(float),
        alpha=0.18
    )

plt.xlabel("Audiovisual spatial disparity (°)")
plt.ylabel("Post-trial confidence rating (1–5)")
plt.title("Post-trial confidence by spatial disparity and explicit report")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "confidence_by_offset_and_report.png"),
    dpi=FIG_DPI,
    bbox_inches="tight"
)
plt.show()

# Plot report-support function.
subj_support_bins = model_df_conf.copy()
subj_support_bins["support_bin"] = pd.cut(
    subj_support_bins["report_support"],
    bins=np.linspace(0, 1, 6),
    include_lowest=True
)

support_subject_bin = (
    subj_support_bins
    .groupby(["subject_id", "support_bin"], as_index=False, observed=True, sort=False)
    .agg(
        report_support_mean=("report_support", "mean"),
        confidence_mean=("confidence_rating", "mean"),
        n_trials=("trialIndex", "nunique")
    )
)

support_group_bin = (
    support_subject_bin
    .groupby("support_bin", as_index=False, observed=True, sort=False)
    .agg(
        report_support_mean=("report_support_mean", "mean"),
        confidence_mean=("confidence_mean", "mean"),
        confidence_sd=("confidence_mean", "std"),
        n_subjects=("subject_id", "nunique"),
        n_trials=("n_trials", "sum")
    )
)

support_group_bin["confidence_sem"] = support_group_bin["confidence_sd"] / np.sqrt(support_group_bin["n_subjects"])
support_group_bin["ci95_low"] = support_group_bin["confidence_mean"] - 1.96 * support_group_bin["confidence_sem"]
support_group_bin["ci95_high"] = support_group_bin["confidence_mean"] + 1.96 * support_group_bin["confidence_sem"]

support_subject_bin.to_csv(
    os.path.join(CSV_DIR, "subject_confidence_by_report_support_bin.csv"),
    index=False
)

support_group_bin.to_csv(
    os.path.join(CSV_DIR, "group_confidence_by_report_support_bin.csv"),
    index=False
)

plt.figure(figsize=(7, 5))

g = support_group_bin.dropna(subset=["report_support_mean", "confidence_mean"]).sort_values("report_support_mean")

plt.plot(
    g["report_support_mean"],
    g["confidence_mean"],
    marker="o",
    linewidth=2.5
)

plt.fill_between(
    g["report_support_mean"].astype(float),
    g["ci95_low"].astype(float),
    g["ci95_high"].astype(float),
    alpha=0.18
)

plt.xlabel("Spatial evidence support for explicit report")
plt.ylabel("Post-trial confidence rating (1–5)")
plt.title("Confidence increases when the report is supported by spatial evidence")
plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "confidence_by_report_support.png"),
    dpi=FIG_DPI,
    bbox_inches="tight"
)
plt.show()


# -------------------------
# SUBJECT-FIRST INTERACTION MODELS
# -------------------------
interaction_formula = "confidence_rating ~ offset_10_c * different_report"
interaction_required_terms = [
    "Intercept",
    "offset_10_c",
    "different_report",
    "offset_10_c:different_report"
]

support_formula = "confidence_rating ~ report_support + different_report"
support_required_terms = [
    "Intercept",
    "report_support",
    "different_report"
]

coef_rows = []

for sid, g in model_df_conf.groupby("subject_id", observed=True, sort=False):
    g = g.dropna(
        subset=["confidence_rating", "offset_10_c", "different_report", "report_support"]
    ).copy()

    # Need variability in report and offset to estimate interaction meaningfully.
    has_report_variation = g["different_report"].nunique() >= 2
    has_offset_variation = g["offset_10_c"].nunique() >= 2

    interaction_out = {}
    if has_report_variation and has_offset_variation:
        interaction_out = safe_fit_subject_ols(
            g,
            formula=interaction_formula,
            required_terms=interaction_required_terms,
            min_n=MIN_OBS_PER_SUBJECT_MODEL
        )
    else:
        interaction_out = {
            "n_obs": len(g),
            "model_r2": np.nan,
            "model_adj_r2": np.nan,
            "model_failed": True,
            "model_error": "Insufficient report or offset variation",
            "rank_deficient": np.nan,
            "condition_number": np.nan
        }
        for term in interaction_required_terms:
            interaction_out[term] = np.nan

    support_out = safe_fit_subject_ols(
        g,
        formula=support_formula,
        required_terms=support_required_terms,
        min_n=MIN_OBS_PER_SUBJECT_MODEL
    )

    b_same_slope = interaction_out.get("offset_10_c", np.nan)
    b_interaction = interaction_out.get("offset_10_c:different_report", np.nan)
    b_diff_slope = (
        b_same_slope + b_interaction
        if np.isfinite(b_same_slope) and np.isfinite(b_interaction)
        else np.nan
    )

    coef_rows.append({
        "subject_id": sid,
        "n_obs": len(g),
        "n_same_reports": int(np.sum(g["same_report"] == 1)),
        "n_different_reports": int(np.sum(g["different_report"] == 1)),
        "n_offsets": int(g["offset_abs"].nunique()),

        "interaction_intercept": interaction_out.get("Intercept", np.nan),
        "interaction_same_report_offset_slope": b_same_slope,
        "interaction_different_report_main_effect": interaction_out.get("different_report", np.nan),
        "interaction_offset_x_different_report": b_interaction,
        "interaction_different_report_offset_slope": b_diff_slope,
        "interaction_model_r2": interaction_out.get("model_r2", np.nan),
        "interaction_model_adj_r2": interaction_out.get("model_adj_r2", np.nan),
        "interaction_rank_deficient": interaction_out.get("rank_deficient", np.nan),
        "interaction_condition_number": interaction_out.get("condition_number", np.nan),
        "interaction_model_failed": interaction_out.get("model_failed", False),
        "interaction_model_error": interaction_out.get("model_error", ""),

        "support_intercept": support_out.get("Intercept", np.nan),
        "support_slope": support_out.get("report_support", np.nan),
        "support_different_report": support_out.get("different_report", np.nan),
        "support_model_r2": support_out.get("model_r2", np.nan),
        "support_model_adj_r2": support_out.get("model_adj_r2", np.nan),
        "support_rank_deficient": support_out.get("rank_deficient", np.nan),
        "support_condition_number": support_out.get("condition_number", np.nan),
        "support_model_failed": support_out.get("model_failed", False),
        "support_model_error": support_out.get("model_error", "")
    })

subject_coef_df = pd.DataFrame(coef_rows)

subject_coef_df.to_csv(
    os.path.join(CSV_DIR, "subject_first_confidence_disparity_report_coefficients.csv"),
    index=False
)

# -------------------------
# GROUP TESTS ON SUBJECT-LEVEL COEFFICIENTS
# -------------------------
tests = []

test_specs = [
    ("interaction_offset_x_different_report", "greater", "Does disparity increasingly favor confidence in different-source reports?"),
    ("interaction_same_report_offset_slope", "less", "Does confidence in same-source reports decrease as disparity increases?"),
    ("interaction_different_report_offset_slope", "greater", "Does confidence in different-source reports increase as disparity increases?"),
    ("support_slope", "greater", "Does confidence increase when the explicit report is supported by disparity?")
]

for metric, alt, desc in test_specs:
    vals = subject_coef_df[metric].dropna().astype(float)
    m, lo, hi, se, n = mean_ci_t(vals)
    p, n_perm = sign_flip_test(vals, alternative=alt, seed=RANDOM_SEED + len(tests) * 10)

    tests.append({
        "metric": metric,
        "description": desc,
        "alternative": alt,
        "n_subjects": n,
        "mean": m,
        "se": se,
        "ci95_low": lo,
        "ci95_high": hi,
        "p_perm": p
    })

test_df = pd.DataFrame(tests)
test_df["p_fdr"] = fdr_bh(test_df["p_perm"].values)

test_df.to_csv(
    os.path.join(CSV_DIR, "subject_first_confidence_disparity_report_signflip_tests.csv"),
    index=False
)

print("\nSubject-first coefficients:")
try:
    display(subject_coef_df.head())
except NameError:
    print(subject_coef_df.head().to_string(index=False))

print("\nSubject-first group tests:")
try:
    display(test_df)
except NameError:
    print(test_df.to_string(index=False))


# -------------------------
# OPTIONAL CLUSTER-ROBUST TRIAL-LEVEL ROBUSTNESS MODELS
# -------------------------
cluster_tables = []

cluster_specs = {
    "confidence_offset_x_report_clustered": "confidence_rating ~ offset_10_c * different_report",
    "confidence_report_support_clustered": "confidence_rating ~ report_support + different_report"
}

for model_name, formula in cluster_specs.items():
    try:
        m = smf.ols(formula, data=model_df_conf).fit(
            cov_type="cluster",
            cov_kwds={"groups": model_df_conf["subject_id"]}
        )

        with open(os.path.join(CSV_DIR, f"{model_name}.txt"), "w") as f:
            f.write(str(m.summary()))

        ci = m.conf_int()
        for term in m.params.index:
            cluster_tables.append({
                "model": model_name,
                "term": term,
                "b": m.params[term],
                "SE_cluster": m.bse[term],
                "z": m.tvalues[term],
                "p": m.pvalues[term],
                "ci95_low": ci.loc[term, 0],
                "ci95_high": ci.loc[term, 1],
                "n_obs": int(m.nobs),
                "r2": m.rsquared,
                "adj_r2": m.rsquared_adj
            })

        print(f"\nClustered robustness model: {model_name}")
        print(m.summary())

    except Exception as e:
        print(f"Clustered model failed for {model_name}: {e}")

cluster_df = pd.DataFrame(cluster_tables)
cluster_df.to_csv(
    os.path.join(CSV_DIR, "cluster_robust_confidence_disparity_report_models.csv"),
    index=False
)

print("\nAnalysis complete.")
print(f"Outputs saved to: {OUT_DIR}")

## 10. Explicit Causal Reports and Residual Visual Bias

Relations between explicit P(common)/causal reports and localization bias, residual-bias analyses, and whole-trial shuffle inference.

In [14]:
# %% ANALYSIS 2 + 3 — EXPLICIT P(COMMON) VS LOCALIZATION BIAS
# ============================================================
# Goal:
# Test whether visual localization bias decreases with explicit
# same-source judgment, and whether residual visual bias remains
# after accounting for causal judgment.
#
# Main variables:
#   Psame = same_response_yes
#       1 = participant reported same/common source
#       0 = participant reported different/separate source
#
#   Bias_AminusV = RMSE_A - RMSE_V
#       Positive = response closer to visual trajectory
#       Negative = response closer to auditory trajectory
#
# Models:
#   Model 1: Bias ~ Psame
#   Model 2: Bias ~ Offset + Psame
#   Model 3: Bias ~ Offset * Psame
#
# Residual test:
#   Fit Bias ~ Psame, extract residuals, then test whether residual
#   bias remains > 0 at 30° and 40°.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import ttest_1samp, t as t_dist
import matplotlib.pyplot as plt

# -------------------------
# OUTPUT DIRECTORY
# -------------------------
try:
    bias_out_dir = os.path.join(out_root, "explicit_causal_judgment_vs_bias")
except NameError:
    bias_out_dir = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\explicit_causal_judgment_vs_bias"

os.makedirs(bias_out_dir, exist_ok=True)

# -------------------------
# REQUIRED DATA CHECK
# -------------------------
if "trial_metrics_df" not in globals():
    raise RuntimeError("trial_metrics_df not found. Run your Cell 3 trial-level metric pipeline first.")

if "trial_df" not in globals():
    raise RuntimeError("trial_df not found. Run your load/resample cell first.")

# -------------------------
# HELPERS
# -------------------------
def parse_yes_no_same_response(x):
    """
    Interprets post_question as the participant's answer to:
    'same source?' yes=1, no=0.
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["y", "yes", "1", "true", "same", "same source", "common", "common source"]:
        return 1.0
    if s in ["n", "no", "0", "false", "different", "diff", "different source", "separate", "separate source"]:
        return 0.0
    return np.nan

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).dropna().astype(float).values
    n = len(vals)
    if n < 2:
        return np.nan, np.nan, np.nan, n
    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)
    return m, m - crit * se, m + crit * se, n

def safe_mixedlm(formula, data, groups, label, out_dir):
    """
    Fit mixed model with random intercept for subject.
    Uses Powell first for stability, then falls back to lbfgs.
    """
    print("\n" + "=" * 90)
    print(label)
    print("=" * 90)
    print(f"Formula: {formula}")
    print(f"Rows: {len(data)}")
    print(f"Subjects: {data[groups].nunique()}")

    try:
        model = smf.mixedlm(formula, data=data, groups=data[groups], re_formula="1")
        result = model.fit(method="powell", reml=False, maxiter=1000, disp=False)
    except Exception as e1:
        print(f"Powell failed: {e1}")
        print("Trying lbfgs...")
        model = smf.mixedlm(formula, data=data, groups=data[groups], re_formula="1")
        result = model.fit(method="lbfgs", reml=False, maxiter=1000)

    print(result.summary())

    safe_label = label.lower().replace(" ", "_").replace(":", "").replace("~", "")
    summary_path = os.path.join(out_dir, f"{safe_label}_summary.txt")
    coef_path = os.path.join(out_dir, f"{safe_label}_coefficients.csv")

    with open(summary_path, "w") as f:
        f.write(str(result.summary()))

    coef_df = pd.DataFrame({
        "term": result.params.index,
        "estimate": result.params.values,
        "se": result.bse.reindex(result.params.index).values,
        "z": result.tvalues.reindex(result.params.index).values,
        "p": result.pvalues.reindex(result.params.index).values,
    })
    coef_df.to_csv(coef_path, index=False)

    return result, coef_df

# -------------------------
# BUILD TRIAL-LEVEL MODEL DATA
# -------------------------
df = trial_metrics_df.copy()

# Add Psame from trial_df if needed.
if "same_response_yes" not in df.columns:
    if "same_response_yes" not in trial_df.columns:
        if "post_question" not in trial_df.columns:
            raise RuntimeError("Neither same_response_yes nor post_question found in trial_df.")
        trial_df = trial_df.copy()
        trial_df["same_response_yes"] = trial_df["post_question"].apply(parse_yes_no_same_response)

    judgment_cols = ["subject", "trialIndex", "same_response_yes"]
    if "post_question" in trial_df.columns:
        judgment_cols.append("post_question")
    if "post_question2" in trial_df.columns:
        judgment_cols.append("post_question2")

    judgment_df = trial_df[judgment_cols].drop_duplicates(subset=["subject", "trialIndex"], keep="first")
    df = df.merge(judgment_df, on=["subject", "trialIndex"], how="left")

# Restrict to fixed-offset trials only.
df = df[df["trialType"].astype(str).str.lower() == "incongruent_spatial"].copy()

df["offset_num"] = df.apply(extract_offset_from_row, axis=1)
df = df[df["offset_num"].isin([10, 20, 30, 40])].copy()

# Compute bias metrics.
if "RMSE_A_raw" not in df.columns or "RMSE_V_raw" not in df.columns:
    raise RuntimeError("RMSE_A_raw and/or RMSE_V_raw missing from trial_metrics_df.")

df["Bias_raw_AminusV"] = df["RMSE_A_raw"] - df["RMSE_V_raw"]

if "RMSE_A_lagcorr" in df.columns and "RMSE_V_lagcorr" in df.columns:
    df["Bias_lagcorr_AminusV"] = df["RMSE_A_lagcorr"] - df["RMSE_V_lagcorr"]
else:
    df["Bias_lagcorr_AminusV"] = np.nan

df["Psame"] = pd.to_numeric(df["same_response_yes"], errors="coerce")
df["Psame_centered"] = df["Psame"] - df["Psame"].mean(skipna=True)

df["offset_cat"] = pd.Categorical(
    df["offset_num"].astype(int).astype(str),
    categories=["10", "20", "30", "40"],
    ordered=True
)

model_df = df.dropna(subset=["subject", "offset_cat", "Psame", "Bias_raw_AminusV"]).copy()

print("\nModel data summary:")
print(model_df.groupby("offset_cat", observed=False).agg(
    n_trials=("trialIndex", "count"),
    n_subjects=("subject", "nunique"),
    Psame_mean=("Psame", "mean"),
    Bias_raw_mean=("Bias_raw_AminusV", "mean"),
    Bias_lagcorr_mean=("Bias_lagcorr_AminusV", "mean"),
))

model_data_csv = os.path.join(bias_out_dir, "trial_level_bias_psame_model_data.csv")
model_df.to_csv(model_data_csv, index=False)
print(f"\nSaved model data: {model_data_csv}")

# -------------------------
# MODEL 1-3 FOR RAW BIAS
# -------------------------
raw_m1, raw_m1_coef = safe_mixedlm(
    formula="Bias_raw_AminusV ~ Psame_centered",
    data=model_df,
    groups="subject",
    label="RAW Model 1 Bias ~ Psame",
    out_dir=bias_out_dir
)

raw_m2, raw_m2_coef = safe_mixedlm(
    formula="Bias_raw_AminusV ~ C(offset_cat) + Psame_centered",
    data=model_df,
    groups="subject",
    label="RAW Model 2 Bias ~ Offset + Psame",
    out_dir=bias_out_dir
)

raw_m3, raw_m3_coef = safe_mixedlm(
    formula="Bias_raw_AminusV ~ C(offset_cat) * Psame_centered",
    data=model_df,
    groups="subject",
    label="RAW Model 3 Bias ~ Offset x Psame",
    out_dir=bias_out_dir
)

# -------------------------
# OPTIONAL: MODEL 1-3 FOR LAG-CORRECTED BIAS
# -------------------------
lag_model_df = model_df.dropna(subset=["Bias_lagcorr_AminusV"]).copy()

if len(lag_model_df) > 0:
    lag_m1, lag_m1_coef = safe_mixedlm(
        formula="Bias_lagcorr_AminusV ~ Psame_centered",
        data=lag_model_df,
        groups="subject",
        label="LAGCORR Model 1 Bias ~ Psame",
        out_dir=bias_out_dir
    )

    lag_m2, lag_m2_coef = safe_mixedlm(
        formula="Bias_lagcorr_AminusV ~ C(offset_cat) + Psame_centered",
        data=lag_model_df,
        groups="subject",
        label="LAGCORR Model 2 Bias ~ Offset + Psame",
        out_dir=bias_out_dir
    )

    lag_m3, lag_m3_coef = safe_mixedlm(
        formula="Bias_lagcorr_AminusV ~ C(offset_cat) * Psame_centered",
        data=lag_model_df,
        groups="subject",
        label="LAGCORR Model 3 Bias ~ Offset x Psame",
        out_dir=bias_out_dir
    )

# -------------------------
# RESIDUAL BIAS AFTER ACCOUNTING FOR Psame
# -------------------------
# Use raw Model 1 residuals: visual bias not explained by explicit causal judgment.
model_df["ResidualBias_after_Psame_raw"] = raw_m1.resid

# Subject × offset means of residual bias.
resid_subject_offset = (
    model_df
    .groupby(["subject", "offset_num"], as_index=False)
    .agg(
        ResidualBias_after_Psame_raw=("ResidualBias_after_Psame_raw", "mean"),
        Bias_raw_AminusV=("Bias_raw_AminusV", "mean"),
        Psame=("Psame", "mean"),
        n_trials=("trialIndex", "count")
    )
)

resid_csv = os.path.join(bias_out_dir, "subject_offset_residual_bias_after_psame.csv")
resid_subject_offset.to_csv(resid_csv, index=False)

# Test residual bias > 0 at 30 and 40 degrees.
resid_test_rows = []

print("\n" + "=" * 90)
print("RESIDUAL BIAS TESTS AFTER ACCOUNTING FOR Psame")
print("=" * 90)

for off in [10, 20, 30, 40]:
    vals = resid_subject_offset.loc[
        resid_subject_offset["offset_num"] == off,
        "ResidualBias_after_Psame_raw"
    ].dropna().astype(float).values

    if len(vals) >= 2:
        m, lo, hi, n = mean_ci(vals)
        tres = ttest_1samp(vals, popmean=0.0, alternative="greater")
        print(f"\nOffset {off}°")
        print(f"n = {n}")
        print(f"Mean residual bias = {m:.4f}")
        print(f"95% CI = [{lo:.4f}, {hi:.4f}]")
        print(f"One-sample t-test > 0: t({n-1}) = {tres.statistic:.4f}, p {p_fmt(tres.pvalue)}")

        resid_test_rows.append({
            "offset": off,
            "n": n,
            "mean_residual_bias": m,
            "ci_low": lo,
            "ci_high": hi,
            "t": tres.statistic,
            "df": n - 1,
            "p_greater_than_zero": tres.pvalue
        })

resid_test_df = pd.DataFrame(resid_test_rows)
resid_test_csv = os.path.join(bias_out_dir, "residual_bias_after_psame_tests_by_offset.csv")
resid_test_df.to_csv(resid_test_csv, index=False)

# -------------------------
# SIMPLE DESCRIPTIVE PLOTS
# -------------------------
fig_dir = os.path.join(bias_out_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)

# Plot Psame by offset and bias by offset.
summary = (
    model_df
    .groupby(["subject", "offset_num"], as_index=False)
    .agg(
        Psame=("Psame", "mean"),
        Bias_raw_AminusV=("Bias_raw_AminusV", "mean"),
        Bias_lagcorr_AminusV=("Bias_lagcorr_AminusV", "mean")
    )
)

group_summary = (
    summary
    .groupby("offset_num", as_index=False)
    .agg(
        Psame_mean=("Psame", "mean"),
        Psame_sem=("Psame", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x)))),
        Bias_raw_mean=("Bias_raw_AminusV", "mean"),
        Bias_raw_sem=("Bias_raw_AminusV", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x)))),
        Bias_lagcorr_mean=("Bias_lagcorr_AminusV", "mean"),
        Bias_lagcorr_sem=("Bias_lagcorr_AminusV", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))))
    )
)

summary_csv = os.path.join(bias_out_dir, "subject_and_group_offset_psame_bias_summary.csv")
summary.to_csv(os.path.join(bias_out_dir, "subject_offset_psame_bias_summary.csv"), index=False)
group_summary.to_csv(summary_csv, index=False)

fig, ax1 = plt.subplots(figsize=(7, 5))
x = group_summary["offset_num"].values.astype(float)

ax1.errorbar(
    x,
    group_summary["Bias_raw_mean"],
    yerr=group_summary["Bias_raw_sem"],
    marker="o",
    linewidth=2,
    label="Bias A−V raw"
)
ax1.axhline(0, linestyle="--", linewidth=1)
ax1.set_xlabel("Offset (degrees)")
ax1.set_ylabel("Bias A−V: RMSE_A - RMSE_V")
ax1.set_title("Visual bias and explicit P(common) by offset")

ax2 = ax1.twinx()
ax2.errorbar(
    x,
    group_summary["Psame_mean"],
    yerr=group_summary["Psame_sem"],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="P(same)"
)
ax2.set_ylabel("P(same/common source)")
ax2.set_ylim(-0.05, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

fig.tight_layout()
fig_path = os.path.join(fig_dir, "offset_bias_and_psame_summary.png")
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print("\nSaved outputs:")
print(bias_out_dir)
print(f"Residual tests: {resid_test_csv}")
print(f"Summary figure: {fig_path}")


Model data summary:
            n_trials  n_subjects  Psame_mean  Bias_raw_mean  Bias_lagcorr_mean
offset_cat                                                                    
10              1000          50       0.732       4.346482           4.435101
20              1000          50       0.591       8.397239           8.190846
30              1000          50       0.307       8.847713           8.069980
40              1000          50       0.096       6.288036           5.142163

Saved model data: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\explicit_causal_judgment_vs_bias\trial_level_bias_psame_model_data.csv

RAW Model 1 Bias ~ Psame
Formula: Bias_raw_AminusV ~ Psame_centered
Rows: 4000
Subjects: 50
            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Bias_raw_AminusV
No. Observations: 4000    Method:             ML              
No. Groups:       50      Scale

# Residual Visual Bias

In [15]:
# %% ANALYSIS 2 + 3 — EXPLICIT P(COMMON) VS LOCALIZATION BIAS
# STABLE WINDOW VERSION: 0.5–17.5 s
# =============================================================================
# GOAL
# =============================================================================
# Test whether visual localization bias is explained by explicit same-source
# judgment and whether residual visual bias remains after accounting for
# causal judgment.
#
# Stable-window analysis:
#   0.5–17.5 s
#
# Main variables:
#
#   Psame = same_response_yes
#       1 = participant reported same/common source
#       0 = participant reported different/separate source
#
#   Bias_AminusV = RMSE_A - RMSE_V
#       Positive = response closer to visual trajectory
#       Negative = response closer to auditory trajectory
#
# Models:
#   Model 1: Bias ~ Psame
#   Model 2: Bias ~ Offset + Psame
#   Model 3: Bias ~ Offset * Psame
#
# Residual test:
#   Fit Bias ~ Psame, extract residuals, then test whether residual
#   visual bias remains > 0 within each physical offset.
#
# PRIMARY PAPER METRIC:
#   Lag-corrected residual bias after accounting for Psame.
#
# ROBUSTNESS METRIC:
#   Raw residual bias after accounting for Psame.
#
# =============================================================================
# IMPORTANT LAG UPDATE
# =============================================================================
# Lag handling now follows the revised notebook logic:
#
#   1. Use LagA_sec / LagV_sec already present in trial_df when available.
#   2. Otherwise merge existing values from xcorr_df.
#   3. Otherwise merge values from trial_metrics_df.
#   4. Any remaining missing lag is COMPUTED from that trial's arrays.
#
# Lag search:
#   0–4 s ONLY
#
# Positive lag:
#   behavioral response follows the stimulus.
#
# NO PEAK-r EXCLUSION is applied here.
#
# Rationale:
#   Lag is being used as a temporal alignment/nuisance correction for
#   audiovisual spatial bias, not as the dependent behavioral outcome.
#   Weak response-stimulus coupling during AV conflict can itself reflect
#   multisensory weighting and therefore should not cause trial deletion.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

from scipy.stats import (
    ttest_1samp,
    t as t_dist,
)


# =============================================================================
# SETTINGS
# =============================================================================

STABLE_START_SEC = 0.5
STABLE_END_SEC = 17.5

FIG_DPI = 180

RESAMPLE_HZ = 20.0

# Revised lag search
MIN_LAG_SEC = 0.0
MAX_LAG_SEC = 4.0

MIN_LAG_VALID_SAMPLES = 10


SPATIAL_ANALYSES_ROOT = (
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1\spatial_analyses"
)

bias_out_dir = os.path.join(
    SPATIAL_ANALYSES_ROOT,
    "05_residual_visual_bias",
)

fig_dir = os.path.join(
    bias_out_dir,
    "figures",
)

lag_diag_dir = os.path.join(
    bias_out_dir,
    "lag_diagnostics",
)


for d in [
    bias_out_dir,
    fig_dir,
    lag_diag_dir,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# =============================================================================
# REQUIRED DATA CHECKS
# =============================================================================

if "trial_df" not in globals():

    raise RuntimeError(
        "trial_df not found. Run your load/resample cell first. "
        "This stable-window residual analysis requires trial_df with "
        "t_grid, mouse_x, audio_x, and blob_x arrays."
    )


if "trial_metrics_df" not in globals():

    print(
        "trial_metrics_df not found in memory. "
        "That is okay: missing lag estimates will be computed directly "
        "from the trial trajectories."
    )


# =============================================================================
# BASIC HELPERS
# =============================================================================

def parse_yes_no_same_response(x):
    """
    Interpret same-source response as:
        Same/Common = 1
        Different/Separate = 0
    """

    if pd.isna(x):
        return np.nan

    if isinstance(x, (bool, np.bool_)):
        return float(x)

    try:
        numeric = float(x)

        if numeric in [0.0, 1.0]:
            return numeric

    except Exception:
        pass

    s = str(x).strip().lower()

    same_labels = [
        "y",
        "yes",
        "1",
        "true",
        "same",
        "same source",
        "same_source",
        "common",
        "common source",
        "common_source",
        "one source",
        "one_source",
    ]

    different_labels = [
        "n",
        "no",
        "0",
        "false",
        "different",
        "diff",
        "different source",
        "different_source",
        "separate",
        "separate source",
        "separate_source",
        "two sources",
        "two_sources",
    ]

    if s in same_labels:
        return 1.0

    if s in different_labels:
        return 0.0

    return np.nan


def extract_offset_from_row(row):
    """
    Extract absolute fixed AV offset in degrees.
    """

    numeric_candidates = [
        "offsetDeg",
        "offset_deg",
        "disparity_deg",
        "offset",
    ]

    for col in numeric_candidates:

        if col in row.index and pd.notna(row[col]):

            try:

                value = abs(
                    float(
                        row[col]
                    )
                )

                value = int(
                    round(
                        value
                    )
                )

                if value in [
                    10,
                    20,
                    30,
                    40,
                ]:
                    return value

            except Exception:
                pass


    text_candidates = [
        "condLabel",
        "condition",
        "physical_condition",
        "condition_detailed",
    ]

    for col in text_candidates:

        if col not in row.index:
            continue

        s = str(
            row.get(
                col,
                "",
            )
        ).lower()

        m = re.search(
            r"(10|20|30|40)",
            s,
        )

        if m:
            return int(
                m.group(1)
            )

    return np.nan


def safe_array(x):

    return np.asarray(
        x,
        dtype=float,
    )


def p_fmt(p):

    if pd.isna(p):
        return "nan"

    if p < .001:
        return "< .001"

    return f"= {p:.4f}"


def mean_ci(
    vals,
    alpha=0.05,
):

    vals = (
        pd.Series(
            vals
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .astype(float)
        .values
    )

    n = len(
        vals
    )

    if n < 2:

        return (
            np.nan,
            np.nan,
            np.nan,
            n,
        )

    m = float(
        np.mean(
            vals
        )
    )

    se = float(
        np.std(
            vals,
            ddof=1,
        )
        /
        np.sqrt(
            n
        )
    )

    crit = float(
        t_dist.ppf(
            1 - alpha / 2,
            df=n - 1,
        )
    )

    return (
        m,
        m - crit * se,
        m + crit * se,
        n,
    )


def sem(x):

    x = (
        pd.Series(
            x
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .astype(float)
    )

    if len(x) < 2:
        return np.nan

    return float(
        x.std(
            ddof=1
        )
        /
        np.sqrt(
            len(x)
        )
    )


def safe_label_to_filename(label):

    safe = label.lower()

    safe = safe.replace(
        " ",
        "_",
    )

    safe = safe.replace(
        ":",
        "",
    )

    safe = safe.replace(
        "~",
        "",
    )

    safe = safe.replace(
        "*",
        "x",
    )

    safe = safe.replace(
        "×",
        "x",
    )

    safe = re.sub(
        r"[^a-z0-9_\-+]+",
        "_",
        safe,
    )

    safe = re.sub(
        r"_+",
        "_",
        safe,
    ).strip(
        "_"
    )

    return safe


# =============================================================================
# CORRELATION / LAG HELPERS
# =============================================================================

def corr_1d(
    x,
    y,
):

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    ok = (
        np.isfinite(
            x
        )
        &
        np.isfinite(
            y
        )
    )

    if ok.sum() < MIN_LAG_VALID_SAMPLES:
        return np.nan

    if (
        np.nanstd(
            x[
                ok
            ]
        ) < 1e-12
        or
        np.nanstd(
            y[
                ok
            ]
        ) < 1e-12
    ):
        return np.nan

    return float(
        np.corrcoef(
            x[
                ok
            ],
            y[
                ok
            ],
        )[0, 1]
    )


def estimate_positive_lag_xcorr(
    response,
    stimulus,
    t_grid=None,
    max_lag_sec=MAX_LAG_SEC,
    fs=None,
):
    """
    Estimate behavioral response delay using response-stimulus
    cross-correlation.

    Search:
        0–4 s only.

    Positive lag:
        response follows stimulus.

    IMPORTANT:
        No peak-r threshold is applied here because lag is a nuisance
        temporal correction for the spatial-bias analysis.

    Returns:
        lag_sec
        lag_samples
        peak_corr
        at_zero_boundary
        at_max_boundary
    """

    response = np.asarray(
        response,
        dtype=float,
    )

    stimulus = np.asarray(
        stimulus,
        dtype=float,
    )


    # -------------------------------------------------------------------------
    # Determine sampling frequency
    # -------------------------------------------------------------------------

    if fs is None:

        if t_grid is None:

            fs = RESAMPLE_HZ

        else:

            tt = np.asarray(
                t_grid,
                dtype=float,
            )

            finite_t = tt[
                np.isfinite(
                    tt
                )
            ]

            if len(
                finite_t
            ) >= 2:

                dt = np.nanmedian(
                    np.diff(
                        finite_t
                    )
                )

            else:

                dt = np.nan


            if (
                np.isfinite(
                    dt
                )
                and dt > 0
            ):

                fs = 1.0 / dt

            else:

                fs = RESAMPLE_HZ


    max_lag_samples = int(
        round(
            max_lag_sec
            *
            fs
        )
    )


    best_corr = np.nan
    best_lag_samples = np.nan


    # -------------------------------------------------------------------------
    # Search NON-NEGATIVE lags only
    # -------------------------------------------------------------------------

    for lag_samples in range(
        0,
        max_lag_samples + 1,
    ):

        if lag_samples == 0:

            aligned_response = response
            aligned_stimulus = stimulus

        else:

            aligned_response = response[
                lag_samples:
            ]

            aligned_stimulus = stimulus[
                :-lag_samples
            ]


        r = corr_1d(
            aligned_response,
            aligned_stimulus,
        )


        if (
            np.isfinite(
                r
            )
            and
            (
                not np.isfinite(
                    best_corr
                )
                or r > best_corr
            )
        ):

            best_corr = r
            best_lag_samples = lag_samples


    if not np.isfinite(
        best_lag_samples
    ):

        return {
            "lag_sec": np.nan,
            "lag_samples": np.nan,
            "peak_corr": np.nan,
            "at_zero_boundary": False,
            "at_max_boundary": False,
        }


    lag_sec = float(
        best_lag_samples
        /
        fs
    )


    return {
        "lag_sec":
            lag_sec,

        "lag_samples":
            int(
                best_lag_samples
            ),

        "peak_corr":
            float(
                best_corr
            ),

        "at_zero_boundary":
            bool(
                best_lag_samples == 0
            ),

        "at_max_boundary":
            bool(
                best_lag_samples
                ==
                max_lag_samples
            ),
    }


def shift_response_by_lag(
    response,
    t_grid,
    lag_sec,
):
    """
    Temporally align delayed behavioral response with stimulus time.

    If lag > 0:
        response(t) corresponds approximately to stimulus(t - lag)

    Therefore:
        corrected response at stimulus time t = response(t + lag).
    """

    response = np.asarray(
        response,
        dtype=float,
    )

    t_grid = np.asarray(
        t_grid,
        dtype=float,
    )

    if not np.isfinite(
        lag_sec
    ):

        return np.full_like(
            response,
            np.nan,
            dtype=float,
        )


    return np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan,
    )


# =============================================================================
# MIXED-MODEL HELPER
# =============================================================================

def safe_mixedlm(
    formula,
    data,
    groups,
    label,
    out_dir,
):
    """
    Random-intercept mixed model.
    """

    print(
        "\n"
        +
        "=" * 90
    )

    print(
        label
    )

    print(
        "=" * 90
    )

    print(
        f"Formula: {formula}"
    )

    print(
        f"Rows: {len(data)}"
    )

    print(
        f"Subjects: {data[groups].nunique()}"
    )


    try:

        model = smf.mixedlm(
            formula,
            data=data,
            groups=data[
                groups
            ],
            re_formula="1",
        )

        result = model.fit(
            method="powell",
            reml=False,
            maxiter=1000,
            disp=False,
        )

    except Exception as e1:

        print(
            f"Powell failed: {e1}"
        )

        print(
            "Trying lbfgs..."
        )

        model = smf.mixedlm(
            formula,
            data=data,
            groups=data[
                groups
            ],
            re_formula="1",
        )

        result = model.fit(
            method="lbfgs",
            reml=False,
            maxiter=1000,
        )


    print(
        result.summary()
    )


    safe_label = safe_label_to_filename(
        label
    )

    summary_path = os.path.join(
        out_dir,
        f"{safe_label}_summary.txt",
    )

    coef_path = os.path.join(
        out_dir,
        f"{safe_label}_coefficients.csv",
    )


    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            str(
                result.summary()
            )
        )


    coef_df = pd.DataFrame(
        {
            "term":
                result.params.index,

            "estimate":
                result.params.values,

            "se":
                result.bse.reindex(
                    result.params.index
                ).values,

            "z":
                result.tvalues.reindex(
                    result.params.index
                ).values,

            "p":
                result.pvalues.reindex(
                    result.params.index
                ).values,
        }
    )


    coef_df.to_csv(
        coef_path,
        index=False,
    )


    return (
        result,
        coef_df,
    )


# =============================================================================
# RESIDUAL TEST HELPER
# =============================================================================

def test_residuals_by_offset(
    resid_subject_offset,
    resid_col,
    out_csv,
    label,
):
    """
    One-sample tests of residual visual bias against zero within each offset.
    """

    rows = []


    print(
        "\n"
        +
        "=" * 90
    )

    print(
        label
    )

    print(
        "=" * 90
    )


    for off in [
        10,
        20,
        30,
        40,
    ]:

        vals = (
            resid_subject_offset.loc[
                resid_subject_offset[
                    "offset_num"
                ] == off,
                resid_col,
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
            .dropna()
            .astype(float)
            .values
        )


        if len(
            vals
        ) < 2:

            continue


        m, lo, hi, n = mean_ci(
            vals
        )


        t_two = ttest_1samp(
            vals,
            popmean=0.0,
        )

        t_greater = ttest_1samp(
            vals,
            popmean=0.0,
            alternative="greater",
        )


        print(
            f"\nOffset {off}°"
        )

        print(
            f"n = {n}"
        )

        print(
            f"Mean residual bias = {m:.4f}"
        )

        print(
            f"95% CI = [{lo:.4f}, {hi:.4f}]"
        )

        print(
            "Two-sided one-sample t-test vs 0: "
            f"t({n - 1}) = {t_two.statistic:.4f}, "
            f"p {p_fmt(t_two.pvalue)}"
        )

        print(
            "One-sided one-sample t-test > 0: "
            f"t({n - 1}) = {t_greater.statistic:.4f}, "
            f"p {p_fmt(t_greater.pvalue)}"
        )


        rows.append(
            {
                "offset":
                    off,

                "n":
                    n,

                "mean_residual_bias":
                    m,

                "ci_low":
                    lo,

                "ci_high":
                    hi,

                "t_two_sided":
                    t_two.statistic,

                "df":
                    n - 1,

                "p_two_sided":
                    t_two.pvalue,

                "t_greater_than_zero":
                    t_greater.statistic,

                "p_greater_than_zero":
                    t_greater.pvalue,
            }
        )


    test_df = pd.DataFrame(
        rows
    )

    test_df.to_csv(
        out_csv,
        index=False,
    )


    print(
        f"\nSaved residual test table:\n{out_csv}"
    )


    return test_df


# =============================================================================
# STANDARDIZE KEYS
# =============================================================================

def standardize_merge_keys(df):

    df = df.copy()

    if "subject" in df.columns:

        df[
            "subject"
        ] = (
            df[
                "subject"
            ]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    if "trialIndex" in df.columns:

        df[
            "trialIndex"
        ] = pd.to_numeric(
            df[
                "trialIndex"
            ],
            errors="coerce",
        )

    return df


# =============================================================================
# MERGE EXISTING LAGS, THEN COMPUTE MISSING LAGS
# =============================================================================

def obtain_complete_trial_lags(df):
    """
    Create LagA_sec and LagV_sec for every trial possible.

    Priority:
        1. Existing columns already in trial_df.
        2. xcorr_df.
        3. trial_metrics_df.
        4. Compute remaining missing values directly from arrays.

    The fallback computation uses the agreed non-negative 0–4 s
    cross-correlation lag search and DOES NOT use a peak-r exclusion.
    """

    df = standardize_merge_keys(
        df
    )


    # -------------------------------------------------------------------------
    # Ensure lag columns exist
    # -------------------------------------------------------------------------

    for lag_col in [
        "LagA_sec",
        "LagV_sec",
    ]:

        if lag_col not in df.columns:

            df[
                lag_col
            ] = np.nan

        else:

            df[
                lag_col
            ] = pd.to_numeric(
                df[
                    lag_col
                ],
                errors="coerce",
            )


    # -------------------------------------------------------------------------
    # Source labels
    # -------------------------------------------------------------------------

    df[
        "LagA_source"
    ] = np.where(
        np.isfinite(
            df[
                "LagA_sec"
            ]
        ),
        "trial_df",
        "missing",
    )

    df[
        "LagV_source"
    ] = np.where(
        np.isfinite(
            df[
                "LagV_sec"
            ]
        ),
        "trial_df",
        "missing",
    )


    print(
        "\nInitial lag availability in trial_df:"
    )

    print(
        df[
            [
                "LagA_sec",
                "LagV_sec",
            ]
        ].notna().sum()
    )


    # =========================================================================
    # SOURCE 2 — XCORR_DF
    # =========================================================================

    if "xcorr_df" in globals():

        xdf = standardize_merge_keys(
            xcorr_df
        )

        if all(
            c in xdf.columns
            for c in [
                "LagA_sec",
                "LagV_sec",
            ]
        ):

            print(
                "\nFound LagA_sec / LagV_sec in xcorr_df."
            )


            # -----------------------------------------------------------------
            # Preferred merge: subject + trialIndex
            # -----------------------------------------------------------------

            if all(
                c in xdf.columns
                for c in [
                    "subject",
                    "trialIndex",
                ]
            ):

                lookup = (
                    xdf[
                        [
                            "subject",
                            "trialIndex",
                            "LagA_sec",
                            "LagV_sec",
                        ]
                    ]
                    .drop_duplicates(
                        subset=[
                            "subject",
                            "trialIndex",
                        ],
                        keep="first",
                    )
                    .rename(
                        columns={
                            "LagA_sec":
                                "__xcorr_LagA_sec",

                            "LagV_sec":
                                "__xcorr_LagV_sec",
                        }
                    )
                )


                df = df.merge(
                    lookup,
                    on=[
                        "subject",
                        "trialIndex",
                    ],
                    how="left",
                )


            # -----------------------------------------------------------------
            # Fallback merge: original trial_df index
            # -----------------------------------------------------------------

            elif (
                "trial_df_index" in xdf.columns
                and
                "__trial_df_index__" in df.columns
            ):

                lookup = (
                    xdf[
                        [
                            "trial_df_index",
                            "LagA_sec",
                            "LagV_sec",
                        ]
                    ]
                    .drop_duplicates(
                        subset=[
                            "trial_df_index"
                        ],
                        keep="first",
                    )
                    .rename(
                        columns={
                            "LagA_sec":
                                "__xcorr_LagA_sec",

                            "LagV_sec":
                                "__xcorr_LagV_sec",
                        }
                    )
                )


                df = df.merge(
                    lookup,
                    left_on="__trial_df_index__",
                    right_on="trial_df_index",
                    how="left",
                )


            else:

                lookup = None

                print(
                    "WARNING: xcorr_df has lag columns but no usable "
                    "trial merge keys."
                )


            if (
                "__xcorr_LagA_sec"
                in df.columns
            ):

                df[
                    "__xcorr_LagA_sec"
                ] = pd.to_numeric(
                    df[
                        "__xcorr_LagA_sec"
                    ],
                    errors="coerce",
                )

                use = (
                    df[
                        "LagA_sec"
                    ].isna()
                    &
                    df[
                        "__xcorr_LagA_sec"
                    ].notna()
                )

                df.loc[
                    use,
                    "LagA_sec",
                ] = df.loc[
                    use,
                    "__xcorr_LagA_sec",
                ]

                df.loc[
                    use,
                    "LagA_source",
                ] = "xcorr_df"


            if (
                "__xcorr_LagV_sec"
                in df.columns
            ):

                df[
                    "__xcorr_LagV_sec"
                ] = pd.to_numeric(
                    df[
                        "__xcorr_LagV_sec"
                    ],
                    errors="coerce",
                )

                use = (
                    df[
                        "LagV_sec"
                    ].isna()
                    &
                    df[
                        "__xcorr_LagV_sec"
                    ].notna()
                )

                df.loc[
                    use,
                    "LagV_sec",
                ] = df.loc[
                    use,
                    "__xcorr_LagV_sec",
                ]

                df.loc[
                    use,
                    "LagV_source",
                ] = "xcorr_df"


            drop_cols = [
                c for c in [
                    "__xcorr_LagA_sec",
                    "__xcorr_LagV_sec",
                    "trial_df_index",
                ]
                if c in df.columns
            ]

            if len(
                drop_cols
            ) > 0:

                df = df.drop(
                    columns=drop_cols
                )


    # =========================================================================
    # SOURCE 3 — TRIAL_METRICS_DF
    # =========================================================================

    if "trial_metrics_df" in globals():

        tmdf = standardize_merge_keys(
            trial_metrics_df
        )


        audio_candidates = [
            "LagA_sec",
            "lagA_sec",
            "LagAudio_sec",
            "AudioLag_sec",
            "audio_lag_sec",
            "lag_audio_sec",
            "best_audio_lag_sec",
        ]

        visual_candidates = [
            "LagV_sec",
            "lagV_sec",
            "LagVisual_sec",
            "VisualLag_sec",
            "visual_lag_sec",
            "lag_visual_sec",
            "best_visual_lag_sec",
        ]


        audio_col = next(
            (
                c for c in audio_candidates
                if c in tmdf.columns
            ),
            None,
        )

        visual_col = next(
            (
                c for c in visual_candidates
                if c in tmdf.columns
            ),
            None,
        )


        if (
            audio_col is not None
            or
            visual_col is not None
        ):

            if all(
                c in tmdf.columns
                for c in [
                    "subject",
                    "trialIndex",
                ]
            ):

                lookup_cols = [
                    "subject",
                    "trialIndex",
                ]

                rename_dict = {}


                if audio_col is not None:

                    lookup_cols.append(
                        audio_col
                    )

                    rename_dict[
                        audio_col
                    ] = "__metric_LagA_sec"


                if visual_col is not None:

                    lookup_cols.append(
                        visual_col
                    )

                    rename_dict[
                        visual_col
                    ] = "__metric_LagV_sec"


                lookup = (
                    tmdf[
                        lookup_cols
                    ]
                    .drop_duplicates(
                        subset=[
                            "subject",
                            "trialIndex",
                        ],
                        keep="first",
                    )
                    .rename(
                        columns=rename_dict
                    )
                )


                df = df.merge(
                    lookup,
                    on=[
                        "subject",
                        "trialIndex",
                    ],
                    how="left",
                )


                if (
                    "__metric_LagA_sec"
                    in df.columns
                ):

                    df[
                        "__metric_LagA_sec"
                    ] = pd.to_numeric(
                        df[
                            "__metric_LagA_sec"
                        ],
                        errors="coerce",
                    )

                    use = (
                        df[
                            "LagA_sec"
                        ].isna()
                        &
                        df[
                            "__metric_LagA_sec"
                        ].notna()
                    )

                    df.loc[
                        use,
                        "LagA_sec",
                    ] = df.loc[
                        use,
                        "__metric_LagA_sec",
                    ]

                    df.loc[
                        use,
                        "LagA_source",
                    ] = "trial_metrics_df"


                if (
                    "__metric_LagV_sec"
                    in df.columns
                ):

                    df[
                        "__metric_LagV_sec"
                    ] = pd.to_numeric(
                        df[
                            "__metric_LagV_sec"
                        ],
                        errors="coerce",
                    )

                    use = (
                        df[
                            "LagV_sec"
                        ].isna()
                        &
                        df[
                            "__metric_LagV_sec"
                        ].notna()
                    )

                    df.loc[
                        use,
                        "LagV_sec",
                    ] = df.loc[
                        use,
                        "__metric_LagV_sec",
                    ]

                    df.loc[
                        use,
                        "LagV_source",
                    ] = "trial_metrics_df"


                drop_cols = [
                    c for c in [
                        "__metric_LagA_sec",
                        "__metric_LagV_sec",
                    ]
                    if c in df.columns
                ]

                if len(
                    drop_cols
                ) > 0:

                    df = df.drop(
                        columns=drop_cols
                    )


    # =========================================================================
    # INVALIDATE LEGACY LAGS OUTSIDE REVISED RANGE
    # =========================================================================

    for lag_col, source_col in [
        (
            "LagA_sec",
            "LagA_source",
        ),
        (
            "LagV_sec",
            "LagV_source",
        ),
    ]:

        invalid = (
            np.isfinite(
                df[
                    lag_col
                ]
            )
            &
            (
                (
                    df[
                        lag_col
                    ] < MIN_LAG_SEC
                )
                |
                (
                    df[
                        lag_col
                    ] > MAX_LAG_SEC
                )
            )
        )


        if invalid.any():

            print(
                f"\nInvalidating {int(invalid.sum())} "
                f"legacy {lag_col} estimates outside "
                f"{MIN_LAG_SEC:.1f}–{MAX_LAG_SEC:.1f} s."
            )

            df.loc[
                invalid,
                lag_col,
            ] = np.nan

            df.loc[
                invalid,
                source_col,
            ] = "missing_revised_range"


    # =========================================================================
    # REPORT BEFORE FALLBACK COMPUTATION
    # =========================================================================

    print(
        "\nLag availability after merging existing notebook outputs:"
    )

    print(
        df[
            [
                "LagA_sec",
                "LagV_sec",
            ]
        ].describe()
    )


    print(
        "\nMissing lag counts before fallback computation:"
    )

    print(
        df[
            [
                "LagA_sec",
                "LagV_sec",
            ]
        ].isna().sum()
    )


    # =========================================================================
    # SOURCE 4 — COMPUTE ALL REMAINING MISSING VALUES
    # =========================================================================

    required_arrays = [
        "t_grid",
        "mouse_x",
        "audio_x",
        "blob_x",
    ]

    missing_arrays = [
        c for c in required_arrays
        if c not in df.columns
    ]


    if len(
        missing_arrays
    ) > 0:

        raise RuntimeError(
            "Cannot compute missing lags because these trajectory columns "
            f"are absent: {missing_arrays}"
        )


    need_audio = (
        df[
            "LagA_sec"
        ].isna()
    )

    need_visual = (
        df[
            "LagV_sec"
        ].isna()
    )


    print(
        "\nComputing remaining missing lags directly from trial trajectories..."
    )

    print(
        f"Audio lags to compute:  {int(need_audio.sum())}"
    )

    print(
        f"Visual lags to compute: {int(need_visual.sum())}"
    )


    # Diagnostic columns
    df[
        "LagA_peak_corr"
    ] = np.nan

    df[
        "LagV_peak_corr"
    ] = np.nan

    df[
        "LagA_zero_boundary"
    ] = False

    df[
        "LagV_zero_boundary"
    ] = False

    df[
        "LagA_max_boundary"
    ] = False

    df[
        "LagV_max_boundary"
    ] = False


    for idx, row in df.iterrows():

        if (
            not pd.isna(
                row[
                    "LagA_sec"
                ]
            )
            and
            not pd.isna(
                row[
                    "LagV_sec"
                ]
            )
        ):
            continue


        t = safe_array(
            row[
                "t_grid"
            ]
        )

        resp = safe_array(
            row[
                "mouse_x"
            ]
        )

        aud = safe_array(
            row[
                "audio_x"
            ]
        )

        vis = safe_array(
            row[
                "blob_x"
            ]
        )


        # ---------------------------------------------------------------------
        # Auditory lag
        # ---------------------------------------------------------------------

        if not np.isfinite(
            row[
                "LagA_sec"
            ]
        ):

            lag_a = estimate_positive_lag_xcorr(
                response=resp,
                stimulus=aud,
                t_grid=t,
                max_lag_sec=MAX_LAG_SEC,
            )

            df.at[
                idx,
                "LagA_sec",
            ] = lag_a[
                "lag_sec"
            ]

            df.at[
                idx,
                "LagA_peak_corr",
            ] = lag_a[
                "peak_corr"
            ]

            df.at[
                idx,
                "LagA_zero_boundary",
            ] = lag_a[
                "at_zero_boundary"
            ]

            df.at[
                idx,
                "LagA_max_boundary",
            ] = lag_a[
                "at_max_boundary"
            ]

            if np.isfinite(
                lag_a[
                    "lag_sec"
                ]
            ):

                df.at[
                    idx,
                    "LagA_source",
                ] = "computed_xcorr_0to4s"


        # ---------------------------------------------------------------------
        # Visual lag
        # ---------------------------------------------------------------------

        if not np.isfinite(
            row[
                "LagV_sec"
            ]
        ):

            lag_v = estimate_positive_lag_xcorr(
                response=resp,
                stimulus=vis,
                t_grid=t,
                max_lag_sec=MAX_LAG_SEC,
            )

            df.at[
                idx,
                "LagV_sec",
            ] = lag_v[
                "lag_sec"
            ]

            df.at[
                idx,
                "LagV_peak_corr",
            ] = lag_v[
                "peak_corr"
            ]

            df.at[
                idx,
                "LagV_zero_boundary",
            ] = lag_v[
                "at_zero_boundary"
            ]

            df.at[
                idx,
                "LagV_max_boundary",
            ] = lag_v[
                "at_max_boundary"
            ]

            if np.isfinite(
                lag_v[
                    "lag_sec"
                ]
            ):

                df.at[
                    idx,
                    "LagV_source",
                ] = "computed_xcorr_0to4s"


    # =========================================================================
    # FINAL DIAGNOSTICS
    # =========================================================================

    print(
        "\nFinal lag availability:"
    )

    print(
        df[
            [
                "LagA_sec",
                "LagV_sec",
            ]
        ].describe()
    )


    print(
        "\nFinal missing lag counts:"
    )

    print(
        df[
            [
                "LagA_sec",
                "LagV_sec",
            ]
        ].isna().sum()
    )


    print(
        "\nAudio lag source counts:"
    )

    print(
        df[
            "LagA_source"
        ].value_counts(
            dropna=False
        )
    )


    print(
        "\nVisual lag source counts:"
    )

    print(
        df[
            "LagV_source"
        ].value_counts(
            dropna=False
        )
    )


    return df


# =============================================================================
# BUILD STABLE-WINDOW TRIAL-LEVEL DATASET
# =============================================================================

df = trial_df.copy()


# Preserve original trial_df index just in case xcorr_df uses it.
df[
    "__trial_df_index__"
] = df.index


# =============================================================================
# SAME / DIFFERENT JUDGMENT
# =============================================================================

if "same_response_yes" not in df.columns:

    if (
        "trial_metrics_df" in globals()
        and
        "same_response_yes" in trial_metrics_df.columns
        and
        all(
            c in trial_metrics_df.columns
            for c in [
                "subject",
                "trialIndex",
            ]
        )
    ):

        judgment_df = standardize_merge_keys(
            trial_metrics_df[
                [
                    "subject",
                    "trialIndex",
                    "same_response_yes",
                ]
            ].copy()
        )

        judgment_df = judgment_df.drop_duplicates(
            subset=[
                "subject",
                "trialIndex",
            ],
            keep="first",
        )


        df = standardize_merge_keys(
            df
        )

        df = df.merge(
            judgment_df,
            on=[
                "subject",
                "trialIndex",
            ],
            how="left",
        )


    elif "post_question" in df.columns:

        df[
            "same_response_yes"
        ] = df[
            "post_question"
        ].apply(
            parse_yes_no_same_response
        )


    else:

        raise RuntimeError(
            "Neither same_response_yes nor post_question was found in "
            "trial_df, and same_response_yes was not available from "
            "trial_metrics_df."
        )


# =============================================================================
# RESTRICT TO FIXED-OFFSET TRIALS BEFORE EXPENSIVE FALLBACK LAG COMPUTATION
# =============================================================================

if "trialType" not in df.columns:

    raise RuntimeError(
        "trialType is missing from trial_df."
    )


fixed_mask = (
    df[
        "trialType"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq(
        "incongruent_spatial"
    )
)


df = df[
    fixed_mask
].copy()


df[
    "offset_num"
] = df.apply(
    extract_offset_from_row,
    axis=1,
)


df = df[
    df[
        "offset_num"
    ].isin(
        [
            10,
            20,
            30,
            40,
        ]
    )
].copy()


print(
    "\n"
    +
    "=" * 90
)

print(
    "FIXED-OFFSET TRIALS BEFORE LAG COMPLETION"
)

print(
    "=" * 90
)

print(
    f"Trials: {len(df)}"
)

print(
    f"Subjects: {df['subject'].nunique()}"
)


# =============================================================================
# REQUIRED TRAJECTORY ARRAYS
# =============================================================================

required_array_cols = [
    "t_grid",
    "mouse_x",
    "audio_x",
    "blob_x",
]


missing_array_cols = [
    c for c in required_array_cols
    if c not in df.columns
]


if len(
    missing_array_cols
) > 0:

    raise RuntimeError(
        "The following required trajectory columns are missing from "
        f"trial_df: {missing_array_cols}"
    )


# =============================================================================
# COMPLETE LAG ESTIMATES
# =============================================================================

df = obtain_complete_trial_lags(
    df
)


# =============================================================================
# Psame
# =============================================================================

df[
    "Psame"
] = pd.to_numeric(
    df[
        "same_response_yes"
    ],
    errors="coerce",
)


df = df.dropna(
    subset=[
        "Psame"
    ]
).copy()


df[
    "offset_cat"
] = pd.Categorical(
    df[
        "offset_num"
    ]
    .astype(int)
    .astype(str),
    categories=[
        "10",
        "20",
        "30",
        "40",
    ],
    ordered=True,
)


print(
    "\nFixed-offset trials available before stable-window metric computation:"
)

print(
    df.groupby(
        [
            "offset_num",
            "Psame",
        ],
        observed=False,
    ).size()
)


print(
    "\nLag-column check after completion:"
)

print(
    df[
        [
            "LagA_sec",
            "LagV_sec",
        ]
    ].describe()
)


print(
    "\nMissing lag counts after completion:"
)

print(
    df[
        [
            "LagA_sec",
            "LagV_sec",
        ]
    ].isna().sum()
)


# =============================================================================
# SAVE TRIAL-LEVEL LAG DIAGNOSTICS
# =============================================================================

lag_diag_cols = [
    c for c in [
        "subject",
        "trialIndex",
        "offset_num",
        "Psame",

        "LagA_sec",
        "LagV_sec",

        "LagA_source",
        "LagV_source",

        "LagA_peak_corr",
        "LagV_peak_corr",

        "LagA_zero_boundary",
        "LagV_zero_boundary",

        "LagA_max_boundary",
        "LagV_max_boundary",
    ]
    if c in df.columns
]


lag_diagnostics_df = df[
    lag_diag_cols
].copy()


lag_diagnostics_csv = os.path.join(
    lag_diag_dir,
    "fixed_offset_trial_lag_diagnostics.csv",
)


lag_diagnostics_df.to_csv(
    lag_diagnostics_csv,
    index=False,
)


# =============================================================================
# LAG SUMMARY BY OFFSET
# =============================================================================

lag_summary_rows = []


for off, sub in df.groupby(
    "offset_num"
):

    for modality, lag_col in [
        (
            "audio",
            "LagA_sec",
        ),
        (
            "visual",
            "LagV_sec",
        ),
    ]:

        vals = (
            sub[
                lag_col
            ]
            .dropna()
            .astype(float)
            .values
        )

        if len(
            vals
        ) == 0:
            continue


        lag_summary_rows.append(
            {
                "offset_deg":
                    off,

                "modality":
                    modality,

                "n_trials":
                    len(
                        vals
                    ),

                "mean_lag_sec":
                    np.mean(
                        vals
                    ),

                "median_lag_sec":
                    np.median(
                        vals
                    ),

                "sd_lag_sec":
                    (
                        np.std(
                            vals,
                            ddof=1,
                        )
                        if len(
                            vals
                        ) >= 2
                        else np.nan
                    ),

                "prop_zero_boundary":
                    np.mean(
                        np.isclose(
                            vals,
                            0.0
                        )
                    ),

                "prop_4s_boundary":
                    np.mean(
                        np.isclose(
                            vals,
                            MAX_LAG_SEC
                        )
                    ),
            }
        )


lag_summary_df = pd.DataFrame(
    lag_summary_rows
)


lag_summary_csv = os.path.join(
    lag_diag_dir,
    "fixed_offset_lag_summary_by_offset.csv",
)


lag_summary_df.to_csv(
    lag_summary_csv,
    index=False,
)


print(
    "\nLag summary by offset:"
)

print(
    lag_summary_df.to_string(
        index=False
    )
)


print(
    f"\nSaved lag diagnostics:\n{lag_diagnostics_csv}"
)

print(
    f"Saved lag summary:\n{lag_summary_csv}"
)


# =============================================================================
# COMPUTE STABLE-WINDOW TRIAL-LEVEL RMSE AND BIAS
# =============================================================================

stable_rows = []


for _, row in df.iterrows():

    subj = row[
        "subject"
    ]

    trial_index = row[
        "trialIndex"
    ]

    offset = int(
        row[
            "offset_num"
        ]
    )

    psame = float(
        row[
            "Psame"
        ]
    )


    t = np.asarray(
        row[
            "t_grid"
        ],
        dtype=float,
    )

    resp = np.asarray(
        row[
            "mouse_x"
        ],
        dtype=float,
    )

    aud = np.asarray(
        row[
            "audio_x"
        ],
        dtype=float,
    )

    vis = np.asarray(
        row[
            "blob_x"
        ],
        dtype=float,
    )


    if not (
        len(t)
        ==
        len(resp)
        ==
        len(aud)
        ==
        len(vis)
    ):

        continue


    stable_mask = (
        (t >= STABLE_START_SEC)
        &
        (t <= STABLE_END_SEC)
    )


    if stable_mask.sum() < 2:
        continue


    # =========================================================================
    # RAW ERRORS
    # =========================================================================

    audio_sqerr_raw = (
        resp
        -
        aud
    ) ** 2


    visual_sqerr_raw = (
        resp
        -
        vis
    ) ** 2


    audio_abs_raw = np.abs(
        resp
        -
        aud
    )


    visual_abs_raw = np.abs(
        resp
        -
        vis
    )


    # =========================================================================
    # LAG-CORRECTED ERRORS
    # =========================================================================

    lag_a = pd.to_numeric(
        row.get(
            "LagA_sec",
            np.nan,
        ),
        errors="coerce",
    )


    lag_v = pd.to_numeric(
        row.get(
            "LagV_sec",
            np.nan,
        ),
        errors="coerce",
    )


    resp_a_lagcorr = shift_response_by_lag(
        response=resp,
        t_grid=t,
        lag_sec=lag_a,
    )


    resp_v_lagcorr = shift_response_by_lag(
        response=resp,
        t_grid=t,
        lag_sec=lag_v,
    )


    audio_sqerr_lagcorr = (
        resp_a_lagcorr
        -
        aud
    ) ** 2


    visual_sqerr_lagcorr = (
        resp_v_lagcorr
        -
        vis
    ) ** 2


    audio_abs_lagcorr = np.abs(
        resp_a_lagcorr
        -
        aud
    )


    visual_abs_lagcorr = np.abs(
        resp_v_lagcorr
        -
        vis
    )


    # =========================================================================
    # STABLE-WINDOW SUMMARIES
    # =========================================================================

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )


        RMSE_A_raw_stable = np.sqrt(
            np.nanmean(
                audio_sqerr_raw[
                    stable_mask
                ]
            )
        )


        RMSE_V_raw_stable = np.sqrt(
            np.nanmean(
                visual_sqerr_raw[
                    stable_mask
                ]
            )
        )


        RMSE_A_lagcorr_stable = np.sqrt(
            np.nanmean(
                audio_sqerr_lagcorr[
                    stable_mask
                ]
            )
        )


        RMSE_V_lagcorr_stable = np.sqrt(
            np.nanmean(
                visual_sqerr_lagcorr[
                    stable_mask
                ]
            )
        )


        Abs_A_raw_stable = np.nanmean(
            audio_abs_raw[
                stable_mask
            ]
        )


        Abs_V_raw_stable = np.nanmean(
            visual_abs_raw[
                stable_mask
            ]
        )


        Abs_A_lagcorr_stable = np.nanmean(
            audio_abs_lagcorr[
                stable_mask
            ]
        )


        Abs_V_lagcorr_stable = np.nanmean(
            visual_abs_lagcorr[
                stable_mask
            ]
        )


    # =========================================================================
    # SAVE TRIAL
    # =========================================================================

    stable_rows.append(
        {
            "subject":
                subj,

            "trialIndex":
                trial_index,

            "trialType":
                row.get(
                    "trialType",
                    np.nan,
                ),

            "condLabel":
                row.get(
                    "condLabel",
                    np.nan,
                ),

            "offsetDeg":
                row.get(
                    "offsetDeg",
                    np.nan,
                ),

            "offset_num":
                offset,

            "offset_cat":
                str(
                    offset
                ),

            "Psame":
                psame,

            "same_response_yes":
                psame,

            "post_question":
                row.get(
                    "post_question",
                    np.nan,
                ),

            "post_question2":
                row.get(
                    "post_question2",
                    np.nan,
                ),


            # Lag values
            "LagA_sec":
                lag_a,

            "LagV_sec":
                lag_v,

            "LagA_source":
                row.get(
                    "LagA_source",
                    np.nan,
                ),

            "LagV_source":
                row.get(
                    "LagV_source",
                    np.nan,
                ),


            # Stable window
            "stable_start_sec":
                STABLE_START_SEC,

            "stable_end_sec":
                STABLE_END_SEC,

            "n_timepoints_stable":
                int(
                    np.sum(
                        stable_mask
                    )
                ),

            "n_valid_audio_lagcorr":
                int(
                    np.sum(
                        np.isfinite(
                            audio_sqerr_lagcorr[
                                stable_mask
                            ]
                        )
                    )
                ),

            "n_valid_visual_lagcorr":
                int(
                    np.sum(
                        np.isfinite(
                            visual_sqerr_lagcorr[
                                stable_mask
                            ]
                        )
                    )
                ),


            # RMSE
            "RMSE_A_raw_stable":
                RMSE_A_raw_stable,

            "RMSE_V_raw_stable":
                RMSE_V_raw_stable,

            "RMSE_A_lagcorr_stable":
                RMSE_A_lagcorr_stable,

            "RMSE_V_lagcorr_stable":
                RMSE_V_lagcorr_stable,


            # Absolute error
            "Abs_A_raw_stable":
                Abs_A_raw_stable,

            "Abs_V_raw_stable":
                Abs_V_raw_stable,

            "Abs_A_lagcorr_stable":
                Abs_A_lagcorr_stable,

            "Abs_V_lagcorr_stable":
                Abs_V_lagcorr_stable,


            # Bias
            "Bias_raw_AminusV_stable":
                (
                    RMSE_A_raw_stable
                    -
                    RMSE_V_raw_stable
                ),

            "Bias_lagcorr_AminusV_stable":
                (
                    RMSE_A_lagcorr_stable
                    -
                    RMSE_V_lagcorr_stable
                ),

            "AbsBias_raw_AminusV_stable":
                (
                    Abs_A_raw_stable
                    -
                    Abs_V_raw_stable
                ),

            "AbsBias_lagcorr_AminusV_stable":
                (
                    Abs_A_lagcorr_stable
                    -
                    Abs_V_lagcorr_stable
                ),
        }
    )


# =============================================================================
# CREATE MODEL DATAFRAME
# =============================================================================

model_df = pd.DataFrame(
    stable_rows
)


if len(
    model_df
) == 0:

    raise RuntimeError(
        "No stable-window model rows were created. Check the fixed-offset "
        "trial arrays in trial_df."
    )


model_df[
    "offset_cat"
] = pd.Categorical(
    model_df[
        "offset_num"
    ]
    .astype(int)
    .astype(str),
    categories=[
        "10",
        "20",
        "30",
        "40",
    ],
    ordered=True,
)


model_df[
    "Psame_centered"
] = (
    model_df[
        "Psame"
    ]
    -
    model_df[
        "Psame"
    ].mean(
        skipna=True
    )
)


# Require valid RAW outcome for the raw analysis.
model_df = model_df.dropna(
    subset=[
        "subject",
        "offset_cat",
        "Psame",
        "Bias_raw_AminusV_stable",
    ]
).copy()


# =============================================================================
# MODEL-DATA DIAGNOSTICS
# =============================================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "STABLE-WINDOW MODEL DATA SUMMARY"
)

print(
    "=" * 90
)


print(
    model_df.groupby(
        "offset_cat",
        observed=False,
    ).agg(

        n_trials=(
            "trialIndex",
            "count",
        ),

        n_subjects=(
            "subject",
            "nunique",
        ),

        Psame_mean=(
            "Psame",
            "mean",
        ),

        Bias_raw_mean=(
            "Bias_raw_AminusV_stable",
            "mean",
        ),

        Bias_lagcorr_mean=(
            "Bias_lagcorr_AminusV_stable",
            "mean",
        ),

        AbsBias_raw_mean=(
            "AbsBias_raw_AminusV_stable",
            "mean",
        ),

        AbsBias_lagcorr_mean=(
            "AbsBias_lagcorr_AminusV_stable",
            "mean",
        ),

        n_valid_lagcorr_bias=(
            "Bias_lagcorr_AminusV_stable",
            "count",
        ),
    )
)


print(
    "\nOverall valid lag-corrected bias rows:"
)

print(
    f"{model_df['Bias_lagcorr_AminusV_stable'].notna().sum()} "
    f"/ {len(model_df)}"
)


# =============================================================================
# SAVE MODEL DATA
# =============================================================================

model_data_csv = os.path.join(
    bias_out_dir,
    "trial_level_bias_psame_model_data.csv",
)


model_df.to_csv(
    model_data_csv,
    index=False,
)


print(
    f"\nSaved stable-window model data:\n"
    f"{model_data_csv}"
)


# =============================================================================
# RAW MODELS — ROBUSTNESS
# =============================================================================

raw_m1, raw_m1_coef = safe_mixedlm(

    formula=(
        "Bias_raw_AminusV_stable ~ "
        "Psame_centered"
    ),

    data=model_df,

    groups="subject",

    label="RAW Model 1 Bias ~ Psame",

    out_dir=bias_out_dir,
)


raw_m2, raw_m2_coef = safe_mixedlm(

    formula=(
        "Bias_raw_AminusV_stable ~ "
        "C(offset_cat) + "
        "Psame_centered"
    ),

    data=model_df,

    groups="subject",

    label="RAW Model 2 Bias ~ Offset + Psame",

    out_dir=bias_out_dir,
)


raw_m3, raw_m3_coef = safe_mixedlm(

    formula=(
        "Bias_raw_AminusV_stable ~ "
        "C(offset_cat) * Psame_centered"
    ),

    data=model_df,

    groups="subject",

    label="RAW Model 3 Bias ~ Offset x Psame",

    out_dir=bias_out_dir,
)


# =============================================================================
# LAG-CORRECTED MODEL DATA
# =============================================================================

lag_model_df = model_df.dropna(
    subset=[
        "Bias_lagcorr_AminusV_stable"
    ]
).copy()


print(
    "\n"
    +
    "=" * 90
)

print(
    "LAG-CORRECTED MODEL DATA CHECK"
)

print(
    "=" * 90
)


print(
    f"Valid lag-corrected trials: "
    f"{len(lag_model_df)} / {len(model_df)}"
)

print(
    f"Subjects represented: "
    f"{lag_model_df['subject'].nunique()}"
)


print(
    "\nValid lag-corrected trials by offset:"
)

print(
    lag_model_df[
        "offset_num"
    ].value_counts().sort_index()
)


if len(
    lag_model_df
) == 0:

    raise RuntimeError(
        "No valid lag-corrected stable-window bias rows remain even after "
        "fallback lag computation. Inspect fixed_offset_trial_lag_diagnostics.csv."
    )


# Re-center Psame for the actual lag-corrected analytic sample.
lag_model_df[
    "Psame_centered"
] = (
    lag_model_df[
        "Psame"
    ]
    -
    lag_model_df[
        "Psame"
    ].mean()
)


# =============================================================================
# LAG-CORRECTED MODELS — PRIMARY
# =============================================================================

lag_m1, lag_m1_coef = safe_mixedlm(

    formula=(
        "Bias_lagcorr_AminusV_stable ~ "
        "Psame_centered"
    ),

    data=lag_model_df,

    groups="subject",

    label="LAGCORR Model 1 Bias ~ Psame",

    out_dir=bias_out_dir,
)


lag_m2, lag_m2_coef = safe_mixedlm(

    formula=(
        "Bias_lagcorr_AminusV_stable ~ "
        "C(offset_cat) + "
        "Psame_centered"
    ),

    data=lag_model_df,

    groups="subject",

    label="LAGCORR Model 2 Bias ~ Offset + Psame",

    out_dir=bias_out_dir,
)


lag_m3, lag_m3_coef = safe_mixedlm(

    formula=(
        "Bias_lagcorr_AminusV_stable ~ "
        "C(offset_cat) * Psame_centered"
    ),

    data=lag_model_df,

    groups="subject",

    label="LAGCORR Model 3 Bias ~ Offset x Psame",

    out_dir=bias_out_dir,
)


# =============================================================================
# RESIDUAL BIAS AFTER ACCOUNTING FOR Psame
# =============================================================================

# -----------------------------------------------------------------------------
# RAW — robustness
# -----------------------------------------------------------------------------

model_df[
    "ResidualBias_after_Psame_raw"
] = raw_m1.resid


resid_subject_offset_raw = (
    model_df
    .groupby(
        [
            "subject",
            "offset_num",
        ],
        as_index=False,
    )
    .agg(

        ResidualBias_after_Psame_raw=(
            "ResidualBias_after_Psame_raw",
            "mean",
        ),

        Bias_raw_AminusV_stable=(
            "Bias_raw_AminusV_stable",
            "mean",
        ),

        Bias_lagcorr_AminusV_stable=(
            "Bias_lagcorr_AminusV_stable",
            "mean",
        ),

        Psame=(
            "Psame",
            "mean",
        ),

        n_trials=(
            "trialIndex",
            "count",
        ),
    )
)


raw_resid_csv = os.path.join(
    bias_out_dir,
    "subject_offset_residual_bias_after_psame_raw.csv",
)


resid_subject_offset_raw.to_csv(
    raw_resid_csv,
    index=False,
)


raw_resid_test_csv = os.path.join(
    bias_out_dir,
    "residual_bias_after_psame_tests_by_offset_raw.csv",
)


raw_resid_test_df = test_residuals_by_offset(

    resid_subject_offset=
        resid_subject_offset_raw,

    resid_col=
        "ResidualBias_after_Psame_raw",

    out_csv=
        raw_resid_test_csv,

    label=(
        "RAW RESIDUAL BIAS TESTS "
        "AFTER ACCOUNTING FOR Psame"
    ),
)


# -----------------------------------------------------------------------------
# LAG CORRECTED — primary
# -----------------------------------------------------------------------------

lag_model_df = lag_model_df.copy()


lag_model_df[
    "ResidualBias_after_Psame_lagcorr"
] = lag_m1.resid


resid_subject_offset_lagcorr = (
    lag_model_df
    .groupby(
        [
            "subject",
            "offset_num",
        ],
        as_index=False,
    )
    .agg(

        ResidualBias_after_Psame_lagcorr=(
            "ResidualBias_after_Psame_lagcorr",
            "mean",
        ),

        Bias_lagcorr_AminusV_stable=(
            "Bias_lagcorr_AminusV_stable",
            "mean",
        ),

        Bias_raw_AminusV_stable=(
            "Bias_raw_AminusV_stable",
            "mean",
        ),

        Psame=(
            "Psame",
            "mean",
        ),

        n_trials=(
            "trialIndex",
            "count",
        ),
    )
)


lagcorr_resid_csv = os.path.join(
    bias_out_dir,
    "subject_offset_residual_bias_after_psame_lagcorr.csv",
)


resid_subject_offset_lagcorr.to_csv(
    lagcorr_resid_csv,
    index=False,
)


lagcorr_resid_test_csv = os.path.join(
    bias_out_dir,
    "residual_bias_after_psame_tests_by_offset_lagcorr.csv",
)


lagcorr_resid_test_df = test_residuals_by_offset(

    resid_subject_offset=
        resid_subject_offset_lagcorr,

    resid_col=
        "ResidualBias_after_Psame_lagcorr",

    out_csv=
        lagcorr_resid_test_csv,

    label=(
        "LAG-CORRECTED RESIDUAL BIAS TESTS "
        "AFTER ACCOUNTING FOR Psame"
    ),
)


# =============================================================================
# GENERIC PRIMARY FILENAMES FOR DOWNSTREAM NOTEBOOK CODE
# =============================================================================

generic_resid_csv = os.path.join(
    bias_out_dir,
    "subject_offset_residual_bias_after_psame.csv",
)


generic_resid_test_csv = os.path.join(
    bias_out_dir,
    "residual_bias_after_psame_tests_by_offset.csv",
)


resid_subject_offset_lagcorr.to_csv(
    generic_resid_csv,
    index=False,
)


lagcorr_resid_test_df.to_csv(
    generic_resid_test_csv,
    index=False,
)


print(
    "\nPrimary residual-bias files for the manuscript "
    "use LAG-CORRECTED stable-window bias:"
)

print(
    f"  {generic_resid_csv}"
)

print(
    f"  {generic_resid_test_csv}"
)


# =============================================================================
# SIMPLE DESCRIPTIVE SUMMARIES
# =============================================================================

summary = (
    model_df
    .groupby(
        [
            "subject",
            "offset_num",
        ],
        as_index=False,
    )
    .agg(

        Psame=(
            "Psame",
            "mean",
        ),

        Bias_raw_AminusV_stable=(
            "Bias_raw_AminusV_stable",
            "mean",
        ),

        Bias_lagcorr_AminusV_stable=(
            "Bias_lagcorr_AminusV_stable",
            "mean",
        ),

        AbsBias_raw_AminusV_stable=(
            "AbsBias_raw_AminusV_stable",
            "mean",
        ),

        AbsBias_lagcorr_AminusV_stable=(
            "AbsBias_lagcorr_AminusV_stable",
            "mean",
        ),

        n_trials=(
            "trialIndex",
            "count",
        ),
    )
)


group_summary = (
    summary
    .groupby(
        "offset_num",
        as_index=False,
    )
    .agg(

        Psame_mean=(
            "Psame",
            "mean",
        ),

        Psame_sem=(
            "Psame",
            sem,
        ),


        Bias_raw_mean=(
            "Bias_raw_AminusV_stable",
            "mean",
        ),

        Bias_raw_sem=(
            "Bias_raw_AminusV_stable",
            sem,
        ),


        Bias_lagcorr_mean=(
            "Bias_lagcorr_AminusV_stable",
            "mean",
        ),

        Bias_lagcorr_sem=(
            "Bias_lagcorr_AminusV_stable",
            sem,
        ),


        AbsBias_raw_mean=(
            "AbsBias_raw_AminusV_stable",
            "mean",
        ),

        AbsBias_raw_sem=(
            "AbsBias_raw_AminusV_stable",
            sem,
        ),


        AbsBias_lagcorr_mean=(
            "AbsBias_lagcorr_AminusV_stable",
            "mean",
        ),

        AbsBias_lagcorr_sem=(
            "AbsBias_lagcorr_AminusV_stable",
            sem,
        ),


        n_subjects=(
            "subject",
            "nunique",
        ),
    )
)


summary_subject_csv = os.path.join(
    bias_out_dir,
    "subject_offset_psame_bias_summary.csv",
)


summary_group_csv = os.path.join(
    bias_out_dir,
    "subject_and_group_offset_psame_bias_summary.csv",
)


summary.to_csv(
    summary_subject_csv,
    index=False,
)


group_summary.to_csv(
    summary_group_csv,
    index=False,
)


print(
    "\nSaved subject and group summaries:"
)

print(
    f"  {summary_subject_csv}"
)

print(
    f"  {summary_group_csv}"
)


# =============================================================================
# DESCRIPTIVE FIGURE — PRIMARY LAG-CORRECTED METRIC
# =============================================================================

x = group_summary[
    "offset_num"
].values.astype(float)


fig, ax1 = plt.subplots(
    figsize=(
        7,
        5,
    )
)


ax1.errorbar(
    x,
    group_summary[
        "Bias_lagcorr_mean"
    ],
    yerr=group_summary[
        "Bias_lagcorr_sem"
    ],
    marker="o",
    linewidth=2,
    label="Bias A−V lag-corrected",
)


ax1.axhline(
    0,
    linestyle="--",
    linewidth=1,
)


ax1.set_xlabel(
    "Offset (degrees)"
)


ax1.set_ylabel(
    "Bias A−V: RMSE_A - RMSE_V"
)


ax1.set_title(
    "Stable-window visual bias and explicit P(common)\n"
    "0.5–17.5 s"
)


ax2 = ax1.twinx()


ax2.errorbar(
    x,
    group_summary[
        "Psame_mean"
    ],
    yerr=group_summary[
        "Psame_sem"
    ],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="P(same)",
)


ax2.set_ylabel(
    "P(same/common source)"
)


ax2.set_ylim(
    -0.05,
    1.05,
)


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()


ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best",
)


fig.tight_layout()


fig_path = os.path.join(
    fig_dir,
    "offset_bias_and_psame_summary.png",
)


fig.savefig(
    fig_path,
    dpi=FIG_DPI,
    bbox_inches="tight",
)


plt.close(
    fig
)


# =============================================================================
# DESCRIPTIVE FIGURE — RAW ROBUSTNESS
# =============================================================================

fig, ax1 = plt.subplots(
    figsize=(
        7,
        5,
    )
)


ax1.errorbar(
    x,
    group_summary[
        "Bias_raw_mean"
    ],
    yerr=group_summary[
        "Bias_raw_sem"
    ],
    marker="o",
    linewidth=2,
    label="Bias A−V raw",
)


ax1.axhline(
    0,
    linestyle="--",
    linewidth=1,
)


ax1.set_xlabel(
    "Offset (degrees)"
)


ax1.set_ylabel(
    "Bias A−V: RMSE_A - RMSE_V"
)


ax1.set_title(
    "Stable-window raw visual bias and explicit P(common)\n"
    "0.5–17.5 s"
)


ax2 = ax1.twinx()


ax2.errorbar(
    x,
    group_summary[
        "Psame_mean"
    ],
    yerr=group_summary[
        "Psame_sem"
    ],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="P(same)",
)


ax2.set_ylabel(
    "P(same/common source)"
)


ax2.set_ylim(
    -0.05,
    1.05,
)


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()


ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best",
)


fig.tight_layout()


fig_path_raw = os.path.join(
    fig_dir,
    "offset_bias_and_psame_summary_raw.png",
)


fig.savefig(
    fig_path_raw,
    dpi=FIG_DPI,
    bbox_inches="tight",
)


plt.close(
    fig
)


print(
    "\nSaved figures:"
)

print(
    f"  {fig_path}"
)

print(
    f"  {fig_path_raw}"
)


# =============================================================================
# MODEL SUMMARY INDEX
# =============================================================================

summary_index_rows = [

    {
        "analysis":
            "stable_0p5_17p5_raw_model_1_bias_psame",

        "primary":
            False,

        "outcome":
            "Bias_raw_AminusV_stable",

        "formula":
            "Bias_raw_AminusV_stable ~ Psame_centered",
    },

    {
        "analysis":
            "stable_0p5_17p5_raw_model_2_bias_offset_psame",

        "primary":
            False,

        "outcome":
            "Bias_raw_AminusV_stable",

        "formula":
            "Bias_raw_AminusV_stable ~ C(offset_cat) + Psame_centered",
    },

    {
        "analysis":
            "stable_0p5_17p5_raw_model_3_bias_offset_x_psame",

        "primary":
            False,

        "outcome":
            "Bias_raw_AminusV_stable",

        "formula":
            "Bias_raw_AminusV_stable ~ C(offset_cat) * Psame_centered",
    },

    {
        "analysis":
            "stable_0p5_17p5_lagcorr_model_1_bias_psame",

        "primary":
            True,

        "outcome":
            "Bias_lagcorr_AminusV_stable",

        "formula":
            "Bias_lagcorr_AminusV_stable ~ Psame_centered",
    },

    {
        "analysis":
            "stable_0p5_17p5_lagcorr_model_2_bias_offset_psame",

        "primary":
            True,

        "outcome":
            "Bias_lagcorr_AminusV_stable",

        "formula":
            "Bias_lagcorr_AminusV_stable ~ C(offset_cat) + Psame_centered",
    },

    {
        "analysis":
            "stable_0p5_17p5_lagcorr_model_3_bias_offset_x_psame",

        "primary":
            True,

        "outcome":
            "Bias_lagcorr_AminusV_stable",

        "formula":
            "Bias_lagcorr_AminusV_stable ~ C(offset_cat) * Psame_centered",
    },
]


summary_index = pd.DataFrame(
    summary_index_rows
)


summary_index_csv = os.path.join(
    bias_out_dir,
    "stable_0p5_17p5_model_summary_index.csv",
)


summary_index.to_csv(
    summary_index_csv,
    index=False,
)


# =============================================================================
# FINAL OUTPUT SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "STABLE-WINDOW P(SAME) / VISUAL-BIAS ANALYSIS COMPLETE"
)

print(
    "=" * 90
)


print(
    "\nKey lag diagnostics:"
)

print(
    f"  Trial lag diagnostics: {lag_diagnostics_csv}"
)

print(
    f"  Lag summary:           {lag_summary_csv}"
)


print(
    "\nKey manuscript outputs:"
)

print(
    f"  Stable model data: "
    f"{model_data_csv}"
)

print(
    f"  Lag-corrected residual tests: "
    f"{generic_resid_test_csv}"
)

print(
    f"  Lag-corrected residual subject × offset means: "
    f"{generic_resid_csv}"
)

print(
    f"  Group P(same)/bias summary: "
    f"{summary_group_csv}"
)

print(
    f"  Primary figure: "
    f"{fig_path}"
)


print(
    "\nInterpretation:"
)

print(
    "- Positive Bias A−V means the response was closer to vision."
)

print(
    "- Lag correction uses trial-specific auditory and visual response delays."
)

print(
    "- Existing notebook lags are retained whenever available."
)

print(
    "- Missing lags are estimated with non-negative 0–4 s xcorr."
)

print(
    "- No peak-r exclusion is applied because lag is a nuisance correction "
    "for spatial weighting rather than the dependent variable."
)


FIXED-OFFSET TRIALS BEFORE LAG COMPLETION
Trials: 4000
Subjects: 50

Initial lag availability in trial_df:
LagA_sec    3952
LagV_sec    3952
dtype: int64

Found LagA_sec / LagV_sec in xcorr_df.

Invalidating 1205 legacy LagA_sec estimates outside 0.0–4.0 s.

Invalidating 1205 legacy LagV_sec estimates outside 0.0–4.0 s.

Lag availability after merging existing notebook outputs:
          LagA_sec     LagV_sec
count  2747.000000  2747.000000
mean      1.605442     1.605442
std       1.340856     1.340856
min       0.000000     0.000000
25%       0.450000     0.450000
50%       1.200000     1.200000
75%       2.600000     2.600000
max       4.000000     4.000000

Missing lag counts before fallback computation:
LagA_sec    1253
LagV_sec    1253
dtype: int64

Computing remaining missing lags directly from trial trajectories...
Audio lags to compute:  1253
Visual lags to compute: 1253

Final lag availability:
          LagA_sec     LagV_sec
count  3952.000000  3952.000000
mean      1.40370

In [ ]:
# %% FIXED-OFFSET RESIDUAL BIAS — WHOLE-TRIAL CONDITION-LABEL SHUFFLE
# ============================================================
# Purpose:
#   Test whether fixed-offset residual visual bias remains systematically
#   structured by offset condition after accounting for explicit P(same).
#
# Null:
#   Offset labels are shuffled across complete trials within subject.
#
# This preserves:
#   - subject-level dependence
#   - complete trial residual values
#   - number of trials per offset within subject
#
# This breaks:
#   - the mapping between residual bias and fixed-offset condition.
#
# Primary test:
#   Empirical mean residual bias at each offset is compared against the
#   condition-label shuffle null for that same offset.
#
# Recommended alternative:
#   "two-sided" because residual bias can be meaningfully negative or positive:
#       10° residuals may be below the condition-shuffle null
#       30° residuals may be above the condition-shuffle null
#
# Change CONDITION_LABEL_SHUFFLE_ALTERNATIVE to "greater" only for a strictly
# positive residual-visual-bias hypothesis.
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t as t_dist

# ============================================================
# SETTINGS
# ============================================================

OFFSETS = [10, 20, 30, 40]

N_PERM_FIXED_RESIDUAL_LABEL_SHUFFLE = 1000
RANDOM_SEED_FIXED_RESIDUAL_LABEL_SHUFFLE = 12345

# Use "two-sided" for fixed-offset residual structure.
# This lets negative 10° residuals and positive 30° residuals both count
# as meaningful departures from the condition-label shuffle null.
CONDITION_LABEL_SHUFFLE_ALTERNATIVE = "two-sided"

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

try:
    fixed_resid_out_dir = bias_perm_out_dir
except NameError:
    try:
        fixed_resid_out_dir = bias_out_dir
    except NameError:
        fixed_resid_out_dir = os.getcwd()

os.makedirs(fixed_resid_out_dir, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def _clean_numeric(values):
    return (
        pd.Series(values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
        .values
    )


def _mean_ci_test(values, popmean=0.0, alpha=0.05):
    vals = _clean_numeric(values)
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "se": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p_two_sided": np.nan,
            "p_greater": np.nan,
            "p_less": np.nan,
        }

    m = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1))
    se = float(sd / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))

    t_stat, p_two = stats.ttest_1samp(vals, popmean=popmean, nan_policy="omit")

    return {
        "n": int(n),
        "mean": m,
        "se": se,
        "ci95_low": float(m - crit * se),
        "ci95_high": float(m + crit * se),
        "t": float(t_stat),
        "df": int(n - 1),
        "p_two_sided": float(p_two),
        "p_greater": float(stats.t.sf(t_stat, df=n - 1)),
        "p_less": float(stats.t.cdf(t_stat, df=n - 1)),
    }


def _fdr_bh(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return qvals

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    q = np.empty_like(q_ranked)
    q[order] = q_ranked
    qvals[finite] = q

    return qvals


def _holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    adj = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return adj

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    ranked_adj = np.empty(m)

    for i, pval in enumerate(ranked):
        ranked_adj[i] = (m - i) * pval

    ranked_adj = np.maximum.accumulate(ranked_adj)
    ranked_adj = np.clip(ranked_adj, 0, 1)

    out = np.empty_like(ranked_adj)
    out[order] = ranked_adj
    adj[finite] = out

    return adj


def _empirical_pvalue(observed, null_values, alternative="two-sided"):
    null_values = _clean_numeric(null_values)

    if len(null_values) == 0 or not np.isfinite(observed):
        return np.nan

    if alternative == "greater":
        return float((np.sum(null_values >= observed) + 1) / (len(null_values) + 1))

    if alternative == "less":
        return float((np.sum(null_values <= observed) + 1) / (len(null_values) + 1))

    if alternative == "two-sided":
        # Two-sided relative to the center of the permutation null.
        null_center = float(np.nanmean(null_values))
        observed_distance = abs(observed - null_center)
        null_distance = np.abs(null_values - null_center)

        return float((np.sum(null_distance >= observed_distance) + 1) / (len(null_values) + 1))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")


def _format_p_for_text(p):
    if not np.isfinite(p):
        return "NA"

    if p < .001:
        return "< .001"

    return f"= {p:.4f}".replace("0.", ".")


def _choose_primary_parametric_p(row, alternative):
    if alternative == "greater":
        return row["p_greater"]
    if alternative == "less":
        return row["p_less"]
    return row["p_two_sided"]


# ============================================================
# BUILD TRIAL-LEVEL FIXED-OFFSET RESIDUAL INPUT
# ============================================================
# Preferred input:
#   stable_residual_trial_summary
#
# Required columns:
#   subject
#   trialIndex
#   offset_num
#   ResidualAbsBias_after_Psame_lagcorr_stable
# ============================================================

if "stable_residual_trial_summary" in globals():

    required_cols = [
        "subject",
        "trialIndex",
        "offset_num",
        "ResidualAbsBias_after_Psame_lagcorr_stable",
    ]

    missing_cols = [
        c for c in required_cols
        if c not in stable_residual_trial_summary.columns
    ]

    if missing_cols:
        raise RuntimeError(
            "stable_residual_trial_summary exists but is missing required columns: "
            f"{missing_cols}"
        )

    fixed_resid_trial_df = (
        stable_residual_trial_summary
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "subject",
                "trialIndex",
                "offset_num",
                "ResidualAbsBias_after_Psame_lagcorr_stable",
            ]
        )
        .query("offset_num in @OFFSETS")
        .copy()
    )

    fixed_resid_trial_df = fixed_resid_trial_df[
        [
            "subject",
            "trialIndex",
            "offset_num",
            "ResidualAbsBias_after_Psame_lagcorr_stable",
        ]
    ].copy()

elif "model_df" in globals() and "ResidualAbsBias_after_Psame_lagcorr_stable" in model_df.columns:

    fixed_resid_trial_df = (
        model_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "subject",
                "trialIndex",
                "offset_num",
                "ResidualAbsBias_after_Psame_lagcorr_stable",
            ]
        )
        .query("offset_num in @OFFSETS")
        .copy()
    )

    fixed_resid_trial_df = fixed_resid_trial_df[
        [
            "subject",
            "trialIndex",
            "offset_num",
            "ResidualAbsBias_after_Psame_lagcorr_stable",
        ]
    ].copy()

elif "model_df" in globals() and "ResidualBias_after_Psame_stable" in model_df.columns:

    fixed_resid_trial_df = (
        model_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "subject",
                "trialIndex",
                "offset_num",
                "ResidualBias_after_Psame_stable",
            ]
        )
        .query("offset_num in @OFFSETS")
        .copy()
    )

    fixed_resid_trial_df = fixed_resid_trial_df[
        [
            "subject",
            "trialIndex",
            "offset_num",
            "ResidualBias_after_Psame_stable",
        ]
    ].copy()

    fixed_resid_trial_df = fixed_resid_trial_df.rename(
        columns={
            "ResidualBias_after_Psame_stable": "ResidualAbsBias_after_Psame_lagcorr_stable"
        }
    )

else:
    raise RuntimeError(
        "Could not find trial-level residual-bias values. Run the residual-bias "
        "analysis first so that stable_residual_trial_summary exists with column "
        "'ResidualAbsBias_after_Psame_lagcorr_stable'."
    )

fixed_resid_trial_df["offset_num"] = fixed_resid_trial_df["offset_num"].astype(int)

fixed_resid_trial_csv = os.path.join(
    fixed_resid_out_dir,
    "fixed_offset_residual_bias_trial_level_input_for_condition_label_shuffle.csv"
)
fixed_resid_trial_df.to_csv(fixed_resid_trial_csv, index=False)

print("\nFixed-offset residual-bias trial-level input:")
print(f"Rows: {len(fixed_resid_trial_df)}")
print(f"Subjects: {fixed_resid_trial_df['subject'].nunique()}")
print("Offset counts:")
print(fixed_resid_trial_df.groupby("offset_num").size())
print(f"Saved: {fixed_resid_trial_csv}")

# ============================================================
# OBSERVED SUBJECT × OFFSET SUMMARY
# ============================================================

observed_subject_offset_df = (
    fixed_resid_trial_df
    .groupby(["subject", "offset_num"], as_index=False)
    .agg(
        ResidualBias_after_Psame_stable=(
            "ResidualAbsBias_after_Psame_lagcorr_stable",
            "mean"
        ),
        n_trials=("trialIndex", "nunique"),
    )
)

observed_subject_offset_csv = os.path.join(
    fixed_resid_out_dir,
    "fixed_offset_residual_bias_subject_offset_observed.csv"
)
observed_subject_offset_df.to_csv(observed_subject_offset_csv, index=False)

# ============================================================
# OBSERVED TESTS AGAINST ZERO
# ============================================================
# These are descriptive/parametric tests of residual bias against zero.
# The permutation p values below come from the condition-label shuffle null.
# ============================================================

observed_rows = []

for offset in OFFSETS:
    sub = observed_subject_offset_df[
        observed_subject_offset_df["offset_num"] == offset
    ].copy()

    param = _mean_ci_test(
        sub["ResidualBias_after_Psame_stable"],
        popmean=0.0
    )

    primary_parametric_p = _choose_primary_parametric_p(
        param,
        CONDITION_LABEL_SHUFFLE_ALTERNATIVE
    )

    observed_rows.append({
        "analysis": "fixed_offset_residual_bias_condition_label_shuffle",
        "offset_num": int(offset),
        "offset_label": f"{int(offset)}° offset",
        "mean_residual_bias": param["mean"],
        "se": param["se"],
        "ci95_low": param["ci95_low"],
        "ci95_high": param["ci95_high"],
        "t": param["t"],
        "df": param["df"],
        "n_subjects": param["n"],
        "parametric_p_two_sided": param["p_two_sided"],
        "parametric_p_greater": param["p_greater"],
        "parametric_p_less": param["p_less"],
        "parametric_p_primary": primary_parametric_p,
    })

observed_tests_df = pd.DataFrame(observed_rows)

# ============================================================
# WHOLE-TRIAL CONDITION-LABEL SHUFFLE
# ============================================================

def _shuffle_offset_labels_within_subject(trial_df, rng):
    """
    Shuffle fixed-offset condition labels across complete trials within subject.

    This preserves:
      - subject
      - complete trial residual values
      - number of offset labels within each subject

    This breaks:
      - the mapping between residual value and physical offset condition.
    """
    out = trial_df.copy()
    out["permuted_offset_num"] = np.nan

    for subj, idx in out.groupby("subject").groups.items():
        idx = list(idx)
        offsets = out.loc[idx, "offset_num"].astype(int).values.copy()

        if len(offsets) > 1:
            out.loc[idx, "permuted_offset_num"] = rng.permutation(offsets)
        else:
            out.loc[idx, "permuted_offset_num"] = offsets

    out["permuted_offset_num"] = pd.to_numeric(
        out["permuted_offset_num"],
        errors="coerce"
    ).astype("Int64")

    return out


def _compute_permuted_offset_group_means(permuted_trial_df):
    """
    Compute group-level mean residual bias for each permuted offset.

    First averages within subject × permuted offset, then averages across subjects.
    """
    subject_offset_perm = (
        permuted_trial_df
        .dropna(subset=["permuted_offset_num"])
        .groupby(["subject", "permuted_offset_num"], as_index=False)
        .agg(
            ResidualBias_after_Psame_stable=(
                "ResidualAbsBias_after_Psame_lagcorr_stable",
                "mean"
            ),
            n_trials=("trialIndex", "nunique"),
        )
        .rename(columns={"permuted_offset_num": "offset_num"})
    )

    subject_offset_perm["offset_num"] = subject_offset_perm["offset_num"].astype(int)

    rows = []

    for offset in OFFSETS:
        vals = subject_offset_perm.loc[
            subject_offset_perm["offset_num"] == offset,
            "ResidualBias_after_Psame_stable"
        ]

        param = _mean_ci_test(vals, popmean=0.0)

        rows.append({
            "offset_num": int(offset),
            "mean_residual_bias": param["mean"],
            "t": param["t"],
            "n_subjects": param["n"],
        })

    return pd.DataFrame(rows)


rng = np.random.default_rng(RANDOM_SEED_FIXED_RESIDUAL_LABEL_SHUFFLE)

null_rows = []

for perm_i in range(N_PERM_FIXED_RESIDUAL_LABEL_SHUFFLE):

    permuted_trial_df = _shuffle_offset_labels_within_subject(
        fixed_resid_trial_df,
        rng=rng
    )

    perm_summary = _compute_permuted_offset_group_means(permuted_trial_df)

    for _, row in perm_summary.iterrows():
        null_rows.append({
            "perm_i": perm_i,
            "offset_num": int(row["offset_num"]),
            "offset_label": f"{int(row['offset_num'])}° offset",
            "null_mean_residual_bias": row["mean_residual_bias"],
            "null_t": row["t"],
            "null_n_subjects": row["n_subjects"],
        })

    if (perm_i + 1) % 1000 == 0:
        print(
            f"Completed fixed-offset residual condition-label shuffle "
            f"{perm_i + 1}/{N_PERM_FIXED_RESIDUAL_LABEL_SHUFFLE}"
        )

condition_label_shuffle_null_df = pd.DataFrame(null_rows)

condition_label_shuffle_null_csv = os.path.join(
    fixed_resid_out_dir,
    "fixed_offset_residual_bias_condition_label_shuffle_null_long.csv"
)
condition_label_shuffle_null_df.to_csv(condition_label_shuffle_null_csv, index=False)

# ============================================================
# PERMUTATION P VALUES
# ============================================================

test_rows = []

for _, obs in observed_tests_df.iterrows():
    offset = int(obs["offset_num"])

    null_sub = condition_label_shuffle_null_df[
        condition_label_shuffle_null_df["offset_num"] == offset
    ].copy()

    null_means = _clean_numeric(null_sub["null_mean_residual_bias"])
    null_t = _clean_numeric(null_sub["null_t"])

    observed_effect = obs["mean_residual_bias"]
    observed_t = obs["t"]

    p_effect = _empirical_pvalue(
        observed_effect,
        null_means,
        alternative=CONDITION_LABEL_SHUFFLE_ALTERNATIVE
    )

    p_t = _empirical_pvalue(
        observed_t,
        null_t,
        alternative=CONDITION_LABEL_SHUFFLE_ALTERNATIVE
    )

    if len(null_means) > 0:
        null_mean = float(np.nanmean(null_means))
        null_sd = float(np.nanstd(null_means, ddof=1))
        null_ci95_low = float(np.nanpercentile(null_means, 2.5))
        null_ci95_high = float(np.nanpercentile(null_means, 97.5))
    else:
        null_mean = np.nan
        null_sd = np.nan
        null_ci95_low = np.nan
        null_ci95_high = np.nan

    test_rows.append({
        "analysis": obs["analysis"],
        "offset_num": offset,
        "offset_label": obs["offset_label"],
        "alternative": CONDITION_LABEL_SHUFFLE_ALTERNATIVE,

        "observed_effect": observed_effect,
        "observed_ci95_low": obs["ci95_low"],
        "observed_ci95_high": obs["ci95_high"],
        "observed_t": obs["t"],
        "observed_df": obs["df"],
        "observed_n_subjects": obs["n_subjects"],

        "parametric_p_two_sided": obs["parametric_p_two_sided"],
        "parametric_p_greater": obs["parametric_p_greater"],
        "parametric_p_less": obs["parametric_p_less"],
        "parametric_p_primary": obs["parametric_p_primary"],

        "null_effect_mean": null_mean,
        "null_effect_sd": null_sd,
        "null_effect_ci95_low": null_ci95_low,
        "null_effect_ci95_high": null_ci95_high,

        "permutation_p_effect": p_effect,
        "permutation_p_t": p_t,
        "n_permutations": N_PERM_FIXED_RESIDUAL_LABEL_SHUFFLE,
    })

fixed_offset_residual_label_shuffle_tests_df = pd.DataFrame(test_rows)

fixed_offset_residual_label_shuffle_tests_df["permutation_p_effect_fdr_bh"] = _fdr_bh(
    fixed_offset_residual_label_shuffle_tests_df["permutation_p_effect"].values
)

fixed_offset_residual_label_shuffle_tests_df["permutation_p_t_fdr_bh"] = _fdr_bh(
    fixed_offset_residual_label_shuffle_tests_df["permutation_p_t"].values
)

fixed_offset_residual_label_shuffle_tests_df["permutation_p_effect_holm"] = _holm_bonferroni(
    fixed_offset_residual_label_shuffle_tests_df["permutation_p_effect"].values
)

fixed_offset_residual_label_shuffle_tests_df["permutation_p_t_holm"] = _holm_bonferroni(
    fixed_offset_residual_label_shuffle_tests_df["permutation_p_t"].values
)

fixed_offset_residual_label_shuffle_tests_df["parametric_p_primary_fdr_bh"] = _fdr_bh(
    fixed_offset_residual_label_shuffle_tests_df["parametric_p_primary"].values
)

fixed_offset_residual_label_shuffle_tests_csv = os.path.join(
    fixed_resid_out_dir,
    "fixed_offset_residual_bias_condition_label_shuffle_tests.csv"
)
fixed_offset_residual_label_shuffle_tests_df.to_csv(
    fixed_offset_residual_label_shuffle_tests_csv,
    index=False
)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\nFixed-offset residual-bias condition-label shuffle tests:")
try:
    display(fixed_offset_residual_label_shuffle_tests_df)
except NameError:
    print(fixed_offset_residual_label_shuffle_tests_df.to_string(index=False))

print(f"\nSaved observed subject-offset input: {observed_subject_offset_csv}")
print(f"Saved condition-label shuffle null:  {condition_label_shuffle_null_csv}")
print(f"Saved condition-label shuffle tests: {fixed_offset_residual_label_shuffle_tests_csv}")

print("\nManuscript-ready fixed-offset residual-bias results:")
for _, row in fixed_offset_residual_label_shuffle_tests_df.iterrows():

    p_param = _format_p_for_text(row["parametric_p_primary"])
    p_perm = _format_p_for_text(row["permutation_p_effect"])
    p_fdr = _format_p_for_text(row["permutation_p_effect_fdr_bh"])

    print(
        f"At {int(row['offset_num'])}° offsets, residual bias was "
        f"{row['observed_effect']:.2f}°, "
        f"95% CI [{row['observed_ci95_low']:.2f}, {row['observed_ci95_high']:.2f}], "
        f"t({int(row['observed_df'])}) = {row['observed_t']:.2f}, "
        f"p {p_param}, "
        f"permutation p {p_perm}, "
        f"FDR-corrected p {p_fdr}."
    )

print("\nInterpretation note:")
print(
    "These permutation p values test whether each fixed-offset residual-bias "
    "estimate is unusually high or low relative to a whole-trial condition-label "
    "shuffle null. They do not test residual bias against zero directly; the "
    "parametric t tests provide the residual-vs-zero descriptive inference."
)

## 11. Continuous Residual Recalibration and Breakpoint Dynamics

Time-resolved residual visual bias, auditory-anchor/breakpoint classification, dynamic capture metrics, and breakpoint permutation inference.

In [26]:
# %% CONTINUOUS RECALIBRATION ANALYSIS: RESIDUAL VISUAL BIAS OVER TIME
# ============================================================
# Tests whether visual bias remains after accounting for explicit causal judgment.
#
# Core metric:
#   Bias_AminusV(t) = |response(t) - audio(t)| - |response(t) - visual(t)|
#
# Interpretation:
#   Bias > 0 = response closer to visual trajectory
#   Bias < 0 = response closer to auditory trajectory
#
# Analysis:
#   1) Compute time-resolved bias for each fixed-offset trial.
#   2) Residualize bias after accounting for Psame.
#   3) Test whether residual visual bias changes across early/middle/late trial periods.
#   4) Plot residual bias over time by offset.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import ttest_1samp, t as t_dist

# -------------------------
# SETTINGS
# -------------------------
FIG_DPI = 180

try:
    OUT_DIR = os.path.join(out_root, "continuous_recalibration_residual_bias")
except NameError:
    OUT_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\continuous_recalibration_residual_bias"

CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# -------------------------
# REQUIRED DATA
# -------------------------
if "trial_df" not in globals():
    raise RuntimeError("trial_df not found. Run your load/resample cell first.")

if "trial_metrics_df" not in globals():
    print("trial_metrics_df not found. Lag-corrected bias will be skipped unless lag columns are in trial_df.")
    trial_metrics_df = None

# -------------------------
# HELPERS
# -------------------------
def parse_yes_no_same_response(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["y", "yes", "1", "true", "same", "same source", "common", "common source"]:
        return 1.0
    if s in ["n", "no", "0", "false", "different", "diff", "different source", "separate", "separate source"]:
        return 0.0
    return np.nan

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def shift_response_by_lag(response, t_grid, lag_sec):
    response = np.asarray(response, dtype=float)
    t_grid = np.asarray(t_grid, dtype=float)

    if not np.isfinite(lag_sec):
        return np.full_like(response, np.nan, dtype=float)

    return np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan
    )

def sem(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return x.std(ddof=1) / np.sqrt(len(x))

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).dropna().astype(float).values
    n = len(vals)
    if n < 2:
        return np.nan, np.nan, np.nan, n
    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)
    return m, m - crit * se, m + crit * se, n

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def save_fig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

def safe_mixedlm(formula, data, groups="subject", label="model"):
    print("\n" + "=" * 90)
    print(label)
    print("=" * 90)
    print(f"Formula: {formula}")
    print(f"Rows: {len(data)}")
    print(f"Subjects: {data[groups].nunique()}")

    model = smf.mixedlm(
        formula,
        data=data,
        groups=data[groups],
        re_formula="1"
    )

    try:
        result = model.fit(method="powell", reml=False, maxiter=1000, disp=False)
    except Exception as e:
        print(f"Powell failed: {e}")
        print("Trying lbfgs...")
        result = model.fit(method="lbfgs", reml=False, maxiter=1000)

    print(result.summary())
    return result

# -------------------------
# PREPARE TRIAL DATA
# -------------------------
df = trial_df.copy()

# Add same/different judgment
if "same_response_yes" not in df.columns:
    if "post_question" not in df.columns:
        raise RuntimeError("Neither same_response_yes nor post_question found in trial_df.")
    df["same_response_yes"] = df["post_question"].apply(parse_yes_no_same_response)

df["Psame"] = pd.to_numeric(df["same_response_yes"], errors="coerce")
df["judgment_label"] = np.where(
    df["Psame"] == 1, "Same source",
    np.where(df["Psame"] == 0, "Different source", np.nan)
)

# Merge lag columns if available
if trial_metrics_df is not None:
    lag_cols = ["subject", "trialIndex"]
    for c in ["LagA_sec", "LagV_sec"]:
        if c in trial_metrics_df.columns:
            lag_cols.append(c)

    if "LagA_sec" in lag_cols and "LagV_sec" in lag_cols:
        lag_df = trial_metrics_df[lag_cols].drop_duplicates(
            subset=["subject", "trialIndex"],
            keep="first"
        )
        for c in ["LagA_sec", "LagV_sec"]:
            if c in df.columns:
                df = df.drop(columns=[c])
        df = df.merge(lag_df, on=["subject", "trialIndex"], how="left")

has_lag = ("LagA_sec" in df.columns) and ("LagV_sec" in df.columns)

# Fixed-offset trials only
df = df[df["trialType"].astype(str).str.lower() == "incongruent_spatial"].copy()
df["offset_num"] = df.apply(extract_offset_from_row, axis=1)
df = df[df["offset_num"].isin([10, 20, 30, 40])].copy()
df = df.dropna(subset=["Psame"]).copy()

df["offset_cat"] = pd.Categorical(
    df["offset_num"].astype(int).astype(str),
    categories=["10", "20", "30", "40"],
    ordered=True
)

df["Psame_centered"] = df["Psame"] - df["Psame"].mean(skipna=True)

print("\nFixed-offset trials included:")
print(df.groupby(["offset_num", "judgment_label"]).size())

# -------------------------
# COMPUTE TRIAL × TIMEPOINT BIAS
# -------------------------
trial_time_rows = []

for _, row in df.iterrows():
    subj = row["subject"]
    trial_index = row["trialIndex"]
    cond = row.get("condLabel", "")
    offset = int(row["offset_num"])
    psame = float(row["Psame"])
    psame_centered = float(row["Psame_centered"])
    judgment = row["judgment_label"]

    t = np.asarray(row["t_grid"], dtype=float)
    resp = np.asarray(row["mouse_x"], dtype=float)
    aud = np.asarray(row["audio_x"], dtype=float)
    vis = np.asarray(row["blob_x"], dtype=float)

    # Raw bias
    audio_abs_raw = np.abs(resp - aud)
    visual_abs_raw = np.abs(resp - vis)
    bias_raw_t = audio_abs_raw - visual_abs_raw

    # Lag-corrected bias, if lags are available
    if has_lag:
        resp_a_lag = shift_response_by_lag(resp, t, row.get("LagA_sec", np.nan))
        resp_v_lag = shift_response_by_lag(resp, t, row.get("LagV_sec", np.nan))

        audio_abs_lagcorr = np.abs(resp_a_lag - aud)
        visual_abs_lagcorr = np.abs(resp_v_lag - vis)
        bias_lagcorr_t = audio_abs_lagcorr - visual_abs_lagcorr
    else:
        bias_lagcorr_t = np.full_like(bias_raw_t, np.nan, dtype=float)

    # Normalize time for binning
    valid_t = np.isfinite(t)
    t_min = np.nanmin(t[valid_t])
    t_max = np.nanmax(t[valid_t])
    t_norm = (t - t_min) / (t_max - t_min)

    time_bin = np.where(
        t_norm < 1/3, "early",
        np.where(t_norm < 2/3, "middle", "late")
    )

    for i, time_sec in enumerate(t):
        trial_time_rows.append({
            "subject": subj,
            "trialIndex": trial_index,
            "condLabel": cond,
            "offset_num": offset,
            "offset_cat": str(offset),
            "Psame": psame,
            "Psame_centered": psame_centered,
            "judgment_label": judgment,
            "time_sec": time_sec,
            "time_norm": t_norm[i],
            "time_bin": time_bin[i],
            "Bias_raw_AminusV_t": bias_raw_t[i],
            "Bias_lagcorr_AminusV_t": bias_lagcorr_t[i],
        })

trial_time_df = pd.DataFrame(trial_time_rows)

trial_time_csv = os.path.join(CSV_DIR, "trial_timepoint_bias_for_continuous_recalibration.csv")
trial_time_df.to_csv(trial_time_csv, index=False)

print(f"\nSaved trial-timepoint bias file:\n{trial_time_csv}")

# -------------------------
# TRIAL × BIN SUMMARY
# -------------------------
trial_bin_df = (
    trial_time_df
    .groupby(
        ["subject", "trialIndex", "offset_num", "offset_cat", "Psame", "Psame_centered", "judgment_label", "time_bin"],
        as_index=False
    )
    .agg(
        Bias_raw_AminusV=("Bias_raw_AminusV_t", "mean"),
        Bias_lagcorr_AminusV=("Bias_lagcorr_AminusV_t", "mean"),
        mean_time_sec=("time_sec", "mean"),
        n_timepoints=("time_sec", "count")
    )
)

trial_bin_df["time_bin"] = pd.Categorical(
    trial_bin_df["time_bin"],
    categories=["early", "middle", "late"],
    ordered=True
)

trial_bin_df["offset_cat"] = pd.Categorical(
    trial_bin_df["offset_cat"],
    categories=["10", "20", "30", "40"],
    ordered=True
)

trial_bin_csv = os.path.join(CSV_DIR, "trial_bin_bias_for_continuous_recalibration.csv")
trial_bin_df.to_csv(trial_bin_csv, index=False)

print(f"Saved trial-bin bias file:\n{trial_bin_csv}")

# -------------------------
# MIXED MODELS: DOES RESIDUAL BIAS CHANGE OVER TIME?
# -------------------------
model_df = trial_bin_df.dropna(
    subset=["Bias_raw_AminusV", "Psame_centered", "offset_cat", "time_bin"]
).copy()

# Main model: raw bias as function of offset, time bin, and Psame
raw_time_model = safe_mixedlm(
    formula="Bias_raw_AminusV ~ C(offset_cat) * C(time_bin) + Psame_centered",
    data=model_df,
    groups="subject",
    label="RAW bias: Offset × TimeBin + Psame"
)

# Direct recalibration model: does Psame-adjusted residual bias change over time?
raw_psame_model = safe_mixedlm(
    formula="Bias_raw_AminusV ~ Psame_centered",
    data=model_df,
    groups="subject",
    label="RAW bias: Psame-only model for residualization"
)

model_df["ResidualBias_raw_after_Psame"] = raw_psame_model.resid

raw_resid_time_model = safe_mixedlm(
    formula="ResidualBias_raw_after_Psame ~ C(offset_cat) * C(time_bin)",
    data=model_df,
    groups="subject",
    label="RAW residual bias after Psame: Offset × TimeBin"
)

# Optional lag-corrected versions
if model_df["Bias_lagcorr_AminusV"].notna().sum() > 0:
    lag_model_df = model_df.dropna(subset=["Bias_lagcorr_AminusV"]).copy()

    lag_time_model = safe_mixedlm(
        formula="Bias_lagcorr_AminusV ~ C(offset_cat) * C(time_bin) + Psame_centered",
        data=lag_model_df,
        groups="subject",
        label="LAGCORR bias: Offset × TimeBin + Psame"
    )

    lag_psame_model = safe_mixedlm(
        formula="Bias_lagcorr_AminusV ~ Psame_centered",
        data=lag_model_df,
        groups="subject",
        label="LAGCORR bias: Psame-only model for residualization"
    )

    lag_model_df["ResidualBias_lagcorr_after_Psame"] = lag_psame_model.resid

    lag_resid_time_model = safe_mixedlm(
        formula="ResidualBias_lagcorr_after_Psame ~ C(offset_cat) * C(time_bin)",
        data=lag_model_df,
        groups="subject",
        label="LAGCORR residual bias after Psame: Offset × TimeBin"
    )

# -------------------------
# SUBJECT-LEVEL RESIDUAL TESTS BY OFFSET × TIME BIN
# -------------------------
subject_bin_df = (
    model_df
    .groupby(["subject", "offset_num", "time_bin"], as_index=False)
    .agg(
        ResidualBias_raw_after_Psame=("ResidualBias_raw_after_Psame", "mean"),
        Bias_raw_AminusV=("Bias_raw_AminusV", "mean"),
        Psame=("Psame", "mean"),
        n_trials=("trialIndex", "nunique")
    )
)

subject_bin_csv = os.path.join(CSV_DIR, "subject_offset_timebin_residual_bias_after_psame.csv")
subject_bin_df.to_csv(subject_bin_csv, index=False)

test_rows = []

print("\n" + "=" * 90)
print("RESIDUAL VISUAL BIAS > 0 BY OFFSET × TIME BIN")
print("=" * 90)

for off in [10, 20, 30, 40]:
    for tb in ["early", "middle", "late"]:
        vals = subject_bin_df.loc[
            (subject_bin_df["offset_num"] == off) &
            (subject_bin_df["time_bin"].astype(str) == tb),
            "ResidualBias_raw_after_Psame"
        ].dropna().astype(float).values

        if len(vals) >= 2:
            m, lo, hi, n = mean_ci(vals)
            tres = ttest_1samp(vals, popmean=0.0, alternative="greater")

            print(f"\nOffset {off}°, {tb}")
            print(f"n = {n}")
            print(f"Mean residual bias = {m:.4f}")
            print(f"95% CI = [{lo:.4f}, {hi:.4f}]")
            print(f"t({n-1}) = {tres.statistic:.4f}, p {p_fmt(tres.pvalue)}")

            test_rows.append({
                "offset_num": off,
                "time_bin": tb,
                "n": n,
                "mean_residual_bias": m,
                "ci_low": lo,
                "ci_high": hi,
                "t": tres.statistic,
                "df": n - 1,
                "p_greater_than_zero": tres.pvalue
            })

test_df = pd.DataFrame(test_rows)
test_csv = os.path.join(CSV_DIR, "residual_bias_after_psame_tests_by_offset_timebin.csv")
test_df.to_csv(test_csv, index=False)

# -------------------------
# TIME-RESOLVED RESIDUAL CURVES
# -------------------------
# Residualize each timepoint using Psame-only fixed effects from trial-bin model.
# This keeps the time course while removing the average effect of explicit judgment.
fixed_intercept = raw_psame_model.fe_params.get("Intercept", 0.0)
fixed_beta_psame = raw_psame_model.fe_params.get("Psame_centered", 0.0)

trial_time_df["ResidualBias_raw_after_Psame_t"] = (
    trial_time_df["Bias_raw_AminusV_t"]
    - (fixed_intercept + fixed_beta_psame * trial_time_df["Psame_centered"])
)

if model_df["Bias_lagcorr_AminusV"].notna().sum() > 0:
    fixed_intercept_lag = lag_psame_model.fe_params.get("Intercept", 0.0)
    fixed_beta_psame_lag = lag_psame_model.fe_params.get("Psame_centered", 0.0)

    trial_time_df["ResidualBias_lagcorr_after_Psame_t"] = (
        trial_time_df["Bias_lagcorr_AminusV_t"]
        - (fixed_intercept_lag + fixed_beta_psame_lag * trial_time_df["Psame_centered"])
    )
else:
    trial_time_df["ResidualBias_lagcorr_after_Psame_t"] = np.nan

# Subject-first time-resolved residuals
subject_time_df = (
    trial_time_df
    .groupby(["subject", "offset_num", "time_sec"], as_index=False)
    .agg(
        ResidualBias_raw_after_Psame_t=("ResidualBias_raw_after_Psame_t", "mean"),
        ResidualBias_lagcorr_after_Psame_t=("ResidualBias_lagcorr_after_Psame_t", "mean"),
        Bias_raw_AminusV_t=("Bias_raw_AminusV_t", "mean"),
        Bias_lagcorr_AminusV_t=("Bias_lagcorr_AminusV_t", "mean"),
        Psame=("Psame", "mean"),
        n_trials=("trialIndex", "nunique")
    )
)

subject_time_csv = os.path.join(CSV_DIR, "subject_time_resolved_residual_bias_after_psame.csv")
subject_time_df.to_csv(subject_time_csv, index=False)

group_time_df = (
    subject_time_df
    .groupby(["offset_num", "time_sec"], as_index=False)
    .agg(
        ResidualBias_raw_mean=("ResidualBias_raw_after_Psame_t", "mean"),
        ResidualBias_raw_sem=("ResidualBias_raw_after_Psame_t", sem),
        ResidualBias_lagcorr_mean=("ResidualBias_lagcorr_after_Psame_t", "mean"),
        ResidualBias_lagcorr_sem=("ResidualBias_lagcorr_after_Psame_t", sem),
        Bias_raw_mean=("Bias_raw_AminusV_t", "mean"),
        Bias_raw_sem=("Bias_raw_AminusV_t", sem),
        n_subjects=("subject", "nunique")
    )
)

group_time_df["ResidualBias_raw_ci95"] = 1.96 * group_time_df["ResidualBias_raw_sem"]
group_time_df["ResidualBias_lagcorr_ci95"] = 1.96 * group_time_df["ResidualBias_lagcorr_sem"]
group_time_df["Bias_raw_ci95"] = 1.96 * group_time_df["Bias_raw_sem"]

group_time_csv = os.path.join(CSV_DIR, "group_time_resolved_residual_bias_after_psame.csv")
group_time_df.to_csv(group_time_csv, index=False)

# -------------------------
# PLOTS
# -------------------------

# Plot 1: residual bias over time by offset
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, off in zip(axes, [10, 20, 30, 40]):
    cdf = group_time_df[group_time_df["offset_num"] == off].sort_values("time_sec")

    x = cdf["time_sec"].values.astype(float)
    y = cdf["ResidualBias_raw_mean"].values.astype(float)
    ci = cdf["ResidualBias_raw_ci95"].values.astype(float)

    ax.plot(x, y, linewidth=2.4, label="Residual bias after Psame")
    ax.fill_between(x, y - ci, y + ci, alpha=0.18)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(f"{off}° offset")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Residual Bias A−V")
    ax.legend(fontsize=8)

fig.suptitle(
    "Time-resolved residual visual bias after accounting for explicit causal judgment",
    fontsize=15,
    y=1.02
)
save_fig(fig, os.path.join(FIG_DIR, "time_resolved_residual_bias_after_psame_by_offset.png"))

# Plot 2: early/middle/late residual bias
bin_summary = (
    subject_bin_df
    .groupby(["offset_num", "time_bin"], as_index=False)
    .agg(
        mean_residual=("ResidualBias_raw_after_Psame", "mean"),
        sem_residual=("ResidualBias_raw_after_Psame", sem),
        n_subjects=("subject", "nunique")
    )
)

bin_summary["ci95"] = 1.96 * bin_summary["sem_residual"]

fig, ax = plt.subplots(figsize=(9, 5))

for off in [10, 20, 30, 40]:
    cdf = bin_summary[bin_summary["offset_num"] == off].copy()
    cdf["time_bin"] = pd.Categorical(cdf["time_bin"], categories=["early", "middle", "late"], ordered=True)
    cdf = cdf.sort_values("time_bin")

    ax.errorbar(
        cdf["time_bin"].astype(str),
        cdf["mean_residual"],
        yerr=cdf["ci95"],
        marker="o",
        linewidth=2,
        label=f"{off}°"
    )

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Trial period")
ax.set_ylabel("Residual Bias A−V after Psame")
ax.set_title("Residual visual bias across early, middle, and late trial periods")
ax.legend(title="Offset")
save_fig(fig, os.path.join(FIG_DIR, "early_middle_late_residual_bias_after_psame.png"))

# Plot 3: focus only 30° and 40°
fig, ax = plt.subplots(figsize=(8, 5))

for off in [30, 40]:
    cdf = group_time_df[group_time_df["offset_num"] == off].sort_values("time_sec")

    x = cdf["time_sec"].values.astype(float)
    y = cdf["ResidualBias_raw_mean"].values.astype(float)
    ci = cdf["ResidualBias_raw_ci95"].values.astype(float)

    ax.plot(x, y, linewidth=2.5, label=f"{off}°")
    ax.fill_between(x, y - ci, y + ci, alpha=0.16)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Residual Bias A−V after Psame")
ax.set_title("Persistent residual visual bias at large offsets")
ax.legend(title="Offset")
save_fig(fig, os.path.join(FIG_DIR, "large_offsets_30_40_time_resolved_residual_bias.png"))

# Optional lag-corrected residual plot
if group_time_df["ResidualBias_lagcorr_mean"].notna().sum() > 0:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
    axes = axes.flatten()

    for ax, off in zip(axes, [10, 20, 30, 40]):
        cdf = group_time_df[group_time_df["offset_num"] == off].sort_values("time_sec")

        x = cdf["time_sec"].values.astype(float)
        y = cdf["ResidualBias_lagcorr_mean"].values.astype(float)
        ci = cdf["ResidualBias_lagcorr_ci95"].values.astype(float)

        ax.plot(x, y, linewidth=2.4, label="Lag-corrected residual bias")
        ax.fill_between(x, y - ci, y + ci, alpha=0.18)
        ax.axhline(0, linestyle="--", linewidth=1)
        ax.set_title(f"{off}° offset")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Lag-corrected Residual Bias A−V")
        ax.legend(fontsize=8)

    fig.suptitle(
        "Lag-corrected time-resolved residual visual bias after Psame",
        fontsize=15,
        y=1.02
    )
    save_fig(fig, os.path.join(FIG_DIR, "lagcorr_time_resolved_residual_bias_after_psame_by_offset.png"))

# -------------------------
# FINAL OUTPUT SUMMARY
# -------------------------
print("\nDONE.")
print(f"Outputs saved to: {OUT_DIR}")
print(f"Trial-timepoint CSV: {trial_time_csv}")
print(f"Trial-bin CSV:       {trial_bin_csv}")
print(f"Subject-bin CSV:     {subject_bin_csv}")
print(f"Test CSV:            {test_csv}")
print(f"Subject-time CSV:    {subject_time_csv}")
print(f"Group-time CSV:      {group_time_csv}")
print(f"Figures:             {FIG_DIR}")


Fixed-offset trials included:
offset_num  judgment_label  
10          Different source    268
            Same source         732
20          Different source    409
            Same source         591
30          Different source    693
            Same source         307
40          Different source    904
            Same source          96
dtype: int64

Saved trial-timepoint bias file:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\continuous_recalibration_residual_bias\csv_outputs\trial_timepoint_bias_for_continuous_recalibration.csv
Saved trial-bin bias file:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\continuous_recalibration_residual_bias\csv_outputs\trial_bin_bias_for_continuous_recalibration.csv

RAW bias: Offset × TimeBin + Psame
Formula: Bias_raw_AminusV ~ C(offset_cat) * C(time_bin) + Psame_centered
Rows: 12000
Subjects: 50
          

C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


                        Mixed Linear Model Regression Results
Model:                 MixedLM    Dependent Variable:    ResidualBias_raw_after_Psame
No. Observations:      12000      Method:                ML                          
No. Groups:            50         Scale:                 115.3421                    
Min. group size:       240        Log-Likelihood:        -45514.6764                 
Max. group size:       240        Converged:             Yes                         
Mean group size:       240.0                                                         
-------------------------------------------------------------------------------------
                                          Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------------------
Intercept                                 -3.505    0.340 -10.321 0.000 -4.171 -2.840
C(offset_cat)[T.20]                        5.724    0.480  11.919 0.000  4.783

C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\AppData\Local\Temp\ipykernel_29840\874549313.py:369: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["subject", "offset_num", "time_bin"], as_index=False)


                        Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   ResidualBias_lagcorr_after_Psame
No. Observations:    11827     Method:               ML                              
No. Groups:          50        Scale:                116.4206                        
Min. group size:     138       Log-Likelihood:       -44913.5454                     
Max. group size:     240       Converged:            Yes                             
Mean group size:     236.5                                                           
-------------------------------------------------------------------------------------
                                          Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------------------
Intercept                                 -4.067    0.343 -11.842 0.000 -4.740 -3.394
C(offset_cat)[T.20]                        5.155    0.485  10.619 0.000  4.204

C:\Users\pkell\AppData\Local\Temp\ipykernel_29840\874549313.py:516: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["offset_num", "time_bin"], as_index=False)



DONE.
Outputs saved to: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\continuous_recalibration_residual_bias
Trial-timepoint CSV: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\continuous_recalibration_residual_bias\csv_outputs\trial_timepoint_bias_for_continuous_recalibration.csv
Trial-bin CSV:       C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\continuous_recalibration_residual_bias\csv_outputs\trial_bin_bias_for_continuous_recalibration.csv
Subject-bin CSV:     C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\00_core_spatial_metrics\continuous_recalibration_residual_bias\csv_outputs\subject_offset_timebin_residual_bias_after_psame.csv
Test CSV:            C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_an

In [27]:
# %% CHINESE RESTAURANT / AUDITORY-ANCHOR BREAKPOINT + DYNAMIC CAPTURE ANALYSIS
# ============================================================
# Goal:
# Test whether participants settle into a stable visually biased
# auditory-location estimate during fixed-offset multisensory trials,
# while also testing whether non-stabilized trials remain dynamically
# updating over time.
#
# Main analyses:
#   1. Compute continuous visual capture across sliding windows.
#   2. For each subject × trial × offset, fit:
#        a) constant model
#        b) linear model
#        c) piecewise / segmented model with estimated breakpoint
#   3. Do NOT force a breakpoint:
#        A trial is classified as stabilized only if the piecewise model
#        is better than linear drift and the post-breakpoint slope is small.
#   4. Add dynamic capture analyses:
#        A trial that does not stabilize can be classified as dynamically
#        updating if it shows evidence of non-flat capture over time:
#          - nonconstant model support
#          - non-negligible whole-trial slope and/or post-breakpoint slope
#          - non-negligible mean absolute capture derivative
#   5. Extract:
#        - stabilization time / breakpoint time
#        - pre-breakpoint capture slope
#        - post-breakpoint capture slope
#        - capture value at breakpoint
#        - anchored auditory estimate in degrees toward vision
#        - plateau / stabilized capture value
#        - plateau / stabilized auditory estimate in degrees toward vision
#        - dynamic updating probability
#        - whole-trial capture slope
#        - mean absolute capture derivative
#        - late-trial capture derivative
#        - dynamicity among non-stabilized trials
#   6. Average trial-level estimates within subject × offset,
#      then average within subjects and across subjects for group-level estimates.
#
# Outputs:
#   - window-level metrics
#   - trial-level breakpoint + dynamic metrics
#   - subject × offset breakpoint + dynamic summaries
#   - subject-level breakpoint + dynamic summaries
#   - group-level estimates and tests
#   - offset models
#   - plots
# ============================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_1samp, ttest_rel, t as t_dist
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

SPATIAL_ANALYSES_ROOT = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses"
OUT_DIR = os.path.join(SPATIAL_ANALYSES_ROOT, "06_stabilization_breakpoint")
CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

EXCLUDE_PRACTICE = True
FIXED_OFFSETS = [10, 20, 30, 40]

RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0
TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1 / RESAMPLE_HZ, 1 / RESAMPLE_HZ)

# Sliding-window settings
WINDOW_SEC = 1.0
STEP_SEC = 0.25

# Lag correction settings
# NOTE:
# This preserves your original trial-level visual-lag correction so you can compare
# raw capture and lag-corrected capture. If you later compute subject-level baseline
# lags from unimodal/congruent trials, replace the lag_sec value below with that
# subject-level baseline lag.
MAX_LAG_SEC = 4.0

# Descriptive phase bins retained for plots and sanity checks
EARLY_RANGE = (0.0, 6.0)
MIDDLE_RANGE = (6.0, 12.0)
LATE_RANGE = (12.0, 18.0)

MIN_VALID_SAMPLES = 8
MIN_ABS_DISPARITY_FOR_CAPTURE = 0.01

FIG_DPI = 180

# =========================
# BREAKPOINT SETTINGS
# =========================
BREAKPOINT_METRICS = [
    ("RawVisualCapture", "Raw visual capture"),
    ("LagCorrectedVisualCapture", "Lag-corrected visual capture"),
]

# Candidate breakpoints are searched between these times.
# Avoid values too close to the start/end because slopes are unstable there.
BREAKPOINT_MIN_SEC = 2.0
BREAKPOINT_MAX_SEC = 15.0
BREAKPOINT_STEP_SEC = STEP_SEC

# Minimum amount of data required on each side of the candidate breakpoint.
MIN_PRE_WINDOWS = 8      # 8 windows × .25 step ≈ 2 s worth of window centers
MIN_POST_WINDOWS = 12    # 12 windows × .25 step ≈ 3 s worth of window centers
MIN_TOTAL_WINDOWS_FOR_BREAKPOINT = 24

# Optional smoothing of the windowed capture trajectory before fitting.
# This reduces sensitivity to one noisy window but still preserves continuous time.
SMOOTH_CAPTURE_FOR_BREAKPOINT = True
SMOOTH_N_WINDOWS = 3

# Exclude extreme capture values from the breakpoint fit.
# This protects the segmented regression from rare denominator/tracking artifacts.
CAPTURE_FIT_MIN = -1.0
CAPTURE_FIT_MAX = 2.0

# Model-selection rule:
# Positive BIC improvement means piecewise is better than linear.
# 2 = positive evidence; 6 = stronger evidence.
BIC_DELTA_TO_ACCEPT_PIECEWISE = 2.0

# Stabilization rule:
# The post-breakpoint slope must be small in capture units per second.
# Example: .01 capture/sec means only .06 capture units over 6 seconds.
POST_SLOPE_ABS_MAX_CAPTURE_PER_SEC = 0.01

# The post-breakpoint slope must be meaningfully smaller than the pre-breakpoint slope.
SLOPE_REDUCTION_RATIO = 0.50

# Require some meaningful pre-breakpoint updating before calling it a stabilization transition.
PRE_SLOPE_ABS_MIN_CAPTURE_PER_SEC = 0.005

# Equivalence bound for group-level post-breakpoint slope.
# Used for TOST equivalence test.
POST_SLOPE_EQUIVALENCE_BOUND = 0.01

# =========================
# DYNAMIC CAPTURE SETTINGS
# =========================
# These thresholds define "continued dynamic capture."
# Keep them aligned with your stabilization threshold so the logic is coherent:
#   post slope <= .01 capture/sec = practically stable
#   absolute slope/derivative > .01 capture/sec = practically dynamic
DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC = 0.01
DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC = 0.01

# Require evidence that a non-flat model is better than a constant model.
BIC_DELTA_TO_ACCEPT_NONCONSTANT = 2.0

# If True, a trial must be nonconstant by model selection before being called dynamic.
REQUIRE_NONCONSTANT_MODEL_FOR_DYNAMIC = True

# Dynamic classification is only applied to trials that are not stable/stabilized.
# Stable/stabilized trials are not labeled dynamically updating even if they have
# early pre-breakpoint change.
CLASSIFY_DYNAMIC_ONLY_IF_NOT_STABLE = True

# =========================
# HELPER FUNCTIONS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else Path(filepath).stem

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def is_sweep_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    return ("sweep" in trial_type) or ("sweep" in cond) or ("diverge" in cond) or ("converge" in cond)

def is_fixed_offset_av_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    offset = extract_offset_from_row(row)

    if not np.isfinite(offset):
        return False

    if int(offset) not in FIXED_OFFSETS:
        return False

    if is_sweep_trial(row):
        return False

    if trial_type in ["audio_only", "visual_only", "congruent"]:
        return False

    if "incongruent" in trial_type or "fixed" in trial_type or "static" in trial_type:
        return True

    if "incongruent" in cond or "fixed" in cond or "static" in cond:
        return True

    return True

def resample_trial(df_trial, time_grid):
    needed = ["t", "mouse_x", "blob_x", "audio_x"]
    for c in needed:
        if c not in df_trial.columns:
            raise ValueError(f"Missing required column: {c}")

    d = df_trial.copy()
    d = d.sort_values("t")
    d = d[np.isfinite(d["t"])]
    d = d.drop_duplicates(subset="t", keep="first")

    if len(d) < 5:
        return None

    t_original = d["t"].values.astype(float)
    t_original = t_original - np.nanmin(t_original)

    if np.nanmax(t_original) < 1.0:
        return None

    def _interp(col):
        y = d[col].values.astype(float)
        ok = np.isfinite(t_original) & np.isfinite(y)

        if ok.sum() < 3:
            return np.full_like(time_grid, np.nan, dtype=float)

        return np.interp(
            time_grid,
            t_original[ok],
            y[ok],
            left=np.nan,
            right=np.nan
        )

    return {
        "t_grid": time_grid.copy(),
        "mouse_x": _interp("mouse_x"),
        "blob_x": _interp("blob_x"),
        "audio_x": _interp("audio_x"),
    }

def visual_capture(response, visual, audio):
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    denom = visual - audio
    ok = (
        np.isfinite(response)
        & np.isfinite(visual)
        & np.isfinite(audio)
        & (np.abs(denom) > MIN_ABS_DISPARITY_FOR_CAPTURE)
    )

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean((response[ok] - audio[ok]) / denom[ok]))

def window_mean(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean(x[ok]))

def window_sd(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanstd(x[ok], ddof=1))

def corr_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    x = x[ok]
    y = y[ok]

    if np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])

def lag_scanned_corr(response, stimulus, fs=20.0, max_lag_sec=4.0):
    """
    Estimate a non-negative response lag for temporal alignment.

    Positive lag means response lags the stimulus. Negative lags are not
    searched. No peak-correlation threshold is imposed here because this
    lag is used to construct the lag-corrected spatial-capture trajectory;
    low correlation during AV conflict is not, by itself, a reason to
    discard the spatial trial.
    """
    response = np.asarray(response, dtype=float)
    stimulus = np.asarray(stimulus, dtype=float)

    max_lag_samples = int(round(max_lag_sec * fs))
    lag_samples_grid = np.arange(0, max_lag_samples + 1)

    best_corr = np.nan
    best_lag_samples = np.nan

    for lag_samples in lag_samples_grid:
        if lag_samples == 0:
            r = response
            st = stimulus
        else:
            r = response[lag_samples:]
            st = stimulus[:-lag_samples]

        c = corr_1d(r, st)

        if np.isfinite(c) and (not np.isfinite(best_corr) or c > best_corr):
            best_corr = c
            best_lag_samples = lag_samples

    return {
        "peak_corr": best_corr,
        "lag_samples": best_lag_samples,
        "lag_sec": best_lag_samples / fs if np.isfinite(best_lag_samples) else np.nan,
    }

def apply_response_lag_correction(response, t_grid, lag_sec):
    """
    Positive lag_sec means response lags the stimulus.
    To lag-correct the response, shift response earlier by lag_sec:
        corrected_response(t) = observed_response(t + lag)
    """
    response = np.asarray(response, dtype=float)
    t_grid = np.asarray(t_grid, dtype=float)

    if not np.isfinite(lag_sec):
        return response.copy()

    corrected = np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan
    )

    return corrected

def get_time_bin(center):
    if EARLY_RANGE[0] <= center < EARLY_RANGE[1]:
        return "Early"
    if MIDDLE_RANGE[0] <= center < MIDDLE_RANGE[1]:
        return "Middle"
    if LATE_RANGE[0] <= center <= LATE_RANGE[1]:
        return "Late"
    return np.nan

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)

    return m, m - crit * se, m + crit * se, n

def sem(vals):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    if len(vals) < 2:
        return np.nan
    return np.std(vals, ddof=1) / np.sqrt(len(vals))

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def one_sample_test(vals, popmean=0.0):
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
        }

    m, lo, hi, n = mean_ci(vals)
    test = ttest_1samp(vals, popmean)

    return {
        "n": n,
        "mean": m,
        "ci_low": lo,
        "ci_high": hi,
        "t": test.statistic,
        "df": n - 1,
        "p": test.pvalue,
    }

def one_sample_greater_test(vals, threshold=0.0):
    """
    One-sided test: mean(vals) > threshold.
    Useful for asking whether non-stabilized dynamicity is greater than
    a practical dynamicity threshold.
    """
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p_one_sided": np.nan,
        }

    m, lo, hi, n = mean_ci(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)

    if se <= 0 or not np.isfinite(se):
        return {
            "n": n,
            "mean": m,
            "ci_low": lo,
            "ci_high": hi,
            "t": np.nan,
            "df": n - 1,
            "p_one_sided": np.nan,
        }

    t_val = (m - threshold) / se
    p_one = 1 - t_dist.cdf(t_val, df=n - 1)

    return {
        "n": n,
        "mean": m,
        "ci_low": lo,
        "ci_high": hi,
        "t": t_val,
        "df": n - 1,
        "p_one_sided": p_one,
    }

def paired_test(x, y):
    d = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    n = len(d)

    if n < 2:
        return {
            "n": n,
            "mean_x": np.nan,
            "mean_y": np.nan,
            "mean_diff_x_minus_y": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
        }

    test = ttest_rel(d["x"].values, d["y"].values)

    return {
        "n": n,
        "mean_x": float(d["x"].mean()),
        "mean_y": float(d["y"].mean()),
        "mean_diff_x_minus_y": float((d["x"] - d["y"]).mean()),
        "t": test.statistic,
        "df": n - 1,
        "p": test.pvalue,
    }

def tost_equivalence_test(vals, lower_bound, upper_bound, alpha=0.05):
    """
    Two one-sided tests for equivalence.
    H0_lower: mean <= lower_bound
    H0_upper: mean >= upper_bound
    Equivalent if both one-sided tests are significant.
    """
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "t_lower": np.nan,
            "p_lower": np.nan,
            "t_upper": np.nan,
            "p_upper": np.nan,
            "p_tost": np.nan,
            "equivalent": np.nan,
        }

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    df = n - 1

    if se <= 0 or not np.isfinite(se):
        return {
            "n": n,
            "mean": m,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "t_lower": np.nan,
            "p_lower": np.nan,
            "t_upper": np.nan,
            "p_upper": np.nan,
            "p_tost": np.nan,
            "equivalent": np.nan,
        }

    # Test mean > lower_bound
    t_lower = (m - lower_bound) / se
    p_lower = 1 - t_dist.cdf(t_lower, df)

    # Test mean < upper_bound
    t_upper = (m - upper_bound) / se
    p_upper = t_dist.cdf(t_upper, df)

    p_tost = max(p_lower, p_upper)
    equivalent = int((p_lower < alpha) and (p_upper < alpha))

    return {
        "n": n,
        "mean": m,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "t_lower": t_lower,
        "p_lower": p_lower,
        "t_upper": t_upper,
        "p_upper": p_upper,
        "p_tost": p_tost,
        "equivalent": equivalent,
    }

# =========================
# OLS / BREAKPOINT HELPER FUNCTIONS
# =========================
def fit_ols_design(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    X = X[ok]
    y = y[ok]

    n = len(y)
    p = X.shape[1]

    if n <= p:
        return None

    try:
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    except Exception:
        return None

    pred = X @ beta
    resid = y - pred
    rss = float(np.sum(resid ** 2))

    rss_per_n = max(rss / n, 1e-12)
    bic = float(n * np.log(rss_per_n) + p * np.log(n))
    aic = float(n * np.log(rss_per_n) + 2 * p)

    tss = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if tss <= 1e-12 else float(1 - rss / tss)

    return {
        "beta": beta,
        "pred": pred,
        "resid": resid,
        "rss": rss,
        "bic": bic,
        "aic": aic,
        "r_squared": r2,
        "n": n,
        "p": p,
    }

def fit_constant_model(t, y):
    X = np.column_stack([np.ones_like(t)])
    return fit_ols_design(X, y)

def fit_linear_model(t, y):
    X = np.column_stack([np.ones_like(t), t])
    return fit_ols_design(X, y)

def fit_piecewise_model(t, y, knot):
    hinge = np.maximum(0.0, t - knot)
    X = np.column_stack([np.ones_like(t), t, hinge])
    fit = fit_ols_design(X, y)

    if fit is None:
        return None

    beta = fit["beta"]
    pre_slope = float(beta[1])
    slope_change = float(beta[2])
    post_slope = float(beta[1] + beta[2])

    fit["knot"] = float(knot)
    fit["pre_slope"] = pre_slope
    fit["slope_change"] = slope_change
    fit["post_slope"] = post_slope

    return fit

def piecewise_predict(t, beta, knot):
    t = np.asarray(t, dtype=float)
    hinge = np.maximum(0.0, t - knot)
    X = np.column_stack([np.ones_like(t), t, hinge])
    return X @ beta

def compute_capture_derivative_metrics(t, y):
    """
    Computes continuous dynamicity metrics from the window-level capture trajectory.

    Main dynamicity metric:
      mean_abs_capture_derivative_per_sec =
          mean absolute window-to-window change in capture divided by time.

    This metric is useful for showing whether visual capture remains dynamic
    even when no accepted stabilization breakpoint is found.
    """
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(t) & np.isfinite(y)
    t = t[ok]
    y = y[ok]

    if len(t) < 3:
        return {
            "mean_abs_capture_derivative_per_sec": np.nan,
            "median_abs_capture_derivative_per_sec": np.nan,
            "max_abs_capture_derivative_per_sec": np.nan,
            "early_mean_abs_capture_derivative_per_sec": np.nan,
            "middle_mean_abs_capture_derivative_per_sec": np.nan,
            "late_mean_abs_capture_derivative_per_sec": np.nan,
            "late_minus_early_abs_derivative_per_sec": np.nan,
        }

    order = np.argsort(t)
    t = t[order]
    y = y[order]

    dt = np.diff(t)
    dy = np.diff(y)

    ok_diff = np.isfinite(dt) & np.isfinite(dy) & (dt > 0)

    if ok_diff.sum() < 2:
        return {
            "mean_abs_capture_derivative_per_sec": np.nan,
            "median_abs_capture_derivative_per_sec": np.nan,
            "max_abs_capture_derivative_per_sec": np.nan,
            "early_mean_abs_capture_derivative_per_sec": np.nan,
            "middle_mean_abs_capture_derivative_per_sec": np.nan,
            "late_mean_abs_capture_derivative_per_sec": np.nan,
            "late_minus_early_abs_derivative_per_sec": np.nan,
        }

    derivative = np.full_like(dy, np.nan, dtype=float)
    derivative[ok_diff] = dy[ok_diff] / dt[ok_diff]

    abs_derivative = np.abs(derivative)

    # Use midpoint times for derivative intervals.
    t_mid = (t[:-1] + t[1:]) / 2

    def phase_mean_abs_derivative(time_range):
        mask = (
            np.isfinite(abs_derivative)
            & (t_mid >= time_range[0])
            & (t_mid < time_range[1])
        )

        if mask.sum() < 2:
            return np.nan

        return float(np.nanmean(abs_derivative[mask]))

    early_dyn = phase_mean_abs_derivative(EARLY_RANGE)
    middle_dyn = phase_mean_abs_derivative(MIDDLE_RANGE)

    late_mask_range = (
        np.isfinite(abs_derivative)
        & (t_mid >= LATE_RANGE[0])
        & (t_mid <= LATE_RANGE[1])
    )
    late_dyn = float(np.nanmean(abs_derivative[late_mask_range])) if late_mask_range.sum() >= 2 else np.nan

    return {
        "mean_abs_capture_derivative_per_sec": float(np.nanmean(abs_derivative)),
        "median_abs_capture_derivative_per_sec": float(np.nanmedian(abs_derivative)),
        "max_abs_capture_derivative_per_sec": float(np.nanmax(abs_derivative)),
        "early_mean_abs_capture_derivative_per_sec": early_dyn,
        "middle_mean_abs_capture_derivative_per_sec": middle_dyn,
        "late_mean_abs_capture_derivative_per_sec": late_dyn,
        "late_minus_early_abs_derivative_per_sec": (
            late_dyn - early_dyn
            if np.isfinite(late_dyn) and np.isfinite(early_dyn)
            else np.nan
        ),
    }

def fit_stabilizing_breakpoint_for_trial(trial_window_df, metric):
    """
    Fits constant, linear, and candidate piecewise models to one trial's capture trajectory.

    A stabilizing breakpoint is accepted only if:
      1. Piecewise BIC improves over linear BIC by BIC_DELTA_TO_ACCEPT_PIECEWISE.
      2. Piecewise is also better than constant.
      3. Post-breakpoint slope is small.
      4. Post-breakpoint slope is meaningfully reduced relative to pre-breakpoint slope.
      5. Pre-breakpoint slope is meaningful.
      6. Breakpoint is not too close to start/end.

    Dynamic capture classification:
      A non-stabilized trial can be classified as dynamically updating if:
        1. it is not stable or stabilized,
        2. a nonconstant model is supported,
        3. and it shows non-negligible whole-trial slope, candidate post-slope,
           or mean absolute capture derivative.

    Returns both stabilization and dynamicity metrics.
    """
    d = trial_window_df.copy()
    d = d.sort_values("window_center_sec")

    needed = [
        "window_center_sec",
        metric,
        "offset_deg",
        "MeanDisparitySigned_deg",
        "MeanAbsDisparity_deg",
    ]

    for c in needed:
        if c not in d.columns:
            d[c] = np.nan

    d = d[needed].replace([np.inf, -np.inf], np.nan).dropna(subset=["window_center_sec", metric])
    d = d[(d[metric] >= CAPTURE_FIT_MIN) & (d[metric] <= CAPTURE_FIT_MAX)].copy()

    base_fail = {
        "metric": metric,
        "n_windows_used": len(d),
        "status": "too_few_windows",
        "best_model_type": np.nan,
        "has_stabilizing_breakpoint": 0,
        "stable_from_start": 0,
        "stable_or_stabilized": 0,
        "nonconstant_model_supported": 0,
        "dynamically_updating": 0,
        "ambiguous_or_noisy": 1,
        "trial_dynamic_class": "insufficient_data",
    }

    if len(d) < MIN_TOTAL_WINDOWS_FOR_BREAKPOINT:
        return base_fail

    t = d["window_center_sec"].values.astype(float)
    y_observed = d[metric].values.astype(float)

    if SMOOTH_CAPTURE_FOR_BREAKPOINT and len(y_observed) >= SMOOTH_N_WINDOWS:
        y_fit = (
            pd.Series(y_observed)
            .rolling(window=SMOOTH_N_WINDOWS, center=True, min_periods=1)
            .mean()
            .values
            .astype(float)
        )
    else:
        y_fit = y_observed.copy()

    derivative_metrics = compute_capture_derivative_metrics(t, y_fit)

    offset_abs = float(np.nanmedian(np.abs(d["offset_deg"].values.astype(float))))
    mean_signed_disparity = float(np.nanmean(d["MeanDisparitySigned_deg"].values.astype(float)))
    mean_abs_disparity = float(np.nanmean(d["MeanAbsDisparity_deg"].values.astype(float)))

    if not np.isfinite(offset_abs):
        offset_abs = np.nan

    if not np.isfinite(mean_abs_disparity):
        mean_abs_disparity = offset_abs

    const_fit = fit_constant_model(t, y_fit)
    linear_fit = fit_linear_model(t, y_fit)

    if const_fit is None or linear_fit is None:
        fail = base_fail.copy()
        fail.update({
            "status": "model_fit_failed",
        })
        fail.update(derivative_metrics)
        return fail

    linear_slope = float(linear_fit["beta"][1])
    whole_trial_abs_linear_slope = abs(linear_slope)
    linear_bic_improvement_vs_constant = float(const_fit["bic"] - linear_fit["bic"])

    # Candidate breakpoints
    candidate_knots = np.arange(
        BREAKPOINT_MIN_SEC,
        BREAKPOINT_MAX_SEC + 1e-9,
        BREAKPOINT_STEP_SEC
    )

    candidate_fits = []

    for knot in candidate_knots:
        n_pre = int(np.sum(t < knot))
        n_post = int(np.sum(t >= knot))

        if n_pre < MIN_PRE_WINDOWS or n_post < MIN_POST_WINDOWS:
            continue

        pw_fit = fit_piecewise_model(t, y_fit, knot)

        if pw_fit is not None:
            pw_fit["n_pre_windows"] = n_pre
            pw_fit["n_post_windows"] = n_post
            candidate_fits.append(pw_fit)

    if len(candidate_fits) == 0:
        stable_from_start = int(
            (const_fit["bic"] <= linear_fit["bic"])
            and (whole_trial_abs_linear_slope <= POST_SLOPE_ABS_MAX_CAPTURE_PER_SEC)
        )

        nonconstant_model_supported = int(
            linear_bic_improvement_vs_constant >= BIC_DELTA_TO_ACCEPT_NONCONSTANT
        )

        whole_trial_slope_dynamic = int(
            whole_trial_abs_linear_slope >= DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC
        )

        derivative_dynamic = int(
            np.isfinite(derivative_metrics["mean_abs_capture_derivative_per_sec"])
            and derivative_metrics["mean_abs_capture_derivative_per_sec"] >= DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC
        )

        late_derivative_dynamic = int(
            np.isfinite(derivative_metrics["late_mean_abs_capture_derivative_per_sec"])
            and derivative_metrics["late_mean_abs_capture_derivative_per_sec"] >= DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC
        )

        dynamic_condition = (
            (whole_trial_slope_dynamic == 1)
            or (derivative_dynamic == 1)
            or (late_derivative_dynamic == 1)
        )

        if REQUIRE_NONCONSTANT_MODEL_FOR_DYNAMIC:
            dynamic_condition = dynamic_condition and (nonconstant_model_supported == 1)

        dynamically_updating = int((stable_from_start == 0) and dynamic_condition)
        ambiguous_or_noisy = int((stable_from_start == 0) and (dynamically_updating == 0))

        if stable_from_start == 1:
            trial_dynamic_class = "stable_from_start"
        elif dynamically_updating == 1:
            trial_dynamic_class = "dynamically_updating"
        else:
            trial_dynamic_class = "ambiguous_or_noisy"

        out = {
            "metric": metric,
            "n_windows_used": len(d),
            "status": "no_valid_candidate_knots",
            "best_model_type": "constant" if const_fit["bic"] < linear_fit["bic"] else "linear",

            "constant_bic": const_fit["bic"],
            "linear_bic": linear_fit["bic"],
            "constant_r_squared": const_fit["r_squared"],
            "linear_r_squared": linear_fit["r_squared"],

            "linear_bic_improvement_vs_constant": linear_bic_improvement_vs_constant,
            "linear_slope_capture_per_sec": linear_slope,
            "whole_trial_abs_linear_slope_capture_per_sec": whole_trial_abs_linear_slope,

            "has_stabilizing_breakpoint": 0,
            "stable_from_start": stable_from_start,
            "stable_or_stabilized": int(stable_from_start == 1),

            "nonconstant_model_supported": nonconstant_model_supported,
            "whole_trial_slope_dynamic": whole_trial_slope_dynamic,
            "candidate_post_slope_dynamic": np.nan,
            "derivative_dynamic": derivative_dynamic,
            "late_derivative_dynamic": late_derivative_dynamic,
            "dynamically_updating": dynamically_updating,
            "ambiguous_or_noisy": ambiguous_or_noisy,
            "trial_dynamic_class": trial_dynamic_class,
        }
        out.update(derivative_metrics)
        return out

    best_pw = min(candidate_fits, key=lambda z: z["bic"])

    model_bics = {
        "constant": const_fit["bic"],
        "linear": linear_fit["bic"],
        "piecewise": best_pw["bic"],
    }

    best_model_type = min(model_bics, key=model_bics.get)

    knot = float(best_pw["knot"])
    beta = best_pw["beta"]
    pred = piecewise_predict(t, beta, knot)

    pre_slope = float(best_pw["pre_slope"])
    post_slope = float(best_pw["post_slope"])
    slope_change = float(best_pw["slope_change"])

    bic_improvement_vs_linear = float(linear_fit["bic"] - best_pw["bic"])
    bic_improvement_vs_constant = float(const_fit["bic"] - best_pw["bic"])

    post_slope_abs = abs(post_slope)
    pre_slope_abs = abs(pre_slope)

    post_slope_small = int(post_slope_abs <= POST_SLOPE_ABS_MAX_CAPTURE_PER_SEC)
    pre_slope_meaningful = int(pre_slope_abs >= PRE_SLOPE_ABS_MIN_CAPTURE_PER_SEC)

    if pre_slope_abs > 0:
        slope_reduction_ratio_observed = post_slope_abs / pre_slope_abs
    else:
        slope_reduction_ratio_observed = np.nan

    slope_reduced = int(
        np.isfinite(slope_reduction_ratio_observed)
        and slope_reduction_ratio_observed <= SLOPE_REDUCTION_RATIO
    )

    piecewise_supported = int(
        (bic_improvement_vs_linear >= BIC_DELTA_TO_ACCEPT_PIECEWISE)
        and (bic_improvement_vs_constant >= 0)
        and (best_model_type == "piecewise")
    )

    has_stabilizing_breakpoint = int(
        piecewise_supported
        and post_slope_small
        and slope_reduced
        and pre_slope_meaningful
    )

    stable_from_start = int(
        (best_model_type == "constant")
        and (whole_trial_abs_linear_slope <= POST_SLOPE_ABS_MAX_CAPTURE_PER_SEC)
    )

    stable_or_stabilized = int(
        (has_stabilizing_breakpoint == 1)
        or (stable_from_start == 1)
    )

    nonconstant_model_supported = int(
        (linear_bic_improvement_vs_constant >= BIC_DELTA_TO_ACCEPT_NONCONSTANT)
        or (bic_improvement_vs_constant >= BIC_DELTA_TO_ACCEPT_NONCONSTANT)
    )

    whole_trial_slope_dynamic = int(
        whole_trial_abs_linear_slope >= DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC
    )

    candidate_post_slope_dynamic = int(
        piecewise_supported
        and (post_slope_abs >= DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC)
    )

    derivative_dynamic = int(
        np.isfinite(derivative_metrics["mean_abs_capture_derivative_per_sec"])
        and derivative_metrics["mean_abs_capture_derivative_per_sec"] >= DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC
    )

    late_derivative_dynamic = int(
        np.isfinite(derivative_metrics["late_mean_abs_capture_derivative_per_sec"])
        and derivative_metrics["late_mean_abs_capture_derivative_per_sec"] >= DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC
    )

    dynamic_condition = (
        (whole_trial_slope_dynamic == 1)
        or (candidate_post_slope_dynamic == 1)
        or (derivative_dynamic == 1)
        or (late_derivative_dynamic == 1)
    )

    if REQUIRE_NONCONSTANT_MODEL_FOR_DYNAMIC:
        dynamic_condition = dynamic_condition and (nonconstant_model_supported == 1)

    if CLASSIFY_DYNAMIC_ONLY_IF_NOT_STABLE:
        dynamically_updating = int((stable_or_stabilized == 0) and dynamic_condition)
    else:
        dynamically_updating = int(dynamic_condition)

    ambiguous_or_noisy = int(
        (stable_or_stabilized == 0)
        and (dynamically_updating == 0)
    )

    if stable_from_start == 1:
        trial_dynamic_class = "stable_from_start"
    elif has_stabilizing_breakpoint == 1:
        trial_dynamic_class = "stabilized_after_breakpoint"
    elif dynamically_updating == 1:
        trial_dynamic_class = "dynamically_updating"
    else:
        trial_dynamic_class = "ambiguous_or_noisy"

    # Candidate anchor values
    break_idx = int(np.nanargmin(np.abs(t - knot)))
    breakpoint_capture_observed_nearest = float(y_observed[break_idx])
    breakpoint_capture_predicted = float(piecewise_predict(np.array([knot]), beta, knot)[0])

    post_mask = t >= knot
    if post_mask.sum() > 0:
        plateau_capture_observed = float(np.nanmean(y_observed[post_mask]))
        plateau_capture_predicted = float(np.nanmean(pred[post_mask]))
        plateau_capture_sd_observed = float(np.nanstd(y_observed[post_mask], ddof=1)) if post_mask.sum() > 1 else np.nan
        post_model_residual_sd = float(np.nanstd((y_observed - pred)[post_mask], ddof=1)) if post_mask.sum() > 1 else np.nan

        post_derivative_metrics = compute_capture_derivative_metrics(t[post_mask], y_fit[post_mask])
        post_breakpoint_mean_abs_derivative = post_derivative_metrics["mean_abs_capture_derivative_per_sec"]
        post_breakpoint_median_abs_derivative = post_derivative_metrics["median_abs_capture_derivative_per_sec"]
    else:
        plateau_capture_observed = np.nan
        plateau_capture_predicted = np.nan
        plateau_capture_sd_observed = np.nan
        post_model_residual_sd = np.nan
        post_breakpoint_mean_abs_derivative = np.nan
        post_breakpoint_median_abs_derivative = np.nan

    # Convert capture into degrees of auditory-location displacement.
    # capture = 0.25 at 40 deg means the auditory estimate is stabilized
    # about 10 deg from the auditory source toward the visual source.
    candidate_breakpoint_anchor_shift_toward_visual_deg = breakpoint_capture_predicted * offset_abs
    candidate_plateau_anchor_shift_toward_visual_deg = plateau_capture_observed * offset_abs

    candidate_breakpoint_anchor_shift_signed_physical_deg = breakpoint_capture_predicted * mean_signed_disparity
    candidate_plateau_anchor_shift_signed_physical_deg = plateau_capture_observed * mean_signed_disparity

    accepted = has_stabilizing_breakpoint == 1

    def accepted_value(x):
        return x if accepted else np.nan

    out = {
        "metric": metric,
        "n_windows_used": len(d),
        "status": "fit_ok",
        "best_model_type": best_model_type,

        "constant_bic": const_fit["bic"],
        "linear_bic": linear_fit["bic"],
        "piecewise_bic": best_pw["bic"],

        "constant_r_squared": const_fit["r_squared"],
        "linear_r_squared": linear_fit["r_squared"],
        "piecewise_r_squared": best_pw["r_squared"],

        "linear_bic_improvement_vs_constant": linear_bic_improvement_vs_constant,
        "linear_slope_capture_per_sec": linear_slope,
        "whole_trial_abs_linear_slope_capture_per_sec": whole_trial_abs_linear_slope,

        "candidate_breakpoint_sec": knot,
        "candidate_pre_slope_capture_per_sec": pre_slope,
        "candidate_post_slope_capture_per_sec": post_slope,
        "candidate_slope_change_capture_per_sec": slope_change,
        "candidate_abs_pre_slope_capture_per_sec": pre_slope_abs,
        "candidate_abs_post_slope_capture_per_sec": post_slope_abs,
        "candidate_abs_slope_reduction_capture_per_sec": pre_slope_abs - post_slope_abs,
        "candidate_slope_reduction_ratio": slope_reduction_ratio_observed,

        "bic_improvement_vs_linear": bic_improvement_vs_linear,
        "bic_improvement_vs_constant": bic_improvement_vs_constant,

        "piecewise_supported": piecewise_supported,
        "post_slope_small": post_slope_small,
        "slope_reduced": slope_reduced,
        "pre_slope_meaningful": pre_slope_meaningful,

        "has_stabilizing_breakpoint": has_stabilizing_breakpoint,
        "stable_from_start": stable_from_start,
        "stable_or_stabilized": stable_or_stabilized,

        "nonconstant_model_supported": nonconstant_model_supported,
        "whole_trial_slope_dynamic": whole_trial_slope_dynamic,
        "candidate_post_slope_dynamic": candidate_post_slope_dynamic,
        "derivative_dynamic": derivative_dynamic,
        "late_derivative_dynamic": late_derivative_dynamic,
        "dynamically_updating": dynamically_updating,
        "ambiguous_or_noisy": ambiguous_or_noisy,
        "trial_dynamic_class": trial_dynamic_class,

        "n_pre_windows": best_pw["n_pre_windows"],
        "n_post_windows": best_pw["n_post_windows"],

        "mean_abs_offset_deg": offset_abs,
        "mean_signed_disparity_deg": mean_signed_disparity,
        "mean_abs_disparity_deg": mean_abs_disparity,

        # Candidate values are saved for diagnostics, even if not accepted.
        "candidate_breakpoint_capture_predicted": breakpoint_capture_predicted,
        "candidate_breakpoint_capture_observed_nearest": breakpoint_capture_observed_nearest,
        "candidate_plateau_capture_observed": plateau_capture_observed,
        "candidate_plateau_capture_predicted": plateau_capture_predicted,
        "candidate_plateau_capture_sd_observed": plateau_capture_sd_observed,
        "candidate_post_model_residual_sd": post_model_residual_sd,

        "candidate_breakpoint_anchor_shift_toward_visual_deg": candidate_breakpoint_anchor_shift_toward_visual_deg,
        "candidate_breakpoint_anchor_shift_abs_deg": abs(candidate_breakpoint_anchor_shift_toward_visual_deg),
        "candidate_plateau_anchor_shift_toward_visual_deg": candidate_plateau_anchor_shift_toward_visual_deg,
        "candidate_plateau_anchor_shift_abs_deg": abs(candidate_plateau_anchor_shift_toward_visual_deg),

        "candidate_breakpoint_anchor_shift_signed_physical_deg": candidate_breakpoint_anchor_shift_signed_physical_deg,
        "candidate_plateau_anchor_shift_signed_physical_deg": candidate_plateau_anchor_shift_signed_physical_deg,

        # Accepted stabilization values are NaN if the trial did not meet stabilization criteria.
        "stabilization_time_sec": accepted_value(knot),
        "pre_slope_capture_per_sec": accepted_value(pre_slope),
        "post_slope_capture_per_sec": accepted_value(post_slope),
        "slope_change_capture_per_sec": accepted_value(slope_change),
        "abs_pre_slope_capture_per_sec": accepted_value(pre_slope_abs),
        "abs_post_slope_capture_per_sec": accepted_value(post_slope_abs),
        "abs_slope_reduction_capture_per_sec": accepted_value(pre_slope_abs - post_slope_abs),
        "slope_reduction_ratio": accepted_value(slope_reduction_ratio_observed),

        "breakpoint_capture_predicted": accepted_value(breakpoint_capture_predicted),
        "breakpoint_capture_observed_nearest": accepted_value(breakpoint_capture_observed_nearest),
        "plateau_capture_observed": accepted_value(plateau_capture_observed),
        "plateau_capture_predicted": accepted_value(plateau_capture_predicted),
        "plateau_capture_sd_observed": accepted_value(plateau_capture_sd_observed),
        "post_model_residual_sd": accepted_value(post_model_residual_sd),

        "breakpoint_anchor_shift_toward_visual_deg": accepted_value(candidate_breakpoint_anchor_shift_toward_visual_deg),
        "breakpoint_anchor_shift_abs_deg": accepted_value(abs(candidate_breakpoint_anchor_shift_toward_visual_deg)),
        "plateau_anchor_shift_toward_visual_deg": accepted_value(candidate_plateau_anchor_shift_toward_visual_deg),
        "plateau_anchor_shift_abs_deg": accepted_value(abs(candidate_plateau_anchor_shift_toward_visual_deg)),

        "breakpoint_anchor_shift_signed_physical_deg": accepted_value(candidate_breakpoint_anchor_shift_signed_physical_deg),
        "plateau_anchor_shift_signed_physical_deg": accepted_value(candidate_plateau_anchor_shift_signed_physical_deg),

        # Dynamicity after candidate breakpoint.
        # These are useful especially for trials that do not meet stabilization criteria.
        "post_breakpoint_mean_abs_capture_derivative_per_sec": post_breakpoint_mean_abs_derivative,
        "post_breakpoint_median_abs_capture_derivative_per_sec": post_breakpoint_median_abs_derivative,
    }

    out.update(derivative_metrics)
    return out

# =========================
# STEP 1 — LOAD FIXED-OFFSET TRIALS AND COMPUTE WINDOW METRICS
# =========================
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in {DATA_DIR}")

window_rows = []
trial_lag_rows = []

window_starts = np.arange(0, TRIAL_DURATION_SEC - WINDOW_SEC + 1e-9, STEP_SEC)

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    df = pd.read_csv(fpath)

    required_cols = [
        "trialIndex", "trialType", "condLabel", "isPractice",
        "t", "mouse_x", "blob_x", "audio_x"
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"Skipping {subj}: missing {missing}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"],
        sort=False
    ):
        first_row = df_trial.iloc[0]
        row_series = pd.Series(first_row)

        if not is_fixed_offset_av_trial(row_series):
            continue

        offset = extract_offset_from_row(row_series)

        res = resample_trial(df_trial, TIME_GRID)
        if res is None:
            continue

        t = np.asarray(res["t_grid"], dtype=float)
        response_raw = np.asarray(res["mouse_x"], dtype=float)
        visual_x = np.asarray(res["blob_x"], dtype=float)
        audio_x = np.asarray(res["audio_x"], dtype=float)

        # Preserve original lag-correction logic for comparison.
        visual_lag_res = lag_scanned_corr(
            response=response_raw,
            stimulus=visual_x,
            fs=RESAMPLE_HZ,
            max_lag_sec=MAX_LAG_SEC
        )

        lag_sec = visual_lag_res["lag_sec"]
        response_lagcorr = apply_response_lag_correction(response_raw, t, lag_sec)

        trial_lag_rows.append({
            "subject": subj,
            "source_file": Path(fpath).name,
            "trialIndex": trial_index,
            "trialType": trial_type,
            "condLabel": cond_label,
            "offset_deg": offset,
            "visual_lag_sec": lag_sec,
            "visual_peak_corr": visual_lag_res["peak_corr"],
        })

        response_velocity_raw = np.gradient(response_raw, 1 / RESAMPLE_HZ)
        response_velocity_lagcorr = np.gradient(response_lagcorr, 1 / RESAMPLE_HZ)

        for start in window_starts:
            end = start + WINDOW_SEC
            center = start + WINDOW_SEC / 2

            mask = (t >= start) & (t < end)

            if mask.sum() < MIN_VALID_SAMPLES:
                continue

            raw_capture = visual_capture(
                response_raw[mask],
                visual_x[mask],
                audio_x[mask]
            )

            lagcorr_capture = visual_capture(
                response_lagcorr[mask],
                visual_x[mask],
                audio_x[mask]
            )

            raw_velocity_abs = window_mean(np.abs(response_velocity_raw[mask]))
            lagcorr_velocity_abs = window_mean(np.abs(response_velocity_lagcorr[mask]))

            raw_velocity_signed = window_mean(response_velocity_raw[mask])
            lagcorr_velocity_signed = window_mean(response_velocity_lagcorr[mask])

            raw_response_sd = window_sd(response_raw[mask])
            lagcorr_response_sd = window_sd(response_lagcorr[mask])

            disparity_signed = visual_x[mask] - audio_x[mask]
            disparity_abs = np.abs(disparity_signed)

            window_rows.append({
                "subject": subj,
                "source_file": Path(fpath).name,
                "trialIndex": trial_index,
                "trialType": trial_type,
                "condLabel": cond_label,
                "offset_deg": offset,

                "window_start_sec": start,
                "window_end_sec": end,
                "window_center_sec": center,
                "time_bin": get_time_bin(center),

                "visual_lag_sec": lag_sec,
                "visual_peak_corr": visual_lag_res["peak_corr"],

                "RawVisualCapture": raw_capture,
                "LagCorrectedVisualCapture": lagcorr_capture,

                "AbsResponseVelocity_raw": raw_velocity_abs,
                "AbsResponseVelocity_lagcorr": lagcorr_velocity_abs,

                "SignedResponseVelocity_raw": raw_velocity_signed,
                "SignedResponseVelocity_lagcorr": lagcorr_velocity_signed,

                "ResponseSD_raw": raw_response_sd,
                "ResponseSD_lagcorr": lagcorr_response_sd,

                "MeanDisparitySigned_deg": window_mean(disparity_signed),
                "MeanAbsDisparity_deg": window_mean(disparity_abs),
            })

window_df = pd.DataFrame(window_rows)
trial_lag_df = pd.DataFrame(trial_lag_rows)

if len(window_df) == 0:
    raise RuntimeError("No window-level metrics were created.")

window_csv = os.path.join(CSV_DIR, "anchor_window_level_fixed_offset_metrics.csv")
trial_lag_csv = os.path.join(CSV_DIR, "anchor_trial_level_visual_lag_metrics.csv")

window_df.to_csv(window_csv, index=False)
trial_lag_df.to_csv(trial_lag_csv, index=False)

print(f"Saved window-level metrics: {window_csv}")
print(f"Saved trial-level lag metrics: {trial_lag_csv}")
print(f"Subjects: {window_df['subject'].nunique()}")
print(f"Window rows: {len(window_df)}")

# =========================
# STEP 2 — DESCRIPTIVE TIME-BIN SUMMARIES
# =========================
metric_cols = [
    "RawVisualCapture",
    "LagCorrectedVisualCapture",
    "AbsResponseVelocity_raw",
    "AbsResponseVelocity_lagcorr",
    "ResponseSD_raw",
    "ResponseSD_lagcorr",
]

subject_offset_bin_df = (
    window_df
    .dropna(subset=["time_bin"])
    .groupby(["subject", "offset_deg", "time_bin"], as_index=False)
    .agg(
        n_windows=("window_center_sec", "count"),
        **{m: (m, "mean") for m in metric_cols}
    )
)

time_order = {"Early": 0, "Middle": 1, "Late": 2}
subject_offset_bin_df["time_bin_order"] = subject_offset_bin_df["time_bin"].map(time_order)
subject_offset_bin_df = subject_offset_bin_df.sort_values(["subject", "offset_deg", "time_bin_order"])

subject_offset_bin_csv = os.path.join(CSV_DIR, "anchor_subject_offset_timebin_metrics.csv")
subject_offset_bin_df.to_csv(subject_offset_bin_csv, index=False)

# Wide subject × offset descriptive table
wide_rows = []

for (subj, offset), sub in subject_offset_bin_df.groupby(["subject", "offset_deg"]):
    row = {
        "subject": subj,
        "offset_deg": offset,
    }

    for metric in metric_cols:
        for tb in ["Early", "Middle", "Late"]:
            vals = sub[sub["time_bin"] == tb][metric]
            row[f"{metric}_{tb}"] = vals.iloc[0] if len(vals) > 0 else np.nan

        early = row.get(f"{metric}_Early", np.nan)
        middle = row.get(f"{metric}_Middle", np.nan)
        late = row.get(f"{metric}_Late", np.nan)

        row[f"{metric}_MiddleMinusEarly"] = middle - early if np.isfinite(middle) and np.isfinite(early) else np.nan
        row[f"{metric}_LateMinusMiddle"] = late - middle if np.isfinite(late) and np.isfinite(middle) else np.nan
        row[f"{metric}_LateMinusEarly"] = late - early if np.isfinite(late) and np.isfinite(early) else np.nan

    wide_rows.append(row)

subject_offset_anchor_df = pd.DataFrame(wide_rows)

subject_offset_anchor_csv = os.path.join(CSV_DIR, "anchor_subject_offset_timebin_descriptive_metrics.csv")
subject_offset_anchor_df.to_csv(subject_offset_anchor_csv, index=False)

# =========================
# STEP 3 — TRIAL-LEVEL BREAKPOINT + DYNAMIC CAPTURE ANALYSIS
# =========================
breakpoint_rows = []

trial_group_cols = [
    "subject",
    "source_file",
    "trialIndex",
    "trialType",
    "condLabel",
    "offset_deg",
]

for keys, sub in window_df.groupby(trial_group_cols, dropna=False):
    subject, source_file, trial_index, trial_type, cond_label, offset_deg = keys

    base = {
        "subject": subject,
        "source_file": source_file,
        "trialIndex": trial_index,
        "trialType": trial_type,
        "condLabel": cond_label,
        "offset_deg": offset_deg,
    }

    for metric, metric_label in BREAKPOINT_METRICS:
        res = fit_stabilizing_breakpoint_for_trial(sub, metric=metric)

        row = base.copy()
        row.update(res)
        row["metric_label"] = metric_label

        breakpoint_rows.append(row)

breakpoint_trial_df = pd.DataFrame(breakpoint_rows)

# Make sure binary columns exist and are numeric.
binary_cols = [
    "piecewise_supported",
    "has_stabilizing_breakpoint",
    "stable_from_start",
    "stable_or_stabilized",
    "nonconstant_model_supported",
    "whole_trial_slope_dynamic",
    "candidate_post_slope_dynamic",
    "derivative_dynamic",
    "late_derivative_dynamic",
    "dynamically_updating",
    "ambiguous_or_noisy",
]

for col in binary_cols:
    if col not in breakpoint_trial_df.columns:
        breakpoint_trial_df[col] = 0
    breakpoint_trial_df[col] = breakpoint_trial_df[col].fillna(0).astype(int)

# Add non-stabilized-only and dynamically-updating-only versions of dynamicity metrics.
# These are used to quantify ongoing dynamics specifically in trials that did not stabilize.
dynamic_metric_cols = [
    "linear_slope_capture_per_sec",
    "whole_trial_abs_linear_slope_capture_per_sec",
    "candidate_post_slope_capture_per_sec",
    "candidate_abs_post_slope_capture_per_sec",
    "mean_abs_capture_derivative_per_sec",
    "median_abs_capture_derivative_per_sec",
    "early_mean_abs_capture_derivative_per_sec",
    "middle_mean_abs_capture_derivative_per_sec",
    "late_mean_abs_capture_derivative_per_sec",
    "late_minus_early_abs_derivative_per_sec",
    "post_breakpoint_mean_abs_capture_derivative_per_sec",
    "post_breakpoint_median_abs_capture_derivative_per_sec",
]

for col in dynamic_metric_cols:
    if col in breakpoint_trial_df.columns:
        breakpoint_trial_df[f"{col}_nonstabilized"] = np.where(
            breakpoint_trial_df["stable_or_stabilized"] == 0,
            breakpoint_trial_df[col],
            np.nan
        )

        breakpoint_trial_df[f"{col}_dynamic_trials"] = np.where(
            breakpoint_trial_df["dynamically_updating"] == 1,
            breakpoint_trial_df[col],
            np.nan
        )

breakpoint_trial_csv = os.path.join(CSV_DIR, "anchor_trial_level_breakpoint_DYNAMIC_metrics.csv")
breakpoint_trial_df.to_csv(breakpoint_trial_csv, index=False)

print(f"Saved trial-level breakpoint + dynamic metrics: {breakpoint_trial_csv}")

# =========================
# STEP 4 — SUBJECT × OFFSET BREAKPOINT + DYNAMIC SUMMARIES
# =========================
# These are the core repeated-measures summary rows.
# Each subject contributes one row per offset per metric.
agg_spec = dict(
    n_trials=("trialIndex", "count"),
    n_fit_ok=("status", lambda x: int(np.sum(pd.Series(x) == "fit_ok"))),

    n_piecewise_supported=("piecewise_supported", "sum"),
    n_stabilized_trials=("has_stabilizing_breakpoint", "sum"),
    n_stable_from_start_trials=("stable_from_start", "sum"),
    n_stable_or_stabilized_trials=("stable_or_stabilized", "sum"),
    n_nonconstant_trials=("nonconstant_model_supported", "sum"),
    n_dynamic_trials=("dynamically_updating", "sum"),
    n_ambiguous_or_noisy_trials=("ambiguous_or_noisy", "sum"),

    piecewise_supported_probability=("piecewise_supported", "mean"),
    stabilization_probability=("has_stabilizing_breakpoint", "mean"),
    stable_from_start_probability=("stable_from_start", "mean"),
    stable_or_stabilized_probability=("stable_or_stabilized", "mean"),
    nonconstant_model_probability=("nonconstant_model_supported", "mean"),
    dynamic_probability=("dynamically_updating", "mean"),
    ambiguous_or_noisy_probability=("ambiguous_or_noisy", "mean"),

    stabilization_time_sec=("stabilization_time_sec", "mean"),

    pre_slope_capture_per_sec=("pre_slope_capture_per_sec", "mean"),
    post_slope_capture_per_sec=("post_slope_capture_per_sec", "mean"),
    slope_change_capture_per_sec=("slope_change_capture_per_sec", "mean"),
    abs_pre_slope_capture_per_sec=("abs_pre_slope_capture_per_sec", "mean"),
    abs_post_slope_capture_per_sec=("abs_post_slope_capture_per_sec", "mean"),
    abs_slope_reduction_capture_per_sec=("abs_slope_reduction_capture_per_sec", "mean"),
    slope_reduction_ratio=("slope_reduction_ratio", "mean"),

    breakpoint_capture_predicted=("breakpoint_capture_predicted", "mean"),
    breakpoint_capture_observed_nearest=("breakpoint_capture_observed_nearest", "mean"),
    plateau_capture_observed=("plateau_capture_observed", "mean"),
    plateau_capture_predicted=("plateau_capture_predicted", "mean"),
    plateau_capture_sd_observed=("plateau_capture_sd_observed", "mean"),
    post_model_residual_sd=("post_model_residual_sd", "mean"),

    breakpoint_anchor_shift_toward_visual_deg=("breakpoint_anchor_shift_toward_visual_deg", "mean"),
    breakpoint_anchor_shift_abs_deg=("breakpoint_anchor_shift_abs_deg", "mean"),
    plateau_anchor_shift_toward_visual_deg=("plateau_anchor_shift_toward_visual_deg", "mean"),
    plateau_anchor_shift_abs_deg=("plateau_anchor_shift_abs_deg", "mean"),

    breakpoint_anchor_shift_signed_physical_deg=("breakpoint_anchor_shift_signed_physical_deg", "mean"),
    plateau_anchor_shift_signed_physical_deg=("plateau_anchor_shift_signed_physical_deg", "mean"),

    candidate_breakpoint_sec=("candidate_breakpoint_sec", "mean"),
    candidate_pre_slope_capture_per_sec=("candidate_pre_slope_capture_per_sec", "mean"),
    candidate_post_slope_capture_per_sec=("candidate_post_slope_capture_per_sec", "mean"),
    candidate_abs_post_slope_capture_per_sec=("candidate_abs_post_slope_capture_per_sec", "mean"),
    candidate_breakpoint_anchor_shift_toward_visual_deg=("candidate_breakpoint_anchor_shift_toward_visual_deg", "mean"),
    candidate_plateau_anchor_shift_toward_visual_deg=("candidate_plateau_anchor_shift_toward_visual_deg", "mean"),

    linear_slope_capture_per_sec=("linear_slope_capture_per_sec", "mean"),
    whole_trial_abs_linear_slope_capture_per_sec=("whole_trial_abs_linear_slope_capture_per_sec", "mean"),
    mean_abs_capture_derivative_per_sec=("mean_abs_capture_derivative_per_sec", "mean"),
    median_abs_capture_derivative_per_sec=("median_abs_capture_derivative_per_sec", "mean"),
    early_mean_abs_capture_derivative_per_sec=("early_mean_abs_capture_derivative_per_sec", "mean"),
    middle_mean_abs_capture_derivative_per_sec=("middle_mean_abs_capture_derivative_per_sec", "mean"),
    late_mean_abs_capture_derivative_per_sec=("late_mean_abs_capture_derivative_per_sec", "mean"),
    late_minus_early_abs_derivative_per_sec=("late_minus_early_abs_derivative_per_sec", "mean"),
    post_breakpoint_mean_abs_capture_derivative_per_sec=("post_breakpoint_mean_abs_capture_derivative_per_sec", "mean"),
)

# Add conditional dynamicity columns if available.
for col in dynamic_metric_cols:
    nonstab_col = f"{col}_nonstabilized"
    dyn_col = f"{col}_dynamic_trials"

    if nonstab_col in breakpoint_trial_df.columns:
        agg_spec[nonstab_col] = (nonstab_col, "mean")

    if dyn_col in breakpoint_trial_df.columns:
        agg_spec[dyn_col] = (dyn_col, "mean")

breakpoint_subject_offset_df = (
    breakpoint_trial_df
    .groupby(["metric", "metric_label", "subject", "offset_deg"], as_index=False)
    .agg(**agg_spec)
)

breakpoint_subject_offset_csv = os.path.join(CSV_DIR, "anchor_subject_offset_breakpoint_DYNAMIC_metrics.csv")
breakpoint_subject_offset_df.to_csv(breakpoint_subject_offset_csv, index=False)

# =========================
# STEP 5 — SUBJECT-LEVEL BREAKPOINT + DYNAMIC SUMMARIES
# =========================
# Average across offsets within subject first.
# This is the correct table for group-level one-sample tests.
breakpoint_subject_df = (
    breakpoint_subject_offset_df
    .groupby(["metric", "metric_label", "subject"], as_index=False)
    .mean(numeric_only=True)
)

breakpoint_subject_csv = os.path.join(CSV_DIR, "anchor_subject_level_breakpoint_DYNAMIC_metrics.csv")
breakpoint_subject_df.to_csv(breakpoint_subject_csv, index=False)

# =========================
# STEP 6 — GROUP-LEVEL ESTIMATES AND TESTS
# =========================
group_outcomes = [
    ("piecewise_supported_probability", "Probability that piecewise model improved over linear drift"),
    ("stabilization_probability", "Probability of accepted stabilizing breakpoint"),
    ("stable_from_start_probability", "Probability of stable-from-start trials"),
    ("stable_or_stabilized_probability", "Probability of stable-from-start or stabilized trials"),
    ("nonconstant_model_probability", "Probability of nonconstant model support"),
    ("dynamic_probability", "Probability of dynamically updating non-stabilized trials"),
    ("ambiguous_or_noisy_probability", "Probability of ambiguous/noisy non-stabilized trials"),

    ("stabilization_time_sec", "Time to accepted stabilizing breakpoint, seconds"),

    ("pre_slope_capture_per_sec", "Pre-breakpoint visual-capture slope"),
    ("post_slope_capture_per_sec", "Post-breakpoint visual-capture slope"),
    ("abs_pre_slope_capture_per_sec", "Absolute pre-breakpoint slope"),
    ("abs_post_slope_capture_per_sec", "Absolute post-breakpoint slope"),
    ("abs_slope_reduction_capture_per_sec", "Absolute slope reduction"),

    ("breakpoint_capture_predicted", "Predicted capture at breakpoint"),
    ("plateau_capture_observed", "Observed plateau capture after breakpoint"),
    ("plateau_capture_predicted", "Predicted plateau capture after breakpoint"),

    ("breakpoint_anchor_shift_toward_visual_deg", "Auditory-estimate shift toward vision at breakpoint, deg"),
    ("plateau_anchor_shift_toward_visual_deg", "Stabilized auditory-estimate shift toward vision, deg"),
    ("breakpoint_anchor_shift_abs_deg", "Absolute auditory-estimate shift at breakpoint, deg"),
    ("plateau_anchor_shift_abs_deg", "Absolute stabilized auditory-estimate shift, deg"),

    ("post_model_residual_sd", "Post-breakpoint model residual SD"),

    # Whole-trial dynamicity
    ("whole_trial_abs_linear_slope_capture_per_sec", "Whole-trial absolute linear capture slope"),
    ("mean_abs_capture_derivative_per_sec", "Mean absolute visual-capture derivative"),
    ("late_mean_abs_capture_derivative_per_sec", "Late mean absolute visual-capture derivative"),
    ("late_minus_early_abs_derivative_per_sec", "Late minus early absolute derivative"),

    # Non-stabilized-only dynamicity
    ("whole_trial_abs_linear_slope_capture_per_sec_nonstabilized", "Non-stabilized trials: whole-trial absolute linear capture slope"),
    ("mean_abs_capture_derivative_per_sec_nonstabilized", "Non-stabilized trials: mean absolute visual-capture derivative"),
    ("late_mean_abs_capture_derivative_per_sec_nonstabilized", "Non-stabilized trials: late absolute visual-capture derivative"),
    ("post_breakpoint_mean_abs_capture_derivative_per_sec_nonstabilized", "Non-stabilized trials: post-candidate-breakpoint absolute derivative"),
    ("candidate_abs_post_slope_capture_per_sec_nonstabilized", "Non-stabilized trials: absolute candidate post-breakpoint slope"),

    # Dynamically-updating-only dynamicity
    ("whole_trial_abs_linear_slope_capture_per_sec_dynamic_trials", "Dynamic trials: whole-trial absolute linear capture slope"),
    ("mean_abs_capture_derivative_per_sec_dynamic_trials", "Dynamic trials: mean absolute visual-capture derivative"),
    ("late_mean_abs_capture_derivative_per_sec_dynamic_trials", "Dynamic trials: late absolute visual-capture derivative"),
]

group_estimate_rows = []

for (metric, metric_label), sub in breakpoint_subject_df.groupby(["metric", "metric_label"]):
    for outcome, label in group_outcomes:
        if outcome not in sub.columns:
            continue

        vals = sub[outcome].replace([np.inf, -np.inf], np.nan).dropna()
        m, lo, hi, n = mean_ci(vals)

        group_estimate_rows.append({
            "metric": metric,
            "metric_label": metric_label,
            "outcome": outcome,
            "label": label,
            "n_subjects": n,
            "mean": m,
            "ci_low": lo,
            "ci_high": hi,
        })

group_estimate_df = pd.DataFrame(group_estimate_rows)

group_estimate_csv = os.path.join(CSV_DIR, "anchor_breakpoint_DYNAMIC_group_estimates.csv")
group_estimate_df.to_csv(group_estimate_csv, index=False)

# Group tests on subject-level summaries
group_test_rows = []

test_vs_zero_outcomes = [
    ("pre_slope_capture_per_sec", "Pre-breakpoint visual-capture slope vs 0", 0.0),
    ("post_slope_capture_per_sec", "Post-breakpoint visual-capture slope vs 0", 0.0),
    ("breakpoint_capture_predicted", "Predicted capture at breakpoint vs 0", 0.0),
    ("plateau_capture_observed", "Observed plateau capture vs 0", 0.0),
    ("breakpoint_anchor_shift_toward_visual_deg", "Auditory-estimate shift at breakpoint vs 0 deg", 0.0),
    ("plateau_anchor_shift_toward_visual_deg", "Stabilized auditory-estimate shift vs 0 deg", 0.0),
    ("post_model_residual_sd", "Post-breakpoint residual SD vs 0", 0.0),
]

dynamic_greater_than_threshold_outcomes = [
    (
        "whole_trial_abs_linear_slope_capture_per_sec_nonstabilized",
        "Non-stabilized trials: whole-trial absolute linear slope > dynamic threshold",
        DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC
    ),
    (
        "mean_abs_capture_derivative_per_sec_nonstabilized",
        "Non-stabilized trials: mean absolute capture derivative > dynamic threshold",
        DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC
    ),
    (
        "late_mean_abs_capture_derivative_per_sec_nonstabilized",
        "Non-stabilized trials: late absolute capture derivative > dynamic threshold",
        DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC
    ),
    (
        "candidate_abs_post_slope_capture_per_sec_nonstabilized",
        "Non-stabilized trials: candidate absolute post-breakpoint slope > dynamic threshold",
        DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC
    ),
]

for (metric, metric_label), sub in breakpoint_subject_df.groupby(["metric", "metric_label"]):
    for outcome, label, popmean in test_vs_zero_outcomes:
        if outcome not in sub.columns:
            continue

        stats = one_sample_test(sub[outcome], popmean=popmean)

        row = {
            "analysis": "subject_level_one_sample_test",
            "metric": metric,
            "metric_label": metric_label,
            "outcome": outcome,
            "label": label,
            "popmean": popmean,
        }
        row.update(stats)
        group_test_rows.append(row)

    # Dynamicity tests: one-sided greater-than-threshold tests.
    for outcome, label, threshold in dynamic_greater_than_threshold_outcomes:
        if outcome not in sub.columns:
            continue

        stats = one_sample_greater_test(sub[outcome], threshold=threshold)

        row = {
            "analysis": "subject_level_one_sided_dynamicity_test",
            "metric": metric,
            "metric_label": metric_label,
            "outcome": outcome,
            "label": label,
            "threshold": threshold,
        }
        row.update(stats)
        group_test_rows.append(row)

    # Paired test: absolute pre-slope vs absolute post-slope for stabilized trials.
    if (
        "abs_pre_slope_capture_per_sec" in sub.columns
        and "abs_post_slope_capture_per_sec" in sub.columns
    ):
        stats = paired_test(
            sub["abs_pre_slope_capture_per_sec"],
            sub["abs_post_slope_capture_per_sec"]
        )

        group_test_rows.append({
            "analysis": "subject_level_paired_test",
            "metric": metric,
            "metric_label": metric_label,
            "outcome": "abs_pre_slope_vs_abs_post_slope",
            "label": "Absolute pre-breakpoint slope vs absolute post-breakpoint slope",
            **stats,
        })

    # Equivalence test: post-breakpoint slope is small enough to count as stable.
    if "post_slope_capture_per_sec" in sub.columns:
        eq = tost_equivalence_test(
            sub["post_slope_capture_per_sec"],
            lower_bound=-POST_SLOPE_EQUIVALENCE_BOUND,
            upper_bound=POST_SLOPE_EQUIVALENCE_BOUND,
            alpha=0.05
        )

        group_test_rows.append({
            "analysis": "subject_level_TOST_equivalence",
            "metric": metric,
            "metric_label": metric_label,
            "outcome": "post_slope_capture_per_sec",
            "label": f"Post-breakpoint slope equivalent to 0 within ±{POST_SLOPE_EQUIVALENCE_BOUND} capture/sec",
            **eq,
        })

group_test_df = pd.DataFrame(group_test_rows)

group_test_csv = os.path.join(CSV_DIR, "anchor_breakpoint_DYNAMIC_group_tests.csv")
group_test_df.to_csv(group_test_csv, index=False)

# =========================
# STEP 7 — OFFSET MODELS
# =========================
# These use subject × offset summaries and cluster SE by subject.
offset_model_rows = []

offset_model_outcomes = [
    "piecewise_supported_probability",
    "stabilization_probability",
    "stable_from_start_probability",
    "stable_or_stabilized_probability",
    "nonconstant_model_probability",
    "dynamic_probability",
    "ambiguous_or_noisy_probability",

    "stabilization_time_sec",
    "pre_slope_capture_per_sec",
    "post_slope_capture_per_sec",
    "abs_pre_slope_capture_per_sec",
    "abs_post_slope_capture_per_sec",
    "abs_slope_reduction_capture_per_sec",
    "breakpoint_capture_predicted",
    "plateau_capture_observed",
    "breakpoint_anchor_shift_toward_visual_deg",
    "plateau_anchor_shift_toward_visual_deg",
    "post_model_residual_sd",

    "whole_trial_abs_linear_slope_capture_per_sec",
    "mean_abs_capture_derivative_per_sec",
    "late_mean_abs_capture_derivative_per_sec",
    "late_minus_early_abs_derivative_per_sec",

    "whole_trial_abs_linear_slope_capture_per_sec_nonstabilized",
    "mean_abs_capture_derivative_per_sec_nonstabilized",
    "late_mean_abs_capture_derivative_per_sec_nonstabilized",
    "post_breakpoint_mean_abs_capture_derivative_per_sec_nonstabilized",
    "candidate_abs_post_slope_capture_per_sec_nonstabilized",

    "whole_trial_abs_linear_slope_capture_per_sec_dynamic_trials",
    "mean_abs_capture_derivative_per_sec_dynamic_trials",
    "late_mean_abs_capture_derivative_per_sec_dynamic_trials",
]

for (metric, metric_label), sub_metric in breakpoint_subject_offset_df.groupby(["metric", "metric_label"]):
    for outcome in offset_model_outcomes:
        if outcome not in sub_metric.columns:
            continue

        d = sub_metric[["subject", "offset_deg", outcome]].replace([np.inf, -np.inf], np.nan).dropna().copy()

        if len(d) < 20 or d["subject"].nunique() < 3 or d["offset_deg"].nunique() < 2:
            continue

        d["offset_c"] = d["offset_deg"] - d["offset_deg"].mean()

        try:
            fit = smf.ols(
                f"{outcome} ~ offset_c",
                data=d
            ).fit(
                cov_type="cluster",
                cov_kwds={"groups": d["subject"]}
            )
        except Exception as e:
            offset_model_rows.append({
                "metric": metric,
                "metric_label": metric_label,
                "outcome": outcome,
                "analysis": "offset_model_failed",
                "error": str(e),
            })
            continue

        for term in fit.params.index:
            offset_model_rows.append({
                "analysis": "subject_offset_OLS_clustered_by_subject",
                "metric": metric,
                "metric_label": metric_label,
                "outcome": outcome,
                "term": term,
                "n_rows": len(d),
                "n_subjects": d["subject"].nunique(),
                "estimate": fit.params[term],
                "se": fit.bse[term],
                "t": fit.tvalues[term],
                "p": fit.pvalues[term],
                "r_squared": fit.rsquared,
            })

offset_model_df = pd.DataFrame(offset_model_rows)

offset_model_csv = os.path.join(CSV_DIR, "anchor_breakpoint_DYNAMIC_offset_model_results.csv")
offset_model_df.to_csv(offset_model_csv, index=False)

# =========================
# STEP 8 — PLOTS
# =========================
def summarize_offset_for_plot(df, metric, outcome):
    if outcome not in df.columns:
        return pd.DataFrame()

    d = df[
        (df["metric"] == metric)
        & np.isfinite(df[outcome])
    ].copy()

    if len(d) == 0:
        return pd.DataFrame()

    return (
        d
        .groupby("offset_deg", as_index=False)
        .agg(
            mean=(outcome, "mean"),
            se=(outcome, sem),
            n=("subject", "nunique")
        )
        .sort_values("offset_deg")
    )

def plot_offset_outcome(df, metric, metric_label, outcome, ylabel, title, filename, ylim=None, hlines=None):
    summary = summarize_offset_for_plot(df, metric, outcome)

    if len(summary) < 2:
        return

    fig, ax = plt.subplots(figsize=(7.4, 5.2))

    ax.errorbar(
        summary["offset_deg"],
        summary["mean"],
        yerr=summary["se"],
        marker="o",
        linewidth=2.2,
        capsize=4
    )

    ax.set_xlabel("A-V offset magnitude (deg)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    if ylim is not None:
        ax.set_ylim(ylim)

    if hlines is not None:
        for yval, linestyle, alpha in hlines:
            ax.axhline(yval, linestyle=linestyle, linewidth=1, alpha=alpha)

    fig.tight_layout()
    fig_path = os.path.join(FIG_DIR, filename)
    fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# A. Group time-course plots by offset with average accepted breakpoint overlay.
for metric, metric_label in BREAKPOINT_METRICS:
    group_tc = (
        window_df
        .groupby(["offset_deg", "window_center_sec"], as_index=False)
        .agg(
            mean=(metric, "mean"),
            se=(metric, sem),
            n=("subject", "nunique")
        )
    )

    bp_summary = (
        breakpoint_subject_offset_df[
            (breakpoint_subject_offset_df["metric"] == metric)
            & np.isfinite(breakpoint_subject_offset_df["stabilization_time_sec"])
        ]
        .groupby("offset_deg", as_index=False)
        .agg(
            mean_breakpoint=("stabilization_time_sec", "mean"),
            se_breakpoint=("stabilization_time_sec", sem),
            n=("subject", "nunique")
        )
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), sharex=True, sharey=True)
    axes = axes.flatten()

    for ax, offset in zip(axes, FIXED_OFFSETS):
        sub = group_tc[group_tc["offset_deg"] == offset].copy()

        if len(sub) == 0:
            continue

        ax.plot(
            sub["window_center_sec"],
            sub["mean"],
            linewidth=2.2
        )

        ax.fill_between(
            sub["window_center_sec"],
            sub["mean"] - sub["se"],
            sub["mean"] + sub["se"],
            alpha=0.20
        )

        bp = bp_summary[bp_summary["offset_deg"] == offset]

        if len(bp) > 0 and np.isfinite(bp["mean_breakpoint"].iloc[0]):
            mean_bp = bp["mean_breakpoint"].iloc[0]
            ax.axvline(
                mean_bp,
                linestyle="-",
                linewidth=2.0,
                alpha=0.8,
                label="Mean accepted breakpoint"
            )

        ax.axhline(0.0, linestyle="--", linewidth=1, alpha=0.6)
        ax.axhline(0.5, linestyle="--", linewidth=1, alpha=0.5)
        ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.5)

        ax.set_title(f"Offset {offset}°")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(metric_label)

    handles, labels = axes[0].get_legend_handles_labels()
    if len(handles) > 0:
        fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)

    fig.suptitle(f"Visual capture over time with accepted stabilization breakpoint: {metric_label}", fontsize=15)
    fig.tight_layout(rect=[0, 0.04, 1, 0.95])

    fig_path = os.path.join(FIG_DIR, f"timecourse_with_breakpoint_{metric}.png")
    fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# B. Breakpoint and anchor outcome plots.
for metric, metric_label in BREAKPOINT_METRICS:
    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "stabilization_probability",
        "Proportion of trials",
        f"Probability of accepted stabilizing breakpoint: {metric_label}",
        f"breakpoint_probability_by_offset_{metric}.png",
        ylim=(-0.05, 1.05)
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "stable_or_stabilized_probability",
        "Proportion of trials",
        f"Probability of stable or stabilized trials: {metric_label}",
        f"stable_or_stabilized_probability_by_offset_{metric}.png",
        ylim=(-0.05, 1.05)
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "dynamic_probability",
        "Proportion of trials",
        f"Probability of dynamically updating non-stabilized trials: {metric_label}",
        f"dynamic_probability_by_offset_{metric}.png",
        ylim=(-0.05, 1.05)
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "ambiguous_or_noisy_probability",
        "Proportion of trials",
        f"Probability of ambiguous/noisy non-stabilized trials: {metric_label}",
        f"ambiguous_probability_by_offset_{metric}.png",
        ylim=(-0.05, 1.05)
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "stabilization_time_sec",
        "Time to accepted breakpoint (s)",
        f"Time to stabilization by offset: {metric_label}",
        f"stabilization_time_by_offset_{metric}.png"
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "breakpoint_anchor_shift_toward_visual_deg",
        "Auditory-estimate shift toward vision (deg)",
        f"Auditory anchor location at breakpoint: {metric_label}",
        f"breakpoint_anchor_shift_deg_by_offset_{metric}.png"
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "plateau_anchor_shift_toward_visual_deg",
        "Stabilized auditory-estimate shift toward vision (deg)",
        f"Stabilized auditory anchor location: {metric_label}",
        f"plateau_anchor_shift_deg_by_offset_{metric}.png"
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "breakpoint_capture_predicted",
        "Visual capture at breakpoint",
        f"Visual capture at accepted breakpoint: {metric_label}",
        f"breakpoint_capture_by_offset_{metric}.png",
        hlines=[(0, "--", 0.6), (0.5, "--", 0.5), (1, "--", 0.5)]
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "plateau_capture_observed",
        "Observed plateau visual capture",
        f"Observed plateau capture after breakpoint: {metric_label}",
        f"plateau_capture_by_offset_{metric}.png",
        hlines=[(0, "--", 0.6), (0.5, "--", 0.5), (1, "--", 0.5)]
    )

# C. Dynamicity plots for non-stabilized trials.
for metric, metric_label in BREAKPOINT_METRICS:
    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "mean_abs_capture_derivative_per_sec_nonstabilized",
        "Mean |Δ capture / Δ time|",
        f"Dynamicity of non-stabilized trials: {metric_label}",
        f"nonstabilized_mean_abs_derivative_by_offset_{metric}.png",
        hlines=[(DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC, "--", 0.7)]
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "late_mean_abs_capture_derivative_per_sec_nonstabilized",
        "Late mean |Δ capture / Δ time|",
        f"Late dynamicity of non-stabilized trials: {metric_label}",
        f"nonstabilized_late_abs_derivative_by_offset_{metric}.png",
        hlines=[(DYNAMIC_DERIVATIVE_ABS_MIN_CAPTURE_PER_SEC, "--", 0.7)]
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "whole_trial_abs_linear_slope_capture_per_sec_nonstabilized",
        "Absolute whole-trial slope",
        f"Whole-trial slope of non-stabilized trials: {metric_label}",
        f"nonstabilized_whole_trial_abs_slope_by_offset_{metric}.png",
        hlines=[(DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC, "--", 0.7)]
    )

    plot_offset_outcome(
        breakpoint_subject_offset_df,
        metric,
        metric_label,
        "candidate_abs_post_slope_capture_per_sec_nonstabilized",
        "Absolute candidate post-breakpoint slope",
        f"Residual drift in non-stabilized trials: {metric_label}",
        f"nonstabilized_candidate_abs_post_slope_by_offset_{metric}.png",
        hlines=[(DYNAMIC_SLOPE_ABS_MIN_CAPTURE_PER_SEC, "--", 0.7)]
    )

# D. Trial-classification probability plot.
# Shows stable-from-start, stabilized, dynamic, and ambiguous trial proportions.
for metric, metric_label in BREAKPOINT_METRICS:
    d = breakpoint_subject_offset_df[
        breakpoint_subject_offset_df["metric"] == metric
    ].copy()

    needed = [
        "stable_from_start_probability",
        "stabilization_probability",
        "dynamic_probability",
        "ambiguous_or_noisy_probability",
    ]

    if not all(col in d.columns for col in needed):
        continue

    class_long = pd.concat([
        d[["subject", "offset_deg", "stable_from_start_probability"]]
        .rename(columns={"stable_from_start_probability": "probability"})
        .assign(classification="Stable from start"),

        d[["subject", "offset_deg", "stabilization_probability"]]
        .rename(columns={"stabilization_probability": "probability"})
        .assign(classification="Stabilized after breakpoint"),

        d[["subject", "offset_deg", "dynamic_probability"]]
        .rename(columns={"dynamic_probability": "probability"})
        .assign(classification="Dynamically updating"),

        d[["subject", "offset_deg", "ambiguous_or_noisy_probability"]]
        .rename(columns={"ambiguous_or_noisy_probability": "probability"})
        .assign(classification="Ambiguous/noisy"),
    ], ignore_index=True)

    class_summary = (
        class_long
        .groupby(["offset_deg", "classification"], as_index=False)
        .agg(
            mean=("probability", "mean"),
            se=("probability", sem),
            n=("subject", "nunique")
        )
        .sort_values(["offset_deg", "classification"])
    )

    fig, ax = plt.subplots(figsize=(9.2, 5.8))

    for classification in [
        "Stable from start",
        "Stabilized after breakpoint",
        "Dynamically updating",
        "Ambiguous/noisy",
    ]:
        sub = class_summary[class_summary["classification"] == classification].sort_values("offset_deg")

        if len(sub) == 0:
            continue

        ax.errorbar(
            sub["offset_deg"],
            sub["mean"],
            yerr=sub["se"],
            marker="o",
            linewidth=2.2,
            capsize=4,
            label=classification
        )

    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("A-V offset magnitude (deg)")
    ax.set_ylabel("Proportion of trials")
    ax.set_title(f"Trial dynamic classification by offset: {metric_label}")
    ax.legend(frameon=False)

    fig.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"trial_dynamic_classification_by_offset_{metric}.png")
    fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# E. Pre-vs-post slope plot by offset for stabilized trials.
for metric, metric_label in BREAKPOINT_METRICS:
    d = breakpoint_subject_offset_df[
        (breakpoint_subject_offset_df["metric"] == metric)
    ].copy()

    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=["pre_slope_capture_per_sec", "post_slope_capture_per_sec"])

    if len(d) < 5:
        continue

    long_d = pd.concat([
        d[["subject", "offset_deg", "pre_slope_capture_per_sec"]]
        .rename(columns={"pre_slope_capture_per_sec": "slope"})
        .assign(phase="Pre-breakpoint"),

        d[["subject", "offset_deg", "post_slope_capture_per_sec"]]
        .rename(columns={"post_slope_capture_per_sec": "slope"})
        .assign(phase="Post-breakpoint"),
    ], ignore_index=True)

    slope_summary = (
        long_d
        .groupby(["offset_deg", "phase"], as_index=False)
        .agg(
            mean=("slope", "mean"),
            se=("slope", sem),
            n=("subject", "nunique")
        )
    )

    fig, ax = plt.subplots(figsize=(8.2, 5.6))

    for phase in ["Pre-breakpoint", "Post-breakpoint"]:
        sub = slope_summary[slope_summary["phase"] == phase].sort_values("offset_deg")

        ax.errorbar(
            sub["offset_deg"],
            sub["mean"],
            yerr=sub["se"],
            marker="o",
            linewidth=2.2,
            capsize=4,
            label=phase
        )

    ax.axhline(0, linestyle="--", linewidth=1, alpha=0.7)
    ax.set_xlabel("A-V offset magnitude (deg)")
    ax.set_ylabel("Visual-capture slope (capture/sec)")
    ax.set_title(f"Pre- vs post-breakpoint visual-capture slopes: {metric_label}")
    ax.legend(frameon=False)

    fig.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"pre_post_capture_slopes_by_offset_{metric}.png")
    fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# F. Descriptive early/middle/late plots retained.
plot_metrics = [
    ("RawVisualCapture", "Raw visual capture"),
    ("LagCorrectedVisualCapture", "Lag-corrected visual capture"),
    ("AbsResponseVelocity_raw", "Raw absolute response velocity"),
    ("AbsResponseVelocity_lagcorr", "Lag-corrected absolute response velocity"),
]

for metric, ylabel in plot_metrics:
    bin_group = (
        subject_offset_bin_df
        .groupby(["offset_deg", "time_bin", "time_bin_order"], as_index=False)
        .agg(
            mean=(metric, "mean"),
            se=(metric, sem),
            n=("subject", "nunique")
        )
        .sort_values(["offset_deg", "time_bin_order"])
    )

    if len(bin_group) == 0:
        continue

    fig, ax = plt.subplots(figsize=(8.4, 5.8))

    for offset in FIXED_OFFSETS:
        sub = bin_group[bin_group["offset_deg"] == offset].copy()

        if len(sub) == 0:
            continue

        ax.errorbar(
            sub["time_bin_order"],
            sub["mean"],
            yerr=sub["se"],
            marker="o",
            linewidth=2.2,
            capsize=4,
            label=f"{offset}°"
        )

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["Early", "Middle", "Late"])

    if "Capture" in metric:
        ax.axhline(0.0, linestyle="--", linewidth=1, alpha=0.5)
        ax.axhline(0.5, linestyle="--", linewidth=1, alpha=0.6)
        ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.5)

    ax.set_xlabel("Trial phase")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Descriptive early → middle → late pattern: {ylabel}")
    ax.legend(title="Offset", frameon=False)

    fig.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"descriptive_early_middle_late_{metric}.png")
    fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# =========================
# FINAL PRINT SUMMARY
# =========================
print("\n============================================================")
print("CHINESE RESTAURANT / AUDITORY-ANCHOR BREAKPOINT + DYNAMIC CAPTURE ANALYSIS COMPLETE")
print("============================================================")
print(f"Window-level metrics:                         {window_csv}")
print(f"Trial-level lag metrics:                      {trial_lag_csv}")
print(f"Subject-offset descriptive time bins:         {subject_offset_bin_csv}")
print(f"Subject-offset descriptive metrics:           {subject_offset_anchor_csv}")
print(f"Trial-level breakpoint + dynamic metrics:     {breakpoint_trial_csv}")
print(f"Subject-offset breakpoint + dynamic metrics:  {breakpoint_subject_offset_csv}")
print(f"Subject-level breakpoint + dynamic metrics:   {breakpoint_subject_csv}")
print(f"Group-level estimates:                        {group_estimate_csv}")
print(f"Group-level tests:                            {group_test_csv}")
print(f"Offset model results:                         {offset_model_csv}")
print(f"Figures folder:                               {FIG_DIR}")

print("\n============================================================")
print("KEY GROUP ESTIMATES")
print("============================================================")
if len(group_estimate_df) > 0:
    print(group_estimate_df.to_string(index=False))

print("\n============================================================")
print("KEY GROUP TESTS")
print("============================================================")
if len(group_test_df) > 0:
    print(group_test_df.to_string(index=False))

print("\n============================================================")
print("OFFSET MODELS")
print("============================================================")
if len(offset_model_df) > 0:
    print(offset_model_df.to_string(index=False))

print("\n============================================================")
print("INTERPRETATION GUIDE")
print("============================================================")
print("- has_stabilizing_breakpoint = 1 only if the piecewise model beats linear drift and the post-breakpoint slope is small.")
print("- stabilization_time_sec = estimated time at which visual capture transitions into a stable regime.")
print("- pre_slope_capture_per_sec = how quickly visual capture changes before the breakpoint.")
print("- post_slope_capture_per_sec = residual drift after the breakpoint; values near zero support stabilization.")
print("- TOST equivalence tests ask whether the post-breakpoint slope is statistically equivalent to zero.")
print("- breakpoint_capture_predicted = estimated visual capture at the accepted breakpoint.")
print("- plateau_capture_observed = average observed visual capture after the accepted breakpoint.")
print("- breakpoint_anchor_shift_toward_visual_deg = capture at breakpoint × offset magnitude.")
print("- plateau_anchor_shift_toward_visual_deg = stabilized capture × offset magnitude.")
print("- Example: plateau capture = 0.25 at a 40° offset means the auditory estimate stabilized 10° toward the visual stimulus.")
print("- stable_from_start marks trials that were already approximately flat, rather than trials with a transition breakpoint.")
print("- dynamically_updating marks non-stabilized trials that show evidence of continued non-flat capture dynamics.")
print("- dynamic_probability = proportion of trials classified as dynamically updating rather than stable/stabilized.")
print("- mean_abs_capture_derivative_per_sec_nonstabilized quantifies ongoing capture change among trials that did not stabilize.")
print("- late_mean_abs_capture_derivative_per_sec_nonstabilized tests whether non-stabilized trials remain dynamic late in the trial.")
print("- ambiguous_or_noisy marks trials that were neither stable/stabilized nor clearly dynamically updating.")
print("- Trials with no accepted breakpoint are not forced into the stabilization-time estimate; their stabilization_time_sec is NaN.")

Saved window-level metrics: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\06_stabilization_breakpoint\csv_outputs\anchor_window_level_fixed_offset_metrics.csv
Saved trial-level lag metrics: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\06_stabilization_breakpoint\csv_outputs\anchor_trial_level_visual_lag_metrics.csv
Subjects: 50
Window rows: 276000
Saved trial-level breakpoint + dynamic metrics: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\06_stabilization_breakpoint\csv_outputs\anchor_trial_level_breakpoint_DYNAMIC_metrics.csv

CHINESE RESTAURANT / AUDITORY-ANCHOR BREAKPOINT + DYNAMIC CAPTURE ANALYSIS COMPLETE
Window-level metrics:                         C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\06_stabilization_breakpoint\csv_outputs\anchor_window_level_fixed_offset_metrics.csv
Trial-level lag metrics

In [29]:
# %% MOTOR-RESIDUALIZED SPATIAL-LOCALIZATION BREAKPOINT ANALYSIS
# =============================================================================
# Run this cell AFTER the original
# "CHINESE RESTAURANT / AUDITORY-ANCHOR BREAKPOINT + DYNAMIC CAPTURE ANALYSIS".
#
# Goal:
#   Re-run the SAME SL breakpoint/dynamicity algorithm after removing
#   within-trial visual-capture variance linearly associated with:
#       1) signed lag-corrected cursor velocity
#       2) absolute lag-corrected cursor velocity
#
# Motor predictors are centered within trial and only their predicted
# fluctuation is removed. The fitted intercept/mean capture level is retained,
# so the adjusted trajectory stays in visual-capture units and the ORIGINAL
# breakpoint thresholds can be used unchanged.
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp, ttest_rel, t as t_dist


# -----------------------------------------------------------------------------
# 1. REQUIRE THE ORIGINAL BREAKPOINT CELL
# -----------------------------------------------------------------------------

_required = [
    "fit_stabilizing_breakpoint_for_trial",
    "CAPTURE_FIT_MIN",
    "CAPTURE_FIT_MAX",
    "MIN_TOTAL_WINDOWS_FOR_BREAKPOINT",
]

_missing = [
    x for x in _required
    if x not in globals()
]

if _missing:
    raise RuntimeError(
        "Run the original SL breakpoint analysis cell first. Missing:\n"
        + "\n".join(_missing)
    )


# -----------------------------------------------------------------------------
# 2. PATHS / SETTINGS
# -----------------------------------------------------------------------------

SPATIAL_ANALYSES_ROOT_MOTOR = (
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1\spatial_analyses"
)

ORIGINAL_BP_DIR_MOTOR = os.path.join(
    SPATIAL_ANALYSES_ROOT_MOTOR,
    "06_stabilization_breakpoint"
)

ORIGINAL_CSV_DIR_MOTOR = os.path.join(
    ORIGINAL_BP_DIR_MOTOR,
    "csv_outputs"
)

ORIGINAL_WINDOW_CSV_MOTOR = os.path.join(
    ORIGINAL_CSV_DIR_MOTOR,
    "anchor_window_level_fixed_offset_metrics.csv"
)

ORIGINAL_TRIAL_BP_CSV_MOTOR = os.path.join(
    ORIGINAL_CSV_DIR_MOTOR,
    "anchor_trial_level_breakpoint_DYNAMIC_metrics.csv"
)


OUT_DIR_MOTOR = os.path.join(
    SPATIAL_ANALYSES_ROOT_MOTOR,
    "07_motor_residualized_stabilization_breakpoint"
)

CSV_DIR_MOTOR = os.path.join(
    OUT_DIR_MOTOR,
    "csv_outputs"
)

FIG_DIR_MOTOR = os.path.join(
    OUT_DIR_MOTOR,
    "figures"
)

for _d in [
    OUT_DIR_MOTOR,
    CSV_DIR_MOTOR,
    FIG_DIR_MOTOR,
]:
    os.makedirs(
        _d,
        exist_ok=True
    )


ORIGINAL_CAPTURE_METRIC_MOTOR = (
    "LagCorrectedVisualCapture"
)

MOTOR_RESID_METRIC = (
    "MotorResidualizedLagCorrectedVisualCapture"
)

MOTOR_RESID_METRIC_LABEL = (
    "Lag-corrected visual capture residualized "
    "for signed and absolute cursor velocity"
)

MOTOR_PREDICTORS = [
    "SignedResponseVelocity_lagcorr",
    "AbsResponseVelocity_lagcorr",
]

# Same minimum as the breakpoint analysis.
MIN_WINDOWS_FOR_MOTOR_MODEL = (
    MIN_TOTAL_WINDOWS_FOR_BREAKPOINT
)

FIG_DPI_MOTOR = 180


# -----------------------------------------------------------------------------
# 3. HELPERS
# -----------------------------------------------------------------------------

def motor_mean_ci(
    vals,
    alpha=.05,
):

    vals = (
        pd.Series(vals)
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
        .astype(float)
        .values
    )

    n = len(vals)

    if n < 2:
        return (
            np.nan,
            np.nan,
            np.nan,
            n,
        )

    m = float(
        np.mean(vals)
    )

    se = float(
        np.std(
            vals,
            ddof=1
        )
        /
        np.sqrt(n)
    )

    crit = float(
        t_dist.ppf(
            1 - alpha / 2,
            df=n - 1
        )
    )

    return (
        m,
        m - crit * se,
        m + crit * se,
        n,
    )


def motor_sem(
    vals,
):

    vals = (
        pd.Series(vals)
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
        .astype(float)
        .values
    )

    if len(vals) < 2:
        return np.nan

    return float(
        np.std(
            vals,
            ddof=1
        )
        /
        np.sqrt(
            len(vals)
        )
    )


def motor_one_sample(
    vals,
    popmean=0.,
    alternative="two-sided",
):

    vals = (
        pd.Series(vals)
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
        .astype(float)
        .values
    )

    n = len(vals)

    if n < 2:

        return dict(
            n=n,
            mean=np.nan,
            ci95_low=np.nan,
            ci95_high=np.nan,
            t=np.nan,
            df=np.nan,
            p=np.nan,
            dz=np.nan,
        )

    tt = ttest_1samp(
        vals,
        popmean
    )

    tval = float(
        tt.statistic
    )

    p2 = float(
        tt.pvalue
    )

    if alternative == "greater":

        p = (
            p2 / 2
            if tval >= 0
            else
            1 - p2 / 2
        )

    elif alternative == "less":

        p = (
            p2 / 2
            if tval <= 0
            else
            1 - p2 / 2
        )

    else:

        p = p2

    m, lo, hi, _ = (
        motor_mean_ci(
            vals
        )
    )

    sd = np.std(
        vals,
        ddof=1
    )

    dz = (
        (
            np.mean(vals)
            -
            popmean
        )
        /
        sd
        if sd > 0
        else np.nan
    )

    return dict(
        n=n,
        mean=m,
        ci95_low=lo,
        ci95_high=hi,
        t=tval,
        df=n - 1,
        p=float(p),
        dz=(
            float(dz)
            if np.isfinite(dz)
            else np.nan
        ),
    )


def motor_paired(
    original,
    residualized,
):

    d = pd.DataFrame(
        {
            "original": original,
            "residualized": residualized,
        }
    )

    d = (
        d
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    n = len(d)

    if n < 2:

        return dict(
            n=n,
            original_mean=np.nan,
            residualized_mean=np.nan,
            mean_difference_residualized_minus_original=np.nan,
            ci95_difference_low=np.nan,
            ci95_difference_high=np.nan,
            t=np.nan,
            df=np.nan,
            p=np.nan,
            dz=np.nan,
        )

    x = (
        d[
            "original"
        ]
        .astype(float)
        .values
    )

    y = (
        d[
            "residualized"
        ]
        .astype(float)
        .values
    )

    diff = (
        y
        -
        x
    )

    tt = ttest_rel(
        y,
        x
    )

    md, lo, hi, _ = (
        motor_mean_ci(
            diff
        )
    )

    sd = np.std(
        diff,
        ddof=1
    )

    dz = (
        np.mean(diff)
        /
        sd
        if sd > 0
        else np.nan
    )

    return dict(

        n=n,

        original_mean=float(
            np.mean(x)
        ),

        residualized_mean=float(
            np.mean(y)
        ),

        mean_difference_residualized_minus_original=float(
            md
        ),

        ci95_difference_low=float(
            lo
        ),

        ci95_difference_high=float(
            hi
        ),

        t=float(
            tt.statistic
        ),

        df=n - 1,

        p=float(
            tt.pvalue
        ),

        dz=(
            float(dz)
            if np.isfinite(dz)
            else np.nan
        ),
    )


def motor_safe_corr(
    x,
    y,
):

    x = np.asarray(
        x,
        float
    )

    y = np.asarray(
        y,
        float
    )

    ok = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    if ok.sum() < 3:
        return np.nan

    x = x[
        ok
    ]

    y = y[
        ok
    ]

    if (
        np.std(x) <= 1e-12
        or
        np.std(y) <= 1e-12
    ):
        return np.nan

    return float(
        np.corrcoef(
            x,
            y
        )[0, 1]
    )


# -----------------------------------------------------------------------------
# 4. LOAD ORIGINAL WINDOW-LEVEL DATA
# -----------------------------------------------------------------------------

if (
    "window_df" in globals()
    and
    isinstance(
        window_df,
        pd.DataFrame
    )
    and
    len(window_df) > 0
    and
    ORIGINAL_CAPTURE_METRIC_MOTOR
    in window_df.columns
):

    motor_source_window_df = (
        window_df.copy()
    )

    print(
        "Using in-memory window_df "
        "from original breakpoint analysis."
    )

else:

    if not os.path.exists(
        ORIGINAL_WINDOW_CSV_MOTOR
    ):

        raise FileNotFoundError(
            "Original window-level file "
            "not found:\n"
            f"{ORIGINAL_WINDOW_CSV_MOTOR}"
        )

    motor_source_window_df = (
        pd.read_csv(
            ORIGINAL_WINDOW_CSV_MOTOR
        )
    )

    print(
        f"Loaded:\n"
        f"{ORIGINAL_WINDOW_CSV_MOTOR}"
    )


needed_cols = [

    "subject",
    "source_file",
    "trialIndex",
    "trialType",
    "condLabel",
    "offset_deg",

    "window_center_sec",

    "MeanDisparitySigned_deg",
    "MeanAbsDisparity_deg",

    ORIGINAL_CAPTURE_METRIC_MOTOR,

    *MOTOR_PREDICTORS,
]


missing_cols = [
    c
    for c
    in needed_cols
    if c
    not in motor_source_window_df.columns
]


if missing_cols:

    raise ValueError(
        "Missing required columns:\n"
        +
        "\n".join(
            missing_cols
        )
    )


motor_source_window_df[
    "subject"
] = (
    motor_source_window_df[
        "subject"
    ]
    .astype(str)
    .str.lower()
)


motor_source_window_df[
    "trialIndex"
] = pd.to_numeric(
    motor_source_window_df[
        "trialIndex"
    ],
    errors="coerce"
)


motor_source_window_df[
    "offset_deg"
] = pd.to_numeric(
    motor_source_window_df[
        "offset_deg"
    ],
    errors="coerce"
)


# -----------------------------------------------------------------------------
# 5. WITHIN-TRIAL MOTOR RESIDUALIZATION
# -----------------------------------------------------------------------------

def residualize_one_trial_for_motor(
    trial_df,
):

    """
    Model within one trial:

        capture
        ~ centered signed velocity
        + centered absolute velocity

    Then construct:

        adjusted_capture
        = fitted intercept + residual

    Thus the trial mean is retained while
    motor-linked fluctuation is removed.
    """

    d = (
        trial_df
        .copy()
        .sort_values(
            "window_center_sec"
        )
    )


    y = pd.to_numeric(
        d[
            ORIGINAL_CAPTURE_METRIC_MOTOR
        ],
        errors="coerce"
    ).values.astype(float)


    signed_velocity = pd.to_numeric(
        d[
            "SignedResponseVelocity_lagcorr"
        ],
        errors="coerce"
    ).values.astype(float)


    absolute_velocity = pd.to_numeric(
        d[
            "AbsResponseVelocity_lagcorr"
        ],
        errors="coerce"
    ).values.astype(float)


    valid = (

        np.isfinite(y)

        &

        np.isfinite(
            signed_velocity
        )

        &

        np.isfinite(
            absolute_velocity
        )

        &

        (
            y
            >=
            CAPTURE_FIT_MIN
        )

        &

        (
            y
            <=
            CAPTURE_FIT_MAX
        )
    )


    d[
        MOTOR_RESID_METRIC
    ] = np.nan


    diagnostics = dict(

        n_windows_motor_model=int(
            valid.sum()
        ),

        motor_model_ok=0,

        motor_intercept=np.nan,

        beta_signed_velocity=np.nan,

        beta_absolute_velocity=np.nan,

        motor_model_r_squared=np.nan,

        corr_capture_signed_velocity_before=np.nan,

        corr_capture_signed_velocity_after=np.nan,

        corr_capture_abs_velocity_before=np.nan,

        corr_capture_abs_velocity_after=np.nan,
    )


    if (
        valid.sum()
        <
        MIN_WINDOWS_FOR_MOTOR_MODEL
    ):

        return (
            d,
            diagnostics
        )


    y_fit = y[
        valid
    ]


    signed_fit = signed_velocity[
        valid
    ]


    absolute_fit = absolute_velocity[
        valid
    ]


    signed_centered = (
        signed_fit
        -
        np.mean(
            signed_fit
        )
    )


    absolute_centered = (
        absolute_fit
        -
        np.mean(
            absolute_fit
        )
    )


    X = np.column_stack(
        [
            np.ones(
                len(y_fit)
            ),

            signed_centered,

            absolute_centered,
        ]
    )


    try:

        beta, _, _, _ = (
            np.linalg.lstsq(
                X,
                y_fit,
                rcond=None
            )
        )

    except Exception:

        return (
            d,
            diagnostics
        )


    predicted = (
        X
        @
        beta
    )


    residual = (
        y_fit
        -
        predicted
    )


    # Preserve the trial-level mean / intercept.
    adjusted = (
        beta[0]
        +
        residual
    )


    adjusted_all = np.full(
        len(d),
        np.nan,
        dtype=float
    )


    adjusted_all[
        valid
    ] = adjusted


    d[
        MOTOR_RESID_METRIC
    ] = adjusted_all


    rss = float(
        np.sum(
            (
                y_fit
                -
                predicted
            )
            ** 2
        )
    )


    tss = float(
        np.sum(
            (
                y_fit
                -
                np.mean(
                    y_fit
                )
            )
            ** 2
        )
    )


    motor_r_squared = (
        np.nan
        if tss <= 1e-12
        else
        float(
            1
            -
            rss
            /
            tss
        )
    )


    diagnostics.update(

        motor_model_ok=1,

        motor_intercept=float(
            beta[0]
        ),

        beta_signed_velocity=float(
            beta[1]
        ),

        beta_absolute_velocity=float(
            beta[2]
        ),

        motor_model_r_squared=
            motor_r_squared,

        corr_capture_signed_velocity_before=
            motor_safe_corr(
                y_fit,
                signed_fit
            ),

        corr_capture_signed_velocity_after=
            motor_safe_corr(
                adjusted,
                signed_fit
            ),

        corr_capture_abs_velocity_before=
            motor_safe_corr(
                y_fit,
                absolute_fit
            ),

        corr_capture_abs_velocity_after=
            motor_safe_corr(
                adjusted,
                absolute_fit
            ),
    )


    return (
        d,
        diagnostics
    )


trial_group_cols_motor = [

    "subject",
    "source_file",
    "trialIndex",
    "trialType",
    "condLabel",
    "offset_deg",
]


motor_window_parts = []

motor_diag_rows = []


for (
    keys,
    sub
) in motor_source_window_df.groupby(

    trial_group_cols_motor,

    dropna=False,

    sort=False,
):

    (
        subject,
        source_file,
        trial_index,
        trial_type,
        cond_label,
        offset_deg,
    ) = keys


    adjusted_sub, diagnostics = (
        residualize_one_trial_for_motor(
            sub
        )
    )


    motor_window_parts.append(
        adjusted_sub
    )


    motor_diag_rows.append(
        {

            "subject":
                subject,

            "source_file":
                source_file,

            "trialIndex":
                trial_index,

            "trialType":
                trial_type,

            "condLabel":
                cond_label,

            "offset_deg":
                offset_deg,

            **diagnostics,
        }
    )


motor_window_df = pd.concat(
    motor_window_parts,
    ignore_index=True
)


motor_diagnostics_df = pd.DataFrame(
    motor_diag_rows
)


motor_window_csv = os.path.join(
    CSV_DIR_MOTOR,
    "motor_residualized_window_level_metrics.csv"
)


motor_diag_csv = os.path.join(
    CSV_DIR_MOTOR,
    "motor_residualization_trial_diagnostics.csv"
)


motor_window_df.to_csv(
    motor_window_csv,
    index=False
)


motor_diagnostics_df.to_csv(
    motor_diag_csv,
    index=False
)


valid_motor = motor_diagnostics_df[
    motor_diagnostics_df[
        "motor_model_ok"
    ] == 1
].copy()


print(
    "\n"
    +
    "=" * 90
)

print(
    "MOTOR RESIDUALIZATION DIAGNOSTICS"
)

print(
    "=" * 90
)


print(
    f"Trials attempted: "
    f"{len(motor_diagnostics_df)}"
)


print(
    f"Trials with usable motor model: "
    f"{len(valid_motor)}"
)


if len(
    valid_motor
) > 0:

    print(
        "Mean within-trial motor-model R²: "
        f"{valid_motor['motor_model_r_squared'].mean():.4f}"
    )

    print(
        "Median within-trial motor-model R²: "
        f"{valid_motor['motor_model_r_squared'].median():.4f}"
    )

    print(
        "Mean |capture-signed velocity r| "
        "before -> after: "
        f"{valid_motor['corr_capture_signed_velocity_before'].abs().mean():.4f}"
        " -> "
        f"{valid_motor['corr_capture_signed_velocity_after'].abs().mean():.4f}"
    )

    print(
        "Mean |capture-absolute velocity r| "
        "before -> after: "
        f"{valid_motor['corr_capture_abs_velocity_before'].abs().mean():.4f}"
        " -> "
        f"{valid_motor['corr_capture_abs_velocity_after'].abs().mean():.4f}"
    )


# -----------------------------------------------------------------------------
# 6. RE-RUN THE ORIGINAL BREAKPOINT FUNCTION
#    ON MOTOR-RESIDUALIZED CAPTURE
# -----------------------------------------------------------------------------

motor_bp_rows = []


for (
    keys,
    sub
) in motor_window_df.groupby(

    trial_group_cols_motor,

    dropna=False,

    sort=False,
):

    (
        subject,
        source_file,
        trial_index,
        trial_type,
        cond_label,
        offset_deg,
    ) = keys


    result = (
        fit_stabilizing_breakpoint_for_trial(

            sub,

            metric=MOTOR_RESID_METRIC,
        )
    )


    motor_bp_rows.append(
        {

            "subject":
                subject,

            "source_file":
                source_file,

            "trialIndex":
                trial_index,

            "trialType":
                trial_type,

            "condLabel":
                cond_label,

            "offset_deg":
                offset_deg,

            **result,

            "metric_label":
                MOTOR_RESID_METRIC_LABEL,
        }
    )


motor_breakpoint_trial_df = pd.DataFrame(
    motor_bp_rows
)


binary_cols = [

    "piecewise_supported",

    "has_stabilizing_breakpoint",

    "stable_from_start",

    "stable_or_stabilized",

    "nonconstant_model_supported",

    "whole_trial_slope_dynamic",

    "candidate_post_slope_dynamic",

    "derivative_dynamic",

    "late_derivative_dynamic",

    "dynamically_updating",

    "ambiguous_or_noisy",
]


for col in binary_cols:

    if (
        col
        not in motor_breakpoint_trial_df.columns
    ):

        motor_breakpoint_trial_df[
            col
        ] = 0


    motor_breakpoint_trial_df[
        col
    ] = (

        pd.to_numeric(

            motor_breakpoint_trial_df[
                col
            ],

            errors="coerce"
        )

        .fillna(0)

        .astype(int)
    )


dynamic_cols = [

    "linear_slope_capture_per_sec",

    "whole_trial_abs_linear_slope_capture_per_sec",

    "candidate_post_slope_capture_per_sec",

    "candidate_abs_post_slope_capture_per_sec",

    "mean_abs_capture_derivative_per_sec",

    "median_abs_capture_derivative_per_sec",

    "early_mean_abs_capture_derivative_per_sec",

    "middle_mean_abs_capture_derivative_per_sec",

    "late_mean_abs_capture_derivative_per_sec",

    "late_minus_early_abs_derivative_per_sec",

    "post_breakpoint_mean_abs_capture_derivative_per_sec",

    "post_breakpoint_median_abs_capture_derivative_per_sec",
]


for col in dynamic_cols:

    if (
        col
        in motor_breakpoint_trial_df.columns
    ):

        motor_breakpoint_trial_df[
            f"{col}_nonstabilized"
        ] = np.where(

            motor_breakpoint_trial_df[
                "stable_or_stabilized"
            ] == 0,

            motor_breakpoint_trial_df[
                col
            ],

            np.nan,
        )


motor_trial_csv = os.path.join(

    CSV_DIR_MOTOR,

    "motor_residualized_trial_level_breakpoint_DYNAMIC_metrics.csv",
)


motor_breakpoint_trial_df.to_csv(
    motor_trial_csv,
    index=False
)


# -----------------------------------------------------------------------------
# 7. SUBJECT x OFFSET AND SUBJECT-LEVEL SUMMARIES
# -----------------------------------------------------------------------------

agg = dict(

    n_trials=(
        "trialIndex",
        "count"
    ),

    n_stabilized_trials=(
        "has_stabilizing_breakpoint",
        "sum"
    ),

    n_stable_from_start_trials=(
        "stable_from_start",
        "sum"
    ),

    n_stable_or_stabilized_trials=(
        "stable_or_stabilized",
        "sum"
    ),

    n_dynamic_trials=(
        "dynamically_updating",
        "sum"
    ),

    n_ambiguous_trials=(
        "ambiguous_or_noisy",
        "sum"
    ),

    stabilization_probability=(
        "has_stabilizing_breakpoint",
        "mean"
    ),

    stable_from_start_probability=(
        "stable_from_start",
        "mean"
    ),

    stable_or_stabilized_probability=(
        "stable_or_stabilized",
        "mean"
    ),

    dynamic_probability=(
        "dynamically_updating",
        "mean"
    ),

    ambiguous_or_noisy_probability=(
        "ambiguous_or_noisy",
        "mean"
    ),

    stabilization_time_sec=(
        "stabilization_time_sec",
        "mean"
    ),

    whole_trial_abs_linear_slope_capture_per_sec=(
        "whole_trial_abs_linear_slope_capture_per_sec",
        "mean"
    ),

    mean_abs_capture_derivative_per_sec=(
        "mean_abs_capture_derivative_per_sec",
        "mean"
    ),

    late_mean_abs_capture_derivative_per_sec=(
        "late_mean_abs_capture_derivative_per_sec",
        "mean"
    ),
)


for col in [

    "whole_trial_abs_linear_slope_capture_per_sec_nonstabilized",

    "mean_abs_capture_derivative_per_sec_nonstabilized",

    "late_mean_abs_capture_derivative_per_sec_nonstabilized",

    "candidate_abs_post_slope_capture_per_sec_nonstabilized",
]:

    if (
        col
        in motor_breakpoint_trial_df.columns
    ):

        agg[
            col
        ] = (
            col,
            "mean"
        )


motor_subject_offset_df = (

    motor_breakpoint_trial_df

    .groupby(

        [
            "subject",
            "offset_deg"
        ],

        as_index=False
    )

    .agg(
        **agg
    )
)


motor_subject_df = (

    motor_subject_offset_df

    .groupby(
        "subject",
        as_index=False
    )

    .mean(
        numeric_only=True
    )
)


motor_subject_offset_csv = os.path.join(

    CSV_DIR_MOTOR,

    "motor_residualized_subject_offset_breakpoint_DYNAMIC_metrics.csv",
)


motor_subject_csv = os.path.join(

    CSV_DIR_MOTOR,

    "motor_residualized_subject_level_breakpoint_DYNAMIC_metrics.csv",
)


motor_subject_offset_df.to_csv(
    motor_subject_offset_csv,
    index=False
)


motor_subject_df.to_csv(
    motor_subject_csv,
    index=False
)


# -----------------------------------------------------------------------------
# 8. MOTOR-RESIDUALIZED GROUP ESTIMATES
# -----------------------------------------------------------------------------

group_outcomes = [

    "dynamic_probability",

    "stable_or_stabilized_probability",

    "stabilization_probability",

    "stable_from_start_probability",

    "ambiguous_or_noisy_probability",

    "stabilization_time_sec",

    "whole_trial_abs_linear_slope_capture_per_sec",

    "mean_abs_capture_derivative_per_sec",

    "late_mean_abs_capture_derivative_per_sec",

    "whole_trial_abs_linear_slope_capture_per_sec_nonstabilized",

    "mean_abs_capture_derivative_per_sec_nonstabilized",

    "late_mean_abs_capture_derivative_per_sec_nonstabilized",

    "candidate_abs_post_slope_capture_per_sec_nonstabilized",
]


group_rows = []


for outcome in group_outcomes:

    if (
        outcome
        not in motor_subject_df.columns
    ):
        continue


    m, lo, hi, n = (
        motor_mean_ci(
            motor_subject_df[
                outcome
            ]
        )
    )


    group_rows.append(
        {

            "outcome":
                outcome,

            "n_subjects":
                n,

            "mean":
                m,

            "ci95_low":
                lo,

            "ci95_high":
                hi,
        }
    )


motor_group_estimates_df = pd.DataFrame(
    group_rows
)


motor_group_estimates_csv = os.path.join(

    CSV_DIR_MOTOR,

    "motor_residualized_breakpoint_group_estimates.csv",
)


motor_group_estimates_df.to_csv(
    motor_group_estimates_csv,
    index=False
)


# -----------------------------------------------------------------------------
# 9. KEY REVIEWER-CONTROL TESTS
# -----------------------------------------------------------------------------

key_test_rows = []


# Main test 1:
# Does >50% of behavior remain dynamically updating?
if (
    "dynamic_probability"
    in motor_subject_df.columns
):

    stats = (
        motor_one_sample(

            motor_subject_df[
                "dynamic_probability"
            ],

            popmean=.50,

            alternative="greater",
        )
    )


    key_test_rows.append(
        {

            "analysis":
                "motor_residualized_dynamic_probability_gt_0.50",

            "comparison":
                "dynamic_probability > .50",

            **stats,
        }
    )


# Main test 2:
# Are dynamic trials still more common
# than stable/stabilized trials?
if {

    "dynamic_probability",

    "stable_or_stabilized_probability",

}.issubset(
    motor_subject_df.columns
):

    diff = (

        motor_subject_df[
            "dynamic_probability"
        ]

        -

        motor_subject_df[
            "stable_or_stabilized_probability"
        ]
    )


    stats = motor_one_sample(

        diff,

        popmean=0.,

        alternative="greater"
    )


    key_test_rows.append(
        {

            "analysis":
                "motor_residualized_dynamic_gt_stable",

            "comparison":
                (
                    "dynamic_probability - "
                    "stable_or_stabilized_probability > 0"
                ),

            **stats,
        }
    )


motor_key_tests_df = pd.DataFrame(
    key_test_rows
)


motor_key_tests_csv = os.path.join(

    CSV_DIR_MOTOR,

    "motor_residualized_key_group_tests.csv",
)


motor_key_tests_df.to_csv(
    motor_key_tests_csv,
    index=False
)


# -----------------------------------------------------------------------------
# 10. LOAD ORIGINAL BREAKPOINT RESULTS
#     FOR DIRECT COMPARISON
# -----------------------------------------------------------------------------

if (

    "breakpoint_trial_df"
    in globals()

    and

    isinstance(
        breakpoint_trial_df,
        pd.DataFrame
    )

    and

    len(
        breakpoint_trial_df
    ) > 0
):

    original_bp = (
        breakpoint_trial_df.copy()
    )


else:

    if not os.path.exists(
        ORIGINAL_TRIAL_BP_CSV_MOTOR
    ):

        raise FileNotFoundError(

            "Original trial breakpoint "
            "file not found:\n"

            f"{ORIGINAL_TRIAL_BP_CSV_MOTOR}"
        )


    original_bp = pd.read_csv(
        ORIGINAL_TRIAL_BP_CSV_MOTOR
    )


original_bp[
    "subject"
] = (
    original_bp[
        "subject"
    ]
    .astype(str)
    .str.lower()
)


original_bp[
    "trialIndex"
] = pd.to_numeric(

    original_bp[
        "trialIndex"
    ],

    errors="coerce"
)


original_bp[
    "offset_deg"
] = pd.to_numeric(

    original_bp[
        "offset_deg"
    ],

    errors="coerce"
)


# Keep only the original
# lag-corrected capture metric.
if (
    "metric"
    in original_bp.columns
):

    original_bp = (

        original_bp[

            original_bp[
                "metric"
            ]

            .astype(str)

            .eq(
                ORIGINAL_CAPTURE_METRIC_MOTOR
            )
        ]

        .copy()
    )


# -----------------------------------------------------------------------------
# 11. TRIAL-LEVEL ORIGINAL vs MOTOR-RESIDUALIZED CLASSIFICATION
# -----------------------------------------------------------------------------

keep = [

    "subject",

    "trialIndex",

    "offset_deg",

    "trial_dynamic_class",

    "dynamically_updating",

    "stable_from_start",

    "has_stabilizing_breakpoint",

    "stable_or_stabilized",

    "ambiguous_or_noisy",

    "stabilization_time_sec",
]


orig_trial = (

    original_bp[
        [
            c
            for c
            in keep
            if c
            in original_bp.columns
        ]
    ]

    .copy()
)


motor_trial = (

    motor_breakpoint_trial_df[
        [
            c
            for c
            in keep
            if c
            in motor_breakpoint_trial_df.columns
        ]
    ]

    .copy()
)


orig_trial = (

    orig_trial

    .rename(
        columns={
            c:
                f"original_{c}"

            for c
            in orig_trial.columns

            if c
            not in [
                "subject",
                "trialIndex",
                "offset_deg",
            ]
        }
    )
)


motor_trial = (

    motor_trial

    .rename(
        columns={
            c:
                f"motor_{c}"

            for c
            in motor_trial.columns

            if c
            not in [
                "subject",
                "trialIndex",
                "offset_deg",
            ]
        }
    )
)


trial_compare_df = (

    orig_trial

    .merge(

        motor_trial,

        on=[
            "subject",
            "trialIndex",
            "offset_deg",
        ],

        how="inner",

        validate="one_to_one",
    )
)


trial_compare_csv = os.path.join(

    CSV_DIR_MOTOR,

    "original_vs_motor_residualized_trial_comparison.csv",
)


trial_compare_df.to_csv(
    trial_compare_csv,
    index=False
)


classification_crosstab = pd.crosstab(

    trial_compare_df[
        "original_trial_dynamic_class"
    ],

    trial_compare_df[
        "motor_trial_dynamic_class"
    ],

    rownames=[
        "Original class"
    ],

    colnames=[
        "Motor-residualized class"
    ],

    dropna=False,
)


classification_crosstab_csv = os.path.join(

    CSV_DIR_MOTOR,

    "original_vs_motor_residualized_classification_crosstab.csv",
)


classification_crosstab.to_csv(
    classification_crosstab_csv
)


# -----------------------------------------------------------------------------
# 12. DYNAMIC-TRIAL RETENTION
# -----------------------------------------------------------------------------

retention_rows = []


for (
    subject,
    sub
) in trial_compare_df.groupby(
    "subject"
):


    original_dynamic = (

        pd.to_numeric(

            sub[
                "original_dynamically_updating"
            ],

            errors="coerce"
        )

        .fillna(0)

        .astype(int)
    )


    motor_dynamic = (

        pd.to_numeric(

            sub[
                "motor_dynamically_updating"
            ],

            errors="coerce"
        )

        .fillna(0)

        .astype(int)
    )


    mask = (
        original_dynamic
        == 1
    )


    retention = (

        float(
            motor_dynamic[
                mask
            ]
            .mean()
        )

        if mask.sum() > 0

        else np.nan
    )


    retention_rows.append(
        {

            "subject":
                subject,

            "n_original_dynamic_trials":
                int(
                    mask.sum()
                ),

            "dynamic_retention_probability":
                retention,

            "original_dynamic_probability_trial_weighted":
                float(
                    original_dynamic.mean()
                ),

            "motor_dynamic_probability_trial_weighted":
                float(
                    motor_dynamic.mean()
                ),
        }
    )


dynamic_retention_subject_df = pd.DataFrame(
    retention_rows
)


retention_csv = os.path.join(

    CSV_DIR_MOTOR,

    "dynamic_trial_retention_after_motor_residualization_by_subject.csv",
)


dynamic_retention_subject_df.to_csv(
    retention_csv,
    index=False
)


ret_m, ret_lo, ret_hi, ret_n = (
    motor_mean_ci(

        dynamic_retention_subject_df[
            "dynamic_retention_probability"
        ]
    )
)


dynamic_retention_group_df = pd.DataFrame(
    [
        {

            "n_subjects":
                ret_n,

            "mean_dynamic_retention_probability":
                ret_m,

            "ci95_low":
                ret_lo,

            "ci95_high":
                ret_hi,
        }
    ]
)


dynamic_retention_group_csv = os.path.join(

    CSV_DIR_MOTOR,

    "dynamic_trial_retention_after_motor_residualization_group.csv",
)


dynamic_retention_group_df.to_csv(
    dynamic_retention_group_csv,
    index=False
)


# -----------------------------------------------------------------------------
# 13. MATCHED SUBJECT-LEVEL
#     ORIGINAL vs MOTOR COMPARISON
# -----------------------------------------------------------------------------

original_subject_offset = (

    original_bp

    .groupby(

        [
            "subject",
            "offset_deg",
        ],

        as_index=False
    )

    .agg(

        original_dynamic_probability=(
            "dynamically_updating",
            "mean"
        ),

        original_stable_or_stabilized_probability=(
            "stable_or_stabilized",
            "mean"
        ),

        original_stabilization_probability=(
            "has_stabilizing_breakpoint",
            "mean"
        ),

        original_stable_from_start_probability=(
            "stable_from_start",
            "mean"
        ),

        original_ambiguous_probability=(
            "ambiguous_or_noisy",
            "mean"
        ),
    )
)


original_subject = (

    original_subject_offset

    .groupby(
        "subject",
        as_index=False
    )

    .mean(
        numeric_only=True
    )
)


motor_subject_for_compare = (

    motor_subject_df[
        [

            "subject",

            "dynamic_probability",

            "stable_or_stabilized_probability",

            "stabilization_probability",

            "stable_from_start_probability",

            "ambiguous_or_noisy_probability",
        ]
    ]

    .rename(
        columns={

            "dynamic_probability":
                "motor_dynamic_probability",

            "stable_or_stabilized_probability":
                "motor_stable_or_stabilized_probability",

            "stabilization_probability":
                "motor_stabilization_probability",

            "stable_from_start_probability":
                "motor_stable_from_start_probability",

            "ambiguous_or_noisy_probability":
                "motor_ambiguous_probability",
        }
    )
)


subject_compare_df = (

    original_subject

    .merge(

        motor_subject_for_compare,

        on="subject",

        how="inner",

        validate="one_to_one",
    )
)


subject_compare_csv = os.path.join(

    CSV_DIR_MOTOR,

    "original_vs_motor_residualized_subject_comparison.csv",
)


subject_compare_df.to_csv(
    subject_compare_csv,
    index=False
)


paired_specs = [

    (
        "dynamic_probability",
        "original_dynamic_probability",
        "motor_dynamic_probability",
    ),

    (
        "stable_or_stabilized_probability",
        "original_stable_or_stabilized_probability",
        "motor_stable_or_stabilized_probability",
    ),

    (
        "stabilization_probability",
        "original_stabilization_probability",
        "motor_stabilization_probability",
    ),

    (
        "stable_from_start_probability",
        "original_stable_from_start_probability",
        "motor_stable_from_start_probability",
    ),

    (
        "ambiguous_probability",
        "original_ambiguous_probability",
        "motor_ambiguous_probability",
    ),
]


paired_rows = []


for (
    outcome,
    original_col,
    motor_col
) in paired_specs:


    stats = motor_paired(

        subject_compare_df[
            original_col
        ],

        subject_compare_df[
            motor_col
        ],
    )


    paired_rows.append(
        {

            "outcome":
                outcome,

            "difference_definition":
                "motor_residualized_minus_original",

            **stats,
        }
    )


paired_subject_tests_df = pd.DataFrame(
    paired_rows
)


paired_subject_tests_csv = os.path.join(

    CSV_DIR_MOTOR,

    "original_vs_motor_residualized_paired_subject_tests.csv",
)


paired_subject_tests_df.to_csv(
    paired_subject_tests_csv,
    index=False
)


# -----------------------------------------------------------------------------
# 14. PLOT:
#     RESIDUALIZED CLASSIFICATION BY OFFSET
# -----------------------------------------------------------------------------

plot_specs = [

    (
        "stable_from_start_probability",
        "Stable from start"
    ),

    (
        "stabilization_probability",
        "Stabilized after breakpoint"
    ),

    (
        "dynamic_probability",
        "Dynamically updating"
    ),

    (
        "ambiguous_or_noisy_probability",
        "Ambiguous/noisy"
    ),
]


fig, ax = plt.subplots(
    figsize=(
        9.2,
        5.8
    )
)


for (
    outcome,
    label
) in plot_specs:


    summary = (

        motor_subject_offset_df

        .groupby(
            "offset_deg",
            as_index=False
        )

        .agg(

            mean=(
                outcome,
                "mean"
            ),

            se=(
                outcome,
                motor_sem
            ),
        )

        .sort_values(
            "offset_deg"
        )
    )


    ax.errorbar(

        summary[
            "offset_deg"
        ],

        summary[
            "mean"
        ],

        yerr=summary[
            "se"
        ],

        marker="o",

        linewidth=2,

        capsize=4,

        label=label,
    )


ax.set_ylim(
    -.05,
    1.05
)


ax.set_xlabel(
    "A-V offset magnitude (deg)"
)


ax.set_ylabel(
    "Proportion of trials"
)


ax.set_title(
    "Motor-residualized spatial-weighting dynamics"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


fig.savefig(

    os.path.join(

        FIG_DIR_MOTOR,

        "motor_residualized_trial_classification_by_offset.png",
    ),

    dpi=FIG_DPI_MOTOR,

    bbox_inches="tight"
)


plt.show()


# -----------------------------------------------------------------------------
# 15. PLOT:
#     ORIGINAL vs MOTOR-RESIDUALIZED
#     DYNAMIC PROBABILITY BY OFFSET
# -----------------------------------------------------------------------------

original_plot = (

    original_subject_offset

    .groupby(
        "offset_deg",
        as_index=False
    )

    .agg(

        mean=(
            "original_dynamic_probability",
            "mean"
        ),

        se=(
            "original_dynamic_probability",
            motor_sem
        ),
    )

    .sort_values(
        "offset_deg"
    )
)


motor_plot = (

    motor_subject_offset_df

    .groupby(
        "offset_deg",
        as_index=False
    )

    .agg(

        mean=(
            "dynamic_probability",
            "mean"
        ),

        se=(
            "dynamic_probability",
            motor_sem
        ),
    )

    .sort_values(
        "offset_deg"
    )
)


fig, ax = plt.subplots(
    figsize=(
        8.5,
        5.6
    )
)


ax.errorbar(

    original_plot[
        "offset_deg"
    ],

    original_plot[
        "mean"
    ],

    yerr=original_plot[
        "se"
    ],

    marker="o",

    linewidth=2,

    capsize=4,

    label=
        "Original lag-corrected capture",
)


ax.errorbar(

    motor_plot[
        "offset_deg"
    ],

    motor_plot[
        "mean"
    ],

    yerr=motor_plot[
        "se"
    ],

    marker="o",

    linewidth=2,

    capsize=4,

    label=
        "Motor-residualized capture",
)


ax.set_ylim(
    -.05,
    1.05
)


ax.set_xlabel(
    "A-V offset magnitude (deg)"
)


ax.set_ylabel(
    "Probability of dynamically updating trial"
)


ax.set_title(
    "Spatial dynamicity before and after motor residualization"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


fig.savefig(

    os.path.join(

        FIG_DIR_MOTOR,

        "original_vs_motor_residualized_dynamic_probability_by_offset.png",
    ),

    dpi=FIG_DPI_MOTOR,

    bbox_inches="tight"
)


plt.show()


# -----------------------------------------------------------------------------
# 16. PLOT:
#     SUBJECT-LEVEL ORIGINAL vs
#     MOTOR-RESIDUALIZED DYNAMICITY
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(
        6.2,
        6.0
    )
)


ax.scatter(

    subject_compare_df[
        "original_dynamic_probability"
    ],

    subject_compare_df[
        "motor_dynamic_probability"
    ],

    alpha=.8,
)


ax.plot(

    [0, 1],

    [0, 1],

    linestyle="--",

    linewidth=1.2,
)


ax.set_xlim(
    -.02,
    1.02
)


ax.set_ylim(
    -.02,
    1.02
)


ax.set_xlabel(
    "Original dynamic probability"
)


ax.set_ylabel(
    "Motor-residualized dynamic probability"
)


ax.set_title(
    "Participant-level robustness of SL dynamicity"
)


fig.tight_layout()


fig.savefig(

    os.path.join(

        FIG_DIR_MOTOR,

        "subject_original_vs_motor_residualized_dynamic_probability.png",
    ),

    dpi=FIG_DPI_MOTOR,

    bbox_inches="tight"
)


plt.show()


# -----------------------------------------------------------------------------
# 17. FINAL SUMMARY
# -----------------------------------------------------------------------------

print(
    "\n"
    +
    "=" * 100
)


print(
    "MOTOR-RESIDUALIZED SL BREAKPOINT ANALYSIS COMPLETE"
)


print(
    "=" * 100
)


print(
    "\nKEY MOTOR-RESIDUALIZED GROUP ESTIMATES"
)


print(
    "-" * 100
)


print(
    motor_group_estimates_df.to_string(
        index=False
    )
)


print(
    "\nKEY REVIEWER-CONTROL TESTS"
)


print(
    "-" * 100
)


print(
    motor_key_tests_df.to_string(
        index=False
    )
)


print(
    "\nORIGINAL vs MOTOR-RESIDUALIZED "
    "PAIRED SUBJECT TESTS"
)


print(
    "-" * 100
)


print(
    paired_subject_tests_df.to_string(
        index=False
    )
)


print(
    "\nDYNAMIC-TRIAL RETENTION"
)


print(
    "-" * 100
)


print(
    dynamic_retention_group_df.to_string(
        index=False
    )
)


print(
    "\nTRIAL-CLASSIFICATION CROSSTAB"
)


print(
    "-" * 100
)


print(
    classification_crosstab.to_string()
)


print(
    "\nOUTPUT DIRECTORY"
)


print(
    "-" * 100
)


print(
    OUT_DIR_MOTOR
)


print(
    "\nINTERPRETATION GUIDE"
)


print(
    "-" * 100
)


print(
    "1) The primary quantity is motor-residualized dynamic_probability.\n"
    "2) If it remains high after residualization, ongoing SL recalibration is not "
    "explained simply by continuous mouse motion.\n"
    "3) The paired original-vs-motor test shows how much dynamic probability changes "
    "after removing motor-associated variance.\n"
    "4) dynamic_retention_probability is the fraction of originally dynamic trials "
    "that remain dynamic after the motor control.\n"
    "5) The before/after velocity correlations verify that the linear motor association "
    "was actually removed.\n"
    "6) Because the ORIGINAL fit_stabilizing_breakpoint_for_trial() function is reused, "
    "all breakpoint/BIC/slope/dynamicity criteria remain identical."
)

Using in-memory window_df from original breakpoint analysis.

MOTOR RESIDUALIZATION DIAGNOSTICS
Trials attempted: 4000
Trials with usable motor model: 3971
Mean within-trial motor-model R²: 0.1683
Median within-trial motor-model R²: 0.1063
Mean |capture-signed velocity r| before -> after: 0.1891 -> 0.0000
Mean |capture-absolute velocity r| before -> after: 0.2778 -> 0.0000

MOTOR-RESIDUALIZED SL BREAKPOINT ANALYSIS COMPLETE

KEY MOTOR-RESIDUALIZED GROUP ESTIMATES
----------------------------------------------------------------------------------------------------
                                                   outcome  n_subjects     mean  ci95_low  ci95_high
                                       dynamic_probability          50 0.719000  0.693200   0.744800
                          stable_or_stabilized_probability          50 0.252500  0.226140   0.278860
                                 stabilization_probability          50 0.199000  0.177268   0.220732
                           

In [ ]:
# %% ADD-ON: PERMUTATION INFERENCE FOR BREAKPOINT + DYNAMIC CAPTURE ANALYSIS
# FLAT OUTPUT-PATH VERSION
# ============================================================
# Run this AFTER:
#   # %% CHINESE RESTAURANT / AUDITORY-ANCHOR BREAKPOINT + DYNAMIC CAPTURE ANALYSIS
#
# This add-on performs:
#
# 1) Whole-trial fixed-offset condition-label shuffle
#    ------------------------------------------------
#    Offset labels are shuffled across complete fixed-offset trials
#    within subject. This preserves subject-level dependence and
#    complete trial-level breakpoint/dynamic metrics, while breaking
#    the mapping between each trial's metrics and its physical offset.
#
# 2) Cluster-based permutation analysis over time
#    ------------------------------------------------
#    Uses the window-level capture timecourses from window_df.
#    Offset labels are shuffled at the complete-trial level within subject,
#    then group timecourses are recomputed. This creates a null distribution
#    of timecourses for each metric × offset.
#
# FLAT OUTPUT:
#   All CSV and PNG outputs are saved directly into:
#       spatial_chinese_restaurant_anchor_BREAKPOINT_DYNAMIC_analysis
#
#   No nested output folders are created.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import t as t_dist

try:
    import statsmodels.formula.api as smf
except ImportError:
    smf = None

# ============================================================
# SETTINGS
# ============================================================

N_PERM_BREAKPOINT_LABEL_SHUFFLE = 10000

# Cluster permutation can be slower because it recomputes full timecourses.
# Use 1000 for checking; increase to 5000 or 10000 for final.
N_PERM_BREAKPOINT_CLUSTER = 1000

RANDOM_SEED_BREAKPOINT_PERM = 12345

# Recommended: two-sided, because condition shuffling tests whether values
# are unusually high or unusually low relative to the offset-label null.
BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE = "two-sided"
BREAKPOINT_CLUSTER_ALTERNATIVE = "two-sided"

CLUSTER_FORMING_ALPHA = 0.05

try:
    FIXED_OFFSETS
except NameError:
    FIXED_OFFSETS = [10, 20, 30, 40]

try:
    BREAKPOINT_METRICS
except NameError:
    BREAKPOINT_METRICS = [
        ("RawVisualCapture", "Raw visual capture"),
        ("LagCorrectedVisualCapture", "Lag-corrected visual capture"),
    ]

try:
    FIG_DPI
except NameError:
    FIG_DPI = 180

# ============================================================
# FLAT OUTPUT DIRECTORY
# ============================================================

OUTPUT_FOLDER_NAME = "06_stabilization_breakpoint"

def _find_ancestor_named(path, ancestor_name):
    if path is None:
        return None

    try:
        cur = os.path.abspath(str(path))
    except Exception:
        return None

    base = os.path.basename(cur)

    if "." in base and not os.path.isdir(cur):
        cur = os.path.dirname(cur)

    while True:
        if os.path.basename(cur).lower() == ancestor_name.lower():
            return cur

        parent = os.path.dirname(cur)

        if parent == cur:
            return None

        cur = parent

def _resolve_flat_breakpoint_output_dir():
    """
    Forces output directly into:
        spatial_chinese_restaurant_anchor_BREAKPOINT_DYNAMIC_analysis

    Priority:
      1. Existing OUT_DIR if it points to that folder.
      2. Parent of CSV_DIR/FIG_DIR if available.
      3. DATA_DIR / spatial_chinese_restaurant_anchor_BREAKPOINT_DYNAMIC_analysis.
      4. Current working directory / spatial_chinese_restaurant_anchor_BREAKPOINT_DYNAMIC_analysis.
    """
    candidate_paths = []

    if "OUT_DIR" in globals():
        candidate_paths.append(OUT_DIR)

    if "CSV_DIR" in globals():
        candidate_paths.append(CSV_DIR)

    if "FIG_DIR" in globals():
        candidate_paths.append(FIG_DIR)

    if "DATA_DIR" in globals():
        candidate_paths.append(DATA_DIR)

    candidate_paths.append(os.getcwd())

    for candidate in candidate_paths:
        root = _find_ancestor_named(candidate, OUTPUT_FOLDER_NAME)
        if root is not None:
            return root

    if "DATA_DIR" in globals():
        return os.path.join(r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses", OUTPUT_FOLDER_NAME)

    return os.path.join(os.getcwd(), OUTPUT_FOLDER_NAME)

PERM_OUT_DIR = _resolve_flat_breakpoint_output_dir()
os.makedirs(PERM_OUT_DIR, exist_ok=True)

# Overwrite these so all saves go flat.
OUT_DIR = PERM_OUT_DIR
CSV_DIR = PERM_OUT_DIR
FIG_DIR = PERM_OUT_DIR
PERM_CSV_DIR = PERM_OUT_DIR
PERM_FIG_DIR = PERM_OUT_DIR

def _flat_output_path(filepath_or_filename):
    """
    Always save directly into PERM_OUT_DIR using only the basename.
    This prevents nested folders from being created accidentally.
    """
    filename = os.path.basename(str(filepath_or_filename))
    return os.path.join(PERM_OUT_DIR, filename)

def _safe_to_csv_perm(df, filename, index=False):
    outpath = _flat_output_path(filename)
    os.makedirs(PERM_OUT_DIR, exist_ok=True)
    df.to_csv(outpath, index=index)
    return outpath

def _safe_savefig_perm(fig, filename):
    outpath = _flat_output_path(filename)
    os.makedirs(PERM_OUT_DIR, exist_ok=True)
    fig.tight_layout()
    fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return outpath

print("\n============================================================")
print("BREAKPOINT + DYNAMIC CAPTURE PERMUTATION ADD-ON")
print("FLAT OUTPUT-PATH VERSION")
print("============================================================")
print(f"Whole-trial condition-label shuffles: {N_PERM_BREAKPOINT_LABEL_SHUFFLE}")
print(f"Cluster permutations: {N_PERM_BREAKPOINT_CLUSTER}")
print(f"Condition-label alternative: {BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE}")
print(f"Cluster alternative: {BREAKPOINT_CLUSTER_ALTERNATIVE}")
print(f"All outputs saved directly to:\n{PERM_OUT_DIR}")

# ============================================================
# REQUIRED OBJECT CHECKS
# ============================================================

required_objects = [
    "window_df",
    "breakpoint_trial_df",
    "breakpoint_subject_offset_df",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise RuntimeError(
        "Missing required objects from the parent breakpoint/dynamic analysis: "
        f"{missing_objects}. Run the breakpoint/dynamic analysis cell first."
    )

required_trial_cols = [
    "subject",
    "source_file",
    "trialIndex",
    "trialType",
    "condLabel",
    "offset_deg",
    "metric",
    "metric_label",
]

missing_trial_cols = [c for c in required_trial_cols if c not in breakpoint_trial_df.columns]

if missing_trial_cols:
    raise RuntimeError(
        f"breakpoint_trial_df is missing required columns: {missing_trial_cols}"
    )

required_window_cols = [
    "subject",
    "source_file",
    "trialIndex",
    "trialType",
    "condLabel",
    "offset_deg",
    "window_center_sec",
]

missing_window_cols = [c for c in required_window_cols if c not in window_df.columns]

if missing_window_cols:
    raise RuntimeError(
        f"window_df is missing required columns: {missing_window_cols}"
    )

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def _clean_numeric(values):
    return (
        pd.Series(values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
        .values
    )

def _mean_ci_test(values, popmean=0.0, alpha=0.05):
    vals = _clean_numeric(values)
    n = len(vals)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "se": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p_two_sided": np.nan,
            "p_greater": np.nan,
            "p_less": np.nan,
        }

    m = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1))
    se = float(sd / np.sqrt(n))
    crit = float(t_dist.ppf(1 - alpha / 2, df=n - 1))
    t_stat, p_two = stats.ttest_1samp(vals, popmean=popmean, nan_policy="omit")

    return {
        "n": int(n),
        "mean": m,
        "se": se,
        "ci95_low": float(m - crit * se),
        "ci95_high": float(m + crit * se),
        "t": float(t_stat),
        "df": int(n - 1),
        "p_two_sided": float(p_two),
        "p_greater": float(stats.t.sf(t_stat, df=n - 1)),
        "p_less": float(stats.t.cdf(t_stat, df=n - 1)),
    }

def _empirical_pvalue(observed, null_values, alternative="two-sided"):
    null_values = _clean_numeric(null_values)

    if len(null_values) == 0 or not np.isfinite(observed):
        return np.nan

    if alternative == "greater":
        return float((np.sum(null_values >= observed) + 1) / (len(null_values) + 1))

    if alternative == "less":
        return float((np.sum(null_values <= observed) + 1) / (len(null_values) + 1))

    if alternative == "two-sided":
        null_center = float(np.nanmean(null_values))
        observed_distance = abs(observed - null_center)
        null_distance = np.abs(null_values - null_center)
        return float((np.sum(null_distance >= observed_distance) + 1) / (len(null_values) + 1))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")

def _fdr_bh(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return qvals

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    q = np.empty_like(q_ranked)
    q[order] = q_ranked
    qvals[finite] = q

    return qvals

def _holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    adj = np.full_like(pvals, np.nan, dtype=float)

    finite = np.isfinite(pvals)
    p = pvals[finite]

    if len(p) == 0:
        return adj

    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    ranked_adj = np.empty(m)

    for i, pval in enumerate(ranked):
        ranked_adj[i] = (m - i) * pval

    ranked_adj = np.maximum.accumulate(ranked_adj)
    ranked_adj = np.clip(ranked_adj, 0, 1)

    out = np.empty_like(ranked_adj)
    out[order] = ranked_adj
    adj[finite] = out

    return adj

def _p_for_text(p):
    if not np.isfinite(p):
        return "NA"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}".replace("0.", ".")

def _safe_filename(label):
    safe = str(label)
    safe = safe.replace(" ", "_")
    safe = safe.replace("|", "")
    safe = safe.replace(">", "gt")
    safe = safe.replace("<", "lt")
    safe = safe.replace("−", "minus")
    safe = safe.replace("-", "minus")
    safe = safe.replace("°", "deg")
    safe = safe.replace("/", "_")
    safe = re.sub(r"[^A-Za-z0-9_]+", "_", safe)
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe

def _offset_label(offset):
    if pd.isna(offset):
        return "Collapsed across offsets"
    return f"{int(float(offset))}° offset"

def _choose_primary_parametric_p(row, alternative):
    if alternative == "greater":
        return row.get("p_greater", np.nan)
    if alternative == "less":
        return row.get("p_less", np.nan)
    return row.get("p_two_sided", np.nan)

def _slope_from_subject_offset(df, value_col, offset_col="offset_deg"):
    d = (
        df[[offset_col, value_col]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    if len(d) < 3 or d[offset_col].nunique() < 2:
        return np.nan

    x = d[offset_col].astype(float).values
    y = d[value_col].astype(float).values

    x = x - np.nanmean(x)

    try:
        beta = np.linalg.lstsq(
            np.column_stack([np.ones_like(x), x]),
            y,
            rcond=None
        )[0]
        return float(beta[1])
    except Exception:
        return np.nan

def _apply_fdr_by_group(df, p_col, group_cols, out_col):
    df[out_col] = np.nan

    if df.empty or p_col not in df.columns:
        return df

    for _, idx in df.groupby(group_cols, dropna=False).groups.items():
        idx = list(idx)
        df.loc[idx, out_col] = _fdr_bh(df.loc[idx, p_col].values)

    return df

# ============================================================
# OUTCOME SPECIFICATION
# ============================================================

BREAKPOINT_PERM_OUTCOME_SPECS = [
    ("piecewise_supported_probability", "piecewise_supported", "mean", "Probability that piecewise model improved over linear drift"),
    ("stabilization_probability", "has_stabilizing_breakpoint", "mean", "Probability of accepted stabilizing breakpoint"),
    ("stable_from_start_probability", "stable_from_start", "mean", "Probability of stable-from-start trials"),
    ("stable_or_stabilized_probability", "stable_or_stabilized", "mean", "Probability of stable-from-start or stabilized trials"),
    ("nonconstant_model_probability", "nonconstant_model_supported", "mean", "Probability of nonconstant model support"),
    ("dynamic_probability", "dynamically_updating", "mean", "Probability of dynamically updating non-stabilized trials"),
    ("ambiguous_or_noisy_probability", "ambiguous_or_noisy", "mean", "Probability of ambiguous/noisy non-stabilized trials"),

    ("stabilization_time_sec", "stabilization_time_sec", "mean", "Time to accepted stabilizing breakpoint, seconds"),
    ("pre_slope_capture_per_sec", "pre_slope_capture_per_sec", "mean", "Pre-breakpoint visual-capture slope"),
    ("post_slope_capture_per_sec", "post_slope_capture_per_sec", "mean", "Post-breakpoint visual-capture slope"),
    ("abs_pre_slope_capture_per_sec", "abs_pre_slope_capture_per_sec", "mean", "Absolute pre-breakpoint slope"),
    ("abs_post_slope_capture_per_sec", "abs_post_slope_capture_per_sec", "mean", "Absolute post-breakpoint slope"),
    ("abs_slope_reduction_capture_per_sec", "abs_slope_reduction_capture_per_sec", "mean", "Absolute slope reduction"),

    ("breakpoint_capture_predicted", "breakpoint_capture_predicted", "mean", "Predicted capture at breakpoint"),
    ("breakpoint_capture_observed_nearest", "breakpoint_capture_observed_nearest", "mean", "Observed capture nearest breakpoint"),
    ("plateau_capture_observed", "plateau_capture_observed", "mean", "Observed plateau capture after breakpoint"),
    ("plateau_capture_predicted", "plateau_capture_predicted", "mean", "Predicted plateau capture after breakpoint"),
    ("plateau_capture_sd_observed", "plateau_capture_sd_observed", "mean", "Observed plateau capture SD"),
    ("post_model_residual_sd", "post_model_residual_sd", "mean", "Post-breakpoint model residual SD"),

    ("breakpoint_anchor_shift_toward_visual_deg", "breakpoint_anchor_shift_toward_visual_deg", "mean", "Auditory-estimate shift toward vision at breakpoint, deg"),
    ("plateau_anchor_shift_toward_visual_deg", "plateau_anchor_shift_toward_visual_deg", "mean", "Stabilized auditory-estimate shift toward vision, deg"),
    ("breakpoint_anchor_shift_abs_deg", "breakpoint_anchor_shift_abs_deg", "mean", "Absolute auditory-estimate shift at breakpoint, deg"),
    ("plateau_anchor_shift_abs_deg", "plateau_anchor_shift_abs_deg", "mean", "Absolute stabilized auditory-estimate shift, deg"),

    ("whole_trial_abs_linear_slope_capture_per_sec", "whole_trial_abs_linear_slope_capture_per_sec", "mean", "Whole-trial absolute linear capture slope"),
    ("mean_abs_capture_derivative_per_sec", "mean_abs_capture_derivative_per_sec", "mean", "Mean absolute visual-capture derivative"),
    ("late_mean_abs_capture_derivative_per_sec", "late_mean_abs_capture_derivative_per_sec", "mean", "Late mean absolute visual-capture derivative"),
    ("late_minus_early_abs_derivative_per_sec", "late_minus_early_abs_derivative_per_sec", "mean", "Late minus early absolute derivative"),

    ("whole_trial_abs_linear_slope_capture_per_sec_nonstabilized", "whole_trial_abs_linear_slope_capture_per_sec_nonstabilized", "mean", "Non-stabilized trials: whole-trial absolute linear capture slope"),
    ("mean_abs_capture_derivative_per_sec_nonstabilized", "mean_abs_capture_derivative_per_sec_nonstabilized", "mean", "Non-stabilized trials: mean absolute visual-capture derivative"),
    ("late_mean_abs_capture_derivative_per_sec_nonstabilized", "late_mean_abs_capture_derivative_per_sec_nonstabilized", "mean", "Non-stabilized trials: late absolute visual-capture derivative"),
    ("post_breakpoint_mean_abs_capture_derivative_per_sec_nonstabilized", "post_breakpoint_mean_abs_capture_derivative_per_sec_nonstabilized", "mean", "Non-stabilized trials: post-candidate-breakpoint absolute derivative"),
    ("candidate_abs_post_slope_capture_per_sec_nonstabilized", "candidate_abs_post_slope_capture_per_sec_nonstabilized", "mean", "Non-stabilized trials: absolute candidate post-breakpoint slope"),

    ("whole_trial_abs_linear_slope_capture_per_sec_dynamic_trials", "whole_trial_abs_linear_slope_capture_per_sec_dynamic_trials", "mean", "Dynamic trials: whole-trial absolute linear capture slope"),
    ("mean_abs_capture_derivative_per_sec_dynamic_trials", "mean_abs_capture_derivative_per_sec_dynamic_trials", "mean", "Dynamic trials: mean absolute visual-capture derivative"),
    ("late_mean_abs_capture_derivative_per_sec_dynamic_trials", "late_mean_abs_capture_derivative_per_sec_dynamic_trials", "mean", "Dynamic trials: late absolute visual-capture derivative"),
]

BREAKPOINT_PERM_OUTCOME_SPECS = [
    spec for spec in BREAKPOINT_PERM_OUTCOME_SPECS
    if spec[1] in breakpoint_trial_df.columns
]

print(f"\nPermutation outcome count: {len(BREAKPOINT_PERM_OUTCOME_SPECS)}")
print("Outcomes included:")
for outcome, source_col, _, label in BREAKPOINT_PERM_OUTCOME_SPECS:
    print(f"  - {outcome} <- {source_col}")

# ============================================================
# PART 1 — PREPARE TRIAL-LEVEL DATA AND OFFSET LABEL SHUFFLES
# ============================================================

TRIAL_KEY_COLS = [
    "subject",
    "source_file",
    "trialIndex",
    "trialType",
    "condLabel",
]

breakpoint_perm_trial_df = breakpoint_trial_df.copy()
breakpoint_perm_trial_df = breakpoint_perm_trial_df[
    breakpoint_perm_trial_df["offset_deg"].isin(FIXED_OFFSETS)
].copy()

breakpoint_perm_trial_df["offset_deg"] = breakpoint_perm_trial_df["offset_deg"].astype(int)

unique_trial_offset_df = (
    breakpoint_perm_trial_df[
        TRIAL_KEY_COLS + ["offset_deg"]
    ]
    .drop_duplicates()
    .copy()
)

if unique_trial_offset_df.duplicated(subset=TRIAL_KEY_COLS).any():
    duplicated = unique_trial_offset_df[
        unique_trial_offset_df.duplicated(subset=TRIAL_KEY_COLS, keep=False)
    ].sort_values(TRIAL_KEY_COLS)

    raise RuntimeError(
        "Some trial keys map to multiple offset labels. Inspect duplicated trial keys:\n"
        f"{duplicated.head(20)}"
    )

unique_trial_offset_csv = _safe_to_csv_perm(
    unique_trial_offset_df,
    "breakpoint_DYNAMIC_unique_trial_offset_labels_for_permutation.csv",
    index=False
)

print("\nPermutation trial-label input:")
print(f"Unique trials: {len(unique_trial_offset_df)}")
print(f"Subjects: {unique_trial_offset_df['subject'].nunique()}")
print("Offset counts:")
print(unique_trial_offset_df.groupby("offset_deg").size())
print(f"Saved: {unique_trial_offset_csv}")

def _shuffle_offsets_within_subject(unique_trial_df, rng, permuted_col="permuted_offset_deg"):
    out = unique_trial_df.copy()
    out[permuted_col] = np.nan

    for subj, idx in out.groupby("subject").groups.items():
        idx = list(idx)
        labels = out.loc[idx, "offset_deg"].astype(int).values.copy()

        if len(labels) > 1:
            out.loc[idx, permuted_col] = rng.permutation(labels)
        else:
            out.loc[idx, permuted_col] = labels

    out[permuted_col] = pd.to_numeric(out[permuted_col], errors="coerce").astype("Int64")

    return out

def _attach_permuted_offsets_to_trial_metrics(trial_metric_df, shuffled_trial_labels_df):
    use_labels = shuffled_trial_labels_df[
        TRIAL_KEY_COLS + ["permuted_offset_deg"]
    ].copy()

    out = trial_metric_df.merge(
        use_labels,
        on=TRIAL_KEY_COLS,
        how="left",
        validate="many_to_one"
    )

    out["permuted_offset_deg"] = pd.to_numeric(
        out["permuted_offset_deg"],
        errors="coerce"
    )

    return out

def _summarize_subject_offset_from_trial_metrics(trial_metric_df, offset_col="offset_deg"):
    d = trial_metric_df.copy()
    d[offset_col] = pd.to_numeric(d[offset_col], errors="coerce")
    d = d.dropna(subset=[offset_col]).copy()
    d[offset_col] = d[offset_col].astype(int)

    group_cols = ["metric", "metric_label", "subject", offset_col]

    rows = []

    for group_key, sub in d.groupby(group_cols, dropna=False):
        metric, metric_label, subject, offset = group_key

        row = {
            "metric": metric,
            "metric_label": metric_label,
            "subject": subject,
            "offset_deg": int(offset),
            "n_trials": int(sub["trialIndex"].nunique()),
        }

        for outcome, source_col, aggfunc, label in BREAKPOINT_PERM_OUTCOME_SPECS:
            vals = _clean_numeric(sub[source_col])

            if len(vals) == 0:
                row[outcome] = np.nan
            elif aggfunc == "mean":
                row[outcome] = float(np.nanmean(vals))
            elif aggfunc == "sum":
                row[outcome] = float(np.nansum(vals))
            else:
                raise ValueError(f"Unsupported aggfunc: {aggfunc}")

        rows.append(row)

    return pd.DataFrame(rows)

def _summarize_group_offset_empirical(subject_offset_df):
    rows = []

    for (metric, metric_label, offset), sub in subject_offset_df.groupby(
        ["metric", "metric_label", "offset_deg"],
        dropna=False
    ):
        for outcome, source_col, aggfunc, label in BREAKPOINT_PERM_OUTCOME_SPECS:
            if outcome not in sub.columns:
                continue

            stats_dict = _mean_ci_test(sub[outcome], popmean=0.0)

            primary_param_p = _choose_primary_parametric_p(
                stats_dict,
                BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE
            )

            rows.append({
                "analysis": "offset_empirical_value_condition_label_shuffle",
                "metric": metric,
                "metric_label": metric_label,
                "offset_deg": int(offset),
                "offset_label": _offset_label(offset),
                "outcome": outcome,
                "source_col": source_col,
                "label": label,
                "observed_effect": stats_dict["mean"],
                "observed_ci95_low": stats_dict["ci95_low"],
                "observed_ci95_high": stats_dict["ci95_high"],
                "observed_se": stats_dict["se"],
                "observed_t": stats_dict["t"],
                "observed_df": stats_dict["df"],
                "observed_n_subjects": stats_dict["n"],
                "parametric_p_two_sided": stats_dict["p_two_sided"],
                "parametric_p_greater": stats_dict["p_greater"],
                "parametric_p_less": stats_dict["p_less"],
                "parametric_p_primary": primary_param_p,
            })

    return pd.DataFrame(rows)

def _summarize_offset_slopes(subject_offset_df):
    rows = []

    for (metric, metric_label), sub_metric in subject_offset_df.groupby(
        ["metric", "metric_label"],
        dropna=False
    ):
        for outcome, source_col, aggfunc, label in BREAKPOINT_PERM_OUTCOME_SPECS:
            if outcome not in sub_metric.columns:
                continue

            d = (
                sub_metric[["subject", "offset_deg", outcome]]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
                .copy()
            )

            if len(d) < 8 or d["subject"].nunique() < 3 or d["offset_deg"].nunique() < 2:
                continue

            slope = _slope_from_subject_offset(d, value_col=outcome, offset_col="offset_deg")

            param_est = np.nan
            param_se = np.nan
            param_t = np.nan
            param_p = np.nan
            r2 = np.nan
            n_rows = len(d)
            n_subjects = d["subject"].nunique()

            if smf is not None:
                try:
                    d["offset_c"] = d["offset_deg"] - d["offset_deg"].mean()
                    fit = smf.ols(
                        f"{outcome} ~ offset_c",
                        data=d
                    ).fit(
                        cov_type="cluster",
                        cov_kwds={"groups": d["subject"]}
                    )

                    param_est = float(fit.params.get("offset_c", np.nan))
                    param_se = float(fit.bse.get("offset_c", np.nan))
                    param_t = float(fit.tvalues.get("offset_c", np.nan))
                    param_p = float(fit.pvalues.get("offset_c", np.nan))
                    r2 = float(fit.rsquared)

                except Exception:
                    pass

            rows.append({
                "analysis": "offset_slope_condition_label_shuffle",
                "metric": metric,
                "metric_label": metric_label,
                "outcome": outcome,
                "source_col": source_col,
                "label": label,
                "observed_slope": slope,
                "observed_clustered_estimate": param_est,
                "observed_clustered_se": param_se,
                "observed_clustered_t": param_t,
                "observed_clustered_p": param_p,
                "observed_r_squared": r2,
                "n_rows": n_rows,
                "n_subjects": n_subjects,
            })

    return pd.DataFrame(rows)

# ============================================================
# PART 2 — OBSERVED EMPIRICAL VALUES AND SLOPES
# ============================================================

observed_breakpoint_subject_offset_for_perm_df = _summarize_subject_offset_from_trial_metrics(
    breakpoint_perm_trial_df,
    offset_col="offset_deg"
)

observed_breakpoint_subject_offset_for_perm_csv = _safe_to_csv_perm(
    observed_breakpoint_subject_offset_for_perm_df,
    "breakpoint_DYNAMIC_observed_subject_offset_values_for_permutation.csv",
    index=False
)

observed_offset_empirical_df = _summarize_group_offset_empirical(
    observed_breakpoint_subject_offset_for_perm_df
)

observed_offset_slopes_df = _summarize_offset_slopes(
    observed_breakpoint_subject_offset_for_perm_df
)

observed_offset_empirical_csv = _safe_to_csv_perm(
    observed_offset_empirical_df,
    "breakpoint_DYNAMIC_observed_offset_empirical_values.csv",
    index=False
)

observed_offset_slopes_csv = _safe_to_csv_perm(
    observed_offset_slopes_df,
    "breakpoint_DYNAMIC_observed_offset_slope_values.csv",
    index=False
)

print("\nObserved empirical offset values:")
try:
    display(observed_offset_empirical_df)
except NameError:
    print(observed_offset_empirical_df.head(30).to_string(index=False))

print("\nObserved offset slopes:")
try:
    display(observed_offset_slopes_df)
except NameError:
    print(observed_offset_slopes_df.head(30).to_string(index=False))

# ============================================================
# PART 3 — WHOLE-TRIAL CONDITION-LABEL SHUFFLE NULL
# ============================================================

rng_label = np.random.default_rng(RANDOM_SEED_BREAKPOINT_PERM)

label_shuffle_empirical_null_rows = []
label_shuffle_slope_null_rows = []

for perm_i in range(N_PERM_BREAKPOINT_LABEL_SHUFFLE):

    shuffled_labels = _shuffle_offsets_within_subject(
        unique_trial_offset_df,
        rng=rng_label,
        permuted_col="permuted_offset_deg"
    )

    perm_trial_metric_df = _attach_permuted_offsets_to_trial_metrics(
        breakpoint_perm_trial_df,
        shuffled_labels
    )

    perm_subject_offset_df = _summarize_subject_offset_from_trial_metrics(
        perm_trial_metric_df,
        offset_col="permuted_offset_deg"
    )

    perm_empirical_df = _summarize_group_offset_empirical(
        perm_subject_offset_df
    )

    if not perm_empirical_df.empty:
        for _, row in perm_empirical_df.iterrows():
            label_shuffle_empirical_null_rows.append({
                "perm_i": perm_i,
                "metric": row["metric"],
                "metric_label": row["metric_label"],
                "offset_deg": row["offset_deg"],
                "offset_label": row["offset_label"],
                "outcome": row["outcome"],
                "label": row["label"],
                "null_effect": row["observed_effect"],
                "null_t": row["observed_t"],
                "null_n_subjects": row["observed_n_subjects"],
            })

    perm_slopes_df = _summarize_offset_slopes(
        perm_subject_offset_df
    )

    if not perm_slopes_df.empty:
        for _, row in perm_slopes_df.iterrows():
            label_shuffle_slope_null_rows.append({
                "perm_i": perm_i,
                "metric": row["metric"],
                "metric_label": row["metric_label"],
                "outcome": row["outcome"],
                "label": row["label"],
                "null_slope": row["observed_slope"],
            })

    if (perm_i + 1) % 250 == 0:
        print(
            f"Completed breakpoint/dynamic whole-trial condition-label shuffle "
            f"{perm_i + 1}/{N_PERM_BREAKPOINT_LABEL_SHUFFLE}"
        )

breakpoint_label_shuffle_empirical_null_df = pd.DataFrame(label_shuffle_empirical_null_rows)
breakpoint_label_shuffle_slope_null_df = pd.DataFrame(label_shuffle_slope_null_rows)

breakpoint_label_shuffle_empirical_null_csv = _safe_to_csv_perm(
    breakpoint_label_shuffle_empirical_null_df,
    "breakpoint_DYNAMIC_condition_label_shuffle_empirical_null_long.csv",
    index=False
)

breakpoint_label_shuffle_slope_null_csv = _safe_to_csv_perm(
    breakpoint_label_shuffle_slope_null_df,
    "breakpoint_DYNAMIC_condition_label_shuffle_slope_null_long.csv",
    index=False
)

# ============================================================
# PART 4 — PERMUTATION P VALUES FOR OFFSET EMPIRICAL VALUES
# ============================================================

empirical_test_rows = []

for _, obs in observed_offset_empirical_df.iterrows():

    null_sub = breakpoint_label_shuffle_empirical_null_df[
        (breakpoint_label_shuffle_empirical_null_df["metric"] == obs["metric"])
        & (breakpoint_label_shuffle_empirical_null_df["offset_deg"] == obs["offset_deg"])
        & (breakpoint_label_shuffle_empirical_null_df["outcome"] == obs["outcome"])
    ].copy()

    null_effects = _clean_numeric(null_sub["null_effect"])
    null_t = _clean_numeric(null_sub["null_t"])

    if len(null_effects) > 0:
        null_effect_mean = float(np.nanmean(null_effects))
        null_effect_sd = float(np.nanstd(null_effects, ddof=1))
        null_effect_ci95_low = float(np.nanpercentile(null_effects, 2.5))
        null_effect_ci95_high = float(np.nanpercentile(null_effects, 97.5))
    else:
        null_effect_mean = np.nan
        null_effect_sd = np.nan
        null_effect_ci95_low = np.nan
        null_effect_ci95_high = np.nan

    p_effect = _empirical_pvalue(
        obs["observed_effect"],
        null_effects,
        alternative=BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE
    )

    p_t = _empirical_pvalue(
        obs["observed_t"],
        null_t,
        alternative=BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE
    )

    empirical_test_rows.append({
        "null_type": "within_subject_whole_trial_condition_label_shuffle",
        "analysis": "offset_empirical_value_condition_label_shuffle",
        "metric": obs["metric"],
        "metric_label": obs["metric_label"],
        "offset_deg": obs["offset_deg"],
        "offset_label": obs["offset_label"],
        "outcome": obs["outcome"],
        "label": obs["label"],
        "observed_effect": obs["observed_effect"],
        "observed_ci95_low": obs["observed_ci95_low"],
        "observed_ci95_high": obs["observed_ci95_high"],
        "observed_se": obs["observed_se"],
        "observed_t": obs["observed_t"],
        "observed_df": obs["observed_df"],
        "observed_n_subjects": obs["observed_n_subjects"],
        "parametric_p_two_sided": obs["parametric_p_two_sided"],
        "parametric_p_greater": obs["parametric_p_greater"],
        "parametric_p_less": obs["parametric_p_less"],
        "parametric_p_primary": obs["parametric_p_primary"],
        "null_effect_mean": null_effect_mean,
        "null_effect_sd": null_effect_sd,
        "null_effect_ci95_low": null_effect_ci95_low,
        "null_effect_ci95_high": null_effect_ci95_high,
        "permutation_p_effect": p_effect,
        "permutation_p_t": p_t,
        "alternative": BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE,
        "n_permutations": N_PERM_BREAKPOINT_LABEL_SHUFFLE,
    })

breakpoint_label_shuffle_empirical_tests_df = pd.DataFrame(empirical_test_rows)

for p_col in ["permutation_p_effect", "permutation_p_t"]:
    breakpoint_label_shuffle_empirical_tests_df[f"{p_col}_fdr_all_tests"] = _fdr_bh(
        breakpoint_label_shuffle_empirical_tests_df[p_col].values
    )

    breakpoint_label_shuffle_empirical_tests_df[f"{p_col}_holm_all_tests"] = _holm_bonferroni(
        breakpoint_label_shuffle_empirical_tests_df[p_col].values
    )

    breakpoint_label_shuffle_empirical_tests_df = _apply_fdr_by_group(
        breakpoint_label_shuffle_empirical_tests_df,
        p_col=p_col,
        group_cols=["metric", "outcome"],
        out_col=f"{p_col}_fdr_within_metric_outcome"
    )

    breakpoint_label_shuffle_empirical_tests_df = _apply_fdr_by_group(
        breakpoint_label_shuffle_empirical_tests_df,
        p_col=p_col,
        group_cols=["outcome"],
        out_col=f"{p_col}_fdr_within_outcome"
    )

breakpoint_label_shuffle_empirical_tests_csv = _safe_to_csv_perm(
    breakpoint_label_shuffle_empirical_tests_df,
    "breakpoint_DYNAMIC_condition_label_shuffle_empirical_tests.csv",
    index=False
)

# ============================================================
# PART 5 — PERMUTATION P VALUES FOR OFFSET SLOPES
# ============================================================

slope_test_rows = []

for _, obs in observed_offset_slopes_df.iterrows():

    null_sub = breakpoint_label_shuffle_slope_null_df[
        (breakpoint_label_shuffle_slope_null_df["metric"] == obs["metric"])
        & (breakpoint_label_shuffle_slope_null_df["outcome"] == obs["outcome"])
    ].copy()

    null_slopes = _clean_numeric(null_sub["null_slope"])

    if len(null_slopes) > 0:
        null_slope_mean = float(np.nanmean(null_slopes))
        null_slope_sd = float(np.nanstd(null_slopes, ddof=1))
        null_slope_ci95_low = float(np.nanpercentile(null_slopes, 2.5))
        null_slope_ci95_high = float(np.nanpercentile(null_slopes, 97.5))
    else:
        null_slope_mean = np.nan
        null_slope_sd = np.nan
        null_slope_ci95_low = np.nan
        null_slope_ci95_high = np.nan

    p_slope = _empirical_pvalue(
        obs["observed_slope"],
        null_slopes,
        alternative=BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE
    )

    slope_test_rows.append({
        "null_type": "within_subject_whole_trial_condition_label_shuffle",
        "analysis": "offset_slope_condition_label_shuffle",
        "metric": obs["metric"],
        "metric_label": obs["metric_label"],
        "outcome": obs["outcome"],
        "label": obs["label"],
        "observed_slope": obs["observed_slope"],
        "observed_clustered_estimate": obs["observed_clustered_estimate"],
        "observed_clustered_se": obs["observed_clustered_se"],
        "observed_clustered_t": obs["observed_clustered_t"],
        "observed_clustered_p": obs["observed_clustered_p"],
        "observed_r_squared": obs["observed_r_squared"],
        "n_rows": obs["n_rows"],
        "n_subjects": obs["n_subjects"],
        "null_slope_mean": null_slope_mean,
        "null_slope_sd": null_slope_sd,
        "null_slope_ci95_low": null_slope_ci95_low,
        "null_slope_ci95_high": null_slope_ci95_high,
        "permutation_p_slope": p_slope,
        "alternative": BREAKPOINT_LABEL_SHUFFLE_ALTERNATIVE,
        "n_permutations": N_PERM_BREAKPOINT_LABEL_SHUFFLE,
    })

breakpoint_label_shuffle_slope_tests_df = pd.DataFrame(slope_test_rows)

breakpoint_label_shuffle_slope_tests_df["permutation_p_slope_fdr_all_tests"] = _fdr_bh(
    breakpoint_label_shuffle_slope_tests_df["permutation_p_slope"].values
)

breakpoint_label_shuffle_slope_tests_df["permutation_p_slope_holm_all_tests"] = _holm_bonferroni(
    breakpoint_label_shuffle_slope_tests_df["permutation_p_slope"].values
)

breakpoint_label_shuffle_slope_tests_df = _apply_fdr_by_group(
    breakpoint_label_shuffle_slope_tests_df,
    p_col="permutation_p_slope",
    group_cols=["metric"],
    out_col="permutation_p_slope_fdr_within_metric"
)

breakpoint_label_shuffle_slope_tests_csv = _safe_to_csv_perm(
    breakpoint_label_shuffle_slope_tests_df,
    "breakpoint_DYNAMIC_condition_label_shuffle_slope_tests.csv",
    index=False
)

print("\nCondition-label shuffle empirical tests:")
try:
    display(breakpoint_label_shuffle_empirical_tests_df)
except NameError:
    print(breakpoint_label_shuffle_empirical_tests_df.head(40).to_string(index=False))

print("\nCondition-label shuffle offset-slope tests:")
try:
    display(breakpoint_label_shuffle_slope_tests_df)
except NameError:
    print(breakpoint_label_shuffle_slope_tests_df.head(40).to_string(index=False))

# ============================================================
# PART 6 — CLUSTER-BASED PERMUTATION ANALYSIS OVER TIME
# ============================================================

WINDOW_TRIAL_KEY_COLS = [
    "subject",
    "source_file",
    "trialIndex",
    "trialType",
    "condLabel",
]

window_perm_df = window_df.copy()
window_perm_df = window_perm_df[
    window_perm_df["offset_deg"].isin(FIXED_OFFSETS)
].copy()
window_perm_df["offset_deg"] = window_perm_df["offset_deg"].astype(int)
window_perm_df["time_sec_rounded"] = window_perm_df["window_center_sec"].astype(float).round(6)

window_metric_names = [m for m, _ in BREAKPOINT_METRICS if m in window_perm_df.columns]

if len(window_metric_names) == 0:
    raise RuntimeError(
        "None of the BREAKPOINT_METRICS were found in window_df. "
        f"Expected one of: {[m for m, _ in BREAKPOINT_METRICS]}"
    )

unique_window_trial_offset_df = (
    window_perm_df[
        WINDOW_TRIAL_KEY_COLS + ["offset_deg"]
    ]
    .drop_duplicates()
    .copy()
)

if unique_window_trial_offset_df.duplicated(subset=WINDOW_TRIAL_KEY_COLS).any():
    duplicated = unique_window_trial_offset_df[
        unique_window_trial_offset_df.duplicated(subset=WINDOW_TRIAL_KEY_COLS, keep=False)
    ].sort_values(WINDOW_TRIAL_KEY_COLS)

    raise RuntimeError(
        "Some window trial keys map to multiple offset labels. Inspect duplicated keys:\n"
        f"{duplicated.head(20)}"
    )

def _attach_permuted_offsets_to_window_df(window_metric_df, shuffled_trial_labels_df):
    use_labels = shuffled_trial_labels_df[
        WINDOW_TRIAL_KEY_COLS + ["permuted_offset_deg"]
    ].copy()

    out = window_metric_df.merge(
        use_labels,
        on=WINDOW_TRIAL_KEY_COLS,
        how="left",
        validate="many_to_one"
    )

    out["permuted_offset_deg"] = pd.to_numeric(
        out["permuted_offset_deg"],
        errors="coerce"
    )

    return out

def _compute_group_timecourse(window_metric_df, metric, offset_col="offset_deg"):
    d = window_metric_df.copy()
    d[offset_col] = pd.to_numeric(d[offset_col], errors="coerce")
    d = d.dropna(subset=[offset_col, "time_sec_rounded", metric]).copy()
    d[offset_col] = d[offset_col].astype(int)

    subj_offset_time = (
        d
        .groupby(["subject", offset_col, "time_sec_rounded"], as_index=False)
        .agg(value=(metric, "mean"))
        .rename(columns={offset_col: "offset_deg"})
    )

    group_tc = (
        subj_offset_time
        .groupby(["offset_deg", "time_sec_rounded"], as_index=False)
        .agg(
            mean=("value", "mean"),
            se=("value", lambda x: np.nanstd(x, ddof=1) / np.sqrt(np.sum(np.isfinite(x))) if np.sum(np.isfinite(x)) > 1 else np.nan),
            n_subjects=("subject", "nunique")
        )
    )

    return subj_offset_time, group_tc

def _build_null_timecourse_array(null_long_df, metric, offset):
    d = null_long_df[
        (null_long_df["metric"] == metric)
        & (null_long_df["offset_deg"] == offset)
    ].copy()

    if d.empty:
        return np.array([]), np.empty((0, 0))

    times = np.sort(d["time_sec"].unique().astype(float))
    perms = np.sort(d["perm_i"].unique().astype(int))

    wide = d.pivot_table(
        index="perm_i",
        columns="time_sec",
        values="null_mean",
        aggfunc="mean"
    )

    wide = wide.reindex(index=perms, columns=times)

    return times, wide.values.astype(float)

def _timecourse_empirical_pvalues(observed_curve, null_matrix, alternative="two-sided"):
    observed_curve = np.asarray(observed_curve, dtype=float)
    null_matrix = np.asarray(null_matrix, dtype=float)

    n_time = len(observed_curve)
    p_vals = np.full(n_time, np.nan)
    null_mean = np.full(n_time, np.nan)
    null_sd = np.full(n_time, np.nan)
    z_obs = np.full(n_time, np.nan)

    for j in range(n_time):
        null_vals = _clean_numeric(null_matrix[:, j])

        if len(null_vals) == 0 or not np.isfinite(observed_curve[j]):
            continue

        null_mean[j] = float(np.nanmean(null_vals))
        null_sd[j] = float(np.nanstd(null_vals, ddof=1))

        if null_sd[j] > 0 and np.isfinite(null_sd[j]):
            z_obs[j] = (observed_curve[j] - null_mean[j]) / null_sd[j]

        p_vals[j] = _empirical_pvalue(
            observed_curve[j],
            null_vals,
            alternative=alternative
        )

    return p_vals, null_mean, null_sd, z_obs

def _find_contiguous_clusters(mask):
    mask = np.asarray(mask, dtype=bool)
    clusters = []
    start = None

    for i, val in enumerate(mask):
        if val and start is None:
            start = i

        if (not val or i == len(mask) - 1) and start is not None:
            end = i if not val else i + 1
            clusters.append(np.arange(start, end))
            start = None

    return clusters

def _cluster_stat_from_z(z_vals, cluster_idx, alternative="two-sided"):
    vals = np.asarray(z_vals, dtype=float)[cluster_idx]
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan

    if alternative == "greater":
        return float(np.sum(vals))

    if alternative == "less":
        return float(np.sum(-vals))

    if alternative == "two-sided":
        return float(np.sum(np.abs(vals)))

    raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")

def _cluster_mask_from_p_z(p_vals, z_vals, alpha=0.05, alternative="two-sided"):
    p_vals = np.asarray(p_vals, dtype=float)
    z_vals = np.asarray(z_vals, dtype=float)

    mask = np.isfinite(p_vals) & (p_vals < alpha)

    if alternative == "greater":
        mask = mask & (z_vals > 0)
    elif alternative == "less":
        mask = mask & (z_vals < 0)
    elif alternative == "two-sided":
        mask = mask & np.isfinite(z_vals)
    else:
        raise ValueError("alternative must be 'greater', 'less', or 'two-sided'.")

    return mask

def _cluster_correct_from_label_shuffle_null(
    observed_curve,
    null_matrix,
    timepoints,
    alternative="two-sided",
    cluster_alpha=0.05
):
    observed_curve = np.asarray(observed_curve, dtype=float)
    null_matrix = np.asarray(null_matrix, dtype=float)
    timepoints = np.asarray(timepoints, dtype=float)

    if null_matrix.size == 0 or len(timepoints) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    p_obs, null_mean, null_sd, z_obs = _timecourse_empirical_pvalues(
        observed_curve,
        null_matrix,
        alternative=alternative
    )

    obs_cluster_mask = _cluster_mask_from_p_z(
        p_obs,
        z_obs,
        alpha=cluster_alpha,
        alternative=alternative
    )

    obs_clusters = _find_contiguous_clusters(obs_cluster_mask)

    n_perm, n_time = null_matrix.shape
    max_cluster_stats = np.zeros(n_perm, dtype=float)

    for perm_i in range(n_perm):
        perm_curve = null_matrix[perm_i, :].astype(float)

        z_perm = np.full(n_time, np.nan)
        p_perm = np.full(n_time, np.nan)

        for j in range(n_time):
            null_vals = _clean_numeric(null_matrix[:, j])

            if len(null_vals) == 0 or not np.isfinite(perm_curve[j]):
                continue

            sd_j = np.nanstd(null_vals, ddof=1)
            mean_j = np.nanmean(null_vals)

            if sd_j > 0 and np.isfinite(sd_j):
                z_perm[j] = (perm_curve[j] - mean_j) / sd_j

            p_perm[j] = _empirical_pvalue(
                perm_curve[j],
                null_vals,
                alternative=alternative
            )

        perm_mask = _cluster_mask_from_p_z(
            p_perm,
            z_perm,
            alpha=cluster_alpha,
            alternative=alternative
        )

        perm_clusters = _find_contiguous_clusters(perm_mask)

        if len(perm_clusters) == 0:
            max_cluster_stats[perm_i] = 0.0
        else:
            perm_cluster_stats = [
                _cluster_stat_from_z(z_perm, cidx, alternative=alternative)
                for cidx in perm_clusters
            ]
            max_cluster_stats[perm_i] = float(np.nanmax(perm_cluster_stats))

    cluster_rows = []

    for cluster_i, cidx in enumerate(obs_clusters, start=1):
        c_stat = _cluster_stat_from_z(z_obs, cidx, alternative=alternative)

        p_cluster = float(
            (np.sum(max_cluster_stats >= c_stat) + 1)
            / (len(max_cluster_stats) + 1)
        )

        cluster_rows.append({
            "cluster_id": cluster_i,
            "start_time_sec": float(timepoints[cidx[0]]),
            "end_time_sec": float(timepoints[cidx[-1]]),
            "n_timepoints": int(len(cidx)),
            "cluster_stat": c_stat,
            "cluster_corrected_p": p_cluster,
            "mean_observed_in_cluster": float(np.nanmean(observed_curve[cidx])),
            "mean_null_in_cluster": float(np.nanmean(null_mean[cidx])),
            "mean_observed_minus_null_in_cluster": float(np.nanmean(observed_curve[cidx] - null_mean[cidx])),
            "max_abs_z_in_cluster": float(np.nanmax(np.abs(z_obs[cidx]))),
            "peak_z_in_cluster": float(np.nanmax(z_obs[cidx])),
            "min_z_in_cluster": float(np.nanmin(z_obs[cidx])),
        })

    binwise_df = pd.DataFrame({
        "time_sec": timepoints,
        "observed_mean": observed_curve,
        "null_mean": null_mean,
        "observed_minus_null": observed_curve - null_mean,
        "null_sd": null_sd,
        "z_observed": z_obs,
        "p_uncorrected": p_obs,
        "passes_cluster_forming_threshold": obs_cluster_mask,
    })

    null_max_df = pd.DataFrame({
        "perm_i": np.arange(n_perm),
        "max_cluster_stat": max_cluster_stats,
    })

    cluster_df = pd.DataFrame(cluster_rows)

    return binwise_df, cluster_df, null_max_df

# Observed timecourses
observed_timecourse_subject_rows = []
observed_timecourse_group_rows = []

for metric in window_metric_names:
    subj_tc, group_tc = _compute_group_timecourse(
        window_perm_df,
        metric=metric,
        offset_col="offset_deg"
    )

    subj_tc["metric"] = metric
    group_tc["metric"] = metric

    observed_timecourse_subject_rows.append(subj_tc)
    observed_timecourse_group_rows.append(group_tc)

observed_timecourse_subject_df = pd.concat(observed_timecourse_subject_rows, ignore_index=True)
observed_timecourse_group_df = pd.concat(observed_timecourse_group_rows, ignore_index=True)

observed_timecourse_subject_csv = _safe_to_csv_perm(
    observed_timecourse_subject_df,
    "breakpoint_DYNAMIC_observed_subject_offset_timecourses.csv",
    index=False
)

observed_timecourse_group_csv = _safe_to_csv_perm(
    observed_timecourse_group_df,
    "breakpoint_DYNAMIC_observed_group_offset_timecourses.csv",
    index=False
)

# Null timecourses
rng_cluster = np.random.default_rng(RANDOM_SEED_BREAKPOINT_PERM + 900000)

cluster_null_rows = []

for perm_i in range(N_PERM_BREAKPOINT_CLUSTER):

    shuffled_labels = _shuffle_offsets_within_subject(
        unique_window_trial_offset_df,
        rng=rng_cluster,
        permuted_col="permuted_offset_deg"
    )

    perm_window_df = _attach_permuted_offsets_to_window_df(
        window_perm_df,
        shuffled_labels
    )

    for metric in window_metric_names:
        _, perm_group_tc = _compute_group_timecourse(
            perm_window_df,
            metric=metric,
            offset_col="permuted_offset_deg"
        )

        if perm_group_tc.empty:
            continue

        for _, row in perm_group_tc.iterrows():
            cluster_null_rows.append({
                "perm_i": perm_i,
                "metric": metric,
                "offset_deg": int(row["offset_deg"]),
                "offset_label": _offset_label(row["offset_deg"]),
                "time_sec": float(row["time_sec_rounded"]),
                "null_mean": row["mean"],
                "null_n_subjects": row["n_subjects"],
            })

    if (perm_i + 1) % 100 == 0:
        print(
            f"Completed breakpoint/dynamic cluster condition-label shuffle "
            f"{perm_i + 1}/{N_PERM_BREAKPOINT_CLUSTER}"
        )

breakpoint_cluster_null_timecourse_df = pd.DataFrame(cluster_null_rows)

breakpoint_cluster_null_timecourse_csv = _safe_to_csv_perm(
    breakpoint_cluster_null_timecourse_df,
    "breakpoint_DYNAMIC_cluster_condition_label_shuffle_null_timecourses_long.csv",
    index=False
)

# Cluster tests
cluster_binwise_rows = []
cluster_corrected_rows = []
cluster_max_null_rows = []

for metric in window_metric_names:
    for offset in FIXED_OFFSETS:

        obs_sub = observed_timecourse_group_df[
            (observed_timecourse_group_df["metric"] == metric)
            & (observed_timecourse_group_df["offset_deg"] == offset)
        ].copy()

        if obs_sub.empty:
            continue

        obs_sub = obs_sub.sort_values("time_sec_rounded")
        obs_times = obs_sub["time_sec_rounded"].astype(float).values

        null_times, null_matrix = _build_null_timecourse_array(
            breakpoint_cluster_null_timecourse_df,
            metric=metric,
            offset=offset
        )

        if null_matrix.size == 0:
            continue

        common_times = np.intersect1d(obs_times, null_times)

        if len(common_times) == 0:
            continue

        obs_aligned = (
            obs_sub
            .set_index("time_sec_rounded")
            .reindex(common_times)["mean"]
            .astype(float)
            .values
        )

        null_df_for_metric_offset = pd.DataFrame(
            null_matrix,
            columns=null_times
        )

        null_aligned = null_df_for_metric_offset[common_times].values.astype(float)

        binwise_df, cluster_df, max_null_df = _cluster_correct_from_label_shuffle_null(
            observed_curve=obs_aligned,
            null_matrix=null_aligned,
            timepoints=common_times,
            alternative=BREAKPOINT_CLUSTER_ALTERNATIVE,
            cluster_alpha=CLUSTER_FORMING_ALPHA
        )

        if not binwise_df.empty:
            binwise_df["analysis"] = "cluster_timecourse_condition_label_shuffle"
            binwise_df["metric"] = metric
            binwise_df["offset_deg"] = offset
            binwise_df["offset_label"] = _offset_label(offset)
            binwise_df["alternative"] = BREAKPOINT_CLUSTER_ALTERNATIVE
            binwise_df["n_permutations"] = N_PERM_BREAKPOINT_CLUSTER

            binwise_df["p_uncorrected_fdr_within_metric_offset"] = _fdr_bh(
                binwise_df["p_uncorrected"].values
            )

            cluster_binwise_rows.append(binwise_df)

        if not cluster_df.empty:
            cluster_df["analysis"] = "cluster_timecourse_condition_label_shuffle"
            cluster_df["metric"] = metric
            cluster_df["offset_deg"] = offset
            cluster_df["offset_label"] = _offset_label(offset)
            cluster_df["alternative"] = BREAKPOINT_CLUSTER_ALTERNATIVE
            cluster_df["cluster_forming_alpha"] = CLUSTER_FORMING_ALPHA
            cluster_df["n_permutations"] = N_PERM_BREAKPOINT_CLUSTER

            cluster_corrected_rows.append(cluster_df)

        if not max_null_df.empty:
            max_null_df["analysis"] = "cluster_timecourse_condition_label_shuffle"
            max_null_df["metric"] = metric
            max_null_df["offset_deg"] = offset
            max_null_df["offset_label"] = _offset_label(offset)
            max_null_df["alternative"] = BREAKPOINT_CLUSTER_ALTERNATIVE

            cluster_max_null_rows.append(max_null_df)

        print(f"Completed cluster test: {metric}, {offset}°")

breakpoint_cluster_binwise_df = (
    pd.concat(cluster_binwise_rows, ignore_index=True)
    if len(cluster_binwise_rows) > 0
    else pd.DataFrame()
)

breakpoint_cluster_corrected_df = (
    pd.concat(cluster_corrected_rows, ignore_index=True)
    if len(cluster_corrected_rows) > 0
    else pd.DataFrame()
)

breakpoint_cluster_max_null_df = (
    pd.concat(cluster_max_null_rows, ignore_index=True)
    if len(cluster_max_null_rows) > 0
    else pd.DataFrame()
)

if not breakpoint_cluster_binwise_df.empty:
    breakpoint_cluster_binwise_df["p_uncorrected_fdr_all_time_tests"] = _fdr_bh(
        breakpoint_cluster_binwise_df["p_uncorrected"].values
    )

    breakpoint_cluster_binwise_df["p_uncorrected_holm_all_time_tests"] = _holm_bonferroni(
        breakpoint_cluster_binwise_df["p_uncorrected"].values
    )

if not breakpoint_cluster_corrected_df.empty:
    breakpoint_cluster_corrected_df["cluster_corrected_p_fdr_all_clusters"] = _fdr_bh(
        breakpoint_cluster_corrected_df["cluster_corrected_p"].values
    )

    breakpoint_cluster_corrected_df["cluster_corrected_p_holm_all_clusters"] = _holm_bonferroni(
        breakpoint_cluster_corrected_df["cluster_corrected_p"].values
    )

    breakpoint_cluster_corrected_df = _apply_fdr_by_group(
        breakpoint_cluster_corrected_df,
        p_col="cluster_corrected_p",
        group_cols=["metric"],
        out_col="cluster_corrected_p_fdr_within_metric"
    )

breakpoint_cluster_binwise_csv = _safe_to_csv_perm(
    breakpoint_cluster_binwise_df,
    "breakpoint_DYNAMIC_cluster_binwise_timecourse_tests.csv",
    index=False
)

breakpoint_cluster_corrected_csv = _safe_to_csv_perm(
    breakpoint_cluster_corrected_df,
    "breakpoint_DYNAMIC_cluster_corrected_timecourse_tests.csv",
    index=False
)

breakpoint_cluster_max_null_csv = _safe_to_csv_perm(
    breakpoint_cluster_max_null_df,
    "breakpoint_DYNAMIC_cluster_max_null_distribution.csv",
    index=False
)

print("\nCluster-corrected timecourse tests:")
try:
    display(breakpoint_cluster_corrected_df)
except NameError:
    print(breakpoint_cluster_corrected_df.to_string(index=False))

# ============================================================
# PART 7 — CLUSTER PLOTS
# ============================================================

def _plot_cluster_timecourse(metric, offset):
    obs = observed_timecourse_group_df[
        (observed_timecourse_group_df["metric"] == metric)
        & (observed_timecourse_group_df["offset_deg"] == offset)
    ].copy()

    bins = breakpoint_cluster_binwise_df[
        (breakpoint_cluster_binwise_df["metric"] == metric)
        & (breakpoint_cluster_binwise_df["offset_deg"] == offset)
    ].copy()

    clusters = breakpoint_cluster_corrected_df[
        (breakpoint_cluster_corrected_df["metric"] == metric)
        & (breakpoint_cluster_corrected_df["offset_deg"] == offset)
    ].copy()

    if obs.empty or bins.empty:
        return

    obs = obs.sort_values("time_sec_rounded")
    bins = bins.sort_values("time_sec")

    fig, ax = plt.subplots(figsize=(8.8, 5.0))

    x = obs["time_sec_rounded"].astype(float).values
    y = obs["mean"].astype(float).values
    se = obs["se"].astype(float).values

    ax.plot(x, y, linewidth=2.3, label="Observed")
    ax.fill_between(x, y - se, y + se, alpha=0.20)

    ax.plot(
        bins["time_sec"].astype(float).values,
        bins["null_mean"].astype(float).values,
        linestyle="--",
        linewidth=1.8,
        label="Condition-label null mean"
    )

    ax.axhline(0, linestyle="--", linewidth=1, alpha=0.5)
    ax.axhline(0.5, linestyle="--", linewidth=1, alpha=0.4)
    ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.4)

    sig_clusters = clusters[
        clusters["cluster_corrected_p"] < 0.05
    ].copy()

    if not sig_clusters.empty:
        y_min, y_max = ax.get_ylim()
        y_pos = y_max - 0.08 * (y_max - y_min)
        label_added = False

        for _, crow in sig_clusters.iterrows():
            ax.axvspan(
                crow["start_time_sec"],
                crow["end_time_sec"],
                alpha=0.12,
                label="cluster p < .05" if not label_added else "_nolegend_"
            )

            ax.text(
                x=(crow["start_time_sec"] + crow["end_time_sec"]) / 2,
                y=y_pos,
                s=f"p={crow['cluster_corrected_p']:.4f}",
                ha="center",
                va="center",
                fontsize=8
            )

            label_added = True

    ax.set_xlabel("Time (s)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric}: observed vs condition-label null, {offset}°")
    ax.legend(frameon=False)

    _safe_savefig_perm(
        fig,
        f"breakpoint_DYNAMIC_cluster_timecourse_{_safe_filename(metric)}_{int(offset)}deg.png"
    )

for metric in window_metric_names:
    for offset in FIXED_OFFSETS:
        _plot_cluster_timecourse(metric, offset)

# ============================================================
# FINAL PRINT SUMMARY
# ============================================================

print("\n============================================================")
print("BREAKPOINT + DYNAMIC CAPTURE PERMUTATION ADD-ON COMPLETE")
print("FLAT OUTPUT-PATH VERSION")
print("============================================================")

print("\nAll outputs were saved directly to:")
print(PERM_OUT_DIR)

print("\nPrimary files:")
print(f"Unique trial labels:                         {unique_trial_offset_csv}")
print(f"Observed subject-offset values:              {observed_breakpoint_subject_offset_for_perm_csv}")
print(f"Observed offset empirical values:            {observed_offset_empirical_csv}")
print(f"Observed offset slopes:                      {observed_offset_slopes_csv}")
print(f"Condition-label empirical null:              {breakpoint_label_shuffle_empirical_null_csv}")
print(f"Condition-label empirical tests:             {breakpoint_label_shuffle_empirical_tests_csv}")
print(f"Condition-label slope null:                  {breakpoint_label_shuffle_slope_null_csv}")
print(f"Condition-label slope tests:                 {breakpoint_label_shuffle_slope_tests_csv}")
print(f"Observed timecourse subject values:          {observed_timecourse_subject_csv}")
print(f"Observed timecourse group values:            {observed_timecourse_group_csv}")
print(f"Cluster null timecourses:                    {breakpoint_cluster_null_timecourse_csv}")
print(f"Cluster binwise tests:                       {breakpoint_cluster_binwise_csv}")
print(f"Cluster-corrected timecourse tests:          {breakpoint_cluster_corrected_csv}")
print(f"Cluster max-null distribution:               {breakpoint_cluster_max_null_csv}")

print("\n============================================================")
print("MANUSCRIPT-READY EMPIRICAL VALUE TESTS")
print("============================================================")

primary_empirical_cols = [
    "metric_label",
    "offset_label",
    "outcome",
    "observed_effect",
    "observed_ci95_low",
    "observed_ci95_high",
    "observed_t",
    "observed_df",
    "parametric_p_primary",
    "permutation_p_effect",
    "permutation_p_effect_fdr_all_tests",
    "permutation_p_effect_fdr_within_metric_outcome",
]

available_empirical_cols = [
    c for c in primary_empirical_cols
    if c in breakpoint_label_shuffle_empirical_tests_df.columns
]

if len(breakpoint_label_shuffle_empirical_tests_df) > 0:
    print(
        breakpoint_label_shuffle_empirical_tests_df[available_empirical_cols]
        .to_string(index=False)
    )

print("\n============================================================")
print("MANUSCRIPT-READY OFFSET SLOPE TESTS")
print("============================================================")

primary_slope_cols = [
    "metric_label",
    "outcome",
    "observed_slope",
    "observed_clustered_estimate",
    "observed_clustered_se",
    "observed_clustered_t",
    "observed_clustered_p",
    "permutation_p_slope",
    "permutation_p_slope_fdr_all_tests",
    "permutation_p_slope_fdr_within_metric",
]

available_slope_cols = [
    c for c in primary_slope_cols
    if c in breakpoint_label_shuffle_slope_tests_df.columns
]

if len(breakpoint_label_shuffle_slope_tests_df) > 0:
    print(
        breakpoint_label_shuffle_slope_tests_df[available_slope_cols]
        .to_string(index=False)
    )

print("\n============================================================")
print("CLUSTER-CORRECTED TIMECOURSE RESULTS")
print("============================================================")

primary_cluster_cols = [
    "metric",
    "offset_label",
    "start_time_sec",
    "end_time_sec",
    "n_timepoints",
    "mean_observed_in_cluster",
    "mean_null_in_cluster",
    "mean_observed_minus_null_in_cluster",
    "cluster_corrected_p",
    "cluster_corrected_p_fdr_all_clusters",
    "cluster_corrected_p_fdr_within_metric",
]

available_cluster_cols = [
    c for c in primary_cluster_cols
    if c in breakpoint_cluster_corrected_df.columns
]

if len(breakpoint_cluster_corrected_df) > 0:
    print(
        breakpoint_cluster_corrected_df[available_cluster_cols]
        .to_string(index=False)
    )
else:
    print("No cluster-corrected timecourse clusters were found.")

print("\n============================================================")
print("INTERPRETATION GUIDE")
print("============================================================")
print("- Empirical-value permutation p values test whether each metric × offset value is unusually high/low relative to a whole-trial condition-label shuffle null.")
print("- Offset-slope permutation p values test whether the monotonic offset effect is stronger than expected after shuffling complete-trial offset labels within subject.")
print("- Cluster tests identify contiguous time ranges where the empirical capture trajectory differs from the condition-label shuffle null.")
print("- Use permutation_p_effect and permutation_p_effect_fdr_all_tests for offset-specific empirical values.")
print("- Use permutation_p_slope and permutation_p_slope_fdr_all_tests for offset-slope effects.")
print("- Use cluster_corrected_p and cluster_corrected_p_fdr_all_clusters for timecourse clusters.")

In [30]:
# %% =============================================================================
# TIME-RESOLVED REPORTED-AUDITORY POSITION RELATIVE TO VISION — DEGREES
# =============================================================================
#
# PRIMARY QUESTION
# ----------------
# After the spatial breakpoint analysis showed that most fixed-offset trials
# remain dynamically updating, WHAT FORM does that updating take?
#
# Does the reported auditory position:
#
#     - progressively approach the visual stimulus?
#     - move back toward the physical auditory stimulus?
#     - remain within a relatively stable spatial zone?
#     - fluctuate back and forth without strong net drift?
#
#
# PRIMARY DEGREE-SPACE MEASURES
# -----------------------------
#
# Let:
#
#     R(t) = participant's reported location
#     A(t) = physical auditory location
#     V(t) = visual location
#
# after lag-correcting R(t) to the visual trajectory.
#
#
# 1. VISUAL-DIRECTED REPORTED SHIFT
#
#     B(t) = [R(t) - A(t)] * sign[V(t) - A(t)]
#
# Interpretation:
#
#     0°            = report at the auditory stimulus
#     offset°       = report at the visual stimulus
#     positive      = shifted toward vision
#     negative      = shifted away from vision
#
#
# 2. REMAINING REPORTED-TO-VISUAL GAP — PRIMARY
#
#     G(t) = [V(t) - R(t)] * sign[V(t) - A(t)]
#
# Interpretation:
#
#     0°            = report at the visual stimulus
#     offset°       = report at the auditory stimulus
#
#     G(t) decreasing:
#         reported sound is moving TOWARD the visual stimulus
#
#     G(t) increasing:
#         reported sound is moving AWAY FROM vision / toward audition
#
#     G(t) approximately constant:
#         report remains in a stable spatial zone
#
#     alternating increases/decreases:
#         dynamic fluctuation / recalibration
#
#
# 3. ABSOLUTE REPORT-TO-VISUAL DISTANCE
#
#     |R(t) - V(t)|
#
# This discards overshoot direction and gives pure separation in degrees.
#
#
# 4. NORMALIZED VISUAL CAPTURE
#
#     [R(t) - A(t)] / [V(t) - A(t)]
#
#     0 = auditory stimulus
#     1 = visual stimulus
#
# Included as a bridge to the existing breakpoint/capture analysis.
#
#
# TEMPORAL ANALYSIS
# -----------------
#
# A. 250-ms trial-first bins
# B. subject-first condition-average trajectories
# C. group mean ± 95% CI over time
# D. temporal derivative of reported-to-visual gap
# E. cluster-based sign-flip tests of derivative versus 0
# F. time × degree density heatmaps
# G. participant-level:
#       - early vs late change
#       - whole-trial slope
#       - total variation
#       - mean absolute velocity
#       - directionality index
#       - fluctuation excess
#
#
# DIRECTIONALITY INDEX
# --------------------
#
#     |net change| / total variation
#
# approximately 1:
#     mostly monotonic directional movement
#
# approximately 0:
#     substantial back-and-forth fluctuation with little net displacement
#
#
# PRIMARY FIXED CONDITIONS
# ------------------------
#
#     10°, 20°, 30°, 40°
#
# Congruent trials are not used because there is no meaningful A-V axis when
# auditory and visual locations coincide.
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning
)


# =============================================================================
# 2. SETTINGS
# =============================================================================

FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]


# Same stable analysis window used throughout the SL analyses.
ANALYSIS_START_S = 0.50
ANALYSIS_END_S = 17.50


# Preserve temporal resolution while reducing pointwise mouse noise.
TIME_BIN_S = 0.25


# Derivative is calculated from subject-level trajectories after only mild
# centered smoothing.
DERIVATIVE_SMOOTH_BINS = 3


# Early and late windows used for interpretable scalar summaries.
EARLY_WINDOW = (
    0.50,
    3.00,
)

LATE_WINDOW = (
    15.00,
    17.50,
)


# Minimum actual A-V separation required for degree-axis calculations.
MIN_AV_DISPARITY_DEG = 1.0


# Extreme normalized capture values are almost always malformed samples.
# If your existing breakpoint cell defined its own safety limit, use it.
CAPTURE_SAFETY_LIMIT = float(
    globals().get(
        "SL_ABSOLUTE_CAPTURE_SAFETY_LIMIT",
        3.0
    )
)


# Lag fallback.
MAX_LAG_FALLBACK_S = 4.0
MIN_LAG_CORR_SAMPLES = 20


# Cluster permutation inference.
N_CLUSTER_PERM = 5000
CLUSTER_ALPHA = 0.05
MIN_CLUSTER_SUBJECTS = 10
RANDOM_SEED = 20260930


# Sign-flip scalar tests.
N_SIGNFLIP = 10000


# Heatmap.
HEATMAP_DEG_BIN = 1.0
HEATMAP_Y_MIN = -5.0
HEATMAP_Y_MAX = 45.0


# Figures.
FIG_DPI = 220


# =============================================================================
# 3. OUTPUT DIRECTORY
# =============================================================================

if "DATA_DIR" in globals():

    GAP_ROOT = (
        Path(DATA_DIR)
        /
        "spatial_behavior_tonelli_lag_corrected_subject_first"
        /
        "time_resolved_reported_audio_visual_gap"
    )

elif "SPATIAL_ANALYSES_ROOT" in globals():

    GAP_ROOT = (
        Path(SPATIAL_ANALYSES_ROOT)
        /
        "time_resolved_reported_audio_visual_gap"
    )

else:

    GAP_ROOT = (
        Path.cwd()
        /
        "time_resolved_reported_audio_visual_gap"
    )


CSV_DIR = (
    GAP_ROOT
    /
    "csv_outputs"
)

FIG_DIR = (
    GAP_ROOT
    /
    "figures"
)


for directory in [
    GAP_ROOT,
    CSV_DIR,
    FIG_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# =============================================================================
# 4. INPUT CHECK
# =============================================================================

if "trial_df" not in globals():

    raise RuntimeError(
        "trial_df was not found.\n\n"
        "Run the SL load/resample cells first. "
        "This analysis expects trial_df with arrays:\n"
        "    t_grid\n"
        "    mouse_x\n"
        "    audio_x\n"
        "    blob_x"
    )


sl_gap_source = (
    trial_df
    .copy()
)


required_cols = [
    "subject",
    "trialIndex",
    "trialType",
    "condLabel",
    "t_grid",
    "mouse_x",
    "audio_x",
    "blob_x",
]


missing = [
    c
    for c in required_cols
    if c not in sl_gap_source.columns
]


if missing:

    raise RuntimeError(
        "trial_df is missing required columns:\n"
        f"{missing}"
    )


# =============================================================================
# 5. HELPERS
# =============================================================================

def gap_safe_array(
    x
):

    try:

        return np.asarray(
            x,
            dtype=float
        ).reshape(-1)

    except Exception:

        return np.asarray(
            [],
            dtype=float
        )



def gap_normalize_subject(
    value
):

    text = str(
        value
    ).strip().lower()

    match = re.search(
        r"(\d+)",
        text
    )

    if match:

        return (
            f"pid{int(match.group(1)):03d}"
        )

    return text



def gap_normalize_trial(
    value
):

    return re.sub(
        r"\.0$",
        "",
        str(value).strip()
    )



def gap_extract_offset(
    row
):

    # Prefer explicit offsetDeg.
    if (
        "offsetDeg" in row.index
        and
        pd.notna(
            row[
                "offsetDeg"
            ]
        )
    ):

        try:

            value = abs(
                float(
                    row[
                        "offsetDeg"
                    ]
                )
            )

            if int(
                round(
                    value
                )
            ) in FIXED_OFFSETS:

                return int(
                    round(
                        value
                    )
                )

        except Exception:

            pass


    # Fallback: parse condLabel.
    text = str(
        row.get(
            "condLabel",
            ""
        )
    ).lower()


    match = re.search(
        r"(10|20|30|40)",
        text
    )


    if match:

        return int(
            match.group(
                1
            )
        )


    return np.nan



def gap_shift_response(
    response,
    time_s,
    lag_sec
):

    """
    Positive lag means behavior occurs after the stimulus.

    Corrected response at stimulus time t:

        R_corrected(t) = R_observed(t + lag)
    """

    response = np.asarray(
        response,
        dtype=float
    )

    time_s = np.asarray(
        time_s,
        dtype=float
    )


    if (
        len(response) != len(time_s)
        or
        len(response) < 2
        or
        not np.isfinite(lag_sec)
    ):

        return np.full(
            len(time_s),
            np.nan,
            dtype=float
        )


    good = (
        np.isfinite(response)
        &
        np.isfinite(time_s)
    )


    if good.sum() < 2:

        return np.full(
            len(time_s),
            np.nan,
            dtype=float
        )


    t_valid = time_s[
        good
    ]

    r_valid = response[
        good
    ]


    order = np.argsort(
        t_valid
    )


    t_valid = t_valid[
        order
    ]

    r_valid = r_valid[
        order
    ]


    unique_t, unique_idx = np.unique(
        t_valid,
        return_index=True
    )


    r_valid = r_valid[
        unique_idx
    ]


    return np.interp(
        time_s
        +
        float(
            lag_sec
        ),
        unique_t,
        r_valid,
        left=np.nan,
        right=np.nan
    )



def gap_estimate_visual_lag(
    response,
    visual,
    time_s,
    max_lag_s=MAX_LAG_FALLBACK_S
):

    """
    Fallback only.

    Finds the non-negative response delay maximizing response-visual
    correlation.

    Existing LagV_sec values are preferred whenever available.
    """

    response = np.asarray(
        response,
        dtype=float
    )

    visual = np.asarray(
        visual,
        dtype=float
    )

    time_s = np.asarray(
        time_s,
        dtype=float
    )


    n = min(
        len(response),
        len(visual),
        len(time_s)
    )


    if n < MIN_LAG_CORR_SAMPLES:

        return np.nan


    response = response[:n]
    visual = visual[:n]
    time_s = time_s[:n]


    dt = np.nanmedian(
        np.diff(
            time_s
        )
    )


    if (
        not np.isfinite(dt)
        or
        dt <= 0
    ):

        return np.nan


    max_samples = int(
        round(
            max_lag_s
            /
            dt
        )
    )


    best_corr = -np.inf
    best_lag = np.nan


    for lag_samples in range(
        0,
        max_samples + 1
    ):

        if lag_samples == 0:

            r = response
            v = visual

        else:

            r = response[
                lag_samples:
            ]

            v = visual[
                :-lag_samples
            ]


        good = (
            np.isfinite(r)
            &
            np.isfinite(v)
        )


        if good.sum() < MIN_LAG_CORR_SAMPLES:

            continue


        if (
            np.nanstd(
                r[
                    good
                ]
            ) < 1e-10
            or
            np.nanstd(
                v[
                    good
                ]
            ) < 1e-10
        ):

            continue


        corr = np.corrcoef(
            r[
                good
            ],
            v[
                good
            ]
        )[
            0,
            1
        ]


        if (
            np.isfinite(
                corr
            )
            and
            corr > best_corr
        ):

            best_corr = corr

            best_lag = (
                lag_samples
                *
                dt
            )


    return float(
        best_lag
    ) if np.isfinite(
        best_lag
    ) else np.nan



def gap_mean_ci(
    values,
    alpha=0.05
):

    x = (
        pd.Series(
            values
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan
        )
        .dropna()
        .astype(float)
        .to_numpy()
    )


    n = len(
        x
    )


    if n == 0:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            0
        )


    mean = float(
        np.mean(
            x
        )
    )


    if n < 2:

        return (
            mean,
            np.nan,
            np.nan,
            np.nan,
            n
        )


    sd = float(
        np.std(
            x,
            ddof=1
        )
    )


    se = (
        sd
        /
        np.sqrt(
            n
        )
    )


    crit = stats.t.ppf(
        1
        -
        alpha / 2,
        df=
            n - 1
    )


    return (
        mean,
        mean
        -
        crit * se,
        mean
        +
        crit * se,
        se,
        n
    )



def gap_fdr_bh(
    p_values
):

    p = np.asarray(
        p_values,
        dtype=float
    )


    result = np.full(
        len(p),
        np.nan
    )


    valid = np.isfinite(
        p
    )


    if valid.sum() == 0:

        return result


    pv = p[
        valid
    ]


    order = np.argsort(
        pv
    )


    ranked = pv[
        order
    ]


    m = len(
        ranked
    )


    q = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )


    q = np.minimum.accumulate(
        q[
            ::-1
        ]
    )[
        ::-1
    ]


    q = np.clip(
        q,
        0,
        1
    )


    back = np.empty(
        m
    )


    back[
        order
    ] = q


    result[
        valid
    ] = back


    return result



def gap_signflip_test(
    values,
    n_perm=N_SIGNFLIP,
    seed=RANDOM_SEED
):

    x = (
        pd.Series(
            values
        )
        .dropna()
        .astype(float)
        .to_numpy()
    )


    if len(
        x
    ) < 2:

        return np.nan


    observed = abs(
        np.mean(
            x
        )
    )


    rng = np.random.default_rng(
        seed
    )


    count = 0


    for _ in range(
        n_perm
    ):

        signs = rng.choice(
            [
                -1.0,
                1.0,
            ],
            size=
                len(
                    x
                )
        )


        permuted = abs(
            np.mean(
                x
                *
                signs
            )
        )


        if permuted >= observed:

            count += 1


    return (
        count
        +
        1
    ) / (
        n_perm
        +
        1
    )


# =============================================================================
# 6. PREPARE FIXED-OFFSET TRIAL TABLE
# =============================================================================

sl_fixed_df = (
    sl_gap_source
    .copy()
    .reset_index(
        drop=True
    )
)


sl_fixed_df[
    "subject"
] = (
    sl_fixed_df[
        "subject"
    ]
    .apply(
        gap_normalize_subject
    )
)


sl_fixed_df[
    "trial_id"
] = (
    sl_fixed_df[
        "trialIndex"
    ]
    .apply(
        gap_normalize_trial
    )
)


sl_fixed_df[
    "offset_deg"
] = (
    sl_fixed_df
    .apply(
        gap_extract_offset,
        axis=1
    )
)


# Fixed incongruent spatial trials only.
sl_fixed_df = sl_fixed_df[
    (
        sl_fixed_df[
            "trialType"
        ]
        .astype(str)
        .str.lower()
        .eq(
            "incongruent_spatial"
        )
    )
    &
    (
        sl_fixed_df[
            "offset_deg"
        ]
        .isin(
            FIXED_OFFSETS
        )
    )
].copy()


if sl_fixed_df.empty:

    raise RuntimeError(
        "No 10/20/30/40° fixed-offset trials were found."
    )


print(
    "\n"
    +
    "=" * 90
)


print(
    "FIXED-OFFSET INPUT"
)


print(
    "=" * 90
)


print(
    f"Participants: "
    f"{sl_fixed_df['subject'].nunique()}"
)


print(
    f"Trials: "
    f"{len(sl_fixed_df)}"
)


print(
    "\nTrial counts:"
)


print(
    sl_fixed_df[
        "offset_deg"
    ]
    .value_counts()
    .sort_index()
)


# =============================================================================
# 7. RESOLVE VISUAL-LAG VALUES
# =============================================================================
#
# Priority:
#
# 1. LagV_sec or equivalent already stored in trial_df
# 2. matching value from trial_metrics_df
# 3. trajectory-based fallback estimate
#
# =============================================================================

visual_lag_candidates = [
    "LagV_sec",
    "lagV_sec",
    "lag_v_sec",
    "LagVisual_sec",
    "VisualLag_sec",
    "visual_lag_sec",
    "lag_visual_sec",
    "best_visual_lag_sec",
]


def first_existing_column(
    dataframe,
    candidates
):

    if (
        dataframe is None
        or
        not isinstance(
            dataframe,
            pd.DataFrame
        )
    ):

        return None


    lookup = {
        str(c).lower():
            c
        for c in dataframe.columns
    }


    for candidate in candidates:

        if candidate.lower() in lookup:

            return lookup[
                candidate.lower()
            ]


    return None


direct_lag_col = first_existing_column(
    sl_fixed_df,
    visual_lag_candidates
)


metric_lag_lookup = {}


if (
    "trial_metrics_df" in globals()
    and
    isinstance(
        trial_metrics_df,
        pd.DataFrame
    )
    and
    not trial_metrics_df.empty
):

    metric_subject_col = first_existing_column(
        trial_metrics_df,
        [
            "subject",
            "participant_id",
            "participant",
            "subj",
        ]
    )


    metric_trial_col = first_existing_column(
        trial_metrics_df,
        [
            "trialIndex",
            "trial_id",
            "trial",
            "trial_index",
        ]
    )


    metric_visual_lag_col = first_existing_column(
        trial_metrics_df,
        visual_lag_candidates
    )


    if (
        metric_subject_col is not None
        and
        metric_trial_col is not None
        and
        metric_visual_lag_col is not None
    ):

        lookup_df = trial_metrics_df[
            [
                metric_subject_col,
                metric_trial_col,
                metric_visual_lag_col,
            ]
        ].copy()


        lookup_df[
            "_subject"
        ] = (
            lookup_df[
                metric_subject_col
            ]
            .apply(
                gap_normalize_subject
            )
        )


        lookup_df[
            "_trial"
        ] = (
            lookup_df[
                metric_trial_col
            ]
            .apply(
                gap_normalize_trial
            )
        )


        lookup_df[
            "_lag"
        ] = pd.to_numeric(
            lookup_df[
                metric_visual_lag_col
            ],
            errors="coerce"
        )


        lookup_df = (
            lookup_df
            .dropna(
                subset=[
                    "_lag"
                ]
            )
            .groupby(
                [
                    "_subject",
                    "_trial",
                ],
                as_index=False
            )
            .agg(
                _lag=(
                    "_lag",
                    "median"
                )
            )
        )


        metric_lag_lookup = {
            (
                row[
                    "_subject"
                ],
                row[
                    "_trial"
                ]
            ):
                float(
                    row[
                        "_lag"
                    ]
                )
            for _, row
            in lookup_df.iterrows()
        }


print(
    "\nVisual lag resolution:"
)


print(
    f"Direct trial_df lag column: "
    f"{direct_lag_col}"
)


print(
    f"trial_metrics_df lookup entries: "
    f"{len(metric_lag_lookup)}"
)


# =============================================================================
# 8. BUILD TIMEPOINT-LEVEL DEGREE-SPACE DATA
# =============================================================================

gap_rows = []


lag_source_counts = {
    "trial_df":
        0,
    "trial_metrics_df":
        0,
    "estimated_fallback":
        0,
    "unresolved":
        0,
}


for _, row in sl_fixed_df.iterrows():

    t = gap_safe_array(
        row[
            "t_grid"
        ]
    )


    response_raw = gap_safe_array(
        row[
            "mouse_x"
        ]
    )


    audio = gap_safe_array(
        row[
            "audio_x"
        ]
    )


    visual = gap_safe_array(
        row[
            "blob_x"
        ]
    )


    n = min(
        len(t),
        len(response_raw),
        len(audio),
        len(visual)
    )


    if n < 5:

        continue


    t = t[
        :n
    ]

    response_raw = response_raw[
        :n
    ]

    audio = audio[
        :n
    ]

    visual = visual[
        :n
    ]


    # -------------------------------------------------------------------------
    # Resolve visual lag.
    # -------------------------------------------------------------------------

    visual_lag = np.nan

    lag_source = (
        "unresolved"
    )


    if (
        direct_lag_col is not None
    ):

        try:

            candidate = float(
                row[
                    direct_lag_col
                ]
            )


            if (
                np.isfinite(
                    candidate
                )
                and
                abs(
                    candidate
                ) <= 5.0
            ):

                visual_lag = candidate
                lag_source = "trial_df"

        except Exception:

            pass


    if not np.isfinite(
        visual_lag
    ):

        key = (
            row[
                "subject"
            ],
            row[
                "trial_id"
            ]
        )


        if key in metric_lag_lookup:

            candidate = metric_lag_lookup[
                key
            ]


            if (
                np.isfinite(
                    candidate
                )
                and
                abs(
                    candidate
                ) <= 5.0
            ):

                visual_lag = float(
                    candidate
                )

                lag_source = (
                    "trial_metrics_df"
                )


    if not np.isfinite(
        visual_lag
    ):

        visual_lag = (
            gap_estimate_visual_lag(
                response_raw,
                visual,
                t
            )
        )


        if np.isfinite(
            visual_lag
        ):

            lag_source = (
                "estimated_fallback"
            )


    if not np.isfinite(
        visual_lag
    ):

        lag_source_counts[
            "unresolved"
        ] += 1

        continue


    lag_source_counts[
        lag_source
    ] += 1


    # -------------------------------------------------------------------------
    # Lag-correct behavioral report.
    # -------------------------------------------------------------------------

    response_lagcorr = (
        gap_shift_response(
            response_raw,
            t,
            visual_lag
        )
    )


    # -------------------------------------------------------------------------
    # Physical A-V geometry.
    # -------------------------------------------------------------------------

    signed_av = (
        visual
        -
        audio
    )


    abs_av = np.abs(
        signed_av
    )


    visual_direction = np.sign(
        signed_av
    )


    valid = (
        np.isfinite(
            t
        )
        &
        np.isfinite(
            response_lagcorr
        )
        &
        np.isfinite(
            response_raw
        )
        &
        np.isfinite(
            audio
        )
        &
        np.isfinite(
            visual
        )
        &
        np.isfinite(
            visual_direction
        )
        &
        (
            abs_av
            >=
            MIN_AV_DISPARITY_DEG
        )
        &
        (
            t
            >=
            ANALYSIS_START_S
        )
        &
        (
            t
            <=
            ANALYSIS_END_S
        )
    )


    if valid.sum() < 10:

        continue


    # -------------------------------------------------------------------------
    # PRIMARY:
    # Remaining gap from the reported sound to the visual stimulus.
    #
    # 0° = report at vision
    # offset° = report at audition
    # -------------------------------------------------------------------------

    gap_to_visual_lagcorr = (
        (
            visual
            -
            response_lagcorr
        )
        *
        visual_direction
    )


    # Raw version for robustness.
    gap_to_visual_raw = (
        (
            visual
            -
            response_raw
        )
        *
        visual_direction
    )


    # -------------------------------------------------------------------------
    # Companion:
    # visual-directed displacement away from auditory position.
    #
    # 0° = auditory stimulus
    # offset° = visual stimulus
    # -------------------------------------------------------------------------

    shift_from_audio_lagcorr = (
        (
            response_lagcorr
            -
            audio
        )
        *
        visual_direction
    )


    shift_from_audio_raw = (
        (
            response_raw
            -
            audio
        )
        *
        visual_direction
    )


    # -------------------------------------------------------------------------
    # Absolute response-to-visual distance.
    # -------------------------------------------------------------------------

    abs_gap_to_visual_lagcorr = np.abs(
        response_lagcorr
        -
        visual
    )


    # -------------------------------------------------------------------------
    # Normalized visual capture.
    # -------------------------------------------------------------------------

    capture_fraction = (
        shift_from_audio_lagcorr
        /
        abs_av
    )


    capture_ok = (
        np.isfinite(
            capture_fraction
        )
        &
        (
            np.abs(
                capture_fraction
            )
            <=
            CAPTURE_SAFETY_LIMIT
        )
    )


    valid &= capture_ok


    for idx in np.where(
        valid
    )[
        0
    ]:

        gap_rows.append(
            {
                "subject":
                    row[
                        "subject"
                    ],

                "trial_id":
                    row[
                        "trial_id"
                    ],

                "offset_deg":
                    float(
                        row[
                            "offset_deg"
                        ]
                    ),

                "condition":
                    (
                        f"offset_"
                        f"{int(row['offset_deg'])}"
                    ),

                "time_s":
                    float(
                        t[
                            idx
                        ]
                    ),

                "LagV_sec":
                    float(
                        visual_lag
                    ),

                "LagV_source":
                    lag_source,

                "audio_x":
                    float(
                        audio[
                            idx
                        ]
                    ),

                "visual_x":
                    float(
                        visual[
                            idx
                        ]
                    ),

                "response_raw":
                    float(
                        response_raw[
                            idx
                        ]
                    ),

                "response_lagcorr":
                    float(
                        response_lagcorr[
                            idx
                        ]
                    ),

                "actual_av_disparity_deg":
                    float(
                        abs_av[
                            idx
                        ]
                    ),

                # PRIMARY
                "reported_to_visual_gap_deg":
                    float(
                        gap_to_visual_lagcorr[
                            idx
                        ]
                    ),

                # Pure unsigned distance
                "absolute_reported_visual_gap_deg":
                    float(
                        abs_gap_to_visual_lagcorr[
                            idx
                        ]
                    ),

                # Complementary degree-space capture
                "visual_directed_shift_deg":
                    float(
                        shift_from_audio_lagcorr[
                            idx
                        ]
                    ),

                # Normalized bridge to breakpoint measure
                "visual_capture_fraction":
                    float(
                        capture_fraction[
                            idx
                        ]
                    ),

                # Raw robustness
                "reported_to_visual_gap_deg_raw":
                    float(
                        gap_to_visual_raw[
                            idx
                        ]
                    ),

                "visual_directed_shift_deg_raw":
                    float(
                        shift_from_audio_raw[
                            idx
                        ]
                    ),
            }
        )


gap_timepoint_df = pd.DataFrame(
    gap_rows
)


if gap_timepoint_df.empty:

    raise RuntimeError(
        "No valid fixed-offset degree-space timepoints were created."
    )


gap_timepoint_df.to_csv(
    CSV_DIR
    /
    "01_fixed_offset_reported_visual_gap_timepoints.csv",
    index=False
)


print(
    "\n"
    +
    "=" * 90
)


print(
    "TIMEPOINT DATA CREATED"
)


print(
    "=" * 90
)


print(
    f"Rows: "
    f"{len(gap_timepoint_df):,}"
)


print(
    f"Participants: "
    f"{gap_timepoint_df['subject'].nunique()}"
)


print(
    f"Trials: "
    f"{gap_timepoint_df[['subject','trial_id']].drop_duplicates().shape[0]}"
)


print(
    "\nVisual-lag sources:"
)


print(
    lag_source_counts
)


# =============================================================================
# 9. 250-MS TRIAL-FIRST BINNING
# =============================================================================

gap_timepoint_df[
    "time_bin"
] = np.floor(
    (
        gap_timepoint_df[
            "time_s"
        ]
        -
        ANALYSIS_START_S
    )
    /
    TIME_BIN_S
).astype(
    int
)


trial_time_df = (
    gap_timepoint_df

    .groupby(
        [
            "subject",
            "trial_id",
            "condition",
            "offset_deg",
            "time_bin",
        ],
        as_index=False,
        observed=True
    )

    .agg(
        time_s=(
            "time_s",
            "mean"
        ),

        reported_to_visual_gap_deg=(
            "reported_to_visual_gap_deg",
            "mean"
        ),

        absolute_reported_visual_gap_deg=(
            "absolute_reported_visual_gap_deg",
            "mean"
        ),

        visual_directed_shift_deg=(
            "visual_directed_shift_deg",
            "mean"
        ),

        visual_capture_fraction=(
            "visual_capture_fraction",
            "mean"
        ),

        actual_av_disparity_deg=(
            "actual_av_disparity_deg",
            "mean"
        ),
    )
)


trial_time_df.to_csv(
    CSV_DIR
    /
    "02_trial_timebin_reported_visual_gap.csv",
    index=False
)


# =============================================================================
# 10. SUBJECT-FIRST TEMPORAL TRAJECTORIES
# =============================================================================
#
# Average trials within participant × offset × time.
# =============================================================================

subject_time_df = (
    trial_time_df

    .groupby(
        [
            "subject",
            "condition",
            "offset_deg",
            "time_bin",
        ],
        as_index=False,
        observed=True
    )

    .agg(
        time_s=(
            "time_s",
            "mean"
        ),

        reported_to_visual_gap_deg=(
            "reported_to_visual_gap_deg",
            "mean"
        ),

        absolute_reported_visual_gap_deg=(
            "absolute_reported_visual_gap_deg",
            "mean"
        ),

        visual_directed_shift_deg=(
            "visual_directed_shift_deg",
            "mean"
        ),

        visual_capture_fraction=(
            "visual_capture_fraction",
            "mean"
        ),

        actual_av_disparity_deg=(
            "actual_av_disparity_deg",
            "mean"
        ),

        n_trials=(
            "trial_id",
            "nunique"
        ),
    )

    .sort_values(
        [
            "subject",
            "offset_deg",
            "time_s",
        ]
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 11. SUBJECT-LEVEL TEMPORAL DERIVATIVE
# =============================================================================
#
# dG/dt:
#
#     negative = report moving toward vision
#     positive = report moving away from vision / toward audition
#
# =============================================================================

derivative_parts = []


for (
    subject,
    condition
), g in subject_time_df.groupby(
    [
        "subject",
        "condition",
    ],
    sort=False
):

    g = (
        g
        .sort_values(
            "time_s"
        )
        .copy()
    )


    g[
        "gap_smoothed_deg"
    ] = (
        g[
            "reported_to_visual_gap_deg"
        ]
        .rolling(
            DERIVATIVE_SMOOTH_BINS,
            center=True,
            min_periods=1
        )
        .mean()
    )


    t = g[
        "time_s"
    ].to_numpy(
        dtype=float
    )


    y = g[
        "gap_smoothed_deg"
    ].to_numpy(
        dtype=float
    )


    if len(
        g
    ) >= 3:

        g[
            "gap_velocity_deg_per_s"
        ] = np.gradient(
            y,
            t
        )

    else:

        g[
            "gap_velocity_deg_per_s"
        ] = np.nan


    derivative_parts.append(
        g
    )


subject_time_df = pd.concat(
    derivative_parts,
    ignore_index=True
)


subject_time_df.to_csv(
    CSV_DIR
    /
    "03_subject_time_resolved_reported_visual_gap.csv",
    index=False
)


# =============================================================================
# 12. GROUP TIME-RESOLVED SUMMARY
# =============================================================================

group_rows = []


for (
    condition,
    offset,
    time_bin
), g in subject_time_df.groupby(
    [
        "condition",
        "offset_deg",
        "time_bin",
    ],
    sort=False,
    observed=True
):

    row = {
        "condition":
            condition,

        "offset_deg":
            offset,

        "time_bin":
            time_bin,

        "time_s":
            float(
                g[
                    "time_s"
                ].mean()
            ),
    }


    for metric in [
        "reported_to_visual_gap_deg",
        "absolute_reported_visual_gap_deg",
        "visual_directed_shift_deg",
        "visual_capture_fraction",
        "gap_velocity_deg_per_s",
    ]:

        (
            mean,
            low,
            high,
            se,
            n
        ) = gap_mean_ci(
            g[
                metric
            ]
        )


        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95_low"
        ] = low

        row[
            f"{metric}_ci95_high"
        ] = high

        row[
            f"{metric}_sem"
        ] = se

        row[
            f"{metric}_n"
        ] = n


    group_rows.append(
        row
    )


group_time_df = pd.DataFrame(
    group_rows
)


group_time_df.to_csv(
    CSV_DIR
    /
    "04_group_time_resolved_reported_visual_gap.csv",
    index=False
)


# =============================================================================
# 13. PARTICIPANT-LEVEL DIRECTIONAL / FLUCTUATION METRICS
# =============================================================================

subject_metric_rows = []


for (
    subject,
    condition,
    offset
), g in subject_time_df.groupby(
    [
        "subject",
        "condition",
        "offset_deg",
    ],
    sort=False
):

    g = (
        g
        .sort_values(
            "time_s"
        )
        .dropna(
            subset=[
                "reported_to_visual_gap_deg"
            ]
        )
        .copy()
    )


    if len(
        g
    ) < 5:

        continue


    t = g[
        "time_s"
    ].to_numpy(
        dtype=float
    )


    gap = g[
        "reported_to_visual_gap_deg"
    ].to_numpy(
        dtype=float
    )


    vel = g[
        "gap_velocity_deg_per_s"
    ].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Early / late means.
    # -------------------------------------------------------------------------

    early_mask = (
        (
            t
            >=
            EARLY_WINDOW[
                0
            ]
        )
        &
        (
            t
            <=
            EARLY_WINDOW[
                1
            ]
        )
    )


    late_mask = (
        (
            t
            >=
            LATE_WINDOW[
                0
            ]
        )
        &
        (
            t
            <=
            LATE_WINDOW[
                1
            ]
        )
    )


    early_gap = (
        float(
            np.mean(
                gap[
                    early_mask
                ]
            )
        )
        if
        early_mask.sum() > 0
        else
        np.nan
    )


    late_gap = (
        float(
            np.mean(
                gap[
                    late_mask
                ]
            )
        )
        if
        late_mask.sum() > 0
        else
        np.nan
    )


    late_minus_early = (
        late_gap
        -
        early_gap
        if
        np.isfinite(
            early_gap
        )
        and
        np.isfinite(
            late_gap
        )
        else
        np.nan
    )


    # -------------------------------------------------------------------------
    # Whole-trial linear slope.
    # -------------------------------------------------------------------------

    whole_slope = float(
        np.polyfit(
            t,
            gap,
            1
        )[
            0
        ]
    )


    # -------------------------------------------------------------------------
    # Total variation:
    # total degree-space path traveled by the subject-average gap trajectory.
    # -------------------------------------------------------------------------

    total_variation = float(
        np.sum(
            np.abs(
                np.diff(
                    gap
                )
            )
        )
    )


    endpoint_net_change = float(
        gap[
            -1
        ]
        -
        gap[
            0
        ]
    )


    # -------------------------------------------------------------------------
    # Directionality:
    #
    # 1 -> nearly all movement in one direction
    # 0 -> substantial back-and-forth movement
    # -------------------------------------------------------------------------

    directionality_index = (
        abs(
            endpoint_net_change
        )
        /
        total_variation
        if
        total_variation > 1e-12
        else
        np.nan
    )


    fluctuation_excess_deg = (
        total_variation
        -
        abs(
            endpoint_net_change
        )
    )


    mean_abs_velocity = (
        float(
            np.nanmean(
                np.abs(
                    vel
                )
            )
        )
        if
        np.isfinite(
            vel
        ).any()
        else
        np.nan
    )


    late_sd = (
        float(
            np.std(
                gap[
                    late_mask
                ],
                ddof=1
            )
        )
        if
        late_mask.sum() >= 2
        else
        np.nan
    )


    subject_metric_rows.append(
        {
            "subject":
                subject,

            "condition":
                condition,

            "offset_deg":
                offset,

            "early_gap_deg":
                early_gap,

            "late_gap_deg":
                late_gap,

            "late_minus_early_gap_deg":
                late_minus_early,

            "whole_trial_gap_slope_deg_per_s":
                whole_slope,

            "endpoint_net_change_deg":
                endpoint_net_change,

            "total_variation_deg":
                total_variation,

            "mean_abs_velocity_deg_per_s":
                mean_abs_velocity,

            "directionality_index":
                directionality_index,

            "fluctuation_excess_deg":
                fluctuation_excess_deg,

            "late_gap_sd_deg":
                late_sd,
        }
    )


subject_metrics_df = pd.DataFrame(
    subject_metric_rows
)


subject_metrics_df.to_csv(
    CSV_DIR
    /
    "05_subject_gap_direction_fluctuation_metrics.csv",
    index=False
)


# =============================================================================
# 14. INFERENCE:
#     DOES THE VISUAL GAP GET BIGGER OR SMALLER?
# =============================================================================
#
# Primary:
#
#     late gap - early gap
#
# negative:
#     gap SHRINKS over trial -> reported sound moves toward vision
#
# positive:
#     gap GROWS over trial -> reported sound moves toward audition
#
# Also test whole-trial slope.
#
# =============================================================================

test_rows = []


for metric in [
    "late_minus_early_gap_deg",
    "whole_trial_gap_slope_deg_per_s",
]:

    for offset in FIXED_OFFSETS:

        values = (
            subject_metrics_df.loc[
                subject_metrics_df[
                    "offset_deg"
                ]
                ==
                offset,
                metric
            ]
            .dropna()
            .astype(float)
            .to_numpy()
        )


        if len(
            values
        ) < 2:

            continue


        (
            mean,
            low,
            high,
            se,
            n
        ) = gap_mean_ci(
            values
        )


        tt = stats.ttest_1samp(
            values,
            0.0
        )


        sd = np.std(
            values,
            ddof=1
        )


        dz = (
            mean
            /
            sd
            if
            sd > 0
            else
            np.nan
        )


        perm_p = gap_signflip_test(
            values,
            seed=
                RANDOM_SEED
                +
                int(
                    offset
                )
        )


        test_rows.append(
            {
                "metric":
                    metric,

                "offset_deg":
                    offset,

                "n":
                    n,

                "mean":
                    mean,

                "ci95_low":
                    low,

                "ci95_high":
                    high,

                "se":
                    se,

                "t":
                    float(
                        tt.statistic
                    ),

                "df":
                    n
                    -
                    1,

                "p":
                    float(
                        tt.pvalue
                    ),

                "dz":
                    dz,

                "signflip_p":
                    perm_p,
            }
        )


gap_tests_df = pd.DataFrame(
    test_rows
)


for metric in gap_tests_df[
    "metric"
].unique():

    mask = (
        gap_tests_df[
            "metric"
        ]
        ==
        metric
    )


    gap_tests_df.loc[
        mask,
        "p_fdr"
    ] = gap_fdr_bh(
        gap_tests_df.loc[
            mask,
            "p"
        ]
    )


    gap_tests_df.loc[
        mask,
        "signflip_p_fdr"
    ] = gap_fdr_bh(
        gap_tests_df.loc[
            mask,
            "signflip_p"
        ]
    )


gap_tests_df.to_csv(
    CSV_DIR
    /
    "06_gap_early_late_and_slope_tests.csv",
    index=False
)


# =============================================================================
# 15. CLUSTER-BASED SIGN-FLIP TEST OF TEMPORAL DERIVATIVE
# =============================================================================
#
# Identifies CONTIGUOUS periods where:
#
#     dG/dt < 0:
#         report systematically moves toward vision
#
#     dG/dt > 0:
#         report systematically moves away from vision
#
# This preserves the temporal structure and controls family-wise error over
# time within each offset.
#
# =============================================================================

def find_signed_clusters(
    t_stats,
    thresholds
):

    t_stats = np.asarray(
        t_stats,
        dtype=float
    )


    thresholds = np.asarray(
        thresholds,
        dtype=float
    )


    supra = (
        np.isfinite(
            t_stats
        )
        &
        np.isfinite(
            thresholds
        )
        &
        (
            np.abs(
                t_stats
            )
            >
            thresholds
        )
    )


    clusters = []

    start = None
    current_sign = None


    for i in range(
        len(
            t_stats
        )
    ):

        if supra[
            i
        ]:

            sign_i = np.sign(
                t_stats[
                    i
                ]
            )


            if (
                start is None
                or
                sign_i != current_sign
            ):

                if start is not None:

                    clusters.append(
                        (
                            start,
                            i - 1
                        )
                    )


                start = i
                current_sign = sign_i

        else:

            if start is not None:

                clusters.append(
                    (
                        start,
                        i - 1
                    )
                )


                start = None
                current_sign = None


    if start is not None:

        clusters.append(
            (
                start,
                len(
                    t_stats
                )
                -
                1
            )
        )


    return clusters



def column_t_statistics(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    t_values = np.full(
        matrix.shape[
            1
        ],
        np.nan
    )


    n_values = np.zeros(
        matrix.shape[
            1
        ],
        dtype=int
    )


    thresholds = np.full(
        matrix.shape[
            1
        ],
        np.nan
    )


    for j in range(
        matrix.shape[
            1
        ]
    ):

        x = matrix[
            :,
            j
        ]


        x = x[
            np.isfinite(
                x
            )
        ]


        n = len(
            x
        )


        n_values[
            j
        ] = n


        if n < MIN_CLUSTER_SUBJECTS:

            continue


        sd = np.std(
            x,
            ddof=1
        )


        if sd < 1e-12:

            continue


        t_values[
            j
        ] = (
            np.mean(
                x
            )
            /
            (
                sd
                /
                np.sqrt(
                    n
                )
            )
        )


        thresholds[
            j
        ] = stats.t.ppf(
            1
            -
            CLUSTER_ALPHA / 2,
            df=
                n - 1
        )


    return (
        t_values,
        n_values,
        thresholds
    )



def cluster_signflip_derivative(
    subject_time_condition_df,
    n_perm=N_CLUSTER_PERM,
    seed=RANDOM_SEED
):

    wide = (
        subject_time_condition_df

        .pivot_table(
            index=
                "subject",
            columns=
                "time_bin",
            values=
                "gap_velocity_deg_per_s",
            aggfunc=
                "mean"
        )

        .sort_index(
            axis=1
        )
    )


    time_lookup = (
        subject_time_condition_df

        .groupby(
            "time_bin"
        )[
            "time_s"
        ]

        .mean()

        .reindex(
            wide.columns
        )
    )


    matrix = wide.to_numpy(
        dtype=float
    )


    times = time_lookup.to_numpy(
        dtype=float
    )


    observed_t, observed_n, thresholds = (
        column_t_statistics(
            matrix
        )
    )


    observed_clusters = find_signed_clusters(
        observed_t,
        thresholds
    )


    observed_masses = []


    for start, end in observed_clusters:

        mass = float(
            np.sum(
                np.abs(
                    observed_t[
                        start:
                        end + 1
                    ]
                )
            )
        )


        observed_masses.append(
            mass
        )


    rng = np.random.default_rng(
        seed
    )


    max_null_masses = np.zeros(
        n_perm,
        dtype=float
    )


    for p in range(
        n_perm
    ):

        signs = rng.choice(
            [
                -1.0,
                1.0,
            ],
            size=
                matrix.shape[
                    0
                ]
        )[
            :,
            None
        ]


        perm_matrix = (
            matrix
            *
            signs
        )


        perm_t, _, perm_thresholds = (
            column_t_statistics(
                perm_matrix
            )
        )


        perm_clusters = find_signed_clusters(
            perm_t,
            perm_thresholds
        )


        if len(
            perm_clusters
        ) == 0:

            max_null_masses[
                p
            ] = 0.0

            continue


        masses = [
            np.sum(
                np.abs(
                    perm_t[
                        start:
                        end + 1
                    ]
                )
            )
            for start, end
            in perm_clusters
        ]


        max_null_masses[
            p
        ] = np.max(
            masses
        )


    result_rows = []


    for (
        cluster_index,
        (
            start,
            end
        ),
        mass
    ) in zip(
        range(
            len(
                observed_clusters
            )
        ),
        observed_clusters,
        observed_masses
    ):

        p_cluster = (
            1
            +
            np.sum(
                max_null_masses
                >=
                mass
            )
        ) / (
            n_perm
            +
            1
        )


        mean_t = np.nanmean(
            observed_t[
                start:
                end + 1
            ]
        )


        direction = (
            "toward_visual"
            if
            mean_t < 0
            else
            "away_from_visual"
        )


        result_rows.append(
            {
                "cluster":
                    cluster_index + 1,

                "start_time_s":
                    float(
                        times[
                            start
                        ]
                    ),

                "end_time_s":
                    float(
                        times[
                            end
                        ]
                    ),

                "cluster_mass":
                    mass,

                "mean_t":
                    mean_t,

                "direction":
                    direction,

                "cluster_p_fwer":
                    p_cluster,
            }
        )


    return pd.DataFrame(
        result_rows
    )


cluster_parts = []


for offset in FIXED_OFFSETS:

    d = subject_time_df[
        subject_time_df[
            "offset_deg"
        ]
        ==
        offset
    ].copy()


    clusters = (
        cluster_signflip_derivative(
            d,
            seed=
                RANDOM_SEED
                +
                offset
        )
    )


    if not clusters.empty:

        clusters[
            "offset_deg"
        ] = offset


        cluster_parts.append(
            clusters
        )


if cluster_parts:

    derivative_clusters_df = pd.concat(
        cluster_parts,
        ignore_index=True
    )

else:

    derivative_clusters_df = pd.DataFrame(
        columns=[
            "cluster",
            "start_time_s",
            "end_time_s",
            "cluster_mass",
            "mean_t",
            "direction",
            "cluster_p_fwer",
            "offset_deg",
        ]
    )


derivative_clusters_df.to_csv(
    CSV_DIR
    /
    "07_gap_velocity_cluster_tests.csv",
    index=False
)


# =============================================================================
# 16. FIGURE 1:
#     PRIMARY DEGREE GAP OVER TIME
# =============================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        15,
        10
    ),
    sharex=True,
    sharey=True
)


axes = axes.ravel()


for ax, offset in zip(
    axes,
    FIXED_OFFSETS
):

    d = (
        group_time_df[
            group_time_df[
                "offset_deg"
            ]
            ==
            offset
        ]
        .sort_values(
            "time_s"
        )
    )


    x = d[
        "time_s"
    ].to_numpy(
        dtype=float
    )


    y = d[
        "reported_to_visual_gap_deg_mean"
    ].to_numpy(
        dtype=float
    )


    low = d[
        "reported_to_visual_gap_deg_ci95_low"
    ].to_numpy(
        dtype=float
    )


    high = d[
        "reported_to_visual_gap_deg_ci95_high"
    ].to_numpy(
        dtype=float
    )


    ax.plot(
        x,
        y,
        linewidth=2.5,
        label=
            "Reported-to-visual gap"
    )


    ax.fill_between(
        x,
        low,
        high,
        alpha=0.20
    )


    # Visual stimulus reference.
    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1.25,
        label=
            "Visual location"
    )


    # Physical auditory stimulus reference in this aligned coordinate system.
    ax.axhline(
        float(
            offset
        ),
        linestyle=":",
        linewidth=1.5,
        label=
            "Auditory location"
    )


    ax.set_title(
        f"{offset}° A-V offset"
    )


    ax.set_xlim(
        ANALYSIS_START_S,
        ANALYSIS_END_S
    )


    ax.set_ylim(
        HEATMAP_Y_MIN,
        HEATMAP_Y_MAX
    )


    ax.set_xlabel(
        "Time (s)"
    )


    ax.set_ylabel(
        "Reported auditory distance from visual (deg)"
    )


axes[
    0
].legend(
    frameon=False,
    fontsize=9
)


fig.suptitle(
    "How does the reported auditory position move relative to vision?",
    fontsize=16
)


fig.tight_layout()


fig.savefig(
    FIG_DIR
    /
    "01_reported_to_visual_gap_deg_over_time.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)


plt.show()
plt.close(
    fig
)


# =============================================================================
# 17. FIGURE 2:
#     VISUAL-DIRECTED SHIFT FROM AUDITION
# =============================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        15,
        10
    ),
    sharex=True,
    sharey=True
)


axes = axes.ravel()


for ax, offset in zip(
    axes,
    FIXED_OFFSETS
):

    d = (
        group_time_df[
            group_time_df[
                "offset_deg"
            ]
            ==
            offset
        ]
        .sort_values(
            "time_s"
        )
    )


    x = d[
        "time_s"
    ].to_numpy(
        dtype=float
    )


    y = d[
        "visual_directed_shift_deg_mean"
    ].to_numpy(
        dtype=float
    )


    low = d[
        "visual_directed_shift_deg_ci95_low"
    ].to_numpy(
        dtype=float
    )


    high = d[
        "visual_directed_shift_deg_ci95_high"
    ].to_numpy(
        dtype=float
    )


    ax.plot(
        x,
        y,
        linewidth=2.5
    )


    ax.fill_between(
        x,
        low,
        high,
        alpha=0.20
    )


    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1.25
    )


    ax.axhline(
        float(
            offset
        ),
        linestyle=":",
        linewidth=1.5
    )


    ax.set_title(
        f"{offset}° A-V offset"
    )


    ax.set_xlim(
        ANALYSIS_START_S,
        ANALYSIS_END_S
    )


    ax.set_ylim(
        HEATMAP_Y_MIN,
        HEATMAP_Y_MAX
    )


    ax.set_xlabel(
        "Time (s)"
    )


    ax.set_ylabel(
        "Visual-directed shift from auditory location (deg)"
    )


fig.suptitle(
    "Visual capture expressed directly in degrees",
    fontsize=16
)


fig.tight_layout()


fig.savefig(
    FIG_DIR
    /
    "02_visual_directed_shift_deg_over_time.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)


plt.show()
plt.close(
    fig
)


# =============================================================================
# 18. FIGURE 3:
#     VELOCITY OF THE REPORTED-TO-VISUAL GAP
# =============================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        15,
        10
    ),
    sharex=True
)


axes = axes.ravel()


for ax, offset in zip(
    axes,
    FIXED_OFFSETS
):

    d = (
        group_time_df[
            group_time_df[
                "offset_deg"
            ]
            ==
            offset
        ]
        .sort_values(
            "time_s"
        )
    )


    x = d[
        "time_s"
    ].to_numpy(
        dtype=float
    )


    y = d[
        "gap_velocity_deg_per_s_mean"
    ].to_numpy(
        dtype=float
    )


    low = d[
        "gap_velocity_deg_per_s_ci95_low"
    ].to_numpy(
        dtype=float
    )


    high = d[
        "gap_velocity_deg_per_s_ci95_high"
    ].to_numpy(
        dtype=float
    )


    ax.plot(
        x,
        y,
        linewidth=2.0
    )


    ax.fill_between(
        x,
        low,
        high,
        alpha=0.20
    )


    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1.25
    )


    # -------------------------------------------------------------------------
    # Shade significant cluster periods.
    # -------------------------------------------------------------------------

    clusters = derivative_clusters_df[
        (
            derivative_clusters_df[
                "offset_deg"
            ]
            ==
            offset
        )
        &
        (
            derivative_clusters_df[
                "cluster_p_fwer"
            ]
            <
            0.05
        )
    ]


    for _, cluster in clusters.iterrows():

        ax.axvspan(
            cluster[
                "start_time_s"
            ],
            cluster[
                "end_time_s"
            ],
            alpha=0.15
        )


    ax.set_title(
        f"{offset}° A-V offset"
    )


    ax.set_xlim(
        ANALYSIS_START_S,
        ANALYSIS_END_S
    )


    ax.set_xlabel(
        "Time (s)"
    )


    ax.set_ylabel(
        "Change in visual gap (deg/s)\n"
        "negative = toward vision"
    )


fig.suptitle(
    "When does localization move toward or away from vision?",
    fontsize=16
)


fig.tight_layout()


fig.savefig(
    FIG_DIR
    /
    "03_reported_visual_gap_velocity_over_time.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)


plt.show()
plt.close(
    fig
)


# =============================================================================
# 19. FIGURE 4:
#     TIME × DEGREE DISTRIBUTION HEATMAP
# =============================================================================
#
# This is especially useful for your "specific zone" question.
#
# Each column is normalized separately, so the heatmap shows WHERE the
# participant-average reported auditory position is distributed at each
# moment in time.
#
# =============================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        15,
        10
    ),
    sharex=True,
    sharey=True
)


axes = axes.ravel()


degree_edges = np.arange(
    HEATMAP_Y_MIN,
    HEATMAP_Y_MAX
    +
    HEATMAP_DEG_BIN,
    HEATMAP_DEG_BIN
)


time_edges = np.arange(
    ANALYSIS_START_S,
    ANALYSIS_END_S
    +
    TIME_BIN_S,
    TIME_BIN_S
)


for ax, offset in zip(
    axes,
    FIXED_OFFSETS
):

    d = subject_time_df[
        subject_time_df[
            "offset_deg"
        ]
        ==
        offset
    ].dropna(
        subset=[
            "time_s",
            "reported_to_visual_gap_deg",
        ]
    )


    H, x_edges, y_edges = np.histogram2d(
        d[
            "time_s"
        ].to_numpy(
            dtype=float
        ),
        d[
            "reported_to_visual_gap_deg"
        ].to_numpy(
            dtype=float
        ),
        bins=[
            time_edges,
            degree_edges,
        ]
    )


    # Normalize each time column to probability.
    column_sums = H.sum(
        axis=1,
        keepdims=True
    )


    H_norm = np.divide(
        H,
        column_sums,
        out=np.zeros_like(
            H,
            dtype=float
        ),
        where=
            column_sums > 0
    )


    mesh = ax.pcolormesh(
        x_edges,
        y_edges,
        H_norm.T,
        shading="auto"
    )


    # Visual location.
    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1.1
    )


    # Auditory location.
    ax.axhline(
        float(
            offset
        ),
        linestyle=":",
        linewidth=1.4
    )


    ax.set_title(
        f"{offset}° A-V offset"
    )


    ax.set_xlim(
        ANALYSIS_START_S,
        ANALYSIS_END_S
    )


    ax.set_ylim(
        HEATMAP_Y_MIN,
        HEATMAP_Y_MAX
    )


    ax.set_xlabel(
        "Time (s)"
    )


    ax.set_ylabel(
        "Reported auditory distance from visual (deg)"
    )


fig.colorbar(
    mesh,
    ax=axes.tolist(),
    label=
        "Proportion of participant trajectories at time bin",
    shrink=0.8
)


fig.suptitle(
    "Where in degree-space does the reported auditory position reside over time?",
    fontsize=16
)


fig.subplots_adjust(
    top=0.92,
    right=0.88,
    wspace=0.15,
    hspace=0.22
)


fig.savefig(
    FIG_DIR
    /
    "04_time_by_degree_gap_density_heatmap.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)


plt.show()
plt.close(
    fig
)


# =============================================================================
# 20. FIGURE 5:
#     EARLY VS LATE DEGREE GAP
# =============================================================================

summary_rows = []


for offset in FIXED_OFFSETS:

    d = subject_metrics_df[
        subject_metrics_df[
            "offset_deg"
        ]
        ==
        offset
    ]


    for metric, label in [
        (
            "early_gap_deg",
            "Early"
        ),
        (
            "late_gap_deg",
            "Late"
        ),
    ]:

        (
            mean,
            low,
            high,
            se,
            n
        ) = gap_mean_ci(
            d[
                metric
            ]
        )


        summary_rows.append(
            {
                "offset_deg":
                    offset,

                "period":
                    label,

                "mean":
                    mean,

                "ci95_low":
                    low,

                "ci95_high":
                    high,

                "n":
                    n,
            }
        )


early_late_plot_df = pd.DataFrame(
    summary_rows
)


fig, ax = plt.subplots(
    figsize=(
        10,
        6
    )
)


x = np.arange(
    len(
        FIXED_OFFSETS
    )
)


width = 0.34


for j, period in enumerate(
    [
        "Early",
        "Late",
    ]
):

    d = (
        early_late_plot_df[
            early_late_plot_df[
                "period"
            ]
            ==
            period
        ]
        .set_index(
            "offset_deg"
        )
        .reindex(
            FIXED_OFFSETS
        )
    )


    y = d[
        "mean"
    ].to_numpy(
        dtype=float
    )


    low = d[
        "ci95_low"
    ].to_numpy(
        dtype=float
    )


    high = d[
        "ci95_high"
    ].to_numpy(
        dtype=float
    )


    ax.bar(
        x
        +
        (
            j - 0.5
        )
        *
        width,
        y,
        width=
            width,
        yerr=np.vstack(
            [
                np.clip(
                    y - low,
                    0,
                    None
                ),
                np.clip(
                    high - y,
                    0,
                    None
                ),
            ]
        ),
        capsize=4,
        label=
            period
    )


ax.set_xticks(
    x
)


ax.set_xticklabels(
    [
        f"{offset}°"
        for offset
        in FIXED_OFFSETS
    ]
)


ax.set_ylabel(
    "Reported auditory distance from visual (deg)"
)


ax.set_xlabel(
    "Fixed A-V offset"
)


ax.set_title(
    "Does the reported auditory–visual gap change from early to late trial?"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


fig.savefig(
    FIG_DIR
    /
    "05_early_vs_late_reported_visual_gap.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)


plt.show()
plt.close(
    fig
)


# =============================================================================
# 21. FIGURE 6:
#     DIRECTIONAL DRIFT VS FLUCTUATION
# =============================================================================
#
# X:
#     directionality index
#
# Y:
#     total variation in degrees
#
# A trajectory with:
#
#     high directionality + high TV
#         = strong monotonic movement
#
#     low directionality + high TV
#         = substantial back-and-forth recalibration
#
#     low TV
#         = relatively spatially stable
#
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        10,
        7
    )
)


for offset in FIXED_OFFSETS:

    d = subject_metrics_df[
        subject_metrics_df[
            "offset_deg"
        ]
        ==
        offset
    ]


    ax.scatter(
        d[
            "directionality_index"
        ],
        d[
            "total_variation_deg"
        ],
        alpha=0.60,
        label=
            f"{offset}°"
    )


ax.set_xlim(
    -0.02,
    1.02
)


ax.set_xlabel(
    "Directionality index\n"
    "0 = back-and-forth; 1 = monotonic"
)


ax.set_ylabel(
    "Total variation of reported visual gap (deg)"
)


ax.set_title(
    "Is continued spatial updating directional or fluctuating?"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


fig.savefig(
    FIG_DIR
    /
    "06_directionality_vs_total_variation.png",
    dpi=FIG_DPI,
    bbox_inches="tight"
)


plt.show()
plt.close(
    fig
)


# =============================================================================
# 22. GROUP SUMMARY OF DYNAMIC FORM
# =============================================================================

dynamic_summary_rows = []


for offset in FIXED_OFFSETS:

    d = subject_metrics_df[
        subject_metrics_df[
            "offset_deg"
        ]
        ==
        offset
    ]


    for metric in [
        "early_gap_deg",
        "late_gap_deg",
        "late_minus_early_gap_deg",
        "whole_trial_gap_slope_deg_per_s",
        "endpoint_net_change_deg",
        "total_variation_deg",
        "mean_abs_velocity_deg_per_s",
        "directionality_index",
        "fluctuation_excess_deg",
        "late_gap_sd_deg",
    ]:

        (
            mean,
            low,
            high,
            se,
            n
        ) = gap_mean_ci(
            d[
                metric
            ]
        )


        dynamic_summary_rows.append(
            {
                "offset_deg":
                    offset,

                "metric":
                    metric,

                "n":
                    n,

                "mean":
                    mean,

                "ci95_low":
                    low,

                "ci95_high":
                    high,

                "se":
                    se,
            }
        )


dynamic_summary_df = pd.DataFrame(
    dynamic_summary_rows
)


dynamic_summary_df.to_csv(
    CSV_DIR
    /
    "08_group_dynamic_form_summary.csv",
    index=False
)


# =============================================================================
# 23. PRINT PRIMARY RESULTS
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)


print(
    "PRIMARY QUESTION: DOES THE REPORTED AUDITORY–VISUAL GAP GET BIGGER OR SMALLER?"
)


print(
    "=" * 100
)


primary_tests = gap_tests_df[
    gap_tests_df[
        "metric"
    ]
    ==
    "late_minus_early_gap_deg"
]


for _, row in primary_tests.iterrows():

    direction = (
        "TOWARD vision (gap shrank)"
        if
        row[
            "mean"
        ] < 0
        else
        "AWAY from vision / toward audition (gap grew)"
    )


    print(
        f"\n{int(row['offset_deg'])}°:"
    )


    print(
        f"  Late - early gap = "
        f"{row['mean']:.3f}° "
        f"[95% CI "
        f"{row['ci95_low']:.3f}, "
        f"{row['ci95_high']:.3f}]"
    )


    print(
        f"  Direction: {direction}"
    )


    print(
        f"  t({int(row['df'])}) = "
        f"{row['t']:.3f}, "
        f"p = {row['p']:.6g}, "
        f"FDR p = {row['p_fdr']:.6g}"
    )


    print(
        f"  sign-flip p = "
        f"{row['signflip_p']:.6g}, "
        f"FDR = "
        f"{row['signflip_p_fdr']:.6g}"
    )


print(
    "\n"
    +
    "=" * 100
)


print(
    "TEMPORALLY LOCALIZED DIRECTIONAL CHANGE"
)


print(
    "=" * 100
)


if derivative_clusters_df.empty:

    print(
        "No supra-threshold derivative clusters were found."
    )

else:

    significant_clusters = (
        derivative_clusters_df[
            derivative_clusters_df[
                "cluster_p_fwer"
            ]
            <
            0.05
        ]
    )


    if significant_clusters.empty:

        print(
            "No derivative clusters survived FWER correction."
        )

    else:

        print(
            significant_clusters[
                [
                    "offset_deg",
                    "start_time_s",
                    "end_time_s",
                    "direction",
                    "cluster_p_fwer",
                ]
            ]
            .sort_values(
                [
                    "offset_deg",
                    "start_time_s",
                ]
            )
            .to_string(
                index=False
            )
        )


print(
    "\n"
    +
    "=" * 100
)


print(
    "DIRECTIONALITY VS FLUCTUATION"
)


print(
    "=" * 100
)


for offset in FIXED_OFFSETS:

    d = dynamic_summary_df[
        dynamic_summary_df[
            "offset_deg"
        ]
        ==
        offset
    ]


    directionality = d[
        d[
            "metric"
        ]
        ==
        "directionality_index"
    ]


    tv = d[
        d[
            "metric"
        ]
        ==
        "total_variation_deg"
    ]


    fluct = d[
        d[
            "metric"
        ]
        ==
        "fluctuation_excess_deg"
    ]


    if (
        not directionality.empty
        and
        not tv.empty
        and
        not fluct.empty
    ):

        print(
            f"\n{offset}°:"
        )


        print(
            f"  Directionality index = "
            f"{directionality['mean'].iloc[0]:.3f}"
        )


        print(
            f"  Total variation = "
            f"{tv['mean'].iloc[0]:.3f}°"
        )


        print(
            f"  Back-and-forth fluctuation excess = "
            f"{fluct['mean'].iloc[0]:.3f}°"
        )


# =============================================================================
# 24. INTERPRETATION GUIDE
# =============================================================================

print(
    """
================================================================================
HOW TO INTERPRET THESE OUTPUTS
================================================================================

PRIMARY TRAJECTORY:

    reported_to_visual_gap_deg

Reference geometry:

    0°              = reported sound is at the VISUAL stimulus
    nominal offset  = reported sound is at the AUDITORY stimulus


If the curve DECREASES over time:

    reported sound is being progressively pulled TOWARD VISION.

If the curve INCREASES over time:

    reported sound is moving AWAY FROM VISION / back toward AUDITION.

If the curve stays around one degree value:

    localization is occupying a relatively stable spatial zone.

If the curve repeatedly rises and falls:

    the spatial estimate is dynamically recalibrating / fluctuating.


DERIVATIVE:

    gap_velocity_deg_per_s < 0
        movement toward vision

    gap_velocity_deg_per_s > 0
        movement away from vision

Cluster-corrected periods tell you WHEN that directional movement is
consistent across participants.


DIRECTIONALITY INDEX:

    near 1
        most temporal movement is in one direction

    near 0
        substantial back-and-forth movement with little net displacement


TOTAL VARIATION:

    total amount of degree-space movement through the trial, regardless of
    direction.

Therefore:

    high total variation + low directionality
        = persistent fluctuation / recalibration within a zone

    high total variation + high directionality
        = sustained directional drift

    low total variation
        = spatially stable estimate


HEATMAP:

    This is the clearest answer to the "specific zone" question.

    A narrow horizontal band:
        reports concentrate in a stable spatial zone.

    A sloped band:
        systematic movement toward/away from vision.

    A broad band:
        large between-participant variability.

    A band that repeatedly bends:
        non-monotonic dynamic recalibration.
================================================================================
"""
)


# =============================================================================
# 25. FINAL OUTPUT LIST
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)


print(
    "TIME-RESOLVED DEGREE-SPACE ANALYSIS COMPLETE"
)


print(
    "=" * 100
)


print(
    f"\nOutput directory:\n{GAP_ROOT}"
)


print(
    "\nPrimary CSVs:"
)


for filename in [
    "01_fixed_offset_reported_visual_gap_timepoints.csv",
    "02_trial_timebin_reported_visual_gap.csv",
    "03_subject_time_resolved_reported_visual_gap.csv",
    "04_group_time_resolved_reported_visual_gap.csv",
    "05_subject_gap_direction_fluctuation_metrics.csv",
    "06_gap_early_late_and_slope_tests.csv",
    "07_gap_velocity_cluster_tests.csv",
    "08_group_dynamic_form_summary.csv",
]:

    print(
        "  ",
        CSV_DIR
        /
        filename
    )


print(
    "\nPrimary figures:"
)


for filename in [
    "01_reported_to_visual_gap_deg_over_time.png",
    "02_visual_directed_shift_deg_over_time.png",
    "03_reported_visual_gap_velocity_over_time.png",
    "04_time_by_degree_gap_density_heatmap.png",
    "05_early_vs_late_reported_visual_gap.png",
    "06_directionality_vs_total_variation.png",
]:

    print(
        "  ",
        FIG_DIR
        /
        filename
    )


print(
    "\nKey objects now in memory:"
)


print(
    "  gap_timepoint_df"
)


print(
    "  trial_time_df"
)


print(
    "  subject_time_df"
)


print(
    "  group_time_df"
)


print(
    "  subject_metrics_df"
)


print(
    "  gap_tests_df"
)


print(
    "  derivative_clusters_df"
)


print(
    "  dynamic_summary_df"
)


FIXED-OFFSET INPUT
Participants: 50
Trials: 4000

Trial counts:
offset_deg
10.0    1000
20.0    1000
30.0    1000
40.0    1000
Name: count, dtype: int64

Visual lag resolution:
Direct trial_df lag column: LagV_sec
trial_metrics_df lookup entries: 5932

TIMEPOINT DATA CREATED
Rows: 1,166,448
Participants: 50
Trials: 3952

Visual-lag sources:
{'trial_df': 3952, 'trial_metrics_df': 0, 'estimated_fallback': 0, 'unresolved': 48}

PRIMARY QUESTION: DOES THE REPORTED AUDITORY–VISUAL GAP GET BIGGER OR SMALLER?

10°:
  Late - early gap = 0.533° [95% CI -0.238, 1.304]
  Direction: AWAY from vision / toward audition (gap grew)
  t(49) = 1.390, p = 0.1709, FDR p = 0.227866
  sign-flip p = 0.175082, FDR = 0.233443

20°:
  Late - early gap = 0.049° [95% CI -0.873, 0.971]
  Direction: AWAY from vision / toward audition (gap grew)
  t(49) = 0.107, p = 0.915387, FDR p = 0.915387
  sign-flip p = 0.923808, FDR = 0.923808

30°:
  Late - early gap = 1.100° [95% CI 0.092, 2.108]
  Direction: AWAY from visi

## 12. Psychometric Boundary and Individual-Difference Analyses

Subject-level PSE/psychometric-slope prediction, strong-versus-weak integrator analyses, and mixed-effects psychometric robustness checks.

## 13. Response Variance and Metacognitive Confidence

Response-variance/uncertainty analyses and post-trial confidence as support for explicit same/different reports.

In [ ]:
# %% CELL 2 — RESPONSE VARIANCE / UNCERTAINTY ANALYSES
# ============================================================
# Goal:
# Test whether response uncertainty tracks the causal binding boundary.
#
# Uses two predictors:
#   1. Predicted same-source probability from the mixed psychometric model
#      - internally named PredictedSameProb
#      - figures label this as "Predicted same-source probability"
#
#   2. Empirical subject-level PSE
#      - tests whether uncertainty is higher near each subject's empirical PSE
#
# Analyses:
#   A. Variance as a function of predicted same-source probability:
#        ResponseVariance ~ PredictedSameProb + PredictedSameProb²
#
#   B. Time-resolved uncertainty trajectories:
#        Variance(t) by offset
#        Variance(t) by predicted same-source probability bin
#
#   C. Same vs different judgment split:
#        Variance(t) and capture(t) split by post-trial judgment
#
# IMPORTANT:
# - This analysis does NOT derive true P(common) from variance.
# - It treats time-resolved response variance as a behavioral marker of
#   causal uncertainty / response instability.
# ============================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as t_dist, ttest_1samp
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

PSE_DIR = os.path.join(DATA_DIR, "spatial_mixed_effects_psychometric_pse")
PSE_CSV_DIR = os.path.join(PSE_DIR, "csv_outputs")

MIXED_SUBJECT_OFFSET_PRED_CSV = os.path.join(
    PSE_CSV_DIR,
    "mixed_model_subject_offset_predicted_same_probability.csv"
)

PSE_COMPARE_CSV = os.path.join(
    PSE_CSV_DIR,
    "mixed_model_predicted_pse_vs_empirical_pse.csv"
)

OUT_DIR = os.path.join(DATA_DIR, "spatial_response_uncertainty_binding_boundary_analysis")
CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

EXCLUDE_PRACTICE = True
FIXED_OFFSETS = [10, 20, 30, 40]

RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0
TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1.0 / RESAMPLE_HZ, 1.0 / RESAMPLE_HZ)

WINDOW_SEC = 1.0
STEP_SEC = 0.25
MAX_LAG_SEC = 2.0

MIN_VALID_SAMPLES = 8
MIN_TRIALS_FOR_VARIANCE = 2
MIN_ABS_DISPARITY_FOR_CAPTURE = 0.01

FIG_DPI = 180

# =========================
# HELPERS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else Path(filepath).stem

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def parse_same_response(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    same_tokens = [
        "same", "same source", "common", "common source",
        "together", "one", "1", "yes", "y", "true"
    ]

    diff_tokens = [
        "different", "diff", "different source", "separate",
        "separate source", "apart", "two", "2", "0",
        "no", "n", "false"
    ]

    if s in same_tokens:
        return 1.0
    if s in diff_tokens:
        return 0.0
    if "same" in s or "common" in s:
        return 1.0
    if "different" in s or "separate" in s or "diff" in s:
        return 0.0

    return np.nan

def judgment_label(x):
    if not np.isfinite(x):
        return np.nan
    return "Same" if int(x) == 1 else "Different"

def is_sweep_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    return ("sweep" in trial_type) or ("sweep" in cond) or ("diverge" in cond) or ("converge" in cond)

def is_fixed_offset_av_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    offset = extract_offset_from_row(row)

    if not np.isfinite(offset):
        return False

    if int(offset) not in FIXED_OFFSETS:
        return False

    if is_sweep_trial(row):
        return False

    if trial_type in ["audio_only", "visual_only", "congruent"]:
        return False

    if "incongruent" in trial_type or "fixed" in trial_type or "static" in trial_type:
        return True

    if "incongruent" in cond or "fixed" in cond or "static" in cond:
        return True

    return True

def resample_trial(df_trial, time_grid):
    needed = ["t", "mouse_x", "blob_x", "audio_x"]

    for c in needed:
        if c not in df_trial.columns:
            raise ValueError(f"Missing required column: {c}")

    d = df_trial.copy()
    d = d.sort_values("t")
    d = d[np.isfinite(d["t"])]
    d = d.drop_duplicates(subset="t", keep="first")

    if len(d) < 5:
        return None

    t_original = d["t"].values.astype(float)
    t_original = t_original - np.nanmin(t_original)

    if np.nanmax(t_original) < 1.0:
        return None

    def _interp(col):
        y = d[col].values.astype(float)
        ok = np.isfinite(t_original) & np.isfinite(y)

        if ok.sum() < 3:
            return np.full_like(time_grid, np.nan, dtype=float)

        return np.interp(
            time_grid,
            t_original[ok],
            y[ok],
            left=np.nan,
            right=np.nan
        )

    return {
        "t_grid": time_grid.copy(),
        "mouse_x": _interp("mouse_x"),
        "blob_x": _interp("blob_x"),
        "audio_x": _interp("audio_x"),
    }

def corr_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    x = x[ok]
    y = y[ok]

    if np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])

def lag_scanned_corr(response, stimulus, fs=20.0, max_lag_sec=2.0):
    response = np.asarray(response, dtype=float)
    stimulus = np.asarray(stimulus, dtype=float)

    max_lag_samples = int(round(max_lag_sec * fs))
    lag_samples_grid = np.arange(-max_lag_samples, max_lag_samples + 1)

    best_corr = np.nan
    best_lag_samples = np.nan

    for lag_samples in lag_samples_grid:
        if lag_samples > 0:
            r = response[lag_samples:]
            s = stimulus[:-lag_samples]
        elif lag_samples < 0:
            r = response[:lag_samples]
            s = stimulus[-lag_samples:]
        else:
            r = response
            s = stimulus

        c = corr_1d(r, s)

        if np.isfinite(c) and (not np.isfinite(best_corr) or c > best_corr):
            best_corr = c
            best_lag_samples = lag_samples

    return {
        "peak_corr": best_corr,
        "lag_samples": best_lag_samples,
        "lag_sec": best_lag_samples / fs if np.isfinite(best_lag_samples) else np.nan,
    }

def apply_response_lag_correction(response, t_grid, lag_sec):
    response = np.asarray(response, dtype=float)
    t_grid = np.asarray(t_grid, dtype=float)

    if not np.isfinite(lag_sec):
        return response.copy()

    return np.interp(
        t_grid + lag_sec,
        t_grid,
        response,
        left=np.nan,
        right=np.nan
    )

def visual_capture_pointwise(response, visual, audio):
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    denom = visual - audio
    out = np.full_like(response, np.nan, dtype=float)

    ok = (
        np.isfinite(response)
        & np.isfinite(visual)
        & np.isfinite(audio)
        & (np.abs(denom) > MIN_ABS_DISPARITY_FOR_CAPTURE)
    )

    out[ok] = (response[ok] - audio[ok]) / denom[ok]
    return out

def window_mean(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean(x[ok]))

def window_var(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)

    if ok.sum() < MIN_TRIALS_FOR_VARIANCE:
        return np.nan

    return float(np.nanvar(x[ok], ddof=1))

def sem(vals):
    vals = pd.Series(vals).dropna().astype(float).values

    if len(vals) < 2:
        return np.nan

    return np.std(vals, ddof=1) / np.sqrt(len(vals))

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

# =========================
# STEP 1 — LOAD PSE / PREDICTED VALUES
# =========================
if not os.path.exists(MIXED_SUBJECT_OFFSET_PRED_CSV):
    raise FileNotFoundError(
        "Mixed model subject-offset predicted values not found. "
        "Run Cell 1 first.\n"
        f"Missing: {MIXED_SUBJECT_OFFSET_PRED_CSV}"
    )

if not os.path.exists(PSE_COMPARE_CSV):
    raise FileNotFoundError(
        "Predicted vs empirical PSE file not found. "
        "Run Cell 1 first.\n"
        f"Missing: {PSE_COMPARE_CSV}"
    )

pred_subject_offset_df = pd.read_csv(MIXED_SUBJECT_OFFSET_PRED_CSV)
pse_compare_df = pd.read_csv(PSE_COMPARE_CSV)

pred_subject_offset_df["subject"] = pred_subject_offset_df["subject"].astype(str).str.lower()
pse_compare_df["subject"] = pse_compare_df["subject"].astype(str).str.lower()

# =========================
# STEP 2 — BUILD TRIAL × WINDOW CAPTURE TABLE
# =========================
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in {DATA_DIR}")

window_starts = np.arange(0, TRIAL_DURATION_SEC - WINDOW_SEC + 1e-9, STEP_SEC)

trial_window_rows = []

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    df = pd.read_csv(fpath)

    required_cols = [
        "trialIndex", "trialType", "condLabel", "isPractice",
        "t", "mouse_x", "blob_x", "audio_x"
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"Skipping {subj}: missing {missing}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"],
        sort=False
    ):
        first_row = df_trial.iloc[0]
        row_series = pd.Series(first_row)

        if not is_fixed_offset_av_trial(row_series):
            continue

        offset = extract_offset_from_row(row_series)

        same_response = np.nan
        if "post_question" in df_trial.columns:
            same_response = parse_same_response(first_row.get("post_question", np.nan))
        if not np.isfinite(same_response) and "post_question2" in df_trial.columns:
            same_response = parse_same_response(first_row.get("post_question2", np.nan))

        res = resample_trial(df_trial, TIME_GRID)
        if res is None:
            continue

        t = res["t_grid"]
        response_raw = res["mouse_x"]
        visual_x = res["blob_x"]
        audio_x = res["audio_x"]

        lag_res = lag_scanned_corr(
            response=response_raw,
            stimulus=visual_x,
            fs=RESAMPLE_HZ,
            max_lag_sec=MAX_LAG_SEC
        )

        response_lagcorr = apply_response_lag_correction(response_raw, t, lag_res["lag_sec"])

        raw_capture_ts = visual_capture_pointwise(response_raw, visual_x, audio_x)
        lagcorr_capture_ts = visual_capture_pointwise(response_lagcorr, visual_x, audio_x)

        for start in window_starts:
            end = start + WINDOW_SEC
            center = start + WINDOW_SEC / 2

            mask = (t >= start) & (t < end)

            if mask.sum() < MIN_VALID_SAMPLES:
                continue

            trial_window_rows.append({
                "subject": subj,
                "source_file": Path(fpath).name,
                "trialIndex": trial_index,
                "trialType": trial_type,
                "condLabel": cond_label,
                "offset_deg": float(offset),
                "same_response": same_response,
                "judgment": judgment_label(same_response),

                "window_start_sec": start,
                "window_end_sec": end,
                "window_center_sec": center,

                "visual_lag_sec": lag_res["lag_sec"],
                "visual_peak_corr": lag_res["peak_corr"],

                "RawCapture_mean": window_mean(raw_capture_ts[mask]),
                "LagCorrectedCapture_mean": window_mean(lagcorr_capture_ts[mask]),

                "RawResponse_mean": window_mean(response_raw[mask]),
                "LagCorrectedResponse_mean": window_mean(response_lagcorr[mask]),
            })

trial_window_df = pd.DataFrame(trial_window_rows)

if len(trial_window_df) == 0:
    raise RuntimeError("No trial-window rows were created.")

trial_window_df["subject"] = trial_window_df["subject"].astype(str).str.lower()

trial_window_csv = os.path.join(CSV_DIR, "trial_window_capture_for_uncertainty.csv")
trial_window_df.to_csv(trial_window_csv, index=False)

print(f"Saved trial-window capture table: {trial_window_csv}")
print(f"Subjects: {trial_window_df['subject'].nunique()}")
print(f"Rows: {len(trial_window_df)}")

# =========================
# STEP 3 — RESPONSE UNCERTAINTY ACROSS REPEATED TRIALS
# =========================
# Primary uncertainty = variance across repeated trials in lag-corrected capture.
# Also compute raw capture variance and response-position variance.

uncertainty_df = (
    trial_window_df
    .groupby(["subject", "offset_deg", "window_center_sec"], as_index=False)
    .agg(
        n_trials=("trialIndex", "nunique"),
        LagCorrectedCaptureVariance=("LagCorrectedCapture_mean", window_var),
        RawCaptureVariance=("RawCapture_mean", window_var),
        LagCorrectedResponseVariance=("LagCorrectedResponse_mean", window_var),
        RawResponseVariance=("RawResponse_mean", window_var),
        LagCorrectedCaptureMean=("LagCorrectedCapture_mean", "mean"),
        RawCaptureMean=("RawCapture_mean", "mean"),
    )
)

uncertainty_df = uncertainty_df[uncertainty_df["n_trials"] >= MIN_TRIALS_FOR_VARIANCE].copy()

# Merge predicted same-source probability and empirical/predicted PSE
uncertainty_df = pd.merge(
    uncertainty_df,
    pred_subject_offset_df[["subject", "offset_deg", "PredictedSameProb", "PredictedPSE_deg"]],
    on=["subject", "offset_deg"],
    how="left"
)

uncertainty_df = pd.merge(
    uncertainty_df,
    pse_compare_df[["subject", "EmpiricalPSE_deg"]],
    on="subject",
    how="left"
)

uncertainty_df["PredictedSameProb_centered"] = uncertainty_df["PredictedSameProb"] - 0.5
uncertainty_df["PredictedSameProb_centered_sq"] = uncertainty_df["PredictedSameProb_centered"] ** 2

uncertainty_df["DistanceFromEmpiricalPSE_deg"] = np.abs(
    uncertainty_df["offset_deg"] - uncertainty_df["EmpiricalPSE_deg"]
)

uncertainty_df["DistanceFromPredictedPSE_deg"] = np.abs(
    uncertainty_df["offset_deg"] - uncertainty_df["PredictedPSE_deg"]
)

uncertainty_df["NearEmpiricalPSE"] = -uncertainty_df["DistanceFromEmpiricalPSE_deg"]
uncertainty_df["NearPredictedPSE"] = -uncertainty_df["DistanceFromPredictedPSE_deg"]

uncertainty_csv = os.path.join(CSV_DIR, "subject_offset_time_response_uncertainty.csv")
uncertainty_df.to_csv(uncertainty_csv, index=False)

# =========================
# STEP 4 — JUDGMENT-SPLIT UNCERTAINTY AND CAPTURE
# =========================
judgment_uncertainty_df = (
    trial_window_df
    .dropna(subset=["judgment"])
    .groupby(["subject", "offset_deg", "judgment", "window_center_sec"], as_index=False)
    .agg(
        n_trials=("trialIndex", "nunique"),
        LagCorrectedCaptureVariance=("LagCorrectedCapture_mean", window_var),
        RawCaptureVariance=("RawCapture_mean", window_var),
        LagCorrectedCaptureMean=("LagCorrectedCapture_mean", "mean"),
        RawCaptureMean=("RawCapture_mean", "mean"),
    )
)

judgment_uncertainty_df = judgment_uncertainty_df[
    judgment_uncertainty_df["n_trials"] >= MIN_TRIALS_FOR_VARIANCE
].copy()

judgment_uncertainty_df = pd.merge(
    judgment_uncertainty_df,
    pred_subject_offset_df[["subject", "offset_deg", "PredictedSameProb", "PredictedPSE_deg"]],
    on=["subject", "offset_deg"],
    how="left"
)

judgment_uncertainty_df = pd.merge(
    judgment_uncertainty_df,
    pse_compare_df[["subject", "EmpiricalPSE_deg"]],
    on="subject",
    how="left"
)

judgment_uncertainty_csv = os.path.join(CSV_DIR, "same_different_judgment_split_uncertainty.csv")
judgment_uncertainty_df.to_csv(judgment_uncertainty_csv, index=False)

# =========================
# STEP 5 — MODELS
# =========================
model_rows = []

outcomes = [
    ("LagCorrectedCaptureVariance", "Lag-corrected capture variance"),
    ("RawCaptureVariance", "Raw capture variance"),
    ("LagCorrectedResponseVariance", "Lag-corrected response variance"),
    ("RawResponseVariance", "Raw response variance"),
]

model_specs = []

for outcome, label in outcomes:
    model_specs.append((
        f"{outcome}_predicted_same_probability_quadratic",
        outcome,
        label,
        f"{outcome} ~ PredictedSameProb_centered + PredictedSameProb_centered_sq"
    ))

    model_specs.append((
        f"{outcome}_near_empirical_pse",
        outcome,
        label,
        f"{outcome} ~ NearEmpiricalPSE"
    ))

    model_specs.append((
        f"{outcome}_near_predicted_pse",
        outcome,
        label,
        f"{outcome} ~ NearPredictedPSE"
    ))

    model_specs.append((
        f"{outcome}_time_offset",
        outcome,
        label,
        f"{outcome} ~ window_center_sec * offset_deg"
    ))

for model_name, outcome, label, formula in model_specs:
    needed = ["subject", outcome]

    for candidate in [
        "PredictedSameProb_centered",
        "PredictedSameProb_centered_sq",
        "NearEmpiricalPSE",
        "NearPredictedPSE",
        "window_center_sec",
        "offset_deg",
    ]:
        if candidate in formula:
            needed.append(candidate)

    d = uncertainty_df[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(d) < 20 or d["subject"].nunique() < 5:
        continue

    fit = smf.ols(formula, data=d).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["subject"]}
    )

    for term in fit.params.index:
        model_rows.append({
            "model_name": model_name,
            "outcome": outcome,
            "outcome_label": label,
            "term": term,
            "n_rows": len(d),
            "n_subjects": d["subject"].nunique(),
            "estimate": fit.params[term],
            "se": fit.bse[term],
            "t": fit.tvalues[term],
            "p": fit.pvalues[term],
            "r_squared": fit.rsquared,
            "formula": formula,
        })

# Judgment split models
judgment_model_specs = [
    ("LagCorrectedCaptureVariance", "Lag-corrected capture variance"),
    ("RawCaptureVariance", "Raw capture variance"),
    ("LagCorrectedCaptureMean", "Lag-corrected capture mean"),
    ("RawCaptureMean", "Raw capture mean"),
]

for outcome, label in judgment_model_specs:
    d = judgment_uncertainty_df[
        ["subject", "offset_deg", "window_center_sec", "judgment", outcome]
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(d) < 20 or d["subject"].nunique() < 5:
        continue

    d["judgment"] = pd.Categorical(d["judgment"], categories=["Different", "Same"])

    formula = f"{outcome} ~ C(judgment) * offset_deg + window_center_sec"

    fit = smf.ols(formula, data=d).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["subject"]}
    )

    for term in fit.params.index:
        model_rows.append({
            "model_name": f"judgment_split_{outcome}",
            "outcome": outcome,
            "outcome_label": label,
            "term": term,
            "n_rows": len(d),
            "n_subjects": d["subject"].nunique(),
            "estimate": fit.params[term],
            "se": fit.bse[term],
            "t": fit.tvalues[term],
            "p": fit.pvalues[term],
            "r_squared": fit.rsquared,
            "formula": formula,
        })

model_results_df = pd.DataFrame(model_rows)

model_results_csv = os.path.join(CSV_DIR, "response_uncertainty_model_results.csv")
model_results_df.to_csv(model_results_csv, index=False)

# =========================
# STEP 6 — FIGURES
# =========================

# Figure 1: Uncertainty over time by offset
for outcome, label in [
    ("LagCorrectedCaptureVariance", "Lag-corrected capture variance"),
    ("RawCaptureVariance", "Raw capture variance"),
]:
    group_tc = (
        uncertainty_df
        .groupby(["offset_deg", "window_center_sec"], as_index=False)
        .agg(
            mean=(outcome, "mean"),
            se=(outcome, sem),
            n=("subject", "nunique")
        )
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), sharex=True)
    axes = axes.flatten()

    for ax, offset in zip(axes, FIXED_OFFSETS):
        sub = group_tc[group_tc["offset_deg"] == offset].copy()

        ax.plot(sub["window_center_sec"], sub["mean"], linewidth=2.2)
        ax.fill_between(
            sub["window_center_sec"],
            sub["mean"] - sub["se"],
            sub["mean"] + sub["se"],
            alpha=0.20
        )

        ax.set_title(f"Offset {offset}°")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(label)

    fig.suptitle(f"Time-resolved response uncertainty: {label}", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    fig_path = os.path.join(FIG_DIR, f"timecourse_by_offset_{outcome}.png")
    fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# Figure 2: Variance as function of predicted same-source probability
for outcome, label in [
    ("LagCorrectedCaptureVariance", "Lag-corrected capture variance"),
    ("RawCaptureVariance", "Raw capture variance"),
]:
    d = uncertainty_df[
        ["PredictedSameProb", "PredictedSameProb_centered", "PredictedSameProb_centered_sq", outcome]
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(d) >= 20:
        fit = smf.ols(
            f"{outcome} ~ PredictedSameProb_centered + PredictedSameProb_centered_sq",
            data=d
        ).fit()

        x_grid = np.linspace(d["PredictedSameProb"].min(), d["PredictedSameProb"].max(), 150)
        pred_df = pd.DataFrame({
            "PredictedSameProb": x_grid,
            "PredictedSameProb_centered": x_grid - 0.5,
            "PredictedSameProb_centered_sq": (x_grid - 0.5) ** 2,
        })

        pred = fit.get_prediction(pred_df).summary_frame(alpha=0.05)

        fig, ax = plt.subplots(figsize=(7.2, 5.6))

        ax.scatter(
            d["PredictedSameProb"],
            d[outcome],
            alpha=0.25,
            s=22
        )

        ax.plot(x_grid, pred["mean"], linewidth=2.4)
        ax.fill_between(
            x_grid,
            pred["mean_ci_lower"].values,
            pred["mean_ci_upper"].values,
            alpha=0.20
        )

        ax.axvline(0.5, linestyle="--", linewidth=1.2, alpha=0.75)
        ax.set_xlabel("Predicted same-source probability")
        ax.set_ylabel(label)
        ax.set_title(f"Does uncertainty peak near the binding boundary?")

        txt = (
            f"Linear β = {fit.params['PredictedSameProb_centered']:.4f}, p {p_fmt(fit.pvalues['PredictedSameProb_centered'])}\n"
            f"Quadratic β = {fit.params['PredictedSameProb_centered_sq']:.4f}, p {p_fmt(fit.pvalues['PredictedSameProb_centered_sq'])}"
        )

        ax.text(
            0.04,
            0.96,
            txt,
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

        fig.tight_layout()
        fig_path = os.path.join(FIG_DIR, f"uncertainty_by_predicted_same_probability_{outcome}.png")
        fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()

# Figure 3: Variance as function of distance from empirical PSE
for outcome, label in [
    ("LagCorrectedCaptureVariance", "Lag-corrected capture variance"),
    ("RawCaptureVariance", "Raw capture variance"),
]:
    d = uncertainty_df[
        ["DistanceFromEmpiricalPSE_deg", outcome]
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(d) >= 20:
        fit = smf.ols(f"{outcome} ~ DistanceFromEmpiricalPSE_deg", data=d).fit()

        x_grid = np.linspace(d["DistanceFromEmpiricalPSE_deg"].min(), d["DistanceFromEmpiricalPSE_deg"].max(), 150)
        pred = fit.get_prediction(
            pd.DataFrame({"DistanceFromEmpiricalPSE_deg": x_grid})
        ).summary_frame(alpha=0.05)

        fig, ax = plt.subplots(figsize=(7.2, 5.6))

        ax.scatter(d["DistanceFromEmpiricalPSE_deg"], d[outcome], alpha=0.25, s=22)
        ax.plot(x_grid, pred["mean"], linewidth=2.4)
        ax.fill_between(
            x_grid,
            pred["mean_ci_lower"].values,
            pred["mean_ci_upper"].values,
            alpha=0.20
        )

        ax.set_xlabel("Distance from empirical PSE (deg)")
        ax.set_ylabel(label)
        ax.set_title("Does uncertainty increase near empirical PSE?")

        txt = f"β = {fit.params['DistanceFromEmpiricalPSE_deg']:.4f}, p {p_fmt(fit.pvalues['DistanceFromEmpiricalPSE_deg'])}"
        ax.text(
            0.04,
            0.96,
            txt,
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

        fig.tight_layout()
        fig_path = os.path.join(FIG_DIR, f"uncertainty_by_distance_from_empirical_pse_{outcome}.png")
        fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.show()

# Figure 4: Judgment split over time by offset
for outcome, label in [
    ("LagCorrectedCaptureVariance", "Lag-corrected capture variance"),
    ("LagCorrectedCaptureMean", "Lag-corrected capture mean"),
]:
    group_j = (
        judgment_uncertainty_df
        .groupby(["offset_deg", "judgment", "window_center_sec"], as_index=False)
        .agg(
            mean=(outcome, "mean"),
            se=(outcome, sem),
            n=("subject", "nunique")
        )
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), sharex=True)
    axes = axes.flatten()

    for ax, offset in zip(axes, FIXED_OFFSETS):
        sub_offset = group_j[group_j["offset_deg"] == offset].copy()

        for judgment in ["Same", "Different"]:
            sub = sub_offset[sub_offset["judgment"] == judgment].copy()

            if len(sub) == 0:
                continue

            ax.plot(
                sub["window_center_sec"],
                sub["mean"],
                linewidth=2.2,
                label=judgment
            )

            ax.fill_between(
                sub["window_center_sec"],
                sub["mean"] - sub["se"],
                sub["mean"] + sub["se"],
                alpha=0.18
            )

        if "CaptureMean" in outcome:
            ax.axhline(0.5, linestyle="--", linewidth=1.0, alpha=0.6)

        ax.set_title(f"Offset {offset}°")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(label)
        ax.legend(frameon=False)

    fig.suptitle(f"Same vs different judgment split: {label}", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    fig_path = os.path.join(FIG_DIR, f"same_different_timecourse_{outcome}.png")
    fig.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# =========================
# FINAL PRINT
# =========================
print("\n============================================================")
print("RESPONSE UNCERTAINTY / BINDING BOUNDARY ANALYSIS COMPLETE")
print("============================================================")
print(f"Trial-window table:              {trial_window_csv}")
print(f"Subject-offset-time uncertainty: {uncertainty_csv}")
print(f"Judgment-split uncertainty:      {judgment_uncertainty_csv}")
print(f"Model results:                   {model_results_csv}")
print(f"Figures folder:                  {FIG_DIR}")

print("\nKey model results:")
if len(model_results_df) > 0:
    print(model_results_df[[
        "model_name", "outcome_label", "term", "estimate", "se", "t", "p", "n_rows", "n_subjects", "r_squared"
    ]].to_string(index=False))

print("\nInterpretation guide:")
print("- A significant quadratic effect for predicted same-source probability means uncertainty changes nonlinearly around the binding boundary.")
print("- If the quadratic term is negative, uncertainty is highest near predicted same-source probability = .50.")
print("- A negative distance-from-PSE coefficient means uncertainty is higher closer to the empirical PSE.")
print("- Same vs different split tests whether subjective causal judgment separates response stability trajectories at the same physical offset.")
print("- Use 'Predicted PSE' or 'Empirical PSE' language in write-ups; avoid describing variance as directly deriving P(common).")

In [ ]:
# %% POST-TRIAL CONFIDENCE AS METACOGNITIVE SUPPORT FOR EXPLICIT SAME/DIFFERENT REPORTS
# ============================================================
# Question:
# Does post-trial confidence track whether the participant's explicit
# same/different report is supported by spatial disparity?
#
# Core model, fit subject-first:
#   confidence ~ spatial_disparity * reported_different
#
# Interpretation:
#   - same-source reports should be more confident at small disparities
#   - different-source reports should be more confident at large disparities
#   - a positive interaction means confidence increasingly favors
#     "different" reports as disparity grows
#
# Complementary support-score model:
#   report_support =
#       1 - disparity/max_disparity, if reported same
#       disparity/max_disparity,     if reported different
#
#   confidence ~ report_support + reported_different
#
# Interpretation:
#   Higher confidence when the explicit report is more supported by
#   the spatial evidence.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# -------------------------
# SETTINGS
# -------------------------
RANDOM_SEED = 12345
N_PERM = 10000
MAX_FIXED_OFFSET = 40.0
MIN_OBS_PER_SUBJECT_MODEL = 12
FIG_DPI = 200

try:
    OUT_DIR = os.path.join(out_root, "posttrial_confidence_report_by_disparity")
except NameError:
    OUT_DIR = os.path.join(os.getcwd(), "posttrial_confidence_report_by_disparity")

CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

if "trial_df" not in globals():
    raise RuntimeError("trial_df not found. Run the spatial notebook load/resample cell first.")

# -------------------------
# HELPERS
# -------------------------
def parse_yes_no_same_response(x):
    """
    Interprets post_question as the participant's same-source response.
    same/common/yes = 1
    different/separate/no = 0
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    same_tokens = [
        "y", "yes", "1", "true",
        "same", "same source", "common", "common source"
    ]

    different_tokens = [
        "n", "no", "0", "false",
        "different", "diff", "different source",
        "separate", "separate source"
    ]

    if s in same_tokens:
        return 1.0
    if s in different_tokens:
        return 0.0

    if "same" in s or "common" in s:
        return 1.0
    if "different" in s or "separate" in s or "diff" in s:
        return 0.0

    return np.nan


def parse_confidence(x):
    """
    Parse post_question2 confidence rating.
    Expected scale: 1-5.
    """
    val = pd.to_numeric(x, errors="coerce")
    if pd.isna(val):
        return np.nan
    return float(val)


def extract_offset_abs(row):
    """
    Extract absolute audiovisual spatial offset.
    Congruent trials are assigned 0.
    """
    trial_type = str(row.get("trialType", "")).lower()

    if trial_type == "congruent":
        return 0.0

    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return abs(float(row["offsetDeg"]))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return float(m.group(1))

    return np.nan


def extract_side_label(row):
    """
    Extract left/right/center label for descriptive output.
    """
    trial_type = str(row.get("trialType", "")).lower()
    if trial_type == "congruent":
        return "center"

    side = row.get("sideSign", np.nan)
    try:
        side_num = float(side)
        if side_num < 0:
            return "left"
        if side_num > 0:
            return "right"
    except Exception:
        pass

    cond = str(row.get("condLabel", "")).lower()
    if "left" in cond:
        return "left"
    if "right" in cond:
        return "right"

    return "unknown"


def mean_ci_t(values, alpha=0.05):
    vals = pd.Series(values).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)

    return m, m - crit * se, m + crit * se, se, n


def sign_flip_test(values, popmean=0.0, alternative="two-sided", n_perm=N_PERM, seed=RANDOM_SEED):
    vals = pd.Series(values).dropna().astype(float).values
    diffs = vals - popmean
    n = len(diffs)

    if n < 2:
        return np.nan, n

    obs = np.mean(diffs)
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1, 1], size=(n_perm, n))
    null = np.mean(signs * diffs, axis=1)

    if alternative == "greater":
        p = (np.sum(null >= obs) + 1) / (n_perm + 1)
    elif alternative == "less":
        p = (np.sum(null <= obs) + 1) / (n_perm + 1)
    else:
        p = (np.sum(np.abs(null) >= np.abs(obs)) + 1) / (n_perm + 1)

    return p, n


def fdr_bh(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)

    valid = np.isfinite(p)
    pv = p[valid]

    if len(pv) == 0:
        return out

    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)

    adj = ranked * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)

    tmp = np.empty_like(adj)
    tmp[order] = adj
    out[valid] = tmp

    return out


def safe_fit_subject_ols(data, formula, required_terms, min_n=MIN_OBS_PER_SUBJECT_MODEL):
    out = {
        "n_obs": len(data),
        "model_r2": np.nan,
        "model_adj_r2": np.nan,
        "model_failed": False,
        "model_error": "",
        "rank_deficient": np.nan,
        "condition_number": np.nan
    }

    for term in required_terms:
        out[term] = np.nan

    if len(data) < min_n:
        out["model_failed"] = True
        out["model_error"] = f"Too few observations: {len(data)}"
        return out

    try:
        m = smf.ols(formula, data=data).fit()

        out["model_r2"] = m.rsquared
        out["model_adj_r2"] = m.rsquared_adj

        exog = np.asarray(m.model.exog, dtype=float)
        rank = np.linalg.matrix_rank(exog)
        n_params = exog.shape[1]

        out["rank_deficient"] = bool(rank < n_params)
        out["condition_number"] = np.linalg.cond(exog)

        for term in required_terms:
            out[term] = m.params.get(term, np.nan)

    except Exception as e:
        out["model_failed"] = True
        out["model_error"] = str(e)

    return out


# -------------------------
# BUILD ANALYSIS DATASET
# -------------------------
df_conf = trial_df.copy()

subject_col = "subject" if "subject" in df_conf.columns else "participant_id"
if subject_col not in df_conf.columns:
    raise RuntimeError("Could not find a subject column named 'subject' or 'participant_id'.")

if "same_response_yes" not in df_conf.columns:
    if "post_question" not in df_conf.columns:
        raise RuntimeError("Need either same_response_yes or post_question in trial_df.")
    df_conf["same_response_yes"] = df_conf["post_question"].apply(parse_yes_no_same_response)

if "confidence_rating" not in df_conf.columns:
    if "post_question2" not in df_conf.columns:
        raise RuntimeError("Need either confidence_rating or post_question2 in trial_df.")
    df_conf["confidence_rating"] = df_conf["post_question2"].apply(parse_confidence)

df_conf["offset_abs"] = df_conf.apply(extract_offset_abs, axis=1)
df_conf["side_label"] = df_conf.apply(extract_side_label, axis=1)

# Fixed spatial evidence trials: congruent + fixed spatial offsets.
# Excludes dynamic sweeps because their disparity changes continuously.
fixed_mask = df_conf["trialType"].astype(str).str.lower().isin([
    "congruent",
    "incongruent_spatial"
])

model_df_conf = df_conf[fixed_mask].copy()

model_df_conf = model_df_conf.dropna(
    subset=[subject_col, "trialIndex", "confidence_rating", "same_response_yes", "offset_abs"]
).copy()

model_df_conf["same_report"] = model_df_conf["same_response_yes"].astype(float)
model_df_conf["different_report"] = 1.0 - model_df_conf["same_report"]

model_df_conf["report_label"] = np.where(
    model_df_conf["same_report"] == 1,
    "same",
    "different"
)

model_df_conf["offset_10"] = model_df_conf["offset_abs"] / 10.0
model_df_conf["offset_10_c"] = model_df_conf["offset_10"] - model_df_conf["offset_10"].mean()

# Evidence support for the report:
# same reports are most supported at low disparity; different reports at high disparity.
model_df_conf["offset_norm_0_1"] = model_df_conf["offset_abs"] / MAX_FIXED_OFFSET
model_df_conf["report_support"] = np.where(
    model_df_conf["different_report"] == 1,
    model_df_conf["offset_norm_0_1"],
    1.0 - model_df_conf["offset_norm_0_1"]
)

model_df_conf["condition_fixed"] = np.where(
    model_df_conf["offset_abs"] == 0,
    "congruent",
    "spatial_" + model_df_conf["offset_abs"].astype(int).astype(str)
)

model_df_conf["condition_fixed_side"] = np.where(
    model_df_conf["offset_abs"] == 0,
    "congruent",
    model_df_conf["condition_fixed"] + "_" + model_df_conf["side_label"]
)

model_df_conf = model_df_conf.rename(columns={subject_col: "subject_id"})

model_df_conf.to_csv(
    os.path.join(CSV_DIR, "posttrial_confidence_disparity_report_model_df.csv"),
    index=False
)

print("\nAnalysis dataset:")
print(f"Rows: {len(model_df_conf):,}")
print(f"Subjects: {model_df_conf['subject_id'].nunique()}")
print(model_df_conf[["subject_id", "trialIndex", "trialType", "condition_fixed_side", "offset_abs", "report_label", "confidence_rating", "report_support"]].head())


# -------------------------
# DESCRIPTIVE SUBJECT-FIRST MEANS BY OFFSET × REPORT
# -------------------------
subj_offset_report = (
    model_df_conf
    .groupby(["subject_id", "offset_abs", "report_label"], as_index=False, observed=True, sort=False)
    .agg(
        confidence_mean=("confidence_rating", "mean"),
        n_trials=("trialIndex", "nunique")
    )
)

group_offset_report = (
    subj_offset_report
    .groupby(["offset_abs", "report_label"], as_index=False, observed=True, sort=False)
    .agg(
        confidence_mean=("confidence_mean", "mean"),
        confidence_sd=("confidence_mean", "std"),
        n_subjects=("subject_id", "nunique"),
        n_trials=("n_trials", "sum")
    )
)

group_offset_report["confidence_sem"] = (
    group_offset_report["confidence_sd"] / np.sqrt(group_offset_report["n_subjects"])
)
group_offset_report["ci95_low"] = group_offset_report.apply(
    lambda r: r["confidence_mean"] - t_dist.ppf(.975, r["n_subjects"] - 1) * r["confidence_sem"]
    if r["n_subjects"] > 1 else np.nan,
    axis=1
)
group_offset_report["ci95_high"] = group_offset_report.apply(
    lambda r: r["confidence_mean"] + t_dist.ppf(.975, r["n_subjects"] - 1) * r["confidence_sem"]
    if r["n_subjects"] > 1 else np.nan,
    axis=1
)

subj_offset_report.to_csv(
    os.path.join(CSV_DIR, "subject_confidence_by_offset_and_report.csv"),
    index=False
)

group_offset_report.to_csv(
    os.path.join(CSV_DIR, "group_confidence_by_offset_and_report.csv"),
    index=False
)

# Plot confidence by offset and report.
plt.figure(figsize=(8, 5))

for label in ["same", "different"]:
    g = group_offset_report[group_offset_report["report_label"] == label].sort_values("offset_abs")
    if len(g) == 0:
        continue

    plt.plot(
        g["offset_abs"],
        g["confidence_mean"],
        marker="o",
        linewidth=2.5,
        label=f"Reported {label}"
    )

    plt.fill_between(
        g["offset_abs"].astype(float),
        g["ci95_low"].astype(float),
        g["ci95_high"].astype(float),
        alpha=0.18
    )

plt.xlabel("Audiovisual spatial disparity (°)")
plt.ylabel("Post-trial confidence rating (1–5)")
plt.title("Post-trial confidence by spatial disparity and explicit report")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "confidence_by_offset_and_report.png"),
    dpi=FIG_DPI,
    bbox_inches="tight"
)
plt.show()

# Plot report-support function.
subj_support_bins = model_df_conf.copy()
subj_support_bins["support_bin"] = pd.cut(
    subj_support_bins["report_support"],
    bins=np.linspace(0, 1, 6),
    include_lowest=True
)

support_subject_bin = (
    subj_support_bins
    .groupby(["subject_id", "support_bin"], as_index=False, observed=True, sort=False)
    .agg(
        report_support_mean=("report_support", "mean"),
        confidence_mean=("confidence_rating", "mean"),
        n_trials=("trialIndex", "nunique")
    )
)

support_group_bin = (
    support_subject_bin
    .groupby("support_bin", as_index=False, observed=True, sort=False)
    .agg(
        report_support_mean=("report_support_mean", "mean"),
        confidence_mean=("confidence_mean", "mean"),
        confidence_sd=("confidence_mean", "std"),
        n_subjects=("subject_id", "nunique"),
        n_trials=("n_trials", "sum")
    )
)

support_group_bin["confidence_sem"] = support_group_bin["confidence_sd"] / np.sqrt(support_group_bin["n_subjects"])
support_group_bin["ci95_low"] = support_group_bin["confidence_mean"] - 1.96 * support_group_bin["confidence_sem"]
support_group_bin["ci95_high"] = support_group_bin["confidence_mean"] + 1.96 * support_group_bin["confidence_sem"]

support_subject_bin.to_csv(
    os.path.join(CSV_DIR, "subject_confidence_by_report_support_bin.csv"),
    index=False
)

support_group_bin.to_csv(
    os.path.join(CSV_DIR, "group_confidence_by_report_support_bin.csv"),
    index=False
)

plt.figure(figsize=(7, 5))

g = support_group_bin.dropna(subset=["report_support_mean", "confidence_mean"]).sort_values("report_support_mean")

plt.plot(
    g["report_support_mean"],
    g["confidence_mean"],
    marker="o",
    linewidth=2.5
)

plt.fill_between(
    g["report_support_mean"].astype(float),
    g["ci95_low"].astype(float),
    g["ci95_high"].astype(float),
    alpha=0.18
)

plt.xlabel("Spatial evidence support for explicit report")
plt.ylabel("Post-trial confidence rating (1–5)")
plt.title("Confidence increases when the report is supported by spatial evidence")
plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "confidence_by_report_support.png"),
    dpi=FIG_DPI,
    bbox_inches="tight"
)
plt.show()


# -------------------------
# SUBJECT-FIRST INTERACTION MODELS
# -------------------------
interaction_formula = "confidence_rating ~ offset_10_c * different_report"
interaction_required_terms = [
    "Intercept",
    "offset_10_c",
    "different_report",
    "offset_10_c:different_report"
]

support_formula = "confidence_rating ~ report_support + different_report"
support_required_terms = [
    "Intercept",
    "report_support",
    "different_report"
]

coef_rows = []

for sid, g in model_df_conf.groupby("subject_id", observed=True, sort=False):
    g = g.dropna(
        subset=["confidence_rating", "offset_10_c", "different_report", "report_support"]
    ).copy()

    # Need variability in report and offset to estimate interaction meaningfully.
    has_report_variation = g["different_report"].nunique() >= 2
    has_offset_variation = g["offset_10_c"].nunique() >= 2

    interaction_out = {}
    if has_report_variation and has_offset_variation:
        interaction_out = safe_fit_subject_ols(
            g,
            formula=interaction_formula,
            required_terms=interaction_required_terms,
            min_n=MIN_OBS_PER_SUBJECT_MODEL
        )
    else:
        interaction_out = {
            "n_obs": len(g),
            "model_r2": np.nan,
            "model_adj_r2": np.nan,
            "model_failed": True,
            "model_error": "Insufficient report or offset variation",
            "rank_deficient": np.nan,
            "condition_number": np.nan
        }
        for term in interaction_required_terms:
            interaction_out[term] = np.nan

    support_out = safe_fit_subject_ols(
        g,
        formula=support_formula,
        required_terms=support_required_terms,
        min_n=MIN_OBS_PER_SUBJECT_MODEL
    )

    b_same_slope = interaction_out.get("offset_10_c", np.nan)
    b_interaction = interaction_out.get("offset_10_c:different_report", np.nan)
    b_diff_slope = (
        b_same_slope + b_interaction
        if np.isfinite(b_same_slope) and np.isfinite(b_interaction)
        else np.nan
    )

    coef_rows.append({
        "subject_id": sid,
        "n_obs": len(g),
        "n_same_reports": int(np.sum(g["same_report"] == 1)),
        "n_different_reports": int(np.sum(g["different_report"] == 1)),
        "n_offsets": int(g["offset_abs"].nunique()),

        "interaction_intercept": interaction_out.get("Intercept", np.nan),
        "interaction_same_report_offset_slope": b_same_slope,
        "interaction_different_report_main_effect": interaction_out.get("different_report", np.nan),
        "interaction_offset_x_different_report": b_interaction,
        "interaction_different_report_offset_slope": b_diff_slope,
        "interaction_model_r2": interaction_out.get("model_r2", np.nan),
        "interaction_model_adj_r2": interaction_out.get("model_adj_r2", np.nan),
        "interaction_rank_deficient": interaction_out.get("rank_deficient", np.nan),
        "interaction_condition_number": interaction_out.get("condition_number", np.nan),
        "interaction_model_failed": interaction_out.get("model_failed", False),
        "interaction_model_error": interaction_out.get("model_error", ""),

        "support_intercept": support_out.get("Intercept", np.nan),
        "support_slope": support_out.get("report_support", np.nan),
        "support_different_report": support_out.get("different_report", np.nan),
        "support_model_r2": support_out.get("model_r2", np.nan),
        "support_model_adj_r2": support_out.get("model_adj_r2", np.nan),
        "support_rank_deficient": support_out.get("rank_deficient", np.nan),
        "support_condition_number": support_out.get("condition_number", np.nan),
        "support_model_failed": support_out.get("model_failed", False),
        "support_model_error": support_out.get("model_error", "")
    })

subject_coef_df = pd.DataFrame(coef_rows)

subject_coef_df.to_csv(
    os.path.join(CSV_DIR, "subject_first_confidence_disparity_report_coefficients.csv"),
    index=False
)

# -------------------------
# GROUP TESTS ON SUBJECT-LEVEL COEFFICIENTS
# -------------------------
tests = []

test_specs = [
    ("interaction_offset_x_different_report", "greater", "Does disparity increasingly favor confidence in different-source reports?"),
    ("interaction_same_report_offset_slope", "less", "Does confidence in same-source reports decrease as disparity increases?"),
    ("interaction_different_report_offset_slope", "greater", "Does confidence in different-source reports increase as disparity increases?"),
    ("support_slope", "greater", "Does confidence increase when the explicit report is supported by disparity?")
]

for metric, alt, desc in test_specs:
    vals = subject_coef_df[metric].dropna().astype(float)
    m, lo, hi, se, n = mean_ci_t(vals)
    p, n_perm = sign_flip_test(vals, alternative=alt, seed=RANDOM_SEED + len(tests) * 10)

    tests.append({
        "metric": metric,
        "description": desc,
        "alternative": alt,
        "n_subjects": n,
        "mean": m,
        "se": se,
        "ci95_low": lo,
        "ci95_high": hi,
        "p_perm": p
    })

test_df = pd.DataFrame(tests)
test_df["p_fdr"] = fdr_bh(test_df["p_perm"].values)

test_df.to_csv(
    os.path.join(CSV_DIR, "subject_first_confidence_disparity_report_signflip_tests.csv"),
    index=False
)

print("\nSubject-first coefficients:")
try:
    display(subject_coef_df.head())
except NameError:
    print(subject_coef_df.head().to_string(index=False))

print("\nSubject-first group tests:")
try:
    display(test_df)
except NameError:
    print(test_df.to_string(index=False))


# -------------------------
# OPTIONAL CLUSTER-ROBUST TRIAL-LEVEL ROBUSTNESS MODELS
# -------------------------
cluster_tables = []

cluster_specs = {
    "confidence_offset_x_report_clustered": "confidence_rating ~ offset_10_c * different_report",
    "confidence_report_support_clustered": "confidence_rating ~ report_support + different_report"
}

for model_name, formula in cluster_specs.items():
    try:
        m = smf.ols(formula, data=model_df_conf).fit(
            cov_type="cluster",
            cov_kwds={"groups": model_df_conf["subject_id"]}
        )

        with open(os.path.join(CSV_DIR, f"{model_name}.txt"), "w") as f:
            f.write(str(m.summary()))

        ci = m.conf_int()
        for term in m.params.index:
            cluster_tables.append({
                "model": model_name,
                "term": term,
                "b": m.params[term],
                "SE_cluster": m.bse[term],
                "z": m.tvalues[term],
                "p": m.pvalues[term],
                "ci95_low": ci.loc[term, 0],
                "ci95_high": ci.loc[term, 1],
                "n_obs": int(m.nobs),
                "r2": m.rsquared,
                "adj_r2": m.rsquared_adj
            })

        print(f"\nClustered robustness model: {model_name}")
        print(m.summary())

    except Exception as e:
        print(f"Clustered model failed for {model_name}: {e}")

cluster_df = pd.DataFrame(cluster_tables)
cluster_df.to_csv(
    os.path.join(CSV_DIR, "cluster_robust_confidence_disparity_report_models.csv"),
    index=False
)

print("\nAnalysis complete.")
print(f"Outputs saved to: {OUT_DIR}")

## 14. Spatial Localization Hysteresis

Symmetric bias-versus-disparity hysteresis analysis, compatibility updates, and exact split-half reliability.

In [ ]:
# %% SPATIAL LOCALIZATION HYSTERESIS — SYMMETRIC BIAS VS DISPARITY ANALYSIS
# ============================================================
# Goal:
# Test whether the same A-V disparity produces different localization bias
# depending on sweep history.
#
# Main metric:
#   Localization hysteresis = Converge NormalizedVisualBias - Diverge NormalizedVisualBias
#
# Interpretation:
#   Positive value:
#       stronger visual capture during converge than diverge at the same disparity
#
#   Negative value:
#       stronger visual capture during diverge than converge at the same disparity
#
# This avoids asymmetric unbind/rebind threshold definitions.
# ============================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp, ttest_rel, t as t_dist

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

SPATIAL_ANALYSES_ROOT = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses"
OUT_DIR = os.path.join(SPATIAL_ANALYSES_ROOT, "07_spatial_hysteresis")
os.makedirs(OUT_DIR, exist_ok=True)

RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0
TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1.0 / RESAMPLE_HZ, 1.0 / RESAMPLE_HZ)

EXCLUDE_PRACTICE = True

WINDOW_SEC = 2.0
STEP_SEC = 0.25
MIN_VALID_SAMPLES = 8

# Use only post-change phase of ramp, where disparity is changing.
RAMP_CHANGE_TIME_SEC = 12.0

# Disparity bins for matched converge/diverge comparison
BIN_WIDTH_DEG = 2.0
MIN_DISPARITY_DEG = 5.0
MAX_DISPARITY_DEG = 35.0

# Optional: focus summary index on theoretically useful bins
SUMMARY_BIN_LOW = 5.0
SUMMARY_BIN_HIGH = 35.0

MIN_ABS_DISPARITY_FOR_CAPTURE = 0.01
FIG_DPI = 180

# =========================
# HELPER FUNCTIONS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else Path(filepath).stem

def resample_trial(df_trial, time_grid):
    needed = ["t", "mouse_x", "blob_x", "audio_x"]
    for c in needed:
        if c not in df_trial.columns:
            raise ValueError(f"Missing required column: {c}")

    d = df_trial.copy()
    d = d.sort_values("t")
    d = d[np.isfinite(d["t"])]
    d = d.drop_duplicates(subset="t", keep="first")

    if len(d) < 5:
        return None

    t_original = d["t"].values.astype(float)
    t_original = t_original - np.nanmin(t_original)

    if np.nanmax(t_original) < 1.0:
        return None

    def _interp(col):
        y = d[col].values.astype(float)
        ok = np.isfinite(t_original) & np.isfinite(y)

        if ok.sum() < 3:
            return np.full_like(time_grid, np.nan, dtype=float)

        return np.interp(
            time_grid,
            t_original[ok],
            y[ok],
            left=np.nan,
            right=np.nan
        )

    return {
        "t_grid": time_grid.copy(),
        "mouse_x": _interp("mouse_x"),
        "blob_x": _interp("blob_x"),
        "audio_x": _interp("audio_x"),
    }

def sliding_window_indices(t_grid, window_sec, step_sec):
    starts = np.arange(t_grid[0], t_grid[-1] - window_sec + 1e-9, step_sec)
    windows = []

    for start in starts:
        end = start + window_sec
        idx = np.where((t_grid >= start) & (t_grid < end))[0]

        if len(idx) >= MIN_VALID_SAMPLES:
            windows.append((start, end, idx))

    return windows

def normalized_visual_bias(response, visual, audio, min_abs_disparity=0.01):
    """
    Normalized visual bias / visual capture:
        (response - audio) / (visual - audio)

    0   = response at auditory location
    0.5 = midpoint
    1   = response at visual location
    """
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    denom = visual - audio

    ok = (
        np.isfinite(response)
        & np.isfinite(visual)
        & np.isfinite(audio)
        & (np.abs(denom) > min_abs_disparity)
    )

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean((response[ok] - audio[ok]) / denom[ok]))

def visual_directed_bias_deg(response, visual, audio, min_abs_disparity=0.01):
    """
    Bias toward vision in degrees.
    Positive = response shifted toward visual stimulus.
    """
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    disparity_signed = visual - audio
    visual_direction = np.sign(disparity_signed)

    ok = (
        np.isfinite(response)
        & np.isfinite(visual)
        & np.isfinite(audio)
        & np.isfinite(visual_direction)
        & (np.abs(disparity_signed) > min_abs_disparity)
    )

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean((response[ok] - audio[ok]) * visual_direction[ok]))

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)

    return m, m - crit * se, m + crit * se, n

def sem(vals):
    vals = pd.Series(vals).dropna().astype(float).values

    if len(vals) < 2:
        return np.nan

    return np.std(vals, ddof=1) / np.sqrt(len(vals))

def print_one_sample(label, vals, popmean=0.0, unit=""):
    vals = pd.Series(vals).dropna().astype(float).values

    if len(vals) < 2:
        print(f"{label}: not enough data")
        return

    m, lo, hi, n = mean_ci(vals)
    res = ttest_1samp(vals, popmean)
    p_txt = "< .001" if res.pvalue < .001 else f"= {res.pvalue:.4f}"

    print(f"\n{label}")
    print("-" * len(label))
    print(f"n = {n}")
    print(f"Mean = {m:.4f}{unit}")
    print(f"95% CI = [{lo:.4f}, {hi:.4f}]{unit}")
    print(f"t({n-1}) = {res.statistic:.4f}, p {p_txt}")

# =========================
# STEP 1 — LOAD AND WINDOW RAMP TRIALS
# =========================
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in {DATA_DIR}")

window_rows = []

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    df = pd.read_csv(fpath)

    required_cols = [
        "trialIndex", "trialType", "condLabel", "isPractice",
        "t", "mouse_x", "blob_x", "audio_x"
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"Skipping {subj}: missing {missing}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    df = df[
        (df["trialType"].astype(str) == "incongruent_sweep")
        & (
            df["condLabel"].astype(str).str.lower().str.contains("diverge")
            | df["condLabel"].astype(str).str.lower().str.contains("converge")
        )
    ].copy()

    if len(df) == 0:
        continue

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"],
        sort=False
    ):
        cond_lower = str(cond_label).lower()

        if "diverge" in cond_lower:
            sweep_direction = "Diverge"
        elif "converge" in cond_lower:
            sweep_direction = "Converge"
        else:
            continue

        resampled = resample_trial(df_trial, TIME_GRID)

        if resampled is None:
            continue

        t = np.asarray(resampled["t_grid"], dtype=float)
        mouse_x = np.asarray(resampled["mouse_x"], dtype=float)
        blob_x = np.asarray(resampled["blob_x"], dtype=float)
        audio_x = np.asarray(resampled["audio_x"], dtype=float)

        disparity_abs = np.abs(blob_x - audio_x)

        windows = sliding_window_indices(t, WINDOW_SEC, STEP_SEC)

        for start, end, inds in windows:
            center = (start + end) / 2

            # Focus on post-change phase only
            if center < RAMP_CHANGE_TIME_SEC:
                continue

            mean_disp = np.nanmean(disparity_abs[inds])

            norm_bias = normalized_visual_bias(
                mouse_x[inds],
                blob_x[inds],
                audio_x[inds],
                min_abs_disparity=MIN_ABS_DISPARITY_FOR_CAPTURE
            )

            visual_bias_deg = visual_directed_bias_deg(
                mouse_x[inds],
                blob_x[inds],
                audio_x[inds],
                min_abs_disparity=MIN_ABS_DISPARITY_FOR_CAPTURE
            )

            window_rows.append({
                "subject": subj,
                "source_file": Path(fpath).name,
                "trialIndex": trial_index,
                "trialType": trial_type,
                "condLabel": cond_label,
                "SweepDirection": sweep_direction,
                "window_start_sec": start,
                "window_end_sec": end,
                "window_center_sec": center,
                "mean_disparity_window": mean_disp,
                "NormalizedVisualBias_window": norm_bias,
                "VisualDirectedBias_deg_window": visual_bias_deg,
            })

window_df = pd.DataFrame(window_rows)

if len(window_df) == 0:
    raise RuntimeError("No valid diverge/converge ramp windows found.")

window_csv = os.path.join(OUT_DIR, "spatial_hysteresis_window_level_bias_by_disparity.csv")
window_df.to_csv(window_csv, index=False)

print(f"Saved window-level hysteresis input: {window_csv}")
print(f"Subjects: {window_df['subject'].nunique()}")
print(f"Windows: {len(window_df)}")

# =========================
# STEP 2 — BIN BY DISPARITY
# =========================
bins = np.arange(MIN_DISPARITY_DEG, MAX_DISPARITY_DEG + BIN_WIDTH_DEG, BIN_WIDTH_DEG)

window_df = window_df[
    np.isfinite(window_df["mean_disparity_window"])
    & np.isfinite(window_df["NormalizedVisualBias_window"])
].copy()

window_df = window_df[
    (window_df["mean_disparity_window"] >= MIN_DISPARITY_DEG)
    & (window_df["mean_disparity_window"] <= MAX_DISPARITY_DEG)
].copy()

window_df["disparity_bin"] = pd.cut(
    window_df["mean_disparity_window"],
    bins=bins,
    include_lowest=True
)

window_df["disparity_bin_center"] = window_df["disparity_bin"].apply(
    lambda x: x.mid if pd.notna(x) else np.nan
).astype(float)

# Subject × bin × direction mean
subject_bin_summary = (
    window_df
    .groupby(["subject", "SweepDirection", "disparity_bin_center"], observed=True)
    .agg(
        mean_disparity=("mean_disparity_window", "mean"),
        mean_NormalizedVisualBias=("NormalizedVisualBias_window", "mean"),
        mean_VisualDirectedBias_deg=("VisualDirectedBias_deg_window", "mean"),
        n_windows=("NormalizedVisualBias_window", "count"),
    )
    .reset_index()
)

subject_bin_csv = os.path.join(OUT_DIR, "subject_bin_direction_bias_summary.csv")
subject_bin_summary.to_csv(subject_bin_csv, index=False)

# =========================
# STEP 3 — COMPUTE SUBJECT-LEVEL HYSTERESIS INDEX
# =========================
pivot_capture = subject_bin_summary.pivot_table(
    index=["subject", "disparity_bin_center"],
    columns="SweepDirection",
    values="mean_NormalizedVisualBias"
).reset_index()

pivot_bias_deg = subject_bin_summary.pivot_table(
    index=["subject", "disparity_bin_center"],
    columns="SweepDirection",
    values="mean_VisualDirectedBias_deg"
).reset_index()

for df_piv in [pivot_capture, pivot_bias_deg]:
    if "Converge" not in df_piv.columns:
        df_piv["Converge"] = np.nan
    if "Diverge" not in df_piv.columns:
        df_piv["Diverge"] = np.nan

pivot_capture["CaptureHysteresis_ConvergeMinusDiverge"] = (
    pivot_capture["Converge"] - pivot_capture["Diverge"]
)

pivot_bias_deg["BiasDegHysteresis_ConvergeMinusDiverge"] = (
    pivot_bias_deg["Converge"] - pivot_bias_deg["Diverge"]
)

bin_hysteresis_df = pd.merge(
    pivot_capture[
        [
            "subject",
            "disparity_bin_center",
            "Diverge",
            "Converge",
            "CaptureHysteresis_ConvergeMinusDiverge",
        ]
    ].rename(
        columns={
            "Diverge": "Diverge_NormalizedVisualBias",
            "Converge": "Converge_NormalizedVisualBias",
        }
    ),
    pivot_bias_deg[
        [
            "subject",
            "disparity_bin_center",
            "Diverge",
            "Converge",
            "BiasDegHysteresis_ConvergeMinusDiverge",
        ]
    ].rename(
        columns={
            "Diverge": "Diverge_VisualDirectedBias_deg",
            "Converge": "Converge_VisualDirectedBias_deg",
        }
    ),
    on=["subject", "disparity_bin_center"],
    how="outer"
)

# Restrict subject-level summary to pre-specified bins, e.g., 10–30 deg
summary_bin_mask = (
    (bin_hysteresis_df["disparity_bin_center"] >= SUMMARY_BIN_LOW)
    & (bin_hysteresis_df["disparity_bin_center"] <= SUMMARY_BIN_HIGH)
)

subject_hysteresis_df = (
    bin_hysteresis_df[summary_bin_mask]
    .groupby("subject", as_index=False)
    .agg(
        n_matched_bins=("CaptureHysteresis_ConvergeMinusDiverge", lambda x: x.notna().sum()),
        mean_CaptureHysteresis=("CaptureHysteresis_ConvergeMinusDiverge", "mean"),
        mean_BiasDegHysteresis=("BiasDegHysteresis_ConvergeMinusDiverge", "mean"),
        mean_Diverge_Capture=("Diverge_NormalizedVisualBias", "mean"),
        mean_Converge_Capture=("Converge_NormalizedVisualBias", "mean"),
        mean_Diverge_BiasDeg=("Diverge_VisualDirectedBias_deg", "mean"),
        mean_Converge_BiasDeg=("Converge_VisualDirectedBias_deg", "mean"),
    )
)

# Keep only subjects with at least 2 matched bins in summary range
subject_hysteresis_df = subject_hysteresis_df[
    subject_hysteresis_df["n_matched_bins"] >= 2
].copy()

bin_hysteresis_csv = os.path.join(OUT_DIR, "bin_level_hysteresis_converge_minus_diverge.csv")
subject_hysteresis_csv = os.path.join(OUT_DIR, "subject_level_localization_hysteresis_index.csv")

bin_hysteresis_df.to_csv(bin_hysteresis_csv, index=False)
subject_hysteresis_df.to_csv(subject_hysteresis_csv, index=False)

print(f"Saved bin-level hysteresis:     {bin_hysteresis_csv}")
print(f"Saved subject hysteresis index: {subject_hysteresis_csv}")

# =========================
# STEP 4 — GROUP STATS
# =========================
print("\n============================================================")
print("SPATIAL LOCALIZATION HYSTERESIS: CONVERGE − DIVERGE")
print("============================================================")
print(f"Summary disparity range: {SUMMARY_BIN_LOW:.1f}–{SUMMARY_BIN_HIGH:.1f} deg")
print(f"Valid subjects: n = {len(subject_hysteresis_df)}")

print_one_sample(
    "Subject localization hysteresis index: normalized visual bias",
    subject_hysteresis_df["mean_CaptureHysteresis"],
    popmean=0.0,
    unit=""
)

print_one_sample(
    "Subject localization hysteresis index: visual-directed bias in degrees",
    subject_hysteresis_df["mean_BiasDegHysteresis"],
    popmean=0.0,
    unit=" deg"
)

# Paired test for average capture across matched bins
paired_df = subject_hysteresis_df.dropna(
    subset=["mean_Converge_Capture", "mean_Diverge_Capture"]
).copy()

if len(paired_df) >= 2:
    paired = ttest_rel(
        paired_df["mean_Converge_Capture"],
        paired_df["mean_Diverge_Capture"]
    )
    p_txt = "< .001" if paired.pvalue < .001 else f"= {paired.pvalue:.4f}"

    print("\nPaired test: mean Converge capture vs mean Diverge capture")
    print("---------------------------------------------------------")
    print(f"n = {len(paired_df)}")
    print(f"Converge mean = {paired_df['mean_Converge_Capture'].mean():.4f}")
    print(f"Diverge mean  = {paired_df['mean_Diverge_Capture'].mean():.4f}")
    print(f"t({len(paired_df)-1}) = {paired.statistic:.4f}, p {p_txt}")

# Save group stats
group_rows = []

for metric, popmean, unit, label in [
    ("mean_CaptureHysteresis", 0.0, "", "Capture hysteresis: Converge - Diverge"),
    ("mean_BiasDegHysteresis", 0.0, "deg", "Bias hysteresis deg: Converge - Diverge"),
]:
    vals = subject_hysteresis_df[metric].dropna().astype(float).values

    if len(vals) >= 2:
        m, lo, hi, n = mean_ci(vals)
        test = ttest_1samp(vals, popmean=popmean)
        group_rows.append({
            "label": label,
            "metric": metric,
            "unit": unit,
            "n": n,
            "mean": m,
            "ci_low": lo,
            "ci_high": hi,
            "t": test.statistic,
            "df": n - 1,
            "p": test.pvalue,
            "summary_bin_low": SUMMARY_BIN_LOW,
            "summary_bin_high": SUMMARY_BIN_HIGH,
        })

group_stats_df = pd.DataFrame(group_rows)
group_stats_csv = os.path.join(OUT_DIR, "group_localization_hysteresis_stats.csv")
group_stats_df.to_csv(group_stats_csv, index=False)

# =========================
# STEP 5 — GROUP HYSTERESIS CURVE
# =========================
group_curve = (
    subject_bin_summary
    .groupby(["SweepDirection", "disparity_bin_center"], observed=True)
    .agg(
        mean_capture=("mean_NormalizedVisualBias", "mean"),
        sem_capture=("mean_NormalizedVisualBias", sem),
        mean_bias_deg=("mean_VisualDirectedBias_deg", "mean"),
        sem_bias_deg=("mean_VisualDirectedBias_deg", sem),
        n_subject_bins=("subject", "nunique"),
    )
    .reset_index()
)

group_curve_csv = os.path.join(OUT_DIR, "group_hysteresis_curve_by_disparity_bin.csv")
group_curve.to_csv(group_curve_csv, index=False)

# Plot normalized visual bias curve
fig, ax = plt.subplots(figsize=(7.6, 5.8))

for direction in ["Diverge", "Converge"]:
    sub = group_curve[group_curve["SweepDirection"] == direction].copy()
    sub = sub.dropna(subset=["disparity_bin_center", "mean_capture"])

    ax.plot(
        sub["disparity_bin_center"],
        sub["mean_capture"],
        marker="o",
        linewidth=2,
        label=direction
    )

    ax.fill_between(
        sub["disparity_bin_center"],
        sub["mean_capture"] - sub["sem_capture"],
        sub["mean_capture"] + sub["sem_capture"],
        alpha=0.2
    )

ax.axhline(0.5, linestyle="--", linewidth=1.0, alpha=0.8, label="Midpoint")
ax.axhline(1.0, linestyle=":", linewidth=1.0, alpha=0.8, label="Visual")
ax.axhline(0.0, linestyle=":", linewidth=1.0, alpha=0.8, label="Auditory")

ax.axvspan(
    SUMMARY_BIN_LOW,
    SUMMARY_BIN_HIGH,
    alpha=0.08,
    label=f"summary range {SUMMARY_BIN_LOW:.0f}–{SUMMARY_BIN_HIGH:.0f}°"
)

ax.set_xlabel("A-V disparity (deg)")
ax.set_ylabel("Normalized visual bias")
ax.set_title("Spatial localization hysteresis: bias vs disparity")
ax.set_ylim(-0.25, 1.35)
ax.legend()

curve_capture_path = os.path.join(OUT_DIR, "hysteresis_curve_normalized_visual_bias.png")
fig.tight_layout()
fig.savefig(curve_capture_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# Plot visual-directed bias in degrees
fig, ax = plt.subplots(figsize=(7.6, 5.8))

for direction in ["Diverge", "Converge"]:
    sub = group_curve[group_curve["SweepDirection"] == direction].copy()
    sub = sub.dropna(subset=["disparity_bin_center", "mean_bias_deg"])

    ax.plot(
        sub["disparity_bin_center"],
        sub["mean_bias_deg"],
        marker="o",
        linewidth=2,
        label=direction
    )

    ax.fill_between(
        sub["disparity_bin_center"],
        sub["mean_bias_deg"] - sub["sem_bias_deg"],
        sub["mean_bias_deg"] + sub["sem_bias_deg"],
        alpha=0.2
    )

ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.8)
ax.axvspan(
    SUMMARY_BIN_LOW,
    SUMMARY_BIN_HIGH,
    alpha=0.08,
    label=f"summary range {SUMMARY_BIN_LOW:.0f}–{SUMMARY_BIN_HIGH:.0f}°"
)

ax.set_xlabel("A-V disparity (deg)")
ax.set_ylabel("Visual-directed bias (deg)")
ax.set_title("Spatial localization hysteresis: visual-directed bias vs disparity")
ax.legend()

curve_bias_path = os.path.join(OUT_DIR, "hysteresis_curve_visual_directed_bias_deg.png")
fig.tight_layout()
fig.savefig(curve_bias_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# Plot subject hysteresis index
fig, ax = plt.subplots(figsize=(6.4, 5.4))

plot_df = subject_hysteresis_df.sort_values("mean_CaptureHysteresis").reset_index(drop=True)

ax.axhline(0, linestyle="--", linewidth=1.0)
ax.scatter(
    np.arange(len(plot_df)),
    plot_df["mean_CaptureHysteresis"],
    alpha=0.8
)

ax.set_xlabel("Subject sorted by hysteresis index")
ax.set_ylabel("Mean Converge − Diverge capture")
ax.set_title("Subject-level localization hysteresis index")

subject_index_path = os.path.join(OUT_DIR, "subject_localization_hysteresis_index.png")
fig.tight_layout()
fig.savefig(subject_index_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

print("\n============================================================")
print("LOCALIZATION HYSTERESIS ANALYSIS COMPLETE")
print("============================================================")
print(f"Window-level file:        {window_csv}")
print(f"Subject-bin summary:      {subject_bin_csv}")
print(f"Bin-level hysteresis:     {bin_hysteresis_csv}")
print(f"Subject hysteresis index: {subject_hysteresis_csv}")
print(f"Group stats:              {group_stats_csv}")
print(f"Group curve data:         {group_curve_csv}")
print(f"Capture curve plot:       {curve_capture_path}")
print(f"Bias-deg curve plot:      {curve_bias_path}")
print(f"Subject index plot:       {subject_index_path}")

print("\nInterpretation guide:")
print("- mean_CaptureHysteresis = mean normalized visual bias during converge minus diverge at matched disparity bins.")
print("- Positive values mean stronger visual capture during converge than diverge at the same physical disparity.")
print("- Negative values mean stronger visual capture during diverge than converge at the same physical disparity.")
print("- This is the clean spatial-localization hysteresis metric because both directions use the same dependent variable.")

In [ ]:
# %% SL RELIABILITY COMPATIBILITY FIX
# ============================================================
# This cell replaces the SL half-hysteresis function with a version
# compatible with both older and newer NumPy releases.
#
# NumPy compatibility:
#   newer NumPy: np.trapezoid
#   older NumPy: np.trapz
#
# After running this cell, rerun the reliability code beginning at:
#
#   # A. ODD–EVEN SPLIT
#
# You do not need to recreate sl_trial_window_df.
# ============================================================

import numpy as np
import pandas as pd


def sl_trapezoidal_integral(y, x):
    """
    NumPy-version-compatible trapezoidal integration.

    Uses np.trapezoid when available and falls back to np.trapz
    for older NumPy versions.
    """
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)

    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x=x))

    return float(np.trapz(y, x=x))


def sl_compute_half_hysteresis_metrics(
    source_window_df,
    assignments,
):
    """
    Recompute SL localization hysteresis independently in each trial half.

    Pipeline:
      1. Merge whole-trial half assignments with the lag-corrected
         participant × trial × window table.
      2. Bin windows by instantaneous absolute A–V disparity.
      3. Average windows within trial × disparity bin.
      4. Average whole trials within participant × half × direction × bin.
      5. Match converge and diverge curves from 5° through 35°.
      6. Compute converge-minus-diverge normalized visual capture.
      7. Derive:
           - mean CaptureHysteresis, the primary manuscript index;
           - signed hysteresis area;
           - area-normalized hysteresis index.
    """
    if assignments is None or assignments.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
        )

    required_source_columns = {
        "subject",
        "trialIndex",
        "analysis_group",
        "abs_disparity_deg",
        "normalized_visual_capture",
        "window_center_s",
    }

    missing_source_columns = (
        required_source_columns
        - set(source_window_df.columns)
    )

    if missing_source_columns:
        raise ValueError(
            "source_window_df is missing required columns:\n"
            f"{sorted(missing_source_columns)}"
        )

    required_assignment_columns = {
        "subject",
        "trialIndex",
        "analysis_group",
        "half",
    }

    missing_assignment_columns = (
        required_assignment_columns
        - set(assignments.columns)
    )

    if missing_assignment_columns:
        raise ValueError(
            "assignments is missing required columns:\n"
            f"{sorted(missing_assignment_columns)}"
        )

    working = source_window_df.merge(
        assignments[
            [
                "subject",
                "trialIndex",
                "analysis_group",
                "half",
            ]
        ],
        on=[
            "subject",
            "trialIndex",
            "analysis_group",
        ],
        how="inner",
        validate="many_to_one",
    )

    working["abs_disparity_deg"] = pd.to_numeric(
        working["abs_disparity_deg"],
        errors="coerce",
    )

    working["normalized_visual_capture"] = (
        pd.to_numeric(
            working["normalized_visual_capture"],
            errors="coerce",
        )
    )

    working["window_center_s"] = pd.to_numeric(
        working["window_center_s"],
        errors="coerce",
    )

    working = (
        working
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "subject",
                "trialIndex",
                "analysis_group",
                "half",
                "abs_disparity_deg",
                "normalized_visual_capture",
                "window_center_s",
            ]
        )
        .copy()
    )

    if working.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
        )

    disparity_edges = np.arange(
        float(SL_MIN_MATCHED_DISPARITY_DEG),
        float(SL_MAX_MATCHED_DISPARITY_DEG)
        + float(SL_DISPARITY_BIN_WIDTH_DEG)
        + 1e-9,
        float(SL_DISPARITY_BIN_WIDTH_DEG),
    )

    if len(disparity_edges) < 2:
        raise ValueError(
            "The SL disparity-bin settings produced fewer than two edges."
        )

    disparity_centers = (
        disparity_edges[:-1]
        + disparity_edges[1:]
    ) / 2.0

    working["disparity_bin"] = pd.cut(
        working["abs_disparity_deg"],
        bins=disparity_edges,
        labels=disparity_centers,
        include_lowest=True,
        right=False,
    )

    working = working.dropna(
        subset=["disparity_bin"]
    ).copy()

    working["disparity_bin_center"] = pd.to_numeric(
        working["disparity_bin"].astype(str),
        errors="coerce",
    )

    working = working.dropna(
        subset=["disparity_bin_center"]
    ).copy()

    # Average overlapping windows within each whole trial and disparity bin.
    trial_binned = (
        working
        .groupby(
            [
                "subject",
                "half",
                "trialIndex",
                "analysis_group",
                "disparity_bin_center",
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            normalized_visual_capture=(
                "normalized_visual_capture",
                "mean",
            ),
            n_windows=(
                "window_center_s",
                "count",
            ),
        )
    )

    # Give every whole trial equal weight within each half.
    subject_binned = (
        trial_binned
        .groupby(
            [
                "subject",
                "half",
                "analysis_group",
                "disparity_bin_center",
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            normalized_visual_capture=(
                "normalized_visual_capture",
                "mean",
            ),
            n_trials=(
                "trialIndex",
                "nunique",
            ),
        )
    )

    metric_rows = []
    curve_rows = []

    for (
        subject,
        half,
    ), group in subject_binned.groupby(
        ["subject", "half"],
        sort=False,
    ):
        wide = group.pivot_table(
            index="disparity_bin_center",
            columns="analysis_group",
            values="normalized_visual_capture",
            aggfunc="mean",
        )

        required_directions = {
            "sweep_diverge",
            "sweep_converge",
        }

        if not required_directions.issubset(
            set(wide.columns)
        ):
            continue

        wide = (
            wide
            .replace([np.inf, -np.inf], np.nan)
            .dropna(
                subset=[
                    "sweep_diverge",
                    "sweep_converge",
                ]
            )
            .sort_index()
        )

        wide = wide[
            (
                wide.index
                >= SL_MIN_MATCHED_DISPARITY_DEG
            )
            & (
                wide.index
                <= SL_MAX_MATCHED_DISPARITY_DEG
            )
        ]

        if len(wide) < 2:
            continue

        disparity = wide.index.to_numpy(
            dtype=float
        )

        # Primary manuscript sign:
        # converge minus diverge normalized visual capture.
        difference = (
            wide["sweep_converge"].to_numpy(
                dtype=float
            )
            - wide["sweep_diverge"].to_numpy(
                dtype=float
            )
        )

        valid = (
            np.isfinite(disparity)
            & np.isfinite(difference)
        )

        disparity = disparity[valid]
        difference = difference[valid]

        if len(disparity) < 2:
            continue

        sort_order = np.argsort(disparity)

        disparity = disparity[sort_order]
        difference = difference[sort_order]

        primary_index = float(
            np.nanmean(difference)
        )

        signed_area = sl_trapezoidal_integral(
            difference,
            disparity,
        )

        disparity_range = float(
            np.nanmax(disparity)
            - np.nanmin(disparity)
        )

        area_normalized_index = (
            signed_area / disparity_range
            if disparity_range > 0
            else np.nan
        )

        metric_rows.append(
            {
                "subject": subject,
                "half": int(half),
                "mean_CaptureHysteresis":
                    primary_index,
                "capture_hysteresis_signed_area":
                    signed_area,
                "capture_hysteresis_area_normalized":
                    area_normalized_index,
                "n_common_bins":
                    int(len(disparity)),
                "minimum_common_disparity_deg":
                    float(np.nanmin(disparity)),
                "maximum_common_disparity_deg":
                    float(np.nanmax(disparity)),
                "common_disparity_range_deg":
                    disparity_range,
            }
        )

        for (
            disparity_value,
            difference_value,
        ) in zip(
            disparity,
            difference,
        ):
            curve_rows.append(
                {
                    "subject":
                        subject,
                    "half":
                        int(half),
                    "disparity_bin_center":
                        float(disparity_value),
                    "capture_converge_minus_diverge":
                        float(difference_value),
                }
            )

    metrics = pd.DataFrame(metric_rows)
    curves = pd.DataFrame(curve_rows)

    return (
        metrics,
        curves,
        trial_binned,
        subject_binned,
    )


print("SL compatibility fix installed successfully.")
print(f"NumPy version: {np.__version__}")
print(
    "Integration function: "
    + (
        "np.trapezoid"
        if hasattr(np, "trapezoid")
        else "np.trapz"
    )
)
print(
    "\nNow rerun the SL reliability code beginning at:\n"
    "# A. ODD–EVEN SPLIT"
)
# ============================================================
# A. ODD–EVEN SPLIT
# ============================================================

sl_odd_even_assignments = (
    sl_create_trial_assignment(
        sl_trial_window_df,
        split_type="odd_even",
        split_number=0,
        seed=SL_RELIABILITY_RANDOM_SEED,
    )
)

(
    sl_odd_even_metrics_long,
    sl_odd_even_difference_curves,
    sl_odd_even_trial_binned,
    sl_odd_even_subject_binned,
) = sl_compute_half_hysteresis_metrics(
    sl_trial_window_df,
    sl_odd_even_assignments,
)

sl_odd_even_metrics_wide = (
    sl_widen_half_metrics(
        sl_odd_even_metrics_long
    )
)

sl_odd_even_reliability = (
    sl_summarize_split_reliability(
        sl_odd_even_metrics_wide,
        split_type="odd_even",
        split_number=0,
        run_bootstrap=True,
    )
)


# ============================================================
# B. REPEATED RANDOM STRATIFIED SPLITS
# ============================================================

sl_random_reliability_rows = []
sl_random_participant_rows = []

for split_number in range(
    SL_N_RANDOM_SPLITS
):
    assignments = sl_create_trial_assignment(
        sl_trial_window_df,
        split_type="random",
        split_number=split_number,
        seed=SL_RELIABILITY_RANDOM_SEED,
    )

    split_output = (
        sl_compute_half_hysteresis_metrics(
            sl_trial_window_df,
            assignments,
        )
    )

    if isinstance(split_output, pd.DataFrame):
        continue

    split_metrics_long = split_output[0]

    split_metrics_wide = (
        sl_widen_half_metrics(
            split_metrics_long
        )
    )

    if split_metrics_wide.empty:
        continue

    split_reliability = (
        sl_summarize_split_reliability(
            split_metrics_wide,
            split_type="random",
            split_number=split_number + 1,
            run_bootstrap=False,
        )
    )

    if not split_reliability.empty:
        sl_random_reliability_rows.append(
            split_reliability
        )

    split_metrics_wide = (
        split_metrics_wide.copy()
    )

    split_metrics_wide["split_number"] = (
        split_number + 1
    )

    sl_random_participant_rows.append(
        split_metrics_wide
    )

    if (
        (split_number + 1) % 100 == 0
        or split_number == 0
    ):
        print(
            f"Completed SL random split "
            f"{split_number + 1}/"
            f"{SL_N_RANDOM_SPLITS}"
        )


if len(sl_random_reliability_rows) > 0:
    sl_random_split_reliability = pd.concat(
        sl_random_reliability_rows,
        ignore_index=True,
    )
else:
    sl_random_split_reliability = (
        pd.DataFrame()
    )

if len(sl_random_participant_rows) > 0:
    sl_random_split_participant_estimates = (
        pd.concat(
            sl_random_participant_rows,
            ignore_index=True,
        )
    )
else:
    sl_random_split_participant_estimates = (
        pd.DataFrame()
    )


# ============================================================
# SUMMARIZE RANDOM-SPLIT DISTRIBUTIONS
# ============================================================

sl_random_summary_rows = []

if not sl_random_split_reliability.empty:
    for metric, group in (
        sl_random_split_reliability.groupby(
            "metric"
        )
    ):
        for reliability_column in [
            "pearson_r_half",
            "pearson_spearman_brown",
            "spearman_rho_half",
            "spearman_brown_from_spearman",
        ]:
            values = pd.to_numeric(
                group[reliability_column],
                errors="coerce",
            ).dropna()

            if len(values) == 0:
                continue

            sl_random_summary_rows.append(
                {
                    "metric": metric,
                    "reliability_statistic":
                        reliability_column,
                    "n_successful_splits":
                        int(len(values)),
                    "mean": float(values.mean()),
                    "median": float(values.median()),
                    "sd": float(
                        values.std(ddof=1)
                    ),
                    "percentile_2p5":
                        float(
                            np.percentile(
                                values,
                                2.5,
                            )
                        ),
                    "percentile_25":
                        float(
                            np.percentile(
                                values,
                                25,
                            )
                        ),
                    "percentile_75":
                        float(
                            np.percentile(
                                values,
                                75,
                            )
                        ),
                    "percentile_97p5":
                        float(
                            np.percentile(
                                values,
                                97.5,
                            )
                        ),
                    "proportion_positive":
                        float(
                            (values > 0).mean()
                        ),
                }
            )

sl_random_split_summary = pd.DataFrame(
    sl_random_summary_rows
)


# ============================================================
# SAVE OUTPUTS
# ============================================================

sl_odd_even_assignments.to_csv(
    SL_RELIABILITY_DIR
    / "sl_odd_even_trial_assignments.csv",
    index=False,
)

sl_odd_even_metrics_long.to_csv(
    SL_RELIABILITY_DIR
    / "sl_odd_even_half_hysteresis_metrics_long.csv",
    index=False,
)

sl_odd_even_metrics_wide.to_csv(
    SL_RELIABILITY_DIR
    / "sl_odd_even_half_hysteresis_metrics_wide.csv",
    index=False,
)

sl_odd_even_difference_curves.to_csv(
    SL_RELIABILITY_DIR
    / "sl_odd_even_half_difference_curves.csv",
    index=False,
)

sl_odd_even_reliability.to_csv(
    SL_RELIABILITY_DIR
    / "sl_odd_even_reliability_summary.csv",
    index=False,
)

sl_random_split_reliability.to_csv(
    SL_RELIABILITY_DIR
    / "sl_random_split_reliability_all_splits.csv",
    index=False,
)

sl_random_split_participant_estimates.to_csv(
    SL_RELIABILITY_DIR
    / "sl_random_split_participant_estimates.csv",
    index=False,
)

sl_random_split_summary.to_csv(
    SL_RELIABILITY_DIR
    / "sl_random_split_reliability_distribution_summary.csv",
    index=False,
)


# ============================================================
# OPTIONAL VALIDATION AGAINST THE EXISTING FULL-DATA INDEX
# ============================================================

existing_hysteresis_path = (
    SL_HYSTERESIS_BASE_DIR
    / "spatial_localization_hysteresis_bias_vs_disparity"
    / "subject_level_localization_hysteresis_index.csv"
)

sl_full_reconstructed_rows = []

# Assign every trial to one artificial "full" half.
sl_full_assignments = (
    sl_trial_window_df[
        [
            "subject",
            "trialIndex",
            "analysis_group",
        ]
    ]
    .drop_duplicates()
    .copy()
)

sl_full_assignments["half"] = 1

sl_full_output = (
    sl_compute_half_hysteresis_metrics(
        sl_trial_window_df,
        sl_full_assignments,
    )
)

if not isinstance(sl_full_output, pd.DataFrame):
    sl_full_reconstructed = (
        sl_full_output[0]
        .rename(
            columns={
                "mean_CaptureHysteresis":
                    "mean_CaptureHysteresis_reconstructed"
            }
        )
        .drop(columns=["half"])
    )

    sl_full_reconstructed.to_csv(
        SL_RELIABILITY_DIR
        / "sl_full_data_reconstructed_hysteresis_index.csv",
        index=False,
    )

    if existing_hysteresis_path.exists():
        existing_hysteresis = pd.read_csv(
            existing_hysteresis_path
        )

        if (
            "subject"
            in existing_hysteresis.columns
            and "mean_CaptureHysteresis"
            in existing_hysteresis.columns
        ):
            existing_hysteresis["subject"] = (
                existing_hysteresis["subject"]
                .apply(sl_normalize_subject)
            )

            validation = (
                existing_hysteresis[
                    [
                        "subject",
                        "mean_CaptureHysteresis",
                    ]
                ]
                .merge(
                    sl_full_reconstructed[
                        [
                            "subject",
                            "mean_CaptureHysteresis_reconstructed",
                        ]
                    ],
                    on="subject",
                    how="inner",
                )
                .dropna()
            )

            validation[
                "difference_reconstructed_minus_existing"
            ] = (
                validation[
                    "mean_CaptureHysteresis_reconstructed"
                ]
                - validation[
                    "mean_CaptureHysteresis"
                ]
            )

            validation.to_csv(
                SL_RELIABILITY_DIR
                / "sl_reconstructed_vs_existing_hysteresis_validation.csv",
                index=False,
            )

            validation_r, validation_p, validation_n = (
                sl_safe_pearson(
                    validation[
                        "mean_CaptureHysteresis"
                    ],
                    validation[
                        "mean_CaptureHysteresis_reconstructed"
                    ],
                )
            )

            print("\nFull-data reconstruction check")
            print("------------------------------")
            print(
                f"Matched participants: "
                f"{validation_n}"
            )
            print(
                "Existing versus reconstructed index: "
                f"r = {validation_r:.4f}, "
                f"p = {validation_p:.6f}"
            )
            print(
                "Mean absolute discrepancy: "
                f"{validation['difference_reconstructed_minus_existing'].abs().mean():.6f}"
            )
            print(
                "Maximum absolute discrepancy: "
                f"{validation['difference_reconstructed_minus_existing'].abs().max():.6f}"
            )


# ============================================================
# FIGURES
# ============================================================

for metric in [
    "mean_CaptureHysteresis",
    "capture_hysteresis_signed_area",
    "capture_hysteresis_area_normalized",
]:
    half_1 = f"{metric}_half1"
    half_2 = f"{metric}_half2"

    if (
        half_1 not in sl_odd_even_metrics_wide.columns
        or half_2 not in sl_odd_even_metrics_wide.columns
    ):
        continue

    plot_df = sl_odd_even_metrics_wide[
        ["subject", half_1, half_2]
    ].dropna()

    if len(plot_df) < SL_MIN_PARTICIPANTS:
        continue

    pearson_r, pearson_p, n_value = (
        sl_safe_pearson(
            plot_df[half_1],
            plot_df[half_2],
        )
    )

    sb_value = sl_spearman_brown(
        pearson_r
    )

    x_values = plot_df[half_1].to_numpy(
        dtype=float
    )

    y_values = plot_df[half_2].to_numpy(
        dtype=float
    )

    plt.figure(figsize=(6.5, 6.0))

    plt.scatter(
        x_values,
        y_values,
        s=55,
        alpha=0.80,
    )

    if np.std(x_values) > 0:
        slope, intercept = np.polyfit(
            x_values,
            y_values,
            deg=1,
        )

        x_grid = np.linspace(
            x_values.min(),
            x_values.max(),
            200,
        )

        plt.plot(
            x_grid,
            intercept + slope * x_grid,
            linewidth=2,
        )

    minimum = np.nanmin(
        np.concatenate(
            [x_values, y_values]
        )
    )

    maximum = np.nanmax(
        np.concatenate(
            [x_values, y_values]
        )
    )

    plt.plot(
        [minimum, maximum],
        [minimum, maximum],
        linestyle="--",
        linewidth=1.2,
        alpha=0.60,
    )

    plt.xlabel(f"Odd trials: {metric}")
    plt.ylabel(f"Even trials: {metric}")

    plt.title(
        "SL hysteresis odd–even reliability\n"
        f"n = {n_value}, r = {pearson_r:.3f}, "
        f"Spearman–Brown = {sb_value:.3f}"
    )

    plt.tight_layout()

    plt.savefig(
        SL_RELIABILITY_FIG_DIR
        / f"sl_odd_even_{metric}.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()


if not sl_random_split_reliability.empty:
    for metric in (
        sl_random_split_reliability[
            "metric"
        ].unique()
    ):
        values = (
            sl_random_split_reliability.loc[
                sl_random_split_reliability[
                    "metric"
                ] == metric,
                "pearson_spearman_brown",
            ]
            .dropna()
        )

        if len(values) == 0:
            continue

        plt.figure(figsize=(7.0, 5.0))

        plt.hist(
            values,
            bins=35,
            edgecolor="black",
            alpha=0.80,
        )

        plt.axvline(
            values.median(),
            linestyle="--",
            linewidth=2,
            label=(
                "Median corrected reliability = "
                f"{values.median():.3f}"
            ),
        )

        plt.axvline(
            0,
            linewidth=1,
        )

        plt.xlabel(
            "Spearman–Brown-corrected Pearson reliability"
        )

        plt.ylabel(
            "Number of random splits"
        )

        plt.title(
            "SL repeated split-half reliability\n"
            f"{metric}"
        )

        plt.legend()
        plt.tight_layout()

        plt.savefig(
            SL_RELIABILITY_FIG_DIR
            / f"sl_random_split_distribution_{metric}.png",
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n")
print("=" * 78)
print("SL HYSTERESIS SPLIT-HALF RELIABILITY")
print("=" * 78)

print("\nOdd–even reliability:")
if sl_odd_even_reliability.empty:
    print(
        "No valid odd–even reliability estimates were obtained."
    )
else:
    print(
        sl_odd_even_reliability.to_string(
            index=False
        )
    )

print("\nRepeated random-split distribution summary:")
if sl_random_split_summary.empty:
    print(
        "No valid repeated random-split estimates were obtained."
    )
else:
    print(
        sl_random_split_summary.to_string(
            index=False
        )
    )

primary_half_columns = [
    "mean_CaptureHysteresis_half1",
    "mean_CaptureHysteresis_half2",
]

if all(
    column in sl_odd_even_metrics_wide.columns
    for column in primary_half_columns
):
    primary_n = (
        sl_odd_even_metrics_wide[
            primary_half_columns
        ]
        .dropna()
        .shape[0]
    )
else:
    primary_n = 0

print("\nParticipant inclusion:")
print(
    "Odd–even participants with complete primary index estimates: "
    f"{primary_n}"
)

print("\nOutputs saved to:")
print(SL_RELIABILITY_DIR)

print("\nPrimary files:")
print(
    SL_RELIABILITY_DIR
    / "sl_odd_even_reliability_summary.csv"
)
print(
    SL_RELIABILITY_DIR
    / "sl_random_split_reliability_distribution_summary.csv"
)
print(
    SL_RELIABILITY_DIR
    / "sl_random_split_reliability_all_splits.csv"
)

print("\nInterpretation:")
print(
    "- mean_CaptureHysteresis is the primary localization measure: "
    "mean converge-minus-diverge normalized capture across matched "
    "5°–35° disparities."
)
print(
    "- pearson_r_half is the uncorrected association between independent "
    "sets of sweep trials."
)
print(
    "- pearson_spearman_brown estimates the reliability expected for the "
    "full trial set."
)
print(
    "- The median across repeated random splits should be emphasized over "
    "a single odd-even partition."
)
print(
    "- Check the full-data reconstruction output before interpreting "
    "reliability. A very high correspondence with the existing saved "
    "hysteresis index confirms that the reconstructed pipeline matches "
    "the original analysis."
)

In [ ]:
# %% SPATIAL LOCALIZATION HYSTERESIS — EXACT SPLIT-HALF RELIABILITY
# =============================================================================
# PURPOSE
# =============================================================================
# Estimate split-half reliability of the exact spatial-localization hysteresis
# index used in the manuscript.
#
# This code deliberately reuses the original hysteresis analysis's window_df
# rather than reconstructing the response trajectories from trial_df.
#
# It preserves the original manuscript pipeline:
#
#   1. Original raw-response 2-s windows from window_df.
#   2. Original post-change window selection.
#   3. Original normalized visual-bias values.
#   4. Original 5°–35° disparity range.
#   5. Original 2° disparity bins.
#   6. Original pd.cut settings.
#   7. Original window-weighted averaging:
#
#          subject × direction × disparity bin
#
#      No additional trial-level averaging is introduced.
#
#   8. Original hysteresis definition:
#
#          Converge normalized visual bias
#          minus
#          Diverge normalized visual bias
#
#   9. Original participant summary:
#      mean difference across matched 5°–35° bins, with at least two
#      matched bins required.
#
# Whole trials are split separately within:
#
#      subject × sweep direction
#
# Reliability analyses:
#
#   A. Deterministic odd–even trial split
#   B. Repeated random stratified split halves
#   C. Spearman–Brown correction
#   D. Participant-bootstrap CI for the odd–even estimate
#   E. Exact full-data reconstruction validation against the original
#      subject_level_localization_hysteresis_index.csv
#
# IMPORTANT
# =============================================================================
# Run the original manuscript hysteresis analysis before this cell whenever
# possible. The code will use its in-memory window_df directly.
# =============================================================================

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =============================================================================
# USER SETTINGS
# =============================================================================

SL_RELIABILITY_RANDOM_SEED = 20260721

# Number of repeated random stratified split halves.
SL_N_RANDOM_SPLITS = 1000

# Number of participant-bootstrap samples for the odd-even CI.
SL_N_BOOTSTRAPS = 10000

# Minimum participant count required for a reliability correlation.
SL_MIN_PARTICIPANTS = 8

# A participant needs at least two trials in each direction so each half can
# contain at least one converge and one diverge trial.
SL_MIN_TRIALS_PER_DIRECTION = 2

# These settings must match the original hysteresis analysis exactly.
SL_BIN_WIDTH_DEG = 2.0
SL_MIN_DISPARITY_DEG = 5.0
SL_MAX_DISPARITY_DEG = 35.0
SL_SUMMARY_BIN_LOW = 5.0
SL_SUMMARY_BIN_HIGH = 35.0

# Require the same minimum number of matched disparity bins as the original.
SL_MIN_MATCHED_BINS = 2


# =============================================================================
# OUTPUT PATHS
# =============================================================================

# Prefer the original hysteresis analysis's OUT_DIR.
if "OUT_DIR" in globals():
    SL_ORIGINAL_HYSTERESIS_DIR = Path(OUT_DIR)

elif "DATA_DIR" in globals():
    SL_ORIGINAL_HYSTERESIS_DIR = (
        Path(DATA_DIR)
        / "spatial_localization_hysteresis_bias_vs_disparity"
    )

else:
    raise RuntimeError(
        "Neither OUT_DIR nor DATA_DIR was found.\n"
        "Run the original spatial-localization hysteresis analysis before "
        "this cell, or define DATA_DIR manually."
    )

SL_RELIABILITY_DIR = (
    SL_ORIGINAL_HYSTERESIS_DIR
    / "exact_metric_split_half_reliability"
)

SL_RELIABILITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SL_RELIABILITY_FIG_DIR = (
    SL_RELIABILITY_DIR
    / "figures"
)

SL_RELIABILITY_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SL_ORIGINAL_WINDOW_PATH = (
    SL_ORIGINAL_HYSTERESIS_DIR
    / "spatial_hysteresis_window_level_bias_by_disparity.csv"
)

SL_ORIGINAL_SUBJECT_INDEX_PATH = (
    SL_ORIGINAL_HYSTERESIS_DIR
    / "subject_level_localization_hysteresis_index.csv"
)


# =============================================================================
# LOAD THE EXACT ORIGINAL WINDOW TABLE
# =============================================================================

if (
    "window_df" in globals()
    and isinstance(window_df, pd.DataFrame)
    and len(window_df) > 0
):
    sl_exact_window_df = window_df.copy()
    sl_window_source = "in-memory window_df"

elif SL_ORIGINAL_WINDOW_PATH.exists():
    sl_exact_window_df = pd.read_csv(
        SL_ORIGINAL_WINDOW_PATH
    )
    sl_window_source = str(
        SL_ORIGINAL_WINDOW_PATH
    )

else:
    raise FileNotFoundError(
        "The original window_df was not found in memory, and the saved "
        "window-level CSV was not found at:\n"
        f"{SL_ORIGINAL_WINDOW_PATH}\n\n"
        "Run the original manuscript hysteresis analysis first."
    )


# =============================================================================
# INPUT VALIDATION
# =============================================================================

required_window_columns = {
    "subject",
    "trialIndex",
    "SweepDirection",
    "window_start_sec",
    "window_end_sec",
    "window_center_sec",
    "mean_disparity_window",
    "NormalizedVisualBias_window",
    "VisualDirectedBias_deg_window",
}

missing_window_columns = (
    required_window_columns
    - set(sl_exact_window_df.columns)
)

if missing_window_columns:
    raise ValueError(
        "The original window table is missing required columns:\n"
        f"{sorted(missing_window_columns)}"
    )


def sl_normalize_subject(value):
    """
    Normalize subject labels for validation merges without changing
    participant identity.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    match = re.search(
        r"pid\s*0*(\d+)",
        text,
    )

    if match:
        return f"pid{int(match.group(1)):03d}"

    match = re.search(
        r"(\d+)",
        text,
    )

    if match:
        return f"pid{int(match.group(1)):03d}"

    return text


sl_exact_window_df["subject"] = (
    sl_exact_window_df["subject"]
    .apply(sl_normalize_subject)
)

# Use a string trial identifier to prevent merge mismatches caused by
# integer-versus-float CSV representations.
sl_exact_window_df["trialIndex"] = (
    sl_exact_window_df["trialIndex"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

sl_exact_window_df["SweepDirection"] = (
    sl_exact_window_df["SweepDirection"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        {
            "diverge": "Diverge",
            "converge": "Converge",
        }
    )
)

numeric_window_columns = [
    "window_start_sec",
    "window_end_sec",
    "window_center_sec",
    "mean_disparity_window",
    "NormalizedVisualBias_window",
    "VisualDirectedBias_deg_window",
]

for column in numeric_window_columns:
    sl_exact_window_df[column] = pd.to_numeric(
        sl_exact_window_df[column],
        errors="coerce",
    )

sl_exact_window_df = (
    sl_exact_window_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=[
            "subject",
            "trialIndex",
            "SweepDirection",
            "mean_disparity_window",
            "NormalizedVisualBias_window",
        ]
    )
    .copy()
)

sl_exact_window_df = sl_exact_window_df[
    sl_exact_window_df["SweepDirection"].isin(
        ["Diverge", "Converge"]
    )
].copy()

if sl_exact_window_df.empty:
    raise RuntimeError(
        "No valid converge/diverge rows remained in the original "
        "window-level table."
    )


print("=" * 78)
print("EXACT SL HYSTERESIS RELIABILITY INPUT")
print("=" * 78)

print(f"Window source: {sl_window_source}")
print(f"Rows: {len(sl_exact_window_df):,}")
print(
    "Participants: "
    f"{sl_exact_window_df['subject'].nunique()}"
)
print(
    "Whole trials: "
    f"{sl_exact_window_df[['subject', 'trialIndex', 'SweepDirection']].drop_duplicates().shape[0]}"
)
print(
    "Window-center range: "
    f"{sl_exact_window_df['window_center_sec'].min():.2f}–"
    f"{sl_exact_window_df['window_center_sec'].max():.2f} s"
)

print("\nTrials by direction:")
print(
    sl_exact_window_df[
        [
            "subject",
            "trialIndex",
            "SweepDirection",
        ]
    ]
    .drop_duplicates()
    .groupby("SweepDirection")
    .size()
)


# =============================================================================
# RELIABILITY STATISTIC HELPERS
# =============================================================================

def sl_safe_pearson(x, y):
    """
    Pearson correlation protected against missing or constant data.
    """
    pair = pd.DataFrame(
        {
            "x": x,
            "y": y,
        }
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    if (
        len(pair) < SL_MIN_PARTICIPANTS
        or pair["x"].nunique() < 2
        or pair["y"].nunique() < 2
    ):
        return np.nan, np.nan, int(len(pair))

    result = stats.pearsonr(
        pair["x"],
        pair["y"],
    )

    return (
        float(result.statistic),
        float(result.pvalue),
        int(len(pair)),
    )


def sl_safe_spearman(x, y):
    """
    Spearman correlation protected against missing or constant data.
    """
    pair = pd.DataFrame(
        {
            "x": x,
            "y": y,
        }
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    if (
        len(pair) < SL_MIN_PARTICIPANTS
        or pair["x"].nunique() < 2
        or pair["y"].nunique() < 2
    ):
        return np.nan, np.nan, int(len(pair))

    result = stats.spearmanr(
        pair["x"],
        pair["y"],
    )

    return (
        float(result.statistic),
        float(result.pvalue),
        int(len(pair)),
    )


def sl_spearman_brown(r_half):
    """
    Spearman–Brown correction:

        r_SB = 2r / (1 + r)
    """
    if (
        not np.isfinite(r_half)
        or np.isclose(
            1.0 + r_half,
            0.0,
        )
    ):
        return np.nan

    return float(
        (2.0 * r_half)
        / (1.0 + r_half)
    )


def sl_trapezoidal_integral(y, x):
    """
    NumPy-version-compatible trapezoidal integration.
    """
    y = np.asarray(
        y,
        dtype=float,
    )

    x = np.asarray(
        x,
        dtype=float,
    )

    if hasattr(np, "trapezoid"):
        return float(
            np.trapezoid(
                y,
                x=x,
            )
        )

    return float(
        np.trapz(
            y,
            x=x,
        )
    )


def sl_bootstrap_reliability_ci(
    half_df,
    half_1_column,
    half_2_column,
    method="pearson",
    n_bootstraps=10000,
    seed=12345,
):
    """
    Participant bootstrap for:
      - raw split-half correlation;
      - Spearman–Brown-corrected reliability.

    The bootstrap resamples participants with replacement.
    """
    pair = half_df[
        [
            "subject",
            half_1_column,
            half_2_column,
        ]
    ].replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    if len(pair) < SL_MIN_PARTICIPANTS:
        return {
            "raw_ci_low": np.nan,
            "raw_ci_high": np.nan,
            "sb_ci_low": np.nan,
            "sb_ci_high": np.nan,
            "n_successful_bootstraps": 0,
        }

    x = pair[
        half_1_column
    ].to_numpy(dtype=float)

    y = pair[
        half_2_column
    ].to_numpy(dtype=float)

    rng = np.random.default_rng(seed)

    raw_values = []
    corrected_values = []

    for _ in range(n_bootstraps):
        sample_indices = rng.integers(
            0,
            len(pair),
            size=len(pair),
        )

        x_boot = x[sample_indices]
        y_boot = y[sample_indices]

        if (
            np.nanstd(x_boot) == 0
            or np.nanstd(y_boot) == 0
        ):
            continue

        if method == "pearson":
            r_value = stats.pearsonr(
                x_boot,
                y_boot,
            ).statistic

        elif method == "spearman":
            r_value = stats.spearmanr(
                x_boot,
                y_boot,
            ).statistic

        else:
            raise ValueError(
                "method must be 'pearson' or 'spearman'"
            )

        if not np.isfinite(r_value):
            continue

        raw_values.append(
            float(r_value)
        )

        corrected = sl_spearman_brown(
            float(r_value)
        )

        if np.isfinite(corrected):
            corrected_values.append(
                corrected
            )

    if len(raw_values) == 0:
        return {
            "raw_ci_low": np.nan,
            "raw_ci_high": np.nan,
            "sb_ci_low": np.nan,
            "sb_ci_high": np.nan,
            "n_successful_bootstraps": 0,
        }

    raw_ci = np.percentile(
        raw_values,
        [2.5, 97.5],
    )

    if len(corrected_values) > 0:
        corrected_ci = np.percentile(
            corrected_values,
            [2.5, 97.5],
        )
    else:
        corrected_ci = [
            np.nan,
            np.nan,
        ]

    return {
        "raw_ci_low": float(raw_ci[0]),
        "raw_ci_high": float(raw_ci[1]),
        "sb_ci_low": float(corrected_ci[0]),
        "sb_ci_high": float(corrected_ci[1]),
        "n_successful_bootstraps":
            int(len(raw_values)),
    }


# =============================================================================
# CREATE WHOLE-TRIAL HALF ASSIGNMENTS
# =============================================================================

def sl_create_exact_trial_assignment(
    source_window_df,
    split_type,
    split_number=0,
    seed=12345,
):
    """
    Assign whole trials to half 1 or half 2 separately within:

        subject × SweepDirection

    split_type:
        "odd_even"
        "random"

    For random splits, if a direction contains an odd number of trials,
    one trial is randomly omitted so the two halves are balanced.
    """
    trial_table = (
        source_window_df[
            [
                "subject",
                "trialIndex",
                "SweepDirection",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "subject",
                "SweepDirection",
                "trialIndex",
            ]
        )
        .reset_index(drop=True)
    )

    assignment_rows = []

    rng = np.random.default_rng(
        seed + split_number
    )

    for (
        subject,
        direction,
    ), group in trial_table.groupby(
        [
            "subject",
            "SweepDirection",
        ],
        sort=False,
    ):
        group = (
            group
            .copy()
            .reset_index(drop=True)
        )

        n_trials = len(group)

        if (
            n_trials
            < SL_MIN_TRIALS_PER_DIRECTION
        ):
            continue

        if split_type == "odd_even":
            group["half"] = (
                np.arange(n_trials) % 2
            ) + 1

        elif split_type == "random":
            shuffled_indices = rng.permutation(
                n_trials
            )

            n_per_half = n_trials // 2
            usable_trial_count = (
                2 * n_per_half
            )

            shuffled_indices = (
                shuffled_indices[
                    :usable_trial_count
                ]
            )

            half_labels = np.concatenate(
                [
                    np.ones(
                        n_per_half,
                        dtype=int,
                    ),
                    np.full(
                        n_per_half,
                        2,
                        dtype=int,
                    ),
                ]
            )

            group = group.iloc[
                shuffled_indices
            ].copy()

            group["half"] = half_labels

        else:
            raise ValueError(
                "split_type must be "
                "'odd_even' or 'random'"
            )

        assignment_rows.append(group)

    if len(assignment_rows) == 0:
        return pd.DataFrame(
            columns=[
                "subject",
                "trialIndex",
                "SweepDirection",
                "half",
            ]
        )

    assignments = pd.concat(
        assignment_rows,
        ignore_index=True,
    )

    # Retain subjects who have both sweep directions represented
    # in both halves.
    direction_count_table = (
        assignments
        .groupby(
            [
                "subject",
                "half",
            ],
            as_index=False,
        )
        .agg(
            n_directions=(
                "SweepDirection",
                "nunique",
            )
        )
    )

    valid_subject_counts = (
        direction_count_table[
            direction_count_table[
                "n_directions"
            ] == 2
        ]
        .groupby("subject")
        .size()
    )

    valid_subjects = (
        valid_subject_counts[
            valid_subject_counts == 2
        ]
        .index
    )

    assignments = assignments[
        assignments["subject"].isin(
            valid_subjects
        )
    ].copy()

    return assignments


# =============================================================================
# EXACT REPRODUCTION OF ORIGINAL STEPS 2–3 WITHIN EACH HALF
# =============================================================================

def sl_compute_exact_half_hysteresis(
    source_window_df,
    assignments,
):
    """
    Reproduce the original manuscript hysteresis Steps 2–3 independently
    within each trial half.

    Crucially, this function:
      - uses the original window-level values;
      - uses 2° bins;
      - uses the original pd.cut behavior;
      - directly averages windows within subject × half × direction × bin;
      - does not average within trial first;
      - applies the original two-matched-bin rule.
    """
    if (
        assignments is None
        or assignments.empty
    ):
        return (
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
        )

    working = source_window_df.merge(
        assignments[
            [
                "subject",
                "trialIndex",
                "SweepDirection",
                "half",
            ]
        ],
        on=[
            "subject",
            "trialIndex",
            "SweepDirection",
        ],
        how="inner",
        validate="many_to_one",
    )

    working = (
        working
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                "subject",
                "trialIndex",
                "SweepDirection",
                "half",
                "mean_disparity_window",
                "NormalizedVisualBias_window",
            ]
        )
        .copy()
    )

    # This exactly mirrors the original Step 2 filtering.
    working = working[
        (
            working["mean_disparity_window"]
            >= SL_MIN_DISPARITY_DEG
        )
        & (
            working["mean_disparity_window"]
            <= SL_MAX_DISPARITY_DEG
        )
    ].copy()

    # This exactly mirrors:
    #
    # bins = np.arange(
    #     MIN_DISPARITY_DEG,
    #     MAX_DISPARITY_DEG + BIN_WIDTH_DEG,
    #     BIN_WIDTH_DEG
    # )
    disparity_bins = np.arange(
        SL_MIN_DISPARITY_DEG,
        SL_MAX_DISPARITY_DEG
        + SL_BIN_WIDTH_DEG,
        SL_BIN_WIDTH_DEG,
    )

    working["disparity_bin"] = pd.cut(
        working["mean_disparity_window"],
        bins=disparity_bins,
        include_lowest=True,
    )

    working["disparity_bin_center"] = (
        working["disparity_bin"]
        .apply(
            lambda interval:
                interval.mid
                if pd.notna(interval)
                else np.nan
        )
        .astype(float)
    )

    working = working.dropna(
        subset=[
            "disparity_bin_center",
        ]
    ).copy()

    # -------------------------------------------------------------------------
    # Exact original weighting:
    #
    # Average all windows directly within:
    #
    #     subject × half × direction × disparity bin
    #
    # Do not first average within trial.
    # -------------------------------------------------------------------------
    subject_half_bin_summary = (
        working
        .groupby(
            [
                "subject",
                "half",
                "SweepDirection",
                "disparity_bin_center",
            ],
            observed=True,
        )
        .agg(
            mean_disparity=(
                "mean_disparity_window",
                "mean",
            ),
            mean_NormalizedVisualBias=(
                "NormalizedVisualBias_window",
                "mean",
            ),
            mean_VisualDirectedBias_deg=(
                "VisualDirectedBias_deg_window",
                "mean",
            ),
            n_windows=(
                "NormalizedVisualBias_window",
                "count",
            ),
            n_trials=(
                "trialIndex",
                "nunique",
            ),
        )
        .reset_index()
    )

    # -------------------------------------------------------------------------
    # Exact original normalized-capture pivot.
    # -------------------------------------------------------------------------
    pivot_capture = (
        subject_half_bin_summary
        .pivot_table(
            index=[
                "subject",
                "half",
                "disparity_bin_center",
            ],
            columns="SweepDirection",
            values="mean_NormalizedVisualBias",
        )
        .reset_index()
    )

    # Exact original degree-bias pivot.
    pivot_bias_deg = (
        subject_half_bin_summary
        .pivot_table(
            index=[
                "subject",
                "half",
                "disparity_bin_center",
            ],
            columns="SweepDirection",
            values="mean_VisualDirectedBias_deg",
        )
        .reset_index()
    )

    for pivot_df in [
        pivot_capture,
        pivot_bias_deg,
    ]:
        if "Converge" not in pivot_df.columns:
            pivot_df["Converge"] = np.nan

        if "Diverge" not in pivot_df.columns:
            pivot_df["Diverge"] = np.nan

    pivot_capture[
        "CaptureHysteresis_ConvergeMinusDiverge"
    ] = (
        pivot_capture["Converge"]
        - pivot_capture["Diverge"]
    )

    pivot_bias_deg[
        "BiasDegHysteresis_ConvergeMinusDiverge"
    ] = (
        pivot_bias_deg["Converge"]
        - pivot_bias_deg["Diverge"]
    )

    bin_hysteresis = pd.merge(
        pivot_capture[
            [
                "subject",
                "half",
                "disparity_bin_center",
                "Diverge",
                "Converge",
                "CaptureHysteresis_ConvergeMinusDiverge",
            ]
        ].rename(
            columns={
                "Diverge":
                    "Diverge_NormalizedVisualBias",
                "Converge":
                    "Converge_NormalizedVisualBias",
            }
        ),
        pivot_bias_deg[
            [
                "subject",
                "half",
                "disparity_bin_center",
                "Diverge",
                "Converge",
                "BiasDegHysteresis_ConvergeMinusDiverge",
            ]
        ].rename(
            columns={
                "Diverge":
                    "Diverge_VisualDirectedBias_deg",
                "Converge":
                    "Converge_VisualDirectedBias_deg",
            }
        ),
        on=[
            "subject",
            "half",
            "disparity_bin_center",
        ],
        how="outer",
    )

    # Exact original summary-bin restriction.
    summary_mask = (
        (
            bin_hysteresis[
                "disparity_bin_center"
            ]
            >= SL_SUMMARY_BIN_LOW
        )
        & (
            bin_hysteresis[
                "disparity_bin_center"
            ]
            <= SL_SUMMARY_BIN_HIGH
        )
    )

    summary_bin_data = (
        bin_hysteresis[
            summary_mask
        ]
        .copy()
    )

    # Exact original participant metric.
    subject_half_hysteresis = (
        summary_bin_data
        .groupby(
            [
                "subject",
                "half",
            ],
            as_index=False,
        )
        .agg(
            n_matched_bins=(
                "CaptureHysteresis_ConvergeMinusDiverge",
                lambda values:
                    values.notna().sum(),
            ),
            mean_CaptureHysteresis=(
                "CaptureHysteresis_ConvergeMinusDiverge",
                "mean",
            ),
            mean_BiasDegHysteresis=(
                "BiasDegHysteresis_ConvergeMinusDiverge",
                "mean",
            ),
            mean_Diverge_Capture=(
                "Diverge_NormalizedVisualBias",
                "mean",
            ),
            mean_Converge_Capture=(
                "Converge_NormalizedVisualBias",
                "mean",
            ),
            mean_Diverge_BiasDeg=(
                "Diverge_VisualDirectedBias_deg",
                "mean",
            ),
            mean_Converge_BiasDeg=(
                "Converge_VisualDirectedBias_deg",
                "mean",
            ),
        )
    )

    subject_half_hysteresis = (
        subject_half_hysteresis[
            subject_half_hysteresis[
                "n_matched_bins"
            ] >= SL_MIN_MATCHED_BINS
        ]
        .copy()
    )

    # -------------------------------------------------------------------------
    # Supplementary area metrics derived from the same exact bin curves.
    # These do not replace the primary manuscript metric.
    # -------------------------------------------------------------------------
    area_rows = []

    for (
        subject,
        half,
    ), group in summary_bin_data.groupby(
        [
            "subject",
            "half",
        ],
        sort=False,
    ):
        group = (
            group
            .dropna(
                subset=[
                    "disparity_bin_center",
                    "CaptureHysteresis_ConvergeMinusDiverge",
                ]
            )
            .sort_values(
                "disparity_bin_center"
            )
        )

        if len(group) < SL_MIN_MATCHED_BINS:
            continue

        disparity = group[
            "disparity_bin_center"
        ].to_numpy(dtype=float)

        difference = group[
            "CaptureHysteresis_ConvergeMinusDiverge"
        ].to_numpy(dtype=float)

        signed_area = (
            sl_trapezoidal_integral(
                difference,
                disparity,
            )
        )

        disparity_range = float(
            np.nanmax(disparity)
            - np.nanmin(disparity)
        )

        normalized_area = (
            signed_area / disparity_range
            if disparity_range > 0
            else np.nan
        )

        area_rows.append(
            {
                "subject": subject,
                "half": int(half),
                "capture_hysteresis_signed_area":
                    signed_area,
                "capture_hysteresis_area_normalized":
                    normalized_area,
                "minimum_matched_disparity_deg":
                    float(np.nanmin(disparity)),
                "maximum_matched_disparity_deg":
                    float(np.nanmax(disparity)),
                "matched_disparity_range_deg":
                    disparity_range,
            }
        )

    area_metrics = pd.DataFrame(
        area_rows
    )

    if not area_metrics.empty:
        subject_half_hysteresis = (
            subject_half_hysteresis
            .merge(
                area_metrics,
                on=[
                    "subject",
                    "half",
                ],
                how="left",
            )
        )

    return (
        subject_half_hysteresis,
        bin_hysteresis,
        subject_half_bin_summary,
    )


# =============================================================================
# WIDEN HALF METRICS
# =============================================================================

def sl_widen_exact_half_metrics(
    metrics_long,
):
    """
    Convert long-format half metrics to one row per participant.
    """
    if metrics_long.empty:
        return pd.DataFrame()

    metric_columns = [
        "mean_CaptureHysteresis",
        "mean_BiasDegHysteresis",
        "capture_hysteresis_signed_area",
        "capture_hysteresis_area_normalized",
        "n_matched_bins",
    ]

    available_metrics = [
        metric
        for metric in metric_columns
        if metric in metrics_long.columns
    ]

    wide_parts = []

    for metric in available_metrics:
        metric_wide = (
            metrics_long
            .pivot(
                index="subject",
                columns="half",
                values=metric,
            )
            .rename(
                columns={
                    1: f"{metric}_half1",
                    2: f"{metric}_half2",
                }
            )
        )

        wide_parts.append(
            metric_wide
        )

    if len(wide_parts) == 0:
        return pd.DataFrame()

    return (
        pd.concat(
            wide_parts,
            axis=1,
        )
        .reset_index()
    )


# =============================================================================
# SUMMARIZE RELIABILITY FOR ONE SPLIT
# =============================================================================

def sl_summarize_exact_split_reliability(
    metrics_wide,
    split_type,
    split_number,
    run_bootstrap=False,
):
    """
    Calculate Pearson, Spearman, and Spearman–Brown reliability.
    """
    metrics_to_test = [
        "mean_CaptureHysteresis",
        "mean_BiasDegHysteresis",
        "capture_hysteresis_signed_area",
        "capture_hysteresis_area_normalized",
    ]

    rows = []

    for metric in metrics_to_test:
        half_1_column = (
            f"{metric}_half1"
        )

        half_2_column = (
            f"{metric}_half2"
        )

        if (
            half_1_column
            not in metrics_wide.columns
            or half_2_column
            not in metrics_wide.columns
        ):
            continue

        (
            pearson_r,
            pearson_p,
            pearson_n,
        ) = sl_safe_pearson(
            metrics_wide[
                half_1_column
            ],
            metrics_wide[
                half_2_column
            ],
        )

        (
            spearman_rho,
            spearman_p,
            spearman_n,
        ) = sl_safe_spearman(
            metrics_wide[
                half_1_column
            ],
            metrics_wide[
                half_2_column
            ],
        )

        row = {
            "split_type":
                split_type,
            "split_number":
                split_number,
            "metric":
                metric,
            "n_participants":
                pearson_n,
            "pearson_r_half":
                pearson_r,
            "pearson_p":
                pearson_p,
            "pearson_spearman_brown":
                sl_spearman_brown(
                    pearson_r
                ),
            "spearman_rho_half":
                spearman_rho,
            "spearman_p":
                spearman_p,
            "spearman_brown_from_spearman":
                sl_spearman_brown(
                    spearman_rho
                ),
        }

        if run_bootstrap:
            pearson_ci = (
                sl_bootstrap_reliability_ci(
                    metrics_wide,
                    half_1_column,
                    half_2_column,
                    method="pearson",
                    n_bootstraps=
                        SL_N_BOOTSTRAPS,
                    seed=
                        SL_RELIABILITY_RANDOM_SEED,
                )
            )

            spearman_ci = (
                sl_bootstrap_reliability_ci(
                    metrics_wide,
                    half_1_column,
                    half_2_column,
                    method="spearman",
                    n_bootstraps=
                        SL_N_BOOTSTRAPS,
                    seed=
                        SL_RELIABILITY_RANDOM_SEED
                        + 1,
                )
            )

            row.update(
                {
                    "pearson_half_ci_low":
                        pearson_ci[
                            "raw_ci_low"
                        ],
                    "pearson_half_ci_high":
                        pearson_ci[
                            "raw_ci_high"
                        ],
                    "pearson_sb_ci_low":
                        pearson_ci[
                            "sb_ci_low"
                        ],
                    "pearson_sb_ci_high":
                        pearson_ci[
                            "sb_ci_high"
                        ],
                    "spearman_half_ci_low":
                        spearman_ci[
                            "raw_ci_low"
                        ],
                    "spearman_half_ci_high":
                        spearman_ci[
                            "raw_ci_high"
                        ],
                    "spearman_sb_ci_low":
                        spearman_ci[
                            "sb_ci_low"
                        ],
                    "spearman_sb_ci_high":
                        spearman_ci[
                            "sb_ci_high"
                        ],
                }
            )

        rows.append(row)

    return pd.DataFrame(rows)


# =============================================================================
# FULL-DATA RECONSTRUCTION BEFORE RELIABILITY
# =============================================================================
#
# Every trial is assigned to one artificial full-data group. This must
# reproduce the original manuscript index before the reliability estimates
# should be interpreted.
# =============================================================================

sl_full_assignments = (
    sl_exact_window_df[
        [
            "subject",
            "trialIndex",
            "SweepDirection",
        ]
    ]
    .drop_duplicates()
    .copy()
)

sl_full_assignments["half"] = 1

(
    sl_full_reconstructed_long,
    sl_full_reconstructed_bins,
    sl_full_reconstructed_bin_summary,
) = sl_compute_exact_half_hysteresis(
    sl_exact_window_df,
    sl_full_assignments,
)

sl_full_reconstructed = (
    sl_full_reconstructed_long
    .drop(columns=["half"])
    .copy()
)

sl_full_reconstructed.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_full_data_reconstructed_hysteresis_index.csv",
    index=False,
)

sl_full_reconstructed_bins.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_full_data_reconstructed_bin_hysteresis.csv",
    index=False,
)

sl_full_reconstructed_bin_summary.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_full_data_reconstructed_subject_bin_summary.csv",
    index=False,
)


# =============================================================================
# VALIDATE AGAINST ORIGINAL SUBJECT INDEX
# =============================================================================

sl_full_validation = pd.DataFrame()
sl_validation_passed = False

if SL_ORIGINAL_SUBJECT_INDEX_PATH.exists():
    sl_original_subject_index = pd.read_csv(
        SL_ORIGINAL_SUBJECT_INDEX_PATH
    )

    if (
        "subject"
        not in sl_original_subject_index.columns
        or "mean_CaptureHysteresis"
        not in sl_original_subject_index.columns
    ):
        raise ValueError(
            "The original subject-level hysteresis CSV does not contain "
            "'subject' and 'mean_CaptureHysteresis'."
        )

    sl_original_subject_index[
        "subject"
    ] = (
        sl_original_subject_index[
            "subject"
        ]
        .apply(sl_normalize_subject)
    )

    sl_full_validation = (
        sl_original_subject_index[
            [
                "subject",
                "mean_CaptureHysteresis",
            ]
        ]
        .rename(
            columns={
                "mean_CaptureHysteresis":
                    "mean_CaptureHysteresis_original"
            }
        )
        .merge(
            sl_full_reconstructed[
                [
                    "subject",
                    "mean_CaptureHysteresis",
                ]
            ].rename(
                columns={
                    "mean_CaptureHysteresis":
                        "mean_CaptureHysteresis_reconstructed"
                }
            ),
            on="subject",
            how="outer",
            indicator=True,
        )
    )

    sl_full_validation[
        "difference_reconstructed_minus_original"
    ] = (
        sl_full_validation[
            "mean_CaptureHysteresis_reconstructed"
        ]
        - sl_full_validation[
            "mean_CaptureHysteresis_original"
        ]
    )

    sl_full_validation.to_csv(
        SL_RELIABILITY_DIR
        / "sl_exact_reconstructed_vs_original_validation.csv",
        index=False,
    )

    sl_matched_validation = (
        sl_full_validation[
            sl_full_validation[
                "_merge"
            ] == "both"
        ]
        .dropna(
            subset=[
                "mean_CaptureHysteresis_original",
                "mean_CaptureHysteresis_reconstructed",
            ]
        )
        .copy()
    )

    (
        validation_r,
        validation_p,
        validation_n,
    ) = sl_safe_pearson(
        sl_matched_validation[
            "mean_CaptureHysteresis_original"
        ],
        sl_matched_validation[
            "mean_CaptureHysteresis_reconstructed"
        ],
    )

    validation_mean_difference = float(
        sl_matched_validation[
            "difference_reconstructed_minus_original"
        ].mean()
    )

    validation_mean_absolute_difference = float(
        sl_matched_validation[
            "difference_reconstructed_minus_original"
        ].abs().mean()
    )

    validation_max_absolute_difference = float(
        sl_matched_validation[
            "difference_reconstructed_minus_original"
        ].abs().max()
    )

    # Tight tolerance because the same window table and same aggregation
    # operations should reproduce the original result almost exactly.
    sl_validation_passed = bool(
        (
            validation_n > 0
        )
        and np.isfinite(validation_r)
        and (
            validation_r >= 0.999999
        )
        and (
            validation_max_absolute_difference
            <= 1e-8
        )
    )

    print("\n")
    print("=" * 78)
    print("EXACT FULL-DATA RECONSTRUCTION VALIDATION")
    print("=" * 78)

    print(
        f"Original participants: "
        f"{sl_original_subject_index['subject'].nunique()}"
    )

    print(
        f"Reconstructed participants: "
        f"{sl_full_reconstructed['subject'].nunique()}"
    )

    print(
        f"Matched participants: "
        f"{validation_n}"
    )

    print(
        "Existing versus reconstructed index:\n"
        f"  r = {validation_r:.10f}\n"
        f"  p = {validation_p:.10g}"
    )

    print(
        "Original group mean: "
        f"{sl_matched_validation['mean_CaptureHysteresis_original'].mean():.10f}"
    )

    print(
        "Reconstructed group mean: "
        f"{sl_matched_validation['mean_CaptureHysteresis_reconstructed'].mean():.10f}"
    )

    print(
        "Mean signed discrepancy: "
        f"{validation_mean_difference:.12f}"
    )

    print(
        "Mean absolute discrepancy: "
        f"{validation_mean_absolute_difference:.12f}"
    )

    print(
        "Maximum absolute discrepancy: "
        f"{validation_max_absolute_difference:.12f}"
    )

    if sl_validation_passed:
        print(
            "\nVALIDATION PASSED:\n"
            "The reliability pipeline reproduces the original full-data "
            "manuscript metric within numerical tolerance."
        )

    else:
        print(
            "\nVALIDATION WARNING:\n"
            "The full-data reconstruction does not yet match the original "
            "metric within the strict numerical tolerance. Inspect the "
            "validation CSV before interpreting reliability."
        )

else:
    print(
        "\nOriginal subject-level hysteresis file was not found:\n"
        f"{SL_ORIGINAL_SUBJECT_INDEX_PATH}\n"
        "The reconstructed full-data index was saved, but exact validation "
        "could not be completed."
    )


# =============================================================================
# A. DETERMINISTIC ODD–EVEN SPLIT
# =============================================================================

sl_odd_even_assignments = (
    sl_create_exact_trial_assignment(
        sl_exact_window_df,
        split_type="odd_even",
        split_number=0,
        seed=SL_RELIABILITY_RANDOM_SEED,
    )
)

(
    sl_odd_even_metrics_long,
    sl_odd_even_bin_hysteresis,
    sl_odd_even_subject_bin_summary,
) = sl_compute_exact_half_hysteresis(
    sl_exact_window_df,
    sl_odd_even_assignments,
)

sl_odd_even_metrics_wide = (
    sl_widen_exact_half_metrics(
        sl_odd_even_metrics_long
    )
)

sl_odd_even_reliability = (
    sl_summarize_exact_split_reliability(
        sl_odd_even_metrics_wide,
        split_type="odd_even",
        split_number=0,
        run_bootstrap=True,
    )
)


# =============================================================================
# B. REPEATED RANDOM STRATIFIED SPLITS
# =============================================================================

sl_random_reliability_rows = []
sl_random_participant_rows = []

for split_number in range(
    SL_N_RANDOM_SPLITS
):
    split_assignments = (
        sl_create_exact_trial_assignment(
            sl_exact_window_df,
            split_type="random",
            split_number=split_number,
            seed=SL_RELIABILITY_RANDOM_SEED,
        )
    )

    (
        split_metrics_long,
        _,
        _,
    ) = sl_compute_exact_half_hysteresis(
        sl_exact_window_df,
        split_assignments,
    )

    split_metrics_wide = (
        sl_widen_exact_half_metrics(
            split_metrics_long
        )
    )

    if split_metrics_wide.empty:
        continue

    split_reliability = (
        sl_summarize_exact_split_reliability(
            split_metrics_wide,
            split_type="random",
            split_number=split_number + 1,
            run_bootstrap=False,
        )
    )

    if not split_reliability.empty:
        sl_random_reliability_rows.append(
            split_reliability
        )

    split_metrics_wide = (
        split_metrics_wide.copy()
    )

    split_metrics_wide[
        "split_number"
    ] = (
        split_number + 1
    )

    sl_random_participant_rows.append(
        split_metrics_wide
    )

    if (
        split_number == 0
        or (split_number + 1) % 100 == 0
    ):
        print(
            "Completed exact SL random split "
            f"{split_number + 1}/"
            f"{SL_N_RANDOM_SPLITS}"
        )


if len(sl_random_reliability_rows) > 0:
    sl_random_split_reliability = (
        pd.concat(
            sl_random_reliability_rows,
            ignore_index=True,
        )
    )
else:
    sl_random_split_reliability = (
        pd.DataFrame()
    )

if len(sl_random_participant_rows) > 0:
    sl_random_split_participant_estimates = (
        pd.concat(
            sl_random_participant_rows,
            ignore_index=True,
        )
    )
else:
    sl_random_split_participant_estimates = (
        pd.DataFrame()
    )


# =============================================================================
# SUMMARIZE RANDOM-SPLIT DISTRIBUTIONS
# =============================================================================

sl_random_summary_rows = []

if not sl_random_split_reliability.empty:
    for (
        metric,
        metric_group,
    ) in sl_random_split_reliability.groupby(
        "metric"
    ):
        for reliability_statistic in [
            "pearson_r_half",
            "pearson_spearman_brown",
            "spearman_rho_half",
            "spearman_brown_from_spearman",
        ]:
            values = pd.to_numeric(
                metric_group[
                    reliability_statistic
                ],
                errors="coerce",
            ).dropna()

            if len(values) == 0:
                continue

            sl_random_summary_rows.append(
                {
                    "metric":
                        metric,
                    "reliability_statistic":
                        reliability_statistic,
                    "n_successful_splits":
                        int(len(values)),
                    "mean":
                        float(values.mean()),
                    "median":
                        float(values.median()),
                    "sd":
                        float(
                            values.std(ddof=1)
                        ),
                    "percentile_2p5":
                        float(
                            np.percentile(
                                values,
                                2.5,
                            )
                        ),
                    "percentile_25":
                        float(
                            np.percentile(
                                values,
                                25,
                            )
                        ),
                    "percentile_75":
                        float(
                            np.percentile(
                                values,
                                75,
                            )
                        ),
                    "percentile_97p5":
                        float(
                            np.percentile(
                                values,
                                97.5,
                            )
                        ),
                    "proportion_positive":
                        float(
                            (values > 0).mean()
                        ),
                }
            )

sl_random_split_summary = pd.DataFrame(
    sl_random_summary_rows
)


# =============================================================================
# SAVE ALL OUTPUTS
# =============================================================================

sl_odd_even_assignments.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_odd_even_trial_assignments.csv",
    index=False,
)

sl_odd_even_metrics_long.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_odd_even_half_hysteresis_metrics_long.csv",
    index=False,
)

sl_odd_even_metrics_wide.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_odd_even_half_hysteresis_metrics_wide.csv",
    index=False,
)

sl_odd_even_bin_hysteresis.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_odd_even_half_bin_hysteresis.csv",
    index=False,
)

sl_odd_even_subject_bin_summary.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_odd_even_half_subject_bin_summary.csv",
    index=False,
)

sl_odd_even_reliability.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_odd_even_reliability_summary.csv",
    index=False,
)

sl_random_split_reliability.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_random_split_reliability_all_splits.csv",
    index=False,
)

sl_random_split_participant_estimates.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_random_split_participant_estimates.csv",
    index=False,
)

sl_random_split_summary.to_csv(
    SL_RELIABILITY_DIR
    / "sl_exact_random_split_reliability_distribution_summary.csv",
    index=False,
)


# =============================================================================
# ODD–EVEN RELIABILITY FIGURES
# =============================================================================

figure_metrics = [
    "mean_CaptureHysteresis",
    "mean_BiasDegHysteresis",
    "capture_hysteresis_signed_area",
    "capture_hysteresis_area_normalized",
]

for metric in figure_metrics:
    half_1_column = (
        f"{metric}_half1"
    )

    half_2_column = (
        f"{metric}_half2"
    )

    if (
        half_1_column
        not in sl_odd_even_metrics_wide.columns
        or half_2_column
        not in sl_odd_even_metrics_wide.columns
    ):
        continue

    plot_data = (
        sl_odd_even_metrics_wide[
            [
                "subject",
                half_1_column,
                half_2_column,
            ]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
    )

    if (
        len(plot_data)
        < SL_MIN_PARTICIPANTS
    ):
        continue

    (
        pearson_r,
        pearson_p,
        participant_n,
    ) = sl_safe_pearson(
        plot_data[
            half_1_column
        ],
        plot_data[
            half_2_column
        ],
    )

    corrected_reliability = (
        sl_spearman_brown(
            pearson_r
        )
    )

    x_values = plot_data[
        half_1_column
    ].to_numpy(dtype=float)

    y_values = plot_data[
        half_2_column
    ].to_numpy(dtype=float)

    plt.figure(
        figsize=(6.5, 6.0)
    )

    plt.scatter(
        x_values,
        y_values,
        s=55,
        alpha=0.80,
    )

    if np.nanstd(x_values) > 0:
        slope, intercept = np.polyfit(
            x_values,
            y_values,
            deg=1,
        )

        x_grid = np.linspace(
            np.nanmin(x_values),
            np.nanmax(x_values),
            200,
        )

        plt.plot(
            x_grid,
            intercept
            + slope * x_grid,
            linewidth=2,
        )

    common_minimum = float(
        np.nanmin(
            np.concatenate(
                [
                    x_values,
                    y_values,
                ]
            )
        )
    )

    common_maximum = float(
        np.nanmax(
            np.concatenate(
                [
                    x_values,
                    y_values,
                ]
            )
        )
    )

    plt.plot(
        [
            common_minimum,
            common_maximum,
        ],
        [
            common_minimum,
            common_maximum,
        ],
        linestyle="--",
        linewidth=1.2,
        alpha=0.60,
    )

    plt.xlabel(
        f"Odd trials: {metric}"
    )

    plt.ylabel(
        f"Even trials: {metric}"
    )

    plt.title(
        "Exact SL hysteresis odd–even reliability\n"
        f"n = {participant_n}, "
        f"r = {pearson_r:.3f}, "
        f"Spearman–Brown = "
        f"{corrected_reliability:.3f}"
    )

    plt.tight_layout()

    plt.savefig(
        SL_RELIABILITY_FIG_DIR
        / f"sl_exact_odd_even_{metric}.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()


# =============================================================================
# RANDOM-SPLIT DISTRIBUTION FIGURES
# =============================================================================

if not sl_random_split_reliability.empty:
    for metric in (
        sl_random_split_reliability[
            "metric"
        ].unique()
    ):
        values = (
            sl_random_split_reliability.loc[
                sl_random_split_reliability[
                    "metric"
                ] == metric,
                "pearson_spearman_brown",
            ]
            .dropna()
        )

        if len(values) == 0:
            continue

        plt.figure(
            figsize=(7.0, 5.0)
        )

        plt.hist(
            values,
            bins=35,
            edgecolor="black",
            alpha=0.80,
        )

        plt.axvline(
            values.median(),
            linestyle="--",
            linewidth=2,
            label=(
                "Median corrected reliability = "
                f"{values.median():.3f}"
            ),
        )

        plt.axvline(
            0,
            linewidth=1,
        )

        plt.xlabel(
            "Spearman–Brown-corrected "
            "Pearson reliability"
        )

        plt.ylabel(
            "Number of random splits"
        )

        plt.title(
            "Exact SL repeated split-half reliability\n"
            f"{metric}"
        )

        plt.legend()

        plt.tight_layout()

        plt.savefig(
            SL_RELIABILITY_FIG_DIR
            / (
                "sl_exact_random_split_"
                f"distribution_{metric}.png"
            ),
            dpi=200,
            bbox_inches="tight",
        )

        plt.show()


# =============================================================================
# PRINT FINAL RESULTS
# =============================================================================

print("\n")
print("=" * 78)
print("EXACT SL HYSTERESIS SPLIT-HALF RELIABILITY")
print("=" * 78)

print("\nPrimary manuscript metric:")
print(
    "mean_CaptureHysteresis = mean Converge-minus-Diverge normalized "
    "visual bias across matched 5°–35° disparity bins."
)

print("\nOdd–even reliability:")
if sl_odd_even_reliability.empty:
    print(
        "No valid odd–even reliability estimates were obtained."
    )
else:
    print(
        sl_odd_even_reliability.to_string(
            index=False
        )
    )

print(
    "\nRepeated random-split distribution summary:"
)

if sl_random_split_summary.empty:
    print(
        "No valid random-split reliability estimates were obtained."
    )
else:
    print(
        sl_random_split_summary.to_string(
            index=False
        )
    )

primary_columns = [
    "mean_CaptureHysteresis_half1",
    "mean_CaptureHysteresis_half2",
]

if all(
    column
    in sl_odd_even_metrics_wide.columns
    for column in primary_columns
):
    primary_participant_n = (
        sl_odd_even_metrics_wide[
            primary_columns
        ]
        .dropna()
        .shape[0]
    )
else:
    primary_participant_n = 0

print("\nParticipant inclusion:")
print(
    "Odd–even participants with complete primary estimates: "
    f"{primary_participant_n}"
)

print("\nFull-data validation status:")
if SL_ORIGINAL_SUBJECT_INDEX_PATH.exists():
    print(
        "PASSED"
        if sl_validation_passed
        else "WARNING — inspect validation output"
    )
else:
    print(
        "Original subject index was unavailable."
    )

print("\nOutputs saved to:")
print(
    SL_RELIABILITY_DIR
)

print("\nPrimary output files:")
print(
    SL_RELIABILITY_DIR
    / "sl_exact_reconstructed_vs_original_validation.csv"
)

print(
    SL_RELIABILITY_DIR
    / "sl_exact_odd_even_reliability_summary.csv"
)

print(
    SL_RELIABILITY_DIR
    / "sl_exact_random_split_reliability_distribution_summary.csv"
)

print(
    SL_RELIABILITY_DIR
    / "sl_exact_random_split_reliability_all_splits.csv"
)

print("\nReporting guidance:")
print(
    "1. First verify that the exact full-data reconstruction passes."
)

print(
    "2. Treat mean_CaptureHysteresis as the primary reliability target."
)

print(
    "3. Report the odd–even estimate as a transparent deterministic check."
)

print(
    "4. Emphasize the median Spearman–Brown-corrected Pearson reliability "
    "across repeated random stratified splits."
)

print(
    "5. Signed-area and degree-bias reliability estimates are supplementary."
)

## 15. Complementary Lag Analyses

Shared preparation and complementary analyses linking lag to disparity, causal judgment, continuous visual bias, sweep history, stabilization, confidence, and continuous multisensory state.

In [20]:
# %% COMPLEMENTARY LAG ANALYSES — SHARED PREPARATION
# =============================================================================
# PURPOSE
# =============================================================================
# Shared setup for the complementary lag analyses:
#
#   1. Lag by fixed audiovisual disparity
#   2. Lag × visual bias
#   3. Temporal lag hysteresis during sweeps
#   4. Lag before vs after spatial stabilization
#   5. Confidence × causal judgment predicting lag
#   6. Continuous multisensory integration-state analysis
#
# IMPORTANT
# =============================================================================
# For analyses in which lag itself is interpreted as a behavioral outcome,
# this code uses the already-generated VALID lag table from the exact
# collaborator-style analysis:
#
#   - 20 Hz
#   - auditory-relative tracking lag
#   - non-negative lag search: 0–4 s
#   - peak r > .40
#   - auditory stimulus SD >= 3 deg
#
# The valid-lag table is therefore treated as authoritative.
# =============================================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import (
    ttest_1samp,
    ttest_rel,
    pearsonr,
    spearmanr,
    t as t_dist,
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# PATHS
# =============================================================================

DATA_DIR = (
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1\individual_data_SL"
)

FILE_PATTERN = "*master_trials*.csv"

SPATIAL_ANALYSES_ROOT = (
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1\spatial_analyses"
)

# Exact collaborator-style valid lag table
VALID_LAG_CSV = os.path.join(
    SPATIAL_ANALYSES_ROOT,
    "02_same_different_tracking_lag",
    "csv_outputs",
    "03_VALID_trial_lag_estimates_after_exact_collaborator_QC.csv",
)

# Existing trial-level spatial metrics
SPATIAL_METRICS_CSV = os.path.join(
    SPATIAL_ANALYSES_ROOT,
    "00_core_spatial_metrics",
    "trial_level_metrics",
    "cell3_trial_level_lag_corrected_spatial_metrics.csv",
)

# Existing breakpoint results
BREAKPOINT_CSV = os.path.join(
    SPATIAL_ANALYSES_ROOT,
    "06_stabilization_breakpoint",
    "csv_outputs",
    "anchor_trial_level_breakpoint_DYNAMIC_metrics.csv",
)

# New complementary lag analyses root
COMPLEMENTARY_ROOT = os.path.join(
    SPATIAL_ANALYSES_ROOT,
    "08_complementary_lag_analyses",
)
os.makedirs(COMPLEMENTARY_ROOT, exist_ok=True)

# =============================================================================
# GENERAL SETTINGS
# =============================================================================

RANDOM_SEED = 12345
N_PERM = 10000
N_BOOT = 10000
FIG_DPI = 180

RESAMPLE_HZ = 20.0
DT_SEC = 1.0 / RESAMPLE_HZ
TRIAL_DURATION_SEC = 18.0

# =============================================================================
# HELPERS
# =============================================================================

def mean_ci_t(values, alpha=0.05):
    vals = pd.Series(values).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    sd = np.std(vals, ddof=1)
    se = sd / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)

    return m, m - crit * se, m + crit * se, se, n


def sign_flip_test(values, popmean=0.0, alternative="two-sided",
                   n_perm=N_PERM, seed=RANDOM_SEED):

    vals = pd.Series(values).dropna().astype(float).values
    diffs = vals - popmean
    n = len(diffs)

    if n < 2:
        return np.nan

    obs = np.mean(diffs)

    rng = np.random.default_rng(seed)

    # Memory-safe permutation loop
    count = 0

    for _ in range(n_perm):
        signs = rng.choice([-1.0, 1.0], size=n)
        null_val = np.mean(signs * diffs)

        if alternative == "greater":
            count += null_val >= obs

        elif alternative == "less":
            count += null_val <= obs

        else:
            count += abs(null_val) >= abs(obs)

    return (count + 1) / (n_perm + 1)


def bootstrap_mean_ci(values, n_boot=N_BOOT, seed=RANDOM_SEED):

    vals = pd.Series(values).dropna().astype(float).values

    if len(vals) < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)

    boots = np.empty(n_boot)

    for b in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots[b] = np.mean(sample)

    return np.percentile(boots, [2.5, 97.5])


def parse_same_response(x):

    if pd.isna(x):
        return np.nan

    if isinstance(x, (bool, np.bool_)):
        return float(x)

    try:
        numeric = float(x)

        if numeric in [0, 1]:
            return numeric

    except Exception:
        pass

    s = str(x).strip().lower()

    if s in [
        "yes", "y", "same", "one", "one source", "1",
        "common", "same source"
    ]:
        return 1.0

    if s in [
        "no", "n", "different", "two", "two sources", "0",
        "separate", "different source"
    ]:
        return 0.0

    return np.nan


def parse_confidence(x):

    if pd.isna(x):
        return np.nan

    try:
        val = float(x)

        if np.isfinite(val):
            return val

    except Exception:
        pass

    s = str(x).lower()

    m = re.search(r"([1-5](?:\.\d+)?)", s)

    if m:
        return float(m.group(1))

    return np.nan


def extract_subject_id(filepath):

    name = Path(filepath).name

    m = re.search(
        r"(pid\d+)",
        name,
        flags=re.IGNORECASE
    )

    return (
        m.group(1).lower()
        if m
        else Path(filepath).stem.lower()
    )


def load_trial_level_posttrial_metadata():

    files = sorted(
        glob.glob(
            os.path.join(
                DATA_DIR,
                FILE_PATTERN
            )
        )
    )

    rows = []

    for fpath in files:

        subject = extract_subject_id(fpath)

        df = pd.read_csv(fpath)

        if "isPractice" in df.columns:
            df = df[
                pd.to_numeric(
                    df["isPractice"],
                    errors="coerce"
                ).fillna(0) == 0
            ].copy()

        if "trialIndex" not in df.columns:
            continue

        # One row per trial
        first = (
            df
            .sort_values("t")
            .groupby(
                "trialIndex",
                as_index=False
            )
            .first()
        )

        for _, row in first.iterrows():

            same = np.nan

            if "same_response_yes" in row.index:
                same = parse_same_response(
                    row["same_response_yes"]
                )

            if (
                not np.isfinite(same)
                and "post_question" in row.index
            ):
                same = parse_same_response(
                    row["post_question"]
                )

            confidence = np.nan

            if "confidence_rating" in row.index:
                confidence = parse_confidence(
                    row["confidence_rating"]
                )

            if (
                not np.isfinite(confidence)
                and "post_question2" in row.index
            ):
                confidence = parse_confidence(
                    row["post_question2"]
                )

            rows.append(
                {
                    "subject": subject,
                    "trialIndex": row["trialIndex"],
                    "same_response_meta": same,
                    "confidence_rating": confidence,
                    "trialType_meta": row.get(
                        "trialType",
                        np.nan
                    ),
                    "condLabel_meta": row.get(
                        "condLabel",
                        np.nan
                    ),
                }
            )

    return pd.DataFrame(rows)


# =============================================================================
# LOAD VALID AUDITORY-LAG DATA
# =============================================================================

if not os.path.exists(VALID_LAG_CSV):
    raise FileNotFoundError(
        "Valid collaborator-style lag table not found:\n"
        f"{VALID_LAG_CSV}\n\n"
        "Run the exact Same/Different auditory-lag analysis first."
    )

lag_valid_df = pd.read_csv(
    VALID_LAG_CSV
)

required_lag_cols = [
    "subject",
    "trialIndex",
    "physical_condition",
    "offset_deg",
    "lag_audio_sec",
    "peak_audio_corr",
]

missing = [
    c for c in required_lag_cols
    if c not in lag_valid_df.columns
]

if missing:
    raise RuntimeError(
        "Valid lag file is missing columns: "
        + ", ".join(missing)
    )

lag_valid_df["subject"] = (
    lag_valid_df["subject"]
    .astype(str)
    .str.lower()
)

lag_valid_df["trialIndex"] = pd.to_numeric(
    lag_valid_df["trialIndex"],
    errors="coerce"
)

lag_valid_df["lag_audio_sec"] = pd.to_numeric(
    lag_valid_df["lag_audio_sec"],
    errors="coerce"
)

lag_valid_df["peak_audio_corr"] = pd.to_numeric(
    lag_valid_df["peak_audio_corr"],
    errors="coerce"
)

lag_valid_df["offset_deg"] = pd.to_numeric(
    lag_valid_df["offset_deg"],
    errors="coerce"
)

# Explicit judgment
if "same_response" in lag_valid_df.columns:

    lag_valid_df["same_report"] = (
        lag_valid_df["same_response"]
        .apply(parse_same_response)
    )

elif "judgment" in lag_valid_df.columns:

    lag_valid_df["same_report"] = (
        lag_valid_df["judgment"]
        .apply(parse_same_response)
    )

else:
    lag_valid_df["same_report"] = np.nan

lag_valid_df["different_report"] = (
    1.0 - lag_valid_df["same_report"]
)

# Clean disparity variable
lag_valid_df["disparity_deg"] = (
    lag_valid_df["offset_deg"].abs()
)

lag_valid_df.loc[
    lag_valid_df["physical_condition"]
    .astype(str)
    .str.lower()
    .eq("congruent"),
    "disparity_deg"
] = 0.0


# =============================================================================
# LOAD SPATIAL BIAS METRICS
# =============================================================================

if os.path.exists(SPATIAL_METRICS_CSV):

    spatial_metrics_df = pd.read_csv(
        SPATIAL_METRICS_CSV
    )

    spatial_metrics_df["subject"] = (
        spatial_metrics_df["subject"]
        .astype(str)
        .str.lower()
    )

    spatial_metrics_df["trialIndex"] = pd.to_numeric(
        spatial_metrics_df["trialIndex"],
        errors="coerce"
    )

else:

    spatial_metrics_df = pd.DataFrame()

    print(
        "\nWARNING: trial-level spatial metric file not found:"
    )
    print(SPATIAL_METRICS_CSV)
    print(
        "Lag × visual-bias and integration-state analyses "
        "will require that file."
    )


# =============================================================================
# LOAD POST-TRIAL JUDGMENT + CONFIDENCE METADATA
# =============================================================================

posttrial_meta_df = (
    load_trial_level_posttrial_metadata()
)

posttrial_meta_df["subject"] = (
    posttrial_meta_df["subject"]
    .astype(str)
    .str.lower()
)

posttrial_meta_df["trialIndex"] = pd.to_numeric(
    posttrial_meta_df["trialIndex"],
    errors="coerce"
)


# =============================================================================
# BUILD MASTER TRIAL-LEVEL COMPLEMENTARY-LAG DATASET
# =============================================================================

lag_master_df = lag_valid_df.copy()

lag_master_df = lag_master_df.merge(
    posttrial_meta_df[
        [
            "subject",
            "trialIndex",
            "same_response_meta",
            "confidence_rating",
        ]
    ],
    on=[
        "subject",
        "trialIndex",
    ],
    how="left",
)

# Prefer exact lag-table response if available.
lag_master_df["same_report"] = np.where(
    lag_master_df["same_report"].notna(),
    lag_master_df["same_report"],
    lag_master_df["same_response_meta"],
)

lag_master_df["different_report"] = (
    1.0 - lag_master_df["same_report"]
)

lag_master_df["judgment_label"] = np.where(
    lag_master_df["same_report"] == 1,
    "Same",
    np.where(
        lag_master_df["same_report"] == 0,
        "Different",
        np.nan,
    )
)


# Merge spatial metrics when available.
if len(spatial_metrics_df) > 0:

    spatial_keep_cols = [
        c for c in [
            "subject",
            "trialIndex",
            "trialType",
            "condLabel",
            "offsetDeg",
            "Capture_rmse_lagcorr",
            "Capture_dev_lagcorr",
            "VisualCapture_audioLag",
            "VisualCapture_visualLag",
            "VisualCapture_raw",
            "RMSE_A_lagcorr",
            "RMSE_V_lagcorr",
        ]
        if c in spatial_metrics_df.columns
    ]

    lag_master_df = lag_master_df.merge(
        spatial_metrics_df[
            spatial_keep_cols
        ],
        on=[
            "subject",
            "trialIndex",
        ],
        how="left",
        suffixes=(
            "",
            "_spatial",
        ),
    )


# =============================================================================
# SAVE MASTER MERGED DATASET
# =============================================================================

master_csv = os.path.join(
    COMPLEMENTARY_ROOT,
    "complementary_lag_master_valid_trials.csv",
)

lag_master_df.to_csv(
    master_csv,
    index=False,
)

print("\n" + "=" * 90)
print("COMPLEMENTARY LAG ANALYSES — SHARED PREPARATION COMPLETE")
print("=" * 90)

print(
    f"\nValid lag trials: {len(lag_master_df)}"
)

print(
    f"Subjects: {lag_master_df['subject'].nunique()}"
)

print(
    "\nPhysical conditions:"
)

print(
    lag_master_df[
        "physical_condition"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nJudgment counts:"
)

print(
    lag_master_df[
        "judgment_label"
    ].value_counts(
        dropna=False
    )
)

print(
    f"\nSaved merged complementary-lag table:\n{master_csv}"
)


COMPLEMENTARY LAG ANALYSES — SHARED PREPARATION COMPLETE

Valid lag trials: 3274
Subjects: 50

Physical conditions:
physical_condition
fixed_10          889
fixed_20          811
congruent         457
fixed_30          438
sweep_converge    345
sweep_diverge     334
Name: count, dtype: int64

Judgment counts:
judgment_label
Same         1885
Different    1389
Name: count, dtype: int64

Saved merged complementary-lag table:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\spatial_analyses\08_complementary_lag_analyses\complementary_lag_master_valid_trials.csv


In [21]:
# %% COMPLEMENTARY LAG ANALYSIS 1 — FIXED DISPARITY × CAUSAL JUDGMENT
# =============================================================================
# QUESTIONS
# =============================================================================
# 1. Does auditory-relative tracking lag increase as audiovisual disparity grows?
#
# 2. Does explicit causal judgment predict lag after controlling for disparity?
#
# 3. Does the disparity-lag slope itself differ between trials judged Same
#    versus Different?
#
# Primary model:
#
#   Lag ~ Disparity + Judgment + Disparity × Judgment
#
# Participant-first inferential strategy:
#   - median valid lag within subject × disparity
#   - median valid lag within subject × disparity × judgment
#   - subject-specific regression coefficients
#   - group-level one-sample tests on coefficients
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_1samp
import statsmodels.formula.api as smf

# =============================================================================
# OUTPUT
# =============================================================================

OUT_DIR = os.path.join(
    COMPLEMENTARY_ROOT,
    "01_fixed_disparity_lag",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures",
)

for d in [
    OUT_DIR,
    CSV_DIR,
    FIG_DIR,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# =============================================================================
# FIXED AV TRIALS ONLY
# =============================================================================

fixed_conditions = [
    "congruent",
    "fixed_10",
    "fixed_20",
    "fixed_30",
    "fixed_40",
]

df = lag_master_df[
    lag_master_df[
        "physical_condition"
    ].isin(
        fixed_conditions
    )
].copy()

df = df.dropna(
    subset=[
        "subject",
        "lag_audio_sec",
        "disparity_deg",
    ]
)

df["disparity_10"] = (
    df["disparity_deg"] / 10.0
)

print(
    f"Fixed valid lag trials: {len(df)}"
)

print(
    f"Subjects: {df['subject'].nunique()}"
)


# =============================================================================
# ANALYSIS 1A — MEDIAN LAG BY DISPARITY
# =============================================================================

subject_disparity = (
    df
    .groupby(
        [
            "subject",
            "disparity_deg",
        ],
        as_index=False,
    )
    .agg(
        median_lag_sec=(
            "lag_audio_sec",
            "median",
        ),
        mean_lag_sec=(
            "lag_audio_sec",
            "mean",
        ),
        n_valid_trials=(
            "lag_audio_sec",
            "count",
        ),
        mean_peak_r=(
            "peak_audio_corr",
            "mean",
        ),
    )
)

subject_disparity.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_median_lag_by_fixed_disparity.csv",
    ),
    index=False,
)


group_rows = []

for disparity, g in subject_disparity.groupby(
    "disparity_deg"
):

    m, lo, hi, se, n = mean_ci_t(
        g["median_lag_sec"]
    )

    group_rows.append(
        {
            "disparity_deg": disparity,
            "mean_subject_median_lag_sec": m,
            "ci95_low": lo,
            "ci95_high": hi,
            "sem": se,
            "n_subjects": n,
            "mean_valid_trials_per_subject": g[
                "n_valid_trials"
            ].mean(),
        }
    )


group_disparity = pd.DataFrame(
    group_rows
).sort_values(
    "disparity_deg"
)

group_disparity.to_csv(
    os.path.join(
        CSV_DIR,
        "group_lag_by_fixed_disparity.csv",
    ),
    index=False,
)


# =============================================================================
# PARTICIPANT-FIRST LINEAR DISPARITY SLOPE
# =============================================================================

subject_slope_rows = []

for subject, g in df.groupby(
    "subject"
):

    # Reduce repeated trials to participant × disparity medians first.
    sg = (
        g
        .groupby(
            "disparity_deg",
            as_index=False,
        )
        .agg(
            lag_audio_sec=(
                "lag_audio_sec",
                "median",
            )
        )
    )

    if len(sg) < 3:
        continue

    sg["disparity_10"] = (
        sg["disparity_deg"] / 10.0
    )

    try:

        model = smf.ols(
            "lag_audio_sec ~ disparity_10",
            data=sg,
        ).fit()

        subject_slope_rows.append(
            {
                "subject": subject,
                "n_disparities": len(sg),
                "intercept_sec": model.params.get(
                    "Intercept",
                    np.nan,
                ),
                "disparity_slope_sec_per_10deg": model.params.get(
                    "disparity_10",
                    np.nan,
                ),
                "r2": model.rsquared,
            }
        )

    except Exception:
        continue


subject_slopes = pd.DataFrame(
    subject_slope_rows
)

subject_slopes.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_first_disparity_lag_slopes.csv",
    ),
    index=False,
)


slope_vals = (
    subject_slopes[
        "disparity_slope_sec_per_10deg"
    ]
    .dropna()
    .values
)

if len(slope_vals) >= 2:

    t_result = ttest_1samp(
        slope_vals,
        0.0,
    )

    m, lo, hi, se, n = mean_ci_t(
        slope_vals
    )

    p_perm = sign_flip_test(
        slope_vals,
        popmean=0.0,
        alternative="greater",
    )

    slope_result = pd.DataFrame(
        [
            {
                "effect": "disparity_slope",
                "mean_sec_per_10deg": m,
                "ci95_low": lo,
                "ci95_high": hi,
                "t": t_result.statistic,
                "df": n - 1,
                "p_two_sided": t_result.pvalue,
                "p_signflip_greater": p_perm,
                "n_subjects": n,
            }
        ]
    )

else:

    slope_result = pd.DataFrame()


slope_result.to_csv(
    os.path.join(
        CSV_DIR,
        "group_disparity_lag_slope_test.csv",
    ),
    index=False,
)


# =============================================================================
# ANALYSIS 1B — SUBJECT × DISPARITY × JUDGMENT MEDIANS
# =============================================================================

judgment_df = df.dropna(
    subset=[
        "same_report",
    ]
).copy()

subject_disparity_judgment = (
    judgment_df
    .groupby(
        [
            "subject",
            "disparity_deg",
            "judgment_label",
        ],
        as_index=False,
    )
    .agg(
        median_lag_sec=(
            "lag_audio_sec",
            "median",
        ),
        n_valid_trials=(
            "lag_audio_sec",
            "count",
        ),
    )
)

subject_disparity_judgment.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_median_lag_by_disparity_and_judgment.csv",
    ),
    index=False,
)


# =============================================================================
# PARTICIPANT-FIRST INTERACTION MODEL
#
# Lag ~ disparity + Different + disparity × Different
# =============================================================================

subject_model_rows = []

for subject, g in judgment_df.groupby(
    "subject"
):

    # Median within each participant × disparity × judgment cell.
    sg = (
        g
        .groupby(
            [
                "disparity_deg",
                "different_report",
            ],
            as_index=False,
        )
        .agg(
            lag_audio_sec=(
                "lag_audio_sec",
                "median",
            )
        )
    )

    if (
        len(sg) < 5
        or sg[
            "different_report"
        ].nunique() < 2
        or sg[
            "disparity_deg"
        ].nunique() < 2
    ):
        continue

    sg["disparity_10"] = (
        sg["disparity_deg"] / 10.0
    )

    try:

        model = smf.ols(
            (
                "lag_audio_sec ~ "
                "disparity_10 * different_report"
            ),
            data=sg,
        ).fit()

        subject_model_rows.append(
            {
                "subject": subject,
                "n_cells": len(sg),
                "n_disparities": sg[
                    "disparity_deg"
                ].nunique(),

                "intercept_same_sec":
                    model.params.get(
                        "Intercept",
                        np.nan,
                    ),

                "disparity_slope_same_sec_per_10deg":
                    model.params.get(
                        "disparity_10",
                        np.nan,
                    ),

                "different_minus_same_intercept_sec":
                    model.params.get(
                        "different_report",
                        np.nan,
                    ),

                "disparity_x_different_interaction_sec_per_10deg":
                    model.params.get(
                        "disparity_10:different_report",
                        np.nan,
                    ),

                "disparity_slope_different_sec_per_10deg":
                    (
                        model.params.get(
                            "disparity_10",
                            np.nan,
                        )
                        +
                        model.params.get(
                            "disparity_10:different_report",
                            np.nan,
                        )
                    ),

                "r2": model.rsquared,
            }
        )

    except Exception:
        continue


subject_models = pd.DataFrame(
    subject_model_rows
)

subject_models.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_first_disparity_x_judgment_models.csv",
    ),
    index=False,
)


# =============================================================================
# GROUP TESTS ON SUBJECT-SPECIFIC COEFFICIENTS
# =============================================================================

effects = [
    (
        "disparity_slope_same_sec_per_10deg",
        "greater",
    ),
    (
        "different_minus_same_intercept_sec",
        "greater",
    ),
    (
        "disparity_x_different_interaction_sec_per_10deg",
        "two-sided",
    ),
    (
        "disparity_slope_different_sec_per_10deg",
        "greater",
    ),
]

result_rows = []

for effect, direction in effects:

    vals = (
        subject_models[
            effect
        ]
        .dropna()
        .values
    )

    if len(vals) < 2:
        continue

    t_result = ttest_1samp(
        vals,
        0.0,
    )

    m, lo, hi, se, n = mean_ci_t(
        vals
    )

    perm_p = sign_flip_test(
        vals,
        popmean=0.0,
        alternative=direction,
    )

    result_rows.append(
        {
            "effect": effect,
            "mean": m,
            "ci95_low": lo,
            "ci95_high": hi,
            "t": t_result.statistic,
            "df": n - 1,
            "p_two_sided": t_result.pvalue,
            "signflip_alternative": direction,
            "p_signflip": perm_p,
            "n_subjects": n,
        }
    )


group_model_results = pd.DataFrame(
    result_rows
)

group_model_results.to_csv(
    os.path.join(
        CSV_DIR,
        "group_disparity_x_judgment_tests.csv",
    ),
    index=False,
)


# =============================================================================
# PLOT 1 — LAG BY DISPARITY
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)

ax.errorbar(
    group_disparity[
        "disparity_deg"
    ],
    group_disparity[
        "mean_subject_median_lag_sec"
    ],
    yerr=[
        group_disparity[
            "mean_subject_median_lag_sec"
        ]
        -
        group_disparity[
            "ci95_low"
        ],

        group_disparity[
            "ci95_high"
        ]
        -
        group_disparity[
            "mean_subject_median_lag_sec"
        ],
    ],
    marker="o",
    capsize=4,
)

ax.set_xlabel(
    "Audiovisual spatial disparity (deg)"
)

ax.set_ylabel(
    "Auditory-relative tracking lag (s)"
)

ax.set_title(
    "Tracking latency across fixed audiovisual disparity"
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "lag_by_fixed_disparity.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


# =============================================================================
# PLOT 2 — SAME VS DIFFERENT
# =============================================================================

group_judgment_rows = []

for (
    disparity,
    judgment
), g in subject_disparity_judgment.groupby(
    [
        "disparity_deg",
        "judgment_label",
    ]
):

    m, lo, hi, se, n = mean_ci_t(
        g[
            "median_lag_sec"
        ]
    )

    group_judgment_rows.append(
        {
            "disparity_deg": disparity,
            "judgment_label": judgment,
            "mean_lag_sec": m,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_subjects": n,
        }
    )


group_judgment = pd.DataFrame(
    group_judgment_rows
)

group_judgment.to_csv(
    os.path.join(
        CSV_DIR,
        "group_lag_by_disparity_and_judgment.csv",
    ),
    index=False,
)


fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)

for label in [
    "Same",
    "Different",
]:

    g = group_judgment[
        group_judgment[
            "judgment_label"
        ] == label
    ].sort_values(
        "disparity_deg"
    )

    if len(g) == 0:
        continue

    ax.errorbar(
        g[
            "disparity_deg"
        ],
        g[
            "mean_lag_sec"
        ],
        yerr=[
            g[
                "mean_lag_sec"
            ]
            -
            g[
                "ci95_low"
            ],

            g[
                "ci95_high"
            ]
            -
            g[
                "mean_lag_sec"
            ],
        ],
        marker="o",
        capsize=4,
        label=label,
    )

ax.set_xlabel(
    "Audiovisual spatial disparity (deg)"
)

ax.set_ylabel(
    "Auditory-relative tracking lag (s)"
)

ax.set_title(
    "Tracking latency by disparity and causal judgment"
)

ax.legend(
    title="Post-trial judgment"
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "lag_by_disparity_and_judgment.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


print("\n" + "=" * 90)
print("FIXED DISPARITY LAG ANALYSIS COMPLETE")
print("=" * 90)

print(
    "\nGroup lag by disparity:"
)
print(
    group_disparity.to_string(
        index=False
    )
)

print(
    "\nParticipant-first disparity slope:"
)
print(
    slope_result.to_string(
        index=False
    )
)

print(
    "\nDisparity × judgment group tests:"
)
print(
    group_model_results.to_string(
        index=False
    )
)

print(
    f"\nOutputs saved to:\n{OUT_DIR}"
)

Fixed valid lag trials: 2595
Subjects: 50

FIXED DISPARITY LAG ANALYSIS COMPLETE

Group lag by disparity:
 disparity_deg  mean_subject_median_lag_sec  ci95_low  ci95_high      sem  n_subjects  mean_valid_trials_per_subject
           0.0                       0.8230  0.630800   1.015200 0.095642          50                           9.14
          10.0                       0.7230  0.532145   0.913855 0.094973          50                          17.78
          20.0                       0.8915  0.704976   1.078024 0.092818          50                          16.22
          30.0                       1.2150  0.969671   1.460329 0.122080          50                           8.76

Participant-first disparity slope:
         effect  mean_sec_per_10deg  ci95_low  ci95_high        t  df  p_two_sided  p_signflip_greater  n_subjects
disparity_slope             0.13445   0.05758    0.21132 3.514867  49     0.000958              0.0005          50

Disparity × judgment group tests:
        

In [22]:
# %% COMPLEMENTARY LAG ANALYSIS 2 — AUDITORY LAG × CONTINUOUS VISUAL BIAS
# =============================================================================
# QUESTIONS
# =============================================================================
# 1. Does stronger visual bias predict SHORTER auditory-relative lag after
#    controlling for physical disparity?
#
#       Lag ~ VisualBias + Disparity
#
# 2. Does visual bias predict lag beyond explicit causal report?
#
#       Lag ~ VisualBias + P(same) + Disparity
#
# 3. Does explicit causal report predict lag beyond visual bias?
#
# This tests whether the temporal and spatial signatures of audiovisual
# integration carry overlapping versus unique information.
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_1samp
import statsmodels.formula.api as smf

# =============================================================================
# OUTPUT
# =============================================================================

OUT_DIR = os.path.join(
    COMPLEMENTARY_ROOT,
    "02_lag_visual_bias_relationship",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures",
)

for d in [
    OUT_DIR,
    CSV_DIR,
    FIG_DIR,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# =============================================================================
# REQUIRE SPATIAL BIAS
# =============================================================================

if "Capture_rmse_lagcorr" not in lag_master_df.columns:

    raise RuntimeError(
        "Capture_rmse_lagcorr was not found in lag_master_df.\n"
        "Run the shared preparation cell after first running the notebook's "
        "trial-level spatial metric analysis."
    )


# Fixed audiovisual trials
fixed_conditions = [
    "congruent",
    "fixed_10",
    "fixed_20",
    "fixed_30",
    "fixed_40",
]

df = lag_master_df[
    lag_master_df[
        "physical_condition"
    ].isin(
        fixed_conditions
    )
].copy()

df["visual_bias_deg"] = pd.to_numeric(
    df[
        "Capture_rmse_lagcorr"
    ],
    errors="coerce",
)

df["disparity_10"] = (
    pd.to_numeric(
        df[
            "disparity_deg"
        ],
        errors="coerce",
    )
    / 10.0
)

df = df.dropna(
    subset=[
        "subject",
        "lag_audio_sec",
        "visual_bias_deg",
        "disparity_10",
    ]
)

print(
    f"Valid lag + visual-bias trials: {len(df)}"
)

print(
    f"Subjects: {df['subject'].nunique()}"
)


# =============================================================================
# CENTER VISUAL BIAS WITHIN PARTICIPANT
#
# This makes the primary coefficient predominantly a within-observer effect:
# on trials where an observer is MORE visually biased than usual, are they
# also faster than usual?
# =============================================================================

df[
    "visual_bias_subject_mean"
] = (
    df
    .groupby(
        "subject"
    )[
        "visual_bias_deg"
    ]
    .transform(
        "mean"
    )
)

df[
    "visual_bias_within"
] = (
    df[
        "visual_bias_deg"
    ]
    -
    df[
        "visual_bias_subject_mean"
    ]
)

df[
    "disparity_subject_mean"
] = (
    df
    .groupby(
        "subject"
    )[
        "disparity_10"
    ]
    .transform(
        "mean"
    )
)

df[
    "disparity_within"
] = (
    df[
        "disparity_10"
    ]
    -
    df[
        "disparity_subject_mean"
    ]
)

df.to_csv(
    os.path.join(
        CSV_DIR,
        "lag_visual_bias_model_trials.csv",
    ),
    index=False,
)


# =============================================================================
# MODEL A — VISUAL BIAS + DISPARITY
# =============================================================================

model_a_df = df.copy()

model_a = smf.mixedlm(
    (
        "lag_audio_sec ~ "
        "visual_bias_within + "
        "visual_bias_subject_mean + "
        "disparity_10"
    ),
    data=model_a_df,
    groups=model_a_df[
        "subject"
    ],
)

try:

    fit_a = model_a.fit(
        reml=False,
        method="lbfgs",
    )

except Exception:

    fit_a = model_a.fit(
        reml=False,
        method="powell",
    )


# =============================================================================
# MODEL B — VISUAL BIAS + EXPLICIT JUDGMENT + DISPARITY
# =============================================================================

model_b_df = df.dropna(
    subset=[
        "same_report",
    ]
).copy()

model_b = smf.mixedlm(
    (
        "lag_audio_sec ~ "
        "visual_bias_within + "
        "visual_bias_subject_mean + "
        "same_report + "
        "disparity_10"
    ),
    data=model_b_df,
    groups=model_b_df[
        "subject"
    ],
)

try:

    fit_b = model_b.fit(
        reml=False,
        method="lbfgs",
    )

except Exception:

    fit_b = model_b.fit(
        reml=False,
        method="powell",
    )


# =============================================================================
# SAVE MIXED-MODEL SUMMARIES
# =============================================================================

with open(
    os.path.join(
        CSV_DIR,
        "mixed_model_A_visual_bias_plus_disparity.txt",
    ),
    "w",
) as f:

    f.write(
        fit_a.summary().as_text()
    )


with open(
    os.path.join(
        CSV_DIR,
        "mixed_model_B_visual_bias_judgment_disparity.txt",
    ),
    "w",
) as f:

    f.write(
        fit_b.summary().as_text()
    )


def model_coeff_table(
    fit,
    model_name,
):

    ci = fit.conf_int()

    rows = []

    for term in fit.params.index:

        if term == "Group Var":
            continue

        rows.append(
            {
                "model": model_name,
                "term": term,
                "estimate": fit.params[
                    term
                ],
                "se": fit.bse.get(
                    term,
                    np.nan,
                ),
                "z": fit.tvalues.get(
                    term,
                    np.nan,
                ),
                "p": fit.pvalues.get(
                    term,
                    np.nan,
                ),
                "ci95_low": (
                    ci.loc[
                        term,
                        0,
                    ]
                    if term in ci.index
                    else np.nan
                ),
                "ci95_high": (
                    ci.loc[
                        term,
                        1,
                    ]
                    if term in ci.index
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


mixed_coeffs = pd.concat(
    [
        model_coeff_table(
            fit_a,
            "A_bias_plus_disparity",
        ),
        model_coeff_table(
            fit_b,
            "B_bias_plus_judgment_plus_disparity",
        ),
    ],
    ignore_index=True,
)

mixed_coeffs.to_csv(
    os.path.join(
        CSV_DIR,
        "mixed_model_coefficients.csv",
    ),
    index=False,
)


# =============================================================================
# PARTICIPANT-FIRST ROBUSTNESS
#
# Trial-level OLS within each observer:
#
#   Lag ~ VisualBias + Disparity
#
# and
#
#   Lag ~ VisualBias + Same + Disparity
# =============================================================================

subject_rows = []

for subject, g in df.groupby(
    "subject"
):

    if len(g) < 10:
        continue

    # Standardize within participant to improve interpretability.
    sg = g.copy()

    vb_sd = sg[
        "visual_bias_deg"
    ].std(
        ddof=1
    )

    if (
        not np.isfinite(vb_sd)
        or vb_sd <= 0
    ):
        continue

    sg[
        "visual_bias_z"
    ] = (
        sg[
            "visual_bias_deg"
        ]
        -
        sg[
            "visual_bias_deg"
        ].mean()
    ) / vb_sd

    # Model A
    try:

        ma = smf.ols(
            (
                "lag_audio_sec ~ "
                "visual_bias_z + disparity_10"
            ),
            data=sg,
        ).fit()

        bias_a = ma.params.get(
            "visual_bias_z",
            np.nan,
        )

        disp_a = ma.params.get(
            "disparity_10",
            np.nan,
        )

    except Exception:

        bias_a = np.nan
        disp_a = np.nan


    # Model B
    sg_b = sg.dropna(
        subset=[
            "same_report",
        ]
    )

    if (
        len(sg_b) >= 10
        and sg_b[
            "same_report"
        ].nunique() >= 2
    ):

        try:

            mb = smf.ols(
                (
                    "lag_audio_sec ~ "
                    "visual_bias_z + "
                    "same_report + "
                    "disparity_10"
                ),
                data=sg_b,
            ).fit()

            bias_b = mb.params.get(
                "visual_bias_z",
                np.nan,
            )

            same_b = mb.params.get(
                "same_report",
                np.nan,
            )

            disp_b = mb.params.get(
                "disparity_10",
                np.nan,
            )

        except Exception:

            bias_b = np.nan
            same_b = np.nan
            disp_b = np.nan

    else:

        bias_b = np.nan
        same_b = np.nan
        disp_b = np.nan


    subject_rows.append(
        {
            "subject": subject,

            "modelA_visual_bias_slope_sec_per_1SD":
                bias_a,

            "modelA_disparity_slope_sec_per_10deg":
                disp_a,

            "modelB_visual_bias_slope_sec_per_1SD":
                bias_b,

            "modelB_same_report_effect_sec":
                same_b,

            "modelB_disparity_slope_sec_per_10deg":
                disp_b,
        }
    )


subject_coeffs = pd.DataFrame(
    subject_rows
)

subject_coeffs.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_first_lag_visual_bias_coefficients.csv",
    ),
    index=False,
)


# =============================================================================
# GROUP TESTS
# =============================================================================

tests = [
    (
        "modelA_visual_bias_slope_sec_per_1SD",
        "less",
        "Greater visual bias predicts shorter lag",
    ),
    (
        "modelB_visual_bias_slope_sec_per_1SD",
        "less",
        "Visual bias predicts shorter lag beyond causal report",
    ),
    (
        "modelB_same_report_effect_sec",
        "less",
        "Same-source report predicts shorter lag beyond visual bias",
    ),
]

result_rows = []

for col, direction, interpretation in tests:

    vals = (
        subject_coeffs[
            col
        ]
        .dropna()
        .values
    )

    if len(vals) < 2:
        continue

    t_result = ttest_1samp(
        vals,
        0.0,
    )

    m, lo, hi, se, n = mean_ci_t(
        vals
    )

    p_perm = sign_flip_test(
        vals,
        alternative=direction,
    )

    result_rows.append(
        {
            "effect": col,
            "prediction": interpretation,
            "mean_coefficient": m,
            "ci95_low": lo,
            "ci95_high": hi,
            "t": t_result.statistic,
            "df": n - 1,
            "p_two_sided": t_result.pvalue,
            "signflip_alternative": direction,
            "p_signflip": p_perm,
            "n_subjects": n,
        }
    )


group_tests = pd.DataFrame(
    result_rows
)

group_tests.to_csv(
    os.path.join(
        CSV_DIR,
        "group_subject_first_lag_visual_bias_tests.csv",
    ),
    index=False,
)


# =============================================================================
# PLOT — VISUAL BIAS BINNED FOR DISPLAY ONLY
# =============================================================================

plot_df = df.copy()

plot_df[
    "visual_bias_bin"
] = pd.qcut(
    plot_df[
        "visual_bias_deg"
    ],
    q=5,
    duplicates="drop",
)

subject_bin = (
    plot_df
    .groupby(
        [
            "subject",
            "visual_bias_bin",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        lag_sec=(
            "lag_audio_sec",
            "median",
        ),
        visual_bias_deg=(
            "visual_bias_deg",
            "mean",
        ),
    )
)

group_bin_rows = []

for bin_label, g in subject_bin.groupby(
    "visual_bias_bin",
    observed=True,
):

    x = g[
        "visual_bias_deg"
    ].mean()

    y, lo, hi, se, n = mean_ci_t(
        g[
            "lag_sec"
        ]
    )

    group_bin_rows.append(
        {
            "bin": str(
                bin_label
            ),
            "mean_visual_bias_deg": x,
            "mean_lag_sec": y,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_subjects": n,
        }
    )


group_bins = pd.DataFrame(
    group_bin_rows
).sort_values(
    "mean_visual_bias_deg"
)

group_bins.to_csv(
    os.path.join(
        CSV_DIR,
        "group_lag_by_visual_bias_quantile.csv",
    ),
    index=False,
)


fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)

ax.errorbar(
    group_bins[
        "mean_visual_bias_deg"
    ],
    group_bins[
        "mean_lag_sec"
    ],
    yerr=[
        group_bins[
            "mean_lag_sec"
        ]
        -
        group_bins[
            "ci95_low"
        ],

        group_bins[
            "ci95_high"
        ]
        -
        group_bins[
            "mean_lag_sec"
        ],
    ],
    marker="o",
    capsize=4,
)

ax.set_xlabel(
    "Lag-corrected visual bias (deg)"
)

ax.set_ylabel(
    "Auditory-relative tracking lag (s)"
)

ax.set_title(
    "Temporal tracking latency across continuous visual weighting"
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "lag_by_visual_bias_quantile.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


print("\n" + "=" * 90)
print("LAG × VISUAL BIAS ANALYSIS COMPLETE")
print("=" * 90)

print(
    "\nMixed-model coefficients:"
)
print(
    mixed_coeffs.to_string(
        index=False
    )
)

print(
    "\nParticipant-first group tests:"
)
print(
    group_tests.to_string(
        index=False
    )
)

print(
    f"\nOutputs saved to:\n{OUT_DIR}"
)

Valid lag + visual-bias trials: 2595
Subjects: 50


C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pke


LAG × VISUAL BIAS ANALYSIS COMPLETE

Mixed-model coefficients:
                              model                     term      estimate       se          z            p  ci95_low  ci95_high
              A_bias_plus_disparity                Intercept -1.795989e-16      NaN        NaN          NaN       NaN        NaN
              A_bias_plus_disparity       visual_bias_within -4.005847e-02 0.002395 -16.727457 8.269413e-63 -0.044752  -0.035365
              A_bias_plus_disparity visual_bias_subject_mean -4.822567e-16      NaN        NaN          NaN       NaN        NaN
              A_bias_plus_disparity             disparity_10  2.951632e-01 0.021905  13.474718 2.203263e-41  0.252230   0.338096
B_bias_plus_judgment_plus_disparity                Intercept  3.156997e-16      NaN        NaN          NaN       NaN        NaN
B_bias_plus_judgment_plus_disparity       visual_bias_within -3.402195e-02 0.002744 -12.397574 2.693569e-35 -0.039401  -0.028643
B_bias_plus_judgment_plus_dispari

In [ ]:
# %% COMPLEMENTARY LAG ANALYSIS 3 — TEMPORAL LAG HYSTERESIS DURING SWEEPS
# =============================================================================
# QUESTION
# =============================================================================
# At the SAME instantaneous audiovisual disparity, does auditory-relative
# tracking lag differ depending on whether that disparity was reached by:
#
#   Divergence: initially aligned -> separating
#   Convergence: initially separated -> approaching
#
# =============================================================================
# REVISED MATCHING LOGIC
# =============================================================================
# This version:
#
#   1. uses exact matched disparity targets:
#         5, 10, 15, 20, 25, 30, 35 degrees
#
#   2. calculates the ACTUAL disparity range traversed by every sweep trial
#      during the dynamic 12–18 s period;
#
#   3. determines whether each exact target is genuinely contained within
#      that observed trial-specific disparity range;
#
#   4. interpolates the exact time at which the sweep crosses each target;
#
#   5. estimates local auditory-relative lag in a 4-s window centered on
#      that crossing;
#
#   6. does NOT exclude observations because of peak correlation or
#      auditory-position SD;
#
#   7. saves detailed diagnostics distinguishing:
#         - target disparity genuinely not reached
#         - lag mathematically not estimable
#
# This avoids treating a 4-s mean disparity as the instantaneous disparity
# and avoids the previous +/-2.5-degree nearest-point rule.
# =============================================================================


import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, ttest_1samp


# =============================================================================
# OUTPUT
# =============================================================================

OUT_DIR = os.path.join(
    COMPLEMENTARY_ROOT,
    "03_temporal_lag_hysteresis",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures",
)

for d in [
    OUT_DIR,
    CSV_DIR,
    FIG_DIR,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# =============================================================================
# SETTINGS
# =============================================================================

RESAMPLE_HZ_LOCAL = 20.0
DT_LOCAL = 1.0 / RESAMPLE_HZ_LOCAL

TIME_GRID_LOCAL = np.arange(
    0.0,
    18.0 + DT_LOCAL,
    DT_LOCAL,
)

WINDOW_SEC = 4.0

MAX_LOCAL_LAG_SEC = 2.0

MAX_LOCAL_LAG_SAMPLES = int(
    round(
        MAX_LOCAL_LAG_SEC
        * RESAMPLE_HZ_LOCAL
    )
)

# Mathematical minimum only.
# This is NOT a behavioral QC exclusion.
MIN_LOCAL_VALID_SAMPLES = 20

RAMP_CHANGE_TIME_SEC = 12.0
TRIAL_END_SEC = 18.0


# =============================================================================
# EXACT MATCHED DISPARITIES
# =============================================================================

MATCHED_DISPARITIES = np.array(
    [
        5.0,
        10.0,
        15.0,
        20.0,
        25.0,
        30.0,
        35.0,
    ],
    dtype=float,
)

MIN_MATCHED_DISPARITY = float(
    MATCHED_DISPARITIES.min()
)

MAX_MATCHED_DISPARITY = float(
    MATCHED_DISPARITIES.max()
)


print(
    "\nExact matched disparities:",
    MATCHED_DISPARITIES,
)


# =============================================================================
# HELPERS
# =============================================================================

def resample_sweep_trial(df_trial):

    d = (
        df_trial
        .sort_values("t")
        .drop_duplicates(
            subset="t",
            keep="first",
        )
        .copy()
    )

    if len(d) < 5:
        return None

    t = pd.to_numeric(
        d["t"],
        errors="coerce",
    ).values.astype(float)

    if np.sum(np.isfinite(t)) < 3:
        return None

    # Normalize trial time to start at zero.
    t = t - np.nanmin(t)

    def interp_col(col):

        y = pd.to_numeric(
            d[col],
            errors="coerce",
        ).values.astype(float)

        ok = (
            np.isfinite(t)
            &
            np.isfinite(y)
        )

        if ok.sum() < 3:

            return np.full(
                len(TIME_GRID_LOCAL),
                np.nan,
            )

        return np.interp(
            TIME_GRID_LOCAL,
            t[ok],
            y[ok],
            left=np.nan,
            right=np.nan,
        )

    return {
        "t": TIME_GRID_LOCAL.copy(),
        "mouse": interp_col("mouse_x"),
        "audio": interp_col("audio_x"),
        "visual": interp_col("blob_x"),
    }


def local_positive_lag(
    response,
    stimulus,
):
    """
    Estimate auditory-relative response lag within one local window.

    Searches non-negative lags from 0 to MAX_LOCAL_LAG_SEC.

    IMPORTANT:
    No peak-r threshold and no auditory-SD threshold are imposed.

    A lag is missing only if Pearson correlation cannot be estimated
    for any candidate lag.
    """

    response = np.asarray(
        response,
        dtype=float,
    )

    stimulus = np.asarray(
        stimulus,
        dtype=float,
    )

    best_r = -np.inf
    best_lag = np.nan

    for lag_samples in range(
        0,
        MAX_LOCAL_LAG_SAMPLES + 1,
    ):

        if lag_samples == 0:

            r_resp = response
            r_stim = stimulus

        else:

            r_resp = response[
                lag_samples:
            ]

            r_stim = stimulus[
                :-lag_samples
            ]

        ok = (
            np.isfinite(r_resp)
            &
            np.isfinite(r_stim)
        )

        if ok.sum() < MIN_LOCAL_VALID_SAMPLES:
            continue

        resp_sd = np.nanstd(
            r_resp[ok]
        )

        stim_sd = np.nanstd(
            r_stim[ok]
        )

        # Pearson r is undefined for constant signals.
        # This is mathematical necessity, not behavioral QC.
        if (
            resp_sd <= 0
            or
            stim_sd <= 0
        ):
            continue

        r = pearsonr(
            r_resp[ok],
            r_stim[ok],
        )[0]

        if (
            np.isfinite(r)
            and
            r > best_r
        ):

            best_r = r

            best_lag = (
                lag_samples
                /
                RESAMPLE_HZ_LOCAL
            )

    if not np.isfinite(best_lag):

        return (
            np.nan,
            np.nan,
        )

    return (
        best_lag,
        best_r,
    )


def find_exact_crossing_time(
    t,
    disparity,
    target,
    direction,
):
    """
    Estimate the time at which the observed AV disparity crosses an exact
    target value.

    For Diverge, the preferred crossing is upward:
        disparity[i] <= target <= disparity[i+1]

    For Converge, the preferred crossing is downward:
        disparity[i] >= target >= disparity[i+1]

    Linear interpolation is used between the two surrounding samples.

    If small trajectory irregularities prevent finding the expected
    directional crossing, a generic sign-change crossing is attempted.

    Returns:
        crossing_time
        crossing_method
    """

    t = np.asarray(
        t,
        dtype=float,
    )

    disparity = np.asarray(
        disparity,
        dtype=float,
    )

    ok = (
        np.isfinite(t)
        &
        np.isfinite(disparity)
    )

    t = t[ok]
    disparity = disparity[ok]

    if len(t) < 2:

        return (
            np.nan,
            "insufficient_samples",
        )


    # -------------------------------------------------------------------------
    # Exact sampled hit
    # -------------------------------------------------------------------------

    exact_idx = np.where(
        np.isclose(
            disparity,
            target,
            atol=1e-10,
        )
    )[0]

    if len(exact_idx) > 0:

        # If multiple exact hits exist, use the first during divergence
        # and the last during convergence to respect sweep direction.
        if direction == "Diverge":
            idx = exact_idx[0]
        else:
            idx = exact_idx[-1]

        return (
            float(t[idx]),
            "exact_sample",
        )


    # -------------------------------------------------------------------------
    # Preferred directional crossing
    # -------------------------------------------------------------------------

    candidate_crossings = []

    for i in range(
        len(disparity) - 1
    ):

        d0 = disparity[i]
        d1 = disparity[i + 1]

        t0 = t[i]
        t1 = t[i + 1]

        if direction == "Diverge":

            crossed = (
                d0 <= target
                and
                d1 >= target
            )

        else:

            crossed = (
                d0 >= target
                and
                d1 <= target
            )

        if not crossed:
            continue

        if d1 == d0:

            crossing_time = (
                t0 + t1
            ) / 2.0

        else:

            frac = (
                target - d0
            ) / (
                d1 - d0
            )

            crossing_time = (
                t0
                +
                frac
                *
                (
                    t1 - t0
                )
            )

        candidate_crossings.append(
            crossing_time
        )


    if len(candidate_crossings) > 0:

        if direction == "Diverge":

            crossing_time = np.min(
                candidate_crossings
            )

        else:

            crossing_time = np.max(
                candidate_crossings
            )

        return (
            float(crossing_time),
            "directional_interpolation",
        )


    # -------------------------------------------------------------------------
    # Generic fallback crossing
    # -------------------------------------------------------------------------
    # This protects against tiny local reversals/noise in the observed
    # disparity trajectory.

    generic_crossings = []

    centered = (
        disparity
        -
        target
    )

    for i in range(
        len(centered) - 1
    ):

        y0 = centered[i]
        y1 = centered[i + 1]

        if (
            y0 == 0
            or
            y1 == 0
            or
            y0 * y1 < 0
        ):

            d0 = disparity[i]
            d1 = disparity[i + 1]

            t0 = t[i]
            t1 = t[i + 1]

            if d1 == d0:

                crossing_time = (
                    t0 + t1
                ) / 2.0

            else:

                frac = (
                    target - d0
                ) / (
                    d1 - d0
                )

                crossing_time = (
                    t0
                    +
                    frac
                    *
                    (
                        t1 - t0
                    )
                )

            generic_crossings.append(
                crossing_time
            )


    if len(generic_crossings) > 0:

        if direction == "Diverge":

            crossing_time = np.min(
                generic_crossings
            )

        else:

            crossing_time = np.max(
                generic_crossings
            )

        return (
            float(crossing_time),
            "generic_interpolation",
        )


    return (
        np.nan,
        "crossing_not_found",
    )


# =============================================================================
# LOAD SWEEP TRIALS
# =============================================================================

files = sorted(
    glob.glob(
        os.path.join(
            DATA_DIR,
            FILE_PATTERN,
        )
    )
)

window_rows = []
trial_diagnostic_rows = []


for fpath in files:

    subject = extract_subject_id(
        fpath
    )

    raw = pd.read_csv(
        fpath
    )

    needed = [
        "trialIndex",
        "trialType",
        "condLabel",
        "t",
        "mouse_x",
        "audio_x",
        "blob_x",
    ]

    missing = [
        c
        for c in needed
        if c not in raw.columns
    ]

    if missing:

        print(
            f"Skipping {fpath}; missing columns:",
            missing,
        )

        continue


    # -------------------------------------------------------------------------
    # Remove practice trials
    # -------------------------------------------------------------------------

    if "isPractice" in raw.columns:

        raw = raw[
            pd.to_numeric(
                raw["isPractice"],
                errors="coerce",
            )
            .fillna(0)
            == 0
        ]


    # -------------------------------------------------------------------------
    # Sweep trials only
    # -------------------------------------------------------------------------

    sweeps = raw[
        raw["trialType"]
        .astype(str)
        .str.lower()
        .eq(
            "incongruent_sweep"
        )
    ].copy()


    for (
        trial_index,
        cond_label
    ), trial in sweeps.groupby(
        [
            "trialIndex",
            "condLabel",
        ],
        sort=False,
    ):

        label = str(
            cond_label
        ).lower()

        if "diverge" in label:

            direction = "Diverge"

        elif "converge" in label:

            direction = "Converge"

        else:

            continue


        rs = resample_sweep_trial(
            trial
        )

        if rs is None:

            trial_diagnostic_rows.append(
                {
                    "subject":
                        subject,

                    "trialIndex":
                        trial_index,

                    "condLabel":
                        cond_label,

                    "direction":
                        direction,

                    "trial_resampled":
                        False,

                    "ramp_min_disparity_deg":
                        np.nan,

                    "ramp_max_disparity_deg":
                        np.nan,

                    "ramp_start_disparity_deg":
                        np.nan,

                    "ramp_end_disparity_deg":
                        np.nan,

                    "ramp_disparity_change_deg":
                        np.nan,
                }
            )

            continue


        t = rs["t"]
        mouse = rs["mouse"]
        audio = rs["audio"]
        visual = rs["visual"]


        # ---------------------------------------------------------------------
        # Instantaneous absolute AV disparity
        # -------------------------------------------------------------------------

        instantaneous_disparity = np.abs(
            visual
            -
            audio
        )


        # ---------------------------------------------------------------------
        # Dynamic sweep portion only
        # -------------------------------------------------------------------------

        ramp_mask = (
            (t >= RAMP_CHANGE_TIME_SEC)
            &
            (t <= TRIAL_END_SEC)
            &
            np.isfinite(
                instantaneous_disparity
            )
        )

        ramp_idx = np.where(
            ramp_mask
        )[0]

        if len(ramp_idx) < 2:

            trial_diagnostic_rows.append(
                {
                    "subject":
                        subject,

                    "trialIndex":
                        trial_index,

                    "condLabel":
                        cond_label,

                    "direction":
                        direction,

                    "trial_resampled":
                        True,

                    "ramp_min_disparity_deg":
                        np.nan,

                    "ramp_max_disparity_deg":
                        np.nan,

                    "ramp_start_disparity_deg":
                        np.nan,

                    "ramp_end_disparity_deg":
                        np.nan,

                    "ramp_disparity_change_deg":
                        np.nan,
                }
            )

            continue


        ramp_t = t[
            ramp_idx
        ]

        ramp_disparity = (
            instantaneous_disparity[
                ramp_idx
            ]
        )


        # ---------------------------------------------------------------------
        # ACTUAL disparity range traversed by this trial
        # -------------------------------------------------------------------------

        ramp_min = np.nanmin(
            ramp_disparity
        )

        ramp_max = np.nanmax(
            ramp_disparity
        )

        ramp_start = ramp_disparity[
            0
        ]

        ramp_end = ramp_disparity[
            -1
        ]

        ramp_change = (
            ramp_end
            -
            ramp_start
        )


        trial_diagnostic_rows.append(
            {
                "subject":
                    subject,

                "trialIndex":
                    trial_index,

                "condLabel":
                    cond_label,

                "direction":
                    direction,

                "trial_resampled":
                    True,

                "ramp_min_disparity_deg":
                    ramp_min,

                "ramp_max_disparity_deg":
                    ramp_max,

                "ramp_start_disparity_deg":
                    ramp_start,

                "ramp_end_disparity_deg":
                    ramp_end,

                "ramp_disparity_change_deg":
                    ramp_change,
            }
        )


        # =====================================================================
        # ONE LOCAL-LAG ESTIMATE PER EXACT TARGET DISPARITY
        # =====================================================================

        for target_disparity in MATCHED_DISPARITIES:


            # -----------------------------------------------------------------
            # First determine whether this trial ACTUALLY traversed the target.
            #
            # No arbitrary +/-2.5-degree tolerance is used.
            # -----------------------------------------------------------------

            target_in_observed_range = (
                target_disparity >= ramp_min
                and
                target_disparity <= ramp_max
            )


            if not target_in_observed_range:

                window_rows.append(
                    {
                        "subject":
                            subject,

                        "trialIndex":
                            trial_index,

                        "condLabel":
                            cond_label,

                        "direction":
                            direction,

                        "target_disparity_deg":
                            target_disparity,

                        "trial_ramp_min_disparity_deg":
                            ramp_min,

                        "trial_ramp_max_disparity_deg":
                            ramp_max,

                        "target_in_observed_range":
                            False,

                        "crossing_method":
                            "not_in_observed_range",

                        "window_center_sec":
                            np.nan,

                        "window_start_sec":
                            np.nan,

                        "window_end_sec":
                            np.nan,

                        "instantaneous_disparity_deg":
                            np.nan,

                        "window_mean_disparity_deg":
                            np.nan,

                        "audio_sd_deg":
                            np.nan,

                        "lag_audio_sec":
                            np.nan,

                        "peak_audio_corr":
                            np.nan,

                        "lag_estimable":
                            False,

                        "failure_reason":
                            "target_disparity_not_reached",
                    }
                )

                continue


            # -----------------------------------------------------------------
            # Find exact crossing time by interpolation
            # -----------------------------------------------------------------

            center_time, crossing_method = (
                find_exact_crossing_time(
                    ramp_t,
                    ramp_disparity,
                    target_disparity,
                    direction,
                )
            )


            if not np.isfinite(
                center_time
            ):

                window_rows.append(
                    {
                        "subject":
                            subject,

                        "trialIndex":
                            trial_index,

                        "condLabel":
                            cond_label,

                        "direction":
                            direction,

                        "target_disparity_deg":
                            target_disparity,

                        "trial_ramp_min_disparity_deg":
                            ramp_min,

                        "trial_ramp_max_disparity_deg":
                            ramp_max,

                        "target_in_observed_range":
                            True,

                        "crossing_method":
                            crossing_method,

                        "window_center_sec":
                            np.nan,

                        "window_start_sec":
                            np.nan,

                        "window_end_sec":
                            np.nan,

                        "instantaneous_disparity_deg":
                            target_disparity,

                        "window_mean_disparity_deg":
                            np.nan,

                        "audio_sd_deg":
                            np.nan,

                        "lag_audio_sec":
                            np.nan,

                        "peak_audio_corr":
                            np.nan,

                        "lag_estimable":
                            False,

                        "failure_reason":
                            "crossing_time_not_found",
                    }
                )

                continue


            # -----------------------------------------------------------------
            # Construct 4-s lag-estimation window centered on exact crossing.
            #
            # The window is clipped at trial boundaries rather than discarded.
            # -----------------------------------------------------------------

            start = max(
                0.0,
                center_time
                -
                WINDOW_SEC / 2.0,
            )

            end = min(
                TRIAL_END_SEC,
                center_time
                +
                WINDOW_SEC / 2.0,
            )


            idx = np.where(
                (t >= start)
                &
                (t <= end)
            )[0]


            local_mouse = mouse[
                idx
            ]

            local_audio = audio[
                idx
            ]

            local_visual = visual[
                idx
            ]


            # -----------------------------------------------------------------
            # Diagnostics only
            # -----------------------------------------------------------------

            audio_sd = np.nanstd(
                local_audio
            )

            window_mean_disparity = np.nanmean(
                np.abs(
                    local_visual
                    -
                    local_audio
                )
            )


            # -----------------------------------------------------------------
            # Local auditory-relative lag
            # -----------------------------------------------------------------

            lag_sec, peak_r = local_positive_lag(
                local_mouse,
                local_audio,
            )


            lag_estimable = (
                np.isfinite(
                    lag_sec
                )
                and
                np.isfinite(
                    peak_r
                )
            )


            window_rows.append(
                {
                    "subject":
                        subject,

                    "trialIndex":
                        trial_index,

                    "condLabel":
                        cond_label,

                    "direction":
                        direction,

                    "target_disparity_deg":
                        target_disparity,

                    "trial_ramp_min_disparity_deg":
                        ramp_min,

                    "trial_ramp_max_disparity_deg":
                        ramp_max,

                    "target_in_observed_range":
                        True,

                    "crossing_method":
                        crossing_method,

                    "window_center_sec":
                        center_time,

                    "window_start_sec":
                        start,

                    "window_end_sec":
                        end,

                    # Exact matched disparity by construction.
                    "instantaneous_disparity_deg":
                        target_disparity,

                    # Diagnostic only; NOT used for matching.
                    "window_mean_disparity_deg":
                        window_mean_disparity,

                    # Diagnostic only; NOT used as exclusion.
                    "audio_sd_deg":
                        audio_sd,

                    "lag_audio_sec":
                        lag_sec,

                    # Diagnostic only; NOT used as exclusion.
                    "peak_audio_corr":
                        peak_r,

                    "lag_estimable":
                        lag_estimable,

                    "failure_reason":
                        (
                            ""
                            if lag_estimable
                            else
                            "lag_not_estimable"
                        ),
                }
            )


# =============================================================================
# CREATE DATAFRAMES
# =============================================================================

window_df = pd.DataFrame(
    window_rows
)

trial_diagnostics = pd.DataFrame(
    trial_diagnostic_rows
)


# =============================================================================
# SAVE TRIAL-RANGE DIAGNOSTICS
# =============================================================================

trial_diagnostics.to_csv(
    os.path.join(
        CSV_DIR,
        "sweep_trial_observed_disparity_ranges.csv",
    ),
    index=False,
)


window_df.to_csv(
    os.path.join(
        CSV_DIR,
        "all_local_sweep_lag_windows.csv",
    ),
    index=False,
)


# =============================================================================
# PRIMARY ANALYSIS DATA
# =============================================================================
# No peak-r or auditory-SD filtering.
#
# We retain every lag that is mathematically estimable.

valid = window_df[
    window_df[
        "lag_estimable"
    ] == True
].copy()


# =============================================================================
# DIAGNOSTIC 1 — OBSERVED SWEEP RANGES
# =============================================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "OBSERVED SWEEP DISPARITY RANGES"
)

print(
    "=" * 90
)


range_summary = (
    trial_diagnostics[
        trial_diagnostics[
            "trial_resampled"
        ] == True
    ]
    .groupby(
        "direction"
    )
    .agg(
        n_trials=(
            "trialIndex",
            "count",
        ),

        mean_min_disparity=(
            "ramp_min_disparity_deg",
            "mean",
        ),

        median_min_disparity=(
            "ramp_min_disparity_deg",
            "median",
        ),

        mean_max_disparity=(
            "ramp_max_disparity_deg",
            "mean",
        ),

        median_max_disparity=(
            "ramp_max_disparity_deg",
            "median",
        ),

        mean_start_disparity=(
            "ramp_start_disparity_deg",
            "mean",
        ),

        mean_end_disparity=(
            "ramp_end_disparity_deg",
            "mean",
        ),
    )
)


print(
    range_summary.to_string()
)


# =============================================================================
# DIAGNOSTIC 2 — HOW MANY TRIALS ACTUALLY REACH EACH TARGET?
# =============================================================================

target_reach = (
    window_df
    .groupby(
        [
            "direction",
            "target_disparity_deg",
        ],
        as_index=False,
    )
    .agg(
        n_expected=(
            "trialIndex",
            "count",
        ),

        n_target_reached=(
            "target_in_observed_range",
            "sum",
        ),

        n_lag_estimable=(
            "lag_estimable",
            "sum",
        ),
    )
)


target_reach[
    "prop_target_reached"
] = (
    target_reach[
        "n_target_reached"
    ]
    /
    target_reach[
        "n_expected"
    ]
)


target_reach[
    "prop_lag_estimable"
] = (
    target_reach[
        "n_lag_estimable"
    ]
    /
    target_reach[
        "n_expected"
    ]
)


target_reach.to_csv(
    os.path.join(
        CSV_DIR,
        "target_disparity_reach_diagnostics.csv",
    ),
    index=False,
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "TARGET DISPARITY REACH DIAGNOSTICS"
)

print(
    "=" * 90
)

print(
    target_reach.to_string(
        index=False
    )
)


# =============================================================================
# DIAGNOSTIC 3 — FAILURE REASONS
# =============================================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "FAILURE REASONS"
)

print(
    "=" * 90
)


print(
    window_df[
        "failure_reason"
    ]
    .replace(
        "",
        "none"
    )
    .value_counts(
        dropna=False
    )
)


# =============================================================================
# MEDIAN WITHIN SUBJECT × DIRECTION × EXACT DISPARITY
# =============================================================================

subject_bin = (
    valid
    .groupby(
        [
            "subject",
            "direction",
            "target_disparity_deg",
        ],
        as_index=False,
    )
    .agg(
        median_lag_sec=(
            "lag_audio_sec",
            "median",
        ),

        n_lag_estimates=(
            "lag_audio_sec",
            "count",
        ),

        mean_peak_r=(
            "peak_audio_corr",
            "mean",
        ),

        mean_audio_sd_deg=(
            "audio_sd_deg",
            "mean",
        ),

        mean_window_disparity_deg=(
            "window_mean_disparity_deg",
            "mean",
        ),
    )
)


subject_bin.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_direction_disparity_local_lags.csv",
    ),
    index=False,
)


# =============================================================================
# MATCH DIVERGE AND CONVERGE WITHIN PARTICIPANT × EXACT DISPARITY
# =============================================================================

wide = subject_bin.pivot_table(
    index=[
        "subject",
        "target_disparity_deg",
    ],
    columns="direction",
    values="median_lag_sec",
)


wide = wide.reset_index()

wide.columns.name = None


if (
    "Diverge" not in wide.columns
    or
    "Converge" not in wide.columns
):

    raise RuntimeError(
        "Could not find both Diverge and Converge "
        "local-lag estimates."
    )


wide[
    "lag_hysteresis_converge_minus_diverge_sec"
] = (
    wide[
        "Converge"
    ]
    -
    wide[
        "Diverge"
    ]
)


wide[
    "matched_both_directions"
] = (
    wide[
        "Diverge"
    ].notna()
    &
    wide[
        "Converge"
    ].notna()
)


wide.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_bin_temporal_hysteresis.csv",
    ),
    index=False,
)


# =============================================================================
# IMPORTANT:
# PRIMARY HYSTERESIS INDEX USES ONLY EXACT DISPARITIES FOR WHICH THE
# PARTICIPANT CONTRIBUTES BOTH DIVERGE AND CONVERGE.
# =============================================================================

matched_wide = wide[
    wide[
        "matched_both_directions"
    ]
].copy()


# =============================================================================
# SUBJECT-LEVEL HYSTERESIS
# =============================================================================

subject_hysteresis = (
    matched_wide
    .groupby(
        "subject",
        as_index=False,
    )
    .agg(
        temporal_hysteresis_sec=(
            "lag_hysteresis_converge_minus_diverge_sec",
            "mean",
        ),

        n_matched_disparities=(
            "lag_hysteresis_converge_minus_diverge_sec",
            "count",
        ),
    )
)


subject_hysteresis.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_temporal_hysteresis_index.csv",
    ),
    index=False,
)


# =============================================================================
# GROUP INFERENCE
# =============================================================================

vals = subject_hysteresis[
    "temporal_hysteresis_sec"
].dropna().values


if len(vals) >= 2:

    t_result = ttest_1samp(
        vals,
        0.0,
    )

    m, lo, hi, se, n = mean_ci_t(
        vals
    )

    p_perm = sign_flip_test(
        vals,
        alternative="two-sided",
    )


    result = pd.DataFrame(
        [
            {
                "effect":
                    "converge_minus_diverge_temporal_lag",

                "mean_sec":
                    m,

                "ci95_low":
                    lo,

                "ci95_high":
                    hi,

                "t":
                    t_result.statistic,

                "df":
                    n - 1,

                "p_two_sided":
                    t_result.pvalue,

                "p_signflip_two_sided":
                    p_perm,

                "n_subjects":
                    n,
            }
        ]
    )

else:

    result = pd.DataFrame()


result.to_csv(
    os.path.join(
        CSV_DIR,
        "group_temporal_hysteresis_result.csv",
    ),
    index=False,
)


# =============================================================================
# GROUP TRAJECTORIES
# =============================================================================
# For the figure, calculate each direction independently at each exact
# disparity. Participant counts are saved and printed below.

group_rows = []


for (
    direction,
    target_disparity
), g in subject_bin.groupby(
    [
        "direction",
        "target_disparity_deg",
    ]
):

    m, lo, hi, se, n = mean_ci_t(
        g[
            "median_lag_sec"
        ]
    )

    group_rows.append(
        {
            "direction":
                direction,

            "disparity_deg":
                target_disparity,

            "mean_lag_sec":
                m,

            "ci95_low":
                lo,

            "ci95_high":
                hi,

            "n_subjects":
                n,
        }
    )


group_bin = pd.DataFrame(
    group_rows
)


group_bin.to_csv(
    os.path.join(
        CSV_DIR,
        "group_local_lag_by_direction_and_disparity.csv",
    ),
    index=False,
)


# =============================================================================
# PARTICIPANT COUNTS AT EACH EXACT DISPARITY
# =============================================================================

counts = group_bin.pivot(
    index="disparity_deg",
    columns="direction",
    values="n_subjects",
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "PARTICIPANTS CONTRIBUTING TO EACH EXACT DISPARITY"
)

print(
    "=" * 90
)


print(
    counts.to_string()
)


# =============================================================================
# MATCHED PARTICIPANT COUNTS
# =============================================================================

matched_counts = (
    matched_wide
    .groupby(
        "target_disparity_deg"
    )[
        "subject"
    ]
    .nunique()
)


print(
    "\nMatched participants contributing BOTH directions:"
)

print(
    matched_counts.to_string()
)


# =============================================================================
# GROUP PLOT
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        7.5,
        5.5,
    )
)


for direction in [
    "Diverge",
    "Converge",
]:

    g = group_bin[
        group_bin[
            "direction"
        ] == direction
    ].sort_values(
        "disparity_deg"
    )


    ax.errorbar(
        g[
            "disparity_deg"
        ],

        g[
            "mean_lag_sec"
        ],

        yerr=[
            g[
                "mean_lag_sec"
            ]
            -
            g[
                "ci95_low"
            ],

            g[
                "ci95_high"
            ]
            -
            g[
                "mean_lag_sec"
            ],
        ],

        marker="o",
        capsize=4,
        label=direction,
    )


ax.set_xlim(
    MIN_MATCHED_DISPARITY - 1.0,
    MAX_MATCHED_DISPARITY + 1.0,
)


ax.set_xticks(
    MATCHED_DISPARITIES
)


ax.set_xlabel(
    "Instantaneous audiovisual disparity (deg)"
)


ax.set_ylabel(
    "Local auditory-relative tracking lag (s)"
)


ax.set_title(
    "Temporal lag hysteresis at matched audiovisual disparity"
)


ax.legend()


fig.tight_layout()


fig.savefig(
    os.path.join(
        FIG_DIR,
        "temporal_lag_hysteresis_by_disparity.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)


plt.show()


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "TEMPORAL LAG HYSTERESIS ANALYSIS COMPLETE"
)

print(
    "=" * 90
)


print(
    f"\nTotal trial × target observations: "
    f"{len(window_df)}"
)


print(
    f"Targets actually reached: "
    f"{int(window_df['target_in_observed_range'].sum())}"
)


print(
    f"Estimable local-lag observations: "
    f"{len(valid)}"
)


print(
    f"\nParticipants with at least one matched "
    f"Diverge/Converge disparity: "
    f"{subject_hysteresis['temporal_hysteresis_sec'].notna().sum()}"
)


print(
    "\nNumber of exact matched disparities per participant:"
)


print(
    subject_hysteresis[
        "n_matched_disparities"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nGroup temporal hysteresis result:"
)


print(
    result.to_string(
        index=False
    )
)


print(
    "\nDiagnostic files:"
)


print(
    os.path.join(
        CSV_DIR,
        "sweep_trial_observed_disparity_ranges.csv",
    )
)


print(
    os.path.join(
        CSV_DIR,
        "target_disparity_reach_diagnostics.csv",
    )
)


print(
    f"\nAll outputs saved to:\n{OUT_DIR}"
)

In [ ]:
# %% COMPLEMENTARY LAG ANALYSIS 4 — LAG BEFORE VS AFTER SPATIAL STABILIZATION
# =============================================================================
# QUESTION
# =============================================================================
# When a trial transitions from active spatial recalibration into a stable
# visually shifted estimate, does auditory-relative tracking latency also change?
#
# Primary quantity:
#
#   Delta lag = Post-stabilization lag - Pre-stabilization lag
#
# Negative:
#   tracking becomes faster after stabilization
#
# Positive:
#   tracking becomes slower after stabilization
#
# Only trials with an accepted stabilizing breakpoint are analyzed.
# =============================================================================

import os
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, ttest_1samp

# =============================================================================
# OUTPUT
# =============================================================================

OUT_DIR = os.path.join(
    COMPLEMENTARY_ROOT,
    "04_lag_pre_post_stabilization",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

for d in [
    OUT_DIR,
    CSV_DIR,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# =============================================================================
# SETTINGS
# =============================================================================

LOCAL_MAX_LAG_SEC = 4.0
LOCAL_MAX_LAG_SAMPLES = int(
    LOCAL_MAX_LAG_SEC
    *
    RESAMPLE_HZ
)

MIN_PEAK_R = 0.40
MIN_AUDIO_SD_DEG = 3.0
MIN_VALID_SAMPLES_LOCAL = 30

# Minimum temporal data on each side of the breakpoint
MIN_PRE_DURATION_SEC = 2.0
MIN_POST_DURATION_SEC = 3.0

# Avoid immediate transition vicinity
BREAKPOINT_BUFFER_SEC = 0.5


# =============================================================================
# LOAD BREAKPOINTS
# =============================================================================

if not os.path.exists(
    BREAKPOINT_CSV
):

    raise FileNotFoundError(
        "Breakpoint trial table not found:\n"
        f"{BREAKPOINT_CSV}"
    )


bp = pd.read_csv(
    BREAKPOINT_CSV
)

bp[
    "subject"
] = (
    bp[
        "subject"
    ]
    .astype(str)
    .str.lower()
)

bp[
    "trialIndex"
] = pd.to_numeric(
    bp[
        "trialIndex"
    ],
    errors="coerce",
)

# Prefer lag-corrected visual-capture breakpoint metric
if "metric" in bp.columns:

    possible = bp[
        "metric"
    ].astype(
        str
    ).unique()

    preferred = [
        x for x in possible
        if "LagCorrectedVisualCapture".lower()
        in x.lower()
    ]

    if len(preferred) > 0:

        bp = bp[
            bp[
                "metric"
            ].astype(str) == preferred[0]
        ].copy()


bp = bp[
    pd.to_numeric(
        bp[
            "has_stabilizing_breakpoint"
        ],
        errors="coerce",
    ).fillna(
        0
    ) == 1
].copy()

bp[
    "stabilization_time_sec"
] = pd.to_numeric(
    bp[
        "stabilization_time_sec"
    ],
    errors="coerce",
)

bp = bp.dropna(
    subset=[
        "subject",
        "trialIndex",
        "stabilization_time_sec",
    ]
)

print(
    f"Accepted stabilizing-breakpoint trials: {len(bp)}"
)


# =============================================================================
# POSITIVE-LAG HELPER
# =============================================================================

def estimate_segment_lag(
    response,
    audio,
):

    response = np.asarray(
        response,
        dtype=float,
    )

    audio = np.asarray(
        audio,
        dtype=float,
    )

    if np.nanstd(
        audio
    ) < MIN_AUDIO_SD_DEG:

        return {
            "lag_sec": np.nan,
            "peak_r": np.nan,
            "audio_sd": np.nanstd(
                audio
            ),
            "valid": False,
        }

    best_r = -np.inf
    best_lag_samples = np.nan

    for lag_samples in range(
        0,
        LOCAL_MAX_LAG_SAMPLES + 1,
    ):

        if lag_samples == 0:

            rr = response
            ss = audio

        else:

            rr = response[
                lag_samples:
            ]

            ss = audio[
                :-lag_samples
            ]

        ok = (
            np.isfinite(
                rr
            )
            &
            np.isfinite(
                ss
            )
        )

        if ok.sum() < MIN_VALID_SAMPLES_LOCAL:
            continue

        if (
            np.nanstd(
                rr[
                    ok
                ]
            ) <= 0
            or
            np.nanstd(
                ss[
                    ok
                ]
            ) <= 0
        ):
            continue

        r = pearsonr(
            rr[
                ok
            ],
            ss[
                ok
            ],
        )[0]

        if (
            np.isfinite(r)
            and r > best_r
        ):

            best_r = r
            best_lag_samples = lag_samples


    lag_sec = (
        best_lag_samples
        /
        RESAMPLE_HZ
        if np.isfinite(
            best_lag_samples
        )
        else np.nan
    )

    valid = (
        np.isfinite(
            lag_sec
        )
        and
        np.isfinite(
            best_r
        )
        and
        best_r > MIN_PEAK_R
    )

    return {
        "lag_sec": lag_sec,
        "peak_r": best_r,
        "audio_sd": np.nanstd(
            audio
        ),
        "valid": valid,
    }


# =============================================================================
# LOAD MASTER FILES AND MATCH STABILIZED TRIALS
# =============================================================================

files = sorted(
    glob.glob(
        os.path.join(
            DATA_DIR,
            FILE_PATTERN,
        )
    )
)

lag_rows = []

for fpath in files:

    subject = extract_subject_id(
        fpath
    )

    subject_bp = bp[
        bp[
            "subject"
        ] == subject
    ]

    if len(
        subject_bp
    ) == 0:
        continue

    raw = pd.read_csv(
        fpath
    )

    if "isPractice" in raw.columns:

        raw = raw[
            pd.to_numeric(
                raw[
                    "isPractice"
                ],
                errors="coerce",
            ).fillna(
                0
            ) == 0
        ].copy()

    for _, bp_row in subject_bp.iterrows():

        trial_index = bp_row[
            "trialIndex"
        ]

        breakpoint = bp_row[
            "stabilization_time_sec"
        ]

        tr = raw[
            pd.to_numeric(
                raw[
                    "trialIndex"
                ],
                errors="coerce",
            ) == trial_index
        ].copy()

        if len(
            tr
        ) < 5:
            continue

        tr = (
            tr
            .sort_values(
                "t"
            )
            .drop_duplicates(
                subset="t",
                keep="first",
            )
        )

        t = pd.to_numeric(
            tr[
                "t"
            ],
            errors="coerce",
        ).values.astype(float)

        t = (
            t
            -
            np.nanmin(
                t
            )
        )

        mouse = pd.to_numeric(
            tr[
                "mouse_x"
            ],
            errors="coerce",
        ).values.astype(float)

        audio = pd.to_numeric(
            tr[
                "audio_x"
            ],
            errors="coerce",
        ).values.astype(float)

        # Pre-breakpoint segment
        pre_end = (
            breakpoint
            -
            BREAKPOINT_BUFFER_SEC
        )

        # Post-breakpoint segment
        post_start = (
            breakpoint
            +
            BREAKPOINT_BUFFER_SEC
        )

        pre_mask = (
            np.isfinite(t)
            &
            (t >= 0.0)
            &
            (t < pre_end)
        )

        post_mask = (
            np.isfinite(t)
            &
            (t >= post_start)
            &
            (t <= TRIAL_DURATION_SEC)
        )

        pre_duration = (
            np.nanmax(
                t[
                    pre_mask
                ]
            )
            -
            np.nanmin(
                t[
                    pre_mask
                ]
            )
            if pre_mask.sum() > 1
            else 0.0
        )

        post_duration = (
            np.nanmax(
                t[
                    post_mask
                ]
            )
            -
            np.nanmin(
                t[
                    post_mask
                ]
            )
            if post_mask.sum() > 1
            else 0.0
        )

        if (
            pre_duration < MIN_PRE_DURATION_SEC
            or
            post_duration < MIN_POST_DURATION_SEC
        ):
            continue

        pre = estimate_segment_lag(
            mouse[
                pre_mask
            ],
            audio[
                pre_mask
            ],
        )

        post = estimate_segment_lag(
            mouse[
                post_mask
            ],
            audio[
                post_mask
            ],
        )

        lag_rows.append(
            {
                "subject": subject,
                "trialIndex": trial_index,
                "offset": bp_row.get(
                    "offset",
                    bp_row.get(
                        "offset_deg",
                        np.nan,
                    ),
                ),
                "stabilization_time_sec": breakpoint,

                "pre_lag_sec": pre[
                    "lag_sec"
                ],

                "pre_peak_r": pre[
                    "peak_r"
                ],

                "pre_audio_sd_deg": pre[
                    "audio_sd"
                ],

                "pre_valid": pre[
                    "valid"
                ],

                "post_lag_sec": post[
                    "lag_sec"
                ],

                "post_peak_r": post[
                    "peak_r"
                ],

                "post_audio_sd_deg": post[
                    "audio_sd"
                ],

                "post_valid": post[
                    "valid"
                ],
            }
        )


trial_results = pd.DataFrame(
    lag_rows
)

trial_results[
    "both_valid"
] = (
    trial_results[
        "pre_valid"
    ].fillna(
        False
    )
    &
    trial_results[
        "post_valid"
    ].fillna(
        False
    )
)

trial_results[
    "post_minus_pre_lag_sec"
] = np.where(
    trial_results[
        "both_valid"
    ],
    trial_results[
        "post_lag_sec"
    ]
    -
    trial_results[
        "pre_lag_sec"
    ],
    np.nan,
)

trial_results.to_csv(
    os.path.join(
        CSV_DIR,
        "trial_pre_post_stabilization_lags.csv",
    ),
    index=False,
)


# =============================================================================
# PARTICIPANT MEDIANS
# =============================================================================

valid_pairs = trial_results[
    trial_results[
        "both_valid"
    ]
].copy()

subject_results = (
    valid_pairs
    .groupby(
        "subject",
        as_index=False,
    )
    .agg(
        median_pre_lag_sec=(
            "pre_lag_sec",
            "median",
        ),

        median_post_lag_sec=(
            "post_lag_sec",
            "median",
        ),

        median_post_minus_pre_lag_sec=(
            "post_minus_pre_lag_sec",
            "median",
        ),

        n_valid_stabilized_trials=(
            "post_minus_pre_lag_sec",
            "count",
        ),
    )
)

subject_results.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_pre_post_stabilization_lag.csv",
    ),
    index=False,
)


# =============================================================================
# GROUP TEST
# =============================================================================

vals = subject_results[
    "median_post_minus_pre_lag_sec"
].dropna().values

if len(
    vals
) >= 2:

    t_result = ttest_1samp(
        vals,
        0.0,
    )

    m, lo, hi, se, n = mean_ci_t(
        vals
    )

    p_two = sign_flip_test(
        vals,
        alternative="two-sided",
    )

    p_less = sign_flip_test(
        vals,
        alternative="less",
    )

    group_result = pd.DataFrame(
        [
            {
                "effect":
                    "post_minus_pre_tracking_lag",

                "mean_sec": m,
                "ci95_low": lo,
                "ci95_high": hi,
                "t": t_result.statistic,
                "df": n - 1,
                "p_two_sided": t_result.pvalue,
                "p_signflip_two_sided": p_two,
                "p_signflip_less_predicted_faster_post": p_less,
                "n_subjects": n,
            }
        ]
    )

else:

    group_result = pd.DataFrame()


group_result.to_csv(
    os.path.join(
        CSV_DIR,
        "group_pre_post_stabilization_lag_result.csv",
    ),
    index=False,
)


print("\n" + "=" * 90)
print("PRE- VS POST-STABILIZATION LAG ANALYSIS COMPLETE")
print("=" * 90)

print(
    f"\nTrials with accepted breakpoints: {len(bp)}"
)

print(
    f"Trials with valid pre + post lag: {len(valid_pairs)}"
)

print(
    f"Participants contributing: {len(subject_results)}"
)

print(
    "\nGroup result:"
)

print(
    group_result.to_string(
        index=False
    )
)

print(
    f"\nOutputs saved to:\n{OUT_DIR}"
)

In [ ]:
# %% COMPLEMENTARY LAG ANALYSIS 5 — CONFIDENCE × CAUSAL JUDGMENT
# =============================================================================
# QUESTION
# =============================================================================
# Does the Same/Different lag difference become stronger as confidence increases?
#
# Model:
#
#   Lag ~ Judgment × Confidence + Disparity
#
# Predicted qualitative ordering:
#
#   High-confidence Same       -> shortest lag
#   Low-confidence Same        -> intermediate
#   Low-confidence Different   -> intermediate
#   High-confidence Different  -> longest lag
#
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
from scipy.stats import ttest_1samp

# =============================================================================
# OUTPUT
# =============================================================================

OUT_DIR = os.path.join(
    COMPLEMENTARY_ROOT,
    "05_confidence_judgment_lag",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures",
)

for d in [
    OUT_DIR,
    CSV_DIR,
    FIG_DIR,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# =============================================================================
# DATA
# =============================================================================

df = lag_master_df.copy()

df = df[
    df[
        "physical_condition"
    ].isin(
        [
            "congruent",
            "fixed_10",
            "fixed_20",
            "fixed_30",
            "fixed_40",
        ]
    )
].copy()

df[
    "confidence_rating"
] = pd.to_numeric(
    df[
        "confidence_rating"
    ],
    errors="coerce",
)

df[
    "disparity_10"
] = (
    pd.to_numeric(
        df[
            "disparity_deg"
        ],
        errors="coerce",
    )
    /
    10.0
)

df = df.dropna(
    subset=[
        "subject",
        "lag_audio_sec",
        "different_report",
        "confidence_rating",
        "disparity_10",
    ]
)

# Center confidence around the scale midpoint or grand mean.
df[
    "confidence_c"
] = (
    df[
        "confidence_rating"
    ]
    -
    df[
        "confidence_rating"
    ].mean()
)

df.to_csv(
    os.path.join(
        CSV_DIR,
        "confidence_judgment_lag_model_trials.csv",
    ),
    index=False,
)


# =============================================================================
# MIXED MODEL
# =============================================================================

model = smf.mixedlm(
    (
        "lag_audio_sec ~ "
        "different_report * confidence_c + "
        "disparity_10"
    ),
    data=df,
    groups=df[
        "subject"
    ],
)

try:

    fit = model.fit(
        reml=False,
        method="lbfgs",
    )

except Exception:

    fit = model.fit(
        reml=False,
        method="powell",
    )


with open(
    os.path.join(
        CSV_DIR,
        "mixed_model_confidence_x_judgment_lag.txt",
    ),
    "w",
) as f:

    f.write(
        fit.summary().as_text()
    )


ci = fit.conf_int()

coeff_rows = []

for term in fit.params.index:

    if term == "Group Var":
        continue

    coeff_rows.append(
        {
            "term": term,
            "estimate": fit.params[
                term
            ],
            "se": fit.bse.get(
                term,
                np.nan,
            ),
            "z": fit.tvalues.get(
                term,
                np.nan,
            ),
            "p": fit.pvalues.get(
                term,
                np.nan,
            ),
            "ci95_low": (
                ci.loc[
                    term,
                    0,
                ]
                if term in ci.index
                else np.nan
            ),
            "ci95_high": (
                ci.loc[
                    term,
                    1,
                ]
                if term in ci.index
                else np.nan
            ),
        }
    )


coeff_df = pd.DataFrame(
    coeff_rows
)

coeff_df.to_csv(
    os.path.join(
        CSV_DIR,
        "mixed_model_confidence_x_judgment_coefficients.csv",
    ),
    index=False,
)


# =============================================================================
# PARTICIPANT-FIRST INTERACTION
# =============================================================================

subject_rows = []

for subject, g in df.groupby(
    "subject"
):

    if (
        len(g) < 10
        or g[
            "different_report"
        ].nunique() < 2
        or g[
            "confidence_rating"
        ].nunique() < 2
    ):
        continue

    sg = g.copy()

    sg[
        "confidence_subject_c"
    ] = (
        sg[
            "confidence_rating"
        ]
        -
        sg[
            "confidence_rating"
        ].mean()
    )

    try:

        m = smf.ols(
            (
                "lag_audio_sec ~ "
                "different_report * confidence_subject_c + "
                "disparity_10"
            ),
            data=sg,
        ).fit()

        subject_rows.append(
            {
                "subject": subject,
                "n_trials": len(
                    sg
                ),

                "same_confidence_slope_sec_per_unit":
                    m.params.get(
                        "confidence_subject_c",
                        np.nan,
                    ),

                "different_minus_same_effect_sec":
                    m.params.get(
                        "different_report",
                        np.nan,
                    ),

                "judgment_x_confidence_interaction_sec_per_unit":
                    m.params.get(
                        "different_report:confidence_subject_c",
                        np.nan,
                    ),

                "different_confidence_slope_sec_per_unit":
                    (
                        m.params.get(
                            "confidence_subject_c",
                            np.nan,
                        )
                        +
                        m.params.get(
                            "different_report:confidence_subject_c",
                            np.nan,
                        )
                    ),

                "disparity_slope_sec_per_10deg":
                    m.params.get(
                        "disparity_10",
                        np.nan,
                    ),
            }
        )

    except Exception:
        continue


subject_coeffs = pd.DataFrame(
    subject_rows
)

subject_coeffs.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_first_confidence_x_judgment_lag_coefficients.csv",
    ),
    index=False,
)


# =============================================================================
# GROUP TESTS
# =============================================================================

effects = [
    (
        "same_confidence_slope_sec_per_unit",
        "less",
    ),
    (
        "different_minus_same_effect_sec",
        "greater",
    ),
    (
        "judgment_x_confidence_interaction_sec_per_unit",
        "greater",
    ),
    (
        "different_confidence_slope_sec_per_unit",
        "greater",
    ),
]

rows = []

for col, direction in effects:

    vals = (
        subject_coeffs[
            col
        ]
        .dropna()
        .values
    )

    if len(
        vals
    ) < 2:
        continue

    t_result = ttest_1samp(
        vals,
        0.0,
    )

    m, lo, hi, se, n = mean_ci_t(
        vals
    )

    p_perm = sign_flip_test(
        vals,
        alternative=direction,
    )

    rows.append(
        {
            "effect": col,
            "mean": m,
            "ci95_low": lo,
            "ci95_high": hi,
            "t": t_result.statistic,
            "df": n - 1,
            "p_two_sided": t_result.pvalue,
            "signflip_alternative": direction,
            "p_signflip": p_perm,
            "n_subjects": n,
        }
    )


group_tests = pd.DataFrame(
    rows
)

group_tests.to_csv(
    os.path.join(
        CSV_DIR,
        "group_confidence_x_judgment_lag_tests.csv",
    ),
    index=False,
)


# =============================================================================
# DESCRIPTIVE PLOT
# =============================================================================

subject_conf = (
    df
    .groupby(
        [
            "subject",
            "confidence_rating",
            "judgment_label",
        ],
        as_index=False,
    )
    .agg(
        median_lag_sec=(
            "lag_audio_sec",
            "median",
        )
    )
)

plot_rows = []

for (
    confidence,
    judgment
), g in subject_conf.groupby(
    [
        "confidence_rating",
        "judgment_label",
    ]
):

    m, lo, hi, se, n = mean_ci_t(
        g[
            "median_lag_sec"
        ]
    )

    plot_rows.append(
        {
            "confidence_rating": confidence,
            "judgment_label": judgment,
            "mean_lag_sec": m,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_subjects": n,
        }
    )


group_conf = pd.DataFrame(
    plot_rows
)

group_conf.to_csv(
    os.path.join(
        CSV_DIR,
        "group_lag_by_confidence_and_judgment.csv",
    ),
    index=False,
)


fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)

for judgment in [
    "Same",
    "Different",
]:

    g = group_conf[
        group_conf[
            "judgment_label"
        ] == judgment
    ].sort_values(
        "confidence_rating"
    )

    if len(
        g
    ) == 0:
        continue

    ax.errorbar(
        g[
            "confidence_rating"
        ],
        g[
            "mean_lag_sec"
        ],
        yerr=[
            g[
                "mean_lag_sec"
            ]
            -
            g[
                "ci95_low"
            ],

            g[
                "ci95_high"
            ]
            -
            g[
                "mean_lag_sec"
            ],
        ],
        marker="o",
        capsize=4,
        label=judgment,
    )


ax.set_xlabel(
    "Post-trial confidence"
)

ax.set_ylabel(
    "Auditory-relative tracking lag (s)"
)

ax.set_title(
    "Tracking latency by causal judgment and confidence"
)

ax.legend(
    title="Judgment"
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "lag_by_confidence_and_judgment.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


print("\n" + "=" * 90)
print("CONFIDENCE × CAUSAL-JUDGMENT LAG ANALYSIS COMPLETE")
print("=" * 90)

print(
    "\nMixed-model coefficients:"
)
print(
    coeff_df.to_string(
        index=False
    )
)

print(
    "\nParticipant-first group tests:"
)
print(
    group_tests.to_string(
        index=False
    )
)

print(
    f"\nOutputs saved to:\n{OUT_DIR}"
)

In [26]:
# %% COMPLEMENTARY LAG ANALYSIS 6 — CONTINUOUS MULTISENSORY INTEGRATION STATE
# =============================================================================
# PURPOSE
# =============================================================================
# Ask whether three complementary behavioral signatures covary after removing
# the portion explained by physical audiovisual disparity:
#
#   1. Explicit causal report:
#         Same = more bound
#
#   2. Continuous spatial weighting:
#         Greater visual bias = more bound
#
#   3. Temporal tracking:
#         Shorter auditory-relative lag = more bound
#
# Theoretically coherent bound-state direction:
#
#   + Same
#   + Visual bias
#   - Auditory lag
#
# This analysis has two components:
#
# A. Theory-signed integration score
# B. Exploratory PCA of the three disparity-residualized measures
#
# IMPORTANT:
# Do not interpret the PCA as proving a latent causal state.
# It is a descriptive test of shared covariance.
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =============================================================================
# OUTPUT
# =============================================================================

OUT_DIR = os.path.join(
    COMPLEMENTARY_ROOT,
    "06_continuous_integration_state",
)

CSV_DIR = os.path.join(
    OUT_DIR,
    "csv_outputs",
)

FIG_DIR = os.path.join(
    OUT_DIR,
    "figures",
)

for d in [
    OUT_DIR,
    CSV_DIR,
    FIG_DIR,
]:
    os.makedirs(
        d,
        exist_ok=True,
    )


# =============================================================================
# DATA
# =============================================================================

if "Capture_rmse_lagcorr" not in lag_master_df.columns:

    raise RuntimeError(
        "Capture_rmse_lagcorr is required for this analysis."
    )


df = lag_master_df[
    lag_master_df[
        "physical_condition"
    ].isin(
        [
            "congruent",
            "fixed_10",
            "fixed_20",
            "fixed_30",
            "fixed_40",
        ]
    )
].copy()

df[
    "visual_bias_deg"
] = pd.to_numeric(
    df[
        "Capture_rmse_lagcorr"
    ],
    errors="coerce",
)

df[
    "disparity_10"
] = (
    pd.to_numeric(
        df[
            "disparity_deg"
        ],
        errors="coerce",
    )
    /
    10.0
)

df = df.dropna(
    subset=[
        "subject",
        "lag_audio_sec",
        "visual_bias_deg",
        "same_report",
        "disparity_10",
    ]
)

print(
    f"Complete trials: {len(df)}"
)

print(
    f"Subjects: {df['subject'].nunique()}"
)


# =============================================================================
# DISPARITY-RESIDUALIZE EACH MEASURE WITHIN PARTICIPANT
# =============================================================================

residual_rows = []

for subject, g in df.groupby(
    "subject"
):

    if (
        len(g) < 10
        or g[
            "disparity_10"
        ].nunique() < 2
    ):
        continue

    sg = g.copy()

    try:

        lag_model = smf.ols(
            (
                "lag_audio_sec ~ "
                "disparity_10"
            ),
            data=sg,
        ).fit()

        bias_model = smf.ols(
            (
                "visual_bias_deg ~ "
                "disparity_10"
            ),
            data=sg,
        ).fit()

        # Linear probability residual is intentional here because we only
        # need a continuous residualized report tendency.
        same_model = smf.ols(
            (
                "same_report ~ "
                "disparity_10"
            ),
            data=sg,
        ).fit()

        sg[
            "lag_resid"
        ] = lag_model.resid

        sg[
            "visual_bias_resid"
        ] = bias_model.resid

        sg[
            "same_resid"
        ] = same_model.resid

        residual_rows.append(
            sg
        )

    except Exception:
        continue


resid_df = pd.concat(
    residual_rows,
    ignore_index=True,
)

# =============================================================================
# WITHIN-SUBJECT STANDARDIZATION
# =============================================================================

def z_within_subject(
    s,
):

    sd = s.std(
        ddof=1
    )

    if (
        not np.isfinite(
            sd
        )
        or sd <= 0
    ):

        return pd.Series(
            np.nan,
            index=s.index,
        )

    return (
        s
        -
        s.mean()
    ) / sd


resid_df[
    "lag_resid_z"
] = (
    resid_df
    .groupby(
        "subject"
    )[
        "lag_resid"
    ]
    .transform(
        z_within_subject
    )
)

resid_df[
    "visual_bias_resid_z"
] = (
    resid_df
    .groupby(
        "subject"
    )[
        "visual_bias_resid"
    ]
    .transform(
        z_within_subject
    )
)

resid_df[
    "same_resid_z"
] = (
    resid_df
    .groupby(
        "subject"
    )[
        "same_resid"
    ]
    .transform(
        z_within_subject
    )
)


# =============================================================================
# THEORY-SIGNED INTEGRATION SCORE
#
# Greater = more bound:
#   + visual bias
#   + same-source tendency
#   - auditory lag
# =============================================================================

resid_df[
    "integration_score"
] = (
    resid_df[
        "visual_bias_resid_z"
    ]
    +
    resid_df[
        "same_resid_z"
    ]
    -
    resid_df[
        "lag_resid_z"
    ]
) / 3.0


resid_df.to_csv(
    os.path.join(
        CSV_DIR,
        "trial_level_disparity_residualized_integration_measures.csv",
    ),
    index=False,
)


# =============================================================================
# PAIRWISE WITHIN-SUBJECT ASSOCIATIONS
# =============================================================================

corr_rows = []

for subject, g in resid_df.groupby(
    "subject"
):

    pairs = [
        (
            "visual_bias_resid",
            "lag_resid",
            "visual_bias_vs_lag",
        ),
        (
            "same_resid",
            "lag_resid",
            "same_vs_lag",
        ),
        (
            "same_resid",
            "visual_bias_resid",
            "same_vs_visual_bias",
        ),
    ]

    row = {
        "subject": subject,
    }

    for xcol, ycol, name in pairs:

        tmp = g[
            [
                xcol,
                ycol,
            ]
        ].dropna()

        if (
            len(tmp) >= 5
            and tmp[
                xcol
            ].std() > 0
            and tmp[
                ycol
            ].std() > 0
        ):

            r, p = pearsonr(
                tmp[
                    xcol
                ],
                tmp[
                    ycol
                ],
            )

        else:

            r = np.nan
            p = np.nan

        row[
            f"r_{name}"
        ] = r

        row[
            f"p_{name}"
        ] = p

    corr_rows.append(
        row
    )


subject_corr = pd.DataFrame(
    corr_rows
)

subject_corr.to_csv(
    os.path.join(
        CSV_DIR,
        "subject_within_trial_integration_correlations.csv",
    ),
    index=False,
)


# Fisher-z group summaries
corr_summary_rows = []

for col in [
    "r_visual_bias_vs_lag",
    "r_same_vs_lag",
    "r_same_vs_visual_bias",
]:

    vals = (
        subject_corr[
            col
        ]
        .dropna()
        .clip(
            -0.999999,
            0.999999,
        )
        .values
    )

    if len(
        vals
    ) < 2:
        continue

    z = np.arctanh(
        vals
    )

    z_mean, z_lo, z_hi, z_se, n = mean_ci_t(
        z
    )

    corr_summary_rows.append(
        {
            "association": col,
            "mean_fisher_z": z_mean,
            "ci95_low_fisher_z": z_lo,
            "ci95_high_fisher_z": z_hi,
            "backtransformed_mean_r": np.tanh(
                z_mean
            ),
            "backtransformed_ci95_low_r": np.tanh(
                z_lo
            ),
            "backtransformed_ci95_high_r": np.tanh(
                z_hi
            ),
            "p_signflip_fisher_z_two_sided":
                sign_flip_test(
                    z,
                    alternative="two-sided",
                ),
            "n_subjects": n,
        }
    )


corr_summary = pd.DataFrame(
    corr_summary_rows
)

corr_summary.to_csv(
    os.path.join(
        CSV_DIR,
        "group_integration_correlation_summary.csv",
    ),
    index=False,
)


# =============================================================================
# EXPLORATORY PCA
# =============================================================================

pca_df = resid_df.dropna(
    subset=[
        "visual_bias_resid_z",
        "lag_resid_z",
        "same_resid_z",
    ]
).copy()

X = pca_df[
    [
        "visual_bias_resid_z",
        "lag_resid_z",
        "same_resid_z",
    ]
].values

# Already within-subject standardized, but center globally for PCA.
X_scaled = StandardScaler(
    with_mean=True,
    with_std=True,
).fit_transform(
    X
)

pca = PCA(
    n_components=3
)

scores = pca.fit_transform(
    X_scaled
)

loadings = pd.DataFrame(
    pca.components_.T,
    index=[
        "visual_bias",
        "auditory_lag",
        "same_report",
    ],
    columns=[
        "PC1",
        "PC2",
        "PC3",
    ],
)

variance = pd.DataFrame(
    {
        "component": [
            "PC1",
            "PC2",
            "PC3",
        ],
        "variance_explained": pca.explained_variance_ratio_,
    }
)

loadings.to_csv(
    os.path.join(
        CSV_DIR,
        "integration_state_pca_loadings.csv",
    )
)

variance.to_csv(
    os.path.join(
        CSV_DIR,
        "integration_state_pca_variance_explained.csv",
    ),
    index=False,
)


pca_df[
    "PC1_raw"
] = scores[
    :,
    0
]

# Orient PC1 toward theoretical "more bound":
#   + visual bias
#   - lag
#   + Same
theory_vector = np.array(
    [
        1.0,
        -1.0,
        1.0,
    ]
)

pc1_loadings = pca.components_[
    0,
    :
]

if np.dot(
    pc1_loadings,
    theory_vector,
) < 0:

    pca_df[
        "PC1_integration_oriented"
    ] = (
        -pca_df[
            "PC1_raw"
        ]
    )

    oriented_loadings = (
        -pc1_loadings
    )

else:

    pca_df[
        "PC1_integration_oriented"
    ] = pca_df[
        "PC1_raw"
    ]

    oriented_loadings = pc1_loadings


oriented_loading_df = pd.DataFrame(
    {
        "measure": [
            "visual_bias",
            "auditory_lag",
            "same_report",
        ],
        "PC1_oriented_loading": oriented_loadings,
    }
)

oriented_loading_df.to_csv(
    os.path.join(
        CSV_DIR,
        "integration_state_pc1_oriented_loadings.csv",
    ),
    index=False,
)


pca_df.to_csv(
    os.path.join(
        CSV_DIR,
        "trial_level_integration_state_PCA_scores.csv",
    ),
    index=False,
)


# =============================================================================
# COMPARE THEORY-SIGNED SCORE TO PCA AXIS
# =============================================================================

score_merge = resid_df[
    [
        "subject",
        "trialIndex",
        "integration_score",
    ]
].merge(
    pca_df[
        [
            "subject",
            "trialIndex",
            "PC1_integration_oriented",
        ]
    ],
    on=[
        "subject",
        "trialIndex",
    ],
    how="inner",
)

r_score_pc1, p_score_pc1 = pearsonr(
    score_merge[
        "integration_score"
    ],
    score_merge[
        "PC1_integration_oriented"
    ],
)

score_pc1_result = pd.DataFrame(
    [
        {
            "comparison":
                "theory_signed_score_vs_oriented_PC1",
            "r": r_score_pc1,
            "p": p_score_pc1,
            "n_trials": len(
                score_merge
            ),
        }
    ]
)

score_pc1_result.to_csv(
    os.path.join(
        CSV_DIR,
        "theory_score_vs_pca_axis.csv",
    ),
    index=False,
)


# =============================================================================
# FIGURE — PCA LOADINGS
# =============================================================================

fig, ax = plt.subplots(
    figsize=(7.0, 5.0)
)

ax.bar(
    oriented_loading_df[
        "measure"
    ],
    oriented_loading_df[
        "PC1_oriented_loading"
    ],
)

ax.axhline(
    0,
    linewidth=1,
)

ax.set_ylabel(
    "Oriented PC1 loading"
)

ax.set_title(
    "Shared spatiotemporal integration axis"
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        FIG_DIR,
        "integration_state_pc1_loadings.png",
    ),
    dpi=FIG_DPI,
    bbox_inches="tight",
)

plt.show()


print("\n" + "=" * 90)
print("CONTINUOUS INTEGRATION-STATE ANALYSIS COMPLETE")
print("=" * 90)

print(
    "\nPairwise disparity-residualized associations:"
)

print(
    corr_summary.to_string(
        index=False
    )
)

print(
    "\nPCA variance explained:"
)

print(
    variance.to_string(
        index=False
    )
)

print(
    "\nOriented PC1 loadings:"
)

print(
    oriented_loading_df.to_string(
        index=False
    )
)

print(
    "\nTheory score vs PCA:"
)

print(
    score_pc1_result.to_string(
        index=False
    )
)

print(
    f"\nOutputs saved to:\n{OUT_DIR}"
)

Complete trials: 2595
Subjects: 50

CONTINUOUS INTEGRATION-STATE ANALYSIS COMPLETE

Pairwise disparity-residualized associations:
          association  mean_fisher_z  ci95_low_fisher_z  ci95_high_fisher_z  backtransformed_mean_r  backtransformed_ci95_low_r  backtransformed_ci95_high_r  p_signflip_fisher_z_two_sided  n_subjects
 r_visual_bias_vs_lag      -0.373656          -0.432896           -0.314416               -0.357185                   -0.407738                    -0.304449                         0.0001          50
        r_same_vs_lag      -0.224984          -0.279605           -0.170364               -0.221264                   -0.272540                    -0.168734                         0.0001          50
r_same_vs_visual_bias       0.586101           0.498667            0.673535                0.527085                    0.461068                     0.587300                         0.0001          50

PCA variance explained:
component  variance_explained
      PC1      

In [29]:
# %% ==========================================================================
# CROSS-TASK UNITY–SPATIAL READOUT PCA + INCREMENTAL MIXED MODEL
# =============================================================================
#
# PURPOSE
# -------
# Build a participant × disparity cross-task dataset linking:
#
#   Spatial Localization (SL):
#       1. lag-corrected visual bias
#       2. auditory-relative tracking lag
#       3. explicit Same-source probability
#
#   Causal Inference (CI):
#       4. stable/plateau continuous unity
#
# Then:
#   1. residualize all four measures for participant baseline and physical disparity
#   2. report residualized pairwise associations
#   3. run the original 3-variable PCA: bias + lag + Same
#   4. run a continuous-unity 3-variable PCA: bias + lag + unity
#   5. run a full 4-variable PCA: bias + lag + Same + unity
#   6. compare PC1 loadings + PC1/PC1+PC2 communalities
#   7. fit a mixed model testing whether CI unity adds unique prediction of
#      SL visual bias beyond disparity, Same probability, and auditory lag
#
# IMPORTANT CI CHOICE
# -------------------
# The primary CI unity variable is computed from fixed_trial_breakpoints.csv
# using plateau_perceived_unity for ALL trials classified:
#
#       broad_class == "stable_or_stabilizing"
#
# This includes:
#       - stable_from_onset trials
#       - stabilizing trials
#
# That is preferable here to fixed_subject_breakpoints.csv because the latter's
# mean_plateau_unity is conditional on accepted "stabilizing" trials only.
#
# The code also computes a stabilizing-only unity measure as a sensitivity
# variable, but the primary PCA uses all stable/stabilizing causal states.
#
# IMPORTANT RESIDUALIZATION CHOICE
# --------------------------------
# Once data are aggregated to one row per participant × disparity, fitting a
# separate categorical disparity model within each participant would be saturated.
# Therefore physical disparity is removed with a two-way fixed-effect model:
#
#       measure ~ C(subject) + C(disparity)
#
# This removes:
#       - participant baseline differences
#       - arbitrary/nonlinear mean effects of physical disparity
#
# The resulting residuals are then z-scored within participant before PCA.
# This is more conservative than assuming a linear disparity effect, especially
# because the CI unity function is clearly nonlinear across 0–40°.
#
# RUN LOCATION
# ------------
# Paste this as ONE CELL after the shared complementary-lag preparation in the
# Continuous_SLv3_rebuilt notebook (i.e., after lag_master_df has been created).
#
# ==============================================================================


from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    pearsonr,
    ttest_1samp,
    chi2,
    t as t_dist,
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)


# =============================================================================
# 0. USER / PATH SETTINGS
# =============================================================================

RANDOM_SEED_CROSS = 20260914
N_PERM_CROSS = 1000
FIG_DPI_CROSS = 180


SPATIAL_ANALYSES_ROOT_CROSS = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1\spatial_analyses"
)


# This matches the Continuous_SLv3_rebuilt notebook.
if "COMPLEMENTARY_ROOT" in globals():

    COMPLEMENTARY_ROOT_CROSS = Path(
        COMPLEMENTARY_ROOT
    )

else:

    COMPLEMENTARY_ROOT_CROSS = (
        SPATIAL_ANALYSES_ROOT_CROSS
        /
        "08_complementary_lag_analyses"
    )


OUT_DIR_CROSS = (
    COMPLEMENTARY_ROOT_CROSS
    /
    "07_cross_task_unity_pca"
)


CSV_DIR_CROSS = (
    OUT_DIR_CROSS
    /
    "csv_outputs"
)


FIG_DIR_CROSS = (
    OUT_DIR_CROSS
    /
    "figures"
)


CSV_DIR_CROSS.mkdir(
    parents=True,
    exist_ok=True,
)


FIG_DIR_CROSS.mkdir(
    parents=True,
    exist_ok=True,
)


# CI analysis root used in your CI notebook.
CI_ANALYSIS_ROOT_CROSS = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1\individual_data_CI"
    r"\causal_inference_analyses"
)


# OPTIONAL:
# If automatic search does not find the correct file, paste the exact Windows
# path here. Otherwise leave as None.
CI_FIXED_TRIAL_BP_CSV_MANUAL = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_CI\CI_unity_breakpoints\fixed_trial_breakpoints.csv"

# Example:
# CI_FIXED_TRIAL_BP_CSV_MANUAL = (
#     r"C:\...\fixed_trial_breakpoints.csv"
# )


# =============================================================================
# 1. HELPERS
# =============================================================================

def _mean_ci(
    values,
    alpha=0.05,
):

    vals = (
        pd.Series(
            values
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .astype(float)
        .values
    )


    n = len(
        vals
    )


    if n < 2:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            n,
        )


    m = float(
        np.mean(
            vals
        )
    )


    sd = float(
        np.std(
            vals,
            ddof=1,
        )
    )


    se = (
        sd
        /
        np.sqrt(
            n
        )
    )


    crit = float(
        t_dist.ppf(
            1
            -
            alpha / 2,
            df=n - 1,
        )
    )


    return (
        m,
        m - crit * se,
        m + crit * se,
        se,
        n,
    )


def _sign_flip_test(
    values,
    popmean=0.0,
    alternative="two-sided",
    n_perm=N_PERM_CROSS,
    seed=RANDOM_SEED_CROSS,
):

    vals = (
        pd.Series(
            values
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .astype(float)
        .values
    )


    if len(
        vals
    ) < 2:

        return np.nan


    diffs = (
        vals
        -
        popmean
    )


    obs = float(
        np.mean(
            diffs
        )
    )


    rng = np.random.default_rng(
        seed
    )


    count = 0


    for _ in range(
        n_perm
    ):

        signs = rng.choice(
            [
                -1.0,
                1.0,
            ],
            size=len(
                diffs
            ),
        )


        null_value = float(
            np.mean(
                signs
                *
                diffs
            )
        )


        if alternative == "greater":

            count += int(
                null_value
                >=
                obs
            )


        elif alternative == "less":

            count += int(
                null_value
                <=
                obs
            )


        else:

            count += int(
                abs(
                    null_value
                )
                >=
                abs(
                    obs
                )
            )


    return (
        count + 1
    ) / (
        n_perm + 1
    )


def _z_within_subject(
    series,
):

    series = pd.to_numeric(
        series,
        errors="coerce",
    )


    sd = series.std(
        ddof=1
    )


    if (
        not np.isfinite(
            sd
        )
        or
        sd <= 0
    ):

        return pd.Series(
            np.nan,
            index=series.index,
        )


    return (
        series
        -
        series.mean()
    ) / sd


def _find_ci_breakpoint_file():
    """
    Find fixed_trial_breakpoints.csv robustly.
    """

    if (
        CI_FIXED_TRIAL_BP_CSV_MANUAL
        is not None
    ):

        manual = Path(
            CI_FIXED_TRIAL_BP_CSV_MANUAL
        )


        if not manual.exists():

            raise FileNotFoundError(
                "CI_FIXED_TRIAL_BP_CSV_MANUAL does not exist:\n"
                f"{manual}"
            )


        return manual


    search_roots = [
        CI_ANALYSIS_ROOT_CROSS,
        Path.cwd(),
    ]


    candidates = []


    for root in search_roots:

        if not root.exists():

            continue


        # Prefer exact filename.
        exact = list(
            root.rglob(
                "fixed_trial_breakpoints.csv"
            )
        )


        candidates.extend(
            exact
        )


        # Also allow downloaded/copied names such as "(1)".
        if len(
            exact
        ) == 0:

            candidates.extend(
                root.rglob(
                    "fixed_trial_breakpoints*.csv"
                )
            )


    # Deduplicate.
    candidates = list(
        dict.fromkeys(
            candidates
        )
    )


    if len(
        candidates
    ) == 0:

        raise FileNotFoundError(
            "Could not locate fixed_trial_breakpoints.csv automatically.\n\n"
            "Set CI_FIXED_TRIAL_BP_CSV_MANUAL at the top of this cell "
            "to the exact path of your CI fixed_trial_breakpoints.csv file."
        )


    # If several exist, use the most recently modified and print all candidates.
    candidates = sorted(
        candidates,
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )


    print(
        "\nCI breakpoint-file candidates:"
    )


    for p in candidates:

        print(
            f"  {p}"
        )


    chosen = candidates[
        0
    ]


    print(
        "\nUsing CI breakpoint file:"
    )


    print(
        chosen
    )


    return chosen


def _two_way_residualize(
    data,
    outcome,
):

    """
    Remove participant baseline + arbitrary disparity-level means:

        outcome ~ C(subject) + C(disparity_cat)

    Returns residuals aligned to data.index.
    """

    model_data = (
        data[
            [
                "subject",
                "disparity_cat",
                outcome,
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .copy()
    )


    fit = smf.ols(
        f"{outcome} ~ C(subject) + C(disparity_cat)",
        data=model_data,
    ).fit()


    residuals = pd.Series(
        np.nan,
        index=data.index,
        dtype=float,
    )


    residuals.loc[
        model_data.index
    ] = fit.resid.values


    return (
        residuals,
        fit,
    )


def _safe_pearson(
    x,
    y,
):

    d = (
        pd.DataFrame(
            {
                "x": x,
                "y": y,
            }
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    if (
        len(
            d
        ) < 3
        or
        d[
            "x"
        ].std(
            ddof=1
        ) <= 0
        or
        d[
            "y"
        ].std(
            ddof=1
        ) <= 0
    ):

        return (
            np.nan,
            np.nan,
            len(
                d
            ),
        )


    r, p = pearsonr(
        d[
            "x"
        ],
        d[
            "y"
        ],
    )


    return (
        float(
            r
        ),
        float(
            p
        ),
        len(
            d
        ),
    )


def _fit_mixedlm_with_fallback(
    formula,
    data,
    group_col="subject",
):

    """
    Fit ML MixedLM using several optimizers if necessary.
    """

    last_error = None


    for method in [
        "lbfgs",
        "powell",
        "cg",
    ]:

        try:

            fit = smf.mixedlm(
                formula,
                data=data,
                groups=data[
                    group_col
                ],
            ).fit(
                reml=False,
                method=method,
                maxiter=2000,
                disp=False,
            )


            return (
                fit,
                method,
            )


        except Exception as exc:

            last_error = exc


    raise RuntimeError(
        "MixedLM failed with all optimizers.\n"
        f"Last error: {last_error}"
    )


def _nakagawa_r2_random_intercept(
    fit,
):

    """
    Approximate Nakagawa marginal/conditional R² for a Gaussian
    random-intercept MixedLM.
    """

    fixed_pred = np.asarray(
        fit.model.exog
        @
        fit.fe_params.values,
        dtype=float,
    )


    var_fixed = float(
        np.var(
            fixed_pred,
            ddof=1,
        )
    )


    try:

        var_random = float(
            fit.cov_re.iloc[
                0,
                0,
            ]
        )


    except Exception:

        var_random = 0.0


    var_resid = float(
        fit.scale
    )


    total = (
        var_fixed
        +
        var_random
        +
        var_resid
    )


    if total <= 0:

        return {
            "marginal_r2":
                np.nan,

            "conditional_r2":
                np.nan,

            "var_fixed":
                var_fixed,

            "var_random":
                var_random,

            "var_resid":
                var_resid,
        }


    return {
        "marginal_r2":
            var_fixed
            /
            total,

        "conditional_r2":
            (
                var_fixed
                +
                var_random
            )
            /
            total,

        "var_fixed":
            var_fixed,

        "var_random":
            var_random,

        "var_resid":
            var_resid,
    }


# =============================================================================
# 2. LOAD / VALIDATE SL TRIAL-LEVEL DATA FROM Continuous_SLv3_rebuilt
# =============================================================================

if (
    "lag_master_df" in globals()
    and
    isinstance(
        lag_master_df,
        pd.DataFrame,
    )
    and
    len(
        lag_master_df
    ) > 0
):

    sl_trial_df = (
        lag_master_df.copy()
    )


    print(
        "\nUsing in-memory lag_master_df from the SL notebook."
    )


else:

    lag_master_csv = (
        COMPLEMENTARY_ROOT_CROSS
        /
        "complementary_lag_master_valid_trials.csv"
    )


    if not lag_master_csv.exists():

        raise FileNotFoundError(
            "lag_master_df is not in memory and the saved master file "
            "was not found:\n"
            f"{lag_master_csv}\n\n"
            "Run the shared complementary-lag preparation cell first."
        )


    sl_trial_df = pd.read_csv(
        lag_master_csv
    )


    print(
        "\nLoaded saved SL complementary-lag master:"
    )


    print(
        lag_master_csv
    )


required_sl_cols = [
    "subject",
    "trialIndex",
    "physical_condition",
    "disparity_deg",
    "lag_audio_sec",
    "same_report",
    "Capture_rmse_lagcorr",
]


missing_sl = [
    c
    for c
    in required_sl_cols
    if c
    not in sl_trial_df.columns
]


if missing_sl:

    raise RuntimeError(
        "SL lag_master_df is missing required columns:\n"
        +
        "\n".join(
            missing_sl
        )
    )


sl_trial_df[
    "subject"
] = (
    sl_trial_df[
        "subject"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
)


sl_trial_df[
    "disparity_deg"
] = pd.to_numeric(
    sl_trial_df[
        "disparity_deg"
    ],
    errors="coerce",
)


sl_trial_df[
    "lag_audio_sec"
] = pd.to_numeric(
    sl_trial_df[
        "lag_audio_sec"
    ],
    errors="coerce",
)


sl_trial_df[
    "same_report"
] = pd.to_numeric(
    sl_trial_df[
        "same_report"
    ],
    errors="coerce",
)


sl_trial_df[
    "visual_bias_deg"
] = pd.to_numeric(
    sl_trial_df[
        "Capture_rmse_lagcorr"
    ],
    errors="coerce",
)


# Match the fixed conditions used in the existing PCA.
sl_fixed_df = sl_trial_df[
    sl_trial_df[
        "physical_condition"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            "congruent",
            "fixed_10",
            "fixed_20",
            "fixed_30",
            "fixed_40",
        ]
    )
].copy()


# Force congruent to exactly 0° disparity.
sl_fixed_df.loc[
    sl_fixed_df[
        "physical_condition"
    ]
    .astype(str)
    .str.lower()
    .eq(
        "congruent"
    ),
    "disparity_deg"
] = 0.0


sl_fixed_df[
    "disparity_deg"
] = (
    sl_fixed_df[
        "disparity_deg"
    ]
    .abs()
)


# One row per participant × physical disparity.
#
# Means are used here because the original PCA treats each valid trial
# as an observation; the participant × disparity mean is its natural
# cross-task condition-level counterpart.
sl_subject_disparity = (
    sl_fixed_df
    .groupby(
        [
            "subject",
            "disparity_deg",
        ],
        as_index=False,
    )
    .agg(

        n_sl_trials=(
            "trialIndex",
            "nunique",
        ),

        visual_bias_deg=(
            "visual_bias_deg",
            "mean",
        ),

        auditory_lag_sec=(
            "lag_audio_sec",
            "mean",
        ),

        same_probability=(
            "same_report",
            "mean",
        ),
    )
)


print(
    "\nSL participant × disparity rows:"
)


print(
    len(
        sl_subject_disparity
    )
)


print(
    "SL subjects:"
)


print(
    sl_subject_disparity[
        "subject"
    ].nunique()
)


print(
    "SL disparity levels:"
)


print(
    sorted(
        sl_subject_disparity[
            "disparity_deg"
        ]
        .dropna()
        .unique()
    )
)


# =============================================================================
# 3. LOAD CI FIXED-TRIAL BREAKPOINT DATA AND BUILD STABLE UNITY
# =============================================================================

ci_bp_csv = (
    _find_ci_breakpoint_file()
)


ci_trial_bp = pd.read_csv(
    ci_bp_csv
)


required_ci_cols = [
    "participant_id",
    "trial_id",
    "analysis_group",
    "trial_class",
    "broad_class",
    "plateau_perceived_unity",
]


missing_ci = [
    c
    for c
    in required_ci_cols
    if c
    not in ci_trial_bp.columns
]


if missing_ci:

    raise RuntimeError(
        "CI fixed_trial_breakpoints file is missing columns:\n"
        +
        "\n".join(
            missing_ci
        )
    )


ci_trial_bp[
    "subject"
] = (
    ci_trial_bp[
        "participant_id"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
)


ci_trial_bp[
    "plateau_perceived_unity"
] = pd.to_numeric(
    ci_trial_bp[
        "plateau_perceived_unity"
    ],
    errors="coerce",
)


analysis_group_to_disparity = {

    "congruent":
        0.0,

    "fixed_10":
        10.0,

    "fixed_20":
        20.0,

    "fixed_30":
        30.0,

    "fixed_40":
        40.0,
}


ci_trial_bp[
    "disparity_deg"
] = (
    ci_trial_bp[
        "analysis_group"
    ]
    .map(
        analysis_group_to_disparity
    )
)


ci_fixed_bp = ci_trial_bp[
    ci_trial_bp[
        "analysis_group"
    ]
    .isin(
        analysis_group_to_disparity.keys()
    )
].copy()


# -------------------------------------------------------------------------
# PRIMARY UNITY MEASURE
#
# Stable causal state across BOTH:
#   - stable_from_onset trials
#   - stabilizing trials
# -------------------------------------------------------------------------

ci_stable_trials = ci_fixed_bp[
    ci_fixed_bp[
        "broad_class"
    ]
    .astype(str)
    .eq(
        "stable_or_stabilizing"
    )
    &
    ci_fixed_bp[
        "plateau_perceived_unity"
    ]
    .notna()
].copy()


ci_stable_subject_disparity = (
    ci_stable_trials
    .groupby(
        [
            "subject",
            "disparity_deg",
        ],
        as_index=False,
    )
    .agg(

        stable_unity=(
            "plateau_perceived_unity",
            "mean",
        ),

        n_ci_stable_trials=(
            "trial_id",
            "nunique",
        ),
    )
)


# -------------------------------------------------------------------------
# SENSITIVITY UNITY MEASURE
#
# Accepted stabilizing trials only.
# This corresponds conceptually to the mean_plateau_unity column in
# fixed_subject_breakpoints.csv.
# -------------------------------------------------------------------------

ci_stabilizing_only = ci_fixed_bp[
    ci_fixed_bp[
        "trial_class"
    ]
    .astype(str)
    .eq(
        "stabilizing"
    )
    &
    ci_fixed_bp[
        "plateau_perceived_unity"
    ]
    .notna()
].copy()


ci_stabilizing_only_subject_disparity = (
    ci_stabilizing_only
    .groupby(
        [
            "subject",
            "disparity_deg",
        ],
        as_index=False,
    )
    .agg(

        stabilizing_only_unity=(
            "plateau_perceived_unity",
            "mean",
        ),

        n_ci_stabilizing_only_trials=(
            "trial_id",
            "nunique",
        ),
    )
)


# Total CI fixed-trial count for diagnostics.
ci_total_counts = (
    ci_fixed_bp
    .groupby(
        [
            "subject",
            "disparity_deg",
        ],
        as_index=False,
    )
    .agg(

        n_ci_total_trials=(
            "trial_id",
            "nunique",
        )
    )
)


ci_subject_disparity = (
    ci_stable_subject_disparity

    .merge(
        ci_stabilizing_only_subject_disparity,
        on=[
            "subject",
            "disparity_deg",
        ],
        how="left",
    )

    .merge(
        ci_total_counts,
        on=[
            "subject",
            "disparity_deg",
        ],
        how="left",
    )
)


ci_subject_disparity[
    "ci_stable_fraction"
] = (
    ci_subject_disparity[
        "n_ci_stable_trials"
    ]
    /
    ci_subject_disparity[
        "n_ci_total_trials"
    ]
)


print(
    "\nCI stable-state participant × disparity rows:"
)


print(
    len(
        ci_subject_disparity
    )
)


print(
    "CI subjects:"
)


print(
    ci_subject_disparity[
        "subject"
    ].nunique()
)


print(
    "\nPrimary stable-unity group means:"
)


print(
    ci_subject_disparity
    .groupby(
        "disparity_deg"
    )[
        "stable_unity"
    ]
    .agg(
        [
            "mean",
            "std",
            "count",
        ]
    )
    .to_string()
)


# =============================================================================
# 4. CONSTRUCT PARTICIPANT × DISPARITY CROSS-TASK DATASET
# =============================================================================

cross_task_df = (
    sl_subject_disparity
    .merge(
        ci_subject_disparity,
        on=[
            "subject",
            "disparity_deg",
        ],
        how="inner",
        validate="one_to_one",
    )
)


# Keep exact fixed levels only.
cross_task_df = cross_task_df[
    cross_task_df[
        "disparity_deg"
    ]
    .isin(
        [
            0.0,
            10.0,
            20.0,
            30.0,
            40.0,
        ]
    )
].copy()


cross_task_df[
    "disparity_cat"
] = (
    cross_task_df[
        "disparity_deg"
    ]
    .astype(int)
    .astype(str)
)


raw_cross_csv = (
    CSV_DIR_CROSS
    /
    "cross_task_participant_by_disparity_raw.csv"
)


cross_task_df.to_csv(
    raw_cross_csv,
    index=False,
)


print(
    "\n"
    +
    "=" * 90
)


print(
    "CROSS-TASK PARTICIPANT × DISPARITY DATASET"
)


print(
    "=" * 90
)


print(
    f"Rows: "
    f"{len(cross_task_df)}"
)


print(
    f"Participants: "
    f"{cross_task_df['subject'].nunique()}"
)


print(
    "\nRows per disparity:"
)


print(
    cross_task_df[
        "disparity_deg"
    ]
    .value_counts()
    .sort_index()
)


# =============================================================================
# 5. RESIDUALIZE ALL MEASURES FOR SUBJECT + PHYSICAL DISPARITY
# =============================================================================

residual_specs = {

    "visual_bias_deg":
        "visual_bias_resid",

    "auditory_lag_sec":
        "lag_resid",

    "same_probability":
        "same_resid",

    "stable_unity":
        "unity_resid",
}


residualization_model_rows = []


for outcome, resid_col in residual_specs.items():

    residuals, fit = (
        _two_way_residualize(
            cross_task_df,
            outcome,
        )
    )


    cross_task_df[
        resid_col
    ] = residuals


    residualization_model_rows.append(
        {

            "outcome":
                outcome,

            "n_rows":
                int(
                    fit.nobs
                ),

            "r_squared_full_subject_plus_disparity":
                float(
                    fit.rsquared
                ),

            "adj_r_squared_full_subject_plus_disparity":
                float(
                    fit.rsquared_adj
                ),
        }
    )


residualization_models_df = pd.DataFrame(
    residualization_model_rows
)


# Within-participant z scores for PCA.
for raw_resid_col, z_col in [

    (
        "visual_bias_resid",
        "visual_bias_resid_z",
    ),

    (
        "lag_resid",
        "lag_resid_z",
    ),

    (
        "same_resid",
        "same_resid_z",
    ),

    (
        "unity_resid",
        "unity_resid_z",
    ),
]:

    cross_task_df[
        z_col
    ] = (
        cross_task_df
        .groupby(
            "subject"
        )[
            raw_resid_col
        ]
        .transform(
            _z_within_subject
        )
    )


# Use identical complete rows for ALL PCA models.
analysis_df = (
    cross_task_df
    .dropna(
        subset=[
            "visual_bias_resid_z",
            "lag_resid_z",
            "same_resid_z",
            "unity_resid_z",
        ]
    )
    .copy()
)


# Require at least 4 disparity levels per participant for meaningful
# within-participant association estimates.
n_levels_per_subject = (
    analysis_df
    .groupby(
        "subject"
    )[
        "disparity_deg"
    ]
    .nunique()
)


eligible_subjects = n_levels_per_subject[
    n_levels_per_subject
    >=
    4
].index


analysis_df = analysis_df[
    analysis_df[
        "subject"
    ]
    .isin(
        eligible_subjects
    )
].copy()


resid_cross_csv = (
    CSV_DIR_CROSS
    /
    "cross_task_participant_by_disparity_residualized.csv"
)


analysis_df.to_csv(
    resid_cross_csv,
    index=False,
)


residualization_models_df.to_csv(
    CSV_DIR_CROSS
    /
    "residualization_model_summary.csv",
    index=False,
)


print(
    "\nComplete cross-task PCA rows:"
)


print(
    len(
        analysis_df
    )
)


print(
    "Complete participants:"
)


print(
    analysis_df[
        "subject"
    ].nunique()
)


# =============================================================================
# 6. PAIRWISE RESIDUALIZED ASSOCIATIONS
# =============================================================================

pair_specs = [

    (
        "visual_bias_resid_z",
        "lag_resid_z",
        "visual_bias_vs_lag",
    ),

    (
        "visual_bias_resid_z",
        "same_resid_z",
        "visual_bias_vs_same",
    ),

    (
        "visual_bias_resid_z",
        "unity_resid_z",
        "visual_bias_vs_unity",
    ),

    (
        "lag_resid_z",
        "same_resid_z",
        "lag_vs_same",
    ),

    (
        "lag_resid_z",
        "unity_resid_z",
        "lag_vs_unity",
    ),

    (
        "same_resid_z",
        "unity_resid_z",
        "same_vs_unity",
    ),
]


subject_pair_rows = []


for subject, g in analysis_df.groupby(
    "subject"
):

    row = {
        "subject":
            subject,
    }


    for xcol, ycol, label in pair_specs:

        r, p, n = _safe_pearson(
            g[
                xcol
            ],
            g[
                ycol
            ],
        )


        row[
            f"r_{label}"
        ] = r


        row[
            f"p_{label}"
        ] = p


        row[
            f"n_{label}"
        ] = n


    subject_pair_rows.append(
        row
    )


subject_pair_df = pd.DataFrame(
    subject_pair_rows
)


subject_pair_df.to_csv(
    CSV_DIR_CROSS
    /
    "pairwise_residualized_associations_by_subject.csv",
    index=False,
)


# Fisher-z group summaries.
pair_summary_rows = []


for xcol, ycol, label in pair_specs:

    r_col = (
        f"r_{label}"
    )


    r_values = (
        subject_pair_df[
            r_col
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .clip(
            -0.999999,
            0.999999,
        )
        .values
    )


    if len(
        r_values
    ) < 2:

        continue


    fisher_z = np.arctanh(
        r_values
    )


    (
        z_mean,
        z_lo,
        z_hi,
        z_se,
        n_subjects,
    ) = _mean_ci(
        fisher_z
    )


    ttest = ttest_1samp(
        fisher_z,
        0.0,
    )


    # Descriptive pooled r across all participant × disparity residuals.
    pooled_r, pooled_p, pooled_n = _safe_pearson(
        analysis_df[
            xcol
        ],
        analysis_df[
            ycol
        ],
    )


    pair_summary_rows.append(
        {

            "association":
                label,

            "n_subjects":
                n_subjects,

            "mean_fisher_z":
                z_mean,

            "ci95_low_fisher_z":
                z_lo,

            "ci95_high_fisher_z":
                z_hi,

            "backtransformed_mean_r":
                np.tanh(
                    z_mean
                ),

            "backtransformed_ci95_low_r":
                np.tanh(
                    z_lo
                ),

            "backtransformed_ci95_high_r":
                np.tanh(
                    z_hi
                ),

            "t_fisher_z_vs_0":
                float(
                    ttest.statistic
                ),

            "df":
                n_subjects - 1,

            "p_parametric_fisher_z":
                float(
                    ttest.pvalue
                ),

            "p_signflip_fisher_z":
                _sign_flip_test(
                    fisher_z,
                    alternative=
                        "two-sided",
                ),

            "pooled_descriptive_r":
                pooled_r,

            "pooled_descriptive_p":
                pooled_p,

            "pooled_n_rows":
                pooled_n,
        }
    )


pair_summary_df = pd.DataFrame(
    pair_summary_rows
)


pair_summary_df.to_csv(
    CSV_DIR_CROSS
    /
    "pairwise_residualized_associations_group_summary.csv",
    index=False,
)


# =============================================================================
# 7. PCA HELPER
# =============================================================================

def run_cross_task_pca(
    data,
    variable_columns,
    variable_labels,
    theory_signs,
    model_name,
):

    """
    Run PCA on the same complete cross-task rows.

    Returns:
      variance_df
      loading_df
      score_df

    "PC loading" below means the correlation of the standardized observed
    variable with the PC score, not merely sklearn's eigenvector weight.

    PC1 is sign-oriented so that:
        + visual bias
        - auditory lag
        + Same
        + Unity

    corresponds to "more bound/integrated".
    """


    d = data.dropna(
        subset=
            variable_columns
    ).copy()


    X_raw = d[
        variable_columns
    ].values.astype(
        float
    )


    X = StandardScaler(
        with_mean=True,
        with_std=True,
    ).fit_transform(
        X_raw
    )


    n_components = len(
        variable_columns
    )


    pca = PCA(
        n_components=
            n_components
    )


    scores = pca.fit_transform(
        X
    )


    # Orient PC1 toward the hypothesized bound/integrated direction.
    pc1_weights = pca.components_[
        0,
        :
    ].copy()


    theory_signs = np.asarray(
        theory_signs,
        dtype=float,
    )


    if np.dot(
        pc1_weights,
        theory_signs,
    ) < 0:

        pca.components_[
            0,
            :
        ] *= -1.0


        scores[
            :,
            0
        ] *= -1.0


    component_names = [
        f"PC{i + 1}"
        for i
        in range(
            n_components
        )
    ]


    # Conventional variable-PC loadings:
    # correlation(observed standardized variable, PC score)
    loading_matrix = np.full(
        (
            len(
                variable_columns
            ),
            n_components,
        ),
        np.nan,
        dtype=float,
    )


    for j in range(
        len(
            variable_columns
        )
    ):

        for k in range(
            n_components
        ):

            loading_matrix[
                j,
                k
            ] = np.corrcoef(
                X[
                    :,
                    j
                ],
                scores[
                    :,
                    k
                ],
            )[0, 1]


    variance_df = pd.DataFrame(
        {

            "model":
                model_name,

            "component":
                component_names,

            "variance_explained":
                pca.explained_variance_ratio_,

            "cumulative_variance_explained":
                np.cumsum(
                    pca.explained_variance_ratio_
                ),
        }
    )


    loading_rows = []


    for j, label in enumerate(
        variable_labels
    ):

        pc1_loading = float(
            loading_matrix[
                j,
                0
            ]
        )


        pc1_communality = (
            pc1_loading
            **
            2
        )


        if n_components >= 2:

            pc12_communality = float(
                np.sum(
                    loading_matrix[
                        j,
                        :2
                    ]
                    **
                    2
                )
            )


        else:

            pc12_communality = np.nan


        row = {

            "model":
                model_name,

            "measure":
                label,

            "PC1_loading":
                pc1_loading,

            "PC1_communality":
                pc1_communality,

            "PC1_plus_PC2_communality":
                pc12_communality,

            "PC1_sklearn_weight":
                float(
                    pca.components_[
                        0,
                        j
                    ]
                ),

            "PC1_variance_explained":
                float(
                    pca.explained_variance_ratio_[
                        0
                    ]
                ),
        }


        for k, component_name in enumerate(
            component_names
        ):

            row[
                f"{component_name}_loading"
            ] = float(
                loading_matrix[
                    j,
                    k
                ]
            )


        loading_rows.append(
            row
        )


    loading_df = pd.DataFrame(
        loading_rows
    )


    score_df = d[
        [
            "subject",
            "disparity_deg",
        ]
    ].copy()


    for k, component_name in enumerate(
        component_names
    ):

        score_df[
            component_name
        ] = scores[
            :,
            k
        ]


    score_df[
        "model"
    ] = model_name


    return (
        variance_df,
        loading_df,
        score_df,
    )


# =============================================================================
# 8. PCA 1 — CURRENT 3-VARIABLE REFERENCE
#    bias + lag + Same
# =============================================================================

(
    pca1_variance,
    pca1_loadings,
    pca1_scores,
) = run_cross_task_pca(

    analysis_df,

    variable_columns=[
        "visual_bias_resid_z",
        "lag_resid_z",
        "same_resid_z",
    ],

    variable_labels=[
        "visual_bias",
        "auditory_lag",
        "same_report",
    ],

    theory_signs=[
        1.0,
        -1.0,
        1.0,
    ],

    model_name=
        "PCA_1_reference_bias_lag_same",
)


# =============================================================================
# 9. PCA 2 — CONTINUOUS UNITY 3-VARIABLE PCA
#    bias + lag + CI unity
# =============================================================================

(
    pca2_variance,
    pca2_loadings,
    pca2_scores,
) = run_cross_task_pca(

    analysis_df,

    variable_columns=[
        "visual_bias_resid_z",
        "lag_resid_z",
        "unity_resid_z",
    ],

    variable_labels=[
        "visual_bias",
        "auditory_lag",
        "continuous_unity",
    ],

    theory_signs=[
        1.0,
        -1.0,
        1.0,
    ],

    model_name=
        "PCA_2_bias_lag_continuous_unity",
)


# =============================================================================
# 10. PCA 3 — FULL 4-VARIABLE PCA
#     bias + lag + Same + CI unity
# =============================================================================

(
    pca3_variance,
    pca3_loadings,
    pca3_scores,
) = run_cross_task_pca(

    analysis_df,

    variable_columns=[
        "visual_bias_resid_z",
        "lag_resid_z",
        "same_resid_z",
        "unity_resid_z",
    ],

    variable_labels=[
        "visual_bias",
        "auditory_lag",
        "same_report",
        "continuous_unity",
    ],

    theory_signs=[
        1.0,
        -1.0,
        1.0,
        1.0,
    ],

    model_name=
        "PCA_3_full_bias_lag_same_unity",
)


# =============================================================================
# 11. SAVE + COMPARE PCA VARIANCE, LOADINGS, COMMUNALITIES
# =============================================================================

pca_variance_all = pd.concat(
    [
        pca1_variance,
        pca2_variance,
        pca3_variance,
    ],
    ignore_index=True,
)


pca_loadings_all = pd.concat(
    [
        pca1_loadings,
        pca2_loadings,
        pca3_loadings,
    ],
    ignore_index=True,
)


pca_scores_all = pd.concat(
    [
        pca1_scores,
        pca2_scores,
        pca3_scores,
    ],
    ignore_index=True,
)


pca_variance_all.to_csv(
    CSV_DIR_CROSS
    /
    "cross_task_pca_variance_explained_all_models.csv",
    index=False,
)


pca_loadings_all.to_csv(
    CSV_DIR_CROSS
    /
    "cross_task_pca_loadings_and_communalities_all_models.csv",
    index=False,
)


pca_scores_all.to_csv(
    CSV_DIR_CROSS
    /
    "cross_task_pca_scores_all_models.csv",
    index=False,
)


# Compact PC1 comparison table.
pc1_comparison_df = (
    pca_loadings_all[
        [
            "model",
            "measure",
            "PC1_loading",
            "PC1_communality",
            "PC1_plus_PC2_communality",
            "PC1_variance_explained",
        ]
    ]
    .copy()
)


pc1_comparison_df.to_csv(
    CSV_DIR_CROSS
    /
    "cross_task_pc1_loading_communality_comparison.csv",
    index=False,
)


# =============================================================================
# 12. MIXED MODEL
#     DOES CI UNITY ADD UNIQUE PREDICTION OF SL VISUAL BIAS?
# =============================================================================
#
# IMPORTANT:
# Use the RAW participant × disparity values here, not the two-way residuals.
#
# Physical disparity is controlled flexibly as C(disparity_cat), while a random
# participant intercept accounts for repeated observations.
#
# BASE:
#
#   visual bias
#       ~ disparity
#       + Same probability
#       + auditory lag
#       + (1 | subject)
#
# EXPANDED:
#
#   visual bias
#       ~ disparity
#       + Same probability
#       + auditory lag
#       + stable CI unity
#       + (1 | subject)
#
# This directly tests the incremental contribution of continuous CI unity.
# =============================================================================

mixed_df = (
    cross_task_df
    .dropna(
        subset=[
            "subject",
            "disparity_cat",
            "visual_bias_deg",
            "auditory_lag_sec",
            "same_probability",
            "stable_unity",
        ]
    )
    .copy()
)


# Standardize continuous variables globally for interpretable coefficients.
for col in [

    "visual_bias_deg",

    "auditory_lag_sec",

    "same_probability",

    "stable_unity",
]:

    values = pd.to_numeric(
        mixed_df[
            col
        ],
        errors="coerce",
    ).values.reshape(
        -1,
        1
    )


    mixed_df[
        f"{col}_z"
    ] = StandardScaler().fit_transform(
        values
    ).ravel()


base_formula = (
    "visual_bias_deg_z ~ "
    "C(disparity_cat) + "
    "same_probability_z + "
    "auditory_lag_sec_z"
)


expanded_formula = (
    "visual_bias_deg_z ~ "
    "C(disparity_cat) + "
    "same_probability_z + "
    "auditory_lag_sec_z + "
    "stable_unity_z"
)


base_fit, base_optimizer = (
    _fit_mixedlm_with_fallback(
        base_formula,
        mixed_df,
        group_col="subject",
    )
)


expanded_fit, expanded_optimizer = (
    _fit_mixedlm_with_fallback(
        expanded_formula,
        mixed_df,
        group_col="subject",
    )
)


base_r2 = (
    _nakagawa_r2_random_intercept(
        base_fit
    )
)


expanded_r2 = (
    _nakagawa_r2_random_intercept(
        expanded_fit
    )
)


lr_stat = float(
    2.0
    *
    (
        expanded_fit.llf
        -
        base_fit.llf
    )
)


lr_df = int(
    len(
        expanded_fit.fe_params
    )
    -
    len(
        base_fit.fe_params
    )
)


lr_p = float(
    chi2.sf(
        lr_stat,
        df=max(
            lr_df,
            1,
        ),
    )
)


unity_term = (
    "stable_unity_z"
)


unity_estimate = float(
    expanded_fit.params[
        unity_term
    ]
)


unity_se = float(
    expanded_fit.bse[
        unity_term
    ]
)


unity_z = (
    unity_estimate
    /
    unity_se
)


unity_p = float(
    expanded_fit.pvalues[
        unity_term
    ]
)


unity_ci = (
    expanded_fit
    .conf_int()
    .loc[
        unity_term
    ]
    .values
)


mixed_model_comparison_df = pd.DataFrame(
    [
        {

            "model":
                "base",

            "formula":
                base_formula,

            "optimizer":
                base_optimizer,

            "n_rows":
                int(
                    base_fit.nobs
                ),

            "n_subjects":
                mixed_df[
                    "subject"
                ].nunique(),

            "log_likelihood":
                float(
                    base_fit.llf
                ),

            "AIC":
                float(
                    base_fit.aic
                ),

            "BIC":
                float(
                    base_fit.bic
                ),

            "marginal_R2":
                base_r2[
                    "marginal_r2"
                ],

            "conditional_R2":
                base_r2[
                    "conditional_r2"
                ],
        },

        {

            "model":
                "expanded_with_CI_unity",

            "formula":
                expanded_formula,

            "optimizer":
                expanded_optimizer,

            "n_rows":
                int(
                    expanded_fit.nobs
                ),

            "n_subjects":
                mixed_df[
                    "subject"
                ].nunique(),

            "log_likelihood":
                float(
                    expanded_fit.llf
                ),

            "AIC":
                float(
                    expanded_fit.aic
                ),

            "BIC":
                float(
                    expanded_fit.bic
                ),

            "marginal_R2":
                expanded_r2[
                    "marginal_r2"
                ],

            "conditional_R2":
                expanded_r2[
                    "conditional_r2"
                ],
        },
    ]
)


incremental_unity_df = pd.DataFrame(
    [
        {

            "comparison":
                "expanded_with_CI_unity_vs_base",

            "LR_chi2":
                lr_stat,

            "LR_df":
                lr_df,

            "LR_p":
                lr_p,

            "delta_AIC_expanded_minus_base":
                float(
                    expanded_fit.aic
                    -
                    base_fit.aic
                ),

            "delta_BIC_expanded_minus_base":
                float(
                    expanded_fit.bic
                    -
                    base_fit.bic
                ),

            "base_marginal_R2":
                base_r2[
                    "marginal_r2"
                ],

            "expanded_marginal_R2":
                expanded_r2[
                    "marginal_r2"
                ],

            "delta_marginal_R2":
                (
                    expanded_r2[
                        "marginal_r2"
                    ]
                    -
                    base_r2[
                        "marginal_r2"
                    ]
                ),

            "base_conditional_R2":
                base_r2[
                    "conditional_r2"
                ],

            "expanded_conditional_R2":
                expanded_r2[
                    "conditional_r2"
                ],

            "unity_standardized_beta":
                unity_estimate,

            "unity_SE":
                unity_se,

            "unity_z":
                unity_z,

            "unity_p":
                unity_p,

            "unity_ci95_low":
                float(
                    unity_ci[
                        0
                    ]
                ),

            "unity_ci95_high":
                float(
                    unity_ci[
                        1
                    ]
                ),
        }
    ]
)


# Save all fixed effects from expanded model.
fixed_effect_rows = []


for term in expanded_fit.fe_params.index:

    ci = (
        expanded_fit
        .conf_int()
        .loc[
            term
        ]
        .values
    )


    fixed_effect_rows.append(
        {

            "term":
                term,

            "estimate":
                float(
                    expanded_fit.fe_params[
                        term
                    ]
                ),

            "SE":
                float(
                    expanded_fit.bse[
                        term
                    ]
                ),

            "z":
                float(
                    expanded_fit.fe_params[
                        term
                    ]
                    /
                    expanded_fit.bse[
                        term
                    ]
                ),

            "p":
                float(
                    expanded_fit.pvalues[
                        term
                    ]
                ),

            "ci95_low":
                float(
                    ci[
                        0
                    ]
                ),

            "ci95_high":
                float(
                    ci[
                        1
                    ]
                ),
        }
    )


expanded_fixed_effects_df = pd.DataFrame(
    fixed_effect_rows
)


mixed_model_comparison_df.to_csv(
    CSV_DIR_CROSS
    /
    "mixed_model_base_vs_unity_model_comparison.csv",
    index=False,
)


incremental_unity_df.to_csv(
    CSV_DIR_CROSS
    /
    "mixed_model_incremental_CI_unity_test.csv",
    index=False,
)


expanded_fixed_effects_df.to_csv(
    CSV_DIR_CROSS
    /
    "mixed_model_expanded_fixed_effects.csv",
    index=False,
)


# Save full text summaries too.
with open(
    OUT_DIR_CROSS
    /
    "mixed_model_base_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        base_fit
        .summary()
        .as_text()
    )


with open(
    OUT_DIR_CROSS
    /
    "mixed_model_expanded_with_unity_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        expanded_fit
        .summary()
        .as_text()
    )


# =============================================================================
# 13. FIGURE — PC1 LOADINGS ACROSS THE THREE PCA MODELS
# =============================================================================

plot_df = (
    pc1_comparison_df[
        [
            "model",
            "measure",
            "PC1_loading",
        ]
    ]
    .copy()
)


models = [

    "PCA_1_reference_bias_lag_same",

    "PCA_2_bias_lag_continuous_unity",

    "PCA_3_full_bias_lag_same_unity",
]


measure_order = [

    "visual_bias",

    "auditory_lag",

    "same_report",

    "continuous_unity",
]


fig, ax = plt.subplots(
    figsize=(
        10.5,
        6.0,
    )
)


x = np.arange(
    len(
        measure_order
    )
)


width = 0.24


for idx, model_name in enumerate(
    models
):

    sub = plot_df[
        plot_df[
            "model"
        ].eq(
            model_name
        )
    ]


    value_map = dict(
        zip(
            sub[
                "measure"
            ],
            sub[
                "PC1_loading"
            ],
        )
    )


    vals = [
        value_map.get(
            measure,
            np.nan,
        )

        for measure
        in measure_order
    ]


    ax.bar(
        x
        +
        (
            idx - 1
        )
        *
        width,

        vals,

        width=width,

        label=model_name,
    )


ax.axhline(
    0,
    linewidth=1,
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    [
        "Visual bias",
        "Auditory lag",
        "Same report",
        "Continuous unity",
    ]
)


ax.set_ylabel(
    "Oriented PC1 loading"
)


ax.set_title(
    "Shared integration-axis loadings across PCA specifications"
)


ax.legend(
    frameon=False,
    fontsize=8,
)


fig.tight_layout()


fig.savefig(
    FIG_DIR_CROSS
    /
    "cross_task_pc1_loadings_across_pca_models.png",
    dpi=FIG_DPI_CROSS,
    bbox_inches="tight",
)


plt.show()


# =============================================================================
# 14. FIGURE — PC1 VARIANCE EXPLAINED ACROSS MODELS
# =============================================================================

pc1_variance_plot = (
    pca_variance_all[
        pca_variance_all[
            "component"
        ].eq(
            "PC1"
        )
    ]
    .copy()
)


fig, ax = plt.subplots(
    figsize=(
        8.0,
        5.0,
    )
)


ax.bar(
    pc1_variance_plot[
        "model"
    ],
    pc1_variance_plot[
        "variance_explained"
    ],
)


ax.set_ylim(
    0,
    1,
)


ax.set_ylabel(
    "PC1 variance explained"
)


ax.set_title(
    "Variance captured by the shared integration axis"
)


ax.tick_params(
    axis="x",
    rotation=25,
)


fig.tight_layout()


fig.savefig(
    FIG_DIR_CROSS
    /
    "cross_task_pc1_variance_explained.png",
    dpi=FIG_DPI_CROSS,
    bbox_inches="tight",
)


plt.show()


# =============================================================================
# 15. FINAL CONSOLE REPORT
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)


print(
    "CROSS-TASK UNITY × SPATIAL READOUT ANALYSIS COMPLETE"
)


print(
    "=" * 100
)


print(
    "\nPRIMARY CI UNITY VARIABLE"
)


print(
    "-" * 100
)


print(
    "stable_unity = participant × disparity mean plateau_perceived_unity "
    "across CI trials classified stable_from_onset OR stabilizing."
)


print(
    "\nPAIRWISE RESIDUALIZED ASSOCIATIONS"
)


print(
    "-" * 100
)


print(
    pair_summary_df[
        [
            "association",
            "n_subjects",
            "backtransformed_mean_r",
            "backtransformed_ci95_low_r",
            "backtransformed_ci95_high_r",
            "p_parametric_fisher_z",
            "p_signflip_fisher_z",
        ]
    ]
    .to_string(
        index=False
    )
)


print(
    "\nPCA VARIANCE EXPLAINED"
)


print(
    "-" * 100
)


print(
    pca_variance_all.to_string(
        index=False
    )
)


print(
    "\nPC1 LOADINGS + COMMUNALITIES"
)


print(
    "-" * 100
)


print(
    pc1_comparison_df.to_string(
        index=False
    )
)


print(
    "\nMIXED MODEL — INCREMENTAL CONTRIBUTION OF CI UNITY"
)


print(
    "-" * 100
)


print(
    incremental_unity_df.to_string(
        index=False
    )
)


print(
    "\nEXPANDED MIXED-MODEL FIXED EFFECTS"
)


print(
    "-" * 100
)


print(
    expanded_fixed_effects_df.to_string(
        index=False
    )
)


print(
    "\nINTERPRETATION GUIDE"
)


print(
    "-" * 100
)


print(
    "1) PCA 1 is the participant × disparity analogue of the existing "
    "SL bias + lag + Same integration axis.\n"
    "2) PCA 2 asks whether continuous CI unity aligns with spatial bias "
    "and temporal lag when Same report is replaced by the graded CI state.\n"
    "3) PCA 3 asks whether Same report and continuous unity coexist on the "
    "same shared axis or separate across components.\n"
    "4) Compare PC1_loading and PC1_communality for visual bias across models; "
    "do NOT interpret a larger PC1 variance percentage by itself as unique "
    "variance explained by unity.\n"
    "5) The mixed-model unity coefficient, likelihood-ratio test, AIC/BIC change, "
    "and delta_marginal_R2 are the direct tests of whether continuous CI unity "
    "adds unique prediction of SL visual bias beyond disparity, Same probability, "
    "and auditory lag."
)


print(
    "\nOUTPUT DIRECTORY"
)


print(
    "-" * 100
)


print(
    OUT_DIR_CROSS
)


Using in-memory lag_master_df from the SL notebook.

SL participant × disparity rows:
200
SL subjects:
50
SL disparity levels:
[0.0, 10.0, 20.0, 30.0]

CI stable-state participant × disparity rows:
250
CI subjects:
50

Primary stable-unity group means:
                   mean       std  count
disparity_deg                           
0.0            0.877024  0.162064     50
10.0           0.828311  0.177921     50
20.0           0.657754  0.216323     50
30.0           0.376711  0.278526     50
40.0           0.104237  0.203977     50

CROSS-TASK PARTICIPANT × DISPARITY DATASET
Rows: 200
Participants: 50

Rows per disparity:
disparity_deg
0.0     50
10.0    50
20.0    50
30.0    50
Name: count, dtype: int64

Complete cross-task PCA rows:
200
Complete participants:
50


C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)



CROSS-TASK UNITY × SPATIAL READOUT ANALYSIS COMPLETE

PRIMARY CI UNITY VARIABLE
----------------------------------------------------------------------------------------------------
stable_unity = participant × disparity mean plateau_perceived_unity across CI trials classified stable_from_onset OR stabilizing.

PAIRWISE RESIDUALIZED ASSOCIATIONS
----------------------------------------------------------------------------------------------------
         association  n_subjects  backtransformed_mean_r  backtransformed_ci95_low_r  backtransformed_ci95_high_r  p_parametric_fisher_z  p_signflip_fisher_z
  visual_bias_vs_lag          50               -0.419166                   -0.664671                    -0.091964               0.014581             0.016983
 visual_bias_vs_same          50                0.594946                    0.304544                     0.784149               0.000522             0.001998
visual_bias_vs_unity          50                0.511848                    0

## Individual Differences in Causal Boundary 


In [36]:
# %% SUBJECT-LEVEL PSE / PSYCHOMETRIC SLOPE PREDICTION ANALYSES
# ============================================================
# Analyses:
# 2. Does PSE predict visual capture?
# 3. Does PSE predict localization hysteresis?
# 4. Does PSE predict recalibration / residual visual bias?
# 5. Do psychometric slopes predict capture, hysteresis, recalibration?
#
# Outputs:
# - subject-level PSE and psychometric slope table
# - subject-level visual capture table
# - subject-level recalibration/residual-bias table
# - merged prediction table
# - regression/correlation results
# - publication-friendly scatterplots with regression lines
# ============================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr, t as t_dist
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

OUT_DIR = os.path.join(DATA_DIR, "spatial_pse_capture_hysteresis_recalibration_predictions")
CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# Existing symmetric localization hysteresis output
HYSTERESIS_CSV = os.path.join(
    DATA_DIR,
    "spatial_localization_hysteresis_bias_vs_disparity",
    "subject_level_localization_hysteresis_index.csv"
)

EXCLUDE_PRACTICE = True
FIXED_OFFSETS = [10, 20, 30, 40]

MIN_VALID_SAMPLES = 8
MIN_ABS_DISPARITY_FOR_BIAS = 0.01

TRIAL_ANALYSIS_START_SEC = 0.0
TRIAL_ANALYSIS_END_SEC = 18.0

FIG_DPI = 180

# =========================
# HELPER FUNCTIONS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else Path(filepath).stem

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def parse_same_response(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    same_tokens = [
        "same", "same source", "common", "common source",
        "together", "one", "1", "yes", "y", "true"
    ]

    diff_tokens = [
        "different", "diff", "different source", "separate", "separate source",
        "apart", "two", "2", "0", "no", "n", "false"
    ]

    if s in same_tokens:
        return 1.0
    if s in diff_tokens:
        return 0.0
    if "same" in s or "common" in s:
        return 1.0
    if "different" in s or "separate" in s or "diff" in s:
        return 0.0

    return np.nan

def is_sweep_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    return ("sweep" in trial_type) or ("sweep" in cond) or ("diverge" in cond) or ("converge" in cond)

def is_fixed_offset_av_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    offset = extract_offset_from_row(row)

    if not np.isfinite(offset):
        return False

    if int(offset) not in FIXED_OFFSETS:
        return False

    if is_sweep_trial(row):
        return False

    if trial_type in ["audio_only", "visual_only", "congruent"]:
        return False

    if "incongruent" in trial_type or "fixed" in trial_type or "static" in trial_type:
        return True

    if "incongruent" in cond or "fixed" in cond or "static" in cond:
        return True

    return True

def normalized_visual_bias(response, visual, audio, min_abs_disparity=0.01):
    """
    Normalized visual bias / visual capture:
        (response - audio) / (visual - audio)

    0   = auditory location
    0.5 = midpoint
    1   = visual location
    """
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    denom = visual - audio

    ok = (
        np.isfinite(response)
        & np.isfinite(visual)
        & np.isfinite(audio)
        & (np.abs(denom) > min_abs_disparity)
    )

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean((response[ok] - audio[ok]) / denom[ok]))

def visual_directed_bias_deg(response, visual, audio, min_abs_disparity=0.01):
    """
    Bias toward visual stimulus in degrees.
    Positive = response shifted toward visual.
    """
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    disparity_signed = visual - audio
    visual_direction = np.sign(disparity_signed)

    ok = (
        np.isfinite(response)
        & np.isfinite(visual)
        & np.isfinite(audio)
        & np.isfinite(visual_direction)
        & (np.abs(disparity_signed) > min_abs_disparity)
    )

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean((response[ok] - audio[ok]) * visual_direction[ok]))

def bias_a_minus_v(response, visual, audio):
    """
    Bias_AminusV = |response - audio| - |response - visual|

    Positive = response is closer to visual than audio.
    Negative = response is closer to audio than visual.
    """
    response = np.asarray(response, dtype=float)
    visual = np.asarray(visual, dtype=float)
    audio = np.asarray(audio, dtype=float)

    ok = np.isfinite(response) & np.isfinite(visual) & np.isfinite(audio)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.nanmean(np.abs(response[ok] - audio[ok]) - np.abs(response[ok] - visual[ok])))

def monotonic_nonincreasing(y):
    y = np.asarray(y, dtype=float)
    out = y.copy()

    for i in range(1, len(out)):
        if np.isfinite(out[i]) and np.isfinite(out[i - 1]):
            out[i] = min(out[i], out[i - 1])

    return out

def summarize_psame_by_offset(offsets, same_vals, fixed_offsets=FIXED_OFFSETS, smoothing=0.5):
    d = pd.DataFrame({"offset": offsets, "same": same_vals}).dropna()
    rows = []

    for off in fixed_offsets:
        sub = d[d["offset"] == off]
        n = len(sub)
        n_same = int(np.nansum(sub["same"])) if n > 0 else 0

        if n > 0:
            psame_raw = n_same / n
            psame_smooth = (n_same + smoothing) / (n + 2 * smoothing)
        else:
            psame_raw = np.nan
            psame_smooth = np.nan

        rows.append({
            "offset_deg": float(off),
            "n": n,
            "n_same": n_same,
            "psame_raw": psame_raw,
            "psame_smooth": psame_smooth,
        })

    out = pd.DataFrame(rows)
    out["psame_monotonic"] = monotonic_nonincreasing(out["psame_smooth"].values)
    return out

def bounded_empirical_pse(offsets, same_vals, fixed_offsets=FIXED_OFFSETS):
    """
    Stable PSE estimate:
    - uses smoothed empirical P(same) at each offset
    - enforces monotonic non-increasing P(same)
    - interpolates P(same)=.5
    - bounded to tested offset range
    """
    ps = summarize_psame_by_offset(offsets, same_vals, fixed_offsets=fixed_offsets)
    ps_valid = ps.dropna(subset=["psame_monotonic"]).copy()

    if len(ps_valid) == 0:
        return np.nan, "no_valid_psame", ps

    x = ps_valid["offset_deg"].values.astype(float)
    y = ps_valid["psame_monotonic"].values.astype(float)

    if np.all(y >= 0.5):
        return float(np.max(x)), "bounded_empirical_all_psame_ge_0.5", ps

    if np.all(y < 0.5):
        return float(np.min(x)), "bounded_empirical_all_psame_lt_0.5", ps

    for i in range(len(x) - 1):
        y1, y2 = y[i], y[i + 1]
        x1, x2 = x[i], x[i + 1]

        if y1 >= 0.5 and y2 < 0.5:
            if abs(y2 - y1) < 1e-12:
                pse = x1
            else:
                pse = x1 + (0.5 - y1) * (x2 - x1) / (y2 - y1)

            pse = float(np.clip(pse, min(fixed_offsets), max(fixed_offsets)))
            return pse, "bounded_empirical_interpolated_psame50", ps

    eligible = ps_valid[ps_valid["psame_monotonic"] >= 0.5]
    if len(eligible) > 0:
        return float(eligible["offset_deg"].max()), "bounded_empirical_largest_offset_psame_ge_0.5", ps

    return float(np.min(x)), "bounded_empirical_fallback_min", ps

def logistic_slope_and_pse(offsets, same_vals, fixed_offsets=FIXED_OFFSETS):
    """
    Diagnostic logistic fit:
        P(same) ~ offset

    Returns:
        logistic_pse_deg
        logistic_slope_raw
        logistic_precision = -slope
            Larger positive values = sharper decrease in P(same) with offset.
    """
    d = pd.DataFrame({"offset": offsets, "same": same_vals}).dropna()

    if len(d) < 4:
        return np.nan, np.nan, np.nan, "too_few_trials"

    if d["same"].nunique() < 2:
        return np.nan, np.nan, np.nan, "no_response_variation"

    try:
        X = sm.add_constant(d["offset"].astype(float).values)
        y = d["same"].astype(float).values

        model = sm.GLM(y, X, family=sm.families.Binomial()).fit()

        intercept = float(model.params[0])
        slope = float(model.params[1])

        if not np.isfinite(slope) or abs(slope) < 1e-12:
            return np.nan, slope, np.nan, "invalid_slope"

        pse = -intercept / slope

        if not np.isfinite(pse):
            pse = np.nan

        # Do not use invalid out-of-range PSE as primary, but still keep slope if direction is sensible.
        if slope < 0:
            precision = -slope
            method = "valid_negative_slope"
        else:
            precision = np.nan
            method = "positive_slope_rejected_for_precision"

        if np.isfinite(pse):
            pse = float(np.clip(pse, min(fixed_offsets), max(fixed_offsets)))

        return pse, slope, precision, method

    except Exception as e:
        return np.nan, np.nan, np.nan, f"logistic_failed_{type(e).__name__}"

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)
    return m, m - crit * se, m + crit * se, n

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def safe_corr(x, y):
    d = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()

    if len(d) < 3:
        return {
            "n": len(d),
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "spearman_rho": np.nan,
            "spearman_p": np.nan,
        }

    pr, pp = pearsonr(d["x"], d["y"])
    sr, sp = spearmanr(d["x"], d["y"])

    return {
        "n": len(d),
        "pearson_r": pr,
        "pearson_p": pp,
        "spearman_rho": sr,
        "spearman_p": sp,
    }

def fit_subject_regression(df, x, y):
    d = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(d) < 4 or d[x].std(ddof=1) == 0:
        return None, None

    model = smf.ols(f"{y} ~ {x}", data=d).fit()
    return model, d

def scatter_regression_plot(df, x, y, xlabel, ylabel, title, outpath):
    d = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(d) < 4:
        print(f"Skipping plot, not enough data: {title}")
        return

    model = smf.ols(f"{y} ~ {x}", data=d).fit()
    corr = safe_corr(d[x], d[y])

    x_grid = np.linspace(d[x].min(), d[x].max(), 100)
    pred = model.get_prediction(pd.DataFrame({x: x_grid})).summary_frame(alpha=0.05)

    fig, ax = plt.subplots(figsize=(6.4, 5.4))

    ax.scatter(d[x], d[y], alpha=0.75)

    ax.plot(
        x_grid,
        pred["mean"],
        linewidth=2.2,
        label="OLS fit"
    )

    ax.fill_between(
        x_grid,
        pred["mean_ci_lower"].values,
        pred["mean_ci_upper"].values,
        alpha=0.20,
        label="95% CI"
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    txt = (
        f"n = {corr['n']}\n"
        f"r = {corr['pearson_r']:.3f}, p {p_fmt(corr['pearson_p'])}\n"
        f"β = {model.params[x]:.4f}, p {p_fmt(model.pvalues[x])}"
    )

    ax.text(
        0.04,
        0.96,
        txt,
        transform=ax.transAxes,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
    )

    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# =========================
# STEP 1 — BUILD FIXED-OFFSET TRIAL-LEVEL DATA
# =========================
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in {DATA_DIR}")

trial_rows = []

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    df = pd.read_csv(fpath)

    required_cols = [
        "trialIndex", "trialType", "condLabel", "isPractice",
        "t", "mouse_x", "blob_x", "audio_x"
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"Skipping {subj}: missing {missing}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"],
        sort=False
    ):
        d = df_trial.copy().sort_values("t")
        first_row = d.iloc[0]
        row_series = pd.Series(first_row)

        if not is_fixed_offset_av_trial(row_series):
            continue

        t = d["t"].values.astype(float)
        t = t - np.nanmin(t)

        analysis_mask = (
            np.isfinite(t)
            & (t >= TRIAL_ANALYSIS_START_SEC)
            & (t <= TRIAL_ANALYSIS_END_SEC)
        )

        if analysis_mask.sum() < MIN_VALID_SAMPLES:
            continue

        mouse_x = d["mouse_x"].values.astype(float)[analysis_mask]
        blob_x = d["blob_x"].values.astype(float)[analysis_mask]
        audio_x = d["audio_x"].values.astype(float)[analysis_mask]

        offset = extract_offset_from_row(row_series)

        same_response = np.nan
        if "post_question" in d.columns:
            same_response = parse_same_response(first_row.get("post_question", np.nan))

        if not np.isfinite(same_response) and "post_question2" in d.columns:
            same_response = parse_same_response(first_row.get("post_question2", np.nan))

        norm_capture = normalized_visual_bias(
            mouse_x,
            blob_x,
            audio_x,
            min_abs_disparity=MIN_ABS_DISPARITY_FOR_BIAS
        )

        visual_bias_deg = visual_directed_bias_deg(
            mouse_x,
            blob_x,
            audio_x,
            min_abs_disparity=MIN_ABS_DISPARITY_FOR_BIAS
        )

        bias_amv = bias_a_minus_v(mouse_x, blob_x, audio_x)

        trial_rows.append({
            "subject": subj,
            "source_file": Path(fpath).name,
            "trialIndex": trial_index,
            "trialType": trial_type,
            "condLabel": cond_label,
            "offset_deg": offset,
            "same_response": same_response,
            "different_response": 1.0 - same_response if np.isfinite(same_response) else np.nan,

            "NormalizedVisualBias": norm_capture,
            "VisualDirectedBias_deg": visual_bias_deg,
            "AbsVisualDirectedBias_deg": np.abs(visual_bias_deg) if np.isfinite(visual_bias_deg) else np.nan,
            "Bias_AminusV": bias_amv,

            "post_question": first_row.get("post_question", np.nan),
            "post_question2": first_row.get("post_question2", np.nan),
        })

fixed_trial_df = pd.DataFrame(trial_rows)

if len(fixed_trial_df) == 0:
    raise RuntimeError("No fixed-offset audiovisual trials were found.")

fixed_trial_csv = os.path.join(CSV_DIR, "fixed_offset_trial_level_capture_psame_bias.csv")
fixed_trial_df.to_csv(fixed_trial_csv, index=False)

print(f"Saved fixed-offset trial-level table: {fixed_trial_csv}")
print(f"Subjects: {fixed_trial_df['subject'].nunique()}")
print(f"Fixed-offset trials: {len(fixed_trial_df)}")

# =========================
# STEP 2 — SUBJECT-LEVEL PSE + PSYCHOMETRIC SLOPE
# =========================
pse_rows = []
psame_rows = []

for subj, sub in fixed_trial_df.groupby("subject"):
    pse, pse_method, ps_table = bounded_empirical_pse(
        sub["offset_deg"].values,
        sub["same_response"].values,
        fixed_offsets=FIXED_OFFSETS
    )

    logistic_pse, logistic_slope, psychometric_precision, logistic_method = logistic_slope_and_pse(
        sub["offset_deg"].values,
        sub["same_response"].values,
        fixed_offsets=FIXED_OFFSETS
    )

    ps_table = ps_table.copy()
    ps_table["subject"] = subj
    psame_rows.append(ps_table)

    pse_rows.append({
        "subject": subj,
        "n_fixed_trials": len(sub),
        "n_psame_trials": int(sub["same_response"].notna().sum()),
        "n_same": int((sub["same_response"] == 1).sum()),
        "n_different": int((sub["same_response"] == 0).sum()),

        "PSE_empirical_deg": pse,
        "PSE_empirical_method": pse_method,

        "PSE_logistic_deg": logistic_pse,
        "psychometric_slope_raw": logistic_slope,
        "psychometric_precision": psychometric_precision,
        "psychometric_slope_method": logistic_method,
    })

pse_df = pd.DataFrame(pse_rows)
psame_by_offset_df = pd.concat(psame_rows, ignore_index=True)

pse_csv = os.path.join(CSV_DIR, "subject_level_pse_psychometric_slope.csv")
psame_csv = os.path.join(CSV_DIR, "subject_psame_by_offset.csv")

pse_df.to_csv(pse_csv, index=False)
psame_by_offset_df.to_csv(psame_csv, index=False)

print(f"Saved subject PSE/slope table: {pse_csv}")
print(f"Saved subject P(same) by offset: {psame_csv}")

# =========================
# STEP 3 — SUBJECT-LEVEL CAPTURE
# =========================
capture_subject = (
    fixed_trial_df
    .groupby("subject", as_index=False)
    .agg(
        mean_capture_overall=("NormalizedVisualBias", "mean"),
        mean_visual_bias_deg_overall=("VisualDirectedBias_deg", "mean"),
        mean_bias_AminusV_overall=("Bias_AminusV", "mean"),
        n_fixed_trials=("NormalizedVisualBias", "count"),
    )
)

offset_capture = (
    fixed_trial_df
    .groupby(["subject", "offset_deg"], as_index=False)
    .agg(
        mean_capture=("NormalizedVisualBias", "mean"),
        mean_visual_bias_deg=("VisualDirectedBias_deg", "mean"),
        mean_bias_AminusV=("Bias_AminusV", "mean"),
        n_trials=("NormalizedVisualBias", "count"),
    )
)

for offset in FIXED_OFFSETS:
    sub = offset_capture[offset_capture["offset_deg"] == offset][
        ["subject", "mean_capture", "mean_visual_bias_deg", "mean_bias_AminusV"]
    ].copy()

    sub = sub.rename(columns={
        "mean_capture": f"mean_capture_{offset}deg",
        "mean_visual_bias_deg": f"mean_visual_bias_deg_{offset}deg",
        "mean_bias_AminusV": f"mean_bias_AminusV_{offset}deg",
    })

    capture_subject = pd.merge(
        capture_subject,
        sub,
        on="subject",
        how="left"
    )

capture_csv = os.path.join(CSV_DIR, "subject_level_visual_capture_by_offset.csv")
capture_subject.to_csv(capture_csv, index=False)

print(f"Saved subject visual-capture table: {capture_csv}")

# =========================
# STEP 4 — SUBJECT-LEVEL RECALIBRATION / RESIDUAL VISUAL BIAS
# =========================
# Trial-level residual: Bias_AminusV residualized for same/different judgment and offset.
# Positive residual = more visually biased than expected from offset + explicit judgment.

recal_trial_df = fixed_trial_df.copy()
recal_trial_df = recal_trial_df.replace([np.inf, -np.inf], np.nan)

resid_model_df = recal_trial_df.dropna(
    subset=["Bias_AminusV", "offset_deg", "same_response"]
).copy()

if len(resid_model_df) >= 10 and resid_model_df["subject"].nunique() >= 3:
    resid_model = smf.ols(
        "Bias_AminusV ~ C(offset_deg) + same_response",
        data=resid_model_df
    ).fit()

    resid_model_df["ResidualBias_AminusV"] = resid_model.resid

    recal_trial_df = pd.merge(
        recal_trial_df,
        resid_model_df[["subject", "trialIndex", "condLabel", "ResidualBias_AminusV"]],
        on=["subject", "trialIndex", "condLabel"],
        how="left"
    )
else:
    print("Not enough data for residual-bias model. ResidualBias_AminusV will be NaN.")
    recal_trial_df["ResidualBias_AminusV"] = np.nan

recal_subject = (
    recal_trial_df
    .groupby("subject", as_index=False)
    .agg(
        mean_residual_bias_all_offsets=("ResidualBias_AminusV", "mean"),
        mean_raw_bias_AminusV_all_offsets=("Bias_AminusV", "mean"),
    )
)

for offset in [30, 40]:
    sub = (
        recal_trial_df[recal_trial_df["offset_deg"] == offset]
        .groupby("subject", as_index=False)
        .agg(
            mean_residual_bias=("ResidualBias_AminusV", "mean"),
            mean_raw_bias_AminusV=("Bias_AminusV", "mean"),
            mean_capture=("NormalizedVisualBias", "mean"),
            mean_visual_bias_deg=("VisualDirectedBias_deg", "mean"),
        )
    )

    sub = sub.rename(columns={
        "mean_residual_bias": f"mean_residual_bias_{offset}deg",
        "mean_raw_bias_AminusV": f"mean_raw_bias_AminusV_{offset}deg",
        "mean_capture": f"mean_capture_{offset}deg_for_recal",
        "mean_visual_bias_deg": f"mean_visual_bias_deg_{offset}deg_for_recal",
    })

    recal_subject = pd.merge(
        recal_subject,
        sub,
        on="subject",
        how="left"
    )

recal_subject["mean_residual_bias_30_40deg"] = recal_subject[
    ["mean_residual_bias_30deg", "mean_residual_bias_40deg"]
].mean(axis=1)

recal_subject["mean_raw_bias_AminusV_30_40deg"] = recal_subject[
    ["mean_raw_bias_AminusV_30deg", "mean_raw_bias_AminusV_40deg"]
].mean(axis=1)

recal_trial_csv = os.path.join(CSV_DIR, "trial_level_residual_bias_recalibration.csv")
recal_subject_csv = os.path.join(CSV_DIR, "subject_level_recalibration_residual_bias.csv")

recal_trial_df.to_csv(recal_trial_csv, index=False)
recal_subject.to_csv(recal_subject_csv, index=False)

print(f"Saved trial-level recalibration table: {recal_trial_csv}")
print(f"Saved subject-level recalibration table: {recal_subject_csv}")

# =========================
# STEP 5 — LOAD HYSTERESIS INDEX
# =========================
if os.path.exists(HYSTERESIS_CSV):
    hysteresis_df = pd.read_csv(HYSTERESIS_CSV)
    hysteresis_df["subject"] = hysteresis_df["subject"].astype(str).str.lower()
    print(f"Loaded hysteresis index: {HYSTERESIS_CSV}")
else:
    print(f"WARNING: Hysteresis file not found. Hysteresis predictions will be skipped:\n{HYSTERESIS_CSV}")
    hysteresis_df = pd.DataFrame({"subject": pse_df["subject"]})

# =========================
# STEP 6 — MERGE SUBJECT-LEVEL TABLE
# =========================
subject_df = pse_df.copy()
subject_df["subject"] = subject_df["subject"].astype(str).str.lower()

for df_to_merge in [capture_subject, recal_subject, hysteresis_df]:
    if "subject" in df_to_merge.columns:
        df_to_merge = df_to_merge.copy()
        df_to_merge["subject"] = df_to_merge["subject"].astype(str).str.lower()
        subject_df = pd.merge(subject_df, df_to_merge, on="subject", how="left")

subject_merged_csv = os.path.join(CSV_DIR, "subject_level_pse_capture_hysteresis_recalibration_merged.csv")
subject_df.to_csv(subject_merged_csv, index=False)

print(f"Saved merged subject-level prediction table: {subject_merged_csv}")

# =========================
# STEP 7 — REGRESSION / CORRELATION TESTS
# =========================
prediction_specs = [
    # PSE -> visual capture
    {
        "analysis": "PSE predicts overall visual capture",
        "x": "PSE_empirical_deg",
        "y": "mean_capture_overall",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Mean normalized visual capture",
        "title": "Does PSE predict visual capture?",
        "filename": "pse_predicts_mean_visual_capture.png",
    },
    {
        "analysis": "PSE predicts 10deg visual capture",
        "x": "PSE_empirical_deg",
        "y": "mean_capture_10deg",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Mean visual capture at 10°",
        "title": "PSE predicting visual capture at 10°",
        "filename": "pse_predicts_capture_10deg.png",
    },
    {
        "analysis": "PSE predicts 20deg visual capture",
        "x": "PSE_empirical_deg",
        "y": "mean_capture_20deg",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Mean visual capture at 20°",
        "title": "PSE predicting visual capture at 20°",
        "filename": "pse_predicts_capture_20deg.png",
    },
    {
        "analysis": "PSE predicts 30deg visual capture",
        "x": "PSE_empirical_deg",
        "y": "mean_capture_30deg",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Mean visual capture at 30°",
        "title": "PSE predicting visual capture at 30°",
        "filename": "pse_predicts_capture_30deg.png",
    },
    {
        "analysis": "PSE predicts 40deg visual capture",
        "x": "PSE_empirical_deg",
        "y": "mean_capture_40deg",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Mean visual capture at 40°",
        "title": "PSE predicting visual capture at 40°",
        "filename": "pse_predicts_capture_40deg.png",
    },

    # PSE -> hysteresis
    {
        "analysis": "PSE predicts localization hysteresis",
        "x": "PSE_empirical_deg",
        "y": "mean_CaptureHysteresis",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Localization hysteresis index\n(Converge − Diverge capture)",
        "title": "Does PSE predict localization hysteresis?",
        "filename": "pse_predicts_localization_hysteresis.png",
    },

    # PSE -> recalibration
    {
        "analysis": "PSE predicts residual bias 30/40deg",
        "x": "PSE_empirical_deg",
        "y": "mean_residual_bias_30_40deg",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Residual visual bias at 30°/40°\n(Bias_AminusV residual)",
        "title": "Does PSE predict residual visual bias?",
        "filename": "pse_predicts_residual_bias_30_40.png",
    },
    {
        "analysis": "PSE predicts raw bias 30/40deg",
        "x": "PSE_empirical_deg",
        "y": "mean_raw_bias_AminusV_30_40deg",
        "xlabel": "Subject PSE: P(same)=.5 boundary (deg)",
        "ylabel": "Raw visual bias at 30°/40°\n(|resp-audio| - |resp-visual|)",
        "title": "Does PSE predict raw visual bias after segregation?",
        "filename": "pse_predicts_raw_bias_30_40.png",
    },

    # Psychometric precision -> capture
    {
        "analysis": "Psychometric precision predicts visual capture",
        "x": "psychometric_precision",
        "y": "mean_capture_overall",
        "xlabel": "Psychometric precision (-logistic slope)",
        "ylabel": "Mean normalized visual capture",
        "title": "Does psychometric precision predict visual capture?",
        "filename": "slope_predicts_mean_visual_capture.png",
    },

    # Psychometric precision -> hysteresis
    {
        "analysis": "Psychometric precision predicts hysteresis",
        "x": "psychometric_precision",
        "y": "mean_CaptureHysteresis",
        "xlabel": "Psychometric precision (-logistic slope)",
        "ylabel": "Localization hysteresis index\n(Converge − Diverge capture)",
        "title": "Does psychometric precision predict hysteresis?",
        "filename": "slope_predicts_localization_hysteresis.png",
    },

    # Psychometric precision -> recalibration
    {
        "analysis": "Psychometric precision predicts residual bias 30/40deg",
        "x": "psychometric_precision",
        "y": "mean_residual_bias_30_40deg",
        "xlabel": "Psychometric precision (-logistic slope)",
        "ylabel": "Residual visual bias at 30°/40°",
        "title": "Does psychometric precision predict residual visual bias?",
        "filename": "slope_predicts_residual_bias_30_40.png",
    },
]

regression_rows = []

for spec in prediction_specs:
    x = spec["x"]
    y = spec["y"]

    if x not in subject_df.columns or y not in subject_df.columns:
        print(f"Skipping {spec['analysis']}: missing {x} or {y}")
        continue

    model, d = fit_subject_regression(subject_df, x=x, y=y)

    if model is None:
        print(f"Skipping {spec['analysis']}: not enough valid data.")
        continue

    corr = safe_corr(d[x], d[y])
    m_y, lo_y, hi_y, n_y = mean_ci(d[y])

    regression_rows.append({
        "analysis": spec["analysis"],
        "x": x,
        "y": y,
        "n": len(d),
        "x_mean": d[x].mean(),
        "x_sd": d[x].std(ddof=1),
        "y_mean": d[y].mean(),
        "y_ci_low": lo_y,
        "y_ci_high": hi_y,
        "intercept": model.params.get("Intercept", np.nan),
        "beta": model.params.get(x, np.nan),
        "se": model.bse.get(x, np.nan),
        "t": model.tvalues.get(x, np.nan),
        "p": model.pvalues.get(x, np.nan),
        "r_squared": model.rsquared,
        "pearson_r": corr["pearson_r"],
        "pearson_p": corr["pearson_p"],
        "spearman_rho": corr["spearman_rho"],
        "spearman_p": corr["spearman_p"],
    })

    outpath = os.path.join(FIG_DIR, spec["filename"])

    scatter_regression_plot(
        subject_df,
        x=x,
        y=y,
        xlabel=spec["xlabel"],
        ylabel=spec["ylabel"],
        title=spec["title"],
        outpath=outpath
    )

regression_results_df = pd.DataFrame(regression_rows)
regression_csv = os.path.join(CSV_DIR, "subject_level_pse_slope_prediction_results.csv")
regression_results_df.to_csv(regression_csv, index=False)

print(f"Saved prediction results: {regression_csv}")

# =========================
# STEP 8 — SUMMARY PRINT
# =========================
print("\n============================================================")
print("SUBJECT-LEVEL PSE / SLOPE PREDICTION ANALYSES COMPLETE")
print("============================================================")

print("\nPSE summary:")
print(pse_df[[
    "PSE_empirical_deg",
    "psychometric_slope_raw",
    "psychometric_precision"
]].describe())

print("\nKey prediction results:")
if len(regression_results_df) > 0:
    display_cols = [
        "analysis",
        "n",
        "beta",
        "se",
        "t",
        "p",
        "r_squared",
        "pearson_r",
        "pearson_p",
        "spearman_rho",
        "spearman_p",
    ]
    print(regression_results_df[display_cols].to_string(index=False))
else:
    print("No prediction models were run.")

print("\nSaved files:")
print(f"Fixed trial-level table:  {fixed_trial_csv}")
print(f"PSE / slope table:       {pse_csv}")
print(f"P(same) by offset table: {psame_csv}")
print(f"Capture table:           {capture_csv}")
print(f"Recal trial table:       {recal_trial_csv}")
print(f"Recal subject table:     {recal_subject_csv}")
print(f"Merged subject table:    {subject_merged_csv}")
print(f"Prediction results:      {regression_csv}")
print(f"Figures folder:          {FIG_DIR}")

print("\nInterpretation guide:")
print("- PSE_empirical_deg = causal tolerance; higher values mean the subject reports 'same' at larger disparities.")
print("- psychometric_precision = -logistic slope; higher values mean sharper same-to-different transition.")
print("- mean_capture_overall and mean_capture_*deg index localization visual capture.")
print("- mean_CaptureHysteresis is the symmetric localization hysteresis index: Converge minus Diverge capture.")
print("- mean_residual_bias_30_40deg indexes residual visual bias after accounting for offset and same/different judgment.")




Saved fixed-offset trial-level table: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_pse_capture_hysteresis_recalibration_predictions\csv_outputs\fixed_offset_trial_level_capture_psame_bias.csv
Subjects: 50
Fixed-offset trials: 4000
Saved subject PSE/slope table: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_pse_capture_hysteresis_recalibration_predictions\csv_outputs\subject_level_pse_psychometric_slope.csv
Saved subject P(same) by offset: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_pse_capture_hysteresis_recalibration_predictions\csv_outputs\subject_psame_by_offset.csv
Saved subject visual-capture table: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_pse_capture_hysteresis_recalibration_predictions\csv_outputs\subject_level_visual_capture_by_offset.csv
Saved trial-lev

In [37]:
# %% CELL 1 — MIXED-EFFECTS PSYCHOMETRIC CURVE + PREDICTED PSE ROBUSTNESS CHECK
# ============================================================
# Goal:
# 1. Fit a group-level mixed-effects psychometric curve:
#       same_response ~ offset + random subject intercept/slope
#
# 2. Extract:
#       - group-level psychometric curve
#       - model-predicted subject-level PSEs
#       - empirical subject-level PSEs
#
# 3. Correlate:
#       Predicted PSE vs Empirical PSE
#
# Notes:
# - The primary empirical PSE is computed as in your prior notebook:
#     smoothed empirical P(same) by offset
#     monotonic non-increasing correction
#     bounded interpolation at P(same)=.50
#
# - The mixed-effects model uses statsmodels BinomialBayesMixedGLM.
# - If random slope extraction fails, the code will still save the group curve
#   and will fall back to fixed-effect-only predicted PSEs for transparency.
# ============================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit
from scipy.stats import pearsonr, spearmanr, t as t_dist
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

OUT_DIR = os.path.join(DATA_DIR, "spatial_mixed_effects_psychometric_pse")
CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

EXCLUDE_PRACTICE = True
FIXED_OFFSETS = [10, 20, 30, 40]
FIG_DPI = 180

# =========================
# HELPERS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else Path(filepath).stem

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def parse_same_response(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    same_tokens = [
        "same", "same source", "common", "common source",
        "together", "one", "1", "yes", "y", "true"
    ]

    diff_tokens = [
        "different", "diff", "different source", "separate",
        "separate source", "apart", "two", "2", "0",
        "no", "n", "false"
    ]

    if s in same_tokens:
        return 1.0
    if s in diff_tokens:
        return 0.0
    if "same" in s or "common" in s:
        return 1.0
    if "different" in s or "separate" in s or "diff" in s:
        return 0.0

    return np.nan

def is_sweep_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    return ("sweep" in trial_type) or ("sweep" in cond) or ("diverge" in cond) or ("converge" in cond)

def is_fixed_offset_av_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    offset = extract_offset_from_row(row)

    if not np.isfinite(offset):
        return False

    if int(offset) not in FIXED_OFFSETS:
        return False

    if is_sweep_trial(row):
        return False

    if trial_type in ["audio_only", "visual_only", "congruent"]:
        return False

    if "incongruent" in trial_type or "fixed" in trial_type or "static" in trial_type:
        return True

    if "incongruent" in cond or "fixed" in cond or "static" in cond:
        return True

    return True

def monotonic_nonincreasing(y):
    y = np.asarray(y, dtype=float)
    out = y.copy()

    for i in range(1, len(out)):
        if np.isfinite(out[i]) and np.isfinite(out[i - 1]):
            out[i] = min(out[i], out[i - 1])

    return out

def summarize_psame_by_offset(offsets, same_vals, fixed_offsets=FIXED_OFFSETS, smoothing=0.5):
    d = pd.DataFrame({"offset": offsets, "same": same_vals}).dropna()
    rows = []

    for off in fixed_offsets:
        sub = d[d["offset"] == off]
        n = len(sub)
        n_same = int(np.nansum(sub["same"])) if n > 0 else 0

        if n > 0:
            psame_raw = n_same / n
            psame_smooth = (n_same + smoothing) / (n + 2 * smoothing)
        else:
            psame_raw = np.nan
            psame_smooth = np.nan

        rows.append({
            "offset_deg": float(off),
            "n": n,
            "n_same": n_same,
            "psame_raw": psame_raw,
            "psame_smooth": psame_smooth,
        })

    out = pd.DataFrame(rows)
    out["psame_monotonic"] = monotonic_nonincreasing(out["psame_smooth"].values)
    return out

def bounded_empirical_pse(offsets, same_vals, fixed_offsets=FIXED_OFFSETS):
    ps = summarize_psame_by_offset(offsets, same_vals, fixed_offsets=fixed_offsets)
    ps_valid = ps.dropna(subset=["psame_monotonic"]).copy()

    if len(ps_valid) == 0:
        return np.nan, "no_valid_psame", ps

    x = ps_valid["offset_deg"].values.astype(float)
    y = ps_valid["psame_monotonic"].values.astype(float)

    if np.all(y >= 0.5):
        return float(np.max(x)), "bounded_all_psame_ge_0.5", ps

    if np.all(y < 0.5):
        return float(np.min(x)), "bounded_all_psame_lt_0.5", ps

    for i in range(len(x) - 1):
        y1, y2 = y[i], y[i + 1]
        x1, x2 = x[i], x[i + 1]

        if y1 >= 0.5 and y2 < 0.5:
            pse = x1 + (0.5 - y1) * (x2 - x1) / (y2 - y1)
            pse = float(np.clip(pse, min(fixed_offsets), max(fixed_offsets)))
            return pse, "interpolated_psame50", ps

    eligible = ps_valid[ps_valid["psame_monotonic"] >= 0.5]
    if len(eligible) > 0:
        return float(eligible["offset_deg"].max()), "largest_offset_psame_ge_0.5", ps

    return float(np.min(x)), "fallback_min", ps

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)
    return m, m - crit * se, m + crit * se, n

def extract_random_effects_from_bayes_model(result, subjects):
    """
    Attempts to extract subject random intercepts/slopes from BinomialBayesMixedGLM.
    This function is intentionally robust because statsmodels names can vary.
    """
    re_intercepts = {s: 0.0 for s in subjects}
    re_slopes = {s: 0.0 for s in subjects}

    try:
        re_df = result.random_effects()

        if callable(re_df):
            re_df = re_df()

        if isinstance(re_df, pd.DataFrame):
            for idx, row in re_df.iterrows():
                name = str(idx)
                val = float(row["Mean"]) if "Mean" in row.index else float(row.iloc[0])

                for subj in subjects:
                    if subj in name:
                        if "offset_deg_c" in name or ":offset" in name or "offset" in name:
                            re_slopes[subj] += val
                        else:
                            re_intercepts[subj] += val

        return re_intercepts, re_slopes, True

    except Exception as e:
        print(f"WARNING: Could not extract random effects cleanly: {type(e).__name__}: {e}")
        return re_intercepts, re_slopes, False

# =========================
# STEP 1 — BUILD FIXED-OFFSET SAME/DIFFERENT TABLE
# =========================
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No files matching {FILE_PATTERN} found in {DATA_DIR}")

rows = []

for fpath in csv_files:
    subj = extract_subject_id(fpath)
    df = pd.read_csv(fpath)

    required_cols = ["trialIndex", "trialType", "condLabel", "isPractice", "post_question", "post_question2"]
    missing_core = [c for c in ["trialIndex", "trialType", "condLabel", "isPractice"] if c not in df.columns]
    if missing_core:
        print(f"Skipping {subj}: missing {missing_core}")
        continue

    if EXCLUDE_PRACTICE:
        df = df[df["isPractice"] == 0].copy()

    for (trial_index, trial_type, cond_label), df_trial in df.groupby(
        ["trialIndex", "trialType", "condLabel"],
        sort=False
    ):
        first_row = df_trial.iloc[0]
        row_series = pd.Series(first_row)

        if not is_fixed_offset_av_trial(row_series):
            continue

        offset = extract_offset_from_row(row_series)

        same_response = np.nan

        if "post_question" in df_trial.columns:
            same_response = parse_same_response(first_row.get("post_question", np.nan))

        if not np.isfinite(same_response) and "post_question2" in df_trial.columns:
            same_response = parse_same_response(first_row.get("post_question2", np.nan))

        rows.append({
            "subject": subj,
            "source_file": Path(fpath).name,
            "trialIndex": trial_index,
            "trialType": trial_type,
            "condLabel": cond_label,
            "offset_deg": float(offset),
            "same_response": same_response,
            "post_question": first_row.get("post_question", np.nan),
            "post_question2": first_row.get("post_question2", np.nan),
        })

fixed_psych_df = pd.DataFrame(rows)
fixed_psych_df = fixed_psych_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["subject", "offset_deg", "same_response"]
).copy()

fixed_psych_df["same_response"] = fixed_psych_df["same_response"].astype(int)
fixed_psych_df["subject"] = fixed_psych_df["subject"].astype(str).str.lower()

if len(fixed_psych_df) == 0:
    raise RuntimeError("No fixed-offset same/different trials were found.")

mean_offset = fixed_psych_df["offset_deg"].mean()
fixed_psych_df["offset_deg_c"] = fixed_psych_df["offset_deg"] - mean_offset

fixed_psych_csv = os.path.join(CSV_DIR, "fixed_offset_same_different_trial_table.csv")
fixed_psych_df.to_csv(fixed_psych_csv, index=False)

print(f"Saved fixed-offset psychometric trial table: {fixed_psych_csv}")
print(f"Subjects: {fixed_psych_df['subject'].nunique()}")
print(f"Trials:   {len(fixed_psych_df)}")

# =========================
# STEP 2 — EMPIRICAL SUBJECT-LEVEL PSE
# =========================
emp_rows = []
psame_rows = []

for subj, sub in fixed_psych_df.groupby("subject"):
    pse, method, ps_table = bounded_empirical_pse(
        sub["offset_deg"].values,
        sub["same_response"].values
    )

    ps_table = ps_table.copy()
    ps_table["subject"] = subj
    psame_rows.append(ps_table)

    emp_rows.append({
        "subject": subj,
        "n_trials": len(sub),
        "n_same": int(sub["same_response"].sum()),
        "n_different": int((sub["same_response"] == 0).sum()),
        "EmpiricalPSE_deg": pse,
        "EmpiricalPSE_method": method,
    })

empirical_pse_df = pd.DataFrame(emp_rows)
empirical_psame_by_offset_df = pd.concat(psame_rows, ignore_index=True)

empirical_pse_csv = os.path.join(CSV_DIR, "empirical_subject_level_pse.csv")
empirical_psame_csv = os.path.join(CSV_DIR, "empirical_subject_psame_by_offset.csv")

empirical_pse_df.to_csv(empirical_pse_csv, index=False)
empirical_psame_by_offset_df.to_csv(empirical_psame_csv, index=False)

# =========================
# STEP 3 — MIXED-EFFECTS PSYCHOMETRIC CURVE
# =========================
subjects = sorted(fixed_psych_df["subject"].unique())

mixed_success = False
random_effects_success = False

try:
    vc_formulas = {
        "subject_intercept": "0 + C(subject)",
        "subject_slope": "0 + C(subject):offset_deg_c",
    }

    model = sm.BinomialBayesMixedGLM.from_formula(
        "same_response ~ offset_deg_c",
        vc_formulas,
        fixed_psych_df
    )

    mixed_result = model.fit_vb()
    mixed_success = True

    fixed_intercept = float(mixed_result.fe_mean[0])
    fixed_slope = float(mixed_result.fe_mean[1])

    print("\nMixed-effects psychometric model fit succeeded.")
    print(f"Fixed intercept = {fixed_intercept:.4f}")
    print(f"Fixed offset slope = {fixed_slope:.4f}")

    re_intercepts, re_slopes, random_effects_success = extract_random_effects_from_bayes_model(
        mixed_result,
        subjects
    )

except Exception as e:
    print(f"\nWARNING: Mixed-effects model failed: {type(e).__name__}: {e}")
    print("Falling back to pooled logistic model for group curve.")
    mixed_result = None
    fixed_psych_df_for_glm = fixed_psych_df.copy()

    glm = sm.GLM(
        fixed_psych_df_for_glm["same_response"],
        sm.add_constant(fixed_psych_df_for_glm["offset_deg_c"]),
        family=sm.families.Binomial()
    ).fit()

    fixed_intercept = float(glm.params["const"])
    fixed_slope = float(glm.params["offset_deg_c"])

    re_intercepts = {s: 0.0 for s in subjects}
    re_slopes = {s: 0.0 for s in subjects}

# =========================
# STEP 4 — PREDICTED SUBJECT-LEVEL PSE FROM MIXED MODEL
# =========================
pred_rows = []

for subj in subjects:
    subj_intercept = fixed_intercept + re_intercepts.get(subj, 0.0)
    subj_slope = fixed_slope + re_slopes.get(subj, 0.0)

    if np.isfinite(subj_slope) and subj_slope < 0:
        pse_c = -subj_intercept / subj_slope
        predicted_pse = pse_c + mean_offset
        predicted_pse_bounded = float(np.clip(predicted_pse, min(FIXED_OFFSETS), max(FIXED_OFFSETS)))
        method = "mixed_subject_intercept_slope"
    else:
        predicted_pse = np.nan
        predicted_pse_bounded = np.nan
        method = "invalid_or_nonnegative_slope"

    pred_rows.append({
        "subject": subj,
        "fixed_intercept": fixed_intercept,
        "fixed_slope": fixed_slope,
        "subject_random_intercept": re_intercepts.get(subj, 0.0),
        "subject_random_slope": re_slopes.get(subj, 0.0),
        "subject_model_intercept": subj_intercept,
        "subject_model_slope": subj_slope,
        "PredictedPSE_deg_unbounded": predicted_pse,
        "PredictedPSE_deg": predicted_pse_bounded,
        "PredictedPSE_method": method,
        "mixed_model_success": mixed_success,
        "random_effects_success": random_effects_success,
    })

predicted_pse_df = pd.DataFrame(pred_rows)

# If all subject predicted PSE values are invalid, fall back to group fixed PSE
if predicted_pse_df["PredictedPSE_deg"].notna().sum() == 0:
    print("WARNING: All subject-specific predicted PSEs were invalid. Using group fixed-effect PSE for all subjects.")
    group_pse_c = -fixed_intercept / fixed_slope
    group_pse = float(np.clip(group_pse_c + mean_offset, min(FIXED_OFFSETS), max(FIXED_OFFSETS)))

    predicted_pse_df["PredictedPSE_deg"] = group_pse
    predicted_pse_df["PredictedPSE_deg_unbounded"] = group_pse
    predicted_pse_df["PredictedPSE_method"] = "group_fixed_effect_fallback"

pse_compare_df = pd.merge(
    empirical_pse_df,
    predicted_pse_df,
    on="subject",
    how="inner"
)

pse_compare_csv = os.path.join(CSV_DIR, "mixed_model_predicted_pse_vs_empirical_pse.csv")
pse_compare_df.to_csv(pse_compare_csv, index=False)

# =========================
# STEP 5 — SUBJECT × OFFSET PREDICTED SAME-SOURCE PROBABILITY
# =========================
pred_grid_rows = []

for _, r in predicted_pse_df.iterrows():
    subj = r["subject"]
    subj_intercept = r["subject_model_intercept"]
    subj_slope = r["subject_model_slope"]

    for offset in FIXED_OFFSETS:
        offset_c = offset - mean_offset
        predicted_same_prob = expit(subj_intercept + subj_slope * offset_c)

        pred_grid_rows.append({
            "subject": subj,
            "offset_deg": float(offset),
            "PredictedSameProb": predicted_same_prob,
            "PredictedPSE_deg": r["PredictedPSE_deg"],
            "subject_model_intercept": subj_intercept,
            "subject_model_slope": subj_slope,
        })

subject_offset_pred_df = pd.DataFrame(pred_grid_rows)

subject_offset_pred_csv = os.path.join(CSV_DIR, "mixed_model_subject_offset_predicted_same_probability.csv")
subject_offset_pred_df.to_csv(subject_offset_pred_csv, index=False)

# =========================
# STEP 6 — GROUP PSYCHOMETRIC CURVE PLOT
# =========================
x_grid = np.linspace(min(FIXED_OFFSETS), max(FIXED_OFFSETS), 200)
x_grid_c = x_grid - mean_offset
group_pred = expit(fixed_intercept + fixed_slope * x_grid_c)

group_pse = float(np.clip((-fixed_intercept / fixed_slope) + mean_offset, min(FIXED_OFFSETS), max(FIXED_OFFSETS)))

group_summary = (
    fixed_psych_df
    .groupby("offset_deg", as_index=False)
    .agg(
        mean_same=("same_response", "mean"),
        n=("same_response", "count")
    )
)

fig, ax = plt.subplots(figsize=(7.4, 5.6))

# Subject empirical means
for subj, sub in empirical_psame_by_offset_df.groupby("subject"):
    ax.plot(
        sub["offset_deg"],
        sub["psame_raw"],
        linewidth=0.8,
        alpha=0.18
    )

# Group raw points
ax.scatter(
    group_summary["offset_deg"],
    group_summary["mean_same"],
    s=70,
    label="Observed group proportion"
)

# Mixed model curve
ax.plot(
    x_grid,
    group_pred,
    linewidth=2.6,
    label="Mixed-effects psychometric curve"
)

ax.axhline(0.5, linestyle="--", linewidth=1.1, alpha=0.75)
ax.axvline(group_pse, linestyle="--", linewidth=1.1, alpha=0.75)

ax.text(
    0.05,
    0.08,
    f"Group Predicted PSE = {group_pse:.2f}°",
    transform=ax.transAxes,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)

ax.set_xlabel("A-V offset magnitude (deg)")
ax.set_ylabel("Probability of same-source judgment")
ax.set_title("Mixed-effects psychometric curve")
ax.set_ylim(-0.05, 1.05)
ax.legend(frameon=False)

fig.tight_layout()
group_curve_path = os.path.join(FIG_DIR, "mixed_effects_group_psychometric_curve.png")
fig.savefig(group_curve_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# =========================
# STEP 7 — ROBUSTNESS CHECK: PREDICTED PSE VS EMPIRICAL PSE
# =========================
d = pse_compare_df[["EmpiricalPSE_deg", "PredictedPSE_deg"]].replace([np.inf, -np.inf], np.nan).dropna()

if len(d) >= 3:
    pr, pp = pearsonr(d["EmpiricalPSE_deg"], d["PredictedPSE_deg"])
    sr, sp = spearmanr(d["EmpiricalPSE_deg"], d["PredictedPSE_deg"])
else:
    pr = pp = sr = sp = np.nan

fig, ax = plt.subplots(figsize=(6.4, 5.6))

ax.scatter(
    d["EmpiricalPSE_deg"],
    d["PredictedPSE_deg"],
    alpha=0.8
)

if len(d) >= 3:
    fit = np.polyfit(d["EmpiricalPSE_deg"], d["PredictedPSE_deg"], 1)
    xline = np.linspace(d["EmpiricalPSE_deg"].min(), d["EmpiricalPSE_deg"].max(), 100)
    yline = fit[0] * xline + fit[1]
    ax.plot(xline, yline, linewidth=2.2, label="OLS fit")

ax.plot(
    [min(FIXED_OFFSETS), max(FIXED_OFFSETS)],
    [min(FIXED_OFFSETS), max(FIXED_OFFSETS)],
    linestyle="--",
    linewidth=1.0,
    alpha=0.7,
    label="Identity line"
)

txt = (
    f"n = {len(d)}\n"
    f"Pearson r = {pr:.3f}, p {p_fmt(pp)}\n"
    f"Spearman ρ = {sr:.3f}, p {p_fmt(sp)}"
)

ax.text(
    0.05,
    0.95,
    txt,
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
)

ax.set_xlabel("Empirical PSE (deg)")
ax.set_ylabel("Predicted PSE (deg)")
ax.set_title("Robustness check: Predicted PSE vs Empirical PSE")
ax.legend(frameon=False)

fig.tight_layout()
pse_corr_path = os.path.join(FIG_DIR, "predicted_pse_vs_empirical_pse.png")
fig.savefig(pse_corr_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# =========================
# STEP 8 — SAVE GROUP CURVE
# =========================
group_curve_df = pd.DataFrame({
    "offset_deg": x_grid,
    "predicted_same_probability": group_pred,
    "group_predicted_pse_deg": group_pse,
    "fixed_intercept": fixed_intercept,
    "fixed_slope": fixed_slope,
    "mean_offset_centering": mean_offset,
})

group_curve_csv = os.path.join(CSV_DIR, "mixed_effects_group_psychometric_curve.csv")
group_curve_df.to_csv(group_curve_csv, index=False)

# =========================
# FINAL PRINT
# =========================
print("\n============================================================")
print("MIXED-EFFECTS PSYCHOMETRIC PSE ANALYSIS COMPLETE")
print("============================================================")
print(f"Fixed trial table:                {fixed_psych_csv}")
print(f"Empirical subject PSE:            {empirical_pse_csv}")
print(f"Empirical P(same) by offset:      {empirical_psame_csv}")
print(f"Predicted vs empirical PSE:       {pse_compare_csv}")
print(f"Subject-offset predicted values:  {subject_offset_pred_csv}")
print(f"Group psychometric curve CSV:     {group_curve_csv}")
print(f"Group curve figure:               {group_curve_path}")
print(f"PSE robustness figure:            {pse_corr_path}")

print("\nGroup-level psychometric parameters:")
print(f"Group Predicted PSE = {group_pse:.4f} deg")
print(f"Fixed intercept = {fixed_intercept:.4f}")
print(f"Fixed slope = {fixed_slope:.4f}")

print("\nPredicted PSE vs Empirical PSE robustness check:")
print(f"n = {len(d)}")
print(f"Pearson r = {pr:.4f}, p {p_fmt(pp)}")
print(f"Spearman rho = {sr:.4f}, p {p_fmt(sp)}")

print("\nSubject-level PSE summary:")
print(pse_compare_df[["EmpiricalPSE_deg", "PredictedPSE_deg"]].describe())

Saved fixed-offset psychometric trial table: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL\spatial_mixed_effects_psychometric_pse\csv_outputs\fixed_offset_same_different_trial_table.csv
Subjects: 50
Trials:   4000

Mixed-effects psychometric model fit succeeded.
Fixed intercept = -0.4506
Fixed offset slope = -0.1345


OSError: [Errno 22] Invalid argument: 'C:\\Users\\pkell\\OneDrive\\Documents\\Grad School\\Shams Lab\\Continuous AVT v1\\individual_data_SL\\spatial_mixed_effects_psychometric_pse\\figures\\mixed_effects_group_psychometric_curve.png'

In [ ]:
# %% ANALYSIS 8 — EXPLORATORY STRONG VS WEAK INTEGRATORS BY PSE
# ============================================================
# Question:
# Do high-PSE participants differ from low-PSE participants in:
#   - visual capture
#   - localization hysteresis
#   - residual visual bias / recalibration
#   - lag
#   - lag-corrected RMSE
#
# Grouping:
#   Median split on subject-level PSE.
#
# Note:
#   This is descriptive/exploratory. Keep continuous PSE regressions as the
#   primary analyses; use median split for visualization and interpretation.
# ============================================================

import os
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, mannwhitneyu, t as t_dist

# =========================
# USER SETTINGS
# =========================
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\individual_data_SL"
FILE_PATTERN = "*master_trials*.csv"

OUT_DIR = os.path.join(DATA_DIR, "spatial_strong_vs_weak_integrators_by_PSE")
CSV_DIR = os.path.join(OUT_DIR, "csv_outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, CSV_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

MERGED_PSE_CSV = os.path.join(
    DATA_DIR,
    "spatial_pse_capture_hysteresis_recalibration_predictions",
    "csv_outputs",
    "subject_level_pse_capture_hysteresis_recalibration_merged.csv"
)

PSE_CSV = os.path.join(
    DATA_DIR,
    "spatial_pse_capture_hysteresis_recalibration_predictions",
    "csv_outputs",
    "subject_level_pse_psychometric_slope.csv"
)

CAPTURE_CSV = os.path.join(
    DATA_DIR,
    "spatial_pse_capture_hysteresis_recalibration_predictions",
    "csv_outputs",
    "subject_level_visual_capture_by_offset.csv"
)

RECAL_CSV = os.path.join(
    DATA_DIR,
    "spatial_pse_capture_hysteresis_recalibration_predictions",
    "csv_outputs",
    "subject_level_recalibration_residual_bias.csv"
)

HYSTERESIS_CSV = os.path.join(
    DATA_DIR,
    "spatial_localization_hysteresis_bias_vs_disparity",
    "subject_level_localization_hysteresis_index.csv"
)

EXCLUDE_PRACTICE = True
MIN_VALID_SAMPLES = 8
FIXED_OFFSETS = [10, 20, 30, 40]

# Lag analysis settings
RESAMPLE_HZ = 20.0
TRIAL_DURATION_SEC = 18.0
TIME_GRID = np.arange(0, TRIAL_DURATION_SEC + 1.0 / RESAMPLE_HZ, 1.0 / RESAMPLE_HZ)

MAX_LAG_SEC = 2.0
LAG_STEP_SEC = 0.05

TRIAL_ANALYSIS_START_SEC = 0.0
TRIAL_ANALYSIS_END_SEC = 18.0

FIG_DPI = 180

# =========================
# HELPERS
# =========================
def extract_subject_id(filepath):
    name = Path(filepath).name
    m = re.search(r"(pid\d+)", name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else Path(filepath).stem

def mean_ci(vals, alpha=0.05):
    vals = pd.Series(vals).dropna().astype(float).values
    n = len(vals)

    if n < 2:
        return np.nan, np.nan, np.nan, n

    m = np.mean(vals)
    se = np.std(vals, ddof=1) / np.sqrt(n)
    crit = t_dist.ppf(1 - alpha / 2, df=n - 1)
    return m, m - crit * se, m + crit * se, n

def sem(vals):
    vals = pd.Series(vals).dropna().astype(float).values
    if len(vals) < 2:
        return np.nan
    return np.std(vals, ddof=1) / np.sqrt(len(vals))

def p_fmt(p):
    if pd.isna(p):
        return "nan"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"

def cohens_d_independent(x1, x2):
    x1 = pd.Series(x1).dropna().astype(float).values
    x2 = pd.Series(x2).dropna().astype(float).values

    if len(x1) < 2 or len(x2) < 2:
        return np.nan

    n1, n2 = len(x1), len(x2)
    s1, s2 = np.var(x1, ddof=1), np.var(x2, ddof=1)
    pooled = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))

    if pooled == 0 or not np.isfinite(pooled):
        return np.nan

    return (np.mean(x1) - np.mean(x2)) / pooled

def extract_offset_from_row(row):
    if "offsetDeg" in row.index and pd.notna(row["offsetDeg"]):
        try:
            return int(round(abs(float(row["offsetDeg"]))))
        except Exception:
            pass

    cond = str(row.get("condLabel", "")).lower()
    m = re.search(r"(10|20|30|40)", cond)
    if m:
        return int(m.group(1))

    return np.nan

def is_sweep_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    return ("sweep" in trial_type) or ("sweep" in cond) or ("diverge" in cond) or ("converge" in cond)

def is_fixed_offset_av_trial(row):
    trial_type = str(row.get("trialType", "")).lower()
    cond = str(row.get("condLabel", "")).lower()
    offset = extract_offset_from_row(row)

    if not np.isfinite(offset):
        return False

    if int(offset) not in FIXED_OFFSETS:
        return False

    if is_sweep_trial(row):
        return False

    if trial_type in ["audio_only", "visual_only", "congruent"]:
        return False

    if "incongruent" in trial_type or "fixed" in trial_type or "static" in trial_type:
        return True

    if "incongruent" in cond or "fixed" in cond or "static" in cond:
        return True

    return True

def resample_trial(df_trial, time_grid):
    needed = ["t", "mouse_x", "blob_x", "audio_x"]
    for c in needed:
        if c not in df_trial.columns:
            raise ValueError(f"Missing required column: {c}")

    d = df_trial.copy()
    d = d.sort_values("t")
    d = d[np.isfinite(d["t"])]
    d = d.drop_duplicates(subset="t", keep="first")

    if len(d) < 5:
        return None

    t_original = d["t"].values.astype(float)
    t_original = t_original - np.nanmin(t_original)

    if np.nanmax(t_original) < 1.0:
        return None

    def _interp(col):
        y = d[col].values.astype(float)
        ok = np.isfinite(t_original) & np.isfinite(y)

        if ok.sum() < 3:
            return np.full_like(time_grid, np.nan, dtype=float)

        return np.interp(
            time_grid,
            t_original[ok],
            y[ok],
            left=np.nan,
            right=np.nan
        )

    return {
        "t_grid": time_grid.copy(),
        "mouse_x": _interp("mouse_x"),
        "blob_x": _interp("blob_x"),
        "audio_x": _interp("audio_x"),
    }

def rmse(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)

    if ok.sum() < MIN_VALID_SAMPLES:
        return np.nan

    return float(np.sqrt(np.nanmean((x[ok] - y[ok]) ** 2)))

def best_lag_and_rmse(response, target, t_grid, max_lag_sec=2.0, lag_step_sec=0.05):
    """
    Finds response lag that minimizes RMSE between response(t - lag) and target(t).

    Positive best_lag_sec means the response is delayed relative to the target.
    """
    response = np.asarray(response, dtype=float)
    target = np.asarray(target, dtype=float)
    t_grid = np.asarray(t_grid, dtype=float)

    lag_grid = np.arange(-max_lag_sec, max_lag_sec + 1e-9, lag_step_sec)

    best_lag = np.nan
    best_rmse = np.nan

    for lag in lag_grid:
        shifted_response = np.interp(
            t_grid,
            t_grid + lag,
            response,
            left=np.nan,
            right=np.nan
        )

        val = rmse(shifted_response, target)

        if np.isfinite(val) and (not np.isfinite(best_rmse) or val < best_rmse):
            best_rmse = val
            best_lag = lag

    return float(best_lag), float(best_rmse)

# =========================
# LOAD / MERGE SUBJECT-LEVEL DATA
# =========================
if os.path.exists(MERGED_PSE_CSV):
    subject_df = pd.read_csv(MERGED_PSE_CSV)
    print(f"Loaded merged subject file: {MERGED_PSE_CSV}")
else:
    dfs = []

    for path, label in [
        (PSE_CSV, "PSE"),
        (CAPTURE_CSV, "Capture"),
        (RECAL_CSV, "Recalibration"),
        (HYSTERESIS_CSV, "Hysteresis"),
    ]:
        if os.path.exists(path):
            d = pd.read_csv(path)
            d["subject"] = d["subject"].astype(str).str.lower()
            dfs.append(d)
            print(f"Loaded {label}: {path}")
        else:
            print(f"WARNING: Missing {label}: {path}")

    if len(dfs) == 0:
        raise FileNotFoundError("No subject-level input files found.")

    subject_df = dfs[0]
    for d in dfs[1:]:
        subject_df = pd.merge(subject_df, d, on="subject", how="outer")

subject_df["subject"] = subject_df["subject"].astype(str).str.lower()

if "PSE_empirical_deg" not in subject_df.columns:
    raise ValueError("PSE_empirical_deg not found in the subject-level table.")

# =========================
# COMPUTE LAG AND LAG-CORRECTED RMSE IF NOT ALREADY PRESENT
# =========================
need_lag = not all(c in subject_df.columns for c in [
    "mean_best_lag_visual_sec",
    "mean_lag_corrected_RMSE_visual",
    "mean_uncorrected_RMSE_visual"
])

if need_lag:
    print("Lag/RMSE columns not found. Computing trial-level lag and lag-corrected RMSE from master trial files...")

    csv_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN)))

    lag_rows = []

    for fpath in csv_files:
        subj = extract_subject_id(fpath)
        df = pd.read_csv(fpath)

        required_cols = [
            "trialIndex", "trialType", "condLabel", "isPractice",
            "t", "mouse_x", "blob_x", "audio_x"
        ]

        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            print(f"Skipping {subj}: missing {missing}")
            continue

        if EXCLUDE_PRACTICE:
            df = df[df["isPractice"] == 0].copy()

        for (trial_index, trial_type, cond_label), df_trial in df.groupby(
            ["trialIndex", "trialType", "condLabel"],
            sort=False
        ):
            first_row = df_trial.iloc[0]
            row_series = pd.Series(first_row)

            if not is_fixed_offset_av_trial(row_series):
                continue

            resampled = resample_trial(df_trial, TIME_GRID)

            if resampled is None:
                continue

            t = np.asarray(resampled["t_grid"], dtype=float)
            mouse_x = np.asarray(resampled["mouse_x"], dtype=float)
            blob_x = np.asarray(resampled["blob_x"], dtype=float)

            mask = (
                np.isfinite(t)
                & (t >= TRIAL_ANALYSIS_START_SEC)
                & (t <= TRIAL_ANALYSIS_END_SEC)
            )

            if mask.sum() < MIN_VALID_SAMPLES:
                continue

            best_lag, corrected_rmse = best_lag_and_rmse(
                response=mouse_x[mask],
                target=blob_x[mask],
                t_grid=t[mask],
                max_lag_sec=MAX_LAG_SEC,
                lag_step_sec=LAG_STEP_SEC
            )

            uncorrected_rmse = rmse(mouse_x[mask], blob_x[mask])

            lag_rows.append({
                "subject": subj,
                "trialIndex": trial_index,
                "trialType": trial_type,
                "condLabel": cond_label,
                "offset_deg": extract_offset_from_row(row_series),
                "best_lag_visual_sec": best_lag,
                "lag_corrected_RMSE_visual": corrected_rmse,
                "uncorrected_RMSE_visual": uncorrected_rmse,
            })

    lag_trial_df = pd.DataFrame(lag_rows)

    if len(lag_trial_df) > 0:
        lag_subject = (
            lag_trial_df
            .groupby("subject", as_index=False)
            .agg(
                mean_best_lag_visual_sec=("best_lag_visual_sec", "mean"),
                mean_lag_corrected_RMSE_visual=("lag_corrected_RMSE_visual", "mean"),
                mean_uncorrected_RMSE_visual=("uncorrected_RMSE_visual", "mean"),
            )
        )

        lag_trial_csv = os.path.join(CSV_DIR, "trial_level_lag_corrected_rmse.csv")
        lag_subject_csv = os.path.join(CSV_DIR, "subject_level_lag_corrected_rmse.csv")

        lag_trial_df.to_csv(lag_trial_csv, index=False)
        lag_subject.to_csv(lag_subject_csv, index=False)

        subject_df = pd.merge(subject_df, lag_subject, on="subject", how="left")

        print(f"Saved trial-level lag/RMSE file:   {lag_trial_csv}")
        print(f"Saved subject-level lag/RMSE file: {lag_subject_csv}")
    else:
        print("No lag/RMSE rows were created. Lag/RMSE group comparisons will be skipped.")

# =========================
# MEDIAN SPLIT
# =========================
pse_median = subject_df["PSE_empirical_deg"].median(skipna=True)

subject_df["PSE_Group"] = np.where(
    subject_df["PSE_empirical_deg"] >= pse_median,
    "High PSE",
    "Low PSE"
)

subject_df["PSE_Group"] = pd.Categorical(
    subject_df["PSE_Group"],
    categories=["Low PSE", "High PSE"],
    ordered=True
)

# =========================
# OUTCOME VARIABLES
# =========================
candidate_outcomes = [
    ("mean_capture_overall", "Mean visual capture"),
    ("mean_capture_10deg", "Visual capture 10°"),
    ("mean_capture_20deg", "Visual capture 20°"),
    ("mean_capture_30deg", "Visual capture 30°"),
    ("mean_capture_40deg", "Visual capture 40°"),

    ("mean_CaptureHysteresis", "Localization hysteresis\n(Converge − Diverge capture)"),
    ("mean_BiasDegHysteresis", "Hysteresis in degrees"),

    ("mean_residual_bias_30_40deg", "Residual visual bias 30°/40°"),
    ("mean_raw_bias_AminusV_30_40deg", "Raw visual bias 30°/40°"),

    ("mean_best_lag_visual_sec", "Best visual lag (s)"),
    ("mean_lag_corrected_RMSE_visual", "Lag-corrected visual RMSE"),
    ("mean_uncorrected_RMSE_visual", "Uncorrected visual RMSE"),
]

outcomes = [(c, label) for c, label in candidate_outcomes if c in subject_df.columns]

if len(outcomes) == 0:
    raise RuntimeError("No outcome columns found for high/low PSE comparison.")

# =========================
# GROUP COMPARISONS
# =========================
summary_rows = []

for col, label in outcomes:
    d = subject_df[["subject", "PSE_Group", col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    low = d[d["PSE_Group"] == "Low PSE"][col]
    high = d[d["PSE_Group"] == "High PSE"][col]

    low_mean, low_lo, low_hi, low_n = mean_ci(low)
    high_mean, high_lo, high_hi, high_n = mean_ci(high)

    if low_n >= 2 and high_n >= 2:
        t_res = ttest_ind(high, low, equal_var=False, nan_policy="omit")
        try:
            mw_res = mannwhitneyu(high, low, alternative="two-sided")
            mw_p = mw_res.pvalue
        except Exception:
            mw_p = np.nan

        d_val = cohens_d_independent(high, low)
    else:
        t_res = None
        mw_p = np.nan
        d_val = np.nan

    summary_rows.append({
        "outcome": col,
        "label": label,
        "n_low": low_n,
        "mean_low": low_mean,
        "ci_low_low": low_lo,
        "ci_high_low": low_hi,
        "n_high": high_n,
        "mean_high": high_mean,
        "ci_low_high": high_lo,
        "ci_high_high": high_hi,
        "high_minus_low": high_mean - low_mean if np.isfinite(high_mean) and np.isfinite(low_mean) else np.nan,
        "welch_t": t_res.statistic if t_res is not None else np.nan,
        "welch_p": t_res.pvalue if t_res is not None else np.nan,
        "mannwhitney_p": mw_p,
        "cohens_d_high_minus_low": d_val,
        "pse_median": pse_median,
    })

summary_df = pd.DataFrame(summary_rows)

subject_out_csv = os.path.join(CSV_DIR, "subject_level_high_low_pse_integrator_table.csv")
summary_csv = os.path.join(CSV_DIR, "high_low_pse_group_comparisons.csv")

subject_df.to_csv(subject_out_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print("\n============================================================")
print("HIGH-PSE VS LOW-PSE EXPLORATORY GROUP COMPARISONS")
print("============================================================")
print(f"PSE median split value: {pse_median:.4f} deg")
print(f"Low PSE n = {(subject_df['PSE_Group'] == 'Low PSE').sum()}")
print(f"High PSE n = {(subject_df['PSE_Group'] == 'High PSE').sum()}")

display_cols = [
    "label",
    "n_low",
    "mean_low",
    "n_high",
    "mean_high",
    "high_minus_low",
    "welch_t",
    "welch_p",
    "cohens_d_high_minus_low",
]

print(summary_df[display_cols].to_string(index=False))

# =========================
# PLOTS
# =========================
def plot_group_bar(subject_df, col, label, outpath):
    d = subject_df[["subject", "PSE_Group", col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if d["PSE_Group"].nunique() < 2:
        print(f"Skipping plot for {col}: missing one group.")
        return

    summary = (
        d
        .groupby("PSE_Group", observed=True)
        .agg(
            mean=(col, "mean"),
            sem=(col, sem),
            n=(col, "count")
        )
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(5.8, 5.2))

    x = np.arange(len(summary))

    ax.bar(
        x,
        summary["mean"],
        yerr=summary["sem"],
        capsize=5,
        alpha=0.82
    )

    for i, group in enumerate(summary["PSE_Group"]):
        vals = d[d["PSE_Group"] == group][col].values
        jitter = np.random.normal(loc=0, scale=0.04, size=len(vals))
        ax.scatter(
            np.full(len(vals), i) + jitter,
            vals,
            alpha=0.65,
            s=28
        )

    ax.set_xticks(x)
    ax.set_xticklabels(summary["PSE_Group"].astype(str))
    ax.set_ylabel(label)
    ax.set_title(f"{label}\nHigh vs Low PSE")

    fig.tight_layout()
    fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# Individual plots
for col, label in outcomes:
    outpath = os.path.join(FIG_DIR, f"high_low_pse_{col}.png")
    plot_group_bar(subject_df, col, label, outpath)

# Compact multi-panel plot for key outcomes
key_plot_outcomes = [
    ("mean_capture_overall", "Capture"),
    ("mean_CaptureHysteresis", "Hysteresis"),
    ("mean_residual_bias_30_40deg", "Residual bias"),
    ("mean_best_lag_visual_sec", "Lag"),
    ("mean_lag_corrected_RMSE_visual", "Lag-corrected RMSE"),
]

key_plot_outcomes = [(c, lab) for c, lab in key_plot_outcomes if c in subject_df.columns]

if len(key_plot_outcomes) > 0:
    ncols = 3
    nrows = int(np.ceil(len(key_plot_outcomes) / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.2 * ncols, 4.6 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, (col, label) in zip(axes, key_plot_outcomes):
        d = subject_df[["subject", "PSE_Group", col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

        summary = (
            d
            .groupby("PSE_Group", observed=True)
            .agg(
                mean=(col, "mean"),
                sem=(col, sem),
                n=(col, "count")
            )
            .reset_index()
        )

        x = np.arange(len(summary))

        ax.bar(
            x,
            summary["mean"],
            yerr=summary["sem"],
            capsize=5,
            alpha=0.82
        )

        for i, group in enumerate(summary["PSE_Group"]):
            vals = d[d["PSE_Group"] == group][col].values
            jitter = np.random.normal(loc=0, scale=0.04, size=len(vals))
            ax.scatter(
                np.full(len(vals), i) + jitter,
                vals,
                alpha=0.60,
                s=22
            )

        ax.set_xticks(x)
        ax.set_xticklabels(summary["PSE_Group"].astype(str), rotation=0)
        ax.set_title(label)
        ax.set_ylabel(label)

    for ax in axes[len(key_plot_outcomes):]:
        ax.axis("off")

    fig.suptitle("Exploratory strong vs weak integrators by PSE", fontsize=15, y=1.02)

    multi_path = os.path.join(FIG_DIR, "high_low_pse_key_outcomes_multipanel.png")
    fig.tight_layout()
    fig.savefig(multi_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

print("\n============================================================")
print("STRONG VS WEAK INTEGRATOR ANALYSIS COMPLETE")
print("============================================================")
print(f"Saved subject table:   {subject_out_csv}")
print(f"Saved group summary:   {summary_csv}")
print(f"Figures folder:        {FIG_DIR}")

print("\nInterpretation guide:")
print("- High PSE = broader causal binding window / stronger integrator group.")
print("- This median split is exploratory and descriptive.")
print("- The continuous PSE regressions should remain your primary inferential analyses.")
print("- high_minus_low > 0 means the High-PSE group showed larger values on that outcome.")



In [44]:
# %% AIM 1 — RELIABILITY OF THE INDIVIDUAL-DIFFERENCE PHENOTYPE
# Run after the corresponding SL notebook analyses have generated their CSV outputs.
# Outputs are written to:
#   C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\
#   Individual_differences_analyses\Aim_1_reliability

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------------
# PATHS / SETTINGS
# ---------------------------------------------------------------------
ROOT = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1"
)

SL_DATA = ROOT / "individual_data_SL"
SPATIAL = ROOT / "spatial_analyses"

OUT = (
    ROOT
    / "Individual_differences_analyses"
    / "Aim_1_reliability"
)

CSV_OUT = OUT / "csv_outputs"
FIG_OUT = OUT / "figures"

CSV_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

RNG_SEED = 20261010
N_BOOT = 10000

FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]


def require_file(
    path,
    label,
):
    path = Path(
        path
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nMissing {label}:\n"
            f"{path}\n\n"
            "Run the corresponding section of "
            "Continuous_SLv3_rebuilt first."
        )

    return path


FIXED_CAPTURE_CSV = require_file(
    SL_DATA
    / "spatial_pse_capture_hysteresis_recalibration_predictions"
    / "csv_outputs"
    / "fixed_offset_trial_level_capture_psame_bias.csv",
    "Section 12 fixed-offset trial table",
)

STABLE_BIAS_CSV = require_file(
    SPATIAL
    / "05_residual_visual_bias"
    / "trial_level_bias_psame_model_data.csv",
    "Section 10 stable-window residual-bias model data",
)

VALID_LAG_CSV = require_file(
    SPATIAL
    / "02_same_different_tracking_lag"
    / "csv_outputs"
    / "03_VALID_trial_lag_estimates_after_exact_collaborator_QC.csv",
    "valid collaborator-style auditory-lag table",
)

HYST_WINDOW_CSV = require_file(
    SPATIAL
    / "07_spatial_hysteresis"
    / "spatial_hysteresis_window_level_bias_by_disparity.csv",
    "Section 14 spatial-hysteresis window table",
)

BREAKPOINT_CSV = require_file(
    SPATIAL
    / "06_stabilization_breakpoint"
    / "csv_outputs"
    / "anchor_trial_level_breakpoint_DYNAMIC_metrics.csv",
    "Section 11 trial-level breakpoint table",
)


# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def clean_subject(
    s,
):
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )


def assign_odd_even(
    df,
    strata,
):
    """
    Deterministic odd/even split.

    Splitting is performed separately within participant × stratum so that
    offset/direction composition is balanced as closely as possible.
    """

    d = df.copy()

    d[
        "subject"
    ] = clean_subject(
        d[
            "subject"
        ]
    )

    d[
        "trialIndex"
    ] = pd.to_numeric(
        d[
            "trialIndex"
        ],
        errors="coerce",
    )

    d = (
        d.sort_values(
            [
                "subject",
            ]
            +
            strata
            +
            [
                "trialIndex",
            ]
        )
        .copy()
    )

    d[
        "_within_stratum_order"
    ] = (
        d.groupby(
            [
                "subject",
            ]
            +
            strata
        )
        .cumcount()
    )

    d[
        "half"
    ] = np.where(
        d[
            "_within_stratum_order"
        ]
        % 2
        == 0,
        "odd",
        "even",
    )

    return d


def monotonic_nonincreasing(
    y,
):

    y = np.asarray(
        y,
        dtype=float,
    ).copy()

    for i in range(
        1,
        len(
            y
        ),
    ):

        if (
            np.isfinite(
                y[
                    i
                ]
            )
            and
            np.isfinite(
                y[
                    i - 1
                ]
            )
        ):

            y[
                i
            ] = min(
                y[
                    i
                ],
                y[
                    i - 1
                ],
            )

    return y


def bounded_empirical_pse(
    offsets,
    same_vals,
    smoothing=0.5,
):
    """
    Matches Section 12's PRIMARY PSE logic:

      1. Empirical P(same) at each fixed offset.
      2. Jeffreys-like 0.5 smoothing.
      3. Enforce monotonic non-increasing P(same).
      4. Interpolate P(same)=.50.
      5. Bound the estimate to the tested 10–40° range.
    """

    x = (
        pd.DataFrame(
            {
                "offset":
                    offsets,

                "same":
                    same_vals,
            }
        )
        .dropna()
    )

    rows = []

    for off in FIXED_OFFSETS:

        g = x[
            x[
                "offset"
            ]
            ==
            off
        ]

        n = len(
            g
        )

        if n == 0:

            rows.append(
                (
                    off,
                    np.nan,
                )
            )

            continue

        n_same = float(
            g[
                "same"
            ].sum()
        )

        ps = (
            n_same
            +
            smoothing
        ) / (
            n
            +
            2
            *
            smoothing
        )

        rows.append(
            (
                off,
                ps,
            )
        )

    tab = (
        pd.DataFrame(
            rows,
            columns=[
                "offset",
                "psame",
            ],
        )
        .dropna()
    )

    if tab.empty:

        return np.nan

    xx = tab[
        "offset"
    ].to_numpy(
        float
    )

    yy = monotonic_nonincreasing(
        tab[
            "psame"
        ].to_numpy(
            float
        )
    )

    if np.all(
        yy >= .5
    ):

        return float(
            xx.max()
        )

    if np.all(
        yy < .5
    ):

        return float(
            xx.min()
        )

    for j in range(
        len(
            xx
        )
        -
        1
    ):

        if (
            yy[
                j
            ]
            >= .5
            and
            yy[
                j + 1
            ]
            < .5
        ):

            if np.isclose(
                yy[
                    j
                ],
                yy[
                    j + 1
                ],
            ):

                return float(
                    xx[
                        j
                    ]
                )

            pse = (
                xx[
                    j
                ]
                +
                (
                    .5
                    -
                    yy[
                        j
                    ]
                )
                *
                (
                    xx[
                        j + 1
                    ]
                    -
                    xx[
                        j
                    ]
                )
                /
                (
                    yy[
                        j + 1
                    ]
                    -
                    yy[
                        j
                    ]
                )
            )

            return float(
                np.clip(
                    pse,
                    min(
                        FIXED_OFFSETS
                    ),
                    max(
                        FIXED_OFFSETS
                    ),
                )
            )

    return np.nan


def logistic_pse(
    offsets,
    same_vals,
):
    """
    Logistic PSE sensitivity estimate.

    This is NOT the primary reliability measure; the bounded empirical PSE
    above matches the Section 12 primary PSE.
    """

    d = (
        pd.DataFrame(
            {
                "offset":
                    offsets,

                "same":
                    same_vals,
            }
        )
        .dropna()
    )

    if (
        len(
            d
        )
        < 4
        or
        d[
            "same"
        ].nunique()
        < 2
    ):

        return np.nan

    try:

        X = sm.add_constant(
            d[
                "offset"
            ].astype(
                float
            )
        )

        fit = sm.GLM(
            d[
                "same"
            ].astype(
                float
            ),
            X,
            family=sm.families.Binomial(),
        ).fit()

        b0 = float(
            fit.params.iloc[
                0
            ]
        )

        b1 = float(
            fit.params.iloc[
                1
            ]
        )

        if (
            not np.isfinite(
                b1
            )
            or
            b1 >= 0
            or
            abs(
                b1
            )
            < 1e-12
        ):

            return np.nan

        return float(
            np.clip(
                -b0 / b1,
                min(
                    FIXED_OFFSETS
                ),
                max(
                    FIXED_OFFSETS
                ),
            )
        )

    except Exception:

        return np.nan


def sb_correct(
    r,
):

    if (
        not np.isfinite(
            r
        )
        or
        np.isclose(
            1 + r,
            0,
        )
    ):

        return np.nan

    return (
        2
        *
        r
        /
        (
            1
            +
            r
        )
    )


def paired_reliability(
    df,
    x="odd",
    y="even",
    n_boot=N_BOOT,
    seed=RNG_SEED,
):

    d = (
        df[
            [
                x,
                y,
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )

    if (
        len(
            d
        )
        < 4
        or
        d[
            x
        ].std(
            ddof=1
        )
        == 0
        or
        d[
            y
        ].std(
            ddof=1
        )
        == 0
    ):

        return {
            "n":
                len(
                    d
                ),

            "pearson_r":
                np.nan,

            "spearman_rho":
                np.nan,

            "spearman_brown":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,
        }

    r = pearsonr(
        d[
            x
        ],
        d[
            y
        ],
    )[0]

    rho = spearmanr(
        d[
            x
        ],
        d[
            y
        ],
    )[0]

    sb = sb_correct(
        r
    )

    rng = np.random.default_rng(
        seed
    )

    boot = []

    arr = d[
        [
            x,
            y,
        ]
    ].to_numpy(
        float
    )

    for _ in range(
        n_boot
    ):

        s = arr[
            rng.integers(
                0,
                len(
                    arr
                ),
                len(
                    arr
                ),
            )
        ]

        if (
            np.std(
                s[
                    :,
                    0
                ],
                ddof=1,
            )
            == 0
            or
            np.std(
                s[
                    :,
                    1
                ],
                ddof=1,
            )
            == 0
        ):

            continue

        rb = np.corrcoef(
            s[
                :,
                0
            ],
            s[
                :,
                1
            ],
        )[
            0,
            1
        ]

        boot.append(
            sb_correct(
                rb
            )
        )

    if len(
        boot
    ) < 100:

        lo = np.nan
        hi = np.nan

    else:

        lo, hi = np.nanpercentile(
            boot,
            [
                2.5,
                97.5,
            ],
        )

    return {
        "n":
            len(
                d
            ),

        "pearson_r":
            r,

        "spearman_rho":
            rho,

        "spearman_brown":
            sb,

        "ci_low":
            lo,

        "ci_high":
            hi,
    }


def half_metric_table(
    df,
    value_col,
    strata,
):

    d = assign_odd_even(
        df,
        strata,
    )

    out = (
        d.groupby(
            [
                "subject",
                "half",
            ],
            as_index=False,
        )[
            value_col
        ]
        .mean()
        .pivot(
            index="subject",
            columns="half",
            values=value_col,
        )
        .reset_index()
    )

    return out


def exact_half_hysteresis(
    window_df,
):
    """
    Reproduces the manuscript SL hysteresis metric independently in each half:

      - split WHOLE trials within participant × sweep direction
      - use 5–35° disparity range
      - use 2° disparity bins
      - window-weighted means
      - Converge minus Diverge
      - average matched-bin differences within participant
      - require >=2 matched bins
    """

    w = window_df.copy()

    w[
        "subject"
    ] = clean_subject(
        w[
            "subject"
        ]
    )

    w[
        "trialIndex"
    ] = pd.to_numeric(
        w[
            "trialIndex"
        ],
        errors="coerce",
    )

    w[
        "SweepDirection"
    ] = (
        w[
            "SweepDirection"
        ]
        .astype(str)
        .str.title()
    )

    trials = (
        w[
            [
                "subject",
                "trialIndex",
                "SweepDirection",
            ]
        ]
        .drop_duplicates()
    )

    trials = assign_odd_even(
        trials,
        [
            "SweepDirection",
        ],
    )

    w = w.merge(
        trials[
            [
                "subject",
                "trialIndex",
                "SweepDirection",
                "half",
            ]
        ],
        on=[
            "subject",
            "trialIndex",
            "SweepDirection",
        ],
        how="inner",
    )

    w[
        "mean_disparity_window"
    ] = pd.to_numeric(
        w[
            "mean_disparity_window"
        ],
        errors="coerce",
    )

    w[
        "NormalizedVisualBias_window"
    ] = pd.to_numeric(
        w[
            "NormalizedVisualBias_window"
        ],
        errors="coerce",
    )

    w = w[
        w[
            "mean_disparity_window"
        ].between(
            5.0,
            35.0,
        )
        &
        w[
            "NormalizedVisualBias_window"
        ].notna()
    ].copy()

    bins = np.arange(
        5.0,
        35.0
        +
        2.0,
        2.0,
    )

    w[
        "disparity_bin"
    ] = pd.cut(
        w[
            "mean_disparity_window"
        ],
        bins=bins,
        include_lowest=True,
    )

    w[
        "bin_center"
    ] = (
        w[
            "disparity_bin"
        ]
        .apply(
            lambda z:
                z.mid
                if pd.notna(
                    z
                )
                else np.nan
        )
        .astype(
            float
        )
    )

    s = (
        w.groupby(
            [
                "subject",
                "half",
                "SweepDirection",
                "bin_center",
            ],
            observed=True,
        )
        .agg(
            capture=(
                "NormalizedVisualBias_window",
                "mean",
            )
        )
        .reset_index()
    )

    p = (
        s.pivot_table(
            index=[
                "subject",
                "half",
                "bin_center",
            ],
            columns="SweepDirection",
            values="capture",
        )
        .reset_index()
    )

    if "Converge" not in p.columns:

        p[
            "Converge"
        ] = np.nan

    if "Diverge" not in p.columns:

        p[
            "Diverge"
        ] = np.nan

    p[
        "hyst"
    ] = (
        p[
            "Converge"
        ]
        -
        p[
            "Diverge"
        ]
    )

    subj = (
        p.groupby(
            [
                "subject",
                "half",
            ],
            as_index=False,
        )
        .agg(
            n_bins=(
                "hyst",
                lambda x:
                    x.notna().sum(),
            ),

            value=(
                "hyst",
                "mean",
            ),
        )
    )

    subj.loc[
        subj[
            "n_bins"
        ]
        < 2,
        "value",
    ] = np.nan

    return (
        subj.pivot(
            index="subject",
            columns="half",
            values="value",
        )
        .reset_index()
    )


def make_residual_bias_halves(
    model_df,
):
    """
    Reliability of participant residual visual bias.

    The nuisance model is re-fit INDEPENDENTLY in each half:

        stable lag-corrected bias ~ offset + explicit Same/Different judgment

    Residuals are then averaged within participant.
    """

    d = model_df.copy()

    d[
        "subject"
    ] = clean_subject(
        d[
            "subject"
        ]
    )

    d[
        "offset_num"
    ] = pd.to_numeric(
        d[
            "offset_num"
        ],
        errors="coerce",
    )

    d[
        "Psame"
    ] = pd.to_numeric(
        d[
            "Psame"
        ],
        errors="coerce",
    )

    d[
        "Bias_lagcorr_AminusV_stable"
    ] = pd.to_numeric(
        d[
            "Bias_lagcorr_AminusV_stable"
        ],
        errors="coerce",
    )

    d = d[
        d[
            "offset_num"
        ].isin(
            FIXED_OFFSETS
        )
    ].dropna(
        subset=[
            "subject",
            "trialIndex",
            "offset_num",
            "Psame",
            "Bias_lagcorr_AminusV_stable",
        ]
    )

    d = assign_odd_even(
        d,
        [
            "offset_num",
        ],
    )

    rows = []

    for half, h in d.groupby(
        "half"
    ):

        try:

            fit = smf.ols(
                "Bias_lagcorr_AminusV_stable ~ "
                "C(offset_num) + Psame",
                data=h,
            ).fit()

            hh = h.copy()

            hh[
                "resid"
            ] = fit.resid

            z = (
                hh.groupby(
                    "subject",
                    as_index=False,
                )[
                    "resid"
                ]
                .mean()
            )

            z[
                "half"
            ] = half

            rows.append(
                z
            )

        except Exception:

            pass

    if not rows:

        return pd.DataFrame(
            columns=[
                "subject",
                "odd",
                "even",
            ]
        )

    long = pd.concat(
        rows,
        ignore_index=True,
    )

    return (
        long.pivot(
            index="subject",
            columns="half",
            values="resid",
        )
        .reset_index()
    )


# ---------------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------------
fixed = pd.read_csv(
    FIXED_CAPTURE_CSV
)

fixed[
    "subject"
] = clean_subject(
    fixed[
        "subject"
    ]
)

fixed[
    "offset_deg"
] = pd.to_numeric(
    fixed[
        "offset_deg"
    ],
    errors="coerce",
)

fixed[
    "same_response"
] = pd.to_numeric(
    fixed[
        "same_response"
    ],
    errors="coerce",
)

fixed[
    "NormalizedVisualBias"
] = pd.to_numeric(
    fixed[
        "NormalizedVisualBias"
    ],
    errors="coerce",
)

fixed = fixed[
    fixed[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


stable_bias = pd.read_csv(
    STABLE_BIAS_CSV
)

stable_bias[
    "subject"
] = clean_subject(
    stable_bias[
        "subject"
    ]
)


lag = pd.read_csv(
    VALID_LAG_CSV
)

lag[
    "subject"
] = clean_subject(
    lag[
        "subject"
    ]
)

lag[
    "offset_deg"
] = pd.to_numeric(
    lag[
        "offset_deg"
    ],
    errors="coerce",
)

lag[
    "lag_audio_sec"
] = pd.to_numeric(
    lag[
        "lag_audio_sec"
    ],
    errors="coerce",
)

lag = lag[
    lag[
        "offset_deg"
    ].abs().isin(
        FIXED_OFFSETS
    )
].copy()

lag[
    "offset_deg"
] = lag[
    "offset_deg"
].abs()


hwin = pd.read_csv(
    HYST_WINDOW_CSV
)


bp = pd.read_csv(
    BREAKPOINT_CSV
)

bp[
    "subject"
] = clean_subject(
    bp[
        "subject"
    ]
)

bp[
    "offset_deg"
] = pd.to_numeric(
    bp[
        "offset_deg"
    ],
    errors="coerce",
)

if "metric" in bp.columns:

    bp = bp[
        bp[
            "metric"
        ].astype(str)
        .eq(
            "LagCorrectedVisualCapture"
        )
    ].copy()

bp = bp[
    bp[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()

bp[
    "dynamically_updating"
] = pd.to_numeric(
    bp[
        "dynamically_updating"
    ],
    errors="coerce",
)


# ---------------------------------------------------------------------
# 1A. PSE ODD/EVEN RELIABILITY
# ---------------------------------------------------------------------
fhalf = assign_odd_even(
    fixed.dropna(
        subset=[
            "same_response",
        ]
    ),
    [
        "offset_deg",
    ],
)

pse_rows = []

for (
    subject,
    half,
), g in fhalf.groupby(
    [
        "subject",
        "half",
    ]
):

    pse_rows.append(
        {
            "subject":
                subject,

            "half":
                half,

            "PSE_empirical":
                bounded_empirical_pse(
                    g[
                        "offset_deg"
                    ],
                    g[
                        "same_response"
                    ],
                ),

            "PSE_logistic":
                logistic_pse(
                    g[
                        "offset_deg"
                    ],
                    g[
                        "same_response"
                    ],
                ),
        }
    )

pse_long = pd.DataFrame(
    pse_rows
)

pse_emp = (
    pse_long.pivot(
        index="subject",
        columns="half",
        values="PSE_empirical",
    )
    .reset_index()
)

pse_log = (
    pse_long.pivot(
        index="subject",
        columns="half",
        values="PSE_logistic",
    )
    .reset_index()
)


# ---------------------------------------------------------------------
# 1B. SPATIAL-WEIGHTING RELIABILITY
# ---------------------------------------------------------------------
capture_half = half_metric_table(
    fixed.dropna(
        subset=[
            "NormalizedVisualBias",
        ]
    ),
    "NormalizedVisualBias",
    strata=[
        "offset_deg",
    ],
)


stable_bias[
    "offset_num"
] = pd.to_numeric(
    stable_bias[
        "offset_num"
    ],
    errors="coerce",
)

stable_bias[
    "Bias_lagcorr_AminusV_stable"
] = pd.to_numeric(
    stable_bias[
        "Bias_lagcorr_AminusV_stable"
    ],
    errors="coerce",
)

rmse_half = half_metric_table(
    stable_bias[
        stable_bias[
            "offset_num"
        ].isin(
            FIXED_OFFSETS
        )
    ].dropna(
        subset=[
            "Bias_lagcorr_AminusV_stable",
        ]
    ),
    "Bias_lagcorr_AminusV_stable",
    strata=[
        "offset_num",
    ],
)


resid_half = make_residual_bias_halves(
    stable_bias
)


# ---------------------------------------------------------------------
# 1C. TEMPORAL RELIABILITY
# ---------------------------------------------------------------------
lag_half = half_metric_table(
    lag.dropna(
        subset=[
            "lag_audio_sec",
        ]
    ),
    "lag_audio_sec",
    strata=[
        "offset_deg",
    ],
)


hyst_half = exact_half_hysteresis(
    hwin
)


bp_half = half_metric_table(
    bp.dropna(
        subset=[
            "dynamically_updating",
        ]
    ),
    "dynamically_updating",
    strata=[
        "offset_deg",
    ],
)


# ---------------------------------------------------------------------
# COMBINE AND ESTIMATE RELIABILITY
# ---------------------------------------------------------------------
metric_tables = {
    "PSE_empirical":
        pse_emp,

    "PSE_logistic_sensitivity":
        pse_log,

    "Normalized_visual_capture":
        capture_half,

    "Lagcorrected_RMSE_bias":
        rmse_half,

    "Residual_visual_bias":
        resid_half,

    "Auditory_tracking_lag":
        lag_half,

    "SL_hysteresis":
        hyst_half,

    "Breakpoint_dynamicity":
        bp_half,
}


rel_rows = []

all_halves = []

for k, tab in metric_tables.items():

    if not {
        "odd",
        "even",
    }.issubset(
        tab.columns
    ):

        print(
            f"Skipping {k}: "
            "both halves were not available."
        )

        continue

    tmp = tab[
        [
            "subject",
            "odd",
            "even",
        ]
    ].copy()

    tmp[
        "metric"
    ] = k

    all_halves.append(
        tmp
    )

    r = paired_reliability(
        tmp,
        "odd",
        "even",
    )

    rel_rows.append(
        {
            "metric":
                k,

            **r,
        }
    )


reliability = (
    pd.DataFrame(
        rel_rows
    )
    .sort_values(
        "spearman_brown",
        ascending=False,
    )
)

halves_long = pd.concat(
    all_halves,
    ignore_index=True,
)


reliability.to_csv(
    CSV_OUT
    /
    "aim1_split_half_reliability_summary.csv",
    index=False,
)

halves_long.to_csv(
    CSV_OUT
    /
    "aim1_participant_odd_even_metrics.csv",
    index=False,
)

pse_long.to_csv(
    CSV_OUT
    /
    "aim1_pse_half_estimates.csv",
    index=False,
)


# ---------------------------------------------------------------------
# 1D. RELIABILITY CEILINGS
#
# r_max ≈ sqrt(r_xx * r_yy)
# ---------------------------------------------------------------------
primary = reliability[
    reliability[
        "metric"
    ]
    !=
    "PSE_logistic_sensitivity"
].copy()


names = primary[
    "metric"
].tolist()


rmap = (
    primary
    .set_index(
        "metric"
    )[
        "spearman_brown"
    ]
    .to_dict()
)


ceiling = pd.DataFrame(
    index=names,
    columns=names,
    dtype=float,
)


for a in names:

    for b in names:

        ra = rmap.get(
            a,
            np.nan,
        )

        rb = rmap.get(
            b,
            np.nan,
        )

        if (
            np.isfinite(
                ra
            )
            and
            np.isfinite(
                rb
            )
            and
            ra >= 0
            and
            rb >= 0
        ):

            ceiling.loc[
                a,
                b,
            ] = np.sqrt(
                ra
                *
                rb
            )

        else:

            ceiling.loc[
                a,
                b,
            ] = np.nan


ceiling.to_csv(
    CSV_OUT
    /
    "aim1_reliability_ceiling_matrix.csv"
)


# ---------------------------------------------------------------------
# FIGURE 1 — ODD/EVEN SCATTER PANELS
# ---------------------------------------------------------------------
plot_metrics = [
    m
    for m in names
    if m in metric_tables
]

ncols = 3

nrows = int(
    np.ceil(
        len(
            plot_metrics
        )
        /
        ncols
    )
)


fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(
        15,
        4.5
        *
        nrows,
    ),
)

axes = np.atleast_1d(
    axes
).ravel()


for ax, metric in zip(
    axes,
    plot_metrics,
):

    d = (
        metric_tables[
            metric
        ][
            [
                "odd",
                "even",
            ]
        ]
        .dropna()
    )

    ax.scatter(
        d[
            "odd"
        ],
        d[
            "even"
        ],
        alpha=.72,
    )

    if (
        len(
            d
        )
        >= 4
        and
        d[
            "odd"
        ].std()
        > 0
    ):

        b = np.polyfit(
            d[
                "odd"
            ],
            d[
                "even"
            ],
            1,
        )

        xx = np.linspace(
            d[
                "odd"
            ].min(),
            d[
                "odd"
            ].max(),
            100,
        )

        ax.plot(
            xx,
            b[
                0
            ]
            *
            xx
            +
            b[
                1
            ],
            linewidth=2,
        )

    row = reliability[
        reliability[
            "metric"
        ]
        ==
        metric
    ]

    if len(
        row
    ):

        rr = row.iloc[
            0
        ]

        ax.set_title(
            f"{metric}\n"
            f"SB r={rr['spearman_brown']:.2f} "
            f"[{rr['ci_low']:.2f}, "
            f"{rr['ci_high']:.2f}]"
        )

    else:

        ax.set_title(
            metric
        )

    ax.set_xlabel(
        "Odd-half estimate"
    )

    ax.set_ylabel(
        "Even-half estimate"
    )


for ax in axes[
    len(
        plot_metrics
    ):
]:

    ax.axis(
        "off"
    )


fig.suptitle(
    "Odd–even split-half reliability "
    "of individual-difference measures",
    fontsize=16,
    y=1.01,
)

fig.tight_layout()

fig.savefig(
    FIG_OUT
    /
    "aim1_odd_even_reliability_scatter_panels.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()


# ---------------------------------------------------------------------
# FIGURE 2 — RANKED RELIABILITY
# ---------------------------------------------------------------------
p = (
    primary
    .sort_values(
        "spearman_brown"
    )
    .copy()
)


fig, ax = plt.subplots(
    figsize=(
        9,
        6,
    )
)


y = np.arange(
    len(
        p
    )
)


ax.errorbar(
    p[
        "spearman_brown"
    ],
    y,
    xerr=np.vstack(
        [
            p[
                "spearman_brown"
            ]
            -
            p[
                "ci_low"
            ],

            p[
                "ci_high"
            ]
            -
            p[
                "spearman_brown"
            ],
        ]
    ),
    fmt="o",
    capsize=4,
)


ax.axvline(
    0,
    linestyle="--",
    linewidth=1,
)


ax.set_yticks(
    y
)

ax.set_yticklabels(
    p[
        "metric"
    ]
)

ax.set_xlabel(
    "Spearman–Brown corrected "
    "split-half reliability"
)

ax.set_title(
    "Reliability of candidate "
    "individual-difference phenotypes"
)


fig.tight_layout()

fig.savefig(
    FIG_OUT
    /
    "aim1_ranked_reliability.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()


# ---------------------------------------------------------------------
# FIGURE 3 — RELIABILITY-IMPOSED CORRELATION CEILINGS
# ---------------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(
        10,
        8,
    )
)


mat = ceiling.to_numpy(
    float
)


im = ax.imshow(
    mat,
    vmin=0,
    vmax=1,
    aspect="auto",
)


ax.set_xticks(
    np.arange(
        len(
            names
        )
    )
)

ax.set_yticks(
    np.arange(
        len(
            names
        )
    )
)

ax.set_xticklabels(
    names,
    rotation=45,
    ha="right",
)

ax.set_yticklabels(
    names
)


for i in range(
    len(
        names
    )
):

    for j in range(
        len(
            names
        )
    ):

        if np.isfinite(
            mat[
                i,
                j
            ]
        ):

            ax.text(
                j,
                i,
                f"{mat[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
            )


fig.colorbar(
    im,
    ax=ax,
    label=(
        "Maximum expected "
        "observed |r|"
    ),
)


ax.set_title(
    "Reliability ceiling for "
    "cross-measure individual-difference correlations"
)


fig.tight_layout()

fig.savefig(
    FIG_OUT
    /
    "aim1_reliability_ceiling_heatmap.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()


print(
    "\nAIM 1 COMPLETE"
)

print(
    reliability.to_string(
        index=False
    )
)

print(
    f"\nSaved outputs to:\n{OUT}"
)


AIM 1 COMPLETE
                   metric  n  pearson_r  spearman_rho  spearman_brown   ci_low  ci_high
   Lagcorrected_RMSE_bias 50   0.948205      0.931717        0.973414 0.953829 0.985762
     Residual_visual_bias 50   0.939866      0.925954        0.969001 0.947979 0.982306
 PSE_logistic_sensitivity 49   0.840049      0.831359        0.913073 0.851427 0.953037
            PSE_empirical 50   0.799064      0.810253        0.888311 0.797242 0.945607
Normalized_visual_capture 50   0.754470      0.777959        0.860054 0.746351 0.938877
    Auditory_tracking_lag 50   0.745046      0.739160        0.853898 0.738573 0.927035
            SL_hysteresis 49   0.615005      0.685204        0.761614 0.574794 0.889198
    Breakpoint_dynamicity 50   0.474947      0.398916        0.644019 0.375321 0.791554

Saved outputs to:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_1_reliability


In [46]:
# %% AIM 1 — PUBLICATION/PRESENTATION CORRELATION FIGURES WITH 95% CI
#
# Paste this cell AFTER the full Aim 1 analysis cell has run.
#
# Uses objects already created by Aim 1:
#   metric_tables
#   reliability
#   FIG_OUT
#
# Produces:
#   - one high-resolution figure per reliability metric
#   - one combined panel
#
# Style:
#   blue points + OLS line
#   orange 95% CI
#   n, Pearson r/p, regression beta/p
#   Spearman–Brown reliability + bootstrap CI

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import pearsonr

# ---------------------------------------------------------------------
# OUTPUT DIRECTORY
# ---------------------------------------------------------------------
CORR_FIG_OUT = (
    Path(FIG_OUT)
    / "correlation_figures_95CI"
)

CORR_FIG_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------------------
# STYLE
# ---------------------------------------------------------------------
POINT_COLOR = "#1f77b4"
LINE_COLOR = "#1f77b4"
CI_COLOR = "#ff7f0e"

POINT_ALPHA = 0.72
CI_ALPHA = 0.22


def format_p(p):
    if not np.isfinite(p):
        return "NA"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"


def corr_panel(
    ax,
    df,
    x,
    y,
    xlabel,
    ylabel,
    title,
    extra_text=None,
):
    """
    Participant-level correlation plot.

    Displays:
      n
      Pearson r / p
      HC3 OLS beta / p
      OLS mean regression line
      95% CI of the mean regression line
    """

    d = (
        df[
            [x, y]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .copy()
    )

    if len(d) < 4:
        ax.text(
            .5,
            .5,
            "Insufficient data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title(title)
        return None

    # --------------------------------------------------------------
    # Pearson correlation
    # --------------------------------------------------------------
    r_obj = pearsonr(
        d[x],
        d[y],
    )

    r = float(
        r_obj.statistic
    )

    p_r = float(
        r_obj.pvalue
    )

    # --------------------------------------------------------------
    # HC3 robust OLS
    # --------------------------------------------------------------
    X = sm.add_constant(
        d[[x]],
        has_constant="add",
    )

    fit = sm.OLS(
        d[y],
        X,
    ).fit(
        cov_type="HC3"
    )

    beta = float(
        fit.params[x]
    )

    p_beta = float(
        fit.pvalues[x]
    )

    # --------------------------------------------------------------
    # Prediction grid + 95% mean CI
    # --------------------------------------------------------------
    x_grid = np.linspace(
        d[x].min(),
        d[x].max(),
        250,
    )

    X_grid = sm.add_constant(
        pd.DataFrame(
            {
                x: x_grid,
            }
        ),
        has_constant="add",
    )

    pred = (
        fit.get_prediction(
            X_grid
        )
        .summary_frame(
            alpha=.05
        )
    )

    # --------------------------------------------------------------
    # Plot
    # --------------------------------------------------------------
    ax.scatter(
        d[x],
        d[y],
        s=58,
        alpha=POINT_ALPHA,
        color=POINT_COLOR,
        edgecolor=POINT_COLOR,
    )

    ax.fill_between(
        x_grid,
        pred["mean_ci_lower"].to_numpy(),
        pred["mean_ci_upper"].to_numpy(),
        color=CI_COLOR,
        alpha=CI_ALPHA,
        label="95% CI",
        linewidth=0,
    )

    ax.plot(
        x_grid,
        pred["mean"].to_numpy(),
        color=LINE_COLOR,
        linewidth=2.7,
        label="OLS fit",
    )

    ax.set_xlabel(
        xlabel,
        fontsize=12,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=12,
    )

    ax.set_title(
        title,
        fontsize=14,
        pad=10,
    )

    # --------------------------------------------------------------
    # Stats box
    # --------------------------------------------------------------
    stats_text = (
        f"n = {len(d)}\n"
        f"r = {r:.3f}, p {format_p(p_r)}\n"
        f"β = {beta:.4f}, p {format_p(p_beta)}"
    )

    if extra_text:
        stats_text += (
            "\n"
            + extra_text
        )

    ax.text(
        .04,
        .96,
        stats_text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="black",
            alpha=.88,
        ),
    )

    ax.legend(
        frameon=False,
        loc="upper right",
    )

    return fit


# ---------------------------------------------------------------------
# METRIC LABELS
# ---------------------------------------------------------------------
pretty_labels = {
    "PSE_empirical":
        "Empirical PSE (deg)",

    "PSE_logistic_sensitivity":
        "Logistic PSE (deg)",

    "Normalized_visual_capture":
        "Normalized visual capture",

    "Lagcorrected_RMSE_bias":
        "Lag-corrected visual bias",

    "Residual_visual_bias":
        "Residual visual bias",

    "Auditory_tracking_lag":
        "Auditory tracking lag (s)",

    "SL_hysteresis":
        "SL hysteresis",

    "Breakpoint_dynamicity":
        "Breakpoint dynamicity",
}


# ---------------------------------------------------------------------
# INDIVIDUAL FIGURES
# ---------------------------------------------------------------------
available_metrics = []

for metric, tab in metric_tables.items():

    if not {"odd", "even"}.issubset(
        tab.columns
    ):
        continue

    available_metrics.append(
        metric
    )

    rel_row = reliability[
        reliability["metric"]
        ==
        metric
    ]

    if len(rel_row):

        rr = rel_row.iloc[0]

        extra = (
            f"SB r = "
            f"{rr['spearman_brown']:.3f}\n"
            f"95% CI "
            f"[{rr['ci_low']:.3f}, "
            f"{rr['ci_high']:.3f}]"
        )

    else:

        extra = None

    label = pretty_labels.get(
        metric,
        metric,
    )

    fig, ax = plt.subplots(
        figsize=(7.2, 6.0)
    )

    corr_panel(
        ax=ax,
        df=tab,
        x="odd",
        y="even",
        xlabel=f"Odd-half {label}",
        ylabel=f"Even-half {label}",
        title=f"Split-half reliability: {label}",
        extra_text=extra,
    )

    fig.tight_layout()

    safe_name = (
        metric
        .replace(" ", "_")
        .replace("/", "_")
    )

    fig.savefig(
        CORR_FIG_OUT
        /
        f"aim1_{safe_name}_reliability_95CI.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# ---------------------------------------------------------------------
# COMBINED PANEL
# ---------------------------------------------------------------------
ncols = 3

nrows = int(
    np.ceil(
        len(available_metrics)
        /
        ncols
    )
)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(
        15,
        4.8 * nrows,
    ),
)

axes = np.atleast_1d(
    axes
).ravel()

for ax, metric in zip(
    axes,
    available_metrics,
):

    tab = metric_tables[
        metric
    ]

    rel_row = reliability[
        reliability["metric"]
        ==
        metric
    ]

    if len(rel_row):

        rr = rel_row.iloc[0]

        extra = (
            f"SB r = "
            f"{rr['spearman_brown']:.3f}\n"
            f"95% CI "
            f"[{rr['ci_low']:.3f}, "
            f"{rr['ci_high']:.3f}]"
        )

    else:

        extra = None

    label = pretty_labels.get(
        metric,
        metric,
    )

    corr_panel(
        ax=ax,
        df=tab,
        x="odd",
        y="even",
        xlabel="Odd-half estimate",
        ylabel="Even-half estimate",
        title=label,
        extra_text=extra,
    )


for ax in axes[
    len(available_metrics):
]:
    ax.axis("off")


fig.suptitle(
    "Reliability of candidate individual-difference phenotypes",
    fontsize=18,
    y=1.01,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim1_all_reliability_correlations_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


print(
    "\nUpdated Aim 1 correlation figures saved to:"
)

print(
    CORR_FIG_OUT
)


Updated Aim 1 correlation figures saved to:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_1_reliability\figures\correlation_figures_95CI


In [52]:
# %% AIM 2 — DOES CAUSAL TOLERANCE PREDICT PERCEPTUAL EXPRESSION?
# Revised repeated-measures analysis.
#
# IMPORTANT CHANGE:
# A MixedLM is no longer accepted simply because an optimizer returns.
#
# Each trial-level analysis now:
#   1. attempts an appropriate random-intercept + random-slope LME;
#   2. checks convergence;
#   3. checks that fixed-effect estimates, SEs, and p-values are finite;
#   4. checks whether the random-effects covariance is singular;
#   5. simplifies the random-effects structure if necessary;
#   6. if no defensible LME exists, uses Gaussian GEE with participant
#      clustering rather than reporting a singular mixed model;
#   7. uses cluster-robust OLS only as a final fallback.
#
# This directly addresses the previous:
#   Group Var = 0
#   PSE SE = NaN
#   PSE p = NaN
# problem.

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.stats.multitest import multipletests
from statsmodels.tools.sm_exceptions import ConvergenceWarning


# ---------------------------------------------------------------------
# WARNINGS
# ---------------------------------------------------------------------
# Do not globally suppress convergence warnings.
# The fitting helper captures them and saves them to the diagnostics files.
warnings.filterwarnings(
    "default",
    category=ConvergenceWarning,
)


# ---------------------------------------------------------------------
# PATHS / SETTINGS
# ---------------------------------------------------------------------
ROOT = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1"
)

SL_DATA = ROOT / "individual_data_SL"

SPATIAL = ROOT / "spatial_analyses"

OUT = (
    ROOT
    / "Individual_differences_analyses"
    / "Aim_2_perceptual_expression"
)

CSV_OUT = OUT / "csv_outputs"

FIG_OUT = OUT / "figures"

MODEL_OUT = OUT / "model_summaries"

DIAG_OUT = OUT / "model_diagnostics"


for d in [
    CSV_OUT,
    FIG_OUT,
    MODEL_OUT,
    DIAG_OUT,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]


# Numerical tolerance used to flag singular random-effects covariance matrices.
SINGULAR_EIG_TOL = 1e-7


# ---------------------------------------------------------------------
# FILE CHECK
# ---------------------------------------------------------------------
def require_file(
    path,
    label,
):

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"\nMissing {label}:\n"
            f"{path}\n\n"
            "Run the corresponding section of "
            "Continuous_SLv3_rebuilt first."
        )

    return path


PSE_CSV = require_file(
    SL_DATA
    / "spatial_pse_capture_hysteresis_recalibration_predictions"
    / "csv_outputs"
    / "subject_level_pse_psychometric_slope.csv",
    "Section 12 subject PSE table",
)


TRIAL_METRICS_CSV = require_file(
    SPATIAL
    / "00_core_spatial_metrics"
    / "trial_level_metrics"
    / "cell3_trial_level_lag_corrected_spatial_metrics.csv",
    "lag-corrected trial-level spatial metrics",
)


STABLE_BIAS_CSV = require_file(
    SPATIAL
    / "05_residual_visual_bias"
    / "trial_level_bias_psame_model_data.csv",
    "stable-window bias + explicit judgment trial table",
)


VALID_LAG_CSV = require_file(
    SPATIAL
    / "02_same_different_tracking_lag"
    / "csv_outputs"
    / "03_VALID_trial_lag_estimates_after_exact_collaborator_QC.csv",
    "valid auditory-lag table",
)


# =====================================================================
# GENERAL HELPERS
# =====================================================================
def clean_subject(
    s,
):

    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )


def add_zscore(
    df,
    source_col,
    new_col,
):
    """
    Add a z-scored copy of a variable.

    Z-scored versions are used for random-slope estimation because they
    improve numerical conditioning. Primary fixed-effect coefficients
    remain expressed in the original units.
    """

    x = pd.to_numeric(
        df[source_col],
        errors="coerce",
    )

    sd = x.std(
        ddof=0
    )

    if (
        not np.isfinite(sd)
        or
        sd <= 0
    ):

        df[new_col] = np.nan

    else:

        df[new_col] = (
            x
            -
            x.mean()
        ) / sd

    return df


def safe_corr(
    x,
    y,
):

    d = (
        pd.DataFrame(
            {
                "x": x,
                "y": y,
            }
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )

    if (
        len(d) < 4
        or
        d["x"].std(ddof=1) == 0
        or
        d["y"].std(ddof=1) == 0
    ):

        return {
            "n":
                len(d),

            "pearson_r":
                np.nan,

            "pearson_p":
                np.nan,

            "spearman_rho":
                np.nan,

            "spearman_p":
                np.nan,
        }

    pr = pearsonr(
        d["x"],
        d["y"],
    )

    sr = spearmanr(
        d["x"],
        d["y"],
    )

    return {
        "n":
            len(d),

        "pearson_r":
            float(pr.statistic),

        "pearson_p":
            float(pr.pvalue),

        "spearman_rho":
            float(sr.statistic),

        "spearman_p":
            float(sr.pvalue),
    }


# =====================================================================
# MIXED-MODEL DIAGNOSTICS
# =====================================================================
def _mixedlm_diagnostics(
    fit,
):
    """
    Determine whether a fitted MixedLM is suitable for inference.

    A fit is accepted only when:

      1. optimization converged;
      2. all fixed-effect estimates are finite;
      3. all fixed-effect SEs are finite;
      4. all fixed-effect p-values are finite;
      5. the random-effects covariance matrix is not singular.
    """

    fe_names = list(
        fit.fe_params.index
    )

    bse_fe = fit.bse.reindex(
        fe_names
    )

    p_fe = fit.pvalues.reindex(
        fe_names
    )

    converged = bool(
        getattr(
            fit,
            "converged",
            False,
        )
    )

    finite_fixed = bool(
        np.isfinite(
            fit.fe_params.to_numpy(
                dtype=float
            )
        ).all()
        and
        np.isfinite(
            bse_fe.to_numpy(
                dtype=float
            )
        ).all()
        and
        np.isfinite(
            p_fe.to_numpy(
                dtype=float
            )
        ).all()
    )

    try:

        cov_re = np.asarray(
            fit.cov_re,
            dtype=float,
        )

        eigvals = np.linalg.eigvalsh(
            cov_re
        )

        min_eig = (
            float(
                np.nanmin(
                    eigvals
                )
            )
            if eigvals.size
            else np.nan
        )

        max_eig = (
            float(
                np.nanmax(
                    eigvals
                )
            )
            if eigvals.size
            else np.nan
        )

        rank = int(
            np.linalg.matrix_rank(
                cov_re,
                tol=SINGULAR_EIG_TOL,
            )
        )

        expected_rank = int(
            cov_re.shape[0]
        )

        singular = (
            not np.isfinite(
                eigvals
            ).all()
            or
            rank < expected_rank
            or
            max_eig <= SINGULAR_EIG_TOL
            or
            min_eig < SINGULAR_EIG_TOL
        )

    except Exception:

        min_eig = np.nan
        max_eig = np.nan
        rank = np.nan
        expected_rank = np.nan

        singular = True


    usable = (
        converged
        and
        finite_fixed
        and
        not singular
    )


    return {
        "converged":
            converged,

        "finite_fixed_effect_inference":
            finite_fixed,

        "singular_random_effects":
            bool(singular),

        "random_effect_min_eigenvalue":
            min_eig,

        "random_effect_max_eigenvalue":
            max_eig,

        "random_effect_rank":
            rank,

        "random_effect_expected_rank":
            expected_rank,

        "usable_mixedlm":
            bool(usable),
    }


# =====================================================================
# ROBUST REPEATED-MEASURES FITTING FUNCTION
# =====================================================================
def fit_repeated_linear(
    formula,
    data,
    group="subject",
    re_formula_candidates=None,
    model_name="model",
):
    """
    Fit a continuous repeated-measures model.

    Priority
    --------
    1. MixedLM with random intercept + theoretically appropriate random slope.
    2. Progressively simpler MixedLM random-effects structures.
    3. Gaussian GEE with exchangeable participant correlation.
    4. Cluster-robust OLS as a final fallback.

    Why use GEE if the LME becomes singular?
    ----------------------------------------
    A random-effect variance estimated at zero means the requested LME
    structure is not empirically supported. Forcing that model can produce
    NaN SEs and invalid inference for participant-level predictors such as
    PSE. GEE preserves the repeated-measures structure without requiring a
    nonzero random-effects variance.
    """

    d = (
        data.copy()
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    if group not in d.columns:

        raise ValueError(
            f"Grouping column '{group}' is missing."
        )


    n_groups = d[group].nunique()


    if n_groups < 8:

        raise ValueError(
            f"Only {n_groups} participant clusters "
            f"are available for {model_name}."
        )


    if re_formula_candidates is None:

        re_formula_candidates = [
            "1",
        ]


    diagnostics_rows = []


    # -----------------------------------------------------------------
    # FIRST: ATTEMPT MIXED MODELS
    # Richest justified random-effects structure first.
    # -----------------------------------------------------------------
    for re_formula in re_formula_candidates:

        for method in [
            "lbfgs",
            "powell",
            "cg",
        ]:

            caught_messages = []

            try:

                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always"
                    )

                    model = smf.mixedlm(
                        formula,
                        data=d,
                        groups=d[group],
                        re_formula=re_formula,
                    )

                    fit = model.fit(
                        reml=False,
                        method=method,
                        maxiter=5000,
                        disp=False,
                    )

                    caught_messages = [
                        str(
                            w.message
                        )
                        for w in caught
                    ]


                diag = _mixedlm_diagnostics(
                    fit
                )


                diag.update(
                    {
                        "model_name":
                            model_name,

                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "n_rows":
                            len(d),

                        "n_groups":
                            n_groups,

                        "warnings":
                            " | ".join(
                                caught_messages
                            ),
                    }
                )


                diagnostics_rows.append(
                    diag
                )


                if diag[
                    "usable_mixedlm"
                ]:

                    diagnostics = pd.DataFrame(
                        diagnostics_rows
                    )

                    diagnostics.to_csv(
                        DIAG_OUT
                        /
                        f"{model_name}_fit_diagnostics.csv",
                        index=False,
                    )

                    return (
                        fit,
                        (
                            f"MixedLM["
                            f"{re_formula}; "
                            f"{method}]"
                        ),
                        diagnostics,
                    )


            except Exception as exc:

                diagnostics_rows.append(
                    {
                        "model_name":
                            model_name,

                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "n_rows":
                            len(d),

                        "n_groups":
                            n_groups,

                        "converged":
                            False,

                        "finite_fixed_effect_inference":
                            False,

                        "singular_random_effects":
                            True,

                        "usable_mixedlm":
                            False,

                        "warnings":
                            " | ".join(
                                caught_messages
                            ),

                        "error":
                            repr(
                                exc
                            ),
                    }
                )


    # -----------------------------------------------------------------
    # SECOND: GEE
    #
    # If all defensible LME structures are singular/unsupported,
    # use population-average repeated-measures inference rather than
    # reporting an invalid boundary MixedLM.
    # -----------------------------------------------------------------
    try:

        gee = smf.gee(
            formula,
            groups=group,
            data=d,
            family=sm.families.Gaussian(),
            cov_struct=Exchangeable(),
        ).fit()


        diagnostics_rows.append(
            {
                "model_name":
                    model_name,

                "attempt":
                    "GEE",

                "re_formula":
                    "NA",

                "optimizer":
                    "NA",

                "n_rows":
                    len(d),

                "n_groups":
                    n_groups,

                "converged":
                    bool(
                        getattr(
                            gee,
                            "converged",
                            True,
                        )
                    ),

                "finite_fixed_effect_inference":
                    bool(
                        np.isfinite(
                            gee.params
                        ).all()
                        and
                        np.isfinite(
                            gee.bse
                        ).all()
                        and
                        np.isfinite(
                            gee.pvalues
                        ).all()
                    ),

                "singular_random_effects":
                    False,

                "usable_mixedlm":
                    False,

                "warnings":
                    (
                        "LME unsupported/singular; "
                        "Gaussian GEE used."
                    ),
            }
        )


        diagnostics = pd.DataFrame(
            diagnostics_rows
        )


        diagnostics.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            gee,
            "GEE_Gaussian_Exchangeable",
            diagnostics,
        )


    except Exception as gee_exc:

        # -------------------------------------------------------------
        # FINAL SAFETY FALLBACK
        # -------------------------------------------------------------
        fit = smf.ols(
            formula,
            d,
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups":
                    d[group],
            },
        )


        diagnostics_rows.append(
            {
                "model_name":
                    model_name,

                "attempt":
                    "OLS_cluster",

                "re_formula":
                    "NA",

                "optimizer":
                    "NA",

                "n_rows":
                    len(d),

                "n_groups":
                    n_groups,

                "converged":
                    True,

                "finite_fixed_effect_inference":
                    bool(
                        np.isfinite(
                            fit.params
                        ).all()
                        and
                        np.isfinite(
                            fit.bse
                        ).all()
                        and
                        np.isfinite(
                            fit.pvalues
                        ).all()
                    ),

                "singular_random_effects":
                    False,

                "usable_mixedlm":
                    False,

                "warnings":
                    (
                        f"GEE failed: "
                        f"{repr(gee_exc)}"
                    ),
            }
        )


        diagnostics = pd.DataFrame(
            diagnostics_rows
        )


        diagnostics.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            fit,
            "OLS_cluster_fallback",
            diagnostics,
        )


# =====================================================================
# MODEL OUTPUT HELPERS
# =====================================================================
def fixed_effect_terms(
    fit,
):
    """
    Return fixed-effect names only.

    This prevents MixedLM variance parameters from being inserted into
    the fixed-effect coefficient CSV.
    """

    if hasattr(
        fit,
        "fe_params",
    ):

        return list(
            fit.fe_params.index
        )

    return list(
        fit.params.index
    )


def coef_table(
    fit,
    model_name,
    fit_type,
):

    terms = fixed_effect_terms(
        fit
    )

    params = fit.params.reindex(
        terms
    )

    bse = fit.bse.reindex(
        terms
    )

    stat = fit.tvalues.reindex(
        terms
    )

    pvals = fit.pvalues.reindex(
        terms
    )

    ci = fit.conf_int().reindex(
        terms
    )


    return pd.DataFrame(
        {
            "model":
                model_name,

            "fit_type":
                fit_type,

            "term":
                terms,

            "estimate":
                params.to_numpy(),

            "se":
                bse.to_numpy(),

            "stat":
                stat.to_numpy(),

            "p":
                pvals.to_numpy(),

            "ci95_low":
                ci.iloc[
                    :,
                    0
                ].to_numpy(),

            "ci95_high":
                ci.iloc[
                    :,
                    1
                ].to_numpy(),
        }
    )


def save_model(
    fit,
    name,
    fit_type,
    diagnostics=None,
):

    with open(
        MODEL_OUT
        /
        f"{name}.txt",
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            f"Fit type: {fit_type}\n\n"
        )

        if diagnostics is not None:

            f.write(
                "FIT DIAGNOSTICS\n"
            )

            f.write(
                diagnostics.to_string(
                    index=False
                )
            )

            f.write(
                "\n\nMODEL SUMMARY\n"
            )

        f.write(
            str(
                fit.summary()
            )
        )


    tab = coef_table(
        fit,
        name,
        fit_type,
    )


    tab.to_csv(
        CSV_OUT
        /
        f"{name}_coefficients.csv",
        index=False,
    )


    return tab


# =====================================================================
# PARTICIPANT-LEVEL VISUALIZATION HELPER
# =====================================================================
def scatter_with_fit(
    df,
    x,
    y,
    xlabel,
    ylabel,
    title,
    path,
):
    """
    Participant-level descriptive figure.

    This is separate from the trial-level repeated-measures inference.
    """

    d = (
        df[
            [
                x,
                y,
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .copy()
    )


    if len(d) < 4:

        return


    X = sm.add_constant(
        d[
            [
                x
            ]
        ],
        has_constant="add",
    )


    fit = sm.OLS(
        d[y],
        X,
    ).fit(
        cov_type="HC3"
    )


    xx = np.linspace(
        d[x].min(),
        d[x].max(),
        200,
    )


    XX = sm.add_constant(
        pd.DataFrame(
            {
                x:
                    xx,
            }
        ),
        has_constant="add",
    )


    pred = (
        fit.get_prediction(
            XX
        )
        .summary_frame(
            alpha=.05
        )
    )


    c = safe_corr(
        d[x],
        d[y],
    )


    fig, ax = plt.subplots(
        figsize=(
            7.0,
            5.6,
        )
    )


    ax.scatter(
        d[x],
        d[y],
        s=55,
        alpha=.72,
    )


    ax.plot(
        xx,
        pred[
            "mean"
        ].to_numpy(),
        linewidth=2.5,
    )


    ax.fill_between(
        xx,
        pred[
            "mean_ci_lower"
        ].to_numpy(),
        pred[
            "mean_ci_upper"
        ].to_numpy(),
        alpha=.18,
    )


    ax.set_xlabel(
        xlabel
    )

    ax.set_ylabel(
        ylabel
    )

    ax.set_title(
        title
    )


    ax.text(
        .04,
        .96,
        (
            f"n={c['n']}\n"
            f"r={c['pearson_r']:.3f}, "
            f"p={c['pearson_p']:.4g}\n"
            f"rho={c['spearman_rho']:.3f}"
        ),
        transform=ax.transAxes,
        va="top",
        bbox=dict(
            boxstyle="round,pad=.3",
            facecolor="white",
            alpha=.85,
        ),
    )


    fig.tight_layout()


    fig.savefig(
        path,
        dpi=250,
        bbox_inches="tight",
    )


    plt.show()


# =====================================================================
# LOAD PSE
# =====================================================================
pse = pd.read_csv(
    PSE_CSV
)


pse[
    "subject"
] = clean_subject(
    pse[
        "subject"
    ]
)


pse[
    "PSE_empirical_deg"
] = pd.to_numeric(
    pse[
        "PSE_empirical_deg"
    ],
    errors="coerce",
)


pse = (
    pse[
        [
            "subject",
            "PSE_empirical_deg",
            "psychometric_precision",
        ]
    ]
    .drop_duplicates(
        "subject"
    )
    .copy()
)


# Primary PSE remains in degrees.
# This preserves interpretable beta coefficients:
# change in outcome per 1-degree increase in PSE.
pse[
    "PSE_c"
] = (
    pse[
        "PSE_empirical_deg"
    ]
    -
    pse[
        "PSE_empirical_deg"
    ].mean()
)


# Standardized version retained for numerical/sensitivity purposes.
pse = add_zscore(
    pse,
    "PSE_empirical_deg",
    "PSE_z",
)


pse.to_csv(
    CSV_OUT
    /
    "aim2_subject_pse_predictor_table.csv",
    index=False,
)


# =====================================================================
# 2A. PSE -> VISUAL CAPTURE
# =====================================================================
cap = pd.read_csv(
    TRIAL_METRICS_CSV
)


cap[
    "subject"
] = clean_subject(
    cap[
        "subject"
    ]
)


cap[
    "offset_deg"
] = pd.to_numeric(
    cap[
        "offsetDeg"
    ],
    errors="coerce",
).abs()


cap[
    "VisualCapture_audioLag"
] = pd.to_numeric(
    cap[
        "VisualCapture_audioLag"
    ],
    errors="coerce",
)


sweep_flag = (
    cap.get(
        "trialType",
        pd.Series(
            "",
            index=cap.index,
        ),
    )
    .astype(str)
    .str.lower()
    .str.contains(
        "sweep"
    )
    |
    cap.get(
        "condLabel",
        pd.Series(
            "",
            index=cap.index,
        ),
    )
    .astype(str)
    .str.lower()
    .str.contains(
        "sweep|diverge|converge",
        regex=True,
    )
)


cap = cap[
    cap[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
    &
    ~sweep_flag
].copy()


cap = cap.merge(
    pse[
        [
            "subject",
            "PSE_empirical_deg",
            "PSE_c",
            "PSE_z",
        ]
    ],
    on="subject",
    how="inner",
)


cap = cap.dropna(
    subset=[
        "VisualCapture_audioLag",
        "offset_deg",
        "PSE_empirical_deg",
    ]
).copy()


cap[
    "offset_c"
] = (
    cap[
        "offset_deg"
    ]
    -
    np.mean(
        FIXED_OFFSETS
    )
)


# Standardized offset is used only for the random slope.
cap = add_zscore(
    cap,
    "offset_deg",
    "offset_z",
)


# ---------------------------------------------------------------------
# PRIMARY 2A MODEL
#
# Fixed:
#   visual capture ~ offset * PSE
#
# Random:
#   participant intercept + participant-specific offset slope
#
# If unsupported:
#   random intercept only -> GEE -> clustered OLS.
# ---------------------------------------------------------------------
cap_model, cap_fit_type, cap_diag = fit_repeated_linear(
    formula=(
        "VisualCapture_audioLag ~ "
        "offset_c * PSE_c"
    ),
    data=cap,
    group="subject",
    re_formula_candidates=[
        "1 + offset_z",
        "1",
    ],
    model_name="aim2A_capture_model",
)


cap_coef = save_model(
    cap_model,
    "aim2A_capture_mixed_model",
    cap_fit_type,
    cap_diag,
)


# ---------------------------------------------------------------------
# PARTICIPANT-LEVEL DESCRIPTIVE ASSOCIATION
# ---------------------------------------------------------------------
cap_subj = (
    cap.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        mean_capture=(
            "VisualCapture_audioLag",
            "mean",
        ),

        PSE_empirical_deg=(
            "PSE_empirical_deg",
            "first",
        ),
    )
)


cap_corr = safe_corr(
    cap_subj[
        "PSE_empirical_deg"
    ],
    cap_subj[
        "mean_capture"
    ],
)


# ---------------------------------------------------------------------
# OFFSET-SPECIFIC EXPLORATORY CORRELATIONS
# ---------------------------------------------------------------------
cap_offset_rows = []


for off, g in cap.groupby(
    "offset_deg"
):

    s = (
        g.groupby(
            "subject",
            as_index=False,
        )
        .agg(
            capture=(
                "VisualCapture_audioLag",
                "mean",
            ),

            PSE=(
                "PSE_empirical_deg",
                "first",
            ),
        )
    )


    cc = safe_corr(
        s[
            "PSE"
        ],
        s[
            "capture"
        ],
    )


    cap_offset_rows.append(
        {
            "offset_deg":
                off,

            **cc,
        }
    )


cap_offset = (
    pd.DataFrame(
        cap_offset_rows
    )
    .sort_values(
        "offset_deg"
    )
)


if len(
    cap_offset
):

    valid = cap_offset[
        "pearson_p"
    ].notna()


    cap_offset.loc[
        valid,
        "pearson_p_fdr",
    ] = multipletests(
        cap_offset.loc[
            valid,
            "pearson_p",
        ],
        method="fdr_bh",
    )[
        1
    ]


cap_offset.to_csv(
    CSV_OUT
    /
    "aim2A_capture_offset_specific_correlations.csv",
    index=False,
)


cap_subj.to_csv(
    CSV_OUT
    /
    "aim2A_subject_mean_capture.csv",
    index=False,
)


scatter_with_fit(
    cap_subj,
    "PSE_empirical_deg",
    "mean_capture",
    "SL PSE (deg)",
    "Mean lag-corrected normalized visual capture",
    "Causal tolerance predicts spatial visual capture",
    FIG_OUT
    /
    "aim2A_pse_vs_visual_capture.png",
)


# ---------------------------------------------------------------------
# MODEL-PREDICTED CURVES:
# 25th / median / 75th percentile PSE
# ---------------------------------------------------------------------
q = (
    pse[
        "PSE_empirical_deg"
    ]
    .dropna()
    .quantile(
        [
            .25,
            .50,
            .75,
        ]
    )
    .to_dict()
)


grid_rows = []


for label, pval in [
    (
        "25th percentile",
        q[
            .25
        ],
    ),
    (
        "Median",
        q[
            .50
        ],
    ),
    (
        "75th percentile",
        q[
            .75
        ],
    ),
]:

    for off in FIXED_OFFSETS:

        grid_rows.append(
            {
                "PSE_group":
                    label,

                "PSE_empirical_deg":
                    pval,

                "PSE_c":
                    (
                        pval
                        -
                        pse[
                            "PSE_empirical_deg"
                        ].mean()
                    ),

                "offset_deg":
                    off,

                "offset_c":
                    (
                        off
                        -
                        np.mean(
                            FIXED_OFFSETS
                        )
                    ),
            }
        )


grid = pd.DataFrame(
    grid_rows
)


try:

    grid[
        "pred_capture"
    ] = cap_model.predict(
        grid
    )


    fig, ax = plt.subplots(
        figsize=(
            7,
            5.3,
        )
    )


    for label, g in grid.groupby(
        "PSE_group",
        sort=False,
    ):

        ax.plot(
            g[
                "offset_deg"
            ],
            g[
                "pred_capture"
            ],
            marker="o",
            label=label,
        )


    ax.set_xlabel(
        "A-V offset magnitude (deg)"
    )

    ax.set_ylabel(
        "Predicted lag-corrected visual capture"
    )

    ax.set_title(
        "Model-predicted visual capture "
        "across PSE levels"
    )

    ax.legend(
        frameon=False
    )


    fig.tight_layout()


    fig.savefig(
        FIG_OUT
        /
        "aim2A_capture_predicted_by_offset_and_pse.png",
        dpi=250,
        bbox_inches="tight",
    )


    plt.show()


except Exception as e:

    print(
        "Prediction plot skipped:",
        e,
    )


# =====================================================================
# 2B. PSE -> RESIDUAL VISUAL BIAS / JUDGMENT INTERACTION
# =====================================================================
bias = pd.read_csv(
    STABLE_BIAS_CSV
)


bias[
    "subject"
] = clean_subject(
    bias[
        "subject"
    ]
)


for c in [
    "offset_num",
    "Psame",
    "Bias_lagcorr_AminusV_stable",
]:

    bias[c] = pd.to_numeric(
        bias[c],
        errors="coerce",
    )


bias = bias[
    bias[
        "offset_num"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


bias = bias.merge(
    pse[
        [
            "subject",
            "PSE_empirical_deg",
            "PSE_c",
            "PSE_z",
        ]
    ],
    on="subject",
    how="inner",
)


bias = bias.dropna(
    subset=[
        "offset_num",
        "Psame",
        "Bias_lagcorr_AminusV_stable",
        "PSE_c",
    ]
).copy()


bias[
    "Psame_c"
] = (
    bias[
        "Psame"
    ]
    -
    bias[
        "Psame"
    ].mean()
)


bias[
    "offset_c"
] = (
    bias[
        "offset_num"
    ]
    -
    np.mean(
        FIXED_OFFSETS
    )
)


bias = add_zscore(
    bias,
    "offset_num",
    "offset_z",
)


# ---------------------------------------------------------------------
# PRIMARY 2B MODEL
#
# Fixed:
#   stable visual bias ~ offset + judgment + PSE + judgment × PSE
#
# Random-effect candidates, richest -> simplest:
#   intercept + offset slope + judgment slope
#   intercept + judgment slope
#   intercept + offset slope
#   intercept only
#
# GEE is used if none of these is supported.
# ---------------------------------------------------------------------
bias_model, bias_fit_type, bias_diag = fit_repeated_linear(
    formula=(
        "Bias_lagcorr_AminusV_stable ~ "
        "C(offset_num) + "
        "Psame_c + "
        "PSE_c + "
        "Psame_c:PSE_c"
    ),
    data=bias,
    group="subject",
    re_formula_candidates=[
        "1 + offset_z + Psame_c",
        "1 + Psame_c",
        "1 + offset_z",
        "1",
    ],
    model_name="aim2B_bias_pse_by_judgment_model",
)


bias_coef = save_model(
    bias_model,
    "aim2B_bias_pse_by_judgment_mixed_model",
    bias_fit_type,
    bias_diag,
)


# ---------------------------------------------------------------------
# DIFFERENT-SOURCE-ONLY ANALYSIS
# ---------------------------------------------------------------------
diff = bias[
    bias[
        "Psame"
    ]
    ==
    0
].copy()


diff_model, diff_fit_type, diff_diag = fit_repeated_linear(
    formula=(
        "Bias_lagcorr_AminusV_stable ~ "
        "C(offset_num) + "
        "PSE_c"
    ),
    data=diff,
    group="subject",
    re_formula_candidates=[
        "1 + offset_z",
        "1",
    ],
    model_name="aim2B_different_only_bias_model",
)


diff_coef = save_model(
    diff_model,
    "aim2B_different_only_bias_mixed_model",
    diff_fit_type,
    diff_diag,
)


# ---------------------------------------------------------------------
# PARTICIPANT-LEVEL VISUALIZATION
#
# Remove average offset differences within Different-source trials.
# ---------------------------------------------------------------------
offset_fit = smf.ols(
    "Bias_lagcorr_AminusV_stable ~ "
    "C(offset_num)",
    data=diff,
).fit()


diff[
    "Bias_resid_after_offset"
] = offset_fit.resid


diff_subj = (
    diff.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        mean_residual_visual_bias=(
            "Bias_resid_after_offset",
            "mean",
        ),

        PSE_empirical_deg=(
            "PSE_empirical_deg",
            "first",
        ),

        n_different_trials=(
            "trialIndex",
            "count",
        ),
    )
)


diff_subj.to_csv(
    CSV_OUT
    /
    "aim2B_different_source_residual_bias_subjects.csv",
    index=False,
)


scatter_with_fit(
    diff_subj,
    "PSE_empirical_deg",
    "mean_residual_visual_bias",
    "SL PSE (deg)",
    (
        "Different-source visual bias residual\n"
        "(after removing offset means)"
    ),
    (
        "Does integration propensity persist "
        "after explicit segregation?"
    ),
    FIG_OUT
    /
    "aim2B_pse_vs_different_source_residual_bias.png",
)


# =====================================================================
# 2C. PSE -> AUDITORY TRACKING LAG
# =====================================================================
lag = pd.read_csv(
    VALID_LAG_CSV
)


lag[
    "subject"
] = clean_subject(
    lag[
        "subject"
    ]
)


lag[
    "offset_deg"
] = pd.to_numeric(
    lag[
        "offset_deg"
    ],
    errors="coerce",
).abs()


lag[
    "lag_audio_sec"
] = pd.to_numeric(
    lag[
        "lag_audio_sec"
    ],
    errors="coerce",
)


lag = lag[
    lag[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


lag = lag.merge(
    pse[
        [
            "subject",
            "PSE_empirical_deg",
            "PSE_c",
            "PSE_z",
        ]
    ],
    on="subject",
    how="inner",
)


lag = lag.dropna(
    subset=[
        "lag_audio_sec",
        "offset_deg",
        "PSE_c",
    ]
).copy()


lag[
    "offset_c"
] = (
    lag[
        "offset_deg"
    ]
    -
    np.mean(
        FIXED_OFFSETS
    )
)


lag = add_zscore(
    lag,
    "offset_deg",
    "offset_z",
)


# ---------------------------------------------------------------------
# PRIMARY 2C MODEL
#
# Fixed:
#   lag ~ offset × PSE
#
# Random:
#   participant intercept + participant-specific offset slope
# ---------------------------------------------------------------------
lag_model, lag_fit_type, lag_diag = fit_repeated_linear(
    formula=(
        "lag_audio_sec ~ "
        "offset_c * PSE_c"
    ),
    data=lag,
    group="subject",
    re_formula_candidates=[
        "1 + offset_z",
        "1",
    ],
    model_name="aim2C_lag_model",
)


lag_coef = save_model(
    lag_model,
    "aim2C_lag_mixed_model",
    lag_fit_type,
    lag_diag,
)


# ---------------------------------------------------------------------
# PARTICIPANT-LEVEL DESCRIPTIVE ASSOCIATION
# ---------------------------------------------------------------------
lag_subj = (
    lag.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        mean_auditory_lag_sec=(
            "lag_audio_sec",
            "mean",
        ),

        PSE_empirical_deg=(
            "PSE_empirical_deg",
            "first",
        ),
    )
)


lag_subj.to_csv(
    CSV_OUT
    /
    "aim2C_subject_mean_auditory_lag.csv",
    index=False,
)


scatter_with_fit(
    lag_subj,
    "PSE_empirical_deg",
    "mean_auditory_lag_sec",
    "SL PSE (deg)",
    "Mean auditory-relative tracking lag (s)",
    (
        "Does causal tolerance predict "
        "temporal tracking?"
    ),
    FIG_OUT
    /
    "aim2C_pse_vs_auditory_lag.png",
)


# =====================================================================
# PRIMARY TEST SUMMARY
# =====================================================================
primary_rows = []


def add_primary(
    label,
    fit,
    fit_type,
    term,
):

    primary_rows.append(
        {
            "analysis":
                label,

            "fit_type":
                fit_type,

            "term":
                term,

            "estimate":
                fit.params.get(
                    term,
                    np.nan,
                ),

            "se":
                fit.bse.get(
                    term,
                    np.nan,
                ),

            "stat":
                fit.tvalues.get(
                    term,
                    np.nan,
                ),

            "p":
                fit.pvalues.get(
                    term,
                    np.nan,
                ),
        }
    )


add_primary(
    "2A capture: main PSE effect",
    cap_model,
    cap_fit_type,
    "PSE_c",
)


add_primary(
    "2A capture: disparity x PSE",
    cap_model,
    cap_fit_type,
    "offset_c:PSE_c",
)


add_primary(
    "2B bias: main PSE effect",
    bias_model,
    bias_fit_type,
    "PSE_c",
)


add_primary(
    "2B bias: judgment x PSE",
    bias_model,
    bias_fit_type,
    "Psame_c:PSE_c",
)


add_primary(
    "2B Different-only bias: PSE",
    diff_model,
    diff_fit_type,
    "PSE_c",
)


add_primary(
    "2C lag: main PSE effect",
    lag_model,
    lag_fit_type,
    "PSE_c",
)


add_primary(
    "2C lag: disparity x PSE",
    lag_model,
    lag_fit_type,
    "offset_c:PSE_c",
)


primary = pd.DataFrame(
    primary_rows
)


valid = primary[
    "p"
].notna()


if valid.any():

    primary.loc[
        valid,
        "p_fdr",
    ] = multipletests(
        primary.loc[
            valid,
            "p",
        ],
        method="fdr_bh",
    )[
        1
    ]


primary.to_csv(
    CSV_OUT
    /
    "aim2_primary_tests_summary.csv",
    index=False,
)


# =====================================================================
# 3-PANEL PARTICIPANT-LEVEL SUMMARY FIGURE
# =====================================================================
fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        16,
        5,
    ),
)


panels = [
    (
        cap_subj,
        "PSE_empirical_deg",
        "mean_capture",
        "Mean visual capture",
        "Spatial weighting",
    ),

    (
        diff_subj,
        "PSE_empirical_deg",
        "mean_residual_visual_bias",
        "Residual visual bias",
        "Bias after Different report",
    ),

    (
        lag_subj,
        "PSE_empirical_deg",
        "mean_auditory_lag_sec",
        "Auditory lag (s)",
        "Temporal tracking",
    ),
]


for ax, (
    d,
    x,
    y,
    ylab,
    title,
) in zip(
    axes,
    panels,
):

    qd = (
        d[
            [
                x,
                y,
            ]
        ]
        .dropna()
        .copy()
    )


    ax.scatter(
        qd[x],
        qd[y],
        alpha=.72,
    )


    if (
        len(qd) >= 4
        and
        qd[x].std() > 0
    ):

        X = sm.add_constant(
            qd[
                [
                    x
                ]
            ],
            has_constant="add",
        )


        fit = sm.OLS(
            qd[y],
            X,
        ).fit(
            cov_type="HC3"
        )


        xx = np.linspace(
            qd[x].min(),
            qd[x].max(),
            150,
        )


        XX = sm.add_constant(
            pd.DataFrame(
                {
                    x:
                        xx,
                }
            ),
            has_constant="add",
        )


        pred = (
            fit.get_prediction(
                XX
            )
            .summary_frame(
                alpha=.05
            )
        )


        ax.plot(
            xx,
            pred[
                "mean"
            ],
            linewidth=2,
        )


        ax.fill_between(
            xx,
            pred[
                "mean_ci_lower"
            ].to_numpy(),
            pred[
                "mean_ci_upper"
            ].to_numpy(),
            alpha=.15,
        )


    ax.set_xlabel(
        "SL PSE (deg)"
    )

    ax.set_ylabel(
        ylab
    )

    ax.set_title(
        title
    )


fig.suptitle(
    (
        "Aim 2: Does causal tolerance predict "
        "perceptual expression?"
    ),
    fontsize=16,
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim2_perceptual_expression_three_panel.png",
    dpi=250,
    bbox_inches="tight",
)


plt.show()


# =====================================================================
# PRINT DIAGNOSTIC SUMMARY
# =====================================================================
print(
    "\nAIM 2 COMPLETE"
)


print(
    "\nPrimary tests:"
)


print(
    primary.to_string(
        index=False
    )
)


print(
    "\nChosen repeated-measures estimators:"
)


for label, fit_type in [
    (
        "2A capture",
        cap_fit_type,
    ),
    (
        "2B bias",
        bias_fit_type,
    ),
    (
        "2B Different-only",
        diff_fit_type,
    ),
    (
        "2C lag",
        lag_fit_type,
    ),
]:

    print(
        f"  {label}: {fit_type}"
    )


print(
    "\nIMPORTANT:"
)


print(
    "A MixedLM is used only when the random-effects covariance is "
    "supported and all fixed-effect SEs/p-values are finite. "
    "If a random effect collapses to the boundary, the code "
    "automatically uses Gaussian GEE rather than reporting a "
    "singular MixedLM."
)


print(
    f"\nSaved outputs to:\n{OUT}"
)


AIM 2 COMPLETE

Primary tests:
                   analysis                               fit_type           term  estimate       se      stat        p    p_fdr
2A capture: main PSE effect           MixedLM[1 + offset_z; lbfgs]          PSE_c  0.007416 0.001945  3.813935 0.000137 0.000957
2A capture: disparity x PSE           MixedLM[1 + offset_z; lbfgs] offset_c:PSE_c -0.000093 0.000102 -0.918246 0.358490 0.534600
   2B bias: main PSE effect MixedLM[1 + offset_z + Psame_c; lbfgs]          PSE_c -0.023263 0.056805 -0.409517 0.682160 0.795854
    2B bias: judgment x PSE MixedLM[1 + offset_z + Psame_c; lbfgs]  Psame_c:PSE_c -0.196920 0.094238 -2.089599 0.036654 0.085526
2B Different-only bias: PSE           MixedLM[1 + offset_z; lbfgs]          PSE_c  0.072616 0.083039  0.874479 0.381857 0.534600
    2C lag: main PSE effect           MixedLM[1 + offset_z; lbfgs]          PSE_c -0.026177 0.009028 -2.899607 0.003736 0.013077
    2C lag: disparity x PSE           MixedLM[1 + offset_z; lbfgs

In [48]:
# %% AIM 2 — PUBLICATION/PRESENTATION CORRELATION FIGURES WITH 95% CI
#
# Paste AFTER the full Aim 2 cell.
#
# Uses objects from Aim 2:
#   cap
#   cap_subj
#   diff_subj
#   lag
#   lag_subj
#   FIG_OUT
#
# Produces:
#   - overall PSE -> visual capture
#   - PSE -> capture at 10°, 20°, 30°, 40°
#   - PSE -> residual visual bias after Different reports
#   - PSE -> overall auditory lag
#   - PSE -> lag at each offset
#
# Plot style matches the supplied example.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import pearsonr

CORR_FIG_OUT = (
    Path(FIG_OUT)
    / "correlation_figures_95CI"
)

CORR_FIG_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

POINT_COLOR = "#1f77b4"
LINE_COLOR = "#1f77b4"
CI_COLOR = "#ff7f0e"


def format_p(p):
    if not np.isfinite(p):
        return "NA"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"


def plot_corr_ci(
    ax,
    df,
    x,
    y,
    xlabel,
    ylabel,
    title,
    point_size=62,
):
    """
    Participant-level OLS figure with:
      Pearson r/p
      HC3 beta/p
      fitted mean line
      95% CI of regression mean
    """

    d = (
        df[
            [x, y]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .copy()
    )

    if len(d) < 4:

        ax.text(
            .5,
            .5,
            "Insufficient data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

        ax.set_title(
            title
        )

        return None

    # Pearson r
    rr = pearsonr(
        d[x],
        d[y],
    )

    r = float(
        rr.statistic
    )

    p_r = float(
        rr.pvalue
    )

    # Robust OLS
    X = sm.add_constant(
        d[[x]],
        has_constant="add",
    )

    fit = sm.OLS(
        d[y],
        X,
    ).fit(
        cov_type="HC3"
    )

    beta = float(
        fit.params[x]
    )

    p_beta = float(
        fit.pvalues[x]
    )

    # Prediction interval for mean
    xx = np.linspace(
        d[x].min(),
        d[x].max(),
        250,
    )

    XX = sm.add_constant(
        pd.DataFrame(
            {
                x: xx,
            }
        ),
        has_constant="add",
    )

    pred = (
        fit.get_prediction(
            XX
        )
        .summary_frame(
            alpha=.05
        )
    )

    # Plot
    ax.scatter(
        d[x],
        d[y],
        s=point_size,
        alpha=.72,
        color=POINT_COLOR,
        edgecolor=POINT_COLOR,
    )

    ax.fill_between(
        xx,
        pred["mean_ci_lower"].to_numpy(),
        pred["mean_ci_upper"].to_numpy(),
        color=CI_COLOR,
        alpha=.22,
        linewidth=0,
        label="95% CI",
    )

    ax.plot(
        xx,
        pred["mean"].to_numpy(),
        color=LINE_COLOR,
        linewidth=2.8,
        label="OLS fit",
    )

    ax.set_xlabel(
        xlabel,
        fontsize=12,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=12,
    )

    ax.set_title(
        title,
        fontsize=15,
        pad=10,
    )

    stats_text = (
        f"n = {len(d)}\n"
        f"r = {r:.3f}, p {format_p(p_r)}\n"
        f"β = {beta:.4f}, p {format_p(p_beta)}"
    )

    ax.text(
        .04,
        .96,
        stats_text,
        transform=ax.transAxes,
        va="top",
        fontsize=11.5,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="black",
            alpha=.88,
        ),
    )

    ax.legend(
        frameon=False,
        loc="upper right",
    )

    return fit


# =====================================================================
# 2A — OVERALL PSE -> VISUAL CAPTURE
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.3, 6.1)
)

plot_corr_ci(
    ax=ax,
    df=cap_subj,
    x="PSE_empirical_deg",
    y="mean_capture",
    xlabel="Subject PSE: P(same)=.5 boundary (deg)",
    ylabel="Mean lag-corrected visual capture",
    title="PSE predicting mean visual capture",
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim2A_PSE_predicting_overall_visual_capture_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 2A — OFFSET-SPECIFIC CAPTURE CORRELATIONS
# =====================================================================
offset_capture_tables = {}

for off in [10, 20, 30, 40]:

    g = (
        cap[
            cap["offset_deg"]
            ==
            off
        ]
        .groupby(
            "subject",
            as_index=False,
        )
        .agg(
            PSE_empirical_deg=(
                "PSE_empirical_deg",
                "first",
            ),
            mean_capture=(
                "VisualCapture_audioLag",
                "mean",
            ),
        )
    )

    offset_capture_tables[
        off
    ] = g

    fig, ax = plt.subplots(
        figsize=(7.3, 6.1)
    )

    plot_corr_ci(
        ax=ax,
        df=g,
        x="PSE_empirical_deg",
        y="mean_capture",
        xlabel="Subject PSE: P(same)=.5 boundary (deg)",
        ylabel=f"Mean visual capture at {off}°",
        title=f"PSE predicting visual capture at {off}°",
    )

    fig.tight_layout()

    fig.savefig(
        CORR_FIG_OUT
        /
        f"aim2A_PSE_predicting_visual_capture_{off}deg_95CI.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# Combined 2 × 2 offset panel
fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.5, 11.0),
)

axes = axes.ravel()

for ax, off in zip(
    axes,
    [10, 20, 30, 40],
):

    plot_corr_ci(
        ax=ax,
        df=offset_capture_tables[off],
        x="PSE_empirical_deg",
        y="mean_capture",
        xlabel="Subject PSE (deg)",
        ylabel=f"Visual capture at {off}°",
        title=f"{off}° offset",
        point_size=48,
    )

fig.suptitle(
    "PSE predicting visual capture across fixed disparities",
    fontsize=19,
    y=1.01,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim2A_PSE_capture_all_offsets_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 2B — PSE -> RESIDUAL BIAS AFTER DIFFERENT-SOURCE REPORT
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.3, 6.1)
)

plot_corr_ci(
    ax=ax,
    df=diff_subj,
    x="PSE_empirical_deg",
    y="mean_residual_visual_bias",
    xlabel="Subject PSE: P(same)=.5 boundary (deg)",
    ylabel=(
        "Residual visual bias after\n"
        "Different-source report"
    ),
    title=(
        "PSE predicting residual visual bias "
        "after explicit segregation"
    ),
)

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=.6,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim2B_PSE_predicting_Different_source_residual_bias_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 2C — OVERALL PSE -> AUDITORY TRACKING LAG
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.3, 6.1)
)

plot_corr_ci(
    ax=ax,
    df=lag_subj,
    x="PSE_empirical_deg",
    y="mean_auditory_lag_sec",
    xlabel="Subject PSE: P(same)=.5 boundary (deg)",
    ylabel="Mean auditory-relative tracking lag (s)",
    title="PSE predicting auditory tracking lag",
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim2C_PSE_predicting_overall_auditory_lag_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 2C — OFFSET-SPECIFIC AUDITORY-LAG CORRELATIONS
# =====================================================================
offset_lag_tables = {}

for off in [10, 20, 30, 40]:

    g = (
        lag[
            lag["offset_deg"]
            ==
            off
        ]
        .groupby(
            "subject",
            as_index=False,
        )
        .agg(
            PSE_empirical_deg=(
                "PSE_empirical_deg",
                "first",
            ),
            mean_lag=(
                "lag_audio_sec",
                "mean",
            ),
        )
    )

    offset_lag_tables[
        off
    ] = g


fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.5, 11.0),
)

axes = axes.ravel()

for ax, off in zip(
    axes,
    [10, 20, 30, 40],
):

    plot_corr_ci(
        ax=ax,
        df=offset_lag_tables[off],
        x="PSE_empirical_deg",
        y="mean_lag",
        xlabel="Subject PSE (deg)",
        ylabel=f"Auditory lag at {off}° (s)",
        title=f"{off}° offset",
        point_size=48,
    )

fig.suptitle(
    "PSE predicting auditory tracking lag across fixed disparities",
    fontsize=19,
    y=1.01,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim2C_PSE_lag_all_offsets_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


print(
    "\nUpdated Aim 2 correlation figures saved to:"
)

print(
    CORR_FIG_OUT
)

KeyError: "Column(s) ['PSE_empirical_deg'] do not exist"

In [60]:
# %% AIM 3 — ARE SPATIAL DYNAMICS ORGANIZED AROUND EACH OBSERVER'S CAUSAL BOUNDARY?
#
# Analyses
#   3A. Re-express visual capture as D* = physical disparity - participant PSE.
#   3B. Test whether breakpoint dynamicity is greatest near |D - PSE| = 0.
#   3C. Original LOPO comparison: physical disparity vs participant-relative disparity.
#   3D. REFINED boundary test:
#       (a) compare physical disparity + PSE against observer-relative D*;
#       (b) test for a slope change specifically at D*=0;
#       (c) for dynamicity, test whether D*=0 is a local maximum;
#       (d) compare the refined models using leave-one-participant-out prediction.
#
# Outputs:
#   ...\Individual_differences_analyses\Aim_3_boundary_relative_dynamics

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.cov_struct import Exchangeable

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, log_loss, roc_auc_score

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------------
# PATHS / SETTINGS
# ---------------------------------------------------------------------
ROOT = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1"
)

SL_DATA = ROOT / "individual_data_SL"
SPATIAL = ROOT / "spatial_analyses"

OUT = (
    ROOT
    / "Individual_differences_analyses"
    / "Aim_3_boundary_relative_dynamics"
)

CSV_OUT = OUT / "csv_outputs"
FIG_OUT = OUT / "figures"
MODEL_OUT = OUT / "model_summaries"

for d in [CSV_OUT, FIG_OUT, MODEL_OUT]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )

FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]

RNG_SEED = 20261010
N_PERM = 10000


# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def require_file(
    path,
    label,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"\nMissing {label}:\n"
            f"{path}\n\n"
            "Run the corresponding section of "
            "Continuous_SLv3_rebuilt first."
        )

    return path


PSE_CSV = require_file(
    SL_DATA
    / "spatial_pse_capture_hysteresis_recalibration_predictions"
    / "csv_outputs"
    / "subject_level_pse_psychometric_slope.csv",
    "Section 12 subject PSE table",
)

TRIAL_METRICS_CSV = require_file(
    SPATIAL
    / "00_core_spatial_metrics"
    / "trial_level_metrics"
    / "cell3_trial_level_lag_corrected_spatial_metrics.csv",
    "lag-corrected trial spatial metrics",
)

BREAKPOINT_CSV = require_file(
    SPATIAL
    / "06_stabilization_breakpoint"
    / "csv_outputs"
    / "anchor_trial_level_breakpoint_DYNAMIC_metrics.csv",
    "Section 11 breakpoint trial table",
)


def clean_subject(
    s,
):
    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )


def save_result(
    result,
    name,
):
    with open(
        MODEL_OUT / f"{name}.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            str(
                result.summary()
            )
        )

    ci = result.conf_int()

    tab = pd.DataFrame(
        {
            "term":
                result.params.index,

            "estimate":
                result.params.values,

            "se":
                result.bse.reindex(
                    result.params.index
                ).values,

            "stat":
                result.tvalues.reindex(
                    result.params.index
                ).values,

            "p":
                result.pvalues.reindex(
                    result.params.index
                ).values,

            "ci95_low":
                ci.reindex(
                    result.params.index
                ).iloc[:, 0].values,

            "ci95_high":
                ci.reindex(
                    result.params.index
                ).iloc[:, 1].values,
        }
    )

    tab.to_csv(
        CSV_OUT
        / f"{name}_coefficients.csv",
        index=False,
    )

    return tab


def paired_signflip_test(
    differences,
    n_perm=N_PERM,
    seed=RNG_SEED,
):
    """
    Two-sided participant-level sign-flip test.

    Positive differences mean the participant-relative boundary model
    performed BETTER than the physical + PSE model.
    """

    x = np.asarray(
        differences,
        dtype=float,
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    if len(x) < 3:
        return {
            "n":
                len(x),

            "mean_improvement":
                np.nan,

            "median_improvement":
                np.nan,

            "prop_boundary_better":
                np.nan,

            "signflip_p":
                np.nan,
        }

    observed = float(
        np.mean(
            x
        )
    )

    rng = np.random.default_rng(
        seed
    )

    null = np.empty(
        n_perm,
        dtype=float,
    )

    for k in range(
        n_perm
    ):
        signs = rng.choice(
            [
                -1.0,
                1.0,
            ],
            size=len(x),
            replace=True,
        )

        null[k] = np.mean(
            x
            *
            signs
        )

    p = (
        np.sum(
            np.abs(
                null
            )
            >=
            abs(
                observed
            )
        )
        +
        1
    ) / (
        n_perm
        +
        1
    )

    return {
        "n":
            len(x),

        "mean_improvement":
            observed,

        "median_improvement":
            float(
                np.median(
                    x
                )
            ),

        "prop_boundary_better":
            float(
                np.mean(
                    x
                    >
                    0
                )
            ),

        "signflip_p":
            float(
                p
            ),
    }


# ---------------------------------------------------------------------
# LOAD PSE
# ---------------------------------------------------------------------
pse = pd.read_csv(
    PSE_CSV
)

pse[
    "subject"
] = clean_subject(
    pse[
        "subject"
    ]
)

pse[
    "PSE_empirical_deg"
] = pd.to_numeric(
    pse[
        "PSE_empirical_deg"
    ],
    errors="coerce",
)

pse = (
    pse[
        [
            "subject",
            "PSE_empirical_deg",
        ]
    ]
    .drop_duplicates(
        "subject"
    )
    .dropna()
    .copy()
)

pse[
    "PSE_c"
] = (
    pse[
        "PSE_empirical_deg"
    ]
    -
    pse[
        "PSE_empirical_deg"
    ].mean()
)


# =====================================================================
# 3A. PARTICIPANT-RELATIVE DISPARITY -> VISUAL CAPTURE
# =====================================================================
cap = pd.read_csv(
    TRIAL_METRICS_CSV
)

cap[
    "subject"
] = clean_subject(
    cap[
        "subject"
    ]
)

cap[
    "offset_deg"
] = pd.to_numeric(
    cap[
        "offsetDeg"
    ],
    errors="coerce",
).abs()

cap[
    "VisualCapture_audioLag"
] = pd.to_numeric(
    cap[
        "VisualCapture_audioLag"
    ],
    errors="coerce",
)


sweep_flag = (
    cap.get(
        "trialType",
        pd.Series(
            "",
            index=cap.index,
        ),
    )
    .astype(str)
    .str.lower()
    .str.contains(
        "sweep"
    )
    |
    cap.get(
        "condLabel",
        pd.Series(
            "",
            index=cap.index,
        ),
    )
    .astype(str)
    .str.lower()
    .str.contains(
        "sweep|diverge|converge",
        regex=True,
    )
)


cap = cap[
    cap[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
    &
    ~sweep_flag
].copy()


cap = cap.merge(
    pse,
    on="subject",
    how="inner",
)


cap = cap.dropna(
    subset=[
        "VisualCapture_audioLag",
        "offset_deg",
        "PSE_empirical_deg",
    ]
).copy()


cap[
    "Dstar"
] = (
    cap[
        "offset_deg"
    ]
    -
    cap[
        "PSE_empirical_deg"
    ]
)


cap[
    "boundary_proximity"
] = cap[
    "Dstar"
].abs()


cap[
    "above_boundary"
] = (
    cap[
        "Dstar"
    ]
    > 0
).astype(
    int
)


# ---------------------------------------------------------------------
# Flexible participant-relative model
# ---------------------------------------------------------------------
capture_gee = smf.gee(
    (
        "VisualCapture_audioLag ~ "
        "bs(Dstar, df=4, degree=3)"
    ),
    groups="subject",
    data=cap,
    family=sm.families.Gaussian(),
    cov_struct=Exchangeable(),
).fit()


capture_gee_tab = save_result(
    capture_gee,
    "aim3A_capture_by_participant_relative_disparity_GEE",
)


# ---------------------------------------------------------------------
# Physical-disparity comparator using offset as a categorical factor
#
# There are only four tested fixed offsets (10/20/30/40°), so treating
# offset as categorical avoids overparameterizing a spline basis.
# ---------------------------------------------------------------------
capture_phys_gee = smf.gee(
    (
        "VisualCapture_audioLag ~ "
        "C(offset_deg)"
    ),
    groups="subject",
    data=cap,
    family=sm.families.Gaussian(),
    cov_struct=Exchangeable(),
).fit()


save_result(
    capture_phys_gee,
    "aim3A_capture_by_physical_disparity_GEE",
)


cap.to_csv(
    CSV_OUT
    /
    "aim3_fixed_trial_boundary_relative_capture.csv",
    index=False,
)


# ---------------------------------------------------------------------
# Subject-first D* bins for visualization
# ---------------------------------------------------------------------
bin_edges = np.arange(
    np.floor(
        cap[
            "Dstar"
        ].min()
        /
        5
    )
    *
    5,

    np.ceil(
        cap[
            "Dstar"
        ].max()
        /
        5
    )
    *
    5
    +
    5,

    5,
)


cap[
    "Dstar_bin"
] = pd.cut(
    cap[
        "Dstar"
    ],
    bins=bin_edges,
    include_lowest=True,
)


cap[
    "Dstar_bin_center"
] = (
    cap[
        "Dstar_bin"
    ]
    .apply(
        lambda z:
            z.mid
            if pd.notna(
                z
            )
            else np.nan
    )
    .astype(
        float
    )
)


cap_subj_bin = (
    cap.groupby(
        [
            "subject",
            "Dstar_bin_center",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        mean_capture=(
            "VisualCapture_audioLag",
            "mean",
        )
    )
)


cap_group_bin = (
    cap_subj_bin.groupby(
        "Dstar_bin_center",
        as_index=False,
    )
    .agg(
        mean=(
            "mean_capture",
            "mean",
        ),

        sd=(
            "mean_capture",
            "std",
        ),

        n=(
            "subject",
            "nunique",
        ),
    )
)


cap_group_bin[
    "se"
] = (
    cap_group_bin[
        "sd"
    ]
    /
    np.sqrt(
        cap_group_bin[
            "n"
        ]
    )
)


fig, ax = plt.subplots(
    figsize=(
        8,
        5.5,
    )
)


ax.errorbar(
    cap_group_bin[
        "Dstar_bin_center"
    ],
    cap_group_bin[
        "mean"
    ],
    yerr=cap_group_bin[
        "se"
    ],
    marker="o",
    capsize=4,
    linewidth=2,
)


ax.axvline(
    0,
    linestyle="--",
    linewidth=1.5,
)


ax.set_xlabel(
    "Participant-relative disparity "
    "D* = physical disparity − PSE (deg)"
)


ax.set_ylabel(
    "Lag-corrected normalized visual capture"
)


ax.set_title(
    "Spatial weighting in observer-relative "
    "causal coordinates"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim3A_visual_capture_by_Dstar.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()


# =====================================================================
# 3B. BOUNDARY PROXIMITY -> BREAKPOINT DYNAMICITY
# =====================================================================
bp = pd.read_csv(
    BREAKPOINT_CSV
)

bp[
    "subject"
] = clean_subject(
    bp[
        "subject"
    ]
)

bp[
    "offset_deg"
] = pd.to_numeric(
    bp[
        "offset_deg"
    ],
    errors="coerce",
)


if "metric" in bp.columns:
    bp = bp[
        bp[
            "metric"
        ]
        .astype(str)
        .eq(
            "LagCorrectedVisualCapture"
        )
    ].copy()


bp = bp[
    bp[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


bp[
    "dynamically_updating"
] = pd.to_numeric(
    bp[
        "dynamically_updating"
    ],
    errors="coerce",
)


bp = bp.merge(
    pse,
    on="subject",
    how="inner",
)


bp = bp.dropna(
    subset=[
        "offset_deg",
        "PSE_empirical_deg",
        "dynamically_updating",
    ]
).copy()


bp[
    "Dstar"
] = (
    bp[
        "offset_deg"
    ]
    -
    bp[
        "PSE_empirical_deg"
    ]
)


bp[
    "boundary_proximity"
] = bp[
    "Dstar"
].abs()


bp[
    "proximity_c"
] = (
    bp[
        "boundary_proximity"
    ]
    -
    bp[
        "boundary_proximity"
    ].mean()
)


bp[
    "above_boundary"
] = (
    bp[
        "Dstar"
    ]
    > 0
).astype(
    int
)


# ---------------------------------------------------------------------
# Original asymmetric proximity model
# ---------------------------------------------------------------------
dynamic_gee = smf.gee(
    (
        "dynamically_updating ~ "
        "proximity_c * above_boundary + "
        "PSE_c"
    ),
    groups="subject",
    data=bp,
    family=sm.families.Binomial(),
    cov_struct=Exchangeable(),
).fit()


dynamic_gee_tab = save_result(
    dynamic_gee,
    "aim3B_dynamicity_by_boundary_proximity_GEE",
)


bp.to_csv(
    CSV_OUT
    /
    "aim3_breakpoint_trials_with_boundary_coordinates.csv",
    index=False,
)


# ---------------------------------------------------------------------
# Participant-first proximity bins
# ---------------------------------------------------------------------
prox_edges = np.arange(
    0,

    np.ceil(
        bp[
            "boundary_proximity"
        ].max()
        /
        5
    )
    *
    5
    +
    5,

    5,
)


bp[
    "proximity_bin"
] = pd.cut(
    bp[
        "boundary_proximity"
    ],
    bins=prox_edges,
    include_lowest=True,
)


bp[
    "proximity_bin_center"
] = (
    bp[
        "proximity_bin"
    ]
    .apply(
        lambda z:
            z.mid
            if pd.notna(
                z
            )
            else np.nan
    )
    .astype(
        float
    )
)


bp_subj_bin = (
    bp.groupby(
        [
            "subject",
            "proximity_bin_center",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        p_dynamic=(
            "dynamically_updating",
            "mean",
        )
    )
)


bp_group_bin = (
    bp_subj_bin.groupby(
        "proximity_bin_center",
        as_index=False,
    )
    .agg(
        mean=(
            "p_dynamic",
            "mean",
        ),

        sd=(
            "p_dynamic",
            "std",
        ),

        n=(
            "subject",
            "nunique",
        ),
    )
)


bp_group_bin[
    "se"
] = (
    bp_group_bin[
        "sd"
    ]
    /
    np.sqrt(
        bp_group_bin[
            "n"
        ]
    )
)


fig, ax = plt.subplots(
    figsize=(
        8,
        5.5,
    )
)


ax.errorbar(
    bp_group_bin[
        "proximity_bin_center"
    ],
    bp_group_bin[
        "mean"
    ],
    yerr=bp_group_bin[
        "se"
    ],
    marker="o",
    capsize=4,
    linewidth=2,
)


ax.set_xlabel(
    "Distance from participant-specific PSE, "
    "|D − PSE| (deg)"
)


ax.set_ylabel(
    "Probability of dynamically updating SL trial"
)


ax.set_ylim(
    0,
    1,
)


ax.set_title(
    "Is spatial recalibration greatest near "
    "each observer's causal boundary?"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim3B_dynamicity_by_boundary_proximity.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()


# =====================================================================
# 3C. ORIGINAL LEAVE-ONE-PARTICIPANT-OUT PREDICTION
# =====================================================================
logo = LeaveOneGroupOut()


# ---------------------------------------------------------------------
# CONTINUOUS OUTCOME: VISUAL CAPTURE
# ---------------------------------------------------------------------
cont = (
    cap[
        [
            "subject",
            "offset_deg",
            "Dstar",
            "VisualCapture_audioLag",
        ]
    ]
    .dropna()
    .copy()
)


groups = cont[
    "subject"
].to_numpy()


y = cont[
    "VisualCapture_audioLag"
].to_numpy(
    float
)


pred_rows = []


for model_name, xcol in [
    (
        "physical_disparity",
        "offset_deg",
    ),
    (
        "participant_relative_Dstar",
        "Dstar",
    ),
]:

    preds = np.full(
        len(
            cont
        ),
        np.nan,
    )

    for train, test in logo.split(
        cont,
        y,
        groups,
    ):

        pipe = Pipeline(
            [
                (
                    "poly",
                    PolynomialFeatures(
                        degree=3,
                        include_bias=False,
                    ),
                ),

                (
                    "scale",
                    StandardScaler(),
                ),

                (
                    "ridge",
                    Ridge(
                        alpha=1.0
                    ),
                ),
            ]
        )

        pipe.fit(
            cont.iloc[
                train
            ][
                [
                    xcol
                ]
            ].to_numpy(
                float
            ),
            y[
                train
            ],
        )

        preds[
            test
        ] = pipe.predict(
            cont.iloc[
                test
            ][
                [
                    xcol
                ]
            ].to_numpy(
                float
            )
        )

    for i in range(
        len(
            cont
        )
    ):

        pred_rows.append(
            {
                "outcome":
                    "visual_capture",

                "model":
                    model_name,

                "subject":
                    cont.iloc[
                        i
                    ][
                        "subject"
                    ],

                "observed":
                    y[
                        i
                    ],

                "predicted":
                    preds[
                        i
                    ],
            }
        )


cont_pred = pd.DataFrame(
    pred_rows
)


cont_pred.to_csv(
    CSV_OUT
    /
    "aim3C_LOPO_capture_predictions.csv",
    index=False,
)


cont_perf_rows = []


for model_name, g in cont_pred.groupby(
    "model"
):

    cont_perf_rows.append(
        {
            "outcome":
                "visual_capture",

            "model":
                model_name,

            "RMSE":
                np.sqrt(
                    mean_squared_error(
                        g[
                            "observed"
                        ],
                        g[
                            "predicted"
                        ],
                    )
                ),

            "R2":
                r2_score(
                    g[
                        "observed"
                    ],
                    g[
                        "predicted"
                    ],
                ),
        }
    )


cont_perf = pd.DataFrame(
    cont_perf_rows
)


# ---------------------------------------------------------------------
# BINARY OUTCOME: BREAKPOINT DYNAMICITY
# ---------------------------------------------------------------------
bd = (
    bp[
        [
            "subject",
            "offset_deg",
            "Dstar",
            "boundary_proximity",
            "dynamically_updating",
        ]
    ]
    .dropna()
    .copy()
)


g2 = bd[
    "subject"
].to_numpy()


yb = (
    bd[
        "dynamically_updating"
    ]
    .astype(
        int
    )
    .to_numpy()
)


binary_pred_rows = []


feature_sets = {
    "physical_disparity":
        [
            "offset_deg",
        ],

    "participant_relative_boundary":
        [
            "Dstar",
            "boundary_proximity",
        ],
}


for model_name, cols in feature_sets.items():

    probs = np.full(
        len(
            bd
        ),
        np.nan,
    )

    for train, test in logo.split(
        bd,
        yb,
        g2,
    ):

        if len(
            np.unique(
                yb[
                    train
                ]
            )
        ) < 2:
            continue

        if model_name == "physical_disparity":

            pipe = Pipeline(
                [
                    (
                        "poly",
                        PolynomialFeatures(
                            degree=2,
                            include_bias=False,
                        ),
                    ),

                    (
                        "scale",
                        StandardScaler(),
                    ),

                    (
                        "logit",
                        LogisticRegression(
                            C=1.0,
                            max_iter=5000,
                        ),
                    ),
                ]
            )

        else:

            pipe = Pipeline(
                [
                    (
                        "scale",
                        StandardScaler(),
                    ),

                    (
                        "logit",
                        LogisticRegression(
                            C=1.0,
                            max_iter=5000,
                        ),
                    ),
                ]
            )

        pipe.fit(
            bd.iloc[
                train
            ][
                cols
            ].to_numpy(
                float
            ),
            yb[
                train
            ],
        )

        probs[
            test
        ] = (
            pipe.predict_proba(
                bd.iloc[
                    test
                ][
                    cols
                ].to_numpy(
                    float
                )
            )[
                :,
                1
            ]
        )

    for i in range(
        len(
            bd
        )
    ):

        binary_pred_rows.append(
            {
                "outcome":
                    "breakpoint_dynamicity",

                "model":
                    model_name,

                "subject":
                    bd.iloc[
                        i
                    ][
                        "subject"
                    ],

                "observed":
                    yb[
                        i
                    ],

                "predicted_probability":
                    probs[
                        i
                    ],
            }
        )


bin_pred = (
    pd.DataFrame(
        binary_pred_rows
    )
    .dropna(
        subset=[
            "predicted_probability",
        ]
    )
)


bin_pred.to_csv(
    CSV_OUT
    /
    "aim3C_LOPO_dynamicity_predictions.csv",
    index=False,
)


bin_perf_rows = []


for model_name, g in bin_pred.groupby(
    "model"
):

    auc = np.nan

    if g[
        "observed"
    ].nunique() == 2:

        auc = roc_auc_score(
            g[
                "observed"
            ],
            g[
                "predicted_probability"
            ],
        )

    bin_perf_rows.append(
        {
            "outcome":
                "breakpoint_dynamicity",

            "model":
                model_name,

            "log_loss":
                log_loss(
                    g[
                        "observed"
                    ],
                    g[
                        "predicted_probability"
                    ],
                    labels=[
                        0,
                        1,
                    ],
                ),

            "AUC":
                auc,
        }
    )


bin_perf = pd.DataFrame(
    bin_perf_rows
)


perf = pd.concat(
    [
        cont_perf,
        bin_perf,
    ],
    ignore_index=True,
    sort=False,
)


perf.to_csv(
    CSV_OUT
    /
    "aim3C_LOPO_model_comparison_summary.csv",
    index=False,
)


# ---------------------------------------------------------------------
# ORIGINAL HELD-OUT MODEL COMPARISON FIGURE
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        11,
        4.8,
    ),
)


cp = cont_perf.set_index(
    "model"
)


axes[
    0
].bar(
    [
        "Physical D",
        "Participant-relative D*",
    ],
    [
        cp.loc[
            "physical_disparity",
            "RMSE",
        ],

        cp.loc[
            "participant_relative_Dstar",
            "RMSE",
        ],
    ],
)


axes[
    0
].set_ylabel(
    "Held-out RMSE"
)


axes[
    0
].set_title(
    "Visual capture prediction\n"
    "(lower is better)"
)


bpperf = bin_perf.set_index(
    "model"
)


axes[
    1
].bar(
    [
        "Physical D",
        "Personal boundary",
    ],
    [
        bpperf.loc[
            "physical_disparity",
            "log_loss",
        ],

        bpperf.loc[
            "participant_relative_boundary",
            "log_loss",
        ],
    ],
)


axes[
    1
].set_ylabel(
    "Held-out log loss"
)


axes[
    1
].set_title(
    "Dynamicity prediction\n"
    "(lower is better)"
)


fig.suptitle(
    "Do observer-relative coordinates "
    "improve held-out prediction?",
    fontsize=15,
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim3C_LOPO_coordinate_model_comparison.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()


# =====================================================================
# 3D. REFINED TEST: IS D* = 0 A SPECIAL COMPUTATIONAL POINT?
# =====================================================================
#
# Why this is necessary:
#
# D* = D - PSE already contains information about BOTH physical disparity
# and PSE. Therefore, showing that D* outperforms D alone cannot establish
# that the participant's .50 boundary itself is special.
#
# We now ask:
#
#   1. Does an observer-relative boundary representation compete with a
#      model explicitly given BOTH physical disparity and PSE?
#
#   2. Is there a statistically detectable slope change specifically at
#      D*=0?
#
#   3. For dynamicity, is D*=0 actually a local maximum?
#
# Continuous hinge variables:
#
#       Dstar_below = min(D*, 0)
#       Dstar_above = max(D*, 0)
#
# The two fitted segments therefore meet exactly at D*=0.
# =====================================================================


# ---------------------------------------------------------------------
# Construct boundary-hinge variables
# ---------------------------------------------------------------------
for d in [
    cap,
    bp,
]:

    d[
        "Dstar_below"
    ] = np.minimum(
        d[
            "Dstar"
        ].to_numpy(
            float
        ),
        0.0,
    )

    d[
        "Dstar_above"
    ] = np.maximum(
        d[
            "Dstar"
        ].to_numpy(
            float
        ),
        0.0,
    )

    d[
        "offset_c_refined"
    ] = (
        d[
            "offset_deg"
        ]
        -
        d[
            "offset_deg"
        ].mean()
    )


# =====================================================================
# 3D-A. VISUAL CAPTURE — INFERENTIAL MODELS
# =====================================================================

# ---------------------------------------------------------------------
# Additive physical model:
#
#   Visual capture ~ physical disparity + PSE
#
# Physical disparity is treated as a categorical factor because there are
# only four tested fixed offsets (10/20/30/40°). This avoids the rank/SE
# instability that occurred when a 4-df cubic spline was fit to only four
# unique offset values while still controlling flexibly for physical disparity.
# ---------------------------------------------------------------------
capture_additive_gee = smf.gee(
    (
        "VisualCapture_audioLag ~ "
        "C(offset_deg) + "
        "PSE_c"
    ),
    groups="subject",
    data=cap,
    family=sm.families.Gaussian(),
    cov_struct=Exchangeable(),
).fit()


capture_additive_tab = save_result(
    capture_additive_gee,
    "aim3D_capture_additive_physical_plus_PSE_GEE",
)


# ---------------------------------------------------------------------
# Continuous piecewise-linear boundary model
#
# The pre-specified knot is exactly D*=0.
# ---------------------------------------------------------------------
capture_boundary_hinge_gee = smf.gee(
    (
        "VisualCapture_audioLag ~ "
        "Dstar_below + "
        "Dstar_above"
    ),
    groups="subject",
    data=cap,
    family=sm.families.Gaussian(),
    cov_struct=Exchangeable(),
).fit()


capture_boundary_hinge_tab = save_result(
    capture_boundary_hinge_gee,
    "aim3D_capture_boundary_hinge_GEE",
)


# ---------------------------------------------------------------------
# Is there a slope change at D*=0?
# ---------------------------------------------------------------------
try:

    capture_kink_test = (
        capture_boundary_hinge_gee
        .wald_test(
            "Dstar_below = Dstar_above",
            scalar=True,
        )
    )

    capture_kink_stat = float(
        np.asarray(
            capture_kink_test.statistic
        ).squeeze()
    )

    capture_kink_p = float(
        np.asarray(
            capture_kink_test.pvalue
        ).squeeze()
    )

except Exception:

    capture_kink_stat = np.nan
    capture_kink_p = np.nan


capture_below_beta = (
    capture_boundary_hinge_gee
    .params
    .get(
        "Dstar_below",
        np.nan,
    )
)


capture_above_beta = (
    capture_boundary_hinge_gee
    .params
    .get(
        "Dstar_above",
        np.nan,
    )
)


capture_boundary_summary = pd.DataFrame(
    [
        {
            "outcome":
                "visual_capture",

            "below_boundary_slope":
                capture_below_beta,

            "below_boundary_p":
                capture_boundary_hinge_gee
                .pvalues
                .get(
                    "Dstar_below",
                    np.nan,
                ),

            "above_boundary_slope":
                capture_above_beta,

            "above_boundary_p":
                capture_boundary_hinge_gee
                .pvalues
                .get(
                    "Dstar_above",
                    np.nan,
                ),

            "slope_change_wald_stat":
                capture_kink_stat,

            "slope_change_p":
                capture_kink_p,
        }
    ]
)


capture_boundary_summary.to_csv(
    CSV_OUT
    /
    "aim3D_capture_boundary_knot_test.csv",
    index=False,
)


# =====================================================================
# 3D-B. BREAKPOINT DYNAMICITY — INFERENTIAL MODELS
# =====================================================================

# ---------------------------------------------------------------------
# Additive physical model:
#
#   Dynamicity ~ physical disparity + physical disparity^2 + PSE
#
# The quadratic term is retained because Aim 3B explicitly concerns
# possible nonlinear dynamicity.
# ---------------------------------------------------------------------
dynamic_additive_gee = smf.gee(
    (
        "dynamically_updating ~ "
        "offset_c_refined + "
        "I(offset_c_refined**2) + "
        "PSE_c"
    ),
    groups="subject",
    data=bp,
    family=sm.families.Binomial(),
    cov_struct=Exchangeable(),
).fit()


dynamic_additive_tab = save_result(
    dynamic_additive_gee,
    "aim3D_dynamicity_additive_physical_plus_PSE_GEE",
)


# ---------------------------------------------------------------------
# Boundary-knot model
# ---------------------------------------------------------------------
dynamic_boundary_hinge_gee = smf.gee(
    (
        "dynamically_updating ~ "
        "Dstar_below + "
        "Dstar_above"
    ),
    groups="subject",
    data=bp,
    family=sm.families.Binomial(),
    cov_struct=Exchangeable(),
).fit()


dynamic_boundary_hinge_tab = save_result(
    dynamic_boundary_hinge_gee,
    "aim3D_dynamicity_boundary_hinge_GEE",
)


# ---------------------------------------------------------------------
# Test slope change at D*=0
# ---------------------------------------------------------------------
try:

    dynamic_kink_test = (
        dynamic_boundary_hinge_gee
        .wald_test(
            "Dstar_below = Dstar_above",
            scalar=True,
        )
    )

    dynamic_kink_stat = float(
        np.asarray(
            dynamic_kink_test.statistic
        ).squeeze()
    )

    dynamic_kink_p = float(
        np.asarray(
            dynamic_kink_test.pvalue
        ).squeeze()
    )

except Exception:

    dynamic_kink_stat = np.nan
    dynamic_kink_p = np.nan


dynamic_below_beta = (
    dynamic_boundary_hinge_gee
    .params
    .get(
        "Dstar_below",
        np.nan,
    )
)


dynamic_above_beta = (
    dynamic_boundary_hinge_gee
    .params
    .get(
        "Dstar_above",
        np.nan,
    )
)


# A local MAXIMUM at D*=0 requires:
#
#   below-boundary slope > 0
#   above-boundary slope < 0
#
# i.e., dynamicity rises while approaching the boundary from below,
# then falls after crossing beyond it.
local_peak_pattern = bool(
    np.isfinite(
        dynamic_below_beta
    )
    and
    np.isfinite(
        dynamic_above_beta
    )
    and
    dynamic_below_beta > 0
    and
    dynamic_above_beta < 0
)


dynamic_boundary_summary = pd.DataFrame(
    [
        {
            "outcome":
                "breakpoint_dynamicity",

            "below_boundary_logodds_slope":
                dynamic_below_beta,

            "below_boundary_OR_per_degree":
                (
                    np.exp(
                        dynamic_below_beta
                    )
                    if np.isfinite(
                        dynamic_below_beta
                    )
                    else np.nan
                ),

            "below_boundary_p":
                dynamic_boundary_hinge_gee
                .pvalues
                .get(
                    "Dstar_below",
                    np.nan,
                ),

            "above_boundary_logodds_slope":
                dynamic_above_beta,

            "above_boundary_OR_per_degree":
                (
                    np.exp(
                        dynamic_above_beta
                    )
                    if np.isfinite(
                        dynamic_above_beta
                    )
                    else np.nan
                ),

            "above_boundary_p":
                dynamic_boundary_hinge_gee
                .pvalues
                .get(
                    "Dstar_above",
                    np.nan,
                ),

            "slope_change_wald_stat":
                dynamic_kink_stat,

            "slope_change_p":
                dynamic_kink_p,

            "local_maximum_at_Dstar_zero":
                local_peak_pattern,
        }
    ]
)


dynamic_boundary_summary.to_csv(
    CSV_OUT
    /
    "aim3D_dynamicity_boundary_knot_test.csv",
    index=False,
)


# =====================================================================
# 3D-C. REFINED LEAVE-ONE-PARTICIPANT-OUT MODEL COMPARISON
# =====================================================================
#
# Both competing models receive FOUR engineered predictors so that
# predictive flexibility is roughly matched.
#
# PHYSICAL + PSE:
#
#   D
#   D^2
#   D^3
#   PSE
#
# D* BOUNDARY MODEL:
#
#   D*
#   D*^2
#   D*^3
#   max(D*, 0)
#
# The final term explicitly allows a slope change at D*=0.
# =====================================================================


# ---------------------------------------------------------------------
# VISUAL CAPTURE REFINED DATA
# ---------------------------------------------------------------------
cap_refined = (
    cap[
        [
            "subject",
            "offset_deg",
            "PSE_empirical_deg",
            "Dstar",
            "Dstar_above",
            "VisualCapture_audioLag",
        ]
    ]
    .dropna()
    .copy()
)


cap_refined[
    "offset_sq"
] = (
    cap_refined[
        "offset_deg"
    ]
    ** 2
)


cap_refined[
    "offset_cu"
] = (
    cap_refined[
        "offset_deg"
    ]
    ** 3
)


cap_refined[
    "Dstar_sq"
] = (
    cap_refined[
        "Dstar"
    ]
    ** 2
)


cap_refined[
    "Dstar_cu"
] = (
    cap_refined[
        "Dstar"
    ]
    ** 3
)


capture_feature_sets = {
    "physical_plus_PSE":
        [
            "offset_deg",
            "offset_sq",
            "offset_cu",
            "PSE_empirical_deg",
        ],

    "Dstar_boundary_model":
        [
            "Dstar",
            "Dstar_sq",
            "Dstar_cu",
            "Dstar_above",
        ],
}


groups_refined = (
    cap_refined[
        "subject"
    ]
    .to_numpy()
)


y_refined = (
    cap_refined[
        "VisualCapture_audioLag"
    ]
    .to_numpy(
        float
    )
)


capture_refined_prediction_rows = []


for model_name, cols in capture_feature_sets.items():

    preds = np.full(
        len(
            cap_refined
        ),
        np.nan,
    )

    for train, test in logo.split(
        cap_refined,
        y_refined,
        groups_refined,
    ):

        pipe = Pipeline(
            [
                (
                    "scale",
                    StandardScaler(),
                ),

                (
                    "ridge",
                    Ridge(
                        alpha=1.0
                    ),
                ),
            ]
        )

        pipe.fit(
            cap_refined.iloc[
                train
            ][
                cols
            ].to_numpy(
                float
            ),
            y_refined[
                train
            ],
        )

        preds[
            test
        ] = pipe.predict(
            cap_refined.iloc[
                test
            ][
                cols
            ].to_numpy(
                float
            )
        )

    for i in range(
        len(
            cap_refined
        )
    ):

        capture_refined_prediction_rows.append(
            {
                "outcome":
                    "visual_capture",

                "model":
                    model_name,

                "subject":
                    cap_refined.iloc[
                        i
                    ][
                        "subject"
                    ],

                "observed":
                    y_refined[
                        i
                    ],

                "predicted":
                    preds[
                        i
                    ],
            }
        )


capture_refined_pred = pd.DataFrame(
    capture_refined_prediction_rows
)


capture_refined_pred.to_csv(
    CSV_OUT
    /
    "aim3D_LOPO_capture_additive_vs_boundary_predictions.csv",
    index=False,
)


capture_refined_perf_rows = []


for model_name, g in capture_refined_pred.groupby(
    "model"
):

    capture_refined_perf_rows.append(
        {
            "outcome":
                "visual_capture",

            "model":
                model_name,

            "RMSE":
                np.sqrt(
                    mean_squared_error(
                        g[
                            "observed"
                        ],
                        g[
                            "predicted"
                        ],
                    )
                ),

            "R2":
                r2_score(
                    g[
                        "observed"
                    ],
                    g[
                        "predicted"
                    ],
                ),
        }
    )


capture_refined_perf = pd.DataFrame(
    capture_refined_perf_rows
)


# ---------------------------------------------------------------------
# Participant-wise capture RMSE
# ---------------------------------------------------------------------
capture_subject_perf_rows = []


for (
    subject,
    model_name,
), g in capture_refined_pred.groupby(
    [
        "subject",
        "model",
    ]
):

    capture_subject_perf_rows.append(
        {
            "subject":
                subject,

            "model":
                model_name,

            "RMSE":
                np.sqrt(
                    mean_squared_error(
                        g[
                            "observed"
                        ],
                        g[
                            "predicted"
                        ],
                    )
                ),
        }
    )


capture_subject_perf = pd.DataFrame(
    capture_subject_perf_rows
)


capture_subject_wide = (
    capture_subject_perf
    .pivot(
        index="subject",
        columns="model",
        values="RMSE",
    )
    .reset_index()
)


capture_subject_wide[
    "boundary_improvement"
] = (
    capture_subject_wide[
        "physical_plus_PSE"
    ]
    -
    capture_subject_wide[
        "Dstar_boundary_model"
    ]
)


capture_cv_test = paired_signflip_test(
    capture_subject_wide[
        "boundary_improvement"
    ]
)


capture_subject_wide.to_csv(
    CSV_OUT
    /
    "aim3D_capture_subjectwise_CV_comparison.csv",
    index=False,
)


# ---------------------------------------------------------------------
# DYNAMICITY REFINED DATA
# ---------------------------------------------------------------------
dyn_refined = (
    bp[
        [
            "subject",
            "offset_deg",
            "PSE_empirical_deg",
            "Dstar",
            "Dstar_above",
            "dynamically_updating",
        ]
    ]
    .dropna()
    .copy()
)


dyn_refined[
    "offset_sq"
] = (
    dyn_refined[
        "offset_deg"
    ]
    ** 2
)


dyn_refined[
    "offset_cu"
] = (
    dyn_refined[
        "offset_deg"
    ]
    ** 3
)


dyn_refined[
    "Dstar_sq"
] = (
    dyn_refined[
        "Dstar"
    ]
    ** 2
)


dyn_refined[
    "Dstar_cu"
] = (
    dyn_refined[
        "Dstar"
    ]
    ** 3
)


dynamic_feature_sets_refined = {
    "physical_plus_PSE":
        [
            "offset_deg",
            "offset_sq",
            "offset_cu",
            "PSE_empirical_deg",
        ],

    "Dstar_boundary_model":
        [
            "Dstar",
            "Dstar_sq",
            "Dstar_cu",
            "Dstar_above",
        ],
}


groups_dyn_refined = (
    dyn_refined[
        "subject"
    ]
    .to_numpy()
)


y_dyn_refined = (
    dyn_refined[
        "dynamically_updating"
    ]
    .astype(
        int
    )
    .to_numpy()
)


dynamic_refined_prediction_rows = []


for model_name, cols in dynamic_feature_sets_refined.items():

    probs = np.full(
        len(
            dyn_refined
        ),
        np.nan,
    )

    for train, test in logo.split(
        dyn_refined,
        y_dyn_refined,
        groups_dyn_refined,
    ):

        if len(
            np.unique(
                y_dyn_refined[
                    train
                ]
            )
        ) < 2:
            continue

        pipe = Pipeline(
            [
                (
                    "scale",
                    StandardScaler(),
                ),

                (
                    "logit",
                    LogisticRegression(
                        C=1.0,
                        max_iter=5000,
                    ),
                ),
            ]
        )

        pipe.fit(
            dyn_refined.iloc[
                train
            ][
                cols
            ].to_numpy(
                float
            ),
            y_dyn_refined[
                train
            ],
        )

        probs[
            test
        ] = (
            pipe.predict_proba(
                dyn_refined.iloc[
                    test
                ][
                    cols
                ].to_numpy(
                    float
                )
            )[
                :,
                1
            ]
        )

    for i in range(
        len(
            dyn_refined
        )
    ):

        dynamic_refined_prediction_rows.append(
            {
                "outcome":
                    "breakpoint_dynamicity",

                "model":
                    model_name,

                "subject":
                    dyn_refined.iloc[
                        i
                    ][
                        "subject"
                    ],

                "observed":
                    y_dyn_refined[
                        i
                    ],

                "predicted_probability":
                    probs[
                        i
                    ],
            }
        )


dynamic_refined_pred = (
    pd.DataFrame(
        dynamic_refined_prediction_rows
    )
    .dropna(
        subset=[
            "predicted_probability",
        ]
    )
)


dynamic_refined_pred.to_csv(
    CSV_OUT
    /
    "aim3D_LOPO_dynamicity_additive_vs_boundary_predictions.csv",
    index=False,
)


dynamic_refined_perf_rows = []


for model_name, g in dynamic_refined_pred.groupby(
    "model"
):

    auc = np.nan

    if g[
        "observed"
    ].nunique() == 2:

        auc = roc_auc_score(
            g[
                "observed"
            ],
            g[
                "predicted_probability"
            ],
        )

    dynamic_refined_perf_rows.append(
        {
            "outcome":
                "breakpoint_dynamicity",

            "model":
                model_name,

            "log_loss":
                log_loss(
                    g[
                        "observed"
                    ],
                    g[
                        "predicted_probability"
                    ],
                    labels=[
                        0,
                        1,
                    ],
                ),

            "AUC":
                auc,
        }
    )


dynamic_refined_perf = pd.DataFrame(
    dynamic_refined_perf_rows
)


# ---------------------------------------------------------------------
# Participant-wise dynamicity log loss
# ---------------------------------------------------------------------
dynamic_subject_perf_rows = []


for (
    subject,
    model_name,
), g in dynamic_refined_pred.groupby(
    [
        "subject",
        "model",
    ]
):

    dynamic_subject_perf_rows.append(
        {
            "subject":
                subject,

            "model":
                model_name,

            "log_loss":
                log_loss(
                    g[
                        "observed"
                    ],
                    g[
                        "predicted_probability"
                    ],
                    labels=[
                        0,
                        1,
                    ],
                ),
        }
    )


dynamic_subject_perf = pd.DataFrame(
    dynamic_subject_perf_rows
)


dynamic_subject_wide = (
    dynamic_subject_perf
    .pivot(
        index="subject",
        columns="model",
        values="log_loss",
    )
    .reset_index()
)


dynamic_subject_wide[
    "boundary_improvement"
] = (
    dynamic_subject_wide[
        "physical_plus_PSE"
    ]
    -
    dynamic_subject_wide[
        "Dstar_boundary_model"
    ]
)


dynamic_cv_test = paired_signflip_test(
    dynamic_subject_wide[
        "boundary_improvement"
    ]
)


dynamic_subject_wide.to_csv(
    CSV_OUT
    /
    "aim3D_dynamicity_subjectwise_CV_comparison.csv",
    index=False,
)


# ---------------------------------------------------------------------
# Save compact refined-model summaries
# ---------------------------------------------------------------------
refined_perf = pd.concat(
    [
        capture_refined_perf,
        dynamic_refined_perf,
    ],
    ignore_index=True,
    sort=False,
)


refined_perf.to_csv(
    CSV_OUT
    /
    "aim3D_LOPO_physical_plus_PSE_vs_boundary_summary.csv",
    index=False,
)


refined_cv_tests = pd.DataFrame(
    [
        {
            "outcome":
                "visual_capture",

            "metric":
                "RMSE",

            **capture_cv_test,
        },

        {
            "outcome":
                "breakpoint_dynamicity",

            "metric":
                "log_loss",

            **dynamic_cv_test,
        },
    ]
)


refined_cv_tests.to_csv(
    CSV_OUT
    /
    "aim3D_participantwise_boundary_model_tests.csv",
    index=False,
)


# =====================================================================
# 3D FIGURE 1 — PIECEWISE BEHAVIOR AROUND D*=0
# =====================================================================
cap_plot = (
    cap.groupby(
        [
            "subject",
            "offset_deg",
        ],
        as_index=False,
    )
    .agg(
        Dstar=(
            "Dstar",
            "first",
        ),

        mean_capture=(
            "VisualCapture_audioLag",
            "mean",
        ),
    )
)


bp_plot = (
    bp.groupby(
        [
            "subject",
            "offset_deg",
        ],
        as_index=False,
    )
    .agg(
        Dstar=(
            "Dstar",
            "first",
        ),

        P_dynamic=(
            "dynamically_updating",
            "mean",
        ),
    )
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        13.5,
        5.3,
    ),
)


# ---------------------------------------------------------------------
# Capture
# ---------------------------------------------------------------------
axes[
    0
].scatter(
    cap_plot[
        "Dstar"
    ],
    cap_plot[
        "mean_capture"
    ],
    alpha=.30,
    s=28,
)


x_left_cap = np.linspace(
    cap[
        "Dstar"
    ].min(),
    0,
    100,
)


x_right_cap = np.linspace(
    0,
    cap[
        "Dstar"
    ].max(),
    100,
)


capture_intercept = (
    capture_boundary_hinge_gee
    .params
    .get(
        "Intercept",
        np.nan,
    )
)


axes[
    0
].plot(
    x_left_cap,
    (
        capture_intercept
        +
        capture_below_beta
        *
        x_left_cap
    ),
    linewidth=2.5,
)


axes[
    0
].plot(
    x_right_cap,
    (
        capture_intercept
        +
        capture_above_beta
        *
        x_right_cap
    ),
    linewidth=2.5,
)


axes[
    0
].axvline(
    0,
    linestyle="--",
    linewidth=1.5,
)


axes[
    0
].set_xlabel(
    "D* = physical disparity − PSE (deg)"
)


axes[
    0
].set_ylabel(
    "Mean visual capture"
)


axes[
    0
].set_title(
    "Visual capture around "
    "the personal causal boundary"
)


# ---------------------------------------------------------------------
# Dynamicity
# ---------------------------------------------------------------------
axes[
    1
].scatter(
    bp_plot[
        "Dstar"
    ],
    bp_plot[
        "P_dynamic"
    ],
    alpha=.30,
    s=28,
)


x_left_dyn = np.linspace(
    bp[
        "Dstar"
    ].min(),
    0,
    100,
)


x_right_dyn = np.linspace(
    0,
    bp[
        "Dstar"
    ].max(),
    100,
)


dynamic_intercept = (
    dynamic_boundary_hinge_gee
    .params
    .get(
        "Intercept",
        np.nan,
    )
)


eta_left = (
    dynamic_intercept
    +
    dynamic_below_beta
    *
    x_left_dyn
)


eta_right = (
    dynamic_intercept
    +
    dynamic_above_beta
    *
    x_right_dyn
)


p_left = (
    1
    /
    (
        1
        +
        np.exp(
            -eta_left
        )
    )
)


p_right = (
    1
    /
    (
        1
        +
        np.exp(
            -eta_right
        )
    )
)


axes[
    1
].plot(
    x_left_dyn,
    p_left,
    linewidth=2.5,
)


axes[
    1
].plot(
    x_right_dyn,
    p_right,
    linewidth=2.5,
)


axes[
    1
].axvline(
    0,
    linestyle="--",
    linewidth=1.5,
)


axes[
    1
].set_xlabel(
    "D* = physical disparity − PSE (deg)"
)


axes[
    1
].set_ylabel(
    "Probability dynamically updating"
)


axes[
    1
].set_ylim(
    0,
    1,
)


axes[
    1
].set_title(
    "Dynamicity around "
    "the personal causal boundary"
)


fig.suptitle(
    "Refined Aim 3: Is D*=0 "
    "a special computational point?",
    fontsize=15,
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim3D_boundary_knot_models.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()


# =====================================================================
# 3D FIGURE 2 — REFINED HELD-OUT COMPARISON
# =====================================================================
capture_perf_idx = (
    capture_refined_perf
    .set_index(
        "model"
    )
)


dynamic_perf_idx = (
    dynamic_refined_perf
    .set_index(
        "model"
    )
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        11.5,
        4.8,
    ),
)


axes[
    0
].bar(
    [
        "Physical D\n+ PSE",
        "D* boundary\nmodel",
    ],
    [
        capture_perf_idx.loc[
            "physical_plus_PSE",
            "RMSE",
        ],

        capture_perf_idx.loc[
            "Dstar_boundary_model",
            "RMSE",
        ],
    ],
)


axes[
    0
].set_ylabel(
    "Held-out RMSE"
)


axes[
    0
].set_title(
    "Visual capture\n"
    "(lower is better)"
)


axes[
    1
].bar(
    [
        "Physical D\n+ PSE",
        "D* boundary\nmodel",
    ],
    [
        dynamic_perf_idx.loc[
            "physical_plus_PSE",
            "log_loss",
        ],

        dynamic_perf_idx.loc[
            "Dstar_boundary_model",
            "log_loss",
        ],
    ],
)


axes[
    1
].set_ylabel(
    "Held-out log loss"
)


axes[
    1
].set_title(
    "Breakpoint dynamicity\n"
    "(lower is better)"
)


fig.suptitle(
    "Does a D*=0 boundary model outperform "
    "separate physical D + PSE information?",
    fontsize=14,
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim3D_LOPO_physical_plus_PSE_vs_boundary.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()


# =====================================================================
# PRINT
# =====================================================================
print(
    "\nAIM 3 COMPLETE"
)


print(
    "\nOriginal capture D* GEE:\n"
)

print(
    capture_gee_tab.to_string(
        index=False
    )
)


print(
    "\nOriginal dynamicity GEE:\n"
)

print(
    dynamic_gee_tab.to_string(
        index=False
    )
)


print(
    "\nOriginal held-out model comparison:\n"
)

print(
    perf.to_string(
        index=False
    )
)


print(
    "\nREFINED AIM 3D — capture boundary-knot test:\n"
)

print(
    capture_boundary_summary.to_string(
        index=False
    )
)


print(
    "\nREFINED AIM 3D — dynamicity boundary-knot test:\n"
)

print(
    dynamic_boundary_summary.to_string(
        index=False
    )
)


print(
    "\nREFINED AIM 3D — held-out "
    "physical + PSE vs boundary model:\n"
)

print(
    refined_perf.to_string(
        index=False
    )
)


print(
    "\nREFINED AIM 3D — participant-wise paired CV tests "
    "(positive improvement = D* boundary model better):\n"
)

print(
    refined_cv_tests.to_string(
        index=False
    )
)


print(
    f"\nSaved outputs to:\n{OUT}"
)


AIM 3 COMPLETE

Original capture D* GEE:

                        term  estimate       se      stat            p  ci95_low  ci95_high
                   Intercept  0.805854 0.075144 10.724087 7.845806e-27  0.658574   0.953134
bs(Dstar, df=4, degree=3)[0]  0.043197 0.103687  0.416609 6.769642e-01 -0.160026   0.246421
bs(Dstar, df=4, degree=3)[1] -0.111535 0.081612 -1.366650 1.717350e-01 -0.271491   0.048421
bs(Dstar, df=4, degree=3)[2] -0.295890 0.089075 -3.321802 8.943807e-04 -0.470474  -0.121306
bs(Dstar, df=4, degree=3)[3] -0.245810 0.082305 -2.986571 2.821250e-03 -0.407125  -0.084495

Original dynamicity GEE:

                      term  estimate       se      stat            p  ci95_low  ci95_high
                 Intercept  1.742793 0.121357 14.360851 9.110195e-47  1.504938   1.980649
               proximity_c  0.060851 0.014198  4.285900 1.820009e-05  0.033023   0.088678
            above_boundary -1.067867 0.133737 -7.984843 1.407010e-15 -1.329986  -0.805748
proximity_c:above_

In [56]:
# %% AIM 3 — CORRELATION-STYLE FIGURES WITH 95% CI
#
# Paste AFTER the full Aim 3 analysis cell.
#
# Aim 3 uses repeated participant × condition observations.
# Therefore:
#
#   r / p      = descriptive Pearson correlation
#   beta / p   = OLS coefficient with SE clustered by participant
#
# The latter is the more appropriate inferential p-value.
#
# Uses:
#   cap
#   bp
#   FIG_OUT

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import pearsonr

CORR_FIG_OUT = (
    Path(FIG_OUT)
    / "correlation_figures_95CI"
)

CORR_FIG_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

POINT_COLOR = "#1f77b4"
LINE_COLOR = "#1f77b4"
CI_COLOR = "#ff7f0e"


def format_p(p):
    if not np.isfinite(p):
        return "NA"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"


def clustered_corr_plot(
    ax,
    df,
    x,
    y,
    subject_col,
    xlabel,
    ylabel,
    title,
):
    """
    For repeated participant-condition observations:

      - Pearson r / p are descriptive.
      - beta / p use participant-clustered OLS.
      - CI uses clustered covariance matrix.
    """

    d = (
        df[
            [
                subject_col,
                x,
                y,
            ]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .copy()
    )

    if len(d) < 6:

        ax.text(
            .5,
            .5,
            "Insufficient data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

        ax.set_title(
            title
        )

        return None

    # --------------------------------------------------------------
    # Descriptive Pearson correlation
    # --------------------------------------------------------------
    rr = pearsonr(
        d[x],
        d[y],
    )

    r = float(
        rr.statistic
    )

    p_r = float(
        rr.pvalue
    )

    # --------------------------------------------------------------
    # Participant-clustered OLS
    # --------------------------------------------------------------
    X = sm.add_constant(
        d[[x]],
        has_constant="add",
    )

    fit = sm.OLS(
        d[y],
        X,
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[subject_col],
        },
    )

    beta = float(
        fit.params[x]
    )

    p_beta = float(
        fit.pvalues[x]
    )

    # --------------------------------------------------------------
    # Mean fitted line + CI using clustered covariance
    # --------------------------------------------------------------
    xx = np.linspace(
        d[x].min(),
        d[x].max(),
        250,
    )

    XX = sm.add_constant(
        pd.DataFrame(
            {
                x: xx,
            }
        ),
        has_constant="add",
    )

    pred = (
        fit.get_prediction(
            XX
        )
        .summary_frame(
            alpha=.05
        )
    )

    # --------------------------------------------------------------
    # Plot
    # --------------------------------------------------------------
    ax.scatter(
        d[x],
        d[y],
        s=55,
        alpha=.62,
        color=POINT_COLOR,
        edgecolor=POINT_COLOR,
    )

    ax.fill_between(
        xx,
        pred["mean_ci_lower"].to_numpy(),
        pred["mean_ci_upper"].to_numpy(),
        color=CI_COLOR,
        alpha=.22,
        linewidth=0,
        label="95% CI",
    )

    ax.plot(
        xx,
        pred["mean"].to_numpy(),
        color=LINE_COLOR,
        linewidth=2.8,
        label="OLS fit",
    )

    ax.set_xlabel(
        xlabel,
        fontsize=12,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=12,
    )

    ax.set_title(
        title,
        fontsize=15,
        pad=10,
    )

    n_subj = d[
        subject_col
    ].nunique()

    stats_text = (
        f"n = {n_subj} participants\n"
        f"r = {r:.3f}, p {format_p(p_r)}\n"
        f"β = {beta:.4f}, "
        f"cluster-p {format_p(p_beta)}"
    )

    ax.text(
        .04,
        .96,
        stats_text,
        transform=ax.transAxes,
        va="top",
        fontsize=11.2,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="black",
            alpha=.88,
        ),
    )

    ax.legend(
        frameon=False,
        loc="upper right",
    )

    return fit


# =====================================================================
# 3A — PARTICIPANT-RELATIVE DISPARITY -> VISUAL CAPTURE
#
# Average to participant × physical-offset level first.
# This prevents participants with more surviving trials from contributing
# disproportionately many plotted points.
# =====================================================================
cap_plot = (
    cap.groupby(
        [
            "subject",
            "offset_deg",
        ],
        as_index=False,
    )
    .agg(
        Dstar=(
            "Dstar",
            "first",
        ),
        boundary_proximity=(
            "boundary_proximity",
            "first",
        ),
        mean_capture=(
            "VisualCapture_audioLag",
            "mean",
        ),
    )
)

fig, ax = plt.subplots(
    figsize=(7.5, 6.2)
)

clustered_corr_plot(
    ax=ax,
    df=cap_plot,
    x="Dstar",
    y="mean_capture",
    subject_col="subject",
    xlabel=(
        "Participant-relative disparity\n"
        "D* = physical disparity − PSE (deg)"
    ),
    ylabel="Mean lag-corrected visual capture",
    title=(
        "Visual capture relative to each observer's "
        "causal boundary"
    ),
)

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1.2,
    alpha=.6,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim3A_Dstar_predicting_visual_capture_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 3B — DISTANCE FROM PERSONAL BOUNDARY -> DYNAMICITY
#
# Participant × offset mean probability of being dynamically updating.
# =====================================================================
bp_plot = (
    bp.groupby(
        [
            "subject",
            "offset_deg",
        ],
        as_index=False,
    )
    .agg(
        boundary_proximity=(
            "boundary_proximity",
            "first",
        ),
        Dstar=(
            "Dstar",
            "first",
        ),
        P_dynamic=(
            "dynamically_updating",
            "mean",
        ),
    )
)

fig, ax = plt.subplots(
    figsize=(7.5, 6.2)
)

clustered_corr_plot(
    ax=ax,
    df=bp_plot,
    x="boundary_proximity",
    y="P_dynamic",
    subject_col="subject",
    xlabel=(
        "Distance from participant-specific PSE\n"
        "|D − PSE| (deg)"
    ),
    ylabel=(
        "Proportion of trials dynamically updating"
    ),
    title=(
        "Spatial dynamicity as a function of "
        "causal-boundary proximity"
    ),
)

ax.set_ylim(
    -0.05,
    1.05,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim3B_boundary_proximity_predicting_dynamicity_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# COMBINED AIM 3 FIGURE
# =====================================================================
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14.3, 5.8),
)

clustered_corr_plot(
    ax=axes[0],
    df=cap_plot,
    x="Dstar",
    y="mean_capture",
    subject_col="subject",
    xlabel="D* = physical disparity − PSE (deg)",
    ylabel="Mean visual capture",
    title="Perceptual weighting",
)

axes[0].axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=.6,
)

clustered_corr_plot(
    ax=axes[1],
    df=bp_plot,
    x="boundary_proximity",
    y="P_dynamic",
    subject_col="subject",
    xlabel="|D − PSE| (deg)",
    ylabel="Proportion dynamically updating",
    title="Spatial dynamicity",
)

axes[1].set_ylim(
    -0.05,
    1.05,
)

fig.suptitle(
    "Are spatial dynamics organized around each observer's causal boundary?",
    fontsize=18,
    y=1.01,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim3_boundary_relative_summary_correlations_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


print(
    "\nUpdated Aim 3 correlation figures saved to:"
)

print(
    CORR_FIG_OUT
)


Updated Aim 3 correlation figures saved to:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_3_boundary_relative_dynamics\figures\correlation_figures_95CI


In [58]:
# %% AIM 4 — DOES STRONG INTEGRATION MEAN PERSISTENCE OR FLEXIBILITY?
#
# Requested subset only:
#   4C. Decompose SL hysteresis into:
#         Diverge
#         Converge
#         matched-disparity hysteresis gap
#
#   4D. Test whether PSE predicts:
#         breakpoint dynamicity
#         stabilization timing
#
# NO SL UNBINDING / REBINDING ANALYSES ARE INCLUDED.
#
# IMPORTANT REVISION:
#   The repeated-measures linear models now use the same singularity-aware
#   strategy as revised Aim 2:
#
#       richest justified MixedLM
#           -> simpler MixedLM
#           -> Gaussian GEE if random effects collapse / inference is invalid
#           -> participant-clustered OLS only as final fallback
#
# This resolves the prior Group Var = 0 / NaN-SE problem.
#
# Outputs:
#   ...\Individual_differences_analyses\Aim_4_temporal_phenotype

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    pearsonr,
    spearmanr,
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.tools.sm_exceptions import ConvergenceWarning


# ---------------------------------------------------------------------
# WARNINGS
# ---------------------------------------------------------------------
warnings.filterwarnings(
    "default",
    category=ConvergenceWarning,
)


# ---------------------------------------------------------------------
# PATHS / SETTINGS
# ---------------------------------------------------------------------
ROOT = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1"
)

SL_DATA = ROOT / "individual_data_SL"

SPATIAL = ROOT / "spatial_analyses"


OUT = (
    ROOT
    /
    "Individual_differences_analyses"
    /
    "Aim_4_temporal_phenotype"
)

CSV_OUT = OUT / "csv_outputs"

FIG_OUT = OUT / "figures"

MODEL_OUT = OUT / "model_summaries"

DIAG_OUT = OUT / "model_diagnostics"


for d in [
    CSV_OUT,
    FIG_OUT,
    MODEL_OUT,
    DIAG_OUT,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )


FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]


SINGULAR_EIG_TOL = 1e-7


# ---------------------------------------------------------------------
# FILE CHECK
# ---------------------------------------------------------------------
def require_file(
    path,
    label,
):

    path = Path(
        path
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nMissing {label}:\n"
            f"{path}\n\n"
            "Run the corresponding section of "
            "Continuous_SLv3_rebuilt first."
        )

    return path


PSE_CSV = require_file(
    SL_DATA
    /
    "spatial_pse_capture_hysteresis_recalibration_predictions"
    /
    "csv_outputs"
    /
    "subject_level_pse_psychometric_slope.csv",
    "Section 12 subject PSE table",
)


HYST_BIN_CSV = require_file(
    SPATIAL
    /
    "07_spatial_hysteresis"
    /
    "subject_bin_direction_bias_summary.csv",
    "Section 14 subject × disparity-bin × direction hysteresis table",
)


BREAKPOINT_CSV = require_file(
    SPATIAL
    /
    "06_stabilization_breakpoint"
    /
    "csv_outputs"
    /
    "anchor_trial_level_breakpoint_DYNAMIC_metrics.csv",
    "Section 11 breakpoint trial table",
)


# =====================================================================
# HELPERS
# =====================================================================
def clean_subject(
    s,
):

    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )


def safe_corr(
    x,
    y,
):

    d = (
        pd.DataFrame(
            {
                "x":
                    x,

                "y":
                    y,
            }
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )


    if (
        len(
            d
        )
        < 4
        or
        d[
            "x"
        ].std(
            ddof=1
        )
        == 0
        or
        d[
            "y"
        ].std(
            ddof=1
        )
        == 0
    ):

        return {
            "n":
                len(
                    d
                ),

            "pearson_r":
                np.nan,

            "pearson_p":
                np.nan,

            "spearman_rho":
                np.nan,

            "spearman_p":
                np.nan,
        }


    pr = pearsonr(
        d[
            "x"
        ],
        d[
            "y"
        ],
    )


    sr = spearmanr(
        d[
            "x"
        ],
        d[
            "y"
        ],
    )


    return {
        "n":
            len(
                d
            ),

        "pearson_r":
            float(
                pr.statistic
            ),

        "pearson_p":
            float(
                pr.pvalue
            ),

        "spearman_rho":
            float(
                sr.statistic
            ),

        "spearman_p":
            float(
                sr.pvalue
            ),
    }


def add_zscore(
    df,
    source_col,
    new_col,
):

    x = pd.to_numeric(
        df[
            source_col
        ],
        errors="coerce",
    )

    sd = x.std(
        ddof=0
    )

    if (
        not np.isfinite(
            sd
        )
        or
        sd <= 0
    ):

        df[
            new_col
        ] = np.nan

    else:

        df[
            new_col
        ] = (
            x
            -
            x.mean()
        ) / sd

    return df


# =====================================================================
# MIXED MODEL DIAGNOSTICS
# =====================================================================
def _mixedlm_diagnostics(
    fit,
):
    """
    Reject boundary/singular MixedLM fits before using their inference.

    A MixedLM is retained only if:

      1. optimization converged;
      2. all fixed-effect estimates are finite;
      3. all fixed-effect SEs are finite;
      4. all fixed-effect p-values are finite;
      5. the random-effects covariance matrix is nonsingular.
    """

    fe_names = list(
        fit.fe_params.index
    )


    bse_fe = fit.bse.reindex(
        fe_names
    )


    p_fe = fit.pvalues.reindex(
        fe_names
    )


    converged = bool(
        getattr(
            fit,
            "converged",
            False,
        )
    )


    finite_fixed = bool(
        np.isfinite(
            fit.fe_params.to_numpy(
                dtype=float
            )
        ).all()
        and
        np.isfinite(
            bse_fe.to_numpy(
                dtype=float
            )
        ).all()
        and
        np.isfinite(
            p_fe.to_numpy(
                dtype=float
            )
        ).all()
    )


    try:

        cov_re = np.asarray(
            fit.cov_re,
            dtype=float,
        )


        eigvals = np.linalg.eigvalsh(
            cov_re
        )


        min_eig = (
            float(
                np.nanmin(
                    eigvals
                )
            )
            if eigvals.size
            else np.nan
        )


        max_eig = (
            float(
                np.nanmax(
                    eigvals
                )
            )
            if eigvals.size
            else np.nan
        )


        rank = int(
            np.linalg.matrix_rank(
                cov_re,
                tol=SINGULAR_EIG_TOL,
            )
        )


        expected_rank = int(
            cov_re.shape[
                0
            ]
        )


        singular = (
            not np.isfinite(
                eigvals
            ).all()
            or
            rank < expected_rank
            or
            max_eig <= SINGULAR_EIG_TOL
            or
            min_eig < SINGULAR_EIG_TOL
        )


    except Exception:

        min_eig = np.nan
        max_eig = np.nan
        rank = np.nan
        expected_rank = np.nan

        singular = True


    usable = (
        converged
        and
        finite_fixed
        and
        not singular
    )


    return {
        "converged":
            converged,

        "finite_fixed_effect_inference":
            finite_fixed,

        "singular_random_effects":
            bool(
                singular
            ),

        "random_effect_min_eigenvalue":
            min_eig,

        "random_effect_max_eigenvalue":
            max_eig,

        "random_effect_rank":
            rank,

        "random_effect_expected_rank":
            expected_rank,

        "usable_mixedlm":
            bool(
                usable
            ),
    }


# =====================================================================
# ROBUST REPEATED-MEASURES FITTING
# =====================================================================
def fit_repeated_linear(
    formula,
    data,
    group="subject",
    re_formula_candidates=None,
    model_name="model",
):
    """
    Continuous repeated-measures fitting hierarchy:

      1. MixedLM with richest justified random-effects structure.
      2. Simpler MixedLM structures if the richer model is unsupported.
      3. Gaussian GEE if all valid MixedLM structures are singular/invalid.
      4. Cluster-robust OLS only as a final emergency fallback.

    This prevents Group Var = 0 / NaN-SE MixedLM solutions from being
    interpreted as legitimate inferential models.
    """

    d = (
        data.copy()
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    if group not in d.columns:

        raise ValueError(
            f"Grouping column '{group}' is missing."
        )


    n_groups = d[
        group
    ].nunique()


    if n_groups < 8:

        raise ValueError(
            f"Only {n_groups} participant clusters "
            f"are available for {model_name}."
        )


    if re_formula_candidates is None:

        re_formula_candidates = [
            "1",
        ]


    diagnostics_rows = []


    # -----------------------------------------------------------------
    # FIRST: ATTEMPT MIXED MODELS
    # -----------------------------------------------------------------
    for re_formula in re_formula_candidates:

        for method in [
            "lbfgs",
            "powell",
            "cg",
        ]:

            caught_messages = []


            try:

                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always"
                    )


                    model = smf.mixedlm(
                        formula,
                        data=d,
                        groups=d[
                            group
                        ],
                        re_formula=re_formula,
                    )


                    fit = model.fit(
                        reml=False,
                        method=method,
                        maxiter=5000,
                        disp=False,
                    )


                    caught_messages = [
                        str(
                            w.message
                        )
                        for w in caught
                    ]


                diag = _mixedlm_diagnostics(
                    fit
                )


                diag.update(
                    {
                        "model_name":
                            model_name,

                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "n_rows":
                            len(
                                d
                            ),

                        "n_groups":
                            n_groups,

                        "warnings":
                            " | ".join(
                                caught_messages
                            ),
                    }
                )


                diagnostics_rows.append(
                    diag
                )


                if diag[
                    "usable_mixedlm"
                ]:

                    diagnostics = pd.DataFrame(
                        diagnostics_rows
                    )


                    diagnostics.to_csv(
                        DIAG_OUT
                        /
                        f"{model_name}_fit_diagnostics.csv",
                        index=False,
                    )


                    return (
                        fit,
                        (
                            f"MixedLM["
                            f"{re_formula}; "
                            f"{method}]"
                        ),
                        diagnostics,
                    )


            except Exception as exc:

                diagnostics_rows.append(
                    {
                        "model_name":
                            model_name,

                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "n_rows":
                            len(
                                d
                            ),

                        "n_groups":
                            n_groups,

                        "converged":
                            False,

                        "finite_fixed_effect_inference":
                            False,

                        "singular_random_effects":
                            True,

                        "usable_mixedlm":
                            False,

                        "warnings":
                            " | ".join(
                                caught_messages
                            ),

                        "error":
                            repr(
                                exc
                            ),
                    }
                )


    # -----------------------------------------------------------------
    # SECOND: GEE
    # -----------------------------------------------------------------
    try:

        gee = smf.gee(
            formula,
            groups=group,
            data=d,
            family=sm.families.Gaussian(),
            cov_struct=Exchangeable(),
        ).fit()


        diagnostics_rows.append(
            {
                "model_name":
                    model_name,

                "attempt":
                    "GEE",

                "re_formula":
                    "NA",

                "optimizer":
                    "NA",

                "n_rows":
                    len(
                        d
                    ),

                "n_groups":
                    n_groups,

                "converged":
                    bool(
                        getattr(
                            gee,
                            "converged",
                            True,
                        )
                    ),

                "finite_fixed_effect_inference":
                    bool(
                        np.isfinite(
                            gee.params
                        ).all()
                        and
                        np.isfinite(
                            gee.bse
                        ).all()
                        and
                        np.isfinite(
                            gee.pvalues
                        ).all()
                    ),

                "singular_random_effects":
                    False,

                "usable_mixedlm":
                    False,

                "warnings":
                    (
                        "LME unsupported/singular; "
                        "Gaussian GEE used."
                    ),
            }
        )


        diagnostics = pd.DataFrame(
            diagnostics_rows
        )


        diagnostics.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            gee,
            "GEE_Gaussian_Exchangeable",
            diagnostics,
        )


    except Exception as gee_exc:

        # -------------------------------------------------------------
        # FINAL SAFETY FALLBACK
        # -------------------------------------------------------------
        fit = smf.ols(
            formula,
            d,
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups":
                    d[
                        group
                    ],
            },
        )


        diagnostics_rows.append(
            {
                "model_name":
                    model_name,

                "attempt":
                    "OLS_cluster",

                "re_formula":
                    "NA",

                "optimizer":
                    "NA",

                "n_rows":
                    len(
                        d
                    ),

                "n_groups":
                    n_groups,

                "converged":
                    True,

                "finite_fixed_effect_inference":
                    bool(
                        np.isfinite(
                            fit.params
                        ).all()
                        and
                        np.isfinite(
                            fit.bse
                        ).all()
                        and
                        np.isfinite(
                            fit.pvalues
                        ).all()
                    ),

                "singular_random_effects":
                    False,

                "usable_mixedlm":
                    False,

                "warnings":
                    (
                        f"GEE failed: "
                        f"{repr(gee_exc)}"
                    ),
            }
        )


        diagnostics = pd.DataFrame(
            diagnostics_rows
        )


        diagnostics.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            fit,
            "OLS_cluster_fallback",
            diagnostics,
        )


# =====================================================================
# MODEL OUTPUT HELPERS
# =====================================================================
def fixed_effect_terms(
    fit,
):

    if hasattr(
        fit,
        "fe_params",
    ):

        return list(
            fit.fe_params.index
        )

    return list(
        fit.params.index
    )


def save_model(
    fit,
    name,
    fit_type="",
    diagnostics=None,
):

    with open(
        MODEL_OUT
        /
        f"{name}.txt",
        "w",
        encoding="utf-8",
    ) as f:

        if fit_type:

            f.write(
                f"Fit type: {fit_type}\n\n"
            )


        if diagnostics is not None:

            f.write(
                "FIT DIAGNOSTICS\n"
            )

            f.write(
                diagnostics.to_string(
                    index=False
                )
            )

            f.write(
                "\n\nMODEL SUMMARY\n"
            )


        f.write(
            str(
                fit.summary()
            )
        )


    terms = fixed_effect_terms(
        fit
    )


    ci = fit.conf_int().reindex(
        terms
    )


    tab = pd.DataFrame(
        {
            "term":
                terms,

            "estimate":
                fit.params.reindex(
                    terms
                ).values,

            "se":
                fit.bse.reindex(
                    terms
                ).values,

            "stat":
                fit.tvalues.reindex(
                    terms
                ).values,

            "p":
                fit.pvalues.reindex(
                    terms
                ).values,

            "ci95_low":
                ci.iloc[
                    :,
                    0
                ].values,

            "ci95_high":
                ci.iloc[
                    :,
                    1
                ].values,

            "fit_type":
                fit_type,
        }
    )


    tab.to_csv(
        CSV_OUT
        /
        f"{name}_coefficients.csv",
        index=False,
    )


    return tab


# =====================================================================
# PARTICIPANT-LEVEL PLOT HELPER
# =====================================================================
def scatter_with_fit(
    df,
    x,
    y,
    xlabel,
    ylabel,
    title,
    path,
):

    d = (
        df[
            [
                x,
                y,
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .copy()
    )


    if len(
        d
    ) < 4:

        return


    fit = smf.ols(
        f"{y} ~ {x}",
        d,
    ).fit(
        cov_type="HC3"
    )


    xx = np.linspace(
        d[
            x
        ].min(),
        d[
            x
        ].max(),
        120,
    )


    pred = (
        fit.get_prediction(
            pd.DataFrame(
                {
                    x:
                        xx,
                }
            )
        )
        .summary_frame(
            alpha=.05
        )
    )


    c = safe_corr(
        d[
            x
        ],
        d[
            y
        ],
    )


    fig, ax = plt.subplots(
        figsize=(
            6.5,
            5.1,
        )
    )


    ax.scatter(
        d[
            x
        ],
        d[
            y
        ],
        alpha=.72,
    )


    ax.plot(
        xx,
        pred[
            "mean"
        ],
        linewidth=2,
    )


    ax.fill_between(
        xx,
        pred[
            "mean_ci_lower"
        ].to_numpy(),
        pred[
            "mean_ci_upper"
        ].to_numpy(),
        alpha=.18,
    )


    ax.set_xlabel(
        xlabel
    )


    ax.set_ylabel(
        ylabel
    )


    ax.set_title(
        title
    )


    ax.text(
        .04,
        .96,
        (
            f"n={c['n']}\n"
            f"r={c['pearson_r']:.3f}, "
            f"p={c['pearson_p']:.4g}"
        ),
        transform=ax.transAxes,
        va="top",
    )


    fig.tight_layout()


    fig.savefig(
        path,
        dpi=220,
        bbox_inches="tight",
    )


    plt.show()


# ---------------------------------------------------------------------
# LOAD PSE
# ---------------------------------------------------------------------
pse = pd.read_csv(
    PSE_CSV
)


pse[
    "subject"
] = clean_subject(
    pse[
        "subject"
    ]
)


pse[
    "PSE_empirical_deg"
] = pd.to_numeric(
    pse[
        "PSE_empirical_deg"
    ],
    errors="coerce",
)


pse = (
    pse[
        [
            "subject",
            "PSE_empirical_deg",
        ]
    ]
    .drop_duplicates(
        "subject"
    )
    .dropna()
    .copy()
)


pse[
    "PSE_c"
] = (
    pse[
        "PSE_empirical_deg"
    ]
    -
    pse[
        "PSE_empirical_deg"
    ].mean()
)


# =====================================================================
# 4C. DECOMPOSE SPATIAL-LOCALIZATION HYSTERESIS
#
# Existing sign convention:
#
#   H_SL = Converge capture - Diverge capture
#
# Negative H_SL = stronger visual capture during divergence.
# =====================================================================
h = pd.read_csv(
    HYST_BIN_CSV
)


h[
    "subject"
] = clean_subject(
    h[
        "subject"
    ]
)


h[
    "SweepDirection"
] = (
    h[
        "SweepDirection"
    ]
    .astype(str)
    .str.title()
)


h[
    "disparity_bin_center"
] = pd.to_numeric(
    h[
        "disparity_bin_center"
    ],
    errors="coerce",
)


h[
    "mean_NormalizedVisualBias"
] = pd.to_numeric(
    h[
        "mean_NormalizedVisualBias"
    ],
    errors="coerce",
)


# ---------------------------------------------------------------------
# Preserve manuscript's 5–35° matched-disparity range
# ---------------------------------------------------------------------
h = h[
    h[
        "disparity_bin_center"
    ].between(
        5,
        35,
    )
    &
    h[
        "SweepDirection"
    ].isin(
        [
            "Diverge",
            "Converge",
        ]
    )
].copy()


h = h.merge(
    pse,
    on="subject",
    how="inner",
)


# ---------------------------------------------------------------------
# Participant-level decomposition
# ---------------------------------------------------------------------
sub_dir = (
    h.groupby(
        [
            "subject",
            "SweepDirection",
        ],
        as_index=False,
    )
    .agg(
        mean_capture=(
            "mean_NormalizedVisualBias",
            "mean",
        ),

        PSE_empirical_deg=(
            "PSE_empirical_deg",
            "first",
        ),
    )
)


wide = (
    sub_dir
    .pivot(
        index="subject",
        columns="SweepDirection",
        values="mean_capture",
    )
    .reset_index()
)


wide = wide.merge(
    pse[
        [
            "subject",
            "PSE_empirical_deg",
            "PSE_c",
        ]
    ],
    on="subject",
    how="left",
)


if "Diverge" not in wide.columns:

    wide[
        "Diverge"
    ] = np.nan


if "Converge" not in wide.columns:

    wide[
        "Converge"
    ] = np.nan


wide[
    "H_SL_ConvergeMinusDiverge"
] = (
    wide[
        "Converge"
    ]
    -
    wide[
        "Diverge"
    ]
)


wide.to_csv(
    CSV_OUT
    /
    "aim4C_subject_hysteresis_decomposition.csv",
    index=False,
)


# ---------------------------------------------------------------------
# Participant-level regressions
# ---------------------------------------------------------------------
rows = []


for outcome, label in [
    (
        "Diverge",
        "Mean Diverge visual capture",
    ),
    (
        "Converge",
        "Mean Converge visual capture",
    ),
    (
        "H_SL_ConvergeMinusDiverge",
        "SL hysteresis: Converge − Diverge",
    ),
]:

    d = wide[
        [
            "PSE_empirical_deg",
            outcome,
        ]
    ].dropna()


    c = safe_corr(
        d[
            "PSE_empirical_deg"
        ],
        d[
            outcome
        ],
    )


    fit = smf.ols(
        f"{outcome} ~ PSE_empirical_deg",
        d,
    ).fit(
        cov_type="HC3"
    )


    rows.append(
        {
            "outcome":
                outcome,

            "label":
                label,

            **c,

            "beta_PSE":
                fit.params.get(
                    "PSE_empirical_deg",
                    np.nan,
                ),

            "se_PSE":
                fit.bse.get(
                    "PSE_empirical_deg",
                    np.nan,
                ),

            "p_PSE":
                fit.pvalues.get(
                    "PSE_empirical_deg",
                    np.nan,
                ),

            "R2":
                fit.rsquared,
        }
    )


pd.DataFrame(
    rows
).to_csv(
    CSV_OUT
    /
    "aim4C_pse_hysteresis_component_regressions.csv",
    index=False,
)


# =====================================================================
# REVISED 4C REPEATED-MEASURES HYSTERESIS MODEL
# =====================================================================
#
# Critical question:
#
# Does PSE alter the Diverge-vs-Converge difference while controlling
# matched physical disparity?
#
# Fixed effect of interest:
#
#   C(SweepDirection) × PSE_c
#
# IMPORTANT:
#
# We do NOT attempt a random slope for PSE because PSE is constant within
# participant and is therefore a between-participant predictor.
#
# Instead, the richest justified model permits participant-specific
# slopes for:
#
#   - sweep direction
#   - physical disparity
#
# If that covariance structure is unsupported, the code simplifies it.
# If all MixedLM structures are singular, Gaussian GEE is used.
# =====================================================================


h[
    "disparity_c"
] = (
    h[
        "disparity_bin_center"
    ]
    -
    h[
        "disparity_bin_center"
    ].mean()
)


h = add_zscore(
    h,
    "disparity_bin_center",
    "disparity_z",
)


# Effect-coded direction for RANDOM-EFFECT estimation only.
#
# Diverge  = -0.5
# Converge = +0.5
h[
    "direction_c"
] = np.where(
    h[
        "SweepDirection"
    ].eq(
        "Converge"
    ),
    0.5,
    -0.5,
)


hyst_model, hyst_fit_type, hyst_diag = fit_repeated_linear(
    formula=(
        "mean_NormalizedVisualBias ~ "
        "C(SweepDirection) * PSE_c + "
        "disparity_c + "
        "I(disparity_c**2)"
    ),
    data=h,
    group="subject",
    re_formula_candidates=[
        "1 + disparity_z + direction_c",
        "1 + direction_c",
        "1 + disparity_z",
        "1",
    ],
    model_name="aim4C_direction_by_PSE_model",
)


hyst_coef = save_model(
    hyst_model,
    "aim4C_direction_by_PSE_mixed_model",
    hyst_fit_type,
    hyst_diag,
)


# ---------------------------------------------------------------------
# 3-PANEL FIGURE
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        16,
        5,
    ),
)


panels = [
    (
        "Diverge",
        "Diverge visual capture",
    ),

    (
        "Converge",
        "Converge visual capture",
    ),

    (
        "H_SL_ConvergeMinusDiverge",
        "Hysteresis\n(Converge − Diverge)",
    ),
]


for ax, (
    y,
    title,
) in zip(
    axes,
    panels,
):

    d = wide[
        [
            "PSE_empirical_deg",
            y,
        ]
    ].dropna()


    ax.scatter(
        d[
            "PSE_empirical_deg"
        ],
        d[
            y
        ],
        alpha=.72,
    )


    if (
        len(
            d
        )
        >= 4
        and
        d[
            "PSE_empirical_deg"
        ].std()
        > 0
    ):

        fit = smf.ols(
            f"{y} ~ PSE_empirical_deg",
            d,
        ).fit(
            cov_type="HC3"
        )


        xx = np.linspace(
            d[
                "PSE_empirical_deg"
            ].min(),
            d[
                "PSE_empirical_deg"
            ].max(),
            100,
        )


        pred = (
            fit.get_prediction(
                pd.DataFrame(
                    {
                        "PSE_empirical_deg":
                            xx,
                    }
                )
            )
            .summary_frame(
                alpha=.05
            )
        )


        ax.plot(
            xx,
            pred[
                "mean"
            ],
            linewidth=2,
        )


        ax.fill_between(
            xx,
            pred[
                "mean_ci_lower"
            ].to_numpy(),
            pred[
                "mean_ci_upper"
            ].to_numpy(),
            alpha=.15,
        )


    if y == "H_SL_ConvergeMinusDiverge":

        ax.axhline(
            0,
            linestyle="--",
            linewidth=1,
        )


    ax.set_xlabel(
        "SL PSE (deg)"
    )


    ax.set_ylabel(
        title
    )


    ax.set_title(
        title
    )


fig.suptitle(
    "Aim 4C: What component of spatial hysteresis "
    "varies across integrators?",
    fontsize=15,
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim4C_hysteresis_decomposition_by_PSE.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()


# =====================================================================
# 4D. PSE -> BREAKPOINT DYNAMICITY / STABILIZATION TIME
# =====================================================================
bp = pd.read_csv(
    BREAKPOINT_CSV
)


bp[
    "subject"
] = clean_subject(
    bp[
        "subject"
    ]
)


bp[
    "offset_deg"
] = pd.to_numeric(
    bp[
        "offset_deg"
    ],
    errors="coerce",
)


if "metric" in bp.columns:

    bp = bp[
        bp[
            "metric"
        ]
        .astype(str)
        .eq(
            "LagCorrectedVisualCapture"
        )
    ].copy()


bp = bp[
    bp[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


for c in [
    "dynamically_updating",
    "has_stabilizing_breakpoint",
    "stabilization_time_sec",
]:

    if c in bp.columns:

        bp[
            c
        ] = pd.to_numeric(
            bp[
                c
            ],
            errors="coerce",
        )


bp = bp.merge(
    pse,
    on="subject",
    how="inner",
)


bp[
    "offset_c"
] = (
    bp[
        "offset_deg"
    ]
    -
    np.mean(
        FIXED_OFFSETS
    )
)


bp = add_zscore(
    bp,
    "offset_deg",
    "offset_z",
)


# =====================================================================
# PRIMARY:
# Trial-level breakpoint dynamicity
#
# This analysis was already a valid Binomial GEE and is left unchanged.
# =====================================================================
dyn = bp.dropna(
    subset=[
        "dynamically_updating",
        "PSE_c",
        "offset_c",
    ]
).copy()


dynamic_gee = smf.gee(
    (
        "dynamically_updating ~ "
        "PSE_c * offset_c"
    ),
    groups="subject",
    data=dyn,
    family=sm.families.Binomial(),
    cov_struct=Exchangeable(),
).fit()


dynamic_tab = save_model(
    dynamic_gee,
    "aim4D_dynamicity_PSE_by_offset_GEE",
    "GEE_binomial",
)


# ---------------------------------------------------------------------
# Participant-level overall dynamicity
# ---------------------------------------------------------------------
dyn_subj = (
    dyn.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        P_dynamic=(
            "dynamically_updating",
            "mean",
        ),

        PSE_empirical_deg=(
            "PSE_empirical_deg",
            "first",
        ),
    )
)


dyn_subj.to_csv(
    CSV_OUT
    /
    "aim4D_subject_breakpoint_dynamicity.csv",
    index=False,
)


scatter_with_fit(
    dyn_subj,
    "PSE_empirical_deg",
    "P_dynamic",
    "SL PSE (deg)",
    "Proportion of SL trials dynamically updating",
    (
        "Does causal tolerance predict "
        "continuous spatial recalibration?"
    ),
    FIG_OUT
    /
    "aim4D_pse_vs_breakpoint_dynamicity.png",
)


# =====================================================================
# SECONDARY:
# STABILIZATION TIME
#
# Among trials with an ACCEPTED stabilizing breakpoint,
# does PSE predict when stabilization occurs?
#
# REVISED:
# The old potentially singular MixedLM is replaced by the same
# singularity-aware hierarchy used for 4C and corrected Aim 2.
# =====================================================================
stable = bp[
    bp[
        "has_stabilizing_breakpoint"
    ]
    ==
    1
].dropna(
    subset=[
        "stabilization_time_sec",
        "PSE_c",
        "offset_c",
        "offset_z",
    ]
).copy()


if (
    len(
        stable
    )
    >= 20
    and
    stable[
        "subject"
    ].nunique()
    >= 5
):

    time_model, time_fit_type, time_diag = fit_repeated_linear(
        formula=(
            "stabilization_time_sec ~ "
            "PSE_c * offset_c"
        ),
        data=stable,
        group="subject",
        re_formula_candidates=[
            "1 + offset_z",
            "1",
        ],
        model_name="aim4D_stabilization_time_model",
    )


    time_tab = save_model(
        time_model,
        "aim4D_stabilization_time_PSE_by_offset",
        time_fit_type,
        time_diag,
    )


    time_subj = (
        stable.groupby(
            "subject",
            as_index=False,
        )
        .agg(
            mean_stabilization_time_sec=(
                "stabilization_time_sec",
                "mean",
            ),

            PSE_empirical_deg=(
                "PSE_empirical_deg",
                "first",
            ),
        )
    )


    time_subj.to_csv(
        CSV_OUT
        /
        "aim4D_subject_stabilization_time.csv",
        index=False,
    )


    scatter_with_fit(
        time_subj,
        "PSE_empirical_deg",
        "mean_stabilization_time_sec",
        "SL PSE (deg)",
        "Mean stabilization breakpoint (s)",
        (
            "Among stabilizing trials, does PSE "
            "predict stabilization time?"
        ),
        FIG_OUT
        /
        "aim4D_pse_vs_stabilization_time.png",
    )


else:

    time_tab = pd.DataFrame()

    time_fit_type = "not_run"

    print(
        "Not enough accepted breakpoints "
        "for stabilization-time model."
    )


# =====================================================================
# MODEL-PREDICTED DYNAMICITY ACROSS PSE LEVELS
#
# Descriptive visualization only.
# PSE remains continuous in the inferential model.
# =====================================================================
q = (
    pse[
        "PSE_empirical_deg"
    ]
    .quantile(
        [
            .25,
            .50,
            .75,
        ]
    )
    .to_dict()
)


grid = []


for label, pval in [
    (
        "25th percentile",
        q[
            .25
        ],
    ),
    (
        "Median",
        q[
            .50
        ],
    ),
    (
        "75th percentile",
        q[
            .75
        ],
    ),
]:

    for off in FIXED_OFFSETS:

        grid.append(
            {
                "PSE_group":
                    label,

                "PSE_empirical_deg":
                    pval,

                "PSE_c":
                    pval
                    -
                    pse[
                        "PSE_empirical_deg"
                    ].mean(),

                "offset_deg":
                    off,

                "offset_c":
                    off
                    -
                    np.mean(
                        FIXED_OFFSETS
                    ),
            }
        )


grid = pd.DataFrame(
    grid
)


try:

    grid[
        "predicted_P_dynamic"
    ] = dynamic_gee.predict(
        grid
    )


    fig, ax = plt.subplots(
        figsize=(
            7,
            5.3,
        )
    )


    for label, g in grid.groupby(
        "PSE_group",
        sort=False,
    ):

        ax.plot(
            g[
                "offset_deg"
            ],
            g[
                "predicted_P_dynamic"
            ],
            marker="o",
            label=label,
        )


    ax.set_xlabel(
        "A-V offset magnitude (deg)"
    )


    ax.set_ylabel(
        "Predicted probability dynamically updating"
    )


    ax.set_ylim(
        0,
        1,
    )


    ax.set_title(
        "Breakpoint dynamicity across PSE levels"
    )


    ax.legend(
        frameon=False
    )


    fig.tight_layout()


    fig.savefig(
        FIG_OUT
        /
        "aim4D_predicted_dynamicity_by_offset_and_PSE.png",
        dpi=220,
        bbox_inches="tight",
    )


    plt.show()


except Exception as e:

    print(
        "Dynamicity prediction plot skipped:",
        e,
    )


# =====================================================================
# COMPACT PRIMARY TEST SUMMARY
# =====================================================================
summary_rows = []


for name, fit, fit_type, terms in [
    (
        "4C hysteresis repeated-measures",
        hyst_model,
        hyst_fit_type,
        [
            t
            for t in fixed_effect_terms(
                hyst_model
            )
            if "PSE_c" in t
        ],
    ),

    (
        "4D breakpoint dynamicity",
        dynamic_gee,
        "GEE_binomial",
        [
            "PSE_c",
            "PSE_c:offset_c",
        ],
    ),
]:

    for term in terms:

        summary_rows.append(
            {
                "analysis":
                    name,

                "fit_type":
                    fit_type,

                "term":
                    term,

                "estimate":
                    fit.params.get(
                        term,
                        np.nan,
                    ),

                "se":
                    fit.bse.get(
                        term,
                        np.nan,
                    ),

                "stat":
                    fit.tvalues.get(
                        term,
                        np.nan,
                    ),

                "p":
                    fit.pvalues.get(
                        term,
                        np.nan,
                    ),
            }
        )


if (
    "time_tab" in globals()
    and
    len(
        time_tab
    )
):

    for term in [
        "PSE_c",
        "PSE_c:offset_c",
    ]:

        summary_rows.append(
            {
                "analysis":
                    "4D stabilization time",

                "fit_type":
                    time_fit_type,

                "term":
                    term,

                "estimate":
                    time_model.params.get(
                        term,
                        np.nan,
                    ),

                "se":
                    time_model.bse.get(
                        term,
                        np.nan,
                    ),

                "stat":
                    time_model.tvalues.get(
                        term,
                        np.nan,
                    ),

                "p":
                    time_model.pvalues.get(
                        term,
                        np.nan,
                    ),
            }
        )


summary = pd.DataFrame(
    summary_rows
)


summary.to_csv(
    CSV_OUT
    /
    "aim4_primary_tests_summary.csv",
    index=False,
)


# =====================================================================
# PRINT DIAGNOSTIC SUMMARY
# =====================================================================
print(
    "\nAIM 4 COMPLETE"
)


print(
    "\nPrimary tests:"
)


print(
    summary.to_string(
        index=False
    )
)


print(
    "\nChosen repeated-measures estimators:"
)


print(
    f"  4C hysteresis: "
    f"{hyst_fit_type}"
)


print(
    "  4D breakpoint dynamicity: "
    "GEE_binomial"
)


print(
    f"  4D stabilization time: "
    f"{time_fit_type}"
)


print(
    "\nA MixedLM is retained only if it converges, "
    "has finite fixed-effect inference, and has a "
    "nonsingular random-effects covariance matrix. "
    "Otherwise the analysis automatically uses Gaussian GEE."
)


print(
    f"\nSaved outputs to:\n{OUT}"
)


AIM 4 COMPLETE

Primary tests:
                       analysis                                      fit_type                               term  estimate       se      stat        p
4C hysteresis repeated-measures MixedLM[1 + disparity_z + direction_c; lbfgs]                              PSE_c  0.005534 0.004074  1.358349 0.174353
4C hysteresis repeated-measures MixedLM[1 + disparity_z + direction_c; lbfgs] C(SweepDirection)[T.Diverge]:PSE_c -0.001177 0.004865 -0.241994 0.808785
       4D breakpoint dynamicity                                  GEE_binomial                              PSE_c -0.004358 0.008021 -0.543319 0.586910
       4D breakpoint dynamicity                                  GEE_binomial                     PSE_c:offset_c -0.000420 0.000824 -0.509322 0.610526
          4D stabilization time                  MixedLM[1 + offset_z; lbfgs]                              PSE_c -0.010487 0.013606 -0.770810 0.440819
          4D stabilization time                  MixedLM[1 + o

In [50]:
# %% AIM 4 — PUBLICATION/PRESENTATION CORRELATION FIGURES WITH 95% CI
#
# Paste AFTER the full Aim 4 cell.
#
# Uses:
#   wide
#   dyn
#   dyn_subj
#   stable / time_subj if available
#   hyst_model
#   FIG_OUT
#
# Includes ONLY the retained Aim 4C / 4D analyses.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import pearsonr

CORR_FIG_OUT = (
    Path(FIG_OUT)
    / "correlation_figures_95CI"
)

CORR_FIG_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

POINT_COLOR = "#1f77b4"
LINE_COLOR = "#1f77b4"
CI_COLOR = "#ff7f0e"


def format_p(p):
    if not np.isfinite(p):
        return "NA"
    if p < .001:
        return "< .001"
    return f"= {p:.4f}"


def plot_corr_ci(
    ax,
    df,
    x,
    y,
    xlabel,
    ylabel,
    title,
    extra_text=None,
    y_limits=None,
):
    """
    One observation per participant.

    Shows:
      Pearson r/p
      HC3 OLS beta/p
      fitted line
      95% CI of fitted mean
    """

    d = (
        df[
            [x, y]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .copy()
    )

    if len(d) < 4:

        ax.text(
            .5,
            .5,
            "Insufficient data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

        ax.set_title(
            title
        )

        return None

    # Pearson
    rr = pearsonr(
        d[x],
        d[y],
    )

    r = float(
        rr.statistic
    )

    p_r = float(
        rr.pvalue
    )

    # HC3 OLS
    X = sm.add_constant(
        d[[x]],
        has_constant="add",
    )

    fit = sm.OLS(
        d[y],
        X,
    ).fit(
        cov_type="HC3"
    )

    beta = float(
        fit.params[x]
    )

    p_beta = float(
        fit.pvalues[x]
    )

    # Prediction
    xx = np.linspace(
        d[x].min(),
        d[x].max(),
        250,
    )

    XX = sm.add_constant(
        pd.DataFrame(
            {
                x: xx,
            }
        ),
        has_constant="add",
    )

    pred = (
        fit.get_prediction(
            XX
        )
        .summary_frame(
            alpha=.05
        )
    )

    # Plot
    ax.scatter(
        d[x],
        d[y],
        s=62,
        alpha=.72,
        color=POINT_COLOR,
        edgecolor=POINT_COLOR,
    )

    ax.fill_between(
        xx,
        pred["mean_ci_lower"].to_numpy(),
        pred["mean_ci_upper"].to_numpy(),
        color=CI_COLOR,
        alpha=.22,
        linewidth=0,
        label="95% CI",
    )

    ax.plot(
        xx,
        pred["mean"].to_numpy(),
        color=LINE_COLOR,
        linewidth=2.8,
        label="OLS fit",
    )

    ax.set_xlabel(
        xlabel,
        fontsize=12,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=12,
    )

    ax.set_title(
        title,
        fontsize=15,
        pad=10,
    )

    if y_limits is not None:
        ax.set_ylim(
            *y_limits
        )

    stats_text = (
        f"n = {len(d)}\n"
        f"r = {r:.3f}, p {format_p(p_r)}\n"
        f"β = {beta:.4f}, p {format_p(p_beta)}"
    )

    if extra_text:
        stats_text += (
            "\n"
            + extra_text
        )

    ax.text(
        .04,
        .96,
        stats_text,
        transform=ax.transAxes,
        va="top",
        fontsize=11.2,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="black",
            alpha=.88,
        ),
    )

    ax.legend(
        frameon=False,
        loc="upper right",
    )

    return fit


# =====================================================================
# 4C — PSE -> DIVERGE VISUAL CAPTURE
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.3, 6.1)
)

plot_corr_ci(
    ax=ax,
    df=wide,
    x="PSE_empirical_deg",
    y="Diverge",
    xlabel="Subject PSE: P(same)=.5 boundary (deg)",
    ylabel="Mean visual capture during divergence",
    title="PSE predicting Diverge visual capture",
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim4C_PSE_predicting_Diverge_visual_capture_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 4C — PSE -> CONVERGE VISUAL CAPTURE
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.3, 6.1)
)

plot_corr_ci(
    ax=ax,
    df=wide,
    x="PSE_empirical_deg",
    y="Converge",
    xlabel="Subject PSE: P(same)=.5 boundary (deg)",
    ylabel="Mean visual capture during convergence",
    title="PSE predicting Converge visual capture",
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim4C_PSE_predicting_Converge_visual_capture_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 4C — PSE -> SPATIAL HYSTERESIS
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.3, 6.1)
)

plot_corr_ci(
    ax=ax,
    df=wide,
    x="PSE_empirical_deg",
    y="H_SL_ConvergeMinusDiverge",
    xlabel="Subject PSE: P(same)=.5 boundary (deg)",
    ylabel="SL hysteresis (Converge − Diverge)",
    title="PSE predicting spatial-localization hysteresis",
)

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=.6,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim4C_PSE_predicting_SL_hysteresis_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 4C — COMBINED DIVERGE / CONVERGE CORRELATION PANEL
# =====================================================================
fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 5.4),
)

plot_corr_ci(
    ax=axes[0],
    df=wide,
    x="PSE_empirical_deg",
    y="Diverge",
    xlabel="SL PSE (deg)",
    ylabel="Visual capture",
    title="Diverge",
)

plot_corr_ci(
    ax=axes[1],
    df=wide,
    x="PSE_empirical_deg",
    y="Converge",
    xlabel="SL PSE (deg)",
    ylabel="Visual capture",
    title="Converge",
)

plot_corr_ci(
    ax=axes[2],
    df=wide,
    x="PSE_empirical_deg",
    y="H_SL_ConvergeMinusDiverge",
    xlabel="SL PSE (deg)",
    ylabel="Converge − Diverge",
    title="Spatial hysteresis",
)

axes[2].axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=.6,
)

fig.suptitle(
    "Does causal tolerance explain the components of spatial hysteresis?",
    fontsize=18,
    y=1.01,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim4C_hysteresis_decomposition_correlations_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 4D — PSE -> OVERALL BREAKPOINT DYNAMICITY
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.3, 6.1)
)

plot_corr_ci(
    ax=ax,
    df=dyn_subj,
    x="PSE_empirical_deg",
    y="P_dynamic",
    xlabel="Subject PSE: P(same)=.5 boundary (deg)",
    ylabel="Proportion of SL trials dynamically updating",
    title="PSE predicting spatial breakpoint dynamicity",
    y_limits=(-0.05, 1.05),
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim4D_PSE_predicting_overall_dynamicity_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# =====================================================================
# 4D — OFFSET-SPECIFIC PSE -> BREAKPOINT DYNAMICITY
# =====================================================================
dynamicity_by_offset = {}

for off in [10, 20, 30, 40]:

    g = (
        dyn[
            dyn["offset_deg"]
            ==
            off
        ]
        .groupby(
            "subject",
            as_index=False,
        )
        .agg(
            PSE_empirical_deg=(
                "PSE_empirical_deg",
                "first",
            ),
            P_dynamic=(
                "dynamically_updating",
                "mean",
            ),
        )
    )

    dynamicity_by_offset[
        off
    ] = g


fig, axes = plt.subplots(
    2,
    2,
    figsize=(13.5, 11.0),
)

axes = axes.ravel()

for ax, off in zip(
    axes,
    [10, 20, 30, 40],
):

    plot_corr_ci(
        ax=ax,
        df=dynamicity_by_offset[off],
        x="PSE_empirical_deg",
        y="P_dynamic",
        xlabel="Subject PSE (deg)",
        ylabel=f"Dynamicity at {off}°",
        title=f"{off}° offset",
        y_limits=(-0.05, 1.05),
    )

fig.suptitle(
    "PSE predicting continuous spatial updating across fixed disparities",
    fontsize=18,
    y=1.01,
)

fig.tight_layout()

fig.savefig(
    CORR_FIG_OUT
    /
    "aim4D_PSE_dynamicity_all_offsets_95CI.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# Individual offset-specific figures too
for off in [10, 20, 30, 40]:

    fig, ax = plt.subplots(
        figsize=(7.3, 6.1)
    )

    plot_corr_ci(
        ax=ax,
        df=dynamicity_by_offset[off],
        x="PSE_empirical_deg",
        y="P_dynamic",
        xlabel="Subject PSE: P(same)=.5 boundary (deg)",
        ylabel=f"Proportion dynamically updating at {off}°",
        title=f"PSE predicting breakpoint dynamicity at {off}°",
        y_limits=(-0.05, 1.05),
    )

    fig.tight_layout()

    fig.savefig(
        CORR_FIG_OUT
        /
        f"aim4D_PSE_predicting_dynamicity_{off}deg_95CI.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# =====================================================================
# 4D — PSE -> STABILIZATION TIME
#
# Only runs if the original Aim 4 cell had enough accepted breakpoints
# to create time_subj.
# =====================================================================
if (
    "time_subj" in globals()
    and
    isinstance(
        time_subj,
        pd.DataFrame,
    )
    and
    len(time_subj) >= 4
):

    fig, ax = plt.subplots(
        figsize=(7.3, 6.1)
    )

    plot_corr_ci(
        ax=ax,
        df=time_subj,
        x="PSE_empirical_deg",
        y="mean_stabilization_time_sec",
        xlabel="Subject PSE: P(same)=.5 boundary (deg)",
        ylabel="Mean stabilization breakpoint (s)",
        title="PSE predicting spatial stabilization time",
    )

    fig.tight_layout()

    fig.savefig(
        CORR_FIG_OUT
        /
        "aim4D_PSE_predicting_stabilization_time_95CI.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

else:

    print(
        "\nNo time_subj table was available; "
        "stabilization-time correlation figure skipped."
    )


print(
    "\nUpdated Aim 4 correlation figures saved to:"
)

print(
    CORR_FIG_OUT
)


Updated Aim 4 correlation figures saved to:
C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_1_reliability\figures\correlation_figures_95CI


## Individual Diffeneces Gaussian PSE 

In [69]:
# %% AIM 1 — FULL VALIDATION OF EXPLICIT CAUSAL-REPORT PHENOTYPE
# 
# Keeps ORIGINAL Aim-1 output paths, filenames, and Gaussian/cross-fit columns.
# NEW: shifted lapse-adjusted model comparisons; nested/paired CV; model influence;
#      explicit threshold quality and censor flags; parameter recovery; calibration;
#      reliability sensitivity; tagged sources for tracking-noise proxies.

from pathlib import Path
import glob, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit, ndtr
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf

# --------------------------- USER SETTINGS ---------------------------
ROOT = Path(r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1")
SL_DATA, SPATIAL = ROOT/'individual_data_SL', ROOT/'spatial_analyses'
OUT = ROOT/'Individual_differences_analyses'/'Aim_1_gaussian_psychometric_reliability'
CSV_OUT, FIG_OUT = OUT/'csv_outputs', OUT/'figures'
for p in (CSV_OUT, FIG_OUT): p.mkdir(parents=True, exist_ok=True)
SEED = 20261011
OFFSETS = [10., 20., 30., 40.]
CELLS = np.array([-40.,-30.,-20.,-10.,0.,10.,20.,30.,40.])
N_CV_FOLDS = 5
N_RELIABILITY_SPLITS = 50
N_BOOT = 2500
N_SPLITS_PER_BOOT = 10
N_RECOVERY_REPS = 8                 # 8 simulated datasets × each fitted participant
MIN_TRIALS = 12
EPS = 1e-7
MU_LIMIT = (-25.,25.)
SIGMA_LIMIT = (2.,100.)
FLOOR_LIMIT = (0.,.30)
PEAK_LIMIT = (.50001,.999)
QUALITY_MIN_PEAK = .60              # sensitivity only; NOT an exclusion rule
QUALITY_MIN_PEAK_FLOOR_GAP = .35   # sensitivity only
# 'gaussian4' remains the phenotype used by existing Aims 2–4.
MODELS = ['gaussian4','gaussian3','gaussian2',
          'logistic_abs','probit_abs',
          'logistic_shifted4','probit_shifted4',
          'logistic_shifted5','probit_shifted5']

STABLE_BIAS_CSV = SPATIAL/'05_residual_visual_bias'/'trial_level_bias_psame_model_data.csv'
VALID_LAG_CSV = SPATIAL/'02_same_different_tracking_lag'/'csv_outputs'/'03_VALID_trial_lag_estimates_after_exact_collaborator_QC.csv'
HYST_WINDOW_CSV = SPATIAL/'07_spatial_hysteresis'/'spatial_hysteresis_window_level_bias_by_disparity.csv'
BREAKPOINT_CSV = SPATIAL/'06_stabilization_breakpoint'/'csv_outputs'/'anchor_trial_level_breakpoint_DYNAMIC_metrics.csv'
KALMAN_NATURAL_CSV = SL_DATA/'spatial_behavior_tonelli_lag_corrected_subject_first'/'kalman_filter_precision_analysis'/'csv_outputs'/'kalman_subject_natural_condition_summary.csv'
KALMAN_LAG_CSV = SL_DATA/'spatial_behavior_tonelli_lag_corrected_subject_first'/'kalman_filter_precision_analysis'/'csv_outputs'/'kalman_subject_baseline_lags.csv'
LEGACY_FIXED_CSV = SL_DATA/'spatial_pse_capture_hysteresis_recalibration_predictions'/'csv_outputs'/'fixed_offset_trial_level_capture_psame_bias.csv'

# ------------------------------- HELPERS -----------------------------
def numeric(x): return pd.to_numeric(x, errors='coerce')
def subject_ids(x): return x.astype(str).str.lower().str.strip()
def parse_same(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if s in {'same','same source','common','common source','yes','y','true','1'}: return 1.
    if s in {'different','different source','diff','separate','no','n','false','0'}: return 0.
    if any(z in s for z in ['different','diff','separate']): return 0.
    if any(z in s for z in ['same','common']): return 1.
    return np.nan

def static_av(row):
    typ=str(row.get('trialType','')).lower()
    label=str(row.get('condLabel','')).lower()
    if any(k in typ+' '+label for k in ('audio_only','visual_only','sweep','diverge','converge')): return False
    off=numeric(pd.Series([row.get('offsetDeg',np.nan)])).iloc[0]
    return typ=='congruent' or (np.isfinite(off) and abs(off) in OFFSETS) or bool(re.search(r'(10|20|30|40)',label))

def make_halves(df,strata,name='half',randomize=False,rng=None,labels=('odd','even')):
    d=df.copy().sort_values(['subject']+list(strata)+['trialIndex'])
    d[name]=labels[0]
    for _,g in d.groupby(['subject']+list(strata),sort=False,dropna=False):
        indexes=g.index.to_numpy().copy()
        swap=False
        if randomize:
            rng.shuffle(indexes)
            swap=bool(rng.integers(0,2))
        one,two=(labels[1],labels[0]) if swap else labels
        d.loc[indexes,name]=np.where(np.arange(len(indexes))%2==0,one,two)
    return d

def gaussian(d,mu,sigma,floor,peak):
    return floor+(peak-floor)*np.exp(-.5*((np.asarray(d,dtype=float)-mu)/sigma)**2)

def shaped_prob(model,d,pars):
    d=np.asarray(d,float)
    if model=='gaussian4': return gaussian(d,*pars)
    if model=='gaussian3': return gaussian(d,pars[0],pars[1],0.,pars[2])
    if model=='gaussian2': return np.exp(-.5*(d/pars[0])**2)
    if model=='logistic_abs': return expit(pars[0]-pars[1]*abs(d)/10.)
    if model=='probit_abs': return ndtr(pars[0]-pars[1]*abs(d)/10.)
    if model.startswith(('logistic_shifted','probit_shifted')):
        if model.endswith('4'):
            mu,theta,scale,peak=pars; floor=0.
        else:
            mu,theta,scale,floor,peak=pars
        F=expit if model.startswith('logistic') else ndtr
        # Explicitly normalized to make p(mu)=peak and p(infinity)=floor.
        radial=abs(d-mu)
        denominator=max(float(F(theta/scale)),1e-8)
        return floor+(peak-floor)*F((theta-radial)/scale)/denominator
    raise ValueError(model)

def log_loss_fn(pars,model,d,y):
    p=np.clip(shaped_prob(model,d,pars),EPS,1-EPS)
    return -float(np.sum(y*np.log(p)+(1-y)*np.log1p(-p)))

def gaussian_derived(mu,sigma,floor,peak):
    ans=dict(mu_deg=float(mu),causal_sigma_deg=float(sigma),floor_same=float(floor),
             peak_same=float(peak),delta50_deg=np.nan,left_boundary50_deg=np.nan,
             right_boundary50_deg=np.nan,slope50_per_deg=np.nan)
    if floor<.5<peak:
        z=(.5-floor)/(peak-floor)
        if 0<z<1:
            delta=float(sigma*np.sqrt(-2*np.log(z)))
            ans.update(delta50_deg=delta,left_boundary50_deg=float(mu-delta),
                       right_boundary50_deg=float(mu+delta),
                       slope50_per_deg=float((.5-floor)*delta/sigma**2))
    return ans

def model_starts(model,d,y):
    peak_guess=np.clip(float(y.mean()+.25),.61,.97)
    mu_guess=float(np.clip(d[np.argmax(np.abs(d)<8)] if (abs(d)<8).any() else 0.,*MU_LIMIT))
    if model=='gaussian4':
        return ([[mu_guess,25.,.03,peak_guess],[0.,18.,.05,.95],[0.,40.,.12,.80]],
                [MU_LIMIT,SIGMA_LIMIT,FLOOR_LIMIT,PEAK_LIMIT])
    if model=='gaussian3':
        return ([[mu_guess,20.,peak_guess],[0.,28.,.95]],
                [MU_LIMIT,SIGMA_LIMIT,PEAK_LIMIT])
    if model=='gaussian2': return ([[18.],[30.]], [SIGMA_LIMIT])
    if model in ('logistic_abs','probit_abs'):
        return ([[2.,1.],[1.,.5]], [(-10.,10.),(.00001,10.)])
    if model.endswith('shifted4'):
        return ([[0.,22.,7.,peak_guess],[mu_guess,28.,12.,.90]],
                [MU_LIMIT,(1.,70.),(1.5,40.),PEAK_LIMIT])
    if model.endswith('shifted5'):
        return ([[0.,22.,7.,.04,peak_guess],[mu_guess,28.,12.,.10,.90]],
                [MU_LIMIT,(1.,70.),(1.5,40.),FLOOR_LIMIT,PEAK_LIMIT])
    raise ValueError(model)

def fit_psychometric(d,y,model='gaussian4'):
    # IMPORTANT: use positional arrays rather than pandas index alignment.
    df=pd.DataFrame({'d':numeric(pd.Series(np.asarray(d))).to_numpy(),
                     'y':numeric(pd.Series(np.asarray(y))).to_numpy()}).replace([np.inf,-np.inf],np.nan).dropna()
    df=df[df.y.isin([0,1])]
    out=dict(model=model,fit_success=False,fit_status='not_fit',n_trials=int(len(df)),
             n_same=int(df.y.sum()),n_different=int(len(df)-df.y.sum()),
             n_unique_signed_disparities=int(df.d.round(2).nunique()),nll=np.nan)
    if len(df)<MIN_TRIALS or df.d.round(2).nunique()<4 or df.y.nunique()<2:
        out['fit_status']='insufficient_trial_or_response_variation'; return out
    x,z=df.d.to_numpy(float),df.y.to_numpy(float)
    starts,bounds=model_starts(model,x,z)
    attempts=[]
    for start in starts:
        try:
            res=minimize(log_loss_fn,np.asarray(start,float),args=(model,x,z),method='L-BFGS-B',
                         bounds=bounds,options={'maxiter':1200,'ftol':1e-10})
            if np.isfinite(res.fun): attempts.append(res)
        except (FloatingPointError,ValueError): pass
    if not attempts: out['fit_status']='optimization_failed'; return out
    successes=[res for res in attempts if res.success]
    fit=min(successes or attempts,key=lambda res: res.fun)
    out.update(fit_success=bool(fit.success),fit_status='ok' if fit.success else str(fit.message),
               nll=float(fit.fun),parameters=fit.x.copy(),
               aic=float(2*len(fit.x)+2*fit.fun),
               bic=float(len(fit.x)*np.log(len(df))+2*fit.fun))
    if model=='gaussian4':
        out.update(gaussian_derived(*fit.x))
        touches=np.isclose(fit.x,[lo for lo,hi in bounds],rtol=0,atol=.001)|np.isclose(fit.x,[hi for lo,hi in bounds],rtol=0,atol=.001)
        out['fit_on_parameter_bound']=bool(touches.any())
        out['bound_mu_or_sigma']=bool(touches[:2].any())
        out['bound_floor_or_peak']=bool(touches[2:].any())
        out['floor_at_zero']=bool(abs(fit.x[2]-FLOOR_LIMIT[0])<.001)
        out['peak_at_ceiling']=bool(abs(fit.x[3]-PEAK_LIMIT[1])<.001)
        L,R=out['left_boundary50_deg'],out['right_boundary50_deg']
        out['boundary50_within_observed_range']=bool(np.isfinite(L) and L>=x.min() and R<=x.max())
        out['boundary50_within_designed_range']=bool(np.isfinite(L) and L>=-40 and R<=40)
        out['threshold_quality_flag']=('optimizer_warning' if not fit.success else
            'boundary_not_observed' if not out['boundary50_within_observed_range'] else
            'mu_sigma_bound' if out['bound_mu_or_sigma'] else 'in_range')
    elif model=='gaussian3':
        out['delta50_deg']=gaussian_derived(fit.x[0],fit.x[1],0.,fit.x[2])['delta50_deg']
    elif model=='gaussian2': out['delta50_deg']=float(fit.x[0]*np.sqrt(2*np.log(2)))
    elif model.endswith('_abs'): out['delta50_deg']=10*fit.x[0]/fit.x[1] if fit.x[0]>=0 else np.nan
    return out

def sb_corr(x,y,rank=False):
    x,y=np.asarray(x,float),np.asarray(y,float)
    good=np.isfinite(x)&np.isfinite(y)
    x,y=x[good],y[good]
    if len(x)<4 or np.std(x)<1e-10 or np.std(y)<1e-10: return np.nan,len(x)
    r=float(spearmanr(x,y).statistic if rank else np.corrcoef(x,y)[0,1])
    return (2*r/(1+r) if np.isfinite(r) and (1+r)>1e-6 else np.nan),len(x)

def paired_boot(table,seed,nboot=N_BOOT):
    p=table[['subject','odd','even']].replace([np.inf,-np.inf],np.nan).dropna()
    xy=p[['odd','even']].to_numpy(float); n=len(xy)
    sb,_=sb_corr(xy[:,0],xy[:,1]); rank,_=sb_corr(xy[:,0],xy[:,1],True)
    rr=np.random.default_rng(seed); vals=[]
    for _ in range(nboot):
        ids=rr.integers(0,n,n)
        q,_=sb_corr(xy[ids,0],xy[ids,1])
        if np.isfinite(q): vals.append(q)
    lo,hi=np.percentile(vals,[2.5,97.5]) if len(vals)>30 else (np.nan,np.nan)
    return dict(n=n,pearson_r=sb/(2-sb) if np.isfinite(sb) else np.nan,
                spearman_rho=rank/(2-rank) if np.isfinite(rank) else np.nan,
                spearman_brown=sb,sb_ci_low=lo,sb_ci_high=hi,
                n_boot_valid=len(vals),spearman_brown_rank=rank)

def mean_halves(df,value,strata):
    d=df.dropna(subset=[value]).copy()
    if d.empty: return pd.DataFrame(columns=['subject','odd','even'])
    return make_halves(d,strata).groupby(['subject','half'])[value].mean().unstack().reset_index()

def halfdecay(offset,capture):
    s=pd.DataFrame({'d':offset,'y':capture}).dropna().groupby('d').y.mean().reindex(OFFSETS).dropna()
    if len(s)<3: return np.nan
    xx=s.index.to_numpy(float); yy=np.minimum.accumulate(s.to_numpy(float))
    if abs(yy[0]-yy[-1])<1e-9: return np.nan
    target=.5*(yy[0]+yy[-1])
    for i in range(len(xx)-1):
        if yy[i]>=target>=yy[i+1]:
            return float(np.interp(target,[yy[i+1],yy[i]],[xx[i+1],xx[i]])) if yy[i]!=yy[i+1] else float(xx[i])
    return np.nan

# ------------------------- TRIAL RECONSTRUCTION ----------------------
lag_lookup={}
if KALMAN_LAG_CSV.exists():
    lc=pd.read_csv(KALMAN_LAG_CSV)
    if {'subject','subject_baseline_lag_sec'}.issubset(lc.columns):
        lag_lookup=dict(zip(subject_ids(lc.subject),numeric(lc.subject_baseline_lag_sec)))
masters=sorted(glob.glob(str(SL_DATA/'*master_trials*.csv')))
if not masters: raise FileNotFoundError('No participant master_trials files in: '+str(SL_DATA))
causal_rows=[]; cal_rows=[]
for f in masters:
    match=re.search(r'(pid\d+)',Path(f).name,re.I)
    pid=match.group(1).lower() if match else Path(f).stem.lower()
    raw=pd.read_csv(f)
    needed={'trialIndex','trialType','condLabel','t','mouse_x','blob_x','audio_x'}
    if not needed.issubset(raw.columns):
        warnings.warn(f'{pid} missing columns {needed-set(raw.columns)}'); continue
    if 'isPractice' in raw: raw=raw[numeric(raw.isPractice).fillna(0).eq(0)]
    for (trial,typ,label),g in raw.groupby(['trialIndex','trialType','condLabel'],sort=False):
        g=g.sort_values('t'); row=g.iloc[0]
        t=numeric(g.t).to_numpy(float)
        if not np.isfinite(t).any(): continue
        t=t-np.nanmin(t)
        m=np.isfinite(t)&(t>=.5)&(t<=17.5)
        if m.sum()<10: continue
        time=t[m]; a=numeric(g.audio_x).to_numpy(float)[m]
        v=numeric(g.blob_x).to_numpy(float)[m]
        mouse=numeric(g.mouse_x).to_numpy(float)[m]
        if static_av(row):
            y=numeric(pd.Series([row.get('same_response_yes',np.nan)])).iloc[0]
            if not np.isfinite(y): y=parse_same(row.get('post_question',np.nan))
            valid=np.isfinite(a)&np.isfinite(v)
            if y in (0.,1.) and valid.sum()>=10:
                d=float(np.mean((v-a)[valid]))
                off=numeric(pd.Series([row.get('offsetDeg',np.nan)])).iloc[0]
                causal_rows.append(dict(subject=pid,source_file=Path(f).name,trialIndex=trial,
                    trialType=typ,condLabel=label,designed_offset_abs_deg=abs(off) if np.isfinite(off) else abs(d),
                    signed_disparity_deg=d,abs_disparity_deg=float(np.mean(abs((v-a)[valid]))),
                    disparity_cell=float(CELLS[np.argmin(abs(CELLS-d))]),same_response=float(y)))
        if str(typ).lower() in ('audio_only','visual_only'):
            target=a if str(typ).lower()=='audio_only' else v
            lag=lag_lookup.get(pid,0.)
            if not np.isfinite(lag): lag=0.
            response=np.interp(time+lag,time,mouse,left=np.nan,right=np.nan)
            valid=np.isfinite(response)&np.isfinite(target)
            if valid.sum()>=10:
                error=response[valid]-target[valid]
                cal_rows.append(dict(subject=pid,trialIndex=trial,
                    modality='Audio' if str(typ).lower()=='audio_only' else 'Visual',
                    mean_error_deg=float(error.mean()),sd_error_deg=float(np.std(error,ddof=1)),
                    n_samples=int(valid.sum()),lag_correction_sec=float(lag)))
causal=pd.DataFrame(causal_rows)
if causal.empty: raise RuntimeError('No valid static causal report trials. Verify master file schema.')
causal.subject=subject_ids(causal.subject)
causal.trialIndex=numeric(causal.trialIndex)
causal=causal.drop_duplicates(['subject','trialIndex']).reset_index(drop=True)
calibration_trial=pd.DataFrame(cal_rows)
if not calibration_trial.empty:
    cal=(calibration_trial.groupby(['subject','modality']).agg(
        localization_bias_deg=('mean_error_deg','mean'),tracking_sd_deg=('sd_error_deg','mean'),
        n_unimodal_trials=('trialIndex','count')).unstack('modality'))
    cal.columns=[f'{a}_{b}' for a,b in cal.columns]
    cal=cal.reset_index()
else: cal=pd.DataFrame({'subject':causal.subject.unique()})
for k in ('Audio','Visual'):
    for prefix in ('localization_bias_deg','tracking_sd_deg'):
        col=f'{prefix}_{k}'
        if col not in cal.columns: cal[col]=np.nan
cal['bias_shift_VminusA_deg']=cal.localization_bias_deg_Visual-cal.localization_bias_deg_Audio
causal=causal.merge(cal[['subject','bias_shift_VminusA_deg']],on='subject',how='left',validate='many_to_one')
causal['signed_disparity_biascorr_deg']=causal.signed_disparity_deg+causal.bias_shift_VminusA_deg
calibration_trial.to_csv(CSV_OUT/'aim1_unimodal_calibration_trial_level.csv',index=False)
cal.to_csv(CSV_OUT/'aim1_unimodal_calibration_subject_level.csv',index=False)

# ----------------------- FIT ALL MODELS / SUBJECTS -------------------
fitrows=[]; candidate_rows=[]
for pid,g in causal.groupby('subject'):
    fits={model:fit_psychometric(g.signed_disparity_deg,g.same_response,model) for model in MODELS}
    f=fits['gaussian4']
    row={'subject':pid,**{k:v for k,v in f.items() if k not in ('parameters','model')}}
    if g.signed_disparity_biascorr_deg.notna().all():
        bc=fit_psychometric(g.signed_disparity_biascorr_deg,g.same_response,'gaussian4')
        row.update({f'biascorr_{k}':v for k,v in bc.items() if k not in ('parameters','model')})
    row['mean_same']=float(g.same_response.mean())
    row['n_disparity_cells']=int(g.disparity_cell.nunique())
    fitrows.append(row)
    for mod,result in fits.items():
        candidate_rows.append({'subject':pid,**{k:v for k,v in result.items() if k!='parameters'}})
pheno=pd.DataFrame(fitrows).merge(cal,on='subject',how='left')
all_models=pd.DataFrame(candidate_rows)

# Explicit proxy source: do not silently conflate Kalman-derived R and tracking SD.
if KALMAN_NATURAL_CSV.exists():
    kn=pd.read_csv(KALMAN_NATURAL_CSV)
    if {'subject','condition_class','mean_sqrt_R'}.issubset(kn.columns):
        kn.subject=subject_ids(kn.subject)
        if 'response_version' in kn and kn.response_version.eq('baseline_lagcorr').any():
            kn=kn[kn.response_version.eq('baseline_lagcorr')]
        kp=kn[kn.condition_class.isin(['Audio-only','Visual-only'])].pivot_table(
            index='subject',columns='condition_class',values='mean_sqrt_R',aggfunc='mean').rename(
            columns={'Audio-only':'kalman_sqrt_R_A_deg','Visual-only':'kalman_sqrt_R_V_deg'}).reset_index()
        pheno=pheno.merge(kp,on='subject',how='left')
for modal,word in [('A','Audio'),('V','Visual')]:
    kc=f'kalman_sqrt_R_{modal}_deg'
    if kc not in pheno: pheno[kc]=np.nan
    tc=f'tracking_sd_deg_{word}'
    kcdata=numeric(pheno[kc])
    tcdata=numeric(pheno[tc])
    validk=kcdata.where(kcdata>0)
    validt=tcdata.where(tcdata>0)
    # Legacy column preserved for old Aims 2–4; source tags make it auditable.
    pheno[f'sensory_noise_{modal}_deg']=validk.fillna(validt)
    pheno[f'sensory_noise_{modal}_source']=np.select(
        [validk.notna(),validt.notna()],['Kalman_sqrt_R','tracking_SD_fallback'],default='missing')
    pheno[f'tracking_noise_proxy_{modal}_deg']=pheno[f'sensory_noise_{modal}_deg']
    pheno[f'tracking_noise_proxy_{modal}_source']=pheno[f'sensory_noise_{modal}_source']
    pheno[f'sensory_precision_{modal}']=1/np.square(pheno[f'sensory_noise_{modal}_deg'])

# Operational quality definitions; NEVER use these to silently exclude subjects.
pheno['delta50_full']=pheno.delta50_deg
pheno['delta50_inrange']=pheno.delta50_deg.where(pheno.boundary50_within_observed_range.fillna(False))
pheno['delta50_highquality_flag']=(pheno.fit_success.fillna(False) &
    pheno.boundary50_within_observed_range.fillna(False) &
    ~pheno.bound_mu_or_sigma.fillna(True) &
    pheno.peak_same.gt(QUALITY_MIN_PEAK) &
    (pheno.peak_same-pheno.floor_same).gt(QUALITY_MIN_PEAK_FLOOR_GAP))
pheno['delta50_highquality']=pheno.delta50_deg.where(pheno.delta50_highquality_flag)
pheno['left_boundary_censored']=pheno.left_boundary50_deg.lt(-40)
pheno['right_boundary_censored']=pheno.right_boundary50_deg.gt(40)
pheno['boundary_censor_status']=np.select(
    [pheno.left_boundary_censored & pheno.right_boundary_censored,
     pheno.left_boundary_censored,pheno.right_boundary_censored],
    ['both_unobserved','left_unobserved','right_unobserved'],default='fully_observed')
pheno.loc[pheno.delta50_deg.isna(),'boundary_censor_status']='threshold_not_identified'
pheno['explicit_causal_tolerance_delta50_deg']=pheno.delta50_deg
pheno['explicit_alignment_mu_deg']=pheno.mu_deg
pheno['explicit_curve_sigma_deg']=pheno.causal_sigma_deg
pheno['explicit_boundary_slope50']=pheno.slope50_per_deg
for d in OFFSETS:
    for signed in (-d,d):
        sign='minus' if signed<0 else 'plus'
        pheno[f'p_same_signed_{sign}{int(d)}']=gaussian(signed,pheno.mu_deg,pheno.causal_sigma_deg,
                                                      pheno.floor_same,pheno.peak_same)
    pheno[f'p_same_abs_{int(d)}']=.5*(pheno[f'p_same_signed_minus{int(d)}']+
                                      pheno[f'p_same_signed_plus{int(d)}'])
pheno.to_csv(CSV_OUT/'aim1_gaussian_psychometric_subject_parameters.csv',index=False)
pheno[['subject','delta50_full','delta50_inrange','delta50_highquality',
       'delta50_highquality_flag','boundary_censor_status']+
      [f'p_same_abs_{int(d)}' for d in OFFSETS]].to_csv(
          CSV_OUT/'aim1_validated_causal_tolerance_predictors.csv',index=False)
all_models.to_csv(CSV_OUT/'aim1_full_fit_competing_model_parameters.csv',index=False)

# --------------------------- HELD-OUT CV -----------------------------
cv_base=causal.copy(); cv_base['cv_fold']=-1
r=np.random.default_rng(SEED+7)
for _,g in cv_base.groupby(['subject','disparity_cell'],sort=False):
    ind=g.index.to_numpy().copy();r.shuffle(ind)
    cv_base.loc[ind,'cv_fold']=np.arange(len(ind))%N_CV_FOLDS
cvrows=[]; cvfit=[]
for (pid,fold),held in cv_base.groupby(['subject','cv_fold'],sort=False):
    train=cv_base[(cv_base.subject==pid)&cv_base.cv_fold.ne(fold)]
    for model in MODELS:
        result=fit_psychometric(train.signed_disparity_deg,train.same_response,model)
        cvfit.append(dict(subject=pid,fold=fold,model=model,train_n=result['n_trials'],
                          fit_success=result['fit_success']))
        if not result['fit_success']: continue
        pred=np.clip(shaped_prob(model,held.signed_disparity_deg,result['parameters']),EPS,1-EPS)
        for rowid,p in zip(held.index,pred):
            cvrows.append(dict(row_id=int(rowid),subject=pid,fold=int(fold),model=model,
                same_response=float(cv_base.loc[rowid,'same_response']),
                disparity_cell=float(cv_base.loc[rowid,'disparity_cell']),p_pred=float(p)))
cvpred=pd.DataFrame(cvrows)
if cvpred.empty: raise RuntimeError('No held-out psychometric fits succeeded.')
cvpred['log_loss']=-(cvpred.same_response*np.log(cvpred.p_pred)+(1-cvpred.same_response)*np.log1p(-cvpred.p_pred))
cvpred['brier']=(cvpred.p_pred-cvpred.same_response)**2
cvpred.to_csv(CSV_OUT/'aim1_cv_trial_predictions_all_models.csv',index=False)
pd.DataFrame(cvfit).to_csv(CSV_OUT/'aim1_cv_fit_status.csv',index=False)
# Compare models on exactly the SAME held-out trial keys (complete-case intersection).
matched=cvpred.groupby('row_id').model.nunique()
common=matched[matched.eq(len(MODELS))].index
cvcommon=cvpred[cvpred.row_id.isin(common)].copy()
cvcommon.to_csv(CSV_OUT/'aim1_cv_common_trial_predictions.csv',index=False)
if len(common)<.75*len(cv_base):
    warnings.warn('Less than 75% of trials had predictions from ALL candidates. Inspect CV fit-status file.')
if cvcommon.empty: raise RuntimeError('No fully paired held-out predictions among candidate models.')
cv_subject=cvcommon.groupby(['subject','model'],as_index=False).agg(
    n_heldout=('row_id','count'),log_loss=('log_loss','mean'),brier=('brier','mean'))
cv_subject.to_csv(CSV_OUT/'aim1_cv_participant_model_scores.csv',index=False)
cv_summary=cv_subject.groupby('model',as_index=False).agg(
    n_subjects=('subject','nunique'),mean_log_loss=('log_loss','mean'),
    median_log_loss=('log_loss','median'),mean_brier=('brier','mean')).sort_values('mean_log_loss')
cv_summary.to_csv(CSV_OUT/'aim1_cv_model_comparison_summary.csv',index=False)
bycell=cvcommon.groupby(['model','disparity_cell'],as_index=False).agg(
    n=('row_id','count'),observed_same=('same_response','mean'),predicted_same=('p_pred','mean'))
bycell['calibration_error']=bycell.predicted_same-bycell.observed_same
bycell.to_csv(CSV_OUT/'aim1_cv_condition_calibration.csv',index=False)
wide=cv_subject.pivot(index='subject',columns='model',values='log_loss')
if {'gaussian4','logistic_abs','probit_abs'}.issubset(wide.columns):
    pd.DataFrame(dict(subject=wide.index,gaussian4_minus_logistic=wide.gaussian4-wide.logistic_abs,
        gaussian4_minus_probit=wide.gaussian4-wide.probit_abs)).to_csv(
        CSV_OUT/'aim1_cv_gaussian_vs_simple_model_subject_differences.csv',index=False)

paired_rows=[]; influence_rows=[]
for alternate in [z for z in MODELS if z!='gaussian4']:
    d=wide[['gaussian4',alternate]].dropna()
    dif=(d.gaussian4-d[alternate]).to_numpy(float) # negative means Gaussian4 better
    n=len(d)
    if n<6: continue
    rr=np.random.default_rng(SEED+len(alternate))
    boot=np.mean(dif[rr.integers(0,n,(N_BOOT,n))],axis=1)
    null=np.mean(dif[None,:]*rr.choice([-1,1],size=(N_BOOT,n)),axis=1)
    paired_rows.append(dict(comparison=f'gaussian4_minus_{alternate}',n=n,
        mean_log_loss_difference=float(dif.mean()),ci95_low=float(np.percentile(boot,2.5)),
        ci95_high=float(np.percentile(boot,97.5)),
        paired_signflip_p=float((1+(abs(null)>=abs(dif.mean())).sum())/(N_BOOT+1)),
        gaussian_better_participant_fraction=float(np.mean(dif<0))))
    for excluded in d.index:
        d0=d.drop(index=excluded)
        influence_rows.append(dict(comparison=f'gaussian4_minus_{alternate}',excluded_subject=excluded,
            n_remaining=len(d0),mean_difference_after_exclusion=float((d0.gaussian4-d0[alternate]).mean())))
    # Predefined descriptive leave-two-influential-subjects-out sensitivity.
    favorable=d.assign(diff=d.gaussian4-d[alternate]).sort_values('diff').head(2)
    trimmed=d.drop(index=favorable.index)
    paired_rows[-1]['mean_difference_excluding_two_most_gaussian_favorable']=float(
        (trimmed.gaussian4-trimmed[alternate]).mean())
    paired_rows[-1]['two_excluded_subjects']=';'.join(favorable.index.tolist())
pd.DataFrame(paired_rows).to_csv(CSV_OUT/'aim1_cv_paired_model_comparisons.csv',index=False)
pd.DataFrame(influence_rows).to_csv(CSV_OUT/'aim1_cv_leave_one_subject_out_influence.csv',index=False)

# --------------------- HELD-OUT CALIBRATION BY PREDICTION ------------
calib_bins=[]; calib_slopes=[]
for model,df in cvcommon.groupby('model'):
    df=df.copy(); df['bin']=pd.cut(df.p_pred,np.linspace(0,1,11),include_lowest=True)
    subject_bin=df.groupby(['subject','bin'],observed=True).agg(
        pred=('p_pred','mean'),obs=('same_response','mean'),n=('row_id','count')).reset_index()
    # Participant-first bins avoid participants with more trials dominating summaries.
    b=subject_bin.groupby('bin',observed=True).agg(
        mean_pred=('pred','mean'),observed_same=('obs','mean'),
        n_subjects=('subject','nunique'),n_trials=('n','sum')).reset_index()
    b['model']=model
    calib_bins.append(b)
    val=df.p_pred.clip(EPS,1-EPS)
    logit=np.log(val/(1-val))
    try:
        X=sm.add_constant(pd.DataFrame({'predicted_logit':logit.to_numpy()},index=df.index))
        reg=sm.GLM(df.same_response.to_numpy(),X,family=sm.families.Binomial()).fit(
            cov_type='cluster',cov_kwds={'groups':df.subject.to_numpy()})
        calib_slopes.append(dict(model=model,n_subjects=df.subject.nunique(),
          intercept=float(reg.params['const']),intercept_se=float(reg.bse['const']),
          slope=float(reg.params['predicted_logit']),slope_se=float(reg.bse['predicted_logit']),
          slope_p=float(reg.pvalues['predicted_logit']),
          note='Descriptive cluster-robust calibration; ideal intercept 0, slope 1'))
    except Exception as exc:
        calib_slopes.append(dict(model=model,error=repr(exc)))
calib_bins=pd.concat(calib_bins,ignore_index=True)
calib_bins.to_csv(CSV_OUT/'aim1_cv_predicted_probability_calibration_bins.csv',index=False)
pd.DataFrame(calib_slopes).to_csv(CSV_OUT/'aim1_cv_calibration_intercept_slope.csv',index=False)

# --------------------- CROSS-FITTED GAUSSIAN FOR AIMS 2–4 ------------
keys=['subject','mu_deg','causal_sigma_deg','floor_same','peak_same',
      'delta50_deg','slope50_per_deg','threshold_quality_flag','boundary50_within_observed_range']
causal=causal.merge(pheno[keys],on='subject',how='left',validate='many_to_one')
causal['p_same_hat_full']=gaussian(causal.signed_disparity_deg,causal.mu_deg,
                                 causal.causal_sigma_deg,causal.floor_same,causal.peak_same)
P=causal.p_same_hat_full.clip(EPS,1-EPS)
causal['causal_logodds_full']=np.log(P/(1-P))
causal['causal_ambiguity_full']=4*P*(1-P)
causal['radial_disparity_from_mu_deg']=abs(causal.signed_disparity_deg-causal.mu_deg)
causal['boundary_distance_deg']=causal.radial_disparity_from_mu_deg-causal.delta50_deg
causal['boundary_z']=causal.boundary_distance_deg/causal.causal_sigma_deg
causal_cf=make_halves(causal,['disparity_cell']).copy()
causal_cf['_rowkey']=causal_cf.index
cf_records=[]; cf_scores=[]
for (pid,half),test in causal_cf.groupby(['subject','half'],sort=False):
    train=causal_cf[(causal_cf.subject==pid)&(causal_cf.half!=half)]
    gaussianfit=fit_psychometric(train.signed_disparity_deg,train.same_response,'gaussian4')
    rec=dict(subject=pid,heldout_half=half)
    rec.update({f'cf_{k}':v for k,v in gaussianfit.items() if k not in ('model','parameters')})
    cf_records.append(rec)
    for model in MODELS:
        fitted=gaussianfit if model=='gaussian4' else fit_psychometric(
            train.signed_disparity_deg,train.same_response,model)
        pred=shaped_prob(model,test.signed_disparity_deg,fitted['parameters']) if fitted['fit_success'] else np.full(len(test),np.nan)
        cf_scores.extend(dict(_rowkey=int(i),model=model,p=float(v)) for i,v in zip(test.index,pred))
cf_fits=pd.DataFrame(cf_records)
causal_cf=causal_cf.merge(cf_fits,left_on=['subject','half'],right_on=['subject','heldout_half'],how='left')
cf_probs=pd.DataFrame(cf_scores).pivot(index='_rowkey',columns='model',values='p')
cf_probs.columns=[f'cf_p_same_{mod}' for mod in cf_probs.columns]
causal_cf=causal_cf.merge(cf_probs,left_on='_rowkey',right_index=True,how='left',validate='one_to_one').drop(columns='_rowkey')
causal_cf['cf_p_same_hat']=causal_cf['cf_p_same_gaussian4']
cfP=causal_cf.cf_p_same_hat.clip(EPS,1-EPS)
causal_cf['cf_causal_logodds']=np.log(cfP/(1-cfP))
causal_cf['cf_causal_ambiguity']=4*cfP*(1-cfP)
causal_cf['cf_radial_disparity_from_mu_deg']=abs(causal_cf.signed_disparity_deg-causal_cf.cf_mu_deg)
causal_cf['cf_boundary_distance_deg']=causal_cf.cf_radial_disparity_from_mu_deg-causal_cf.cf_delta50_deg
causal_cf['cf_boundary_z']=causal_cf.cf_boundary_distance_deg/causal_cf.cf_causal_sigma_deg
causal_cf.to_csv(CSV_OUT/'aim1_static_causal_trial_table_with_crossfit_predictions.csv',index=False)
cf_fits.to_csv(CSV_OUT/'aim1_crossfit_gaussian_fit_parameters.csv',index=False)

# ---------------------- REPEATED SPLIT-HALF RELIABILITY --------------
metrics=['mu_deg','delta50_deg','causal_sigma_deg','slope50_per_deg','peak_same','floor_same']
repeat_rows=[]; reliability_rows=[]
rrng=np.random.default_rng(SEED+37)
for repetition in range(N_RELIABILITY_SPLITS):
    df=make_halves(causal,['disparity_cell'],name='half_rep',randomize=True,rng=rrng,labels=('A','B'))
    fits=[]
    for (pid,half),group in df.groupby(['subject','half_rep']):
        out=fit_psychometric(group.signed_disparity_deg,group.same_response,'gaussian4')
        fits.append(dict(subject=pid,repeat=repetition,half=half,
                         **{k:v for k,v in out.items() if k not in ('model','parameters')}))
    fitdf=pd.DataFrame(fits);repeat_rows.append(fitdf)
    for metric in metrics:
        w=fitdf.pivot(index='subject',columns='half',values=metric)
        if not {'A','B'}.issubset(w.columns): continue
        value,n=sb_corr(w.A,w.B); rank,_=sb_corr(w.A,w.B,True)
        reliability_rows.append(dict(repeat=repetition,metric=metric,n=n,spearman_brown=value,
                         pearson_r=value/(2-value) if np.isfinite(value) else np.nan,
                         spearman_rho=rank/(2-rank) if np.isfinite(rank) else np.nan,
                         spearman_brown_rank=rank))
repeat_params=pd.concat(repeat_rows,ignore_index=True)
repeat_rel=pd.DataFrame(reliability_rows)
repeat_params.to_csv(CSV_OUT/'aim1_gaussian_repeated_split_parameter_estimates.csv',index=False)
repeat_rel.to_csv(CSV_OUT/'aim1_gaussian_repeated_split_reliability_all.csv',index=False)
participants=sorted(repeat_params.subject.unique())
pid_index={pid:k for k,pid in enumerate(participants)}
arr=np.full((len(metrics),N_RELIABILITY_SPLITS,len(participants),2),np.nan)
for row in repeat_params.itertuples(index=False):
    for mi,metric in enumerate(metrics):
        arr[mi,int(row.repeat),pid_index[row.subject],int(row.half=='B')]=getattr(row,metric)
brng=np.random.default_rng(SEED+121)
rsum=[]
for mi,metric in enumerate(metrics):
    subset=repeat_rel[repeat_rel.metric==metric]
    boots=[]
    for _ in range(N_BOOT):
        chosen=brng.integers(0,len(participants),len(participants))
        chosen_splits=brng.integers(0,N_RELIABILITY_SPLITS,N_SPLITS_PER_BOOT)
        values=[]
        for j in chosen_splits:
            q=arr[mi,j,chosen]
            score,_=sb_corr(q[:,0],q[:,1])
            if np.isfinite(score): values.append(score)
        if len(values)>=N_SPLITS_PER_BOOT//2: boots.append(np.median(values))
    low,high=np.percentile(boots,[2.5,97.5]) if len(boots)>100 else (np.nan,np.nan)
    rsum.append(dict(metric=metric,n_repeats=len(subset),median_n_subjects=float(subset.n.median()),
        pearson_r_median=float(subset.pearson_r.median()),
        spearman_rho_median=float(subset.spearman_rho.median()),
        spearman_brown_median=float(subset.spearman_brown.median()),
        spearman_brown_rank_median=float(subset.spearman_brown_rank.median()),
        spearman_brown_ci_low=low,spearman_brown_ci_high=high,n_boot_valid=len(boots),
        method='repeated_stratified_split_half',
        interval_method='participant bootstrap with resampled precomputed random splits'))
repeat_summary=pd.DataFrame(rsum)
repeat_summary.to_csv(CSV_OUT/'aim1_gaussian_repeated_split_reliability_summary.csv',index=False)

# Reliability subsets: no hard-coded subject removals. Fix pandas nullable-bool warning.
sensitivity=[]
for metric in metrics:
    for repetition,br in repeat_params.groupby('repeat'):
        w=br.pivot(index='subject',columns='half',values=metric)
        flag=br.pivot(index='subject',columns='half',values='boundary50_within_observed_range')
        if not {'A','B'}.issubset(w.columns) or not {'A','B'}.issubset(flag.columns): continue
        keep=(flag['A'].astype('boolean').fillna(False) & flag['B'].astype('boolean').fillna(False)).to_numpy(bool)
        value,n=sb_corr(w.loc[keep,'A'],w.loc[keep,'B'])
        sensitivity.append(dict(metric=metric,repeat=repetition,n=n,spearman_brown=value))
sens=pd.DataFrame(sensitivity)
sens.groupby('metric',as_index=False).agg(median_n=('n','median'),median_sb=('spearman_brown','median'),
     min_n=('n','min'),n_valid_repeats=('spearman_brown','count')).to_csv(
     CSV_OUT/'aim1_in_range_only_reliability_sensitivity.csv',index=False)

# ------------------------- DOWNSTREAM RELIABILITIES ------------------
downstream={}
if LEGACY_FIXED_CSV.exists():
    fx=pd.read_csv(LEGACY_FIXED_CSV);fx.subject=subject_ids(fx.subject)
    fx.offset_deg=numeric(fx.offset_deg).abs();fx.NormalizedVisualBias=numeric(fx.NormalizedVisualBias)
    fx=fx[fx.offset_deg.isin(OFFSETS)]
    downstream['Normalized_visual_capture']=mean_halves(fx,'NormalizedVisualBias',['offset_deg'])
    fh=make_halves(fx.dropna(subset=['NormalizedVisualBias']),['offset_deg'])
    hd=pd.DataFrame([dict(subject=pid,half=half,implicit=halfdecay(g.offset_deg,g.NormalizedVisualBias))
                     for (pid,half),g in fh.groupby(['subject','half'])])
    downstream['Implicit_capture_halfdecay']=hd.pivot(index='subject',columns='half',values='implicit').reset_index()
if STABLE_BIAS_CSV.exists():
    sb=pd.read_csv(STABLE_BIAS_CSV);sb.subject=subject_ids(sb.subject)
    sb.offset_num=numeric(sb.offset_num).abs()
    sb['Bias_lagcorr_AminusV_stable']=numeric(sb.Bias_lagcorr_AminusV_stable)
    sb['Psame']=numeric(sb.Psame)
    sb=sb[sb.offset_num.isin(OFFSETS)]
    downstream['Lagcorrected_RMSE_bias']=mean_halves(sb,'Bias_lagcorr_AminusV_stable',['offset_num'])
    sh=make_halves(sb.dropna(subset=['Bias_lagcorr_AminusV_stable','Psame']),['offset_num'])
    residual=[]
    for half,g in sh.groupby('half'):
        if len(g)<15: continue
        reg=smf.ols('Bias_lagcorr_AminusV_stable ~ C(offset_num) + Psame',data=g).fit()
        tmp=g[['subject']].copy();tmp['resid']=reg.resid
        result=tmp.groupby('subject').resid.mean().reset_index();result['half']=half
        residual.append(result)
    if residual:
        downstream['Residual_visual_bias']=pd.concat(residual).pivot(index='subject',columns='half',values='resid').reset_index()
if VALID_LAG_CSV.exists():
    lg=pd.read_csv(VALID_LAG_CSV);lg.subject=subject_ids(lg.subject)
    lg.offset_deg=numeric(lg.offset_deg).abs();lg.lag_audio_sec=numeric(lg.lag_audio_sec)
    downstream['Auditory_tracking_lag']=mean_halves(lg[lg.offset_deg.isin(OFFSETS)],'lag_audio_sec',['offset_deg'])
if BREAKPOINT_CSV.exists():
    bp=pd.read_csv(BREAKPOINT_CSV);bp.subject=subject_ids(bp.subject)
    bp.offset_deg=numeric(bp.offset_deg).abs();bp.dynamically_updating=numeric(bp.dynamically_updating)
    if 'metric' in bp: bp=bp[bp.metric.astype(str).eq('LagCorrectedVisualCapture')]
    downstream['Breakpoint_dynamicity']=mean_halves(bp[bp.offset_deg.isin(OFFSETS)],'dynamically_updating',['offset_deg'])
if HYST_WINDOW_CSV.exists():
    hw=pd.read_csv(HYST_WINDOW_CSV);hw.subject=subject_ids(hw.subject)
    for col in ['trialIndex','mean_disparity_window','NormalizedVisualBias_window']: hw[col]=numeric(hw[col])
    hw.SweepDirection=hw.SweepDirection.astype(str).str.title()
    hw=hw[hw.SweepDirection.isin(['Diverge','Converge'])&hw.mean_disparity_window.between(5,35)]
    keys=hw[['subject','SweepDirection','trialIndex']].drop_duplicates()
    keys=make_halves(keys,['SweepDirection'])
    hw=hw.merge(keys,on=['subject','SweepDirection','trialIndex'],how='left')
    hw['disp_bin']=np.round(hw.mean_disparity_window/2)*2
    row=[]
    for (pid,half),g in hw.groupby(['subject','half']):
        q=g.groupby(['SweepDirection','disp_bin']).NormalizedVisualBias_window.mean().unstack('SweepDirection')
        z=q.dropna(subset=['Diverge','Converge']) if {'Diverge','Converge'}.issubset(q.columns) else pd.DataFrame()
        row.append(dict(subject=pid,half=half,hysteresis=float((z.Converge-z.Diverge).mean()) if len(z)>=2 else np.nan))
    if row: downstream['SL_hysteresis']=pd.DataFrame(row).pivot(index='subject',columns='half',values='hysteresis').reset_index()
relrows=[]; halfrows=[]
for i,(label,tab) in enumerate(downstream.items()):
    if not {'odd','even'}.issubset(tab.columns): continue
    stat=paired_boot(tab,seed=SEED+37*i)
    relrows.append({'metric':label,'method':'deterministic_stratified_odd_even',**stat})
    tmp=tab[['subject','odd','even']].copy();tmp['metric']=label;halfrows.append(tmp)
downstream_rel=pd.DataFrame(relrows)
downstream_rel.to_csv(CSV_OUT/'aim1_downstream_split_half_reliability_summary.csv',index=False)
if halfrows: pd.concat(halfrows,ignore_index=True).to_csv(CSV_OUT/'aim1_downstream_odd_even_metrics.csv',index=False)
relval={f'Gaussian_{row.metric}':row.spearman_brown_median for row in repeat_summary.itertuples()}
relval.update({row.metric:row.spearman_brown for row in downstream_rel.itertuples()})
ceiling=pd.DataFrame(index=relval,columns=relval,dtype=float)
for i,ri in relval.items():
    for j,rj in relval.items(): ceiling.loc[i,j]=np.sqrt(ri*rj) if np.isfinite(ri) and np.isfinite(rj) and ri>=0 and rj>=0 else np.nan
ceiling.to_csv(CSV_OUT/'aim1_reliability_ceiling_matrix.csv')

# ------------------- EMPIRICAL CROSSINGS / QC / CALIBRATION -----------
observed=causal.groupby(['subject','disparity_cell']).same_response.agg(['mean','count']).reset_index().rename(columns={'mean':'observed_same','count':'n_trials'})
observed.to_csv(CSV_OUT/'aim1_observed_signed_psychometric_cells.csv',index=False)
empirical=[]
for pid,g in causal.groupby('subject'):
    empirical_curve=g.groupby(g.disparity_cell.abs()).same_response.mean().sort_index()
    xx=empirical_curve.index.to_numpy(float);y=empirical_curve.to_numpy(float); crosses=[]
    for i in range(len(xx)-1):
        if y[i]>=.5>=y[i+1] and y[i]!=y[i+1]:
            crosses.append(float(xx[i]+(.5-y[i])*(xx[i+1]-xx[i])/(y[i+1]-y[i])))
    empirical.append(dict(subject=pid,empirical_abs_crossing_deg=crosses[0] if crosses else np.nan,
                          n_empirical_crossings=len(crosses)))
qc=pheno.merge(pd.DataFrame(empirical),on='subject',how='left')
qc['fit_boundary_minus_empirical_deg']=qc.delta50_deg-qc.empirical_abs_crossing_deg
qc.to_csv(CSV_OUT/'aim1_fit_qc_and_empirical_crossings.csv',index=False)

# Exploratory calibration relationships using SEPARATE Kalman and tracking SD proxies.
calib_predictors=['bias_shift_VminusA_deg','kalman_sqrt_R_A_deg','kalman_sqrt_R_V_deg',
                  'tracking_sd_deg_Audio','tracking_sd_deg_Visual']
calib_corr=[]
for col in calib_predictors:
    sub=qc[['delta50_deg',col,'boundary50_within_observed_range']].dropna()
    sub=sub[sub.boundary50_within_observed_range]
    if len(sub)>=10 and sub[col].std()>0:
        r,p=pearsonr(sub.delta50_deg,sub[col])
        calib_corr.append(dict(predictor=col,n=len(sub),pearson_r=r,p=p))
pd.DataFrame(calib_corr).to_csv(CSV_OUT/'aim1_calibration_correlations_exploratory.csv',index=False)
for name,preds in [('kalman',['bias_shift_VminusA_deg','kalman_sqrt_R_A_deg','kalman_sqrt_R_V_deg']),
                   ('tracking_sd',['bias_shift_VminusA_deg','tracking_sd_deg_Audio','tracking_sd_deg_Visual'])]:
    sub=qc[['delta50_deg','boundary50_within_observed_range']+preds].dropna()
    sub=sub[sub.boundary50_within_observed_range].copy()
    if len(sub)>=18 and all(sub[col].std()>0 for col in preds):
        for col in preds: sub[col+'_z']=(sub[col]-sub[col].mean())/sub[col].std(ddof=0)
        mod=smf.ols('delta50_deg ~ '+' + '.join(p+'_z' for p in preds),data=sub).fit(cov_type='HC3')
        (CSV_OUT/f'aim1_calibration_HC3_{name}.txt').write_text(str(mod.summary()),encoding='utf-8')
        if name=='kalman':
            (CSV_OUT/'aim1_calibration_multivariable_HC3_exploratory.txt').write_text(str(mod.summary()),encoding='utf-8')
            pd.DataFrame({'term':mod.params.index,'beta':mod.params.values,'p':mod.pvalues.values}).to_csv(
                 CSV_OUT/'aim1_calibration_multivariable_coefficients.csv',index=False)

# ----------------------- GAUSSIAN PARAMETER RECOVERY -----------------
# Conditional recovery: generate new outcomes from participant's fitted curve
# AT THAT PARTICIPANT'S ACTUAL DISPARITIES. This does not validate model truth.
sim_rng=np.random.default_rng(SEED+191)
recovery=[]
for pid,g in causal.groupby('subject'):
    row=pheno.set_index('subject').loc[pid]
    params=np.array([row.mu_deg,row.causal_sigma_deg,row.floor_same,row.peak_same],float)
    if not np.isfinite(params).all(): continue
    xs=g.signed_disparity_deg.to_numpy(float)
    true_prob=np.clip(gaussian(xs,*params),EPS,1-EPS)
    for rep in range(N_RECOVERY_REPS):
        simulated=sim_rng.binomial(1,true_prob)
        fitted=fit_psychometric(xs,simulated,'gaussian4')
        recovery.append(dict(subject=pid,simulation=rep,fit_success=fitted['fit_success'],
            true_mu_deg=params[0],recovered_mu_deg=fitted.get('mu_deg',np.nan),
            true_causal_sigma_deg=params[1],recovered_causal_sigma_deg=fitted.get('causal_sigma_deg',np.nan),
            true_floor_same=params[2],recovered_floor_same=fitted.get('floor_same',np.nan),
            true_peak_same=params[3],recovered_peak_same=fitted.get('peak_same',np.nan),
            true_delta50_deg=row.delta50_deg,recovered_delta50_deg=fitted.get('delta50_deg',np.nan),
            source_inrange=bool(row.boundary50_within_observed_range)))
recovery=pd.DataFrame(recovery)
recovery.to_csv(CSV_OUT/'aim1_gaussian_parameter_recovery_simulations.csv',index=False)
recovery_summary=[]
for par in ('mu_deg','causal_sigma_deg','floor_same','peak_same','delta50_deg'):
    for groupname,sel in [('all',recovery),('source_inrange',recovery[recovery.source_inrange])]:
        dd=sel[[f'true_{par}',f'recovered_{par}']].dropna()
        if dd.empty: continue
        err=dd[f'recovered_{par}']-dd[f'true_{par}']
        recovery_summary.append(dict(parameter=par,subset=groupname,n=len(dd),
            n_subjects=sel.subject.nunique(),bias=float(err.mean()),
            MAE=float(abs(err).mean()),RMSE=float(np.sqrt(np.mean(err**2))),
            recovered_vs_true_r=float(dd.corr().iloc[0,1]) if dd.iloc[:,0].std()>0 and dd.iloc[:,1].std()>0 else np.nan))
pd.DataFrame(recovery_summary).to_csv(CSV_OUT/'aim1_gaussian_parameter_recovery_summary.csv',index=False)

# ------------------------------ FIGURES ------------------------------
xgrid=np.linspace(-45,45,361)
fig,ax=plt.subplots(figsize=(10,6));curves=[]
for row in pheno.itertuples(index=False):
    if row.fit_success and np.isfinite(row.mu_deg) and np.isfinite(row.causal_sigma_deg):
        p=gaussian(xgrid,row.mu_deg,row.causal_sigma_deg,row.floor_same,row.peak_same)
        curves.append(p);ax.plot(xgrid,p,lw=1,alpha=.10)
if curves: ax.plot(xgrid,np.mean(curves,axis=0),lw=3,label='Mean Gaussian fit')
ax.axhline(.5,ls='--',lw=1.2,c='firebrick');ax.axvline(0,ls=':',color='gray')
ax.set(xlabel='Actual signed visual − auditory disparity (deg)',ylabel='Fitted P(Same source)',
       ylim=(-.02,1.02),title='Explicit causal reports: participant Gaussian fits')
ax.legend(frameon=False);fig.tight_layout()
fig.savefig(FIG_OUT/'aim1_participant_gaussian_psychometric_curves.png',dpi=220,bbox_inches='tight');plt.show()

bars=[dict(metric='Gaussian '+r.metric,SB=r.spearman_brown_median,lo=r.spearman_brown_ci_low,
           hi=r.spearman_brown_ci_high) for r in repeat_summary.itertuples()]
bars += [dict(metric=r.metric,SB=r.spearman_brown,lo=r.sb_ci_low,hi=r.sb_ci_high)
         for r in downstream_rel.itertuples()]
plot=pd.DataFrame(bars).dropna(subset=['SB']).sort_values('SB')
fig,ax=plt.subplots(figsize=(10,max(5,.43*len(plot))))
ax.barh(np.arange(len(plot)),plot.SB)
for i,r in enumerate(plot.itertuples()):
    if np.isfinite(r.lo) and np.isfinite(r.hi):
        ax.errorbar(r.SB,i,xerr=[[max(0,r.SB-r.lo)],[max(0,r.hi-r.SB)]],fmt='none',color='black',capsize=3)
ax.set(yticks=np.arange(len(plot)),yticklabels=plot.metric,xlim=(min(0,plot.SB.min()-.05),1.08),
       xlabel='Spearman–Brown corrected split-half reliability',
       title='Reliability of Gaussian causal phenotype and downstream measures')
fig.tight_layout();fig.savefig(FIG_OUT/'aim1_gaussian_and_downstream_reliability.png',dpi=220,bbox_inches='tight');plt.show()

fig,ax=plt.subplots(figsize=(10,5))
for model,g in bycell.groupby('model'):
    g=g.sort_values('disparity_cell')
    ax.plot(g.disparity_cell,g.predicted_same,marker='o',label=model,alpha=.75)
first=bycell[bycell.model=='gaussian4'].sort_values('disparity_cell')
ax.scatter(first.disparity_cell,first.observed_same,marker='x',s=90,c='black',zorder=10,label='Observed')
ax.set(xlabel='Signed audiovisual disparity (deg)',ylabel='P(Same source)',ylim=(-.03,1.03),
       title='Held-out psychometric calibration by stimulus condition')
ax.legend(frameon=False,ncol=2,fontsize=8);fig.tight_layout()
fig.savefig(FIG_OUT/'aim1_crossvalidated_condition_calibration.png',dpi=220,bbox_inches='tight');plt.show()

fig,ax=plt.subplots(figsize=(11,5)); cs=cv_summary.sort_values('mean_log_loss')
ax.bar(cs.model,cs.mean_log_loss)
ax.tick_params(axis='x',rotation=45)
ax.set(ylabel='Mean held-out binomial log loss (lower is better)',
       title='Out-of-sample psychometric model comparison — identical trials')
fig.tight_layout();fig.savefig(FIG_OUT/'aim1_out_of_sample_model_comparison.png',dpi=220,bbox_inches='tight');plt.show()

fig,ax=plt.subplots(figsize=(7,6))
for model in ['gaussian4','logistic_shifted4','probit_shifted4','logistic_shifted5','probit_shifted5']:
    t=calib_bins[calib_bins.model.eq(model)]
    ax.plot(t.mean_pred,t.observed_same,marker='o',label=model)
ax.plot([0,1],[0,1],ls='--',c='black',lw=1,label='Perfect calibration')
ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean held-out predicted P(Same)',
       ylabel='Mean observed Same-report proportion',title='Cross-validated probability calibration')
ax.legend(frameon=False,fontsize=8);fig.tight_layout()
fig.savefig(FIG_OUT/'aim1_heldout_probability_calibration.png',dpi=220,bbox_inches='tight');plt.show()

flagged=qc[(~qc.boundary50_within_observed_range.fillna(False))|
           qc.bound_mu_or_sigma.fillna(False)|
           (qc.fit_boundary_minus_empirical_deg.abs()>10)].head(12)
if len(flagged):
    nrow=int(np.ceil(len(flagged)/3));fig,axes=plt.subplots(nrow,3,figsize=(12.6,nrow*3.2),squeeze=False)
    for ax in axes.flat: ax.set_visible(False)
    for ax,row in zip(axes.flat,flagged.itertuples(index=False)):
        ax.set_visible(True);obs=observed[observed.subject==row.subject]
        ax.scatter(obs.disparity_cell,obs.observed_same,c='black',s=25)
        if np.isfinite(row.mu_deg) and np.isfinite(row.causal_sigma_deg):
            x=np.linspace(-40,40,161)
            ax.plot(x,gaussian(x,row.mu_deg,row.causal_sigma_deg,row.floor_same,row.peak_same))
        ax.axhline(.5,ls='--',lw=1,c='gray')
        ax.set(xlim=(-42,42),ylim=(-.05,1.05),title=f'{row.subject}: Δ50={row.delta50_deg:.1f}°')
    fig.suptitle('Participant Gaussian fits requiring QC');fig.tight_layout()
    fig.savefig(FIG_OUT/'aim1_flagged_participant_fit_diagnostics.png',dpi=200,bbox_inches='tight');plt.show()

print('\n=== AIM 1 VALIDATION COMPLETE ===')
print('Participants:',pheno.subject.nunique(),'| static trials:',len(causal))
print('\nOut-of-sample model comparison:\n',cv_summary.to_string(index=False))
print('\nPaired Gaussian-vs-alternatives:\n',pd.DataFrame(paired_rows).to_string(index=False))
print('\nGaussian QC counts:\n',pheno[['fit_success','fit_on_parameter_bound',
       'bound_mu_or_sigma','boundary50_within_observed_range','delta50_highquality_flag']].sum().to_string())
print('\nGaussian reliability:\n',repeat_summary.to_string(index=False))
print('\nParameter recovery:\n',pd.DataFrame(recovery_summary).to_string(index=False))
print('\nSaved:',OUT)
print('CAVEAT: cross-validation selects a descriptive *report* model, not a neural posterior.')
print('High-quality rules are exploratory sensitivity filters, not exclusion criteria.')


=== AIM 1 VALIDATION COMPLETE ===
Participants: 50 | static trials: 4500

Out-of-sample model comparison:
             model  n_subjects  mean_log_loss  median_log_loss  mean_brier
        gaussian3          50       0.459837         0.469594    0.150304
        gaussian4          50       0.460150         0.469594    0.150484
logistic_shifted4          50       0.466821         0.453785    0.147344
logistic_shifted5          50       0.471214         0.476009    0.148197
     logistic_abs          50       0.475640         0.475159    0.154776
       probit_abs          50       0.482936         0.480721    0.154728
  probit_shifted4          50       0.497552         0.509144    0.147349
  probit_shifted5          50       0.506170         0.508294    0.148550
        gaussian2          50       0.865531         0.774020    0.168387

Paired Gaussian-vs-alternatives:
                        comparison  n  mean_log_loss_difference  ci95_low  ci95_high  paired_signflip_p  gaussian_bett

In [77]:

# %% AFTER AIM 1 — IN-RANGE GAUSSIAN CAUSAL TOLERANCE (N = 45)
#
# Place this cell IMMEDIATELY AFTER Aim 1 and BEFORE Aims 2, 3, 4.
#
# PURPOSE:
#   1. Select participants whose Gaussian 50% causal boundaries
#      are within their observed audiovisual disparity range.
#   2. Keep all trials from the 45 eligible participants.
#   3. Automatically filter subject-indexed CSVs loaded by
#      subsequent Aim 2–4 cells using pd.read_csv().
#   4. Preserve the original N=50 Aim 1 CSV files.
#   5. Save an auditable participant inclusion manifest.
#
# IMPORTANT:
#   Run Aims 2–4 in the SAME Jupyter kernel.
#   Run restore_full_sample_csv_reads() before rerunning Aim 1.

from pathlib import Path
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# 1. READ ORIGINAL AIM 1 PHENOTYPE TABLE
# ---------------------------------------------------------------

# Recover the original pandas reader if this cell was already run.
_ORIGINAL_CSV_READER = getattr(
    pd.read_csv,
    '_avt_inrange45_original',
    pd.read_csv
)

_AIM1_FULL_FILE = (
    ROOT
    / 'Individual_differences_analyses'
    / 'Aim_1_gaussian_psychometric_reliability'
    / 'csv_outputs'
    / 'aim1_gaussian_psychometric_subject_parameters.csv'
)

if not _AIM1_FULL_FILE.is_file():
    raise FileNotFoundError(
        f'Run Aim 1 first. Missing:\n{_AIM1_FULL_FILE}'
    )

_full_pheno = _ORIGINAL_CSV_READER(_AIM1_FULL_FILE)

_required = {
    'subject',
    'delta50_deg',
    'boundary50_within_observed_range',
    'fit_success'
}

if not _required.issubset(_full_pheno.columns):
    raise KeyError(
        f'Missing Aim 1 columns: '
        f'{_required - set(_full_pheno.columns)}'
    )

_full_pheno['subject'] = (
    _full_pheno['subject']
    .astype(str)
    .str.strip()
    .str.lower()
)

if _full_pheno['subject'].duplicated().any():
    raise ValueError(
        'Duplicate participant IDs found in Aim 1 phenotype table.'
    )

# ---------------------------------------------------------------
# 2. IDENTIFY PARTICIPANTS WITH IN-RANGE CAUSAL BOUNDARIES
# ---------------------------------------------------------------

# Use the ACTUAL Aim 1 fit-quality flag.
# Do not apply a new arbitrary Delta50 cutoff.

_is_inrange = (
    _full_pheno['boundary50_within_observed_range']
    .astype(str)
    .str.lower()
    .eq('true')
)

_is_good_fit = (
    _full_pheno['fit_success']
    .astype(str)
    .str.lower()
    .eq('true')
)

_has_delta = pd.to_numeric(
    _full_pheno['delta50_deg'],
    errors='coerce'
).notna()

INRANGE_45_SUBJECTS = set(
    _full_pheno.loc[
        _is_inrange & _is_good_fit & _has_delta,
        'subject'
    ]
)

EXCLUDED_OUT_OF_RANGE_SUBJECTS = (
    set(_full_pheno['subject']) - INRANGE_45_SUBJECTS
)

# Fail safely if the expected sample was not reproduced.
if (
    len(_full_pheno) != 50
    or len(INRANGE_45_SUBJECTS) != 45
):
    raise RuntimeError(
        'Expected N=50 in Aim 1 and N=45 with in-range '
        'boundaries.\n'
        f'Found full N={len(_full_pheno)}, '
        f'in-range N={len(INRANGE_45_SUBJECTS)}.\n'
        'Check the Aim 1 fit-quality flags.'
    )

# ---------------------------------------------------------------
# 3. SAVE PARTICIPANT INCLUSION MANIFEST
# ---------------------------------------------------------------

INRANGE45_OUT = (
    ROOT
    / 'Individual_differences_analyses'
    / 'Aim_1_gaussian_psychometric_reliability'
    / 'inrange_N45_reference'
)

INRANGE45_OUT.mkdir(
    parents=True,
    exist_ok=True
)

_manifest = _full_pheno[
    [
        'subject',
        'fit_success',
        'delta50_deg',
        'boundary50_within_observed_range'
    ]
].copy()

_manifest['include_in_aims_2_3_4'] = (
    _manifest['subject'].isin(INRANGE_45_SUBJECTS)
)

_manifest.to_csv(
    INRANGE45_OUT / 'subject_inclusion_manifest.csv',
    index=False
)

_full_pheno[
    _full_pheno.subject.isin(INRANGE_45_SUBJECTS)
].to_csv(
    INRANGE45_OUT / 'gaussian_subject_parameters_N45.csv',
    index=False
)

# ---------------------------------------------------------------
# 4. FILTER EXISTING AIM 1 IN-MEMORY DATAFRAMES
# ---------------------------------------------------------------

# These variables are retained by the original Aim 1 cell.
# Only filter them when a subject column exists.

for _var in (
    'pheno',
    'causal',
    'causal_cf',
    'cf_fits'
):

    if (
        _var in globals()
        and isinstance(globals()[_var], pd.DataFrame)
    ):

        _df = globals()[_var]

        if 'subject' in _df.columns:

            _mask = (
                _df['subject']
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(INRANGE_45_SUBJECTS)
            )

            globals()[_var] = _df.loc[_mask].copy()

# ---------------------------------------------------------------
# 5. AUTOMATICALLY FILTER CSV INPUTS FOR AIMS 2, 3, AND 4
# ---------------------------------------------------------------

# IMPORTANT:
# Subsequent aims may reload their datasets from CSV files.
# Filtering pheno alone would NOT reduce those datasets.
#
# This wrapper applies the N=45 restriction whenever pd.read_csv
# reads a subject-indexed CSV under the AVT project directory.
#
# Original CSVs are NEVER changed.
# The original reader can be restored at any point.

_AVT45_CSV_READ_LOG = []

_AVT45_PROJECT_ROOT = (
    Path(ROOT).expanduser().resolve()
)


def _read_csv_inrange45(filepath, *args, **kwargs):

    # Read the original data normally.
    frame = _ORIGINAL_CSV_READER(
        filepath,
        *args,
        **kwargs
    )

    # Determine whether the file belongs to this project.
    try:

        path = Path(filepath).expanduser().resolve()

        in_project = (
            path == _AVT45_PROJECT_ROOT
            or _AVT45_PROJECT_ROOT in path.parents
        )

    except (TypeError, ValueError, OSError):

        in_project = False
        path = None

    # Apply the sample restriction only to subject-indexed tables.
    if (
        in_project
        and isinstance(frame, pd.DataFrame)
        and 'subject' in frame.columns
    ):

        count_before = len(frame)

        mask = (
            frame['subject']
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(INRANGE_45_SUBJECTS)
        )

        frame = frame.loc[mask].copy()

        removed = count_before - len(frame)

        _AVT45_CSV_READ_LOG.append(
            (
                str(path),
                count_before,
                len(frame),
                removed
            )
        )

        if removed:

            print(
                f'[N45 filter] {path.name}: '
                f'{count_before:,} -> {len(frame):,} rows'
            )

    return frame


# Store the original reader for safe restoration.
_read_csv_inrange45._avt_inrange45_original = (
    _ORIGINAL_CSV_READER
)

# Activate automatic filtering for subsequent cells.
pd.read_csv = _read_csv_inrange45


# ---------------------------------------------------------------
# 6. RESTORE FULL SAMPLE WHEN NEEDED
# ---------------------------------------------------------------

def restore_full_sample_csv_reads():

    """
    Restore standard pandas CSV loading.

    Run this before rerunning Aim 1 or conducting
    any full-sample N=50 analyses.
    """

    pd.read_csv = _ORIGINAL_CSV_READER

    print(
        'Restored pandas.read_csv. '
        'Subsequent CSV loads use the full sample.'
    )


def show_inrange45_read_log():

    """
    Display which datasets were filtered by this cell.
    """

    return pd.DataFrame(
        _AVT45_CSV_READ_LOG,
        columns=[
            'path',
            'rows_original',
            'rows_returned',
            'rows_removed'
        ]
    )


# ---------------------------------------------------------------
# 7. VALIDATION / OUTPUT
# ---------------------------------------------------------------

print('\n' + '=' * 65)
print('IN-RANGE GAUSSIAN CAUSAL-TOLERANCE SAMPLE ACTIVE')
print('=' * 65)

print(
    f'\nOriginal Aim 1 sample: '
    f'{len(_full_pheno)} participants'
)

print(
    f'In-range sample for Aims 2–4: '
    f'{len(INRANGE_45_SUBJECTS)} participants'
)

print('\nExcluded participants:')

for _pid in sorted(EXCLUDED_OUT_OF_RANGE_SUBJECTS):

    print(f'  {_pid}')

print('\nAutomatic filtering is now active.')

print(
    'Subject-indexed CSV files loaded using pd.read_csv() '
    'under ROOT will be restricted to N=45.'
)

print(
    '\nOriginal Aim 1 results have not been modified.'
)

print(
    '\nTo restore full-sample CSV reading, run:'
    '\n    restore_full_sample_csv_reads()'
)

print(
    '\nTo inspect filtered datasets, run:'
    '\n    show_inrange45_read_log()'
)

print(
    '\nParticipant manifest saved to:\n',
    INRANGE45_OUT / 'subject_inclusion_manifest.csv'
)



IN-RANGE GAUSSIAN CAUSAL-TOLERANCE SAMPLE ACTIVE

Original Aim 1 sample: 50 participants
In-range sample for Aims 2–4: 45 participants

Excluded participants:
  pid003
  pid024
  pid029
  pid039
  pid053

Automatic filtering is now active.
Subject-indexed CSV files loaded using pd.read_csv() under ROOT will be restricted to N=45.

Original Aim 1 results have not been modified.

To restore full-sample CSV reading, run:
    restore_full_sample_csv_reads()

To inspect filtered datasets, run:
    show_inrange45_read_log()

Participant manifest saved to:
 C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_1_gaussian_psychometric_reliability\inrange_N45_reference\subject_inclusion_manifest.csv


In [79]:
# %% AIM 2 — DOES THE UPDATED GAUSSIAN CAUSAL PHENOTYPE PREDICT PERCEPTUAL EXPRESSION?
#
# UPDATED QUESTIONS
# -----------------
# 2A. Does explicit causal tolerance (Delta50) predict lag-corrected visual capture?
# 2B. Does Delta50 predict residual visual bias after accounting for current
#     explicit Same/Different report, and does that relation survive on trials
#     explicitly judged Different?
# 2C. Does Delta50 predict auditory tracking lag?
# 2D. Does psychometric SHARPNESS (slope at the 50% boundary) explain variance
#     beyond boundary location?
# 2E. Do Delta50 effects survive adjustment for participant auditory/visual
#     sensory noise (Kalman sqrt(R), when available)?
#
# IMPORTANT INTERPRETATION
# ------------------------
# Delta50 is the half-width of the fitted explicit causal-report function around
# its subjective alignment center mu. It is NOT the Gaussian center and is NOT
# treated as a hard cutoff in implicit spatial perception.
#
# Run revised Aim 1 first.

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings(
    "default",
    category=ConvergenceWarning,
)

# =====================================================================
# PATHS / SETTINGS
# =====================================================================
ROOT = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1"
)

SL_DATA = ROOT / "individual_data_SL"
SPATIAL = ROOT / "spatial_analyses"

AIM1 = (
    ROOT
    / "Individual_differences_analyses"
    / "Aim_1_gaussian_psychometric_reliability"
)

GAUSSIAN_CSV = (
    AIM1
    / "csv_outputs"
    / "aim1_gaussian_psychometric_subject_parameters.csv"
)

TRIAL_METRICS_CSV = (
    SPATIAL
    / "00_core_spatial_metrics"
    / "trial_level_metrics"
    / "cell3_trial_level_lag_corrected_spatial_metrics.csv"
)

STABLE_BIAS_CSV = (
    SPATIAL
    / "05_residual_visual_bias"
    / "trial_level_bias_psame_model_data.csv"
)

VALID_LAG_CSV = (
    SPATIAL
    / "02_same_different_tracking_lag"
    / "csv_outputs"
    / "03_VALID_trial_lag_estimates_after_exact_collaborator_QC.csv"
)

OUT = (
    ROOT
    / "Individual_differences_analyses"
    / "Aim_2_gaussian_perceptual_expression"
)

CSV_OUT = OUT / "csv_outputs"
FIG_OUT = OUT / "figures"
MODEL_OUT = OUT / "model_summaries"
DIAG_OUT = OUT / "model_diagnostics"

for d in [
    CSV_OUT,
    FIG_OUT,
    MODEL_OUT,
    DIAG_OUT,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )

FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]

SINGULAR_EIG_TOL = 1e-7

REQUIRE_50_BOUNDARY_WITHIN_OBSERVED_RANGE = False
EXCLUDE_PARAMETER_BOUND_FITS = False


# =====================================================================
# HELPERS
# =====================================================================
def require_file(
    path,
    label,
):
    path = Path(
        path
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nMissing {label}:\n"
            f"{path}"
        )

    return path


for p, label in [
    (
        GAUSSIAN_CSV,
        "revised Aim-1 Gaussian participant table",
    ),
    (
        TRIAL_METRICS_CSV,
        "core lag-corrected trial metrics",
    ),
    (
        STABLE_BIAS_CSV,
        "stable residual-bias trial table",
    ),
    (
        VALID_LAG_CSV,
        "valid auditory-lag trial table",
    ),
]:

    require_file(
        p,
        label,
    )


def clean_subject(
    s,
):

    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )


def zscore_series(
    x,
):

    x = pd.to_numeric(
        x,
        errors="coerce",
    )

    sd = x.std(
        ddof=0
    )

    return (
        (
            x
            -
            x.mean()
        )
        /
        sd
        if
        np.isfinite(
            sd
        )
        and
        sd
        >
        0
        else
        pd.Series(
            np.nan,
            index=x.index,
        )
    )


def safe_corr(
    x,
    y,
):

    d = (
        pd.DataFrame(
            {
                "x":
                    x,

                "y":
                    y,
            }
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )

    if (
        len(
            d
        )
        <
        4
        or
        d[
            "x"
        ].std(
            ddof=1
        )
        ==
        0
        or
        d[
            "y"
        ].std(
            ddof=1
        )
        ==
        0
    ):

        return {
            "n":
                len(
                    d
                ),

            "pearson_r":
                np.nan,

            "pearson_p":
                np.nan,

            "spearman_rho":
                np.nan,

            "spearman_p":
                np.nan,
        }

    pr = pearsonr(
        d[
            "x"
        ],
        d[
            "y"
        ],
    )

    sr = spearmanr(
        d[
            "x"
        ],
        d[
            "y"
        ],
    )

    return {
        "n":
            len(
                d
            ),

        "pearson_r":
            float(
                pr.statistic
            ),

        "pearson_p":
            float(
                pr.pvalue
            ),

        "spearman_rho":
            float(
                sr.statistic
            ),

        "spearman_p":
            float(
                sr.pvalue
            ),
    }


# =====================================================================
# MIXED MODEL DIAGNOSTICS
# =====================================================================
def _mixedlm_diagnostics(
    fit,
):

    fe_names = list(
        fit.fe_params.index
    )

    bse = fit.bse.reindex(
        fe_names
    )

    pvals = fit.pvalues.reindex(
        fe_names
    )

    converged = bool(
        getattr(
            fit,
            "converged",
            False,
        )
    )

    finite = bool(
        np.isfinite(
            fit.fe_params.to_numpy(
                float
            )
        ).all()
        and
        np.isfinite(
            bse.to_numpy(
                float
            )
        ).all()
        and
        np.isfinite(
            pvals.to_numpy(
                float
            )
        ).all()
    )

    try:

        cov = np.asarray(
            fit.cov_re,
            dtype=float,
        )

        eig = np.linalg.eigvalsh(
            cov
        )

        singular = (
            not np.isfinite(
                eig
            ).all()
            or
            np.linalg.matrix_rank(
                cov,
                tol=SINGULAR_EIG_TOL,
            )
            <
            cov.shape[
                0
            ]
            or
            np.nanmax(
                eig
            )
            <=
            SINGULAR_EIG_TOL
            or
            np.nanmin(
                eig
            )
            <
            SINGULAR_EIG_TOL
        )

        min_eig = float(
            np.nanmin(
                eig
            )
        )

        max_eig = float(
            np.nanmax(
                eig
            )
        )

    except Exception:

        singular = True
        min_eig = np.nan
        max_eig = np.nan

    return {
        "converged":
            converged,

        "finite_fixed_inference":
            finite,

        "singular_random_effects":
            bool(
                singular
            ),

        "min_re_eigenvalue":
            min_eig,

        "max_re_eigenvalue":
            max_eig,

        "usable":
            bool(
                converged
                and
                finite
                and
                not singular
            ),
    }


def fit_repeated_linear(
    formula,
    data,
    re_formula_candidates,
    model_name,
    group="subject",
):

    d = (
        data
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .copy()
    )

    diagnostics = []


    for re_formula in re_formula_candidates:

        for method in [
            "lbfgs",
            "powell",
            "cg",
        ]:

            try:

                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always"
                    )

                    fit = smf.mixedlm(
                        formula,
                        data=d,
                        groups=d[
                            group
                        ],
                        re_formula=re_formula,
                    ).fit(
                        reml=False,
                        method=method,
                        maxiter=5000,
                        disp=False,
                    )


                diag = _mixedlm_diagnostics(
                    fit
                )


                diag.update(
                    {
                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "warnings":
                            " | ".join(
                                str(
                                    w.message
                                )
                                for w in caught
                            ),

                        "n_rows":
                            len(
                                d
                            ),

                        "n_subjects":
                            d[
                                group
                            ].nunique(),
                    }
                )


                diagnostics.append(
                    diag
                )


                if diag[
                    "usable"
                ]:

                    out = pd.DataFrame(
                        diagnostics
                    )

                    out.to_csv(
                        DIAG_OUT
                        /
                        f"{model_name}_fit_diagnostics.csv",
                        index=False,
                    )

                    return (
                        fit,
                        (
                            f"MixedLM["
                            f"{re_formula}; "
                            f"{method}]"
                        ),
                        out,
                    )


            except Exception as exc:

                diagnostics.append(
                    {
                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "usable":
                            False,

                        "error":
                            repr(
                                exc
                            ),

                        "n_rows":
                            len(
                                d
                            ),

                        "n_subjects":
                            d[
                                group
                            ].nunique(),
                    }
                )


    try:

        fit = smf.gee(
            formula,
            groups=group,
            data=d,
            family=sm.families.Gaussian(),
            cov_struct=Exchangeable(),
        ).fit()


        diagnostics.append(
            {
                "attempt":
                    "GEE",

                "usable":
                    True,

                "n_rows":
                    len(
                        d
                    ),

                "n_subjects":
                    d[
                        group
                    ].nunique(),
            }
        )


        out = pd.DataFrame(
            diagnostics
        )


        out.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            fit,
            "GEE_Gaussian_Exchangeable",
            out,
        )


    except Exception as exc:

        fit = smf.ols(
            formula,
            data=d,
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups":
                    d[
                        group
                    ],
            },
        )


        diagnostics.append(
            {
                "attempt":
                    "OLS_cluster",

                "usable":
                    True,

                "error_before_fallback":
                    repr(
                        exc
                    ),

                "n_rows":
                    len(
                        d
                    ),

                "n_subjects":
                    d[
                        group
                    ].nunique(),
            }
        )


        out = pd.DataFrame(
            diagnostics
        )


        out.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            fit,
            "OLS_cluster_fallback",
            out,
        )


def fixed_terms(
    fit,
):

    return (
        list(
            fit.fe_params.index
        )
        if
        hasattr(
            fit,
            "fe_params",
        )
        else
        list(
            fit.params.index
        )
    )


def save_model(
    fit,
    name,
    fit_type,
    diagnostics=None,
):

    with open(
        MODEL_OUT
        /
        f"{name}.txt",
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            f"Fit type: "
            f"{fit_type}\n\n"
        )

        if diagnostics is not None:

            f.write(
                "FIT DIAGNOSTICS\n"
            )

            f.write(
                diagnostics.to_string(
                    index=False
                )
            )

            f.write(
                "\n\nMODEL SUMMARY\n"
            )

        f.write(
            str(
                fit.summary()
            )
        )


    terms = fixed_terms(
        fit
    )


    ci = fit.conf_int().reindex(
        terms
    )


    tab = pd.DataFrame(
        {
            "term":
                terms,

            "estimate":
                fit.params.reindex(
                    terms
                ).values,

            "se":
                fit.bse.reindex(
                    terms
                ).values,

            "stat":
                fit.tvalues.reindex(
                    terms
                ).values,

            "p":
                fit.pvalues.reindex(
                    terms
                ).values,

            "ci95_low":
                ci.iloc[
                    :,
                    0
                ].values,

            "ci95_high":
                ci.iloc[
                    :,
                    1
                ].values,

            "fit_type":
                fit_type,
        }
    )


    tab.to_csv(
        CSV_OUT
        /
        f"{name}_coefficients.csv",
        index=False,
    )


    return tab


def get_term_row(
    tab,
    term,
):

    x = tab[
        tab[
            "term"
        ].eq(
            term
        )
    ]

    if x.empty:

        return {
            "estimate":
                np.nan,

            "se":
                np.nan,

            "stat":
                np.nan,

            "p":
                np.nan,

            "ci95_low":
                np.nan,

            "ci95_high":
                np.nan,
        }

    return x.iloc[
        0
    ].to_dict()


def plot_subject_relation(
    df,
    x,
    y,
    xlabel,
    ylabel,
    title,
    filename,
):

    d = (
        df[
            [
                x,
                y,
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .copy()
    )

    if len(
        d
    ) < 4:
        return


    fit = smf.ols(
        f"{y} ~ {x}",
        data=d,
    ).fit(
        cov_type="HC3"
    )


    xx = np.linspace(
        d[
            x
        ].min(),
        d[
            x
        ].max(),
        150,
    )


    pred = fit.get_prediction(
        pd.DataFrame(
            {
                x:
                    xx,
            }
        )
    ).summary_frame(
        alpha=.05
    )


    corr = safe_corr(
        d[
            x
        ],
        d[
            y
        ],
    )


    fig, ax = plt.subplots(
        figsize=(
            6.6,
            5.2,
        )
    )


    ax.scatter(
        d[
            x
        ],
        d[
            y
        ],
        alpha=.72,
    )


    ax.plot(
        xx,
        pred[
            "mean"
        ],
        linewidth=2,
    )


    ax.fill_between(
        xx,
        pred[
            "mean_ci_lower"
        ].to_numpy(),
        pred[
            "mean_ci_upper"
        ].to_numpy(),
        alpha=.18,
    )


    ax.set_xlabel(
        xlabel
    )


    ax.set_ylabel(
        ylabel
    )


    ax.set_title(
        title
    )


    ax.text(
        .04,
        .96,
        (
            f"n={corr['n']}\n"
            f"r={corr['pearson_r']:.3f}, "
            f"p={corr['pearson_p']:.4g}\n"
            f"HC3 beta={fit.params[x]:.4f}, "
            f"p={fit.pvalues[x]:.4g}"
        ),
        transform=ax.transAxes,
        va="top",
    )


    fig.tight_layout()


    fig.savefig(
        FIG_OUT
        /
        filename,
        dpi=220,
        bbox_inches="tight",
    )


    plt.show()


# =====================================================================
# LOAD UPDATED GAUSSIAN PHENOTYPE
# =====================================================================
pheno = pd.read_csv(
    GAUSSIAN_CSV
)


pheno[
    "subject"
] = clean_subject(
    pheno[
        "subject"
    ]
)


for c in [
    "delta50_deg",
    "mu_deg",
    "causal_sigma_deg",
    "slope50_per_deg",
    "peak_same",
    "floor_same",
    "sensory_noise_A_deg",
    "sensory_noise_V_deg",
]:

    pheno[
        c
    ] = pd.to_numeric(
        pheno[
            c
        ],
        errors="coerce",
    )


pheno = pheno[
    pheno[
        "delta50_deg"
    ].notna()
].copy()


if REQUIRE_50_BOUNDARY_WITHIN_OBSERVED_RANGE:

    pheno = pheno[
        pheno[
            "boundary50_within_observed_range"
        ].astype(
            bool
        )
    ].copy()


if EXCLUDE_PARAMETER_BOUND_FITS:

    pheno = pheno[
        ~pheno[
            "fit_on_parameter_bound"
        ].astype(
            bool
        )
    ].copy()


pheno[
    "delta50_c"
] = (
    pheno[
        "delta50_deg"
    ]
    -
    pheno[
        "delta50_deg"
    ].mean()
)


pheno[
    "slope50_z"
] = zscore_series(
    pheno[
        "slope50_per_deg"
    ]
)


pheno[
    "sigma_curve_z"
] = zscore_series(
    pheno[
        "causal_sigma_deg"
    ]
)


pheno[
    "noiseA_z"
] = zscore_series(
    pheno[
        "sensory_noise_A_deg"
    ]
)


pheno[
    "noiseV_z"
] = zscore_series(
    pheno[
        "sensory_noise_V_deg"
    ]
)


pheno.to_csv(
    CSV_OUT
    /
    "aim2_gaussian_predictor_table.csv",
    index=False,
)


# =====================================================================
# 2A. DELTA50 / CURVE SHAPE -> LAG-CORRECTED VISUAL CAPTURE
# =====================================================================
cap = pd.read_csv(
    TRIAL_METRICS_CSV
)


cap[
    "subject"
] = clean_subject(
    cap[
        "subject"
    ]
)


cap[
    "offset_deg"
] = pd.to_numeric(
    cap[
        "offsetDeg"
    ],
    errors="coerce",
).abs()


cap[
    "VisualCapture_audioLag"
] = pd.to_numeric(
    cap[
        "VisualCapture_audioLag"
    ],
    errors="coerce",
)


sweep_flag = (
    cap.get(
        "trialType",
        pd.Series(
            "",
            index=cap.index,
        ),
    )
    .astype(str)
    .str.lower()
    .str.contains(
        "sweep"
    )
    |
    cap.get(
        "condLabel",
        pd.Series(
            "",
            index=cap.index,
        ),
    )
    .astype(str)
    .str.lower()
    .str.contains(
        "sweep|diverge|converge",
        regex=True,
    )
)


cap = cap[
    cap[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
    &
    ~sweep_flag
].copy()


cap = cap.merge(
    pheno,
    on="subject",
    how="inner",
)


cap = cap.dropna(
    subset=[
        "VisualCapture_audioLag",
        "offset_deg",
        "delta50_c",
    ]
).copy()


cap[
    "offset_c"
] = (
    cap[
        "offset_deg"
    ]
    -
    np.mean(
        FIXED_OFFSETS
    )
)


cap[
    "offset_z"
] = zscore_series(
    cap[
        "offset_deg"
    ]
)


cap_loc, cap_loc_type, cap_loc_diag = fit_repeated_linear(
    (
        "VisualCapture_audioLag ~ "
        "C(offset_deg) + delta50_c"
    ),
    cap,
    [
        "1 + offset_z",
        "1",
    ],
    "aim2A_capture_delta50",
)


cap_loc_tab = save_model(
    cap_loc,
    "aim2A_capture_delta50",
    cap_loc_type,
    cap_loc_diag,
)


cap_shape, cap_shape_type, cap_shape_diag = fit_repeated_linear(
    (
        "VisualCapture_audioLag ~ "
        "C(offset_deg) + delta50_c + slope50_z"
    ),
    cap.dropna(
        subset=[
            "slope50_z",
        ]
    ),
    [
        "1 + offset_z",
        "1",
    ],
    "aim2A_capture_delta50_plus_slope",
)


cap_shape_tab = save_model(
    cap_shape,
    "aim2A_capture_delta50_plus_slope",
    cap_shape_type,
    cap_shape_diag,
)


cap_int, cap_int_type, cap_int_diag = fit_repeated_linear(
    (
        "VisualCapture_audioLag ~ "
        "C(offset_deg) + delta50_c + "
        "offset_c:delta50_c"
    ),
    cap,
    [
        "1 + offset_z",
        "1",
    ],
    "aim2A_capture_delta50_by_offset",
)


cap_int_tab = save_model(
    cap_int,
    "aim2A_capture_delta50_by_offset",
    cap_int_type,
    cap_int_diag,
)


cap_noise = cap.dropna(
    subset=[
        "slope50_z",
        "noiseA_z",
        "noiseV_z",
    ]
).copy()


if (
    cap_noise[
        "subject"
    ].nunique()
    >=
    20
):

    cap_adj, cap_adj_type, cap_adj_diag = fit_repeated_linear(
        (
            "VisualCapture_audioLag ~ "
            "C(offset_deg) + delta50_c + "
            "slope50_z + noiseA_z + noiseV_z"
        ),
        cap_noise,
        [
            "1 + offset_z",
            "1",
        ],
        "aim2A_capture_sensory_noise_adjusted",
    )


    cap_adj_tab = save_model(
        cap_adj,
        "aim2A_capture_sensory_noise_adjusted",
        cap_adj_type,
        cap_adj_diag,
    )

else:

    cap_adj = None
    cap_adj_tab = pd.DataFrame()


cap_subj = (
    cap.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        mean_capture=(
            "VisualCapture_audioLag",
            "mean",
        ),

        delta50_deg=(
            "delta50_deg",
            "first",
        ),

        slope50_per_deg=(
            "slope50_per_deg",
            "first",
        ),
    )
)


cap_subj.to_csv(
    CSV_OUT
    /
    "aim2A_subject_mean_capture_gaussian_predictors.csv",
    index=False,
)


plot_subject_relation(
    cap_subj,
    "delta50_deg",
    "mean_capture",
    "Explicit causal tolerance, Delta50 (deg)",
    "Mean lag-corrected normalized visual capture",
    "Explicit causal tolerance predicts spatial weighting",
    "aim2A_delta50_vs_mean_capture.png",
)


# =====================================================================
# 2B. DELTA50 / CURVE SHAPE -> RESIDUAL VISUAL BIAS
# =====================================================================
bias = pd.read_csv(
    STABLE_BIAS_CSV
)


bias[
    "subject"
] = clean_subject(
    bias[
        "subject"
    ]
)


bias[
    "offset_num"
] = pd.to_numeric(
    bias[
        "offset_num"
    ],
    errors="coerce",
).abs()


bias[
    "Psame"
] = pd.to_numeric(
    bias[
        "Psame"
    ],
    errors="coerce",
)


bias[
    "Bias_lagcorr_AminusV_stable"
] = pd.to_numeric(
    bias[
        "Bias_lagcorr_AminusV_stable"
    ],
    errors="coerce",
)


bias = bias[
    bias[
        "offset_num"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


bias = bias.merge(
    pheno,
    on="subject",
    how="inner",
)


bias = bias.dropna(
    subset=[
        "Bias_lagcorr_AminusV_stable",
        "Psame",
        "delta50_c",
    ]
).copy()


bias[
    "Psame_c"
] = (
    bias[
        "Psame"
    ]
    -
    bias[
        "Psame"
    ].mean()
)


bias[
    "offset_z"
] = zscore_series(
    bias[
        "offset_num"
    ]
)


bias_model, bias_type, bias_diag = fit_repeated_linear(
    (
        "Bias_lagcorr_AminusV_stable ~ "
        "C(offset_num) + Psame_c + "
        "delta50_c + slope50_z + "
        "Psame_c:delta50_c"
    ),
    bias.dropna(
        subset=[
            "slope50_z",
        ]
    ),
    [
        "1 + offset_z + Psame_c",
        "1 + offset_z",
        "1 + Psame_c",
        "1",
    ],
    "aim2B_bias_delta50_by_judgment",
)


bias_tab = save_model(
    bias_model,
    "aim2B_bias_delta50_by_judgment",
    bias_type,
    bias_diag,
)


diff = bias[
    np.isclose(
        bias[
            "Psame"
        ],
        0.0,
    )
].copy()


if (
    len(
        diff
    )
    >=
    30
    and
    diff[
        "subject"
    ].nunique()
    >=
    15
):

    diff_model, diff_type, diff_diag = fit_repeated_linear(
        (
            "Bias_lagcorr_AminusV_stable ~ "
            "C(offset_num) + delta50_c + slope50_z"
        ),
        diff.dropna(
            subset=[
                "slope50_z",
            ]
        ),
        [
            "1 + offset_z",
            "1",
        ],
        "aim2B_different_only_bias_gaussian_predictors",
    )


    diff_tab = save_model(
        diff_model,
        "aim2B_different_only_bias_gaussian_predictors",
        diff_type,
        diff_diag,
    )

else:

    diff_model = None
    diff_tab = pd.DataFrame()


bias_noise = bias.dropna(
    subset=[
        "slope50_z",
        "noiseA_z",
        "noiseV_z",
    ]
).copy()


if (
    bias_noise[
        "subject"
    ].nunique()
    >=
    20
):

    bias_adj, bias_adj_type, bias_adj_diag = fit_repeated_linear(
        (
            "Bias_lagcorr_AminusV_stable ~ "
            "C(offset_num) + Psame_c + "
            "delta50_c + slope50_z + "
            "Psame_c:delta50_c + "
            "noiseA_z + noiseV_z"
        ),
        bias_noise,
        [
            "1 + offset_z + Psame_c",
            "1 + offset_z",
            "1",
        ],
        "aim2B_bias_sensory_noise_adjusted",
    )


    bias_adj_tab = save_model(
        bias_adj,
        "aim2B_bias_sensory_noise_adjusted",
        bias_adj_type,
        bias_adj_diag,
    )

else:

    bias_adj = None
    bias_adj_tab = pd.DataFrame()


diff_subj = (
    diff.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        mean_different_bias=(
            "Bias_lagcorr_AminusV_stable",
            "mean",
        ),

        n_different_trials=(
            "Bias_lagcorr_AminusV_stable",
            "count",
        ),

        delta50_deg=(
            "delta50_deg",
            "first",
        ),
    )
)


diff_subj.to_csv(
    CSV_OUT
    /
    "aim2B_subject_different_only_bias.csv",
    index=False,
)


plot_subject_relation(
    diff_subj,
    "delta50_deg",
    "mean_different_bias",
    "Explicit causal tolerance, Delta50 (deg)",
    "Mean visual bias on Different-source trials (deg)",
    "Implicit spatial attraction after explicit segregation",
    "aim2B_delta50_vs_different_only_bias.png",
)


# =====================================================================
# 2C. DELTA50 / CURVE SHAPE -> AUDITORY TRACKING LAG
# =====================================================================
lag = pd.read_csv(
    VALID_LAG_CSV
)


lag[
    "subject"
] = clean_subject(
    lag[
        "subject"
    ]
)


lag[
    "offset_deg"
] = pd.to_numeric(
    lag[
        "offset_deg"
    ],
    errors="coerce",
).abs()


lag[
    "lag_audio_sec"
] = pd.to_numeric(
    lag[
        "lag_audio_sec"
    ],
    errors="coerce",
)


lag = lag[
    lag[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


lag = lag.merge(
    pheno,
    on="subject",
    how="inner",
)


lag = lag.dropna(
    subset=[
        "lag_audio_sec",
        "delta50_c",
    ]
).copy()


lag[
    "offset_c"
] = (
    lag[
        "offset_deg"
    ]
    -
    np.mean(
        FIXED_OFFSETS
    )
)


lag[
    "offset_z"
] = zscore_series(
    lag[
        "offset_deg"
    ]
)


lag_loc, lag_loc_type, lag_loc_diag = fit_repeated_linear(
    (
        "lag_audio_sec ~ "
        "C(offset_deg) + delta50_c"
    ),
    lag,
    [
        "1 + offset_z",
        "1",
    ],
    "aim2C_lag_delta50",
)


lag_loc_tab = save_model(
    lag_loc,
    "aim2C_lag_delta50",
    lag_loc_type,
    lag_loc_diag,
)


lag_shape, lag_shape_type, lag_shape_diag = fit_repeated_linear(
    (
        "lag_audio_sec ~ "
        "C(offset_deg) + delta50_c + slope50_z"
    ),
    lag.dropna(
        subset=[
            "slope50_z",
        ]
    ),
    [
        "1 + offset_z",
        "1",
    ],
    "aim2C_lag_delta50_plus_slope",
)


lag_shape_tab = save_model(
    lag_shape,
    "aim2C_lag_delta50_plus_slope",
    lag_shape_type,
    lag_shape_diag,
)


lag_int, lag_int_type, lag_int_diag = fit_repeated_linear(
    (
        "lag_audio_sec ~ "
        "C(offset_deg) + delta50_c + "
        "offset_c:delta50_c"
    ),
    lag,
    [
        "1 + offset_z",
        "1",
    ],
    "aim2C_lag_delta50_by_offset",
)


lag_int_tab = save_model(
    lag_int,
    "aim2C_lag_delta50_by_offset",
    lag_int_type,
    lag_int_diag,
)


lag_noise = lag.dropna(
    subset=[
        "slope50_z",
        "noiseA_z",
        "noiseV_z",
    ]
).copy()


if (
    lag_noise[
        "subject"
    ].nunique()
    >=
    20
):

    lag_adj, lag_adj_type, lag_adj_diag = fit_repeated_linear(
        (
            "lag_audio_sec ~ "
            "C(offset_deg) + delta50_c + "
            "slope50_z + noiseA_z + noiseV_z"
        ),
        lag_noise,
        [
            "1 + offset_z",
            "1",
        ],
        "aim2C_lag_sensory_noise_adjusted",
    )


    lag_adj_tab = save_model(
        lag_adj,
        "aim2C_lag_sensory_noise_adjusted",
        lag_adj_type,
        lag_adj_diag,
    )

else:

    lag_adj = None
    lag_adj_tab = pd.DataFrame()


lag_subj = (
    lag.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        mean_lag_sec=(
            "lag_audio_sec",
            "mean",
        ),

        delta50_deg=(
            "delta50_deg",
            "first",
        ),

        slope50_per_deg=(
            "slope50_per_deg",
            "first",
        ),
    )
)


lag_subj.to_csv(
    CSV_OUT
    /
    "aim2C_subject_mean_lag_gaussian_predictors.csv",
    index=False,
)


plot_subject_relation(
    lag_subj,
    "delta50_deg",
    "mean_lag_sec",
    "Explicit causal tolerance, Delta50 (deg)",
    "Mean auditory tracking lag (s)",
    "Explicit causal tolerance predicts temporal tracking",
    "aim2C_delta50_vs_mean_lag.png",
)


# =====================================================================
# COMPACT PRIMARY TEST FAMILY + BH-FDR
# =====================================================================
summary_rows = []


def add_summary(
    analysis,
    model_tab,
    term,
    fit_type,
):

    r = get_term_row(
        model_tab,
        term,
    )

    summary_rows.append(
        {
            "analysis":
                analysis,

            "term":
                term,

            "fit_type":
                fit_type,

            "estimate":
                r.get(
                    "estimate",
                    np.nan,
                ),

            "se":
                r.get(
                    "se",
                    np.nan,
                ),

            "stat":
                r.get(
                    "stat",
                    np.nan,
                ),

            "p":
                r.get(
                    "p",
                    np.nan,
                ),

            "ci95_low":
                r.get(
                    "ci95_low",
                    np.nan,
                ),

            "ci95_high":
                r.get(
                    "ci95_high",
                    np.nan,
                ),
        }
    )


add_summary(
    "2A capture — causal tolerance",
    cap_loc_tab,
    "delta50_c",
    cap_loc_type,
)


add_summary(
    "2A capture — psychometric sharpness",
    cap_shape_tab,
    "slope50_z",
    cap_shape_type,
)


add_summary(
    "2A capture — disparity x tolerance",
    cap_int_tab,
    "offset_c:delta50_c",
    cap_int_type,
)


add_summary(
    "2B bias — causal tolerance",
    bias_tab,
    "delta50_c",
    bias_type,
)


add_summary(
    "2B bias — judgment x tolerance",
    bias_tab,
    "Psame_c:delta50_c",
    bias_type,
)


add_summary(
    "2B bias — psychometric sharpness",
    bias_tab,
    "slope50_z",
    bias_type,
)


if not diff_tab.empty:

    add_summary(
        "2B Different-only bias — causal tolerance",
        diff_tab,
        "delta50_c",
        diff_type,
    )


add_summary(
    "2C lag — causal tolerance",
    lag_loc_tab,
    "delta50_c",
    lag_loc_type,
)


add_summary(
    "2C lag — psychometric sharpness",
    lag_shape_tab,
    "slope50_z",
    lag_shape_type,
)


add_summary(
    "2C lag — disparity x tolerance",
    lag_int_tab,
    "offset_c:delta50_c",
    lag_int_type,
)


summary = pd.DataFrame(
    summary_rows
)


valid = summary[
    "p"
].notna()


summary[
    "p_FDR_BH"
] = np.nan


if valid.any():

    summary.loc[
        valid,
        "p_FDR_BH",
    ] = multipletests(
        summary.loc[
            valid,
            "p",
        ].to_numpy(
            float
        ),
        method="fdr_bh",
    )[
        1
    ]


summary.to_csv(
    CSV_OUT
    /
    "aim2_primary_tests_summary_gaussian.csv",
    index=False,
)


# =====================================================================
# SENSORY-PRECISION ADJUSTED SENSITIVITY RESULTS
# =====================================================================
adj_rows = []


for (
    analysis,
    tab,
    fit_type,
) in [
    (
        "2A capture sensory-adjusted",
        cap_adj_tab,
        (
            cap_adj_type
            if
            cap_adj is not None
            else
            "not_run"
        ),
    ),
    (
        "2B bias sensory-adjusted",
        bias_adj_tab,
        (
            bias_adj_type
            if
            bias_adj is not None
            else
            "not_run"
        ),
    ),
    (
        "2C lag sensory-adjusted",
        lag_adj_tab,
        (
            lag_adj_type
            if
            lag_adj is not None
            else
            "not_run"
        ),
    ),
]:

    if not tab.empty:

        for term in [
            "delta50_c",
            "slope50_z",
            "noiseA_z",
            "noiseV_z",
        ]:

            if term in tab[
                "term"
            ].values:

                r = get_term_row(
                    tab,
                    term,
                )

                adj_rows.append(
                    {
                        "analysis":
                            analysis,

                        "fit_type":
                            fit_type,

                        "term":
                            term,

                        "estimate":
                            r.get(
                                "estimate",
                                np.nan,
                            ),

                        "se":
                            r.get(
                                "se",
                                np.nan,
                            ),

                        "stat":
                            r.get(
                                "stat",
                                np.nan,
                            ),

                        "p":
                            r.get(
                                "p",
                                np.nan,
                            ),

                        "ci95_low":
                            r.get(
                                "ci95_low",
                                np.nan,
                            ),

                        "ci95_high":
                            r.get(
                                "ci95_high",
                                np.nan,
                            ),
                    }
                )


pd.DataFrame(
    adj_rows
).to_csv(
    CSV_OUT
    /
    "aim2_sensory_precision_adjusted_sensitivity_summary.csv",
    index=False,
)


print(
    "\nAIM 2 COMPLETE"
)


print(
    "\nPrimary Gaussian-phenotype tests:"
)


print(
    summary.to_string(
        index=False
    )
)


print(
    "\nInterpretive reminder: "
    "Delta50 is explicit causal-tolerance half-width; "
    "mu is the curve center."
)


print(
    f"\nSaved outputs to:\n{OUT}"
)

[N45 filter] aim1_gaussian_psychometric_subject_parameters.csv: 50 -> 45 rows
[N45 filter] cell3_trial_level_lag_corrected_spatial_metrics.csv: 6,500 -> 5,850 rows
[N45 filter] trial_level_bias_psame_model_data.csv: 4,000 -> 3,600 rows
[N45 filter] 03_VALID_trial_lag_estimates_after_exact_collaborator_QC.csv: 3,274 -> 2,919 rows

AIM 2 COMPLETE

Primary Gaussian-phenotype tests:
                                 analysis               term                               fit_type      estimate       se      stat        p  ci95_low  ci95_high  p_FDR_BH
            2A capture — causal tolerance          delta50_c          MixedLM[1 + offset_z; powell]  9.581447e-03 0.002023  4.735638 0.000002  0.005616   0.013547  0.000022
      2A capture — psychometric sharpness          slope50_z          MixedLM[1 + offset_z; powell] -2.617113e-02 0.016238 -1.611720 0.107023 -0.057997   0.005655  0.267557
       2A capture — disparity x tolerance offset_c:delta50_c          MixedLM[1 + offset_z; powell]

In [96]:
# %% AIM 3 — EXPLICIT CAUSAL PHENOTYPES, IMPLICIT CAPTURE AND DYNAMIC BCI
# Self-contained replacement for the original Aim-3 cell.
# Keeps Aim 3A/3B/3C (legacy descriptive half-decay)/3D and original export names.
# Adds model posterior at 20°, equal-cell posterior average, posterior AUC,
# zero-crossings with identification flags, sweep-history crossings/hysteresis,
# calibration against independent-of-model-fit SL post-trial reports, parameter QC.
# No BCI posterior is treated as a fitted behavioral report probability.
# Existing pandas N45 monkey-patch is explicitly bypassed; sample is filtered here.

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit, logit
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, log_loss, roc_auc_score

ROOT = Path(r'C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1')
SPATIAL = ROOT/'spatial_analyses'
AIM1 = ROOT/'Individual_differences_analyses'/'Aim_1_gaussian_psychometric_reliability'/'csv_outputs'
GAUSSIAN_CSV = AIM1/'aim1_gaussian_psychometric_subject_parameters.csv'
CAUSAL_TRIAL_CSV = AIM1/'aim1_static_causal_trial_table_with_crossfit_predictions.csv'
TRIAL_METRICS_CSV = SPATIAL/'00_core_spatial_metrics'/'trial_level_metrics'/'cell3_trial_level_lag_corrected_spatial_metrics.csv'
BREAKPOINT_CSV = SPATIAL/'06_stabilization_breakpoint'/'csv_outputs'/'anchor_trial_level_breakpoint_DYNAMIC_metrics.csv'
BCI = ROOT/'dynamic BCI'
BCI_TIME_CSV = BCI/'model_timeresolved_SL_20Hz.csv'
BCI_PARAM_CSV = BCI/'dynamic'/'parameter_estimates.csv'
BCI_RESTART_CSV = BCI/'dynamic'/'parameter_estimates_all.csv'
BCI_LAG_CSV = BCI/'dynamic'/'lag_test.csv'

# Trial/window-level behavioral spatial hysteresis used previously in Aim 1.
# Positive NormalizedVisualBias_window = visual-directed spatial capture.
HYST_WINDOW_CSV = SPATIAL/'07_spatial_hysteresis'/'spatial_hysteresis_window_level_bias_by_disparity.csv'
HYST_EMPIRICAL_COLUMN = 'NormalizedVisualBias_window'

# Optional: override above path if your manuscript-level window table lives elsewhere.
# Do NOT substitute a participant-level already-averaged summary: this code deliberately
# computes signed matched-disparity, trial-first hysteresis from raw window records.
MIN_MATCHED_HYST_BINS = 8  # of sixteen 2-degree bins centered 5,7,...35

# Held-out fit R^2 thresholds are diagnostic sensitivity filters, not reliability proofs.
FIT_R2_MINIMAL = 0.0
FIT_R2_STRICT = 0.50

COHORT = 'inrange45'      # 'inrange45' (sensitivity) or 'full50' (all fitted Aim 1 subjects)
WINDOW_MAIN = (2.0, 15.0)
WINDOW_LATE = (12.0, 15.0)
N_BOOT, N_PERM, SEED = 2000, 10000, 20261011
EPS = 1e-6
OFFSETS = [10.,20.,30.,40.]
ALL_OFFSETS = [0.,10.,20.,30.,40.]
SWEEP_BINS = np.arange(5.,35.01,2.)
OUT = ROOT/'Individual_differences_analyses'/f'Aim_3_gaussian_explicit_implicit_mapping_{COHORT}_dynamicBCI'
CSV_OUT, FIG_OUT, MODEL_OUT = OUT/'csv_outputs', OUT/'figures', OUT/'model_summaries'
for p in (CSV_OUT,FIG_OUT,MODEL_OUT): p.mkdir(parents=True,exist_ok=True)

# Previously installed N45 wrapper filters 'pid001_001' incorrectly. Bypass it.
READ = getattr(pd.read_csv, '_avt_inrange45_original', pd.read_csv)

def check_file(path,required=True):
    if not path.is_file():
        if required: raise FileNotFoundError(f'Missing required file: {path}')
        warnings.warn(f'Optional file absent: {path}')
        return False
    return True

for p in [GAUSSIAN_CSV,CAUSAL_TRIAL_CSV,TRIAL_METRICS_CSV,BREAKPOINT_CSV,BCI_TIME_CSV,BCI_PARAM_CSV]:
    check_file(p)

def pid(x):
    return x.astype(str).str.extract(r'(pid\d+)',expand=False).str.lower()

def num(df,cols):
    for c in cols:
        if c in df: df[c]=pd.to_numeric(df[c],errors='coerce')
    return df

def require_columns(df,cols,label):
    absent=set(cols)-set(df.columns)
    if absent: raise KeyError(f'{label} missing columns {sorted(absent)}')

def persist(df,name):
    df.to_csv(CSV_OUT/name,index=False)
    return df

def result_table(res,name):
    (MODEL_OUT/f'{name}.txt').write_text(str(res.summary()),encoding='utf-8')
    ci=res.conf_int()
    tbl=pd.DataFrame({'term':res.params.index,'estimate':res.params.values,
        'se':res.bse.reindex(res.params.index).values,
        'stat':res.tvalues.reindex(res.params.index).values,
        'p':res.pvalues.reindex(res.params.index).values,
        'ci95_low':ci.iloc[:,0].reindex(res.params.index).values,
        'ci95_high':ci.iloc[:,1].reindex(res.params.index).values})
    return persist(tbl,f'{name}_coefficients.csv')

def test_signflip(d,seed=SEED,nperm=N_PERM):
    v=np.asarray(d,dtype=float); v=v[np.isfinite(v)]
    if len(v)<3: return {'n':len(v),'mean_improvement':np.nan,'median_improvement':np.nan,
                            'prop_gaussian_better':np.nan,'signflip_p':np.nan}
    rng=np.random.default_rng(seed); obs=v.mean(); count=0
    for signs in np.array_split(rng.choice([-1.,1.],size=(nperm,len(v))), max(1,int(np.ceil(nperm/1000)))):
        count+=np.sum(np.abs((signs*v).mean(axis=1))>=abs(obs))
    return {'n':len(v),'mean_improvement':obs,'median_improvement':float(np.median(v)),
            'prop_gaussian_better':float((v>0).mean()),'signflip_p':(1+count)/(1+nperm)}

def correlations(df,x,y,label,nboot=N_BOOT,seed=SEED):
    dd=df[[x,y]].replace([np.inf,-np.inf],np.nan).dropna()
    row={'comparison':label,'x':x,'y':y,'n':len(dd),'pearson_r':np.nan,'pearson_p':np.nan,
         'spearman_rho':np.nan,'spearman_p':np.nan,'pearson_ci_low':np.nan,'pearson_ci_high':np.nan,
         'permutation_p':np.nan}
    if len(dd)<5 or min(dd[x].std(),dd[y].std())<=1e-12: return row
    a,b=dd[x].to_numpy(float),dd[y].to_numpy(float)
    r,p=pearsonr(a,b);rho,sp=spearmanr(a,b)
    rng=np.random.default_rng(seed);boots=[];n=len(a);count=0
    for _ in range(nboot):
        ind=rng.integers(0,n,n)
        if np.std(a[ind])>1e-10 and np.std(b[ind])>1e-10:
            boots.append(np.corrcoef(a[ind],b[ind])[0,1])
    for _ in range(N_PERM):
        if abs(np.corrcoef(a,rng.permutation(b))[0,1])>=abs(r): count+=1
    row.update(pearson_r=r,pearson_p=p,spearman_rho=rho,spearman_p=sp,
               pearson_ci_low=np.percentile(boots,2.5) if boots else np.nan,
               pearson_ci_high=np.percentile(boots,97.5) if boots else np.nan,
               permutation_p=(1+count)/(1+N_PERM))
    return row

def halfdecay(offset,capture):
    v=pd.DataFrame({'d':offset,'c':capture}).dropna().groupby('d').c.mean().reindex(OFFSETS).dropna()
    if len(v)<3:return np.nan
    x=v.index.to_numpy(float); y=np.minimum.accumulate(v.to_numpy(float))
    if abs(y[0]-y[-1])<1e-9:return np.nan
    target=(y[0]+y[-1])/2
    for i in range(len(x)-1):
        if y[i]>=target>=y[i+1] and y[i]>y[i+1]:
            return float(np.interp(target,[y[i+1],y[i]],[x[i+1],x[i]]))
    return np.nan

def find_crossing(x,p,criterion=.5):
    """Never manufacture crossings: at most one descending interpolated intersection."""
    q=pd.DataFrame({'d':x,'p':p}).replace([np.inf,-np.inf],np.nan).dropna().sort_values('d')
    q=q.drop_duplicates('d')
    out={'boundary_deg':np.nan,'boundary_status':'insufficient','n_crossings':0,
         'p_low':np.nan,'p_high':np.nan,'min_sampled_deg':np.nan,'max_sampled_deg':np.nan}
    if len(q)<2:return out
    d=q.d.to_numpy(float); z=q.p.to_numpy(float); eps=1e-9
    out.update(p_low=z[0],p_high=z[-1],min_sampled_deg=d[0],max_sampled_deg=d[-1])
    cross=[]
    for j in range(len(d)-1):
        if z[j]>=criterion and z[j+1]<=criterion and z[j]>z[j+1]:
            cross.append(float(d[j]+(criterion-z[j])*(d[j+1]-d[j])/(z[j+1]-z[j])))
    out['n_crossings']=len(cross)
    if len(cross)==1:
        up=sum((z[:-1]<criterion)&(z[1:]>=criterion))
        if up==0:
            out.update(boundary_deg=cross[0],boundary_status='observed_unique')
        else: out['boundary_status']='multiple_directions_nonmonotonic'
    elif len(cross)>1:out['boundary_status']='multiple_descending_crossings'
    elif np.all(z>criterion+eps):out['boundary_status']='above_50_at_max_disparity'
    elif np.all(z<criterion-eps):out['boundary_status']='below_50_at_min_disparity'
    elif np.isclose(z[0],criterion,atol=eps):
        out.update(boundary_deg=d[0],boundary_status='observed_at_lower_endpoint')
    elif np.isclose(z[-1],criterion,atol=eps):
        out.update(boundary_deg=d[-1],boundary_status='observed_at_upper_endpoint')
    else:out['boundary_status']='nonmonotonic_or_unidentified'
    return out

# ============================================================================
# LOAD EXPLICIT AIM-1 TABLES WITH AN EXPLICIT COHORT (NO MONKEY PATCH)
# ============================================================================
pheno=READ(GAUSSIAN_CSV)
require_columns(pheno,['subject','delta50_deg','mu_deg','causal_sigma_deg','slope50_per_deg','fit_success','boundary50_within_observed_range'],'Aim1')
pheno.subject=pid(pheno.subject)
num(pheno,['delta50_deg','mu_deg','causal_sigma_deg','slope50_per_deg','floor_same','peak_same'])
if pheno.subject.duplicated().any():raise ValueError('Duplicate Aim1 participant IDs')
fit_ok=pheno.fit_success.astype(str).str.lower().eq('true')
inrange=pheno.boundary50_within_observed_range.astype(str).str.lower().eq('true')
include=fit_ok&pheno.delta50_deg.notna()
if COHORT=='inrange45': include&=inrange
elif COHORT!='full50':raise ValueError('COHORT must be inrange45 or full50')
pheno=pheno.loc[include].copy(); cohort=set(pheno.subject)
print(f'Aim3 cohort: {COHORT}, N={len(cohort)}. These restrictions are post-hoc sensitivity when N45.')
persist(pheno[['subject','delta50_deg','mu_deg','fit_success','boundary50_within_observed_range']],
        'aim3_subject_cohort_manifest.csv')

causal=READ(CAUSAL_TRIAL_CSV)
causal.subject=pid(causal.subject)
require_columns(causal,['subject','trialIndex','same_response','signed_disparity_deg','cf_p_same_hat',
    'cf_causal_ambiguity','cf_boundary_distance_deg','cf_boundary_z'],'Aim1 trial')
num(causal,['trialIndex','signed_disparity_deg','abs_disparity_deg','same_response','cf_p_same_hat',
    'cf_causal_ambiguity','cf_boundary_distance_deg','cf_boundary_z'])
causal=causal[causal.subject.isin(cohort)].drop_duplicates(['subject','trialIndex'])
causal_cols=['subject','trialIndex','signed_disparity_deg','same_response','cf_p_same_hat',
             'cf_causal_ambiguity','cf_boundary_distance_deg','cf_boundary_z']
causal_small=causal[causal_cols].copy()

cap=READ(TRIAL_METRICS_CSV); cap.subject=pid(cap.subject)
require_columns(cap,['subject','trialIndex','offsetDeg','VisualCapture_audioLag'],'capture')
num(cap,['trialIndex','offsetDeg','VisualCapture_audioLag']);cap['offset_deg']=cap.offsetDeg.abs()
flags=(cap.get('trialType',pd.Series('',index=cap.index)).astype(str)+' '+
       cap.get('condLabel',pd.Series('',index=cap.index)).astype(str)).str.lower()
cap=cap[cap.subject.isin(cohort)&cap.offset_deg.isin(OFFSETS)&
        ~flags.str.contains('sweep|diverge|converge',regex=True)].copy()
cap=cap.merge(causal_small,on=['subject','trialIndex'],how='inner',validate='many_to_one')
cap=cap.merge(pheno[['subject','mu_deg','delta50_deg','causal_sigma_deg','slope50_per_deg']],
              on='subject',how='inner',validate='many_to_one')
cap=cap.dropna(subset=['VisualCapture_audioLag','cf_p_same_hat','cf_boundary_distance_deg']).copy()

bp=READ(BREAKPOINT_CSV);bp.subject=pid(bp.subject)
require_columns(bp,['subject','trialIndex','offset_deg','dynamically_updating'],'breakpoint')
num(bp,['trialIndex','offset_deg','dynamically_updating']);bp.offset_deg=bp.offset_deg.abs()
if 'metric' in bp: bp=bp[bp.metric.astype(str).eq('LagCorrectedVisualCapture')].copy()
# 3B fix: do NOT classify ambiguous/noisy trials as confirmed non-updating.
for col in ['classification','classification_label','state_class','trajectory_class','status']:
    if col in bp:
        drop=bp[col].astype(str).str.lower().str.contains(r'ambiguous|unclassified|noisy|invalid',regex=True)
        if drop.any():print(f'3B: excluding {drop.sum()} ambiguous/noisy trials by {col}')
        bp=bp[~drop].copy()
        break
bp=bp[bp.subject.isin(cohort)&bp.offset_deg.isin(OFFSETS)&bp.dynamically_updating.isin([0,1])].copy()
bp=bp.merge(causal_small,on=['subject','trialIndex'],how='inner',validate='many_to_one')
bp=bp.merge(pheno[['subject','mu_deg','delta50_deg','causal_sigma_deg','slope50_per_deg']],
            on='subject',how='inner',validate='many_to_one')
bp=bp.dropna(subset=['cf_causal_ambiguity','cf_p_same_hat','cf_boundary_distance_deg']).copy()
print('Behavior: capture',len(cap),'rows;',cap.subject.nunique(),'subjects; dynamicity',len(bp),'rows;',bp.subject.nunique(),'subjects')

# ============================================================================
# 3A + 3B: ORIGINAL GEE OUTPUTS, PLUS A WITHIN-BETWEEN AMBIGUITY DECOMPOSITION
# ============================================================================
def gee(formula,data,binary=False):
    return smf.gee(formula,groups='subject',data=data,
       family=sm.families.Binomial() if binary else sm.families.Gaussian(),
       cov_struct=Exchangeable()).fit()

ca=gee('VisualCapture_audioLag ~ C(offset_deg) + cf_p_same_hat',cap)
ca_tab=result_table(ca,'aim3A_capture_by_crossfit_gaussian_psame_GEE')
result_table(gee('VisualCapture_audioLag ~ C(offset_deg) + cf_causal_ambiguity',cap),
             'aim3A_capture_by_crossfit_causal_ambiguity_GEE')
amb=gee('dynamically_updating ~ C(offset_deg) + cf_causal_ambiguity',bp,True)
amb_tab=result_table(amb,'aim3B_dynamicity_by_crossfit_causal_ambiguity_GEE')
result_table(gee('dynamically_updating ~ C(offset_deg) + cf_p_same_hat',bp,True),
             'aim3B_dynamicity_by_crossfit_gaussian_psame_GEE')
# Fix pooled within/between interpretive problem: explicit within-subject deviation and
# between-subject mean, with categorical disparity controlled; exploratory.
bp['amb_subject_mean']=bp.groupby('subject').cf_causal_ambiguity.transform('mean')
bp['amb_within_subject']=bp.cf_causal_ambiguity-bp.amb_subject_mean
result_table(gee('dynamically_updating ~ C(offset_deg) + amb_within_subject + amb_subject_mean',bp,True),
             'aim3B_dynamicity_within_between_causal_ambiguity_GEE')
# Stratified offset-specific association: avoids interpreting pooled ambiguity bins as adjusted.
offset_models=[]
for off,sub in bp.groupby('offset_deg'):
    if sub.subject.nunique()>8 and sub.cf_causal_ambiguity.std()>1e-6:
        rr=gee('dynamically_updating ~ cf_causal_ambiguity',sub,True)
        offset_models.append({'offset_deg':off,'n_subjects':sub.subject.nunique(),
           'n_trials':len(sub),'beta_ambiguity':rr.params.get('cf_causal_ambiguity'),
           'p':rr.pvalues.get('cf_causal_ambiguity')})
offtest=pd.DataFrame(offset_models)
if len(offtest):offtest['p_FDR_BH']=multipletests(offtest.p,method='fdr_bh')[1]
persist(offtest,'aim3B_offset_specific_ambiguity_GEE.csv')

# ============================================================================
# 3C LEGACY DESCRIPTIVE HALF-DECAY (PRESERVED, NOT CALLED AN IMPLICIT CAUSAL PSE)
# ============================================================================
implicit_rows=[]
for subject,g in cap.groupby('subject'):
    delta=float(g.delta50_deg.iloc[0]);radial=abs(g.signed_disparity_deg-g.mu_deg)
    inside=radial<=delta;outside=~inside
    implicit_rows.append({'subject':subject,'explicit_delta50_deg':delta,
       'implicit_capture_halfdecay_deg':halfdecay(g.offset_deg,g.VisualCapture_audioLag),
       'mean_capture_inside_explicit_boundary':g.loc[inside,'VisualCapture_audioLag'].mean(),
       'mean_capture_outside_explicit_boundary':g.loc[outside,'VisualCapture_audioLag'].mean(),
       'n_trials_inside':int(inside.sum()),'n_trials_outside':int(outside.sum())})
implicit=pd.DataFrame(implicit_rows)
implicit['outside_minus_inside_capture']=implicit.mean_capture_outside_explicit_boundary-implicit.mean_capture_inside_explicit_boundary
persist(implicit,'aim3C_explicit_vs_implicit_boundary_subject_summary.csv')
legacy_corr=correlations(implicit,'explicit_delta50_deg','implicit_capture_halfdecay_deg','legacy_halfdecay')
persist(pd.DataFrame([legacy_corr]),'aim3C_explicit_delta50_vs_implicit_capture_halfdecay_correlation.csv')
d=implicit.dropna(subset=['explicit_delta50_deg','implicit_capture_halfdecay_deg'])
if len(d)>=4:
    m=smf.ols('implicit_capture_halfdecay_deg ~ explicit_delta50_deg',d).fit(cov_type='HC3')
    xx=np.linspace(d.explicit_delta50_deg.min(),d.explicit_delta50_deg.max(),150)
    fr=m.get_prediction(pd.DataFrame({'explicit_delta50_deg':xx})).summary_frame()
    fig,ax=plt.subplots(figsize=(6.5,5.2))
    ax.scatter(d.explicit_delta50_deg,d.implicit_capture_halfdecay_deg,alpha=.7)
    ax.plot(xx,fr['mean']);ax.fill_between(xx,fr.mean_ci_lower.to_numpy(),fr.mean_ci_upper.to_numpy(),alpha=.18)
    ax.set(xlabel='Explicit Gaussian Delta50 (deg)',ylabel='Legacy capture half-decay (deg)',
           title='Legacy descriptive capture midpoint (not a causal PSE)')
    fig.tight_layout();fig.savefig(FIG_OUT/'aim3C_explicit_vs_implicit_boundary.png',dpi=220);plt.show()

# ============================================================================
# 3C NEW: STREAM LARGE MODEL OUTPUT IN CHUNKS; TRIAL-FIRST AVERAGES
# ============================================================================
params=READ(BCI_PARAM_CSV);require_columns(params,['subject','p_common','tau_p','kappa'],'BCI parameters')
params['model_subject']=params.subject.astype(str);params.subject=pid(params.subject)
num(params,['p_common','tau_p','kappa','tau_C_derived','sigma_p','sigma_eA','sigma_eV','r2_holdout','r2_norm','n_at_bound'])
params=params.drop_duplicates('subject')
if check_file(BCI_RESTART_CSV,False):
    restarts=READ(BCI_RESTART_CSV);restarts.subject=pid(restarts.subject)
    num(restarts,['p_common','tau_p','kappa','sigma_p','sigma_eA','r2_holdout','chi2_uni'])
    restart_summary=restarts.groupby('subject').agg(n_restart=('subject','size'),
       prior_sd_across_restarts=('p_common','std'),tau_p_sd_across_restarts=('tau_p','std'),
       kappa_sd_across_restarts=('kappa','std')).reset_index()
    params=params.merge(restart_summary,on='subject',how='left')
if check_file(BCI_LAG_CSV,False):
    lg=READ(BCI_LAG_CSV);lg.subject=pid(lg.subject)
    persist(lg.groupby(['subject','source'],dropna=False).agg(n_rows=('subject','size')).reset_index(),
            'aim3C_BCI_lag_file_coverage_QC.csv')
persist(params,'aim3C_BCI_parameter_QC.csv')

fixed_parts=[];sweep_parts=[];trial_coverage=[];all_bci_pids=set()
usecols=['subject','session','trial','condition','disparity_signed','t','audio_x','visual_x','pC1','loc_pred']
types={'trial':'float64','disparity_signed':'float32','t':'float32',
       'audio_x':'float32','visual_x':'float32','pC1':'float32','loc_pred':'float32'}
for ci,chunk in enumerate(READ(BCI_TIME_CSV,usecols=usecols,dtype=types,chunksize=250_000)):
    chunk=chunk[chunk.session.astype(str).str.upper().eq('SL')].copy()
    chunk['subject']=pid(chunk.subject)
    all_bci_pids.update(chunk.subject.dropna().unique())
    chunk=chunk[chunk.subject.isin(cohort)&chunk.pC1.between(0,1)&chunk.t.between(0,18)].copy()
    if chunk.empty:continue
    f=chunk[chunk.condition.eq('congruent')|chunk.condition.str.startswith('spatial_',na=False)].copy()
    if len(f):
        f['offset_deg']=f.disparity_signed.abs().round(0)
        f=f[f.offset_deg.isin(ALL_OFFSETS)]
        f['side']=np.sign(f.disparity_signed).astype('int8')
        for name,(lo,hi) in [('main',WINDOW_MAIN),('late',WINDOW_LATE)]:
            g=f[f.t.between(lo,hi)]
            if len(g):
                v=g.groupby(['subject','trial','offset_deg','side'],as_index=False).agg(
                    total=('pC1','sum'),n=('pC1','size'))
                v['window']=name;fixed_parts.append(v)
    s=chunk[chunk.condition.str.startswith('sweep_',na=False)].copy()
    if len(s):
        s['direction']=np.where(s.condition.str.contains('diverge',na=False),'Diverge','Converge')
        # Signed actual stimulus separation; nominal sweep disparity stays at +-40.
        s['actual_disparity']=abs(s.audio_x-s.visual_x)
        s=s[s.actual_disparity.between(4.,36.,inclusive='left')]
        if len(s):
            s['side']=np.sign(s.disparity_signed).astype('int8')
            s['disp_bin']=5.+2.*np.rint((s.actual_disparity-5.)/2.)
            s=s[s.disp_bin.isin(SWEEP_BINS)]
            if len(s):
                v=s.groupby(['subject','trial','direction','side','disp_bin'],as_index=False).agg(
                   total=('pC1','sum'),n=('pC1','size'),mean_t=('t','mean'))
                sweep_parts.append(v)
    if ci%8==0: print(f'  Loaded BCI chunk {ci+1}')
if not fixed_parts:raise RuntimeError('No valid fixed-offset BCI rows. Check condition labels and time window.')

fixed_chunk=pd.concat(fixed_parts,ignore_index=True)
fixed_trial=fixed_chunk.groupby(['subject','trial','offset_deg','side','window'],as_index=False).agg(
   total=('total','sum'),n=('n','sum'))
fixed_trial['pC1']=fixed_trial.total/fixed_trial.n
persist(fixed_trial,'aim3C_BCI_fixed_trial_mean_posterior.csv')
# Trial -> side (equal trials) -> magnitude (equal L/R), avoiding unequal trial duration bias.
fixed_side=fixed_trial.groupby(['subject','window','offset_deg','side'],as_index=False).agg(
    pC1=('pC1','mean'),n_trials=('trial','nunique'))
fixed_cell=fixed_side.groupby(['subject','window','offset_deg'],as_index=False).agg(
    pC1=('pC1','mean'),n_sides=('side','nunique'),n_trials=('n_trials','sum'))
persist(fixed_side,'aim3C_BCI_fixed_signed_side_posterior.csv')
persist(fixed_cell,'aim3C_BCI_fixed_offset_posterior.csv')
print('BCI model IDs:',len(all_bci_pids),'matched participants:',len(cohort.intersection(all_bci_pids)),
      'unmatched Aim3:',sorted(cohort-all_bci_pids))

# Posterior summaries for every cohort member who has a dynamic model.
posterior_rows=[];cross_rows=[]
for (subject,window),g in fixed_cell.groupby(['subject','window']):
    w=g.set_index('offset_deg').pC1.reindex(ALL_OFFSETS)
    vals=w.to_numpy(float)
    row={'subject':subject,'window':window,'n_cells':int(w.notna().sum()),
         'bci_pC20':float(w.loc[20.]) if np.isfinite(w.loc[20.]) else np.nan,
         'bci_mean_pC_0to40':float(np.mean(vals)) if np.isfinite(vals).all() else np.nan,
         'bci_pC_auc_0to40':float(np.trapz(vals,ALL_OFFSETS)/40) if np.isfinite(vals).all() else np.nan}
    for o in ALL_OFFSETS:row[f'bci_pC_{int(o)}']=w.loc[o]
    posterior_rows.append(row)
    cross_rows.append({'subject':subject,'window':window,'curve':'magnitude_balanced',
                      **find_crossing(g.offset_deg,g.pC1)})
    sides=fixed_side[(fixed_side.subject==subject)&(fixed_side.window==window)]
    for side in [-1,1]:
        ss=sides[(sides.side==side)|(sides.offset_deg.eq(0))].copy()
        cross_rows.append({'subject':subject,'window':window,'curve':f'signed_side_{side:+d}',
                          **find_crossing(ss.offset_deg,ss.pC1)})
post=pd.DataFrame(posterior_rows).merge(params,on='subject',how='left',validate='many_to_one')
post=post.merge(pheno[['subject','delta50_deg','mu_deg']],on='subject',how='left',validate='many_to_one')
persist(post,'aim3C_BCI_subject_posterior_phenotypes.csv')
cross=pd.DataFrame(cross_rows).rename(columns={'boundary_deg':'bci_boundary50_deg'})
persist(cross,'aim3C_BCI_fixed_boundary50_crossing_status_all.csv')
# Require both signed crossings for a true two-sided counterpart of Gaussian Delta50.
cw=cross[cross.window.eq('main')].pivot(index='subject',columns='curve',values='bci_boundary50_deg')
for key in ['signed_side_-1','signed_side_+1']:
    if key not in cw.columns:cw[key]=np.nan
cw['bci_two_sided_delta50_deg']=((cw['signed_side_-1']+cw['signed_side_+1'])/2).where(
                                    cw[['signed_side_-1','signed_side_+1']].notna().all(axis=1))
cw=cw.reset_index()
persist(cw,'aim3C_BCI_two_sided_boundary50.csv')
base=post[post.window.eq('main')].merge(cw,on='subject',how='left')
persist(base,'aim3C_BCI_explicit_implicit_comparison_subjects.csv')

# ============================================================================
# 3C ADDITION: MODEL-QUALITY / PARAMETER-BOUND SENSITIVITY
# ============================================================================
# Primary association: explicit Gaussian Delta50 vs independent BCI P(C=1) at 20°.
# A *held-out* R^2 is used for model predictive-quality sensitivity. A non-bound fit
# means n_at_bound == 0 (NOT n_at_bound <= 1). Never replace missing model-QC with 0.
# Prespecified nested / non-nested subsets are exploratory checks, not new primary
# participant exclusions. All other analyses retain the original COHORT throughout.
require_columns(base, ['subject','delta50_deg','bci_pC20','r2_holdout','n_at_bound'],
                'BCI model-quality sensitivity base')
quality = base[['subject','delta50_deg','bci_pC20','r2_holdout','n_at_bound']].copy()
if 'r2_norm' in base.columns: quality['r2_norm'] = base['r2_norm']
quality['r2_holdout'] = pd.to_numeric(quality['r2_holdout'], errors='coerce')
quality['n_at_bound'] = pd.to_numeric(quality['n_at_bound'], errors='coerce')
quality['usable_pair'] = quality[['delta50_deg','bci_pC20']].notna().all(axis=1)
quality['heldout_R2_positive'] = quality.r2_holdout > FIT_R2_MINIMAL
quality['heldout_R2_ge_0p50'] = quality.r2_holdout >= FIT_R2_STRICT
quality['no_parameters_at_bounds'] = quality.n_at_bound.eq(0)
quality['positive_R2_and_unbounded'] = quality.heldout_R2_positive & quality.no_parameters_at_bounds
quality['R2_ge_0p50_and_unbounded'] = quality.heldout_R2_ge_0p50 & quality.no_parameters_at_bounds
quality['any_fitted_parameter_at_bound'] = quality.n_at_bound.gt(0)
persist(quality, 'aim3C_BCI_model_quality_subject_flags.csv')

quality_subsets = [
    ('all_matched', quality.usable_pair),
    ('heldout_R2_positive', quality.heldout_R2_positive),
    ('heldout_R2_ge_0p50', quality.heldout_R2_ge_0p50),
    ('no_parameters_at_bounds', quality.no_parameters_at_bounds),
    ('positive_R2_and_unbounded', quality.positive_R2_and_unbounded),
    ('R2_ge_0p50_and_unbounded', quality.R2_ge_0p50_and_unbounded),
]
quality_corr_rows = []
quality_membership_rows = []
for j,(subset_name, select) in enumerate(quality_subsets):
    use = quality.loc[quality.usable_pair & select].copy()
    row = correlations(use, 'delta50_deg','bci_pC20', subset_name,
                       seed=SEED+700+j)
    row['subset'] = subset_name
    row['n_total_model_matched'] = int(quality.usable_pair.sum())
    row['n_excluded'] = int(quality.usable_pair.sum()-len(use))
    row['r2_cutoff_note'] = 'R2>0' if subset_name.startswith(('heldout_R2_positive','positive_R2')) else ('R2>=0.50' if '0p50' in subset_name else 'no_R2_filter')
    quality_corr_rows.append(row)
    for sub in quality.subject:
        quality_membership_rows.append({'subject':sub, 'subset':subset_name,
            'included':bool(sub in set(use.subject))})
quality_corr = pd.DataFrame(quality_corr_rows)
quality_corr['p_FDR_BH_exploratory'] = np.nan
qq = quality_corr.permutation_p.notna()
if qq.any():
    quality_corr.loc[qq,'p_FDR_BH_exploratory'] = multipletests(
        quality_corr.loc[qq,'permutation_p'],method='fdr_bh')[1]
persist(quality_corr, 'aim3C_BCI_model_quality_sensitivity_correlations.csv')
persist(pd.DataFrame(quality_membership_rows),
        'aim3C_BCI_model_quality_sensitivity_membership.csv')

fig, ax = plt.subplots(figsize=(9,5.3))
qq=quality_corr.copy()
xx=np.arange(len(qq))
for i,row in qq.iterrows():
    if np.isfinite(row.pearson_r):
        e=np.array([[max(0,row.pearson_r-row.pearson_ci_low)],
                    [max(0,row.pearson_ci_high-row.pearson_r)]])
        ax.errorbar(i,row.pearson_r,yerr=e,fmt='o',capsize=3)
ax.axhline(0,color='gray',ls='--',lw=1)
ax.set_xticks(xx)
ax.set_xticklabels([f'{s}\nN={n}' for s,n in zip(qq.subset,qq.n)],rotation=25,ha='right')
ax.set(ylabel='Pearson r (95% participant-bootstrap CI)',
       title='Explicit Delta50 vs implicit BCI P(C=1) at 20°: model-quality sensitivity')
fig.tight_layout()
fig.savefig(FIG_OUT/'aim3C_BCI_model_quality_sensitivity_correlations.png',dpi=220,bbox_inches='tight')
plt.show()

corr_specs=[('bci_pC20','primary_model_posterior_20deg'),
            ('bci_mean_pC_0to40','secondary_equal_cell_mean'),
            ('bci_pC_auc_0to40','secondary_integrated_auc'),
            ('bci_two_sided_delta50_deg','secondary_two_sided_crossing')]
corrs=[correlations(base,'delta50_deg',c,label,seed=SEED+i) for i,(c,label) in enumerate(corr_specs)]
for off in ALL_OFFSETS:
    if off!=20:corrs.append(correlations(base,'delta50_deg',f'bci_pC_{int(off)}',
                                        f'exploratory_{int(off)}deg',seed=SEED+int(off)+10))
for par in ['p_common','tau_p','kappa','tau_C_derived','r2_holdout']:
    if par in base:corrs.append(correlations(base,'delta50_deg',par,
                                            f'exploratory_parameter_{par}',seed=SEED+len(corrs)))
corrtab=pd.DataFrame(corrs)
mask=corrtab.permutation_p.notna()
corrtab['p_FDR_BH_all_reported']=np.nan
if mask.any():corrtab.loc[mask,'p_FDR_BH_all_reported']=multipletests(corrtab.loc[mask,'permutation_p'],method='fdr_bh')[1]
persist(corrtab,'aim3C_BCI_explicit_vs_implicit_correlations.csv')
persist(cross.groupby(['window','curve','boundary_status'],as_index=False).size(),
        'aim3C_BCI_fixed_boundary_identification_counts.csv')

# ============================================================================
# 3C HISTORY: SWEEPS MATCHED BY ACTUAL DISPARITY, DIRECTION, AND SIGNED SIDE
# ============================================================================
if sweep_parts:
    sweeps=pd.concat(sweep_parts,ignore_index=True)
    sweeps=sweeps.groupby(['subject','trial','direction','side','disp_bin'],as_index=False).agg(
       total=('total','sum'),n=('n','sum'))
    sweeps['pC1']=sweeps.total/sweeps.n
    # A sweep trial can contribute one bin mean; then equal-weight trials and two directions.
    sidecurve=sweeps.groupby(['subject','direction','side','disp_bin'],as_index=False).agg(
       pC1=('pC1','mean'),n_trials=('trial','nunique'))
    curve=sidecurve.groupby(['subject','direction','disp_bin'],as_index=False).agg(
       pC1=('pC1','mean'),n_sides=('side','nunique'))
    persist(sidecurve,'aim3C_BCI_sweep_posterior_signed_by_disparity.csv')
    persist(curve,'aim3C_BCI_sweep_posterior_matched_disparity.csv')
    sweep_cross=[]
    for (subject,direction),g in curve.groupby(['subject','direction']):
        sweep_cross.append({'subject':subject,'direction':direction,
                 **find_crossing(g.disp_bin,g.pC1)})
    sc=pd.DataFrame(sweep_cross).rename(columns={'boundary_deg':'sweep_boundary50_deg'})
    persist(sc,'aim3C_BCI_sweep_boundary50_crossing_status.csv')
    bounds=sc.pivot(index='subject',columns='direction',values='sweep_boundary50_deg')
    for direction in ['Diverge','Converge']:
        if direction not in bounds.columns:bounds[direction]=np.nan
    bounds['diverge_minus_converge_boundary_deg']=bounds.Diverge-bounds.Converge
    bounds=bounds.reset_index()
    matched=curve.pivot(index=['subject','disp_bin'],columns='direction',values='pC1').dropna()
    matched['diverge_minus_converge_pC']=matched.Diverge-matched.Converge
    hist=matched.groupby('subject').agg(
       bci_hysteresis_mean=('diverge_minus_converge_pC','mean'),
       n_matched_bins=('diverge_minus_converge_pC','size')).reset_index()
    hist=hist.merge(bounds,on='subject',how='left').merge(
       base[['subject','delta50_deg','bci_pC20','p_common','tau_p','kappa','tau_C_derived']],
       on='subject',how='left')
    hist['BCI_CausalPosterior_Hysteresis']=hist.bci_hysteresis_mean
    persist(hist,'aim3C_BCI_history_hysteresis_and_boundaries_by_subject.csv')
    hist_corr=[]
    for ycol in ['bci_hysteresis_mean','diverge_minus_converge_boundary_deg']:
        for xcol in ['delta50_deg','p_common','tau_p','kappa','tau_C_derived','bci_pC20']:
            hist_corr.append(correlations(hist,xcol,ycol,f'{ycol}_vs_{xcol}',seed=SEED+len(hist_corr)+400))
    hist_corr=pd.DataFrame(hist_corr)
    valid=hist_corr.permutation_p.notna();hist_corr['p_FDR_BH']=np.nan
    if valid.any():hist_corr.loc[valid,'p_FDR_BH']=multipletests(hist_corr.loc[valid,'permutation_p'],method='fdr_bh')[1]
    persist(hist_corr,'aim3C_BCI_history_correlations.csv')
    tests=[]
    for metric in ['bci_hysteresis_mean','diverge_minus_converge_boundary_deg']:
        st=test_signflip(hist[metric],seed=SEED+500+len(tests))
        tests.append({'metric':metric,**st})
    persist(pd.DataFrame(tests),'aim3C_BCI_history_group_signflip_tests.csv')
    persist(sc.groupby(['direction','boundary_status'],as_index=False).size(),
            'aim3C_BCI_history_boundary_identification_counts.csv')
else:
    print('No sweep observations available; history analyses omitted.')

# ============================================================================
# 3C ADDITION: OBSERVED SL SPATIAL HYSTERESIS VS MODEL-POSTERIOR HYSTERESIS
# ============================================================================
# Behavioral metric is normalized visual-directed capture, not a causal posterior.
# Both signs are oriented so POSITIVE = Diverge minus Converge. Compare individual
# *magnitudes/covariation* only, NOT numerical identity of the two outcome scales.
# Explicit validation: model posterior derives from fits to the SAME SL trajectories,
# so this is an in-sample mechanistic correspondence, not independent validation.
# At least MIN_MATCHED_HYST_BINS must contribute in BOTH conditions.

if not check_file(HYST_WINDOW_CSV, required=False):
    warnings.warn('Empirical spatial hysteresis analysis NOT run. Set HYST_WINDOW_CSV to the '
                  'window-level SL spatial hysteresis CSV (used in Aim 1). '
                  'Aim 3A-3D and BCI quality sensitivity still run.')
elif not sweep_parts:
    warnings.warn('Empirical-vs-BCI hysteresis requires BCI sweep trials; none found.')
else:
    emp = READ(HYST_WINDOW_CSV)
    need = ['subject','trialIndex','SweepDirection','mean_disparity_window',HYST_EMPIRICAL_COLUMN]
    require_columns(emp,need,'empirical spatial hysteresis window CSV')
    emp['subject'] = pid(emp.subject)
    num(emp,['trialIndex','mean_disparity_window',HYST_EMPIRICAL_COLUMN])
    raw_dir = emp.SweepDirection.astype(str).str.lower().str.strip()
    emp['direction'] = np.where(raw_dir.str.contains('diverg',na=False),'Diverge',
                        np.where(raw_dir.str.contains('converg',na=False),'Converge','Unknown'))
    emp = emp[emp.subject.isin(cohort)&emp.direction.isin(['Diverge','Converge'])].copy()
    emp['physical_disparity'] = emp.mean_disparity_window.abs()
    emp = emp[emp.physical_disparity.between(4.,36.,inclusive='left')].copy()
    emp['disp_bin'] = 5.+2.*np.rint((emp.physical_disparity-5.)/2.)
    emp = emp[emp.disp_bin.isin(SWEEP_BINS)].dropna(subset=[HYST_EMPIRICAL_COLUMN,'trialIndex'])

    # If signed sweep-offset information is available, match the BCI aggregation
    # trial -> side -> magnitude. Otherwise, trial -> magnitude remains valid;
    # the condition was experimentally left/right balanced, which is audited.
    emp['side'] = 0
    signed_col = None
    for candidate in ['disparity_signed','signed_disparity_deg','offsetDeg',
                      'signed_offset_deg','sweep_signed_offset_deg','offset_deg_signed']:
        if candidate in emp.columns:
            trial_sign = pd.to_numeric(emp[candidate],errors='coerce')
            if trial_sign.notna().mean()>.95 and (trial_sign.abs()>0).mean()>.90:
                emp['side'] = np.sign(trial_sign).astype(int)
                signed_col = candidate
                break
    if signed_col is None:
        warnings.warn('No signed sweep side column identified in empirical SL hysteresis CSV. '
                      'Aggregating trials without explicit left/right balancing; verify '
                      'the condition has equal left/right repetitions.')
    print(f'Observed SL hysteresis: {len(emp)} window rows; side source: {signed_col}')

    # Data integrity: each trial should uniquely represent one sweep direction/side.
    side_audit = (emp.groupby(['subject','direction','side'],as_index=False)
                    .agg(n_trials=('trialIndex','nunique'),
                         n_windows=(HYST_EMPIRICAL_COLUMN,'size')))
    persist(side_audit,'aim3C_SL_hysteresis_direction_side_coverage.csv')

    etrial = (emp.groupby(['subject','trialIndex','direction','side','disp_bin'],as_index=False)
                 .agg(visual_capture=(HYST_EMPIRICAL_COLUMN,'mean')))
    eside = (etrial.groupby(['subject','direction','side','disp_bin'],as_index=False)
                  .agg(visual_capture=('visual_capture','mean'),
                       n_trials=('trialIndex','nunique')))
    ecurve = (eside.groupby(['subject','direction','disp_bin'],as_index=False)
                   .agg(visual_capture=('visual_capture','mean'),
                        n_sides=('side','nunique')))
    persist(ecurve,'aim3C_SL_empirical_hysteresis_matched_disparity_curves.csv')
    ew = ecurve.pivot(index=['subject','disp_bin'],columns='direction',values='visual_capture')
    mw = curve.pivot(index=['subject','disp_bin'],columns='direction',values='pC1')
    for dd in ['Diverge','Converge']:
        if dd not in ew.columns: ew[dd]=np.nan
        if dd not in mw.columns: mw[dd]=np.nan
    # Require measurements in both directions for a valid disparity-matched bin.
    ep = ew[['Diverge','Converge']].dropna().copy()
    ep['sl_diverge_minus_converge_visual_capture'] = ep.Diverge-ep.Converge
    mp = mw[['Diverge','Converge']].dropna().copy()
    mp['bci_diverge_minus_converge_pC'] = mp.Diverge-mp.Converge
    epoints = ep[['sl_diverge_minus_converge_visual_capture']].join(
        mp[['bci_diverge_minus_converge_pC']],how='inner')
    epoints = epoints.dropna().reset_index()
    persist(epoints,'aim3C_SL_vs_BCI_hysteresis_disparity_matched_pairs.csv')

    # Crucial: compute BCI hysteresis on THE SAME subject/disparity bins as observed
    # behavior. Compare also with the model's existing full-bin hysteresis index.
    subj_emp = epoints.groupby('subject',as_index=False).agg(
        sl_hysteresis_diverge_minus_converge=('sl_diverge_minus_converge_visual_capture','mean'),
        bci_hysteresis_empirically_matched=('bci_diverge_minus_converge_pC','mean'),
        n_common_bins=('disp_bin','nunique'))
    # Both empirical and modeled indices use 5-35 deg, same 2 deg bins, positive
    # = more visual-directed capture (SL) / more common-source belief (BCI) in Diverge.
    subj_emp['passes_min_bins'] = subj_emp.n_common_bins>=MIN_MATCHED_HYST_BINS
    subj_emp['BCI_CausalPosterior_Hysteresis']=subj_emp.bci_hysteresis_empirically_matched
    if 'hist' in locals():
        subcols=['subject','bci_hysteresis_mean','n_matched_bins']
        subj_emp=subj_emp.merge(hist[subcols],on='subject',how='left',validate='one_to_one')
    subj_emp=subj_emp.merge(
        base[['subject','delta50_deg','bci_pC20','r2_holdout','n_at_bound','p_common','tau_p','kappa']],
        on='subject',how='left',validate='one_to_one')
    persist(subj_emp,'aim3C_SL_vs_BCI_hysteresis_subject_comparison_ALL.csv')
    valid_hyst=subj_emp[subj_emp.passes_min_bins].copy()
    persist(valid_hyst,'aim3C_SL_vs_BCI_hysteresis_subject_comparison_VALID.csv')
    print(f'Empirical-model hysteresis: {len(valid_hyst)} participants with >='
          f'{MIN_MATCHED_HYST_BINS} common disparity bins (out of {len(subj_emp)}).')

    # PRIMARY: observed SL hysteresis vs BCI hysteresis with same-bin matching.
    # SECONDARY: the pre-existing BCI index, and sensory/causal-parameter effects.
    hyst_test_specs=[
      ('bci_hysteresis_empirically_matched','primary_same_bin_model_vs_observed'),
      ('bci_hysteresis_mean','secondary_original_model_vs_observed'),
      ('delta50_deg','exploratory_explicit_delta50_vs_observed'),
      ('p_common','exploratory_common_cause_prior_vs_observed'),
      ('tau_p','exploratory_position_memory_vs_observed'),
      ('kappa','exploratory_causal_update_vs_observed'),
    ]
    hyst_obs_corr=[]
    for j,(xcol,label) in enumerate(hyst_test_specs):
        if xcol in valid_hyst.columns:
            hyst_obs_corr.append(correlations(valid_hyst,xcol,
                'sl_hysteresis_diverge_minus_converge',label,seed=SEED+800+j))
    hyst_obs_corr=pd.DataFrame(hyst_obs_corr)
    hyst_obs_corr['p_FDR_BH_exploratory_family']=np.nan
    mask=hyst_obs_corr.permutation_p.notna()
    # The main hypothesis is explicitly labeled primary; FDR is supplied across
    # all comparisons for a more conservative exploratory sensitivity.
    if mask.any():
        hyst_obs_corr.loc[mask,'p_FDR_BH_exploratory_family']=multipletests(
            hyst_obs_corr.loc[mask,'permutation_p'],method='fdr_bh')[1]
    persist(hyst_obs_corr,'aim3C_SL_vs_BCI_hysteresis_correlations.csv')

    # High-fit/no-bound check for the mechanistic hysteresis correspondence.
    hyst_qc=[]
    hyst_filters=[('all_matched',np.ones(len(valid_hyst),bool)),
          ('heldout_R2_positive',valid_hyst.r2_holdout>FIT_R2_MINIMAL),
          ('heldout_R2_ge_0p50',valid_hyst.r2_holdout>=FIT_R2_STRICT),
          ('no_parameters_at_bounds',valid_hyst.n_at_bound.eq(0)),
          ('positive_R2_and_unbounded',
             (valid_hyst.r2_holdout>FIT_R2_MINIMAL)&valid_hyst.n_at_bound.eq(0)),
          ('R2_ge_0p50_and_unbounded',
             (valid_hyst.r2_holdout>=FIT_R2_STRICT)&valid_hyst.n_at_bound.eq(0))]
    for j,(name,mask) in enumerate(hyst_filters):
        dd=valid_hyst.loc[mask]
        rr=correlations(dd,'bci_hysteresis_empirically_matched',
                 'sl_hysteresis_diverge_minus_converge',name,seed=SEED+900+j)
        rr['subset']=name
        rr['n_excluded_from_valid_hysteresis']=len(valid_hyst)-len(dd)
        hyst_qc.append(rr)
    persist(pd.DataFrame(hyst_qc),'aim3C_SL_vs_BCI_hysteresis_fit_quality_sensitivity.csv')

    # Mechanistic specificity: association after adjusting for *overall* causal
    # propensity and explicit tolerance (not required by the core hypothesis).
    def residualized_hyst_corr(frame,control_cols,label,seed):
        cols=['sl_hysteresis_diverge_minus_converge',
              'bci_hysteresis_empirically_matched']+control_cols
        d=frame[cols].replace([np.inf,-np.inf],np.nan).dropna()
        if len(d)<len(control_cols)+8:
            return {'comparison':label,'controls':'+'.join(control_cols),'n':len(d),
                    'partial_pearson_r':np.nan,'partial_p':np.nan}
        z=sm.add_constant(d[control_cols],has_constant='add')
        x=sm.OLS(d.bci_hysteresis_empirically_matched,z).fit().resid
        y=sm.OLS(d.sl_hysteresis_diverge_minus_converge,z).fit().resid
        rr=correlations(pd.DataFrame({'model_resid':x,'behavior_resid':y}),
                         'model_resid','behavior_resid',label,seed=seed)
        return {'comparison':label,'controls':'+'.join(control_cols),
                'n':rr['n'],'partial_pearson_r':rr['pearson_r'],
                'partial_p':rr['pearson_p'],'permutation_p_residual_shuffle_exploratory':rr['permutation_p']}
    pr=[]
    for j,ctrl in enumerate([['delta50_deg'],['bci_pC20'],['delta50_deg','bci_pC20']]):
        pr.append(residualized_hyst_corr(valid_hyst,ctrl,'exploratory_partial_'+str(j),
                                        SEED+980+j))
    persist(pd.DataFrame(pr),'aim3C_SL_vs_BCI_hysteresis_partial_correlations_exploratory.csv')

    # Primary individual-level comparison visualization.
    d=valid_hyst[['bci_hysteresis_empirically_matched',
                   'sl_hysteresis_diverge_minus_converge']].dropna()
    fig,axes=plt.subplots(1,2,figsize=(12.4,5.2))
    axes[0].scatter(d.bci_hysteresis_empirically_matched,
                    d.sl_hysteresis_diverge_minus_converge,alpha=.75)
    if len(d)>=5 and d.iloc[:,0].std()>1e-12:
        fit=sm.OLS(d.iloc[:,1],sm.add_constant(d.iloc[:,0])).fit(cov_type='HC3')
        xline=np.linspace(d.iloc[:,0].min(),d.iloc[:,0].max(),150)
        pred=fit.get_prediction(sm.add_constant(xline)).summary_frame()
        axes[0].plot(xline,pred['mean'].to_numpy(),lw=2)
        axes[0].fill_between(xline,pred['mean_ci_lower'].to_numpy(),
                            pred['mean_ci_upper'].to_numpy(),alpha=.14)
    primary=hyst_obs_corr[hyst_obs_corr.comparison.eq('primary_same_bin_model_vs_observed')]
    title='BCI causal hysteresis vs observed SL spatial hysteresis'
    if not primary.empty and np.isfinite(primary.iloc[0].pearson_r):
        title+=f"\nr={primary.iloc[0].pearson_r:.3f}; N={int(primary.iloc[0].n)}"
    axes[0].set(xlabel='Modeled Diverge - Converge P(C=1)',
                ylabel='Observed Diverge - Converge normalized visual capture',title=title)

    # Plot disparity-resolved participant-first histories on their OWN axes:
    # causal-posterior differences and visual-capture differences are not identical.
    em=epoints.groupby('disp_bin').agg(
        observed=('sl_diverge_minus_converge_visual_capture','mean'),
        modeled=('bci_diverge_minus_converge_pC','mean')).reset_index()
    axes[1].plot(em.disp_bin,em.observed,'o-',label='Observed normalized capture difference')
    axes[1].plot(em.disp_bin,em.modeled,'s--',label='Modeled causal-posterior difference')
    axes[1].axhline(0,color='gray',lw=.8)
    axes[1].set(xlabel='Matched instantaneous audiovisual disparity (deg)',
                ylabel='Diverge - Converge (respective units)',
                title='Disparity-resolved hysteresis (different constructs)')
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_OUT/'aim3C_SL_vs_BCI_hysteresis_individual_and_disparity.png',
                dpi=220,bbox_inches='tight')
    plt.show()

# ============================================================================
# 3C NEW PRIMARY: OBSERVED vs DYNAMIC-BCI-PREDICTED SPATIAL HYSTERESIS
# ============================================================================
# This section does NOT modify any prior Aim3 result, file name, or figure.
# Units: auditory-directed localization => visual-directed bias in degrees.
# It uses each trajectory's own original stimulus coordinates; observed and
# predicted trajectories are only considered TRIAL-MATCHED if stimulus/time
# alignment passes explicit audits. Otherwise no paired-trial claim is made.
# Latent posterior pC1 belongs to the secondary mechanistic comparison above.
# Model 'loc_pred' is assumed to include motor readout; confirm this with Haocheng.
# No extra motor delay is applied. Lag correction is optional and estimated from
# each source's own auditory-only tracking, never copied from the observed cursor
# to the model. No motor-control ablation is fabricated if absent.

# ---------------------------------------------------------------------------
# CONFIGURE INPUT. A real, ORIGINAL time-resolved SL trajectory CSV is required.
# Set this to your own file containing raw mouse x, audio x, visual x, time and
# participant/trial IDs. Do not use a pre-averaged, lag-corrected bias CSV.
# ---------------------------------------------------------------------------
OBS_SL_TIME_CSV = None  # e.g. ROOT/'individual_data_SL'/'YOUR_RAW_SL_TIMEPOINTS.csv'
OBS_COLUMN_OVERRIDES = {   # use explicit names when your original CSV differs
    'subject': None,       # e.g. 'participant'
    'trial': None,         # e.g. 'trialIndex'
    'condition': None,     # e.g. 'condLabel' or 'trialType'
    'time': None,          # seconds FROM TRIAL ONSET
    'mouse': None,         # ORIGINAL mouse x position (degrees, not normalized)
    'audio': None,         # programmed auditory x position (degrees)
    'visual': None,        # displayed visual x position (degrees)
    'side': None,          # optional signed trial direction
    'direction': None,     # optional SweepDirection when condition only says 'sweep'
}
# If original observed data use a different trial-ID origin, set here ONLY after
# checking the saved trial alignment audit (do not silently auto-correct IDs).
OBS_TRIAL_ID_SHIFT = 0
HYST_SIGN = 'diverge_minus_converge'  # matches the existing Aim3 posterior index
HYST_PRIMARY_BINS = SWEEP_BINS.copy()  # 5,7,...,35 degrees
HYST_MIN_BINS_PER_SUBJECT = MIN_MATCHED_HYST_BINS
SPATIAL_MATCH_STIM_RMSE_TOL_DEG = 1.5
SPATIAL_MATCH_TIME_TOL_SEC = 0.10
SPATIAL_MAX_TIME_INTERPOLATION_GAP = 0.16
SPATIAL_MIN_POINTS_PER_BIN = 2
SPATIAL_MIN_VALID_BINS_PER_TRIAL = 3
SPATIAL_MIN_VALID_TRIALS_PER_DIRECTION_SIDE = 2
SPATIAL_MIN_ABS_DISPARITY = 4.0
SPATIAL_MAX_ABS_DISPARITY = 36.0
# Controls, if Haocheng provides their COUNTERFACTUAL forward-pass exports.
# Expected columns match the full model CSV, including loc_pred and stimuli.
# An absent file leads to an explicit 'not run', NOT simulated control results.
BCI_SPATIAL_CONTROL_FILES = {
    'no_causal_prior_adaptation': None,
    'no_spatial_persistence': None,
    'no_both_memory_components': None,
}
# Short new-output paths avoid the Windows 260-character legacy path limit.
# All original Aim 3A–3D output paths remain unchanged.
SPATIAL_OUT = OUT/'spatial_csv'
SPATIAL_FIG = OUT/'spatial_fig'
SPATIAL_OUT.mkdir(parents=True,exist_ok=True)
SPATIAL_FIG.mkdir(parents=True,exist_ok=True)

def spatial_save(frame,name):
    filepath = SPATIAL_OUT / name
    filepath.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(filepath, index=False)
    return frame

def _canonical_col(header,role):
    candidates={
      'subject':['subject','participant','participant_id','pid','subject_id','sub_id','Participant'],
      'trial':['trialIndex','trial_index','trial','trial_id','trialNum','trial_number','Trial'],
      'condition':['condition','condLabel','trialType','trial_type','condition_name','trialName','SweepDirection'],
      'direction':['SweepDirection','sweep_direction','direction','sweepType'],
      'time':['t','time','time_s','time_sec','time_seconds','elapsed_time',
              'trialTime','trial_time','time_from_onset','Time'],
      'mouse':['mouse_x','mouseX','cursor_x','cursorX','mouse_pos_x',
               'mouse_position_x','mousePosX','response_x','responseX',
               'response_position','reported_x','mouse_deg','mousePos','x_mouse'],
      'audio':['audio_x','audioX','audio_pos','audio_position','audioPos',
               'auditory_x','auditory_position','sound_x','soundPos','stim_audio_x'],
      'visual':['visual_x','visualX','visual_pos','visual_position','visualPos',
                'vis_x','stim_visual_x','stim_x'],
      'side':['side','signed_offset_deg','disparity_signed','signed_disparity_deg',
              'offsetDeg','sweep_signed_offset_deg','signed_offset']
    }
    requested=OBS_COLUMN_OVERRIDES.get(role)
    if requested is not None:
        if requested not in header:
            raise KeyError(f'OBS_COLUMN_OVERRIDES[{role!r}]={requested!r} not in CSV')
        return requested
    for name in candidates[role]:
        if name in header:
            return name
    return None

def _standardize_direction(cond):
    s=str(cond).lower()
    if 'diverg' in s or 'sweep_out' in s:return 'Diverge'
    if 'converg' in s or 'sweep_in' in s:return 'Converge'
    if 'audio_only' in s or 'auditory_only' in s:return 'AuditoryOnly'
    return 'Other'

def _normalize_trial_table(df,observed=False):
    if df.empty: return df
    df=df.copy()
    df['subject']=pid(df.subject)
    for c in ['trial','t','mouse_x','audio_x','visual_x','loc_pred','pC1','side_code']:
        if c in df: df[c]=pd.to_numeric(df[c],errors='coerce')
    df['trial']+=OBS_TRIAL_ID_SHIFT if observed else 0
    df['direction']=df.condition.map(_standardize_direction)
    if 'direction_hint' in df.columns:
        hint=df.direction_hint.map(_standardize_direction)
        df['direction']=df.direction.where(df.direction.ne('Other'),hint)
    df=df[df.subject.isin(cohort)&df.trial.notna()&df.t.between(0,18)].copy()
    # Use signed AUDIO MINUS VISUAL trajectory, not a nominal +/-40 deg column
    # which can have an arbitrary sign unrelated to actual sweep side.
    df['delta_x']=df.audio_x-df.visual_x
    df['actual_disparity']=df.delta_x.abs()
    df['actual_side']=np.sign(df.delta_x).fillna(0).astype('int8')
    if 'side_code' in df.columns:
        coded=np.sign(df.side_code).fillna(0).astype('int8')
        # no automatic override when coded sign conflicts with true trajectory.
        discrepancy=(coded.ne(0)&df.actual_side.ne(0)&coded.ne(df.actual_side))
        if discrepancy.any():
            warnings.warn(f'{int(discrepancy.sum())} rows have side metadata with '
                          'opposite sign from the actual audio-minus-visual trajectory; '
                          'actual trajectory sign is used.')
    return df

def _read_raw_observed(path):
    hdr=list(READ(path,nrows=0).columns)
    selected={k:_canonical_col(hdr,k) for k in
             ['subject','trial','condition','time','mouse','audio','visual','side','direction']}
    required=['subject','trial','condition','time','mouse','audio','visual']
    absent=[k for k in required if selected[k] is None]
    if absent:
        raise ValueError('Raw observed SL CSV is not usable; cannot locate '
                         f'{absent}. Columns supplied: {hdr}. Specify '
                         'OBS_COLUMN_OVERRIDES; do not substitute precomputed capture.')
    if len(set(selected[k] for k in required))!=len(required):
        raise ValueError('Two required roles resolved to one input column. Configure explicit overrides.')
    usecols=sorted(set(c for c in selected.values() if c is not None))
    rename={selected[r]:c for r,c in [('subject','subject'),('trial','trial'),
             ('condition','condition'),('time','t'),('mouse','mouse_x'),
             ('audio','audio_x'),('visual','visual_x'),('side','side_code'),
             ('direction','direction_hint')]
             if selected[r] is not None}
    spatial_save(pd.DataFrame([{'role':k,'source_column':v} for k,v in selected.items()]),
                 '01_observed_input_column_mapping.csv')
    pieces=[]
    for raw in READ(path,usecols=usecols,chunksize=250_000):
        raw=raw.rename(columns=rename)
        raw=_normalize_trial_table(raw,observed=True)
        raw=raw[raw.direction.isin(['Diverge','Converge','AuditoryOnly'])].copy()
        if raw.empty:continue
        pieces.append(raw[['subject','trial','condition','direction','t','mouse_x',
                           'audio_x','visual_x','delta_x','actual_disparity','actual_side']])
    if not pieces:return pd.DataFrame()
    result=pd.concat(pieces,ignore_index=True)
    if result.t.quantile(.95)>18:
        raise ValueError('Observed time appears not to be in trial-relative seconds.')
    # Explicit units check: a normalized [0,1] cursor cannot be compared to
    # a spatial model expressed in degrees. Never silently rescale.
    stim_scale=result[['audio_x','visual_x']].abs().stack().quantile(.95)
    cursor_scale=result.mouse_x.abs().quantile(.95)
    if stim_scale>12 and cursor_scale<2:
        raise ValueError('Observed cursor x appears normalized while model/stimulus use degrees. '
                         'Convert to calibrated degrees using the ORIGINAL task mapping first.')
    if cursor_scale>150:
        raise ValueError('Observed cursor x appears to use pixels or another non-degree unit. '
                         'Convert to the common programmed degree coordinate first.')
    # Mean duplicates after verifying they are not repeated independent trials.
    dup=result.duplicated(['subject','trial','t']).sum()
    spatial_save(pd.DataFrame([{'file':str(path),'n_rows':len(result),
         'n_subjects':result.subject.nunique(),'n_trials':result[['subject','trial']].drop_duplicates().shape[0],
         'duplicate_subject_trial_time_rows':int(dup),'trial_id_shift':OBS_TRIAL_ID_SHIFT}]),
         '02_observed_file_coverage.csv')
    if dup:
        raise ValueError(f'{dup} duplicate subject/trial/time rows. '
                         'Investigate original trial numbering before proceeding.')
    return result

def _read_model_spatial(path,pred_col='loc_pred'):
    header=list(READ(path,nrows=0).columns)
    needs=['subject','session','trial','condition','t','audio_x','visual_x',pred_col]
    require_columns(pd.DataFrame(columns=header),needs,f'BCI spatial export {path}')
    chunks=[]
    for raw in READ(path,usecols=needs,chunksize=250_000):
        raw=raw[raw.session.astype(str).str.upper().eq('SL')].copy()
        if raw.empty:continue
        raw=raw.rename(columns={pred_col:'loc_pred'})
        raw=_normalize_trial_table(raw)
        raw=raw[raw.direction.isin(['Diverge','Converge','AuditoryOnly'])].copy()
        if len(raw):
            chunks.append(raw[['subject','trial','condition','direction','t','loc_pred',
                               'audio_x','visual_x','delta_x','actual_disparity','actual_side']])
    if not chunks:return pd.DataFrame()
    model=pd.concat(chunks,ignore_index=True)
    if model.duplicated(['subject','trial','t']).any():
        raise ValueError('BCI spatial export has duplicate trial/time rows.')
    return model

def _trial_metadata(frame,kind):
    if frame.empty: return pd.DataFrame()
    rows=[]
    for (sub,tr),g in frame.groupby(['subject','trial'],sort=False):
        g=g.sort_values('t'); sweep=g[g.t>=12.]
        useful=sweep[(sweep.actual_disparity>=SPATIAL_MIN_ABS_DISPARITY)&
                     (sweep.actual_disparity<=SPATIAL_MAX_ABS_DISPARITY)]
        sides=useful.actual_side.replace(0,np.nan).dropna()
        side=int(sides.mode().iloc[0]) if len(sides) else 0
        early=g[g.t.between(11.0,11.9)].actual_disparity.median()
        late=g[g.t.between(17.0,18.0)].actual_disparity.median()
        # Approximate detected post-change onset; descriptive audit, not an
        # enforced exact 12.0s due to sampling and round-off.
        if g.direction.iloc[0]=='Diverge':
            onset=g[(g.t>=10)&(g.actual_disparity>1)].t.min()
        elif g.direction.iloc[0]=='Converge':
            onset=g[(g.t>=10)&(g.actual_disparity<39)].t.min()
        else:onset=np.nan
        t=np.sort(g.t.dropna().unique())
        rows.append({'subject':sub,'trial':tr,'source':kind,
          'direction':g.direction.iloc[0],'side':side,
          'condition':str(g.condition.iloc[0]),'n_samples':len(g),
          'time_start':float(t.min()) if len(t) else np.nan,
          'time_end':float(t.max()) if len(t) else np.nan,
          'median_dt':float(np.median(np.diff(t))) if len(t)>1 else np.nan,
          'disparity_at_11_to_12':early,'disparity_at_17_to_18':late,
          'detected_ramp_onset':onset,
          'n_valid_postramp_samples':len(useful),
          'n_unique_postramp_bins':len(set(5.+2.*np.rint((useful.actual_disparity-5.)/2.)))})
    return pd.DataFrame(rows)

def _interpolate_within_overlap(reference_t,reference_y,query_t,max_gap=SPATIAL_MAX_TIME_INTERPOLATION_GAP):
    # Do not interpolate across long gaps in missing cursor data.
    r=pd.DataFrame({'t':reference_t,'y':reference_y}).replace([np.inf,-np.inf],np.nan).dropna()
    r=r.groupby('t',as_index=False).y.mean().sort_values('t')
    times=r.t.to_numpy(float);values=r.y.to_numpy(float)
    query=np.asarray(query_t,float)
    out=np.full(len(query),np.nan)
    if len(times)<2:return out
    m=(query>=times[0])&(query<=times[-1])
    q=query[m];inds=np.searchsorted(times,q,side='left')
    left=np.maximum(inds-1,0);right=np.minimum(inds,len(times)-1)
    delta=times[right]-times[left]
    allowable=(delta<=max_gap)|np.isclose(delta,0)
    interp=np.interp(q,times,values);interp[~allowable]=np.nan
    out[m]=interp
    return out

def _pair_trial_audit(obs,model):
    om=_trial_metadata(obs,'observed');mm=_trial_metadata(model,'full_model')
    spatial_save(om,'03_observed_trial_coverage.csv')
    spatial_save(mm,'04_full_model_trial_coverage.csv')
    o=om.rename(columns={c:'obs_'+c for c in om if c not in ['subject','trial']})
    m=mm.rename(columns={c:'model_'+c for c in mm if c not in ['subject','trial']})
    audit=o.merge(m,on=['subject','trial'],how='outer',indicator=True,validate='one_to_one')
    audit['is_sweep_pair']=(audit.obs_direction.isin(['Diverge','Converge'])&
                            audit.model_direction.isin(['Diverge','Converge']))
    audit['direction_matches']=audit.obs_direction.eq(audit.model_direction)
    audit['side_matches']=audit.obs_side.eq(audit.model_side)&audit.obs_side.ne(0)
    audit['timing_coverage_ok']=(audit.obs_time_end>=15.0)&(audit.model_time_end>=15.0)
    audit['ramp_onset_ok']=(audit.obs_detected_ramp_onset.sub(12).abs()<=1.0)&(
                              audit.model_detected_ramp_onset.sub(12).abs()<=1.0)
    audit['nearest_time_diff_median_seconds']=np.nan
    audit['stim_a_rmse_deg']=np.nan
    audit['stim_v_rmse_deg']=np.nan
    audit['n_stim_matched_samples']=0
    # Compare actual acoustic and visual coordinates, rather than matching IDs alone.
    ogroups={(s,t):g for (s,t),g in obs.groupby(['subject','trial'],sort=False)
             if g.direction.iloc[0] in ['Diverge','Converge']}
    mgroups={(s,t):g for (s,t),g in model.groupby(['subject','trial'],sort=False)
             if g.direction.iloc[0] in ['Diverge','Converge']}
    for ix,row in audit[audit._merge.eq('both')&audit.direction_matches&
                       audit.side_matches].iterrows():
        key=(row.subject,row.trial)
        if key not in ogroups or key not in mgroups:continue
        ag=ogroups[key];bg=mgroups[key]
        q=bg[bg.t.between(12.0,18.0)]
        if len(q)==0:continue
        a=_interpolate_within_overlap(ag.t,ag.audio_x,q.t)
        v=_interpolate_within_overlap(ag.t,ag.visual_x,q.t)
        t_obs=np.sort(ag.t.dropna().unique())
        t_q=q.t.to_numpy(float)
        ii=np.searchsorted(t_obs,t_q)
        li=np.clip(ii-1,0,len(t_obs)-1);ri=np.clip(ii,0,len(t_obs)-1)
        nearest=np.minimum(abs(t_q-t_obs[li]),abs(t_q-t_obs[ri]))
        audit.loc[ix,'nearest_time_diff_median_seconds']=np.median(nearest)
        good=np.isfinite(a)&np.isfinite(v)&np.isfinite(q.audio_x)&np.isfinite(q.visual_x)
        audit.loc[ix,'n_stim_matched_samples']=int(good.sum())
        if good.sum()<20:continue
        audit.loc[ix,'stim_a_rmse_deg']=np.sqrt(np.mean((a[good]-q.audio_x.to_numpy()[good])**2))
        audit.loc[ix,'stim_v_rmse_deg']=np.sqrt(np.mean((v[good]-q.visual_x.to_numpy()[good])**2))
    audit['time_alignment_ok']=audit.nearest_time_diff_median_seconds.le(SPATIAL_MATCH_TIME_TOL_SEC)
    audit['stimulus_alignment_ok']=(audit.n_stim_matched_samples>=20)&(
        audit.stim_a_rmse_deg<=SPATIAL_MATCH_STIM_RMSE_TOL_DEG)&(
        audit.stim_v_rmse_deg<=SPATIAL_MATCH_STIM_RMSE_TOL_DEG)
    audit['strict_trial_match']=audit._merge.eq('both')&audit.is_sweep_pair&audit.direction_matches&\
         audit.side_matches&audit.timing_coverage_ok&audit.ramp_onset_ok&\
         audit.time_alignment_ok&audit.stimulus_alignment_ok
    audit['why_not_strict_match']=np.select([
        audit._merge.ne('both'),~audit.is_sweep_pair,~audit.direction_matches,~audit.side_matches,
        ~audit.timing_coverage_ok,~audit.ramp_onset_ok,~audit.time_alignment_ok,
        ~audit.stimulus_alignment_ok],
        ['missing_other_source','unisensory_or_non_sweep','direction_mismatch','side_mismatch',
         'insufficient_time_coverage','nonstandard_sweep_onset',
         'time_alignment_mismatch','stimulus_mismatch'],
        default='strict_match')
    spatial_save(audit,'05_observed_model_trial_matching_audit.csv')
    spatial_save(audit.groupby(['why_not_strict_match'],dropna=False,as_index=False).agg(
        n_trials=('trial','size'),n_subjects=('subject','nunique')),
        '06_trial_exclusion_summary.csv')
    return audit

def _capture_bin_rows(frame,output_col,source,strict_keys=None,lag_map=None):
    """Trial-first 20-Hz-style sweep histogram; no response information from another task."""
    if frame.empty:return pd.DataFrame()
    out=[]
    for (sub,tr),trial in frame.groupby(['subject','trial'],sort=False):
        if trial.direction.iloc[0] not in ['Diverge','Converge']:continue
        if strict_keys is not None and (sub,tr) not in strict_keys:continue
        g=trial.sort_values('t').copy()
        if lag_map is not None:
            lag=lag_map.get(sub,np.nan)
            if not np.isfinite(lag):continue
            # Lag correction: response at t+lag is compared with stimulus at t.
            # Values outside observed support are explicitly set to NaN.
            g['r']=_interpolate_within_overlap(g.t,g[output_col],g.t+lag)
        else:g['r']=g[output_col]
        # The direction is taken from stimulus, not cursor sign.
        g=g[g.t>=12.0]
        g=g[g.actual_disparity.ge(SPATIAL_MIN_ABS_DISPARITY)&
            g.actual_disparity.lt(SPATIAL_MAX_ABS_DISPARITY)].copy()
        g=g[g.actual_side.ne(0)&np.isfinite(g.r)&np.isfinite(g.audio_x)&
            np.isfinite(g.visual_x)].copy()
        if g.empty:continue
        sides=g.actual_side.mode()
        if sides.empty:continue
        side=int(sides.iloc[0]);g=g[g.actual_side.eq(side)]
        g['disp_bin']=5.+2.*np.rint((g.actual_disparity-5.)/2.)
        g=g[g.disp_bin.isin(HYST_PRIMARY_BINS)]
        if g.empty:continue
        # Visual-directed spatial bias, sign-invariant across left/right trials.
        g['capture_deg']=(g.r-g.audio_x)*np.sign(g.visual_x-g.audio_x)
        g['capture_norm']=g['capture_deg']/g.actual_disparity
        rows=g.groupby('disp_bin',as_index=False).agg(
            visual_capture_deg=('capture_deg','mean'),
            normalized_capture=('capture_norm','mean'),n_samples=('capture_deg','size'))
        rows=rows[rows.n_samples>=SPATIAL_MIN_POINTS_PER_BIN]
        if len(rows)<SPATIAL_MIN_VALID_BINS_PER_TRIAL:continue
        rows['subject']=sub;rows['trial']=tr;rows['direction']=trial.direction.iloc[0]
        rows['side']=side;rows['source']=source
        out.append(rows)
    return pd.concat(out,ignore_index=True) if out else pd.DataFrame()

def _make_hysteresis_subjects(trial_bins,metric,quality_mode):
    if trial_bins.empty:return pd.DataFrame(),pd.DataFrame()
    # Each TRIAL contributes equally within condition/side/disparity; then L/R
    # contribute equally; then disparity bins contribute equally.
    side=trial_bins.groupby(['subject','direction','side','disp_bin'],as_index=False).agg(
        value=(metric,'mean'),n_trials=('trial','nunique'))
    side=side[side.n_trials>=SPATIAL_MIN_VALID_TRIALS_PER_DIRECTION_SIDE]
    sides=side.pivot_table(index=['subject','direction','disp_bin'],columns='side',values='value')
    if not {-1,1}.issubset(sides.columns):return pd.DataFrame(),pd.DataFrame()
    sides=sides[[-1,1]].dropna().copy()  # explicitly require both signs
    sides['value']=(sides[-1]+sides[1])/2.
    curve=sides[['value']].reset_index()
    two=curve.pivot(index=['subject','disp_bin'],columns='direction',values='value')
    if not {'Diverge','Converge'}.issubset(two.columns):return pd.DataFrame(),curve
    two=two[['Diverge','Converge']].dropna().copy()
    two['hysteresis']=two.Diverge-two.Converge
    bysub=two.groupby(level=0).agg(hysteresis=('hysteresis','mean'),
        n_common_bins=('hysteresis','size')).reset_index()
    bysub['passes_min_bins']=bysub.n_common_bins>=HYST_MIN_BINS_PER_SUBJECT
    bysub['source']=trial_bins.source.iloc[0]
    bysub['mode']=quality_mode
    return bysub,curve

def _lag_from_unisensory(data,response_col,source):
    """Estimate lag independently for observed and model output from auditory-only trials."""
    rows=[]
    g=data[data.direction.eq('AuditoryOnly')]
    for (sub,tr),a in g.groupby(['subject','trial'],sort=False):
        a=a.sort_values('t')
        ix=np.arange(max(.5,float(a.t.min())),min(17.5,float(a.t.max()))+.001,.05)
        if len(ix)<80:continue
        stim=_interpolate_within_overlap(a.t,a.audio_x,ix)
        response=_interpolate_within_overlap(a.t,a[response_col],ix)
        if not np.isfinite(stim).all() or np.std(stim)<3.:continue
        vals=[]
        for shift in np.arange(0,4.001,.05):
            rr=_interpolate_within_overlap(ix,response,ix+shift)
            ok=np.isfinite(rr)&np.isfinite(stim)
            if ok.sum()<70 or np.std(rr[ok])<1e-7:vals.append(np.nan)
            else: vals.append(np.corrcoef(stim[ok],rr[ok])[0,1])
        if not np.isfinite(vals).any():continue
        best=int(np.nanargmax(vals))
        if vals[best]<.40:continue
        rows.append({'subject':sub,'trial':tr,'source':source,
             'lag_seconds':best*.05,'peak_r':vals[best]})
    lags=pd.DataFrame(rows)
    if not len(lags):return {},lags
    good=lags.groupby('subject').agg(lag_seconds=('lag_seconds','median'),
                                     n_valid_unisensory_trials=('trial','nunique')).reset_index()
    spatial_save(good,f'12_{source}_independent_auditory_lag_by_subject.csv')
    return dict(zip(good.subject,good.lag_seconds)),lags

def _corr_row(frame,x,y,label,seed):
    rr=correlations(frame,x,y,label,seed=seed)
    return rr

def _reliability_split(trial_bins,metric,mode,source,n_repeats=100):
    if trial_bins.empty:return pd.DataFrame(),pd.DataFrame()
    rng=np.random.default_rng(SEED+4501)
    rows=[]
    for iteration in range(n_repeats):
        allocations=[]
        for (sub,direction,side),g in trial_bins.groupby(['subject','direction','side']):
            ids=np.sort(g.trial.unique())
            if len(ids)<4:continue
            shuffled=rng.permutation(ids)
            split=dict(zip(shuffled,np.arange(len(shuffled))%2))
            alloc=g.copy();alloc['half']=alloc.trial.map(split)
            allocations.append(alloc)
        if not allocations:continue
        half_bins=pd.concat(allocations,ignore_index=True)
        scores=[]
        for half,g in half_bins.groupby('half'):
            sh,_=_make_hysteresis_subjects(g,metric,mode)
            if sh.empty:continue
            sh=sh[sh.passes_min_bins][['subject','hysteresis']].copy()
            sh['half']=half;scores.append(sh)
        if len(scores)!=2:continue
        w=pd.concat(scores).pivot(index='subject',columns='half',values='hysteresis').dropna()
        if len(w)<8 or w[0].std()<1e-9 or w[1].std()<1e-9:continue
        r=float(np.corrcoef(w[0],w[1])[0,1])
        rows.append({'source':source,'mode':mode,'split':iteration,
                     'n_subjects':len(w),'pearson_half_correlation':r,
                     'spearman_brown_corrected':2*r/(1+r) if abs(1+r)>1e-8 else np.nan})
    reli=pd.DataFrame(rows)
    summary=(pd.DataFrame([{'source':source,'mode':mode,'n_successful_splits':len(reli),
       'median_split_half_r':reli.pearson_half_correlation.median() if len(reli) else np.nan,
       'median_spearman_brown':reli.spearman_brown_corrected.median() if len(reli) else np.nan,
       'reliability_CI_low':reli.spearman_brown_corrected.quantile(.025) if len(reli) else np.nan,
       'reliability_CI_high':reli.spearman_brown_corrected.quantile(.975) if len(reli) else np.nan}]))
    return reli,summary

def _spatial_predictions_pair(obs_bins,mod_bins,mode,match_label):
    subj_ob,curve_ob=_make_hysteresis_subjects(obs_bins,'visual_capture_deg',mode)
    subj_md,curve_md=_make_hysteresis_subjects(mod_bins,'visual_capture_deg',mode)
    if subj_ob.empty or subj_md.empty:return None
    obs2=subj_ob[['subject','hysteresis','n_common_bins','passes_min_bins']].rename(
      columns={'hysteresis':'observed_hysteresis_deg',
               'n_common_bins':'observed_n_bins',
               'passes_min_bins':'observed_pass'})
    md2=subj_md[['subject','hysteresis','n_common_bins','passes_min_bins']].rename(
      columns={'hysteresis':'BCI_PredictedSpatial_Hysteresis',
               'n_common_bins':'predicted_n_bins',
               'passes_min_bins':'predicted_pass'})
    # Require identical subject/disparity bins rather than comparing separate ranges.
    co=curve_ob.pivot(index=['subject','disp_bin'],columns='direction',values='value')
    cm=curve_md.pivot(index=['subject','disp_bin'],columns='direction',values='value')
    op=co[['Diverge','Converge']].dropna().copy()
    mp=cm[['Diverge','Converge']].dropna().copy()
    op['observed_deg']=op.Diverge-op.Converge
    mp['predicted_deg']=mp.Diverge-mp.Converge
    shared=op[['observed_deg']].join(mp[['predicted_deg']],how='inner').dropna().reset_index()
    if shared.empty:return None
    matched=shared.groupby('subject',as_index=False).agg(
        observed_hysteresis_deg=('observed_deg','mean'),
        BCI_PredictedSpatial_Hysteresis=('predicted_deg','mean'),
        n_shared_bins=('disp_bin','nunique'))
    matched=matched[matched.n_shared_bins>=HYST_MIN_BINS_PER_SUBJECT]
    matched=matched.merge(base[['subject','r2_holdout','n_at_bound','delta50_deg','bci_pC20']],
                          on='subject',how='left',validate='one_to_one')
    label=match_label+'_'+mode
    spatial_save(shared,f'20_{label}_shared_disparity_bins.csv')
    spatial_save(matched,f'21_{label}_individual_hysteresis.csv')
    if len(matched)>=5:
        r=_corr_row(matched,'BCI_PredictedSpatial_Hysteresis',
              'observed_hysteresis_deg',label,SEED+501)
        err=matched.BCI_PredictedSpatial_Hysteresis-matched.observed_hysteresis_deg
        r.update(mean_signed_prediction_error_deg=float(err.mean()),
          mean_abs_prediction_error_deg=float(err.abs().mean()),
          rmse_prediction_error_deg=float(np.sqrt(np.mean(err**2))))
        spatial_save(pd.DataFrame([r]),f'22_{label}_primary_correlation.csv')
        qual=[]
        for name,keep in [('all',np.ones(len(matched),bool)),
             ('R2_ge_0p50',matched.r2_holdout>=FIT_R2_STRICT),
             ('no_parameter_bounds',matched.n_at_bound.eq(0)),
             ('both',matched.r2_holdout.ge(FIT_R2_STRICT)&matched.n_at_bound.eq(0))]:
            qual.append(_corr_row(matched.loc[keep],
                  'BCI_PredictedSpatial_Hysteresis','observed_hysteresis_deg',
                  name,SEED+502+len(qual)))
        spatial_save(pd.DataFrame(qual),f'23_{label}_model_quality_sensitivity.csv')
        fig,axs=plt.subplots(1,2,figsize=(12.8,5.1))
        axs[0].scatter(matched.BCI_PredictedSpatial_Hysteresis,
                       matched.observed_hysteresis_deg,alpha=.75)
        vals=np.r_[matched.BCI_PredictedSpatial_Hysteresis.to_numpy(),
                    matched.observed_hysteresis_deg.to_numpy()]
        if np.isfinite(vals).any():
            lo,hi=np.nanmin(vals),np.nanmax(vals)
            axs[0].plot([lo,hi],[lo,hi],ls='--',color='gray',label='Identity')
        axs[0].set(xlabel='Predicted spatial hysteresis (deg)',
                 ylabel='Observed spatial hysteresis (deg)',
                 title=f'{match_label}: {mode}\nr={r["pearson_r"]:.3f}; N={r["n"]}')
        axs[0].legend(fontsize=8)
        mm=shared[shared.subject.isin(matched.subject)]
        ss=mm.groupby('disp_bin',as_index=False).agg(observed=('observed_deg','mean'),
                                                     predicted=('predicted_deg','mean'))
        axs[1].plot(ss.disp_bin,ss.observed,'o-',label='Observed')
        axs[1].plot(ss.disp_bin,ss.predicted,'s--',label='BCI predicted')
        axs[1].axhline(0,color='gray',ls=':')
        axs[1].set(xlabel='Matched instantaneous disparity (deg)',
                   ylabel='Diverge − Converge (visual-directed deg)',
                   title='Same units, same participant/disparity bins')
        axs[1].legend(fontsize=8)
        fig.tight_layout();fig.savefig(SPATIAL_FIG/f'{label}_agreement.png',dpi=220)
        plt.show()
        print(f'SPATIAL {label}: N={r["n"]}; r={r["pearson_r"]:.3f}; '
              f'permutation p={r["permutation_p"]:.4g}; RMSE={r["rmse_prediction_error_deg"]:.2f}°')
    # Repeat using normalized capture, without clipping the weights to [0,1].
    ono,oc=_make_hysteresis_subjects(obs_bins,'normalized_capture',mode)
    mno,mc=_make_hysteresis_subjects(mod_bins,'normalized_capture',mode)
    if len(oc) and len(mc):
        oo=oc.pivot(index=['subject','disp_bin'],columns='direction',values='value')
        mp=mc.pivot(index=['subject','disp_bin'],columns='direction',values='value')
        oo=oo[['Diverge','Converge']].dropna()
        mp=mp[['Diverge','Converge']].dropna()
        norm=pd.DataFrame({'obs':oo.Diverge-oo.Converge}).join(
            pd.DataFrame({'pred':mp.Diverge-mp.Converge}),how='inner').dropna().reset_index()
        norm=norm.groupby('subject',as_index=False).agg(
            observed_normalized=('obs','mean'),predicted_normalized=('pred','mean'),
            n_bins=('disp_bin','nunique'))
        norm=norm[norm.n_bins>=HYST_MIN_BINS_PER_SUBJECT]
        spatial_save(norm,f'24_{label}_normalized_capture_secondary.csv')
        if len(norm)>=5:
            spatial_save(pd.DataFrame([_corr_row(norm,'predicted_normalized',
                 'observed_normalized',f'{label}_normalized_secondary',SEED+520)]),
                 f'25_{label}_normalized_correlation.csv')
    return matched

# Invariant-preserving explicit name for the ORIGINAL Aim3 pC1 hysteresis.
if 'hist' in locals() and isinstance(hist,pd.DataFrame):
    hist['BCI_CausalPosterior_Hysteresis']=hist['bci_hysteresis_mean']
    spatial_save(hist[['subject','BCI_CausalPosterior_Hysteresis','n_matched_bins']],
                 '00_BCI_CausalPosterior_Hysteresis_original_unchanged.csv')

spatial_controls=[]
observed_path=Path(OBS_SL_TIME_CSV) if OBS_SL_TIME_CSV is not None else None
if observed_path is None or not observed_path.is_file():
    spatial_save(pd.DataFrame([{'status':'NOT_RUN_MISSING_RAW_OBSERVED_SL',
        'required_input':'OBS_SL_TIME_CSV',
        'explanation':'Set path to original time-resolved observed cursor/stimulus table. '
                      'Window-level hysteresis summaries cannot replace the raw source.'}]),
        '00_SPATIAL_ANALYSIS_STATUS.csv')
    warnings.warn('NEW spatial-prediction analysis not run: set OBS_SL_TIME_CSV '
                  'to an original time-resolved SL file. Original Aim3A-3D unchanged.')
else:
    spatial_save(pd.DataFrame([{'status':'INPUT_FILE_FOUND','path':str(observed_path)}]),
                 '00_SPATIAL_ANALYSIS_STATUS.csv')
    obs_sl=_read_raw_observed(observed_path)
    full_sl=_read_model_spatial(BCI_TIME_CSV)
    if obs_sl.empty or full_sl.empty:
        warnings.warn('No valid time-resolved observed/model sweeps after filtering. New analysis skipped.')
    else:
        matchaudit=_pair_trial_audit(obs_sl,full_sl)
        good_trials=set(map(tuple,matchaudit.loc[matchaudit.strict_trial_match,
                                                  ['subject','trial']].to_numpy()))
        strict_subs=len({x[0] for x in good_trials})
        print(f'Spatial audit: {len(good_trials)} strict trial matches across {strict_subs} participants. '
              'See trial exclusion CSV for details.')
        # PRIMARY: only true stimulus-matched trials. No fallback pretending
        # independent random stimulus trajectories constitute paired trials.
        obs_raw=_capture_bin_rows(obs_sl,'mouse_x','observed_raw',good_trials)
        full_raw=_capture_bin_rows(full_sl,'loc_pred','BCI_full_raw',good_trials)
        if len(obs_raw) and len(full_raw):
            spatial_save(obs_raw,'07_matched_observed_raw_trial_bins.csv')
            spatial_save(full_raw,'08_matched_full_model_raw_trial_bins.csv')
            spatial_primary=_spatial_predictions_pair(obs_raw,full_raw,'raw','strict_trial_matched')
            for source,df in [('observed',obs_raw),('BCI_full',full_raw)]:
                rep,summary=_reliability_split(df,'visual_capture_deg','raw',source)
                spatial_save(rep,f'26_{source}_raw_split_half_iterations.csv')
                spatial_save(summary,f'27_{source}_raw_split_half_summary.csv')
        else:
            spatial_primary=None
        if spatial_primary is None:
            spatial_save(pd.DataFrame([{'status':'INSUFFICIENT_VERIFIED_TRIALS',
                   'n_strict_trial_matches':len(good_trials),'n_subjects':strict_subs,
                   'note':'No primary correlation produced. Inspect trial numbering, '
                          'stimulus trajectories, and raw response quality.'}]),
                   '09_primary_spatial_prediction_not_estimable.csv')
        # SECONDARY: condition/side/disparity matched, but NOT same physical
        # trajectory; explicitly labeled as an exploratory unmatched-stimulus test.
        all_obs=_capture_bin_rows(obs_sl,'mouse_x','observed_all')
        all_full=_capture_bin_rows(full_sl,'loc_pred','full_all')
        if len(all_obs) and len(all_full):
            spatial_save(all_obs,'10_all_observed_sweep_raw_trial_bins.csv')
            spatial_save(all_full,'11_all_model_sweep_raw_trial_bins.csv')
            _spatial_predictions_pair(all_obs,all_full,'raw',
                'exploratory_condition_side_matched_not_same_stimuli')
        # SECONDARY LAG CORRECTION: auditory-only-derived delays computed
        # INDEPENDENTLY for each source. If unisensory files absent it is skipped.
        obs_lag,obs_lag_trials=_lag_from_unisensory(obs_sl,'mouse_x','observed')
        mod_lag,mod_lag_trials=_lag_from_unisensory(full_sl,'loc_pred','BCI_full')
        spatial_save(obs_lag_trials,'13_observed_auditory_only_lag_trials.csv')
        spatial_save(mod_lag_trials,'14_model_auditory_only_lag_trials.csv')
        spatial_save(pd.DataFrame([{'observed_subjects_with_lag':len(obs_lag),
                                    'model_subjects_with_lag':len(mod_lag),
                                    'common_subjects_with_lag':len(set(obs_lag)&set(mod_lag))}]),
                     '15_lag_correction_coverage.csv')
        if obs_lag and mod_lag:
            obs_lagbins=_capture_bin_rows(obs_sl,'mouse_x','observed_lag',good_trials,obs_lag)
            mod_lagbins=_capture_bin_rows(full_sl,'loc_pred','model_lag',good_trials,mod_lag)
            if len(obs_lagbins) and len(mod_lagbins):
                spatial_save(obs_lagbins,'16_lag_corrected_observed_trial_bins.csv')
                spatial_save(mod_lagbins,'17_lag_corrected_model_trial_bins.csv')
                _spatial_predictions_pair(obs_lagbins,mod_lagbins,'lag_corrected',
                                         'strict_trial_matched')
        # Counterfactual controls: read ONLY supplied predictions from Haocheng.
        # They MUST be forward-simulated with same stimuli and unchanged remaining
        # parameters and response/motor stage; refitted controls are DIFFERENT.
        for label,filepath in BCI_SPATIAL_CONTROL_FILES.items():
            status={'control':label,'path':str(filepath) if filepath is not None else '',
                 'mode':'counterfactual_ablation_not_refitted','status':'NOT_PROVIDED'}
            if filepath is not None and Path(filepath).is_file():
                try:
                    cmodel=_read_model_spatial(Path(filepath))
                    cmet=_trial_metadata(cmodel,label)
                    ckey=cmet[['subject','trial','direction','side']]
                    fk=_trial_metadata(full_sl,'full_model')[['subject','trial','direction','side']]
                    both=ckey.merge(fk,on=['subject','trial','direction','side'],how='inner')
                    keys=set(map(tuple,both[['subject','trial']].to_numpy()))&good_trials
                    # Reject ablation exports with different stimulus realizations.
                    cf={key:g for key,g in cmodel.groupby(['subject','trial']) if key in keys}
                    ff={key:g for key,g in full_sl.groupby(['subject','trial']) if key in keys}
                    consistent=set()
                    cf_audit=[]
                    for key in keys:
                        a=cf.get(key);f=ff.get(key)
                        if a is None or f is None:continue
                        tt=f.t.to_numpy(float)
                        aa=_interpolate_within_overlap(a.t,a.audio_x,tt)
                        vv=_interpolate_within_overlap(a.t,a.visual_x,tt)
                        ok=np.isfinite(aa)&np.isfinite(vv)
                        if ok.sum()<40:continue
                        ra=float(np.sqrt(np.mean((aa[ok]-f.audio_x.to_numpy()[ok])**2)))
                        rv=float(np.sqrt(np.mean((vv[ok]-f.visual_x.to_numpy()[ok])**2)))
                        pass_qc=(ra<=SPATIAL_MATCH_STIM_RMSE_TOL_DEG and
                                 rv<=SPATIAL_MATCH_STIM_RMSE_TOL_DEG)
                        cf_audit.append({'subject':key[0],'trial':key[1],
                            'auditory_stim_rmse_deg':ra,'visual_stim_rmse_deg':rv,
                            'identical_stimulus_check_pass':pass_qc})
                        if pass_qc: consistent.add(key)
                    spatial_save(pd.DataFrame(cf_audit),f'29_{label}_control_stimulus_audit.csv')
                    keys=keys&consistent
                    cbins=_capture_bin_rows(cmodel,'loc_pred',label,keys)
                    # Score full model and control on same subject/trials only.
                    control_obs=_capture_bin_rows(obs_sl,'mouse_x','observed_control',keys)
                    control_full=_capture_bin_rows(full_sl,'loc_pred','full_control',keys)
                    if len(cbins) and len(control_obs):
                        spatial_save(cbins,f'30_{label}_trial_bins.csv')
                        cr=_spatial_predictions_pair(control_obs,cbins,'raw',
                              f'counterfactual_{label}')
                        fr=_spatial_predictions_pair(control_obs,control_full,'raw',
                              f'counterfactual_comparator_full_for_{label}') if len(control_full) else None
                        status['status']='RUN' if cr is not None and fr is not None else 'INSUFFICIENT_COVERAGE'
                        status['n_candidate_trial_matches']=len(keys)
                        if cr is not None and fr is not None:
                            compare=fr[['subject','observed_hysteresis_deg',
                                 'BCI_PredictedSpatial_Hysteresis']].rename(
                                 columns={'BCI_PredictedSpatial_Hysteresis':'full_predicted_deg'})
                            compare=compare.merge(cr[['subject','BCI_PredictedSpatial_Hysteresis']].rename(
                                columns={'BCI_PredictedSpatial_Hysteresis':'ablated_predicted_deg'}),
                                on='subject',how='inner',validate='one_to_one')
                            compare['full_absolute_error']=abs(compare.full_predicted_deg-
                                                                 compare.observed_hysteresis_deg)
                            compare['ablation_absolute_error']=abs(compare.ablated_predicted_deg-
                                                                     compare.observed_hysteresis_deg)
                            compare['ablation_minus_full_abs_error']=(compare.ablation_absolute_error-
                                                                        compare.full_absolute_error)
                            spatial_save(compare,f'32_{label}_full_vs_ablation_paired_errors.csv')
                            group_test=test_signflip(compare.ablation_minus_full_abs_error,
                                                     seed=SEED+2800+len(spatial_controls))
                            spatial_save(pd.DataFrame([{'control':label,
                                'n_matched_subjects':len(compare),**group_test}]),
                                f'33_{label}_full_vs_ablation_error_signflip.csv')
                    else:status['status']='INSUFFICIENT_COVERAGE'
                except Exception as exc:
                    status['status']='ERROR';status['error']=repr(exc)
                    warnings.warn(f'Counterfactual control {label} failed: {exc}')
            spatial_controls.append(status)
    spatial_save(pd.DataFrame(spatial_controls if spatial_controls else [
        {'control':k,'status':'NOT_RUN_NO_SPATIAL_INPUT'} for k in BCI_SPATIAL_CONTROL_FILES]),
        '31_counterfactual_model_control_availability.csv')
if not (SPATIAL_OUT/'31_counterfactual_model_control_availability.csv').exists():
    spatial_save(pd.DataFrame([{'control':name,'status':'NOT_RUN_NO_RAW_OBSERVED_INPUT'}
                               for name in BCI_SPATIAL_CONTROL_FILES]),
                 '31_counterfactual_model_control_availability.csv')
print('3C spatial prediction additions complete. See:',SPATIAL_OUT)


# ============================================================================
# 3C CALIBRATION: MODEL P(C=1) VS OBSERVED SL SAME/DIFFERENT REPORT
# Calibration is diagnostic, not part of the independently fitted BCI phenotype.
# The binary report is NOT a literal observation of the model's latent posterior.
# ============================================================================
ft=fixed_trial[fixed_trial.window.eq('main')].copy()
ft['trialIndex']=ft.trial
cal=ft.merge(causal[['subject','trialIndex','same_response','signed_disparity_deg']],
             on=['subject','trialIndex'],how='inner',validate='one_to_one')
cal=cal[cal.same_response.isin([0,1]) & cal.pC1.between(0,1)].copy()
cal['bci_logit']=logit(cal.pC1.clip(EPS,1-EPS))
match_counts={'n_model_fixed_trials':int(len(ft)), 'n_matched_with_explicit_report':int(len(cal)),
              'match_fraction':float(len(cal)/len(ft)) if len(ft) else np.nan,
              'n_matched_subjects':int(cal.subject.nunique())}
persist(pd.DataFrame([match_counts]),'aim3C_BCI_calibration_join_audit.csv')
print('Calibration join audit:',match_counts)
if match_counts['match_fraction']<.5:
    warnings.warn('Poor model trial <-> Aim1 trialIndex match. Calibration skipped; verify trial numbering.')
elif len(cal)>100 and cal.same_response.nunique()==2:
    cl=sm.GLM(cal.same_response,sm.add_constant(cal[['bci_logit']]),
              family=sm.families.Binomial()).fit(cov_type='cluster',cov_kwds={'groups':cal.subject})
    (MODEL_OUT/'aim3C_BCI_report_calibration_cluster_robust.txt').write_text(str(cl.summary()),encoding='utf-8')
    # Test against ideal intercept=0, slope=1 (NOT slope vs zero).
    test_int=cl.t_test('const = 0');test_slope=cl.t_test('bci_logit = 1')
    persist(pd.DataFrame([{'n_trials':len(cal),'n_subjects':cal.subject.nunique(),
       'intercept':cl.params['const'],'intercept_se':cl.bse['const'],
       'slope':cl.params['bci_logit'],'slope_se':cl.bse['bci_logit'],
       'intercept_vs_zero_p':float(test_int.pvalue),
       'slope_vs_one_p':float(test_slope.pvalue)}]),'aim3C_BCI_report_calibration_coefficients.csv')
    # Global leave-one-PARTICIPANT-out calibration: no subject's own Same reports used.
    groups=cal.subject.to_numpy(); yy=cal.same_response.to_numpy(int); xx=cal[['bci_logit']].to_numpy(float)
    gp=np.full(len(cal),np.nan)
    for tr,te in LeaveOneGroupOut().split(xx,yy,groups):
        if len(np.unique(yy[tr]))<2:continue
        mod=LogisticRegression(C=1e4,max_iter=2000).fit(xx[tr],yy[tr])
        gp[te]=mod.predict_proba(xx[te])[:,1]
    cal['p_calibrated_LOPO_global']=gp
    # Exploratory personal decision mapping, held-out *trials* within person;
    # unlike raw BCI pC, this estimator uses explicit reports and is NOT implicit.
    personal=np.full(len(cal),np.nan);rg=np.random.default_rng(SEED+888)
    for subject,ids in cal.groupby('subject').indices.items():
        ids=np.asarray(ids)
        if len(ids)<20 or len(np.unique(yy[ids]))<2:continue
        signed=np.round(cal.iloc[ids].signed_disparity_deg.to_numpy(float)/10).astype(int)
        folds=np.zeros(len(ids),int)
        for val in np.unique(signed):
            ix=np.where(signed==val)[0];rg.shuffle(ix);folds[ix]=np.arange(len(ix))%5
        for k in range(5):
            it=ids[folds!=k];ie=ids[folds==k]
            if len(ie)==0 or len(np.unique(yy[it]))<2:continue
            mod=LogisticRegression(C=1.0,max_iter=2000).fit(xx[it],yy[it])
            personal[ie]=mod.predict_proba(xx[ie])[:,1]
    cal['p_calibrated_personal_CV']=personal
    persist(cal,'aim3C_BCI_report_calibration_trial_predictions.csv')
    pr=[]
    for subject,g in cal.groupby('subject'):
        for model,column in [('raw_model_pC1','pC1'),('global_mapping_LOPO','p_calibrated_LOPO_global'),
                             ('personal_mapping_5fold','p_calibrated_personal_CV')]:
            v=g[g[column].notna()]
            if len(v)<4:continue
            pr.append({'subject':subject,'model':model,'n_trials':len(v),
              'log_loss':log_loss(v.same_response,v[column].clip(EPS,1-EPS),labels=[0,1]),
              'brier':np.mean((v.same_response-v[column])**2)})
    pr=pd.DataFrame(pr);persist(pr,'aim3C_BCI_report_calibration_participant_scores.csv')
    pw=pr.pivot(index='subject',columns='model',values='log_loss')
    comps=[]
    for model in ['global_mapping_LOPO','personal_mapping_5fold']:
        if model in pw:
            mm=pw[['raw_model_pC1',model]].dropna()
            comps.append({'comparison':'raw_minus_'+model,**test_signflip(
               mm.raw_model_pC1-mm[model],seed=SEED+600+len(comps))})
    if {'global_mapping_LOPO','personal_mapping_5fold'}.issubset(pw.columns):
        mm=pw[['global_mapping_LOPO','personal_mapping_5fold']].dropna()
        comps.append({'comparison':'global_minus_personal',**test_signflip(
           mm.global_mapping_LOPO-mm.personal_mapping_5fold,seed=SEED+609)})
    persist(pd.DataFrame(comps),'aim3C_BCI_report_calibration_CV_comparisons.csv')

# ============================================================================
# 3D: ORIGINAL HINGE TESTS
# ============================================================================
for x in [cap,bp]:
    x['boundary_below']=np.minimum(x.cf_boundary_distance_deg,0)
    x['boundary_above']=np.maximum(x.cf_boundary_distance_deg,0)
ch=gee('VisualCapture_audioLag ~ boundary_below + boundary_above',cap)
dh=gee('dynamically_updating ~ boundary_below + boundary_above',bp,True)
result_table(ch,'aim3D_capture_boundary_hinge_crossfit_GEE')
result_table(dh,'aim3D_dynamicity_boundary_hinge_crossfit_GEE')
kr=[]
for label,m in [('visual_capture',ch),('breakpoint_dynamicity',dh)]:
    w=m.wald_test('boundary_below = boundary_above',scalar=True)
    a,b=m.params['boundary_below'],m.params['boundary_above']
    kr.append({'outcome':label,'below_boundary_slope':a,'above_boundary_slope':b,
       'slope_change_wald_stat':float(np.asarray(w.statistic).squeeze()),
       'slope_change_p':float(np.asarray(w.pvalue).squeeze()),
       'local_maximum_at_boundary':bool(a>0 and b<0) if label=='breakpoint_dynamicity' else np.nan})
kink=persist(pd.DataFrame(kr),'aim3D_crossfit_boundary_kink_summary.csv')

# ============================================================================
# 3D: PARTICIPANT-HELD-OUT PREDICTION, ORIGINAL PAIR + PHYSICAL-ONLY BASELINE
# ============================================================================
FEATURES={'physical_only':['offset_deg','offset_sq'],
          'physical_plus_gaussian_parameters':['offset_deg','offset_sq','delta50_deg','slope50_per_deg'],
          'gaussian_causal_coordinates':['cf_p_same_hat','psame_sq','cf_causal_ambiguity','cf_boundary_z']}
pred_all=[];perf_all=[];paired_all=[]
for outcome,binary,frame in [('visual_capture',False,cap),('breakpoint_dynamicity',True,bp)]:
    outcome_col='dynamically_updating' if binary else 'VisualCapture_audioLag'
    need=list(dict.fromkeys(['subject',outcome_col]+sum(FEATURES.values(),[])))
    data=frame.copy();data['offset_sq']=data.offset_deg**2;data['psame_sq']=data.cf_p_same_hat**2
    data=data[need].replace([np.inf,-np.inf],np.nan).dropna().reset_index(drop=True)
    X=data;y=data[outcome_col].to_numpy(dtype=int if binary else float);groups=data.subject.to_numpy()
    logo=LeaveOneGroupOut(); pred=[]
    for name,cols in FEATURES.items():
        pp=np.full(len(data),np.nan)
        for tr,te in logo.split(X,y,groups):
            if binary and len(np.unique(y[tr]))<2:continue
            alg=LogisticRegression(C=1.0,max_iter=5000) if binary else Ridge(alpha=1.0)
            model=Pipeline([('scale',StandardScaler()),('model',alg)])
            model.fit(X.iloc[tr][cols],y[tr]);pp[te]=(model.predict_proba(X.iloc[te][cols])[:,1]
                                                     if binary else model.predict(X.iloc[te][cols]))
        temp=pd.DataFrame({'subject':groups,'outcome':outcome,'model':name,'observed':y})
        temp['predicted_probability' if binary else 'predicted']=pp;pred.append(temp.dropna())
    pred=pd.concat(pred,ignore_index=True)
    persist(pred,'aim3D_LOPO_dynamicity_predictions.csv' if binary else 'aim3D_LOPO_capture_predictions.csv')
    scores=[]
    for (subject,name),g in pred.groupby(['subject','model']):
        if binary:
            val=log_loss(g.observed,g.predicted_probability.clip(EPS,1-EPS),labels=[0,1])
        else:val=np.sqrt(mean_squared_error(g.observed,g.predicted))
        scores.append({'subject':subject,'model':name,'metric':val})
    scores=pd.DataFrame(scores);sw=scores.pivot(index='subject',columns='model',values='metric').reset_index()
    sw['gaussian_coordinate_improvement']=(sw['physical_plus_gaussian_parameters']-
                                             sw['gaussian_causal_coordinates'])
    sw=sw.rename(columns={k:k for k in sw.columns})
    if binary:
        sw=sw.rename(columns={'metric':'log_loss'})
        persist(sw,'aim3D_dynamicity_subjectwise_CV_comparison.csv')
    else:persist(sw,'aim3D_capture_subjectwise_CV_comparison.csv')
    ptest=test_signflip(sw.gaussian_coordinate_improvement,seed=SEED+(1 if binary else 0))
    paired_all.append({'outcome':outcome,'metric':'log_loss' if binary else 'RMSE',**ptest})
    for name,g in pred.groupby('model'):
        if binary:
            perf_all.append({'outcome':outcome,'model':name,
                'log_loss':log_loss(g.observed,g.predicted_probability,labels=[0,1]),
                'AUC':roc_auc_score(g.observed,g.predicted_probability) if g.observed.nunique()==2 else np.nan})
        else:
            perf_all.append({'outcome':outcome,'model':name,'RMSE':np.sqrt(mean_squared_error(g.observed,g.predicted)),
                            'R2':r2_score(g.observed,g.predicted)})
persist(pd.DataFrame(perf_all),'aim3D_LOPO_model_comparison_summary.csv')
persist(pd.DataFrame(paired_all),'aim3D_participantwise_model_comparison_tests.csv')

# ============================================================================
# FIGURES: ORIGINAL DESCRIPTION PLUS *ADJUSTED* 3B PREDICTIONS AND NEW MODEL
# ============================================================================
fig,ax=plt.subplots(1,2,figsize=(12.5,5.1))
for frame,var,ycol,axis in [(cap,'cf_p_same_hat','VisualCapture_audioLag',ax[0]),
                             (bp,'cf_causal_ambiguity','dynamically_updating',ax[1])]:
    g=frame.copy();g['bin']=pd.cut(g[var],bins=np.linspace(0,1,6),include_lowest=True)
    a=g.groupby(['subject','bin'],observed=True).agg(x=(var,'mean'),y=(ycol,'mean')).reset_index()
    b=a.groupby('bin',observed=True).agg(x=('x','mean'),y=('y','mean'),
         se=('y',lambda z:z.std(ddof=1)/np.sqrt(z.notna().sum()))).reset_index()
    axis.errorbar(b.x,b.y,yerr=b.se,marker='o',capsize=4,label='Raw participant-first bins')
ax[0].set(xlabel='Cross-fitted fitted P(Same)',ylabel='Lag-corrected visual capture',
          title='Explicit curve vs implicit weighting')
# 3B statistical fix: plot GEE-adjusted probabilities, holding disparity mix fixed.
xs=np.linspace(.05,.95,100);margins=[];mix=bp[['subject','offset_deg']].drop_duplicates()
for vv in xs:
    z=mix.copy();z['cf_causal_ambiguity']=vv
    margins.append(float(np.mean(amb.predict(z))))
ax[1].plot(xs,margins,linestyle='--',label='Offset-adjusted GEE prediction')
ax[1].set(xlabel='Explicit ambiguity, 4p(1-p)',ylabel='Probability dynamically updating',ylim=(0,1),
          title='Raw bins vs disparity-adjusted updating');ax[1].legend(fontsize=8)
fig.tight_layout();fig.savefig(FIG_OUT/'aim3_gaussian_causal_coordinates_summary.png',dpi=220);plt.show()

fig,axes=plt.subplots(1,2,figsize=(12.5,5))
for subject,g in fixed_cell[fixed_cell.window.eq('main')].groupby('subject'):
    axes[0].plot(g.offset_deg,g.pC1,alpha=.12,lw=.9)
mean_curve=fixed_cell[fixed_cell.window.eq('main')].groupby('offset_deg').pC1.mean()
axes[0].plot(mean_curve.index,mean_curve.values,'ko-',lw=2,label='Subject-first mean')
axes[0].axhline(.5,ls='--',color='grey')
axes[0].set(xlabel='Physical disparity (deg)',ylabel='Model P(C=1)',ylim=(0,1),
            title='Implicit model posterior across fixed offsets');axes[0].legend(fontsize=8)
valid=base[['delta50_deg','bci_pC20']].dropna()
axes[1].scatter(valid.delta50_deg,valid.bci_pC20,alpha=.8)
if len(valid)>=4:
    o=np.polyfit(valid.delta50_deg,valid.bci_pC20,1)
    v=np.linspace(valid.delta50_deg.min(),valid.delta50_deg.max(),100)
    axes[1].plot(v,np.polyval(o,v),lw=2)
axes[1].set(xlabel='Explicit Gaussian Delta50 (deg)',ylabel='BCI P(C=1) at 20°',
            title=f'Implicit propensity vs explicit tolerance (N={len(valid)})')
fig.tight_layout();fig.savefig(FIG_OUT/'aim3C_BCI_implicit_propensity_vs_explicit.png',dpi=220);plt.show()
if sweep_parts:
    fig,ax=plt.subplots(figsize=(7,5))
    for direction,g in curve.groupby('direction'):
        a=g.groupby('disp_bin').pC1.agg(['mean','sem'])
        ax.errorbar(a.index,a['mean'],yerr=a['sem'],marker='o',capsize=2,label=direction)
    ax.axhline(.5,ls='--',color='grey');ax.set(xlabel='Actual instantaneous disparity (deg)',
      ylabel='Dynamic BCI P(C=1)',ylim=(0,1),title='Model posterior by sweep history')
    ax.legend();fig.tight_layout();fig.savefig(FIG_OUT/'aim3C_BCI_sweep_history_posteriors.png',dpi=220);plt.show()

print('\nAIM 3 COMPLETE:',OUT)
print('\n3A Gaussian P(Same) association:\n',ca_tab.to_string(index=False))
print('\n3B ambiguity GEE (do not interpret raw bins as adjusted):\n',amb_tab.to_string(index=False))
print('\n3C original descriptive half-decay:\n',pd.DataFrame([legacy_corr]).to_string(index=False))
print('\n3C model posterior individual differences:\n',corrtab.head(4).to_string(index=False))
print('\n3C 50% crossing identification:\n',cross.groupby(['window','curve','boundary_status']).size().to_string())
if sweep_parts:print('\n3C history mean difference:\n',pd.DataFrame(tests).to_string(index=False))
print('\n3C model-quality correlation sensitivity:\n',
      quality_corr[['subset','n','pearson_r','pearson_ci_low','pearson_ci_high','permutation_p']].to_string(index=False))
if 'hyst_obs_corr' in locals():
    print('\n3C observed SL vs modeled BCI posterior hysteresis:\n',
          hyst_obs_corr.to_string(index=False))
else:
    print('\n3C observed SL vs modeled BCI hysteresis: SKIPPED (verify HYST_WINDOW_CSV).')
print('\n3D hinge tests:\n',kink.to_string(index=False))
print('\n3D held-out performance:\n',pd.DataFrame(perf_all).to_string(index=False))
print('\nNote: cross-sectional correlations do not establish posterior calibration or causality.')

Aim3 cohort: inrange45, N=45. These restrictions are post-hoc sensitivity when N45.
Behavior: capture 3552 rows; 45 subjects; dynamicity 3600 rows; 45 subjects
  Loaded BCI chunk 1
  Loaded BCI chunk 9
BCI model IDs: 46 matched participants: 42 unmatched Aim3: ['pid035', 'pid046', 'pid057']


C:\Users\pkell\AppData\Local\Temp\ipykernel_29840\2570398960.py:596: UserWarning: No signed sweep side column identified in empirical SL hysteresis CSV. Aggregating trials without explicit left/right balancing; verify the condition has equal left/right repetitions.
  warnings.warn('No signed sweep side column identified in empirical SL hysteresis CSV. '


Observed SL hysteresis: 15595 window rows; side source: None
Empirical-model hysteresis: 41 participants with >=8 common disparity bins (out of 41).


C:\Users\pkell\AppData\Local\Temp\ipykernel_29840\2570398960.py:1312: UserWarning: NEW spatial-prediction analysis not run: set OBS_SL_TIME_CSV to an original time-resolved SL file. Original Aim3A-3D unchanged.
  warnings.warn('NEW spatial-prediction analysis not run: set OBS_SL_TIME_CSV '


3C spatial prediction additions complete. See: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_3_gaussian_explicit_implicit_mapping_inrange45_dynamicBCI\spatial_csv
Calibration join audit: {'n_model_fixed_trials': 3780, 'n_matched_with_explicit_report': 3780, 'match_fraction': 1.0, 'n_matched_subjects': 42}

AIM 3 COMPLETE: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_3_gaussian_explicit_implicit_mapping_inrange45_dynamicBCI

3A Gaussian P(Same) association:
                  term  estimate       se     stat            p  ci95_low  ci95_high
            Intercept  0.419294 0.081394 5.151383 2.585722e-07  0.259764   0.578824
C(offset_deg)[T.20.0]  0.080087 0.032716 2.447982 1.436588e-02  0.015966   0.144209
C(offset_deg)[T.30.0]  0.097182 0.054138 1.795070 7.264258e-02 -0.008927   0.203291
C(offset_deg)[T.40.0]  0.082811 0.065708 1.260281 2.075681e-01 -0.045975 

In [93]:
# %% AIM 4 — IS TEMPORAL PERSISTENCE SEPARABLE FROM THE UPDATED GAUSSIAN CAUSAL PHENOTYPE?
#
# UPDATED FRAMEWORK
# -----------------
# Static explicit causal tolerance is characterized by the fitted Gaussian
# psychometric function (especially Delta50 and slope at the 50% boundary).
# Temporal persistence is treated as a distinct question: how strongly does
# previous perceptual state constrain current updating?
#
# 4A. Hysteresis decomposition:
#     Diverge, Converge, and Converge-minus-Diverge visual bias.
#     Test Delta50 AND psychometric sharpness rather than Delta50 alone.
#
# 4B. Breakpoint dynamicity:
#     Does Delta50 / sharpness predict continuous updating after controlling
#     physical disparity?
#
# 4C. Stabilization timing:
#     Among accepted stabilizing trials, does Delta50 / sharpness predict
#     when stabilization occurs?
#
# 4D. STATIC-CURVE RESIDUAL HISTORY EFFECT:
#     Learn the fixed-trial mapping from cross-fitted Gaussian P(Same) to
#     visual capture. Predict what sweep-trial capture should be from the
#     participant's STATIC causal function. Residual Diverge-vs-Converge
#     differences quantify history dependence beyond baseline causal tolerance.
#
# Sensory-noise adjusted sensitivity models are also included.

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    pearsonr,
    spearmanr,
    ttest_1samp,
    t as t_dist,
)

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings(
    "default",
    category=ConvergenceWarning,
)

# =====================================================================
# PATHS / SETTINGS
# =====================================================================
ROOT = Path(
    r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab"
    r"\Continuous AVT v1"
)

SL_DATA = ROOT / "individual_data_SL"
SPATIAL = ROOT / "spatial_analyses"

AIM1 = (
    ROOT
    / "Individual_differences_analyses"
    / "Aim_1_gaussian_psychometric_reliability"
)

GAUSSIAN_CSV = (
    AIM1
    / "csv_outputs"
    / "aim1_gaussian_psychometric_subject_parameters.csv"
)

CAUSAL_TRIAL_CSV = (
    AIM1
    / "csv_outputs"
    / "aim1_static_causal_trial_table_with_crossfit_predictions.csv"
)

HYST_BIN_CSV = (
    SPATIAL
    / "07_spatial_hysteresis"
    / "subject_bin_direction_bias_summary.csv"
)

HYST_WINDOW_CSV = (
    SPATIAL
    / "07_spatial_hysteresis"
    / "spatial_hysteresis_window_level_bias_by_disparity.csv"
)

BREAKPOINT_CSV = (
    SPATIAL
    / "06_stabilization_breakpoint"
    / "csv_outputs"
    / "anchor_trial_level_breakpoint_DYNAMIC_metrics.csv"
)

FIXED_CAPTURE_CSV = (
    SL_DATA
    / "spatial_pse_capture_hysteresis_recalibration_predictions"
    / "csv_outputs"
    / "fixed_offset_trial_level_capture_psame_bias.csv"
)

OUT = (
    ROOT
    / "Individual_differences_analyses"
    / "Aim_4_gaussian_temporal_phenotype"
)

CSV_OUT = OUT / "csv_outputs"
FIG_OUT = OUT / "figures"
MODEL_OUT = OUT / "model_summaries"
DIAG_OUT = OUT / "model_diagnostics"

for d in [
    CSV_OUT,
    FIG_OUT,
    MODEL_OUT,
    DIAG_OUT,
]:

    d.mkdir(
        parents=True,
        exist_ok=True,
    )

FIXED_OFFSETS = [
    10,
    20,
    30,
    40,
]

SINGULAR_EIG_TOL = 1e-7
EPS = 1e-8


# =====================================================================
# HELPERS
# =====================================================================
def require_file(
    path,
    label,
):

    path = Path(
        path
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nMissing {label}:\n"
            f"{path}"
        )

    return path


for p, label in [
    (
        GAUSSIAN_CSV,
        "Aim-1 Gaussian phenotype table",
    ),
    (
        CAUSAL_TRIAL_CSV,
        "Aim-1 cross-fitted static causal table",
    ),
    (
        HYST_BIN_CSV,
        "subject-bin-direction hysteresis table",
    ),
    (
        HYST_WINDOW_CSV,
        "window-level hysteresis table",
    ),
    (
        BREAKPOINT_CSV,
        "breakpoint table",
    ),
    (
        FIXED_CAPTURE_CSV,
        "fixed-trial visual-capture table",
    ),
]:

    require_file(
        p,
        label,
    )


def clean_subject(
    s,
):

    return (
        s.astype(str)
        .str.lower()
        .str.strip()
    )


def zscore_series(
    x,
):

    x = pd.to_numeric(
        x,
        errors="coerce",
    )

    sd = x.std(
        ddof=0
    )

    return (
        (
            x
            -
            x.mean()
        )
        /
        sd
        if
        np.isfinite(
            sd
        )
        and
        sd
        >
        0
        else
        pd.Series(
            np.nan,
            index=x.index,
        )
    )


def gaussian_psame(
    d,
    mu,
    sigma,
    floor,
    peak,
):

    d = np.asarray(
        d,
        dtype=float,
    )

    mu = np.asarray(
        mu,
        dtype=float,
    )

    sigma = np.maximum(
        np.asarray(
            sigma,
            dtype=float,
        ),
        EPS,
    )

    floor = np.asarray(
        floor,
        dtype=float,
    )

    peak = np.asarray(
        peak,
        dtype=float,
    )

    return (
        floor
        +
        (
            peak
            -
            floor
        )
        *
        np.exp(
            -0.5
            *
            (
                (
                    d
                    -
                    mu
                )
                /
                sigma
            )
            ** 2
        )
    )


def safe_corr(
    x,
    y,
):

    d = (
        pd.DataFrame(
            {
                "x":
                    x,

                "y":
                    y,
            }
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
    )

    if (
        len(
            d
        )
        <
        4
        or
        d[
            "x"
        ].std(
            ddof=1
        )
        ==
        0
        or
        d[
            "y"
        ].std(
            ddof=1
        )
        ==
        0
    ):

        return {
            "n":
                len(
                    d
                ),

            "pearson_r":
                np.nan,

            "pearson_p":
                np.nan,

            "spearman_rho":
                np.nan,

            "spearman_p":
                np.nan,
        }


    pr = pearsonr(
        d[
            "x"
        ],
        d[
            "y"
        ],
    )


    sr = spearmanr(
        d[
            "x"
        ],
        d[
            "y"
        ],
    )


    return {
        "n":
            len(
                d
            ),

        "pearson_r":
            float(
                pr.statistic
            ),

        "pearson_p":
            float(
                pr.pvalue
            ),

        "spearman_rho":
            float(
                sr.statistic
            ),

        "spearman_p":
            float(
                sr.pvalue
            ),
    }


# =====================================================================
# ROBUST MIXED-LINEAR MODEL HELPERS
# =====================================================================
def _mixedlm_diagnostics(
    fit,
):

    fe = list(
        fit.fe_params.index
    )

    converged = bool(
        getattr(
            fit,
            "converged",
            False,
        )
    )

    finite = bool(
        np.isfinite(
            fit.fe_params.to_numpy(
                float
            )
        ).all()
        and
        np.isfinite(
            fit.bse.reindex(
                fe
            ).to_numpy(
                float
            )
        ).all()
        and
        np.isfinite(
            fit.pvalues.reindex(
                fe
            ).to_numpy(
                float
            )
        ).all()
    )


    try:

        cov = np.asarray(
            fit.cov_re,
            dtype=float,
        )

        eig = np.linalg.eigvalsh(
            cov
        )

        singular = (
            not np.isfinite(
                eig
            ).all()
            or
            np.linalg.matrix_rank(
                cov,
                tol=SINGULAR_EIG_TOL,
            )
            <
            cov.shape[
                0
            ]
            or
            np.nanmax(
                eig
            )
            <=
            SINGULAR_EIG_TOL
            or
            np.nanmin(
                eig
            )
            <
            SINGULAR_EIG_TOL
        )

        min_eig = float(
            np.nanmin(
                eig
            )
        )

        max_eig = float(
            np.nanmax(
                eig
            )
        )

    except Exception:

        singular = True
        min_eig = np.nan
        max_eig = np.nan


    return {
        "converged":
            converged,

        "finite_fixed_inference":
            finite,

        "singular_random_effects":
            bool(
                singular
            ),

        "min_re_eigenvalue":
            min_eig,

        "max_re_eigenvalue":
            max_eig,

        "usable":
            bool(
                converged
                and
                finite
                and
                not singular
            ),
    }


def fit_repeated_linear(
    formula,
    data,
    re_formula_candidates,
    model_name,
    group="subject",
):

    d = (
        data
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .copy()
    )


    diagnostics = []


    for re_formula in re_formula_candidates:

        for method in [
            "lbfgs",
            "powell",
            "cg",
        ]:

            try:

                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always"
                    )


                    fit = smf.mixedlm(
                        formula,
                        data=d,
                        groups=d[
                            group
                        ],
                        re_formula=re_formula,
                    ).fit(
                        reml=False,
                        method=method,
                        maxiter=5000,
                        disp=False,
                    )


                diag = _mixedlm_diagnostics(
                    fit
                )


                diag.update(
                    {
                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "warnings":
                            " | ".join(
                                str(
                                    w.message
                                )
                                for w in caught
                            ),

                        "n_rows":
                            len(
                                d
                            ),

                        "n_subjects":
                            d[
                                group
                            ].nunique(),
                    }
                )


                diagnostics.append(
                    diag
                )


                if diag[
                    "usable"
                ]:

                    out = pd.DataFrame(
                        diagnostics
                    )


                    out.to_csv(
                        DIAG_OUT
                        /
                        f"{model_name}_fit_diagnostics.csv",
                        index=False,
                    )


                    return (
                        fit,
                        (
                            f"MixedLM["
                            f"{re_formula}; "
                            f"{method}]"
                        ),
                        out,
                    )


            except Exception as exc:

                diagnostics.append(
                    {
                        "attempt":
                            "MixedLM",

                        "re_formula":
                            re_formula,

                        "optimizer":
                            method,

                        "usable":
                            False,

                        "error":
                            repr(
                                exc
                            ),

                        "n_rows":
                            len(
                                d
                            ),

                        "n_subjects":
                            d[
                                group
                            ].nunique(),
                    }
                )


    try:

        fit = smf.gee(
            formula,
            groups=group,
            data=d,
            family=sm.families.Gaussian(),
            cov_struct=Exchangeable(),
        ).fit()


        diagnostics.append(
            {
                "attempt":
                    "GEE",

                "usable":
                    True,

                "n_rows":
                    len(
                        d
                    ),

                "n_subjects":
                    d[
                        group
                    ].nunique(),
            }
        )


        out = pd.DataFrame(
            diagnostics
        )


        out.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            fit,
            "GEE_Gaussian_Exchangeable",
            out,
        )


    except Exception as exc:

        fit = smf.ols(
            formula,
            data=d,
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups":
                    d[
                        group
                    ],
            },
        )


        diagnostics.append(
            {
                "attempt":
                    "OLS_cluster",

                "usable":
                    True,

                "error_before_fallback":
                    repr(
                        exc
                    ),

                "n_rows":
                    len(
                        d
                    ),

                "n_subjects":
                    d[
                        group
                    ].nunique(),
            }
        )


        out = pd.DataFrame(
            diagnostics
        )


        out.to_csv(
            DIAG_OUT
            /
            f"{model_name}_fit_diagnostics.csv",
            index=False,
        )


        return (
            fit,
            "OLS_cluster_fallback",
            out,
        )


def fixed_terms(
    fit,
):

    return (
        list(
            fit.fe_params.index
        )
        if
        hasattr(
            fit,
            "fe_params",
        )
        else
        list(
            fit.params.index
        )
    )


def save_model(
    fit,
    name,
    fit_type="",
    diagnostics=None,
):

    with open(
        MODEL_OUT
        /
        f"{name}.txt",
        "w",
        encoding="utf-8",
    ) as f:

        if fit_type:

            f.write(
                f"Fit type: "
                f"{fit_type}\n\n"
            )


        if diagnostics is not None:

            f.write(
                "FIT DIAGNOSTICS\n"
            )

            f.write(
                diagnostics.to_string(
                    index=False
                )
            )

            f.write(
                "\n\nMODEL SUMMARY\n"
            )


        f.write(
            str(
                fit.summary()
            )
        )


    terms = fixed_terms(
        fit
    )


    ci = fit.conf_int().reindex(
        terms
    )


    tab = pd.DataFrame(
        {
            "term":
                terms,

            "estimate":
                fit.params.reindex(
                    terms
                ).values,

            "se":
                fit.bse.reindex(
                    terms
                ).values,

            "stat":
                fit.tvalues.reindex(
                    terms
                ).values,

            "p":
                fit.pvalues.reindex(
                    terms
                ).values,

            "ci95_low":
                ci.iloc[
                    :,
                    0
                ].values,

            "ci95_high":
                ci.iloc[
                    :,
                    1
                ].values,

            "fit_type":
                fit_type,
        }
    )


    tab.to_csv(
        CSV_OUT
        /
        f"{name}_coefficients.csv",
        index=False,
    )


    return tab


def scatter_with_fit(
    df,
    x,
    y,
    xlabel,
    ylabel,
    title,
    filename,
):

    d = (
        df[
            [
                x,
                y,
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna()
        .copy()
    )


    if len(
        d
    ) < 4:
        return


    fit = smf.ols(
        f"{y} ~ {x}",
        d,
    ).fit(
        cov_type="HC3"
    )


    xx = np.linspace(
        d[
            x
        ].min(),
        d[
            x
        ].max(),
        150,
    )


    pred = fit.get_prediction(
        pd.DataFrame(
            {
                x:
                    xx,
            }
        )
    ).summary_frame()


    corr = safe_corr(
        d[
            x
        ],
        d[
            y
        ],
    )


    fig, ax = plt.subplots(
        figsize=(
            6.5,
            5.1,
        )
    )


    ax.scatter(
        d[
            x
        ],
        d[
            y
        ],
        alpha=.72,
    )


    ax.plot(
        xx,
        pred[
            "mean"
        ],
        linewidth=2,
    )


    ax.fill_between(
        xx,
        pred[
            "mean_ci_lower"
        ].to_numpy(),
        pred[
            "mean_ci_upper"
        ].to_numpy(),
        alpha=.18,
    )


    ax.set_xlabel(
        xlabel
    )


    ax.set_ylabel(
        ylabel
    )


    ax.set_title(
        title
    )


    ax.text(
        .04,
        .96,
        (
            f"n={corr['n']}\n"
            f"r={corr['pearson_r']:.3f}, "
            f"p={corr['pearson_p']:.4g}"
        ),
        transform=ax.transAxes,
        va="top",
    )


    fig.tight_layout()


    fig.savefig(
        FIG_OUT
        /
        filename,
        dpi=220,
        bbox_inches="tight",
    )


    plt.show()


# =====================================================================
# LOAD GAUSSIAN PHENOTYPE
# =====================================================================
pheno = pd.read_csv(
    GAUSSIAN_CSV
)


pheno[
    "subject"
] = clean_subject(
    pheno[
        "subject"
    ]
)


for c in [
    "delta50_deg",
    "slope50_per_deg",
    "causal_sigma_deg",
    "mu_deg",
    "floor_same",
    "peak_same",
    "sensory_noise_A_deg",
    "sensory_noise_V_deg",
]:

    pheno[
        c
    ] = pd.to_numeric(
        pheno[
            c
        ],
        errors="coerce",
    )


pheno = pheno[
    pheno[
        "delta50_deg"
    ].notna()
].copy()


pheno[
    "delta50_c"
] = (
    pheno[
        "delta50_deg"
    ]
    -
    pheno[
        "delta50_deg"
    ].mean()
)


pheno[
    "slope50_z"
] = zscore_series(
    pheno[
        "slope50_per_deg"
    ]
)


pheno[
    "noiseA_z"
] = zscore_series(
    pheno[
        "sensory_noise_A_deg"
    ]
)


pheno[
    "noiseV_z"
] = zscore_series(
    pheno[
        "sensory_noise_V_deg"
    ]
)


# =====================================================================
# 4A. HYSTERESIS DECOMPOSITION
# =====================================================================
h = pd.read_csv(
    HYST_BIN_CSV
)


h[
    "subject"
] = clean_subject(
    h[
        "subject"
    ]
)


h[
    "SweepDirection"
] = (
    h[
        "SweepDirection"
    ]
    .astype(str)
    .str.title()
)


h[
    "disparity_bin_center"
] = pd.to_numeric(
    h[
        "disparity_bin_center"
    ],
    errors="coerce",
)


h[
    "mean_NormalizedVisualBias"
] = pd.to_numeric(
    h[
        "mean_NormalizedVisualBias"
    ],
    errors="coerce",
)


h = h[
    h[
        "disparity_bin_center"
    ].between(
        5,
        35,
    )
    &
    h[
        "SweepDirection"
    ].isin(
        [
            "Diverge",
            "Converge",
        ]
    )
].copy()


h = h.merge(
    pheno,
    on="subject",
    how="inner",
)


h[
    "disparity_c"
] = (
    h[
        "disparity_bin_center"
    ]
    -
    h[
        "disparity_bin_center"
    ].mean()
)


h[
    "disparity_z"
] = zscore_series(
    h[
        "disparity_bin_center"
    ]
)


h[
    "direction_c"
] = np.where(
    h[
        "SweepDirection"
    ].eq(
        "Converge"
    ),
    0.5,
    -0.5,
)


sub_dir = (
    h.groupby(
        [
            "subject",
            "SweepDirection",
        ],
        as_index=False,
    )
    .agg(
        mean_capture=(
            "mean_NormalizedVisualBias",
            "mean",
        ),

        delta50_deg=(
            "delta50_deg",
            "first",
        ),

        delta50_c=(
            "delta50_c",
            "first",
        ),

        slope50_per_deg=(
            "slope50_per_deg",
            "first",
        ),

        slope50_z=(
            "slope50_z",
            "first",
        ),

        noiseA_z=(
            "noiseA_z",
            "first",
        ),

        noiseV_z=(
            "noiseV_z",
            "first",
        ),
    )
)


wide = (
    sub_dir.pivot(
        index="subject",
        columns="SweepDirection",
        values="mean_capture",
    )
    .reset_index()
)


wide = wide.merge(
    pheno[
        [
            "subject",
            "delta50_deg",
            "delta50_c",
            "slope50_per_deg",
            "slope50_z",
            "noiseA_z",
            "noiseV_z",
        ]
    ],
    on="subject",
    how="left",
)


for c in [
    "Diverge",
    "Converge",
]:

    if c not in wide.columns:

        wide[
            c
        ] = np.nan


wide[
    "H_SL_ConvergeMinusDiverge"
] = (
    wide[
        "Converge"
    ]
    -
    wide[
        "Diverge"
    ]
)


wide.to_csv(
    CSV_OUT
    /
    "aim4A_subject_hysteresis_decomposition_gaussian.csv",
    index=False,
)


component_rows = []


for outcome in [
    "Diverge",
    "Converge",
    "H_SL_ConvergeMinusDiverge",
]:

    d = wide[
        [
            outcome,
            "delta50_c",
            "slope50_z",
        ]
    ].dropna().copy()


    if len(
        d
    ) >= 10:

        fit = smf.ols(
            (
                f"{outcome} ~ "
                "delta50_c + slope50_z"
            ),
            d,
        ).fit(
            cov_type="HC3"
        )


        for term in [
            "delta50_c",
            "slope50_z",
        ]:

            component_rows.append(
                {
                    "outcome":
                        outcome,

                    "term":
                        term,

                    "n":
                        len(
                            d
                        ),

                    "estimate":
                        fit.params.get(
                            term,
                            np.nan,
                        ),

                    "se":
                        fit.bse.get(
                            term,
                            np.nan,
                        ),

                    "p":
                        fit.pvalues.get(
                            term,
                            np.nan,
                        ),

                    "R2":
                        fit.rsquared,
                }
            )


pd.DataFrame(
    component_rows
).to_csv(
    CSV_OUT
    /
    "aim4A_hysteresis_component_regressions_gaussian.csv",
    index=False,
)


h_model_data = h.dropna(
    subset=[
        "mean_NormalizedVisualBias",
        "delta50_c",
        "slope50_z",
        "disparity_c",
        "disparity_z",
    ]
).copy()


hyst_model, hyst_type, hyst_diag = fit_repeated_linear(
    (
        "mean_NormalizedVisualBias ~ "
        "C(SweepDirection) * delta50_c + "
        "C(SweepDirection) * slope50_z + "
        "disparity_c + I(disparity_c**2)"
    ),
    h_model_data,
    [
        "1 + disparity_z + direction_c",
        "1 + direction_c",
        "1 + disparity_z",
        "1",
    ],
    "aim4A_hysteresis_delta50_slope",
)


hyst_tab = save_model(
    hyst_model,
    "aim4A_hysteresis_delta50_slope",
    hyst_type,
    hyst_diag,
)


h_noise = h_model_data.dropna(
    subset=[
        "noiseA_z",
        "noiseV_z",
    ]
).copy()


if (
    h_noise[
        "subject"
    ].nunique()
    >=
    20
):

    hyst_adj, hyst_adj_type, hyst_adj_diag = fit_repeated_linear(
        (
            "mean_NormalizedVisualBias ~ "
            "C(SweepDirection) * delta50_c + "
            "C(SweepDirection) * slope50_z + "
            "disparity_c + I(disparity_c**2) + "
            "noiseA_z + noiseV_z"
        ),
        h_noise,
        [
            "1 + disparity_z + direction_c",
            "1 + direction_c",
            "1",
        ],
        "aim4A_hysteresis_sensory_adjusted",
    )


    hyst_adj_tab = save_model(
        hyst_adj,
        "aim4A_hysteresis_sensory_adjusted",
        hyst_adj_type,
        hyst_adj_diag,
    )

else:

    hyst_adj = None
    hyst_adj_tab = pd.DataFrame()


# =====================================================================
# 4B. BREAKPOINT DYNAMICITY
# =====================================================================
bp = pd.read_csv(
    BREAKPOINT_CSV
)


bp[
    "subject"
] = clean_subject(
    bp[
        "subject"
    ]
)


bp[
    "offset_deg"
] = pd.to_numeric(
    bp[
        "offset_deg"
    ],
    errors="coerce",
).abs()


for c in [
    "dynamically_updating",
    "has_stabilizing_breakpoint",
    "stabilization_time_sec",
]:

    if c in bp.columns:

        bp[
            c
        ] = pd.to_numeric(
            bp[
                c
            ],
            errors="coerce",
        )


if "metric" in bp.columns:

    bp = bp[
        bp[
            "metric"
        ]
        .astype(str)
        .eq(
            "LagCorrectedVisualCapture"
        )
    ].copy()


bp = bp[
    bp[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


bp = bp.merge(
    pheno,
    on="subject",
    how="inner",
)


dyn = bp.dropna(
    subset=[
        "dynamically_updating",
        "delta50_c",
        "slope50_z",
    ]
).copy()


dynamic_gee = smf.gee(
    (
        "dynamically_updating ~ "
        "C(offset_deg) + delta50_c + slope50_z"
    ),
    groups="subject",
    data=dyn,
    family=sm.families.Binomial(),
    cov_struct=Exchangeable(),
).fit()


dynamic_tab = save_model(
    dynamic_gee,
    "aim4B_dynamicity_delta50_slope_GEE",
    "GEE_binomial",
)


dyn_noise = dyn.dropna(
    subset=[
        "noiseA_z",
        "noiseV_z",
    ]
).copy()


if (
    dyn_noise[
        "subject"
    ].nunique()
    >=
    20
):

    dynamic_adj = smf.gee(
        (
            "dynamically_updating ~ "
            "C(offset_deg) + delta50_c + slope50_z + "
            "noiseA_z + noiseV_z"
        ),
        groups="subject",
        data=dyn_noise,
        family=sm.families.Binomial(),
        cov_struct=Exchangeable(),
    ).fit()


    dynamic_adj_tab = save_model(
        dynamic_adj,
        "aim4B_dynamicity_sensory_adjusted_GEE",
        "GEE_binomial",
    )

else:

    dynamic_adj = None
    dynamic_adj_tab = pd.DataFrame()


dyn_subj = (
    dyn.groupby(
        "subject",
        as_index=False,
    )
    .agg(
        P_dynamic=(
            "dynamically_updating",
            "mean",
        ),

        delta50_deg=(
            "delta50_deg",
            "first",
        ),

        slope50_per_deg=(
            "slope50_per_deg",
            "first",
        ),
    )
)


dyn_subj.to_csv(
    CSV_OUT
    /
    "aim4B_subject_breakpoint_dynamicity_gaussian.csv",
    index=False,
)


scatter_with_fit(
    dyn_subj,
    "delta50_deg",
    "P_dynamic",
    "Explicit causal tolerance, Delta50 (deg)",
    "Proportion dynamically updating",
    "Static causal tolerance vs continuous spatial updating",
    "aim4B_delta50_vs_dynamicity.png",
)


# =====================================================================
# 4C. STABILIZATION TIME
# =====================================================================
stable = bp[
    bp[
        "has_stabilizing_breakpoint"
    ].eq(
        1
    )
].dropna(
    subset=[
        "stabilization_time_sec",
        "delta50_c",
        "slope50_z",
    ]
).copy()


stable[
    "offset_z"
] = zscore_series(
    stable[
        "offset_deg"
    ]
)


if (
    len(
        stable
    )
    >=
    20
    and
    stable[
        "subject"
    ].nunique()
    >=
    8
):

    time_model, time_type, time_diag = fit_repeated_linear(
        (
            "stabilization_time_sec ~ "
            "C(offset_deg) + delta50_c + slope50_z"
        ),
        stable,
        [
            "1 + offset_z",
            "1",
        ],
        "aim4C_stabilization_delta50_slope",
    )


    time_tab = save_model(
        time_model,
        "aim4C_stabilization_delta50_slope",
        time_type,
        time_diag,
    )

else:

    time_model = None
    time_type = "not_run"
    time_tab = pd.DataFrame()


stable_noise = stable.dropna(
    subset=[
        "noiseA_z",
        "noiseV_z",
    ]
).copy()


if (
    len(
        stable_noise
    )
    >=
    20
    and
    stable_noise[
        "subject"
    ].nunique()
    >=
    20
):

    time_adj, time_adj_type, time_adj_diag = fit_repeated_linear(
        (
            "stabilization_time_sec ~ "
            "C(offset_deg) + delta50_c + slope50_z + "
            "noiseA_z + noiseV_z"
        ),
        stable_noise,
        [
            "1 + offset_z",
            "1",
        ],
        "aim4C_stabilization_sensory_adjusted",
    )


    time_adj_tab = save_model(
        time_adj,
        "aim4C_stabilization_sensory_adjusted",
        time_adj_type,
        time_adj_diag,
    )

else:

    time_adj = None
    time_adj_tab = pd.DataFrame()


if not stable.empty:

    time_subj = (
        stable.groupby(
            "subject",
            as_index=False,
        )
        .agg(
            mean_stabilization_time_sec=(
                "stabilization_time_sec",
                "mean",
            ),

            delta50_deg=(
                "delta50_deg",
                "first",
            ),

            slope50_per_deg=(
                "slope50_per_deg",
                "first",
            ),
        )
    )


    time_subj.to_csv(
        CSV_OUT
        /
        "aim4C_subject_stabilization_time_gaussian.csv",
        index=False,
    )


    scatter_with_fit(
        time_subj,
        "delta50_deg",
        "mean_stabilization_time_sec",
        "Explicit causal tolerance, Delta50 (deg)",
        "Mean stabilization time (s)",
        "Static causal tolerance vs stabilization timing",
        "aim4C_delta50_vs_stabilization_time.png",
    )


# =====================================================================
# 4D. HISTORY BEYOND THE STATIC CAUSAL CURVE
# =====================================================================
# First learn the relationship between cross-fitted static P(Same) and
# fixed-trial raw normalized visual capture.

causal = pd.read_csv(
    CAUSAL_TRIAL_CSV
)


causal[
    "subject"
] = clean_subject(
    causal[
        "subject"
    ]
)


causal[
    "trialIndex"
] = pd.to_numeric(
    causal[
        "trialIndex"
    ],
    errors="coerce",
)


causal[
    "cf_p_same_hat"
] = pd.to_numeric(
    causal[
        "cf_p_same_hat"
    ],
    errors="coerce",
)


fixed = pd.read_csv(
    FIXED_CAPTURE_CSV
)


fixed[
    "subject"
] = clean_subject(
    fixed[
        "subject"
    ]
)


fixed[
    "trialIndex"
] = pd.to_numeric(
    fixed[
        "trialIndex"
    ],
    errors="coerce",
)


fixed[
    "offset_deg"
] = pd.to_numeric(
    fixed[
        "offset_deg"
    ],
    errors="coerce",
).abs()


fixed[
    "NormalizedVisualBias"
] = pd.to_numeric(
    fixed[
        "NormalizedVisualBias"
    ],
    errors="coerce",
)


fixed = fixed[
    fixed[
        "offset_deg"
    ].isin(
        FIXED_OFFSETS
    )
].copy()


fixed = fixed.merge(
    causal[
        [
            "subject",
            "trialIndex",
            "cf_p_same_hat",
        ]
    ]
    .drop_duplicates(
        [
            "subject",
            "trialIndex",
        ]
    ),
    on=[
        "subject",
        "trialIndex",
    ],
    how="inner",
)


fixed = fixed.dropna(
    subset=[
        "NormalizedVisualBias",
        "cf_p_same_hat",
    ]
).copy()


static_mapping = smf.gee(
    (
        "NormalizedVisualBias ~ "
        "cf_p_same_hat + I(cf_p_same_hat**2)"
    ),
    groups="subject",
    data=fixed,
    family=sm.families.Gaussian(),
    cov_struct=Exchangeable(),
).fit()


static_mapping_tab = save_model(
    static_mapping,
    "aim4D_static_psame_to_capture_mapping_GEE",
    "GEE_Gaussian",
)


# ---------------------------------------------------------------------
# SWEEP WINDOW DATA
# ---------------------------------------------------------------------
hw = pd.read_csv(
    HYST_WINDOW_CSV
)


hw[
    "subject"
] = clean_subject(
    hw[
        "subject"
    ]
)


hw[
    "SweepDirection"
] = (
    hw[
        "SweepDirection"
    ]
    .astype(str)
    .str.title()
)


hw[
    "mean_disparity_window"
] = pd.to_numeric(
    hw[
        "mean_disparity_window"
    ],
    errors="coerce",
)


hw[
    "NormalizedVisualBias_window"
] = pd.to_numeric(
    hw[
        "NormalizedVisualBias_window"
    ],
    errors="coerce",
)


hw = hw[
    hw[
        "SweepDirection"
    ].isin(
        [
            "Diverge",
            "Converge",
        ]
    )
    &
    hw[
        "mean_disparity_window"
    ].between(
        5,
        35,
    )
].copy()


# ---------------------------------------------------------------------
# Recover actual SIGNED visual-auditory disparity per sweep window.
# ---------------------------------------------------------------------
hw[
    "signed_disparity_window"
] = np.nan


needed_signed_cols = {
    "source_file",
    "trialIndex",
    "window_start_sec",
    "window_end_sec",
}


if needed_signed_cols.issubset(
    hw.columns
):

    for source_file, hfile in hw.groupby(
        "source_file"
    ):

        raw_path = (
            SL_DATA
            /
            str(
                source_file
            )
        )


        if not raw_path.exists():
            continue


        try:

            raw = pd.read_csv(
                raw_path
            )

        except Exception:
            continue


        if not {
            "trialIndex",
            "t",
            "blob_x",
            "audio_x",
        }.issubset(
            raw.columns
        ):
            continue


        raw[
            "trialIndex"
        ] = pd.to_numeric(
            raw[
                "trialIndex"
            ],
            errors="coerce",
        )


        for trial_index, htrial in hfile.groupby(
            "trialIndex"
        ):

            trial_numeric = pd.to_numeric(
                trial_index,
                errors="coerce",
            )


            r = raw[
                raw[
                    "trialIndex"
                ].eq(
                    trial_numeric
                )
            ].copy()


            if r.empty:
                continue


            t_raw = pd.to_numeric(
                r[
                    "t"
                ],
                errors="coerce",
            ).to_numpy(
                float
            )


            if not np.isfinite(
                t_raw
            ).any():
                continue


            t_raw = (
                t_raw
                -
                np.nanmin(
                    t_raw
                )
            )


            v_raw = pd.to_numeric(
                r[
                    "blob_x"
                ],
                errors="coerce",
            ).to_numpy(
                float
            )


            a_raw = pd.to_numeric(
                r[
                    "audio_x"
                ],
                errors="coerce",
            ).to_numpy(
                float
            )


            for idx, wr in htrial.iterrows():

                ws = pd.to_numeric(
                    wr[
                        "window_start_sec"
                    ],
                    errors="coerce",
                )


                we = pd.to_numeric(
                    wr[
                        "window_end_sec"
                    ],
                    errors="coerce",
                )


                m = (
                    np.isfinite(
                        t_raw
                    )
                    &
                    np.isfinite(
                        v_raw
                    )
                    &
                    np.isfinite(
                        a_raw
                    )
                    &
                    (
                        t_raw
                        >=
                        ws
                    )
                    &
                    (
                        t_raw
                        <=
                        we
                    )
                )


                if m.sum() >= 3:

                    hw.at[
                        idx,
                        "signed_disparity_window",
                    ] = float(
                        np.nanmean(
                            v_raw[
                                m
                            ]
                            -
                            a_raw[
                                m
                            ]
                        )
                    )


hw = hw.merge(
    pheno[
        [
            "subject",
            "mu_deg",
            "causal_sigma_deg",
            "floor_same",
            "peak_same",
            "delta50_deg",
            "delta50_c",
            "slope50_z",
            "slope50_per_deg",
        ]
    ],
    on="subject",
    how="inner",
)


dabs = hw[
    "mean_disparity_window"
].to_numpy(
    float
)


ds_signed = pd.to_numeric(
    hw[
        "signed_disparity_window"
    ],
    errors="coerce",
).to_numpy(
    float
)


# Primary: actual signed disparity.
p_signed = gaussian_psame(
    ds_signed,
    hw[
        "mu_deg"
    ],
    hw[
        "causal_sigma_deg"
    ],
    hw[
        "floor_same"
    ],
    hw[
        "peak_same"
    ],
)


# Fallback if signed window recovery failed.
p_pos = gaussian_psame(
    dabs,
    hw[
        "mu_deg"
    ],
    hw[
        "causal_sigma_deg"
    ],
    hw[
        "floor_same"
    ],
    hw[
        "peak_same"
    ],
)


p_neg = gaussian_psame(
    -dabs,
    hw[
        "mu_deg"
    ],
    hw[
        "causal_sigma_deg"
    ],
    hw[
        "floor_same"
    ],
    hw[
        "peak_same"
    ],
)


p_abs_fallback = (
    0.5
    *
    (
        p_pos
        +
        p_neg
    )
)


hw[
    "static_psame_expected"
] = np.where(
    np.isfinite(
        ds_signed
    ),
    p_signed,
    p_abs_fallback,
)


hw[
    "static_psame_expected"
] = np.clip(
    hw[
        "static_psame_expected"
    ],
    EPS,
    1 - EPS,
)


hw[
    "expected_static_capture"
] = static_mapping.predict(
    pd.DataFrame(
        {
            "cf_p_same_hat":
                hw[
                    "static_psame_expected"
                ],
        }
    )
)


hw[
    "history_residual_capture"
] = (
    hw[
        "NormalizedVisualBias_window"
    ]
    -
    hw[
        "expected_static_capture"
    ]
)


hw.to_csv(
    CSV_OUT
    /
    "aim4D_sweep_windows_static_curve_residuals.csv",
    index=False,
)


# ---------------------------------------------------------------------
# SUBJECT × DIRECTION RESIDUAL HISTORY
# ---------------------------------------------------------------------
resid_dir = (
    hw.groupby(
        [
            "subject",
            "SweepDirection",
        ],
        as_index=False,
    )
    .agg(
        mean_history_residual=(
            "history_residual_capture",
            "mean",
        ),

        delta50_deg=(
            "delta50_deg",
            "first",
        ),

        delta50_c=(
            "delta50_c",
            "first",
        ),

        slope50_per_deg=(
            "slope50_per_deg",
            "first",
        ),

        slope50_z=(
            "slope50_z",
            "first",
        ),
    )
)


resid_wide = (
    resid_dir.pivot(
        index="subject",
        columns="SweepDirection",
        values="mean_history_residual",
    )
    .reset_index()
)


resid_wide = resid_wide.merge(
    pheno[
        [
            "subject",
            "delta50_deg",
            "delta50_c",
            "slope50_per_deg",
            "slope50_z",
        ]
    ],
    on="subject",
    how="left",
)


for c in [
    "Diverge",
    "Converge",
]:

    if c not in resid_wide.columns:

        resid_wide[
            c
        ] = np.nan


resid_wide[
    "residual_hysteresis_ConvergeMinusDiverge"
] = (
    resid_wide[
        "Converge"
    ]
    -
    resid_wide[
        "Diverge"
    ]
)


resid_wide.to_csv(
    CSV_OUT
    /
    "aim4D_subject_static_curve_residual_hysteresis.csv",
    index=False,
)


# ---------------------------------------------------------------------
# GROUP TEST: DOES HISTORY REMAIN AFTER STATIC CAUSAL EXPECTATION?
# ---------------------------------------------------------------------
vals = (
    resid_wide[
        "residual_hysteresis_ConvergeMinusDiverge"
    ]
    .dropna()
    .to_numpy(
        float
    )
)


if len(
    vals
) >= 4:

    tt = ttest_1samp(
        vals,
        0.0,
        nan_policy="omit",
    )


    mean_val = float(
        np.mean(
            vals
        )
    )


    se = float(
        np.std(
            vals,
            ddof=1,
        )
        /
        np.sqrt(
            len(
                vals
            )
        )
    )


    crit = t_dist.ppf(
        .975,
        df=(
            len(
                vals
            )
            -
            1
        ),
    )


    residual_test = pd.DataFrame(
        [
            {
                "n":
                    len(
                        vals
                    ),

                "mean_residual_hysteresis":
                    mean_val,

                "ci95_low":
                    mean_val
                    -
                    crit
                    *
                    se,

                "ci95_high":
                    mean_val
                    +
                    crit
                    *
                    se,

                "t":
                    float(
                        tt.statistic
                    ),

                "p":
                    float(
                        tt.pvalue
                    ),
            }
        ]
    )

else:

    residual_test = pd.DataFrame(
        [
            {
                "n":
                    len(
                        vals
                    ),

                "mean_residual_hysteresis":
                    np.nan,

                "ci95_low":
                    np.nan,

                "ci95_high":
                    np.nan,

                "t":
                    np.nan,

                "p":
                    np.nan,
            }
        ]
    )


residual_test.to_csv(
    CSV_OUT
    /
    "aim4D_static_curve_residual_hysteresis_group_test.csv",
    index=False,
)


# ---------------------------------------------------------------------
# DOES DELTA50 / SHARPNESS PREDICT RESIDUAL HISTORY?
# ---------------------------------------------------------------------
d = resid_wide[
    [
        "residual_hysteresis_ConvergeMinusDiverge",
        "delta50_c",
        "slope50_z",
    ]
].dropna().copy()


if len(
    d
) >= 10:

    residual_reg = smf.ols(
        (
            "residual_hysteresis_ConvergeMinusDiverge ~ "
            "delta50_c + slope50_z"
        ),
        d,
    ).fit(
        cov_type="HC3"
    )


    residual_reg_tab = pd.DataFrame(
        {
            "term":
                residual_reg.params.index,

            "estimate":
                residual_reg.params.values,

            "se":
                residual_reg.bse.reindex(
                    residual_reg.params.index
                ).values,

            "stat":
                residual_reg.tvalues.reindex(
                    residual_reg.params.index
                ).values,

            "p":
                residual_reg.pvalues.reindex(
                    residual_reg.params.index
                ).values,
        }
    )


    residual_reg_tab.to_csv(
        CSV_OUT
        /
        "aim4D_residual_hysteresis_delta50_slope_regression.csv",
        index=False,
    )

else:

    residual_reg = None
    residual_reg_tab = pd.DataFrame()


# =====================================================================
# PRIMARY FOCAL TEST SUMMARY + FDR
# =====================================================================
summary_rows = []


def append_term(
    analysis,
    tab,
    term,
    fit_type,
):

    if (
        tab.empty
        or
        term not in tab[
            "term"
        ].values
    ):
        return


    r = tab[
        tab[
            "term"
        ].eq(
            term
        )
    ].iloc[
        0
    ]


    summary_rows.append(
        {
            "analysis":
                analysis,

            "fit_type":
                fit_type,

            "term":
                term,

            "estimate":
                r.get(
                    "estimate",
                    np.nan,
                ),

            "se":
                r.get(
                    "se",
                    np.nan,
                ),

            "stat":
                r.get(
                    "stat",
                    np.nan,
                ),

            "p":
                r.get(
                    "p",
                    np.nan,
                ),

            "ci95_low":
                r.get(
                    "ci95_low",
                    np.nan,
                ),

            "ci95_high":
                r.get(
                    "ci95_high",
                    np.nan,
                ),
        }
    )


hyst_delta_term = next(
    (
        t
        for t in hyst_tab[
            "term"
        ]
        if
        "C(SweepDirection)"
        in t
        and
        "delta50_c"
        in t
    ),
    None,
)


hyst_slope_term = next(
    (
        t
        for t in hyst_tab[
            "term"
        ]
        if
        "C(SweepDirection)"
        in t
        and
        "slope50_z"
        in t
    ),
    None,
)


if hyst_delta_term:

    append_term(
        "4A hysteresis x Delta50",
        hyst_tab,
        hyst_delta_term,
        hyst_type,
    )


if hyst_slope_term:

    append_term(
        "4A hysteresis x sharpness",
        hyst_tab,
        hyst_slope_term,
        hyst_type,
    )


append_term(
    "4B breakpoint dynamicity — Delta50",
    dynamic_tab,
    "delta50_c",
    "GEE_binomial",
)


append_term(
    "4B breakpoint dynamicity — sharpness",
    dynamic_tab,
    "slope50_z",
    "GEE_binomial",
)


if not time_tab.empty:

    append_term(
        "4C stabilization time — Delta50",
        time_tab,
        "delta50_c",
        time_type,
    )


    append_term(
        "4C stabilization time — sharpness",
        time_tab,
        "slope50_z",
        time_type,
    )


summary = pd.DataFrame(
    summary_rows
)


summary[
    "p_FDR_BH"
] = np.nan


if not summary.empty:

    valid = summary[
        "p"
    ].notna()


    if valid.any():

        summary.loc[
            valid,
            "p_FDR_BH",
        ] = multipletests(
            summary.loc[
                valid,
                "p",
            ].to_numpy(
                float
            ),
            method="fdr_bh",
        )[
            1
        ]


summary.to_csv(
    CSV_OUT
    /
    "aim4_primary_tests_summary_gaussian.csv",
    index=False,
)


# =====================================================================
# FIGURE — TEMPORAL PHENOTYPE OVERVIEW
# =====================================================================
fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        15.5,
        4.8,
    ),
)


dplot = wide[
    [
        "delta50_deg",
        "H_SL_ConvergeMinusDiverge",
    ]
].dropna()


axes[
    0
].scatter(
    dplot[
        "delta50_deg"
    ],
    dplot[
        "H_SL_ConvergeMinusDiverge"
    ],
    alpha=.72,
)


if len(
    dplot
) >= 4:

    b = np.polyfit(
        dplot[
            "delta50_deg"
        ],
        dplot[
            "H_SL_ConvergeMinusDiverge"
        ],
        1,
    )


    xx = np.linspace(
        dplot[
            "delta50_deg"
        ].min(),
        dplot[
            "delta50_deg"
        ].max(),
        100,
    )


    axes[
        0
    ].plot(
        xx,
        b[
            0
        ]
        *
        xx
        +
        b[
            1
        ],
        linewidth=2,
    )


axes[
    0
].axhline(
    0,
    linestyle="--",
    linewidth=1,
)


axes[
    0
].set_xlabel(
    "Explicit Delta50 (deg)"
)


axes[
    0
].set_ylabel(
    "SL hysteresis\n(Converge − Diverge)"
)


axes[
    0
].set_title(
    "Boundary location vs history"
)


axes[
    1
].scatter(
    dyn_subj[
        "delta50_deg"
    ],
    dyn_subj[
        "P_dynamic"
    ],
    alpha=.72,
)


axes[
    1
].set_xlabel(
    "Explicit Delta50 (deg)"
)


axes[
    1
].set_ylabel(
    "Probability dynamically updating"
)


axes[
    1
].set_ylim(
    0,
    1,
)


axes[
    1
].set_title(
    "Boundary location vs dynamicity"
)


axes[
    2
].scatter(
    resid_wide[
        "delta50_deg"
    ],
    resid_wide[
        "residual_hysteresis_ConvergeMinusDiverge"
    ],
    alpha=.72,
)


axes[
    2
].axhline(
    0,
    linestyle="--",
    linewidth=1,
)


axes[
    2
].set_xlabel(
    "Explicit Delta50 (deg)"
)


axes[
    2
].set_ylabel(
    "Residual history gap"
)


axes[
    2
].set_title(
    "History beyond static causal curve"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "aim4_gaussian_temporal_phenotype_overview.png",
    dpi=220,
    bbox_inches="tight",
)


plt.show()


print(
    "\nAIM 4 COMPLETE"
)


print(
    "\nPrimary focal tests:"
)


print(
    summary.to_string(
        index=False
    )
)


print(
    "\nStatic-curve residual hysteresis group test:"
)


print(
    residual_test.to_string(
        index=False
    )
)


print(
    "\nChosen estimators:"
)


print(
    f"  Hysteresis model: "
    f"{hyst_type}"
)


print(
    "  Dynamicity model: "
    "GEE_binomial"
)


print(
    f"  Stabilization model: "
    f"{time_type}"
)


print(
    f"\nSaved outputs to:\n{OUT}"
)

[N45 filter] aim1_gaussian_psychometric_subject_parameters.csv: 50 -> 45 rows
[N45 filter] subject_bin_direction_bias_summary.csv: 1,414 -> 1,269 rows
[N45 filter] anchor_trial_level_breakpoint_DYNAMIC_metrics.csv: 8,000 -> 7,200 rows
[N45 filter] aim1_static_causal_trial_table_with_crossfit_predictions.csv: 4,500 -> 4,050 rows
[N45 filter] fixed_offset_trial_level_capture_psame_bias.csv: 4,000 -> 3,600 rows
[N45 filter] spatial_hysteresis_window_level_bias_by_disparity.csv: 21,000 -> 18,900 rows

AIM 4 COMPLETE

Primary focal tests:
                            analysis                                      fit_type                                   term  estimate       se      stat        p  ci95_low  ci95_high  p_FDR_BH
             4A hysteresis x Delta50 MixedLM[1 + disparity_z + direction_c; lbfgs] C(SweepDirection)[T.Diverge]:delta50_c -0.003131 0.005950 -0.526193 0.598754 -0.014792   0.008531  0.598754
           4A hysteresis x sharpness MixedLM[1 + disparity_z + direction_c; lb

In [100]:
# %% CELL Z0 — INDEPENDENT DATA MANIFEST (RUN AFTER EXISTING AIM 1, 2, 3, 4)
# Does NOT modify old notebook variables, outputs, model fits, or pd.read_csv.
from pathlib import Path
import re, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.cov_struct import Exchangeable

ZROOT = Path(r'C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1')
ZOUT = ZROOT/'Individual_differences_analyses'/'Zhu2026_AddOns'
ZOUT.mkdir(parents=True, exist_ok=True)
ZREAD = getattr(pd.read_csv, '_avt_inrange45_original', pd.read_csv)
ZSEED = 20261025
ZAIM1 = ZROOT/'Individual_differences_analyses'/'Aim_1_gaussian_psychometric_reliability'/'csv_outputs'
ZAIM3 = ZROOT/'Individual_differences_analyses'/'Aim_3_gaussian_explicit_implicit_mapping_inrange45_dynamicBCI'/'csv_outputs'
ZAIM3_FULL = ZROOT/'Individual_differences_analyses'/'Aim_3_gaussian_explicit_implicit_mapping_full50_dynamicBCI'/'csv_outputs'
if (ZAIM3_FULL/'aim3C_BCI_explicit_implicit_comparison_subjects.csv').exists():
    ZAIM3 = ZAIM3_FULL
ZBCI = ZROOT/'dynamic BCI'
ZFILES = {
 'pheno': ZAIM1/'aim1_gaussian_psychometric_subject_parameters.csv',
 'calibration': ZAIM1/'aim1_unimodal_calibration_subject_level.csv',
 'cal_trial': ZAIM1/'aim1_unimodal_calibration_trial_level.csv',
 'explicit_trials': ZAIM1/'aim1_static_causal_trial_table_with_crossfit_predictions.csv',
 'cv_models': ZAIM1/'aim1_cv_participant_model_scores.csv',
 'split_reliability': ZAIM1/'aim1_gaussian_repeated_split_reliability_summary.csv',
 'capture': ZROOT/'spatial_analyses'/'00_core_spatial_metrics'/'trial_level_metrics'/'cell3_trial_level_lag_corrected_spatial_metrics.csv',
 'bci_params': ZBCI/'dynamic'/'parameter_estimates.csv',
 'bci_restarts': ZBCI/'dynamic'/'parameter_estimates_all.csv',
 'bci_posterior': ZAIM3/'aim3C_BCI_explicit_implicit_comparison_subjects.csv',
 'bci_time': ZBCI/'model_timeresolved_SL_20Hz.csv',
}

def zid(col):
    return col.astype(str).str.extract(r'(pid\d+)', flags=re.I, expand=False).str.lower()

def zload(which, required=False):
    path = ZFILES[which] if isinstance(which,str) and which in ZFILES else Path(which)
    if not path.is_file():
        if required: raise FileNotFoundError(f'{which}: {path}')
        print(f'OPTIONAL INPUT MISSING: {which}: {path}')
        return pd.DataFrame()
    out = ZREAD(path)
    if 'subject' in out: out['subject'] = zid(out['subject'])
    return out

def zsave(df, name):
    path = ZOUT/name
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return df

def zcorr(data, x, y, label, bootstrap=2000):
    d=data[[x,y]].replace([np.inf,-np.inf],np.nan).dropna()
    result={'analysis':label,'x':x,'y':y,'n':len(d),'pearson_r':np.nan,'pearson_p':np.nan,
            'spearman_rho':np.nan,'spearman_p':np.nan,'ci_low':np.nan,'ci_high':np.nan}
    if len(d)<8 or d[x].std()<1e-8 or d[y].std()<1e-8: return result
    a,b=d[x].to_numpy(float),d[y].to_numpy(float)
    result.update(pearson_r=pearsonr(a,b).statistic,pearson_p=pearsonr(a,b).pvalue,
                  spearman_rho=spearmanr(a,b).statistic,spearman_p=spearmanr(a,b).pvalue)
    rng=np.random.default_rng(ZSEED); vals=[]
    for _ in range(bootstrap):
        idx=rng.integers(0,len(a),len(a))
        if np.std(a[idx])>1e-9 and np.std(b[idx])>1e-9:
            vals.append(np.corrcoef(a[idx],b[idx])[0,1])
    if vals: result['ci_low'],result['ci_high']=np.percentile(vals,[2.5,97.5])
    return result

zp = zload('pheno',True)
if zp.subject.duplicated().any(): raise ValueError('Duplicate participant IDs in Aim 1 phenotype file')
for c in ['delta50_deg','mu_deg','causal_sigma_deg','slope50_per_deg']:
    zp[c]=pd.to_numeric(zp[c],errors='coerce')

zparams=zload('bci_params')
if not zparams.empty:
    if zparams.subject.duplicated().any(): raise ValueError('Duplicate BCI parameter IDs')
    keep=['subject','p_common','sigma_eA','sigma_eV','sigma_p','tau_p','kappa',
          'tau_C_derived','r2_holdout','n_at_bound','spread_p_common','spread_sigma_eA',
          'spread_sigma_p','spread_kappa','spread_tau_p']
    zparams=zparams[[c for c in keep if c in zparams]]

zpost=zload('bci_posterior')
if not zpost.empty:
    if 'window' in zpost: zpost=zpost[zpost.window.eq('main')].copy()
    zpost=zpost[['subject']+[c for c in ['bci_pC20','bci_mean_pC_0to40','bci_pC_auc_0to40'] if c in zpost]]
    if zpost.subject.duplicated().any():raise ValueError('Repeated BCI posterior subject rows')

zcal=zload('calibration')
if not zcal.empty and zcal.subject.duplicated().any():raise ValueError('Duplicate calibration subject rows')

zpeople=zp.copy()
for source in [zcal,zparams,zpost]:
    if not source.empty: zpeople=zpeople.merge(source,on='subject',how='left',validate='one_to_one')
zpeople['inrange_45']=zpeople['boundary50_within_observed_range'].astype(str).str.lower().eq('true')
zpeople['fit_ok']=zpeople['fit_success'].astype(str).str.lower().eq('true')
zpeople['matched_bci']=zpeople.get('p_common',pd.Series(np.nan,index=zpeople.index)).notna()
for c in ['sigma_eA','sigma_eV','sigma_p','tracking_sd_deg_Audio','tracking_sd_deg_Visual']:
    if c in zpeople:
        vals=pd.to_numeric(zpeople[c],errors='coerce')
        zpeople['log_'+c]=np.log(vals.where(vals>0))
if {'p_common','n_at_bound','r2_holdout'}.issubset(zpeople.columns):
    zpeople['quality_good']=zpeople.r2_holdout.ge(.50)&zpeople.n_at_bound.eq(0)
zsave(zpeople,'Z0_participant_manifest_and_features.csv')
print('Primary full behavioral sample:',len(zpeople),'| in-range sensitivity:',int(zpeople.inrange_45.sum()),
      '| BCI matched:',int(zpeople.matched_bci.sum()))
print('Missing core inputs:',[name for name,p in ZFILES.items() if not p.exists()])

Primary full behavioral sample: 50 | in-range sensitivity: 45 | BCI matched: 46
Missing core inputs: []


In [102]:
# %% CELL Z1 — MEASUREMENT VALIDITY: ALTERNATIVE PSYCHOMETRICS + SENSORY CALIBRATION
# Existing Gaussian fits remain untouched; compare existing CV models and
# recoverability of the original unisensory tracking-variability proxies.
zcv=zload('cv_models')
if not zcv.empty:
    print('CV columns:',list(zcv.columns))
    if {'subject','model','log_loss'}.issubset(zcv.columns):
        score=zcv.groupby('model',as_index=False).agg(n=('subject','nunique'),
                    mean_subject_logloss=('log_loss','mean'),sd=('log_loss','std'))
        zsave(score.sort_values('mean_subject_logloss'),'Z1_psychometric_model_comparison.csv')
        print(score.sort_values('mean_subject_logloss').to_string(index=False))
    elif {'subject','model','mean_log_loss'}.issubset(zcv.columns):
        score=zcv.groupby('model',as_index=False).agg(n=('subject','nunique'),
                   mean_subject_logloss=('mean_log_loss','mean'))
        zsave(score.sort_values('mean_subject_logloss'),'Z1_psychometric_model_comparison.csv')
    else:
        warnings.warn('Review columns for participant-level held-out model scores.')

zctrial=zload('cal_trial')
if not zctrial.empty:
    required={'subject','trialIndex','modality','sd_error_deg','mean_error_deg'}
    if not required.issubset(zctrial.columns):
        raise KeyError(f'Unimodal calibration trial missing: {required-set(zctrial.columns)}')
    zctrial=zctrial.sort_values(['subject','modality','trialIndex']).copy()
    # Odd/even within each modality; correlations represent between-person stability,
    # not a direct estimate of likelihood noise without model validation.
    zctrial['split']=zctrial.groupby(['subject','modality']).cumcount()%2
    summary=(zctrial.groupby(['subject','modality','split'],as_index=False)
             .agg(mean_trial_tracking_sd=('sd_error_deg','mean'),
                  mean_trial_bias=('mean_error_deg','mean'),n_trials=('trialIndex','nunique')))
    zsave(summary,'Z1_unimodal_split_half_values.csv')
    rows=[]
    for modality,g in summary.groupby('modality'):
        for metric in ['mean_trial_tracking_sd','mean_trial_bias']:
            w=g.pivot(index='subject',columns='split',values=metric).dropna()
            if len(w)>=8:
                r=zcorr(w.reset_index(),0,1,f'{modality}_{metric}')
                rho=r['pearson_r']
                r['spearman_brown']=(2*rho/(1+rho)) if np.isfinite(rho) and rho>-0.999 else np.nan
                rows.append(r)
    zsave(pd.DataFrame(rows),'Z1_unimodal_reliability.csv')
    print('Unimodal split-half reliability:',pd.DataFrame(rows)[['analysis','n','pearson_r','spearman_brown']].to_string(index=False))

# Test convergence between phenomenological tracking SD and computational
# sensory uncertainty; these constructs should not be equated by definition.
checks=[]
for proxy,model_par in [('tracking_sd_deg_Audio','sigma_eA'),
                        ('tracking_sd_deg_Visual','sigma_eV'),
                        ('localization_bias_deg_Audio','sigma_eA')]:
    if proxy in zpeople and model_par in zpeople:
        checks.append(zcorr(zpeople,proxy,model_par,f'{proxy}_vs_{model_par}'))
zsave(pd.DataFrame(checks),'Z1_behavioral_noise_vs_BCI_likelihood_parameters.csv')

CV columns: ['subject', 'model', 'n_heldout', 'log_loss', 'brier']
            model  n  mean_subject_logloss       sd
        gaussian3 50              0.459837 0.112119
        gaussian4 50              0.460150 0.112694
logistic_shifted4 50              0.466821 0.113864
logistic_shifted5 50              0.471214 0.113587
     logistic_abs 50              0.475640 0.103979
       probit_abs 50              0.482936 0.094953
  probit_shifted4 50              0.497552 0.127463
  probit_shifted5 50              0.506170 0.132095
        gaussian2 50              0.865531 0.484543
Unimodal split-half reliability:                      analysis  n  pearson_r  spearman_brown
 Audio_mean_trial_tracking_sd 50   0.766915        0.868084
        Audio_mean_trial_bias 50   0.377156        0.547731
Visual_mean_trial_tracking_sd 50   0.748141        0.855928
       Visual_mean_trial_bias 50   0.250376        0.400481


""


In [104]:
# %% CELL Z2 — EXPLANATORY COMPARISONS: PRIOR / PRECISION / COMBINED
# These are observational prediction benchmarks, NOT generative BCI ablations.
# Participant is the unit of holdout and of inferential uncertainty.

def z_lopo_models(df,outcome,predictor_sets,prefix):
    outcomes=[];fits=[]
    for name,predictors in predictor_sets.items():
        if predictors and not set(predictors).issubset(df.columns):
            print(f'SKIP {name}: missing {set(predictors)-set(df.columns)}');continue
        cols=['subject',outcome]+predictors
        work=df[cols].replace([np.inf,-np.inf],np.nan).dropna().copy()
        if len(work)<15 or work[outcome].std()<1e-9:continue
        y=work[outcome].to_numpy(float)
        predictions=np.full(len(work),np.nan)
        for train,test in LeaveOneGroupOut().split(work,y,groups=work.subject):
            if not predictors:
                predictions[test]=float(np.mean(y[train]))
            else:
                algorithm=make_pipeline(StandardScaler(),Ridge(alpha=3.0))
                algorithm.fit(work.iloc[train][predictors],y[train])
                predictions[test]=algorithm.predict(work.iloc[test][predictors])
        work['prediction']=predictions
        work['model']=name
        outcomes.append(work[['subject',outcome,'prediction','model']])
        fits.append({'model':name,'outcome':outcome,'n':len(work),
                     'rmse':np.sqrt(mean_squared_error(y,predictions)),
                     'mae':float(np.mean(np.abs(y-predictions)))})
    if outcomes:
        preds=pd.concat(outcomes,ignore_index=True)
        zsave(preds,f'{prefix}_participant_holdout_predictions.csv')
        zsave(pd.DataFrame(fits),f'{prefix}_heldout_scores.csv')
        # Compare loss differences ONLY where both models predicted the same people.
        wide=preds.assign(error_sq=lambda d:(d[outcome]-d.prediction)**2)
        wide=wide.pivot(index='subject',columns='model',values='error_sq')
        pairs=[]
        for name in predictor_sets:
            if name=='baseline' or name not in wide or 'baseline' not in wide:continue
            both=wide[['baseline',name]].dropna()
            if len(both)<8:continue
            delta=both.baseline-both[name]
            rng=np.random.default_rng(ZSEED);boot=[]
            a=delta.to_numpy(float)
            for _ in range(2000):boot.append(a[rng.integers(len(a),size=len(a))].mean())
            pairs.append({'comparison':name+'_vs_baseline','n':len(a),
                          'mean_MSE_improvement':a.mean(),
                          'ci_low':np.percentile(boot,2.5),'ci_high':np.percentile(boot,97.5)})
        zsave(pd.DataFrame(pairs),f'{prefix}_paired_predictive_improvements.csv')
        return pd.DataFrame(fits)
    return pd.DataFrame()

# Same complete cases are enforced for the planned three families, so an
# improvement cannot be obtained merely by switching the evaluated participants.
for outcome in ['delta50_deg','bci_pC20']:
    if outcome not in zpeople:continue
    features=['p_common','log_sigma_eA','log_sigma_p']
    if not all(k in zpeople for k in features):
        print('Cannot run full parameter comparison, missing:',set(features)-set(zpeople.columns));continue
    sample=zpeople[['subject',outcome]+features].replace([np.inf,-np.inf],np.nan).dropna()
    if len(sample)<15:continue
    comparisons={
       'baseline':[],
       'causal_prior':['p_common'],
       'sensory_and_spatial_uncertainty':['log_sigma_eA','log_sigma_p'],
       'combined':['p_common','log_sigma_eA','log_sigma_p'],
    }
    print('\nHeld-out model comparison:',outcome,'N=',len(sample))
    print(z_lopo_models(sample,outcome,comparisons,'Z2_'+outcome).to_string(index=False))
    # Standardized, participant-level HC3 coefficients: exploratory association.
    X=sample[features].apply(lambda v:(v-v.mean())/v.std(ddof=0))
    X=sm.add_constant(X)
    regression=sm.OLS(sample[outcome],X).fit(cov_type='HC3')
    (ZOUT/f'Z2_{outcome}_explanatory_HC3.txt').write_text(str(regression.summary()),encoding='utf-8')
    print('HC3 p-values:',regression.pvalues.to_dict())

# Parameter bounds, optimizer restarts, and parameter tradeoffs (not recovery).
starts=zload('bci_restarts')
if not starts.empty:
    cols=[c for c in ['p_common','sigma_eA','sigma_p','tau_p','kappa'] if c in starts]
    for c in cols:starts[c]=pd.to_numeric(starts[c],errors='coerce')
    spread=starts.groupby('subject')[cols].agg(['std','min','max','count'])
    spread.columns=[f'{a}_{b}' for a,b in spread.columns]
    zsave(spread.reset_index(),'Z2_multistart_parameter_spread.csv')
    # Multistart stability does not establish recovery: simulated truth is unknown.
    print('Saved multistart dispersion; independent simulation-based parameter recovery still required.')


Held-out model comparison: delta50_deg N= 46
                          model     outcome  n      rmse      mae
                       baseline delta50_deg 46 15.921509 9.437256
                   causal_prior delta50_deg 46 16.016022 9.150751
sensory_and_spatial_uncertainty delta50_deg 46 16.193869 9.743274
                       combined delta50_deg 46 16.305984 9.449508
HC3 p-values: {'const': 1.2005725588206925e-19, 'p_common': 0.29958642989775663, 'log_sigma_eA': 0.6135941634284463, 'log_sigma_p': 0.928825833224097}

Held-out model comparison: bci_pC20 N= 42
                          model  outcome  n     rmse      mae
                       baseline bci_pC20 42 0.270482 0.224859
                   causal_prior bci_pC20 42 0.193951 0.153242
sensory_and_spatial_uncertainty bci_pC20 42 0.266455 0.216414
                       combined bci_pC20 42 0.166753 0.131672
HC3 p-values: {'const': 5.195060717547204e-75, 'p_common': 1.219674782434294e-18, 'log_sigma_eA': 3.6103238700647874e-05

In [106]:
# %% CELL Z3 — EXPLICIT/IMPLICIT READOUT TEST WITH CONTROLLED DISPARITY
# Uses cross-fitted *report* probability and separate model-fitted parameters;
# cannot infer mechanistic independence from a regression alone.
ztr=zload('explicit_trials',True)
zcap=zload('capture',True)
needed={'subject','trialIndex','cf_p_same_hat','signed_disparity_deg'}
if not needed.issubset(ztr):raise KeyError(f'Aim1 trials missing {needed-set(ztr)}')
needed={'subject','trialIndex','VisualCapture_audioLag','offsetDeg'}
if not needed.issubset(zcap):raise KeyError(f'Capture missing {needed-set(zcap)}')
for t in [ztr,zcap]:t['trialIndex']=pd.to_numeric(t.trialIndex,errors='coerce')
zcap['offset_deg']=pd.to_numeric(zcap.offsetDeg,errors='coerce').abs()
flags=(zcap.get('trialType',pd.Series('',index=zcap.index)).astype(str)+' '+
       zcap.get('condLabel',pd.Series('',index=zcap.index)).astype(str)).str.lower()
zcap=zcap[zcap.offset_deg.isin([10,20,30,40])&
          ~flags.str.contains('sweep|diverge|converge')].copy()
if ztr.duplicated(['subject','trialIndex']).any():raise ValueError('Duplicate explicit trials')
if zcap.duplicated(['subject','trialIndex']).any():raise ValueError('Duplicate capture trials')
ztrial=(zcap.merge(ztr[['subject','trialIndex','cf_p_same_hat','signed_disparity_deg']],
                   on=['subject','trialIndex'],how='inner',validate='one_to_one')
              .merge(zpeople[['subject']+[c for c in ['p_common','log_sigma_eA','log_sigma_p','delta50_deg']
                                         if c in zpeople]],on='subject',how='left',validate='many_to_one'))
ztrial=ztrial.dropna(subset=['VisualCapture_audioLag','cf_p_same_hat','offset_deg']).copy()
ztrial['report_subject_mean']=ztrial.groupby('subject').cf_p_same_hat.transform('mean')
ztrial['report_within']=ztrial.cf_p_same_hat-ztrial.report_subject_mean
zsave(ztrial,'Z3_capture_crossfitted_trial_dataset.csv')
# Same complete trial sample for the mechanistic-parameter sensitivity models.
zgee_rows=[]
formulas={
  'physical_only':'VisualCapture_audioLag ~ C(offset_deg)',
  'physical_plus_report':'VisualCapture_audioLag ~ C(offset_deg) + report_within + report_subject_mean',
  'physical_plus_prior':'VisualCapture_audioLag ~ C(offset_deg) + p_common',
  'physical_plus_precision':'VisualCapture_audioLag ~ C(offset_deg) + log_sigma_eA + log_sigma_p',
  'physical_plus_combined':'VisualCapture_audioLag ~ C(offset_deg) + p_common + log_sigma_eA + log_sigma_p',
}
complete=ztrial.dropna(subset=['p_common','log_sigma_eA','log_sigma_p','report_within','report_subject_mean']).copy()
print('Repeated-measures matched sample:',len(complete),'trials,',complete.subject.nunique(),'participants')
for label,formula in formulas.items():
    if complete.subject.nunique()<12:continue
    fit=smf.gee(formula,groups='subject',data=complete,
                family=sm.families.Gaussian(),cov_struct=Exchangeable()).fit()
    (ZOUT/f'Z3_{label}_GEE.txt').write_text(str(fit.summary()),encoding='utf-8')
    for term in fit.params.index:
        zgee_rows.append({'model':label,'term':term,'beta':fit.params[term],
                          'se_cluster':fit.bse[term],'p':fit.pvalues[term],
                          'n_subjects':complete.subject.nunique(),'n_trials':len(complete)})
zsave(pd.DataFrame(zgee_rows),'Z3_capture_GEE_model_family.csv')
print('These are clustered observational associations, not BCI model ablations.')

Repeated-measures matched sample: 3632 trials, 46 participants
These are clustered observational associations, not BCI model ablations.


In [108]:
# %% CELL Z4 — EXPORT ORIGINAL RAW SL TRAJECTORIES FOR EXISTING AIM3 SPATIAL QC
# Existing Aim3 already provides strict match, hysteresis, split-half and lag code.
# This cell supplies OBS_SL_TIME_CSV without editing Aim3. Run BEFORE re-running Aim3.
from glob import glob
Z_MASTER_DIR=ZROOT/'individual_data_SL'
Z_OBS_SL_TIME_CSV=ZOUT/'Z4_original_SL_observed_trajectories.csv'
files=sorted(glob(str(Z_MASTER_DIR/'*master_trials*.csv')))
if not files:
    print('RAW SL MASTER FILES NOT FOUND:',Z_MASTER_DIR)
    print('This cell will not fabricate cursor trajectories. Supply *master_trials*.csv files.')
else:
    parts=[];audit=[]
    for file in files:
        subject_match=re.search(r'(pid\d+)',Path(file).name,re.I)
        if not subject_match:continue
        sub=subject_match.group(1).lower()
        if sub not in set(zpeople.subject):continue
        raw=ZREAD(file)
        req={'trialIndex','trialType','condLabel','t','mouse_x','audio_x','blob_x'}
        if not req.issubset(raw.columns):
            audit.append({'subject':sub,'status':'missing_columns','detail':str(req-set(raw.columns))})
            continue
        if 'isPractice' in raw:raw=raw[pd.to_numeric(raw.isPractice,errors='coerce').eq(0)]
        raw=raw[raw.trialType.astype(str).str.lower().isin(
            ['congruent','incongruent_spatial','incongruent_sweep','audio_only','visual_only'])].copy()
        if raw.empty:continue
        raw['subject']=sub
        # Condition labels contain diverge/converge, unlike generic trialType.
        raw['condition']=raw.condLabel.astype(str)
        raw['trial']=pd.to_numeric(raw.trialIndex,errors='coerce')
        raw['visual_x']=pd.to_numeric(raw.blob_x,errors='coerce')
        for col in ['t','mouse_x','audio_x']:raw[col]=pd.to_numeric(raw[col],errors='coerce')
        raw['t']=raw.t-raw.groupby('trial').t.transform('min')
        # Retain original sample times. Do not rescale cursor/stimulus units.
        rows=raw[['subject','trial','condition','t','mouse_x','audio_x','visual_x']].dropna(subset=['trial','t'])
        dupe=rows.duplicated(['subject','trial','t']).sum()
        audit.append({'subject':sub,'status':'ok' if dupe==0 else 'duplicate_time',
                      'detail':f'{len(rows)} rows; duplicates={dupe}',
                      'n_trials':rows.trial.nunique()})
        if dupe:continue
        parts.append(rows)
    zsave(pd.DataFrame(audit),'Z4_raw_SL_export_QC.csv')
    if parts:
        rawout=pd.concat(parts,ignore_index=True)
        zsave(rawout,Z_OBS_SL_TIME_CSV.name)
        print('Original trajectories exported:',Z_OBS_SL_TIME_CSV,'Rows:',len(rawout))
        print('FOR YOUR EXISTING AIM3 CELL 110: set OBS_SL_TIME_CSV = r\''+str(Z_OBS_SL_TIME_CSV)+'\'')
    else:print('No valid original SL trajectories exported. Inspect Z4_raw_SL_export_QC.csv.')

Original trajectories exported: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Zhu2026_AddOns\Z4_original_SL_observed_trajectories.csv Rows: 11267685
FOR YOUR EXISTING AIM3 CELL 110: set OBS_SL_TIME_CSV = r'C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Zhu2026_AddOns\Z4_original_SL_observed_trajectories.csv'


In [110]:
# %% CELL Z5 — STRICT OBSERVED-VERSUS-MODEL SPATIAL HYSTERESIS (EXISTING AIM3 HELPERS)
# Run AFTER re-running existing Aim3 with OBS_SL_TIME_CSV configured.
# Leverages original strict trajectory matcher and original motor/lag routines.
Z_SPATIAL_BASE=ZROOT/'Individual_differences_analyses'
Z_SPATIAL_NAME=('Aim_3_gaussian_explicit_implicit_mapping_full50_dynamicBCI' if
    (Z_SPATIAL_BASE/'Aim_3_gaussian_explicit_implicit_mapping_full50_dynamicBCI'/'spatial_csv'/
     '05_observed_model_trial_matching_audit.csv').exists() else
    'Aim_3_gaussian_explicit_implicit_mapping_inrange45_dynamicBCI')
Z_SPATIAL_RESULTS=Z_SPATIAL_BASE/Z_SPATIAL_NAME/'spatial_csv'
files_to_check={
  'trial_audit':'05_observed_model_trial_matching_audit.csv',
  'subject_correlation':'22_strict_trial_matched_raw_primary_correlation.csv',
}
for label,fn in files_to_check.items():
    path=Z_SPATIAL_RESULTS/fn
    print(label,'AVAILABLE' if path.exists() else 'NOT FOUND',path)
if (Z_SPATIAL_RESULTS/files_to_check['trial_audit']).exists():
    qa=ZREAD(Z_SPATIAL_RESULTS/files_to_check['trial_audit'])
    if 'strict_trial_match' in qa:
        ok=qa.strict_trial_match.astype(str).str.lower().eq('true')
        print('Strict-matched SL trials:',ok.sum(),'; participants:',qa.loc[ok,'subject'].nunique())
        print(qa.loc[~ok,'why_not_strict_match'].value_counts(dropna=False).to_string())
        qa['strict_trial_match']=ok
        zsave(qa.groupby('subject',as_index=False).agg(
             n_trial_rows=('trial','size'),n_strict=('strict_trial_match','sum')),
             'Z5_trial_alignment_summary.csv')
print('Do not interpret spatial model comparisons until strict matching succeeds.')

trial_audit NOT FOUND C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_3_gaussian_explicit_implicit_mapping_inrange45_dynamicBCI\spatial_csv\05_observed_model_trial_matching_audit.csv
subject_correlation NOT FOUND C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Continuous AVT v1\Individual_differences_analyses\Aim_3_gaussian_explicit_implicit_mapping_inrange45_dynamicBCI\spatial_csv\22_strict_trial_matched_raw_primary_correlation.csv
Do not interpret spatial model comparisons until strict matching succeeds.


In [112]:
# %% CELL Z6 — OPTIONAL OUT-OF-SESSION CI VALIDATION: NO DATA LEAKAGE
# CI continuous judgment and CI model posterior must come from the SAME CI trial
# and time samples, with SL model parameters frozen. Input files not supplied yet.
Z_CI_OBS_FILE = None  # path: subject,trial,t,reported_p_common (0..1)
Z_CI_MODEL_FILE = None # path: subject,trial,t,pC1; SL-fitted forward pass on CI stimuli
if Z_CI_OBS_FILE is None or Z_CI_MODEL_FILE is None:
    zsave(pd.DataFrame([{'status':'NOT_RUN','requires':'CI observed time series + SL-fitted BCI CI forward predictions'}]),
          'Z6_CI_cross_session_status.csv')
    print('CI cross-session validation not run: supply both CI time-series files.')
else:
    obs=ZREAD(Z_CI_OBS_FILE);mdl=ZREAD(Z_CI_MODEL_FILE)
    needed_obs={'subject','trial','t','reported_p_common'}
    needed_model={'subject','trial','t','pC1'}
    if not needed_obs.issubset(obs):raise KeyError(needed_obs-set(obs.columns))
    if not needed_model.issubset(mdl):raise KeyError(needed_model-set(mdl.columns))
    for d in [obs,mdl]:
        d['subject']=zid(d.subject)
        for c in ['trial','t']:d[c]=pd.to_numeric(d[c],errors='coerce')
    obs['reported_p_common']=pd.to_numeric(obs.reported_p_common,errors='coerce')
    mdl['pC1']=pd.to_numeric(mdl.pC1,errors='coerce')
    # True time-grid alignment, never merge rounded timestamps into invented matches.
    joined=[];qc=[]
    model_groups={(s,int(t)):g.sort_values('t') for (s,t),g in mdl.groupby(['subject','trial']) if pd.notna(t)}
    for (s,tr),g in obs.groupby(['subject','trial']):
        if pd.isna(tr) or (s,int(tr)) not in model_groups:continue
        m=model_groups[s,int(tr)].dropna(subset=['t','pC1']).drop_duplicates('t')
        if len(m)<5:continue
        q=g.sort_values('t').copy()
        query=q.t.to_numpy(float);mt=m.t.to_numpy(float)
        nearest=np.minimum(np.abs(query-mt[np.clip(np.searchsorted(mt,query)-1,0,len(mt)-1)]),
                           np.abs(query-mt[np.clip(np.searchsorted(mt,query),0,len(mt)-1)]))
        valid=(query>=mt.min())&(query<=mt.max())&(nearest<=.10)
        q['model_pC1']=np.interp(query,mt,m.pC1.to_numpy(float),left=np.nan,right=np.nan)
        q.loc[~valid,'model_pC1']=np.nan
        qc.append({'subject':s,'trial':tr,'n_obs':len(q),'n_time_matched':valid.sum()})
        joined.append(q)
    zsave(pd.DataFrame(qc),'Z6_CI_trial_time_match_audit.csv')
    if joined:
        data=pd.concat(joined,ignore_index=True).dropna(subset=['reported_p_common','model_pC1'])
        data=data[data.reported_p_common.between(0,1)&data.model_pC1.between(0,1)]
        # Subject-first continuous correspondence (not posterior calibration).
        scores=[]
        for s,g in data.groupby('subject'):
            if len(g)<50:continue
            scores.append({'subject':s,'n_samples':len(g),
                  'corr':g.reported_p_common.corr(g.model_pC1),
                  'MAE_raw':np.mean(abs(g.reported_p_common-g.model_pC1))})
        zsave(pd.DataFrame(scores),'Z6_CI_independent_session_correspondence.csv')
        print('CI participants with matched continuous predictions:',len(scores))
    else:print('No CI time-aligned trials; examine trial IDs and time units.')

CI cross-session validation not run: supply both CI time-series files.


In [114]:
# %% CELL Z7 — OPTIONAL COUNTERFACTUAL MEMORY-ABLATION COMPARISONS
# This evaluates supplied forward simulations; it CANNOT create them from
# parameter CSVs alone. Requires already-executed original Aim 3 cell 110.
Z_ABLATION_FILES = {
 'no_causal_prior_adaptation': None,  # simulator forward pass, same fitted params otherwise
 'no_spatial_persistence': None,
 'no_both_memory_components': None,
}
status=[]
if not (Z_SPATIAL_RESULTS/'05_observed_model_trial_matching_audit.csv').exists():
    status.append({'control':'ALL','status':'MISSING_STRICT_TRIAL_AUDIT'})
elif not all(k in globals() for k in ['_read_model_spatial','_capture_bin_rows',
                                         '_spatial_predictions_pair']):
    status.append({'control':'ALL','status':'RUN_EXISTING_AIM3_FIRST'})
else:
    q=ZREAD(Z_SPATIAL_RESULTS/'05_observed_model_trial_matching_audit.csv')
    q['strict']=q.strict_trial_match.astype(str).str.lower().eq('true')
    verified=set(map(tuple,q.loc[q.strict,['subject','trial']].to_numpy()))
    full= _read_model_spatial(ZFILES['bci_time'])
    primary=Z_SPATIAL_RESULTS/'21_strict_trial_matched_raw_individual_hysteresis.csv'
    full_scores=ZREAD(primary) if primary.exists() else pd.DataFrame()
    for label,path in Z_ABLATION_FILES.items():
        if path is None or not Path(path).exists():
            status.append({'control':label,'status':'NOT_RUN_ABLATION_FORWARD_EXPORT_REQUIRED'})
            continue
        ctrl=_read_model_spatial(Path(path))
        # Reject a control that changes the trial's physical input trajectories.
        a=ctrl[ctrl[['subject','trial']].apply(tuple,axis=1).isin(verified)].copy()
        b=full[full[['subject','trial']].apply(tuple,axis=1).isin(verified)].copy()
        a['time_round']=a.t.round(3);b['time_round']=b.t.round(3)
        stim=a.merge(b[['subject','trial','time_round','audio_x','visual_x']],
                     on=['subject','trial','time_round'],suffixes=('_ctrl','_full'))
        if len(stim)<100:
            status.append({'control':label,'status':'INSUFFICIENT_IDENTICAL_TIMEPOINTS'})
            continue
        stim['err']=((stim.audio_x_ctrl-stim.audio_x_full)**2+
                     (stim.visual_x_ctrl-stim.visual_x_full)**2)/2
        err=stim.groupby(['subject','trial']).err.mean().pow(.5)
        verified_control={key for key,v in err.items() if v<.01}
        if len(verified_control)<10:
            status.append({'control':label,'status':'STIMULUS_MISMATCH',
                           'n_verified_trials':len(verified_control)})
            continue
        obs_path=Z_OBS_SL_TIME_CSV
        if not Path(obs_path).exists():
            status.append({'control':label,'status':'OBSERVED_RAW_SL_MISSING'})
            continue
        obs=obs_sl if 'obs_sl' in globals() else _read_raw_observed(obs_path)
        observed=_capture_bin_rows(obs,'mouse_x','observed_'+label,verified_control)
        predicted=_capture_bin_rows(ctrl,'loc_pred','model_'+label,verified_control)
        full_predicted=_capture_bin_rows(full,'loc_pred','full_'+label,verified_control)
        if observed.empty or predicted.empty or full_predicted.empty:
            status.append({'control':label,'status':'NO_VALID_CAPTURE_BINS'})
            continue
        cmp=_spatial_predictions_pair(observed,predicted,'raw','ablation_'+label)
        full_matched=_spatial_predictions_pair(observed,full_predicted,'raw','full_same_trials_'+label)
        if cmp is None or cmp.empty or full_matched is None or full_matched.empty:
            status.append({'control':label,'status':'NO_IDENTIFIABLE_HYSTERESIS'})
            continue
        status.append({'control':label,'status':'COMPLETED',
                       'n_subjects':len(cmp),'n_verified_trials':len(verified_control)})
        # Full model and ablation are evaluated using the SAME original trials.
        pairing=cmp[['subject','observed_hysteresis_deg','BCI_PredictedSpatial_Hysteresis']].rename(
            columns={'BCI_PredictedSpatial_Hysteresis':'ablation_prediction'}).merge(
            full_matched[['subject','BCI_PredictedSpatial_Hysteresis']].rename(
                columns={'BCI_PredictedSpatial_Hysteresis':'full_prediction'}),
            on='subject',how='inner',validate='one_to_one')
        pairing['full_squared_error']=(pairing.full_prediction-pairing.observed_hysteresis_deg)**2
        pairing['ablated_squared_error']=(pairing.ablation_prediction-pairing.observed_hysteresis_deg)**2
        pairing['improvement_full_over_ablation']=pairing.ablated_squared_error-pairing.full_squared_error
        zsave(pairing,'Z7_'+label+'_participant_comparison.csv')
        print(label,'matched participants',len(pairing),
              'mean full-model MSE advantage:',pairing.improvement_full_over_ablation.mean())
zsave(pd.DataFrame(status),'Z7_counterfactual_ablation_status.csv')
print('Counterfactual ablations require real forward simulations, not fabricated parameter manipulations.')

Counterfactual ablations require real forward simulations, not fabricated parameter manipulations.
